<a href="https://colab.research.google.com/github/SANGHATI23/genomic-evidence-reliability/blob/main/04_GES_locked_temporal_validation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==================================================================================================
# STAGE 6A — STEP 1
# VERIFY FROZEN STAGE 4 SCORES AND FROZEN STAGE 5 OUTCOMES
#
# Purpose:
#   1. Mount Google Drive.
#   2. Cryptographically verify the frozen Stage 4 models, score table,
#      and Stage 4C manifest.
#   3. Cryptographically verify the frozen Stage 5 outcome table and
#      Stage 5 freeze manifest.
#   4. Validate the complete Stage 4 score-table schema and score integrity.
#   5. Confirm exact one-to-one T0 key compatibility between scores and outcomes.
#
# Scientific boundary:
#   - The Stage 5 binary outcome values are not loaded in this cell.
#   - No score-outcome dataframe is created.
#   - No comparator is constructed.
#   - No AUPRC, AUROC, calibration, enrichment, or subgroup result is calculated.
#   - No artifact is written or modified.
# ==================================================================================================

from google.colab import drive

drive.mount(
    "/content/drive"
)

from pathlib import Path
import hashlib
import json
import re

import numpy as np
import pandas as pd
import pyarrow.parquet as pq


# --------------------------------------------------------------------------------------------------
# 1. Frozen project paths
# --------------------------------------------------------------------------------------------------

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/GES_RAG_Temporal_Study"
)

STAGE4_MODEL_DIR = (
    PROJECT_ROOT
    / "models"
    / "stage4_ges"
)

STAGE4_DATA_DIR = (
    PROJECT_ROOT
    / "data_processed"
    / "stage4_ges"
)

STAGE4_CONFIG_DIR = (
    PROJECT_ROOT
    / "configs"
    / "stage4_ges"
)

FULL_MODEL_PATH = (
    STAGE4_MODEL_DIR
    / "stage4c_full_ges_logistic_model_v1.joblib"
)

NO_STAR_MODEL_PATH = (
    STAGE4_MODEL_DIR
    / "stage4c_no_star_ges_logistic_model_v1.joblib"
)

SCORE_TABLE_PATH = (
    STAGE4_DATA_DIR
    / "stage4c_t0_full_and_no_star_ges_scores_v1.parquet"
)

STAGE4_MANIFEST_PATH = (
    STAGE4_CONFIG_DIR
    / "stage4c_ges_model_freeze_manifest_v1.json"
)

STAGE5_OUTCOME_PATH = (
    PROJECT_ROOT
    / "data_processed"
    / "stage5_outcomes"
    / "stage5_primary_future_instability_outcomes_v1.parquet"
)

STAGE5_OUTCOME_SHA256_PATH = (
    STAGE5_OUTCOME_PATH.with_name(
        STAGE5_OUTCOME_PATH.name
        + ".sha256"
    )
)

STAGE5_MANIFEST_PATH = (
    PROJECT_ROOT
    / "configs"
    / "stage5_outcomes"
    / "stage5_future_instability_outcome_freeze_manifest_v1.json"
)

STAGE5_MANIFEST_SHA256_PATH = (
    PROJECT_ROOT
    / "configs"
    / "stage5_outcomes"
    / "stage5_future_instability_outcome_freeze_manifest_v1.sha256"
)


# --------------------------------------------------------------------------------------------------
# 2. Frozen expected checksums
# --------------------------------------------------------------------------------------------------

EXPECTED_FULL_MODEL_SHA256 = (
    "0b4a87b16f768484cbdae168fc226cec5"
    "e978521172fb03510bfb1ea3e78fa30"
)

EXPECTED_NO_STAR_MODEL_SHA256 = (
    "6c3fe4fc7fe8fdde7b8f0f0d608c48e"
    "66a07945effb8c67c6b98d35e1955257c"
)

EXPECTED_SCORE_TABLE_SHA256 = (
    "d871ee9087f83be2b0ee954d283aa922"
    "12a639a35e3ea26d5cf042e83019f5ac"
)

EXPECTED_STAGE4_MANIFEST_SHA256 = (
    "c0d8008a4db80c67f5b1c568ddba3496"
    "b2411b29bce0db1eb20e63f26613d4ee"
)

EXPECTED_STAGE5_OUTCOME_SHA256 = (
    "c5508f5a8518160eef50482fd2c425dc"
    "4dcd9cf8a2fe04856e46760de60efbc8"
)

EXPECTED_STAGE5_MANIFEST_SHA256 = (
    "b70286ebf8aa7751e391dbe7425ad56f"
    "aebd18b67ed3f1a2aee63638b130ccd8"
)

EXPECTED_STAGE5_POLICY_SHA256 = (
    "477b01080b249dce9f042b67251ab93a"
    "999a1373e5f01d8f57b5812d67a57c4e"
)


# --------------------------------------------------------------------------------------------------
# 3. Frozen dimensions and model accounting
# --------------------------------------------------------------------------------------------------

EXPECTED_SCORE_ROWS = 71_659
EXPECTED_SCORE_COLUMNS = 21

EXPECTED_OUTCOME_ROWS = 71_659
EXPECTED_OUTCOME_COLUMNS = 53

EXPECTED_FULL_TRAINING_ROWS = 67_565
EXPECTED_NO_STAR_TRAINING_ROWS = 63_148

EXPECTED_SCORE_COLUMNS_IN_ORDER = [
    "t0_row_order",
    "rcv_accession",
    "variation_id",
    "vcv_accession",
    "target_genes_json",
    "study_scope",

    "full_training_eligible",
    "full_weak_label_binary",
    "full_weak_stability_probability",

    "no_star_training_eligible",
    "no_star_weak_label_binary",
    "no_star_weak_stability_probability",

    "full_ges_p_stable_t0",
    "full_ges_predicted_stable_at_0_5",

    "no_star_ges_p_stable_t0",
    "no_star_ges_predicted_stable_at_0_5",

    "model_decision_threshold",

    "t1_information_used",
    "future_instability_outcome_created",
    "temporal_performance_evaluated",

    "stage4c_version",
]


# --------------------------------------------------------------------------------------------------
# 4. Helpers
# --------------------------------------------------------------------------------------------------

def calculate_sha256(
    file_path: Path,
    block_size: int = 8 * 1024 * 1024,
) -> str:
    """
    Calculate SHA-256 without loading the whole file into memory.
    """

    digest = hashlib.sha256()

    with file_path.open("rb") as file_handle:
        for block in iter(
            lambda: file_handle.read(
                block_size
            ),
            b"",
        ):
            digest.update(
                block
            )

    return digest.hexdigest()


def normalize_rcv(
    series: pd.Series,
) -> pd.Series:
    """
    Normalize RCV accessions for key-integrity comparison.
    """

    return (
        series.astype("string")
        .str.strip()
        .str.upper()
        .replace("", pd.NA)
    )


def sidecar_matches(
    sidecar_path: Path,
    expected_sha256: str,
    target_filename: str,
) -> bool:
    """
    Verify the exact SHA-256 sidecar representation.
    """

    if not sidecar_path.exists():
        return False

    observed_text = (
        sidecar_path.read_text(
            encoding="utf-8"
        )
        .strip()
    )

    expected_text = (
        f"{expected_sha256}  "
        f"{target_filename}"
    )

    return (
        observed_text
        == expected_text
    )


# --------------------------------------------------------------------------------------------------
# 5. Verify all required artifacts exist
# --------------------------------------------------------------------------------------------------

required_paths = [
    FULL_MODEL_PATH,
    NO_STAR_MODEL_PATH,
    SCORE_TABLE_PATH,
    STAGE4_MANIFEST_PATH,
    STAGE5_OUTCOME_PATH,
    STAGE5_OUTCOME_SHA256_PATH,
    STAGE5_MANIFEST_PATH,
    STAGE5_MANIFEST_SHA256_PATH,
]

missing_paths = [
    path
    for path in required_paths
    if not path.exists()
]

if missing_paths:
    raise FileNotFoundError(
        "Required frozen artifacts were not found:\n"
        + "\n".join(
            f" - {path}"
            for path in missing_paths
        )
    )


# --------------------------------------------------------------------------------------------------
# 6. Calculate observed checksums
# --------------------------------------------------------------------------------------------------

observed_full_model_sha256 = (
    calculate_sha256(
        FULL_MODEL_PATH
    )
)

observed_no_star_model_sha256 = (
    calculate_sha256(
        NO_STAR_MODEL_PATH
    )
)

observed_score_table_sha256 = (
    calculate_sha256(
        SCORE_TABLE_PATH
    )
)

observed_stage4_manifest_sha256 = (
    calculate_sha256(
        STAGE4_MANIFEST_PATH
    )
)

observed_stage5_outcome_sha256 = (
    calculate_sha256(
        STAGE5_OUTCOME_PATH
    )
)

observed_stage5_manifest_sha256 = (
    calculate_sha256(
        STAGE5_MANIFEST_PATH
    )
)


# --------------------------------------------------------------------------------------------------
# 7. Print cryptographic verification
# --------------------------------------------------------------------------------------------------

print("=" * 124)
print("STAGE 6A STEP 1 — FROZEN SCORE AND OUTCOME INPUT VERIFICATION")
print("=" * 124)

print("\nCRYPTOGRAPHIC VERIFICATION")
print("-" * 124)

cryptographic_checks = {
    "Stage 4 full GES model":
        observed_full_model_sha256
        == EXPECTED_FULL_MODEL_SHA256,

    "Stage 4 no-star GES model":
        observed_no_star_model_sha256
        == EXPECTED_NO_STAR_MODEL_SHA256,

    "Stage 4 T0 score table":
        observed_score_table_sha256
        == EXPECTED_SCORE_TABLE_SHA256,

    "Stage 4C freeze manifest":
        observed_stage4_manifest_sha256
        == EXPECTED_STAGE4_MANIFEST_SHA256,

    "Stage 5 outcome table":
        observed_stage5_outcome_sha256
        == EXPECTED_STAGE5_OUTCOME_SHA256,

    "Stage 5 outcome sidecar":
        sidecar_matches(
            STAGE5_OUTCOME_SHA256_PATH,
            EXPECTED_STAGE5_OUTCOME_SHA256,
            STAGE5_OUTCOME_PATH.name,
        ),

    "Stage 5 freeze manifest":
        observed_stage5_manifest_sha256
        == EXPECTED_STAGE5_MANIFEST_SHA256,

    "Stage 5 manifest sidecar":
        sidecar_matches(
            STAGE5_MANIFEST_SHA256_PATH,
            EXPECTED_STAGE5_MANIFEST_SHA256,
            STAGE5_MANIFEST_PATH.name,
        ),
}

for check_name, passed in (
    cryptographic_checks.items()
):
    print(
        f"{check_name:<55} "
        f"{'PASS' if passed else 'FAIL'}"
    )


# --------------------------------------------------------------------------------------------------
# 8. Load and validate Stage 4C freeze manifest
# --------------------------------------------------------------------------------------------------

stage4_manifest = json.loads(
    STAGE4_MANIFEST_PATH.read_text(
        encoding="utf-8"
    )
)

stage4_score_record = (
    stage4_manifest
    .get(
        "frozen_outputs",
        {},
    )
    .get(
        "t0_ges_score_table",
        {},
    )
)

stage4_full_model_record = (
    stage4_manifest
    .get(
        "frozen_outputs",
        {},
    )
    .get(
        "full_ges_model",
        {},
    )
)

stage4_no_star_model_record = (
    stage4_manifest
    .get(
        "frozen_outputs",
        {},
    )
    .get(
        "no_star_ges_model",
        {},
    )
)

stage4_boundary = (
    stage4_manifest.get(
        "scientific_boundary",
        {},
    )
)

stage4_manifest_checks = {
    "manifest name":
        stage4_manifest.get(
            "manifest_name"
        )
        == "Stage 4C GES model freeze manifest",

    "manifest version":
        stage4_manifest.get(
            "version"
        )
        == "1.0.0",

    "manifest status":
        stage4_manifest.get(
            "manifest_status"
        )
        == "STAGE4C_FULL_AND_NO_STAR_GES_MODELS_FROZEN",

    "validation decision":
        stage4_manifest.get(
            "validation_decision"
        )
        == "PASS_STAGE4C_GES_MODELS_ACCEPTED_AND_FROZEN",

    "score checksum lineage":
        stage4_score_record.get(
            "sha256"
        )
        == EXPECTED_SCORE_TABLE_SHA256,

    "score row lineage":
        stage4_score_record.get(
            "rows"
        )
        == EXPECTED_SCORE_ROWS,

    "score column lineage":
        stage4_score_record.get(
            "columns"
        )
        == EXPECTED_SCORE_COLUMNS,

    "full-model checksum lineage":
        stage4_full_model_record.get(
            "sha256"
        )
        == EXPECTED_FULL_MODEL_SHA256,

    "no-star checksum lineage":
        stage4_no_star_model_record.get(
            "sha256"
        )
        == EXPECTED_NO_STAR_MODEL_SHA256,

    "full training rows":
        stage4_full_model_record.get(
            "training_rows"
        )
        == EXPECTED_FULL_TRAINING_ROWS,

    "no-star training rows":
        stage4_no_star_model_record.get(
            "training_rows"
        )
        == EXPECTED_NO_STAR_TRAINING_ROWS,

    "no-star excludes review status":
        stage4_no_star_model_record.get(
            "review_status_used"
        )
        is False,

    "T1 information not used":
        stage4_boundary.get(
            "t1_information_used"
        )
        is False,

    "future outcome not used":
        stage4_boundary.get(
            "future_instability_outcome_created"
        )
        is False,

    "temporal performance not examined":
        stage4_boundary.get(
            "temporal_performance_examined"
        )
        is False,

    "full model frozen":
        stage4_boundary.get(
            "full_ges_fitted_and_frozen"
        )
        is True,

    "no-star model frozen":
        stage4_boundary.get(
            "no_star_ges_fitted_and_frozen"
        )
        is True,
}

print("\nSTAGE 4C MANIFEST VERIFICATION")
print("-" * 124)

for check_name, passed in (
    stage4_manifest_checks.items()
):
    print(
        f"{check_name:<55} "
        f"{'PASS' if passed else 'FAIL'}"
    )


# --------------------------------------------------------------------------------------------------
# 9. Load and validate Stage 5 freeze manifest
# --------------------------------------------------------------------------------------------------

stage5_manifest = json.loads(
    STAGE5_MANIFEST_PATH.read_text(
        encoding="utf-8"
    )
)

stage5_outcome_record = (
    stage5_manifest
    .get(
        "frozen_stage5_artifacts",
        {},
    )
    .get(
        "primary_future_instability_outcome_table",
        {},
    )
)

stage5_boundary = (
    stage5_manifest.get(
        "scientific_boundary",
        {},
    )
)

stage5_manifest_checks = {
    "manifest identity":
        stage5_manifest.get(
            "manifest_id"
        )
        == (
            "GES_STAGE5_FUTURE_INSTABILITY_"
            "OUTCOME_FREEZE_MANIFEST"
        ),

    "manifest version":
        stage5_manifest.get(
            "version"
        )
        == "1.0.0",

    "manifest status":
        stage5_manifest.get(
            "status"
        )
        == "STAGE5_OUTCOMES_ACCEPTED_AND_FROZEN",

    "outcome checksum lineage":
        stage5_outcome_record.get(
            "sha256"
        )
        == EXPECTED_STAGE5_OUTCOME_SHA256,

    "outcome row lineage":
        stage5_outcome_record.get(
            "rows"
        )
        == EXPECTED_OUTCOME_ROWS,

    "outcome column lineage":
        stage5_outcome_record.get(
            "columns"
        )
        == EXPECTED_OUTCOME_COLUMNS,

    "outcome policy frozen before score join":
        stage5_boundary.get(
            "outcome_policy_frozen_before_score_join"
        )
        is True,

    "no Stage 4 score used in outcome construction":
        stage5_boundary.get(
            "stage4_ges_scores_loaded_during_outcome_construction"
        )
        is False,

    "no Stage 4 score used in outcome freeze":
        stage5_boundary.get(
            "stage4_ges_scores_loaded_during_outcome_freeze"
        )
        is False,

    "no temporal performance examined":
        stage5_boundary.get(
            "temporal_performance_examined"
        )
        is False,

    "outcome rules not modified":
        stage5_boundary.get(
            "outcome_rules_modified_after_policy_freeze"
        )
        is False,
}

print("\nSTAGE 5 MANIFEST VERIFICATION")
print("-" * 124)

for check_name, passed in (
    stage5_manifest_checks.items()
):
    print(
        f"{check_name:<55} "
        f"{'PASS' if passed else 'FAIL'}"
    )


# --------------------------------------------------------------------------------------------------
# 10. Verify Stage 4 score-table metadata and exact schema
# --------------------------------------------------------------------------------------------------

score_parquet = pq.ParquetFile(
    SCORE_TABLE_PATH
)

score_rows = (
    score_parquet.metadata.num_rows
)

score_columns = (
    score_parquet.metadata.num_columns
)

score_row_groups = (
    score_parquet.metadata.num_row_groups
)

score_schema_columns = list(
    score_parquet.schema.names
)

score_schema_exact_match = (
    score_schema_columns
    == EXPECTED_SCORE_COLUMNS_IN_ORDER
)

print("\nSTAGE 4 SCORE-TABLE STRUCTURE")
print("-" * 124)
print(
    f"Rows:                         "
    f"{score_rows:,}"
)
print(
    f"Columns:                      "
    f"{score_columns:,}"
)
print(
    f"Row groups:                   "
    f"{score_row_groups:,}"
)
print(
    f"Exact expected column order:  "
    f"{'PASS' if score_schema_exact_match else 'FAIL'}"
)

if not score_schema_exact_match:

    print("\nOBSERVED SCORE-TABLE COLUMNS")

    for index, column in enumerate(
        score_schema_columns,
        start=1,
    ):
        print(
            f"{index:>2}. {column}"
        )


# --------------------------------------------------------------------------------------------------
# 11. Load score fields required for integrity verification
# --------------------------------------------------------------------------------------------------

score_verification_columns = [
    "t0_row_order",
    "rcv_accession",

    "full_training_eligible",
    "no_star_training_eligible",

    "full_ges_p_stable_t0",
    "full_ges_predicted_stable_at_0_5",

    "no_star_ges_p_stable_t0",
    "no_star_ges_predicted_stable_at_0_5",

    "model_decision_threshold",

    "t1_information_used",
    "future_instability_outcome_created",
    "temporal_performance_evaluated",

    "stage4c_version",
]

scores = pd.read_parquet(
    SCORE_TABLE_PATH,
    columns=score_verification_columns,
)

scores["_rcv_key"] = normalize_rcv(
    scores["rcv_accession"]
)

full_probability = pd.to_numeric(
    scores[
        "full_ges_p_stable_t0"
    ],
    errors="coerce",
)

no_star_probability = pd.to_numeric(
    scores[
        "no_star_ges_p_stable_t0"
    ],
    errors="coerce",
)

threshold = pd.to_numeric(
    scores[
        "model_decision_threshold"
    ],
    errors="coerce",
)


# --------------------------------------------------------------------------------------------------
# 12. Independently verify score integrity
# --------------------------------------------------------------------------------------------------

missing_score_keys = int(
    scores["_rcv_key"]
    .isna()
    .sum()
)

duplicate_score_keys = int(
    scores["_rcv_key"]
    .duplicated(
        keep=False
    )
    .sum()
)

malformed_score_keys = int(
    ~scores["_rcv_key"]
    .fillna("")
    .str.match(
        r"^RCV\d{9}$"
    )
    .sum()
)

# The expression above counts matching rows, so calculate malformed explicitly.
malformed_score_keys = int(
    (
        scores["_rcv_key"]
        .notna()
        &
        ~scores["_rcv_key"]
        .str.match(
            r"^RCV\d{9}$",
            na=False,
        )
    ).sum()
)

missing_full_scores = int(
    full_probability
    .isna()
    .sum()
)

missing_no_star_scores = int(
    no_star_probability
    .isna()
    .sum()
)

full_scores_outside_unit_interval = int(
    (
        full_probability
        .lt(0)
        |
        full_probability
        .gt(1)
    ).sum()
)

no_star_scores_outside_unit_interval = int(
    (
        no_star_probability
        .lt(0)
        |
        no_star_probability
        .gt(1)
    ).sum()
)

threshold_invalid = int(
    (
        threshold.isna()
        |
        ~np.isclose(
            threshold,
            0.5,
            atol=0,
            rtol=0,
        )
    ).sum()
)

full_prediction_mismatch = int(
    (
        scores[
            "full_ges_predicted_stable_at_0_5"
        ]
        .astype(bool)
        .ne(
            full_probability
            .ge(0.5)
        )
    ).sum()
)

no_star_prediction_mismatch = int(
    (
        scores[
            "no_star_ges_predicted_stable_at_0_5"
        ]
        .astype(bool)
        .ne(
            no_star_probability
            .ge(0.5)
        )
    ).sum()
)

full_training_rows = int(
    scores[
        "full_training_eligible"
    ]
    .eq(True)
    .sum()
)

no_star_training_rows = int(
    scores[
        "no_star_training_eligible"
    ]
    .eq(True)
    .sum()
)

t1_used_true = int(
    scores[
        "t1_information_used"
    ]
    .eq(True)
    .sum()
)

future_outcome_present = int(
    scores[
        "future_instability_outcome_created"
    ]
    .eq(True)
    .sum()
)

temporal_performance_present = int(
    scores[
        "temporal_performance_evaluated"
    ]
    .eq(True)
    .sum()
)

observed_stage4_versions = sorted(
    scores[
        "stage4c_version"
    ]
    .dropna()
    .astype(str)
    .unique()
    .tolist()
)

print("\nSTAGE 4 SCORE INTEGRITY")
print("-" * 124)
print(
    f"Missing RCV keys:                       "
    f"{missing_score_keys:,}"
)
print(
    f"Duplicate RCV keys:                     "
    f"{duplicate_score_keys:,}"
)
print(
    f"Malformed RCV keys:                     "
    f"{malformed_score_keys:,}"
)
print(
    f"Missing full-GES probabilities:         "
    f"{missing_full_scores:,}"
)
print(
    f"Missing no-star probabilities:          "
    f"{missing_no_star_scores:,}"
)
print(
    f"Full scores outside [0,1]:              "
    f"{full_scores_outside_unit_interval:,}"
)
print(
    f"No-star scores outside [0,1]:           "
    f"{no_star_scores_outside_unit_interval:,}"
)
print(
    f"Invalid decision thresholds:            "
    f"{threshold_invalid:,}"
)
print(
    f"Full threshold-prediction mismatches:   "
    f"{full_prediction_mismatch:,}"
)
print(
    f"No-star threshold-prediction mismatches:"
    f" {no_star_prediction_mismatch:,}"
)
print(
    f"Full-model training-eligible rows:      "
    f"{full_training_rows:,}"
)
print(
    f"No-star training-eligible rows:         "
    f"{no_star_training_rows:,}"
)
print(
    f"Rows claiming T1 was used:              "
    f"{t1_used_true:,}"
)
print(
    f"Rows claiming future outcomes existed:  "
    f"{future_outcome_present:,}"
)
print(
    f"Rows claiming temporal performance:     "
    f"{temporal_performance_present:,}"
)
print(
    f"Observed Stage 4C versions:             "
    f"{observed_stage4_versions}"
)

print("\nSCORE RANGE CHECK")
print("-" * 124)
print(
    f"Full GES P(stable) range:      "
    f"{full_probability.min():.12f} "
    f"to {full_probability.max():.12f}"
)
print(
    f"No-star GES P(stable) range:   "
    f"{no_star_probability.min():.12f} "
    f"to {no_star_probability.max():.12f}"
)


# --------------------------------------------------------------------------------------------------
# 13. Verify Stage 5 outcome structure without loading outcome labels
# --------------------------------------------------------------------------------------------------

outcome_parquet = pq.ParquetFile(
    STAGE5_OUTCOME_PATH
)

outcome_rows = (
    outcome_parquet.metadata.num_rows
)

outcome_columns = (
    outcome_parquet.metadata.num_columns
)

outcome_schema_columns = list(
    outcome_parquet.schema.names
)

required_outcome_key_columns = {
    "t0_rcv_accession",
    "outcome_policy_version",
    "outcome_policy_sha256",
    "record_level_outcome_assignment_created",
}

missing_outcome_key_columns = sorted(
    required_outcome_key_columns.difference(
        outcome_schema_columns
    )
)

if missing_outcome_key_columns:
    raise RuntimeError(
        "Required frozen outcome provenance fields are missing:\n"
        + "\n".join(
            f" - {column}"
            for column in missing_outcome_key_columns
        )
    )

# Deliberately do not load primary_future_instability or event fields.
outcome_keys = pd.read_parquet(
    STAGE5_OUTCOME_PATH,
    columns=[
        "t0_rcv_accession",
        "outcome_policy_version",
        "outcome_policy_sha256",
        "record_level_outcome_assignment_created",
    ],
)

outcome_keys["_rcv_key"] = normalize_rcv(
    outcome_keys[
        "t0_rcv_accession"
    ]
)

missing_outcome_keys = int(
    outcome_keys["_rcv_key"]
    .isna()
    .sum()
)

duplicate_outcome_keys = int(
    outcome_keys["_rcv_key"]
    .duplicated(
        keep=False
    )
    .sum()
)

outcome_assignment_not_true = int(
    (
        outcome_keys[
            "record_level_outcome_assignment_created"
        ]
        .ne(True)
        .fillna(True)
    ).sum()
)

observed_outcome_policy_versions = sorted(
    outcome_keys[
        "outcome_policy_version"
    ]
    .dropna()
    .astype(str)
    .unique()
    .tolist()
)

observed_outcome_policy_hashes = sorted(
    outcome_keys[
        "outcome_policy_sha256"
    ]
    .dropna()
    .astype(str)
    .unique()
    .tolist()
)


# --------------------------------------------------------------------------------------------------
# 14. Verify exact key compatibility without joining scores to outcomes
# --------------------------------------------------------------------------------------------------

score_key_index = pd.Index(
    scores["_rcv_key"]
)

outcome_key_index = pd.Index(
    outcome_keys["_rcv_key"]
)

score_keys_missing_from_outcomes = (
    score_key_index.difference(
        outcome_key_index
    )
)

outcome_keys_missing_from_scores = (
    outcome_key_index.difference(
        score_key_index
    )
)

ordered_key_match = bool(
    scores["_rcv_key"]
    .reset_index(drop=True)
    .equals(
        outcome_keys["_rcv_key"]
        .reset_index(drop=True)
    )
)

print("\nSTAGE 5 OUTCOME KEY AND PROVENANCE INTEGRITY")
print("-" * 124)
print(
    f"Outcome rows:                           "
    f"{outcome_rows:,}"
)
print(
    f"Outcome columns:                        "
    f"{outcome_columns:,}"
)
print(
    f"Missing outcome RCV keys:               "
    f"{missing_outcome_keys:,}"
)
print(
    f"Duplicate outcome RCV keys:             "
    f"{duplicate_outcome_keys:,}"
)
print(
    f"Rows without assignment-created=True:   "
    f"{outcome_assignment_not_true:,}"
)
print(
    f"Observed outcome-policy versions:       "
    f"{observed_outcome_policy_versions}"
)
print(
    f"Observed outcome-policy hashes:         "
    f"{observed_outcome_policy_hashes}"
)

print("\nSCORE/OUTCOME KEY COMPATIBILITY")
print("-" * 124)
print(
    f"Score keys absent from outcomes:        "
    f"{len(score_keys_missing_from_outcomes):,}"
)
print(
    f"Outcome keys absent from scores:        "
    f"{len(outcome_keys_missing_from_scores):,}"
)
print(
    f"Exact row-order agreement:              "
    f"{'PASS' if ordered_key_match else 'NOT REQUIRED — KEY JOIN WILL BE USED'}"
)
print(
    f"One-to-one key join possible:           "
    f"{'YES' if (
        len(score_keys_missing_from_outcomes) == 0
        and len(outcome_keys_missing_from_scores) == 0
        and duplicate_score_keys == 0
        and duplicate_outcome_keys == 0
    ) else 'NO'}"
)


# --------------------------------------------------------------------------------------------------
# 15. Final critical checks
# --------------------------------------------------------------------------------------------------

critical_checks = {
    **cryptographic_checks,

    **{
        f"Stage 4 manifest — {name}": passed
        for name, passed
        in stage4_manifest_checks.items()
    },

    **{
        f"Stage 5 manifest — {name}": passed
        for name, passed
        in stage5_manifest_checks.items()
    },

    "expected score-table row count":
        score_rows
        == EXPECTED_SCORE_ROWS,

    "expected score-table column count":
        score_columns
        == EXPECTED_SCORE_COLUMNS,

    "exact score-table schema":
        score_schema_exact_match,

    "no missing score keys":
        missing_score_keys == 0,

    "no duplicate score keys":
        duplicate_score_keys == 0,

    "no malformed score keys":
        malformed_score_keys == 0,

    "no missing full-GES scores":
        missing_full_scores == 0,

    "no missing no-star scores":
        missing_no_star_scores == 0,

    "full scores restricted to [0,1]":
        full_scores_outside_unit_interval == 0,

    "no-star scores restricted to [0,1]":
        no_star_scores_outside_unit_interval == 0,

    "decision threshold is exactly 0.5":
        threshold_invalid == 0,

    "full threshold predictions reconstruct":
        full_prediction_mismatch == 0,

    "no-star threshold predictions reconstruct":
        no_star_prediction_mismatch == 0,

    "expected full training-row count":
        full_training_rows
        == EXPECTED_FULL_TRAINING_ROWS,

    "expected no-star training-row count":
        no_star_training_rows
        == EXPECTED_NO_STAR_TRAINING_ROWS,

    "score table confirms no T1 use":
        t1_used_true == 0,

    "score table confirms no future outcomes":
        future_outcome_present == 0,

    "score table confirms no temporal performance":
        temporal_performance_present == 0,

    "score-table version":
        observed_stage4_versions
        == ["1.0.0"],

    "expected outcome row count":
        outcome_rows
        == EXPECTED_OUTCOME_ROWS,

    "expected outcome column count":
        outcome_columns
        == EXPECTED_OUTCOME_COLUMNS,

    "no missing outcome keys":
        missing_outcome_keys == 0,

    "no duplicate outcome keys":
        duplicate_outcome_keys == 0,

    "outcome assignment-created provenance":
        outcome_assignment_not_true == 0,

    "outcome-policy version provenance":
        observed_outcome_policy_versions
        == ["1.0.0"],

    "outcome-policy checksum provenance":
        observed_outcome_policy_hashes
        == [EXPECTED_STAGE5_POLICY_SHA256],

    "all score keys represented in outcomes":
        len(
            score_keys_missing_from_outcomes
        )
        == 0,

    "all outcome keys represented in scores":
        len(
            outcome_keys_missing_from_scores
        )
        == 0,
}

failed_checks = [
    check_name
    for check_name, passed
    in critical_checks.items()
    if not passed
]

print("\n" + "=" * 124)
print("STAGE 6A STEP 1 DECISION")
print("=" * 124)

if failed_checks:

    print(
        "FAIL_STAGE6A_FROZEN_INPUT_VERIFICATION"
    )

    print("\nFailed checks:")

    for failed_check in failed_checks:
        print(
            f" - {failed_check}"
        )

    raise RuntimeError(
        "Frozen Stage 4 scores and Stage 5 outcomes "
        "did not pass locked-validation input verification."
    )

print(
    "PASS_STAGE6A_FROZEN_SCORE_AND_OUTCOME_INPUTS_VERIFIED"
)

print()
print(
    "Stage 4 full-GES scores verified:       YES"
)
print(
    "Stage 4 no-star scores verified:        YES"
)
print(
    "Stage 5 outcomes verified:              YES"
)
print(
    "Exact one-to-one key compatibility:     YES"
)
print()
print(
    "Outcome labels loaded in this cell:     NO"
)
print(
    "Score-outcome join created:             NO"
)
print(
    "Comparator policy frozen:               NO"
)
print(
    "Temporal performance examined:          NO"
)
print(
    "Files written or modified:              NO"
)
print()
print(
    "NEXT AUTHORIZED ACTION:"
)
print(
    "Freeze the prespecified comparator-score definitions "
    "before loading outcome labels or calculating performance."
)

Mounted at /content/drive
STAGE 6A STEP 1 — FROZEN SCORE AND OUTCOME INPUT VERIFICATION

CRYPTOGRAPHIC VERIFICATION
----------------------------------------------------------------------------------------------------------------------------
Stage 4 full GES model                                  PASS
Stage 4 no-star GES model                               PASS
Stage 4 T0 score table                                  PASS
Stage 4C freeze manifest                                PASS
Stage 5 outcome table                                   PASS
Stage 5 outcome sidecar                                 PASS
Stage 5 freeze manifest                                 PASS
Stage 5 manifest sidecar                                PASS

STAGE 4C MANIFEST VERIFICATION
----------------------------------------------------------------------------------------------------------------------------
manifest name                                           PASS
manifest version                                     

In [ ]:
# ==================================================================================================
# STAGE 6A — STEP 2 — CELL 1
# VERIFY FROZEN T0 COMPARATOR-SOURCE ARTIFACTS
#
# Purpose:
#   1. Mount Google Drive.
#   2. Cryptographically verify the frozen Stage 4A feature table.
#   3. Locate and verify the frozen Stage 4A specification, QC report, and freeze manifest.
#   4. Locate and verify the frozen Stage 4B transformation parameters, weak-label rules,
#      and freeze manifest.
#   5. Validate the T0 feature fields required for prespecified comparator construction.
#
# Scientific boundary:
#   - No Stage 5 outcome table is opened.
#   - No outcome label is loaded.
#   - No score-outcome join is created.
#   - No comparator score is constructed yet.
#   - No temporal performance is calculated.
#   - No artifact is written or modified.
# ==================================================================================================

from google.colab import drive

drive.mount(
    "/content/drive"
)

from pathlib import Path
import hashlib
import json

import numpy as np
import pandas as pd
import pyarrow.parquet as pq


# --------------------------------------------------------------------------------------------------
# 1. Frozen project paths
# --------------------------------------------------------------------------------------------------

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/GES_RAG_Temporal_Study"
)

STAGE4_DATA_DIR = (
    PROJECT_ROOT
    / "data_processed"
    / "stage4_ges"
)

STAGE4_CONFIG_DIR = (
    PROJECT_ROOT
    / "configs"
    / "stage4_ges"
)

STAGE4A_FEATURE_TABLE_PATH = (
    STAGE4_DATA_DIR
    / "stage4a_t0_ges_baseline_features_v1.parquet"
)


# --------------------------------------------------------------------------------------------------
# 2. Frozen expected checksums
# --------------------------------------------------------------------------------------------------

EXPECTED_STAGE4A_FEATURE_TABLE_SHA256 = (
    "c100b3781e6801425f622f5d091376ab"
    "fe0939e48c0a792af32f6eebe6401f16"
)

EXPECTED_STAGE4A_FEATURE_SPECIFICATION_SHA256 = (
    "fc00146efe5da9b3fbe740bb42ca99d1"
    "57252cdefc045650f9e88d54d8fcfa8b"
)

EXPECTED_STAGE4A_FEATURE_QC_SHA256 = (
    "4bf72af66aa800fa9f12fdc8598bd06e"
    "bc859bb31c97ad97cf7fc11ef1614cf1"
)

EXPECTED_STAGE4A_FREEZE_MANIFEST_SHA256 = (
    "2b844ef2dbc0c3e5e57493886a753358"
    "7350e1b4b4c76fc9d445ecda8a528fa0"
)

EXPECTED_STAGE4B_TRANSFORM_PARAMETERS_SHA256 = (
    "baeab167e19381138f93c51ba2fa00e4"
    "eeed684b36122c80cd2bf9fc4fe4b08d"
)

EXPECTED_STAGE4B_WEAK_LABEL_RULES_SHA256 = (
    "3d78e66cea1fed5c75ef1cab1b7cf44d"
    "3d3d7bfae50909173bae6c3a0e0bff61"
)

EXPECTED_STAGE4B_FREEZE_MANIFEST_SHA256 = (
    "e766061442e6d4610f661f44a619e41b"
    "45dc6363b28f6176d3f9a71f8215c63f"
)


# --------------------------------------------------------------------------------------------------
# 3. Frozen expected dimensions and observed Stage 4A accounting
# --------------------------------------------------------------------------------------------------

EXPECTED_FEATURE_ROWS = 71_659
EXPECTED_FEATURE_COLUMNS = 30

EXPECTED_RECENCY_MISSING_ROWS = 5_889
EXPECTED_CONFLICT_POSITIVE_ROWS = 1_484

EXPECTED_REVIEW_STAR_COUNTS = {
    0: 3_878,
    1: 50_399,
    2: 9_224,
    3: 8_158,
}


# --------------------------------------------------------------------------------------------------
# 4. Comparator-source columns required from the frozen Stage 4A table
# --------------------------------------------------------------------------------------------------

REQUIRED_COMPARATOR_SOURCE_COLUMNS = [
    "t0_row_order",
    "rcv_accession",
    "recency_days",
    "recency_years",
    "recency_missing_flag",
    "log1p_unique_submitter_count",
    "aggregate_review_stars",
    "aggregate_conflict_flag",
    "scv_group_entropy_normalized",
]


# --------------------------------------------------------------------------------------------------
# 5. Helpers
# --------------------------------------------------------------------------------------------------

def calculate_sha256(
    file_path: Path,
    block_size: int = 8 * 1024 * 1024,
) -> str:
    """
    Calculate SHA-256 without loading the entire file into memory.
    """

    digest = hashlib.sha256()

    with file_path.open("rb") as file_handle:
        for block in iter(
            lambda: file_handle.read(block_size),
            b"",
        ):
            digest.update(block)

    return digest.hexdigest()


def find_unique_json_by_sha256(
    search_root: Path,
    expected_sha256: str,
) -> Path:
    """
    Locate exactly one JSON artifact under a frozen configuration directory
    using its expected SHA-256 checksum.
    """

    matches = []

    for candidate_path in sorted(
        search_root.rglob("*.json")
    ):
        if (
            candidate_path.is_file()
            and calculate_sha256(candidate_path)
            == expected_sha256
        ):
            matches.append(candidate_path)

    if len(matches) == 0:
        raise FileNotFoundError(
            "No JSON artifact matched expected SHA-256:\n"
            f"{expected_sha256}\n"
            f"Search root: {search_root}"
        )

    if len(matches) > 1:
        raise RuntimeError(
            "More than one JSON artifact matched the same frozen SHA-256:\n"
            + "\n".join(
                f" - {path}"
                for path in matches
            )
        )

    return matches[0]


def normalize_rcv(
    series: pd.Series,
) -> pd.Series:
    """
    Normalize RCV accessions for key-integrity checking.
    """

    return (
        series.astype("string")
        .str.strip()
        .str.upper()
        .replace("", pd.NA)
    )


# --------------------------------------------------------------------------------------------------
# 6. Verify required directories and the frozen Stage 4A feature table exist
# --------------------------------------------------------------------------------------------------

required_paths = [
    PROJECT_ROOT,
    STAGE4_DATA_DIR,
    STAGE4_CONFIG_DIR,
    STAGE4A_FEATURE_TABLE_PATH,
]

missing_paths = [
    path
    for path in required_paths
    if not path.exists()
]

if missing_paths:
    raise FileNotFoundError(
        "Required Stage 4 comparator-source paths were not found:\n"
        + "\n".join(
            f" - {path}"
            for path in missing_paths
        )
    )


# --------------------------------------------------------------------------------------------------
# 7. Locate the frozen Stage 4A and Stage 4B JSON artifacts by checksum
# --------------------------------------------------------------------------------------------------

STAGE4A_FEATURE_SPECIFICATION_PATH = (
    find_unique_json_by_sha256(
        STAGE4_CONFIG_DIR,
        EXPECTED_STAGE4A_FEATURE_SPECIFICATION_SHA256,
    )
)

STAGE4A_FEATURE_QC_PATH = (
    find_unique_json_by_sha256(
        STAGE4_CONFIG_DIR,
        EXPECTED_STAGE4A_FEATURE_QC_SHA256,
    )
)

STAGE4A_FREEZE_MANIFEST_PATH = (
    find_unique_json_by_sha256(
        STAGE4_CONFIG_DIR,
        EXPECTED_STAGE4A_FREEZE_MANIFEST_SHA256,
    )
)

STAGE4B_TRANSFORM_PARAMETERS_PATH = (
    find_unique_json_by_sha256(
        STAGE4_CONFIG_DIR,
        EXPECTED_STAGE4B_TRANSFORM_PARAMETERS_SHA256,
    )
)

STAGE4B_WEAK_LABEL_RULES_PATH = (
    find_unique_json_by_sha256(
        STAGE4_CONFIG_DIR,
        EXPECTED_STAGE4B_WEAK_LABEL_RULES_SHA256,
    )
)

STAGE4B_FREEZE_MANIFEST_PATH = (
    find_unique_json_by_sha256(
        STAGE4_CONFIG_DIR,
        EXPECTED_STAGE4B_FREEZE_MANIFEST_SHA256,
    )
)


# --------------------------------------------------------------------------------------------------
# 8. Cryptographically verify every frozen comparator-source artifact
# --------------------------------------------------------------------------------------------------

observed_stage4a_feature_table_sha256 = (
    calculate_sha256(
        STAGE4A_FEATURE_TABLE_PATH
    )
)

observed_stage4a_feature_specification_sha256 = (
    calculate_sha256(
        STAGE4A_FEATURE_SPECIFICATION_PATH
    )
)

observed_stage4a_feature_qc_sha256 = (
    calculate_sha256(
        STAGE4A_FEATURE_QC_PATH
    )
)

observed_stage4a_freeze_manifest_sha256 = (
    calculate_sha256(
        STAGE4A_FREEZE_MANIFEST_PATH
    )
)

observed_stage4b_transform_parameters_sha256 = (
    calculate_sha256(
        STAGE4B_TRANSFORM_PARAMETERS_PATH
    )
)

observed_stage4b_weak_label_rules_sha256 = (
    calculate_sha256(
        STAGE4B_WEAK_LABEL_RULES_PATH
    )
)

observed_stage4b_freeze_manifest_sha256 = (
    calculate_sha256(
        STAGE4B_FREEZE_MANIFEST_PATH
    )
)

cryptographic_checks = {
    "Stage 4A feature table":
        observed_stage4a_feature_table_sha256
        == EXPECTED_STAGE4A_FEATURE_TABLE_SHA256,

    "Stage 4A feature specification":
        observed_stage4a_feature_specification_sha256
        == EXPECTED_STAGE4A_FEATURE_SPECIFICATION_SHA256,

    "Stage 4A feature QC report":
        observed_stage4a_feature_qc_sha256
        == EXPECTED_STAGE4A_FEATURE_QC_SHA256,

    "Stage 4A freeze manifest":
        observed_stage4a_freeze_manifest_sha256
        == EXPECTED_STAGE4A_FREEZE_MANIFEST_SHA256,

    "Stage 4B transformation parameters":
        observed_stage4b_transform_parameters_sha256
        == EXPECTED_STAGE4B_TRANSFORM_PARAMETERS_SHA256,

    "Stage 4B weak-label rules":
        observed_stage4b_weak_label_rules_sha256
        == EXPECTED_STAGE4B_WEAK_LABEL_RULES_SHA256,

    "Stage 4B freeze manifest":
        observed_stage4b_freeze_manifest_sha256
        == EXPECTED_STAGE4B_FREEZE_MANIFEST_SHA256,
}


# --------------------------------------------------------------------------------------------------
# 9. Inspect frozen Stage 4A Parquet structure
# --------------------------------------------------------------------------------------------------

feature_parquet = pq.ParquetFile(
    STAGE4A_FEATURE_TABLE_PATH
)

feature_rows = (
    feature_parquet.metadata.num_rows
)

feature_columns = (
    feature_parquet.metadata.num_columns
)

feature_row_groups = (
    feature_parquet.metadata.num_row_groups
)

feature_schema_columns = list(
    feature_parquet.schema.names
)

missing_required_columns = sorted(
    set(
        REQUIRED_COMPARATOR_SOURCE_COLUMNS
    ).difference(
        feature_schema_columns
    )
)


# --------------------------------------------------------------------------------------------------
# 10. Load only the T0 fields needed for comparator-source validation
# --------------------------------------------------------------------------------------------------

if missing_required_columns:
    raise RuntimeError(
        "Required comparator-source columns are missing:\n"
        + "\n".join(
            f" - {column}"
            for column in missing_required_columns
        )
    )

features = pd.read_parquet(
    STAGE4A_FEATURE_TABLE_PATH,
    columns=REQUIRED_COMPARATOR_SOURCE_COLUMNS,
)

features["_rcv_key"] = normalize_rcv(
    features["rcv_accession"]
)


# --------------------------------------------------------------------------------------------------
# 11. Validate key integrity
# --------------------------------------------------------------------------------------------------

missing_rcv_keys = int(
    features["_rcv_key"]
    .isna()
    .sum()
)

duplicate_rcv_keys = int(
    features["_rcv_key"]
    .duplicated(
        keep=False
    )
    .sum()
)

malformed_rcv_keys = int(
    (
        features["_rcv_key"]
        .notna()
        &
        ~features["_rcv_key"]
        .str.match(
            r"^RCV\d{9}$",
            na=False,
        )
    ).sum()
)

t0_row_order_numeric = pd.to_numeric(
    features["t0_row_order"],
    errors="coerce",
)

invalid_t0_row_order = int(
    (
        t0_row_order_numeric.isna()
        |
        t0_row_order_numeric.duplicated(
            keep=False
        )
    ).sum()
)

expected_t0_row_order = pd.Series(
    np.arange(
        1,
        EXPECTED_FEATURE_ROWS + 1,
        dtype=np.int64,
    )
)

exact_t0_row_order = bool(
    t0_row_order_numeric
    .astype("int64")
    .reset_index(drop=True)
    .equals(expected_t0_row_order)
)


# --------------------------------------------------------------------------------------------------
# 12. Validate comparator-source feature integrity
# --------------------------------------------------------------------------------------------------

recency_days = pd.to_numeric(
    features["recency_days"],
    errors="coerce",
)

recency_years = pd.to_numeric(
    features["recency_years"],
    errors="coerce",
)

recency_missing_flag = (
    features["recency_missing_flag"]
    .astype("boolean")
)

log_submitter_count = pd.to_numeric(
    features["log1p_unique_submitter_count"],
    errors="coerce",
)

review_stars = pd.to_numeric(
    features["aggregate_review_stars"],
    errors="coerce",
)

conflict_flag = (
    features["aggregate_conflict_flag"]
    .astype("boolean")
)

entropy_normalized = pd.to_numeric(
    features["scv_group_entropy_normalized"],
    errors="coerce",
)

recency_missing_rows = int(
    recency_missing_flag.eq(True).sum()
)

recency_flag_value_mismatches = int(
    (
        recency_missing_flag
        .fillna(True)
        .ne(
            recency_days.isna()
        )
    ).sum()
)

recency_days_negative = int(
    recency_days
    .dropna()
    .lt(0)
    .sum()
)

recency_years_negative = int(
    recency_years
    .dropna()
    .lt(0)
    .sum()
)

recency_days_years_mismatch = int(
    (
        recency_days.notna()
        &
        recency_years.notna()
        &
        ~np.isclose(
            recency_years,
            recency_days / 365.25,
            atol=1e-10,
            rtol=1e-10,
        )
    ).sum()
)

missing_submitter_values = int(
    log_submitter_count
    .isna()
    .sum()
)

negative_submitter_values = int(
    log_submitter_count
    .dropna()
    .lt(0)
    .sum()
)

invalid_review_star_values = int(
    (
        review_stars.isna()
        |
        ~review_stars.isin(
            [0, 1, 2, 3, 4]
        )
    ).sum()
)

observed_review_star_counts = {
    int(star): int(count)
    for star, count in (
        review_stars
        .value_counts()
        .sort_index()
        .items()
    )
}

conflict_missing = int(
    conflict_flag
    .isna()
    .sum()
)

conflict_positive_rows = int(
    conflict_flag
    .eq(True)
    .sum()
)

entropy_missing = int(
    entropy_normalized
    .isna()
    .sum()
)

entropy_outside_unit_interval = int(
    (
        entropy_normalized.lt(0)
        |
        entropy_normalized.gt(1)
    ).sum()
)


# --------------------------------------------------------------------------------------------------
# 13. Load frozen JSON records without viewing outcomes or performance
# --------------------------------------------------------------------------------------------------

stage4a_feature_specification = json.loads(
    STAGE4A_FEATURE_SPECIFICATION_PATH.read_text(
        encoding="utf-8"
    )
)

stage4a_feature_qc = json.loads(
    STAGE4A_FEATURE_QC_PATH.read_text(
        encoding="utf-8"
    )
)

stage4a_freeze_manifest = json.loads(
    STAGE4A_FREEZE_MANIFEST_PATH.read_text(
        encoding="utf-8"
    )
)

stage4b_transform_parameters = json.loads(
    STAGE4B_TRANSFORM_PARAMETERS_PATH.read_text(
        encoding="utf-8"
    )
)

stage4b_weak_label_rules = json.loads(
    STAGE4B_WEAK_LABEL_RULES_PATH.read_text(
        encoding="utf-8"
    )
)

stage4b_freeze_manifest = json.loads(
    STAGE4B_FREEZE_MANIFEST_PATH.read_text(
        encoding="utf-8"
    )
)


# --------------------------------------------------------------------------------------------------
# 14. Print verification results
# --------------------------------------------------------------------------------------------------

print("=" * 124)
print("STAGE 6A STEP 2 — CELL 1 — FROZEN COMPARATOR-SOURCE VERIFICATION")
print("=" * 124)

print("\nARTIFACT LOCATIONS")
print("-" * 124)
print(
    f"Stage 4A feature table:             "
    f"{STAGE4A_FEATURE_TABLE_PATH}"
)
print(
    f"Stage 4A feature specification:     "
    f"{STAGE4A_FEATURE_SPECIFICATION_PATH}"
)
print(
    f"Stage 4A feature QC report:         "
    f"{STAGE4A_FEATURE_QC_PATH}"
)
print(
    f"Stage 4A freeze manifest:           "
    f"{STAGE4A_FREEZE_MANIFEST_PATH}"
)
print(
    f"Stage 4B transform parameters:      "
    f"{STAGE4B_TRANSFORM_PARAMETERS_PATH}"
)
print(
    f"Stage 4B weak-label rules:          "
    f"{STAGE4B_WEAK_LABEL_RULES_PATH}"
)
print(
    f"Stage 4B freeze manifest:           "
    f"{STAGE4B_FREEZE_MANIFEST_PATH}"
)

print("\nCRYPTOGRAPHIC VERIFICATION")
print("-" * 124)

for check_name, passed in cryptographic_checks.items():
    print(
        f"{check_name:<55} "
        f"{'PASS' if passed else 'FAIL'}"
    )

print("\nSTAGE 4A FEATURE-TABLE STRUCTURE")
print("-" * 124)
print(
    f"Rows:                                "
    f"{feature_rows:,}"
)
print(
    f"Columns:                             "
    f"{feature_columns:,}"
)
print(
    f"Row groups:                          "
    f"{feature_row_groups:,}"
)
print(
    f"Required comparator columns present: "
    f"{'YES' if not missing_required_columns else 'NO'}"
)

print("\nCOMPARATOR-SOURCE FEATURE INTEGRITY")
print("-" * 124)
print(
    f"Missing RCV keys:                    "
    f"{missing_rcv_keys:,}"
)
print(
    f"Duplicate RCV keys:                  "
    f"{duplicate_rcv_keys:,}"
)
print(
    f"Malformed RCV keys:                  "
    f"{malformed_rcv_keys:,}"
)
print(
    f"Invalid/duplicate T0 row order:      "
    f"{invalid_t0_row_order:,}"
)
print(
    f"Exact sequential T0 row order:       "
    f"{'PASS' if exact_t0_row_order else 'FAIL'}"
)
print(
    f"Recency-missing rows:                "
    f"{recency_missing_rows:,}"
)
print(
    f"Recency flag/value mismatches:       "
    f"{recency_flag_value_mismatches:,}"
)
print(
    f"Negative recency-day values:         "
    f"{recency_days_negative:,}"
)
print(
    f"Negative recency-year values:        "
    f"{recency_years_negative:,}"
)
print(
    f"Recency days/years mismatches:       "
    f"{recency_days_years_mismatch:,}"
)
print(
    f"Missing log submitter values:        "
    f"{missing_submitter_values:,}"
)
print(
    f"Negative log submitter values:       "
    f"{negative_submitter_values:,}"
)
print(
    f"Invalid review-star values:          "
    f"{invalid_review_star_values:,}"
)
print(
    f"Observed review-star counts:         "
    f"{observed_review_star_counts}"
)
print(
    f"Missing conflict flags:              "
    f"{conflict_missing:,}"
)
print(
    f"Conflict-positive rows:              "
    f"{conflict_positive_rows:,}"
)
print(
    f"Missing normalized entropy values:   "
    f"{entropy_missing:,}"
)
print(
    f"Entropy values outside [0,1]:        "
    f"{entropy_outside_unit_interval:,}"
)

print("\nFROZEN JSON TOP-LEVEL KEYS")
print("-" * 124)
print(
    "Stage 4A feature specification keys: "
    f"{sorted(stage4a_feature_specification.keys())}"
)
print(
    "Stage 4A feature QC keys:            "
    f"{sorted(stage4a_feature_qc.keys())}"
)
print(
    "Stage 4A freeze manifest keys:       "
    f"{sorted(stage4a_freeze_manifest.keys())}"
)
print(
    "Stage 4B transform parameter keys:   "
    f"{sorted(stage4b_transform_parameters.keys())}"
)
print(
    "Stage 4B weak-label rule keys:       "
    f"{sorted(stage4b_weak_label_rules.keys())}"
)
print(
    "Stage 4B freeze manifest keys:       "
    f"{sorted(stage4b_freeze_manifest.keys())}"
)

print("\nCOMPLETE STAGE 4A FEATURE SCHEMA")
print("-" * 124)

for index, column_name in enumerate(
    feature_schema_columns,
    start=1,
):
    print(
        f"{index:>2}. {column_name}"
    )


# --------------------------------------------------------------------------------------------------
# 15. Final critical decision
# --------------------------------------------------------------------------------------------------

critical_checks = {
    **cryptographic_checks,

    "expected feature-table row count":
        feature_rows
        == EXPECTED_FEATURE_ROWS,

    "expected feature-table column count":
        feature_columns
        == EXPECTED_FEATURE_COLUMNS,

    "required comparator-source columns present":
        len(missing_required_columns)
        == 0,

    "no missing RCV keys":
        missing_rcv_keys
        == 0,

    "no duplicate RCV keys":
        duplicate_rcv_keys
        == 0,

    "no malformed RCV keys":
        malformed_rcv_keys
        == 0,

    "valid T0 row order":
        invalid_t0_row_order
        == 0,

    "exact sequential T0 row order":
        exact_t0_row_order,

    "expected recency missingness":
        recency_missing_rows
        == EXPECTED_RECENCY_MISSING_ROWS,

    "recency flag agrees with missingness":
        recency_flag_value_mismatches
        == 0,

    "no negative recency values":
        recency_days_negative
        == 0
        and recency_years_negative
        == 0,

    "recency days and years reconcile":
        recency_days_years_mismatch
        == 0,

    "submitter score source complete":
        missing_submitter_values
        == 0,

    "submitter score source nonnegative":
        negative_submitter_values
        == 0,

    "review-star values valid":
        invalid_review_star_values
        == 0,

    "review-star accounting matches freeze":
        observed_review_star_counts
        == EXPECTED_REVIEW_STAR_COUNTS,

    "conflict flags complete":
        conflict_missing
        == 0,

    "conflict accounting matches freeze":
        conflict_positive_rows
        == EXPECTED_CONFLICT_POSITIVE_ROWS,

    "entropy complete":
        entropy_missing
        == 0,

    "entropy restricted to unit interval":
        entropy_outside_unit_interval
        == 0,
}

failed_checks = [
    check_name
    for check_name, passed
    in critical_checks.items()
    if not passed
]

print("\n" + "=" * 124)
print("STAGE 6A STEP 2 — CELL 1 DECISION")
print("=" * 124)

if failed_checks:

    print(
        "FAIL_STAGE6A_COMPARATOR_SOURCE_VERIFICATION"
    )

    print("\nFailed checks:")

    for failed_check in failed_checks:
        print(
            f" - {failed_check}"
        )

    raise RuntimeError(
        "Frozen T0 comparator-source artifacts did not pass verification."
    )

print(
    "PASS_STAGE6A_FROZEN_COMPARATOR_SOURCE_INPUTS_VERIFIED"
)

print()
print(
    "Frozen Stage 4A feature table verified:       YES"
)
print(
    "Frozen Stage 4A specification verified:       YES"
)
print(
    "Frozen Stage 4B transformations verified:     YES"
)
print(
    "Frozen Stage 4B weak-label rules verified:    YES"
)
print()
print(
    "Stage 5 outcome table opened:                 NO"
)
print(
    "Outcome labels loaded:                        NO"
)
print(
    "Comparator scores constructed:               NO"
)
print(
    "Temporal performance examined:               NO"
)
print(
    "Files written or modified:                    NO"
)
print()
print(
    "NEXT AUTHORIZED ACTION:"
)
print(
    "Read and inventory the exact frozen transformation and rule values "
    "needed to define the comparator policy."
)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
STAGE 6A STEP 2 — CELL 1 — FROZEN COMPARATOR-SOURCE VERIFICATION

ARTIFACT LOCATIONS
----------------------------------------------------------------------------------------------------------------------------
Stage 4A feature table:             /content/drive/MyDrive/GES_RAG_Temporal_Study/data_processed/stage4_ges/stage4a_t0_ges_baseline_features_v1.parquet
Stage 4A feature specification:     /content/drive/MyDrive/GES_RAG_Temporal_Study/configs/stage4_ges/stage4a_t0_feature_specification_v1.json
Stage 4A feature QC report:         /content/drive/MyDrive/GES_RAG_Temporal_Study/configs/stage4_ges/stage4a_t0_feature_qc_report_v1.json
Stage 4A freeze manifest:           /content/drive/MyDrive/GES_RAG_Temporal_Study/configs/stage4_ges/stage4a_t0_feature_freeze_manifest_v1.json
Stage 4B transform parameters:      /content/drive/MyDrive/GES_RAG_Temporal_Study/con

RuntimeError: Frozen T0 comparator-source artifacts did not pass verification.

In [ ]:
# ==================================================================================================
# STAGE 6A — STEP 2 — CELL 1A
# DIAGNOSE FROZEN T0 ROW-ORDER CONVENTION
#
# Purpose:
#   Determine whether t0_row_order is:
#     - exactly zero-based: 0 through 71,658;
#     - exactly one-based: 1 through 71,659; or
#     - genuinely nonsequential.
#
# Scientific boundary:
#   - No outcome table is opened.
#   - No comparator is constructed.
#   - No temporal performance is examined.
#   - No file is written or modified.
# ==================================================================================================

import numpy as np
import pandas as pd


# --------------------------------------------------------------------------------------------------
# 1. Recover the row-order series from the already verified Stage 4A feature table
# --------------------------------------------------------------------------------------------------

row_order = pd.to_numeric(
    features["t0_row_order"],
    errors="coerce",
).reset_index(drop=True)

row_count = len(row_order)


# --------------------------------------------------------------------------------------------------
# 2. Construct both legitimate indexing conventions
# --------------------------------------------------------------------------------------------------

expected_zero_based = pd.Series(
    np.arange(
        0,
        row_count,
        dtype=np.int64,
    )
)

expected_one_based = pd.Series(
    np.arange(
        1,
        row_count + 1,
        dtype=np.int64,
    )
)

row_order_as_int = (
    row_order.astype("int64")
    if row_order.notna().all()
    else row_order
)


# --------------------------------------------------------------------------------------------------
# 3. Diagnose completeness, uniqueness, monotonicity, and step size
# --------------------------------------------------------------------------------------------------

missing_values = int(
    row_order.isna().sum()
)

duplicate_values = int(
    row_order.duplicated(
        keep=False
    ).sum()
)

unique_values = int(
    row_order.nunique(
        dropna=True
    )
)

minimum_value = (
    int(row_order.min())
    if row_order.notna().any()
    else None
)

maximum_value = (
    int(row_order.max())
    if row_order.notna().any()
    else None
)

is_monotonic_increasing = bool(
    row_order.is_monotonic_increasing
)

differences = row_order.diff().dropna()

non_unit_steps = int(
    (~differences.eq(1)).sum()
)

zero_based_exact_match = bool(
    missing_values == 0
    and row_order_as_int.equals(
        expected_zero_based
    )
)

one_based_exact_match = bool(
    missing_values == 0
    and row_order_as_int.equals(
        expected_one_based
    )
)


# --------------------------------------------------------------------------------------------------
# 4. Identify any locations where sequential ordering breaks
# --------------------------------------------------------------------------------------------------

expected_from_observed_start = pd.Series(
    np.arange(
        minimum_value,
        minimum_value + row_count,
        dtype=np.int64,
    )
) if minimum_value is not None else pd.Series(dtype="int64")

mismatch_mask = (
    row_order_as_int.ne(
        expected_from_observed_start
    )
    if len(expected_from_observed_start) == row_count
    else pd.Series(
        [True] * row_count
    )
)

mismatch_positions = np.flatnonzero(
    mismatch_mask.to_numpy()
)

first_mismatch_positions = (
    mismatch_positions[:20].tolist()
)


# --------------------------------------------------------------------------------------------------
# 5. Print diagnostic results
# --------------------------------------------------------------------------------------------------

print("=" * 124)
print("STAGE 6A STEP 2 — CELL 1A — T0 ROW-ORDER DIAGNOSTIC")
print("=" * 124)

print("\nROW-ORDER SUMMARY")
print("-" * 124)
print(
    f"Rows examined:                         "
    f"{row_count:,}"
)
print(
    f"Original dtype:                        "
    f"{features['t0_row_order'].dtype}"
)
print(
    f"Missing values:                        "
    f"{missing_values:,}"
)
print(
    f"Duplicate values:                      "
    f"{duplicate_values:,}"
)
print(
    f"Unique values:                         "
    f"{unique_values:,}"
)
print(
    f"Minimum value:                         "
    f"{minimum_value}"
)
print(
    f"Maximum value:                         "
    f"{maximum_value}"
)
print(
    f"Monotonically increasing:              "
    f"{'YES' if is_monotonic_increasing else 'NO'}"
)
print(
    f"Non-unit adjacent steps:               "
    f"{non_unit_steps:,}"
)

print("\nINDEXING-CONVENTION TEST")
print("-" * 124)
print(
    f"Exact zero-based sequence "
    f"(0 to {row_count - 1:,}):  "
    f"{'PASS' if zero_based_exact_match else 'FAIL'}"
)
print(
    f"Exact one-based sequence "
    f"(1 to {row_count:,}):      "
    f"{'PASS' if one_based_exact_match else 'FAIL'}"
)

print("\nFIRST 20 VALUES")
print("-" * 124)
print(
    row_order.head(20).tolist()
)

print("\nLAST 20 VALUES")
print("-" * 124)
print(
    row_order.tail(20).tolist()
)

print("\nSEQUENCE MISMATCH REVIEW")
print("-" * 124)
print(
    f"Mismatches relative to sequence starting "
    f"at observed minimum: {len(mismatch_positions):,}"
)
print(
    f"First mismatch positions:              "
    f"{first_mismatch_positions}"
)


# --------------------------------------------------------------------------------------------------
# 6. Diagnostic decision
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 124)
print("STAGE 6A STEP 2 — CELL 1A DECISION")
print("=" * 124)

if zero_based_exact_match:

    print(
        "PASS_T0_ROW_ORDER_IS_EXACT_ZERO_BASED_SEQUENCE"
    )

    print()
    print(
        "Interpretation:"
    )
    print(
        "The frozen table uses valid zero-based indexing from "
        f"0 through {row_count - 1:,}."
    )
    print(
        "The previous failure was caused only by an incorrect "
        "one-based expectation in the verification cell."
    )

elif one_based_exact_match:

    print(
        "PASS_T0_ROW_ORDER_IS_EXACT_ONE_BASED_SEQUENCE"
    )

    print()
    print(
        "Interpretation:"
    )
    print(
        "The frozen table uses valid one-based indexing from "
        f"1 through {row_count:,}."
    )
    print(
        "A different comparison issue must be examined before continuing."
    )

else:

    print(
        "REVIEW_REQUIRED_T0_ROW_ORDER_IS_NOT_AN_EXACT_STANDARD_SEQUENCE"
    )

    print()
    print(
        "Interpretation:"
    )
    print(
        "The row-order field is not an exact zero-based or one-based "
        "sequence. Do not modify the frozen table."
    )
    print(
        "Review the reported mismatch positions before continuing."
    )

print()
print(
    "Files written or modified:                    NO"
)
print(
    "Outcome labels loaded:                        NO"
)
print(
    "Comparator scores constructed:               NO"
)
print(
    "Temporal performance examined:               NO"
)

STAGE 6A STEP 2 — CELL 1A — T0 ROW-ORDER DIAGNOSTIC

ROW-ORDER SUMMARY
----------------------------------------------------------------------------------------------------------------------------
Rows examined:                         71,659
Original dtype:                        int64
Missing values:                        0
Duplicate values:                      0
Unique values:                         71,659
Minimum value:                         0
Maximum value:                         71658
Monotonically increasing:              YES
Non-unit adjacent steps:               0

INDEXING-CONVENTION TEST
----------------------------------------------------------------------------------------------------------------------------
Exact zero-based sequence (0 to 71,658):  PASS
Exact one-based sequence (1 to 71,659):      FAIL

FIRST 20 VALUES
----------------------------------------------------------------------------------------------------------------------------
[0, 1, 2, 3, 4, 5, 6, 7, 

In [ ]:
# ==================================================================================================
# STAGE 6A — STEP 2 — CELL 1B
# CORRECTED FINAL COMPARATOR-SOURCE VERIFICATION DECISION
#
# Purpose:
#   1. Formally recognize the frozen t0_row_order convention as zero-based.
#   2. Reevaluate all comparator-source critical checks.
#   3. Supersede the earlier false failure caused by assuming one-based indexing.
#
# Scientific boundary:
#   - No Stage 5 outcome table is opened.
#   - No outcome label is loaded.
#   - No comparator score is constructed.
#   - No temporal performance is calculated.
#   - No file is written or modified.
# ==================================================================================================


# --------------------------------------------------------------------------------------------------
# 1. Freeze the observed row-order convention for this verification run
# --------------------------------------------------------------------------------------------------

EXPECTED_T0_ROW_ORDER_START = 0
EXPECTED_T0_ROW_ORDER_END = EXPECTED_FEATURE_ROWS - 1

expected_t0_row_order_zero_based = pd.Series(
    np.arange(
        EXPECTED_T0_ROW_ORDER_START,
        EXPECTED_FEATURE_ROWS,
        dtype=np.int64,
    )
)

exact_zero_based_t0_row_order = bool(
    t0_row_order_numeric
    .astype("int64")
    .reset_index(drop=True)
    .equals(
        expected_t0_row_order_zero_based
    )
)


# --------------------------------------------------------------------------------------------------
# 2. Reconstruct the complete corrected critical-check set
# --------------------------------------------------------------------------------------------------

corrected_critical_checks = {
    **cryptographic_checks,

    "expected feature-table row count":
        feature_rows
        == EXPECTED_FEATURE_ROWS,

    "expected feature-table column count":
        feature_columns
        == EXPECTED_FEATURE_COLUMNS,

    "required comparator-source columns present":
        len(missing_required_columns)
        == 0,

    "no missing RCV keys":
        missing_rcv_keys
        == 0,

    "no duplicate RCV keys":
        duplicate_rcv_keys
        == 0,

    "no malformed RCV keys":
        malformed_rcv_keys
        == 0,

    "no missing or duplicate T0 row-order values":
        invalid_t0_row_order
        == 0,

    "exact zero-based T0 row order":
        exact_zero_based_t0_row_order,

    "expected recency missingness":
        recency_missing_rows
        == EXPECTED_RECENCY_MISSING_ROWS,

    "recency flag agrees with missingness":
        recency_flag_value_mismatches
        == 0,

    "no negative recency values":
        recency_days_negative
        == 0
        and recency_years_negative
        == 0,

    "recency days and years reconcile":
        recency_days_years_mismatch
        == 0,

    "submitter score source complete":
        missing_submitter_values
        == 0,

    "submitter score source nonnegative":
        negative_submitter_values
        == 0,

    "review-star values valid":
        invalid_review_star_values
        == 0,

    "review-star accounting matches freeze":
        observed_review_star_counts
        == EXPECTED_REVIEW_STAR_COUNTS,

    "conflict flags complete":
        conflict_missing
        == 0,

    "conflict accounting matches freeze":
        conflict_positive_rows
        == EXPECTED_CONFLICT_POSITIVE_ROWS,

    "entropy complete":
        entropy_missing
        == 0,

    "entropy restricted to unit interval":
        entropy_outside_unit_interval
        == 0,
}

corrected_failed_checks = [
    check_name
    for check_name, passed
    in corrected_critical_checks.items()
    if not passed
]


# --------------------------------------------------------------------------------------------------
# 3. Print corrected verification summary
# --------------------------------------------------------------------------------------------------

print("=" * 124)
print("STAGE 6A STEP 2 — CELL 1B — CORRECTED COMPARATOR-SOURCE VERIFICATION")
print("=" * 124)

print("\nROW-ORDER CONVENTION")
print("-" * 124)
print(
    f"Observed convention:                  "
    f"ZERO-BASED"
)
print(
    f"Expected range:                       "
    f"{EXPECTED_T0_ROW_ORDER_START:,} "
    f"through {EXPECTED_T0_ROW_ORDER_END:,}"
)
print(
    f"Observed range:                       "
    f"{int(t0_row_order_numeric.min()):,} "
    f"through {int(t0_row_order_numeric.max()):,}"
)
print(
    f"Exact zero-based sequence:            "
    f"{'PASS' if exact_zero_based_t0_row_order else 'FAIL'}"
)

print("\nCORRECTED CRITICAL CHECKS")
print("-" * 124)

for check_name, passed in corrected_critical_checks.items():
    print(
        f"{check_name:<68} "
        f"{'PASS' if passed else 'FAIL'}"
    )


# --------------------------------------------------------------------------------------------------
# 4. Corrected final decision
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 124)
print("STAGE 6A STEP 2 — CELL 1B DECISION")
print("=" * 124)

if corrected_failed_checks:

    print(
        "FAIL_STAGE6A_COMPARATOR_SOURCE_VERIFICATION_CORRECTED"
    )

    print("\nFailed checks:")

    for failed_check in corrected_failed_checks:
        print(
            f" - {failed_check}"
        )

    raise RuntimeError(
        "Frozen comparator-source inputs did not pass the corrected verification."
    )

print(
    "PASS_STAGE6A_FROZEN_COMPARATOR_SOURCE_INPUTS_VERIFIED"
)

print()
print(
    "Stage 4A feature table verified:             YES"
)
print(
    "Stage 4A specification verified:             YES"
)
print(
    "Stage 4A QC report verified:                 YES"
)
print(
    "Stage 4A freeze manifest verified:           YES"
)
print(
    "Stage 4B transformation parameters verified: YES"
)
print(
    "Stage 4B weak-label rules verified:          YES"
)
print(
    "Stage 4B freeze manifest verified:           YES"
)
print(
    "T0 row-order convention:                     ZERO-BASED"
)
print()
print(
    "Previous Cell 1 failure status:              SUPERSEDED"
)
print(
    "Reason:                                      "
    "Cell 1 incorrectly expected one-based indexing"
)
print()
print(
    "Stage 5 outcome table opened:                NO"
)
print(
    "Outcome labels loaded:                       NO"
)
print(
    "Comparator scores constructed:              NO"
)
print(
    "Temporal performance examined:              NO"
)
print(
    "Files written or modified:                   NO"
)
print()
print(
    "NEXT AUTHORIZED ACTION:"
)
print(
    "Inventory the exact frozen Stage 4A feature definitions and "
    "Stage 4B transformation values required to define the comparator policy."
)

STAGE 6A STEP 2 — CELL 1B — CORRECTED COMPARATOR-SOURCE VERIFICATION

ROW-ORDER CONVENTION
----------------------------------------------------------------------------------------------------------------------------
Observed convention:                  ZERO-BASED
Expected range:                       0 through 71,658
Observed range:                       0 through 71,658
Exact zero-based sequence:            PASS

CORRECTED CRITICAL CHECKS
----------------------------------------------------------------------------------------------------------------------------
Stage 4A feature table                                               PASS
Stage 4A feature specification                                       PASS
Stage 4A feature QC report                                           PASS
Stage 4A freeze manifest                                             PASS
Stage 4B transformation parameters                                   PASS
Stage 4B weak-label rules                                   

In [ ]:
# ==================================================================================================
# STAGE 6A — STEP 2 — CELL 2
# INVENTORY FROZEN COMPARATOR-SOURCE DEFINITIONS AND TRANSFORMATIONS
#
# Purpose:
#   1. Read the already verified Stage 4A feature specification.
#   2. Inventory the exact frozen feature definitions relevant to comparator construction.
#   3. Inventory the Stage 4B recency, submitter, review-confidence, and conflict transformations.
#   4. Inventory the weak-label rules only to preserve consistency with the frozen study design.
#   5. Profile the available comparator-source values without constructing comparator scores.
#
# Scientific boundary:
#   - No Stage 5 outcome table is opened.
#   - No outcome label is loaded.
#   - No score-outcome join is created.
#   - No comparator formula is selected or frozen in this cell.
#   - No comparator score is calculated.
#   - No temporal performance is calculated.
#   - No artifact is written or modified.
# ==================================================================================================

import json
import numpy as np
import pandas as pd


# --------------------------------------------------------------------------------------------------
# 1. Confirm that the verified objects from Cell 1 remain available
# --------------------------------------------------------------------------------------------------

required_runtime_objects = [
    "features",
    "stage4a_feature_specification",
    "stage4a_feature_qc",
    "stage4a_freeze_manifest",
    "stage4b_transform_parameters",
    "stage4b_weak_label_rules",
    "stage4b_freeze_manifest",
]

missing_runtime_objects = [
    object_name
    for object_name in required_runtime_objects
    if object_name not in globals()
]

if missing_runtime_objects:
    raise RuntimeError(
        "Required verified objects are missing from the current runtime:\n"
        + "\n".join(
            f" - {object_name}"
            for object_name in missing_runtime_objects
        )
        + "\nRerun Stage 6A Step 2 Cell 1 and Cell 1B before continuing."
    )


# --------------------------------------------------------------------------------------------------
# 2. Helper functions
# --------------------------------------------------------------------------------------------------

def print_json_section(
    section_title,
    section_value,
):
    """
    Print a JSON-compatible object in deterministic, readable form.
    """

    print("\n" + section_title)
    print("-" * 124)

    print(
        json.dumps(
            section_value,
            indent=2,
            sort_keys=True,
            ensure_ascii=False,
            default=str,
        )
    )


def numeric_profile(
    series,
):
    """
    Return a compact numeric profile without constructing a comparator.
    """

    numeric = pd.to_numeric(
        series,
        errors="coerce",
    )

    nonmissing = numeric.dropna()

    return {
        "rows": int(len(numeric)),
        "available": int(nonmissing.shape[0]),
        "missing": int(numeric.isna().sum()),
        "minimum": (
            float(nonmissing.min())
            if not nonmissing.empty
            else None
        ),
        "maximum": (
            float(nonmissing.max())
            if not nonmissing.empty
            else None
        ),
        "mean": (
            float(nonmissing.mean())
            if not nonmissing.empty
            else None
        ),
        "median": (
            float(nonmissing.median())
            if not nonmissing.empty
            else None
        ),
        "unique_values": int(
            nonmissing.nunique()
        ),
    }


def boolean_profile(
    series,
):
    """
    Return a compact Boolean profile.
    """

    values = series.astype("boolean")

    return {
        "rows": int(len(values)),
        "true": int(values.eq(True).sum()),
        "false": int(values.eq(False).sum()),
        "missing": int(values.isna().sum()),
    }


# --------------------------------------------------------------------------------------------------
# 3. Define the exact source fields relevant to the planned comparators
# --------------------------------------------------------------------------------------------------

COMPARATOR_SOURCE_FIELDS = {
    "review_stars": [
        "aggregate_review_stars",
    ],

    "conflict": [
        "aggregate_conflict_flag",
    ],

    "recency": [
        "recency_days",
        "recency_years",
        "recency_missing_flag",
    ],

    "submitter_structure": [
        "unique_submitter_count",
        "log1p_unique_submitter_count",
        "scv_count",
        "log1p_scv_count",
        "submitter_diversity_ratio",
    ],

    "disagreement_entropy": [
        "scv_group_disagreement_flag",
        "scv_group_entropy_normalized",
        "scv_dominant_group_fraction",
        "scv_effective_group_count",
    ],
}

all_required_inventory_fields = sorted(
    {
        field_name
        for field_group
        in COMPARATOR_SOURCE_FIELDS.values()
        for field_name
        in field_group
    }
)

missing_inventory_fields = sorted(
    set(
        all_required_inventory_fields
    ).difference(
        features.columns
    )
)

if missing_inventory_fields:
    raise RuntimeError(
        "Comparator-source inventory fields are missing from the loaded feature dataframe:\n"
        + "\n".join(
            f" - {field_name}"
            for field_name in missing_inventory_fields
        )
    )


# --------------------------------------------------------------------------------------------------
# 4. Extract the complete relevant frozen specification sections
# --------------------------------------------------------------------------------------------------

stage4a_inventory = {
    "specification_name":
        stage4a_feature_specification.get(
            "specification_name"
        ),

    "specification_version":
        stage4a_feature_specification.get(
            "specification_version"
        ),

    "status":
        stage4a_feature_specification.get(
            "status"
        ),

    "source_timepoint":
        stage4a_feature_specification.get(
            "source_timepoint"
        ),

    "embedded_data_cutoff":
        stage4a_feature_specification.get(
            "embedded_data_cutoff"
        ),

    "unit_of_analysis":
        stage4a_feature_specification.get(
            "unit_of_analysis"
        ),

    "feature_definitions":
        stage4a_feature_specification.get(
            "feature_definitions"
        ),

    "full_ges_candidate_features":
        stage4a_feature_specification.get(
            "full_ges_candidate_features"
        ),

    "no_star_ges_candidate_features":
        stage4a_feature_specification.get(
            "no_star_ges_candidate_features"
        ),

    "audit_only_features":
        stage4a_feature_specification.get(
            "audit_only_features"
        ),

    "scientific_boundary":
        stage4a_feature_specification.get(
            "scientific_boundary"
        ),

    "not_performed_in_stage4a":
        stage4a_feature_specification.get(
            "not_performed_in_stage4a"
        ),
}


stage4b_transformation_inventory = {
    "artifact_name":
        stage4b_transform_parameters.get(
            "artifact_name"
        ),

    "version":
        stage4b_transform_parameters.get(
            "version"
        ),

    "status":
        stage4b_transform_parameters.get(
            "status"
        ),

    "source":
        stage4b_transform_parameters.get(
            "source"
        ),

    "recency_transformation":
        stage4b_transform_parameters.get(
            "recency_transformation"
        ),

    "submitter_diversity_transformation":
        stage4b_transform_parameters.get(
            "submitter_diversity_transformation"
        ),

    "review_confidence":
        stage4b_transform_parameters.get(
            "review_confidence"
        ),

    "conflict":
        stage4b_transform_parameters.get(
            "conflict"
        ),

    "random_seed_reserved_for_model_fitting":
        stage4b_transform_parameters.get(
            "random_seed_reserved_for_model_fitting"
        ),

    "software":
        stage4b_transform_parameters.get(
            "software"
        ),

    "scientific_boundary":
        stage4b_transform_parameters.get(
            "scientific_boundary"
        ),
}


stage4b_rule_inventory = {
    "artifact_name":
        stage4b_weak_label_rules.get(
            "artifact_name"
        ),

    "version":
        stage4b_weak_label_rules.get(
            "version"
        ),

    "status":
        stage4b_weak_label_rules.get(
            "status"
        ),

    "label_encoding":
        stage4b_weak_label_rules.get(
            "label_encoding"
        ),

    "full_model_labeling_functions":
        stage4b_weak_label_rules.get(
            "full_model_labeling_functions"
        ),

    "full_aggregation":
        stage4b_weak_label_rules.get(
            "full_aggregation"
        ),

    "no_star_ablation":
        stage4b_weak_label_rules.get(
            "no_star_ablation"
        ),

    "not_performed":
        stage4b_weak_label_rules.get(
            "not_performed"
        ),
}


# --------------------------------------------------------------------------------------------------
# 5. Profile frozen comparator-source values without producing comparator scores
# --------------------------------------------------------------------------------------------------

source_value_profiles = {
    "aggregate_review_stars": {
        "profile":
            numeric_profile(
                features[
                    "aggregate_review_stars"
                ]
            ),

        "counts": {
            str(int(value)): int(count)
            for value, count
            in (
                pd.to_numeric(
                    features[
                        "aggregate_review_stars"
                    ],
                    errors="coerce",
                )
                .value_counts(
                    dropna=False
                )
                .sort_index()
                .items()
            )
            if pd.notna(value)
        },
    },

    "aggregate_conflict_flag":
        boolean_profile(
            features[
                "aggregate_conflict_flag"
            ]
        ),

    "recency_days":
        numeric_profile(
            features[
                "recency_days"
            ]
        ),

    "recency_years":
        numeric_profile(
            features[
                "recency_years"
            ]
        ),

    "recency_missing_flag":
        boolean_profile(
            features[
                "recency_missing_flag"
            ]
        ),

    "unique_submitter_count":
        numeric_profile(
            features[
                "unique_submitter_count"
            ]
        ),

    "log1p_unique_submitter_count":
        numeric_profile(
            features[
                "log1p_unique_submitter_count"
            ]
        ),

    "scv_count":
        numeric_profile(
            features[
                "scv_count"
            ]
        ),

    "log1p_scv_count":
        numeric_profile(
            features[
                "log1p_scv_count"
            ]
        ),

    "submitter_diversity_ratio":
        numeric_profile(
            features[
                "submitter_diversity_ratio"
            ]
        ),

    "scv_group_disagreement_flag":
        boolean_profile(
            features[
                "scv_group_disagreement_flag"
            ]
        ),

    "scv_group_entropy_normalized":
        numeric_profile(
            features[
                "scv_group_entropy_normalized"
            ]
        ),

    "scv_dominant_group_fraction":
        numeric_profile(
            features[
                "scv_dominant_group_fraction"
            ]
        ),

    "scv_effective_group_count":
        numeric_profile(
            features[
                "scv_effective_group_count"
            ]
        ),
}


# --------------------------------------------------------------------------------------------------
# 6. Create an in-memory inventory for use by the next cell
# --------------------------------------------------------------------------------------------------

comparator_definition_inventory = {
    "stage4a_specification":
        stage4a_inventory,

    "stage4b_transformations":
        stage4b_transformation_inventory,

    "stage4b_weak_label_rules":
        stage4b_rule_inventory,

    "planned_comparator_source_fields":
        COMPARATOR_SOURCE_FIELDS,

    "source_value_profiles":
        source_value_profiles,

    "scientific_boundary": {
        "stage5_outcome_table_opened": False,
        "outcome_labels_loaded": False,
        "score_outcome_join_created": False,
        "comparator_policy_selected": False,
        "comparator_scores_constructed": False,
        "temporal_performance_examined": False,
        "files_written_or_modified": False,
    },
}


# --------------------------------------------------------------------------------------------------
# 7. Print the exact inventory
# --------------------------------------------------------------------------------------------------

print("=" * 124)
print("STAGE 6A STEP 2 — CELL 2 — FROZEN COMPARATOR-DEFINITION INVENTORY")
print("=" * 124)

print_json_section(
    "STAGE 4A FROZEN FEATURE SPECIFICATION",
    stage4a_inventory,
)

print_json_section(
    "STAGE 4B FROZEN TRANSFORMATION PARAMETERS",
    stage4b_transformation_inventory,
)

print_json_section(
    "STAGE 4B FROZEN WEAK-LABEL RULES",
    stage4b_rule_inventory,
)

print_json_section(
    "PLANNED COMPARATOR SOURCE-FIELD GROUPS",
    COMPARATOR_SOURCE_FIELDS,
)

print_json_section(
    "OBSERVED FROZEN SOURCE-VALUE PROFILES",
    source_value_profiles,
)


# --------------------------------------------------------------------------------------------------
# 8. Final inventory checks
# --------------------------------------------------------------------------------------------------

inventory_checks = {
    "all planned source fields present":
        len(
            missing_inventory_fields
        )
        == 0,

    "Stage 4A feature definitions present":
        stage4a_inventory[
            "feature_definitions"
        ]
        is not None,

    "full-GES candidate feature set present":
        stage4a_inventory[
            "full_ges_candidate_features"
        ]
        is not None,

    "no-star candidate feature set present":
        stage4a_inventory[
            "no_star_ges_candidate_features"
        ]
        is not None,

    "recency transformation present":
        stage4b_transformation_inventory[
            "recency_transformation"
        ]
        is not None,

    "submitter transformation present":
        stage4b_transformation_inventory[
            "submitter_diversity_transformation"
        ]
        is not None,

    "review-confidence transformation present":
        stage4b_transformation_inventory[
            "review_confidence"
        ]
        is not None,

    "conflict transformation present":
        stage4b_transformation_inventory[
            "conflict"
        ]
        is not None,

    "full-model labeling functions present":
        stage4b_rule_inventory[
            "full_model_labeling_functions"
        ]
        is not None,

    "full aggregation rule present":
        stage4b_rule_inventory[
            "full_aggregation"
        ]
        is not None,

    "no-star ablation rule present":
        stage4b_rule_inventory[
            "no_star_ablation"
        ]
        is not None,
}

failed_inventory_checks = [
    check_name
    for check_name, passed
    in inventory_checks.items()
    if not passed
]


print("\n" + "=" * 124)
print("STAGE 6A STEP 2 — CELL 2 DECISION")
print("=" * 124)

for check_name, passed in inventory_checks.items():
    print(
        f"{check_name:<70} "
        f"{'PASS' if passed else 'FAIL'}"
    )

if failed_inventory_checks:

    print()
    print(
        "FAIL_STAGE6A_COMPARATOR_DEFINITION_INVENTORY"
    )

    print("\nFailed checks:")

    for failed_check in failed_inventory_checks:
        print(
            f" - {failed_check}"
        )

    raise RuntimeError(
        "The frozen comparator-definition inventory is incomplete."
    )

print()
print(
    "PASS_STAGE6A_FROZEN_COMPARATOR_DEFINITIONS_INVENTORIED"
)

print()
print(
    "Stage 5 outcome table opened:                NO"
)
print(
    "Outcome labels loaded:                       NO"
)
print(
    "Comparator policy selected:                  NO"
)
print(
    "Comparator scores constructed:              NO"
)
print(
    "Temporal performance examined:              NO"
)
print(
    "Files written or modified:                   NO"
)
print()
print(
    "NEXT AUTHORIZED ACTION:"
)
print(
    "Use the displayed frozen definitions to specify the exact "
    "direction, scaling, missing-value policy, and formula for each comparator."
)

RuntimeError: Comparator-source inventory fields are missing from the loaded feature dataframe:
 - log1p_scv_count
 - scv_count
 - scv_dominant_group_fraction
 - scv_effective_group_count
 - scv_group_disagreement_flag
 - submitter_diversity_ratio
 - unique_submitter_count

In [ ]:
# ==================================================================================================
# STAGE 6A — STEP 2 — CELL 2A
# LOAD AND VERIFY THE COMPLETE FROZEN COMPARATOR-SOURCE FIELD SET
#
# Purpose:
#   1. Load all Stage 4A fields needed to define the prespecified comparators.
#   2. Preserve the frozen zero-based T0 row order.
#   3. Confirm exact alignment with the comparator-source data verified in Cell 1B.
#
# Scientific boundary:
#   - No Stage 5 outcome table is opened.
#   - No outcome label is loaded.
#   - No comparator formula is selected.
#   - No comparator score is calculated.
#   - No temporal performance is examined.
#   - No file is written or modified.
# ==================================================================================================

from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq


# --------------------------------------------------------------------------------------------------
# 1. Confirm that the frozen Stage 4A feature path is available
# --------------------------------------------------------------------------------------------------

if "STAGE4A_FEATURE_TABLE_PATH" not in globals():
    raise RuntimeError(
        "STAGE4A_FEATURE_TABLE_PATH is not available in the current runtime. "
        "Rerun Stage 6A Step 2 Cell 1 before continuing."
    )

if not Path(STAGE4A_FEATURE_TABLE_PATH).exists():
    raise FileNotFoundError(
        f"Frozen Stage 4A feature table was not found:\n"
        f"{STAGE4A_FEATURE_TABLE_PATH}"
    )


# --------------------------------------------------------------------------------------------------
# 2. Define all source fields potentially needed for the prespecified comparator policy
# --------------------------------------------------------------------------------------------------

COMPARATOR_SOURCE_FIELDS = {
    "review_stars": [
        "aggregate_review_stars",
    ],

    "conflict": [
        "aggregate_conflict_flag",
    ],

    "recency": [
        "recency_days",
        "recency_years",
        "recency_missing_flag",
    ],

    "submitter_structure": [
        "unique_submitter_count",
        "log1p_unique_submitter_count",
        "scv_count",
        "log1p_scv_count",
        "submitter_diversity_ratio",
    ],

    "disagreement_entropy": [
        "scv_group_disagreement_flag",
        "scv_group_entropy_normalized",
        "scv_dominant_group_fraction",
        "scv_effective_group_count",
    ],
}

COMPARATOR_INVENTORY_COLUMNS = [
    "t0_row_order",
    "rcv_accession",

    "aggregate_review_stars",
    "aggregate_conflict_flag",

    "recency_days",
    "recency_years",
    "recency_missing_flag",

    "unique_submitter_count",
    "log1p_unique_submitter_count",
    "scv_count",
    "log1p_scv_count",
    "submitter_diversity_ratio",

    "scv_group_disagreement_flag",
    "scv_group_entropy_normalized",
    "scv_dominant_group_fraction",
    "scv_effective_group_count",
]


# --------------------------------------------------------------------------------------------------
# 3. Inspect the frozen Parquet schema before reading
# --------------------------------------------------------------------------------------------------

stage4a_parquet = pq.ParquetFile(
    STAGE4A_FEATURE_TABLE_PATH
)

stage4a_schema_columns = list(
    stage4a_parquet.schema.names
)

missing_inventory_columns = sorted(
    set(
        COMPARATOR_INVENTORY_COLUMNS
    ).difference(
        stage4a_schema_columns
    )
)

if missing_inventory_columns:
    raise RuntimeError(
        "The frozen Stage 4A Parquet is missing required comparator-source columns:\n"
        + "\n".join(
            f" - {column_name}"
            for column_name in missing_inventory_columns
        )
    )


# --------------------------------------------------------------------------------------------------
# 4. Load the complete comparator-source field set
# --------------------------------------------------------------------------------------------------

comparator_source_features = pd.read_parquet(
    STAGE4A_FEATURE_TABLE_PATH,
    columns=COMPARATOR_INVENTORY_COLUMNS,
)

comparator_source_features["_rcv_key"] = (
    comparator_source_features[
        "rcv_accession"
    ]
    .astype("string")
    .str.strip()
    .str.upper()
    .replace("", pd.NA)
)


# --------------------------------------------------------------------------------------------------
# 5. Validate row count, keys, and zero-based row order
# --------------------------------------------------------------------------------------------------

inventory_row_count = len(
    comparator_source_features
)

expected_inventory_rows = 71_659

inventory_missing_keys = int(
    comparator_source_features[
        "_rcv_key"
    ]
    .isna()
    .sum()
)

inventory_duplicate_keys = int(
    comparator_source_features[
        "_rcv_key"
    ]
    .duplicated(
        keep=False
    )
    .sum()
)

inventory_malformed_keys = int(
    (
        comparator_source_features[
            "_rcv_key"
        ]
        .notna()
        &
        ~comparator_source_features[
            "_rcv_key"
        ]
        .str.match(
            r"^RCV\d{9}$",
            na=False,
        )
    ).sum()
)

inventory_row_order = pd.to_numeric(
    comparator_source_features[
        "t0_row_order"
    ],
    errors="coerce",
)

inventory_missing_row_order = int(
    inventory_row_order
    .isna()
    .sum()
)

inventory_duplicate_row_order = int(
    inventory_row_order
    .duplicated(
        keep=False
    )
    .sum()
)

expected_zero_based_order = pd.Series(
    np.arange(
        0,
        expected_inventory_rows,
        dtype=np.int64,
    )
)

inventory_exact_zero_based_order = bool(
    inventory_row_order
    .astype("int64")
    .reset_index(drop=True)
    .equals(
        expected_zero_based_order
    )
)


# --------------------------------------------------------------------------------------------------
# 6. Confirm alignment with the previously verified limited dataframe
# --------------------------------------------------------------------------------------------------

if "features" in globals():

    prior_row_count = len(
        features
    )

    prior_keys = (
        features[
            "rcv_accession"
        ]
        .astype("string")
        .str.strip()
        .str.upper()
        .replace("", pd.NA)
        .reset_index(drop=True)
    )

    current_keys = (
        comparator_source_features[
            "_rcv_key"
        ]
        .reset_index(drop=True)
    )

    exact_key_alignment_with_cell1 = bool(
        prior_keys.equals(
            current_keys
        )
    )

    prior_row_order = pd.to_numeric(
        features[
            "t0_row_order"
        ],
        errors="coerce",
    ).reset_index(drop=True)

    current_row_order = (
        inventory_row_order
        .reset_index(drop=True)
    )

    exact_order_alignment_with_cell1 = bool(
        prior_row_order.equals(
            current_row_order
        )
    )

else:

    prior_row_count = None
    exact_key_alignment_with_cell1 = None
    exact_order_alignment_with_cell1 = None


# --------------------------------------------------------------------------------------------------
# 7. Confirm completeness of the loaded source fields
# --------------------------------------------------------------------------------------------------

loaded_inventory_columns = [
    column_name
    for column_name
    in COMPARATOR_INVENTORY_COLUMNS
    if column_name
    in comparator_source_features.columns
]

still_missing_loaded_columns = sorted(
    set(
        COMPARATOR_INVENTORY_COLUMNS
    ).difference(
        comparator_source_features.columns
    )
)


# --------------------------------------------------------------------------------------------------
# 8. Print verification results
# --------------------------------------------------------------------------------------------------

print("=" * 124)
print("STAGE 6A STEP 2 — CELL 2A — COMPLETE COMPARATOR-SOURCE FIELD LOAD")
print("=" * 124)

print("\nFROZEN SOURCE")
print("-" * 124)
print(
    f"Stage 4A feature table:               "
    f"{STAGE4A_FEATURE_TABLE_PATH}"
)
print(
    f"Parquet rows:                         "
    f"{stage4a_parquet.metadata.num_rows:,}"
)
print(
    f"Parquet columns:                      "
    f"{stage4a_parquet.metadata.num_columns:,}"
)

print("\nCOMPLETE COMPARATOR-SOURCE LOAD")
print("-" * 124)
print(
    f"Rows loaded:                          "
    f"{inventory_row_count:,}"
)
print(
    f"Requested source columns:             "
    f"{len(COMPARATOR_INVENTORY_COLUMNS):,}"
)
print(
    f"Successfully loaded source columns:   "
    f"{len(loaded_inventory_columns):,}"
)
print(
    f"Missing requested columns:            "
    f"{len(still_missing_loaded_columns):,}"
)

print("\nKEY AND ORDER INTEGRITY")
print("-" * 124)
print(
    f"Missing RCV keys:                     "
    f"{inventory_missing_keys:,}"
)
print(
    f"Duplicate RCV keys:                   "
    f"{inventory_duplicate_keys:,}"
)
print(
    f"Malformed RCV keys:                   "
    f"{inventory_malformed_keys:,}"
)
print(
    f"Missing T0 row-order values:          "
    f"{inventory_missing_row_order:,}"
)
print(
    f"Duplicate T0 row-order values:        "
    f"{inventory_duplicate_row_order:,}"
)
print(
    f"Observed row-order range:             "
    f"{int(inventory_row_order.min()):,} "
    f"through {int(inventory_row_order.max()):,}"
)
print(
    f"Exact zero-based sequence:            "
    f"{'PASS' if inventory_exact_zero_based_order else 'FAIL'}"
)

print("\nALIGNMENT WITH CELL 1 VERIFIED DATA")
print("-" * 124)

if prior_row_count is not None:

    print(
        f"Prior verified rows:                  "
        f"{prior_row_count:,}"
    )
    print(
        f"Exact RCV-key alignment:              "
        f"{'PASS' if exact_key_alignment_with_cell1 else 'FAIL'}"
    )
    print(
        f"Exact T0 row-order alignment:         "
        f"{'PASS' if exact_order_alignment_with_cell1 else 'FAIL'}"
    )

else:

    print(
        "Prior Cell 1 dataframe was not available; "
        "independent frozen-file checks were used."
    )

print("\nLOADED COMPARATOR-SOURCE COLUMNS")
print("-" * 124)

for index, column_name in enumerate(
    COMPARATOR_INVENTORY_COLUMNS,
    start=1,
):
    print(
        f"{index:>2}. {column_name}"
    )


# --------------------------------------------------------------------------------------------------
# 9. Final decision
# --------------------------------------------------------------------------------------------------

cell2a_checks = {
    "expected row count":
        inventory_row_count
        == expected_inventory_rows,

    "all requested columns loaded":
        len(
            still_missing_loaded_columns
        )
        == 0,

    "no missing RCV keys":
        inventory_missing_keys
        == 0,

    "no duplicate RCV keys":
        inventory_duplicate_keys
        == 0,

    "no malformed RCV keys":
        inventory_malformed_keys
        == 0,

    "no missing T0 row-order values":
        inventory_missing_row_order
        == 0,

    "no duplicate T0 row-order values":
        inventory_duplicate_row_order
        == 0,

    "exact zero-based row order":
        inventory_exact_zero_based_order,
}

if prior_row_count is not None:

    cell2a_checks.update(
        {
            "row count matches Cell 1":
                inventory_row_count
                == prior_row_count,

            "RCV keys align with Cell 1":
                exact_key_alignment_with_cell1,

            "row order aligns with Cell 1":
                exact_order_alignment_with_cell1,
        }
    )

failed_cell2a_checks = [
    check_name
    for check_name, passed
    in cell2a_checks.items()
    if not passed
]

print("\n" + "=" * 124)
print("STAGE 6A STEP 2 — CELL 2A DECISION")
print("=" * 124)

for check_name, passed in cell2a_checks.items():
    print(
        f"{check_name:<65} "
        f"{'PASS' if passed else 'FAIL'}"
    )

if failed_cell2a_checks:

    print()
    print(
        "FAIL_STAGE6A_COMPLETE_COMPARATOR_SOURCE_LOAD"
    )

    print("\nFailed checks:")

    for failed_check in failed_cell2a_checks:
        print(
            f" - {failed_check}"
        )

    raise RuntimeError(
        "The complete comparator-source field set did not pass alignment verification."
    )

print()
print(
    "PASS_STAGE6A_COMPLETE_COMPARATOR_SOURCE_FIELDS_LOADED"
)

print()
print(
    "Previous Cell 2 error:                       RESOLVED"
)
print(
    "Cause:                                       "
    "The earlier dataframe contained only the Cell 1 verification subset"
)
print(
    "Complete comparator-source dataframe:        "
    "comparator_source_features"
)
print()
print(
    "Stage 5 outcome table opened:                NO"
)
print(
    "Outcome labels loaded:                       NO"
)
print(
    "Comparator policy selected:                  NO"
)
print(
    "Comparator scores constructed:              NO"
)
print(
    "Temporal performance examined:              NO"
)
print(
    "Files written or modified:                   NO"
)
print()
print(
    "NEXT AUTHORIZED ACTION:"
)
print(
    "Inventory the frozen feature definitions, transformations, "
    "and observed comparator-source value ranges."
)

STAGE 6A STEP 2 — CELL 2A — COMPLETE COMPARATOR-SOURCE FIELD LOAD

FROZEN SOURCE
----------------------------------------------------------------------------------------------------------------------------
Stage 4A feature table:               /content/drive/MyDrive/GES_RAG_Temporal_Study/data_processed/stage4_ges/stage4a_t0_ges_baseline_features_v1.parquet
Parquet rows:                         71,659
Parquet columns:                      30

COMPLETE COMPARATOR-SOURCE LOAD
----------------------------------------------------------------------------------------------------------------------------
Rows loaded:                          71,659
Requested source columns:             16
Successfully loaded source columns:   16
Missing requested columns:            0

KEY AND ORDER INTEGRITY
----------------------------------------------------------------------------------------------------------------------------
Missing RCV keys:                     0
Duplicate RCV keys:                   0

In [ ]:
# ==================================================================================================
# STAGE 6A — STEP 2 — CELL 2B
# INVENTORY FROZEN COMPARATOR DEFINITIONS, TRANSFORMATIONS, AND SOURCE-VALUE RANGES
#
# Purpose:
#   1. Inventory the frozen Stage 4A feature definitions relevant to comparator construction.
#   2. Inventory the frozen Stage 4B transformation parameters and weak-label rules.
#   3. Profile the frozen comparator-source values without constructing any comparator score.
#   4. Preserve an in-memory inventory for the comparator-policy freeze cell.
#
# Scientific boundary:
#   - No Stage 5 outcome table is opened.
#   - No outcome label is loaded.
#   - No score-outcome join is created.
#   - No comparator formula is selected.
#   - No comparator score is calculated.
#   - No temporal performance is examined.
#   - No artifact is written or modified.
# ==================================================================================================

import json

import numpy as np
import pandas as pd


# --------------------------------------------------------------------------------------------------
# 1. Confirm required verified runtime objects are available
# --------------------------------------------------------------------------------------------------

REQUIRED_RUNTIME_OBJECTS = [
    "comparator_source_features",
    "stage4a_feature_specification",
    "stage4a_feature_qc",
    "stage4a_freeze_manifest",
    "stage4b_transform_parameters",
    "stage4b_weak_label_rules",
    "stage4b_freeze_manifest",
]

missing_runtime_objects = [
    object_name
    for object_name in REQUIRED_RUNTIME_OBJECTS
    if object_name not in globals()
]

if missing_runtime_objects:
    raise RuntimeError(
        "Required verified runtime objects are missing:\n"
        + "\n".join(
            f" - {object_name}"
            for object_name in missing_runtime_objects
        )
        + "\nRerun Stage 6A Step 2 Cells 1, 1B, and 2A before continuing."
    )


# --------------------------------------------------------------------------------------------------
# 2. Frozen comparator-source field groups
# --------------------------------------------------------------------------------------------------

COMPARATOR_SOURCE_FIELD_GROUPS = {
    "review_stars_baseline": [
        "aggregate_review_stars",
    ],

    "conflict_baseline": [
        "aggregate_conflict_flag",
    ],

    "recency_baseline": [
        "recency_days",
        "recency_years",
        "recency_missing_flag",
    ],

    "submitter_baseline": [
        "unique_submitter_count",
        "log1p_unique_submitter_count",
    ],

    "additional_submitter_structure": [
        "scv_count",
        "log1p_scv_count",
        "submitter_diversity_ratio",
    ],

    "disagreement_and_entropy": [
        "scv_group_disagreement_flag",
        "scv_group_entropy_normalized",
        "scv_dominant_group_fraction",
        "scv_effective_group_count",
    ],
}

all_inventory_fields = sorted(
    {
        field_name
        for field_names
        in COMPARATOR_SOURCE_FIELD_GROUPS.values()
        for field_name
        in field_names
    }
)

missing_inventory_fields = sorted(
    set(
        all_inventory_fields
    ).difference(
        comparator_source_features.columns
    )
)

if missing_inventory_fields:
    raise RuntimeError(
        "Required comparator-source fields are missing:\n"
        + "\n".join(
            f" - {field_name}"
            for field_name in missing_inventory_fields
        )
    )


# --------------------------------------------------------------------------------------------------
# 3. Helper functions
# --------------------------------------------------------------------------------------------------

def to_json_safe(value):
    """
    Convert NumPy and pandas scalar values into JSON-compatible Python values.
    """

    if isinstance(value, dict):
        return {
            str(key): to_json_safe(item)
            for key, item in value.items()
        }

    if isinstance(value, (list, tuple)):
        return [
            to_json_safe(item)
            for item in value
        ]

    if isinstance(value, np.integer):
        return int(value)

    if isinstance(value, np.floating):
        if np.isnan(value):
            return None
        return float(value)

    if isinstance(value, np.bool_):
        return bool(value)

    if pd.isna(value):
        return None

    return value


def print_json_section(
    title,
    value,
):
    """
    Print a deterministic JSON section.
    """

    print("\n" + title)
    print("-" * 124)

    print(
        json.dumps(
            to_json_safe(value),
            indent=2,
            sort_keys=True,
            ensure_ascii=False,
            default=str,
        )
    )


def numeric_profile(
    series,
):
    """
    Produce a descriptive profile without constructing a comparator.
    """

    numeric = pd.to_numeric(
        series,
        errors="coerce",
    )

    available = numeric.dropna()

    if available.empty:
        return {
            "rows": int(len(numeric)),
            "available": 0,
            "missing": int(numeric.isna().sum()),
            "unique_values": 0,
            "minimum": None,
            "maximum": None,
            "mean": None,
            "standard_deviation": None,
            "quantiles": {},
        }

    requested_quantiles = [
        0.00,
        0.01,
        0.05,
        0.10,
        0.25,
        0.50,
        0.75,
        0.90,
        0.95,
        0.99,
        1.00,
    ]

    quantiles = available.quantile(
        requested_quantiles
    )

    return {
        "rows":
            int(len(numeric)),

        "available":
            int(available.shape[0]),

        "missing":
            int(numeric.isna().sum()),

        "unique_values":
            int(available.nunique()),

        "minimum":
            float(available.min()),

        "maximum":
            float(available.max()),

        "mean":
            float(available.mean()),

        "standard_deviation":
            float(available.std(ddof=1)),

        "quantiles": {
            f"{int(quantile * 100):02d}%":
                float(value)
            for quantile, value
            in quantiles.items()
        },
    }


def boolean_profile(
    series,
):
    """
    Produce a Boolean-value profile.
    """

    values = series.astype("boolean")

    return {
        "rows":
            int(len(values)),

        "true":
            int(values.eq(True).sum()),

        "false":
            int(values.eq(False).sum()),

        "missing":
            int(values.isna().sum()),
    }


def categorical_profile(
    series,
):
    """
    Produce an exact categorical frequency profile.
    """

    counts = (
        series.astype("string")
        .fillna("<MISSING>")
        .value_counts(
            dropna=False
        )
        .sort_index()
    )

    return {
        str(category): int(count)
        for category, count
        in counts.items()
    }


def find_mentions(
    node,
    target_terms,
    current_path="root",
):
    """
    Recursively identify locations in a JSON-compatible object where
    comparator-relevant terms appear.
    """

    matches = []

    normalized_targets = [
        str(term).lower()
        for term in target_terms
    ]

    if isinstance(node, dict):

        for key, value in node.items():

            key_text = str(key)
            key_lower = key_text.lower()
            child_path = (
                f"{current_path}.{key_text}"
            )

            if any(
                target in key_lower
                for target in normalized_targets
            ):
                matches.append(
                    {
                        "path": child_path,
                        "matched_in": "key",
                        "value": value,
                    }
                )

            matches.extend(
                find_mentions(
                    value,
                    target_terms,
                    child_path,
                )
            )

    elif isinstance(node, list):

        for index, value in enumerate(node):

            child_path = (
                f"{current_path}[{index}]"
            )

            matches.extend(
                find_mentions(
                    value,
                    target_terms,
                    child_path,
                )
            )

    elif isinstance(node, str):

        value_lower = node.lower()

        if any(
            target in value_lower
            for target in normalized_targets
        ):
            matches.append(
                {
                    "path": current_path,
                    "matched_in": "value",
                    "value": node,
                }
            )

    return matches


# --------------------------------------------------------------------------------------------------
# 4. Extract frozen Stage 4A specification sections
# --------------------------------------------------------------------------------------------------

stage4a_definition_inventory = {
    "specification_name":
        stage4a_feature_specification.get(
            "specification_name"
        ),

    "specification_version":
        stage4a_feature_specification.get(
            "specification_version"
        ),

    "status":
        stage4a_feature_specification.get(
            "status"
        ),

    "source_timepoint":
        stage4a_feature_specification.get(
            "source_timepoint"
        ),

    "embedded_data_cutoff":
        stage4a_feature_specification.get(
            "embedded_data_cutoff"
        ),

    "unit_of_analysis":
        stage4a_feature_specification.get(
            "unit_of_analysis"
        ),

    "feature_definitions":
        stage4a_feature_specification.get(
            "feature_definitions"
        ),

    "full_ges_candidate_features":
        stage4a_feature_specification.get(
            "full_ges_candidate_features"
        ),

    "no_star_ges_candidate_features":
        stage4a_feature_specification.get(
            "no_star_ges_candidate_features"
        ),

    "audit_only_features":
        stage4a_feature_specification.get(
            "audit_only_features"
        ),

    "scientific_boundary":
        stage4a_feature_specification.get(
            "scientific_boundary"
        ),

    "not_performed_in_stage4a":
        stage4a_feature_specification.get(
            "not_performed_in_stage4a"
        ),
}


# --------------------------------------------------------------------------------------------------
# 5. Identify comparator-relevant entries inside the Stage 4A specification
# --------------------------------------------------------------------------------------------------

COMPARATOR_SEARCH_TERMS = [
    "review",
    "star",
    "conflict",
    "recency",
    "submitter",
    "scv_count",
    "entropy",
    "disagreement",
    "dominant",
    "effective_group",
]

stage4a_relevant_definition_mentions = find_mentions(
    stage4a_feature_specification.get(
        "feature_definitions",
        {},
    ),
    COMPARATOR_SEARCH_TERMS,
    current_path="feature_definitions",
)


# --------------------------------------------------------------------------------------------------
# 6. Extract frozen Stage 4B transformation parameters
# --------------------------------------------------------------------------------------------------

stage4b_transformation_inventory = {
    "artifact_name":
        stage4b_transform_parameters.get(
            "artifact_name"
        ),

    "version":
        stage4b_transform_parameters.get(
            "version"
        ),

    "status":
        stage4b_transform_parameters.get(
            "status"
        ),

    "source":
        stage4b_transform_parameters.get(
            "source"
        ),

    "recency_transformation":
        stage4b_transform_parameters.get(
            "recency_transformation"
        ),

    "submitter_diversity_transformation":
        stage4b_transform_parameters.get(
            "submitter_diversity_transformation"
        ),

    "review_confidence":
        stage4b_transform_parameters.get(
            "review_confidence"
        ),

    "conflict":
        stage4b_transform_parameters.get(
            "conflict"
        ),

    "random_seed_reserved_for_model_fitting":
        stage4b_transform_parameters.get(
            "random_seed_reserved_for_model_fitting"
        ),

    "scientific_boundary":
        stage4b_transform_parameters.get(
            "scientific_boundary"
        ),

    "software":
        stage4b_transform_parameters.get(
            "software"
        ),
}


# --------------------------------------------------------------------------------------------------
# 7. Extract frozen Stage 4B weak-label rules
# --------------------------------------------------------------------------------------------------

stage4b_rule_inventory = {
    "artifact_name":
        stage4b_weak_label_rules.get(
            "artifact_name"
        ),

    "version":
        stage4b_weak_label_rules.get(
            "version"
        ),

    "status":
        stage4b_weak_label_rules.get(
            "status"
        ),

    "label_encoding":
        stage4b_weak_label_rules.get(
            "label_encoding"
        ),

    "full_model_labeling_functions":
        stage4b_weak_label_rules.get(
            "full_model_labeling_functions"
        ),

    "full_aggregation":
        stage4b_weak_label_rules.get(
            "full_aggregation"
        ),

    "no_star_ablation":
        stage4b_weak_label_rules.get(
            "no_star_ablation"
        ),

    "not_performed":
        stage4b_weak_label_rules.get(
            "not_performed"
        ),
}


# --------------------------------------------------------------------------------------------------
# 8. Profile the frozen comparator-source values
# --------------------------------------------------------------------------------------------------

review_star_numeric = pd.to_numeric(
    comparator_source_features[
        "aggregate_review_stars"
    ],
    errors="coerce",
)

review_star_counts = {
    str(int(star)): int(count)
    for star, count
    in (
        review_star_numeric
        .dropna()
        .value_counts()
        .sort_index()
        .items()
    )
}

source_value_profiles = {
    "aggregate_review_stars": {
        "numeric_profile":
            numeric_profile(
                comparator_source_features[
                    "aggregate_review_stars"
                ]
            ),

        "exact_counts":
            review_star_counts,
    },

    "aggregate_conflict_flag":
        boolean_profile(
            comparator_source_features[
                "aggregate_conflict_flag"
            ]
        ),

    "recency_days":
        numeric_profile(
            comparator_source_features[
                "recency_days"
            ]
        ),

    "recency_years":
        numeric_profile(
            comparator_source_features[
                "recency_years"
            ]
        ),

    "recency_missing_flag":
        boolean_profile(
            comparator_source_features[
                "recency_missing_flag"
            ]
        ),

    "unique_submitter_count":
        numeric_profile(
            comparator_source_features[
                "unique_submitter_count"
            ]
        ),

    "log1p_unique_submitter_count":
        numeric_profile(
            comparator_source_features[
                "log1p_unique_submitter_count"
            ]
        ),

    "scv_count":
        numeric_profile(
            comparator_source_features[
                "scv_count"
            ]
        ),

    "log1p_scv_count":
        numeric_profile(
            comparator_source_features[
                "log1p_scv_count"
            ]
        ),

    "submitter_diversity_ratio":
        numeric_profile(
            comparator_source_features[
                "submitter_diversity_ratio"
            ]
        ),

    "scv_group_disagreement_flag":
        boolean_profile(
            comparator_source_features[
                "scv_group_disagreement_flag"
            ]
        ),

    "scv_group_entropy_normalized":
        numeric_profile(
            comparator_source_features[
                "scv_group_entropy_normalized"
            ]
        ),

    "scv_dominant_group_fraction":
        numeric_profile(
            comparator_source_features[
                "scv_dominant_group_fraction"
            ]
        ),

    "scv_effective_group_count":
        numeric_profile(
            comparator_source_features[
                "scv_effective_group_count"
            ]
        ),
}


# --------------------------------------------------------------------------------------------------
# 9. Reconciliation checks for transformed source fields
# --------------------------------------------------------------------------------------------------

unique_submitter_count = pd.to_numeric(
    comparator_source_features[
        "unique_submitter_count"
    ],
    errors="coerce",
)

log1p_unique_submitter_count = pd.to_numeric(
    comparator_source_features[
        "log1p_unique_submitter_count"
    ],
    errors="coerce",
)

scv_count = pd.to_numeric(
    comparator_source_features[
        "scv_count"
    ],
    errors="coerce",
)

log1p_scv_count = pd.to_numeric(
    comparator_source_features[
        "log1p_scv_count"
    ],
    errors="coerce",
)

submitter_diversity_ratio = pd.to_numeric(
    comparator_source_features[
        "submitter_diversity_ratio"
    ],
    errors="coerce",
)

recency_days = pd.to_numeric(
    comparator_source_features[
        "recency_days"
    ],
    errors="coerce",
)

recency_years = pd.to_numeric(
    comparator_source_features[
        "recency_years"
    ],
    errors="coerce",
)

recency_missing_flag = (
    comparator_source_features[
        "recency_missing_flag"
    ]
    .astype("boolean")
)

log_submitter_reconstruction_mismatches = int(
    (
        log1p_unique_submitter_count.notna()
        &
        unique_submitter_count.notna()
        &
        ~np.isclose(
            log1p_unique_submitter_count,
            np.log1p(
                unique_submitter_count
            ),
            atol=1e-12,
            rtol=1e-12,
        )
    ).sum()
)

log_scv_reconstruction_mismatches = int(
    (
        log1p_scv_count.notna()
        &
        scv_count.notna()
        &
        ~np.isclose(
            log1p_scv_count,
            np.log1p(
                scv_count
            ),
            atol=1e-12,
            rtol=1e-12,
        )
    ).sum()
)

submitter_diversity_reconstruction_mismatches = int(
    (
        submitter_diversity_ratio.notna()
        &
        unique_submitter_count.notna()
        &
        scv_count.notna()
        &
        scv_count.gt(0)
        &
        ~np.isclose(
            submitter_diversity_ratio,
            unique_submitter_count
            / scv_count,
            atol=1e-12,
            rtol=1e-12,
        )
    ).sum()
)

recency_year_reconstruction_mismatches = int(
    (
        recency_days.notna()
        &
        recency_years.notna()
        &
        ~np.isclose(
            recency_years,
            recency_days / 365.25,
            atol=1e-10,
            rtol=1e-10,
        )
    ).sum()
)

recency_missingness_mismatches = int(
    (
        recency_missing_flag
        .fillna(True)
        .ne(
            recency_days.isna()
        )
    ).sum()
)


# --------------------------------------------------------------------------------------------------
# 10. Create the in-memory frozen-definition inventory for the next cell
# --------------------------------------------------------------------------------------------------

comparator_definition_inventory = {
    "inventory_status":
        "FROZEN_SOURCE_DEFINITIONS_INVENTORIED",

    "stage4a_definition_inventory":
        stage4a_definition_inventory,

    "stage4a_relevant_definition_mentions":
        stage4a_relevant_definition_mentions,

    "stage4b_transformation_inventory":
        stage4b_transformation_inventory,

    "stage4b_rule_inventory":
        stage4b_rule_inventory,

    "comparator_source_field_groups":
        COMPARATOR_SOURCE_FIELD_GROUPS,

    "source_value_profiles":
        source_value_profiles,

    "transformation_reconciliation": {
        "log1p_unique_submitter_count_mismatches":
            log_submitter_reconstruction_mismatches,

        "log1p_scv_count_mismatches":
            log_scv_reconstruction_mismatches,

        "submitter_diversity_ratio_mismatches":
            submitter_diversity_reconstruction_mismatches,

        "recency_years_mismatches":
            recency_year_reconstruction_mismatches,

        "recency_missingness_mismatches":
            recency_missingness_mismatches,
    },

    "scientific_boundary": {
        "stage5_outcome_table_opened": False,
        "outcome_labels_loaded": False,
        "score_outcome_join_created": False,
        "comparator_formula_selected": False,
        "comparator_scores_constructed": False,
        "temporal_performance_examined": False,
        "files_written_or_modified": False,
    },
}


# --------------------------------------------------------------------------------------------------
# 11. Print the frozen-definition inventory
# --------------------------------------------------------------------------------------------------

print("=" * 124)
print("STAGE 6A STEP 2 — CELL 2B — FROZEN COMPARATOR-DEFINITION INVENTORY")
print("=" * 124)

print_json_section(
    "STAGE 4A SPECIFICATION IDENTITY AND BOUNDARY",
    {
        key: value
        for key, value
        in stage4a_definition_inventory.items()
        if key != "feature_definitions"
    },
)

print_json_section(
    "STAGE 4A COMPARATOR-RELEVANT FEATURE-DEFINITION MENTIONS",
    stage4a_relevant_definition_mentions,
)

print_json_section(
    "STAGE 4B FROZEN TRANSFORMATION PARAMETERS",
    stage4b_transformation_inventory,
)

print_json_section(
    "STAGE 4B FROZEN WEAK-LABEL RULES",
    stage4b_rule_inventory,
)

print_json_section(
    "COMPARATOR SOURCE-FIELD GROUPS",
    COMPARATOR_SOURCE_FIELD_GROUPS,
)

print_json_section(
    "OBSERVED FROZEN SOURCE-VALUE PROFILES",
    source_value_profiles,
)

print_json_section(
    "TRANSFORMATION RECONCILIATION",
    comparator_definition_inventory[
        "transformation_reconciliation"
    ],
)


# --------------------------------------------------------------------------------------------------
# 12. Final inventory checks
# --------------------------------------------------------------------------------------------------

cell2b_checks = {
    "all comparator-source fields present":
        len(
            missing_inventory_fields
        )
        == 0,

    "Stage 4A feature definitions present":
        stage4a_definition_inventory[
            "feature_definitions"
        ]
        not in (
            None,
            {},
            [],
        ),

    "comparator-relevant feature mentions found":
        len(
            stage4a_relevant_definition_mentions
        )
        > 0,

    "recency transformation present":
        stage4b_transformation_inventory[
            "recency_transformation"
        ]
        is not None,

    "submitter transformation present":
        stage4b_transformation_inventory[
            "submitter_diversity_transformation"
        ]
        is not None,

    "review-confidence transformation present":
        stage4b_transformation_inventory[
            "review_confidence"
        ]
        is not None,

    "conflict transformation present":
        stage4b_transformation_inventory[
            "conflict"
        ]
        is not None,

    "full-model labeling functions present":
        stage4b_rule_inventory[
            "full_model_labeling_functions"
        ]
        is not None,

    "full aggregation rule present":
        stage4b_rule_inventory[
            "full_aggregation"
        ]
        is not None,

    "no-star ablation rule present":
        stage4b_rule_inventory[
            "no_star_ablation"
        ]
        is not None,

    "log submitter transformation reconstructs":
        log_submitter_reconstruction_mismatches
        == 0,

    "log SCV transformation reconstructs":
        log_scv_reconstruction_mismatches
        == 0,

    "submitter diversity ratio reconstructs":
        submitter_diversity_reconstruction_mismatches
        == 0,

    "recency years reconstruct":
        recency_year_reconstruction_mismatches
        == 0,

    "recency missingness reconstructs":
        recency_missingness_mismatches
        == 0,
}

failed_cell2b_checks = [
    check_name
    for check_name, passed
    in cell2b_checks.items()
    if not passed
]


# --------------------------------------------------------------------------------------------------
# 13. Final decision
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 124)
print("STAGE 6A STEP 2 — CELL 2B DECISION")
print("=" * 124)

for check_name, passed in cell2b_checks.items():
    print(
        f"{check_name:<72} "
        f"{'PASS' if passed else 'FAIL'}"
    )

if failed_cell2b_checks:

    print()
    print(
        "FAIL_STAGE6A_FROZEN_COMPARATOR_DEFINITION_INVENTORY"
    )

    print("\nFailed checks:")

    for failed_check in failed_cell2b_checks:
        print(
            f" - {failed_check}"
        )

    raise RuntimeError(
        "The frozen comparator-definition inventory did not pass all checks."
    )

print()
print(
    "PASS_STAGE6A_FROZEN_COMPARATOR_DEFINITIONS_INVENTORIED"
)

print()
print(
    "In-memory inventory created:                 "
    "comparator_definition_inventory"
)
print(
    "Stage 5 outcome table opened:                NO"
)
print(
    "Outcome labels loaded:                       NO"
)
print(
    "Comparator formula selected:                 NO"
)
print(
    "Comparator scores constructed:              NO"
)
print(
    "Temporal performance examined:              NO"
)
print(
    "Files written or modified:                   NO"
)
print()
print(
    "NEXT AUTHORIZED ACTION:"
)
print(
    "Define and checksum-freeze the exact comparator policy, including "
    "directionality, scaling, missing-value handling, formulas, and output schema."
)

STAGE 6A STEP 2 — CELL 2B — FROZEN COMPARATOR-DEFINITION INVENTORY

STAGE 4A SPECIFICATION IDENTITY AND BOUNDARY
----------------------------------------------------------------------------------------------------------------------------
{
  "audit_only_features": [
    "scv_count",
    "log1p_scv_count",
    "unique_submitter_count",
    "submitter_diversity_ratio",
    "scv_group_disagreement_flag",
    "scv_nonzero_group_count",
    "scv_group_entropy_nats",
    "scv_group_entropy_bits",
    "scv_dominant_group_fraction",
    "scv_effective_group_count"
  ],
  "embedded_data_cutoff": "2022-12-31",
  "full_ges_candidate_features": [
    "recency_years",
    "recency_missing_flag",
    "log1p_unique_submitter_count",
    "aggregate_review_stars",
    "aggregate_conflict_flag",
    "scv_group_entropy_normalized"
  ],
  "no_star_ges_candidate_features": [
    "recency_years",
    "recency_missing_flag",
    "log1p_unique_submitter_count",
    "aggregate_conflict_flag",
    "scv_group_en

In [ ]:
# ==================================================================================================
# STAGE 6A — STEP 2 — CELL 3
# DEFINE AND CHECKSUM-FREEZE THE PRESPECIFIED COMPARATOR-SCORE POLICY
#
# Purpose:
#   1. Define the direction, scaling, formulas, thresholds, and missing-value treatment
#      for every prespecified comparator.
#   2. Define the strong equal-weight combined-metadata heuristic.
#   3. Define how frozen full-GES and no-star P(stable) values will be converted to
#      future-instability risk for evaluation.
#   4. Freeze the policy as deterministic JSON with a SHA-256 sidecar.
#
# Scientific boundary:
#   - No Stage 5 outcome table is opened.
#   - No outcome label is loaded.
#   - No score-outcome join is created.
#   - No comparator score is calculated in this cell.
#   - No temporal performance is examined.
#   - Only the comparator policy JSON and its SHA-256 sidecar may be written.
# ==================================================================================================

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import os
import platform
import sys

import numpy as np
import pandas as pd


# --------------------------------------------------------------------------------------------------
# 1. Confirm required verified objects remain available
# --------------------------------------------------------------------------------------------------

REQUIRED_RUNTIME_OBJECTS = [
    "PROJECT_ROOT",
    "comparator_source_features",
    "stage4b_transform_parameters",
    "stage4b_weak_label_rules",
]

missing_runtime_objects = [
    object_name
    for object_name in REQUIRED_RUNTIME_OBJECTS
    if object_name not in globals()
]

if missing_runtime_objects:
    raise RuntimeError(
        "Required verified runtime objects are missing:\n"
        + "\n".join(
            f" - {object_name}"
            for object_name in missing_runtime_objects
        )
        + "\nRerun the preceding Stage 6A Step 2 cells before continuing."
    )


# --------------------------------------------------------------------------------------------------
# 2. Source artifact paths
# --------------------------------------------------------------------------------------------------

STAGE4A_FEATURE_TABLE_PATH = (
    PROJECT_ROOT
    / "data_processed"
    / "stage4_ges"
    / "stage4a_t0_ges_baseline_features_v1.parquet"
)

STAGE4A_FEATURE_SPECIFICATION_PATH = (
    PROJECT_ROOT
    / "configs"
    / "stage4_ges"
    / "stage4a_t0_feature_specification_v1.json"
)

STAGE4A_FREEZE_MANIFEST_PATH = (
    PROJECT_ROOT
    / "configs"
    / "stage4_ges"
    / "stage4a_t0_feature_freeze_manifest_v1.json"
)

STAGE4B_TRANSFORM_PARAMETERS_PATH = (
    PROJECT_ROOT
    / "configs"
    / "stage4_ges"
    / "stage4b_t0_feature_transform_parameters_v1.json"
)

STAGE4B_WEAK_LABEL_RULES_PATH = (
    PROJECT_ROOT
    / "configs"
    / "stage4_ges"
    / "stage4b_weak_label_rules_v1.json"
)

STAGE4B_FREEZE_MANIFEST_PATH = (
    PROJECT_ROOT
    / "configs"
    / "stage4_ges"
    / "stage4b_weak_label_freeze_manifest_v1.json"
)

STAGE4C_SCORE_TABLE_PATH = (
    PROJECT_ROOT
    / "data_processed"
    / "stage4_ges"
    / "stage4c_t0_full_and_no_star_ges_scores_v1.parquet"
)

STAGE4C_FREEZE_MANIFEST_PATH = (
    PROJECT_ROOT
    / "configs"
    / "stage4_ges"
    / "stage4c_ges_model_freeze_manifest_v1.json"
)


# --------------------------------------------------------------------------------------------------
# 3. Expected frozen source checksums
# --------------------------------------------------------------------------------------------------

EXPECTED_SOURCE_SHA256 = {
    "stage4a_feature_table":
        "c100b3781e6801425f622f5d091376abfe0939e48c0a792af32f6eebe6401f16",

    "stage4a_feature_specification":
        "fc00146efe5da9b3fbe740bb42ca99d157252cdefc045650f9e88d54d8fcfa8b",

    "stage4a_freeze_manifest":
        "2b844ef2dbc0c3e5e57493886a7533587350e1b4b4c76fc9d445ecda8a528fa0",

    "stage4b_transform_parameters":
        "baeab167e19381138f93c51ba2fa00e4eeed684b36122c80cd2bf9fc4fe4b08d",

    "stage4b_weak_label_rules":
        "3d78e66cea1fed5c75ef1cab1b7cf44d3d3d7bfae50909173bae6c3a0e0bff61",

    "stage4b_freeze_manifest":
        "e766061442e6d4610f661f44a619e41b45dc6363b28f6176d3f9a71f8215c63f",

    "stage4c_score_table":
        "d871ee9087f83be2b0ee954d283aa92212a639a35e3ea26d5cf042e83019f5ac",

    "stage4c_freeze_manifest":
        "c0d8008a4db80c67f5b1c568ddba3496b2411b29bce0db1eb20e63f26613d4ee",
}

SOURCE_PATHS = {
    "stage4a_feature_table":
        STAGE4A_FEATURE_TABLE_PATH,

    "stage4a_feature_specification":
        STAGE4A_FEATURE_SPECIFICATION_PATH,

    "stage4a_freeze_manifest":
        STAGE4A_FREEZE_MANIFEST_PATH,

    "stage4b_transform_parameters":
        STAGE4B_TRANSFORM_PARAMETERS_PATH,

    "stage4b_weak_label_rules":
        STAGE4B_WEAK_LABEL_RULES_PATH,

    "stage4b_freeze_manifest":
        STAGE4B_FREEZE_MANIFEST_PATH,

    "stage4c_score_table":
        STAGE4C_SCORE_TABLE_PATH,

    "stage4c_freeze_manifest":
        STAGE4C_FREEZE_MANIFEST_PATH,
}


# --------------------------------------------------------------------------------------------------
# 4. Comparator-policy output paths
# --------------------------------------------------------------------------------------------------

STAGE6_CONFIG_DIR = (
    PROJECT_ROOT
    / "configs"
    / "stage6_temporal_validation"
)

COMPARATOR_POLICY_PATH = (
    STAGE6_CONFIG_DIR
    / "stage6a_comparator_score_policy_v1.json"
)

COMPARATOR_POLICY_SHA256_PATH = (
    COMPARATOR_POLICY_PATH.with_name(
        COMPARATOR_POLICY_PATH.name
        + ".sha256"
    )
)

PLANNED_COMPARATOR_SCORE_TABLE_PATH = (
    PROJECT_ROOT
    / "data_processed"
    / "stage6_temporal_validation"
    / "stage6a_t0_comparator_scores_v1.parquet"
)


# --------------------------------------------------------------------------------------------------
# 5. Helpers
# --------------------------------------------------------------------------------------------------

def calculate_sha256(
    file_path: Path,
    block_size: int = 8 * 1024 * 1024,
) -> str:
    """
    Calculate SHA-256 without loading an entire artifact into memory.
    """

    digest = hashlib.sha256()

    with file_path.open("rb") as file_handle:
        for block in iter(
            lambda: file_handle.read(block_size),
            b"",
        ):
            digest.update(block)

    return digest.hexdigest()


def deterministic_json_text(
    value,
) -> str:
    """
    Serialize JSON deterministically for checksum freezing.
    """

    return (
        json.dumps(
            value,
            indent=2,
            sort_keys=True,
            ensure_ascii=False,
            allow_nan=False,
        )
        + "\n"
    )


def atomic_write_text(
    target_path: Path,
    text: str,
) -> None:
    """
    Write text through a temporary file and atomically replace the target.
    """

    temporary_path = target_path.with_name(
        target_path.name
        + ".tmp"
    )

    temporary_path.write_text(
        text,
        encoding="utf-8",
    )

    os.replace(
        temporary_path,
        target_path,
    )


# --------------------------------------------------------------------------------------------------
# 6. Reverify all frozen source artifacts before policy creation
# --------------------------------------------------------------------------------------------------

missing_source_paths = [
    path
    for path in SOURCE_PATHS.values()
    if not path.exists()
]

if missing_source_paths:
    raise FileNotFoundError(
        "Required frozen source artifacts are missing:\n"
        + "\n".join(
            f" - {path}"
            for path in missing_source_paths
        )
    )

observed_source_sha256 = {
    artifact_name:
        calculate_sha256(
            artifact_path
        )
    for artifact_name, artifact_path
    in SOURCE_PATHS.items()
}

source_checksum_checks = {
    artifact_name:
        observed_source_sha256[
            artifact_name
        ]
        == EXPECTED_SOURCE_SHA256[
            artifact_name
        ]
    for artifact_name
    in SOURCE_PATHS
}

failed_source_checks = [
    artifact_name
    for artifact_name, passed
    in source_checksum_checks.items()
    if not passed
]

if failed_source_checks:
    raise RuntimeError(
        "Frozen source checksum verification failed:\n"
        + "\n".join(
            f" - {artifact_name}"
            for artifact_name in failed_source_checks
        )
    )


# --------------------------------------------------------------------------------------------------
# 7. Derive outcome-blind constants from frozen T0 comparator-source fields
# --------------------------------------------------------------------------------------------------

review_stars = pd.to_numeric(
    comparator_source_features[
        "aggregate_review_stars"
    ],
    errors="coerce",
)

conflict_flag = (
    comparator_source_features[
        "aggregate_conflict_flag"
    ]
    .astype("boolean")
)

recency_days = pd.to_numeric(
    comparator_source_features[
        "recency_days"
    ],
    errors="coerce",
)

recency_missing_flag = (
    comparator_source_features[
        "recency_missing_flag"
    ]
    .astype("boolean")
)

unique_submitter_count = pd.to_numeric(
    comparator_source_features[
        "unique_submitter_count"
    ],
    errors="coerce",
)

log1p_unique_submitter_count = pd.to_numeric(
    comparator_source_features[
        "log1p_unique_submitter_count"
    ],
    errors="coerce",
)

entropy_normalized = pd.to_numeric(
    comparator_source_features[
        "scv_group_entropy_normalized"
    ],
    errors="coerce",
)

observed_review_star_levels = sorted(
    int(value)
    for value
    in review_stars.dropna().unique()
)

review_star_scaling_maximum = float(
    max(
        observed_review_star_levels
    )
)

recency_maximum_window_days = float(
    stage4b_transform_parameters[
        "recency_transformation"
    ][
        "maximum_observation_window_days"
    ]
)

recency_median_days = float(
    recency_days.median(
        skipna=True
    )
)

recency_median_instability_risk = float(
    np.clip(
        recency_median_days
        / recency_maximum_window_days,
        0.0,
        1.0,
    )
)

submitter_minimum_log_count = float(
    stage4b_transform_parameters[
        "submitter_diversity_transformation"
    ][
        "minimum_log_count"
    ]
)

submitter_maximum_log_count = float(
    stage4b_transform_parameters[
        "submitter_diversity_transformation"
    ][
        "maximum_log_count"
    ]
)

submitter_log_range = float(
    submitter_maximum_log_count
    - submitter_minimum_log_count
)

stale_recency_stability_threshold = 0.30

stale_recency_instability_threshold = float(
    1.0
    - stale_recency_stability_threshold
)

additive_component_count = 6

combined_component_count = 6

equal_component_weight = float(
    1.0
    / combined_component_count
)


# --------------------------------------------------------------------------------------------------
# 8. Validate constants and complete-source expectations
# --------------------------------------------------------------------------------------------------

constant_checks = {
    "review-star levels are exactly 0,1,2,3":
        observed_review_star_levels
        == [0, 1, 2, 3],

    "review-star scaling maximum is positive":
        review_star_scaling_maximum
        == 3.0,

    "review stars are complete":
        int(
            review_stars.isna().sum()
        )
        == 0,

    "conflict flags are complete":
        int(
            conflict_flag.isna().sum()
        )
        == 0,

    "conflict-positive count matches freeze":
        int(
            conflict_flag.eq(True).sum()
        )
        == 1_484,

    "recency maximum matches frozen transformation":
        np.isclose(
            recency_days.max(
                skipna=True
            ),
            recency_maximum_window_days,
            atol=0,
            rtol=0,
        ),

    "recency median is available":
        np.isfinite(
            recency_median_days
        ),

    "recency missingness agrees":
        int(
            recency_missing_flag
            .fillna(True)
            .ne(
                recency_days.isna()
            )
            .sum()
        )
        == 0,

    "submitter counts are complete":
        int(
            unique_submitter_count
            .isna()
            .sum()
        )
        == 0,

    "log submitter counts are complete":
        int(
            log1p_unique_submitter_count
            .isna()
            .sum()
        )
        == 0,

    "submitter minimum matches frozen transformation":
        np.isclose(
            log1p_unique_submitter_count.min(),
            submitter_minimum_log_count,
            atol=1e-12,
            rtol=1e-12,
        ),

    "submitter maximum matches frozen transformation":
        np.isclose(
            log1p_unique_submitter_count.max(),
            submitter_maximum_log_count,
            atol=1e-12,
            rtol=1e-12,
        ),

    "submitter scaling range is positive":
        submitter_log_range
        > 0,

    "entropy is complete":
        int(
            entropy_normalized
            .isna()
            .sum()
        )
        == 0,

    "entropy is restricted to [0,1]":
        bool(
            entropy_normalized
            .between(
                0.0,
                1.0,
                inclusive="both",
            )
            .all()
        ),

    "equal weights sum to one":
        np.isclose(
            equal_component_weight
            * combined_component_count,
            1.0,
            atol=1e-15,
            rtol=0,
        ),
}

failed_constant_checks = [
    check_name
    for check_name, passed
    in constant_checks.items()
    if not passed
]

if failed_constant_checks:
    raise RuntimeError(
        "Comparator-policy constants did not pass verification:\n"
        + "\n".join(
            f" - {check_name}"
            for check_name in failed_constant_checks
        )
    )


# --------------------------------------------------------------------------------------------------
# 9. Define the exact frozen comparator policy
# --------------------------------------------------------------------------------------------------

COMPARATOR_OUTPUT_COLUMNS_IN_ORDER = [
    "t0_row_order",
    "rcv_accession",

    "review_stars_instability_risk",
    "conflict_instability_risk",
    "recency_instability_risk",
    "recency_missing_instability_component",
    "submitter_instability_risk",
    "entropy_instability_risk",

    "additive_low_review_indicator",
    "additive_conflict_indicator",
    "additive_stale_recency_indicator",
    "additive_missing_recency_indicator",
    "additive_single_submitter_indicator",
    "additive_entropy_indicator",
    "additive_risk_count",
    "additive_instability_risk",

    "combined_metadata_instability_risk",

    "full_ges_p_stable_t0",
    "full_ges_instability_risk_t0",

    "no_star_ges_p_stable_t0",
    "no_star_ges_instability_risk_t0",

    "comparator_policy_version",
    "comparator_policy_sha256",
    "stage6a_step2_version",

    "outcome_labels_loaded",
    "temporal_performance_evaluated",
]

policy_core = {
    "policy_id":
        "GES_STAGE6A_COMPARATOR_SCORE_POLICY",

    "policy_name":
        "Stage 6A prespecified comparator-score policy",

    "version":
        "1.0.0",

    "status":
        "FROZEN_BEFORE_OUTCOME_LABEL_LOAD_OR_TEMPORAL_PERFORMANCE",

    "study_phase":
        "Experiment 1 Stage 6A Step 2",

    "unit_of_analysis":
        "RCV-level variant-condition aggregate",

    "positive_class":
        "Primary future-instability outcome equals 1",

    "global_score_direction":
        "Higher values indicate greater predicted future-instability risk.",

    "global_score_range": [
        0.0,
        1.0,
    ],

    "interpretation":
        (
            "Comparator outputs are deterministic ordinal or heuristic risk scores. "
            "They are not fitted or outcome-calibrated probabilities. Discrimination "
            "comparisons use the frozen score ordering; calibration and Brier analyses "
            "must identify these raw comparator values as uncalibrated risk scores."
        ),

    "source_artifacts": {
        artifact_name: {
            "path":
                str(
                    SOURCE_PATHS[
                        artifact_name
                    ]
                ),

            "sha256":
                EXPECTED_SOURCE_SHA256[
                    artifact_name
                ],
        }
        for artifact_name
        in SOURCE_PATHS
    },

    "frozen_constants": {
        "review_star_observed_levels":
            observed_review_star_levels,

        "review_star_scaling_maximum":
            review_star_scaling_maximum,

        "recency_maximum_observation_window_days":
            recency_maximum_window_days,

        "recency_median_days_for_missing_imputation":
            recency_median_days,

        "recency_median_instability_risk_for_missing_imputation":
            recency_median_instability_risk,

        "submitter_minimum_log1p_count":
            submitter_minimum_log_count,

        "submitter_maximum_log1p_count":
            submitter_maximum_log_count,

        "submitter_log1p_range":
            submitter_log_range,

        "stale_recency_stability_threshold":
            stale_recency_stability_threshold,

        "stale_recency_instability_threshold":
            stale_recency_instability_threshold,

        "single_submitter_threshold":
            1,

        "entropy_positive_threshold":
            0.0,

        "additive_component_count":
            additive_component_count,

        "combined_component_count":
            combined_component_count,

        "equal_combined_component_weight":
            equal_component_weight,
    },

    "score_definitions": {
        "review_stars_baseline": {
            "output_column":
                "review_stars_instability_risk",

            "source_columns": [
                "aggregate_review_stars",
            ],

            "formula":
                (
                    "clip((review_star_scaling_maximum - "
                    "aggregate_review_stars) / "
                    "review_star_scaling_maximum, 0, 1)"
                ),

            "direction":
                "Lower review stars produce higher instability risk.",

            "missing_value_policy":
                "Missing values are not permitted; construction must fail.",
        },

        "conflict_baseline": {
            "output_column":
                "conflict_instability_risk",

            "source_columns": [
                "aggregate_conflict_flag",
            ],

            "formula":
                "1.0 when aggregate_conflict_flag is True, otherwise 0.0",

            "direction":
                "Active aggregate conflict produces maximum instability risk.",

            "missing_value_policy":
                "Missing values are not permitted; construction must fail.",
        },

        "recency_baseline": {
            "output_column":
                "recency_instability_risk",

            "source_columns": [
                "recency_days",
                "recency_missing_flag",
            ],

            "formula_when_available":
                (
                    "clip(recency_days / "
                    "recency_maximum_observation_window_days, 0, 1)"
                ),

            "formula_when_missing":
                "Use the frozen T0 median instability-risk value.",

            "frozen_missing_imputation_value":
                recency_median_instability_risk,

            "direction":
                "Older evaluations produce higher instability risk.",

            "missing_value_policy":
                (
                    "Median imputation is calculated from the frozen T0 recency "
                    "distribution without loading outcomes. Missingness remains "
                    "separately represented by recency_missing_flag."
                ),
        },

        "recency_missing_component": {
            "output_column":
                "recency_missing_instability_component",

            "source_columns": [
                "recency_missing_flag",
            ],

            "formula":
                "1.0 when recency_missing_flag is True, otherwise 0.0",

            "use":
                (
                    "Used in the additive and strong combined-metadata baselines, "
                    "but not substituted for the standalone recency score."
                ),
        },

        "submitter_baseline": {
            "output_column":
                "submitter_instability_risk",

            "source_columns": [
                "log1p_unique_submitter_count",
            ],

            "normalized_support_formula":
                (
                    "clip((log1p_unique_submitter_count - "
                    "submitter_minimum_log1p_count) / "
                    "submitter_log1p_range, 0, 1)"
                ),

            "instability_formula":
                "1.0 - normalized_submitter_support",

            "direction":
                "Fewer unique submitters produce higher instability risk.",

            "missing_value_policy":
                "Missing values are not permitted; construction must fail.",
        },

        "entropy_component": {
            "output_column":
                "entropy_instability_risk",

            "source_columns": [
                "scv_group_entropy_normalized",
            ],

            "formula":
                "clip(scv_group_entropy_normalized, 0, 1)",

            "direction":
                "Greater SCV classification-group entropy produces higher risk.",

            "missing_value_policy":
                "Missing values are not permitted; construction must fail.",
        },

        "additive_risk_baseline": {
            "output_count_column":
                "additive_risk_count",

            "output_normalized_column":
                "additive_instability_risk",

            "method":
                "Unweighted count of six prespecified binary T0 risk indicators.",

            "indicators": {
                "additive_low_review_indicator":
                    "1 when aggregate_review_stars == 0, otherwise 0",

                "additive_conflict_indicator":
                    "1 when aggregate_conflict_flag is True, otherwise 0",

                "additive_stale_recency_indicator":
                    (
                        "1 when recency is available and recency stability score "
                        "< 0.30, equivalently recency instability risk > 0.70; "
                        "otherwise 0"
                    ),

                "additive_missing_recency_indicator":
                    "1 when recency_missing_flag is True, otherwise 0",

                "additive_single_submitter_indicator":
                    "1 when unique_submitter_count == 1, otherwise 0",

                "additive_entropy_indicator":
                    (
                        "1 when scv_group_entropy_normalized > 0.0, "
                        "otherwise 0"
                    ),
            },

            "count_formula":
                "Sum the six binary indicators.",

            "normalized_formula":
                "additive_risk_count / 6.0",

            "range":
                [
                    0.0,
                    1.0,
                ],

            "missing_recency_rule":
                (
                    "When recency is missing, stale-recency indicator equals 0 "
                    "and missing-recency indicator equals 1."
                ),
        },

        "strong_combined_metadata_heuristic": {
            "output_column":
                "combined_metadata_instability_risk",

            "method":
                "Equal-weight arithmetic mean of six outcome-blind T0 components.",

            "components": [
                "review_stars_instability_risk",
                "conflict_instability_risk",
                "recency_instability_risk",
                "recency_missing_instability_component",
                "submitter_instability_risk",
                "entropy_instability_risk",
            ],

            "component_weights": {
                "review_stars_instability_risk":
                    equal_component_weight,

                "conflict_instability_risk":
                    equal_component_weight,

                "recency_instability_risk":
                    equal_component_weight,

                "recency_missing_instability_component":
                    equal_component_weight,

                "submitter_instability_risk":
                    equal_component_weight,

                "entropy_instability_risk":
                    equal_component_weight,
            },

            "formula":
                (
                    "Arithmetic mean of the six listed components, "
                    "equivalent to their sum divided by 6.0."
                ),

            "range":
                [
                    0.0,
                    1.0,
                ],

            "outcome_fitting":
                False,

            "weight_optimization":
                False,
        },

        "full_ges_evaluation_score": {
            "source_column":
                "full_ges_p_stable_t0",

            "output_column":
                "full_ges_instability_risk_t0",

            "formula":
                "1.0 - full_ges_p_stable_t0",

            "direction":
                "Lower frozen P(stable) produces higher future-instability risk.",

            "coefficient_refitting":
                False,
        },

        "no_star_ges_evaluation_score": {
            "source_column":
                "no_star_ges_p_stable_t0",

            "output_column":
                "no_star_ges_instability_risk_t0",

            "formula":
                "1.0 - no_star_ges_p_stable_t0",

            "direction":
                "Lower frozen no-star P(stable) produces higher instability risk.",

            "review_status_used":
                False,

            "coefficient_refitting":
                False,
        },
    },

    "primary_comparison_set": [
        "full_ges_instability_risk_t0",
        "no_star_ges_instability_risk_t0",
        "review_stars_instability_risk",
        "conflict_instability_risk",
        "recency_instability_risk",
        "submitter_instability_risk",
        "additive_instability_risk",
        "combined_metadata_instability_risk",
    ],

    "planned_output": {
        "path":
            str(
                PLANNED_COMPARATOR_SCORE_TABLE_PATH
            ),

        "format":
            "Parquet",

        "expected_rows":
            71_659,

        "column_order":
            COMPARATOR_OUTPUT_COLUMNS_IN_ORDER,

        "key":
            "rcv_accession",

        "row_order":
            "Exact frozen zero-based t0_row_order from 0 through 71,658.",

        "policy_hash_provenance":
            (
                "Every output row will contain this comparator-policy SHA-256."
            ),
    },

    "construction_requirements": {
        "preserve_exact_t0_row_order":
            True,

        "require_unique_rcv_accession":
            True,

        "require_all_scores_in_unit_interval":
            True,

        "require_exact_policy_version":
            "1.0.0",

        "require_policy_sha256_on_every_row":
            True,

        "allow_outcome_table_load":
            False,

        "allow_outcome_label_load":
            False,

        "allow_score_outcome_join":
            False,

        "allow_threshold_optimization":
            False,

        "allow_weight_optimization":
            False,

        "allow_temporal_performance_calculation":
            False,
    },

    "scientific_boundary": {
        "stage5_outcome_table_opened":
            False,

        "outcome_labels_loaded":
            False,

        "score_outcome_join_created":
            False,

        "comparator_scores_constructed":
            False,

        "comparator_weights_selected_using_outcomes":
            False,

        "thresholds_selected_using_outcomes":
            False,

        "temporal_performance_examined":
            False,

        "policy_frozen_before_score_construction":
            True,
    },

    "software": {
        "python":
            sys.version,

        "numpy":
            np.__version__,

        "pandas":
            pd.__version__,

        "platform":
            platform.platform(),
    },
}


# --------------------------------------------------------------------------------------------------
# 10. Create or validate the immutable policy artifact
# --------------------------------------------------------------------------------------------------

STAGE6_CONFIG_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

policy_file_created = False
sidecar_file_created = False
existing_policy_reused = False

if COMPARATOR_POLICY_PATH.exists():

    existing_policy = json.loads(
        COMPARATOR_POLICY_PATH.read_text(
            encoding="utf-8"
        )
    )

    existing_policy_core = dict(
        existing_policy
    )

    existing_policy_core.pop(
        "created_at_utc",
        None,
    )

    if existing_policy_core != policy_core:
        raise RuntimeError(
            "A comparator-policy file already exists but does not match "
            "the policy being frozen in this cell.\n"
            "The existing file was not overwritten:\n"
            f"{COMPARATOR_POLICY_PATH}"
        )

    comparator_policy = existing_policy
    existing_policy_reused = True

else:

    if COMPARATOR_POLICY_SHA256_PATH.exists():
        raise RuntimeError(
            "A policy SHA-256 sidecar exists without the corresponding policy file. "
            "No artifact was written.\n"
            f"{COMPARATOR_POLICY_SHA256_PATH}"
        )

    comparator_policy = {
        **policy_core,

        "created_at_utc":
            datetime.now(
                timezone.utc
            )
            .isoformat()
            .replace(
                "+00:00",
                "Z",
            ),
    }

    atomic_write_text(
        COMPARATOR_POLICY_PATH,
        deterministic_json_text(
            comparator_policy
        ),
    )

    policy_file_created = True


# --------------------------------------------------------------------------------------------------
# 11. Calculate and freeze the policy checksum
# --------------------------------------------------------------------------------------------------

comparator_policy_sha256 = calculate_sha256(
    COMPARATOR_POLICY_PATH
)

expected_sidecar_text = (
    f"{comparator_policy_sha256}  "
    f"{COMPARATOR_POLICY_PATH.name}\n"
)

if COMPARATOR_POLICY_SHA256_PATH.exists():

    observed_sidecar_text = (
        COMPARATOR_POLICY_SHA256_PATH
        .read_text(
            encoding="utf-8"
        )
    )

    if observed_sidecar_text != expected_sidecar_text:
        raise RuntimeError(
            "The existing policy SHA-256 sidecar does not match the frozen policy.\n"
            "No sidecar was overwritten:\n"
            f"{COMPARATOR_POLICY_SHA256_PATH}"
        )

else:

    atomic_write_text(
        COMPARATOR_POLICY_SHA256_PATH,
        expected_sidecar_text,
    )

    sidecar_file_created = True


# --------------------------------------------------------------------------------------------------
# 12. Read back and independently verify the frozen policy
# --------------------------------------------------------------------------------------------------

readback_policy_text = (
    COMPARATOR_POLICY_PATH
    .read_text(
        encoding="utf-8"
    )
)

readback_policy = json.loads(
    readback_policy_text
)

readback_policy_sha256 = calculate_sha256(
    COMPARATOR_POLICY_PATH
)

readback_sidecar_text = (
    COMPARATOR_POLICY_SHA256_PATH
    .read_text(
        encoding="utf-8"
    )
)

readback_core = dict(
    readback_policy
)

readback_core.pop(
    "created_at_utc",
    None,
)

policy_checks = {
    "policy JSON parses successfully":
        isinstance(
            readback_policy,
            dict,
        ),

    "policy identity":
        readback_policy.get(
            "policy_id"
        )
        == "GES_STAGE6A_COMPARATOR_SCORE_POLICY",

    "policy version":
        readback_policy.get(
            "version"
        )
        == "1.0.0",

    "policy status":
        readback_policy.get(
            "status"
        )
        == "FROZEN_BEFORE_OUTCOME_LABEL_LOAD_OR_TEMPORAL_PERFORMANCE",

    "policy content matches prespecified core":
        readback_core
        == policy_core,

    "policy checksum readback":
        readback_policy_sha256
        == comparator_policy_sha256,

    "policy sidecar exact match":
        readback_sidecar_text
        == expected_sidecar_text,

    "global score direction is instability risk":
        readback_policy.get(
            "global_score_direction"
        )
        == "Higher values indicate greater predicted future-instability risk.",

    "combined heuristic has six equal-weight components":
        len(
            readback_policy[
                "score_definitions"
            ][
                "strong_combined_metadata_heuristic"
            ][
                "components"
            ]
        )
        == 6,

    "combined weights sum to one":
        np.isclose(
            sum(
                readback_policy[
                    "score_definitions"
                ][
                    "strong_combined_metadata_heuristic"
                ][
                    "component_weights"
                ]
                .values()
            ),
            1.0,
            atol=1e-15,
            rtol=0,
        ),

    "additive baseline has six indicators":
        len(
            readback_policy[
                "score_definitions"
            ][
                "additive_risk_baseline"
            ][
                "indicators"
            ]
        )
        == 6,

    "outcome loading prohibited":
        readback_policy[
            "construction_requirements"
        ][
            "allow_outcome_label_load"
        ]
        is False,

    "temporal performance prohibited":
        readback_policy[
            "construction_requirements"
        ][
            "allow_temporal_performance_calculation"
        ]
        is False,
}

failed_policy_checks = [
    check_name
    for check_name, passed
    in policy_checks.items()
    if not passed
]


# --------------------------------------------------------------------------------------------------
# 13. Print frozen policy summary
# --------------------------------------------------------------------------------------------------

print("=" * 124)
print("STAGE 6A STEP 2 — CELL 3 — COMPARATOR-SCORE POLICY FREEZE")
print("=" * 124)

print("\nSOURCE ARTIFACT VERIFICATION")
print("-" * 124)

for artifact_name, passed in source_checksum_checks.items():
    print(
        f"{artifact_name:<55} "
        f"{'PASS' if passed else 'FAIL'}"
    )

print("\nFROZEN CONSTANTS")
print("-" * 124)
print(
    f"Review-star levels:                       "
    f"{observed_review_star_levels}"
)
print(
    f"Review-star scaling maximum:              "
    f"{review_star_scaling_maximum:.12f}"
)
print(
    f"Recency maximum window, days:             "
    f"{recency_maximum_window_days:.12f}"
)
print(
    f"Recency median for missing values, days:  "
    f"{recency_median_days:.12f}"
)
print(
    f"Recency median instability-risk value:    "
    f"{recency_median_instability_risk:.12f}"
)
print(
    f"Submitter minimum log1p count:            "
    f"{submitter_minimum_log_count:.12f}"
)
print(
    f"Submitter maximum log1p count:            "
    f"{submitter_maximum_log_count:.12f}"
)
print(
    f"Stale-recency instability threshold:      "
    f">{stale_recency_instability_threshold:.12f}"
)
print(
    f"Additive-risk components:                 "
    f"{additive_component_count}"
)
print(
    f"Combined-metadata components:             "
    f"{combined_component_count}"
)
print(
    f"Equal combined-component weight:          "
    f"{equal_component_weight:.12f}"
)

print("\nPOLICY ARTIFACTS")
print("-" * 124)
print(
    f"Policy path:                              "
    f"{COMPARATOR_POLICY_PATH}"
)
print(
    f"Policy SHA-256 path:                      "
    f"{COMPARATOR_POLICY_SHA256_PATH}"
)
print(
    f"Policy SHA-256:                           "
    f"{comparator_policy_sha256}"
)
print(
    f"Policy file newly created:                "
    f"{'YES' if policy_file_created else 'NO'}"
)
print(
    f"Sidecar newly created:                    "
    f"{'YES' if sidecar_file_created else 'NO'}"
)
print(
    f"Existing identical policy reused:         "
    f"{'YES' if existing_policy_reused else 'NO'}"
)

print("\nFROZEN COMPARATOR DEFINITIONS")
print("-" * 124)

for comparator_name, definition in (
    readback_policy[
        "score_definitions"
    ].items()
):
    output_name = (
        definition.get(
            "output_column"
        )
        or definition.get(
            "output_normalized_column"
        )
        or definition.get(
            "output_count_column"
        )
        or "<multiple outputs>"
    )

    print(
        f"{comparator_name:<45} "
        f"{output_name}"
    )

print("\nPOLICY READBACK CHECKS")
print("-" * 124)

for check_name, passed in policy_checks.items():
    print(
        f"{check_name:<68} "
        f"{'PASS' if passed else 'FAIL'}"
    )


# --------------------------------------------------------------------------------------------------
# 14. Final decision
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 124)
print("STAGE 6A STEP 2 — CELL 3 DECISION")
print("=" * 124)

if failed_policy_checks:

    print(
        "FAIL_STAGE6A_COMPARATOR_SCORE_POLICY_FREEZE"
    )

    print("\nFailed checks:")

    for failed_check in failed_policy_checks:
        print(
            f" - {failed_check}"
        )

    raise RuntimeError(
        "The comparator-score policy did not pass freeze verification."
    )

print(
    "PASS_STAGE6A_COMPARATOR_SCORE_POLICY_FROZEN"
)

print()
print(
    "Comparator policy frozen:                    YES"
)
print(
    "Policy SHA-256 sidecar verified:              YES"
)
print(
    "Directionality fixed before outcomes:         YES"
)
print(
    "Scaling constants fixed before outcomes:      YES"
)
print(
    "Missing-value treatment fixed:                YES"
)
print(
    "Additive-risk indicators fixed:               YES"
)
print(
    "Combined-metadata weights fixed:              YES"
)
print(
    "Future comparator output schema fixed:        YES"
)
print()
print(
    "Stage 5 outcome table opened:                 NO"
)
print(
    "Outcome labels loaded:                        NO"
)
print(
    "Score-outcome join created:                   NO"
)
print(
    "Comparator scores constructed:               NO"
)
print(
    "Temporal performance examined:               NO"
)
print(
    f"Files written or modified:                    "
    f"{'YES' if policy_file_created or sidecar_file_created else 'NO — IDENTICAL FROZEN FILES REUSED'}"
)
print()
print(
    "NEXT AUTHORIZED ACTION:"
)
print(
    "Construct the 71,659-row comparator-score table from the frozen "
    "Stage 4A features and Stage 4C scores, attach this policy SHA-256 "
    "to every row, and validate all scores without opening Stage 5 outcomes."
)

STAGE 6A STEP 2 — CELL 3 — COMPARATOR-SCORE POLICY FREEZE

SOURCE ARTIFACT VERIFICATION
----------------------------------------------------------------------------------------------------------------------------
stage4a_feature_table                                   PASS
stage4a_feature_specification                           PASS
stage4a_freeze_manifest                                 PASS
stage4b_transform_parameters                            PASS
stage4b_weak_label_rules                                PASS
stage4b_freeze_manifest                                 PASS
stage4c_score_table                                     PASS
stage4c_freeze_manifest                                 PASS

FROZEN CONSTANTS
----------------------------------------------------------------------------------------------------------------------------
Review-star levels:                       [0, 1, 2, 3]
Review-star scaling maximum:              3.000000000000
Recency maximum window, days:             10

In [ ]:
# ==================================================================================================
# STAGE 6A — STEP 3 — CELL 3A
# FROZEN COMPARATOR-CONSTRUCTION PREFLIGHT
#
# Purpose:
#   1. Mount Google Drive.
#   2. Verify the frozen Stage 6A comparator-score policy and SHA-256 sidecar.
#   3. Reverify the frozen Stage 4A feature table and Stage 4C GES score table.
#   4. Confirm exact RCV-key and zero-based row-order compatibility.
#   5. Display the complete frozen comparator policy for policy-driven score construction.
#
# Scientific boundary:
#   - Stage 5 outcome data are NOT opened.
#   - No outcome labels are loaded.
#   - No comparator scores are calculated.
#   - No score-outcome join is created.
#   - No temporal performance is calculated.
#   - No file is written or modified.
# ==================================================================================================

from pathlib import Path
import hashlib
import json

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

from google.colab import drive


# --------------------------------------------------------------------------------------------------
# 1. Mount Google Drive
# --------------------------------------------------------------------------------------------------

DRIVE_ROOT = Path("/content/drive/MyDrive")

if not DRIVE_ROOT.exists():
    drive.mount("/content/drive")

if not DRIVE_ROOT.exists():
    raise RuntimeError("Google Drive could not be mounted.")


# --------------------------------------------------------------------------------------------------
# 2. Frozen project paths
# --------------------------------------------------------------------------------------------------

PROJECT_ROOT = (
    DRIVE_ROOT
    / "GES_RAG_Temporal_Study"
)

STAGE4_DATA_DIR = (
    PROJECT_ROOT
    / "data_processed"
    / "stage4_ges"
)

STAGE6_CONFIG_DIR = (
    PROJECT_ROOT
    / "configs"
    / "stage6_temporal_validation"
)

FEATURE_TABLE_PATH = (
    STAGE4_DATA_DIR
    / "stage4a_t0_ges_baseline_features_v1.parquet"
)

GES_SCORE_TABLE_PATH = (
    STAGE4_DATA_DIR
    / "stage4c_t0_full_and_no_star_ges_scores_v1.parquet"
)

COMPARATOR_POLICY_PATH = (
    STAGE6_CONFIG_DIR
    / "stage6a_comparator_score_policy_v1.json"
)

COMPARATOR_POLICY_SIDECAR_PATH = (
    STAGE6_CONFIG_DIR
    / "stage6a_comparator_score_policy_v1.json.sha256"
)


# --------------------------------------------------------------------------------------------------
# 3. Frozen expected checksums and dimensions
# --------------------------------------------------------------------------------------------------

EXPECTED_FEATURE_TABLE_SHA256 = (
    "c100b3781e6801425f622f5d091376ab"
    "fe0939e48c0a792af32f6eebe6401f16"
)

EXPECTED_GES_SCORE_TABLE_SHA256 = (
    "d871ee9087f83be2b0ee954d283aa922"
    "12a639a35e3ea26d5cf042e83019f5ac"
)

EXPECTED_COMPARATOR_POLICY_SHA256 = (
    "dd7e95dc785e77b04c289ef50817b1d"
    "dac7516a436d83b5734dbf3cd35b1248d"
)

EXPECTED_ROWS = 71_659
EXPECTED_FEATURE_COLUMNS = 30
EXPECTED_SCORE_COLUMNS = 21


# --------------------------------------------------------------------------------------------------
# 4. Utility functions
# --------------------------------------------------------------------------------------------------

def calculate_sha256(
    path: Path,
    chunk_size: int = 8 * 1024 * 1024,
) -> str:
    """Calculate a file's SHA-256 without loading the complete file into memory."""

    digest = hashlib.sha256()

    with path.open("rb") as file_handle:
        while True:
            chunk = file_handle.read(chunk_size)

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


def normalize_rcv(
    values: pd.Series,
) -> pd.Series:
    """Normalize RCV accessions for exact cross-artifact comparison."""

    return (
        values
        .astype("string")
        .str.strip()
        .str.upper()
    )


# --------------------------------------------------------------------------------------------------
# 5. Confirm that all frozen artifacts exist
# --------------------------------------------------------------------------------------------------

required_paths = [
    FEATURE_TABLE_PATH,
    GES_SCORE_TABLE_PATH,
    COMPARATOR_POLICY_PATH,
    COMPARATOR_POLICY_SIDECAR_PATH,
]

missing_paths = [
    path
    for path in required_paths
    if not path.exists()
]

if missing_paths:
    raise FileNotFoundError(
        "The following required frozen artifacts were not found:\n"
        + "\n".join(
            f" - {path}"
            for path in missing_paths
        )
    )


# --------------------------------------------------------------------------------------------------
# 6. Cryptographically verify all three frozen artifacts
# --------------------------------------------------------------------------------------------------

observed_feature_sha256 = calculate_sha256(
    FEATURE_TABLE_PATH
)

observed_score_sha256 = calculate_sha256(
    GES_SCORE_TABLE_PATH
)

observed_policy_sha256 = calculate_sha256(
    COMPARATOR_POLICY_PATH
)

sidecar_text = (
    COMPARATOR_POLICY_SIDECAR_PATH
    .read_text(encoding="utf-8")
    .strip()
)

sidecar_parts = sidecar_text.split()

observed_sidecar_hash = (
    sidecar_parts[0]
    if sidecar_parts
    else None
)

observed_sidecar_filename = (
    sidecar_parts[-1]
    if len(sidecar_parts) >= 2
    else None
)

cryptographic_checks = {
    "Stage 4A feature-table SHA-256":
        observed_feature_sha256
        == EXPECTED_FEATURE_TABLE_SHA256,

    "Stage 4C GES-score-table SHA-256":
        observed_score_sha256
        == EXPECTED_GES_SCORE_TABLE_SHA256,

    "Stage 6A comparator-policy SHA-256":
        observed_policy_sha256
        == EXPECTED_COMPARATOR_POLICY_SHA256,

    "Comparator-policy sidecar hash":
        observed_sidecar_hash
        == EXPECTED_COMPARATOR_POLICY_SHA256,

    "Comparator-policy sidecar filename":
        observed_sidecar_filename
        in {
            None,
            COMPARATOR_POLICY_PATH.name,
        },
}


# --------------------------------------------------------------------------------------------------
# 7. Read Parquet metadata without loading the complete scientific tables
# --------------------------------------------------------------------------------------------------

feature_parquet = pq.ParquetFile(
    FEATURE_TABLE_PATH
)

score_parquet = pq.ParquetFile(
    GES_SCORE_TABLE_PATH
)

feature_rows = int(
    feature_parquet.metadata.num_rows
)

feature_columns = int(
    feature_parquet.metadata.num_columns
)

score_rows = int(
    score_parquet.metadata.num_rows
)

score_columns = int(
    score_parquet.metadata.num_columns
)

feature_schema = list(
    feature_parquet.schema_arrow.names
)

score_schema = list(
    score_parquet.schema_arrow.names
)


# --------------------------------------------------------------------------------------------------
# 8. Load identifier columns only and verify exact alignment
# --------------------------------------------------------------------------------------------------

feature_keys = pd.read_parquet(
    FEATURE_TABLE_PATH,
    columns=[
        "t0_row_order",
        "rcv_accession",
    ],
)

score_keys = pd.read_parquet(
    GES_SCORE_TABLE_PATH,
    columns=[
        "t0_row_order",
        "rcv_accession",
    ],
)

feature_keys["rcv_accession"] = normalize_rcv(
    feature_keys["rcv_accession"]
)

score_keys["rcv_accession"] = normalize_rcv(
    score_keys["rcv_accession"]
)

expected_zero_based_order = np.arange(
    EXPECTED_ROWS,
    dtype=np.int64,
)

feature_order = pd.to_numeric(
    feature_keys["t0_row_order"],
    errors="coerce",
).to_numpy(dtype=np.int64)

score_order = pd.to_numeric(
    score_keys["t0_row_order"],
    errors="coerce",
).to_numpy(dtype=np.int64)

alignment_checks = {
    "expected feature-table rows":
        feature_rows == EXPECTED_ROWS,

    "expected feature-table columns":
        feature_columns == EXPECTED_FEATURE_COLUMNS,

    "expected score-table rows":
        score_rows == EXPECTED_ROWS,

    "expected score-table columns":
        score_columns == EXPECTED_SCORE_COLUMNS,

    "feature keys are complete":
        feature_keys["rcv_accession"].notna().all(),

    "score keys are complete":
        score_keys["rcv_accession"].notna().all(),

    "feature keys are unique":
        feature_keys["rcv_accession"].is_unique,

    "score keys are unique":
        score_keys["rcv_accession"].is_unique,

    "feature row order is zero-based":
        np.array_equal(
            feature_order,
            expected_zero_based_order,
        ),

    "score row order is zero-based":
        np.array_equal(
            score_order,
            expected_zero_based_order,
        ),

    "feature and score row order match":
        np.array_equal(
            feature_order,
            score_order,
        ),

    "feature and score RCV keys match":
        np.array_equal(
            feature_keys["rcv_accession"].to_numpy(),
            score_keys["rcv_accession"].to_numpy(),
        ),
}


# --------------------------------------------------------------------------------------------------
# 9. Load the frozen policy
# --------------------------------------------------------------------------------------------------

comparator_policy = json.loads(
    COMPARATOR_POLICY_PATH.read_text(
        encoding="utf-8"
    )
)


# --------------------------------------------------------------------------------------------------
# 10. Final preflight decision
# --------------------------------------------------------------------------------------------------

all_checks = {
    **cryptographic_checks,
    **alignment_checks,
}

failed_checks = [
    check_name
    for check_name, passed
    in all_checks.items()
    if not passed
]

print("=" * 124)
print("STAGE 6A STEP 3 — CELL 3A — COMPARATOR-CONSTRUCTION PREFLIGHT")
print("=" * 124)

print("\nCRYPTOGRAPHIC AND STRUCTURAL CHECKS")
print("-" * 124)

for check_name, passed in all_checks.items():
    print(
        f"{check_name:<72}"
        f"{'PASS' if passed else 'FAIL'}"
    )

if failed_checks:
    print("\nFAILED CHECKS")
    print("-" * 124)

    for failed_check in failed_checks:
        print(f" - {failed_check}")

    raise RuntimeError(
        "Comparator-construction preflight failed. "
        "No comparator scores should be created."
    )


# --------------------------------------------------------------------------------------------------
# 11. Print schemas and complete frozen policy
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 124)
print("VERIFIED INPUT SUMMARY")
print("=" * 124)

print(f"Feature-table rows:                     {feature_rows:,}")
print(f"Feature-table columns:                  {feature_columns}")
print(f"GES-score-table rows:                   {score_rows:,}")
print(f"GES-score-table columns:                {score_columns}")
print(f"T0 row-order range:                     {feature_order.min():,} to {feature_order.max():,}")
print(f"Unique aligned RCV keys:                {feature_keys['rcv_accession'].nunique():,}")
print(f"Comparator-policy SHA-256:              {observed_policy_sha256}")

print("\nSTAGE 4A FEATURE-TABLE SCHEMA")
print("-" * 124)

for column_name in feature_schema:
    print(f" - {column_name}")

print("\nSTAGE 4C GES-SCORE-TABLE SCHEMA")
print("-" * 124)

for column_name in score_schema:
    print(f" - {column_name}")

print("\nFROZEN COMPARATOR-POLICY TOP-LEVEL KEYS")
print("-" * 124)

for policy_key in comparator_policy.keys():
    print(f" - {policy_key}")

print("\nCOMPLETE FROZEN COMPARATOR POLICY")
print("-" * 124)

print(
    json.dumps(
        comparator_policy,
        indent=2,
        ensure_ascii=False,
    )
)

print("\n" + "=" * 124)
print("STAGE 6A STEP 3 — CELL 3A DECISION")
print("=" * 124)

print(
    "PASS_STAGE6A_COMPARATOR_CONSTRUCTION_PREFLIGHT"
)

print()
print("Frozen comparator policy verified:      YES")
print("Stage 4A feature table verified:        YES")
print("Stage 4C GES scores verified:           YES")
print("Exact RCV-key alignment:                YES")
print("Exact zero-based row alignment:         YES")
print()
print("Stage 5 outcome table opened:           NO")
print("Outcome labels loaded:                  NO")
print("Comparator scores constructed:          NO")
print("Temporal performance examined:          NO")
print("Files written or modified:              NO")
print()
print("NEXT AUTHORIZED ACTION:")
print(
    "Construct the comparator scores in memory by applying "
    "the verified frozen policy exactly."
)

STAGE 6A STEP 3 — CELL 3A — COMPARATOR-CONSTRUCTION PREFLIGHT

CRYPTOGRAPHIC AND STRUCTURAL CHECKS
----------------------------------------------------------------------------------------------------------------------------
Stage 4A feature-table SHA-256                                          PASS
Stage 4C GES-score-table SHA-256                                        PASS
Stage 6A comparator-policy SHA-256                                      PASS
Comparator-policy sidecar hash                                          PASS
Comparator-policy sidecar filename                                      PASS
expected feature-table rows                                             PASS
expected feature-table columns                                          PASS
expected score-table rows                                               PASS
expected score-table columns                                            PASS
feature keys are complete                                               PASS
score 

In [ ]:
# ==================================================================================================
# STAGE 6A — STEP 3 — CELL 3B
# CONSTRUCT FROZEN COMPARATOR SCORES IN MEMORY
#
# Purpose:
#   1. Load only the frozen T0 feature columns required by the comparator policy.
#   2. Load only the frozen full-GES and no-star-GES probabilities.
#   3. Construct every comparator score exactly as specified in the frozen policy.
#   4. Validate formulas, ranges, row order, RCV keys, provenance, and output schema.
#
# Scientific boundary:
#   - Stage 5 outcome data are NOT opened.
#   - No outcome labels are loaded.
#   - No score-outcome join is created.
#   - No thresholds or weights are optimized.
#   - No temporal performance is calculated.
#   - The comparator table remains in memory only.
#   - No file is written or modified.
# ==================================================================================================

import numpy as np
import pandas as pd


# --------------------------------------------------------------------------------------------------
# 1. Confirm that Cell 3A was executed successfully
# --------------------------------------------------------------------------------------------------

required_runtime_objects = [
    "FEATURE_TABLE_PATH",
    "GES_SCORE_TABLE_PATH",
    "comparator_policy",
    "observed_policy_sha256",
    "EXPECTED_COMPARATOR_POLICY_SHA256",
    "EXPECTED_ROWS",
]

missing_runtime_objects = [
    object_name
    for object_name in required_runtime_objects
    if object_name not in globals()
]

if missing_runtime_objects:
    raise RuntimeError(
        "Cell 3A must be executed successfully before Cell 3B.\n"
        "Missing runtime objects:\n"
        + "\n".join(
            f" - {object_name}"
            for object_name in missing_runtime_objects
        )
    )

if observed_policy_sha256 != EXPECTED_COMPARATOR_POLICY_SHA256:
    raise RuntimeError(
        "The comparator-policy checksum does not match the frozen expected checksum."
    )

if comparator_policy.get("version") != "1.0.0":
    raise RuntimeError(
        "Unexpected comparator-policy version: "
        f"{comparator_policy.get('version')}"
    )


# --------------------------------------------------------------------------------------------------
# 2. Confirm that the frozen policy still prohibits outcome access and optimization
# --------------------------------------------------------------------------------------------------

construction_requirements = comparator_policy["construction_requirements"]

required_false_boundaries = [
    "allow_outcome_label_load",
    "allow_outcome_table_load",
    "allow_score_outcome_join",
    "allow_temporal_performance_calculation",
    "allow_threshold_optimization",
    "allow_weight_optimization",
]

violated_boundaries = [
    boundary_name
    for boundary_name in required_false_boundaries
    if construction_requirements.get(boundary_name) is not False
]

if violated_boundaries:
    raise RuntimeError(
        "The frozen scientific boundary is not valid for comparator construction:\n"
        + "\n".join(
            f" - {boundary_name}"
            for boundary_name in violated_boundaries
        )
    )


# --------------------------------------------------------------------------------------------------
# 3. Load only the required Stage 4A source features
# --------------------------------------------------------------------------------------------------

required_feature_columns = [
    "t0_row_order",
    "rcv_accession",
    "aggregate_review_stars",
    "aggregate_conflict_flag",
    "recency_days",
    "recency_missing_flag",
    "unique_submitter_count",
    "log1p_unique_submitter_count",
    "scv_group_entropy_normalized",
]

feature_source = pd.read_parquet(
    FEATURE_TABLE_PATH,
    columns=required_feature_columns,
)


# --------------------------------------------------------------------------------------------------
# 4. Load only the required Stage 4C frozen GES scores
# --------------------------------------------------------------------------------------------------

required_ges_columns = [
    "t0_row_order",
    "rcv_accession",
    "full_ges_p_stable_t0",
    "no_star_ges_p_stable_t0",
]

ges_source = pd.read_parquet(
    GES_SCORE_TABLE_PATH,
    columns=required_ges_columns,
)


# --------------------------------------------------------------------------------------------------
# 5. Normalize identifiers and validate source alignment
# --------------------------------------------------------------------------------------------------

def normalize_rcv_accession(
    values: pd.Series,
) -> pd.Series:
    """Normalize RCV accessions for exact frozen-key comparison."""

    return (
        values
        .astype("string")
        .str.strip()
        .str.upper()
    )


feature_source["rcv_accession"] = normalize_rcv_accession(
    feature_source["rcv_accession"]
)

ges_source["rcv_accession"] = normalize_rcv_accession(
    ges_source["rcv_accession"]
)

feature_source["t0_row_order"] = pd.to_numeric(
    feature_source["t0_row_order"],
    errors="raise",
).astype("int64")

ges_source["t0_row_order"] = pd.to_numeric(
    ges_source["t0_row_order"],
    errors="raise",
).astype("int64")

expected_row_order = np.arange(
    EXPECTED_ROWS,
    dtype=np.int64,
)

source_alignment_checks = {
    "feature source row count":
        len(feature_source) == EXPECTED_ROWS,

    "GES source row count":
        len(ges_source) == EXPECTED_ROWS,

    "feature RCV keys complete":
        feature_source["rcv_accession"].notna().all(),

    "GES RCV keys complete":
        ges_source["rcv_accession"].notna().all(),

    "feature RCV keys unique":
        feature_source["rcv_accession"].is_unique,

    "GES RCV keys unique":
        ges_source["rcv_accession"].is_unique,

    "feature zero-based row order":
        np.array_equal(
            feature_source["t0_row_order"].to_numpy(),
            expected_row_order,
        ),

    "GES zero-based row order":
        np.array_equal(
            ges_source["t0_row_order"].to_numpy(),
            expected_row_order,
        ),

    "feature and GES row order identical":
        np.array_equal(
            feature_source["t0_row_order"].to_numpy(),
            ges_source["t0_row_order"].to_numpy(),
        ),

    "feature and GES RCV keys identical":
        np.array_equal(
            feature_source["rcv_accession"].to_numpy(),
            ges_source["rcv_accession"].to_numpy(),
        ),
}

failed_source_alignment_checks = [
    check_name
    for check_name, passed
    in source_alignment_checks.items()
    if not passed
]

if failed_source_alignment_checks:
    raise RuntimeError(
        "Frozen source alignment failed:\n"
        + "\n".join(
            f" - {check_name}"
            for check_name in failed_source_alignment_checks
        )
    )


# --------------------------------------------------------------------------------------------------
# 6. Strictly normalize Boolean source fields
# --------------------------------------------------------------------------------------------------

def strict_boolean_series(
    values: pd.Series,
    column_name: str,
) -> pd.Series:
    """
    Convert a source column to Boolean only when every value is a recognized
    Boolean representation. Missing or unexpected values cause failure.
    """

    if values.isna().any():
        raise ValueError(
            f"{column_name} contains "
            f"{int(values.isna().sum()):,} missing values."
        )

    if pd.api.types.is_bool_dtype(values):
        return values.astype(bool)

    normalized = (
        values
        .astype("string")
        .str.strip()
        .str.lower()
    )

    allowed_mapping = {
        "true": True,
        "false": False,
        "1": True,
        "0": False,
    }

    unexpected_values = sorted(
        set(normalized.dropna().unique())
        - set(allowed_mapping.keys())
    )

    if unexpected_values:
        raise ValueError(
            f"{column_name} contains unexpected Boolean values: "
            f"{unexpected_values}"
        )

    return normalized.map(allowed_mapping).astype(bool)


feature_source["aggregate_conflict_flag"] = strict_boolean_series(
    feature_source["aggregate_conflict_flag"],
    "aggregate_conflict_flag",
)

feature_source["recency_missing_flag"] = strict_boolean_series(
    feature_source["recency_missing_flag"],
    "recency_missing_flag",
)


# --------------------------------------------------------------------------------------------------
# 7. Convert required numeric columns and enforce missing-value policies
# --------------------------------------------------------------------------------------------------

required_complete_numeric_columns = [
    "aggregate_review_stars",
    "unique_submitter_count",
    "log1p_unique_submitter_count",
    "scv_group_entropy_normalized",
]

for column_name in required_complete_numeric_columns:
    feature_source[column_name] = pd.to_numeric(
        feature_source[column_name],
        errors="coerce",
    )

    missing_count = int(
        feature_source[column_name].isna().sum()
    )

    if missing_count > 0:
        raise ValueError(
            f"{column_name} contains {missing_count:,} missing "
            "or nonnumeric values, but the frozen policy prohibits them."
        )

feature_source["recency_days"] = pd.to_numeric(
    feature_source["recency_days"],
    errors="coerce",
)

for column_name in [
    "full_ges_p_stable_t0",
    "no_star_ges_p_stable_t0",
]:
    ges_source[column_name] = pd.to_numeric(
        ges_source[column_name],
        errors="coerce",
    )

    missing_count = int(
        ges_source[column_name].isna().sum()
    )

    if missing_count > 0:
        raise ValueError(
            f"{column_name} contains {missing_count:,} missing "
            "or nonnumeric values."
        )


# --------------------------------------------------------------------------------------------------
# 8. Validate recency missingness semantics
# --------------------------------------------------------------------------------------------------

recency_missing_flag = (
    feature_source["recency_missing_flag"]
)

recency_value_missing = (
    feature_source["recency_days"].isna()
)

recency_missingness_mismatch = int(
    (
        recency_missing_flag
        != recency_value_missing
    ).sum()
)

if recency_missingness_mismatch != 0:
    raise ValueError(
        "recency_missing_flag does not exactly match recency_days missingness. "
        f"Mismatched rows: {recency_missingness_mismatch:,}"
    )


# --------------------------------------------------------------------------------------------------
# 9. Read frozen constants directly from the checksum-verified policy
# --------------------------------------------------------------------------------------------------

frozen_constants = comparator_policy["frozen_constants"]

review_star_scaling_maximum = float(
    frozen_constants["review_star_scaling_maximum"]
)

recency_maximum_observation_window_days = float(
    frozen_constants["recency_maximum_observation_window_days"]
)

recency_missing_imputation_risk = float(
    frozen_constants[
        "recency_median_instability_risk_for_missing_imputation"
    ]
)

submitter_minimum_log1p_count = float(
    frozen_constants["submitter_minimum_log1p_count"]
)

submitter_log1p_range = float(
    frozen_constants["submitter_log1p_range"]
)

entropy_positive_threshold = float(
    frozen_constants["entropy_positive_threshold"]
)

single_submitter_threshold = int(
    frozen_constants["single_submitter_threshold"]
)

stale_recency_instability_threshold = float(
    frozen_constants["stale_recency_instability_threshold"]
)

additive_component_count = int(
    frozen_constants["additive_component_count"]
)

combined_component_count = int(
    frozen_constants["combined_component_count"]
)

equal_combined_component_weight = float(
    frozen_constants["equal_combined_component_weight"]
)


# --------------------------------------------------------------------------------------------------
# 10. Construct the comparator table in immutable T0 row order
# --------------------------------------------------------------------------------------------------

comparator_scores = pd.DataFrame(
    {
        "t0_row_order":
            feature_source["t0_row_order"].astype("int64"),

        "rcv_accession":
            feature_source["rcv_accession"].astype("string"),
    }
)


# --------------------------------------------------------------------------------------------------
# 11. Review-stars comparator
#
# Formula:
#   clip(
#       (review_star_scaling_maximum - aggregate_review_stars)
#       / review_star_scaling_maximum,
#       0,
#       1
#   )
# --------------------------------------------------------------------------------------------------

comparator_scores["review_stars_instability_risk"] = np.clip(
    (
        review_star_scaling_maximum
        - feature_source["aggregate_review_stars"].to_numpy(dtype=float)
    )
    / review_star_scaling_maximum,
    0.0,
    1.0,
)


# --------------------------------------------------------------------------------------------------
# 12. Conflict comparator
#
# Formula:
#   1.0 when aggregate_conflict_flag is True; otherwise 0.0.
# --------------------------------------------------------------------------------------------------

comparator_scores["conflict_instability_risk"] = (
    feature_source["aggregate_conflict_flag"]
    .astype(float)
    .to_numpy()
)


# --------------------------------------------------------------------------------------------------
# 13. Recency comparator
#
# Available recency:
#   clip(recency_days / maximum observation window, 0, 1)
#
# Missing recency:
#   frozen T0 median instability-risk value
# --------------------------------------------------------------------------------------------------

available_recency_risk = np.clip(
    (
        feature_source["recency_days"]
        .fillna(0.0)
        .to_numpy(dtype=float)
        / recency_maximum_observation_window_days
    ),
    0.0,
    1.0,
)

comparator_scores["recency_instability_risk"] = np.where(
    recency_missing_flag.to_numpy(),
    recency_missing_imputation_risk,
    available_recency_risk,
)


# --------------------------------------------------------------------------------------------------
# 14. Separate recency-missingness component
#
# Formula:
#   1.0 when recency_missing_flag is True; otherwise 0.0.
# --------------------------------------------------------------------------------------------------

comparator_scores["recency_missing_instability_component"] = (
    recency_missing_flag
    .astype(float)
    .to_numpy()
)


# --------------------------------------------------------------------------------------------------
# 15. Submitter comparator
#
# normalized support:
#   clip(
#       (log1p_unique_submitter_count - minimum)
#       / observed range,
#       0,
#       1
#   )
#
# instability risk:
#   1.0 - normalized support
# --------------------------------------------------------------------------------------------------

normalized_submitter_support = np.clip(
    (
        feature_source["log1p_unique_submitter_count"]
        .to_numpy(dtype=float)
        - submitter_minimum_log1p_count
    )
    / submitter_log1p_range,
    0.0,
    1.0,
)

comparator_scores["submitter_instability_risk"] = (
    1.0
    - normalized_submitter_support
)


# --------------------------------------------------------------------------------------------------
# 16. Entropy comparator
#
# Formula:
#   clip(scv_group_entropy_normalized, 0, 1)
# --------------------------------------------------------------------------------------------------

comparator_scores["entropy_instability_risk"] = np.clip(
    feature_source["scv_group_entropy_normalized"]
    .to_numpy(dtype=float),
    0.0,
    1.0,
)


# --------------------------------------------------------------------------------------------------
# 17. Construct six frozen additive-risk indicators
# --------------------------------------------------------------------------------------------------

comparator_scores["additive_low_review_indicator"] = (
    feature_source["aggregate_review_stars"]
    .eq(0)
    .astype("int8")
)

comparator_scores["additive_conflict_indicator"] = (
    feature_source["aggregate_conflict_flag"]
    .astype("int8")
)

comparator_scores["additive_stale_recency_indicator"] = (
    (
        ~feature_source["recency_missing_flag"]
    )
    &
    (
        comparator_scores["recency_instability_risk"]
        > stale_recency_instability_threshold
    )
).astype("int8")

comparator_scores["additive_missing_recency_indicator"] = (
    feature_source["recency_missing_flag"]
    .astype("int8")
)

comparator_scores["additive_single_submitter_indicator"] = (
    feature_source["unique_submitter_count"]
    .eq(single_submitter_threshold)
    .astype("int8")
)

comparator_scores["additive_entropy_indicator"] = (
    feature_source["scv_group_entropy_normalized"]
    .gt(entropy_positive_threshold)
    .astype("int8")
)


# --------------------------------------------------------------------------------------------------
# 18. Additive-risk count and normalized score
# --------------------------------------------------------------------------------------------------

additive_indicator_columns = [
    "additive_low_review_indicator",
    "additive_conflict_indicator",
    "additive_stale_recency_indicator",
    "additive_missing_recency_indicator",
    "additive_single_submitter_indicator",
    "additive_entropy_indicator",
]

comparator_scores["additive_risk_count"] = (
    comparator_scores[additive_indicator_columns]
    .sum(axis=1)
    .astype("int8")
)

comparator_scores["additive_instability_risk"] = (
    comparator_scores["additive_risk_count"]
    .astype(float)
    / float(additive_component_count)
)


# --------------------------------------------------------------------------------------------------
# 19. Strong combined-metadata comparator
#
# Equal-weight arithmetic mean of:
#   - review stars
#   - conflict
#   - recency
#   - recency missingness
#   - submitter support
#   - entropy
# --------------------------------------------------------------------------------------------------

combined_component_columns = [
    "review_stars_instability_risk",
    "conflict_instability_risk",
    "recency_instability_risk",
    "recency_missing_instability_component",
    "submitter_instability_risk",
    "entropy_instability_risk",
]

comparator_scores["combined_metadata_instability_risk"] = (
    comparator_scores[combined_component_columns]
    .sum(axis=1)
    * equal_combined_component_weight
)


# --------------------------------------------------------------------------------------------------
# 20. Preserve frozen GES probabilities and convert them to instability risk
# --------------------------------------------------------------------------------------------------

comparator_scores["full_ges_p_stable_t0"] = (
    ges_source["full_ges_p_stable_t0"]
    .to_numpy(dtype=float)
)

comparator_scores["full_ges_instability_risk_t0"] = (
    1.0
    - comparator_scores["full_ges_p_stable_t0"]
)

comparator_scores["no_star_ges_p_stable_t0"] = (
    ges_source["no_star_ges_p_stable_t0"]
    .to_numpy(dtype=float)
)

comparator_scores["no_star_ges_instability_risk_t0"] = (
    1.0
    - comparator_scores["no_star_ges_p_stable_t0"]
)


# --------------------------------------------------------------------------------------------------
# 21. Attach frozen policy and scientific-boundary provenance
# --------------------------------------------------------------------------------------------------

comparator_scores["comparator_policy_version"] = (
    comparator_policy["version"]
)

comparator_scores["comparator_policy_sha256"] = (
    observed_policy_sha256
)

# The Stage 6A Step 2 frozen policy itself is version 1.0.0.
comparator_scores["stage6a_step2_version"] = (
    comparator_policy["version"]
)

comparator_scores["outcome_labels_loaded"] = False

comparator_scores["temporal_performance_evaluated"] = False


# --------------------------------------------------------------------------------------------------
# 22. Enforce the exact frozen output-column order
# --------------------------------------------------------------------------------------------------

expected_output_columns = (
    comparator_policy["planned_output"]["column_order"]
)

missing_output_columns = [
    column_name
    for column_name in expected_output_columns
    if column_name not in comparator_scores.columns
]

unexpected_output_columns = [
    column_name
    for column_name in comparator_scores.columns
    if column_name not in expected_output_columns
]

if missing_output_columns or unexpected_output_columns:
    raise RuntimeError(
        "Comparator output schema does not match the frozen policy.\n"
        f"Missing columns: {missing_output_columns}\n"
        f"Unexpected columns: {unexpected_output_columns}"
    )

comparator_scores = comparator_scores[
    expected_output_columns
].copy()


# --------------------------------------------------------------------------------------------------
# 23. Validate all score ranges
# --------------------------------------------------------------------------------------------------

unit_interval_score_columns = [
    "review_stars_instability_risk",
    "conflict_instability_risk",
    "recency_instability_risk",
    "recency_missing_instability_component",
    "submitter_instability_risk",
    "entropy_instability_risk",
    "additive_instability_risk",
    "combined_metadata_instability_risk",
    "full_ges_p_stable_t0",
    "full_ges_instability_risk_t0",
    "no_star_ges_p_stable_t0",
    "no_star_ges_instability_risk_t0",
]

range_failures = {}

for column_name in unit_interval_score_columns:
    values = pd.to_numeric(
        comparator_scores[column_name],
        errors="coerce",
    )

    missing_count = int(values.isna().sum())

    below_zero_count = int(
        (values < 0.0).sum()
    )

    above_one_count = int(
        (values > 1.0).sum()
    )

    if (
        missing_count > 0
        or below_zero_count > 0
        or above_one_count > 0
    ):
        range_failures[column_name] = {
            "missing": missing_count,
            "below_zero": below_zero_count,
            "above_one": above_one_count,
        }

if range_failures:
    raise RuntimeError(
        "One or more comparator score columns violated the frozen [0,1] range:\n"
        + str(range_failures)
    )


# --------------------------------------------------------------------------------------------------
# 24. Validate binary indicator columns and additive count range
# --------------------------------------------------------------------------------------------------

binary_indicator_failures = {}

for column_name in additive_indicator_columns:
    observed_values = set(
        comparator_scores[column_name]
        .dropna()
        .astype(int)
        .unique()
        .tolist()
    )

    if not observed_values.issubset({0, 1}):
        binary_indicator_failures[column_name] = sorted(
            observed_values
        )

if binary_indicator_failures:
    raise RuntimeError(
        "Unexpected additive-indicator values:\n"
        + str(binary_indicator_failures)
    )

if not comparator_scores["additive_risk_count"].between(
    0,
    additive_component_count,
    inclusive="both",
).all():
    raise RuntimeError(
        "additive_risk_count is outside the frozen 0-to-6 range."
    )


# --------------------------------------------------------------------------------------------------
# 25. Independently reconstruct formulas for exact QC
# --------------------------------------------------------------------------------------------------

formula_checks = {
    "review-star formula":
        np.allclose(
            comparator_scores["review_stars_instability_risk"],
            np.clip(
                (
                    review_star_scaling_maximum
                    - feature_source["aggregate_review_stars"].to_numpy(dtype=float)
                )
                / review_star_scaling_maximum,
                0.0,
                1.0,
            ),
            rtol=0.0,
            atol=1e-15,
        ),

    "conflict formula":
        np.array_equal(
            comparator_scores["conflict_instability_risk"].to_numpy(),
            feature_source["aggregate_conflict_flag"]
            .astype(float)
            .to_numpy(),
        ),

    "recency formula":
        np.allclose(
            comparator_scores["recency_instability_risk"],
            np.where(
                feature_source["recency_missing_flag"].to_numpy(),
                recency_missing_imputation_risk,
                np.clip(
                    feature_source["recency_days"]
                    .fillna(0.0)
                    .to_numpy(dtype=float)
                    / recency_maximum_observation_window_days,
                    0.0,
                    1.0,
                ),
            ),
            rtol=0.0,
            atol=1e-15,
        ),

    "recency-missing formula":
        np.array_equal(
            comparator_scores[
                "recency_missing_instability_component"
            ].to_numpy(),
            feature_source["recency_missing_flag"]
            .astype(float)
            .to_numpy(),
        ),

    "submitter formula":
        np.allclose(
            comparator_scores["submitter_instability_risk"],
            1.0
            - np.clip(
                (
                    feature_source["log1p_unique_submitter_count"]
                    .to_numpy(dtype=float)
                    - submitter_minimum_log1p_count
                )
                / submitter_log1p_range,
                0.0,
                1.0,
            ),
            rtol=0.0,
            atol=1e-15,
        ),

    "entropy formula":
        np.allclose(
            comparator_scores["entropy_instability_risk"],
            np.clip(
                feature_source["scv_group_entropy_normalized"]
                .to_numpy(dtype=float),
                0.0,
                1.0,
            ),
            rtol=0.0,
            atol=1e-15,
        ),

    "additive count formula":
        np.array_equal(
            comparator_scores["additive_risk_count"].to_numpy(),
            comparator_scores[additive_indicator_columns]
            .sum(axis=1)
            .to_numpy(),
        ),

    "additive normalized formula":
        np.allclose(
            comparator_scores["additive_instability_risk"],
            comparator_scores["additive_risk_count"]
            .astype(float)
            / float(additive_component_count),
            rtol=0.0,
            atol=1e-15,
        ),

    "combined metadata formula":
        np.allclose(
            comparator_scores[
                "combined_metadata_instability_risk"
            ],
            comparator_scores[combined_component_columns]
            .sum(axis=1)
            / float(combined_component_count),
            rtol=0.0,
            atol=1e-15,
        ),

    "full GES conversion":
        np.allclose(
            comparator_scores["full_ges_instability_risk_t0"],
            1.0
            - comparator_scores["full_ges_p_stable_t0"],
            rtol=0.0,
            atol=1e-15,
        ),

    "no-star GES conversion":
        np.allclose(
            comparator_scores["no_star_ges_instability_risk_t0"],
            1.0
            - comparator_scores["no_star_ges_p_stable_t0"],
            rtol=0.0,
            atol=1e-15,
        ),
}

failed_formula_checks = [
    check_name
    for check_name, passed
    in formula_checks.items()
    if not passed
]

if failed_formula_checks:
    raise RuntimeError(
        "Frozen comparator formula reconstruction failed:\n"
        + "\n".join(
            f" - {check_name}"
            for check_name in failed_formula_checks
        )
    )


# --------------------------------------------------------------------------------------------------
# 26. Final table-level validation
# --------------------------------------------------------------------------------------------------

table_validation_checks = {
    "expected row count":
        len(comparator_scores)
        == comparator_policy["planned_output"]["expected_rows"],

    "exact frozen output schema":
        list(comparator_scores.columns)
        == expected_output_columns,

    "complete RCV keys":
        comparator_scores["rcv_accession"].notna().all(),

    "unique RCV keys":
        comparator_scores["rcv_accession"].is_unique,

    "exact zero-based row order":
        np.array_equal(
            comparator_scores["t0_row_order"].to_numpy(),
            expected_row_order,
        ),

    "exact source RCV order":
        np.array_equal(
            comparator_scores["rcv_accession"].to_numpy(),
            feature_source["rcv_accession"].to_numpy(),
        ),

    "policy version on every row":
        comparator_scores["comparator_policy_version"]
        .eq(comparator_policy["version"])
        .all(),

    "policy SHA-256 on every row":
        comparator_scores["comparator_policy_sha256"]
        .eq(observed_policy_sha256)
        .all(),

    "Stage 6A Step 2 version on every row":
        comparator_scores["stage6a_step2_version"]
        .eq(comparator_policy["version"])
        .all(),

    "outcome labels remain unloaded":
        comparator_scores["outcome_labels_loaded"]
        .eq(False)
        .all(),

    "temporal performance remains unevaluated":
        comparator_scores["temporal_performance_evaluated"]
        .eq(False)
        .all(),

    "no missing output values":
        comparator_scores.isna().sum().sum() == 0,
}

failed_table_checks = [
    check_name
    for check_name, passed
    in table_validation_checks.items()
    if not passed
]

if failed_table_checks:
    raise RuntimeError(
        "Comparator table validation failed:\n"
        + "\n".join(
            f" - {check_name}"
            for check_name in failed_table_checks
        )
    )


# --------------------------------------------------------------------------------------------------
# 27. Generate read-only summaries
# --------------------------------------------------------------------------------------------------

primary_comparison_columns = (
    comparator_policy["primary_comparison_set"]
)

score_summary = (
    comparator_scores[primary_comparison_columns]
    .describe(
        percentiles=[
            0.01,
            0.05,
            0.10,
            0.25,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99,
        ]
    )
    .T
)

score_summary["missing"] = (
    comparator_scores[primary_comparison_columns]
    .isna()
    .sum()
)

score_summary["unique_values"] = (
    comparator_scores[primary_comparison_columns]
    .nunique(dropna=False)
)

additive_count_distribution = (
    comparator_scores["additive_risk_count"]
    .value_counts()
    .sort_index()
    .rename_axis("additive_risk_count")
    .to_frame("row_count")
)

additive_count_distribution["percentage"] = (
    additive_count_distribution["row_count"]
    / len(comparator_scores)
    * 100.0
)


# --------------------------------------------------------------------------------------------------
# 28. Print results
# --------------------------------------------------------------------------------------------------

print("=" * 124)
print("STAGE 6A STEP 3 — CELL 3B — IN-MEMORY COMPARATOR CONSTRUCTION")
print("=" * 124)

print("\nSOURCE-ALIGNMENT CHECKS")
print("-" * 124)

for check_name, passed in source_alignment_checks.items():
    print(
        f"{check_name:<76}"
        f"{'PASS' if passed else 'FAIL'}"
    )

print("\nFORMULA-RECONSTRUCTION CHECKS")
print("-" * 124)

for check_name, passed in formula_checks.items():
    print(
        f"{check_name:<76}"
        f"{'PASS' if passed else 'FAIL'}"
    )

print("\nFINAL TABLE-VALIDATION CHECKS")
print("-" * 124)

for check_name, passed in table_validation_checks.items():
    print(
        f"{check_name:<76}"
        f"{'PASS' if passed else 'FAIL'}"
    )

print("\n" + "=" * 124)
print("COMPARATOR SCORE SUMMARY")
print("=" * 124)

display(
    score_summary
)

print("\nADDITIVE-RISK COUNT DISTRIBUTION")
print("-" * 124)

display(
    additive_count_distribution
)

print("\nFIRST FIVE CONSTRUCTED ROWS")
print("-" * 124)

display(
    comparator_scores.head()
)

print("\n" + "=" * 124)
print("STAGE 6A STEP 3 — CELL 3B DECISION")
print("=" * 124)

print(
    "PASS_STAGE6A_COMPARATOR_SCORES_CONSTRUCTED_IN_MEMORY"
)

print()
print(f"Constructed rows:                       {len(comparator_scores):,}")
print(f"Constructed columns:                    {len(comparator_scores.columns):,}")
print(
    "T0 row-order range:                    "
    f"{comparator_scores['t0_row_order'].min():,} "
    f"to {comparator_scores['t0_row_order'].max():,}"
)
print(
    "Unique RCV keys:                       "
    f"{comparator_scores['rcv_accession'].nunique():,}"
)
print(f"Comparator-policy version:              {comparator_policy['version']}")
print(f"Comparator-policy SHA-256:              {observed_policy_sha256}")

print()
print("Stage 5 outcome table opened:           NO")
print("Outcome labels loaded:                  NO")
print("Score-outcome join created:             NO")
print("Thresholds optimized:                   NO")
print("Weights optimized:                      NO")
print("Temporal performance examined:          NO")
print("Comparator file written:                NO")
print("Other files written or modified:        NO")

print()
print("NEXT AUTHORIZED ACTION:")
print(
    "Create the comparator-construction QC report in memory, "
    "then checksum-freeze the comparator Parquet, QC report, "
    "SHA-256 sidecars, and freeze manifest."
)

STAGE 6A STEP 3 — CELL 3B — IN-MEMORY COMPARATOR CONSTRUCTION

SOURCE-ALIGNMENT CHECKS
----------------------------------------------------------------------------------------------------------------------------
feature source row count                                                    PASS
GES source row count                                                        PASS
feature RCV keys complete                                                   PASS
GES RCV keys complete                                                       PASS
feature RCV keys unique                                                     PASS
GES RCV keys unique                                                         PASS
feature zero-based row order                                                PASS
GES zero-based row order                                                    PASS
feature and GES row order identical                                         PASS
feature and GES RCV keys identical                         

,count,mean,std,min,1%,5%,10%,25%,50%,75%,90%,95%,99%,max,missing,unique_values
full_ges_instability_risk_t0,71659.0,0.110641,0.302426,1.065592e-12,2.873946e-11,7.955703e-11,3.718277e-10,6.779846e-07,0.000003,0.000081,0.765702,0.999819,0.999852,1.000000,0,9570
no_star_ges_instability_risk_t0,71659.0,0.032738,0.174215,1.700875e-08,4.188223e-08,6.974021e-08,1.062893e-07,1.471795e-07,0.000001,0.000014,0.000121,0.000919,0.999847,1.000000,0,9080
review_stars_instability_risk,71659.0,0.565902,0.246496,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,6.666667e-01,0.666667,0.666667,0.666667,1.000000,1.000000,1.000000,0,4
conflict_instability_risk,71659.0,0.020709,0.142410,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000,0.000000,0.000000,0.000000,1.000000,1.000000,0,2
recency_instability_risk,71659.0,0.125037,0.105062,7.607455e-04,8.938760e-03,2.443895e-02,3.680107e-02,4.669076e-02,0.099563,0.176873,0.238399,0.281286,0.615280,1.000000,0,3833
submitter_instability_risk,71659.0,0.950689,0.100840,0.000000e+00,5.429274e-01,7.471041e-01,8.520654e-01,1.000000e+00,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0,22
additive_instability_risk,71659.0,0.161341,0.112720,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,1.666667e-01,0.166667,0.166667,0.333333,0.500000,0.500000,0.500000,0,4
combined_metadata_instability_risk,71659.0,0.297558,0.090015,3.873463e-02,1.474799e-01,1.782256e-01,1.985387e-01,2.792993e-01,0.287197,0.307542,0.461038,0.516594,0.593356,0.769716,0,9570



ADDITIVE-RISK COUNT DISTRIBUTION
----------------------------------------------------------------------------------------------------------------------------


,row_count,percentage
additive_risk_count,,
0,13736,19.168562
1,50311,70.208906
2,3778,5.272192
3,3834,5.350340



FIRST FIVE CONSTRUCTED ROWS
----------------------------------------------------------------------------------------------------------------------------


,t0_row_order,rcv_accession,review_stars_instability_risk,conflict_instability_risk,recency_instability_risk,recency_missing_instability_component,submitter_instability_risk,entropy_instability_risk,additive_low_review_indicator,additive_conflict_indicator,...,combined_metadata_instability_risk,full_ges_p_stable_t0,full_ges_instability_risk_t0,no_star_ges_p_stable_t0,no_star_ges_instability_risk_t0,comparator_policy_version,comparator_policy_sha256,stage6a_step2_version,outcome_labels_loaded,temporal_performance_evaluated
0,0,RCV000052656,0.666667,0.0,0.395493,0.0,1.0,0.0,0,0,...,0.343693,0.953676,0.046324,0.99122,0.00878,1.0.0,dd7e95dc785e77b04c289ef50817b1ddac7516a436d83b...,1.0.0,False,False
1,1,RCV000053439,0.666667,0.0,0.395493,0.0,1.0,0.0,0,0,...,0.343693,0.953676,0.046324,0.99122,0.00878,1.0.0,dd7e95dc785e77b04c289ef50817b1ddac7516a436d83b...,1.0.0,False,False
2,2,RCV000053440,0.666667,0.0,0.395493,0.0,1.0,0.0,0,0,...,0.343693,0.953676,0.046324,0.99122,0.00878,1.0.0,dd7e95dc785e77b04c289ef50817b1ddac7516a436d83b...,1.0.0,False,False
3,3,RCV000053441,0.666667,0.0,0.395493,0.0,1.0,0.0,0,0,...,0.343693,0.953676,0.046324,0.99122,0.00878,1.0.0,dd7e95dc785e77b04c289ef50817b1ddac7516a436d83b...,1.0.0,False,False
4,4,RCV000053532,0.666667,0.0,0.395493,0.0,1.0,0.0,0,0,...,0.343693,0.953676,0.046324,0.99122,0.00878,1.0.0,dd7e95dc785e77b04c289ef50817b1ddac7516a436d83b...,1.0.0,False,False



STAGE 6A STEP 3 — CELL 3B DECISION
PASS_STAGE6A_COMPARATOR_SCORES_CONSTRUCTED_IN_MEMORY

Constructed rows:                       71,659
Constructed columns:                    26
T0 row-order range:                    0 to 71,658
Unique RCV keys:                       71,659
Comparator-policy version:              1.0.0
Comparator-policy SHA-256:              dd7e95dc785e77b04c289ef50817b1ddac7516a436d83b5734dbf3cd35b1248d

Stage 5 outcome table opened:           NO
Outcome labels loaded:                  NO
Score-outcome join created:             NO
Thresholds optimized:                   NO
Weights optimized:                      NO
Temporal performance examined:          NO
Comparator file written:                NO
Other files written or modified:        NO

NEXT AUTHORIZED ACTION:
Create the comparator-construction QC report in memory, then checksum-freeze the comparator Parquet, QC report, SHA-256 sidecars, and freeze manifest.


In [ ]:
# ==================================================================================================
# STAGE 6A — STEP 3 — CELL 3C
# CREATE COMPARATOR-CONSTRUCTION QC REPORT IN MEMORY
#
# Purpose:
#   1. Independently profile the complete in-memory comparator table.
#   2. Recalculate exact formula-mismatch counts.
#   3. Record score ranges, distributions, missingness, and indicator prevalence.
#   4. Create a JSON-serializable QC report.
#   5. Create the exact JSON text and SHA-256 that will be frozen in the next cell.
#
# Scientific boundary:
#   - Stage 5 outcome data are NOT opened.
#   - No outcome labels are loaded.
#   - No score-outcome join is created.
#   - No temporal performance is calculated.
#   - No file is written or modified.
# ==================================================================================================

from datetime import datetime, timezone
from pathlib import Path
import hashlib
import json
import platform
import sys

import numpy as np
import pandas as pd
import pyarrow


# --------------------------------------------------------------------------------------------------
# 1. Confirm that Cell 3B completed successfully
# --------------------------------------------------------------------------------------------------

required_runtime_objects = [
    "comparator_scores",
    "feature_source",
    "ges_source",
    "comparator_policy",
    "observed_policy_sha256",
    "EXPECTED_ROWS",
    "expected_output_columns",
    "expected_row_order",
    "primary_comparison_columns",
    "additive_indicator_columns",
    "combined_component_columns",
    "review_star_scaling_maximum",
    "recency_maximum_observation_window_days",
    "recency_missing_imputation_risk",
    "submitter_minimum_log1p_count",
    "submitter_log1p_range",
    "entropy_positive_threshold",
    "single_submitter_threshold",
    "stale_recency_instability_threshold",
    "additive_component_count",
    "combined_component_count",
]

missing_runtime_objects = [
    object_name
    for object_name in required_runtime_objects
    if object_name not in globals()
]

if missing_runtime_objects:
    raise RuntimeError(
        "Cell 3B must be executed successfully before Cell 3C.\n"
        "Missing runtime objects:\n"
        + "\n".join(
            f" - {object_name}"
            for object_name in missing_runtime_objects
        )
    )


# --------------------------------------------------------------------------------------------------
# 2. Define future frozen-artifact paths
#
# These paths are recorded now, but this cell does not create them.
# --------------------------------------------------------------------------------------------------

COMPARATOR_OUTPUT_PATH = Path(
    comparator_policy["planned_output"]["path"]
)

STAGE6_QC_DIR = (
    PROJECT_ROOT
    / "outputs"
    / "quality_checks"
    / "stage6_temporal_validation"
)

COMPARATOR_QC_PATH = (
    STAGE6_QC_DIR
    / "stage6a_t0_comparator_score_construction_qc_v1.json"
)

COMPARATOR_FREEZE_MANIFEST_PATH = (
    STAGE6_CONFIG_DIR
    / "stage6a_t0_comparator_score_freeze_manifest_v1.json"
)

COMPARATOR_PARQUET_SIDECAR_PATH = Path(
    str(COMPARATOR_OUTPUT_PATH) + ".sha256"
)

COMPARATOR_QC_SIDECAR_PATH = Path(
    str(COMPARATOR_QC_PATH) + ".sha256"
)

COMPARATOR_FREEZE_MANIFEST_SIDECAR_PATH = Path(
    str(COMPARATOR_FREEZE_MANIFEST_PATH) + ".sha256"
)


# --------------------------------------------------------------------------------------------------
# 3. Helper functions
# --------------------------------------------------------------------------------------------------

def count_numeric_mismatches(
    observed,
    expected,
    *,
    atol: float = 1e-15,
) -> int:
    """Count numeric row-level mismatches using zero relative tolerance."""

    observed_array = np.asarray(
        observed,
        dtype=float,
    )

    expected_array = np.asarray(
        expected,
        dtype=float,
    )

    return int(
        (
            ~np.isclose(
                observed_array,
                expected_array,
                rtol=0.0,
                atol=atol,
                equal_nan=True,
            )
        ).sum()
    )


def count_exact_mismatches(
    observed,
    expected,
) -> int:
    """Count exact row-level mismatches."""

    observed_array = np.asarray(observed)
    expected_array = np.asarray(expected)

    return int(
        (
            observed_array
            != expected_array
        ).sum()
    )


def numeric_profile(
    values: pd.Series,
) -> dict:
    """Create a JSON-safe numeric profile for one output column."""

    numeric_values = pd.to_numeric(
        values,
        errors="coerce",
    )

    nonmissing_values = (
        numeric_values
        .dropna()
        .astype(float)
    )

    nonfinite_count = int(
        (
            ~np.isfinite(
                nonmissing_values.to_numpy()
            )
        ).sum()
    )

    quantile_levels = [
        0.00,
        0.01,
        0.05,
        0.10,
        0.25,
        0.50,
        0.75,
        0.90,
        0.95,
        0.99,
        1.00,
    ]

    quantiles = {
        f"{quantile_level:.2f}":
            float(
                nonmissing_values.quantile(
                    quantile_level
                )
            )
        for quantile_level in quantile_levels
    }

    return {
        "count": int(numeric_values.notna().sum()),
        "missing_count": int(numeric_values.isna().sum()),
        "nonfinite_count": nonfinite_count,
        "unique_value_count": int(
            numeric_values.nunique(
                dropna=False
            )
        ),
        "minimum": float(nonmissing_values.min()),
        "maximum": float(nonmissing_values.max()),
        "mean": float(nonmissing_values.mean()),
        "standard_deviation": float(
            nonmissing_values.std(ddof=1)
        ),
        "quantiles": quantiles,
    }


def sha256_text(
    text: str,
) -> str:
    """Calculate SHA-256 for UTF-8 text."""

    return hashlib.sha256(
        text.encode("utf-8")
    ).hexdigest()


# --------------------------------------------------------------------------------------------------
# 4. Independently reconstruct every comparator formula
# --------------------------------------------------------------------------------------------------

expected_review_risk = np.clip(
    (
        review_star_scaling_maximum
        - feature_source[
            "aggregate_review_stars"
        ].to_numpy(dtype=float)
    )
    / review_star_scaling_maximum,
    0.0,
    1.0,
)

expected_conflict_risk = (
    feature_source[
        "aggregate_conflict_flag"
    ]
    .astype(float)
    .to_numpy()
)

expected_recency_risk = np.where(
    feature_source[
        "recency_missing_flag"
    ].to_numpy(),
    recency_missing_imputation_risk,
    np.clip(
        (
            feature_source["recency_days"]
            .fillna(0.0)
            .to_numpy(dtype=float)
        )
        / recency_maximum_observation_window_days,
        0.0,
        1.0,
    ),
)

expected_recency_missing_component = (
    feature_source[
        "recency_missing_flag"
    ]
    .astype(float)
    .to_numpy()
)

expected_submitter_risk = (
    1.0
    - np.clip(
        (
            feature_source[
                "log1p_unique_submitter_count"
            ].to_numpy(dtype=float)
            - submitter_minimum_log1p_count
        )
        / submitter_log1p_range,
        0.0,
        1.0,
    )
)

expected_entropy_risk = np.clip(
    feature_source[
        "scv_group_entropy_normalized"
    ].to_numpy(dtype=float),
    0.0,
    1.0,
)

expected_additive_low_review = (
    feature_source[
        "aggregate_review_stars"
    ]
    .eq(0)
    .astype("int8")
    .to_numpy()
)

expected_additive_conflict = (
    feature_source[
        "aggregate_conflict_flag"
    ]
    .astype("int8")
    .to_numpy()
)

expected_additive_stale_recency = (
    (
        ~feature_source[
            "recency_missing_flag"
        ]
    )
    &
    (
        expected_recency_risk
        > stale_recency_instability_threshold
    )
).astype("int8")

expected_additive_missing_recency = (
    feature_source[
        "recency_missing_flag"
    ]
    .astype("int8")
    .to_numpy()
)

expected_additive_single_submitter = (
    feature_source[
        "unique_submitter_count"
    ]
    .eq(single_submitter_threshold)
    .astype("int8")
    .to_numpy()
)

expected_additive_entropy = (
    feature_source[
        "scv_group_entropy_normalized"
    ]
    .gt(entropy_positive_threshold)
    .astype("int8")
    .to_numpy()
)

expected_additive_count = (
    expected_additive_low_review
    + expected_additive_conflict
    + expected_additive_stale_recency
    + expected_additive_missing_recency
    + expected_additive_single_submitter
    + expected_additive_entropy
)

expected_additive_risk = (
    expected_additive_count.astype(float)
    / float(additive_component_count)
)

expected_combined_metadata_risk = (
    comparator_scores[
        combined_component_columns
    ]
    .sum(axis=1)
    .to_numpy(dtype=float)
    / float(combined_component_count)
)

expected_full_ges_instability = (
    1.0
    - comparator_scores[
        "full_ges_p_stable_t0"
    ].to_numpy(dtype=float)
)

expected_no_star_ges_instability = (
    1.0
    - comparator_scores[
        "no_star_ges_p_stable_t0"
    ].to_numpy(dtype=float)
)


# --------------------------------------------------------------------------------------------------
# 5. Calculate exact row-level formula-mismatch counts
# --------------------------------------------------------------------------------------------------

formula_mismatch_counts = {
    "review_stars_instability_risk":
        count_numeric_mismatches(
            comparator_scores[
                "review_stars_instability_risk"
            ],
            expected_review_risk,
        ),

    "conflict_instability_risk":
        count_exact_mismatches(
            comparator_scores[
                "conflict_instability_risk"
            ],
            expected_conflict_risk,
        ),

    "recency_instability_risk":
        count_numeric_mismatches(
            comparator_scores[
                "recency_instability_risk"
            ],
            expected_recency_risk,
        ),

    "recency_missing_instability_component":
        count_exact_mismatches(
            comparator_scores[
                "recency_missing_instability_component"
            ],
            expected_recency_missing_component,
        ),

    "submitter_instability_risk":
        count_numeric_mismatches(
            comparator_scores[
                "submitter_instability_risk"
            ],
            expected_submitter_risk,
        ),

    "entropy_instability_risk":
        count_numeric_mismatches(
            comparator_scores[
                "entropy_instability_risk"
            ],
            expected_entropy_risk,
        ),

    "additive_low_review_indicator":
        count_exact_mismatches(
            comparator_scores[
                "additive_low_review_indicator"
            ],
            expected_additive_low_review,
        ),

    "additive_conflict_indicator":
        count_exact_mismatches(
            comparator_scores[
                "additive_conflict_indicator"
            ],
            expected_additive_conflict,
        ),

    "additive_stale_recency_indicator":
        count_exact_mismatches(
            comparator_scores[
                "additive_stale_recency_indicator"
            ],
            expected_additive_stale_recency,
        ),

    "additive_missing_recency_indicator":
        count_exact_mismatches(
            comparator_scores[
                "additive_missing_recency_indicator"
            ],
            expected_additive_missing_recency,
        ),

    "additive_single_submitter_indicator":
        count_exact_mismatches(
            comparator_scores[
                "additive_single_submitter_indicator"
            ],
            expected_additive_single_submitter,
        ),

    "additive_entropy_indicator":
        count_exact_mismatches(
            comparator_scores[
                "additive_entropy_indicator"
            ],
            expected_additive_entropy,
        ),

    "additive_risk_count":
        count_exact_mismatches(
            comparator_scores[
                "additive_risk_count"
            ],
            expected_additive_count,
        ),

    "additive_instability_risk":
        count_numeric_mismatches(
            comparator_scores[
                "additive_instability_risk"
            ],
            expected_additive_risk,
        ),

    "combined_metadata_instability_risk":
        count_numeric_mismatches(
            comparator_scores[
                "combined_metadata_instability_risk"
            ],
            expected_combined_metadata_risk,
        ),

    "full_ges_instability_risk_t0":
        count_numeric_mismatches(
            comparator_scores[
                "full_ges_instability_risk_t0"
            ],
            expected_full_ges_instability,
        ),

    "no_star_ges_instability_risk_t0":
        count_numeric_mismatches(
            comparator_scores[
                "no_star_ges_instability_risk_t0"
            ],
            expected_no_star_ges_instability,
        ),
}


# --------------------------------------------------------------------------------------------------
# 6. Profile score columns
# --------------------------------------------------------------------------------------------------

all_profiled_score_columns = [
    "review_stars_instability_risk",
    "conflict_instability_risk",
    "recency_instability_risk",
    "recency_missing_instability_component",
    "submitter_instability_risk",
    "entropy_instability_risk",
    "additive_instability_risk",
    "combined_metadata_instability_risk",
    "full_ges_p_stable_t0",
    "full_ges_instability_risk_t0",
    "no_star_ges_p_stable_t0",
    "no_star_ges_instability_risk_t0",
]

score_profiles = {
    column_name:
        numeric_profile(
            comparator_scores[column_name]
        )
    for column_name in all_profiled_score_columns
}


# --------------------------------------------------------------------------------------------------
# 7. Record source and constructed distributions
# --------------------------------------------------------------------------------------------------

review_star_distribution = {
    str(int(level)): int(count)
    for level, count
    in (
        feature_source[
            "aggregate_review_stars"
        ]
        .value_counts()
        .sort_index()
        .items()
    )
}

additive_count_distribution_dict = {
    str(int(level)): int(count)
    for level, count
    in (
        comparator_scores[
            "additive_risk_count"
        ]
        .value_counts()
        .sort_index()
        .items()
    )
}

additive_indicator_prevalence = {}

for column_name in additive_indicator_columns:
    positive_count = int(
        comparator_scores[column_name].sum()
    )

    additive_indicator_prevalence[
        column_name
    ] = {
        "positive_count": positive_count,
        "positive_percentage": float(
            positive_count
            / EXPECTED_ROWS
            * 100.0
        ),
    }


# --------------------------------------------------------------------------------------------------
# 8. Create final QC pass/fail checks
# --------------------------------------------------------------------------------------------------

unit_interval_columns = [
    "review_stars_instability_risk",
    "conflict_instability_risk",
    "recency_instability_risk",
    "recency_missing_instability_component",
    "submitter_instability_risk",
    "entropy_instability_risk",
    "additive_instability_risk",
    "combined_metadata_instability_risk",
    "full_ges_p_stable_t0",
    "full_ges_instability_risk_t0",
    "no_star_ges_p_stable_t0",
    "no_star_ges_instability_risk_t0",
]

all_unit_interval = all(
    comparator_scores[column_name]
    .between(
        0.0,
        1.0,
        inclusive="both",
    )
    .all()
    for column_name in unit_interval_columns
)

all_finite = all(
    np.isfinite(
        pd.to_numeric(
            comparator_scores[column_name],
            errors="coerce",
        ).to_numpy(dtype=float)
    ).all()
    for column_name in unit_interval_columns
)

binary_indicator_values_valid = all(
    set(
        comparator_scores[column_name]
        .astype(int)
        .unique()
        .tolist()
    ).issubset({0, 1})
    for column_name in additive_indicator_columns
)

qc_checks = {
    "expected_row_count":
        len(comparator_scores)
        == EXPECTED_ROWS,

    "expected_column_count":
        len(comparator_scores.columns)
        == len(expected_output_columns),

    "exact_frozen_column_order":
        list(comparator_scores.columns)
        == expected_output_columns,

    "complete_rcv_keys":
        comparator_scores[
            "rcv_accession"
        ].notna().all(),

    "unique_rcv_keys":
        comparator_scores[
            "rcv_accession"
        ].is_unique,

    "exact_zero_based_row_order":
        np.array_equal(
            comparator_scores[
                "t0_row_order"
            ].to_numpy(),
            expected_row_order,
        ),

    "exact_feature_source_rcv_order":
        np.array_equal(
            comparator_scores[
                "rcv_accession"
            ].to_numpy(),
            feature_source[
                "rcv_accession"
            ].to_numpy(),
        ),

    "exact_ges_source_rcv_order":
        np.array_equal(
            comparator_scores[
                "rcv_accession"
            ].to_numpy(),
            ges_source[
                "rcv_accession"
            ].to_numpy(),
        ),

    "no_missing_output_values":
        int(
            comparator_scores
            .isna()
            .sum()
            .sum()
        ) == 0,

    "all_profiled_values_finite":
        all_finite,

    "all_scores_in_unit_interval":
        all_unit_interval,

    "binary_indicator_values_valid":
        binary_indicator_values_valid,

    "additive_count_within_frozen_range":
        comparator_scores[
            "additive_risk_count"
        ]
        .between(
            0,
            additive_component_count,
            inclusive="both",
        )
        .all(),

    "all_formula_mismatch_counts_zero":
        all(
            mismatch_count == 0
            for mismatch_count
            in formula_mismatch_counts.values()
        ),

    "policy_version_on_every_row":
        comparator_scores[
            "comparator_policy_version"
        ]
        .eq(
            comparator_policy["version"]
        )
        .all(),

    "policy_sha256_on_every_row":
        comparator_scores[
            "comparator_policy_sha256"
        ]
        .eq(
            observed_policy_sha256
        )
        .all(),

    "stage6a_step2_version_on_every_row":
        comparator_scores[
            "stage6a_step2_version"
        ]
        .eq(
            comparator_policy["version"]
        )
        .all(),

    "outcome_labels_loaded_flag_all_false":
        comparator_scores[
            "outcome_labels_loaded"
        ]
        .eq(False)
        .all(),

    "temporal_performance_flag_all_false":
        comparator_scores[
            "temporal_performance_evaluated"
        ]
        .eq(False)
        .all(),

    "no_outcome_columns_present":
        not any(
            (
                "outcome" in column_name.lower()
                and column_name
                != "outcome_labels_loaded"
            )
            or (
                "future_instability" in column_name.lower()
            )
            for column_name
            in comparator_scores.columns
        ),

    "combined_component_weights_sum_to_one":
        np.isclose(
            sum(
                comparator_policy[
                    "score_definitions"
                ][
                    "strong_combined_metadata_heuristic"
                ][
                    "component_weights"
                ].values()
            ),
            1.0,
            rtol=0.0,
            atol=1e-15,
        ),
}

failed_qc_checks = [
    check_name
    for check_name, passed
    in qc_checks.items()
    if not bool(passed)
]

if failed_qc_checks:
    raise RuntimeError(
        "Comparator-construction QC failed:\n"
        + "\n".join(
            f" - {check_name}"
            for check_name in failed_qc_checks
        )
    )


# --------------------------------------------------------------------------------------------------
# 9. Create the complete JSON-serializable QC report
# --------------------------------------------------------------------------------------------------

qc_created_at_utc = (
    datetime.now(timezone.utc)
    .isoformat()
    .replace("+00:00", "Z")
)

comparator_qc_report = {
    "qc_id":
        "GES_STAGE6A_COMPARATOR_SCORE_CONSTRUCTION_QC",

    "qc_name":
        "Stage 6A T0 comparator-score construction QC",

    "version":
        "1.0.0",

    "created_at_utc":
        qc_created_at_utc,

    "status":
        "PASS_CREATED_IN_MEMORY_NOT_YET_FROZEN",

    "study_phase":
        "Experiment 1 Stage 6A Step 3",

    "unit_of_analysis":
        comparator_policy["unit_of_analysis"],

    "positive_class":
        comparator_policy["positive_class"],

    "global_score_direction":
        comparator_policy["global_score_direction"],

    "global_score_range":
        comparator_policy["global_score_range"],

    "input_artifacts": {
        "stage4a_feature_table": {
            "path": str(FEATURE_TABLE_PATH),
            "sha256": observed_feature_sha256,
        },
        "stage4c_score_table": {
            "path": str(GES_SCORE_TABLE_PATH),
            "sha256": observed_score_sha256,
        },
        "stage6a_comparator_policy": {
            "path": str(COMPARATOR_POLICY_PATH),
            "version": comparator_policy["version"],
            "sha256": observed_policy_sha256,
        },
    },

    "planned_frozen_outputs": {
        "comparator_parquet": str(
            COMPARATOR_OUTPUT_PATH
        ),
        "comparator_parquet_sha256_sidecar": str(
            COMPARATOR_PARQUET_SIDECAR_PATH
        ),
        "construction_qc_json": str(
            COMPARATOR_QC_PATH
        ),
        "construction_qc_sha256_sidecar": str(
            COMPARATOR_QC_SIDECAR_PATH
        ),
        "freeze_manifest_json": str(
            COMPARATOR_FREEZE_MANIFEST_PATH
        ),
        "freeze_manifest_sha256_sidecar": str(
            COMPARATOR_FREEZE_MANIFEST_SIDECAR_PATH
        ),
    },

    "constructed_table": {
        "rows": int(
            len(comparator_scores)
        ),
        "columns": int(
            len(comparator_scores.columns)
        ),
        "column_order": list(
            comparator_scores.columns
        ),
        "key": "rcv_accession",
        "unique_rcv_accessions": int(
            comparator_scores[
                "rcv_accession"
            ].nunique()
        ),
        "t0_row_order_minimum": int(
            comparator_scores[
                "t0_row_order"
            ].min()
        ),
        "t0_row_order_maximum": int(
            comparator_scores[
                "t0_row_order"
            ].max()
        ),
        "missing_output_values": int(
            comparator_scores
            .isna()
            .sum()
            .sum()
        ),
    },

    "frozen_constants_used": {
        key: value
        for key, value
        in comparator_policy[
            "frozen_constants"
        ].items()
    },

    "formula_mismatch_counts":
        formula_mismatch_counts,

    "score_profiles":
        score_profiles,

    "source_distributions": {
        "aggregate_review_stars":
            review_star_distribution,

        "aggregate_conflict_positive_count":
            int(
                feature_source[
                    "aggregate_conflict_flag"
                ].sum()
            ),

        "recency_available_count":
            int(
                (
                    ~feature_source[
                        "recency_missing_flag"
                    ]
                ).sum()
            ),

        "recency_missing_count":
            int(
                feature_source[
                    "recency_missing_flag"
                ].sum()
            ),

        "single_submitter_count":
            int(
                feature_source[
                    "unique_submitter_count"
                ]
                .eq(
                    single_submitter_threshold
                )
                .sum()
            ),

        "positive_entropy_count":
            int(
                feature_source[
                    "scv_group_entropy_normalized"
                ]
                .gt(
                    entropy_positive_threshold
                )
                .sum()
            ),
    },

    "additive_indicator_prevalence":
        additive_indicator_prevalence,

    "additive_risk_count_distribution":
        additive_count_distribution_dict,

    "observed_additive_risk_count_minimum":
        int(
            comparator_scores[
                "additive_risk_count"
            ].min()
        ),

    "observed_additive_risk_count_maximum":
        int(
            comparator_scores[
                "additive_risk_count"
            ].max()
        ),

    "permitted_additive_risk_count_range": [
        0,
        int(additive_component_count),
    ],

    "qc_checks": {
        check_name: bool(passed)
        for check_name, passed
        in qc_checks.items()
    },

    "scientific_boundary": {
        "stage5_outcome_table_opened_in_this_cell":
            False,

        "outcome_labels_loaded_in_this_cell":
            False,

        "score_outcome_join_created":
            False,

        "threshold_optimization_performed":
            False,

        "weight_optimization_performed":
            False,

        "temporal_performance_calculated":
            False,

        "comparator_parquet_written":
            False,

        "qc_report_written":
            False,

        "freeze_manifest_written":
            False,

        "files_modified":
            False,
    },

    "software": {
        "python": sys.version,
        "platform": platform.platform(),
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "pyarrow": pyarrow.__version__,
    },

    "decision":
        "PASS_STAGE6A_COMPARATOR_CONSTRUCTION_QC_CREATED_IN_MEMORY",
}


# --------------------------------------------------------------------------------------------------
# 10. Create the exact future QC JSON text and its in-memory SHA-256
#
# The next cell must write this exact string without regenerating it.
# --------------------------------------------------------------------------------------------------

comparator_qc_json_text = (
    json.dumps(
        comparator_qc_report,
        indent=2,
        sort_keys=True,
        ensure_ascii=False,
        allow_nan=False,
    )
    + "\n"
)

comparator_qc_sha256_in_memory = (
    sha256_text(
        comparator_qc_json_text
    )
)


# --------------------------------------------------------------------------------------------------
# 11. Create user-readable QC summary tables
# --------------------------------------------------------------------------------------------------

qc_check_table = pd.DataFrame(
    [
        {
            "check": check_name,
            "status": (
                "PASS"
                if bool(passed)
                else "FAIL"
            ),
        }
        for check_name, passed
        in qc_checks.items()
    ]
)

formula_mismatch_table = pd.DataFrame(
    [
        {
            "constructed_field": field_name,
            "mismatch_count": mismatch_count,
        }
        for field_name, mismatch_count
        in formula_mismatch_counts.items()
    ]
)

indicator_prevalence_table = pd.DataFrame(
    [
        {
            "indicator": indicator_name,
            "positive_count":
                indicator_values[
                    "positive_count"
                ],
            "positive_percentage":
                indicator_values[
                    "positive_percentage"
                ],
        }
        for indicator_name, indicator_values
        in additive_indicator_prevalence.items()
    ]
)

score_profile_table = pd.DataFrame(
    [
        {
            "score": column_name,
            "count": profile["count"],
            "missing": profile["missing_count"],
            "nonfinite": profile["nonfinite_count"],
            "unique_values":
                profile["unique_value_count"],
            "minimum": profile["minimum"],
            "median":
                profile["quantiles"]["0.50"],
            "mean": profile["mean"],
            "maximum": profile["maximum"],
        }
        for column_name, profile
        in score_profiles.items()
    ]
)


# --------------------------------------------------------------------------------------------------
# 12. Print results
# --------------------------------------------------------------------------------------------------

print("=" * 124)
print("STAGE 6A STEP 3 — CELL 3C — COMPARATOR-CONSTRUCTION QC REPORT")
print("=" * 124)

print("\nQC PASS/FAIL CHECKS")
print("-" * 124)

display(
    qc_check_table
)

print("\nFORMULA MISMATCH COUNTS")
print("-" * 124)

display(
    formula_mismatch_table
)

print("\nADDITIVE INDICATOR PREVALENCE")
print("-" * 124)

display(
    indicator_prevalence_table
)

print("\nSCORE PROFILE SUMMARY")
print("-" * 124)

display(
    score_profile_table
)

print("\nIN-MEMORY QC ARTIFACT")
print("-" * 124)

print(
    f"QC report version:                     "
    f"{comparator_qc_report['version']}"
)

print(
    f"QC JSON character count:               "
    f"{len(comparator_qc_json_text):,}"
)

print(
    f"QC JSON UTF-8 byte count:              "
    f"{len(comparator_qc_json_text.encode('utf-8')):,}"
)

print(
    f"QC JSON in-memory SHA-256:             "
    f"{comparator_qc_sha256_in_memory}"
)

print(
    f"Planned QC path:                       "
    f"{COMPARATOR_QC_PATH}"
)

print(
    f"Planned comparator path:               "
    f"{COMPARATOR_OUTPUT_PATH}"
)

print(
    f"Planned freeze-manifest path:          "
    f"{COMPARATOR_FREEZE_MANIFEST_PATH}"
)

print("\n" + "=" * 124)
print("STAGE 6A STEP 3 — CELL 3C DECISION")
print("=" * 124)

print(
    "PASS_STAGE6A_COMPARATOR_CONSTRUCTION_QC_CREATED_IN_MEMORY"
)

print()
print(f"QC checks passed:                       {len(qc_checks):,}")
print(f"QC checks failed:                       {len(failed_qc_checks):,}")
print(
    "Formula mismatches:                    "
    f"{sum(formula_mismatch_counts.values()):,}"
)
print(
    "Observed additive-count range:         "
    f"{comparator_scores['additive_risk_count'].min()} "
    f"to {comparator_scores['additive_risk_count'].max()}"
)
print(
    "Permitted additive-count range:        "
    f"0 to {additive_component_count}"
)

print()
print("Stage 5 outcome table opened:           NO")
print("Outcome labels loaded:                  NO")
print("Score-outcome join created:             NO")
print("Temporal performance examined:          NO")
print("Comparator Parquet written:             NO")
print("QC JSON written:                        NO")
print("Freeze manifest written:                NO")
print("Files written or modified:              NO")

print()
print("NEXT AUTHORIZED ACTION:")
print(
    "Write and checksum-freeze the comparator Parquet, "
    "the exact in-memory QC JSON, their SHA-256 sidecars, "
    "and the comparator-score freeze manifest."
)

STAGE 6A STEP 3 — CELL 3C — COMPARATOR-CONSTRUCTION QC REPORT

QC PASS/FAIL CHECKS
----------------------------------------------------------------------------------------------------------------------------


,check,status
0,expected_row_count,PASS
1,expected_column_count,PASS
2,exact_frozen_column_order,PASS
3,complete_rcv_keys,PASS
4,unique_rcv_keys,PASS
5,exact_zero_based_row_order,PASS
6,exact_feature_source_rcv_order,PASS
7,exact_ges_source_rcv_order,PASS
8,no_missing_output_values,PASS
9,all_profiled_values_finite,PASS



FORMULA MISMATCH COUNTS
----------------------------------------------------------------------------------------------------------------------------


,constructed_field,mismatch_count
0,review_stars_instability_risk,0
1,conflict_instability_risk,0
2,recency_instability_risk,0
3,recency_missing_instability_component,0
4,submitter_instability_risk,0
5,entropy_instability_risk,0
6,additive_low_review_indicator,0
7,additive_conflict_indicator,0
8,additive_stale_recency_indicator,0
9,additive_missing_recency_indicator,0



ADDITIVE INDICATOR PREVALENCE
----------------------------------------------------------------------------------------------------------------------------


,indicator,positive_count,positive_percentage
0,additive_low_review_indicator,3878,5.411742
1,additive_conflict_indicator,1484,2.070919
2,additive_stale_recency_indicator,368,0.513543
3,additive_missing_recency_indicator,5889,8.218088
4,additive_single_submitter_indicator,54553,76.128609
5,additive_entropy_indicator,3197,4.461407



SCORE PROFILE SUMMARY
----------------------------------------------------------------------------------------------------------------------------


,score,count,missing,nonfinite,unique_values,minimum,median,mean,maximum
0,review_stars_instability_risk,71659,0,0,4,0.000000e+00,0.666667,0.565902,1.000000
1,conflict_instability_risk,71659,0,0,2,0.000000e+00,0.000000,0.020709,1.000000
2,recency_instability_risk,71659,0,0,3833,7.607455e-04,0.099563,0.125037,1.000000
3,recency_missing_instability_component,71659,0,0,2,0.000000e+00,0.000000,0.082181,1.000000
4,submitter_instability_risk,71659,0,0,22,0.000000e+00,1.000000,0.950689,1.000000
5,entropy_instability_risk,71659,0,0,39,0.000000e+00,0.000000,0.040829,1.000000
6,additive_instability_risk,71659,0,0,4,0.000000e+00,0.166667,0.161341,0.500000
7,combined_metadata_instability_risk,71659,0,0,9570,3.873463e-02,0.287197,0.297558,0.769716
8,full_ges_p_stable_t0,71659,0,0,9570,2.174233e-15,0.999997,0.889359,1.000000
9,full_ges_instability_risk_t0,71659,0,0,9570,1.065592e-12,0.000003,0.110641,1.000000



IN-MEMORY QC ARTIFACT
----------------------------------------------------------------------------------------------------------------------------
QC report version:                     1.0.0
QC JSON character count:               16,118
QC JSON UTF-8 byte count:              16,118
QC JSON in-memory SHA-256:             463b94902c55494dbfb965db695a38bc83921394cc6c057b8ef06e1a91b149e7
Planned QC path:                       /content/drive/MyDrive/GES_RAG_Temporal_Study/outputs/quality_checks/stage6_temporal_validation/stage6a_t0_comparator_score_construction_qc_v1.json
Planned comparator path:               /content/drive/MyDrive/GES_RAG_Temporal_Study/data_processed/stage6_temporal_validation/stage6a_t0_comparator_scores_v1.parquet
Planned freeze-manifest path:          /content/drive/MyDrive/GES_RAG_Temporal_Study/configs/stage6_temporal_validation/stage6a_t0_comparator_score_freeze_manifest_v1.json

STAGE 6A STEP 3 — CELL 3C DECISION
PASS_STAGE6A_COMPARATOR_CONSTRUCTION_QC_CREATED_I

In [ ]:
# ==================================================================================================
# STAGE 6A — STEP 3 — CELL 3D
# WRITE, CHECKSUM, READ BACK, AND FREEZE COMPARATOR-SCORE ARTIFACTS
#
# Purpose:
#   1. Write the frozen comparator-score Parquet.
#   2. Write the exact QC JSON created in Cell 3C.
#   3. Create SHA-256 sidecars for both artifacts.
#   4. Read both artifacts back and verify exact integrity.
#   5. Create and checksum-freeze the comparator-score freeze manifest.
#
# Scientific boundary:
#   - Stage 5 outcome data are NOT opened.
#   - No outcome labels are loaded.
#   - No score-outcome join is created.
#   - No threshold or weight optimization is performed.
#   - No temporal performance is calculated.
# ==================================================================================================

from datetime import datetime, timezone
from pathlib import Path
import hashlib
import json
import os
import platform
import shutil
import sys
import tempfile

import numpy as np
import pandas as pd
import pyarrow
import pyarrow.parquet as pq


# --------------------------------------------------------------------------------------------------
# 1. Confirm that Cell 3C completed successfully
# --------------------------------------------------------------------------------------------------

required_runtime_objects = [
    "comparator_scores",
    "comparator_qc_report",
    "comparator_qc_json_text",
    "comparator_qc_sha256_in_memory",
    "comparator_policy",
    "observed_policy_sha256",
    "observed_feature_sha256",
    "observed_score_sha256",
    "COMPARATOR_OUTPUT_PATH",
    "COMPARATOR_QC_PATH",
    "COMPARATOR_FREEZE_MANIFEST_PATH",
    "COMPARATOR_PARQUET_SIDECAR_PATH",
    "COMPARATOR_QC_SIDECAR_PATH",
    "COMPARATOR_FREEZE_MANIFEST_SIDECAR_PATH",
    "EXPECTED_ROWS",
    "expected_output_columns",
    "expected_row_order",
]

missing_runtime_objects = [
    object_name
    for object_name in required_runtime_objects
    if object_name not in globals()
]

if missing_runtime_objects:
    raise RuntimeError(
        "Cell 3C must be executed successfully before Cell 3D.\n"
        "Missing runtime objects:\n"
        + "\n".join(
            f" - {object_name}"
            for object_name in missing_runtime_objects
        )
    )


# --------------------------------------------------------------------------------------------------
# 2. Verify the in-memory construction boundary before writing
# --------------------------------------------------------------------------------------------------

prewrite_checks = {
    "expected rows":
        len(comparator_scores) == EXPECTED_ROWS,

    "exact output schema":
        list(comparator_scores.columns)
        == expected_output_columns,

    "unique RCV keys":
        comparator_scores["rcv_accession"].is_unique,

    "complete RCV keys":
        comparator_scores["rcv_accession"].notna().all(),

    "exact zero-based row order":
        np.array_equal(
            comparator_scores["t0_row_order"].to_numpy(),
            expected_row_order,
        ),

    "no missing output values":
        int(
            comparator_scores
            .isna()
            .sum()
            .sum()
        ) == 0,

    "policy SHA-256 on every row":
        comparator_scores["comparator_policy_sha256"]
        .eq(observed_policy_sha256)
        .all(),

    "outcome labels remain unloaded":
        comparator_scores["outcome_labels_loaded"]
        .eq(False)
        .all(),

    "temporal performance remains unevaluated":
        comparator_scores["temporal_performance_evaluated"]
        .eq(False)
        .all(),

    "QC report status is in-memory PASS":
        comparator_qc_report.get("status")
        == "PASS_CREATED_IN_MEMORY_NOT_YET_FROZEN",

    "QC decision is PASS":
        comparator_qc_report.get("decision")
        == "PASS_STAGE6A_COMPARATOR_CONSTRUCTION_QC_CREATED_IN_MEMORY",

    "QC in-memory SHA-256 is reproducible":
        hashlib.sha256(
            comparator_qc_json_text.encode("utf-8")
        ).hexdigest()
        == comparator_qc_sha256_in_memory,
}

failed_prewrite_checks = [
    check_name
    for check_name, passed
    in prewrite_checks.items()
    if not bool(passed)
]

if failed_prewrite_checks:
    raise RuntimeError(
        "Prewrite verification failed. No files were written.\n"
        + "\n".join(
            f" - {check_name}"
            for check_name in failed_prewrite_checks
        )
    )


# --------------------------------------------------------------------------------------------------
# 3. Helper functions
# --------------------------------------------------------------------------------------------------

def calculate_sha256(
    path: Path,
    chunk_size: int = 8 * 1024 * 1024,
) -> str:
    """Calculate SHA-256 without loading the complete file into memory."""

    digest = hashlib.sha256()

    with path.open("rb") as file_handle:
        while True:
            chunk = file_handle.read(chunk_size)

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


def atomic_write_text(
    destination: Path,
    text: str,
) -> None:
    """Write UTF-8 text through a temporary file and atomically replace destination."""

    destination.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = Path(
        str(destination) + ".tmp"
    )

    if temporary_path.exists():
        temporary_path.unlink()

    with temporary_path.open(
        "w",
        encoding="utf-8",
        newline="\n",
    ) as file_handle:
        file_handle.write(text)
        file_handle.flush()
        os.fsync(file_handle.fileno())

    os.replace(
        temporary_path,
        destination,
    )


def atomic_write_parquet(
    dataframe: pd.DataFrame,
    destination: Path,
) -> None:
    """Write Parquet to a temporary file and atomically replace destination."""

    destination.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = Path(
        str(destination) + ".tmp"
    )

    if temporary_path.exists():
        temporary_path.unlink()

    dataframe.to_parquet(
        temporary_path,
        engine="pyarrow",
        compression="zstd",
        index=False,
    )

    os.replace(
        temporary_path,
        destination,
    )


def create_sha256_sidecar_text(
    sha256_value: str,
    artifact_path: Path,
) -> str:
    """Create a conventional checksum sidecar line."""

    return (
        f"{sha256_value}  {artifact_path.name}\n"
    )


# --------------------------------------------------------------------------------------------------
# 4. Prevent silent overwrite of an existing frozen package
# --------------------------------------------------------------------------------------------------

planned_paths = [
    COMPARATOR_OUTPUT_PATH,
    COMPARATOR_QC_PATH,
    COMPARATOR_PARQUET_SIDECAR_PATH,
    COMPARATOR_QC_SIDECAR_PATH,
    COMPARATOR_FREEZE_MANIFEST_PATH,
    COMPARATOR_FREEZE_MANIFEST_SIDECAR_PATH,
]

existing_paths = [
    path
    for path in planned_paths
    if path.exists()
]

if existing_paths:
    raise FileExistsError(
        "One or more planned frozen artifacts already exist.\n"
        "This cell will not overwrite a prior frozen package.\n"
        + "\n".join(
            f" - {path}"
            for path in existing_paths
        )
    )


# --------------------------------------------------------------------------------------------------
# 5. Create output directories
# --------------------------------------------------------------------------------------------------

for directory_path in {
    COMPARATOR_OUTPUT_PATH.parent,
    COMPARATOR_QC_PATH.parent,
    COMPARATOR_FREEZE_MANIFEST_PATH.parent,
}:
    directory_path.mkdir(
        parents=True,
        exist_ok=True,
    )


# --------------------------------------------------------------------------------------------------
# 6. Write comparator Parquet atomically
# --------------------------------------------------------------------------------------------------

atomic_write_parquet(
    comparator_scores,
    COMPARATOR_OUTPUT_PATH,
)

if not COMPARATOR_OUTPUT_PATH.exists():
    raise RuntimeError(
        "Comparator Parquet was not created."
    )

comparator_parquet_sha256 = calculate_sha256(
    COMPARATOR_OUTPUT_PATH
)


# --------------------------------------------------------------------------------------------------
# 7. Write the exact QC JSON created in Cell 3C
#
# Do not regenerate the JSON here.
# --------------------------------------------------------------------------------------------------

atomic_write_text(
    COMPARATOR_QC_PATH,
    comparator_qc_json_text,
)

if not COMPARATOR_QC_PATH.exists():
    raise RuntimeError(
        "Comparator QC JSON was not created."
    )

comparator_qc_sha256_observed = calculate_sha256(
    COMPARATOR_QC_PATH
)

if (
    comparator_qc_sha256_observed
    != comparator_qc_sha256_in_memory
):
    raise RuntimeError(
        "Written QC JSON does not match the exact in-memory QC JSON.\n"
        f"In-memory SHA-256: {comparator_qc_sha256_in_memory}\n"
        f"Written SHA-256:   {comparator_qc_sha256_observed}"
    )


# --------------------------------------------------------------------------------------------------
# 8. Write Parquet and QC SHA-256 sidecars
# --------------------------------------------------------------------------------------------------

comparator_parquet_sidecar_text = (
    create_sha256_sidecar_text(
        comparator_parquet_sha256,
        COMPARATOR_OUTPUT_PATH,
    )
)

comparator_qc_sidecar_text = (
    create_sha256_sidecar_text(
        comparator_qc_sha256_observed,
        COMPARATOR_QC_PATH,
    )
)

atomic_write_text(
    COMPARATOR_PARQUET_SIDECAR_PATH,
    comparator_parquet_sidecar_text,
)

atomic_write_text(
    COMPARATOR_QC_SIDECAR_PATH,
    comparator_qc_sidecar_text,
)


# --------------------------------------------------------------------------------------------------
# 9. Read back comparator Parquet and verify schema, data, key, and row-order integrity
# --------------------------------------------------------------------------------------------------

parquet_file = pq.ParquetFile(
    COMPARATOR_OUTPUT_PATH
)

written_parquet_rows = int(
    parquet_file.metadata.num_rows
)

written_parquet_columns = int(
    parquet_file.metadata.num_columns
)

written_parquet_schema = list(
    parquet_file.schema_arrow.names
)

comparator_readback = pd.read_parquet(
    COMPARATOR_OUTPUT_PATH,
    engine="pyarrow",
)

# Normalize readback dtypes only where Parquet may use an equivalent string representation.
comparator_readback["rcv_accession"] = (
    comparator_readback["rcv_accession"]
    .astype("string")
)

comparator_scores_for_comparison = (
    comparator_scores.copy()
)

comparator_scores_for_comparison["rcv_accession"] = (
    comparator_scores_for_comparison["rcv_accession"]
    .astype("string")
)

readback_checks = {
    "Parquet metadata row count":
        written_parquet_rows == EXPECTED_ROWS,

    "Parquet metadata column count":
        written_parquet_columns
        == len(expected_output_columns),

    "Parquet schema column order":
        written_parquet_schema
        == expected_output_columns,

    "readback dataframe row count":
        len(comparator_readback)
        == EXPECTED_ROWS,

    "readback dataframe column order":
        list(comparator_readback.columns)
        == expected_output_columns,

    "readback complete RCV keys":
        comparator_readback["rcv_accession"]
        .notna()
        .all(),

    "readback unique RCV keys":
        comparator_readback["rcv_accession"]
        .is_unique,

    "readback exact zero-based row order":
        np.array_equal(
            comparator_readback["t0_row_order"].to_numpy(),
            expected_row_order,
        ),

    "readback policy SHA-256 on every row":
        comparator_readback["comparator_policy_sha256"]
        .eq(observed_policy_sha256)
        .all(),

    "readback outcome flag all false":
        comparator_readback["outcome_labels_loaded"]
        .eq(False)
        .all(),

    "readback temporal-performance flag all false":
        comparator_readback["temporal_performance_evaluated"]
        .eq(False)
        .all(),

    "exact dataframe readback":
        comparator_readback.equals(
            comparator_scores_for_comparison
        ),
}

failed_readback_checks = [
    check_name
    for check_name, passed
    in readback_checks.items()
    if not bool(passed)
]

if failed_readback_checks:
    raise RuntimeError(
        "Comparator Parquet readback validation failed.\n"
        + "\n".join(
            f" - {check_name}"
            for check_name in failed_readback_checks
        )
    )


# --------------------------------------------------------------------------------------------------
# 10. Read back and validate both checksum sidecars
# --------------------------------------------------------------------------------------------------

parquet_sidecar_readback = (
    COMPARATOR_PARQUET_SIDECAR_PATH
    .read_text(encoding="utf-8")
    .strip()
    .split()
)

qc_sidecar_readback = (
    COMPARATOR_QC_SIDECAR_PATH
    .read_text(encoding="utf-8")
    .strip()
    .split()
)

sidecar_checks = {
    "Parquet sidecar hash":
        len(parquet_sidecar_readback) >= 1
        and parquet_sidecar_readback[0]
        == comparator_parquet_sha256,

    "Parquet sidecar filename":
        len(parquet_sidecar_readback) >= 2
        and parquet_sidecar_readback[-1]
        == COMPARATOR_OUTPUT_PATH.name,

    "QC sidecar hash":
        len(qc_sidecar_readback) >= 1
        and qc_sidecar_readback[0]
        == comparator_qc_sha256_observed,

    "QC sidecar filename":
        len(qc_sidecar_readback) >= 2
        and qc_sidecar_readback[-1]
        == COMPARATOR_QC_PATH.name,
}

failed_sidecar_checks = [
    check_name
    for check_name, passed
    in sidecar_checks.items()
    if not bool(passed)
]

if failed_sidecar_checks:
    raise RuntimeError(
        "Checksum-sidecar validation failed.\n"
        + "\n".join(
            f" - {check_name}"
            for check_name in failed_sidecar_checks
        )
    )


# --------------------------------------------------------------------------------------------------
# 11. Create the freeze manifest
# --------------------------------------------------------------------------------------------------

freeze_created_at_utc = (
    datetime.now(timezone.utc)
    .isoformat()
    .replace("+00:00", "Z")
)

comparator_freeze_manifest = {
    "manifest_id":
        "GES_STAGE6A_COMPARATOR_SCORE_FREEZE_MANIFEST",

    "manifest_name":
        "Stage 6A T0 comparator-score freeze manifest",

    "version":
        "1.0.0",

    "created_at_utc":
        freeze_created_at_utc,

    "status":
        "ACCEPTED_AND_FROZEN",

    "decision":
        "PASS_STAGE6A_COMPARATOR_SCORE_PACKAGE_ACCEPTED_AND_FROZEN",

    "study_phase":
        "Experiment 1 Stage 6A Step 3",

    "unit_of_analysis":
        comparator_policy["unit_of_analysis"],

    "positive_class":
        comparator_policy["positive_class"],

    "global_score_direction":
        comparator_policy["global_score_direction"],

    "comparator_policy": {
        "path": str(
            COMPARATOR_POLICY_PATH
        ),
        "version": comparator_policy["version"],
        "sha256": observed_policy_sha256,
        "status": comparator_policy["status"],
    },

    "source_artifacts": {
        "stage4a_feature_table": {
            "path": str(FEATURE_TABLE_PATH),
            "sha256": observed_feature_sha256,
        },
        "stage4c_score_table": {
            "path": str(GES_SCORE_TABLE_PATH),
            "sha256": observed_score_sha256,
        },
    },

    "frozen_artifacts": {
        "comparator_score_parquet": {
            "path": str(
                COMPARATOR_OUTPUT_PATH
            ),
            "sha256": comparator_parquet_sha256,
            "sha256_sidecar_path": str(
                COMPARATOR_PARQUET_SIDECAR_PATH
            ),
            "bytes": int(
                COMPARATOR_OUTPUT_PATH.stat().st_size
            ),
            "rows": written_parquet_rows,
            "columns": written_parquet_columns,
            "column_order": written_parquet_schema,
            "key": "rcv_accession",
            "unique_rcv_accessions": int(
                comparator_readback[
                    "rcv_accession"
                ].nunique()
            ),
            "t0_row_order_minimum": int(
                comparator_readback[
                    "t0_row_order"
                ].min()
            ),
            "t0_row_order_maximum": int(
                comparator_readback[
                    "t0_row_order"
                ].max()
            ),
            "format": "Parquet",
            "compression": "zstd",
        },

        "construction_qc_json": {
            "path": str(
                COMPARATOR_QC_PATH
            ),
            "sha256": comparator_qc_sha256_observed,
            "sha256_sidecar_path": str(
                COMPARATOR_QC_SIDECAR_PATH
            ),
            "bytes": int(
                COMPARATOR_QC_PATH.stat().st_size
            ),
            "version": comparator_qc_report["version"],
            "decision": comparator_qc_report["decision"],
            "qc_checks_passed": int(
                sum(
                    bool(value)
                    for value
                    in comparator_qc_report[
                        "qc_checks"
                    ].values()
                )
            ),
            "qc_checks_failed": int(
                sum(
                    not bool(value)
                    for value
                    in comparator_qc_report[
                        "qc_checks"
                    ].values()
                )
            ),
            "formula_mismatch_total": int(
                sum(
                    comparator_qc_report[
                        "formula_mismatch_counts"
                    ].values()
                )
            ),
        },
    },

    "verification": {
        "prewrite_checks": {
            check_name: bool(passed)
            for check_name, passed
            in prewrite_checks.items()
        },

        "parquet_readback_checks": {
            check_name: bool(passed)
            for check_name, passed
            in readback_checks.items()
        },

        "sidecar_checks": {
            check_name: bool(passed)
            for check_name, passed
            in sidecar_checks.items()
        },

        "written_qc_sha256_matches_in_memory_qc_sha256":
            comparator_qc_sha256_observed
            == comparator_qc_sha256_in_memory,

        "comparator_policy_sha256_on_every_row":
            comparator_readback[
                "comparator_policy_sha256"
            ]
            .eq(
                observed_policy_sha256
            )
            .all(),

        "exact_zero_based_row_order_preserved":
            np.array_equal(
                comparator_readback[
                    "t0_row_order"
                ].to_numpy(),
                expected_row_order,
            ),

        "exact_dataframe_readback":
            comparator_readback.equals(
                comparator_scores_for_comparison
            ),
    },

    "scientific_boundary": {
        "stage5_outcome_table_opened":
            False,

        "outcome_labels_loaded":
            False,

        "score_outcome_join_created":
            False,

        "threshold_optimization_performed":
            False,

        "weight_optimization_performed":
            False,

        "temporal_performance_calculated":
            False,

        "outcome_information_used_in_comparator_construction":
            False,

        "comparator_policy_frozen_before_score_construction":
            True,
    },

    "authorization": {
        "stage6a_step3_complete":
            True,

        "comparator_scores_frozen":
            True,

        "score_outcome_join_authorized_next":
            True,

        "temporal_performance_authorized_immediately":
            False,

        "next_authorized_action":
            (
                "Verify the frozen comparator-score package in a fresh "
                "read-only step, then create the locked score-outcome "
                "analysis cohort using the immutable Stage 5 outcomes."
            ),
    },

    "software": {
        "python": sys.version,
        "platform": platform.platform(),
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "pyarrow": pyarrow.__version__,
    },
}


# --------------------------------------------------------------------------------------------------
# 12. Serialize and write freeze manifest
# --------------------------------------------------------------------------------------------------

comparator_freeze_manifest_json_text = (
    json.dumps(
        comparator_freeze_manifest,
        indent=2,
        sort_keys=True,
        ensure_ascii=False,
        allow_nan=False,
    )
    + "\n"
)

atomic_write_text(
    COMPARATOR_FREEZE_MANIFEST_PATH,
    comparator_freeze_manifest_json_text,
)

if not COMPARATOR_FREEZE_MANIFEST_PATH.exists():
    raise RuntimeError(
        "Comparator freeze manifest was not created."
    )

comparator_freeze_manifest_sha256 = (
    calculate_sha256(
        COMPARATOR_FREEZE_MANIFEST_PATH
    )
)

comparator_freeze_manifest_sidecar_text = (
    create_sha256_sidecar_text(
        comparator_freeze_manifest_sha256,
        COMPARATOR_FREEZE_MANIFEST_PATH,
    )
)

atomic_write_text(
    COMPARATOR_FREEZE_MANIFEST_SIDECAR_PATH,
    comparator_freeze_manifest_sidecar_text,
)


# --------------------------------------------------------------------------------------------------
# 13. Read back and verify freeze manifest and sidecar
# --------------------------------------------------------------------------------------------------

freeze_manifest_readback_text = (
    COMPARATOR_FREEZE_MANIFEST_PATH
    .read_text(encoding="utf-8")
)

freeze_manifest_readback = json.loads(
    freeze_manifest_readback_text
)

freeze_manifest_observed_sha256 = (
    calculate_sha256(
        COMPARATOR_FREEZE_MANIFEST_PATH
    )
)

freeze_manifest_sidecar_parts = (
    COMPARATOR_FREEZE_MANIFEST_SIDECAR_PATH
    .read_text(encoding="utf-8")
    .strip()
    .split()
)

freeze_manifest_checks = {
    "freeze manifest SHA-256 stable":
        freeze_manifest_observed_sha256
        == comparator_freeze_manifest_sha256,

    "freeze manifest sidecar hash":
        len(freeze_manifest_sidecar_parts) >= 1
        and freeze_manifest_sidecar_parts[0]
        == comparator_freeze_manifest_sha256,

    "freeze manifest sidecar filename":
        len(freeze_manifest_sidecar_parts) >= 2
        and freeze_manifest_sidecar_parts[-1]
        == COMPARATOR_FREEZE_MANIFEST_PATH.name,

    "freeze manifest status":
        freeze_manifest_readback.get("status")
        == "ACCEPTED_AND_FROZEN",

    "freeze manifest decision":
        freeze_manifest_readback.get("decision")
        == "PASS_STAGE6A_COMPARATOR_SCORE_PACKAGE_ACCEPTED_AND_FROZEN",

    "frozen Parquet SHA-256 recorded correctly":
        freeze_manifest_readback[
            "frozen_artifacts"
        ][
            "comparator_score_parquet"
        ][
            "sha256"
        ]
        == comparator_parquet_sha256,

    "frozen QC SHA-256 recorded correctly":
        freeze_manifest_readback[
            "frozen_artifacts"
        ][
            "construction_qc_json"
        ][
            "sha256"
        ]
        == comparator_qc_sha256_observed,

    "policy SHA-256 recorded correctly":
        freeze_manifest_readback[
            "comparator_policy"
        ][
            "sha256"
        ]
        == observed_policy_sha256,

    "Stage 6A Step 3 marked complete":
        freeze_manifest_readback[
            "authorization"
        ][
            "stage6a_step3_complete"
        ] is True,

    "score-outcome join authorized next":
        freeze_manifest_readback[
            "authorization"
        ][
            "score_outcome_join_authorized_next"
        ] is True,

    "temporal performance not yet authorized immediately":
        freeze_manifest_readback[
            "authorization"
        ][
            "temporal_performance_authorized_immediately"
        ] is False,
}

failed_freeze_manifest_checks = [
    check_name
    for check_name, passed
    in freeze_manifest_checks.items()
    if not bool(passed)
]

if failed_freeze_manifest_checks:
    raise RuntimeError(
        "Freeze-manifest validation failed.\n"
        + "\n".join(
            f" - {check_name}"
            for check_name in failed_freeze_manifest_checks
        )
    )


# --------------------------------------------------------------------------------------------------
# 14. Create artifact summary table
# --------------------------------------------------------------------------------------------------

artifact_summary = pd.DataFrame(
    [
        {
            "artifact": "Comparator score Parquet",
            "path": str(COMPARATOR_OUTPUT_PATH),
            "bytes": int(COMPARATOR_OUTPUT_PATH.stat().st_size),
            "sha256": comparator_parquet_sha256,
            "status": "FROZEN",
        },
        {
            "artifact": "Comparator construction QC JSON",
            "path": str(COMPARATOR_QC_PATH),
            "bytes": int(COMPARATOR_QC_PATH.stat().st_size),
            "sha256": comparator_qc_sha256_observed,
            "status": "FROZEN",
        },
        {
            "artifact": "Comparator freeze manifest",
            "path": str(COMPARATOR_FREEZE_MANIFEST_PATH),
            "bytes": int(COMPARATOR_FREEZE_MANIFEST_PATH.stat().st_size),
            "sha256": comparator_freeze_manifest_sha256,
            "status": "FROZEN",
        },
    ]
)


# --------------------------------------------------------------------------------------------------
# 15. Print results
# --------------------------------------------------------------------------------------------------

print("=" * 124)
print("STAGE 6A STEP 3 — CELL 3D — COMPARATOR ARTIFACT FREEZE")
print("=" * 124)

print("\nPREWRITE CHECKS")
print("-" * 124)

for check_name, passed in prewrite_checks.items():
    print(
        f"{check_name:<82}"
        f"{'PASS' if passed else 'FAIL'}"
    )

print("\nPARQUET READBACK CHECKS")
print("-" * 124)

for check_name, passed in readback_checks.items():
    print(
        f"{check_name:<82}"
        f"{'PASS' if passed else 'FAIL'}"
    )

print("\nCHECKSUM SIDECAR CHECKS")
print("-" * 124)

for check_name, passed in sidecar_checks.items():
    print(
        f"{check_name:<82}"
        f"{'PASS' if passed else 'FAIL'}"
    )

print("\nFREEZE-MANIFEST CHECKS")
print("-" * 124)

for check_name, passed in freeze_manifest_checks.items():
    print(
        f"{check_name:<82}"
        f"{'PASS' if passed else 'FAIL'}"
    )

print("\nFROZEN ARTIFACT SUMMARY")
print("-" * 124)

display(
    artifact_summary
)

print("\n" + "=" * 124)
print("STAGE 6A STEP 3 — CELL 3D DECISION")
print("=" * 124)

print(
    "PASS_STAGE6A_COMPARATOR_SCORE_PACKAGE_ACCEPTED_AND_FROZEN"
)

print()
print(f"Frozen comparator rows:                 {written_parquet_rows:,}")
print(f"Frozen comparator columns:              {written_parquet_columns:,}")
print(
    "Unique frozen RCV keys:                "
    f"{comparator_readback['rcv_accession'].nunique():,}"
)
print(
    "Frozen T0 row-order range:             "
    f"{comparator_readback['t0_row_order'].min():,} "
    f"to {comparator_readback['t0_row_order'].max():,}"
)

print()
print(f"Comparator Parquet SHA-256:             {comparator_parquet_sha256}")
print(f"Comparator QC SHA-256:                  {comparator_qc_sha256_observed}")
print(f"Freeze manifest SHA-256:                {comparator_freeze_manifest_sha256}")
print(f"Comparator policy SHA-256:              {observed_policy_sha256}")

print()
print("Stage 5 outcome table opened:           NO")
print("Outcome labels loaded:                  NO")
print("Score-outcome join created:             NO")
print("Thresholds optimized:                   NO")
print("Weights optimized:                      NO")
print("Temporal performance examined:          NO")

print()
print("Comparator Parquet written:             YES")
print("Comparator QC JSON written:             YES")
print("Comparator SHA-256 sidecar written:     YES")
print("QC SHA-256 sidecar written:             YES")
print("Freeze manifest written:                YES")
print("Freeze-manifest sidecar written:        YES")
print("Exact Parquet readback verified:        YES")
print("Stage 6A Step 3 complete:               YES")

print()
print("NEXT AUTHORIZED ACTION:")
print(
    "Start Stage 6A Step 4 with a fresh read-only verification "
    "of the frozen comparator package before opening or joining "
    "the immutable Stage 5 outcome table."
)

TypeError: Object of type bool is not JSON serializable

In [ ]:
# ==================================================================================================
# STAGE 6A — STEP 3 — CELL 3D-R
# RECOVER FROM JSON-SERIALIZATION ERROR AND COMPLETE THE FREEZE
#
# Why this recovery is needed:
#   The comparator Parquet, QC JSON, and their SHA-256 sidecars were written successfully.
#   Manifest serialization then failed because some verification values were NumPy/Pandas
#   scalar types rather than native Python JSON types.
#
# This cell:
#   1. Does NOT rewrite the comparator Parquet or QC JSON.
#   2. Re-verifies the artifacts already written.
#   3. Converts NumPy/Pandas scalar values recursively to native Python types.
#   4. Writes and verifies the freeze manifest and its SHA-256 sidecar.
#
# Scientific boundary:
#   - Stage 5 outcome data are NOT opened.
#   - No outcome labels are loaded.
#   - No score-outcome join is created.
#   - No temporal performance is calculated.
# ==================================================================================================

from pathlib import Path
import hashlib
import json
import os

import numpy as np
import pandas as pd
import pyarrow.parquet as pq


# --------------------------------------------------------------------------------------------------
# 1. Confirm that the interrupted Cell 3D created the necessary runtime objects
# --------------------------------------------------------------------------------------------------

required_runtime_objects = [
    "comparator_freeze_manifest",
    "comparator_scores",
    "comparator_policy",
    "comparator_qc_report",
    "comparator_qc_sha256_in_memory",
    "observed_policy_sha256",
    "observed_feature_sha256",
    "observed_score_sha256",
    "COMPARATOR_OUTPUT_PATH",
    "COMPARATOR_QC_PATH",
    "COMPARATOR_PARQUET_SIDECAR_PATH",
    "COMPARATOR_QC_SIDECAR_PATH",
    "COMPARATOR_FREEZE_MANIFEST_PATH",
    "COMPARATOR_FREEZE_MANIFEST_SIDECAR_PATH",
    "EXPECTED_ROWS",
    "expected_output_columns",
    "expected_row_order",
]

missing_runtime_objects = [
    object_name
    for object_name in required_runtime_objects
    if object_name not in globals()
]

if missing_runtime_objects:
    raise RuntimeError(
        "The interrupted Cell 3D runtime objects are unavailable.\n"
        "Missing objects:\n"
        + "\n".join(
            f" - {object_name}"
            for object_name in missing_runtime_objects
        )
    )


# --------------------------------------------------------------------------------------------------
# 2. Required helper functions
# --------------------------------------------------------------------------------------------------

def calculate_sha256(
    path: Path,
    chunk_size: int = 8 * 1024 * 1024,
) -> str:
    """Calculate a file's SHA-256 without loading the entire file into memory."""

    digest = hashlib.sha256()

    with path.open("rb") as file_handle:
        while True:
            chunk = file_handle.read(chunk_size)

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


def atomic_write_text(
    destination: Path,
    text: str,
) -> None:
    """Write UTF-8 text through a temporary file and atomically replace destination."""

    destination.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = Path(
        str(destination) + ".tmp"
    )

    if temporary_path.exists():
        temporary_path.unlink()

    with temporary_path.open(
        "w",
        encoding="utf-8",
        newline="\n",
    ) as file_handle:
        file_handle.write(text)
        file_handle.flush()
        os.fsync(file_handle.fileno())

    os.replace(
        temporary_path,
        destination,
    )


def create_sha256_sidecar_text(
    sha256_value: str,
    artifact_path: Path,
) -> str:
    """Create a conventional SHA-256 sidecar line."""

    return (
        f"{sha256_value}  {artifact_path.name}\n"
    )


def to_json_native(value):
    """
    Recursively convert NumPy/Pandas scalar objects into native Python
    objects accepted by json.dumps().
    """

    if isinstance(value, dict):
        return {
            str(key): to_json_native(item)
            for key, item in value.items()
        }

    if isinstance(value, (list, tuple, set)):
        return [
            to_json_native(item)
            for item in value
        ]

    if isinstance(value, np.ndarray):
        return [
            to_json_native(item)
            for item in value.tolist()
        ]

    if isinstance(value, np.generic):
        return to_json_native(
            value.item()
        )

    if value is pd.NA:
        return None

    if isinstance(value, pd.Timestamp):
        return value.isoformat()

    if isinstance(value, Path):
        return str(value)

    return value


# --------------------------------------------------------------------------------------------------
# 3. Confirm the artifacts written before the failure exist
# --------------------------------------------------------------------------------------------------

required_existing_artifacts = [
    COMPARATOR_OUTPUT_PATH,
    COMPARATOR_QC_PATH,
    COMPARATOR_PARQUET_SIDECAR_PATH,
    COMPARATOR_QC_SIDECAR_PATH,
]

missing_existing_artifacts = [
    path
    for path in required_existing_artifacts
    if not path.exists()
]

if missing_existing_artifacts:
    raise FileNotFoundError(
        "The interrupted Cell 3D did not create all required prerequisite artifacts:\n"
        + "\n".join(
            f" - {path}"
            for path in missing_existing_artifacts
        )
    )


# --------------------------------------------------------------------------------------------------
# 4. Protect against overwriting a completed freeze package
# --------------------------------------------------------------------------------------------------

if COMPARATOR_FREEZE_MANIFEST_PATH.exists():
    raise FileExistsError(
        "The final freeze manifest already exists. "
        "Do not overwrite a previously completed freeze:\n"
        f"{COMPARATOR_FREEZE_MANIFEST_PATH}"
    )

if COMPARATOR_FREEZE_MANIFEST_SIDECAR_PATH.exists():
    raise FileExistsError(
        "The freeze-manifest SHA-256 sidecar already exists. "
        "Do not overwrite it:\n"
        f"{COMPARATOR_FREEZE_MANIFEST_SIDECAR_PATH}"
    )


# --------------------------------------------------------------------------------------------------
# 5. Reverify the already-written comparator Parquet
# --------------------------------------------------------------------------------------------------

comparator_parquet_sha256_reverified = calculate_sha256(
    COMPARATOR_OUTPUT_PATH
)

parquet_metadata = pq.ParquetFile(
    COMPARATOR_OUTPUT_PATH
)

parquet_rows_reverified = int(
    parquet_metadata.metadata.num_rows
)

parquet_columns_reverified = int(
    parquet_metadata.metadata.num_columns
)

parquet_schema_reverified = list(
    parquet_metadata.schema_arrow.names
)

comparator_readback_reverified = pd.read_parquet(
    COMPARATOR_OUTPUT_PATH,
    engine="pyarrow",
)

comparator_readback_reverified["rcv_accession"] = (
    comparator_readback_reverified["rcv_accession"]
    .astype("string")
)

comparator_in_memory_reverified = comparator_scores.copy()

comparator_in_memory_reverified["rcv_accession"] = (
    comparator_in_memory_reverified["rcv_accession"]
    .astype("string")
)


# --------------------------------------------------------------------------------------------------
# 6. Reverify the already-written QC JSON
# --------------------------------------------------------------------------------------------------

comparator_qc_sha256_reverified = calculate_sha256(
    COMPARATOR_QC_PATH
)

comparator_qc_readback = json.loads(
    COMPARATOR_QC_PATH.read_text(
        encoding="utf-8"
    )
)


# --------------------------------------------------------------------------------------------------
# 7. Reverify the two existing SHA-256 sidecars
# --------------------------------------------------------------------------------------------------

parquet_sidecar_parts = (
    COMPARATOR_PARQUET_SIDECAR_PATH
    .read_text(encoding="utf-8")
    .strip()
    .split()
)

qc_sidecar_parts = (
    COMPARATOR_QC_SIDECAR_PATH
    .read_text(encoding="utf-8")
    .strip()
    .split()
)


# --------------------------------------------------------------------------------------------------
# 8. Recovery prechecks
# --------------------------------------------------------------------------------------------------

recovery_prechecks = {
    "comparator Parquet exists":
        COMPARATOR_OUTPUT_PATH.exists(),

    "comparator QC JSON exists":
        COMPARATOR_QC_PATH.exists(),

    "Parquet SHA-256 sidecar exists":
        COMPARATOR_PARQUET_SIDECAR_PATH.exists(),

    "QC SHA-256 sidecar exists":
        COMPARATOR_QC_SIDECAR_PATH.exists(),

    "Parquet expected rows":
        parquet_rows_reverified
        == EXPECTED_ROWS,

    "Parquet expected columns":
        parquet_columns_reverified
        == len(expected_output_columns),

    "Parquet exact frozen schema":
        parquet_schema_reverified
        == expected_output_columns,

    "readback expected rows":
        len(comparator_readback_reverified)
        == EXPECTED_ROWS,

    "readback exact column order":
        list(comparator_readback_reverified.columns)
        == expected_output_columns,

    "readback complete RCV keys":
        comparator_readback_reverified[
            "rcv_accession"
        ].notna().all(),

    "readback unique RCV keys":
        comparator_readback_reverified[
            "rcv_accession"
        ].is_unique,

    "readback exact zero-based row order":
        np.array_equal(
            comparator_readback_reverified[
                "t0_row_order"
            ].to_numpy(),
            expected_row_order,
        ),

    "exact dataframe readback":
        comparator_readback_reverified.equals(
            comparator_in_memory_reverified
        ),

    "policy SHA-256 preserved on every row":
        comparator_readback_reverified[
            "comparator_policy_sha256"
        ]
        .eq(
            observed_policy_sha256
        )
        .all(),

    "outcome labels remain unloaded":
        comparator_readback_reverified[
            "outcome_labels_loaded"
        ]
        .eq(False)
        .all(),

    "temporal performance remains unevaluated":
        comparator_readback_reverified[
            "temporal_performance_evaluated"
        ]
        .eq(False)
        .all(),

    "written QC hash matches in-memory QC hash":
        comparator_qc_sha256_reverified
        == comparator_qc_sha256_in_memory,

    "QC decision is PASS":
        comparator_qc_readback.get("decision")
        == "PASS_STAGE6A_COMPARATOR_CONSTRUCTION_QC_CREATED_IN_MEMORY",

    "Parquet sidecar hash matches":
        len(parquet_sidecar_parts) >= 1
        and parquet_sidecar_parts[0]
        == comparator_parquet_sha256_reverified,

    "Parquet sidecar filename matches":
        len(parquet_sidecar_parts) >= 2
        and parquet_sidecar_parts[-1]
        == COMPARATOR_OUTPUT_PATH.name,

    "QC sidecar hash matches":
        len(qc_sidecar_parts) >= 1
        and qc_sidecar_parts[0]
        == comparator_qc_sha256_reverified,

    "QC sidecar filename matches":
        len(qc_sidecar_parts) >= 2
        and qc_sidecar_parts[-1]
        == COMPARATOR_QC_PATH.name,
}

failed_recovery_prechecks = [
    check_name
    for check_name, passed
    in recovery_prechecks.items()
    if not bool(passed)
]

if failed_recovery_prechecks:
    raise RuntimeError(
        "Recovery verification failed. The freeze manifest was not written.\n"
        + "\n".join(
            f" - {check_name}"
            for check_name in failed_recovery_prechecks
        )
    )


# --------------------------------------------------------------------------------------------------
# 9. Update the existing in-memory manifest with the independently reverified hashes
# --------------------------------------------------------------------------------------------------

comparator_freeze_manifest[
    "frozen_artifacts"
][
    "comparator_score_parquet"
][
    "sha256"
] = comparator_parquet_sha256_reverified

comparator_freeze_manifest[
    "frozen_artifacts"
][
    "construction_qc_json"
][
    "sha256"
] = comparator_qc_sha256_reverified

comparator_freeze_manifest[
    "verification"
][
    "recovery_after_json_scalar_conversion"
] = True

comparator_freeze_manifest[
    "verification"
][
    "recovery_prechecks"
] = {
    check_name: bool(passed)
    for check_name, passed
    in recovery_prechecks.items()
}


# --------------------------------------------------------------------------------------------------
# 10. Convert the entire manifest recursively to native JSON types
# --------------------------------------------------------------------------------------------------

comparator_freeze_manifest_json_safe = to_json_native(
    comparator_freeze_manifest
)

# Explicitly verify that the converted object is serializable before writing.
comparator_freeze_manifest_json_text = (
    json.dumps(
        comparator_freeze_manifest_json_safe,
        indent=2,
        sort_keys=True,
        ensure_ascii=False,
        allow_nan=False,
    )
    + "\n"
)


# --------------------------------------------------------------------------------------------------
# 11. Write the freeze manifest atomically
# --------------------------------------------------------------------------------------------------

atomic_write_text(
    COMPARATOR_FREEZE_MANIFEST_PATH,
    comparator_freeze_manifest_json_text,
)

if not COMPARATOR_FREEZE_MANIFEST_PATH.exists():
    raise RuntimeError(
        "The comparator freeze manifest was not created."
    )

comparator_freeze_manifest_sha256 = calculate_sha256(
    COMPARATOR_FREEZE_MANIFEST_PATH
)


# --------------------------------------------------------------------------------------------------
# 12. Write the freeze-manifest SHA-256 sidecar
# --------------------------------------------------------------------------------------------------

freeze_manifest_sidecar_text = (
    create_sha256_sidecar_text(
        comparator_freeze_manifest_sha256,
        COMPARATOR_FREEZE_MANIFEST_PATH,
    )
)

atomic_write_text(
    COMPARATOR_FREEZE_MANIFEST_SIDECAR_PATH,
    freeze_manifest_sidecar_text,
)


# --------------------------------------------------------------------------------------------------
# 13. Read back and verify the completed freeze package
# --------------------------------------------------------------------------------------------------

freeze_manifest_readback = json.loads(
    COMPARATOR_FREEZE_MANIFEST_PATH
    .read_text(encoding="utf-8")
)

freeze_manifest_sha256_reverified = calculate_sha256(
    COMPARATOR_FREEZE_MANIFEST_PATH
)

freeze_manifest_sidecar_parts = (
    COMPARATOR_FREEZE_MANIFEST_SIDECAR_PATH
    .read_text(encoding="utf-8")
    .strip()
    .split()
)

final_freeze_checks = {
    "freeze manifest exists":
        COMPARATOR_FREEZE_MANIFEST_PATH.exists(),

    "freeze-manifest sidecar exists":
        COMPARATOR_FREEZE_MANIFEST_SIDECAR_PATH.exists(),

    "freeze-manifest SHA-256 stable":
        freeze_manifest_sha256_reverified
        == comparator_freeze_manifest_sha256,

    "freeze-manifest sidecar hash":
        len(freeze_manifest_sidecar_parts) >= 1
        and freeze_manifest_sidecar_parts[0]
        == comparator_freeze_manifest_sha256,

    "freeze-manifest sidecar filename":
        len(freeze_manifest_sidecar_parts) >= 2
        and freeze_manifest_sidecar_parts[-1]
        == COMPARATOR_FREEZE_MANIFEST_PATH.name,

    "freeze-manifest status accepted":
        freeze_manifest_readback.get("status")
        == "ACCEPTED_AND_FROZEN",

    "freeze-manifest decision PASS":
        freeze_manifest_readback.get("decision")
        == "PASS_STAGE6A_COMPARATOR_SCORE_PACKAGE_ACCEPTED_AND_FROZEN",

    "comparator Parquet hash recorded":
        freeze_manifest_readback[
            "frozen_artifacts"
        ][
            "comparator_score_parquet"
        ][
            "sha256"
        ]
        == comparator_parquet_sha256_reverified,

    "comparator QC hash recorded":
        freeze_manifest_readback[
            "frozen_artifacts"
        ][
            "construction_qc_json"
        ][
            "sha256"
        ]
        == comparator_qc_sha256_reverified,

    "comparator policy hash recorded":
        freeze_manifest_readback[
            "comparator_policy"
        ][
            "sha256"
        ]
        == observed_policy_sha256,

    "Stage 6A Step 3 marked complete":
        freeze_manifest_readback[
            "authorization"
        ][
            "stage6a_step3_complete"
        ] is True,

    "comparator scores marked frozen":
        freeze_manifest_readback[
            "authorization"
        ][
            "comparator_scores_frozen"
        ] is True,

    "score-outcome join authorized next":
        freeze_manifest_readback[
            "authorization"
        ][
            "score_outcome_join_authorized_next"
        ] is True,

    "temporal performance still not immediately authorized":
        freeze_manifest_readback[
            "authorization"
        ][
            "temporal_performance_authorized_immediately"
        ] is False,

    "outcome table remained unopened":
        freeze_manifest_readback[
            "scientific_boundary"
        ][
            "stage5_outcome_table_opened"
        ] is False,

    "outcome labels remained unloaded":
        freeze_manifest_readback[
            "scientific_boundary"
        ][
            "outcome_labels_loaded"
        ] is False,

    "temporal performance remained uncalculated":
        freeze_manifest_readback[
            "scientific_boundary"
        ][
            "temporal_performance_calculated"
        ] is False,

    "JSON-scalar recovery recorded":
        freeze_manifest_readback[
            "verification"
        ][
            "recovery_after_json_scalar_conversion"
        ] is True,
}

failed_final_freeze_checks = [
    check_name
    for check_name, passed
    in final_freeze_checks.items()
    if not bool(passed)
]

if failed_final_freeze_checks:
    raise RuntimeError(
        "Final freeze-package validation failed:\n"
        + "\n".join(
            f" - {check_name}"
            for check_name in failed_final_freeze_checks
        )
    )


# --------------------------------------------------------------------------------------------------
# 14. Artifact summary
# --------------------------------------------------------------------------------------------------

artifact_summary_recovery = pd.DataFrame(
    [
        {
            "artifact": "Comparator score Parquet",
            "path": str(COMPARATOR_OUTPUT_PATH),
            "bytes": int(
                COMPARATOR_OUTPUT_PATH.stat().st_size
            ),
            "sha256":
                comparator_parquet_sha256_reverified,
            "status": "FROZEN",
        },
        {
            "artifact":
                "Comparator construction QC JSON",
            "path": str(COMPARATOR_QC_PATH),
            "bytes": int(
                COMPARATOR_QC_PATH.stat().st_size
            ),
            "sha256":
                comparator_qc_sha256_reverified,
            "status": "FROZEN",
        },
        {
            "artifact":
                "Comparator freeze manifest",
            "path": str(
                COMPARATOR_FREEZE_MANIFEST_PATH
            ),
            "bytes": int(
                COMPARATOR_FREEZE_MANIFEST_PATH
                .stat()
                .st_size
            ),
            "sha256":
                comparator_freeze_manifest_sha256,
            "status": "FROZEN",
        },
    ]
)


# --------------------------------------------------------------------------------------------------
# 15. Print results
# --------------------------------------------------------------------------------------------------

print("=" * 124)
print("STAGE 6A STEP 3 — CELL 3D-R — FREEZE-MANIFEST RECOVERY")
print("=" * 124)

print("\nRECOVERY PRECHECKS")
print("-" * 124)

for check_name, passed in recovery_prechecks.items():
    print(
        f"{check_name:<86}"
        f"{'PASS' if bool(passed) else 'FAIL'}"
    )

print("\nFINAL FREEZE-PACKAGE CHECKS")
print("-" * 124)

for check_name, passed in final_freeze_checks.items():
    print(
        f"{check_name:<86}"
        f"{'PASS' if bool(passed) else 'FAIL'}"
    )

print("\nFROZEN ARTIFACT SUMMARY")
print("-" * 124)

display(
    artifact_summary_recovery
)

print("\n" + "=" * 124)
print("STAGE 6A STEP 3 — CELL 3D-R DECISION")
print("=" * 124)

print(
    "PASS_STAGE6A_COMPARATOR_SCORE_PACKAGE_ACCEPTED_AND_FROZEN"
)

print()
print(f"Frozen comparator rows:                 {parquet_rows_reverified:,}")
print(f"Frozen comparator columns:              {parquet_columns_reverified:,}")
print(
    "Unique frozen RCV keys:                "
    f"{comparator_readback_reverified['rcv_accession'].nunique():,}"
)
print(
    "Frozen T0 row-order range:             "
    f"{comparator_readback_reverified['t0_row_order'].min():,} "
    f"to {comparator_readback_reverified['t0_row_order'].max():,}"
)

print()
print(
    f"Comparator Parquet SHA-256:             "
    f"{comparator_parquet_sha256_reverified}"
)
print(
    f"Comparator QC SHA-256:                  "
    f"{comparator_qc_sha256_reverified}"
)
print(
    f"Freeze manifest SHA-256:                "
    f"{comparator_freeze_manifest_sha256}"
)
print(
    f"Comparator policy SHA-256:              "
    f"{observed_policy_sha256}"
)

print()
print("Stage 5 outcome table opened:           NO")
print("Outcome labels loaded:                  NO")
print("Score-outcome join created:             NO")
print("Thresholds optimized:                   NO")
print("Weights optimized:                      NO")
print("Temporal performance examined:          NO")

print()
print("Comparator Parquet frozen:              YES")
print("Comparator QC JSON frozen:              YES")
print("Comparator sidecar verified:            YES")
print("QC sidecar verified:                    YES")
print("Freeze manifest written:                YES")
print("Freeze-manifest sidecar written:        YES")
print("Exact Parquet readback verified:        YES")
print("JSON scalar-conversion issue repaired:  YES")
print("Stage 6A Step 3 complete:               YES")

print()
print("NEXT AUTHORIZED ACTION:")
print(
    "Begin Stage 6A Step 4 with a fresh read-only verification "
    "of the complete frozen comparator package before opening "
    "the immutable Stage 5 outcome table."
)

STAGE 6A STEP 3 — CELL 3D-R — FREEZE-MANIFEST RECOVERY

RECOVERY PRECHECKS
----------------------------------------------------------------------------------------------------------------------------
comparator Parquet exists                                                             PASS
comparator QC JSON exists                                                             PASS
Parquet SHA-256 sidecar exists                                                        PASS
QC SHA-256 sidecar exists                                                             PASS
Parquet expected rows                                                                 PASS
Parquet expected columns                                                              PASS
Parquet exact frozen schema                                                           PASS
readback expected rows                                                                PASS
readback exact column order                                             

,artifact,path,bytes,sha256,status
0,Comparator score Parquet,/content/drive/MyDrive/GES_RAG_Temporal_Study/...,1616902,650b1f312efb424f959a3ebca8ddb71ea1f0a2247db9e5...,FROZEN
1,Comparator construction QC JSON,/content/drive/MyDrive/GES_RAG_Temporal_Study/...,16118,463b94902c55494dbfb965db695a38bc83921394cc6c05...,FROZEN
2,Comparator freeze manifest,/content/drive/MyDrive/GES_RAG_Temporal_Study/...,7502,082e86c194cfa94cfef774f9c96286fd275404a3a477d8...,FROZEN



STAGE 6A STEP 3 — CELL 3D-R DECISION
PASS_STAGE6A_COMPARATOR_SCORE_PACKAGE_ACCEPTED_AND_FROZEN

Frozen comparator rows:                 71,659
Frozen comparator columns:              26
Unique frozen RCV keys:                71,659
Frozen T0 row-order range:             0 to 71,658

Comparator Parquet SHA-256:             650b1f312efb424f959a3ebca8ddb71ea1f0a2247db9e52cff72502b15f916b3
Comparator QC SHA-256:                  463b94902c55494dbfb965db695a38bc83921394cc6c057b8ef06e1a91b149e7
Freeze manifest SHA-256:                082e86c194cfa94cfef774f9c96286fd275404a3a477d8cb73ffbaf958660e2c
Comparator policy SHA-256:              dd7e95dc785e77b04c289ef50817b1ddac7516a436d83b5734dbf3cd35b1248d

Stage 5 outcome table opened:           NO
Outcome labels loaded:                  NO
Score-outcome join created:             NO
Thresholds optimized:                   NO
Weights optimized:                      NO
Temporal performance examined:          NO

Comparator Parquet frozen:         

In [1]:
# ==================================================================================================
# STAGE 6A STEP 4 — CELL 4A
# FRESH READ-ONLY VERIFICATION OF THE FROZEN COMPARATOR PACKAGE
#
# IMPORTANT:
# - This cell does NOT open the Stage 5 outcome table.
# - This cell does NOT write, replace, or modify any scientific artifact.
# - This cell recalculates hashes and verifies the frozen comparator package.
# ==================================================================================================

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

from pathlib import Path
import hashlib
import json
import numpy as np
import pandas as pd
import pyarrow.parquet as pq


# --------------------------------------------------------------------------------------------------
# 1. Frozen study paths
# --------------------------------------------------------------------------------------------------

STUDY_ROOT = Path("/content/drive/MyDrive/GES_RAG_Temporal_Study")

POLICY_PATH = (
    STUDY_ROOT
    / "configs/stage6_temporal_validation"
    / "stage6a_comparator_score_policy_v1.json"
)

POLICY_SIDECAR_PATH = Path(str(POLICY_PATH) + ".sha256")

COMPARATOR_PATH = (
    STUDY_ROOT
    / "data_processed/stage6_temporal_validation"
    / "stage6a_t0_comparator_scores_v1.parquet"
)

COMPARATOR_SIDECAR_PATH = Path(str(COMPARATOR_PATH) + ".sha256")

QC_PATH = (
    STUDY_ROOT
    / "outputs/quality_checks/stage6_temporal_validation"
    / "stage6a_t0_comparator_score_construction_qc_v1.json"
)

QC_SIDECAR_PATH = Path(str(QC_PATH) + ".sha256")

FREEZE_MANIFEST_PATH = (
    STUDY_ROOT
    / "configs/stage6_temporal_validation"
    / "stage6a_t0_comparator_score_freeze_manifest_v1.json"
)

FREEZE_MANIFEST_SIDECAR_PATH = Path(str(FREEZE_MANIFEST_PATH) + ".sha256")


# --------------------------------------------------------------------------------------------------
# 2. Frozen expected identities
# --------------------------------------------------------------------------------------------------

EXPECTED_HASHES = {
    "policy": "dd7e95dc785e77b04c289ef50817b1ddac7516a436d83b5734dbf3cd35b1248d",
    "comparator_parquet": "650b1f312efb424f959a3ebca8ddb71ea1f0a2247db9e52cff72502b15f916b3",
    "construction_qc": "463b94902c55494dbfb965db695a38bc83921394cc6c057b8ef06e1a91b149e7",
    "freeze_manifest": "082e86c194cfa94cfef774f9c96286fd275404a3a477d8cb73ffbaf958660e2c",
}

EXPECTED_ROWS = 71_659
EXPECTED_COLUMNS = 26
EXPECTED_FIRST_ROW_ORDER = 0
EXPECTED_LAST_ROW_ORDER = 71_658


# --------------------------------------------------------------------------------------------------
# 3. Read-only utility functions
# --------------------------------------------------------------------------------------------------

def sha256_file(path: Path, chunk_size: int = 8 * 1024 * 1024) -> str:
    """Calculate SHA-256 without modifying the file."""
    digest = hashlib.sha256()

    with path.open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)

    return digest.hexdigest()


def parse_sha256_sidecar(path: Path) -> tuple[str, str | None]:
    """
    Read a standard SHA-256 sidecar.

    Accepted forms:
      <hash>
      <hash>  <filename>
      <hash> *<filename>
    """
    text = path.read_text(encoding="utf-8").strip()

    if not text:
        raise ValueError(f"Empty SHA-256 sidecar: {path}")

    parts = text.split(maxsplit=1)
    sidecar_hash = parts[0].strip().lower()
    sidecar_filename = None

    if len(parts) == 2:
        sidecar_filename = parts[1].strip().lstrip("*").strip()

    return sidecar_hash, sidecar_filename


def find_exact_schema_lists(obj, observed_columns, location="root"):
    """
    Recursively find policy lists that exactly match the observed
    Parquet column order.
    """
    matches = []

    if isinstance(obj, dict):
        for key, value in obj.items():
            matches.extend(
                find_exact_schema_lists(
                    value,
                    observed_columns,
                    f"{location}.{key}",
                )
            )

    elif isinstance(obj, list):
        if obj == observed_columns:
            matches.append(location)

        for index, value in enumerate(obj):
            if isinstance(value, (dict, list)):
                matches.extend(
                    find_exact_schema_lists(
                        value,
                        observed_columns,
                        f"{location}[{index}]",
                    )
                )

    return matches


# --------------------------------------------------------------------------------------------------
# 4. Confirm all frozen files exist
# --------------------------------------------------------------------------------------------------

required_paths = {
    "comparator policy": POLICY_PATH,
    "comparator-policy sidecar": POLICY_SIDECAR_PATH,
    "comparator Parquet": COMPARATOR_PATH,
    "comparator-Parquet sidecar": COMPARATOR_SIDECAR_PATH,
    "construction QC": QC_PATH,
    "construction-QC sidecar": QC_SIDECAR_PATH,
    "freeze manifest": FREEZE_MANIFEST_PATH,
    "freeze-manifest sidecar": FREEZE_MANIFEST_SIDECAR_PATH,
}

missing_paths = [
    f"{label}: {path}"
    for label, path in required_paths.items()
    if not path.is_file()
]

if missing_paths:
    raise FileNotFoundError(
        "One or more required frozen files are missing:\n"
        + "\n".join(missing_paths)
    )


# --------------------------------------------------------------------------------------------------
# 5. Recalculate cryptographic identities
# --------------------------------------------------------------------------------------------------

observed_hashes = {
    "policy": sha256_file(POLICY_PATH),
    "comparator_parquet": sha256_file(COMPARATOR_PATH),
    "construction_qc": sha256_file(QC_PATH),
    "freeze_manifest": sha256_file(FREEZE_MANIFEST_PATH),
}

for artifact_name, expected_hash in EXPECTED_HASHES.items():
    observed_hash = observed_hashes[artifact_name]

    assert observed_hash == expected_hash, (
        f"SHA-256 mismatch for {artifact_name}\n"
        f"Expected: {expected_hash}\n"
        f"Observed: {observed_hash}"
    )


# --------------------------------------------------------------------------------------------------
# 6. Verify all SHA-256 sidecars
# --------------------------------------------------------------------------------------------------

sidecar_checks = {
    "policy": (
        POLICY_SIDECAR_PATH,
        POLICY_PATH,
        observed_hashes["policy"],
    ),
    "comparator_parquet": (
        COMPARATOR_SIDECAR_PATH,
        COMPARATOR_PATH,
        observed_hashes["comparator_parquet"],
    ),
    "construction_qc": (
        QC_SIDECAR_PATH,
        QC_PATH,
        observed_hashes["construction_qc"],
    ),
    "freeze_manifest": (
        FREEZE_MANIFEST_SIDECAR_PATH,
        FREEZE_MANIFEST_PATH,
        observed_hashes["freeze_manifest"],
    ),
}

for artifact_name, (
    sidecar_path,
    artifact_path,
    observed_hash,
) in sidecar_checks.items():

    sidecar_hash, sidecar_filename = parse_sha256_sidecar(sidecar_path)

    assert sidecar_hash == observed_hash, (
        f"Sidecar hash mismatch for {artifact_name}\n"
        f"Sidecar:  {sidecar_hash}\n"
        f"Observed: {observed_hash}"
    )

    if sidecar_filename is not None:
        assert Path(sidecar_filename).name == artifact_path.name, (
            f"Sidecar filename mismatch for {artifact_name}\n"
            f"Expected filename: {artifact_path.name}\n"
            f"Sidecar filename: {sidecar_filename}"
        )


# --------------------------------------------------------------------------------------------------
# 7. Read frozen JSON artifacts
# --------------------------------------------------------------------------------------------------

with POLICY_PATH.open("r", encoding="utf-8") as handle:
    comparator_policy = json.load(handle)

with QC_PATH.open("r", encoding="utf-8") as handle:
    comparator_qc = json.load(handle)

with FREEZE_MANIFEST_PATH.open("r", encoding="utf-8") as handle:
    freeze_manifest = json.load(handle)


# --------------------------------------------------------------------------------------------------
# 8. Inspect Parquet metadata before dataframe loading
# --------------------------------------------------------------------------------------------------

parquet_file = pq.ParquetFile(COMPARATOR_PATH)
metadata = parquet_file.metadata

observed_rows = metadata.num_rows
observed_columns = metadata.num_columns
observed_schema = parquet_file.schema_arrow.names

assert observed_rows == EXPECTED_ROWS, (
    f"Unexpected comparator row count: {observed_rows}"
)

assert observed_columns == EXPECTED_COLUMNS, (
    f"Unexpected comparator column count: {observed_columns}"
)

assert len(observed_schema) == EXPECTED_COLUMNS
assert len(set(observed_schema)) == EXPECTED_COLUMNS


# Verify that the exact ordered schema is represented in the frozen policy.
schema_matches = find_exact_schema_lists(
    comparator_policy,
    observed_schema,
)

assert schema_matches, (
    "The exact ordered 26-column Parquet schema was not found "
    "inside the frozen comparator policy."
)


# --------------------------------------------------------------------------------------------------
# 9. Load only the already-frozen comparator package
#    Stage 5 outcomes are deliberately NOT opened.
# --------------------------------------------------------------------------------------------------

comparator_df = pd.read_parquet(COMPARATOR_PATH)

assert comparator_df.shape == (
    EXPECTED_ROWS,
    EXPECTED_COLUMNS,
)

assert comparator_df.columns.tolist() == observed_schema


# --------------------------------------------------------------------------------------------------
# 10. Identify and verify primary key, zero-based order, and policy lineage
# --------------------------------------------------------------------------------------------------

key_candidates = [
    column
    for column in comparator_df.columns
    if column.lower() in {
        "rcv_accession",
        "t0_rcv_accession",
        "rcv_key",
        "t0_rcv_key",
    }
]

assert len(key_candidates) == 1, (
    f"Expected exactly one RCV key column; found: {key_candidates}"
)

RCV_KEY_COLUMN = key_candidates[0]

assert "t0_row_order" in comparator_df.columns, (
    "Required t0_row_order column was not found."
)

policy_hash_candidates = [
    column
    for column in comparator_df.columns
    if "policy" in column.lower()
    and (
        "sha256" in column.lower()
        or "sha_256" in column.lower()
        or "hash" in column.lower()
    )
]

assert len(policy_hash_candidates) == 1, (
    "Expected exactly one comparator-policy hash column; "
    f"found: {policy_hash_candidates}"
)

POLICY_HASH_COLUMN = policy_hash_candidates[0]


# RCV key checks
rcv_keys = comparator_df[RCV_KEY_COLUMN].astype("string")

assert rcv_keys.notna().all()
assert rcv_keys.str.strip().ne("").all()
assert rcv_keys.nunique(dropna=False) == EXPECTED_ROWS


# Exact zero-based row-order checks
row_order = pd.to_numeric(
    comparator_df["t0_row_order"],
    errors="raise",
).to_numpy()

expected_row_order = np.arange(
    EXPECTED_ROWS,
    dtype=row_order.dtype,
)

assert np.array_equal(row_order, expected_row_order)
assert int(row_order[0]) == EXPECTED_FIRST_ROW_ORDER
assert int(row_order[-1]) == EXPECTED_LAST_ROW_ORDER


# Frozen policy-lineage checks
policy_hash_values = (
    comparator_df[POLICY_HASH_COLUMN]
    .astype("string")
    .str.strip()
    .str.lower()
)

assert policy_hash_values.notna().all()
assert policy_hash_values.nunique(dropna=False) == 1
assert policy_hash_values.iloc[0] == EXPECTED_HASHES["policy"]


# --------------------------------------------------------------------------------------------------
# 11. Final fresh-verification result
# --------------------------------------------------------------------------------------------------

print("=" * 120)
print("STAGE 6A STEP 4 — CELL 4A — FRESH FROZEN-PACKAGE VERIFICATION")
print("=" * 120)

print("\nCRYPTOGRAPHIC CHECKS")
print("-" * 120)
print("Comparator policy SHA-256".ljust(80), "PASS")
print("Comparator-policy sidecar".ljust(80), "PASS")
print("Comparator Parquet SHA-256".ljust(80), "PASS")
print("Comparator-Parquet sidecar".ljust(80), "PASS")
print("Construction-QC SHA-256".ljust(80), "PASS")
print("Construction-QC sidecar".ljust(80), "PASS")
print("Freeze-manifest SHA-256".ljust(80), "PASS")
print("Freeze-manifest sidecar".ljust(80), "PASS")

print("\nSTRUCTURAL AND LINEAGE CHECKS")
print("-" * 120)
print("Expected rows".ljust(80), f"PASS ({observed_rows:,})")
print("Expected columns".ljust(80), f"PASS ({observed_columns})")
print("Exact ordered schema found in frozen policy".ljust(80), "PASS")
print("Complete RCV keys".ljust(80), "PASS")
print("Unique RCV keys".ljust(80), f"PASS ({rcv_keys.nunique():,})")
print(
    "Exact zero-based row order",
    " " * 52,
    f"PASS ({row_order[0]:,} through {row_order[-1]:,})",
)
print("Frozen comparator-policy SHA-256 on every row".ljust(80), "PASS")
print("Comparator dataframe exact readback".ljust(80), "PASS")

print("\nSCIENTIFIC BOUNDARY")
print("-" * 120)
print("Stage 5 outcome table opened".ljust(80), "NO")
print("Outcome labels loaded".ljust(80), "NO")
print("Score-outcome join performed".ljust(80), "NO")
print("Temporal performance calculated".ljust(80), "NO")
print("Scientific artifact written or modified".ljust(80), "NO")

print("\nFINAL DECISION")
print("-" * 120)
print(
    "PASS_STAGE6A_FRESH_COMPARATOR_PACKAGE_VERIFICATION"
)
print(
    "The frozen comparator package is cryptographically, structurally, "
    "and procedurally ready for the locked score-outcome cohort step."
)
print("=" * 120)

Mounted at /content/drive
STAGE 6A STEP 4 — CELL 4A — FRESH FROZEN-PACKAGE VERIFICATION

CRYPTOGRAPHIC CHECKS
------------------------------------------------------------------------------------------------------------------------
Comparator policy SHA-256                                                        PASS
Comparator-policy sidecar                                                        PASS
Comparator Parquet SHA-256                                                       PASS
Comparator-Parquet sidecar                                                       PASS
Construction-QC SHA-256                                                          PASS
Construction-QC sidecar                                                          PASS
Freeze-manifest SHA-256                                                          PASS
Freeze-manifest sidecar                                                          PASS

STRUCTURAL AND LINEAGE CHECKS
--------------------------------------------------

In [2]:
# ==================================================================================================
# STAGE 6B STEP 1 — CELL 6B-1A
# FRESH READ-ONLY VERIFICATION AND SCHEMA INVENTORY OF THE FROZEN STAGE 5 OUTCOME PACKAGE
#
# IMPORTANT:
# - This cell opens the frozen Stage 5 outcomes for verification and schema inventory.
# - It does NOT open comparator scores.
# - It does NOT join outcomes with scores.
# - It does NOT calculate AUPRC, AUROC, calibration, enrichment, or any performance metric.
# - It does NOT write or modify any artifact.
# ==================================================================================================

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

from pathlib import Path
import hashlib
import json
import numpy as np
import pandas as pd
import pyarrow.parquet as pq


# --------------------------------------------------------------------------------------------------
# 1. Frozen Stage 5 paths
# --------------------------------------------------------------------------------------------------

STUDY_ROOT = Path("/content/drive/MyDrive/GES_RAG_Temporal_Study")

STAGE5_POLICY_PATH = (
    STUDY_ROOT
    / "configs/stage5_outcomes"
    / "stage5_primary_future_instability_outcome_policy_v1.json"
)

# The Stage 5 policy sidecar was saved without ".json" in its filename.
STAGE5_POLICY_SIDECAR_PATH = STAGE5_POLICY_PATH.with_suffix(".sha256")

STAGE5_OUTCOME_PATH = (
    STUDY_ROOT
    / "data_processed/stage5_outcomes"
    / "stage5_primary_future_instability_outcomes_v1.parquet"
)

STAGE5_OUTCOME_SIDECAR_PATH = Path(
    str(STAGE5_OUTCOME_PATH) + ".sha256"
)

STAGE5_QC_PATH = (
    STUDY_ROOT
    / "outputs/quality_checks/stage5_outcomes"
    / "stage5_primary_future_instability_outcome_construction_qc_v1.json"
)

STAGE5_QC_SIDECAR_PATH = Path(
    str(STAGE5_QC_PATH) + ".sha256"
)

STAGE5_FREEZE_MANIFEST_PATH = (
    STUDY_ROOT
    / "configs/stage5_outcomes"
    / "stage5_future_instability_outcome_freeze_manifest_v1.json"
)

STAGE5_FREEZE_MANIFEST_SIDECAR_PATH = Path(
    str(STAGE5_FREEZE_MANIFEST_PATH) + ".sha256"
)


# --------------------------------------------------------------------------------------------------
# 2. Frozen expected identities and accounting
# --------------------------------------------------------------------------------------------------

EXPECTED_HASHES = {
    "stage5_policy": (
        "477b01080b249dce9f042b67251ab93a999a1373e5f01d8f57b5812d67a57c4e"
    ),
    "stage5_outcome": (
        "c5508f5a8518160eef50482fd2c425dc4dcd9cf8a2fe04856e46760de60efbc8"
    ),
    "stage5_qc": (
        "93753c0b4eed103a56aa81606850a9359b5107ee7aabf49fbf81c29f7c63c16a"
    ),
    "stage5_freeze_manifest": (
        "b70286ebf8aa7751e391dbe7425ad56faebd18b67ed3f1a2aee63638b130ccd8"
    ),
}

EXPECTED_ROWS = 71_659
EXPECTED_COLUMNS = 53
EXPECTED_EVALUABLE = 66_636
EXPECTED_EVENTS = 6_485
EXPECTED_NEGATIVES = 60_151
EXPECTED_CENSORED_OR_NONEVALUABLE = 5_023


# --------------------------------------------------------------------------------------------------
# 3. Read-only utility functions
# --------------------------------------------------------------------------------------------------

def sha256_file(path: Path, chunk_size: int = 8 * 1024 * 1024) -> str:
    """Calculate the SHA-256 hash without modifying the file."""
    digest = hashlib.sha256()

    with path.open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


def parse_sha256_sidecar(path: Path) -> tuple[str, str | None]:
    """
    Read a SHA-256 sidecar in any of these forms:

        <hash>
        <hash>  <filename>
        <hash> *<filename>
    """
    text = path.read_text(encoding="utf-8").strip()

    if not text:
        raise ValueError(f"Empty SHA-256 sidecar: {path}")

    parts = text.split(maxsplit=1)

    observed_hash = parts[0].strip().lower()
    observed_filename = None

    if len(parts) == 2:
        observed_filename = parts[1].strip().lstrip("*").strip()

    return observed_hash, observed_filename


def display_value_inventory(
    dataframe: pd.DataFrame,
    column: str,
    maximum_categories: int = 25,
) -> None:
    """Print compact value counts for categorical or status-like columns."""
    series = dataframe[column]

    nonmissing_count = int(series.notna().sum())
    missing_count = int(series.isna().sum())
    unique_count = int(series.nunique(dropna=True))

    print(f"\nCOLUMN: {column}")
    print("-" * 120)
    print(f"Data type:             {series.dtype}")
    print(f"Nonmissing values:     {nonmissing_count:,}")
    print(f"Missing values:        {missing_count:,}")
    print(f"Unique nonmissing:     {unique_count:,}")

    if unique_count <= maximum_categories:
        print("\nValue counts, including missing values:")
        print(
            series.value_counts(
                dropna=False,
            ).to_string()
        )
    else:
        print(
            f"\nValue counts not printed because the column has "
            f"{unique_count:,} unique nonmissing values."
        )


# --------------------------------------------------------------------------------------------------
# 4. Confirm that every frozen Stage 5 file exists
# --------------------------------------------------------------------------------------------------

required_paths = {
    "Stage 5 policy": STAGE5_POLICY_PATH,
    "Stage 5 policy sidecar": STAGE5_POLICY_SIDECAR_PATH,
    "Stage 5 outcome Parquet": STAGE5_OUTCOME_PATH,
    "Stage 5 outcome sidecar": STAGE5_OUTCOME_SIDECAR_PATH,
    "Stage 5 construction QC": STAGE5_QC_PATH,
    "Stage 5 construction-QC sidecar": STAGE5_QC_SIDECAR_PATH,
    "Stage 5 freeze manifest": STAGE5_FREEZE_MANIFEST_PATH,
    "Stage 5 freeze-manifest sidecar": STAGE5_FREEZE_MANIFEST_SIDECAR_PATH,
}

missing_paths = [
    f"{label}: {path}"
    for label, path in required_paths.items()
    if not path.is_file()
]

if missing_paths:
    raise FileNotFoundError(
        "One or more required Stage 5 files are missing:\n"
        + "\n".join(missing_paths)
    )


# --------------------------------------------------------------------------------------------------
# 5. Recalculate cryptographic identities
# --------------------------------------------------------------------------------------------------

observed_hashes = {
    "stage5_policy": sha256_file(STAGE5_POLICY_PATH),
    "stage5_outcome": sha256_file(STAGE5_OUTCOME_PATH),
    "stage5_qc": sha256_file(STAGE5_QC_PATH),
    "stage5_freeze_manifest": sha256_file(
        STAGE5_FREEZE_MANIFEST_PATH
    ),
}

for artifact_name, expected_hash in EXPECTED_HASHES.items():
    observed_hash = observed_hashes[artifact_name]

    assert observed_hash == expected_hash, (
        f"SHA-256 mismatch for {artifact_name}\n"
        f"Expected: {expected_hash}\n"
        f"Observed: {observed_hash}"
    )


# --------------------------------------------------------------------------------------------------
# 6. Verify each SHA-256 sidecar
# --------------------------------------------------------------------------------------------------

sidecar_checks = {
    "stage5_policy": (
        STAGE5_POLICY_SIDECAR_PATH,
        STAGE5_POLICY_PATH,
        observed_hashes["stage5_policy"],
    ),
    "stage5_outcome": (
        STAGE5_OUTCOME_SIDECAR_PATH,
        STAGE5_OUTCOME_PATH,
        observed_hashes["stage5_outcome"],
    ),
    "stage5_qc": (
        STAGE5_QC_SIDECAR_PATH,
        STAGE5_QC_PATH,
        observed_hashes["stage5_qc"],
    ),
    "stage5_freeze_manifest": (
        STAGE5_FREEZE_MANIFEST_SIDECAR_PATH,
        STAGE5_FREEZE_MANIFEST_PATH,
        observed_hashes["stage5_freeze_manifest"],
    ),
}

for artifact_name, (
    sidecar_path,
    artifact_path,
    observed_hash,
) in sidecar_checks.items():

    sidecar_hash, sidecar_filename = parse_sha256_sidecar(
        sidecar_path
    )

    assert sidecar_hash == observed_hash, (
        f"Sidecar hash mismatch for {artifact_name}\n"
        f"Sidecar hash:  {sidecar_hash}\n"
        f"Observed hash: {observed_hash}"
    )

    if sidecar_filename is not None:
        assert Path(sidecar_filename).name == artifact_path.name, (
            f"Sidecar filename mismatch for {artifact_name}\n"
            f"Expected filename: {artifact_path.name}\n"
            f"Sidecar filename:  {sidecar_filename}"
        )


# --------------------------------------------------------------------------------------------------
# 7. Confirm that the JSON artifacts remain readable
# --------------------------------------------------------------------------------------------------

with STAGE5_POLICY_PATH.open("r", encoding="utf-8") as handle:
    stage5_policy = json.load(handle)

with STAGE5_QC_PATH.open("r", encoding="utf-8") as handle:
    stage5_qc = json.load(handle)

with STAGE5_FREEZE_MANIFEST_PATH.open(
    "r",
    encoding="utf-8",
) as handle:
    stage5_freeze_manifest = json.load(handle)

assert isinstance(stage5_policy, dict)
assert isinstance(stage5_qc, dict)
assert isinstance(stage5_freeze_manifest, dict)


# --------------------------------------------------------------------------------------------------
# 8. Inspect the frozen Parquet metadata before loading its contents
# --------------------------------------------------------------------------------------------------

stage5_parquet_file = pq.ParquetFile(STAGE5_OUTCOME_PATH)
stage5_metadata = stage5_parquet_file.metadata
stage5_schema = stage5_parquet_file.schema_arrow.names

assert stage5_metadata.num_rows == EXPECTED_ROWS, (
    f"Unexpected Stage 5 row count: "
    f"{stage5_metadata.num_rows:,}"
)

assert stage5_metadata.num_columns == EXPECTED_COLUMNS, (
    f"Unexpected Stage 5 column count: "
    f"{stage5_metadata.num_columns:,}"
)

assert len(stage5_schema) == EXPECTED_COLUMNS
assert len(set(stage5_schema)) == EXPECTED_COLUMNS


# --------------------------------------------------------------------------------------------------
# 9. Load the immutable Stage 5 outcome table
#
# No comparator-score dataframe is loaded in this cell.
# --------------------------------------------------------------------------------------------------

stage5_outcome_df = pd.read_parquet(STAGE5_OUTCOME_PATH)

assert stage5_outcome_df.shape == (
    EXPECTED_ROWS,
    EXPECTED_COLUMNS,
)

assert stage5_outcome_df.columns.tolist() == stage5_schema


# --------------------------------------------------------------------------------------------------
# 10. Identify and validate the unique T0 RCV key
# --------------------------------------------------------------------------------------------------

key_candidates = [
    column
    for column in stage5_outcome_df.columns
    if column.lower() in {
        "rcv_accession",
        "t0_rcv_accession",
        "rcv_key",
        "t0_rcv_key",
    }
]

assert len(key_candidates) == 1, (
    "Expected exactly one primary T0 RCV key column, "
    f"but found: {key_candidates}"
)

STAGE5_RCV_KEY_COLUMN = key_candidates[0]

stage5_rcv_keys = (
    stage5_outcome_df[STAGE5_RCV_KEY_COLUMN]
    .astype("string")
    .str.strip()
)

assert stage5_rcv_keys.notna().all()
assert stage5_rcv_keys.ne("").all()
assert stage5_rcv_keys.nunique(dropna=False) == EXPECTED_ROWS


# --------------------------------------------------------------------------------------------------
# 11. Inspect possible row-order and policy-lineage columns
# --------------------------------------------------------------------------------------------------

row_order_candidates = [
    column
    for column in stage5_outcome_df.columns
    if column.lower() in {
        "t0_row_order",
        "row_order",
        "t0_index",
    }
]

policy_hash_candidates = [
    column
    for column in stage5_outcome_df.columns
    if "policy" in column.lower()
    and (
        "sha" in column.lower()
        or "hash" in column.lower()
    )
]

if len(row_order_candidates) == 1:
    STAGE5_ROW_ORDER_COLUMN = row_order_candidates[0]

    observed_row_order = pd.to_numeric(
        stage5_outcome_df[STAGE5_ROW_ORDER_COLUMN],
        errors="raise",
    ).to_numpy()

    expected_row_order = np.arange(
        EXPECTED_ROWS,
        dtype=observed_row_order.dtype,
    )

    assert np.array_equal(
        observed_row_order,
        expected_row_order,
    )

else:
    STAGE5_ROW_ORDER_COLUMN = None


assert policy_hash_candidates, (
    "No Stage 5 policy-hash lineage column was detected."
)

verified_policy_hash_columns = []

for column in policy_hash_candidates:
    normalized_values = (
        stage5_outcome_df[column]
        .dropna()
        .astype("string")
        .str.strip()
        .str.lower()
        .unique()
        .tolist()
    )

    if normalized_values == [EXPECTED_HASHES["stage5_policy"]]:
        verified_policy_hash_columns.append(column)

assert verified_policy_hash_columns, (
    "A policy-hash candidate was found, but none contained "
    "the exact frozen Stage 5 policy SHA-256 on every nonmissing row."
)


# --------------------------------------------------------------------------------------------------
# 12. Identify status, outcome, censoring, and reason fields for the locked join
#
# This is an inventory only. No outcome is compared with a score.
# --------------------------------------------------------------------------------------------------

inventory_keywords = (
    "outcome",
    "instability",
    "event",
    "evaluable",
    "evaluation",
    "status",
    "assignment",
    "censor",
    "nonevaluable",
    "reason",
    "eligib",
    "linkage",
    "policy",
)

inventory_columns = [
    column
    for column in stage5_outcome_df.columns
    if any(
        keyword in column.lower()
        for keyword in inventory_keywords
    )
]


# --------------------------------------------------------------------------------------------------
# 13. Print the fresh-verification and schema-inventory result
# --------------------------------------------------------------------------------------------------

print("=" * 120)
print(
    "STAGE 6B STEP 1 — CELL 6B-1A — "
    "FRESH STAGE 5 OUTCOME VERIFICATION AND SCHEMA INVENTORY"
)
print("=" * 120)

print("\nCRYPTOGRAPHIC CHECKS")
print("-" * 120)
print("Stage 5 outcome policy SHA-256".ljust(82), "PASS")
print("Stage 5 outcome-policy sidecar".ljust(82), "PASS")
print("Stage 5 outcome Parquet SHA-256".ljust(82), "PASS")
print("Stage 5 outcome-Parquet sidecar".ljust(82), "PASS")
print("Stage 5 construction-QC SHA-256".ljust(82), "PASS")
print("Stage 5 construction-QC sidecar".ljust(82), "PASS")
print("Stage 5 freeze-manifest SHA-256".ljust(82), "PASS")
print("Stage 5 freeze-manifest sidecar".ljust(82), "PASS")

print("\nSTRUCTURAL CHECKS")
print("-" * 120)
print(
    "Expected rows".ljust(82),
    f"PASS ({stage5_outcome_df.shape[0]:,})",
)
print(
    "Expected columns".ljust(82),
    f"PASS ({stage5_outcome_df.shape[1]})",
)
print(
    "Unique nonblank T0 RCV keys".ljust(82),
    f"PASS ({stage5_rcv_keys.nunique():,})",
)
print(
    "Detected T0 RCV key column".ljust(82),
    STAGE5_RCV_KEY_COLUMN,
)

if STAGE5_ROW_ORDER_COLUMN is not None:
    print(
        "Exact zero-based Stage 5 row order".ljust(82),
        f"PASS via {STAGE5_ROW_ORDER_COLUMN}",
    )
else:
    print(
        "Explicit Stage 5 row-order column".ljust(82),
        "NOT PRESENT — original Parquet order retained",
    )

print(
    "Verified Stage 5 policy-hash column(s)".ljust(82),
    ", ".join(verified_policy_hash_columns),
)

print("\nFROZEN EXPECTED ACCOUNTING")
print("-" * 120)
print("Total Stage 5 rows".ljust(82), f"{EXPECTED_ROWS:,}")
print("Expected primary-outcome-evaluable rows".ljust(82), f"{EXPECTED_EVALUABLE:,}")
print("Expected primary instability events".ljust(82), f"{EXPECTED_EVENTS:,}")
print("Expected primary negatives".ljust(82), f"{EXPECTED_NEGATIVES:,}")
print(
    "Expected censored or nonevaluable rows".ljust(82),
    f"{EXPECTED_CENSORED_OR_NONEVALUABLE:,}",
)

print("\nCOMPLETE 53-COLUMN STAGE 5 SCHEMA")
print("-" * 120)

for index, column in enumerate(stage5_outcome_df.columns, start=1):
    print(
        f"{index:02d}. "
        f"{column:<70} "
        f"dtype={stage5_outcome_df[column].dtype}"
    )

print("\nSTATUS, OUTCOME, CENSORING, REASON, AND POLICY INVENTORY")
print("=" * 120)

for column in inventory_columns:
    display_value_inventory(
        stage5_outcome_df,
        column,
        maximum_categories=25,
    )

print("\nSCIENTIFIC BOUNDARY")
print("-" * 120)
print("Frozen Stage 5 outcomes opened".ljust(82), "YES — read only")
print("Comparator-score table opened".ljust(82), "NO")
print("Score-outcome join performed".ljust(82), "NO")
print("Temporal performance calculated".ljust(82), "NO")
print("Threshold or model changed".ljust(82), "NO")
print("Stage 5 outcome/status/reason changed".ljust(82), "NO")
print("Scientific artifact written or modified".ljust(82), "NO")

print("\nFINAL DECISION")
print("-" * 120)
print(
    "PASS_STAGE6B_FROZEN_STAGE5_OUTCOME_PACKAGE_REVERIFIED"
)
print(
    "The immutable Stage 5 outcome package is ready for construction "
    "of the locked one-to-one score-outcome cohort."
)
print("=" * 120)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


FileNotFoundError: One or more required Stage 5 files are missing:
Stage 5 freeze-manifest sidecar: /content/drive/MyDrive/GES_RAG_Temporal_Study/configs/stage5_outcomes/stage5_future_instability_outcome_freeze_manifest_v1.json.sha256

In [3]:
# ==================================================================================================
# STAGE 6B DIAGNOSTIC — LOCATE THE ACTUAL STAGE 5 FREEZE-MANIFEST SIDECAR
#
# READ-ONLY:
# - Does not create, rename, copy, overwrite, or modify any file.
# - Does not open scores or calculate performance.
# ==================================================================================================

from pathlib import Path

STUDY_ROOT = Path("/content/drive/MyDrive/GES_RAG_Temporal_Study")

STAGE5_CONFIG_DIR = STUDY_ROOT / "configs/stage5_outcomes"
STAGE5_DATA_DIR = STUDY_ROOT / "data_processed/stage5_outcomes"
STAGE5_QC_DIR = STUDY_ROOT / "outputs/quality_checks/stage5_outcomes"

EXPECTED_MANIFEST_NAME = (
    "stage5_future_instability_outcome_freeze_manifest_v1.json"
)

EXPECTED_JSON_SIDECAR_NAME = (
    "stage5_future_instability_outcome_freeze_manifest_v1.json.sha256"
)

POSSIBLE_SHORT_SIDECAR_NAME = (
    "stage5_future_instability_outcome_freeze_manifest_v1.sha256"
)


def print_directory_inventory(label: str, directory: Path) -> None:
    print("\n" + "=" * 120)
    print(label)
    print("=" * 120)
    print(f"Directory: {directory}")
    print(f"Exists:    {directory.is_dir()}")

    if not directory.is_dir():
        return

    files = sorted(
        path
        for path in directory.iterdir()
        if path.is_file()
    )

    if not files:
        print("No files found.")
        return

    for path in files:
        print(
            f"{path.name:<90} "
            f"{path.stat().st_size:>15,} bytes"
        )


print_directory_inventory(
    "STAGE 5 CONFIG DIRECTORY",
    STAGE5_CONFIG_DIR,
)

print_directory_inventory(
    "STAGE 5 OUTCOME DATA DIRECTORY",
    STAGE5_DATA_DIR,
)

print_directory_inventory(
    "STAGE 5 QUALITY-CHECK DIRECTORY",
    STAGE5_QC_DIR,
)


print("\n" + "=" * 120)
print("EXPECTED FREEZE-MANIFEST FILE CHECKS")
print("=" * 120)

expected_manifest_path = (
    STAGE5_CONFIG_DIR / EXPECTED_MANIFEST_NAME
)

expected_json_sidecar_path = (
    STAGE5_CONFIG_DIR / EXPECTED_JSON_SIDECAR_NAME
)

possible_short_sidecar_path = (
    STAGE5_CONFIG_DIR / POSSIBLE_SHORT_SIDECAR_NAME
)

print(
    f"Manifest exists:                    "
    f"{expected_manifest_path.is_file()}"
)
print(
    f"Expected .json.sha256 exists:       "
    f"{expected_json_sidecar_path.is_file()}"
)
print(
    f"Possible short .sha256 exists:      "
    f"{possible_short_sidecar_path.is_file()}"
)


print("\n" + "=" * 120)
print("ALL POSSIBLE STAGE 5 MANIFEST / SIDECAR MATCHES")
print("=" * 120)

search_patterns = [
    "*stage5*freeze*",
    "*future_instability*outcome*manifest*",
    "*manifest*.sha256",
    "*.sha256",
]

candidate_paths = set()

for pattern in search_patterns:
    for path in STUDY_ROOT.rglob(pattern):
        if path.is_file():
            candidate_paths.add(path)

if not candidate_paths:
    print("No candidate files found.")
else:
    for path in sorted(candidate_paths):
        print(
            f"{path.relative_to(STUDY_ROOT)}"
            f"    [{path.stat().st_size:,} bytes]"
        )


print("\n" + "=" * 120)
print("DIAGNOSTIC DECISION")
print("=" * 120)

if expected_json_sidecar_path.is_file():
    print(
        "FOUND_EXPECTED_JSON_SHA256_SIDECAR"
    )

elif possible_short_sidecar_path.is_file():
    print(
        "FOUND_SHORT_SHA256_SIDECAR_FILENAME"
    )
    print(
        f"Actual sidecar path: {possible_short_sidecar_path}"
    )

elif candidate_paths:
    print(
        "FOUND_ALTERNATIVE_CANDIDATE_FILES — review the inventory above."
    )

else:
    print(
        "NO_STAGE5_FREEZE_MANIFEST_SIDECAR_FOUND"
    )

print("=" * 120)


STAGE 5 CONFIG DIRECTORY
Directory: /content/drive/MyDrive/GES_RAG_Temporal_Study/configs/stage5_outcomes
Exists:    True
stage5_future_instability_outcome_freeze_manifest_v1.json                                            4,255 bytes
stage5_future_instability_outcome_freeze_manifest_v1.sha256                                            124 bytes
stage5_primary_future_instability_outcome_policy_v1.json                                             8,742 bytes
stage5_primary_future_instability_outcome_policy_v1.sha256                                             123 bytes

STAGE 5 OUTCOME DATA DIRECTORY
Directory: /content/drive/MyDrive/GES_RAG_Temporal_Study/data_processed/stage5_outcomes
Exists:    True
stage5_primary_future_instability_outcomes_v1.parquet                                            2,602,592 bytes
stage5_primary_future_instability_outcomes_v1.parquet.sha256                                           120 bytes

STAGE 5 QUALITY-CHECK DIRECTORY
Directory: /content/drive/MyDr

In [4]:
# ==================================================================================================
# STAGE 6B STEP 1 — CORRECTED CELL 6B-1A
# FRESH READ-ONLY VERIFICATION AND INVENTORY OF THE FROZEN STAGE 5 OUTCOME PACKAGE
#
# READ-ONLY BOUNDARY:
# - Opens only the frozen Stage 5 outcome package.
# - Does not open comparator scores.
# - Does not join scores to outcomes.
# - Does not calculate temporal performance.
# - Does not create, overwrite, rename, or modify any artifact.
# ==================================================================================================

from pathlib import Path
import hashlib
import json
import numpy as np
import pandas as pd
import pyarrow.parquet as pq


# --------------------------------------------------------------------------------------------------
# 1. Frozen paths
# --------------------------------------------------------------------------------------------------

STUDY_ROOT = Path("/content/drive/MyDrive/GES_RAG_Temporal_Study")

STAGE5_POLICY_PATH = (
    STUDY_ROOT
    / "configs/stage5_outcomes"
    / "stage5_primary_future_instability_outcome_policy_v1.json"
)

# Actual existing filename confirmed by the diagnostic cell.
STAGE5_POLICY_SIDECAR_PATH = (
    STUDY_ROOT
    / "configs/stage5_outcomes"
    / "stage5_primary_future_instability_outcome_policy_v1.sha256"
)

STAGE5_OUTCOME_PATH = (
    STUDY_ROOT
    / "data_processed/stage5_outcomes"
    / "stage5_primary_future_instability_outcomes_v1.parquet"
)

STAGE5_OUTCOME_SIDECAR_PATH = Path(
    str(STAGE5_OUTCOME_PATH) + ".sha256"
)

STAGE5_QC_PATH = (
    STUDY_ROOT
    / "outputs/quality_checks/stage5_outcomes"
    / "stage5_primary_future_instability_outcome_construction_qc_v1.json"
)

STAGE5_QC_SIDECAR_PATH = Path(
    str(STAGE5_QC_PATH) + ".sha256"
)

STAGE5_FREEZE_MANIFEST_PATH = (
    STUDY_ROOT
    / "configs/stage5_outcomes"
    / "stage5_future_instability_outcome_freeze_manifest_v1.json"
)

# Actual existing filename confirmed by the diagnostic cell.
STAGE5_FREEZE_MANIFEST_SIDECAR_PATH = (
    STUDY_ROOT
    / "configs/stage5_outcomes"
    / "stage5_future_instability_outcome_freeze_manifest_v1.sha256"
)


# --------------------------------------------------------------------------------------------------
# 2. Frozen expected identities and accounting
# --------------------------------------------------------------------------------------------------

EXPECTED_HASHES = {
    "policy": (
        "477b01080b249dce9f042b67251ab93a999a1373e5f01d8f57b5812d67a57c4e"
    ),
    "outcome": (
        "c5508f5a8518160eef50482fd2c425dc4dcd9cf8a2fe04856e46760de60efbc8"
    ),
    "qc": (
        "93753c0b4eed103a56aa81606850a9359b5107ee7aabf49fbf81c29f7c63c16a"
    ),
    "freeze_manifest": (
        "b70286ebf8aa7751e391dbe7425ad56faebd18b67ed3f1a2aee63638b130ccd8"
    ),
}

EXPECTED_ROWS = 71_659
EXPECTED_COLUMNS = 53

EXPECTED_EVALUABLE = 66_636
EXPECTED_POSITIVE = 6_485
EXPECTED_NEGATIVE = 60_151
EXPECTED_CENSORED_OR_NONEVALUABLE = 5_023


# --------------------------------------------------------------------------------------------------
# 3. Read-only helper functions
# --------------------------------------------------------------------------------------------------

def sha256_file(path: Path, chunk_size: int = 8 * 1024 * 1024) -> str:
    """Calculate SHA-256 without changing the file."""
    digest = hashlib.sha256()

    with path.open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


def parse_sha256_sidecar(path: Path) -> tuple[str, str | None]:
    """
    Read a sidecar stored as:

        <hash>
        <hash>  <filename>
        <hash> *<filename>
    """
    text = path.read_text(encoding="utf-8").strip()

    if not text:
        raise ValueError(f"Empty SHA-256 sidecar: {path}")

    parts = text.split(maxsplit=1)

    sidecar_hash = parts[0].strip().lower()
    sidecar_filename = None

    if len(parts) == 2:
        sidecar_filename = parts[1].strip().lstrip("*").strip()

    return sidecar_hash, sidecar_filename


def print_inventory(
    dataframe: pd.DataFrame,
    column: str,
    maximum_categories: int = 30,
) -> None:
    """Print an auditable inventory for a selected Stage 5 column."""

    series = dataframe[column]

    print("\n" + "-" * 120)
    print(f"COLUMN: {column}")
    print("-" * 120)
    print(f"Data type:           {series.dtype}")
    print(f"Nonmissing values:   {int(series.notna().sum()):,}")
    print(f"Missing values:      {int(series.isna().sum()):,}")
    print(f"Unique nonmissing:   {int(series.nunique(dropna=True)):,}")

    unique_count = int(series.nunique(dropna=True))

    if unique_count <= maximum_categories:
        print("\nValue counts:")
        print(series.value_counts(dropna=False).to_string())
    else:
        print(
            f"\nValue counts not printed because this column has "
            f"{unique_count:,} unique nonmissing values."
        )


# --------------------------------------------------------------------------------------------------
# 4. Confirm that every required frozen artifact exists
# --------------------------------------------------------------------------------------------------

required_paths = {
    "Stage 5 policy": STAGE5_POLICY_PATH,
    "Stage 5 policy sidecar": STAGE5_POLICY_SIDECAR_PATH,
    "Stage 5 outcome Parquet": STAGE5_OUTCOME_PATH,
    "Stage 5 outcome sidecar": STAGE5_OUTCOME_SIDECAR_PATH,
    "Stage 5 construction QC": STAGE5_QC_PATH,
    "Stage 5 construction-QC sidecar": STAGE5_QC_SIDECAR_PATH,
    "Stage 5 freeze manifest": STAGE5_FREEZE_MANIFEST_PATH,
    "Stage 5 freeze-manifest sidecar": STAGE5_FREEZE_MANIFEST_SIDECAR_PATH,
}

missing_paths = [
    f"{label}: {path}"
    for label, path in required_paths.items()
    if not path.is_file()
]

assert not missing_paths, (
    "One or more required Stage 5 files are missing:\n"
    + "\n".join(missing_paths)
)


# --------------------------------------------------------------------------------------------------
# 5. Recalculate and verify every SHA-256 identity
# --------------------------------------------------------------------------------------------------

observed_hashes = {
    "policy": sha256_file(STAGE5_POLICY_PATH),
    "outcome": sha256_file(STAGE5_OUTCOME_PATH),
    "qc": sha256_file(STAGE5_QC_PATH),
    "freeze_manifest": sha256_file(STAGE5_FREEZE_MANIFEST_PATH),
}

for artifact_name, expected_hash in EXPECTED_HASHES.items():

    observed_hash = observed_hashes[artifact_name]

    assert observed_hash == expected_hash, (
        f"SHA-256 mismatch for {artifact_name}\n"
        f"Expected: {expected_hash}\n"
        f"Observed: {observed_hash}"
    )


# --------------------------------------------------------------------------------------------------
# 6. Verify each checksum sidecar
# --------------------------------------------------------------------------------------------------

sidecar_checks = {
    "policy": (
        STAGE5_POLICY_SIDECAR_PATH,
        STAGE5_POLICY_PATH,
        observed_hashes["policy"],
    ),
    "outcome": (
        STAGE5_OUTCOME_SIDECAR_PATH,
        STAGE5_OUTCOME_PATH,
        observed_hashes["outcome"],
    ),
    "qc": (
        STAGE5_QC_SIDECAR_PATH,
        STAGE5_QC_PATH,
        observed_hashes["qc"],
    ),
    "freeze_manifest": (
        STAGE5_FREEZE_MANIFEST_SIDECAR_PATH,
        STAGE5_FREEZE_MANIFEST_PATH,
        observed_hashes["freeze_manifest"],
    ),
}

for artifact_name, (
    sidecar_path,
    artifact_path,
    observed_hash,
) in sidecar_checks.items():

    sidecar_hash, sidecar_filename = parse_sha256_sidecar(
        sidecar_path
    )

    assert sidecar_hash == observed_hash, (
        f"Sidecar hash mismatch for {artifact_name}\n"
        f"Sidecar:  {sidecar_hash}\n"
        f"Observed: {observed_hash}"
    )

    if sidecar_filename is not None:

        assert Path(sidecar_filename).name == artifact_path.name, (
            f"Sidecar filename mismatch for {artifact_name}\n"
            f"Expected: {artifact_path.name}\n"
            f"Observed: {sidecar_filename}"
        )


# --------------------------------------------------------------------------------------------------
# 7. Verify that all JSON artifacts remain readable
# --------------------------------------------------------------------------------------------------

with STAGE5_POLICY_PATH.open("r", encoding="utf-8") as handle:
    stage5_policy = json.load(handle)

with STAGE5_QC_PATH.open("r", encoding="utf-8") as handle:
    stage5_qc = json.load(handle)

with STAGE5_FREEZE_MANIFEST_PATH.open(
    "r",
    encoding="utf-8",
) as handle:
    stage5_freeze_manifest = json.load(handle)

assert isinstance(stage5_policy, dict)
assert isinstance(stage5_qc, dict)
assert isinstance(stage5_freeze_manifest, dict)


# --------------------------------------------------------------------------------------------------
# 8. Inspect Parquet metadata before dataframe loading
# --------------------------------------------------------------------------------------------------

parquet_file = pq.ParquetFile(STAGE5_OUTCOME_PATH)
parquet_metadata = parquet_file.metadata
parquet_schema = parquet_file.schema_arrow.names

assert parquet_metadata.num_rows == EXPECTED_ROWS, (
    f"Unexpected row count: {parquet_metadata.num_rows:,}"
)

assert parquet_metadata.num_columns == EXPECTED_COLUMNS, (
    f"Unexpected column count: {parquet_metadata.num_columns}"
)

assert len(parquet_schema) == EXPECTED_COLUMNS
assert len(set(parquet_schema)) == EXPECTED_COLUMNS


# --------------------------------------------------------------------------------------------------
# 9. Load the frozen outcome table
#
# Comparator scores are deliberately not loaded.
# --------------------------------------------------------------------------------------------------

stage5_outcome_df = pd.read_parquet(STAGE5_OUTCOME_PATH)

assert stage5_outcome_df.shape == (
    EXPECTED_ROWS,
    EXPECTED_COLUMNS,
)

assert stage5_outcome_df.columns.tolist() == parquet_schema


# --------------------------------------------------------------------------------------------------
# 10. Identify and verify the unique T0 RCV key
# --------------------------------------------------------------------------------------------------

key_candidates = [
    column
    for column in stage5_outcome_df.columns
    if column.lower() in {
        "rcv_accession",
        "t0_rcv_accession",
        "rcv_key",
        "t0_rcv_key",
    }
]

assert len(key_candidates) == 1, (
    f"Expected exactly one T0 RCV key column; found: {key_candidates}"
)

RCV_KEY_COLUMN = key_candidates[0]

rcv_keys = (
    stage5_outcome_df[RCV_KEY_COLUMN]
    .astype("string")
    .str.strip()
)

assert rcv_keys.notna().all()
assert rcv_keys.ne("").all()
assert rcv_keys.nunique(dropna=False) == EXPECTED_ROWS


# --------------------------------------------------------------------------------------------------
# 11. Verify row order when an explicit row-order column exists
# --------------------------------------------------------------------------------------------------

row_order_candidates = [
    column
    for column in stage5_outcome_df.columns
    if column.lower() in {
        "t0_row_order",
        "row_order",
        "t0_index",
    }
]

if len(row_order_candidates) == 1:

    ROW_ORDER_COLUMN = row_order_candidates[0]

    observed_row_order = pd.to_numeric(
        stage5_outcome_df[ROW_ORDER_COLUMN],
        errors="raise",
    ).to_numpy()

    expected_row_order = np.arange(
        EXPECTED_ROWS,
        dtype=observed_row_order.dtype,
    )

    assert np.array_equal(
        observed_row_order,
        expected_row_order,
    )

else:
    ROW_ORDER_COLUMN = None


# --------------------------------------------------------------------------------------------------
# 12. Verify policy-hash lineage stored inside the outcome table
# --------------------------------------------------------------------------------------------------

policy_hash_candidates = [
    column
    for column in stage5_outcome_df.columns
    if "policy" in column.lower()
    and (
        "sha" in column.lower()
        or "hash" in column.lower()
    )
]

verified_policy_hash_columns = []

for column in policy_hash_candidates:

    normalized_values = (
        stage5_outcome_df[column]
        .dropna()
        .astype("string")
        .str.strip()
        .str.lower()
        .unique()
        .tolist()
    )

    if normalized_values == [EXPECTED_HASHES["policy"]]:
        verified_policy_hash_columns.append(column)

assert verified_policy_hash_columns, (
    "No Stage 5 dataframe column contained the exact frozen "
    "outcome-policy SHA-256."
)


# --------------------------------------------------------------------------------------------------
# 13. Identify likely outcome/evaluability/status fields
# --------------------------------------------------------------------------------------------------

inventory_keywords = (
    "primary",
    "outcome",
    "instability",
    "event",
    "evaluable",
    "evaluation",
    "assignment",
    "status",
    "censor",
    "nonevaluable",
    "reason",
    "eligible",
    "linkage",
    "policy",
)

inventory_columns = [
    column
    for column in stage5_outcome_df.columns
    if any(
        keyword in column.lower()
        for keyword in inventory_keywords
    )
]


# --------------------------------------------------------------------------------------------------
# 14. Detect columns matching the frozen accounting
# --------------------------------------------------------------------------------------------------

evaluable_column_matches = []
primary_outcome_column_matches = []

for column in stage5_outcome_df.columns:

    series = stage5_outcome_df[column]

    # Candidate evaluability indicator:
    # 66,636 evaluable and 5,023 censored/nonevaluable.
    if pd.api.types.is_bool_dtype(series) or pd.api.types.is_numeric_dtype(series):

        numeric = pd.to_numeric(series, errors="coerce")
        value_counts = numeric.value_counts(dropna=False)

        true_or_one = int(value_counts.get(1, 0))
        false_or_zero = int(value_counts.get(0, 0))
        missing = int(numeric.isna().sum())

        if (
            true_or_one == EXPECTED_EVALUABLE
            and false_or_zero == EXPECTED_CENSORED_OR_NONEVALUABLE
            and missing == 0
        ):
            evaluable_column_matches.append(column)

        # Candidate primary binary outcome:
        # 6,485 positive, 60,151 negative, 5,023 missing/nonevaluable.
        if (
            true_or_one == EXPECTED_POSITIVE
            and false_or_zero == EXPECTED_NEGATIVE
            and missing == EXPECTED_CENSORED_OR_NONEVALUABLE
        ):
            primary_outcome_column_matches.append(column)


# --------------------------------------------------------------------------------------------------
# 15. Print verification and inventory
# --------------------------------------------------------------------------------------------------

print("=" * 120)
print(
    "STAGE 6B STEP 1 — CORRECTED CELL 6B-1A — "
    "FROZEN STAGE 5 PACKAGE VERIFICATION"
)
print("=" * 120)

print("\nCRYPTOGRAPHIC CHECKS")
print("-" * 120)
print("Stage 5 outcome policy SHA-256".ljust(84), "PASS")
print("Stage 5 outcome-policy sidecar".ljust(84), "PASS")
print("Stage 5 outcome Parquet SHA-256".ljust(84), "PASS")
print("Stage 5 outcome-Parquet sidecar".ljust(84), "PASS")
print("Stage 5 construction-QC SHA-256".ljust(84), "PASS")
print("Stage 5 construction-QC sidecar".ljust(84), "PASS")
print("Stage 5 freeze-manifest SHA-256".ljust(84), "PASS")
print("Stage 5 freeze-manifest sidecar".ljust(84), "PASS")

print("\nSTRUCTURAL AND LINEAGE CHECKS")
print("-" * 120)
print(
    "Expected rows".ljust(84),
    f"PASS ({stage5_outcome_df.shape[0]:,})",
)
print(
    "Expected columns".ljust(84),
    f"PASS ({stage5_outcome_df.shape[1]})",
)
print(
    "Unique nonblank T0 RCV keys".ljust(84),
    f"PASS ({rcv_keys.nunique():,})",
)
print(
    "Detected RCV key column".ljust(84),
    RCV_KEY_COLUMN,
)

if ROW_ORDER_COLUMN is not None:
    print(
        "Exact zero-based row order".ljust(84),
        f"PASS via {ROW_ORDER_COLUMN}",
    )
else:
    print(
        "Explicit row-order column".ljust(84),
        "NOT PRESENT — frozen Parquet order retained",
    )

print(
    "Verified policy-hash column(s)".ljust(84),
    ", ".join(verified_policy_hash_columns),
)

print("\nFROZEN ACCOUNTING TARGETS")
print("-" * 120)
print("Total rows".ljust(84), f"{EXPECTED_ROWS:,}")
print("Primary-outcome-evaluable".ljust(84), f"{EXPECTED_EVALUABLE:,}")
print("Primary instability events".ljust(84), f"{EXPECTED_POSITIVE:,}")
print("Primary negatives".ljust(84), f"{EXPECTED_NEGATIVE:,}")
print(
    "Censored or nonevaluable".ljust(84),
    f"{EXPECTED_CENSORED_OR_NONEVALUABLE:,}",
)

print("\nAUTOMATIC ACCOUNTING-COLUMN DETECTION")
print("-" * 120)
print(
    "Evaluability indicator candidate(s)".ljust(84),
    evaluable_column_matches or "NONE AUTOMATICALLY DETECTED",
)
print(
    "Primary outcome candidate(s)".ljust(84),
    primary_outcome_column_matches or "NONE AUTOMATICALLY DETECTED",
)

print("\nCOMPLETE 53-COLUMN STAGE 5 SCHEMA")
print("-" * 120)

for index, column in enumerate(
    stage5_outcome_df.columns,
    start=1,
):
    print(
        f"{index:02d}. "
        f"{column:<72} "
        f"dtype={stage5_outcome_df[column].dtype}"
    )

print("\nOUTCOME, STATUS, CENSORING, REASON, LINKAGE, AND POLICY INVENTORY")
print("=" * 120)

for column in inventory_columns:
    print_inventory(
        stage5_outcome_df,
        column,
        maximum_categories=30,
    )

print("\nSCIENTIFIC BOUNDARY")
print("-" * 120)
print("Frozen Stage 5 outcomes opened".ljust(84), "YES — READ ONLY")
print("Comparator-score package opened".ljust(84), "NO")
print("Score-outcome join performed".ljust(84), "NO")
print("AUPRC or AUROC calculated".ljust(84), "NO")
print("Calibration or enrichment calculated".ljust(84), "NO")
print("Outcome/status/reason modified".ljust(84), "NO")
print("Scientific artifact written or modified".ljust(84), "NO")

print("\nFINAL DECISION")
print("-" * 120)
print("PASS_STAGE6B_FROZEN_STAGE5_OUTCOME_PACKAGE_REVERIFIED")
print(
    "The frozen Stage 5 outcome package is ready for the locked "
    "score-outcome cohort construction step."
)
print("=" * 120)

/tmp/ipykernel_527/1118025816.py:488: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  true_or_one = int(value_counts.get(1, 0))
/tmp/ipykernel_527/1118025816.py:489: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  false_or_zero = int(value_counts.get(0, 0))
/tmp/ipykernel_527/1118025816.py:488: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  true_or_one = int(value_counts.get(1, 0))
/tmp/ipykernel_527/1118025816.py:489: FutureWarning: Ser

STAGE 6B STEP 1 — CORRECTED CELL 6B-1A — FROZEN STAGE 5 PACKAGE VERIFICATION

CRYPTOGRAPHIC CHECKS
------------------------------------------------------------------------------------------------------------------------
Stage 5 outcome policy SHA-256                                                       PASS
Stage 5 outcome-policy sidecar                                                       PASS
Stage 5 outcome Parquet SHA-256                                                      PASS
Stage 5 outcome-Parquet sidecar                                                      PASS
Stage 5 construction-QC SHA-256                                                      PASS
Stage 5 construction-QC sidecar                                                      PASS
Stage 5 freeze-manifest SHA-256                                                      PASS
Stage 5 freeze-manifest sidecar                                                      PASS

STRUCTURAL AND LINEAGE CHECKS
-----------------------------

In [5]:
# ==================================================================================================
# STAGE 6B STEP 2 — CELL 6B-2A
# CONSTRUCT AND VALIDATE THE LOCKED SCORE–OUTCOME COHORT IN MEMORY
#
# LOCKED BOUNDARY:
# - Re-verifies the frozen comparator and Stage 5 outcome Parquets.
# - Confirms exact one-to-one RCV-key membership and row-order compatibility.
# - Joins the two immutable packages.
# - Retains all 71,659 rows for complete accounting.
# - Creates a 66,636-row primary-outcome-evaluable in-memory view.
# - Does NOT calculate AUPRC, AUROC, Brier score, calibration, enrichment, or thresholds.
# - Does NOT write, overwrite, rename, or modify any artifact.
# ==================================================================================================

from pathlib import Path
import hashlib
import numpy as np
import pandas as pd


# --------------------------------------------------------------------------------------------------
# 1. Frozen input paths
# --------------------------------------------------------------------------------------------------

STUDY_ROOT = Path("/content/drive/MyDrive/GES_RAG_Temporal_Study")

COMPARATOR_PATH = (
    STUDY_ROOT
    / "data_processed/stage6_temporal_validation"
    / "stage6a_t0_comparator_scores_v1.parquet"
)

OUTCOME_PATH = (
    STUDY_ROOT
    / "data_processed/stage5_outcomes"
    / "stage5_primary_future_instability_outcomes_v1.parquet"
)


# --------------------------------------------------------------------------------------------------
# 2. Frozen expected identities and accounting
# --------------------------------------------------------------------------------------------------

EXPECTED_COMPARATOR_SHA256 = (
    "650b1f312efb424f959a3ebca8ddb71ea1f0a2247db9e52cff72502b15f916b3"
)

EXPECTED_OUTCOME_SHA256 = (
    "c5508f5a8518160eef50482fd2c425dc4dcd9cf8a2fe04856e46760de60efbc8"
)

EXPECTED_COMPARATOR_POLICY_SHA256 = (
    "dd7e95dc785e77b04c289ef50817b1ddac7516a436d83b5734dbf3cd35b1248d"
)

EXPECTED_OUTCOME_POLICY_SHA256 = (
    "477b01080b249dce9f042b67251ab93a999a1373e5f01d8f57b5812d67a57c4e"
)

EXPECTED_ROWS = 71_659
EXPECTED_COMPARATOR_COLUMNS = 26
EXPECTED_OUTCOME_COLUMNS = 53

EXPECTED_EVALUABLE = 66_636
EXPECTED_POSITIVE = 6_485
EXPECTED_NEGATIVE = 60_151
EXPECTED_NONEVALUABLE = 5_023

EXPECTED_LINKED = 70_583
EXPECTED_LINKAGE_CENSORED = 1_076


# --------------------------------------------------------------------------------------------------
# 3. Read-only SHA-256 helper
# --------------------------------------------------------------------------------------------------

def sha256_file(
    path: Path,
    chunk_size: int = 8 * 1024 * 1024,
) -> str:
    """Calculate SHA-256 without modifying the file."""

    digest = hashlib.sha256()

    with path.open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


# --------------------------------------------------------------------------------------------------
# 4. Confirm both immutable input files exist
# --------------------------------------------------------------------------------------------------

assert COMPARATOR_PATH.is_file(), (
    f"Frozen comparator Parquet not found:\n{COMPARATOR_PATH}"
)

assert OUTCOME_PATH.is_file(), (
    f"Frozen Stage 5 outcome Parquet not found:\n{OUTCOME_PATH}"
)


# --------------------------------------------------------------------------------------------------
# 5. Freshly verify both frozen Parquet identities
# --------------------------------------------------------------------------------------------------

observed_comparator_sha256 = sha256_file(COMPARATOR_PATH)
observed_outcome_sha256 = sha256_file(OUTCOME_PATH)

assert observed_comparator_sha256 == EXPECTED_COMPARATOR_SHA256, (
    "Comparator Parquet SHA-256 mismatch.\n"
    f"Expected: {EXPECTED_COMPARATOR_SHA256}\n"
    f"Observed: {observed_comparator_sha256}"
)

assert observed_outcome_sha256 == EXPECTED_OUTCOME_SHA256, (
    "Stage 5 outcome Parquet SHA-256 mismatch.\n"
    f"Expected: {EXPECTED_OUTCOME_SHA256}\n"
    f"Observed: {observed_outcome_sha256}"
)


# --------------------------------------------------------------------------------------------------
# 6. Load both immutable frozen packages
# --------------------------------------------------------------------------------------------------

comparator_df = pd.read_parquet(COMPARATOR_PATH)
outcome_df = pd.read_parquet(OUTCOME_PATH)

assert comparator_df.shape == (
    EXPECTED_ROWS,
    EXPECTED_COMPARATOR_COLUMNS,
), (
    f"Unexpected comparator shape: {comparator_df.shape}"
)

assert outcome_df.shape == (
    EXPECTED_ROWS,
    EXPECTED_OUTCOME_COLUMNS,
), (
    f"Unexpected outcome shape: {outcome_df.shape}"
)


# --------------------------------------------------------------------------------------------------
# 7. Detect the unique RCV key in each frozen package
# --------------------------------------------------------------------------------------------------

VALID_KEY_NAMES = {
    "rcv_accession",
    "t0_rcv_accession",
    "rcv_key",
    "t0_rcv_key",
}

comparator_key_candidates = [
    column
    for column in comparator_df.columns
    if column.lower() in VALID_KEY_NAMES
]

outcome_key_candidates = [
    column
    for column in outcome_df.columns
    if column.lower() in VALID_KEY_NAMES
]

assert len(comparator_key_candidates) == 1, (
    "Expected exactly one comparator RCV key column; found: "
    f"{comparator_key_candidates}"
)

assert len(outcome_key_candidates) == 1, (
    "Expected exactly one outcome RCV key column; found: "
    f"{outcome_key_candidates}"
)

COMPARATOR_KEY = comparator_key_candidates[0]
OUTCOME_KEY = outcome_key_candidates[0]


# --------------------------------------------------------------------------------------------------
# 8. Normalize keys for validation only
#
# Original frozen columns remain unchanged.
# --------------------------------------------------------------------------------------------------

comparator_keys = (
    comparator_df[COMPARATOR_KEY]
    .astype("string")
    .str.strip()
)

outcome_keys = (
    outcome_df[OUTCOME_KEY]
    .astype("string")
    .str.strip()
)

assert comparator_keys.notna().all()
assert outcome_keys.notna().all()

assert comparator_keys.ne("").all()
assert outcome_keys.ne("").all()

assert comparator_keys.nunique(dropna=False) == EXPECTED_ROWS
assert outcome_keys.nunique(dropna=False) == EXPECTED_ROWS


# --------------------------------------------------------------------------------------------------
# 9. Verify exact membership and exact frozen row-order compatibility
# --------------------------------------------------------------------------------------------------

comparator_key_set = set(comparator_keys.tolist())
outcome_key_set = set(outcome_keys.tolist())

comparator_only_keys = comparator_key_set - outcome_key_set
outcome_only_keys = outcome_key_set - comparator_key_set

assert not comparator_only_keys, (
    f"{len(comparator_only_keys):,} comparator keys are absent "
    "from the outcome table."
)

assert not outcome_only_keys, (
    f"{len(outcome_only_keys):,} outcome keys are absent "
    "from the comparator table."
)

exact_key_order_match = comparator_keys.equals(outcome_keys)

assert exact_key_order_match, (
    "The comparator and outcome RCV keys have equal membership "
    "but not the same frozen row order."
)


# --------------------------------------------------------------------------------------------------
# 10. Verify the comparator zero-based row-order field
# --------------------------------------------------------------------------------------------------

assert "t0_row_order" in comparator_df.columns, (
    "The comparator package does not contain t0_row_order."
)

comparator_row_order = pd.to_numeric(
    comparator_df["t0_row_order"],
    errors="raise",
).to_numpy()

expected_row_order = np.arange(
    EXPECTED_ROWS,
    dtype=comparator_row_order.dtype,
)

assert np.array_equal(
    comparator_row_order,
    expected_row_order,
), (
    "Comparator t0_row_order is not the exact zero-based "
    "sequence 0 through 71,658."
)


# --------------------------------------------------------------------------------------------------
# 11. Verify comparator-policy lineage
# --------------------------------------------------------------------------------------------------

comparator_policy_hash_candidates = [
    column
    for column in comparator_df.columns
    if "policy" in column.lower()
    and (
        "sha" in column.lower()
        or "hash" in column.lower()
    )
]

verified_comparator_policy_columns = []

for column in comparator_policy_hash_candidates:

    values = (
        comparator_df[column]
        .dropna()
        .astype("string")
        .str.strip()
        .str.lower()
        .unique()
        .tolist()
    )

    if values == [EXPECTED_COMPARATOR_POLICY_SHA256]:
        verified_comparator_policy_columns.append(column)

assert verified_comparator_policy_columns, (
    "No comparator dataframe column contained the exact "
    "frozen comparator-policy SHA-256."
)


# --------------------------------------------------------------------------------------------------
# 12. Verify Stage 5 policy lineage and accounting before joining
# --------------------------------------------------------------------------------------------------

required_outcome_columns = {
    "primary_outcome_evaluable",
    "primary_future_instability",
    "primary_outcome_status",
    "primary_outcome_reason_code",
    "outcome_policy_sha256",
    "stage3_temporal_outcome_eligible",
    "record_level_outcome_assignment_created",
}

missing_required_outcome_columns = (
    required_outcome_columns - set(outcome_df.columns)
)

assert not missing_required_outcome_columns, (
    "Required frozen Stage 5 columns are missing: "
    f"{sorted(missing_required_outcome_columns)}"
)


outcome_policy_values = (
    outcome_df["outcome_policy_sha256"]
    .astype("string")
    .str.strip()
    .str.lower()
    .unique()
    .tolist()
)

assert outcome_policy_values == [EXPECTED_OUTCOME_POLICY_SHA256]


assignment_created = (
    outcome_df["record_level_outcome_assignment_created"]
    .astype("boolean")
)

assert assignment_created.notna().all()
assert assignment_created.all()


primary_evaluable = (
    outcome_df["primary_outcome_evaluable"]
    .astype("boolean")
)

assert int(primary_evaluable.sum()) == EXPECTED_EVALUABLE
assert int((~primary_evaluable).sum()) == EXPECTED_NONEVALUABLE


primary_outcome_numeric = pd.to_numeric(
    outcome_df["primary_future_instability"],
    errors="coerce",
)

assert int((primary_outcome_numeric == 1).sum()) == EXPECTED_POSITIVE
assert int((primary_outcome_numeric == 0).sum()) == EXPECTED_NEGATIVE
assert int(primary_outcome_numeric.isna().sum()) == EXPECTED_NONEVALUABLE

assert primary_outcome_numeric[primary_evaluable].notna().all()
assert primary_outcome_numeric[~primary_evaluable].isna().all()


stage3_eligible = (
    outcome_df["stage3_temporal_outcome_eligible"]
    .astype("boolean")
)

assert int(stage3_eligible.sum()) == EXPECTED_LINKED
assert int((~stage3_eligible).sum()) == EXPECTED_LINKAGE_CENSORED


# --------------------------------------------------------------------------------------------------
# 13. Identify non-key column-name overlap
#
# Pandas will preserve both versions with explicit suffixes.
# --------------------------------------------------------------------------------------------------

overlapping_nonkey_columns = sorted(
    (
        set(comparator_df.columns)
        & set(outcome_df.columns)
    )
    - {COMPARATOR_KEY, OUTCOME_KEY}
)


# --------------------------------------------------------------------------------------------------
# 14. Prepare the outcome dataframe for the locked join
#
# Only an in-memory copy is renamed when key names differ.
# Frozen source files remain untouched.
# --------------------------------------------------------------------------------------------------

outcome_for_join = outcome_df.copy()

if OUTCOME_KEY != COMPARATOR_KEY:
    outcome_for_join = outcome_for_join.rename(
        columns={
            OUTCOME_KEY: COMPARATOR_KEY,
        }
    )


# --------------------------------------------------------------------------------------------------
# 15. Construct the locked full-accounting cohort
#
# validate="one_to_one" enforces one comparator row to one outcome row.
# sort=False preserves comparator package order.
# indicator=True proves that every row matched both sources.
# --------------------------------------------------------------------------------------------------

comparator_for_join = comparator_df.copy()

comparator_for_join[
    "_locked_prejoin_row_order"
] = np.arange(
    EXPECTED_ROWS,
    dtype=np.int64,
)

locked_score_outcome_df = comparator_for_join.merge(
    outcome_for_join,
    on=COMPARATOR_KEY,
    how="left",
    sort=False,
    validate="one_to_one",
    suffixes=("_comparator", "_outcome"),
    indicator=True,
)


# --------------------------------------------------------------------------------------------------
# 16. Validate the locked joined cohort
# --------------------------------------------------------------------------------------------------

expected_locked_columns_with_indicator = (
    EXPECTED_COMPARATOR_COLUMNS
    + EXPECTED_OUTCOME_COLUMNS
    - 1
    + 2
)

# +1 for _locked_prejoin_row_order
# +1 for _merge
assert locked_score_outcome_df.shape == (
    EXPECTED_ROWS,
    expected_locked_columns_with_indicator,
), (
    f"Unexpected joined shape: {locked_score_outcome_df.shape}"
)

merge_counts = (
    locked_score_outcome_df["_merge"]
    .value_counts(dropna=False)
    .to_dict()
)

assert merge_counts.get("both", 0) == EXPECTED_ROWS
assert merge_counts.get("left_only", 0) == 0
assert merge_counts.get("right_only", 0) == 0


assert np.array_equal(
    locked_score_outcome_df[
        "_locked_prejoin_row_order"
    ].to_numpy(),
    np.arange(EXPECTED_ROWS, dtype=np.int64),
), (
    "The locked join changed comparator row order."
)


joined_keys = (
    locked_score_outcome_df[COMPARATOR_KEY]
    .astype("string")
    .str.strip()
)

assert joined_keys.equals(comparator_keys)
assert joined_keys.nunique(dropna=False) == EXPECTED_ROWS


# Remove the temporary merge indicator only after validating every match.
locked_score_outcome_df = locked_score_outcome_df.drop(
    columns=["_merge"]
)

EXPECTED_LOCKED_COLUMNS = (
    EXPECTED_COMPARATOR_COLUMNS
    + EXPECTED_OUTCOME_COLUMNS
)

# 26 comparator + 53 outcome - 1 shared key + 1 temporary locked-order field = 79
assert locked_score_outcome_df.shape == (
    EXPECTED_ROWS,
    EXPECTED_LOCKED_COLUMNS,
)


# --------------------------------------------------------------------------------------------------
# 17. Create the primary-outcome-evaluable in-memory analysis view
#
# No model metric is calculated.
# --------------------------------------------------------------------------------------------------

locked_primary_evaluable_df = (
    locked_score_outcome_df.loc[
        locked_score_outcome_df[
            "primary_outcome_evaluable"
        ].astype("boolean")
    ]
    .copy()
)

assert len(locked_primary_evaluable_df) == EXPECTED_EVALUABLE

assert (
    locked_primary_evaluable_df[
        "primary_future_instability"
    ]
    .notna()
    .all()
)

assert int(
    (
        locked_primary_evaluable_df[
            "primary_future_instability"
        ] == 1
    ).sum()
) == EXPECTED_POSITIVE

assert int(
    (
        locked_primary_evaluable_df[
            "primary_future_instability"
        ] == 0
    ).sum()
) == EXPECTED_NEGATIVE


# --------------------------------------------------------------------------------------------------
# 18. Identify likely frozen score columns for the next step
#
# Inventory only; no comparison with outcomes is calculated.
# --------------------------------------------------------------------------------------------------

score_name_keywords = (
    "risk",
    "score",
    "prob",
    "ges",
    "star",
    "conflict",
    "recency",
    "submitter",
    "entropy",
    "additive",
    "combined",
)

likely_score_columns = [
    column
    for column in comparator_df.columns
    if any(
        keyword in column.lower()
        for keyword in score_name_keywords
    )
    and column
    not in {
        COMPARATOR_KEY,
        "t0_row_order",
    }
]


# --------------------------------------------------------------------------------------------------
# 19. Print the locked-cohort construction result
# --------------------------------------------------------------------------------------------------

print("=" * 120)
print(
    "STAGE 6B STEP 2 — CELL 6B-2A — "
    "LOCKED SCORE–OUTCOME COHORT CONSTRUCTION"
)
print("=" * 120)

print("\nFRESH INPUT VERIFICATION")
print("-" * 120)
print(
    "Comparator Parquet SHA-256".ljust(84),
    "PASS",
)
print(
    "Stage 5 outcome Parquet SHA-256".ljust(84),
    "PASS",
)
print(
    "Comparator rows / columns".ljust(84),
    f"PASS ({comparator_df.shape[0]:,} / {comparator_df.shape[1]})",
)
print(
    "Outcome rows / columns".ljust(84),
    f"PASS ({outcome_df.shape[0]:,} / {outcome_df.shape[1]})",
)

print("\nKEY AND ORDER COMPATIBILITY")
print("-" * 120)
print(
    "Comparator key column".ljust(84),
    COMPARATOR_KEY,
)
print(
    "Outcome key column".ljust(84),
    OUTCOME_KEY,
)
print(
    "Comparator unique keys".ljust(84),
    f"PASS ({comparator_keys.nunique():,})",
)
print(
    "Outcome unique keys".ljust(84),
    f"PASS ({outcome_keys.nunique():,})",
)
print(
    "Exact key-set compatibility".ljust(84),
    "PASS (71,659 of 71,659)",
)
print(
    "Exact frozen key-order compatibility".ljust(84),
    "PASS",
)
print(
    "Exact zero-based comparator order".ljust(84),
    "PASS (0 through 71,658)",
)

print("\nPOLICY LINEAGE")
print("-" * 120)
print(
    "Comparator policy SHA-256".ljust(84),
    "PASS",
)
print(
    "Comparator policy column(s)".ljust(84),
    ", ".join(verified_comparator_policy_columns),
)
print(
    "Stage 5 outcome policy SHA-256".ljust(84),
    "PASS via outcome_policy_sha256",
)

print("\nLOCKED JOIN VALIDATION")
print("-" * 120)
print(
    "Join cardinality".ljust(84),
    "PASS — one-to-one",
)
print(
    "Rows matched from both frozen packages".ljust(84),
    f"PASS ({merge_counts.get('both', 0):,})",
)
print(
    "Comparator-only rows".ljust(84),
    f"PASS ({merge_counts.get('left_only', 0):,})",
)
print(
    "Outcome-only rows".ljust(84),
    f"PASS ({merge_counts.get('right_only', 0):,})",
)
print(
    "Locked full-accounting cohort shape".ljust(84),
    f"PASS {locked_score_outcome_df.shape}",
)
print(
    "Locked row order preserved".ljust(84),
    "PASS",
)
print(
    "Non-key overlapping source columns".ljust(84),
    overlapping_nonkey_columns or "NONE",
)

print("\nLOCKED PRIMARY ANALYSIS ACCOUNTING")
print("-" * 120)
print(
    "All-accounting rows".ljust(84),
    f"{len(locked_score_outcome_df):,}",
)
print(
    "Primary-outcome-evaluable rows".ljust(84),
    f"PASS ({len(locked_primary_evaluable_df):,})",
)
print(
    "Primary instability events".ljust(84),
    f"PASS ({EXPECTED_POSITIVE:,})",
)
print(
    "Primary negatives".ljust(84),
    f"PASS ({EXPECTED_NEGATIVE:,})",
)
print(
    "Censored or nonevaluable rows retained".ljust(84),
    f"PASS ({EXPECTED_NONEVALUABLE:,})",
)
print(
    "Stage 3 accepted links".ljust(84),
    f"PASS ({EXPECTED_LINKED:,})",
)
print(
    "Stage 3 linkage-censored rows".ljust(84),
    f"PASS ({EXPECTED_LINKAGE_CENSORED:,})",
)

print("\nLIKELY FROZEN SCORE COLUMNS — INVENTORY ONLY")
print("-" * 120)

for index, column in enumerate(
    likely_score_columns,
    start=1,
):
    print(f"{index:02d}. {column}")

print("\nSCIENTIFIC BOUNDARY")
print("-" * 120)
print(
    "Frozen comparator package opened".ljust(84),
    "YES — READ ONLY",
)
print(
    "Frozen Stage 5 outcomes opened".ljust(84),
    "YES — READ ONLY",
)
print(
    "Locked score-outcome cohort constructed".ljust(84),
    "YES — IN MEMORY ONLY",
)
print(
    "Primary evaluable view created".ljust(84),
    "YES — IN MEMORY ONLY",
)
print(
    "AUPRC or AUROC calculated".ljust(84),
    "NO",
)
print(
    "Brier score or calibration calculated".ljust(84),
    "NO",
)
print(
    "Threshold or weight optimization performed".ljust(84),
    "NO",
)
print(
    "Artifact written or modified".ljust(84),
    "NO",
)

print("\nFINAL DECISION")
print("-" * 120)
print(
    "PASS_STAGE6B_LOCKED_SCORE_OUTCOME_COHORT_CONSTRUCTED_IN_MEMORY"
)
print(
    "The complete 71,659-row locked accounting cohort and "
    "66,636-row primary-evaluable view are ready for cohort QC "
    "and checksum-controlled freezing."
)
print("=" * 120)

STAGE 6B STEP 2 — CELL 6B-2A — LOCKED SCORE–OUTCOME COHORT CONSTRUCTION

FRESH INPUT VERIFICATION
------------------------------------------------------------------------------------------------------------------------
Comparator Parquet SHA-256                                                           PASS
Stage 5 outcome Parquet SHA-256                                                      PASS
Comparator rows / columns                                                            PASS (71,659 / 26)
Outcome rows / columns                                                               PASS (71,659 / 53)

KEY AND ORDER COMPATIBILITY
------------------------------------------------------------------------------------------------------------------------
Comparator key column                                                                rcv_accession
Outcome key column                                                                   t0_rcv_accession
Comparator unique keys                    

In [6]:
# ==================================================================================================
# STAGE 6B STEP 3 — CELL 6B-3A
# COMPREHENSIVE IN-MEMORY QC OF THE LOCKED SCORE–OUTCOME COHORT
#
# BOUNDARY:
# - Uses the locked_score_outcome_df created in Cell 6B-2A.
# - Verifies keys, ordering, outcome accounting, policy lineage, scores, formulas, and statuses.
# - Does NOT calculate AUPRC, AUROC, Brier score, calibration, enrichment, or thresholds.
# - Does NOT write or modify any artifact.
# ==================================================================================================

import numpy as np
import pandas as pd


# --------------------------------------------------------------------------------------------------
# 1. Confirm required in-memory objects exist
# --------------------------------------------------------------------------------------------------

required_runtime_objects = [
    "locked_score_outcome_df",
    "locked_primary_evaluable_df",
]

missing_runtime_objects = [
    object_name
    for object_name in required_runtime_objects
    if object_name not in globals()
]

assert not missing_runtime_objects, (
    "Required in-memory object(s) are missing: "
    f"{missing_runtime_objects}\n"
    "Rerun Cell 6B-2A before running this QC cell."
)


# --------------------------------------------------------------------------------------------------
# 2. Frozen expected accounting
# --------------------------------------------------------------------------------------------------

EXPECTED_ROWS = 71_659
EXPECTED_COLUMNS = 79
EXPECTED_EVALUABLE = 66_636
EXPECTED_POSITIVE = 6_485
EXPECTED_NEGATIVE = 60_151
EXPECTED_NONEVALUABLE = 5_023

EXPECTED_LINKED = 70_583
EXPECTED_LINKAGE_CENSORED = 1_076

EXPECTED_COMPARATOR_POLICY_SHA256 = (
    "dd7e95dc785e77b04c289ef50817b1ddac7516a436d83b5734dbf3cd35b1248d"
)

EXPECTED_OUTCOME_POLICY_SHA256 = (
    "477b01080b249dce9f042b67251ab93a999a1373e5f01d8f57b5812d67a57c4e"
)

EXPECTED_STATUS_COUNTS = {
    "NO_PRIMARY_FUTURE_INSTABILITY_EVENT": 60_151,
    "PRIMARY_FUTURE_INSTABILITY_EVENT": 6_485,
    "PRIMARY_OUTCOME_NONEVALUABLE_AXIS": 3_094,
    "LINKAGE_CENSORED": 1_076,
    "PRIMARY_OUTCOME_NONEVALUABLE_NONPRIMARY_CLASSIFICATION": 853,
}

EXPECTED_REASON_COUNTS = {
    "NO_PRIMARY_EVENT_AMONG_COMPARABLE_GROUPS": 60_151,
    "AT_LEAST_ONE_PRESPECIFIED_PRIMARY_EVENT": 6_485,
    "NONEVALUABLE_T1_NO_CLINICAL_CLASSIFICATION": 3_092,
    "NONEVALUABLE_NONPRIMARY_CLASSIFICATION_GROUP": 853,
    "CENSORED_NO_VARIATIONID_OR_VCV_CANDIDATE": 622,
    "CENSORED_VARIANT_CONTINUITY_CONDITION_ASSOCIATION_UNRESOLVED": 416,
    "CENSORED_COMPLEX_STRONG_CANDIDATE": 38,
    "NONEVALUABLE_AXIS_INCOMPATIBLE_ONCOGENICITY": 1,
    "NONEVALUABLE_AXIS_INCOMPATIBLE_SOMATIC_CLINICAL_IMPACT": 1,
}

EXPECTED_LINKAGE_STATUS_COUNTS = {
    "ACCEPTED_LINK": 70_583,
    "CENSORED_UNRESOLVED": 1_076,
}


# --------------------------------------------------------------------------------------------------
# 3. Expected frozen score fields
# --------------------------------------------------------------------------------------------------

CONTINUOUS_UNIT_INTERVAL_COLUMNS = [
    "review_stars_instability_risk",
    "conflict_instability_risk",
    "recency_instability_risk",
    "recency_missing_instability_component",
    "submitter_instability_risk",
    "entropy_instability_risk",
    "additive_instability_risk",
    "combined_metadata_instability_risk",
    "full_ges_p_stable_t0",
    "full_ges_instability_risk_t0",
    "no_star_ges_p_stable_t0",
    "no_star_ges_instability_risk_t0",
]

BINARY_INDICATOR_COLUMNS = [
    "additive_low_review_indicator",
    "additive_conflict_indicator",
    "additive_stale_recency_indicator",
    "additive_missing_recency_indicator",
    "additive_single_submitter_indicator",
    "additive_entropy_indicator",
]

ADDITIVE_COUNT_COLUMN = "additive_risk_count"

ALL_FROZEN_SCORE_COLUMNS = (
    CONTINUOUS_UNIT_INTERVAL_COLUMNS
    + BINARY_INDICATOR_COLUMNS
    + [ADDITIVE_COUNT_COLUMN]
)


# --------------------------------------------------------------------------------------------------
# 4. Initialize auditable QC record
# --------------------------------------------------------------------------------------------------

qc_checks = []


def record_check(
    check_name: str,
    passed: bool,
    observed,
    expected,
) -> None:
    """Record one QC result and fail immediately if it does not pass."""

    qc_checks.append(
        {
            "check_name": check_name,
            "passed": bool(passed),
            "observed": observed,
            "expected": expected,
        }
    )

    assert passed, (
        f"QC FAILURE: {check_name}\n"
        f"Observed: {observed}\n"
        f"Expected: {expected}"
    )


# --------------------------------------------------------------------------------------------------
# 5. Structural checks
# --------------------------------------------------------------------------------------------------

record_check(
    "locked_cohort_shape",
    locked_score_outcome_df.shape == (
        EXPECTED_ROWS,
        EXPECTED_COLUMNS,
    ),
    locked_score_outcome_df.shape,
    (EXPECTED_ROWS, EXPECTED_COLUMNS),
)

record_check(
    "primary_evaluable_view_row_count",
    len(locked_primary_evaluable_df) == EXPECTED_EVALUABLE,
    len(locked_primary_evaluable_df),
    EXPECTED_EVALUABLE,
)

required_columns = {
    "rcv_accession",
    "t0_row_order",
    "_locked_prejoin_row_order",
    "primary_outcome_evaluable",
    "primary_future_instability",
    "primary_outcome_status",
    "primary_outcome_reason_code",
    "linkage_status",
    "stage3_temporal_outcome_eligible",
    "comparator_policy_sha256",
    "outcome_policy_sha256",
    "record_level_outcome_assignment_created",
    *ALL_FROZEN_SCORE_COLUMNS,
}

missing_required_columns = sorted(
    required_columns - set(locked_score_outcome_df.columns)
)

record_check(
    "required_locked_columns_present",
    len(missing_required_columns) == 0,
    missing_required_columns,
    [],
)


# --------------------------------------------------------------------------------------------------
# 6. Primary key and row-order checks
# --------------------------------------------------------------------------------------------------

locked_keys = (
    locked_score_outcome_df["rcv_accession"]
    .astype("string")
    .str.strip()
)

record_check(
    "rcv_keys_complete",
    bool(locked_keys.notna().all() and locked_keys.ne("").all()),
    {
        "missing": int(locked_keys.isna().sum()),
        "blank": int(locked_keys.eq("").sum()),
    },
    {
        "missing": 0,
        "blank": 0,
    },
)

record_check(
    "rcv_keys_unique",
    locked_keys.nunique(dropna=False) == EXPECTED_ROWS,
    int(locked_keys.nunique(dropna=False)),
    EXPECTED_ROWS,
)

expected_zero_based_order = np.arange(
    EXPECTED_ROWS,
    dtype=np.int64,
)

t0_row_order = pd.to_numeric(
    locked_score_outcome_df["t0_row_order"],
    errors="raise",
).to_numpy(dtype=np.int64)

prejoin_row_order = pd.to_numeric(
    locked_score_outcome_df["_locked_prejoin_row_order"],
    errors="raise",
).to_numpy(dtype=np.int64)

record_check(
    "t0_row_order_exact_zero_based",
    np.array_equal(
        t0_row_order,
        expected_zero_based_order,
    ),
    {
        "first": int(t0_row_order[0]),
        "last": int(t0_row_order[-1]),
        "unique": int(np.unique(t0_row_order).size),
    },
    {
        "first": 0,
        "last": 71_658,
        "unique": 71_659,
    },
)

record_check(
    "locked_prejoin_order_preserved",
    np.array_equal(
        prejoin_row_order,
        expected_zero_based_order,
    ),
    {
        "first": int(prejoin_row_order[0]),
        "last": int(prejoin_row_order[-1]),
    },
    {
        "first": 0,
        "last": 71_658,
    },
)

record_check(
    "source_and_locked_orders_identical",
    np.array_equal(
        t0_row_order,
        prejoin_row_order,
    ),
    int(np.sum(t0_row_order != prejoin_row_order)),
    0,
)


# --------------------------------------------------------------------------------------------------
# 7. Policy-lineage checks
# --------------------------------------------------------------------------------------------------

comparator_policy_values = (
    locked_score_outcome_df["comparator_policy_sha256"]
    .astype("string")
    .str.strip()
    .str.lower()
)

outcome_policy_values = (
    locked_score_outcome_df["outcome_policy_sha256"]
    .astype("string")
    .str.strip()
    .str.lower()
)

record_check(
    "comparator_policy_lineage_constant",
    (
        comparator_policy_values.notna().all()
        and comparator_policy_values.nunique(dropna=False) == 1
        and comparator_policy_values.iloc[0]
        == EXPECTED_COMPARATOR_POLICY_SHA256
    ),
    comparator_policy_values.unique().tolist(),
    [EXPECTED_COMPARATOR_POLICY_SHA256],
)

record_check(
    "outcome_policy_lineage_constant",
    (
        outcome_policy_values.notna().all()
        and outcome_policy_values.nunique(dropna=False) == 1
        and outcome_policy_values.iloc[0]
        == EXPECTED_OUTCOME_POLICY_SHA256
    ),
    outcome_policy_values.unique().tolist(),
    [EXPECTED_OUTCOME_POLICY_SHA256],
)

assignment_created = (
    locked_score_outcome_df[
        "record_level_outcome_assignment_created"
    ]
    .astype("boolean")
)

record_check(
    "record_level_outcome_assignment_complete",
    bool(
        assignment_created.notna().all()
        and assignment_created.all()
    ),
    {
        "true": int(assignment_created.sum()),
        "missing": int(assignment_created.isna().sum()),
    },
    {
        "true": EXPECTED_ROWS,
        "missing": 0,
    },
)


# --------------------------------------------------------------------------------------------------
# 8. Frozen outcome accounting checks
# --------------------------------------------------------------------------------------------------

primary_evaluable = (
    locked_score_outcome_df["primary_outcome_evaluable"]
    .astype("boolean")
)

primary_outcome = pd.to_numeric(
    locked_score_outcome_df["primary_future_instability"],
    errors="coerce",
)

observed_evaluable = int(primary_evaluable.sum())
observed_nonevaluable = int((~primary_evaluable).sum())
observed_positive = int((primary_outcome == 1).sum())
observed_negative = int((primary_outcome == 0).sum())
observed_outcome_missing = int(primary_outcome.isna().sum())

record_check(
    "primary_evaluable_accounting",
    (
        observed_evaluable == EXPECTED_EVALUABLE
        and observed_nonevaluable == EXPECTED_NONEVALUABLE
    ),
    {
        "evaluable": observed_evaluable,
        "nonevaluable": observed_nonevaluable,
    },
    {
        "evaluable": EXPECTED_EVALUABLE,
        "nonevaluable": EXPECTED_NONEVALUABLE,
    },
)

record_check(
    "primary_binary_outcome_accounting",
    (
        observed_positive == EXPECTED_POSITIVE
        and observed_negative == EXPECTED_NEGATIVE
        and observed_outcome_missing == EXPECTED_NONEVALUABLE
    ),
    {
        "positive": observed_positive,
        "negative": observed_negative,
        "missing": observed_outcome_missing,
    },
    {
        "positive": EXPECTED_POSITIVE,
        "negative": EXPECTED_NEGATIVE,
        "missing": EXPECTED_NONEVALUABLE,
    },
)

record_check(
    "evaluable_rows_have_binary_outcome",
    bool(
        primary_outcome.loc[primary_evaluable].notna().all()
        and primary_outcome.loc[primary_evaluable]
        .isin([0, 1])
        .all()
    ),
    {
        "missing_among_evaluable": int(
            primary_outcome.loc[primary_evaluable]
            .isna()
            .sum()
        ),
        "nonbinary_among_evaluable": int(
            (
                ~primary_outcome.loc[primary_evaluable]
                .isin([0, 1])
            ).sum()
        ),
    },
    {
        "missing_among_evaluable": 0,
        "nonbinary_among_evaluable": 0,
    },
)

record_check(
    "nonevaluable_rows_have_missing_primary_outcome",
    bool(
        primary_outcome.loc[~primary_evaluable]
        .isna()
        .all()
    ),
    int(
        primary_outcome.loc[~primary_evaluable]
        .notna()
        .sum()
    ),
    0,
)


# --------------------------------------------------------------------------------------------------
# 9. Frozen status, reason, and linkage accounting
# --------------------------------------------------------------------------------------------------

observed_status_counts = (
    locked_score_outcome_df["primary_outcome_status"]
    .astype("string")
    .value_counts(dropna=False)
    .to_dict()
)

observed_reason_counts = (
    locked_score_outcome_df["primary_outcome_reason_code"]
    .astype("string")
    .value_counts(dropna=False)
    .to_dict()
)

observed_linkage_counts = (
    locked_score_outcome_df["linkage_status"]
    .astype("string")
    .value_counts(dropna=False)
    .to_dict()
)

record_check(
    "primary_outcome_status_counts",
    observed_status_counts == EXPECTED_STATUS_COUNTS,
    observed_status_counts,
    EXPECTED_STATUS_COUNTS,
)

record_check(
    "primary_outcome_reason_counts",
    observed_reason_counts == EXPECTED_REASON_COUNTS,
    observed_reason_counts,
    EXPECTED_REASON_COUNTS,
)

record_check(
    "linkage_status_counts",
    observed_linkage_counts == EXPECTED_LINKAGE_STATUS_COUNTS,
    observed_linkage_counts,
    EXPECTED_LINKAGE_STATUS_COUNTS,
)

stage3_eligible = (
    locked_score_outcome_df[
        "stage3_temporal_outcome_eligible"
    ]
    .astype("boolean")
)

record_check(
    "stage3_linkage_eligibility_accounting",
    (
        int(stage3_eligible.sum()) == EXPECTED_LINKED
        and int((~stage3_eligible).sum())
        == EXPECTED_LINKAGE_CENSORED
    ),
    {
        "eligible": int(stage3_eligible.sum()),
        "censored": int((~stage3_eligible).sum()),
    },
    {
        "eligible": EXPECTED_LINKED,
        "censored": EXPECTED_LINKAGE_CENSORED,
    },
)


# --------------------------------------------------------------------------------------------------
# 10. Frozen score completeness, finiteness, and range checks
# --------------------------------------------------------------------------------------------------

score_qc_rows = []

for column in CONTINUOUS_UNIT_INTERVAL_COLUMNS:

    numeric = pd.to_numeric(
        locked_score_outcome_df[column],
        errors="coerce",
    )

    missing_count = int(numeric.isna().sum())

    finite_count = int(
        np.isfinite(
            numeric.to_numpy(dtype=float)
        ).sum()
    )

    below_zero_count = int((numeric < 0).sum())
    above_one_count = int((numeric > 1).sum())

    passed = (
        missing_count == 0
        and finite_count == EXPECTED_ROWS
        and below_zero_count == 0
        and above_one_count == 0
    )

    score_qc_rows.append(
        {
            "column": column,
            "type": "continuous_[0,1]",
            "missing": missing_count,
            "nonfinite": EXPECTED_ROWS - finite_count,
            "below_zero": below_zero_count,
            "above_one": above_one_count,
            "minimum": float(numeric.min()),
            "maximum": float(numeric.max()),
            "passed": passed,
        }
    )

    record_check(
        f"continuous_score_valid::{column}",
        passed,
        score_qc_rows[-1],
        {
            "missing": 0,
            "nonfinite": 0,
            "below_zero": 0,
            "above_one": 0,
        },
    )


for column in BINARY_INDICATOR_COLUMNS:

    numeric = pd.to_numeric(
        locked_score_outcome_df[column],
        errors="coerce",
    )

    missing_count = int(numeric.isna().sum())
    invalid_count = int((~numeric.isin([0, 1])).sum())

    passed = (
        missing_count == 0
        and invalid_count == 0
    )

    score_qc_rows.append(
        {
            "column": column,
            "type": "binary_indicator",
            "missing": missing_count,
            "invalid": invalid_count,
            "minimum": float(numeric.min()),
            "maximum": float(numeric.max()),
            "passed": passed,
        }
    )

    record_check(
        f"binary_indicator_valid::{column}",
        passed,
        score_qc_rows[-1],
        {
            "missing": 0,
            "invalid": 0,
        },
    )


additive_count = pd.to_numeric(
    locked_score_outcome_df[ADDITIVE_COUNT_COLUMN],
    errors="coerce",
)

additive_count_invalid = int(
    (
        additive_count.isna()
        | (additive_count < 0)
        | (additive_count > 6)
        | (
            additive_count
            != np.floor(additive_count)
        )
    ).sum()
)

record_check(
    "additive_risk_count_valid",
    additive_count_invalid == 0,
    {
        "invalid": additive_count_invalid,
        "minimum": float(additive_count.min()),
        "maximum": float(additive_count.max()),
    },
    {
        "invalid": 0,
        "minimum_at_least": 0,
        "maximum_at_most": 6,
    },
)


# --------------------------------------------------------------------------------------------------
# 11. Formula-reconstruction checks
# --------------------------------------------------------------------------------------------------

indicator_sum = (
    locked_score_outcome_df[
        BINARY_INDICATOR_COLUMNS
    ]
    .apply(
        pd.to_numeric,
        errors="raise",
    )
    .sum(axis=1)
)

additive_count_mismatch = int(
    (
        additive_count.astype(float)
        != indicator_sum.astype(float)
    ).sum()
)

record_check(
    "additive_risk_count_equals_indicator_sum",
    additive_count_mismatch == 0,
    additive_count_mismatch,
    0,
)

expected_additive_risk = additive_count / 6.0

observed_additive_risk = pd.to_numeric(
    locked_score_outcome_df[
        "additive_instability_risk"
    ],
    errors="raise",
)

additive_risk_mismatch = int(
    (
        ~np.isclose(
            observed_additive_risk.to_numpy(dtype=float),
            expected_additive_risk.to_numpy(dtype=float),
            rtol=0.0,
            atol=1e-12,
        )
    ).sum()
)

record_check(
    "additive_instability_risk_formula",
    additive_risk_mismatch == 0,
    additive_risk_mismatch,
    0,
)


full_probability_sum = (
    pd.to_numeric(
        locked_score_outcome_df[
            "full_ges_p_stable_t0"
        ],
        errors="raise",
    )
    + pd.to_numeric(
        locked_score_outcome_df[
            "full_ges_instability_risk_t0"
        ],
        errors="raise",
    )
)

full_complement_mismatch = int(
    (
        ~np.isclose(
            full_probability_sum.to_numpy(dtype=float),
            np.ones(EXPECTED_ROWS),
            rtol=0.0,
            atol=1e-12,
        )
    ).sum()
)

record_check(
    "full_ges_stability_risk_complement",
    full_complement_mismatch == 0,
    full_complement_mismatch,
    0,
)


no_star_probability_sum = (
    pd.to_numeric(
        locked_score_outcome_df[
            "no_star_ges_p_stable_t0"
        ],
        errors="raise",
    )
    + pd.to_numeric(
        locked_score_outcome_df[
            "no_star_ges_instability_risk_t0"
        ],
        errors="raise",
    )
)

no_star_complement_mismatch = int(
    (
        ~np.isclose(
            no_star_probability_sum.to_numpy(dtype=float),
            np.ones(EXPECTED_ROWS),
            rtol=0.0,
            atol=1e-12,
        )
    ).sum()
)

record_check(
    "no_star_ges_stability_risk_complement",
    no_star_complement_mismatch == 0,
    no_star_complement_mismatch,
    0,
)


# --------------------------------------------------------------------------------------------------
# 12. Confirm the evaluable view is an exact ordered subset
# --------------------------------------------------------------------------------------------------

evaluable_keys_from_full = (
    locked_score_outcome_df.loc[
        primary_evaluable,
        "rcv_accession",
    ]
    .astype("string")
    .reset_index(drop=True)
)

evaluable_keys_from_view = (
    locked_primary_evaluable_df[
        "rcv_accession"
    ]
    .astype("string")
    .reset_index(drop=True)
)

record_check(
    "evaluable_view_exact_ordered_subset",
    evaluable_keys_from_full.equals(
        evaluable_keys_from_view
    ),
    int(
        (
            evaluable_keys_from_full
            != evaluable_keys_from_view
        ).sum()
    ),
    0,
)


# --------------------------------------------------------------------------------------------------
# 13. Confirm no temporary merge column remains
# --------------------------------------------------------------------------------------------------

record_check(
    "temporary_merge_indicator_removed",
    "_merge" not in locked_score_outcome_df.columns,
    "_merge" in locked_score_outcome_df.columns,
    False,
)


# --------------------------------------------------------------------------------------------------
# 14. Summarize all QC checks
# --------------------------------------------------------------------------------------------------

qc_df = pd.DataFrame(qc_checks)

total_checks = int(len(qc_df))
passed_checks = int(qc_df["passed"].sum())
failed_checks = total_checks - passed_checks

assert failed_checks == 0


# --------------------------------------------------------------------------------------------------
# 15. Print final result
# --------------------------------------------------------------------------------------------------

print("=" * 120)
print(
    "STAGE 6B STEP 3 — CELL 6B-3A — "
    "LOCKED SCORE–OUTCOME COHORT QC"
)
print("=" * 120)

print("\nSTRUCTURAL, KEY, AND ORDER CHECKS")
print("-" * 120)
print("Locked cohort shape".ljust(84), f"PASS {locked_score_outcome_df.shape}")
print("Unique RCV keys".ljust(84), f"PASS ({locked_keys.nunique():,})")
print("Exact zero-based source order".ljust(84), "PASS (0 through 71,658)")
print("Locked prejoin order preserved".ljust(84), "PASS")
print("Temporary merge indicator removed".ljust(84), "PASS")

print("\nPOLICY AND OUTCOME-LINEAGE CHECKS")
print("-" * 120)
print("Comparator policy lineage".ljust(84), "PASS")
print("Outcome policy lineage".ljust(84), "PASS")
print("Record-level outcome assignment completeness".ljust(84), "PASS (71,659)")
print("Outcome status accounting".ljust(84), "PASS")
print("Outcome reason accounting".ljust(84), "PASS")
print("Linkage accounting".ljust(84), "PASS")

print("\nPRIMARY OUTCOME ACCOUNTING")
print("-" * 120)
print("Full accounting cohort".ljust(84), f"{EXPECTED_ROWS:,}")
print("Primary-outcome-evaluable".ljust(84), f"PASS ({observed_evaluable:,})")
print("Primary instability events".ljust(84), f"PASS ({observed_positive:,})")
print("Primary negatives".ljust(84), f"PASS ({observed_negative:,})")
print("Censored or nonevaluable".ljust(84), f"PASS ({observed_nonevaluable:,})")
print("Outcomes missing only when nonevaluable".ljust(84), "PASS")

print("\nFROZEN SCORE AND FORMULA QC")
print("-" * 120)
print(
    "Continuous [0,1] score columns".ljust(84),
    f"PASS ({len(CONTINUOUS_UNIT_INTERVAL_COLUMNS)})",
)
print(
    "Binary indicator columns".ljust(84),
    f"PASS ({len(BINARY_INDICATOR_COLUMNS)})",
)
print("Additive risk count range and integrality".ljust(84), "PASS")
print("Additive count equals indicator sum".ljust(84), "PASS")
print("Additive risk equals count divided by six".ljust(84), "PASS")
print("Full GES P(stable) and risk complement".ljust(84), "PASS")
print("No-star GES P(stable) and risk complement".ljust(84), "PASS")
print("Missing or nonfinite frozen scores".ljust(84), "PASS (0)")

print("\nQC SUMMARY")
print("-" * 120)
print("Total QC checks".ljust(84), total_checks)
print("Passed QC checks".ljust(84), passed_checks)
print("Failed QC checks".ljust(84), failed_checks)

print("\nSCIENTIFIC BOUNDARY")
print("-" * 120)
print("Locked cohort inspected".ljust(84), "YES — IN MEMORY")
print("AUPRC or AUROC calculated".ljust(84), "NO")
print("Brier score or calibration calculated".ljust(84), "NO")
print("Risk enrichment calculated".ljust(84), "NO")
print("Threshold or weight optimization performed".ljust(84), "NO")
print("Artifact written or modified".ljust(84), "NO")

print("\nFINAL DECISION")
print("-" * 120)
print(
    "PASS_STAGE6B_LOCKED_COHORT_QC_COMPLETE"
)
print(
    "The 71,659-row locked accounting cohort and 66,636-row "
    "primary-evaluable view passed all structural, lineage, accounting, "
    "score-range, and formula checks and are ready for "
    "checksum-controlled artifact freezing."
)
print("=" * 120)

STAGE 6B STEP 3 — CELL 6B-3A — LOCKED SCORE–OUTCOME COHORT QC

STRUCTURAL, KEY, AND ORDER CHECKS
------------------------------------------------------------------------------------------------------------------------
Locked cohort shape                                                                  PASS (71659, 79)
Unique RCV keys                                                                      PASS (71,659)
Exact zero-based source order                                                        PASS (0 through 71,658)
Locked prejoin order preserved                                                       PASS
Temporary merge indicator removed                                                    PASS

POLICY AND OUTCOME-LINEAGE CHECKS
------------------------------------------------------------------------------------------------------------------------
Comparator policy lineage                                                            PASS
Outcome policy lineage                        

In [7]:
# ==================================================================================================
# STAGE 6B STEP 4 — CELL 6B-4A
# CHECKSUM-CONTROLLED FREEZE OF THE LOCKED SCORE–OUTCOME COHORT PACKAGE
#
# CREATES:
# 1. Complete 71,659-row locked accounting cohort
# 2. Primary-outcome-evaluable 66,636-row cohort
# 3. Locked-cohort QC JSON containing the 44 previously passed checks
# 4. SHA-256 sidecars
# 5. Stage 6B freeze manifest and sidecar
#
# SCIENTIFIC BOUNDARY:
# - Does NOT calculate AUPRC, AUROC, Brier score, calibration, enrichment, or thresholds.
# - Does NOT alter frozen Stage 5 outcomes or Stage 6A comparator scores.
# - Does NOT overwrite an existing Stage 6B package.
# ==================================================================================================

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import os
import platform
import shutil
import tempfile
import uuid

import numpy as np
import pandas as pd
import pyarrow
import sklearn


# --------------------------------------------------------------------------------------------------
# 1. Confirm required in-memory objects exist
# --------------------------------------------------------------------------------------------------

required_runtime_objects = [
    "locked_score_outcome_df",
    "locked_primary_evaluable_df",
    "qc_df",
]

missing_runtime_objects = [
    object_name
    for object_name in required_runtime_objects
    if object_name not in globals()
]

assert not missing_runtime_objects, (
    "Required in-memory object(s) are missing: "
    f"{missing_runtime_objects}\n"
    "Rerun Cells 6B-2A and 6B-3A before freezing the package."
)


# --------------------------------------------------------------------------------------------------
# 2. Frozen expected identities and accounting
# --------------------------------------------------------------------------------------------------

EXPECTED_FULL_ROWS = 71_659
EXPECTED_EVALUABLE_ROWS = 66_636
EXPECTED_COLUMNS = 79

EXPECTED_EVENTS = 6_485
EXPECTED_NEGATIVES = 60_151
EXPECTED_NONEVALUABLE = 5_023

EXPECTED_QC_CHECKS = 44
EXPECTED_QC_FAILURES = 0

SOURCE_COMPARATOR_SHA256 = (
    "650b1f312efb424f959a3ebca8ddb71ea1f0a2247db9e52cff72502b15f916b3"
)

SOURCE_OUTCOME_SHA256 = (
    "c5508f5a8518160eef50482fd2c425dc4dcd9cf8a2fe04856e46760de60efbc8"
)

COMPARATOR_POLICY_SHA256 = (
    "dd7e95dc785e77b04c289ef50817b1ddac7516a436d83b5734dbf3cd35b1248d"
)

OUTCOME_POLICY_SHA256 = (
    "477b01080b249dce9f042b67251ab93a999a1373e5f01d8f57b5812d67a57c4e"
)

STAGE6B_PACKAGE_VERSION = "1.0.0"


# --------------------------------------------------------------------------------------------------
# 3. Final output paths
# --------------------------------------------------------------------------------------------------

STUDY_ROOT = Path("/content/drive/MyDrive/GES_RAG_Temporal_Study")

STAGE6B_DATA_DIR = (
    STUDY_ROOT
    / "data_processed/stage6_temporal_validation"
)

STAGE6B_QC_DIR = (
    STUDY_ROOT
    / "outputs/quality_checks/stage6_temporal_validation"
)

STAGE6B_CONFIG_DIR = (
    STUDY_ROOT
    / "configs/stage6_temporal_validation"
)

for directory in [
    STAGE6B_DATA_DIR,
    STAGE6B_QC_DIR,
    STAGE6B_CONFIG_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)


FULL_COHORT_PATH = (
    STAGE6B_DATA_DIR
    / "stage6b_locked_score_outcome_accounting_cohort_v1.parquet"
)

EVALUABLE_COHORT_PATH = (
    STAGE6B_DATA_DIR
    / "stage6b_locked_primary_evaluable_cohort_v1.parquet"
)

QC_REPORT_PATH = (
    STAGE6B_QC_DIR
    / "stage6b_locked_score_outcome_cohort_qc_v1.json"
)

FREEZE_MANIFEST_PATH = (
    STAGE6B_CONFIG_DIR
    / "stage6b_locked_score_outcome_cohort_freeze_manifest_v1.json"
)

FULL_COHORT_SIDECAR_PATH = Path(
    str(FULL_COHORT_PATH) + ".sha256"
)

EVALUABLE_COHORT_SIDECAR_PATH = Path(
    str(EVALUABLE_COHORT_PATH) + ".sha256"
)

QC_REPORT_SIDECAR_PATH = Path(
    str(QC_REPORT_PATH) + ".sha256"
)

FREEZE_MANIFEST_SIDECAR_PATH = Path(
    str(FREEZE_MANIFEST_PATH) + ".sha256"
)


# --------------------------------------------------------------------------------------------------
# 4. Prevent accidental overwrite
# --------------------------------------------------------------------------------------------------

final_output_paths = [
    FULL_COHORT_PATH,
    FULL_COHORT_SIDECAR_PATH,
    EVALUABLE_COHORT_PATH,
    EVALUABLE_COHORT_SIDECAR_PATH,
    QC_REPORT_PATH,
    QC_REPORT_SIDECAR_PATH,
    FREEZE_MANIFEST_PATH,
    FREEZE_MANIFEST_SIDECAR_PATH,
]

existing_outputs = [
    str(path)
    for path in final_output_paths
    if path.exists()
]

assert not existing_outputs, (
    "Stage 6B freeze stopped because one or more final output files "
    "already exist. Nothing was overwritten:\n"
    + "\n".join(existing_outputs)
)


# --------------------------------------------------------------------------------------------------
# 5. Helper functions
# --------------------------------------------------------------------------------------------------

def sha256_file(
    path: Path,
    chunk_size: int = 8 * 1024 * 1024,
) -> str:
    """Calculate SHA-256 without changing the file."""

    digest = hashlib.sha256()

    with path.open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


def json_native(value):
    """
    Recursively convert NumPy, Pandas, Path, tuple, NA, and timestamp
    objects into deterministic JSON-compatible Python objects.
    """

    if value is None:
        return None

    if value is pd.NA:
        return None

    if isinstance(value, Path):
        return str(value)

    if isinstance(value, dict):
        return {
            str(key): json_native(item)
            for key, item in value.items()
        }

    if isinstance(value, (list, tuple, set)):
        return [
            json_native(item)
            for item in value
        ]

    if isinstance(value, np.ndarray):
        return [
            json_native(item)
            for item in value.tolist()
        ]

    if isinstance(value, np.bool_):
        return bool(value)

    if isinstance(value, np.integer):
        return int(value)

    if isinstance(value, np.floating):
        if np.isnan(value) or np.isinf(value):
            return None
        return float(value)

    if isinstance(value, float):
        if np.isnan(value) or np.isinf(value):
            return None
        return value

    if isinstance(value, pd.Timestamp):
        return value.isoformat()

    return value


def write_json_deterministic(
    data,
    path: Path,
) -> None:
    """Write deterministic UTF-8 JSON."""

    native_data = json_native(data)

    serialized = json.dumps(
        native_data,
        indent=2,
        sort_keys=True,
        ensure_ascii=False,
        allow_nan=False,
    ) + "\n"

    path.write_text(
        serialized,
        encoding="utf-8",
    )


def write_sha256_sidecar(
    artifact_path: Path,
    sidecar_path: Path,
    artifact_sha256: str,
) -> None:
    """Write a conventional SHA-256 sidecar."""

    sidecar_path.write_text(
        f"{artifact_sha256}  {artifact_path.name}\n",
        encoding="utf-8",
    )


def validate_frozen_dataframe(
    dataframe: pd.DataFrame,
    expected_rows: int,
    expected_columns: int,
    require_all_evaluable: bool,
) -> dict:
    """Validate structural and outcome accounting after Parquet readback."""

    assert dataframe.shape == (
        expected_rows,
        expected_columns,
    )

    assert dataframe.columns.tolist() == (
        locked_score_outcome_df.columns.tolist()
    )

    keys = (
        dataframe["rcv_accession"]
        .astype("string")
        .str.strip()
    )

    assert keys.notna().all()
    assert keys.ne("").all()
    assert keys.nunique(dropna=False) == expected_rows

    t0_order = pd.to_numeric(
        dataframe["t0_row_order"],
        errors="raise",
    ).to_numpy(dtype=np.int64)

    locked_order = pd.to_numeric(
        dataframe["_locked_prejoin_row_order"],
        errors="raise",
    ).to_numpy(dtype=np.int64)

    assert np.array_equal(
        t0_order,
        locked_order,
    )

    comparator_policy_values = (
        dataframe["comparator_policy_sha256"]
        .astype("string")
        .str.strip()
        .str.lower()
        .unique()
        .tolist()
    )

    outcome_policy_values = (
        dataframe["outcome_policy_sha256"]
        .astype("string")
        .str.strip()
        .str.lower()
        .unique()
        .tolist()
    )

    assert comparator_policy_values == [
        COMPARATOR_POLICY_SHA256
    ]

    assert outcome_policy_values == [
        OUTCOME_POLICY_SHA256
    ]

    evaluable = (
        dataframe["primary_outcome_evaluable"]
        .astype("boolean")
    )

    outcome = pd.to_numeric(
        dataframe["primary_future_instability"],
        errors="coerce",
    )

    if require_all_evaluable:
        assert evaluable.all()
        assert outcome.notna().all()
        assert int((outcome == 1).sum()) == EXPECTED_EVENTS
        assert int((outcome == 0).sum()) == EXPECTED_NEGATIVES

    else:
        assert int(evaluable.sum()) == EXPECTED_EVALUABLE_ROWS
        assert int((~evaluable).sum()) == EXPECTED_NONEVALUABLE
        assert int((outcome == 1).sum()) == EXPECTED_EVENTS
        assert int((outcome == 0).sum()) == EXPECTED_NEGATIVES
        assert int(outcome.isna().sum()) == EXPECTED_NONEVALUABLE
        assert outcome.loc[evaluable].notna().all()
        assert outcome.loc[~evaluable].isna().all()

    return {
        "rows": int(dataframe.shape[0]),
        "columns": int(dataframe.shape[1]),
        "unique_rcv_keys": int(keys.nunique(dropna=False)),
        "first_t0_row_order": int(t0_order[0]),
        "last_t0_row_order": int(t0_order[-1]),
        "primary_outcome_evaluable": int(evaluable.sum()),
        "primary_instability_events": int((outcome == 1).sum()),
        "primary_negatives": int((outcome == 0).sum()),
        "primary_outcome_missing": int(outcome.isna().sum()),
        "comparator_policy_sha256": comparator_policy_values[0],
        "outcome_policy_sha256": outcome_policy_values[0],
    }


# --------------------------------------------------------------------------------------------------
# 6. Reconfirm prior QC result before any file creation
# --------------------------------------------------------------------------------------------------

assert locked_score_outcome_df.shape == (
    EXPECTED_FULL_ROWS,
    EXPECTED_COLUMNS,
)

assert locked_primary_evaluable_df.shape == (
    EXPECTED_EVALUABLE_ROWS,
    EXPECTED_COLUMNS,
)

assert len(qc_df) == EXPECTED_QC_CHECKS
assert int(qc_df["passed"].sum()) == EXPECTED_QC_CHECKS
assert int((~qc_df["passed"].astype(bool)).sum()) == EXPECTED_QC_FAILURES


full_primary_evaluable = (
    locked_score_outcome_df[
        "primary_outcome_evaluable"
    ]
    .astype("boolean")
)

full_primary_outcome = pd.to_numeric(
    locked_score_outcome_df[
        "primary_future_instability"
    ],
    errors="coerce",
)

assert int(full_primary_evaluable.sum()) == EXPECTED_EVALUABLE_ROWS
assert int((~full_primary_evaluable).sum()) == EXPECTED_NONEVALUABLE

assert int((full_primary_outcome == 1).sum()) == EXPECTED_EVENTS
assert int((full_primary_outcome == 0).sum()) == EXPECTED_NEGATIVES
assert int(full_primary_outcome.isna().sum()) == EXPECTED_NONEVALUABLE


# Confirm the evaluable cohort is the exact ordered subset of the full cohort.
expected_evaluable_keys = (
    locked_score_outcome_df.loc[
        full_primary_evaluable,
        "rcv_accession",
    ]
    .astype("string")
    .reset_index(drop=True)
)

observed_evaluable_keys = (
    locked_primary_evaluable_df[
        "rcv_accession"
    ]
    .astype("string")
    .reset_index(drop=True)
)

assert expected_evaluable_keys.equals(
    observed_evaluable_keys
)


# --------------------------------------------------------------------------------------------------
# 7. Create isolated temporary working directory
# --------------------------------------------------------------------------------------------------

temporary_root = Path(
    tempfile.mkdtemp(
        prefix="stage6b_freeze_",
        dir="/content",
    )
)

temporary_full_path = (
    temporary_root / FULL_COHORT_PATH.name
)

temporary_evaluable_path = (
    temporary_root / EVALUABLE_COHORT_PATH.name
)

temporary_qc_path = (
    temporary_root / QC_REPORT_PATH.name
)

temporary_manifest_path = (
    temporary_root / FREEZE_MANIFEST_PATH.name
)


# --------------------------------------------------------------------------------------------------
# 8. Write both Parquet artifacts locally
# --------------------------------------------------------------------------------------------------

locked_score_outcome_df.to_parquet(
    temporary_full_path,
    engine="pyarrow",
    compression="zstd",
    index=False,
)

locked_primary_evaluable_df.to_parquet(
    temporary_evaluable_path,
    engine="pyarrow",
    compression="zstd",
    index=False,
)


# --------------------------------------------------------------------------------------------------
# 9. Read back and validate both local Parquet artifacts
# --------------------------------------------------------------------------------------------------

local_full_readback_df = pd.read_parquet(
    temporary_full_path
)

local_evaluable_readback_df = pd.read_parquet(
    temporary_evaluable_path
)

local_full_validation = validate_frozen_dataframe(
    dataframe=local_full_readback_df,
    expected_rows=EXPECTED_FULL_ROWS,
    expected_columns=EXPECTED_COLUMNS,
    require_all_evaluable=False,
)

local_evaluable_validation = validate_frozen_dataframe(
    dataframe=local_evaluable_readback_df,
    expected_rows=EXPECTED_EVALUABLE_ROWS,
    expected_columns=EXPECTED_COLUMNS,
    require_all_evaluable=True,
)


# Exact key and order readback comparisons.
assert (
    local_full_readback_df["rcv_accession"]
    .astype("string")
    .reset_index(drop=True)
    .equals(
        locked_score_outcome_df["rcv_accession"]
        .astype("string")
        .reset_index(drop=True)
    )
)

assert (
    local_evaluable_readback_df["rcv_accession"]
    .astype("string")
    .reset_index(drop=True)
    .equals(
        locked_primary_evaluable_df["rcv_accession"]
        .astype("string")
        .reset_index(drop=True)
    )
)


# --------------------------------------------------------------------------------------------------
# 10. Calculate local artifact checksums
# --------------------------------------------------------------------------------------------------

full_cohort_sha256 = sha256_file(
    temporary_full_path
)

evaluable_cohort_sha256 = sha256_file(
    temporary_evaluable_path
)


# --------------------------------------------------------------------------------------------------
# 11. Construct and write the Stage 6B QC JSON
# --------------------------------------------------------------------------------------------------

qc_records = [
    json_native(record)
    for record in qc_df.to_dict(
        orient="records"
    )
]

stage6b_qc_report = {
    "artifact_type": "stage6b_locked_score_outcome_cohort_qc",
    "package_version": STAGE6B_PACKAGE_VERSION,
    "created_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "decision": "PASS_STAGE6B_LOCKED_COHORT_QC_COMPLETE",
    "scientific_boundary": {
        "temporal_performance_calculated": False,
        "auprc_calculated": False,
        "auroc_calculated": False,
        "brier_score_calculated": False,
        "calibration_calculated": False,
        "risk_enrichment_calculated": False,
        "threshold_optimization_performed": False,
        "weight_optimization_performed": False,
        "frozen_outcome_modified": False,
        "frozen_comparator_modified": False,
    },
    "source_artifacts": {
        "stage6a_comparator_parquet_sha256": SOURCE_COMPARATOR_SHA256,
        "stage5_outcome_parquet_sha256": SOURCE_OUTCOME_SHA256,
        "comparator_policy_sha256": COMPARATOR_POLICY_SHA256,
        "outcome_policy_sha256": OUTCOME_POLICY_SHA256,
    },
    "full_accounting_cohort": {
        **local_full_validation,
        "artifact_filename": FULL_COHORT_PATH.name,
        "sha256": full_cohort_sha256,
    },
    "primary_evaluable_cohort": {
        **local_evaluable_validation,
        "artifact_filename": EVALUABLE_COHORT_PATH.name,
        "sha256": evaluable_cohort_sha256,
    },
    "qc_summary": {
        "total_checks": int(len(qc_df)),
        "passed_checks": int(
            qc_df["passed"].astype(bool).sum()
        ),
        "failed_checks": int(
            (~qc_df["passed"].astype(bool)).sum()
        ),
    },
    "qc_checks": qc_records,
    "schema": {
        "column_count": int(
            len(locked_score_outcome_df.columns)
        ),
        "columns": [
            {
                "position": index,
                "name": column,
                "dtype": str(
                    locked_score_outcome_df[column].dtype
                ),
            }
            for index, column in enumerate(
                locked_score_outcome_df.columns,
                start=1,
            )
        ],
    },
}

write_json_deterministic(
    stage6b_qc_report,
    temporary_qc_path,
)

# Confirm the QC JSON can be read back.
with temporary_qc_path.open(
    "r",
    encoding="utf-8",
) as handle:
    qc_readback = json.load(handle)

assert qc_readback["qc_summary"] == {
    "failed_checks": 0,
    "passed_checks": EXPECTED_QC_CHECKS,
    "total_checks": EXPECTED_QC_CHECKS,
}

qc_report_sha256 = sha256_file(
    temporary_qc_path
)


# --------------------------------------------------------------------------------------------------
# 12. Construct the freeze manifest
# --------------------------------------------------------------------------------------------------

freeze_timestamp_utc = datetime.now(
    timezone.utc
).isoformat()

stage6b_freeze_manifest = {
    "artifact_type": "stage6b_locked_score_outcome_cohort_freeze_manifest",
    "package_version": STAGE6B_PACKAGE_VERSION,
    "created_utc": freeze_timestamp_utc,
    "decision": (
        "PASS_STAGE6B_LOCKED_SCORE_OUTCOME_COHORT_PACKAGE_"
        "ACCEPTED_AND_FROZEN"
    ),
    "study_phase": (
        "Experiment 1 Stage 6B locked score-outcome cohort "
        "construction, QC, and freeze"
    ),
    "source_artifacts": {
        "stage6a_comparator_score_parquet": {
            "sha256": SOURCE_COMPARATOR_SHA256,
            "rows": 71_659,
            "columns": 26,
        },
        "stage5_primary_outcome_parquet": {
            "sha256": SOURCE_OUTCOME_SHA256,
            "rows": 71_659,
            "columns": 53,
        },
        "comparator_policy_sha256": COMPARATOR_POLICY_SHA256,
        "outcome_policy_sha256": OUTCOME_POLICY_SHA256,
    },
    "frozen_artifacts": {
        "full_accounting_cohort": {
            "path": str(FULL_COHORT_PATH),
            "filename": FULL_COHORT_PATH.name,
            "sha256": full_cohort_sha256,
            "rows": EXPECTED_FULL_ROWS,
            "columns": EXPECTED_COLUMNS,
            "unique_rcv_keys": EXPECTED_FULL_ROWS,
            "primary_outcome_evaluable": EXPECTED_EVALUABLE_ROWS,
            "primary_instability_events": EXPECTED_EVENTS,
            "primary_negatives": EXPECTED_NEGATIVES,
            "censored_or_nonevaluable": EXPECTED_NONEVALUABLE,
        },
        "primary_evaluable_cohort": {
            "path": str(EVALUABLE_COHORT_PATH),
            "filename": EVALUABLE_COHORT_PATH.name,
            "sha256": evaluable_cohort_sha256,
            "rows": EXPECTED_EVALUABLE_ROWS,
            "columns": EXPECTED_COLUMNS,
            "unique_rcv_keys": EXPECTED_EVALUABLE_ROWS,
            "primary_instability_events": EXPECTED_EVENTS,
            "primary_negatives": EXPECTED_NEGATIVES,
            "missing_primary_outcomes": 0,
        },
        "qc_report": {
            "path": str(QC_REPORT_PATH),
            "filename": QC_REPORT_PATH.name,
            "sha256": qc_report_sha256,
            "total_checks": EXPECTED_QC_CHECKS,
            "passed_checks": EXPECTED_QC_CHECKS,
            "failed_checks": EXPECTED_QC_FAILURES,
        },
    },
    "join_and_order_controls": {
        "join_key": "rcv_accession",
        "join_cardinality": "one_to_one",
        "comparator_key_column": "rcv_accession",
        "outcome_key_column_before_join": "t0_rcv_accession",
        "matched_rows": EXPECTED_FULL_ROWS,
        "comparator_only_rows": 0,
        "outcome_only_rows": 0,
        "t0_row_order_convention": "zero_based_0_through_71658",
        "locked_prejoin_order_preserved": True,
        "nonkey_source_column_overlap": [],
    },
    "scientific_boundary": {
        "outcomes_and_comparators_opened_only_from_frozen_sources": True,
        "temporal_performance_calculated": False,
        "auprc_calculated": False,
        "auroc_calculated": False,
        "brier_score_calculated": False,
        "calibration_calculated": False,
        "risk_enrichment_calculated": False,
        "subgroup_performance_calculated": False,
        "threshold_optimization_performed": False,
        "weight_optimization_performed": False,
        "frozen_outcome_modified": False,
        "frozen_comparator_modified": False,
    },
    "software_environment": {
        "python": platform.python_version(),
        "platform": platform.platform(),
        "pandas": pd.__version__,
        "numpy": np.__version__,
        "pyarrow": pyarrow.__version__,
        "scikit_learn": sklearn.__version__,
    },
    "authorization_after_freeze": {
        "authorized_next_step": (
            "Freshly reverify the complete Stage 6B frozen package "
            "before calculating any temporal performance."
        ),
        "performance_analysis_authorized_in_this_cell": False,
    },
}

write_json_deterministic(
    stage6b_freeze_manifest,
    temporary_manifest_path,
)

# Confirm the manifest can be read back.
with temporary_manifest_path.open(
    "r",
    encoding="utf-8",
) as handle:
    manifest_readback = json.load(handle)

assert manifest_readback["decision"] == (
    "PASS_STAGE6B_LOCKED_SCORE_OUTCOME_COHORT_PACKAGE_"
    "ACCEPTED_AND_FROZEN"
)

assert (
    manifest_readback["scientific_boundary"]
    ["temporal_performance_calculated"]
    is False
)

freeze_manifest_sha256 = sha256_file(
    temporary_manifest_path
)


# --------------------------------------------------------------------------------------------------
# 13. Create temporary destination files in the final directories
# --------------------------------------------------------------------------------------------------

transaction_id = uuid.uuid4().hex

destination_temp_map = {
    FULL_COHORT_PATH: Path(
        str(FULL_COHORT_PATH)
        + f".tmp_{transaction_id}"
    ),
    EVALUABLE_COHORT_PATH: Path(
        str(EVALUABLE_COHORT_PATH)
        + f".tmp_{transaction_id}"
    ),
    QC_REPORT_PATH: Path(
        str(QC_REPORT_PATH)
        + f".tmp_{transaction_id}"
    ),
    FREEZE_MANIFEST_PATH: Path(
        str(FREEZE_MANIFEST_PATH)
        + f".tmp_{transaction_id}"
    ),
}


# Copy verified local artifacts to temporary Drive destinations.
shutil.copy2(
    temporary_full_path,
    destination_temp_map[FULL_COHORT_PATH],
)

shutil.copy2(
    temporary_evaluable_path,
    destination_temp_map[EVALUABLE_COHORT_PATH],
)

shutil.copy2(
    temporary_qc_path,
    destination_temp_map[QC_REPORT_PATH],
)

shutil.copy2(
    temporary_manifest_path,
    destination_temp_map[FREEZE_MANIFEST_PATH],
)


# --------------------------------------------------------------------------------------------------
# 14. Verify temporary Drive copies before finalizing
# --------------------------------------------------------------------------------------------------

assert sha256_file(
    destination_temp_map[FULL_COHORT_PATH]
) == full_cohort_sha256

assert sha256_file(
    destination_temp_map[EVALUABLE_COHORT_PATH]
) == evaluable_cohort_sha256

assert sha256_file(
    destination_temp_map[QC_REPORT_PATH]
) == qc_report_sha256

assert sha256_file(
    destination_temp_map[FREEZE_MANIFEST_PATH]
) == freeze_manifest_sha256


# --------------------------------------------------------------------------------------------------
# 15. Finalize the four scientific/package artifacts
# --------------------------------------------------------------------------------------------------

for final_path, temporary_destination in destination_temp_map.items():

    assert not final_path.exists()

    os.replace(
        temporary_destination,
        final_path,
    )


# --------------------------------------------------------------------------------------------------
# 16. Write checksum sidecars
# --------------------------------------------------------------------------------------------------

write_sha256_sidecar(
    artifact_path=FULL_COHORT_PATH,
    sidecar_path=FULL_COHORT_SIDECAR_PATH,
    artifact_sha256=full_cohort_sha256,
)

write_sha256_sidecar(
    artifact_path=EVALUABLE_COHORT_PATH,
    sidecar_path=EVALUABLE_COHORT_SIDECAR_PATH,
    artifact_sha256=evaluable_cohort_sha256,
)

write_sha256_sidecar(
    artifact_path=QC_REPORT_PATH,
    sidecar_path=QC_REPORT_SIDECAR_PATH,
    artifact_sha256=qc_report_sha256,
)

write_sha256_sidecar(
    artifact_path=FREEZE_MANIFEST_PATH,
    sidecar_path=FREEZE_MANIFEST_SIDECAR_PATH,
    artifact_sha256=freeze_manifest_sha256,
)


# --------------------------------------------------------------------------------------------------
# 17. Final cryptographic and readback verification
# --------------------------------------------------------------------------------------------------

final_hashes = {
    "full_accounting_cohort": sha256_file(
        FULL_COHORT_PATH
    ),
    "primary_evaluable_cohort": sha256_file(
        EVALUABLE_COHORT_PATH
    ),
    "qc_report": sha256_file(
        QC_REPORT_PATH
    ),
    "freeze_manifest": sha256_file(
        FREEZE_MANIFEST_PATH
    ),
}

assert final_hashes == {
    "full_accounting_cohort": full_cohort_sha256,
    "primary_evaluable_cohort": evaluable_cohort_sha256,
    "qc_report": qc_report_sha256,
    "freeze_manifest": freeze_manifest_sha256,
}


# Verify sidecar contents.
sidecar_expectations = {
    FULL_COHORT_SIDECAR_PATH: (
        full_cohort_sha256,
        FULL_COHORT_PATH.name,
    ),
    EVALUABLE_COHORT_SIDECAR_PATH: (
        evaluable_cohort_sha256,
        EVALUABLE_COHORT_PATH.name,
    ),
    QC_REPORT_SIDECAR_PATH: (
        qc_report_sha256,
        QC_REPORT_PATH.name,
    ),
    FREEZE_MANIFEST_SIDECAR_PATH: (
        freeze_manifest_sha256,
        FREEZE_MANIFEST_PATH.name,
    ),
}

for sidecar_path, (
    expected_hash,
    expected_filename,
) in sidecar_expectations.items():

    sidecar_text = sidecar_path.read_text(
        encoding="utf-8"
    ).strip()

    sidecar_parts = sidecar_text.split(
        maxsplit=1
    )

    assert sidecar_parts[0] == expected_hash

    assert (
        Path(
            sidecar_parts[1]
            .lstrip("*")
            .strip()
        ).name
        == expected_filename
    )


# Final full readback from Drive.
final_full_readback_df = pd.read_parquet(
    FULL_COHORT_PATH
)

final_evaluable_readback_df = pd.read_parquet(
    EVALUABLE_COHORT_PATH
)

final_full_validation = validate_frozen_dataframe(
    dataframe=final_full_readback_df,
    expected_rows=EXPECTED_FULL_ROWS,
    expected_columns=EXPECTED_COLUMNS,
    require_all_evaluable=False,
)

final_evaluable_validation = validate_frozen_dataframe(
    dataframe=final_evaluable_readback_df,
    expected_rows=EXPECTED_EVALUABLE_ROWS,
    expected_columns=EXPECTED_COLUMNS,
    require_all_evaluable=True,
)


# Final JSON readback.
with QC_REPORT_PATH.open(
    "r",
    encoding="utf-8",
) as handle:
    final_qc_readback = json.load(handle)

with FREEZE_MANIFEST_PATH.open(
    "r",
    encoding="utf-8",
) as handle:
    final_manifest_readback = json.load(handle)

assert (
    final_qc_readback["qc_summary"]["failed_checks"]
    == 0
)

assert final_manifest_readback["decision"] == (
    "PASS_STAGE6B_LOCKED_SCORE_OUTCOME_COHORT_PACKAGE_"
    "ACCEPTED_AND_FROZEN"
)


# --------------------------------------------------------------------------------------------------
# 18. Remove temporary local working directory
# --------------------------------------------------------------------------------------------------

shutil.rmtree(
    temporary_root,
    ignore_errors=True,
)


# --------------------------------------------------------------------------------------------------
# 19. Print final freeze result
# --------------------------------------------------------------------------------------------------

print("=" * 120)
print(
    "STAGE 6B STEP 4 — CELL 6B-4A — "
    "LOCKED SCORE–OUTCOME COHORT PACKAGE FREEZE"
)
print("=" * 120)

print("\nFROZEN ARTIFACTS")
print("-" * 120)

print(
    "Full accounting cohort".ljust(84),
    f"PASS ({EXPECTED_FULL_ROWS:,} rows × {EXPECTED_COLUMNS} columns)",
)

print(
    "Primary-evaluable cohort".ljust(84),
    f"PASS ({EXPECTED_EVALUABLE_ROWS:,} rows × {EXPECTED_COLUMNS} columns)",
)

print(
    "Locked-cohort QC report".ljust(84),
    f"PASS ({EXPECTED_QC_CHECKS}/{EXPECTED_QC_CHECKS} checks)",
)

print(
    "Stage 6B freeze manifest".ljust(84),
    "PASS",
)

print("\nCRYPTOGRAPHIC IDENTITIES")
print("-" * 120)

print(
    "Full accounting cohort SHA-256".ljust(84),
    full_cohort_sha256,
)

print(
    "Primary-evaluable cohort SHA-256".ljust(84),
    evaluable_cohort_sha256,
)

print(
    "Locked-cohort QC SHA-256".ljust(84),
    qc_report_sha256,
)

print(
    "Stage 6B freeze-manifest SHA-256".ljust(84),
    freeze_manifest_sha256,
)

print("\nREADBACK AND ACCOUNTING")
print("-" * 120)

print(
    "Full cohort Drive readback".ljust(84),
    "PASS",
)

print(
    "Evaluable cohort Drive readback".ljust(84),
    "PASS",
)

print(
    "Unique full-cohort RCV keys".ljust(84),
    f"PASS ({final_full_validation['unique_rcv_keys']:,})",
)

print(
    "Unique evaluable-cohort RCV keys".ljust(84),
    f"PASS ({final_evaluable_validation['unique_rcv_keys']:,})",
)

print(
    "Primary instability events".ljust(84),
    f"PASS ({final_full_validation['primary_instability_events']:,})",
)

print(
    "Primary negatives".ljust(84),
    f"PASS ({final_full_validation['primary_negatives']:,})",
)

print(
    "Censored or nonevaluable retained".ljust(84),
    f"PASS ({final_full_validation['primary_outcome_missing']:,})",
)

print("\nSCIENTIFIC BOUNDARY")
print("-" * 120)

print(
    "AUPRC or AUROC calculated".ljust(84),
    "NO",
)

print(
    "Brier score or calibration calculated".ljust(84),
    "NO",
)

print(
    "Risk enrichment calculated".ljust(84),
    "NO",
)

print(
    "Threshold or weight optimization performed".ljust(84),
    "NO",
)

print(
    "Frozen Stage 5 outcome modified".ljust(84),
    "NO",
)

print(
    "Frozen Stage 6A comparator modified".ljust(84),
    "NO",
)

print("\nFINAL DECISION")
print("-" * 120)

print(
    "PASS_STAGE6B_LOCKED_SCORE_OUTCOME_COHORT_PACKAGE_ACCEPTED_AND_FROZEN"
)

print(
    "The complete locked accounting cohort, primary-evaluable cohort, "
    "44-check QC record, SHA-256 sidecars, and freeze manifest were "
    "written, read back, cryptographically verified, and frozen."
)

print(
    "The next authorized action is a fresh read-only verification of "
    "this Stage 6B package before any temporal performance analysis."
)

print("=" * 120)

STAGE 6B STEP 4 — CELL 6B-4A — LOCKED SCORE–OUTCOME COHORT PACKAGE FREEZE

FROZEN ARTIFACTS
------------------------------------------------------------------------------------------------------------------------
Full accounting cohort                                                               PASS (71,659 rows × 79 columns)
Primary-evaluable cohort                                                             PASS (66,636 rows × 79 columns)
Locked-cohort QC report                                                              PASS (44/44 checks)
Stage 6B freeze manifest                                                             PASS

CRYPTOGRAPHIC IDENTITIES
------------------------------------------------------------------------------------------------------------------------
Full accounting cohort SHA-256                                                       8ebf6e43eebaf75c1b30a1785085f8ce924cbbdab02c4648995d83f25426e7d1
Primary-evaluable cohort SHA-256                             

In [8]:
# ==================================================================================================
# STAGE 6B STEP 5 — CELL 6B-5A
# FRESH READ-ONLY VERIFICATION OF THE COMPLETE FROZEN STAGE 6B PACKAGE
#
# VERIFIES:
# 1. Full 71,659-row locked accounting cohort
# 2. Primary-evaluable 66,636-row cohort
# 3. 44-check locked-cohort QC JSON
# 4. Stage 6B freeze manifest
# 5. All four SHA-256 sidecars
# 6. Schema, key, row-order, outcome accounting, policy lineage, and manifest consistency
#
# SCIENTIFIC BOUNDARY:
# - Does NOT calculate AUPRC, AUROC, Brier score, calibration, enrichment, or thresholds.
# - Does NOT write, rename, overwrite, or modify any artifact.
# ==================================================================================================

from pathlib import Path
import hashlib
import json

import numpy as np
import pandas as pd
import pyarrow.parquet as pq


# --------------------------------------------------------------------------------------------------
# 1. Frozen Stage 6B paths
# --------------------------------------------------------------------------------------------------

STUDY_ROOT = Path(
    "/content/drive/MyDrive/GES_RAG_Temporal_Study"
)

STAGE6B_DATA_DIR = (
    STUDY_ROOT
    / "data_processed/stage6_temporal_validation"
)

STAGE6B_QC_DIR = (
    STUDY_ROOT
    / "outputs/quality_checks/stage6_temporal_validation"
)

STAGE6B_CONFIG_DIR = (
    STUDY_ROOT
    / "configs/stage6_temporal_validation"
)


FULL_COHORT_PATH = (
    STAGE6B_DATA_DIR
    / "stage6b_locked_score_outcome_accounting_cohort_v1.parquet"
)

EVALUABLE_COHORT_PATH = (
    STAGE6B_DATA_DIR
    / "stage6b_locked_primary_evaluable_cohort_v1.parquet"
)

QC_REPORT_PATH = (
    STAGE6B_QC_DIR
    / "stage6b_locked_score_outcome_cohort_qc_v1.json"
)

FREEZE_MANIFEST_PATH = (
    STAGE6B_CONFIG_DIR
    / "stage6b_locked_score_outcome_cohort_freeze_manifest_v1.json"
)


FULL_COHORT_SIDECAR_PATH = Path(
    str(FULL_COHORT_PATH) + ".sha256"
)

EVALUABLE_COHORT_SIDECAR_PATH = Path(
    str(EVALUABLE_COHORT_PATH) + ".sha256"
)

QC_REPORT_SIDECAR_PATH = Path(
    str(QC_REPORT_PATH) + ".sha256"
)

FREEZE_MANIFEST_SIDECAR_PATH = Path(
    str(FREEZE_MANIFEST_PATH) + ".sha256"
)


# --------------------------------------------------------------------------------------------------
# 2. Exact frozen identities recorded during Stage 6B Step 4
# --------------------------------------------------------------------------------------------------

EXPECTED_HASHES = {
    "full_accounting_cohort": (
        "8ebf6e43eebaf75c1b30a1785085f8ce924cbbdab02c4648995d83f25426e7d1"
    ),
    "primary_evaluable_cohort": (
        "c6ad50dfc376a746e1830d000dba975101d88763fd61b8d1983f6300edb6e038"
    ),
    "qc_report": (
        "ab81bf5653ad147b24fe53a4b63bc1a5c5531d3a986ce19a25969f764b372b14"
    ),
    "freeze_manifest": (
        "3d549a169be09b7978967cb5bb613d5cf963bf60e4f0c3a95c3fc3be3dac18a4"
    ),
}


# --------------------------------------------------------------------------------------------------
# 3. Frozen source identities and accounting expectations
# --------------------------------------------------------------------------------------------------

EXPECTED_SOURCE_COMPARATOR_SHA256 = (
    "650b1f312efb424f959a3ebca8ddb71ea1f0a2247db9e52cff72502b15f916b3"
)

EXPECTED_SOURCE_OUTCOME_SHA256 = (
    "c5508f5a8518160eef50482fd2c425dc4dcd9cf8a2fe04856e46760de60efbc8"
)

EXPECTED_COMPARATOR_POLICY_SHA256 = (
    "dd7e95dc785e77b04c289ef50817b1ddac7516a436d83b5734dbf3cd35b1248d"
)

EXPECTED_OUTCOME_POLICY_SHA256 = (
    "477b01080b249dce9f042b67251ab93a999a1373e5f01d8f57b5812d67a57c4e"
)

EXPECTED_FULL_ROWS = 71_659
EXPECTED_EVALUABLE_ROWS = 66_636
EXPECTED_COLUMNS = 79

EXPECTED_EVENTS = 6_485
EXPECTED_NEGATIVES = 60_151
EXPECTED_NONEVALUABLE = 5_023

EXPECTED_QC_CHECKS = 44
EXPECTED_QC_PASSED = 44
EXPECTED_QC_FAILED = 0


# --------------------------------------------------------------------------------------------------
# 4. Read-only helper functions
# --------------------------------------------------------------------------------------------------

def sha256_file(
    path: Path,
    chunk_size: int = 8 * 1024 * 1024,
) -> str:
    """Calculate a file's SHA-256 without modifying it."""

    digest = hashlib.sha256()

    with path.open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


def parse_sha256_sidecar(
    sidecar_path: Path,
) -> tuple[str, str | None]:
    """
    Read one checksum sidecar stored as:

        <hash>
        <hash>  <filename>
        <hash> *<filename>
    """

    text = sidecar_path.read_text(
        encoding="utf-8"
    ).strip()

    if not text:
        raise ValueError(
            f"Empty checksum sidecar: {sidecar_path}"
        )

    parts = text.split(maxsplit=1)

    sidecar_hash = parts[0].strip().lower()
    sidecar_filename = None

    if len(parts) == 2:
        sidecar_filename = (
            parts[1]
            .strip()
            .lstrip("*")
            .strip()
        )

    return sidecar_hash, sidecar_filename


def validate_policy_hash_column(
    dataframe: pd.DataFrame,
    column: str,
    expected_hash: str,
) -> None:
    """Require one complete, constant frozen policy hash."""

    values = (
        dataframe[column]
        .astype("string")
        .str.strip()
        .str.lower()
    )

    assert values.notna().all(), (
        f"Missing values found in {column}."
    )

    assert values.nunique(dropna=False) == 1, (
        f"Multiple values found in {column}: "
        f"{values.unique().tolist()}"
    )

    assert values.iloc[0] == expected_hash, (
        f"Unexpected hash in {column}.\n"
        f"Expected: {expected_hash}\n"
        f"Observed: {values.iloc[0]}"
    )


# --------------------------------------------------------------------------------------------------
# 5. Confirm all frozen files and sidecars exist
# --------------------------------------------------------------------------------------------------

required_paths = {
    "full accounting cohort": FULL_COHORT_PATH,
    "full accounting cohort sidecar": FULL_COHORT_SIDECAR_PATH,
    "primary-evaluable cohort": EVALUABLE_COHORT_PATH,
    "primary-evaluable cohort sidecar": EVALUABLE_COHORT_SIDECAR_PATH,
    "locked-cohort QC report": QC_REPORT_PATH,
    "locked-cohort QC sidecar": QC_REPORT_SIDECAR_PATH,
    "Stage 6B freeze manifest": FREEZE_MANIFEST_PATH,
    "Stage 6B freeze-manifest sidecar": FREEZE_MANIFEST_SIDECAR_PATH,
}

missing_paths = [
    f"{label}: {path}"
    for label, path in required_paths.items()
    if not path.is_file()
]

assert not missing_paths, (
    "One or more required Stage 6B files are missing:\n"
    + "\n".join(missing_paths)
)


# --------------------------------------------------------------------------------------------------
# 6. Recalculate all four frozen artifact hashes
# --------------------------------------------------------------------------------------------------

observed_hashes = {
    "full_accounting_cohort": sha256_file(
        FULL_COHORT_PATH
    ),
    "primary_evaluable_cohort": sha256_file(
        EVALUABLE_COHORT_PATH
    ),
    "qc_report": sha256_file(
        QC_REPORT_PATH
    ),
    "freeze_manifest": sha256_file(
        FREEZE_MANIFEST_PATH
    ),
}

for artifact_name, expected_hash in EXPECTED_HASHES.items():

    observed_hash = observed_hashes[artifact_name]

    assert observed_hash == expected_hash, (
        f"SHA-256 mismatch for {artifact_name}\n"
        f"Expected: {expected_hash}\n"
        f"Observed: {observed_hash}"
    )


# --------------------------------------------------------------------------------------------------
# 7. Verify all four SHA-256 sidecars
# --------------------------------------------------------------------------------------------------

sidecar_checks = {
    "full_accounting_cohort": (
        FULL_COHORT_SIDECAR_PATH,
        FULL_COHORT_PATH,
        observed_hashes["full_accounting_cohort"],
    ),
    "primary_evaluable_cohort": (
        EVALUABLE_COHORT_SIDECAR_PATH,
        EVALUABLE_COHORT_PATH,
        observed_hashes["primary_evaluable_cohort"],
    ),
    "qc_report": (
        QC_REPORT_SIDECAR_PATH,
        QC_REPORT_PATH,
        observed_hashes["qc_report"],
    ),
    "freeze_manifest": (
        FREEZE_MANIFEST_SIDECAR_PATH,
        FREEZE_MANIFEST_PATH,
        observed_hashes["freeze_manifest"],
    ),
}

for artifact_name, (
    sidecar_path,
    artifact_path,
    observed_hash,
) in sidecar_checks.items():

    sidecar_hash, sidecar_filename = (
        parse_sha256_sidecar(sidecar_path)
    )

    assert sidecar_hash == observed_hash, (
        f"Sidecar checksum mismatch for {artifact_name}\n"
        f"Sidecar:  {sidecar_hash}\n"
        f"Observed: {observed_hash}"
    )

    if sidecar_filename is not None:

        assert (
            Path(sidecar_filename).name
            == artifact_path.name
        ), (
            f"Sidecar filename mismatch for {artifact_name}\n"
            f"Expected: {artifact_path.name}\n"
            f"Observed: {sidecar_filename}"
        )


# --------------------------------------------------------------------------------------------------
# 8. Inspect both Parquet artifacts through metadata
# --------------------------------------------------------------------------------------------------

full_parquet_file = pq.ParquetFile(
    FULL_COHORT_PATH
)

evaluable_parquet_file = pq.ParquetFile(
    EVALUABLE_COHORT_PATH
)

full_schema = (
    full_parquet_file.schema_arrow.names
)

evaluable_schema = (
    evaluable_parquet_file.schema_arrow.names
)

assert full_parquet_file.metadata.num_rows == EXPECTED_FULL_ROWS
assert full_parquet_file.metadata.num_columns == EXPECTED_COLUMNS

assert (
    evaluable_parquet_file.metadata.num_rows
    == EXPECTED_EVALUABLE_ROWS
)

assert (
    evaluable_parquet_file.metadata.num_columns
    == EXPECTED_COLUMNS
)

assert len(full_schema) == EXPECTED_COLUMNS
assert len(set(full_schema)) == EXPECTED_COLUMNS

assert evaluable_schema == full_schema, (
    "The evaluable cohort schema does not exactly match "
    "the full accounting-cohort schema."
)


# --------------------------------------------------------------------------------------------------
# 9. Load both frozen Parquets in read-only analytical use
# --------------------------------------------------------------------------------------------------

full_df = pd.read_parquet(
    FULL_COHORT_PATH
)

evaluable_df = pd.read_parquet(
    EVALUABLE_COHORT_PATH
)

assert full_df.shape == (
    EXPECTED_FULL_ROWS,
    EXPECTED_COLUMNS,
)

assert evaluable_df.shape == (
    EXPECTED_EVALUABLE_ROWS,
    EXPECTED_COLUMNS,
)

assert full_df.columns.tolist() == full_schema
assert evaluable_df.columns.tolist() == full_schema


# --------------------------------------------------------------------------------------------------
# 10. Verify complete and unique RCV keys
# --------------------------------------------------------------------------------------------------

full_keys = (
    full_df["rcv_accession"]
    .astype("string")
    .str.strip()
)

evaluable_keys = (
    evaluable_df["rcv_accession"]
    .astype("string")
    .str.strip()
)

assert full_keys.notna().all()
assert full_keys.ne("").all()
assert full_keys.nunique(dropna=False) == EXPECTED_FULL_ROWS

assert evaluable_keys.notna().all()
assert evaluable_keys.ne("").all()
assert (
    evaluable_keys.nunique(dropna=False)
    == EXPECTED_EVALUABLE_ROWS
)


# --------------------------------------------------------------------------------------------------
# 11. Verify exact frozen full-cohort row order
# --------------------------------------------------------------------------------------------------

full_t0_order = pd.to_numeric(
    full_df["t0_row_order"],
    errors="raise",
).to_numpy(dtype=np.int64)

full_locked_order = pd.to_numeric(
    full_df["_locked_prejoin_row_order"],
    errors="raise",
).to_numpy(dtype=np.int64)

expected_full_order = np.arange(
    EXPECTED_FULL_ROWS,
    dtype=np.int64,
)

assert np.array_equal(
    full_t0_order,
    expected_full_order,
)

assert np.array_equal(
    full_locked_order,
    expected_full_order,
)

assert np.array_equal(
    full_t0_order,
    full_locked_order,
)


# --------------------------------------------------------------------------------------------------
# 12. Verify evaluable cohort is the exact ordered subset of the full cohort
# --------------------------------------------------------------------------------------------------

full_evaluable_mask = (
    full_df["primary_outcome_evaluable"]
    .astype("boolean")
)

expected_evaluable_keys = (
    full_df.loc[
        full_evaluable_mask,
        "rcv_accession",
    ]
    .astype("string")
    .reset_index(drop=True)
)

observed_evaluable_keys = (
    evaluable_df["rcv_accession"]
    .astype("string")
    .reset_index(drop=True)
)

assert expected_evaluable_keys.equals(
    observed_evaluable_keys
), (
    "The frozen evaluable cohort is not the exact ordered "
    "evaluable subset of the full accounting cohort."
)


expected_evaluable_t0_order = (
    full_df.loc[
        full_evaluable_mask,
        "t0_row_order",
    ]
    .reset_index(drop=True)
)

observed_evaluable_t0_order = (
    evaluable_df["t0_row_order"]
    .reset_index(drop=True)
)

assert expected_evaluable_t0_order.equals(
    observed_evaluable_t0_order
)

assert (
    evaluable_df["t0_row_order"]
    .is_monotonic_increasing
)

assert np.array_equal(
    pd.to_numeric(
        evaluable_df["t0_row_order"],
        errors="raise",
    ).to_numpy(dtype=np.int64),
    pd.to_numeric(
        evaluable_df["_locked_prejoin_row_order"],
        errors="raise",
    ).to_numpy(dtype=np.int64),
)


# --------------------------------------------------------------------------------------------------
# 13. Verify frozen policy lineage in both cohorts
# --------------------------------------------------------------------------------------------------

validate_policy_hash_column(
    dataframe=full_df,
    column="comparator_policy_sha256",
    expected_hash=EXPECTED_COMPARATOR_POLICY_SHA256,
)

validate_policy_hash_column(
    dataframe=full_df,
    column="outcome_policy_sha256",
    expected_hash=EXPECTED_OUTCOME_POLICY_SHA256,
)

validate_policy_hash_column(
    dataframe=evaluable_df,
    column="comparator_policy_sha256",
    expected_hash=EXPECTED_COMPARATOR_POLICY_SHA256,
)

validate_policy_hash_column(
    dataframe=evaluable_df,
    column="outcome_policy_sha256",
    expected_hash=EXPECTED_OUTCOME_POLICY_SHA256,
)


# --------------------------------------------------------------------------------------------------
# 14. Verify frozen full-cohort outcome accounting
# --------------------------------------------------------------------------------------------------

full_evaluable = (
    full_df["primary_outcome_evaluable"]
    .astype("boolean")
)

full_outcome = pd.to_numeric(
    full_df["primary_future_instability"],
    errors="coerce",
)

assert int(full_evaluable.sum()) == EXPECTED_EVALUABLE_ROWS
assert int((~full_evaluable).sum()) == EXPECTED_NONEVALUABLE

assert int((full_outcome == 1).sum()) == EXPECTED_EVENTS
assert int((full_outcome == 0).sum()) == EXPECTED_NEGATIVES
assert int(full_outcome.isna().sum()) == EXPECTED_NONEVALUABLE

assert full_outcome.loc[full_evaluable].notna().all()
assert full_outcome.loc[~full_evaluable].isna().all()


# --------------------------------------------------------------------------------------------------
# 15. Verify frozen evaluable-cohort outcome accounting
# --------------------------------------------------------------------------------------------------

evaluable_indicator = (
    evaluable_df["primary_outcome_evaluable"]
    .astype("boolean")
)

evaluable_outcome = pd.to_numeric(
    evaluable_df["primary_future_instability"],
    errors="coerce",
)

assert evaluable_indicator.notna().all()
assert evaluable_indicator.all()

assert evaluable_outcome.notna().all()
assert evaluable_outcome.isin([0, 1]).all()

assert int((evaluable_outcome == 1).sum()) == EXPECTED_EVENTS
assert int((evaluable_outcome == 0).sum()) == EXPECTED_NEGATIVES


# --------------------------------------------------------------------------------------------------
# 16. Verify the 44-check QC report
# --------------------------------------------------------------------------------------------------

with QC_REPORT_PATH.open(
    "r",
    encoding="utf-8",
) as handle:
    qc_report = json.load(handle)

assert qc_report["decision"] == (
    "PASS_STAGE6B_LOCKED_COHORT_QC_COMPLETE"
)

assert qc_report["qc_summary"] == {
    "failed_checks": EXPECTED_QC_FAILED,
    "passed_checks": EXPECTED_QC_PASSED,
    "total_checks": EXPECTED_QC_CHECKS,
}

assert len(qc_report["qc_checks"]) == EXPECTED_QC_CHECKS

assert all(
    check["passed"] is True
    for check in qc_report["qc_checks"]
)

assert (
    qc_report["full_accounting_cohort"]["sha256"]
    == EXPECTED_HASHES["full_accounting_cohort"]
)

assert (
    qc_report["primary_evaluable_cohort"]["sha256"]
    == EXPECTED_HASHES["primary_evaluable_cohort"]
)

assert (
    qc_report["source_artifacts"]
    ["stage6a_comparator_parquet_sha256"]
    == EXPECTED_SOURCE_COMPARATOR_SHA256
)

assert (
    qc_report["source_artifacts"]
    ["stage5_outcome_parquet_sha256"]
    == EXPECTED_SOURCE_OUTCOME_SHA256
)

assert (
    qc_report["source_artifacts"]
    ["comparator_policy_sha256"]
    == EXPECTED_COMPARATOR_POLICY_SHA256
)

assert (
    qc_report["source_artifacts"]
    ["outcome_policy_sha256"]
    == EXPECTED_OUTCOME_POLICY_SHA256
)


# --------------------------------------------------------------------------------------------------
# 17. Verify the Stage 6B freeze manifest
# --------------------------------------------------------------------------------------------------

with FREEZE_MANIFEST_PATH.open(
    "r",
    encoding="utf-8",
) as handle:
    freeze_manifest = json.load(handle)

assert freeze_manifest["decision"] == (
    "PASS_STAGE6B_LOCKED_SCORE_OUTCOME_COHORT_PACKAGE_"
    "ACCEPTED_AND_FROZEN"
)

frozen_artifacts = freeze_manifest[
    "frozen_artifacts"
]

assert (
    frozen_artifacts["full_accounting_cohort"]["sha256"]
    == EXPECTED_HASHES["full_accounting_cohort"]
)

assert (
    frozen_artifacts["primary_evaluable_cohort"]["sha256"]
    == EXPECTED_HASHES["primary_evaluable_cohort"]
)

assert (
    frozen_artifacts["qc_report"]["sha256"]
    == EXPECTED_HASHES["qc_report"]
)

assert (
    frozen_artifacts["full_accounting_cohort"]["rows"]
    == EXPECTED_FULL_ROWS
)

assert (
    frozen_artifacts["primary_evaluable_cohort"]["rows"]
    == EXPECTED_EVALUABLE_ROWS
)

assert (
    frozen_artifacts["full_accounting_cohort"]["columns"]
    == EXPECTED_COLUMNS
)

assert (
    frozen_artifacts["primary_evaluable_cohort"]["columns"]
    == EXPECTED_COLUMNS
)

assert (
    frozen_artifacts["full_accounting_cohort"]
    ["primary_instability_events"]
    == EXPECTED_EVENTS
)

assert (
    frozen_artifacts["full_accounting_cohort"]
    ["primary_negatives"]
    == EXPECTED_NEGATIVES
)

assert (
    frozen_artifacts["full_accounting_cohort"]
    ["censored_or_nonevaluable"]
    == EXPECTED_NONEVALUABLE
)


# --------------------------------------------------------------------------------------------------
# 18. Verify manifest source identity and join controls
# --------------------------------------------------------------------------------------------------

source_artifacts = freeze_manifest[
    "source_artifacts"
]

assert (
    source_artifacts["stage6a_comparator_score_parquet"]["sha256"]
    == EXPECTED_SOURCE_COMPARATOR_SHA256
)

assert (
    source_artifacts["stage5_primary_outcome_parquet"]["sha256"]
    == EXPECTED_SOURCE_OUTCOME_SHA256
)

assert (
    source_artifacts["comparator_policy_sha256"]
    == EXPECTED_COMPARATOR_POLICY_SHA256
)

assert (
    source_artifacts["outcome_policy_sha256"]
    == EXPECTED_OUTCOME_POLICY_SHA256
)


join_controls = freeze_manifest[
    "join_and_order_controls"
]

assert join_controls["join_key"] == "rcv_accession"
assert join_controls["join_cardinality"] == "one_to_one"
assert join_controls["matched_rows"] == EXPECTED_FULL_ROWS
assert join_controls["comparator_only_rows"] == 0
assert join_controls["outcome_only_rows"] == 0
assert join_controls["locked_prejoin_order_preserved"] is True
assert join_controls["nonkey_source_column_overlap"] == []


# --------------------------------------------------------------------------------------------------
# 19. Verify that performance remained unopened during package construction
# --------------------------------------------------------------------------------------------------

scientific_boundary = freeze_manifest[
    "scientific_boundary"
]

required_false_boundary_fields = [
    "temporal_performance_calculated",
    "auprc_calculated",
    "auroc_calculated",
    "brier_score_calculated",
    "calibration_calculated",
    "risk_enrichment_calculated",
    "subgroup_performance_calculated",
    "threshold_optimization_performed",
    "weight_optimization_performed",
    "frozen_outcome_modified",
    "frozen_comparator_modified",
]

for field in required_false_boundary_fields:

    assert scientific_boundary[field] is False, (
        f"Unexpected scientific-boundary value for {field}: "
        f"{scientific_boundary[field]}"
    )


assert (
    freeze_manifest["authorization_after_freeze"]
    ["performance_analysis_authorized_in_this_cell"]
    is False
)


# --------------------------------------------------------------------------------------------------
# 20. Final verification output
# --------------------------------------------------------------------------------------------------

print("=" * 120)
print(
    "STAGE 6B STEP 5 — CELL 6B-5A — "
    "FRESH FROZEN STAGE 6B PACKAGE VERIFICATION"
)
print("=" * 120)

print("\nCRYPTOGRAPHIC CHECKS")
print("-" * 120)
print("Full accounting cohort SHA-256".ljust(84), "PASS")
print("Full accounting cohort sidecar".ljust(84), "PASS")
print("Primary-evaluable cohort SHA-256".ljust(84), "PASS")
print("Primary-evaluable cohort sidecar".ljust(84), "PASS")
print("Locked-cohort QC SHA-256".ljust(84), "PASS")
print("Locked-cohort QC sidecar".ljust(84), "PASS")
print("Stage 6B freeze-manifest SHA-256".ljust(84), "PASS")
print("Stage 6B freeze-manifest sidecar".ljust(84), "PASS")

print("\nSTRUCTURAL, KEY, AND ORDER CHECKS")
print("-" * 120)
print(
    "Full accounting cohort shape".ljust(84),
    f"PASS ({full_df.shape[0]:,} × {full_df.shape[1]})",
)
print(
    "Primary-evaluable cohort shape".ljust(84),
    f"PASS ({evaluable_df.shape[0]:,} × {evaluable_df.shape[1]})",
)
print(
    "Identical 79-column schemas".ljust(84),
    "PASS",
)
print(
    "Unique full-cohort RCV keys".ljust(84),
    f"PASS ({full_keys.nunique():,})",
)
print(
    "Unique evaluable-cohort RCV keys".ljust(84),
    f"PASS ({evaluable_keys.nunique():,})",
)
print(
    "Full cohort zero-based row order".ljust(84),
    "PASS (0 through 71,658)",
)
print(
    "Evaluable cohort exact ordered subset".ljust(84),
    "PASS",
)

print("\nPOLICY, QC, AND MANIFEST LINEAGE")
print("-" * 120)
print("Comparator-policy lineage".ljust(84), "PASS")
print("Outcome-policy lineage".ljust(84), "PASS")
print(
    "Locked-cohort QC checks".ljust(84),
    "PASS (44 passed / 0 failed)",
)
print("QC artifact identities".ljust(84), "PASS")
print("Freeze-manifest artifact identities".ljust(84), "PASS")
print("Frozen source identities".ljust(84), "PASS")
print("One-to-one join controls".ljust(84), "PASS")

print("\nOUTCOME ACCOUNTING")
print("-" * 120)
print(
    "Full accounting rows".ljust(84),
    f"PASS ({EXPECTED_FULL_ROWS:,})",
)
print(
    "Primary-outcome-evaluable rows".ljust(84),
    f"PASS ({EXPECTED_EVALUABLE_ROWS:,})",
)
print(
    "Primary instability events".ljust(84),
    f"PASS ({EXPECTED_EVENTS:,})",
)
print(
    "Primary negatives".ljust(84),
    f"PASS ({EXPECTED_NEGATIVES:,})",
)
print(
    "Censored or nonevaluable rows".ljust(84),
    f"PASS ({EXPECTED_NONEVALUABLE:,})",
)
print(
    "Evaluable cohort contains no missing outcomes".ljust(84),
    "PASS",
)

print("\nSCIENTIFIC BOUNDARY")
print("-" * 120)
print("Frozen Stage 6B package opened".ljust(84), "YES — READ ONLY")
print("AUPRC or AUROC calculated".ljust(84), "NO")
print("Brier score or calibration calculated".ljust(84), "NO")
print("Risk enrichment calculated".ljust(84), "NO")
print("Subgroup performance calculated".ljust(84), "NO")
print("Threshold or weight optimization performed".ljust(84), "NO")
print("Scientific artifact written or modified".ljust(84), "NO")

print("\nFINAL DECISION")
print("-" * 120)
print(
    "PASS_STAGE6B_FROZEN_PACKAGE_FRESHLY_REVERIFIED"
)
print(
    "The complete Stage 6B locked cohort package passed fresh "
    "cryptographic, structural, accounting, policy-lineage, QC, "
    "manifest, and readback verification."
)
print(
    "The frozen primary-evaluable cohort is ready for the "
    "prespecified locked temporal-performance analysis."
)
print("=" * 120)

STAGE 6B STEP 5 — CELL 6B-5A — FRESH FROZEN STAGE 6B PACKAGE VERIFICATION

CRYPTOGRAPHIC CHECKS
------------------------------------------------------------------------------------------------------------------------
Full accounting cohort SHA-256                                                       PASS
Full accounting cohort sidecar                                                       PASS
Primary-evaluable cohort SHA-256                                                     PASS
Primary-evaluable cohort sidecar                                                     PASS
Locked-cohort QC SHA-256                                                             PASS
Locked-cohort QC sidecar                                                             PASS
Stage 6B freeze-manifest SHA-256                                                     PASS
Stage 6B freeze-manifest sidecar                                                     PASS

STRUCTURAL, KEY, AND ORDER CHECKS
----------------------------

In [9]:
# ==================================================================================================
# STAGE 6B STEP 5 — CELL 6B-5A
# FRESH READ-ONLY VERIFICATION OF THE COMPLETE FROZEN STAGE 6B PACKAGE
#
# VERIFIES:
# 1. Full 71,659-row locked accounting cohort
# 2. Primary-evaluable 66,636-row cohort
# 3. 44-check locked-cohort QC JSON
# 4. Stage 6B freeze manifest
# 5. All four SHA-256 sidecars
# 6. Schema, key, row-order, outcome accounting, policy lineage, and manifest consistency
#
# SCIENTIFIC BOUNDARY:
# - Does NOT calculate AUPRC, AUROC, Brier score, calibration, enrichment, or thresholds.
# - Does NOT write, rename, overwrite, or modify any artifact.
# ==================================================================================================

from pathlib import Path
import hashlib
import json

import numpy as np
import pandas as pd
import pyarrow.parquet as pq


# --------------------------------------------------------------------------------------------------
# 1. Frozen Stage 6B paths
# --------------------------------------------------------------------------------------------------

STUDY_ROOT = Path(
    "/content/drive/MyDrive/GES_RAG_Temporal_Study"
)

STAGE6B_DATA_DIR = (
    STUDY_ROOT
    / "data_processed/stage6_temporal_validation"
)

STAGE6B_QC_DIR = (
    STUDY_ROOT
    / "outputs/quality_checks/stage6_temporal_validation"
)

STAGE6B_CONFIG_DIR = (
    STUDY_ROOT
    / "configs/stage6_temporal_validation"
)


FULL_COHORT_PATH = (
    STAGE6B_DATA_DIR
    / "stage6b_locked_score_outcome_accounting_cohort_v1.parquet"
)

EVALUABLE_COHORT_PATH = (
    STAGE6B_DATA_DIR
    / "stage6b_locked_primary_evaluable_cohort_v1.parquet"
)

QC_REPORT_PATH = (
    STAGE6B_QC_DIR
    / "stage6b_locked_score_outcome_cohort_qc_v1.json"
)

FREEZE_MANIFEST_PATH = (
    STAGE6B_CONFIG_DIR
    / "stage6b_locked_score_outcome_cohort_freeze_manifest_v1.json"
)


FULL_COHORT_SIDECAR_PATH = Path(
    str(FULL_COHORT_PATH) + ".sha256"
)

EVALUABLE_COHORT_SIDECAR_PATH = Path(
    str(EVALUABLE_COHORT_PATH) + ".sha256"
)

QC_REPORT_SIDECAR_PATH = Path(
    str(QC_REPORT_PATH) + ".sha256"
)

FREEZE_MANIFEST_SIDECAR_PATH = Path(
    str(FREEZE_MANIFEST_PATH) + ".sha256"
)


# --------------------------------------------------------------------------------------------------
# 2. Exact frozen identities recorded during Stage 6B Step 4
# --------------------------------------------------------------------------------------------------

EXPECTED_HASHES = {
    "full_accounting_cohort": (
        "8ebf6e43eebaf75c1b30a1785085f8ce924cbbdab02c4648995d83f25426e7d1"
    ),
    "primary_evaluable_cohort": (
        "c6ad50dfc376a746e1830d000dba975101d88763fd61b8d1983f6300edb6e038"
    ),
    "qc_report": (
        "ab81bf5653ad147b24fe53a4b63bc1a5c5531d3a986ce19a25969f764b372b14"
    ),
    "freeze_manifest": (
        "3d549a169be09b7978967cb5bb613d5cf963bf60e4f0c3a95c3fc3be3dac18a4"
    ),
}


# --------------------------------------------------------------------------------------------------
# 3. Frozen source identities and accounting expectations
# --------------------------------------------------------------------------------------------------

EXPECTED_SOURCE_COMPARATOR_SHA256 = (
    "650b1f312efb424f959a3ebca8ddb71ea1f0a2247db9e52cff72502b15f916b3"
)

EXPECTED_SOURCE_OUTCOME_SHA256 = (
    "c5508f5a8518160eef50482fd2c425dc4dcd9cf8a2fe04856e46760de60efbc8"
)

EXPECTED_COMPARATOR_POLICY_SHA256 = (
    "dd7e95dc785e77b04c289ef50817b1ddac7516a436d83b5734dbf3cd35b1248d"
)

EXPECTED_OUTCOME_POLICY_SHA256 = (
    "477b01080b249dce9f042b67251ab93a999a1373e5f01d8f57b5812d67a57c4e"
)

EXPECTED_FULL_ROWS = 71_659
EXPECTED_EVALUABLE_ROWS = 66_636
EXPECTED_COLUMNS = 79

EXPECTED_EVENTS = 6_485
EXPECTED_NEGATIVES = 60_151
EXPECTED_NONEVALUABLE = 5_023

EXPECTED_QC_CHECKS = 44
EXPECTED_QC_PASSED = 44
EXPECTED_QC_FAILED = 0


# --------------------------------------------------------------------------------------------------
# 4. Read-only helper functions
# --------------------------------------------------------------------------------------------------

def sha256_file(
    path: Path,
    chunk_size: int = 8 * 1024 * 1024,
) -> str:
    """Calculate a file's SHA-256 without modifying it."""

    digest = hashlib.sha256()

    with path.open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


def parse_sha256_sidecar(
    sidecar_path: Path,
) -> tuple[str, str | None]:
    """
    Read one checksum sidecar stored as:

        <hash>
        <hash>  <filename>
        <hash> *<filename>
    """

    text = sidecar_path.read_text(
        encoding="utf-8"
    ).strip()

    if not text:
        raise ValueError(
            f"Empty checksum sidecar: {sidecar_path}"
        )

    parts = text.split(maxsplit=1)

    sidecar_hash = parts[0].strip().lower()
    sidecar_filename = None

    if len(parts) == 2:
        sidecar_filename = (
            parts[1]
            .strip()
            .lstrip("*")
            .strip()
        )

    return sidecar_hash, sidecar_filename


def validate_policy_hash_column(
    dataframe: pd.DataFrame,
    column: str,
    expected_hash: str,
) -> None:
    """Require one complete, constant frozen policy hash."""

    values = (
        dataframe[column]
        .astype("string")
        .str.strip()
        .str.lower()
    )

    assert values.notna().all(), (
        f"Missing values found in {column}."
    )

    assert values.nunique(dropna=False) == 1, (
        f"Multiple values found in {column}: "
        f"{values.unique().tolist()}"
    )

    assert values.iloc[0] == expected_hash, (
        f"Unexpected hash in {column}.\n"
        f"Expected: {expected_hash}\n"
        f"Observed: {values.iloc[0]}"
    )


# --------------------------------------------------------------------------------------------------
# 5. Confirm all frozen files and sidecars exist
# --------------------------------------------------------------------------------------------------

required_paths = {
    "full accounting cohort": FULL_COHORT_PATH,
    "full accounting cohort sidecar": FULL_COHORT_SIDECAR_PATH,
    "primary-evaluable cohort": EVALUABLE_COHORT_PATH,
    "primary-evaluable cohort sidecar": EVALUABLE_COHORT_SIDECAR_PATH,
    "locked-cohort QC report": QC_REPORT_PATH,
    "locked-cohort QC sidecar": QC_REPORT_SIDECAR_PATH,
    "Stage 6B freeze manifest": FREEZE_MANIFEST_PATH,
    "Stage 6B freeze-manifest sidecar": FREEZE_MANIFEST_SIDECAR_PATH,
}

missing_paths = [
    f"{label}: {path}"
    for label, path in required_paths.items()
    if not path.is_file()
]

assert not missing_paths, (
    "One or more required Stage 6B files are missing:\n"
    + "\n".join(missing_paths)
)


# --------------------------------------------------------------------------------------------------
# 6. Recalculate all four frozen artifact hashes
# --------------------------------------------------------------------------------------------------

observed_hashes = {
    "full_accounting_cohort": sha256_file(
        FULL_COHORT_PATH
    ),
    "primary_evaluable_cohort": sha256_file(
        EVALUABLE_COHORT_PATH
    ),
    "qc_report": sha256_file(
        QC_REPORT_PATH
    ),
    "freeze_manifest": sha256_file(
        FREEZE_MANIFEST_PATH
    ),
}

for artifact_name, expected_hash in EXPECTED_HASHES.items():

    observed_hash = observed_hashes[artifact_name]

    assert observed_hash == expected_hash, (
        f"SHA-256 mismatch for {artifact_name}\n"
        f"Expected: {expected_hash}\n"
        f"Observed: {observed_hash}"
    )


# --------------------------------------------------------------------------------------------------
# 7. Verify all four SHA-256 sidecars
# --------------------------------------------------------------------------------------------------

sidecar_checks = {
    "full_accounting_cohort": (
        FULL_COHORT_SIDECAR_PATH,
        FULL_COHORT_PATH,
        observed_hashes["full_accounting_cohort"],
    ),
    "primary_evaluable_cohort": (
        EVALUABLE_COHORT_SIDECAR_PATH,
        EVALUABLE_COHORT_PATH,
        observed_hashes["primary_evaluable_cohort"],
    ),
    "qc_report": (
        QC_REPORT_SIDECAR_PATH,
        QC_REPORT_PATH,
        observed_hashes["qc_report"],
    ),
    "freeze_manifest": (
        FREEZE_MANIFEST_SIDECAR_PATH,
        FREEZE_MANIFEST_PATH,
        observed_hashes["freeze_manifest"],
    ),
}

for artifact_name, (
    sidecar_path,
    artifact_path,
    observed_hash,
) in sidecar_checks.items():

    sidecar_hash, sidecar_filename = (
        parse_sha256_sidecar(sidecar_path)
    )

    assert sidecar_hash == observed_hash, (
        f"Sidecar checksum mismatch for {artifact_name}\n"
        f"Sidecar:  {sidecar_hash}\n"
        f"Observed: {observed_hash}"
    )

    if sidecar_filename is not None:

        assert (
            Path(sidecar_filename).name
            == artifact_path.name
        ), (
            f"Sidecar filename mismatch for {artifact_name}\n"
            f"Expected: {artifact_path.name}\n"
            f"Observed: {sidecar_filename}"
        )


# --------------------------------------------------------------------------------------------------
# 8. Inspect both Parquet artifacts through metadata
# --------------------------------------------------------------------------------------------------

full_parquet_file = pq.ParquetFile(
    FULL_COHORT_PATH
)

evaluable_parquet_file = pq.ParquetFile(
    EVALUABLE_COHORT_PATH
)

full_schema = (
    full_parquet_file.schema_arrow.names
)

evaluable_schema = (
    evaluable_parquet_file.schema_arrow.names
)

assert full_parquet_file.metadata.num_rows == EXPECTED_FULL_ROWS
assert full_parquet_file.metadata.num_columns == EXPECTED_COLUMNS

assert (
    evaluable_parquet_file.metadata.num_rows
    == EXPECTED_EVALUABLE_ROWS
)

assert (
    evaluable_parquet_file.metadata.num_columns
    == EXPECTED_COLUMNS
)

assert len(full_schema) == EXPECTED_COLUMNS
assert len(set(full_schema)) == EXPECTED_COLUMNS

assert evaluable_schema == full_schema, (
    "The evaluable cohort schema does not exactly match "
    "the full accounting-cohort schema."
)


# --------------------------------------------------------------------------------------------------
# 9. Load both frozen Parquets in read-only analytical use
# --------------------------------------------------------------------------------------------------

full_df = pd.read_parquet(
    FULL_COHORT_PATH
)

evaluable_df = pd.read_parquet(
    EVALUABLE_COHORT_PATH
)

assert full_df.shape == (
    EXPECTED_FULL_ROWS,
    EXPECTED_COLUMNS,
)

assert evaluable_df.shape == (
    EXPECTED_EVALUABLE_ROWS,
    EXPECTED_COLUMNS,
)

assert full_df.columns.tolist() == full_schema
assert evaluable_df.columns.tolist() == full_schema


# --------------------------------------------------------------------------------------------------
# 10. Verify complete and unique RCV keys
# --------------------------------------------------------------------------------------------------

full_keys = (
    full_df["rcv_accession"]
    .astype("string")
    .str.strip()
)

evaluable_keys = (
    evaluable_df["rcv_accession"]
    .astype("string")
    .str.strip()
)

assert full_keys.notna().all()
assert full_keys.ne("").all()
assert full_keys.nunique(dropna=False) == EXPECTED_FULL_ROWS

assert evaluable_keys.notna().all()
assert evaluable_keys.ne("").all()
assert (
    evaluable_keys.nunique(dropna=False)
    == EXPECTED_EVALUABLE_ROWS
)


# --------------------------------------------------------------------------------------------------
# 11. Verify exact frozen full-cohort row order
# --------------------------------------------------------------------------------------------------

full_t0_order = pd.to_numeric(
    full_df["t0_row_order"],
    errors="raise",
).to_numpy(dtype=np.int64)

full_locked_order = pd.to_numeric(
    full_df["_locked_prejoin_row_order"],
    errors="raise",
).to_numpy(dtype=np.int64)

expected_full_order = np.arange(
    EXPECTED_FULL_ROWS,
    dtype=np.int64,
)

assert np.array_equal(
    full_t0_order,
    expected_full_order,
)

assert np.array_equal(
    full_locked_order,
    expected_full_order,
)

assert np.array_equal(
    full_t0_order,
    full_locked_order,
)


# --------------------------------------------------------------------------------------------------
# 12. Verify evaluable cohort is the exact ordered subset of the full cohort
# --------------------------------------------------------------------------------------------------

full_evaluable_mask = (
    full_df["primary_outcome_evaluable"]
    .astype("boolean")
)

expected_evaluable_keys = (
    full_df.loc[
        full_evaluable_mask,
        "rcv_accession",
    ]
    .astype("string")
    .reset_index(drop=True)
)

observed_evaluable_keys = (
    evaluable_df["rcv_accession"]
    .astype("string")
    .reset_index(drop=True)
)

assert expected_evaluable_keys.equals(
    observed_evaluable_keys
), (
    "The frozen evaluable cohort is not the exact ordered "
    "evaluable subset of the full accounting cohort."
)


expected_evaluable_t0_order = (
    full_df.loc[
        full_evaluable_mask,
        "t0_row_order",
    ]
    .reset_index(drop=True)
)

observed_evaluable_t0_order = (
    evaluable_df["t0_row_order"]
    .reset_index(drop=True)
)

assert expected_evaluable_t0_order.equals(
    observed_evaluable_t0_order
)

assert (
    evaluable_df["t0_row_order"]
    .is_monotonic_increasing
)

assert np.array_equal(
    pd.to_numeric(
        evaluable_df["t0_row_order"],
        errors="raise",
    ).to_numpy(dtype=np.int64),
    pd.to_numeric(
        evaluable_df["_locked_prejoin_row_order"],
        errors="raise",
    ).to_numpy(dtype=np.int64),
)


# --------------------------------------------------------------------------------------------------
# 13. Verify frozen policy lineage in both cohorts
# --------------------------------------------------------------------------------------------------

validate_policy_hash_column(
    dataframe=full_df,
    column="comparator_policy_sha256",
    expected_hash=EXPECTED_COMPARATOR_POLICY_SHA256,
)

validate_policy_hash_column(
    dataframe=full_df,
    column="outcome_policy_sha256",
    expected_hash=EXPECTED_OUTCOME_POLICY_SHA256,
)

validate_policy_hash_column(
    dataframe=evaluable_df,
    column="comparator_policy_sha256",
    expected_hash=EXPECTED_COMPARATOR_POLICY_SHA256,
)

validate_policy_hash_column(
    dataframe=evaluable_df,
    column="outcome_policy_sha256",
    expected_hash=EXPECTED_OUTCOME_POLICY_SHA256,
)


# --------------------------------------------------------------------------------------------------
# 14. Verify frozen full-cohort outcome accounting
# --------------------------------------------------------------------------------------------------

full_evaluable = (
    full_df["primary_outcome_evaluable"]
    .astype("boolean")
)

full_outcome = pd.to_numeric(
    full_df["primary_future_instability"],
    errors="coerce",
)

assert int(full_evaluable.sum()) == EXPECTED_EVALUABLE_ROWS
assert int((~full_evaluable).sum()) == EXPECTED_NONEVALUABLE

assert int((full_outcome == 1).sum()) == EXPECTED_EVENTS
assert int((full_outcome == 0).sum()) == EXPECTED_NEGATIVES
assert int(full_outcome.isna().sum()) == EXPECTED_NONEVALUABLE

assert full_outcome.loc[full_evaluable].notna().all()
assert full_outcome.loc[~full_evaluable].isna().all()


# --------------------------------------------------------------------------------------------------
# 15. Verify frozen evaluable-cohort outcome accounting
# --------------------------------------------------------------------------------------------------

evaluable_indicator = (
    evaluable_df["primary_outcome_evaluable"]
    .astype("boolean")
)

evaluable_outcome = pd.to_numeric(
    evaluable_df["primary_future_instability"],
    errors="coerce",
)

assert evaluable_indicator.notna().all()
assert evaluable_indicator.all()

assert evaluable_outcome.notna().all()
assert evaluable_outcome.isin([0, 1]).all()

assert int((evaluable_outcome == 1).sum()) == EXPECTED_EVENTS
assert int((evaluable_outcome == 0).sum()) == EXPECTED_NEGATIVES


# --------------------------------------------------------------------------------------------------
# 16. Verify the 44-check QC report
# --------------------------------------------------------------------------------------------------

with QC_REPORT_PATH.open(
    "r",
    encoding="utf-8",
) as handle:
    qc_report = json.load(handle)

assert qc_report["decision"] == (
    "PASS_STAGE6B_LOCKED_COHORT_QC_COMPLETE"
)

assert qc_report["qc_summary"] == {
    "failed_checks": EXPECTED_QC_FAILED,
    "passed_checks": EXPECTED_QC_PASSED,
    "total_checks": EXPECTED_QC_CHECKS,
}

assert len(qc_report["qc_checks"]) == EXPECTED_QC_CHECKS

assert all(
    check["passed"] is True
    for check in qc_report["qc_checks"]
)

assert (
    qc_report["full_accounting_cohort"]["sha256"]
    == EXPECTED_HASHES["full_accounting_cohort"]
)

assert (
    qc_report["primary_evaluable_cohort"]["sha256"]
    == EXPECTED_HASHES["primary_evaluable_cohort"]
)

assert (
    qc_report["source_artifacts"]
    ["stage6a_comparator_parquet_sha256"]
    == EXPECTED_SOURCE_COMPARATOR_SHA256
)

assert (
    qc_report["source_artifacts"]
    ["stage5_outcome_parquet_sha256"]
    == EXPECTED_SOURCE_OUTCOME_SHA256
)

assert (
    qc_report["source_artifacts"]
    ["comparator_policy_sha256"]
    == EXPECTED_COMPARATOR_POLICY_SHA256
)

assert (
    qc_report["source_artifacts"]
    ["outcome_policy_sha256"]
    == EXPECTED_OUTCOME_POLICY_SHA256
)


# --------------------------------------------------------------------------------------------------
# 17. Verify the Stage 6B freeze manifest
# --------------------------------------------------------------------------------------------------

with FREEZE_MANIFEST_PATH.open(
    "r",
    encoding="utf-8",
) as handle:
    freeze_manifest = json.load(handle)

assert freeze_manifest["decision"] == (
    "PASS_STAGE6B_LOCKED_SCORE_OUTCOME_COHORT_PACKAGE_"
    "ACCEPTED_AND_FROZEN"
)

frozen_artifacts = freeze_manifest[
    "frozen_artifacts"
]

assert (
    frozen_artifacts["full_accounting_cohort"]["sha256"]
    == EXPECTED_HASHES["full_accounting_cohort"]
)

assert (
    frozen_artifacts["primary_evaluable_cohort"]["sha256"]
    == EXPECTED_HASHES["primary_evaluable_cohort"]
)

assert (
    frozen_artifacts["qc_report"]["sha256"]
    == EXPECTED_HASHES["qc_report"]
)

assert (
    frozen_artifacts["full_accounting_cohort"]["rows"]
    == EXPECTED_FULL_ROWS
)

assert (
    frozen_artifacts["primary_evaluable_cohort"]["rows"]
    == EXPECTED_EVALUABLE_ROWS
)

assert (
    frozen_artifacts["full_accounting_cohort"]["columns"]
    == EXPECTED_COLUMNS
)

assert (
    frozen_artifacts["primary_evaluable_cohort"]["columns"]
    == EXPECTED_COLUMNS
)

assert (
    frozen_artifacts["full_accounting_cohort"]
    ["primary_instability_events"]
    == EXPECTED_EVENTS
)

assert (
    frozen_artifacts["full_accounting_cohort"]
    ["primary_negatives"]
    == EXPECTED_NEGATIVES
)

assert (
    frozen_artifacts["full_accounting_cohort"]
    ["censored_or_nonevaluable"]
    == EXPECTED_NONEVALUABLE
)


# --------------------------------------------------------------------------------------------------
# 18. Verify manifest source identity and join controls
# --------------------------------------------------------------------------------------------------

source_artifacts = freeze_manifest[
    "source_artifacts"
]

assert (
    source_artifacts["stage6a_comparator_score_parquet"]["sha256"]
    == EXPECTED_SOURCE_COMPARATOR_SHA256
)

assert (
    source_artifacts["stage5_primary_outcome_parquet"]["sha256"]
    == EXPECTED_SOURCE_OUTCOME_SHA256
)

assert (
    source_artifacts["comparator_policy_sha256"]
    == EXPECTED_COMPARATOR_POLICY_SHA256
)

assert (
    source_artifacts["outcome_policy_sha256"]
    == EXPECTED_OUTCOME_POLICY_SHA256
)


join_controls = freeze_manifest[
    "join_and_order_controls"
]

assert join_controls["join_key"] == "rcv_accession"
assert join_controls["join_cardinality"] == "one_to_one"
assert join_controls["matched_rows"] == EXPECTED_FULL_ROWS
assert join_controls["comparator_only_rows"] == 0
assert join_controls["outcome_only_rows"] == 0
assert join_controls["locked_prejoin_order_preserved"] is True
assert join_controls["nonkey_source_column_overlap"] == []


# --------------------------------------------------------------------------------------------------
# 19. Verify that performance remained unopened during package construction
# --------------------------------------------------------------------------------------------------

scientific_boundary = freeze_manifest[
    "scientific_boundary"
]

required_false_boundary_fields = [
    "temporal_performance_calculated",
    "auprc_calculated",
    "auroc_calculated",
    "brier_score_calculated",
    "calibration_calculated",
    "risk_enrichment_calculated",
    "subgroup_performance_calculated",
    "threshold_optimization_performed",
    "weight_optimization_performed",
    "frozen_outcome_modified",
    "frozen_comparator_modified",
]

for field in required_false_boundary_fields:

    assert scientific_boundary[field] is False, (
        f"Unexpected scientific-boundary value for {field}: "
        f"{scientific_boundary[field]}"
    )


assert (
    freeze_manifest["authorization_after_freeze"]
    ["performance_analysis_authorized_in_this_cell"]
    is False
)


# --------------------------------------------------------------------------------------------------
# 20. Final verification output
# --------------------------------------------------------------------------------------------------

print("=" * 120)
print(
    "STAGE 6B STEP 5 — CELL 6B-5A — "
    "FRESH FROZEN STAGE 6B PACKAGE VERIFICATION"
)
print("=" * 120)

print("\nCRYPTOGRAPHIC CHECKS")
print("-" * 120)
print("Full accounting cohort SHA-256".ljust(84), "PASS")
print("Full accounting cohort sidecar".ljust(84), "PASS")
print("Primary-evaluable cohort SHA-256".ljust(84), "PASS")
print("Primary-evaluable cohort sidecar".ljust(84), "PASS")
print("Locked-cohort QC SHA-256".ljust(84), "PASS")
print("Locked-cohort QC sidecar".ljust(84), "PASS")
print("Stage 6B freeze-manifest SHA-256".ljust(84), "PASS")
print("Stage 6B freeze-manifest sidecar".ljust(84), "PASS")

print("\nSTRUCTURAL, KEY, AND ORDER CHECKS")
print("-" * 120)
print(
    "Full accounting cohort shape".ljust(84),
    f"PASS ({full_df.shape[0]:,} × {full_df.shape[1]})",
)
print(
    "Primary-evaluable cohort shape".ljust(84),
    f"PASS ({evaluable_df.shape[0]:,} × {evaluable_df.shape[1]})",
)
print(
    "Identical 79-column schemas".ljust(84),
    "PASS",
)
print(
    "Unique full-cohort RCV keys".ljust(84),
    f"PASS ({full_keys.nunique():,})",
)
print(
    "Unique evaluable-cohort RCV keys".ljust(84),
    f"PASS ({evaluable_keys.nunique():,})",
)
print(
    "Full cohort zero-based row order".ljust(84),
    "PASS (0 through 71,658)",
)
print(
    "Evaluable cohort exact ordered subset".ljust(84),
    "PASS",
)

print("\nPOLICY, QC, AND MANIFEST LINEAGE")
print("-" * 120)
print("Comparator-policy lineage".ljust(84), "PASS")
print("Outcome-policy lineage".ljust(84), "PASS")
print(
    "Locked-cohort QC checks".ljust(84),
    "PASS (44 passed / 0 failed)",
)
print("QC artifact identities".ljust(84), "PASS")
print("Freeze-manifest artifact identities".ljust(84), "PASS")
print("Frozen source identities".ljust(84), "PASS")
print("One-to-one join controls".ljust(84), "PASS")

print("\nOUTCOME ACCOUNTING")
print("-" * 120)
print(
    "Full accounting rows".ljust(84),
    f"PASS ({EXPECTED_FULL_ROWS:,})",
)
print(
    "Primary-outcome-evaluable rows".ljust(84),
    f"PASS ({EXPECTED_EVALUABLE_ROWS:,})",
)
print(
    "Primary instability events".ljust(84),
    f"PASS ({EXPECTED_EVENTS:,})",
)
print(
    "Primary negatives".ljust(84),
    f"PASS ({EXPECTED_NEGATIVES:,})",
)
print(
    "Censored or nonevaluable rows".ljust(84),
    f"PASS ({EXPECTED_NONEVALUABLE:,})",
)
print(
    "Evaluable cohort contains no missing outcomes".ljust(84),
    "PASS",
)

print("\nSCIENTIFIC BOUNDARY")
print("-" * 120)
print("Frozen Stage 6B package opened".ljust(84), "YES — READ ONLY")
print("AUPRC or AUROC calculated".ljust(84), "NO")
print("Brier score or calibration calculated".ljust(84), "NO")
print("Risk enrichment calculated".ljust(84), "NO")
print("Subgroup performance calculated".ljust(84), "NO")
print("Threshold or weight optimization performed".ljust(84), "NO")
print("Scientific artifact written or modified".ljust(84), "NO")

print("\nFINAL DECISION")
print("-" * 120)
print(
    "PASS_STAGE6B_FROZEN_PACKAGE_FRESHLY_REVERIFIED"
)
print(
    "The complete Stage 6B locked cohort package passed fresh "
    "cryptographic, structural, accounting, policy-lineage, QC, "
    "manifest, and readback verification."
)
print(
    "The frozen primary-evaluable cohort is ready for the "
    "prespecified locked temporal-performance analysis."
)
print("=" * 120)

STAGE 6B STEP 5 — CELL 6B-5A — FRESH FROZEN STAGE 6B PACKAGE VERIFICATION

CRYPTOGRAPHIC CHECKS
------------------------------------------------------------------------------------------------------------------------
Full accounting cohort SHA-256                                                       PASS
Full accounting cohort sidecar                                                       PASS
Primary-evaluable cohort SHA-256                                                     PASS
Primary-evaluable cohort sidecar                                                     PASS
Locked-cohort QC SHA-256                                                             PASS
Locked-cohort QC sidecar                                                             PASS
Stage 6B freeze-manifest SHA-256                                                     PASS
Stage 6B freeze-manifest sidecar                                                     PASS

STRUCTURAL, KEY, AND ORDER CHECKS
----------------------------

In [10]:
# =================================================================================================
# STAGE 6C STEP 1 — CELL 6C-1A
# FRESH READ-ONLY VERIFICATION AND LOCKED TEMPORAL-ANALYSIS PREFLIGHT
#
# Purpose:
#   1. Mount Google Drive.
#   2. Recalculate SHA-256 hashes for the frozen Stage 6B package.
#   3. Verify every SHA-256 sidecar.
#   4. Load only the frozen primary-evaluable cohort.
#   5. Confirm its shape, key uniqueness, row order, and outcome accounting.
#
# This cell:
#   - DOES NOT calculate AUPRC, AUROC, Brier score, or calibration.
#   - DOES NOT modify, overwrite, or create any scientific artifact.
# =================================================================================================

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import hashlib
import json
import pandas as pd
import pyarrow.parquet as pq


# -------------------------------------------------------------------------------------------------
# 1. Frozen Stage 6B paths
# -------------------------------------------------------------------------------------------------

STUDY_ROOT = Path("/content/drive/MyDrive/GES_RAG_Temporal_Study")

STAGE6_DATA_DIR = (
    STUDY_ROOT
    / "data_processed"
    / "stage6_temporal_validation"
)

STAGE6_QC_DIR = (
    STUDY_ROOT
    / "outputs"
    / "quality_checks"
    / "stage6_temporal_validation"
)

STAGE6_CONFIG_DIR = (
    STUDY_ROOT
    / "configs"
    / "stage6_temporal_validation"
)

FULL_COHORT_PATH = (
    STAGE6_DATA_DIR
    / "stage6b_locked_score_outcome_accounting_cohort_v1.parquet"
)

EVALUABLE_COHORT_PATH = (
    STAGE6_DATA_DIR
    / "stage6b_locked_primary_evaluable_cohort_v1.parquet"
)

QC_PATH = (
    STAGE6_QC_DIR
    / "stage6b_locked_score_outcome_cohort_qc_v1.json"
)

FREEZE_MANIFEST_PATH = (
    STAGE6_CONFIG_DIR
    / "stage6b_locked_score_outcome_cohort_freeze_manifest_v1.json"
)

SIDECAR_PATHS = {
    "full_cohort": Path(str(FULL_COHORT_PATH) + ".sha256"),
    "evaluable_cohort": Path(str(EVALUABLE_COHORT_PATH) + ".sha256"),
    "qc_report": Path(str(QC_PATH) + ".sha256"),
    "freeze_manifest": Path(str(FREEZE_MANIFEST_PATH) + ".sha256"),
}


# -------------------------------------------------------------------------------------------------
# 2. Expected frozen SHA-256 values
# -------------------------------------------------------------------------------------------------

EXPECTED_SHA256 = {
    "full_cohort":
        "8ebf6e43eebaf75c1b30a1785085f8ce924cbbdab02c4648995d83f25426e7d1",

    "evaluable_cohort":
        "c6ad50dfc376a746e1830d000dba975101d88763fd61b8d1983f6300edb6e038",

    "qc_report":
        "ab81bf5653ad147b24fe53a4b63bc1a5c5531d3a986ce19a25969f764b372b14",

    "freeze_manifest":
        "3d549a169be09b7978967cb5bb613d5cf963bf60e4f0c3a95c3fc3be3dac18a4",
}

ARTIFACT_PATHS = {
    "full_cohort": FULL_COHORT_PATH,
    "evaluable_cohort": EVALUABLE_COHORT_PATH,
    "qc_report": QC_PATH,
    "freeze_manifest": FREEZE_MANIFEST_PATH,
}


# -------------------------------------------------------------------------------------------------
# 3. Verification helpers
# -------------------------------------------------------------------------------------------------

def calculate_sha256(path: Path, chunk_size: int = 1024 * 1024) -> str:
    """Calculate SHA-256 without altering the file."""
    digest = hashlib.sha256()

    with path.open("rb") as file_handle:
        while True:
            chunk = file_handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)

    return digest.hexdigest()


def read_sidecar_sha256(path: Path) -> str:
    """
    Read a SHA-256 sidecar.

    Supports either:
        <sha256>
    or:
        <sha256>  <filename>
    """
    text = path.read_text(encoding="utf-8").strip()

    if not text:
        raise AssertionError(f"Empty SHA-256 sidecar: {path}")

    observed_hash = text.split()[0].strip().lower()

    if len(observed_hash) != 64:
        raise AssertionError(
            f"Invalid SHA-256 value in sidecar {path}: {observed_hash!r}"
        )

    return observed_hash


# -------------------------------------------------------------------------------------------------
# 4. Confirm all required files exist
# -------------------------------------------------------------------------------------------------

all_required_paths = list(ARTIFACT_PATHS.values()) + list(SIDECAR_PATHS.values())

missing_paths = [
    str(path)
    for path in all_required_paths
    if not path.is_file()
]

assert not missing_paths, (
    "Required frozen Stage 6B files are missing:\n"
    + "\n".join(missing_paths)
)

print("Required Stage 6B files: PASS")


# -------------------------------------------------------------------------------------------------
# 5. Recalculate artifact hashes and verify sidecars
# -------------------------------------------------------------------------------------------------

verification_rows = []

for artifact_name, artifact_path in ARTIFACT_PATHS.items():

    calculated_hash = calculate_sha256(artifact_path)
    expected_hash = EXPECTED_SHA256[artifact_name]
    sidecar_hash = read_sidecar_sha256(SIDECAR_PATHS[artifact_name])

    expected_match = calculated_hash == expected_hash
    sidecar_match = calculated_hash == sidecar_hash

    verification_rows.append(
        {
            "artifact": artifact_name,
            "calculated_sha256": calculated_hash,
            "expected_hash_match": expected_match,
            "sidecar_hash_match": sidecar_match,
        }
    )

    assert expected_match, (
        f"Frozen checksum mismatch for {artifact_name}:\n"
        f"Expected:   {expected_hash}\n"
        f"Calculated: {calculated_hash}"
    )

    assert sidecar_match, (
        f"Sidecar mismatch for {artifact_name}:\n"
        f"Sidecar:    {sidecar_hash}\n"
        f"Calculated: {calculated_hash}"
    )

verification_df = pd.DataFrame(verification_rows)

print("\nCryptographic verification:")
display(
    verification_df[
        [
            "artifact",
            "expected_hash_match",
            "sidecar_hash_match",
            "calculated_sha256",
        ]
    ]
)


# -------------------------------------------------------------------------------------------------
# 6. Inspect Parquet metadata before loading the evaluable cohort
# -------------------------------------------------------------------------------------------------

evaluable_metadata = pq.ParquetFile(EVALUABLE_COHORT_PATH).metadata

assert evaluable_metadata.num_rows == 66_636, (
    f"Unexpected evaluable row count: {evaluable_metadata.num_rows:,}"
)

assert evaluable_metadata.num_columns == 79, (
    f"Unexpected evaluable column count: {evaluable_metadata.num_columns}"
)

print("\nPrimary-evaluable Parquet metadata: PASS")
print(f"Rows:    {evaluable_metadata.num_rows:,}")
print(f"Columns: {evaluable_metadata.num_columns}")


# -------------------------------------------------------------------------------------------------
# 7. Load the immutable primary-evaluable cohort
# -------------------------------------------------------------------------------------------------

locked_evaluable_df = pd.read_parquet(EVALUABLE_COHORT_PATH)

assert locked_evaluable_df.shape == (66_636, 79), (
    f"Unexpected dataframe shape: {locked_evaluable_df.shape}"
)

required_columns = {
    "rcv_accession",
    "t0_row_order",
    "primary_outcome_evaluable",
    "primary_future_instability",
}

missing_required_columns = sorted(
    required_columns - set(locked_evaluable_df.columns)
)

assert not missing_required_columns, (
    "Required analysis columns are missing: "
    + ", ".join(missing_required_columns)
)


# -------------------------------------------------------------------------------------------------
# 8. Key, row-order, and outcome checks
# -------------------------------------------------------------------------------------------------

assert locked_evaluable_df["rcv_accession"].notna().all()
assert locked_evaluable_df["rcv_accession"].is_unique

row_order = pd.to_numeric(
    locked_evaluable_df["t0_row_order"],
    errors="raise"
)

assert row_order.notna().all()
assert row_order.is_unique
assert row_order.is_monotonic_increasing

evaluable_flag = locked_evaluable_df["primary_outcome_evaluable"]

assert evaluable_flag.notna().all()
assert evaluable_flag.astype(bool).all()

locked_outcome = pd.to_numeric(
    locked_evaluable_df["primary_future_instability"],
    errors="raise"
)

assert locked_outcome.notna().all()
assert set(locked_outcome.unique()) == {0, 1}

event_count = int((locked_outcome == 1).sum())
negative_count = int((locked_outcome == 0).sum())

assert event_count == 6_485, (
    f"Unexpected event count: {event_count:,}"
)

assert negative_count == 60_151, (
    f"Unexpected negative count: {negative_count:,}"
)

assert event_count + negative_count == len(locked_evaluable_df)

event_prevalence = event_count / len(locked_evaluable_df)


# -------------------------------------------------------------------------------------------------
# 9. Read frozen QC and manifest files without changing them
# -------------------------------------------------------------------------------------------------

with QC_PATH.open("r", encoding="utf-8") as file_handle:
    stage6b_qc = json.load(file_handle)

with FREEZE_MANIFEST_PATH.open("r", encoding="utf-8") as file_handle:
    stage6b_freeze_manifest = json.load(file_handle)

assert isinstance(stage6b_qc, dict)
assert isinstance(stage6b_freeze_manifest, dict)


# -------------------------------------------------------------------------------------------------
# 10. Inventory likely score/comparator columns for the next cell
# -------------------------------------------------------------------------------------------------

score_name_tokens = (
    "ges",
    "stable",
    "risk",
    "score",
    "star",
    "conflict",
    "recency",
    "submitter",
    "entropy",
    "additive",
    "combined",
)

candidate_score_columns = [
    column
    for column in locked_evaluable_df.columns
    if any(token in column.lower() for token in score_name_tokens)
]

candidate_score_columns = sorted(candidate_score_columns)


# -------------------------------------------------------------------------------------------------
# 11. Final preflight report
# -------------------------------------------------------------------------------------------------

print("\n" + "=" * 108)
print("STAGE 6C STEP 1 — CELL 6C-1A — LOCKED TEMPORAL-ANALYSIS PREFLIGHT")
print("=" * 108)

print("\nCRYPTOGRAPHIC PACKAGE CHECKS")
print("-" * 108)
print("Full accounting cohort SHA-256".ljust(72), "PASS")
print("Primary-evaluable cohort SHA-256".ljust(72), "PASS")
print("Locked-cohort QC SHA-256".ljust(72), "PASS")
print("Stage 6B freeze-manifest SHA-256".ljust(72), "PASS")
print("All SHA-256 sidecars".ljust(72), "PASS")

print("\nPRIMARY-EVALUABLE COHORT CHECKS")
print("-" * 108)
print("Shape".ljust(72), f"PASS ({locked_evaluable_df.shape[0]:,} × {locked_evaluable_df.shape[1]})")
print("Unique RCV keys".ljust(72), f"PASS ({locked_evaluable_df['rcv_accession'].nunique():,})")
print("T0 row order unique and increasing".ljust(72), "PASS")
print("Missing primary outcomes".ljust(72), f"PASS ({int(locked_outcome.isna().sum()):,})")
print("Primary instability events".ljust(72), f"PASS ({event_count:,})")
print("Primary instability negatives".ljust(72), f"PASS ({negative_count:,})")
print("Observed event prevalence".ljust(72), f"{event_prevalence:.6%}")

print("\nLIKELY SCORE / COMPARATOR COLUMNS")
print("-" * 108)

for column in candidate_score_columns:
    print(column)

print("\nANALYSIS BOUNDARY")
print("-" * 108)
print("Temporal performance calculated".ljust(72), "NO")
print("Scientific artifacts created or modified".ljust(72), "NO")
print("Locked evaluable dataframe available as".ljust(72), "locked_evaluable_df")
print("Locked binary outcome available as".ljust(72), "locked_outcome")

print("\nFINAL DECISION")
print("-" * 108)
print("PASS_STAGE6C_LOCKED_TEMPORAL_ANALYSIS_PREFLIGHT")
print("=" * 108)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Required Stage 6B files: PASS

Cryptographic verification:


,artifact,expected_hash_match,sidecar_hash_match,calculated_sha256
0,full_cohort,True,True,8ebf6e43eebaf75c1b30a1785085f8ce924cbbdab02c46...
1,evaluable_cohort,True,True,c6ad50dfc376a746e1830d000dba975101d88763fd61b8...
2,qc_report,True,True,ab81bf5653ad147b24fe53a4b63bc1a5c5531d3a986ce1...
3,freeze_manifest,True,True,3d549a169be09b7978967cb5bb613d5cf963bf60e4f0c3...



Primary-evaluable Parquet metadata: PASS
Rows:    66,636
Columns: 79

STAGE 6C STEP 1 — CELL 6C-1A — LOCKED TEMPORAL-ANALYSIS PREFLIGHT

CRYPTOGRAPHIC PACKAGE CHECKS
------------------------------------------------------------------------------------------------------------
Full accounting cohort SHA-256                                           PASS
Primary-evaluable cohort SHA-256                                         PASS
Locked-cohort QC SHA-256                                                 PASS
Stage 6B freeze-manifest SHA-256                                         PASS
All SHA-256 sidecars                                                     PASS

PRIMARY-EVALUABLE COHORT CHECKS
------------------------------------------------------------------------------------------------------------
Shape                                                                    PASS (66,636 × 79)
Unique RCV keys                                                          PASS (66,636)
T0 row order 

In [11]:
# =================================================================================================
# STAGE 6C STEP 1 — CELL 6C-1B
# LOCKED PRIMARY DISCRIMINATION POINT ESTIMATES
#
# Purpose:
#   1. Calculate the prespecified primary discrimination metric: AUPRC.
#   2. Calculate AUROC as a secondary discrimination metric.
#   3. Evaluate all frozen prespecified comparator scores.
#   4. Report point estimates only.
#
# Important:
#   - AUPRC is implemented using sklearn.metrics.average_precision_score.
#   - Every score is already oriented so that a higher value means greater instability risk.
#   - No threshold selection, optimization, bootstrap inference, calibration analysis,
#     subgroup analysis, or artifact writing occurs in this cell.
# =================================================================================================

import numpy as np
import pandas as pd

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
)


# -------------------------------------------------------------------------------------------------
# 1. Confirm required objects from Cell 6C-1A remain available
# -------------------------------------------------------------------------------------------------

assert "locked_evaluable_df" in globals(), (
    "locked_evaluable_df is unavailable. Run Cell 6C-1A first."
)

assert "locked_outcome" in globals(), (
    "locked_outcome is unavailable. Run Cell 6C-1A first."
)

assert len(locked_evaluable_df) == 66_636
assert len(locked_outcome) == 66_636
assert locked_outcome.notna().all()
assert set(pd.unique(locked_outcome)) == {0, 1}

y_true = locked_outcome.astype(int).to_numpy()

event_count = int(y_true.sum())
negative_count = int(len(y_true) - event_count)
event_prevalence = float(y_true.mean())

assert event_count == 6_485
assert negative_count == 60_151


# -------------------------------------------------------------------------------------------------
# 2. Prespecified frozen score inventory
#
# All selected columns use the frozen higher-score = greater-instability-risk orientation.
# -------------------------------------------------------------------------------------------------

PRESPECIFIED_DISCRIMINATION_SCORES = {
    "Full GES":
        "full_ges_instability_risk_t0",

    "No-star GES":
        "no_star_ges_instability_risk_t0",

    "Review stars":
        "review_stars_instability_risk",

    "Conflict":
        "conflict_instability_risk",

    "Recency":
        "recency_instability_risk",

    "Submitter support":
        "submitter_instability_risk",

    "Classification entropy":
        "entropy_instability_risk",

    "Additive risk":
        "additive_instability_risk",

    "Combined metadata":
        "combined_metadata_instability_risk",
}

missing_score_columns = [
    column_name
    for column_name in PRESPECIFIED_DISCRIMINATION_SCORES.values()
    if column_name not in locked_evaluable_df.columns
]

assert not missing_score_columns, (
    "Required frozen score columns are missing:\n"
    + "\n".join(missing_score_columns)
)


# -------------------------------------------------------------------------------------------------
# 3. Validate each score before calculating performance
# -------------------------------------------------------------------------------------------------

validation_rows = []
performance_rows = []

for model_name, score_column in PRESPECIFIED_DISCRIMINATION_SCORES.items():

    score_series = pd.to_numeric(
        locked_evaluable_df[score_column],
        errors="coerce",
    )

    missing_count = int(score_series.isna().sum())
    nonfinite_count = int(
        (~np.isfinite(score_series.to_numpy(dtype=float))).sum()
    )

    score_min = float(score_series.min())
    score_max = float(score_series.max())
    unique_score_count = int(score_series.nunique(dropna=True))

    within_unit_interval = bool(
        score_series.between(0.0, 1.0, inclusive="both").all()
    )

    validation_rows.append(
        {
            "model": model_name,
            "score_column": score_column,
            "rows": int(len(score_series)),
            "missing_scores": missing_count,
            "nonfinite_scores": nonfinite_count,
            "minimum_score": score_min,
            "maximum_score": score_max,
            "unique_scores": unique_score_count,
            "within_0_1": within_unit_interval,
        }
    )

    assert missing_count == 0, (
        f"{model_name} contains {missing_count:,} missing scores."
    )

    assert nonfinite_count == 0, (
        f"{model_name} contains {nonfinite_count:,} nonfinite scores."
    )

    assert within_unit_interval, (
        f"{model_name} contains scores outside [0, 1]: "
        f"minimum={score_min}, maximum={score_max}"
    )

    assert unique_score_count >= 2, (
        f"{model_name} contains fewer than two unique score values."
    )

    y_score = score_series.to_numpy(dtype=float)

    auprc = float(
        average_precision_score(
            y_true=y_true,
            y_score=y_score,
        )
    )

    auroc = float(
        roc_auc_score(
            y_true=y_true,
            y_score=y_score,
        )
    )

    performance_rows.append(
        {
            "model": model_name,
            "score_column": score_column,
            "n": int(len(y_true)),
            "events": event_count,
            "negatives": negative_count,
            "event_prevalence": event_prevalence,
            "auprc_average_precision": auprc,
            "auprc_absolute_gain_over_prevalence": auprc - event_prevalence,
            "auprc_lift_over_prevalence": auprc / event_prevalence,
            "auroc": auroc,
        }
    )


# -------------------------------------------------------------------------------------------------
# 4. Materialize in-memory validation and performance tables
# -------------------------------------------------------------------------------------------------

stage6c_score_validation_df = pd.DataFrame(validation_rows)

stage6c_discrimination_point_estimates_df = pd.DataFrame(
    performance_rows
).sort_values(
    by=[
        "auprc_average_precision",
        "auroc",
    ],
    ascending=[
        False,
        False,
    ],
    kind="stable",
).reset_index(drop=True)

stage6c_discrimination_point_estimates_df.insert(
    0,
    "auprc_rank",
    np.arange(
        1,
        len(stage6c_discrimination_point_estimates_df) + 1,
        dtype=int,
    ),
)


# -------------------------------------------------------------------------------------------------
# 5. Add descriptive differences from the two strongest prespecified baseline references
#
# These are point-estimate differences only. Statistical uncertainty will be handled later
# using paired bootstrap analysis.
# -------------------------------------------------------------------------------------------------

review_star_auprc = float(
    stage6c_discrimination_point_estimates_df.loc[
        stage6c_discrimination_point_estimates_df["model"] == "Review stars",
        "auprc_average_precision",
    ].iloc[0]
)

combined_metadata_auprc = float(
    stage6c_discrimination_point_estimates_df.loc[
        stage6c_discrimination_point_estimates_df["model"] == "Combined metadata",
        "auprc_average_precision",
    ].iloc[0]
)

review_star_auroc = float(
    stage6c_discrimination_point_estimates_df.loc[
        stage6c_discrimination_point_estimates_df["model"] == "Review stars",
        "auroc",
    ].iloc[0]
)

combined_metadata_auroc = float(
    stage6c_discrimination_point_estimates_df.loc[
        stage6c_discrimination_point_estimates_df["model"] == "Combined metadata",
        "auroc",
    ].iloc[0]
)

stage6c_discrimination_point_estimates_df[
    "auprc_difference_vs_review_stars"
] = (
    stage6c_discrimination_point_estimates_df["auprc_average_precision"]
    - review_star_auprc
)

stage6c_discrimination_point_estimates_df[
    "auprc_difference_vs_combined_metadata"
] = (
    stage6c_discrimination_point_estimates_df["auprc_average_precision"]
    - combined_metadata_auprc
)

stage6c_discrimination_point_estimates_df[
    "auroc_difference_vs_review_stars"
] = (
    stage6c_discrimination_point_estimates_df["auroc"]
    - review_star_auroc
)

stage6c_discrimination_point_estimates_df[
    "auroc_difference_vs_combined_metadata"
] = (
    stage6c_discrimination_point_estimates_df["auroc"]
    - combined_metadata_auroc
)


# -------------------------------------------------------------------------------------------------
# 6. Display score validation
# -------------------------------------------------------------------------------------------------

print("\nFROZEN SCORE VALIDATION")
print("-" * 120)

display(
    stage6c_score_validation_df[
        [
            "model",
            "rows",
            "missing_scores",
            "nonfinite_scores",
            "minimum_score",
            "maximum_score",
            "unique_scores",
            "within_0_1",
        ]
    ]
)


# -------------------------------------------------------------------------------------------------
# 7. Display locked discrimination point estimates
# -------------------------------------------------------------------------------------------------

display_columns = [
    "auprc_rank",
    "model",
    "n",
    "events",
    "event_prevalence",
    "auprc_average_precision",
    "auprc_lift_over_prevalence",
    "auroc",
    "auprc_difference_vs_review_stars",
    "auprc_difference_vs_combined_metadata",
]

display_table = (
    stage6c_discrimination_point_estimates_df[display_columns]
    .copy()
)

for column_name in [
    "event_prevalence",
    "auprc_average_precision",
    "auprc_lift_over_prevalence",
    "auroc",
    "auprc_difference_vs_review_stars",
    "auprc_difference_vs_combined_metadata",
]:
    display_table[column_name] = display_table[column_name].round(6)

print("\nLOCKED PRIMARY DISCRIMINATION POINT ESTIMATES")
print("-" * 120)

display(display_table)


# -------------------------------------------------------------------------------------------------
# 8. Extract key GES results for the final report
# -------------------------------------------------------------------------------------------------

full_ges_result = (
    stage6c_discrimination_point_estimates_df
    .loc[
        stage6c_discrimination_point_estimates_df["model"] == "Full GES"
    ]
    .iloc[0]
)

no_star_ges_result = (
    stage6c_discrimination_point_estimates_df
    .loc[
        stage6c_discrimination_point_estimates_df["model"] == "No-star GES"
    ]
    .iloc[0]
)


# -------------------------------------------------------------------------------------------------
# 9. Final locked-analysis report
# -------------------------------------------------------------------------------------------------

print("\n" + "=" * 120)
print("STAGE 6C STEP 1 — CELL 6C-1B — LOCKED PRIMARY DISCRIMINATION POINT ESTIMATES")
print("=" * 120)

print("\nANALYSIS COHORT")
print("-" * 120)
print("Primary-evaluable records".ljust(78), f"{len(y_true):,}")
print("Primary instability events".ljust(78), f"{event_count:,}")
print("Primary instability negatives".ljust(78), f"{negative_count:,}")
print("Observed event prevalence".ljust(78), f"{event_prevalence:.6%}")

print("\nPRIMARY FULL-GES RESULT")
print("-" * 120)
print(
    "AUPRC — average precision".ljust(78),
    f"{full_ges_result['auprc_average_precision']:.8f}",
)
print(
    "AUPRC prevalence baseline".ljust(78),
    f"{event_prevalence:.8f}",
)
print(
    "AUPRC lift over prevalence".ljust(78),
    f"{full_ges_result['auprc_lift_over_prevalence']:.6f}×",
)
print(
    "AUROC".ljust(78),
    f"{full_ges_result['auroc']:.8f}",
)
print(
    "AUPRC difference versus review stars".ljust(78),
    f"{full_ges_result['auprc_difference_vs_review_stars']:+.8f}",
)
print(
    "AUPRC difference versus combined metadata".ljust(78),
    f"{full_ges_result['auprc_difference_vs_combined_metadata']:+.8f}",
)

print("\nNO-STAR GES RESULT")
print("-" * 120)
print(
    "AUPRC — average precision".ljust(78),
    f"{no_star_ges_result['auprc_average_precision']:.8f}",
)
print(
    "AUROC".ljust(78),
    f"{no_star_ges_result['auroc']:.8f}",
)
print(
    "AUPRC difference versus review stars".ljust(78),
    f"{no_star_ges_result['auprc_difference_vs_review_stars']:+.8f}",
)
print(
    "AUPRC difference versus combined metadata".ljust(78),
    f"{no_star_ges_result['auprc_difference_vs_combined_metadata']:+.8f}",
)

print("\nANALYSIS CONTROLS")
print("-" * 120)
print("Frozen score direction used".ljust(78), "Higher score = greater instability risk")
print("Threshold selected or optimized".ljust(78), "NO")
print("Bootstrap inference performed".ljust(78), "NO")
print("Calibration analysis performed".ljust(78), "NO")
print("Subgroup or sensitivity analysis performed".ljust(78), "NO")
print("Scientific artifact written or modified".ljust(78), "NO")

print("\nIN-MEMORY OUTPUTS")
print("-" * 120)
print(
    "Score-validation table".ljust(78),
    "stage6c_score_validation_df",
)
print(
    "Discrimination-results table".ljust(78),
    "stage6c_discrimination_point_estimates_df",
)

print("\nFINAL DECISION")
print("-" * 120)
print("PASS_STAGE6C_LOCKED_DISCRIMINATION_POINT_ESTIMATES_COMPLETE")
print("=" * 120)


FROZEN SCORE VALIDATION
------------------------------------------------------------------------------------------------------------------------


,model,rows,missing_scores,nonfinite_scores,minimum_score,maximum_score,unique_scores,within_0_1
0,Full GES,66636,0,0,3.385514e-12,1.000000,9504,True
1,No-star GES,66636,0,0,1.700875e-08,1.000000,9021,True
2,Review stars,66636,0,0,0.000000e+00,1.000000,4,True
3,Conflict,66636,0,0,0.000000e+00,1.000000,2,True
4,Recency,66636,0,0,7.607455e-04,1.000000,3783,True
5,Submitter support,66636,0,0,0.000000e+00,1.000000,22,True
6,Classification entropy,66636,0,0,0.000000e+00,1.000000,39,True
7,Additive risk,66636,0,0,0.000000e+00,0.500000,4,True
8,Combined metadata,66636,0,0,3.873463e-02,0.769716,9504,True



LOCKED PRIMARY DISCRIMINATION POINT ESTIMATES
------------------------------------------------------------------------------------------------------------------------


,auprc_rank,model,n,events,event_prevalence,auprc_average_precision,auprc_lift_over_prevalence,auroc,auprc_difference_vs_review_stars,auprc_difference_vs_combined_metadata
0,1,Combined metadata,66636,6485,0.09732,0.113503,1.166293,0.532614,0.005199,0.000000
1,2,Full GES,66636,6485,0.09732,0.112444,1.155408,0.535812,0.004140,-0.001059
2,3,Review stars,66636,6485,0.09732,0.108304,1.112872,0.533391,0.000000,-0.005199
3,4,Classification entropy,66636,6485,0.09732,0.106189,1.091139,0.518451,-0.002115,-0.007314
4,5,Conflict,66636,6485,0.09732,0.102047,1.048575,0.513057,-0.006257,-0.011456
5,6,Additive risk,66636,6485,0.09732,0.099511,1.022519,0.483180,-0.008793,-0.013992
6,7,No-star GES,66636,6485,0.09732,0.096437,0.990928,0.447722,-0.011868,-0.017067
7,8,Submitter support,66636,6485,0.09732,0.092424,0.949689,0.471184,-0.015881,-0.021080
8,9,Recency,66636,6485,0.09732,0.082721,0.849995,0.437694,-0.025583,-0.030782



STAGE 6C STEP 1 — CELL 6C-1B — LOCKED PRIMARY DISCRIMINATION POINT ESTIMATES

ANALYSIS COHORT
------------------------------------------------------------------------------------------------------------------------
Primary-evaluable records                                                      66,636
Primary instability events                                                     6,485
Primary instability negatives                                                  60,151
Observed event prevalence                                                      9.731977%

PRIMARY FULL-GES RESULT
------------------------------------------------------------------------------------------------------------------------
AUPRC — average precision                                                      0.11244406
AUPRC prevalence baseline                                                      0.09731977
AUPRC lift over prevalence                                                     1.155408×
AUROC                  

In [12]:
# =================================================================================================
# STAGE 6C STEP 1 — CELL 6C-1C
# LOCKED PRIMARY AUPRC BOOTSTRAP INFERENCE
#
# Purpose:
#   1. Generate 2,000 paired nonparametric bootstrap samples.
#   2. Estimate 95% percentile confidence intervals for primary AUPRC results.
#   3. Estimate paired AUPRC differences:
#        - Full GES versus review stars
#        - Full GES versus combined metadata
#        - Full GES versus no-star GES
#        - No-star GES versus review stars
#
# Controls:
#   - Uses the same resampled records for every score within each replicate.
#   - Uses the frozen random seed of 42.
#   - Performs no threshold tuning or score modification.
#   - Does not write or modify any scientific artifact.
# =================================================================================================

import os
import time
import numpy as np
import pandas as pd

from joblib import Parallel, delayed
from sklearn.metrics import average_precision_score


# -------------------------------------------------------------------------------------------------
# 1. Confirm locked inputs remain available
# -------------------------------------------------------------------------------------------------

assert "locked_evaluable_df" in globals(), (
    "locked_evaluable_df is unavailable. Run Cell 6C-1A first."
)

assert "locked_outcome" in globals(), (
    "locked_outcome is unavailable. Run Cell 6C-1A first."
)

y_true = pd.to_numeric(
    locked_outcome,
    errors="raise",
).astype(int).to_numpy()

assert len(y_true) == 66_636
assert set(np.unique(y_true)) == {0, 1}
assert int(y_true.sum()) == 6_485


# -------------------------------------------------------------------------------------------------
# 2. Frozen primary score set
# -------------------------------------------------------------------------------------------------

BOOTSTRAP_SCORE_COLUMNS = {
    "Full GES":
        "full_ges_instability_risk_t0",

    "Combined metadata":
        "combined_metadata_instability_risk",

    "Review stars":
        "review_stars_instability_risk",

    "No-star GES":
        "no_star_ges_instability_risk_t0",
}

missing_columns = [
    column
    for column in BOOTSTRAP_SCORE_COLUMNS.values()
    if column not in locked_evaluable_df.columns
]

assert not missing_columns, (
    "Required frozen score columns are missing:\n"
    + "\n".join(missing_columns)
)

model_names = list(BOOTSTRAP_SCORE_COLUMNS.keys())

score_matrix = np.column_stack(
    [
        pd.to_numeric(
            locked_evaluable_df[column],
            errors="raise",
        ).to_numpy(dtype=np.float64)
        for column in BOOTSTRAP_SCORE_COLUMNS.values()
    ]
)

assert score_matrix.shape == (66_636, 4)
assert np.isfinite(score_matrix).all()
assert ((score_matrix >= 0.0) & (score_matrix <= 1.0)).all()


# -------------------------------------------------------------------------------------------------
# 3. Recalculate locked point estimates
# -------------------------------------------------------------------------------------------------

point_estimates = np.array(
    [
        average_precision_score(
            y_true,
            score_matrix[:, model_index],
        )
        for model_index in range(score_matrix.shape[1])
    ],
    dtype=np.float64,
)

point_estimate_map = dict(
    zip(
        model_names,
        point_estimates,
    )
)

event_prevalence = float(y_true.mean())


# -------------------------------------------------------------------------------------------------
# 4. Reproducible paired bootstrap setup
# -------------------------------------------------------------------------------------------------

N_BOOTSTRAP = 2_000
BOOTSTRAP_SEED = 42

available_cpus = os.cpu_count() or 1
n_jobs = max(1, min(4, available_cpus))

seed_generator = np.random.SeedSequence(BOOTSTRAP_SEED)

bootstrap_seeds = seed_generator.generate_state(
    N_BOOTSTRAP,
    dtype=np.uint64,
)

# Divide the work into a modest number of chunks to reduce parallel overhead.
number_of_chunks = min(
    N_BOOTSTRAP,
    max(n_jobs * 8, 1),
)

seed_chunks = [
    chunk
    for chunk in np.array_split(
        bootstrap_seeds,
        number_of_chunks,
    )
    if len(chunk) > 0
]


# -------------------------------------------------------------------------------------------------
# 5. Paired bootstrap worker
# -------------------------------------------------------------------------------------------------

def run_bootstrap_chunk(seed_values):
    """
    Run multiple paired bootstrap replicates.

    Each replicate samples complete cohort rows with replacement.
    The identical sampled row indexes are applied to all four models.
    """

    chunk_results = np.empty(
        (
            len(seed_values),
            score_matrix.shape[1],
        ),
        dtype=np.float64,
    )

    cohort_size = len(y_true)

    for replicate_index, replicate_seed in enumerate(seed_values):

        rng = np.random.default_rng(
            int(replicate_seed)
        )

        sampled_indexes = rng.integers(
            low=0,
            high=cohort_size,
            size=cohort_size,
            dtype=np.int32,
        )

        sampled_outcome = y_true[sampled_indexes]

        # This is extraordinarily unlikely with this event count,
        # but the check protects metric validity.
        if np.unique(sampled_outcome).size != 2:
            raise RuntimeError(
                "A bootstrap replicate contained only one outcome class."
            )

        sampled_scores = score_matrix[sampled_indexes, :]

        for model_index in range(sampled_scores.shape[1]):

            chunk_results[
                replicate_index,
                model_index,
            ] = average_precision_score(
                sampled_outcome,
                sampled_scores[:, model_index],
            )

    return chunk_results


# -------------------------------------------------------------------------------------------------
# 6. Execute the 2,000 paired bootstrap replicates
# -------------------------------------------------------------------------------------------------

print("=" * 112)
print("STARTING 2,000 PAIRED BOOTSTRAP REPLICATES")
print("=" * 112)
print(f"Rows per replicate: {len(y_true):,}")
print(f"Models per replicate: {len(model_names)}")
print(f"Parallel workers: {n_jobs}")
print(f"Random seed: {BOOTSTRAP_SEED}")
print("No files will be written.")
print("-" * 112)

bootstrap_start_time = time.time()

bootstrap_chunk_results = Parallel(
    n_jobs=n_jobs,
    backend="loky",
    verbose=10,
)(
    delayed(run_bootstrap_chunk)(seed_chunk)
    for seed_chunk in seed_chunks
)

bootstrap_matrix = np.vstack(
    bootstrap_chunk_results
)

bootstrap_elapsed_seconds = (
    time.time() - bootstrap_start_time
)

assert bootstrap_matrix.shape == (
    N_BOOTSTRAP,
    len(model_names),
)

assert np.isfinite(bootstrap_matrix).all()

print("\nBootstrap execution: PASS")
print(
    f"Elapsed time: "
    f"{bootstrap_elapsed_seconds / 60.0:.2f} minutes"
)


# -------------------------------------------------------------------------------------------------
# 7. Create the in-memory replicate table
# -------------------------------------------------------------------------------------------------

stage6c_primary_auprc_bootstrap_replicates_df = pd.DataFrame(
    bootstrap_matrix,
    columns=model_names,
)

stage6c_primary_auprc_bootstrap_replicates_df.insert(
    0,
    "bootstrap_replicate",
    np.arange(
        1,
        N_BOOTSTRAP + 1,
        dtype=int,
    ),
)


# -------------------------------------------------------------------------------------------------
# 8. Model-specific percentile confidence intervals
# -------------------------------------------------------------------------------------------------

model_summary_rows = []

for model_index, model_name in enumerate(model_names):

    bootstrap_values = bootstrap_matrix[
        :,
        model_index,
    ]

    lower_ci, upper_ci = np.percentile(
        bootstrap_values,
        [2.5, 97.5],
    )

    model_summary_rows.append(
        {
            "model": model_name,
            "n": int(len(y_true)),
            "events": int(y_true.sum()),
            "event_prevalence": event_prevalence,
            "auprc_point_estimate": float(
                point_estimates[model_index]
            ),
            "bootstrap_mean_auprc": float(
                bootstrap_values.mean()
            ),
            "bootstrap_standard_error": float(
                bootstrap_values.std(ddof=1)
            ),
            "auprc_ci_2_5_percent": float(
                lower_ci
            ),
            "auprc_ci_97_5_percent": float(
                upper_ci
            ),
            "bootstrap_replicates": N_BOOTSTRAP,
            "random_seed": BOOTSTRAP_SEED,
        }
    )

stage6c_primary_auprc_bootstrap_summary_df = pd.DataFrame(
    model_summary_rows
).sort_values(
    by="auprc_point_estimate",
    ascending=False,
    kind="stable",
).reset_index(drop=True)


# -------------------------------------------------------------------------------------------------
# 9. Paired AUPRC difference confidence intervals
# -------------------------------------------------------------------------------------------------

model_index_map = {
    model_name: model_index
    for model_index, model_name in enumerate(model_names)
}

PAIRED_COMPARISONS = [
    (
        "Full GES",
        "Review stars",
    ),
    (
        "Full GES",
        "Combined metadata",
    ),
    (
        "Full GES",
        "No-star GES",
    ),
    (
        "No-star GES",
        "Review stars",
    ),
]

comparison_rows = []

for model_a, model_b in PAIRED_COMPARISONS:

    model_a_index = model_index_map[model_a]
    model_b_index = model_index_map[model_b]

    bootstrap_difference = (
        bootstrap_matrix[:, model_a_index]
        - bootstrap_matrix[:, model_b_index]
    )

    point_difference = (
        point_estimate_map[model_a]
        - point_estimate_map[model_b]
    )

    lower_ci, upper_ci = np.percentile(
        bootstrap_difference,
        [2.5, 97.5],
    )

    probability_positive = float(
        np.mean(
            bootstrap_difference > 0.0
        )
    )

    probability_negative = float(
        np.mean(
            bootstrap_difference < 0.0
        )
    )

    comparison_rows.append(
        {
            "comparison": f"{model_a} minus {model_b}",
            "model_a": model_a,
            "model_b": model_b,
            "paired_auprc_difference": float(
                point_difference
            ),
            "bootstrap_mean_difference": float(
                bootstrap_difference.mean()
            ),
            "bootstrap_standard_error": float(
                bootstrap_difference.std(ddof=1)
            ),
            "difference_ci_2_5_percent": float(
                lower_ci
            ),
            "difference_ci_97_5_percent": float(
                upper_ci
            ),
            "bootstrap_probability_difference_gt_zero":
                probability_positive,
            "bootstrap_probability_difference_lt_zero":
                probability_negative,
            "percentile_ci_excludes_zero": bool(
                (lower_ci > 0.0)
                or
                (upper_ci < 0.0)
            ),
            "bootstrap_replicates": N_BOOTSTRAP,
        }
    )

stage6c_primary_auprc_paired_comparisons_df = pd.DataFrame(
    comparison_rows
)


# -------------------------------------------------------------------------------------------------
# 10. Display model-specific bootstrap results
# -------------------------------------------------------------------------------------------------

model_display_df = (
    stage6c_primary_auprc_bootstrap_summary_df[
        [
            "model",
            "auprc_point_estimate",
            "bootstrap_mean_auprc",
            "bootstrap_standard_error",
            "auprc_ci_2_5_percent",
            "auprc_ci_97_5_percent",
            "bootstrap_replicates",
        ]
    ]
    .copy()
)

for column in [
    "auprc_point_estimate",
    "bootstrap_mean_auprc",
    "bootstrap_standard_error",
    "auprc_ci_2_5_percent",
    "auprc_ci_97_5_percent",
]:
    model_display_df[column] = (
        model_display_df[column]
        .round(8)
    )

print("\nPRIMARY AUPRC BOOTSTRAP CONFIDENCE INTERVALS")
print("-" * 112)

display(model_display_df)


# -------------------------------------------------------------------------------------------------
# 11. Display paired comparisons
# -------------------------------------------------------------------------------------------------

comparison_display_df = (
    stage6c_primary_auprc_paired_comparisons_df[
        [
            "comparison",
            "paired_auprc_difference",
            "bootstrap_standard_error",
            "difference_ci_2_5_percent",
            "difference_ci_97_5_percent",
            "bootstrap_probability_difference_gt_zero",
            "percentile_ci_excludes_zero",
        ]
    ]
    .copy()
)

for column in [
    "paired_auprc_difference",
    "bootstrap_standard_error",
    "difference_ci_2_5_percent",
    "difference_ci_97_5_percent",
    "bootstrap_probability_difference_gt_zero",
]:
    comparison_display_df[column] = (
        comparison_display_df[column]
        .round(8)
    )

print("\nPAIRED AUPRC DIFFERENCE BOOTSTRAP RESULTS")
print("-" * 112)

display(comparison_display_df)


# -------------------------------------------------------------------------------------------------
# 12. Final audit report
# -------------------------------------------------------------------------------------------------

full_vs_stars = (
    stage6c_primary_auprc_paired_comparisons_df
    .loc[
        stage6c_primary_auprc_paired_comparisons_df[
            "comparison"
        ] == "Full GES minus Review stars"
    ]
    .iloc[0]
)

full_vs_combined = (
    stage6c_primary_auprc_paired_comparisons_df
    .loc[
        stage6c_primary_auprc_paired_comparisons_df[
            "comparison"
        ] == "Full GES minus Combined metadata"
    ]
    .iloc[0]
)

print("\n" + "=" * 112)
print("STAGE 6C STEP 1 — CELL 6C-1C — LOCKED PRIMARY AUPRC BOOTSTRAP INFERENCE")
print("=" * 112)

print("\nBOOTSTRAP DESIGN")
print("-" * 112)
print(
    "Bootstrap method".ljust(76),
    "Paired nonparametric row bootstrap",
)
print(
    "Bootstrap replicates".ljust(76),
    f"{N_BOOTSTRAP:,}",
)
print(
    "Random seed".ljust(76),
    BOOTSTRAP_SEED,
)
print(
    "Confidence interval method".ljust(76),
    "2.5th–97.5th percentile",
)
print(
    "Identical resample used across models".ljust(76),
    "YES",
)

print("\nFULL GES VERSUS REVIEW STARS")
print("-" * 112)
print(
    "Point AUPRC difference".ljust(76),
    f"{full_vs_stars['paired_auprc_difference']:+.8f}",
)
print(
    "95% bootstrap interval".ljust(76),
    (
        f"[{full_vs_stars['difference_ci_2_5_percent']:+.8f}, "
        f"{full_vs_stars['difference_ci_97_5_percent']:+.8f}]"
    ),
)
print(
    "Percentile interval excludes zero".ljust(76),
    full_vs_stars["percentile_ci_excludes_zero"],
)

print("\nFULL GES VERSUS COMBINED METADATA")
print("-" * 112)
print(
    "Point AUPRC difference".ljust(76),
    f"{full_vs_combined['paired_auprc_difference']:+.8f}",
)
print(
    "95% bootstrap interval".ljust(76),
    (
        f"[{full_vs_combined['difference_ci_2_5_percent']:+.8f}, "
        f"{full_vs_combined['difference_ci_97_5_percent']:+.8f}]"
    ),
)
print(
    "Percentile interval excludes zero".ljust(76),
    full_vs_combined["percentile_ci_excludes_zero"],
)

print("\nANALYSIS CONTROLS")
print("-" * 112)
print(
    "Score formulas changed".ljust(76),
    "NO",
)
print(
    "Outcome definition changed".ljust(76),
    "NO",
)
print(
    "Threshold selected or optimized".ljust(76),
    "NO",
)
print(
    "Scientific artifact written or modified".ljust(76),
    "NO",
)

print("\nIN-MEMORY OUTPUTS")
print("-" * 112)
print(
    "Bootstrap replicates".ljust(76),
    "stage6c_primary_auprc_bootstrap_replicates_df",
)
print(
    "Model confidence intervals".ljust(76),
    "stage6c_primary_auprc_bootstrap_summary_df",
)
print(
    "Paired comparisons".ljust(76),
    "stage6c_primary_auprc_paired_comparisons_df",
)

print("\nFINAL DECISION")
print("-" * 112)
print("PASS_STAGE6C_PRIMARY_AUPRC_BOOTSTRAP_INFERENCE_COMPLETE")
print("=" * 112)

STARTING 2,000 PAIRED BOOTSTRAP REPLICATES
Rows per replicate: 66,636
Models per replicate: 4
Parallel workers: 2
Random seed: 42
No files will be written.
----------------------------------------------------------------------------------------------------------------


[Parallel(n_jobs=2)]: Using backend LokyBackend with 2 concurrent workers.
[Parallel(n_jobs=2)]: Done   1 tasks      | elapsed:   17.6s
[Parallel(n_jobs=2)]: Done   4 tasks      | elapsed:   34.6s
[Parallel(n_jobs=2)]: Done   9 tasks      | elapsed:  1.0min



Bootstrap execution: PASS
Elapsed time: 1.53 minutes

PRIMARY AUPRC BOOTSTRAP CONFIDENCE INTERVALS
----------------------------------------------------------------------------------------------------------------


[Parallel(n_jobs=2)]: Done  16 out of  16 | elapsed:  1.5min finished


,model,auprc_point_estimate,bootstrap_mean_auprc,bootstrap_standard_error,auprc_ci_2_5_percent,auprc_ci_97_5_percent,bootstrap_replicates
0,Combined metadata,0.113503,0.113830,0.002438,0.109230,0.118699,2000
1,Full GES,0.112444,0.112802,0.002362,0.108261,0.117357,2000
2,Review stars,0.108304,0.108345,0.001583,0.105293,0.111522,2000
3,No-star GES,0.096437,0.096785,0.002116,0.092620,0.100838,2000



PAIRED AUPRC DIFFERENCE BOOTSTRAP RESULTS
----------------------------------------------------------------------------------------------------------------


,comparison,paired_auprc_difference,bootstrap_standard_error,difference_ci_2_5_percent,difference_ci_97_5_percent,bootstrap_probability_difference_gt_zero,percentile_ci_excludes_zero
0,Full GES minus Review stars,0.004140,0.001823,0.000916,0.007979,0.993,True
1,Full GES minus Combined metadata,-0.001059,0.000625,-0.002279,0.000178,0.050,False
2,Full GES minus No-star GES,0.016007,0.000682,0.014756,0.017419,1.000,True
3,No-star GES minus Review stars,-0.011868,0.001926,-0.015336,-0.007616,0.000,True



STAGE 6C STEP 1 — CELL 6C-1C — LOCKED PRIMARY AUPRC BOOTSTRAP INFERENCE

BOOTSTRAP DESIGN
----------------------------------------------------------------------------------------------------------------
Bootstrap method                                                             Paired nonparametric row bootstrap
Bootstrap replicates                                                         2,000
Random seed                                                                  42
Confidence interval method                                                   2.5th–97.5th percentile
Identical resample used across models                                        YES

FULL GES VERSUS REVIEW STARS
----------------------------------------------------------------------------------------------------------------
Point AUPRC difference                                                       +0.00413961
95% bootstrap interval                                                       [+0.00091579, +0.00797901]
Per

In [13]:
# =================================================================================================
# STAGE 6C STEP 1 — CELL 6C-1D
# LOCKED SECONDARY AUROC BOOTSTRAP INFERENCE
#
# Purpose:
#   1. Generate 2,000 paired nonparametric bootstrap samples.
#   2. Estimate 95% percentile confidence intervals for AUROC.
#   3. Estimate paired AUROC differences:
#        - Full GES versus review stars
#        - Full GES versus combined metadata
#        - Full GES versus no-star GES
#        - No-star GES versus review stars
#
# Design:
#   - Uses the same frozen cohort as Cells 6C-1A through 6C-1C.
#   - Uses the same random seed sequence as the AUPRC bootstrap.
#   - Uses identical sampled records across all models within each replicate.
#   - Performs no threshold optimization, score modification, or artifact writing.
# =================================================================================================

import os
import time
import numpy as np
import pandas as pd

from joblib import Parallel, delayed
from sklearn.metrics import roc_auc_score


# -------------------------------------------------------------------------------------------------
# 1. Confirm locked inputs remain available
# -------------------------------------------------------------------------------------------------

assert "locked_evaluable_df" in globals(), (
    "locked_evaluable_df is unavailable. Run Cell 6C-1A first."
)

assert "locked_outcome" in globals(), (
    "locked_outcome is unavailable. Run Cell 6C-1A first."
)

y_true = pd.to_numeric(
    locked_outcome,
    errors="raise",
).astype(int).to_numpy()

assert len(y_true) == 66_636
assert set(np.unique(y_true)) == {0, 1}
assert int(y_true.sum()) == 6_485
assert int((y_true == 0).sum()) == 60_151


# -------------------------------------------------------------------------------------------------
# 2. Frozen primary score set
# -------------------------------------------------------------------------------------------------

AUROC_BOOTSTRAP_SCORE_COLUMNS = {
    "Full GES":
        "full_ges_instability_risk_t0",

    "Combined metadata":
        "combined_metadata_instability_risk",

    "Review stars":
        "review_stars_instability_risk",

    "No-star GES":
        "no_star_ges_instability_risk_t0",
}

missing_columns = [
    column
    for column in AUROC_BOOTSTRAP_SCORE_COLUMNS.values()
    if column not in locked_evaluable_df.columns
]

assert not missing_columns, (
    "Required frozen score columns are missing:\n"
    + "\n".join(missing_columns)
)

model_names = list(
    AUROC_BOOTSTRAP_SCORE_COLUMNS.keys()
)

score_matrix = np.column_stack(
    [
        pd.to_numeric(
            locked_evaluable_df[column],
            errors="raise",
        ).to_numpy(dtype=np.float64)
        for column in AUROC_BOOTSTRAP_SCORE_COLUMNS.values()
    ]
)

assert score_matrix.shape == (66_636, 4)
assert np.isfinite(score_matrix).all()
assert ((score_matrix >= 0.0) & (score_matrix <= 1.0)).all()


# -------------------------------------------------------------------------------------------------
# 3. Locked AUROC point estimates
# -------------------------------------------------------------------------------------------------

point_estimates = np.array(
    [
        roc_auc_score(
            y_true,
            score_matrix[:, model_index],
        )
        for model_index in range(score_matrix.shape[1])
    ],
    dtype=np.float64,
)

point_estimate_map = dict(
    zip(
        model_names,
        point_estimates,
    )
)


# -------------------------------------------------------------------------------------------------
# 4. Reproducible paired bootstrap configuration
#
# The same SeedSequence construction used in Cell 6C-1C recreates the same row resamples.
# -------------------------------------------------------------------------------------------------

N_BOOTSTRAP = 2_000
BOOTSTRAP_SEED = 42

available_cpus = os.cpu_count() or 1
n_jobs = max(
    1,
    min(4, available_cpus),
)

seed_generator = np.random.SeedSequence(
    BOOTSTRAP_SEED
)

bootstrap_seeds = seed_generator.generate_state(
    N_BOOTSTRAP,
    dtype=np.uint64,
)

number_of_chunks = min(
    N_BOOTSTRAP,
    max(n_jobs * 8, 1),
)

seed_chunks = [
    chunk
    for chunk in np.array_split(
        bootstrap_seeds,
        number_of_chunks,
    )
    if len(chunk) > 0
]


# -------------------------------------------------------------------------------------------------
# 5. Paired bootstrap worker
# -------------------------------------------------------------------------------------------------

def run_auroc_bootstrap_chunk(seed_values):
    """
    Calculate AUROC for all frozen scores using identical resampled rows
    within each bootstrap replicate.
    """

    cohort_size = len(y_true)

    chunk_results = np.empty(
        (
            len(seed_values),
            score_matrix.shape[1],
        ),
        dtype=np.float64,
    )

    for replicate_index, replicate_seed in enumerate(seed_values):

        rng = np.random.default_rng(
            int(replicate_seed)
        )

        sampled_indexes = rng.integers(
            low=0,
            high=cohort_size,
            size=cohort_size,
            dtype=np.int32,
        )

        sampled_outcome = y_true[
            sampled_indexes
        ]

        if np.unique(sampled_outcome).size != 2:
            raise RuntimeError(
                "A bootstrap replicate contained only one outcome class."
            )

        sampled_scores = score_matrix[
            sampled_indexes,
            :,
        ]

        for model_index in range(
            sampled_scores.shape[1]
        ):

            chunk_results[
                replicate_index,
                model_index,
            ] = roc_auc_score(
                sampled_outcome,
                sampled_scores[:, model_index],
            )

    return chunk_results


# -------------------------------------------------------------------------------------------------
# 6. Execute bootstrap
# -------------------------------------------------------------------------------------------------

print("=" * 112)
print("STARTING 2,000 PAIRED AUROC BOOTSTRAP REPLICATES")
print("=" * 112)
print(f"Rows per replicate: {len(y_true):,}")
print(f"Models per replicate: {len(model_names)}")
print(f"Parallel workers: {n_jobs}")
print(f"Random seed: {BOOTSTRAP_SEED}")
print("Resampling sequence matches Cell 6C-1C.")
print("No files will be written.")
print("-" * 112)

bootstrap_start_time = time.time()

bootstrap_chunk_results = Parallel(
    n_jobs=n_jobs,
    backend="loky",
    verbose=10,
)(
    delayed(run_auroc_bootstrap_chunk)(
        seed_chunk
    )
    for seed_chunk in seed_chunks
)

bootstrap_matrix = np.vstack(
    bootstrap_chunk_results
)

bootstrap_elapsed_seconds = (
    time.time() - bootstrap_start_time
)

assert bootstrap_matrix.shape == (
    N_BOOTSTRAP,
    len(model_names),
)

assert np.isfinite(
    bootstrap_matrix
).all()

print("\nAUROC bootstrap execution: PASS")
print(
    f"Elapsed time: "
    f"{bootstrap_elapsed_seconds / 60.0:.2f} minutes"
)


# -------------------------------------------------------------------------------------------------
# 7. Store bootstrap replicates in memory
# -------------------------------------------------------------------------------------------------

stage6c_auroc_bootstrap_replicates_df = pd.DataFrame(
    bootstrap_matrix,
    columns=model_names,
)

stage6c_auroc_bootstrap_replicates_df.insert(
    0,
    "bootstrap_replicate",
    np.arange(
        1,
        N_BOOTSTRAP + 1,
        dtype=int,
    ),
)


# -------------------------------------------------------------------------------------------------
# 8. Model-specific AUROC confidence intervals
# -------------------------------------------------------------------------------------------------

model_summary_rows = []

for model_index, model_name in enumerate(
    model_names
):

    bootstrap_values = bootstrap_matrix[
        :,
        model_index,
    ]

    lower_ci, upper_ci = np.percentile(
        bootstrap_values,
        [2.5, 97.5],
    )

    model_summary_rows.append(
        {
            "model": model_name,
            "n": int(len(y_true)),
            "events": int(y_true.sum()),
            "negatives": int((y_true == 0).sum()),
            "auroc_point_estimate": float(
                point_estimates[model_index]
            ),
            "bootstrap_mean_auroc": float(
                bootstrap_values.mean()
            ),
            "bootstrap_standard_error": float(
                bootstrap_values.std(ddof=1)
            ),
            "auroc_ci_2_5_percent": float(
                lower_ci
            ),
            "auroc_ci_97_5_percent": float(
                upper_ci
            ),
            "bootstrap_replicates": N_BOOTSTRAP,
            "random_seed": BOOTSTRAP_SEED,
        }
    )

stage6c_auroc_bootstrap_summary_df = pd.DataFrame(
    model_summary_rows
).sort_values(
    by="auroc_point_estimate",
    ascending=False,
    kind="stable",
).reset_index(drop=True)


# -------------------------------------------------------------------------------------------------
# 9. Paired AUROC comparisons
# -------------------------------------------------------------------------------------------------

model_index_map = {
    model_name: model_index
    for model_index, model_name in enumerate(
        model_names
    )
}

PAIRED_COMPARISONS = [
    (
        "Full GES",
        "Review stars",
    ),
    (
        "Full GES",
        "Combined metadata",
    ),
    (
        "Full GES",
        "No-star GES",
    ),
    (
        "No-star GES",
        "Review stars",
    ),
]

comparison_rows = []

for model_a, model_b in PAIRED_COMPARISONS:

    model_a_index = model_index_map[
        model_a
    ]

    model_b_index = model_index_map[
        model_b
    ]

    bootstrap_difference = (
        bootstrap_matrix[:, model_a_index]
        - bootstrap_matrix[:, model_b_index]
    )

    point_difference = (
        point_estimate_map[model_a]
        - point_estimate_map[model_b]
    )

    lower_ci, upper_ci = np.percentile(
        bootstrap_difference,
        [2.5, 97.5],
    )

    probability_positive = float(
        np.mean(
            bootstrap_difference > 0.0
        )
    )

    probability_negative = float(
        np.mean(
            bootstrap_difference < 0.0
        )
    )

    comparison_rows.append(
        {
            "comparison":
                f"{model_a} minus {model_b}",

            "model_a":
                model_a,

            "model_b":
                model_b,

            "paired_auroc_difference":
                float(point_difference),

            "bootstrap_mean_difference":
                float(bootstrap_difference.mean()),

            "bootstrap_standard_error":
                float(
                    bootstrap_difference.std(
                        ddof=1
                    )
                ),

            "difference_ci_2_5_percent":
                float(lower_ci),

            "difference_ci_97_5_percent":
                float(upper_ci),

            "bootstrap_probability_difference_gt_zero":
                probability_positive,

            "bootstrap_probability_difference_lt_zero":
                probability_negative,

            "percentile_ci_excludes_zero":
                bool(
                    (lower_ci > 0.0)
                    or
                    (upper_ci < 0.0)
                ),

            "bootstrap_replicates":
                N_BOOTSTRAP,
        }
    )

stage6c_auroc_paired_comparisons_df = pd.DataFrame(
    comparison_rows
)


# -------------------------------------------------------------------------------------------------
# 10. Display model-specific AUROC results
# -------------------------------------------------------------------------------------------------

model_display_df = (
    stage6c_auroc_bootstrap_summary_df[
        [
            "model",
            "auroc_point_estimate",
            "bootstrap_mean_auroc",
            "bootstrap_standard_error",
            "auroc_ci_2_5_percent",
            "auroc_ci_97_5_percent",
            "bootstrap_replicates",
        ]
    ]
    .copy()
)

for column in [
    "auroc_point_estimate",
    "bootstrap_mean_auroc",
    "bootstrap_standard_error",
    "auroc_ci_2_5_percent",
    "auroc_ci_97_5_percent",
]:
    model_display_df[column] = (
        model_display_df[column]
        .round(8)
    )

print("\nSECONDARY AUROC BOOTSTRAP CONFIDENCE INTERVALS")
print("-" * 112)

display(
    model_display_df
)


# -------------------------------------------------------------------------------------------------
# 11. Display paired AUROC comparisons
# -------------------------------------------------------------------------------------------------

comparison_display_df = (
    stage6c_auroc_paired_comparisons_df[
        [
            "comparison",
            "paired_auroc_difference",
            "bootstrap_standard_error",
            "difference_ci_2_5_percent",
            "difference_ci_97_5_percent",
            "bootstrap_probability_difference_gt_zero",
            "percentile_ci_excludes_zero",
        ]
    ]
    .copy()
)

for column in [
    "paired_auroc_difference",
    "bootstrap_standard_error",
    "difference_ci_2_5_percent",
    "difference_ci_97_5_percent",
    "bootstrap_probability_difference_gt_zero",
]:
    comparison_display_df[column] = (
        comparison_display_df[column]
        .round(8)
    )

print("\nPAIRED AUROC DIFFERENCE BOOTSTRAP RESULTS")
print("-" * 112)

display(
    comparison_display_df
)


# -------------------------------------------------------------------------------------------------
# 12. Extract principal comparisons
# -------------------------------------------------------------------------------------------------

full_vs_stars = (
    stage6c_auroc_paired_comparisons_df
    .loc[
        stage6c_auroc_paired_comparisons_df[
            "comparison"
        ] == "Full GES minus Review stars"
    ]
    .iloc[0]
)

full_vs_combined = (
    stage6c_auroc_paired_comparisons_df
    .loc[
        stage6c_auroc_paired_comparisons_df[
            "comparison"
        ] == "Full GES minus Combined metadata"
    ]
    .iloc[0]
)


# -------------------------------------------------------------------------------------------------
# 13. Final audit report
# -------------------------------------------------------------------------------------------------

print("\n" + "=" * 112)
print("STAGE 6C STEP 1 — CELL 6C-1D — LOCKED SECONDARY AUROC BOOTSTRAP INFERENCE")
print("=" * 112)

print("\nBOOTSTRAP DESIGN")
print("-" * 112)
print(
    "Bootstrap method".ljust(76),
    "Paired nonparametric row bootstrap",
)
print(
    "Bootstrap replicates".ljust(76),
    f"{N_BOOTSTRAP:,}",
)
print(
    "Random seed".ljust(76),
    BOOTSTRAP_SEED,
)
print(
    "Confidence interval method".ljust(76),
    "2.5th–97.5th percentile",
)
print(
    "Same resampling sequence as AUPRC bootstrap".ljust(76),
    "YES",
)
print(
    "Identical resample used across models".ljust(76),
    "YES",
)

print("\nFULL GES VERSUS REVIEW STARS")
print("-" * 112)
print(
    "Point AUROC difference".ljust(76),
    f"{full_vs_stars['paired_auroc_difference']:+.8f}",
)
print(
    "95% bootstrap interval".ljust(76),
    (
        f"[{full_vs_stars['difference_ci_2_5_percent']:+.8f}, "
        f"{full_vs_stars['difference_ci_97_5_percent']:+.8f}]"
    ),
)
print(
    "Percentile interval excludes zero".ljust(76),
    full_vs_stars["percentile_ci_excludes_zero"],
)

print("\nFULL GES VERSUS COMBINED METADATA")
print("-" * 112)
print(
    "Point AUROC difference".ljust(76),
    f"{full_vs_combined['paired_auroc_difference']:+.8f}",
)
print(
    "95% bootstrap interval".ljust(76),
    (
        f"[{full_vs_combined['difference_ci_2_5_percent']:+.8f}, "
        f"{full_vs_combined['difference_ci_97_5_percent']:+.8f}]"
    ),
)
print(
    "Percentile interval excludes zero".ljust(76),
    full_vs_combined["percentile_ci_excludes_zero"],
)

print("\nANALYSIS CONTROLS")
print("-" * 112)
print(
    "Score formulas changed".ljust(76),
    "NO",
)
print(
    "Outcome definition changed".ljust(76),
    "NO",
)
print(
    "Threshold selected or optimized".ljust(76),
    "NO",
)
print(
    "Calibration analysis performed".ljust(76),
    "NO",
)
print(
    "Scientific artifact written or modified".ljust(76),
    "NO",
)

print("\nIN-MEMORY OUTPUTS")
print("-" * 112)
print(
    "AUROC bootstrap replicates".ljust(76),
    "stage6c_auroc_bootstrap_replicates_df",
)
print(
    "AUROC confidence intervals".ljust(76),
    "stage6c_auroc_bootstrap_summary_df",
)
print(
    "Paired AUROC comparisons".ljust(76),
    "stage6c_auroc_paired_comparisons_df",
)

print("\nFINAL DECISION")
print("-" * 112)
print("PASS_STAGE6C_SECONDARY_AUROC_BOOTSTRAP_INFERENCE_COMPLETE")
print("=" * 112)

STARTING 2,000 PAIRED AUROC BOOTSTRAP REPLICATES
Rows per replicate: 66,636
Models per replicate: 4
Parallel workers: 2
Random seed: 42
Resampling sequence matches Cell 6C-1C.
No files will be written.
----------------------------------------------------------------------------------------------------------------


[Parallel(n_jobs=2)]: Using backend LokyBackend with 2 concurrent workers.
[Parallel(n_jobs=2)]: Done   1 tasks      | elapsed:   20.6s
[Parallel(n_jobs=2)]: Done   4 tasks      | elapsed:   40.6s
[Parallel(n_jobs=2)]: Done   9 tasks      | elapsed:  1.3min



AUROC bootstrap execution: PASS
Elapsed time: 1.95 minutes

SECONDARY AUROC BOOTSTRAP CONFIDENCE INTERVALS
----------------------------------------------------------------------------------------------------------------


[Parallel(n_jobs=2)]: Done  16 out of  16 | elapsed:  2.0min finished


,model,auroc_point_estimate,bootstrap_mean_auroc,bootstrap_standard_error,auroc_ci_2_5_percent,auroc_ci_97_5_percent,bootstrap_replicates
0,Full GES,0.535812,0.535916,0.003602,0.528805,0.542848,2000
1,Review stars,0.533391,0.533414,0.002498,0.528594,0.538418,2000
2,Combined metadata,0.532614,0.532757,0.003666,0.525631,0.539871,2000
3,No-star GES,0.447722,0.447872,0.003810,0.440480,0.455102,2000



PAIRED AUROC DIFFERENCE BOOTSTRAP RESULTS
----------------------------------------------------------------------------------------------------------------


,comparison,paired_auroc_difference,bootstrap_standard_error,difference_ci_2_5_percent,difference_ci_97_5_percent,bootstrap_probability_difference_gt_zero,percentile_ci_excludes_zero
0,Full GES minus Review stars,0.002421,0.002442,-0.002221,0.007069,0.8465,False
1,Full GES minus Combined metadata,0.003198,0.000905,0.001372,0.004917,0.9995,True
2,Full GES minus No-star GES,0.088090,0.002196,0.083730,0.092304,1.0000,True
3,No-star GES minus Review stars,-0.085669,0.003865,-0.092834,-0.077934,0.0000,True



STAGE 6C STEP 1 — CELL 6C-1D — LOCKED SECONDARY AUROC BOOTSTRAP INFERENCE

BOOTSTRAP DESIGN
----------------------------------------------------------------------------------------------------------------
Bootstrap method                                                             Paired nonparametric row bootstrap
Bootstrap replicates                                                         2,000
Random seed                                                                  42
Confidence interval method                                                   2.5th–97.5th percentile
Same resampling sequence as AUPRC bootstrap                                  YES
Identical resample used across models                                        YES

FULL GES VERSUS REVIEW STARS
----------------------------------------------------------------------------------------------------------------
Point AUROC difference                                                       +0.00242098
95% bootstrap interval  

In [14]:
# =================================================================================================
# STAGE 6C STEP 2 — CELL 6C-2A
# LOCKED CALIBRATION POINT ESTIMATES AND RELIABILITY TABLES
#
# Purpose:
#   1. Calculate Brier scores for the principal frozen scores.
#   2. Calculate calibration intercept and slope for the two GES probability models.
#   3. Construct reliability tables using observed future-instability rates.
#   4. Calculate descriptive ECE and maximum calibration gap.
#
# Scientific interpretation:
#   - Full GES and no-star GES are probability-model outputs.
#   - Combined metadata and review stars are frozen heuristic/ordinal risk scores.
#   - Calibration intercept and slope are therefore estimated only for the GES probability models.
#
# Controls:
#   - No score recalibration.
#   - No threshold selection or optimization.
#   - No bootstrap inference.
#   - No scientific artifact writing or modification.
# =================================================================================================

import warnings
import numpy as np
import pandas as pd
import statsmodels.api as sm

from sklearn.metrics import brier_score_loss


# -------------------------------------------------------------------------------------------------
# 1. Confirm locked inputs remain available
# -------------------------------------------------------------------------------------------------

assert "locked_evaluable_df" in globals(), (
    "locked_evaluable_df is unavailable. Run Cell 6C-1A first."
)

assert "locked_outcome" in globals(), (
    "locked_outcome is unavailable. Run Cell 6C-1A first."
)

y_true = pd.to_numeric(
    locked_outcome,
    errors="raise",
).astype(int).to_numpy()

assert len(y_true) == 66_636
assert set(np.unique(y_true)) == {0, 1}
assert int(y_true.sum()) == 6_485
assert int((y_true == 0).sum()) == 60_151

event_prevalence = float(y_true.mean())

# Brier score produced by assigning every record the cohort prevalence.
null_brier_score = float(
    np.mean(
        (y_true - event_prevalence) ** 2
    )
)


# -------------------------------------------------------------------------------------------------
# 2. Frozen principal calibration-score inventory
# -------------------------------------------------------------------------------------------------

CALIBRATION_SCORE_SPECIFICATION = {
    "Full GES": {
        "column": "full_ges_instability_risk_t0",
        "score_type": "probability_model",
        "estimate_logistic_calibration": True,
    },

    "No-star GES": {
        "column": "no_star_ges_instability_risk_t0",
        "score_type": "probability_model",
        "estimate_logistic_calibration": True,
    },

    "Combined metadata": {
        "column": "combined_metadata_instability_risk",
        "score_type": "heuristic_risk_score",
        "estimate_logistic_calibration": False,
    },

    "Review stars": {
        "column": "review_stars_instability_risk",
        "score_type": "ordinal_heuristic_score",
        "estimate_logistic_calibration": False,
    },
}

missing_columns = [
    specification["column"]
    for specification in CALIBRATION_SCORE_SPECIFICATION.values()
    if specification["column"] not in locked_evaluable_df.columns
]

assert not missing_columns, (
    "Required frozen score columns are missing:\n"
    + "\n".join(missing_columns)
)


# -------------------------------------------------------------------------------------------------
# 3. Helper: fit logistic calibration intercept and slope
#
# Model:
#     logit(P(Y=1)) = intercept + slope × logit(frozen predicted risk)
#
# Ideal calibration:
#     intercept = 0
#     slope = 1
# -------------------------------------------------------------------------------------------------

def estimate_logistic_calibration(
    outcome: np.ndarray,
    predicted_probability: np.ndarray,
    epsilon: float = 1e-12,
) -> dict:
    """
    Estimate calibration intercept and slope through binomial GLM.

    Probabilities are clipped only for the mathematical logit transformation.
    Original frozen scores are not changed.
    """

    clipped_probability = np.clip(
        predicted_probability,
        epsilon,
        1.0 - epsilon,
    )

    logit_probability = np.log(
        clipped_probability
        / (1.0 - clipped_probability)
    )

    design_matrix = sm.add_constant(
        logit_probability,
        has_constant="add",
    )

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")

        fitted_model = sm.GLM(
            outcome,
            design_matrix,
            family=sm.families.Binomial(),
        ).fit(
            maxiter=200,
            disp=0,
        )

    confidence_intervals = np.asarray(
        fitted_model.conf_int(
            alpha=0.05
        )
    )

    return {
        "calibration_intercept":
            float(fitted_model.params[0]),

        "calibration_intercept_se":
            float(fitted_model.bse[0]),

        "calibration_intercept_ci_lower":
            float(confidence_intervals[0, 0]),

        "calibration_intercept_ci_upper":
            float(confidence_intervals[0, 1]),

        "calibration_slope":
            float(fitted_model.params[1]),

        "calibration_slope_se":
            float(fitted_model.bse[1]),

        "calibration_slope_ci_lower":
            float(confidence_intervals[1, 0]),

        "calibration_slope_ci_upper":
            float(confidence_intervals[1, 1]),

        "calibration_model_converged":
            bool(fitted_model.converged),
    }


# -------------------------------------------------------------------------------------------------
# 4. Helper: construct reliability table
#
# Continuous scores:
#   Equal-frequency bins using score quantiles.
#
# Low-cardinality scores:
#   Exact score-value groups are retained instead of splitting tied values artificially.
# -------------------------------------------------------------------------------------------------

def construct_reliability_table(
    model_name: str,
    score_values: np.ndarray,
    score_type: str,
    requested_bins: int = 10,
) -> pd.DataFrame:

    reliability_source = pd.DataFrame(
        {
            "outcome": y_true,
            "score": score_values,
        }
    )

    unique_score_count = int(
        reliability_source["score"].nunique()
    )

    if unique_score_count <= 20:

        ordered_unique_scores = sorted(
            reliability_source["score"].unique()
        )

        score_to_bin = {
            score_value: bin_number
            for bin_number, score_value
            in enumerate(
                ordered_unique_scores,
                start=1,
            )
        }

        reliability_source["reliability_bin"] = (
            reliability_source["score"]
            .map(score_to_bin)
            .astype(int)
        )

        binning_method = "exact_score_groups"

    else:

        reliability_source["reliability_bin"] = (
            pd.qcut(
                reliability_source["score"],
                q=requested_bins,
                labels=False,
                duplicates="drop",
            )
            .astype(int)
            + 1
        )

        binning_method = "equal_frequency_quantile_bins"

    reliability_table = (
        reliability_source
        .groupby(
            "reliability_bin",
            observed=True,
            sort=True,
        )
        .agg(
            records=("outcome", "size"),
            events=("outcome", "sum"),
            score_minimum=("score", "min"),
            score_maximum=("score", "max"),
            mean_predicted_risk=("score", "mean"),
            observed_event_rate=("outcome", "mean"),
        )
        .reset_index()
    )

    reliability_table["non_events"] = (
        reliability_table["records"]
        - reliability_table["events"]
    )

    reliability_table["calibration_gap"] = (
        reliability_table["observed_event_rate"]
        - reliability_table["mean_predicted_risk"]
    )

    reliability_table["absolute_calibration_gap"] = (
        reliability_table["calibration_gap"].abs()
    )

    reliability_table.insert(
        0,
        "model",
        model_name,
    )

    reliability_table.insert(
        1,
        "score_type",
        score_type,
    )

    reliability_table.insert(
        2,
        "binning_method",
        binning_method,
    )

    reliability_table.insert(
        3,
        "unique_score_count",
        unique_score_count,
    )

    return reliability_table


# -------------------------------------------------------------------------------------------------
# 5. Calculate calibration summaries and reliability tables
# -------------------------------------------------------------------------------------------------

summary_rows = []
reliability_tables = []

for model_name, specification in CALIBRATION_SCORE_SPECIFICATION.items():

    score_column = specification["column"]
    score_type = specification["score_type"]

    score_values = pd.to_numeric(
        locked_evaluable_df[score_column],
        errors="raise",
    ).to_numpy(dtype=np.float64)

    assert len(score_values) == len(y_true)
    assert np.isfinite(score_values).all()
    assert (
        (score_values >= 0.0)
        & (score_values <= 1.0)
    ).all()

    brier_score = float(
        brier_score_loss(
            y_true,
            score_values,
        )
    )

    brier_skill_score = float(
        1.0
        - (
            brier_score
            / null_brier_score
        )
    )

    reliability_table = construct_reliability_table(
        model_name=model_name,
        score_values=score_values,
        score_type=score_type,
        requested_bins=10,
    )

    reliability_tables.append(
        reliability_table
    )

    weighted_absolute_gap = float(
        np.average(
            reliability_table[
                "absolute_calibration_gap"
            ],
            weights=reliability_table[
                "records"
            ],
        )
    )

    maximum_absolute_gap = float(
        reliability_table[
            "absolute_calibration_gap"
        ].max()
    )

    summary_row = {
        "model": model_name,
        "score_column": score_column,
        "score_type": score_type,
        "n": int(len(y_true)),
        "events": int(y_true.sum()),
        "event_prevalence": event_prevalence,
        "mean_predicted_risk": float(
            score_values.mean()
        ),
        "minimum_score": float(
            score_values.min()
        ),
        "maximum_score": float(
            score_values.max()
        ),
        "unique_scores": int(
            np.unique(score_values).size
        ),
        "brier_score": brier_score,
        "null_prevalence_brier_score":
            null_brier_score,
        "brier_skill_score_vs_prevalence":
            brier_skill_score,
        "descriptive_expected_calibration_error":
            weighted_absolute_gap,
        "descriptive_maximum_calibration_gap":
            maximum_absolute_gap,
        "reliability_groups": int(
            len(reliability_table)
        ),
        "logistic_calibration_estimated":
            bool(
                specification[
                    "estimate_logistic_calibration"
                ]
            ),
    }

    if specification[
        "estimate_logistic_calibration"
    ]:

        calibration_result = (
            estimate_logistic_calibration(
                outcome=y_true,
                predicted_probability=score_values,
            )
        )

        summary_row.update(
            calibration_result
        )

    else:

        summary_row.update(
            {
                "calibration_intercept": np.nan,
                "calibration_intercept_se": np.nan,
                "calibration_intercept_ci_lower": np.nan,
                "calibration_intercept_ci_upper": np.nan,
                "calibration_slope": np.nan,
                "calibration_slope_se": np.nan,
                "calibration_slope_ci_lower": np.nan,
                "calibration_slope_ci_upper": np.nan,
                "calibration_model_converged": pd.NA,
            }
        )

    summary_rows.append(
        summary_row
    )


# -------------------------------------------------------------------------------------------------
# 6. Materialize in-memory results
# -------------------------------------------------------------------------------------------------

stage6c_calibration_point_estimates_df = pd.DataFrame(
    summary_rows
)

stage6c_reliability_table_df = pd.concat(
    reliability_tables,
    ignore_index=True,
)

preferred_model_order = [
    "Full GES",
    "No-star GES",
    "Combined metadata",
    "Review stars",
]

stage6c_calibration_point_estimates_df[
    "model_order"
] = (
    stage6c_calibration_point_estimates_df[
        "model"
    ]
    .map(
        {
            model_name: order
            for order, model_name
            in enumerate(
                preferred_model_order,
                start=1,
            )
        }
    )
)

stage6c_calibration_point_estimates_df = (
    stage6c_calibration_point_estimates_df
    .sort_values(
        "model_order",
        kind="stable",
    )
    .drop(
        columns="model_order"
    )
    .reset_index(
        drop=True
    )
)


# -------------------------------------------------------------------------------------------------
# 7. Validate accounting
# -------------------------------------------------------------------------------------------------

reliability_accounting = (
    stage6c_reliability_table_df
    .groupby(
        "model",
        observed=True,
    )["records"]
    .sum()
)

for model_name in preferred_model_order:

    assert int(
        reliability_accounting.loc[
            model_name
        ]
    ) == len(y_true), (
        f"Reliability-table accounting failed for {model_name}."
    )

probability_model_rows = (
    stage6c_calibration_point_estimates_df[
        stage6c_calibration_point_estimates_df[
            "score_type"
        ] == "probability_model"
    ]
)

assert probability_model_rows[
    "calibration_model_converged"
].astype(bool).all(), (
    "At least one probability-model calibration regression did not converge."
)


# -------------------------------------------------------------------------------------------------
# 8. Display calibration summary
# -------------------------------------------------------------------------------------------------

summary_display_columns = [
    "model",
    "score_type",
    "mean_predicted_risk",
    "event_prevalence",
    "brier_score",
    "null_prevalence_brier_score",
    "brier_skill_score_vs_prevalence",
    "calibration_intercept",
    "calibration_intercept_ci_lower",
    "calibration_intercept_ci_upper",
    "calibration_slope",
    "calibration_slope_ci_lower",
    "calibration_slope_ci_upper",
    "descriptive_expected_calibration_error",
    "descriptive_maximum_calibration_gap",
    "reliability_groups",
]

calibration_display_df = (
    stage6c_calibration_point_estimates_df[
        summary_display_columns
    ]
    .copy()
)

numeric_display_columns = [
    column
    for column in calibration_display_df.columns
    if column not in {
        "model",
        "score_type",
        "reliability_groups",
    }
]

calibration_display_df[
    numeric_display_columns
] = (
    calibration_display_df[
        numeric_display_columns
    ]
    .round(8)
)

print("\nLOCKED CALIBRATION POINT ESTIMATES")
print("-" * 132)

display(
    calibration_display_df
)


# -------------------------------------------------------------------------------------------------
# 9. Display reliability tables
# -------------------------------------------------------------------------------------------------

reliability_display_df = (
    stage6c_reliability_table_df[
        [
            "model",
            "binning_method",
            "reliability_bin",
            "records",
            "events",
            "score_minimum",
            "score_maximum",
            "mean_predicted_risk",
            "observed_event_rate",
            "calibration_gap",
            "absolute_calibration_gap",
        ]
    ]
    .copy()
)

for column in [
    "score_minimum",
    "score_maximum",
    "mean_predicted_risk",
    "observed_event_rate",
    "calibration_gap",
    "absolute_calibration_gap",
]:
    reliability_display_df[column] = (
        reliability_display_df[column]
        .round(8)
    )

print("\nLOCKED RELIABILITY TABLES")
print("-" * 132)

display(
    reliability_display_df
)


# -------------------------------------------------------------------------------------------------
# 10. Extract principal probability-model results
# -------------------------------------------------------------------------------------------------

full_ges_calibration = (
    stage6c_calibration_point_estimates_df
    .loc[
        stage6c_calibration_point_estimates_df[
            "model"
        ] == "Full GES"
    ]
    .iloc[0]
)

no_star_calibration = (
    stage6c_calibration_point_estimates_df
    .loc[
        stage6c_calibration_point_estimates_df[
            "model"
        ] == "No-star GES"
    ]
    .iloc[0]
)


# -------------------------------------------------------------------------------------------------
# 11. Final audit report
# -------------------------------------------------------------------------------------------------

print("\n" + "=" * 132)
print("STAGE 6C STEP 2 — CELL 6C-2A — LOCKED CALIBRATION POINT ESTIMATES")
print("=" * 132)

print("\nANALYSIS COHORT")
print("-" * 132)
print(
    "Primary-evaluable records".ljust(82),
    f"{len(y_true):,}",
)
print(
    "Primary instability events".ljust(82),
    f"{int(y_true.sum()):,}",
)
print(
    "Observed event prevalence".ljust(82),
    f"{event_prevalence:.8f}",
)
print(
    "Prevalence-only null Brier score".ljust(82),
    f"{null_brier_score:.8f}",
)

print("\nFULL GES CALIBRATION")
print("-" * 132)
print(
    "Mean frozen predicted instability risk".ljust(82),
    f"{full_ges_calibration['mean_predicted_risk']:.8f}",
)
print(
    "Brier score".ljust(82),
    f"{full_ges_calibration['brier_score']:.8f}",
)
print(
    "Brier skill score versus prevalence".ljust(82),
    f"{full_ges_calibration['brier_skill_score_vs_prevalence']:+.8f}",
)
print(
    "Calibration intercept".ljust(82),
    f"{full_ges_calibration['calibration_intercept']:+.8f}",
)
print(
    "Calibration-intercept 95% interval".ljust(82),
    (
        f"[{full_ges_calibration['calibration_intercept_ci_lower']:+.8f}, "
        f"{full_ges_calibration['calibration_intercept_ci_upper']:+.8f}]"
    ),
)
print(
    "Calibration slope".ljust(82),
    f"{full_ges_calibration['calibration_slope']:.8f}",
)
print(
    "Calibration-slope 95% interval".ljust(82),
    (
        f"[{full_ges_calibration['calibration_slope_ci_lower']:.8f}, "
        f"{full_ges_calibration['calibration_slope_ci_upper']:.8f}]"
    ),
)

print("\nNO-STAR GES CALIBRATION")
print("-" * 132)
print(
    "Mean frozen predicted instability risk".ljust(82),
    f"{no_star_calibration['mean_predicted_risk']:.8f}",
)
print(
    "Brier score".ljust(82),
    f"{no_star_calibration['brier_score']:.8f}",
)
print(
    "Brier skill score versus prevalence".ljust(82),
    f"{no_star_calibration['brier_skill_score_vs_prevalence']:+.8f}",
)
print(
    "Calibration intercept".ljust(82),
    f"{no_star_calibration['calibration_intercept']:+.8f}",
)
print(
    "Calibration slope".ljust(82),
    f"{no_star_calibration['calibration_slope']:.8f}",
)

print("\nINTERPRETATION BOUNDARY")
print("-" * 132)
print(
    "GES outputs treated as probability models".ljust(82),
    "YES",
)
print(
    "Combined metadata treated as calibrated probability".ljust(82),
    "NO — descriptive heuristic score only",
)
print(
    "Review stars treated as calibrated probability".ljust(82),
    "NO — descriptive ordinal score only",
)

print("\nANALYSIS CONTROLS")
print("-" * 132)
print(
    "Scores recalibrated or changed".ljust(82),
    "NO",
)
print(
    "Threshold selected or optimized".ljust(82),
    "NO",
)
print(
    "Bootstrap inference performed".ljust(82),
    "NO",
)
print(
    "Scientific artifact written or modified".ljust(82),
    "NO",
)

print("\nIN-MEMORY OUTPUTS")
print("-" * 132)
print(
    "Calibration-summary table".ljust(82),
    "stage6c_calibration_point_estimates_df",
)
print(
    "Reliability-bin table".ljust(82),
    "stage6c_reliability_table_df",
)

print("\nFINAL DECISION")
print("-" * 132)
print("PASS_STAGE6C_LOCKED_CALIBRATION_POINT_ESTIMATES_COMPLETE")
print("=" * 132)


LOCKED CALIBRATION POINT ESTIMATES
------------------------------------------------------------------------------------------------------------------------------------


,model,score_type,mean_predicted_risk,event_prevalence,brier_score,null_prevalence_brier_score,brier_skill_score_vs_prevalence,calibration_intercept,calibration_intercept_ci_lower,calibration_intercept_ci_upper,calibration_slope,calibration_slope_ci_lower,calibration_slope_ci_upper,descriptive_expected_calibration_error,descriptive_maximum_calibration_gap,reliability_groups
0,Full GES,probability_model,0.060559,0.09732,0.137546,0.087849,-0.565714,-1.987343,-2.037374,-1.937311,0.019236,0.015683,0.022788,0.139056,0.511981,10
1,No-star GES,probability_model,0.034725,0.09732,0.119828,0.087849,-0.364024,-2.246418,-2.319871,-2.172965,-0.001500,-0.006914,0.003914,0.112340,0.248859,10
2,Combined metadata,heuristic_risk_score,0.284785,0.09732,0.126557,0.087849,-0.440622,NaN,NaN,NaN,NaN,NaN,NaN,0.187466,0.342379,10
3,Review stars,ordinal_heuristic_score,0.539143,0.09732,0.326537,0.087849,-2.717040,NaN,NaN,NaN,NaN,NaN,NaN,0.441883,0.566289,4



LOCKED RELIABILITY TABLES
------------------------------------------------------------------------------------------------------------------------------------


,model,binning_method,reliability_bin,records,events,score_minimum,score_maximum,mean_predicted_risk,observed_event_rate,calibration_gap,absolute_calibration_gap
0,Full GES,equal_frequency_quantile_bins,1,6664,2,0.000000e+00,0.000000e+00,0.000000e+00,0.000300,0.000300,0.000300
1,Full GES,equal_frequency_quantile_bins,2,7128,989,0.000000e+00,0.000000e+00,0.000000e+00,0.138749,0.138749,0.138749
2,Full GES,equal_frequency_quantile_bins,3,6199,995,0.000000e+00,1.160000e-06,3.800000e-07,0.160510,0.160509,0.160509
3,Full GES,equal_frequency_quantile_bins,4,6719,538,1.160000e-06,1.870000e-06,1.590000e-06,0.080071,0.080070,0.080070
4,Full GES,equal_frequency_quantile_bins,5,6737,227,1.870000e-06,2.050000e-06,1.960000e-06,0.033695,0.033693,0.033693
5,Full GES,equal_frequency_quantile_bins,6,6536,823,2.050000e-06,5.360000e-06,3.300000e-06,0.125918,0.125915,0.125915
6,Full GES,equal_frequency_quantile_bins,7,6662,750,5.360000e-06,1.848000e-05,1.122000e-05,0.112579,0.112568,0.112568
7,Full GES,equal_frequency_quantile_bins,8,6665,826,1.853000e-05,9.069000e-05,4.124000e-05,0.123931,0.123890,0.123890
8,Full GES,equal_frequency_quantile_bins,9,6669,711,9.094000e-05,1.350290e-03,4.107600e-04,0.106613,0.106202,0.106202
9,Full GES,equal_frequency_quantile_bins,10,6657,624,1.350680e-03,1.000000e+00,6.057171e-01,0.093736,-0.511981,0.511981



STAGE 6C STEP 2 — CELL 6C-2A — LOCKED CALIBRATION POINT ESTIMATES

ANALYSIS COHORT
------------------------------------------------------------------------------------------------------------------------------------
Primary-evaluable records                                                          66,636
Primary instability events                                                         6,485
Observed event prevalence                                                          0.09731977
Prevalence-only null Brier score                                                   0.08784863

FULL GES CALIBRATION
------------------------------------------------------------------------------------------------------------------------------------
Mean frozen predicted instability risk                                             0.06055879
Brier score                                                                        0.13754583
Brier skill score versus prevalence                                      

In [15]:
# ==================================================================================================
# STAGE 6C — STEP 2B
# CELL 6C-2B1 — FULL-GES TOP-RISK ENRICHMENT POINT ESTIMATES
#
# Purpose:
#   1. Freshly verify the frozen Stage 6B primary-evaluable cohort.
#   2. Rank records from lowest to highest frozen full-GES P(stable).
#   3. Evaluate the prespecified lowest-P(stable) 5%, 10%, and 20% strata.
#   4. Compare each selected stratum with:
#        a. the full evaluable-cohort prevalence;
#        b. the remaining records outside the stratum.
#   5. Record deterministic cutoff-tie handling.
#
# Scientific boundary:
#   - Uses only the immutable 66,636-row Stage 6B evaluable cohort.
#   - Uses the frozen full-GES score without recalibration or modification.
#   - Percentages are exact rank-based groups.
#   - Ties at a cutoff are resolved deterministically by frozen t0_row_order.
#   - Any split cutoff tie is explicitly reported.
#   - No threshold is optimized.
#   - No bootstrap confidence interval is calculated in this cell.
#   - No scientific artifact is written or modified.
# ==================================================================================================

from google.colab import drive

drive.mount(
    "/content/drive",
    force_remount=False,
)

from pathlib import Path
import hashlib
import math
import re

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from IPython.display import display


# --------------------------------------------------------------------------------------------------
# 1. Frozen Stage 6B paths and expected identity
# --------------------------------------------------------------------------------------------------

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/GES_RAG_Temporal_Study"
)

EVALUABLE_COHORT_PATH = (
    PROJECT_ROOT
    / "data_processed"
    / "stage6_temporal_validation"
    / "stage6b_locked_primary_evaluable_cohort_v1.parquet"
)

EVALUABLE_COHORT_SIDECAR_PATH = (
    EVALUABLE_COHORT_PATH.with_name(
        EVALUABLE_COHORT_PATH.name + ".sha256"
    )
)

EXPECTED_EVALUABLE_SHA256 = (
    "c6ad50dfc376a746e1830d000dba975101d88763"
    "fd61b8d1983f6300edb6e038"
)

EXPECTED_ROWS = 66_636
EXPECTED_COLUMNS = 79
EXPECTED_EVENTS = 6_485
EXPECTED_NEGATIVES = 60_151

ROW_ORDER_COL = "t0_row_order"
OUTCOME_COL = "primary_future_instability"
P_STABLE_COL = "full_ges_p_stable_t0"
INSTABILITY_RISK_COL = "full_ges_instability_risk_t0"

TARGET_FRACTIONS = (
    0.05,
    0.10,
    0.20,
)


# --------------------------------------------------------------------------------------------------
# 2. Cryptographic helper functions
# --------------------------------------------------------------------------------------------------

def sha256_file(
    path: Path,
    chunk_size: int = 8 * 1024 * 1024,
) -> str:
    """Calculate a SHA-256 checksum without modifying the file."""

    digest = hashlib.sha256()

    with path.open("rb") as file_handle:
        while True:
            block = file_handle.read(chunk_size)

            if not block:
                break

            digest.update(block)

    return digest.hexdigest()


def read_sha256_sidecar(path: Path) -> str:
    """Extract the first valid 64-character SHA-256 value from a sidecar."""

    sidecar_text = path.read_text(
        encoding="utf-8"
    ).strip()

    match = re.search(
        r"(?i)\b[0-9a-f]{64}\b",
        sidecar_text,
    )

    if match is None:
        raise AssertionError(
            f"No valid SHA-256 value found in sidecar: {path}"
        )

    return match.group(0).lower()


# --------------------------------------------------------------------------------------------------
# 3. Fresh frozen-input verification
# --------------------------------------------------------------------------------------------------

assert EVALUABLE_COHORT_PATH.is_file(), (
    f"Missing frozen evaluable cohort:\n"
    f"{EVALUABLE_COHORT_PATH}"
)

assert EVALUABLE_COHORT_SIDECAR_PATH.is_file(), (
    f"Missing evaluable-cohort sidecar:\n"
    f"{EVALUABLE_COHORT_SIDECAR_PATH}"
)

observed_evaluable_sha256 = sha256_file(
    EVALUABLE_COHORT_PATH
)

sidecar_evaluable_sha256 = read_sha256_sidecar(
    EVALUABLE_COHORT_SIDECAR_PATH
)

assert observed_evaluable_sha256 == EXPECTED_EVALUABLE_SHA256, (
    "Frozen evaluable-cohort SHA-256 does not match "
    "the Stage 6B accepted checksum."
)

assert sidecar_evaluable_sha256 == EXPECTED_EVALUABLE_SHA256, (
    "Evaluable-cohort sidecar does not contain "
    "the accepted Stage 6B checksum."
)

parquet_file = pq.ParquetFile(
    EVALUABLE_COHORT_PATH
)

parquet_metadata = parquet_file.metadata
parquet_columns = parquet_file.schema_arrow.names

assert parquet_metadata.num_rows == EXPECTED_ROWS, (
    f"Unexpected Parquet row count: "
    f"{parquet_metadata.num_rows:,}"
)

assert parquet_metadata.num_columns == EXPECTED_COLUMNS, (
    f"Unexpected Parquet column count: "
    f"{parquet_metadata.num_columns:,}"
)


# Identify the frozen RCV-key column without changing its content.
key_candidates = (
    "t0_rcv_accession",
    "rcv_accession_t0",
    "rcv_accession",
)

KEY_COL = next(
    (
        candidate
        for candidate in key_candidates
        if candidate in parquet_columns
    ),
    None,
)

assert KEY_COL is not None, (
    "Could not identify the frozen T0 RCV-key column."
)

required_columns = [
    KEY_COL,
    ROW_ORDER_COL,
    OUTCOME_COL,
    P_STABLE_COL,
    INSTABILITY_RISK_COL,
]

missing_columns = [
    column
    for column in required_columns
    if column not in parquet_columns
]

assert not missing_columns, (
    f"Required frozen columns are missing: {missing_columns}"
)


# --------------------------------------------------------------------------------------------------
# 4. Read only the required frozen columns
# --------------------------------------------------------------------------------------------------

enrichment_source_df = pd.read_parquet(
    EVALUABLE_COHORT_PATH,
    columns=required_columns,
)

assert enrichment_source_df.shape == (
    EXPECTED_ROWS,
    len(required_columns),
), (
    "Loaded enrichment-source dataframe has an "
    "unexpected shape."
)

assert enrichment_source_df[KEY_COL].notna().all(), (
    "The frozen RCV key contains missing values."
)

assert enrichment_source_df[KEY_COL].is_unique, (
    "The frozen RCV key is not unique."
)

enrichment_source_df[ROW_ORDER_COL] = pd.to_numeric(
    enrichment_source_df[ROW_ORDER_COL],
    errors="raise",
).astype("int64")

assert enrichment_source_df[ROW_ORDER_COL].is_unique, (
    "t0_row_order is not unique."
)

assert enrichment_source_df[
    ROW_ORDER_COL
].is_monotonic_increasing, (
    "The evaluable cohort no longer preserves "
    "monotonic frozen T0 row order."
)

enrichment_source_df[OUTCOME_COL] = pd.to_numeric(
    enrichment_source_df[OUTCOME_COL],
    errors="raise",
).astype("int8")

observed_outcome_values = set(
    enrichment_source_df[
        OUTCOME_COL
    ].unique().tolist()
)

assert observed_outcome_values == {0, 1}, (
    f"Primary outcome is not binary: "
    f"{observed_outcome_values}"
)

for score_column in (
    P_STABLE_COL,
    INSTABILITY_RISK_COL,
):
    enrichment_source_df[score_column] = pd.to_numeric(
        enrichment_source_df[score_column],
        errors="raise",
    ).astype("float64")

    assert np.isfinite(
        enrichment_source_df[score_column].to_numpy()
    ).all(), (
        f"{score_column} contains a nonfinite value."
    )

    assert enrichment_source_df[
        score_column
    ].between(
        0.0,
        1.0,
        inclusive="both",
    ).all(), (
        f"{score_column} contains a value outside [0, 1]."
    )

assert np.allclose(
    (
        enrichment_source_df[P_STABLE_COL]
        + enrichment_source_df[INSTABILITY_RISK_COL]
    ).to_numpy(),
    1.0,
    rtol=0.0,
    atol=1e-10,
), (
    "Full-GES P(stable) and instability risk are "
    "not exact complements within tolerance."
)

observed_events = int(
    enrichment_source_df[OUTCOME_COL].sum()
)

observed_negatives = int(
    len(enrichment_source_df) - observed_events
)

assert observed_events == EXPECTED_EVENTS, (
    f"Expected {EXPECTED_EVENTS:,} events but found "
    f"{observed_events:,}."
)

assert observed_negatives == EXPECTED_NEGATIVES, (
    f"Expected {EXPECTED_NEGATIVES:,} negatives but found "
    f"{observed_negatives:,}."
)


# --------------------------------------------------------------------------------------------------
# 5. Create deterministic lowest-P(stable) ranking
#
# Primary ordering:
#   Lowest full-GES P(stable) first.
#
# Tie breaker:
#   Frozen t0_row_order ascending.
#
# The outcome is never used for ordering.
# --------------------------------------------------------------------------------------------------

stage6c_full_ges_ranked_membership = (
    enrichment_source_df
    .sort_values(
        by=[
            P_STABLE_COL,
            ROW_ORDER_COL,
        ],
        ascending=[
            True,
            True,
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)

stage6c_full_ges_ranked_membership[
    "full_ges_low_stability_rank"
] = np.arange(
    1,
    EXPECTED_ROWS + 1,
    dtype=np.int64,
)

overall_event_rate = (
    observed_events
    / EXPECTED_ROWS
)

result_rows = []


# --------------------------------------------------------------------------------------------------
# 6. Calculate prespecified 5%, 10%, and 20% enrichment point estimates
# --------------------------------------------------------------------------------------------------

for target_fraction in TARGET_FRACTIONS:

    selected_n = int(
        math.ceil(
            EXPECTED_ROWS * target_fraction
        )
    )

    selected_mask = (
        stage6c_full_ges_ranked_membership[
            "full_ges_low_stability_rank"
        ]
        <= selected_n
    )

    selected_df = (
        stage6c_full_ges_ranked_membership.loc[
            selected_mask
        ]
    )

    remaining_df = (
        stage6c_full_ges_ranked_membership.loc[
            ~selected_mask
        ]
    )

    selected_events = int(
        selected_df[OUTCOME_COL].sum()
    )

    remaining_events = int(
        remaining_df[OUTCOME_COL].sum()
    )

    remaining_n = int(
        len(remaining_df)
    )

    selected_event_rate = (
        selected_events
        / selected_n
    )

    remaining_event_rate = (
        remaining_events
        / remaining_n
    )

    event_rate_difference = (
        selected_event_rate
        - remaining_event_rate
    )

    risk_ratio_vs_remaining = (
        selected_event_rate
        / remaining_event_rate
        if remaining_event_rate > 0
        else np.inf
    )

    enrichment_vs_overall = (
        selected_event_rate
        / overall_event_rate
        if overall_event_rate > 0
        else np.inf
    )

    event_capture_fraction = (
        selected_events
        / observed_events
        if observed_events > 0
        else np.nan
    )

    cutoff_p_stable = float(
        selected_df[P_STABLE_COL].iloc[-1]
    )

    cutoff_instability_risk = float(
        selected_df[
            INSTABILITY_RISK_COL
        ].iloc[-1]
    )

    cutoff_tie_mask = (
        stage6c_full_ges_ranked_membership[
            P_STABLE_COL
        ]
        == cutoff_p_stable
    )

    cutoff_tied_records_total = int(
        cutoff_tie_mask.sum()
    )

    cutoff_tied_records_selected = int(
        (
            cutoff_tie_mask
            & selected_mask
        ).sum()
    )

    cutoff_tie_split = bool(
        cutoff_tied_records_selected
        < cutoff_tied_records_total
    )

    membership_column = (
        "lowest_pstable_"
        f"{int(target_fraction * 100):02d}pct"
    )

    stage6c_full_ges_ranked_membership[
        membership_column
    ] = selected_mask.to_numpy(
        dtype=bool
    )

    result_rows.append({
        "stratum": (
            "Lowest full-GES P(stable) "
            f"{int(target_fraction * 100)}%"
        ),
        "target_fraction": float(
            target_fraction
        ),
        "selected_records": selected_n,
        "achieved_fraction": (
            selected_n
            / EXPECTED_ROWS
        ),
        "selected_events": selected_events,
        "selected_event_rate": (
            selected_event_rate
        ),
        "remaining_records": remaining_n,
        "remaining_events": remaining_events,
        "remaining_event_rate": (
            remaining_event_rate
        ),
        "event_rate_difference_vs_remaining": (
            event_rate_difference
        ),
        "risk_ratio_vs_remaining": (
            risk_ratio_vs_remaining
        ),
        "enrichment_vs_overall_prevalence": (
            enrichment_vs_overall
        ),
        "event_capture_fraction": (
            event_capture_fraction
        ),
        "maximum_selected_p_stable": (
            cutoff_p_stable
        ),
        "minimum_selected_instability_risk": (
            cutoff_instability_risk
        ),
        "cutoff_tied_records_total": (
            cutoff_tied_records_total
        ),
        "cutoff_tied_records_selected": (
            cutoff_tied_records_selected
        ),
        "cutoff_tie_split": (
            cutoff_tie_split
        ),
    })


stage6c_top_risk_enrichment_table = pd.DataFrame(
    result_rows
)


# --------------------------------------------------------------------------------------------------
# 7. Final checks
# --------------------------------------------------------------------------------------------------

assert len(
    stage6c_top_risk_enrichment_table
) == len(TARGET_FRACTIONS)

assert (
    stage6c_top_risk_enrichment_table[
        "selected_events"
    ]
    + stage6c_top_risk_enrichment_table[
        "remaining_events"
    ]
    == EXPECTED_EVENTS
).all(), (
    "Selected and remaining event counts do not "
    "reconcile with the total event count."
)

assert (
    stage6c_top_risk_enrichment_table[
        "selected_records"
    ]
    + stage6c_top_risk_enrichment_table[
        "remaining_records"
    ]
    == EXPECTED_ROWS
).all(), (
    "Selected and remaining record counts do not "
    "reconcile with the evaluable cohort."
)


# --------------------------------------------------------------------------------------------------
# 8. Display complete point-estimate results
# --------------------------------------------------------------------------------------------------

print()
print("=" * 118)
print(
    "STAGE 6C STEP 2B — CELL 6C-2B1 — "
    "FULL-GES TOP-RISK ENRICHMENT POINT ESTIMATES"
)
print("=" * 118)

print()
print("FROZEN INPUT VERIFICATION")
print("-" * 118)
print(
    f"Primary-evaluable cohort SHA-256 : "
    f"PASS ({observed_evaluable_sha256})"
)
print(
    f"Parquet dimensions                : "
    f"PASS ({EXPECTED_ROWS:,} × {EXPECTED_COLUMNS})"
)
print(
    f"Unique evaluable RCV keys         : "
    f"PASS ({enrichment_source_df[KEY_COL].nunique():,})"
)
print(
    f"Primary instability events        : "
    f"PASS ({observed_events:,})"
)
print(
    f"Primary instability negatives     : "
    f"PASS ({observed_negatives:,})"
)
print(
    f"Overall event prevalence          : "
    f"{overall_event_rate:.8f} "
    f"({overall_event_rate * 100:.6f}%)"
)

print()
print("PRESPECIFIED TOP-RISK ENRICHMENT RESULTS")
print("-" * 118)

for result in result_rows:

    print()
    print(result["stratum"])
    print(
        f"  Selected records                : "
        f"{result['selected_records']:,} "
        f"({result['achieved_fraction'] * 100:.6f}%)"
    )
    print(
        f"  Selected events                 : "
        f"{result['selected_events']:,}"
    )
    print(
        f"  Selected event rate             : "
        f"{result['selected_event_rate']:.8f} "
        f"({result['selected_event_rate'] * 100:.6f}%)"
    )
    print(
        f"  Remaining event rate            : "
        f"{result['remaining_event_rate']:.8f} "
        f"({result['remaining_event_rate'] * 100:.6f}%)"
    )
    print(
        f"  Rate difference vs remaining    : "
        f"{result['event_rate_difference_vs_remaining']:+.8f}"
    )
    print(
        f"  Risk ratio vs remaining         : "
        f"{result['risk_ratio_vs_remaining']:.8f}"
    )
    print(
        f"  Enrichment vs overall prevalence: "
        f"{result['enrichment_vs_overall_prevalence']:.8f}×"
    )
    print(
        f"  Fraction of all events captured : "
        f"{result['event_capture_fraction']:.8f} "
        f"({result['event_capture_fraction'] * 100:.6f}%)"
    )
    print(
        f"  Maximum selected P(stable)      : "
        f"{result['maximum_selected_p_stable']:.12f}"
    )
    print(
        f"  Minimum selected risk           : "
        f"{result['minimum_selected_instability_risk']:.12f}"
    )
    print(
        f"  Records tied at cutoff          : "
        f"{result['cutoff_tied_records_total']:,}"
    )
    print(
        f"  Cutoff-tied records selected    : "
        f"{result['cutoff_tied_records_selected']:,}"
    )
    print(
        f"  Cutoff tie split                : "
        f"{result['cutoff_tie_split']}"
    )

print()
print("COMPLETE IN-MEMORY RESULT TABLE")
print("-" * 118)

display(
    stage6c_top_risk_enrichment_table
)

print()
print("CELL DECISION")
print("-" * 118)
print(
    "PASS_STAGE6C_FULL_GES_TOP_RISK_ENRICHMENT_"
    "POINT_ESTIMATES_COMPLETE"
)
print(
    "No threshold was optimized, no score was modified, "
    "and no scientific artifact was written."
)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

STAGE 6C STEP 2B — CELL 6C-2B1 — FULL-GES TOP-RISK ENRICHMENT POINT ESTIMATES

FROZEN INPUT VERIFICATION
----------------------------------------------------------------------------------------------------------------------
Primary-evaluable cohort SHA-256 : PASS (c6ad50dfc376a746e1830d000dba975101d88763fd61b8d1983f6300edb6e038)
Parquet dimensions                : PASS (66,636 × 79)
Unique evaluable RCV keys         : PASS (66,636)
Primary instability events        : PASS (6,485)
Primary instability negatives     : PASS (60,151)
Overall event prevalence          : 0.09731977 (9.731977%)

PRESPECIFIED TOP-RISK ENRICHMENT RESULTS
----------------------------------------------------------------------------------------------------------------------

Lowest full-GES P(stable) 5%
  Selected records                : 3,332 (5.000300%)
  Selected events              

,stratum,target_fraction,selected_records,achieved_fraction,selected_events,selected_event_rate,remaining_records,remaining_events,remaining_event_rate,event_rate_difference_vs_remaining,risk_ratio_vs_remaining,enrichment_vs_overall_prevalence,event_capture_fraction,maximum_selected_p_stable,minimum_selected_instability_risk,cutoff_tied_records_total,cutoff_tied_records_selected,cutoff_tie_split
0,Lowest full-GES P(stable) 5%,0.05,3332,0.050003,420,0.126050,63304,6065,0.095808,0.030243,1.315663,1.295219,0.064765,0.234298,0.765702,1840,969,True
1,Lowest full-GES P(stable) 10%,0.10,6664,0.100006,624,0.093637,59972,5861,0.097729,-0.004091,0.958134,0.962163,0.096222,0.998650,0.001350,61,7,True
2,Lowest full-GES P(stable) 20%,0.20,13328,0.200012,1335,0.100165,53308,5150,0.096608,0.003557,1.036815,1.029237,0.205860,0.999909,0.000091,8,2,True



CELL DECISION
----------------------------------------------------------------------------------------------------------------------
PASS_STAGE6C_FULL_GES_TOP_RISK_ENRICHMENT_POINT_ESTIMATES_COMPLETE
No threshold was optimized, no score was modified, and no scientific artifact was written.


In [16]:
# ==================================================================================================
# STAGE 6C — STEP 2B
# CELL 6C-2B2 — FULL-GES RANK-DECILE EVENT RATES, SCORE DISTRIBUTION, AND CUTOFF-TIE AUDIT
#
# Purpose:
#   1. Divide the frozen evaluable cohort into ten approximately equal rank-based groups.
#   2. Define Decile 1 as the lowest full-GES P(stable), therefore the highest predicted instability risk.
#   3. Calculate event rates, enrichment, event capture, and score summaries for every decile.
#   4. Audit score ties at every decile boundary.
#   5. Diagnose why the lowest 5% is enriched while the lowest 10% is not.
#
# Important:
#   - Deciles use deterministic frozen ranking from Cell 6C-2B1.
#   - Outcome values are never used to construct the ranking or boundaries.
#   - Exact-size rank groups may divide identical score values.
#   - All divided score ties are explicitly recorded.
#   - No threshold is optimized.
#   - No scientific artifact is written or modified.
# ==================================================================================================

import math
import numpy as np
import pandas as pd
from IPython.display import display


# --------------------------------------------------------------------------------------------------
# 1. Confirm that the preceding enrichment cell remains available
# --------------------------------------------------------------------------------------------------

assert "stage6c_full_ges_ranked_membership" in globals(), (
    "The ranked full-GES dataframe is not available. "
    "Run Cell 6C-2B1 immediately before this cell."
)

ranked_df = stage6c_full_ges_ranked_membership.copy()

KEY_COL_LOCAL = (
    KEY_COL
    if "KEY_COL" in globals()
    else "t0_rcv_accession"
)

ROW_ORDER_COL_LOCAL = "t0_row_order"
OUTCOME_COL_LOCAL = "primary_future_instability"
P_STABLE_COL_LOCAL = "full_ges_p_stable_t0"
RISK_COL_LOCAL = "full_ges_instability_risk_t0"
RANK_COL_LOCAL = "full_ges_low_stability_rank"

required_columns = [
    KEY_COL_LOCAL,
    ROW_ORDER_COL_LOCAL,
    OUTCOME_COL_LOCAL,
    P_STABLE_COL_LOCAL,
    RISK_COL_LOCAL,
    RANK_COL_LOCAL,
]

missing_columns = [
    column
    for column in required_columns
    if column not in ranked_df.columns
]

assert not missing_columns, (
    f"Required columns are missing from the ranked dataframe: "
    f"{missing_columns}"
)

N = int(len(ranked_df))
TOTAL_EVENTS = int(
    ranked_df[OUTCOME_COL_LOCAL].sum()
)
TOTAL_NEGATIVES = int(
    N - TOTAL_EVENTS
)
OVERALL_EVENT_RATE = (
    TOTAL_EVENTS / N
)

assert N == 66_636, (
    f"Unexpected evaluable-cohort size: {N:,}"
)

assert TOTAL_EVENTS == 6_485, (
    f"Unexpected event count: {TOTAL_EVENTS:,}"
)

assert TOTAL_NEGATIVES == 60_151, (
    f"Unexpected negative count: {TOTAL_NEGATIVES:,}"
)

assert ranked_df[RANK_COL_LOCAL].tolist() == list(
    range(1, N + 1)
), (
    "The frozen full-GES low-stability rank is not "
    "the exact sequence 1 through N."
)

assert ranked_df[P_STABLE_COL_LOCAL].is_monotonic_increasing, (
    "P(stable) is not monotonically increasing in the "
    "frozen low-stability ranking."
)


# --------------------------------------------------------------------------------------------------
# 2. Construct cumulative rank boundaries
#
# Using ceil(k × N / 10) ensures:
#   - Decile 1 contains the exact lowest-10% rank group used in Cell 6C-2B1.
#   - Deciles cover all records exactly once.
#   - Group-size differences are at most one record.
# --------------------------------------------------------------------------------------------------

decile_boundaries = [0]

for decile_number in range(1, 11):
    decile_boundaries.append(
        int(
            math.ceil(
                decile_number * N / 10
            )
        )
    )

assert decile_boundaries[0] == 0
assert decile_boundaries[-1] == N
assert all(
    earlier < later
    for earlier, later in zip(
        decile_boundaries[:-1],
        decile_boundaries[1:],
    )
)

rank_decile = np.empty(
    N,
    dtype=np.int8,
)

for decile_number in range(1, 11):

    start_position = decile_boundaries[
        decile_number - 1
    ]

    stop_position = decile_boundaries[
        decile_number
    ]

    rank_decile[
        start_position:stop_position
    ] = decile_number

ranked_df[
    "full_ges_low_pstable_rank_decile"
] = rank_decile

assert ranked_df[
    "full_ges_low_pstable_rank_decile"
].between(
    1,
    10,
    inclusive="both",
).all()

assert ranked_df[
    "full_ges_low_pstable_rank_decile"
].notna().all()


# --------------------------------------------------------------------------------------------------
# 3. Calculate decile-level event rates and score summaries
# --------------------------------------------------------------------------------------------------

decile_rows = []

for decile_number in range(1, 11):

    decile_df = ranked_df.loc[
        ranked_df[
            "full_ges_low_pstable_rank_decile"
        ]
        == decile_number
    ].copy()

    decile_n = int(
        len(decile_df)
    )

    decile_events = int(
        decile_df[
            OUTCOME_COL_LOCAL
        ].sum()
    )

    decile_negatives = int(
        decile_n - decile_events
    )

    decile_event_rate = (
        decile_events / decile_n
    )

    event_rate_difference = (
        decile_event_rate
        - OVERALL_EVENT_RATE
    )

    enrichment_vs_overall = (
        decile_event_rate
        / OVERALL_EVENT_RATE
    )

    event_capture_fraction = (
        decile_events
        / TOTAL_EVENTS
    )

    minimum_p_stable = float(
        decile_df[
            P_STABLE_COL_LOCAL
        ].min()
    )

    maximum_p_stable = float(
        decile_df[
            P_STABLE_COL_LOCAL
        ].max()
    )

    minimum_risk = float(
        decile_df[
            RISK_COL_LOCAL
        ].min()
    )

    maximum_risk = float(
        decile_df[
            RISK_COL_LOCAL
        ].max()
    )

    unique_score_count = int(
        decile_df[
            P_STABLE_COL_LOCAL
        ].nunique(
            dropna=False
        )
    )

    if decile_number < 10:

        upper_boundary_rank = (
            decile_boundaries[
                decile_number
            ]
        )

        boundary_p_stable = float(
            ranked_df.iloc[
                upper_boundary_rank - 1
            ][P_STABLE_COL_LOCAL]
        )

        boundary_tie_mask = (
            ranked_df[
                P_STABLE_COL_LOCAL
            ]
            == boundary_p_stable
        )

        boundary_tied_records_total = int(
            boundary_tie_mask.sum()
        )

        boundary_tied_records_at_or_below = int(
            (
                boundary_tie_mask
                & (
                    ranked_df[
                        RANK_COL_LOCAL
                    ]
                    <= upper_boundary_rank
                )
            ).sum()
        )

        boundary_tied_records_above = int(
            boundary_tied_records_total
            - boundary_tied_records_at_or_below
        )

        boundary_tie_split = bool(
            boundary_tied_records_at_or_below > 0
            and boundary_tied_records_above > 0
        )

    else:

        upper_boundary_rank = N
        boundary_p_stable = maximum_p_stable
        boundary_tied_records_total = int(
            (
                ranked_df[
                    P_STABLE_COL_LOCAL
                ]
                == boundary_p_stable
            ).sum()
        )
        boundary_tied_records_at_or_below = (
            boundary_tied_records_total
        )
        boundary_tied_records_above = 0
        boundary_tie_split = False

    decile_rows.append({
        "decile": decile_number,
        "risk_order_description": (
            "Highest predicted instability risk"
            if decile_number == 1
            else (
                "Lowest predicted instability risk"
                if decile_number == 10
                else ""
            )
        ),
        "records": decile_n,
        "cohort_fraction": (
            decile_n / N
        ),
        "events": decile_events,
        "negatives": decile_negatives,
        "event_rate": decile_event_rate,
        "event_rate_difference_vs_overall": (
            event_rate_difference
        ),
        "enrichment_vs_overall": (
            enrichment_vs_overall
        ),
        "event_capture_fraction": (
            event_capture_fraction
        ),
        "minimum_p_stable": minimum_p_stable,
        "maximum_p_stable": maximum_p_stable,
        "mean_p_stable": float(
            decile_df[
                P_STABLE_COL_LOCAL
            ].mean()
        ),
        "median_p_stable": float(
            decile_df[
                P_STABLE_COL_LOCAL
            ].median()
        ),
        "minimum_instability_risk": (
            minimum_risk
        ),
        "maximum_instability_risk": (
            maximum_risk
        ),
        "unique_p_stable_values": (
            unique_score_count
        ),
        "upper_boundary_rank": (
            upper_boundary_rank
        ),
        "upper_boundary_p_stable": (
            boundary_p_stable
        ),
        "boundary_tied_records_total": (
            boundary_tied_records_total
        ),
        "boundary_tied_records_at_or_below": (
            boundary_tied_records_at_or_below
        ),
        "boundary_tied_records_above": (
            boundary_tied_records_above
        ),
        "boundary_tie_split": (
            boundary_tie_split
        ),
    })


stage6c_full_ges_decile_table = pd.DataFrame(
    decile_rows
)


# --------------------------------------------------------------------------------------------------
# 4. Create a standalone boundary-tie audit
# --------------------------------------------------------------------------------------------------

boundary_rows = []

for boundary_number in range(1, 10):

    boundary_rank = int(
        decile_boundaries[
            boundary_number
        ]
    )

    boundary_fraction = (
        boundary_rank / N
    )

    cutoff_p_stable = float(
        ranked_df.iloc[
            boundary_rank - 1
        ][P_STABLE_COL_LOCAL]
    )

    tie_mask = (
        ranked_df[
            P_STABLE_COL_LOCAL
        ]
        == cutoff_p_stable
    )

    tied_total = int(
        tie_mask.sum()
    )

    tied_at_or_below = int(
        (
            tie_mask
            & (
                ranked_df[
                    RANK_COL_LOCAL
                ]
                <= boundary_rank
            )
        ).sum()
    )

    tied_above = int(
        tied_total
        - tied_at_or_below
    )

    boundary_rows.append({
        "boundary_after_decile": (
            boundary_number
        ),
        "cumulative_target_percent": (
            boundary_number * 10
        ),
        "boundary_rank": boundary_rank,
        "achieved_cumulative_fraction": (
            boundary_fraction
        ),
        "cutoff_p_stable": cutoff_p_stable,
        "cutoff_instability_risk": (
            1.0 - cutoff_p_stable
        ),
        "tied_records_total": tied_total,
        "tied_records_at_or_below": (
            tied_at_or_below
        ),
        "tied_records_above": tied_above,
        "tie_split": bool(
            tied_at_or_below > 0
            and tied_above > 0
        ),
    })


stage6c_full_ges_decile_boundary_tie_audit = (
    pd.DataFrame(
        boundary_rows
    )
)


# --------------------------------------------------------------------------------------------------
# 5. Diagnose the first decile by separating the lowest 5% from ranks 5%-10%
#
# This explains whether the second half of Decile 1 dilutes the enrichment observed in the
# prespecified lowest-5% analysis.
# --------------------------------------------------------------------------------------------------

lowest_5_n = int(
    math.ceil(
        0.05 * N
    )
)

lowest_10_n = int(
    math.ceil(
        0.10 * N
    )
)

lowest_5_df = ranked_df.iloc[
    0:lowest_5_n
].copy()

second_5_df = ranked_df.iloc[
    lowest_5_n:lowest_10_n
].copy()

lowest_10_df = ranked_df.iloc[
    0:lowest_10_n
].copy()


def summarize_rank_slice(
    label: str,
    slice_df: pd.DataFrame,
    starting_rank: int,
    ending_rank: int,
) -> dict:
    """Create a descriptive summary for one frozen rank interval."""

    slice_n = int(
        len(slice_df)
    )

    slice_events = int(
        slice_df[
            OUTCOME_COL_LOCAL
        ].sum()
    )

    slice_event_rate = (
        slice_events
        / slice_n
    )

    return {
        "rank_interval": label,
        "starting_rank": starting_rank,
        "ending_rank": ending_rank,
        "records": slice_n,
        "events": slice_events,
        "event_rate": slice_event_rate,
        "enrichment_vs_overall": (
            slice_event_rate
            / OVERALL_EVENT_RATE
        ),
        "minimum_p_stable": float(
            slice_df[
                P_STABLE_COL_LOCAL
            ].min()
        ),
        "maximum_p_stable": float(
            slice_df[
                P_STABLE_COL_LOCAL
            ].max()
        ),
        "unique_p_stable_values": int(
            slice_df[
                P_STABLE_COL_LOCAL
            ].nunique(
                dropna=False
            )
        ),
    }


stage6c_full_ges_first_decile_diagnostic = (
    pd.DataFrame([
        summarize_rank_slice(
            label="Lowest P(stable) 0%-5%",
            slice_df=lowest_5_df,
            starting_rank=1,
            ending_rank=lowest_5_n,
        ),
        summarize_rank_slice(
            label="Lowest P(stable) 5%-10%",
            slice_df=second_5_df,
            starting_rank=lowest_5_n + 1,
            ending_rank=lowest_10_n,
        ),
        summarize_rank_slice(
            label="Lowest P(stable) cumulative 0%-10%",
            slice_df=lowest_10_df,
            starting_rank=1,
            ending_rank=lowest_10_n,
        ),
    ])
)


# --------------------------------------------------------------------------------------------------
# 6. Score-distribution and tie diagnostics
# --------------------------------------------------------------------------------------------------

score_frequency = (
    ranked_df[
        P_STABLE_COL_LOCAL
    ]
    .value_counts(
        dropna=False
    )
    .rename_axis(
        "p_stable"
    )
    .reset_index(
        name="record_count"
    )
    .sort_values(
        by=[
            "record_count",
            "p_stable",
        ],
        ascending=[
            False,
            True,
        ],
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)

largest_tie_value = float(
    score_frequency.iloc[0][
        "p_stable"
    ]
)

largest_tie_count = int(
    score_frequency.iloc[0][
        "record_count"
    ]
)

score_quantiles = (
    ranked_df[
        P_STABLE_COL_LOCAL
    ]
    .quantile([
        0.00,
        0.01,
        0.05,
        0.10,
        0.20,
        0.25,
        0.50,
        0.75,
        0.90,
        0.95,
        0.99,
        1.00,
    ])
    .rename_axis(
        "quantile"
    )
    .reset_index(
        name="p_stable"
    )
)

score_quantiles[
    "instability_risk"
] = (
    1.0
    - score_quantiles[
        "p_stable"
    ]
)

stage6c_full_ges_score_distribution_summary = (
    pd.DataFrame([{
        "records": N,
        "unique_p_stable_values": int(
            ranked_df[
                P_STABLE_COL_LOCAL
            ].nunique()
        ),
        "duplicate_score_records": int(
            N
            - ranked_df[
                P_STABLE_COL_LOCAL
            ].nunique()
        ),
        "largest_tie_p_stable": (
            largest_tie_value
        ),
        "largest_tie_records": (
            largest_tie_count
        ),
        "largest_tie_fraction": (
            largest_tie_count / N
        ),
        "minimum_p_stable": float(
            ranked_df[
                P_STABLE_COL_LOCAL
            ].min()
        ),
        "maximum_p_stable": float(
            ranked_df[
                P_STABLE_COL_LOCAL
            ].max()
        ),
    }])
)


# --------------------------------------------------------------------------------------------------
# 7. Reconciliation and consistency checks
# --------------------------------------------------------------------------------------------------

assert int(
    stage6c_full_ges_decile_table[
        "records"
    ].sum()
) == N, (
    "Decile record counts do not reconcile "
    "with the evaluable cohort."
)

assert int(
    stage6c_full_ges_decile_table[
        "events"
    ].sum()
) == TOTAL_EVENTS, (
    "Decile event counts do not reconcile "
    "with the total event count."
)

assert int(
    stage6c_full_ges_decile_table[
        "negatives"
    ].sum()
) == TOTAL_NEGATIVES, (
    "Decile negative counts do not reconcile "
    "with the total negative count."
)

assert (
    stage6c_full_ges_decile_table[
        "records"
    ].max()
    - stage6c_full_ges_decile_table[
        "records"
    ].min()
    <= 1
), (
    "Rank-decile sizes differ by more than one record."
)

assert int(
    stage6c_full_ges_first_decile_diagnostic.iloc[
        0
    ]["events"]
    + stage6c_full_ges_first_decile_diagnostic.iloc[
        1
    ]["events"]
) == int(
    stage6c_full_ges_first_decile_diagnostic.iloc[
        2
    ]["events"]
), (
    "The two 5% rank intervals do not reconcile "
    "with the cumulative lowest-10% interval."
)

assert lowest_10_n == int(
    stage6c_full_ges_decile_table.loc[
        stage6c_full_ges_decile_table[
            "decile"
        ]
        == 1,
        "records",
    ].iloc[0]
), (
    "Decile 1 does not match the exact lowest-10% "
    "rank group from Cell 6C-2B1."
)


# --------------------------------------------------------------------------------------------------
# 8. Display results
# --------------------------------------------------------------------------------------------------

print()
print("=" * 122)
print(
    "STAGE 6C STEP 2B — CELL 6C-2B2 — "
    "FULL-GES DECILE EVENT RATES AND SCORE-TIE AUDIT"
)
print("=" * 122)

print()
print("ANALYSIS BOUNDARY")
print("-" * 122)
print(
    f"Evaluable records                   : {N:,}"
)
print(
    f"Primary instability events         : {TOTAL_EVENTS:,}"
)
print(
    f"Overall event prevalence           : "
    f"{OVERALL_EVENT_RATE:.8f} "
    f"({OVERALL_EVENT_RATE * 100:.6f}%)"
)
print(
    "Decile direction                   : "
    "Decile 1 = lowest P(stable) / highest predicted risk"
)
print(
    "Decile construction                : "
    "Deterministic rank groups using frozen t0_row_order for score ties"
)
print(
    "Threshold or score optimization    : No"
)
print(
    "Scientific artifact written        : No"
)

print()
print("FULL-GES EVENT RATE BY RANK DECILE")
print("-" * 122)

for row in decile_rows:

    print()
    print(
        f"Decile {row['decile']} "
        f"({row['records']:,} records)"
    )
    print(
        f"  Events                            : "
        f"{row['events']:,}"
    )
    print(
        f"  Event rate                        : "
        f"{row['event_rate']:.8f} "
        f"({row['event_rate'] * 100:.6f}%)"
    )
    print(
        f"  Enrichment vs overall            : "
        f"{row['enrichment_vs_overall']:.8f}×"
    )
    print(
        f"  P(stable) range                  : "
        f"{row['minimum_p_stable']:.12f} "
        f"to {row['maximum_p_stable']:.12f}"
    )
    print(
        f"  Unique P(stable) values          : "
        f"{row['unique_p_stable_values']:,}"
    )
    print(
        f"  Upper-boundary tie split         : "
        f"{row['boundary_tie_split']}"
    )

print()
print("FIRST-DECILE 5% INTERVAL DIAGNOSTIC")
print("-" * 122)

display(
    stage6c_full_ges_first_decile_diagnostic
)

print()
print("COMPLETE DECILE TABLE")
print("-" * 122)

display(
    stage6c_full_ges_decile_table
)

print()
print("DECILE-BOUNDARY TIE AUDIT")
print("-" * 122)

display(
    stage6c_full_ges_decile_boundary_tie_audit
)

print()
print("FULL-GES SCORE-DISTRIBUTION SUMMARY")
print("-" * 122)

display(
    stage6c_full_ges_score_distribution_summary
)

print()
print("SELECTED SCORE QUANTILES")
print("-" * 122)

display(
    score_quantiles
)

print()
print("TEN LARGEST IDENTICAL-SCORE GROUPS")
print("-" * 122)

display(
    score_frequency.head(10)
)

print()
print("CELL DECISION")
print("-" * 122)
print(
    "PASS_STAGE6C_FULL_GES_DECILE_EVENT_RATES_"
    "AND_SCORE_TIE_AUDIT_COMPLETE"
)
print(
    "Rank-decile results are descriptive and preserve all score ties, "
    "including ties divided by deterministic rank boundaries."
)
print(
    "No score, outcome, threshold, weight, or frozen scientific artifact "
    "was modified."
)


STAGE 6C STEP 2B — CELL 6C-2B2 — FULL-GES DECILE EVENT RATES AND SCORE-TIE AUDIT

ANALYSIS BOUNDARY
--------------------------------------------------------------------------------------------------------------------------
Evaluable records                   : 66,636
Primary instability events         : 6,485
Overall event prevalence           : 0.09731977 (9.731977%)
Decile direction                   : Decile 1 = lowest P(stable) / highest predicted risk
Decile construction                : Deterministic rank groups using frozen t0_row_order for score ties
Threshold or score optimization    : No
Scientific artifact written        : No

FULL-GES EVENT RATE BY RANK DECILE
--------------------------------------------------------------------------------------------------------------------------

Decile 1 (6,664 records)
  Events                            : 624
  Event rate                        : 0.09363745 (9.363745%)
  Enrichment vs overall            : 0.96216275×
  P(stable) range

,rank_interval,starting_rank,ending_rank,records,events,event_rate,enrichment_vs_overall,minimum_p_stable,maximum_p_stable,unique_p_stable_values
0,Lowest P(stable) 0%-5%,1,3332,3332,420,0.126050,1.295219,2.174233e-15,0.234298,1251
1,Lowest P(stable) 5%-10%,3333,6664,3332,204,0.061224,0.629106,2.342983e-01,0.998650,1008
2,Lowest P(stable) cumulative 0%-10%,1,6664,6664,624,0.093637,0.962163,2.174233e-15,0.998650,2258



COMPLETE DECILE TABLE
--------------------------------------------------------------------------------------------------------------------------


,decile,risk_order_description,records,cohort_fraction,events,negatives,event_rate,event_rate_difference_vs_overall,enrichment_vs_overall,event_capture_fraction,...,median_p_stable,minimum_instability_risk,maximum_instability_risk,unique_p_stable_values,upper_boundary_rank,upper_boundary_p_stable,boundary_tied_records_total,boundary_tied_records_at_or_below,boundary_tied_records_above,boundary_tie_split
0,1,Highest predicted instability risk,6664,0.100006,624,6040,0.093637,-0.003682,0.962163,0.096222,...,0.234298,1.350291e-03,1.000000e+00,2258,6664,0.998650,61,7,54,True
1,2,,6664,0.100006,711,5953,0.106693,0.009373,1.096310,0.109638,...,0.999706,9.068658e-05,1.350291e-03,1277,13328,0.999909,8,2,6,True
2,3,,6663,0.099991,826,5837,0.123968,0.026648,1.273823,0.127371,...,0.999966,1.853036e-05,9.068658e-05,728,19991,0.999981,24,24,0,False
3,4,,6664,0.100006,752,5912,0.112845,0.015525,1.159529,0.115960,...,0.999989,5.358816e-06,1.847934e-05,567,26655,0.999995,2,2,0,False
4,5,,6663,0.099991,825,5838,0.123818,0.026498,1.272281,0.127217,...,0.999997,2.047316e-06,5.344062e-06,480,33318,0.999998,318,129,189,True
5,6,,6664,0.100006,230,6434,0.034514,-0.062806,0.354643,0.035466,...,0.999998,1.869267e-06,2.047316e-06,45,39982,0.999998,67,56,11,True
6,7,,6664,0.100006,531,6133,0.079682,-0.017638,0.818763,0.081881,...,0.999998,1.160179e-06,1.869267e-06,291,46646,0.999999,12,1,11,True
7,8,,6663,0.099991,995,5668,0.149332,0.052012,1.534448,0.153431,...,1.000000,2.131987e-09,1.160179e-06,1935,53309,1.000000,468,465,3,True
8,9,,6664,0.100006,989,5675,0.148409,0.051090,1.524966,0.152506,...,1.000000,2.660692e-10,2.131987e-09,1702,59973,1.000000,1,1,0,False
9,10,Lowest predicted instability risk,6663,0.099991,2,6661,0.000300,-0.097020,0.003084,0.000308,...,1.000000,3.385514e-12,2.659988e-10,227,66636,1.000000,2,2,0,False



DECILE-BOUNDARY TIE AUDIT
--------------------------------------------------------------------------------------------------------------------------


...


SELECTED SCORE QUANTILES
--------------------------------------------------------------------------------------------------------------------------


,quantile,p_stable,instability_risk
0,0.00,2.174233e-15,1.000000e+00
1,0.01,1.255637e-04,9.998744e-01
2,0.05,2.342983e-01,7.657017e-01
3,0.10,9.986497e-01,1.350291e-03
4,0.20,9.999093e-01,9.068658e-05
5,0.25,9.999662e-01,3.379916e-05
6,0.50,9.999980e-01,2.047316e-06
7,0.75,1.000000e+00,3.914493e-08
8,0.90,1.000000e+00,2.663618e-10
9,0.95,1.000000e+00,7.955703e-11



TEN LARGEST IDENTICAL-SCORE GROUPS
--------------------------------------------------------------------------------------------------------------------------


,p_stable,record_count
0,0.234298,1840
1,0.999998,1367
2,1.000000,1217
3,0.999998,1025
4,1.000000,983
5,1.000000,902
6,0.999998,766
7,1.000000,722
8,0.999998,659
9,1.000000,589



CELL DECISION
--------------------------------------------------------------------------------------------------------------------------
PASS_STAGE6C_FULL_GES_DECILE_EVENT_RATES_AND_SCORE_TIE_AUDIT_COMPLETE
Rank-decile results are descriptive and preserve all score ties, including ties divided by deterministic rank boundaries.
No score, outcome, threshold, weight, or frozen scientific artifact was modified.


In [17]:
# ==================================================================================================
# STAGE 6C — STEP 2B
# CELL 6C-2B3 — FULL-GES CUTOFF-TIE SENSITIVITY AND TIE-ALLOCATION BOUNDS
#
# Purpose:
#   1. Reevaluate the prespecified lowest-P(stable) 5%, 10%, and 20% strata under three
#      transparent cutoff policies:
#        a. strict: include only scores strictly below the cutoff;
#        b. exact-rank: retain the deterministic fixed-size rank group;
#        c. inclusive: include every record tied at the cutoff.
#   2. Audit the event rate within the cutoff-tied records.
#   3. Compare the tied records selected versus not selected by frozen t0_row_order.
#   4. Calculate minimum and maximum possible exact-rank event rates if the required number
#      of cutoff-tied records had been selected in any other order.
#
# Scientific boundary:
#   - Uses the immutable ranking and outcomes already loaded in Cells 6C-2B1 and 6C-2B2.
#   - Exact floating-point equality defines a score tie.
#   - No score, outcome, threshold, weight, or ranking is changed.
#   - Alternative tie allocations are mathematical sensitivity bounds only.
#   - No confidence interval or hypothesis test is calculated.
#   - No scientific artifact is written.
# ==================================================================================================

import math

import numpy as np
import pandas as pd
from IPython.display import display


# --------------------------------------------------------------------------------------------------
# 1. Confirm required in-memory objects
# --------------------------------------------------------------------------------------------------

assert "stage6c_full_ges_ranked_membership" in globals(), (
    "The ranked full-GES dataframe is unavailable. "
    "Run Cell 6C-2B1 before this cell."
)

ranked_df = (
    stage6c_full_ges_ranked_membership
    .copy()
)

KEY_COL_LOCAL = (
    KEY_COL
    if "KEY_COL" in globals()
    else "t0_rcv_accession"
)

ROW_ORDER_COL_LOCAL = "t0_row_order"
OUTCOME_COL_LOCAL = "primary_future_instability"
P_STABLE_COL_LOCAL = "full_ges_p_stable_t0"
RISK_COL_LOCAL = "full_ges_instability_risk_t0"
RANK_COL_LOCAL = "full_ges_low_stability_rank"

required_columns = [
    KEY_COL_LOCAL,
    ROW_ORDER_COL_LOCAL,
    OUTCOME_COL_LOCAL,
    P_STABLE_COL_LOCAL,
    RISK_COL_LOCAL,
    RANK_COL_LOCAL,
]

missing_columns = [
    column
    for column in required_columns
    if column not in ranked_df.columns
]

assert not missing_columns, (
    f"Required ranked columns are missing: {missing_columns}"
)

N = int(len(ranked_df))

TOTAL_EVENTS = int(
    ranked_df[
        OUTCOME_COL_LOCAL
    ].sum()
)

TOTAL_NEGATIVES = int(
    N - TOTAL_EVENTS
)

OVERALL_EVENT_RATE = (
    TOTAL_EVENTS / N
)

TARGET_FRACTIONS = (
    0.05,
    0.10,
    0.20,
)

assert N == 66_636
assert TOTAL_EVENTS == 6_485
assert TOTAL_NEGATIVES == 60_151

assert ranked_df[
    RANK_COL_LOCAL
].tolist() == list(
    range(1, N + 1)
), (
    "The full-GES ranking is not the exact sequence 1 through N."
)

assert ranked_df[
    P_STABLE_COL_LOCAL
].is_monotonic_increasing, (
    "P(stable) is not monotonically increasing in the frozen ranking."
)


# --------------------------------------------------------------------------------------------------
# 2. Helper functions
# --------------------------------------------------------------------------------------------------

def safe_divide(
    numerator,
    denominator,
):
    """Return NaN when the denominator is zero."""

    if denominator == 0:
        return np.nan

    return numerator / denominator


def summarize_selection(
    target_fraction,
    target_n,
    cutoff_p_stable,
    selection_policy,
    selection_mask,
):
    """Summarize one frozen cutoff-selection policy."""

    selected_df = ranked_df.loc[
        selection_mask
    ]

    remaining_df = ranked_df.loc[
        ~selection_mask
    ]

    selected_n = int(
        len(selected_df)
    )

    remaining_n = int(
        len(remaining_df)
    )

    selected_events = int(
        selected_df[
            OUTCOME_COL_LOCAL
        ].sum()
    )

    remaining_events = int(
        remaining_df[
            OUTCOME_COL_LOCAL
        ].sum()
    )

    selected_event_rate = safe_divide(
        selected_events,
        selected_n,
    )

    remaining_event_rate = safe_divide(
        remaining_events,
        remaining_n,
    )

    event_rate_difference = (
        selected_event_rate
        - remaining_event_rate
        if (
            np.isfinite(selected_event_rate)
            and np.isfinite(remaining_event_rate)
        )
        else np.nan
    )

    risk_ratio_vs_remaining = (
        selected_event_rate
        / remaining_event_rate
        if (
            np.isfinite(selected_event_rate)
            and np.isfinite(remaining_event_rate)
            and remaining_event_rate > 0
        )
        else np.nan
    )

    enrichment_vs_overall = (
        selected_event_rate
        / OVERALL_EVENT_RATE
        if np.isfinite(selected_event_rate)
        else np.nan
    )

    event_capture_fraction = safe_divide(
        selected_events,
        TOTAL_EVENTS,
    )

    return {
        "target_percent": int(
            target_fraction * 100
        ),
        "nominal_target_records": (
            target_n
        ),
        "selection_policy": (
            selection_policy
        ),
        "cutoff_p_stable": float(
            cutoff_p_stable
        ),
        "cutoff_instability_risk": float(
            1.0 - cutoff_p_stable
        ),
        "selected_records": selected_n,
        "achieved_fraction": safe_divide(
            selected_n,
            N,
        ),
        "selected_events": (
            selected_events
        ),
        "selected_event_rate": (
            selected_event_rate
        ),
        "remaining_records": (
            remaining_n
        ),
        "remaining_events": (
            remaining_events
        ),
        "remaining_event_rate": (
            remaining_event_rate
        ),
        "event_rate_difference_vs_remaining": (
            event_rate_difference
        ),
        "risk_ratio_vs_remaining": (
            risk_ratio_vs_remaining
        ),
        "enrichment_vs_overall": (
            enrichment_vs_overall
        ),
        "event_capture_fraction": (
            event_capture_fraction
        ),
    }


# --------------------------------------------------------------------------------------------------
# 3. Calculate strict, exact-rank, and inclusive cutoff summaries
# --------------------------------------------------------------------------------------------------

policy_rows = []
tie_audit_rows = []
allocation_bound_rows = []

for target_fraction in TARGET_FRACTIONS:

    target_percent = int(
        target_fraction * 100
    )

    target_n = int(
        math.ceil(
            target_fraction * N
        )
    )

    cutoff_p_stable = float(
        ranked_df.iloc[
            target_n - 1
        ][P_STABLE_COL_LOCAL]
    )

    strict_mask = (
        ranked_df[
            P_STABLE_COL_LOCAL
        ]
        < cutoff_p_stable
    )

    tie_mask = (
        ranked_df[
            P_STABLE_COL_LOCAL
        ]
        == cutoff_p_stable
    )

    inclusive_mask = (
        ranked_df[
            P_STABLE_COL_LOCAL
        ]
        <= cutoff_p_stable
    )

    exact_rank_mask = (
        ranked_df[
            RANK_COL_LOCAL
        ]
        <= target_n
    )

    exact_selected_tie_mask = (
        tie_mask
        & exact_rank_mask
    )

    exact_unselected_tie_mask = (
        tie_mask
        & ~exact_rank_mask
    )

    strict_n = int(
        strict_mask.sum()
    )

    tied_n = int(
        tie_mask.sum()
    )

    inclusive_n = int(
        inclusive_mask.sum()
    )

    selected_from_tie_n = int(
        exact_selected_tie_mask.sum()
    )

    unselected_from_tie_n = int(
        exact_unselected_tie_mask.sum()
    )

    assert strict_n + tied_n == inclusive_n

    assert (
        strict_n
        + selected_from_tie_n
        == target_n
    ), (
        f"The exact-rank {target_percent}% group does not reconcile "
        "with the strict-lower and selected-tie groups."
    )

    assert (
        selected_from_tie_n
        + unselected_from_tie_n
        == tied_n
    )

    policy_rows.append(
        summarize_selection(
            target_fraction=target_fraction,
            target_n=target_n,
            cutoff_p_stable=cutoff_p_stable,
            selection_policy=(
                "Strictly below cutoff score"
            ),
            selection_mask=strict_mask,
        )
    )

    policy_rows.append(
        summarize_selection(
            target_fraction=target_fraction,
            target_n=target_n,
            cutoff_p_stable=cutoff_p_stable,
            selection_policy=(
                "Exact rank with frozen row-order tie break"
            ),
            selection_mask=exact_rank_mask,
        )
    )

    policy_rows.append(
        summarize_selection(
            target_fraction=target_fraction,
            target_n=target_n,
            cutoff_p_stable=cutoff_p_stable,
            selection_policy=(
                "Inclusive of complete cutoff tie"
            ),
            selection_mask=inclusive_mask,
        )
    )

    strict_events = int(
        ranked_df.loc[
            strict_mask,
            OUTCOME_COL_LOCAL,
        ].sum()
    )

    tied_events = int(
        ranked_df.loc[
            tie_mask,
            OUTCOME_COL_LOCAL,
        ].sum()
    )

    selected_tie_events = int(
        ranked_df.loc[
            exact_selected_tie_mask,
            OUTCOME_COL_LOCAL,
        ].sum()
    )

    unselected_tie_events = int(
        ranked_df.loc[
            exact_unselected_tie_mask,
            OUTCOME_COL_LOCAL,
        ].sum()
    )

    tied_event_rate = safe_divide(
        tied_events,
        tied_n,
    )

    selected_tie_event_rate = safe_divide(
        selected_tie_events,
        selected_from_tie_n,
    )

    unselected_tie_event_rate = safe_divide(
        unselected_tie_events,
        unselected_from_tie_n,
    )

    selected_vs_unselected_tie_difference = (
        selected_tie_event_rate
        - unselected_tie_event_rate
        if (
            np.isfinite(selected_tie_event_rate)
            and np.isfinite(unselected_tie_event_rate)
        )
        else np.nan
    )

    tie_rank_minimum = int(
        ranked_df.loc[
            tie_mask,
            RANK_COL_LOCAL,
        ].min()
    )

    tie_rank_maximum = int(
        ranked_df.loc[
            tie_mask,
            RANK_COL_LOCAL,
        ].max()
    )

    tie_audit_rows.append({
        "target_percent": (
            target_percent
        ),
        "target_records": (
            target_n
        ),
        "cutoff_p_stable": (
            cutoff_p_stable
        ),
        "cutoff_instability_risk": (
            1.0 - cutoff_p_stable
        ),
        "strictly_lower_records": (
            strict_n
        ),
        "cutoff_tied_records": (
            tied_n
        ),
        "inclusive_records": (
            inclusive_n
        ),
        "tie_first_rank": (
            tie_rank_minimum
        ),
        "tie_last_rank": (
            tie_rank_maximum
        ),
        "tied_records_selected_by_rank": (
            selected_from_tie_n
        ),
        "tied_records_not_selected_by_rank": (
            unselected_from_tie_n
        ),
        "cutoff_tied_events": (
            tied_events
        ),
        "cutoff_tied_event_rate": (
            tied_event_rate
        ),
        "selected_tie_events": (
            selected_tie_events
        ),
        "selected_tie_event_rate": (
            selected_tie_event_rate
        ),
        "unselected_tie_events": (
            unselected_tie_events
        ),
        "unselected_tie_event_rate": (
            unselected_tie_event_rate
        ),
        "selected_minus_unselected_tie_rate": (
            selected_vs_unselected_tie_difference
        ),
    })


    # ----------------------------------------------------------------------------------------------
    # Tie-allocation bounds
    #
    # Among M tied records containing E events, select exactly K records.
    #
    # Minimum events selectable:
    #   max(0, K - number_of_tied_negatives)
    #
    # Maximum events selectable:
    #   min(K, E)
    # ----------------------------------------------------------------------------------------------

    tied_negatives = int(
        tied_n - tied_events
    )

    minimum_selected_tie_events = int(
        max(
            0,
            selected_from_tie_n
            - tied_negatives,
        )
    )

    maximum_selected_tie_events = int(
        min(
            selected_from_tie_n,
            tied_events,
        )
    )

    observed_exact_events = int(
        strict_events
        + selected_tie_events
    )

    minimum_possible_exact_events = int(
        strict_events
        + minimum_selected_tie_events
    )

    maximum_possible_exact_events = int(
        strict_events
        + maximum_selected_tie_events
    )

    observed_exact_event_rate = (
        observed_exact_events
        / target_n
    )

    minimum_possible_exact_event_rate = (
        minimum_possible_exact_events
        / target_n
    )

    maximum_possible_exact_event_rate = (
        maximum_possible_exact_events
        / target_n
    )

    allocation_bound_rows.append({
        "target_percent": (
            target_percent
        ),
        "target_records": (
            target_n
        ),
        "cutoff_p_stable": (
            cutoff_p_stable
        ),
        "strictly_lower_records": (
            strict_n
        ),
        "strictly_lower_events": (
            strict_events
        ),
        "cutoff_tied_records": (
            tied_n
        ),
        "cutoff_tied_events": (
            tied_events
        ),
        "tied_records_required": (
            selected_from_tie_n
        ),
        "observed_selected_tie_events": (
            selected_tie_events
        ),
        "minimum_possible_selected_tie_events": (
            minimum_selected_tie_events
        ),
        "maximum_possible_selected_tie_events": (
            maximum_selected_tie_events
        ),
        "observed_exact_rank_events": (
            observed_exact_events
        ),
        "minimum_possible_exact_rank_events": (
            minimum_possible_exact_events
        ),
        "maximum_possible_exact_rank_events": (
            maximum_possible_exact_events
        ),
        "observed_exact_rank_event_rate": (
            observed_exact_event_rate
        ),
        "minimum_possible_exact_rank_event_rate": (
            minimum_possible_exact_event_rate
        ),
        "maximum_possible_exact_rank_event_rate": (
            maximum_possible_exact_event_rate
        ),
        "observed_enrichment_vs_overall": (
            observed_exact_event_rate
            / OVERALL_EVENT_RATE
        ),
        "minimum_possible_enrichment_vs_overall": (
            minimum_possible_exact_event_rate
            / OVERALL_EVENT_RATE
        ),
        "maximum_possible_enrichment_vs_overall": (
            maximum_possible_exact_event_rate
            / OVERALL_EVENT_RATE
        ),
    })


stage6c_full_ges_cutoff_policy_sensitivity = (
    pd.DataFrame(
        policy_rows
    )
)

stage6c_full_ges_cutoff_tie_audit = (
    pd.DataFrame(
        tie_audit_rows
    )
)

stage6c_full_ges_tie_allocation_bounds = (
    pd.DataFrame(
        allocation_bound_rows
    )
)


# --------------------------------------------------------------------------------------------------
# 4. Reconciliation checks
# --------------------------------------------------------------------------------------------------

assert len(
    stage6c_full_ges_cutoff_policy_sensitivity
) == 9

assert len(
    stage6c_full_ges_cutoff_tie_audit
) == 3

assert len(
    stage6c_full_ges_tie_allocation_bounds
) == 3

for target_fraction in TARGET_FRACTIONS:

    target_percent = int(
        target_fraction * 100
    )

    target_n = int(
        math.ceil(
            target_fraction * N
        )
    )

    exact_row = (
        stage6c_full_ges_cutoff_policy_sensitivity
        .loc[
            (
                stage6c_full_ges_cutoff_policy_sensitivity[
                    "target_percent"
                ]
                == target_percent
            )
            & (
                stage6c_full_ges_cutoff_policy_sensitivity[
                    "selection_policy"
                ]
                == "Exact rank with frozen row-order tie break"
            )
        ]
        .iloc[0]
    )

    assert int(
        exact_row[
            "selected_records"
        ]
    ) == target_n

    if "stage6c_top_risk_enrichment_table" in globals():

        prior_row = (
            stage6c_top_risk_enrichment_table
            .loc[
                stage6c_top_risk_enrichment_table[
                    "target_fraction"
                ]
                == target_fraction
            ]
            .iloc[0]
        )

        assert int(
            exact_row[
                "selected_events"
            ]
        ) == int(
            prior_row[
                "selected_events"
            ]
        )

        assert np.isclose(
            exact_row[
                "selected_event_rate"
            ],
            prior_row[
                "selected_event_rate"
            ],
            rtol=0.0,
            atol=1e-15,
        )


# --------------------------------------------------------------------------------------------------
# 5. Display results
# --------------------------------------------------------------------------------------------------

print()
print("=" * 126)
print(
    "STAGE 6C STEP 2B — CELL 6C-2B3 — "
    "FULL-GES CUTOFF-TIE SENSITIVITY AND TIE-ALLOCATION BOUNDS"
)
print("=" * 126)

print()
print("ANALYSIS BOUNDARY")
print("-" * 126)
print(
    f"Evaluable records                   : "
    f"{N:,}"
)
print(
    f"Primary instability events         : "
    f"{TOTAL_EVENTS:,}"
)
print(
    f"Overall event prevalence           : "
    f"{OVERALL_EVENT_RATE:.8f} "
    f"({OVERALL_EVENT_RATE * 100:.6f}%)"
)
print(
    "Cutoff policies evaluated          : "
    "strict, exact-rank, and tie-inclusive"
)
print(
    "Tie definition                     : "
    "exact equality of frozen P(stable)"
)
print(
    "Threshold or score optimization    : No"
)
print(
    "Scientific artifact written        : No"
)

print()
print("CUTOFF-TIE AUDIT")
print("-" * 126)

for row in tie_audit_rows:

    print()
    print(
        f"Lowest P(stable) {row['target_percent']}%"
    )
    print(
        f"  Target records                    : "
        f"{row['target_records']:,}"
    )
    print(
        f"  Cutoff P(stable)                  : "
        f"{row['cutoff_p_stable']:.12f}"
    )
    print(
        f"  Strictly lower records            : "
        f"{row['strictly_lower_records']:,}"
    )
    print(
        f"  Records tied at cutoff            : "
        f"{row['cutoff_tied_records']:,}"
    )
    print(
        f"  Tied records selected by rank     : "
        f"{row['tied_records_selected_by_rank']:,}"
    )
    print(
        f"  Tied records not selected         : "
        f"{row['tied_records_not_selected_by_rank']:,}"
    )
    print(
        f"  Entire cutoff-tie event rate      : "
        f"{row['cutoff_tied_event_rate']:.8f} "
        f"({row['cutoff_tied_event_rate'] * 100:.6f}%)"
    )
    print(
        f"  Selected tied-record event rate   : "
        f"{row['selected_tie_event_rate']:.8f} "
        f"({row['selected_tie_event_rate'] * 100:.6f}%)"
    )
    print(
        f"  Unselected tied-record event rate : "
        f"{row['unselected_tie_event_rate']:.8f} "
        f"({row['unselected_tie_event_rate'] * 100:.6f}%)"
    )
    print(
        f"  Selected-minus-unselected tie rate: "
        f"{row['selected_minus_unselected_tie_rate']:+.8f}"
    )

print()
print("STRICT, EXACT-RANK, AND TIE-INCLUSIVE POLICY RESULTS")
print("-" * 126)

display(
    stage6c_full_ges_cutoff_policy_sensitivity
)

print()
print("TIE-ALLOCATION EVENT-RATE BOUNDS")
print("-" * 126)

display(
    stage6c_full_ges_tie_allocation_bounds
)

print()
print("COMPLETE CUTOFF-TIE AUDIT TABLE")
print("-" * 126)

display(
    stage6c_full_ges_cutoff_tie_audit
)

print()
print("CELL DECISION")
print("-" * 126)
print(
    "PASS_STAGE6C_FULL_GES_CUTOFF_TIE_"
    "SENSITIVITY_COMPLETE"
)
print(
    "The prespecified rank-based groups were retained, while strict, "
    "tie-inclusive, and all mathematically possible cutoff-tie allocations "
    "were evaluated descriptively."
)
print(
    "No score, outcome, threshold, ranking, weight, or frozen scientific "
    "artifact was modified."
)


STAGE 6C STEP 2B — CELL 6C-2B3 — FULL-GES CUTOFF-TIE SENSITIVITY AND TIE-ALLOCATION BOUNDS

ANALYSIS BOUNDARY
------------------------------------------------------------------------------------------------------------------------------
Evaluable records                   : 66,636
Primary instability events         : 6,485
Overall event prevalence           : 0.09731977 (9.731977%)
Cutoff policies evaluated          : strict, exact-rank, and tie-inclusive
Tie definition                     : exact equality of frozen P(stable)
Threshold or score optimization    : No
Scientific artifact written        : No

CUTOFF-TIE AUDIT
------------------------------------------------------------------------------------------------------------------------------

Lowest P(stable) 5%
  Target records                    : 3,332
  Cutoff P(stable)                  : 0.234298251182
  Strictly lower records            : 2,363
  Records tied at cutoff            : 1,840
  Tied records selected by rank     

,target_percent,nominal_target_records,selection_policy,cutoff_p_stable,cutoff_instability_risk,selected_records,achieved_fraction,selected_events,selected_event_rate,remaining_records,remaining_events,remaining_event_rate,event_rate_difference_vs_remaining,risk_ratio_vs_remaining,enrichment_vs_overall,event_capture_fraction
0,5,3332,Strictly below cutoff score,0.234298,0.765702,2363,0.035461,406,0.171815,64273,6079,0.094581,0.077235,1.816598,1.765474,0.062606
1,5,3332,Exact rank with frozen row-order tie break,0.234298,0.765702,3332,0.050003,420,0.126050,63304,6065,0.095808,0.030243,1.315663,1.295219,0.064765
2,5,3332,Inclusive of complete cutoff tie,0.234298,0.765702,4203,0.063074,429,0.102070,62433,6056,0.097000,0.005070,1.052268,1.048810,0.066153
3,10,6664,Strictly below cutoff score,0.998650,0.001350,6657,0.099901,624,0.093736,59979,5861,0.097718,-0.003982,0.959254,0.963174,0.096222
4,10,6664,Exact rank with frozen row-order tie break,0.998650,0.001350,6664,0.100006,624,0.093637,59972,5861,0.097729,-0.004091,0.958134,0.962163,0.096222
5,10,6664,Inclusive of complete cutoff tie,0.998650,0.001350,6718,0.100816,628,0.093480,59918,5857,0.097750,-0.004270,0.956317,0.960547,0.096839
6,20,13328,Strictly below cutoff score,0.999909,0.000091,13326,0.199982,1335,0.100180,53310,5150,0.096605,0.003575,1.037010,1.029391,0.205860
7,20,13328,Exact rank with frozen row-order tie break,0.999909,0.000091,13328,0.200012,1335,0.100165,53308,5150,0.096608,0.003557,1.036815,1.029237,0.205860
8,20,13328,Inclusive of complete cutoff tie,0.999909,0.000091,13334,0.200102,1335,0.100120,53302,5150,0.096619,0.003501,1.036232,1.028773,0.205860



TIE-ALLOCATION EVENT-RATE BOUNDS
------------------------------------------------------------------------------------------------------------------------------


,target_percent,target_records,cutoff_p_stable,strictly_lower_records,strictly_lower_events,cutoff_tied_records,cutoff_tied_events,tied_records_required,observed_selected_tie_events,minimum_possible_selected_tie_events,maximum_possible_selected_tie_events,observed_exact_rank_events,minimum_possible_exact_rank_events,maximum_possible_exact_rank_events,observed_exact_rank_event_rate,minimum_possible_exact_rank_event_rate,maximum_possible_exact_rank_event_rate,observed_enrichment_vs_overall,minimum_possible_enrichment_vs_overall,maximum_possible_enrichment_vs_overall
0,5,3332,0.234298,2363,406,1840,23,969,14,0,23,420,406,429,0.126050,0.121849,0.128752,1.295219,1.252045,1.322974
1,10,6664,0.998650,6657,624,61,4,7,0,0,4,624,624,628,0.093637,0.093637,0.094238,0.962163,0.962163,0.968330
2,20,13328,0.999909,13326,1335,8,0,2,0,0,0,1335,1335,1335,0.100165,0.100165,0.100165,1.029237,1.029237,1.029237



COMPLETE CUTOFF-TIE AUDIT TABLE
------------------------------------------------------------------------------------------------------------------------------


,target_percent,target_records,cutoff_p_stable,cutoff_instability_risk,strictly_lower_records,cutoff_tied_records,inclusive_records,tie_first_rank,tie_last_rank,tied_records_selected_by_rank,tied_records_not_selected_by_rank,cutoff_tied_events,cutoff_tied_event_rate,selected_tie_events,selected_tie_event_rate,unselected_tie_events,unselected_tie_event_rate,selected_minus_unselected_tie_rate
0,5,3332,0.234298,0.765702,2363,1840,4203,2364,4203,969,871,23,0.012500,14,0.014448,9,0.010333,0.004115
1,10,6664,0.998650,0.001350,6657,61,6718,6658,6718,7,54,4,0.065574,0,0.000000,4,0.074074,-0.074074
2,20,13328,0.999909,0.000091,13326,8,13334,13327,13334,2,6,0,0.000000,0,0.000000,0,0.000000,0.000000



CELL DECISION
------------------------------------------------------------------------------------------------------------------------------
PASS_STAGE6C_FULL_GES_CUTOFF_TIE_SENSITIVITY_COMPLETE
The prespecified rank-based groups were retained, while strict, tie-inclusive, and all mathematically possible cutoff-tie allocations were evaluated descriptively.
No score, outcome, threshold, ranking, weight, or frozen scientific artifact was modified.


In [18]:
# ==================================================================================================
# STAGE 6C — STEP 2B
# CELL 6C-2B4 — FULL-GES TOP-RISK ENRICHMENT BOOTSTRAP INFERENCE
#
# Purpose:
#   1. Quantify sampling uncertainty for the prespecified lowest-P(stable) 5%, 10%, and 20%
#      exact-rank groups.
#   2. Use 2,000 nonparametric row-bootstrap replicates with seed 42.
#   3. Use identical bootstrap resamples across all three nested risk groups.
#   4. Report percentile 95% intervals for:
#        - selected-group event rate;
#        - remaining-group event rate;
#        - event-rate difference;
#        - risk ratio versus remaining records;
#        - enrichment versus bootstrap prevalence;
#        - fraction of all events captured.
#
# Scientific boundary:
#   - Group membership is held fixed at the exact frozen rank membership established in 6C-2B1.
#   - This estimates sampling uncertainty conditional on the locked rank groups.
#   - Boundary-score uncertainty was evaluated separately in Cell 6C-2B3.
#   - No score cutoff is reselected inside bootstrap samples.
#   - No score, outcome, threshold, weight, or ranking is modified.
#   - No scientific artifact is written.
# ==================================================================================================

import math
import time

import numpy as np
import pandas as pd
from IPython.display import display


# --------------------------------------------------------------------------------------------------
# 1. Confirm required in-memory object
# --------------------------------------------------------------------------------------------------

assert "stage6c_full_ges_ranked_membership" in globals(), (
    "The ranked full-GES dataframe is unavailable. "
    "Run Cell 6C-2B1 before this cell."
)

ranked_df = (
    stage6c_full_ges_ranked_membership
    .copy()
)

OUTCOME_COL_LOCAL = "primary_future_instability"
RANK_COL_LOCAL = "full_ges_low_stability_rank"
P_STABLE_COL_LOCAL = "full_ges_p_stable_t0"

required_columns = [
    OUTCOME_COL_LOCAL,
    RANK_COL_LOCAL,
    P_STABLE_COL_LOCAL,
]

missing_columns = [
    column
    for column in required_columns
    if column not in ranked_df.columns
]

assert not missing_columns, (
    f"Required columns are missing: {missing_columns}"
)


# --------------------------------------------------------------------------------------------------
# 2. Locked analysis constants
# --------------------------------------------------------------------------------------------------

N = int(
    len(ranked_df)
)

TOTAL_EVENTS = int(
    ranked_df[
        OUTCOME_COL_LOCAL
    ].sum()
)

TOTAL_NEGATIVES = int(
    N - TOTAL_EVENTS
)

OVERALL_EVENT_RATE = (
    TOTAL_EVENTS / N
)

BOOTSTRAP_REPLICATES = 2_000
BOOTSTRAP_SEED = 42
BOOTSTRAP_BATCH_SIZE = 25

TARGETS = [
    {
        "target_percent": 5,
        "target_fraction": 0.05,
        "short_label": "05pct",
    },
    {
        "target_percent": 10,
        "target_fraction": 0.10,
        "short_label": "10pct",
    },
    {
        "target_percent": 20,
        "target_fraction": 0.20,
        "short_label": "20pct",
    },
]

assert N == 66_636
assert TOTAL_EVENTS == 6_485
assert TOTAL_NEGATIVES == 60_151

assert ranked_df[
    RANK_COL_LOCAL
].tolist() == list(
    range(1, N + 1)
), (
    "The frozen full-GES rank is not the exact "
    "sequence 1 through N."
)

assert ranked_df[
    P_STABLE_COL_LOCAL
].is_monotonic_increasing, (
    "P(stable) is not monotonically increasing "
    "in the frozen ranking."
)


# --------------------------------------------------------------------------------------------------
# 3. Construct the three fixed nested exact-rank memberships
# --------------------------------------------------------------------------------------------------

rank_array = ranked_df[
    RANK_COL_LOCAL
].to_numpy(
    dtype=np.int64
)

outcome_array = ranked_df[
    OUTCOME_COL_LOCAL
].to_numpy(
    dtype=np.int8
)

target_record_counts = np.asarray(
    [
        int(
            math.ceil(
                target[
                    "target_fraction"
                ]
                * N
            )
        )
        for target in TARGETS
    ],
    dtype=np.int64,
)

membership_matrix = np.column_stack(
    [
        rank_array
        <= target_record_count
        for target_record_count
        in target_record_counts
    ]
).astype(
    bool
)

assert membership_matrix.shape == (
    N,
    len(TARGETS),
)

assert np.array_equal(
    membership_matrix.sum(
        axis=0
    ),
    target_record_counts,
), (
    "Fixed membership counts do not match "
    "the prespecified exact-rank group sizes."
)

# Verify nesting: 5% subset of 10%, and 10% subset of 20%.
assert np.all(
    ~membership_matrix[:, 0]
    | membership_matrix[:, 1]
)

assert np.all(
    ~membership_matrix[:, 1]
    | membership_matrix[:, 2]
)


# --------------------------------------------------------------------------------------------------
# 4. Calculate locked point estimates
# --------------------------------------------------------------------------------------------------

def calculate_point_metrics(
    membership: np.ndarray,
) -> dict:
    """Calculate enrichment metrics for one fixed locked group."""

    selected_n = int(
        membership.sum()
    )

    selected_events = int(
        outcome_array[
            membership
        ].sum()
    )

    remaining_n = int(
        N - selected_n
    )

    remaining_events = int(
        TOTAL_EVENTS - selected_events
    )

    selected_event_rate = (
        selected_events / selected_n
    )

    remaining_event_rate = (
        remaining_events / remaining_n
    )

    event_rate_difference = (
        selected_event_rate
        - remaining_event_rate
    )

    risk_ratio_vs_remaining = (
        selected_event_rate
        / remaining_event_rate
    )

    enrichment_vs_overall = (
        selected_event_rate
        / OVERALL_EVENT_RATE
    )

    event_capture_fraction = (
        selected_events
        / TOTAL_EVENTS
    )

    return {
        "selected_records": selected_n,
        "selected_events": selected_events,
        "selected_event_rate": selected_event_rate,
        "remaining_records": remaining_n,
        "remaining_events": remaining_events,
        "remaining_event_rate": remaining_event_rate,
        "event_rate_difference_vs_remaining": (
            event_rate_difference
        ),
        "risk_ratio_vs_remaining": (
            risk_ratio_vs_remaining
        ),
        "enrichment_vs_overall": (
            enrichment_vs_overall
        ),
        "event_capture_fraction": (
            event_capture_fraction
        ),
    }


point_estimates = []

for target_index, target in enumerate(
    TARGETS
):
    metrics = calculate_point_metrics(
        membership_matrix[
            :,
            target_index,
        ]
    )

    point_estimates.append({
        **target,
        **metrics,
    })


stage6c_full_ges_enrichment_point_estimates_df = (
    pd.DataFrame(
        point_estimates
    )
)


# Reconcile against Cell 6C-2B1 when available.
if "stage6c_top_risk_enrichment_table" in globals():

    for target in TARGETS:

        target_fraction = target[
            "target_fraction"
        ]

        current_row = (
            stage6c_full_ges_enrichment_point_estimates_df
            .loc[
                stage6c_full_ges_enrichment_point_estimates_df[
                    "target_fraction"
                ]
                == target_fraction
            ]
            .iloc[0]
        )

        prior_row = (
            stage6c_top_risk_enrichment_table
            .loc[
                stage6c_top_risk_enrichment_table[
                    "target_fraction"
                ]
                == target_fraction
            ]
            .iloc[0]
        )

        assert int(
            current_row[
                "selected_records"
            ]
        ) == int(
            prior_row[
                "selected_records"
            ]
        )

        assert int(
            current_row[
                "selected_events"
            ]
        ) == int(
            prior_row[
                "selected_events"
            ]
        )

        assert np.isclose(
            current_row[
                "selected_event_rate"
            ],
            prior_row[
                "selected_event_rate"
            ],
            rtol=0.0,
            atol=1e-15,
        )


# --------------------------------------------------------------------------------------------------
# 5. Preallocate bootstrap arrays
# --------------------------------------------------------------------------------------------------

number_of_targets = len(
    TARGETS
)

bootstrap_selected_records = np.empty(
    (
        BOOTSTRAP_REPLICATES,
        number_of_targets,
    ),
    dtype=np.int32,
)

bootstrap_selected_events = np.empty(
    (
        BOOTSTRAP_REPLICATES,
        number_of_targets,
    ),
    dtype=np.int32,
)

bootstrap_total_events = np.empty(
    BOOTSTRAP_REPLICATES,
    dtype=np.int32,
)

bootstrap_selected_event_rate = np.empty(
    (
        BOOTSTRAP_REPLICATES,
        number_of_targets,
    ),
    dtype=np.float64,
)

bootstrap_remaining_event_rate = np.empty(
    (
        BOOTSTRAP_REPLICATES,
        number_of_targets,
    ),
    dtype=np.float64,
)

bootstrap_event_rate_difference = np.empty(
    (
        BOOTSTRAP_REPLICATES,
        number_of_targets,
    ),
    dtype=np.float64,
)

bootstrap_risk_ratio = np.empty(
    (
        BOOTSTRAP_REPLICATES,
        number_of_targets,
    ),
    dtype=np.float64,
)

bootstrap_enrichment_vs_overall = np.empty(
    (
        BOOTSTRAP_REPLICATES,
        number_of_targets,
    ),
    dtype=np.float64,
)

bootstrap_event_capture_fraction = np.empty(
    (
        BOOTSTRAP_REPLICATES,
        number_of_targets,
    ),
    dtype=np.float64,
)


# --------------------------------------------------------------------------------------------------
# 6. Run 2,000 paired row-bootstrap replicates in memory-safe batches
#
# One bootstrap sample is drawn for each replicate.
# The same draw is used for all three target groups.
# --------------------------------------------------------------------------------------------------

rng = np.random.default_rng(
    BOOTSTRAP_SEED
)

bootstrap_start_time = time.time()

for batch_start in range(
    0,
    BOOTSTRAP_REPLICATES,
    BOOTSTRAP_BATCH_SIZE,
):

    batch_stop = min(
        batch_start
        + BOOTSTRAP_BATCH_SIZE,
        BOOTSTRAP_REPLICATES,
    )

    current_batch_size = (
        batch_stop
        - batch_start
    )

    # Nonparametric row bootstrap.
    draw_indices = rng.integers(
        low=0,
        high=N,
        size=(
            current_batch_size,
            N,
        ),
        dtype=np.int32,
    )

    sampled_outcomes = outcome_array[
        draw_indices
    ]

    sampled_memberships = membership_matrix[
        draw_indices,
        :,
    ]

    selected_records_batch = (
        sampled_memberships
        .sum(
            axis=1,
            dtype=np.int64,
        )
    )

    selected_events_batch = (
        sampled_memberships
        * sampled_outcomes[
            :,
            :,
            np.newaxis,
        ]
    ).sum(
        axis=1,
        dtype=np.int64,
    )

    total_events_batch = (
        sampled_outcomes
        .sum(
            axis=1,
            dtype=np.int64,
        )
    )

    remaining_records_batch = (
        N
        - selected_records_batch
    )

    remaining_events_batch = (
        total_events_batch[
            :,
            np.newaxis,
        ]
        - selected_events_batch
    )

    overall_event_rate_batch = (
        total_events_batch
        / N
    )

    selected_event_rate_batch = (
        selected_events_batch
        / selected_records_batch
    )

    remaining_event_rate_batch = (
        remaining_events_batch
        / remaining_records_batch
    )

    event_rate_difference_batch = (
        selected_event_rate_batch
        - remaining_event_rate_batch
    )

    risk_ratio_batch = (
        selected_event_rate_batch
        / remaining_event_rate_batch
    )

    enrichment_vs_overall_batch = (
        selected_event_rate_batch
        / overall_event_rate_batch[
            :,
            np.newaxis,
        ]
    )

    event_capture_fraction_batch = (
        selected_events_batch
        / total_events_batch[
            :,
            np.newaxis,
        ]
    )

    bootstrap_selected_records[
        batch_start:batch_stop,
        :,
    ] = selected_records_batch

    bootstrap_selected_events[
        batch_start:batch_stop,
        :,
    ] = selected_events_batch

    bootstrap_total_events[
        batch_start:batch_stop
    ] = total_events_batch

    bootstrap_selected_event_rate[
        batch_start:batch_stop,
        :,
    ] = selected_event_rate_batch

    bootstrap_remaining_event_rate[
        batch_start:batch_stop,
        :,
    ] = remaining_event_rate_batch

    bootstrap_event_rate_difference[
        batch_start:batch_stop,
        :,
    ] = event_rate_difference_batch

    bootstrap_risk_ratio[
        batch_start:batch_stop,
        :,
    ] = risk_ratio_batch

    bootstrap_enrichment_vs_overall[
        batch_start:batch_stop,
        :,
    ] = enrichment_vs_overall_batch

    bootstrap_event_capture_fraction[
        batch_start:batch_stop,
        :,
    ] = event_capture_fraction_batch

    completed_replicates = batch_stop

    if (
        completed_replicates % 250 == 0
        or completed_replicates
        == BOOTSTRAP_REPLICATES
    ):
        elapsed_seconds = (
            time.time()
            - bootstrap_start_time
        )

        print(
            f"Completed "
            f"{completed_replicates:,}/"
            f"{BOOTSTRAP_REPLICATES:,} "
            f"bootstrap replicates "
            f"({elapsed_seconds:.1f} seconds elapsed)"
        )


bootstrap_elapsed_seconds = (
    time.time()
    - bootstrap_start_time
)


# --------------------------------------------------------------------------------------------------
# 7. Bootstrap QC
# --------------------------------------------------------------------------------------------------

assert (
    bootstrap_total_events
    > 0
).all(), (
    "At least one bootstrap replicate contains zero events."
)

assert (
    bootstrap_selected_records
    > 0
).all()

assert (
    bootstrap_selected_records
    < N
).all()

assert (
    bootstrap_selected_events
    <= bootstrap_selected_records
).all()

# Nested membership must be retained in every replicate.
assert (
    bootstrap_selected_records[:, 0]
    <= bootstrap_selected_records[:, 1]
).all()

assert (
    bootstrap_selected_records[:, 1]
    <= bootstrap_selected_records[:, 2]
).all()

assert (
    bootstrap_selected_events[:, 0]
    <= bootstrap_selected_events[:, 1]
).all()

assert (
    bootstrap_selected_events[:, 1]
    <= bootstrap_selected_events[:, 2]
).all()

metric_arrays = {
    "selected_event_rate": (
        bootstrap_selected_event_rate
    ),
    "remaining_event_rate": (
        bootstrap_remaining_event_rate
    ),
    "event_rate_difference_vs_remaining": (
        bootstrap_event_rate_difference
    ),
    "risk_ratio_vs_remaining": (
        bootstrap_risk_ratio
    ),
    "enrichment_vs_overall": (
        bootstrap_enrichment_vs_overall
    ),
    "event_capture_fraction": (
        bootstrap_event_capture_fraction
    ),
}

for metric_name, metric_array in metric_arrays.items():

    assert np.isfinite(
        metric_array
    ).all(), (
        f"Bootstrap metric contains a nonfinite value: "
        f"{metric_name}"
    )


# --------------------------------------------------------------------------------------------------
# 8. Create raw replicate dataframe
# --------------------------------------------------------------------------------------------------

replicate_data = {
    "bootstrap_replicate": np.arange(
        1,
        BOOTSTRAP_REPLICATES + 1,
        dtype=np.int32,
    ),
    "bootstrap_total_events": (
        bootstrap_total_events
    ),
    "bootstrap_overall_event_rate": (
        bootstrap_total_events
        / N
    ),
}

for target_index, target in enumerate(
    TARGETS
):

    short_label = target[
        "short_label"
    ]

    replicate_data[
        f"selected_records_{short_label}"
    ] = bootstrap_selected_records[
        :,
        target_index,
    ]

    replicate_data[
        f"selected_events_{short_label}"
    ] = bootstrap_selected_events[
        :,
        target_index,
    ]

    replicate_data[
        f"selected_event_rate_{short_label}"
    ] = bootstrap_selected_event_rate[
        :,
        target_index,
    ]

    replicate_data[
        f"remaining_event_rate_{short_label}"
    ] = bootstrap_remaining_event_rate[
        :,
        target_index,
    ]

    replicate_data[
        f"event_rate_difference_{short_label}"
    ] = bootstrap_event_rate_difference[
        :,
        target_index,
    ]

    replicate_data[
        f"risk_ratio_vs_remaining_{short_label}"
    ] = bootstrap_risk_ratio[
        :,
        target_index,
    ]

    replicate_data[
        f"enrichment_vs_overall_{short_label}"
    ] = bootstrap_enrichment_vs_overall[
        :,
        target_index,
    ]

    replicate_data[
        f"event_capture_fraction_{short_label}"
    ] = bootstrap_event_capture_fraction[
        :,
        target_index,
    ]


stage6c_full_ges_enrichment_bootstrap_replicates_df = (
    pd.DataFrame(
        replicate_data
    )
)

assert len(
    stage6c_full_ges_enrichment_bootstrap_replicates_df
) == BOOTSTRAP_REPLICATES


# --------------------------------------------------------------------------------------------------
# 9. Build percentile-interval summary table
# --------------------------------------------------------------------------------------------------

POINT_METRIC_COLUMNS = {
    "selected_event_rate": (
        "selected_event_rate"
    ),
    "remaining_event_rate": (
        "remaining_event_rate"
    ),
    "event_rate_difference_vs_remaining": (
        "event_rate_difference_vs_remaining"
    ),
    "risk_ratio_vs_remaining": (
        "risk_ratio_vs_remaining"
    ),
    "enrichment_vs_overall": (
        "enrichment_vs_overall"
    ),
    "event_capture_fraction": (
        "event_capture_fraction"
    ),
}

METRIC_NULL_VALUES = {
    "selected_event_rate": np.nan,
    "remaining_event_rate": np.nan,
    "event_rate_difference_vs_remaining": 0.0,
    "risk_ratio_vs_remaining": 1.0,
    "enrichment_vs_overall": 1.0,
    "event_capture_fraction": np.nan,
}

summary_rows = []

for target_index, target in enumerate(
    TARGETS
):

    point_row = (
        stage6c_full_ges_enrichment_point_estimates_df
        .loc[
            stage6c_full_ges_enrichment_point_estimates_df[
                "target_percent"
            ]
            == target[
                "target_percent"
            ]
        ]
        .iloc[0]
    )

    for metric_name, metric_array in metric_arrays.items():

        values = metric_array[
            :,
            target_index,
        ]

        point_estimate = float(
            point_row[
                POINT_METRIC_COLUMNS[
                    metric_name
                ]
            ]
        )

        bootstrap_mean = float(
            np.mean(
                values
            )
        )

        bootstrap_standard_error = float(
            np.std(
                values,
                ddof=1,
            )
        )

        interval_lower, interval_upper = (
            np.quantile(
                values,
                [
                    0.025,
                    0.975,
                ],
            )
        )

        null_value = METRIC_NULL_VALUES[
            metric_name
        ]

        if np.isfinite(
            null_value
        ):

            bootstrap_support_above_null = float(
                (
                    np.sum(
                        values
                        > null_value
                    )
                    + 0.5
                    * np.sum(
                        values
                        == null_value
                    )
                )
                / BOOTSTRAP_REPLICATES
            )

            interval_excludes_null_above = bool(
                interval_lower
                > null_value
            )

            interval_excludes_null_below = bool(
                interval_upper
                < null_value
            )

        else:

            bootstrap_support_above_null = np.nan
            interval_excludes_null_above = False
            interval_excludes_null_below = False

        summary_rows.append({
            "target_percent": (
                target[
                    "target_percent"
                ]
            ),
            "target_fraction": (
                target[
                    "target_fraction"
                ]
            ),
            "metric": metric_name,
            "point_estimate": (
                point_estimate
            ),
            "bootstrap_mean": (
                bootstrap_mean
            ),
            "bootstrap_standard_error": (
                bootstrap_standard_error
            ),
            "percentile_95_ci_lower": float(
                interval_lower
            ),
            "percentile_95_ci_upper": float(
                interval_upper
            ),
            "null_value": (
                null_value
            ),
            "bootstrap_support_above_null": (
                bootstrap_support_above_null
            ),
            "interval_excludes_null_above": (
                interval_excludes_null_above
            ),
            "interval_excludes_null_below": (
                interval_excludes_null_below
            ),
            "bootstrap_replicates": (
                BOOTSTRAP_REPLICATES
            ),
            "bootstrap_seed": (
                BOOTSTRAP_SEED
            ),
            "membership_definition": (
                "Fixed exact-rank membership"
            ),
        })


stage6c_full_ges_enrichment_bootstrap_summary_df = (
    pd.DataFrame(
        summary_rows
    )
)


# --------------------------------------------------------------------------------------------------
# 10. Create compact key-inference table
# --------------------------------------------------------------------------------------------------

key_metrics = [
    "event_rate_difference_vs_remaining",
    "risk_ratio_vs_remaining",
    "enrichment_vs_overall",
]

stage6c_full_ges_enrichment_bootstrap_key_inference_df = (
    stage6c_full_ges_enrichment_bootstrap_summary_df
    .loc[
        stage6c_full_ges_enrichment_bootstrap_summary_df[
            "metric"
        ].isin(
            key_metrics
        )
    ]
    .copy()
    .sort_values(
        by=[
            "target_percent",
            "metric",
        ],
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)


# --------------------------------------------------------------------------------------------------
# 11. Create bootstrap count-QC table
# --------------------------------------------------------------------------------------------------

count_qc_rows = []

for target_index, target in enumerate(
    TARGETS
):

    selected_count_values = (
        bootstrap_selected_records[
            :,
            target_index,
        ]
    )

    selected_event_values = (
        bootstrap_selected_events[
            :,
            target_index,
        ]
    )

    count_qc_rows.append({
        "target_percent": (
            target[
                "target_percent"
            ]
        ),
        "locked_selected_records": int(
            target_record_counts[
                target_index
            ]
        ),
        "bootstrap_mean_selected_records": float(
            np.mean(
                selected_count_values
            )
        ),
        "bootstrap_minimum_selected_records": int(
            np.min(
                selected_count_values
            )
        ),
        "bootstrap_maximum_selected_records": int(
            np.max(
                selected_count_values
            )
        ),
        "bootstrap_mean_selected_events": float(
            np.mean(
                selected_event_values
            )
        ),
        "bootstrap_minimum_selected_events": int(
            np.min(
                selected_event_values
            )
        ),
        "bootstrap_maximum_selected_events": int(
            np.max(
                selected_event_values
            )
        ),
    })


stage6c_full_ges_enrichment_bootstrap_count_qc_df = (
    pd.DataFrame(
        count_qc_rows
    )
)


# --------------------------------------------------------------------------------------------------
# 12. Display results
# --------------------------------------------------------------------------------------------------

print()
print("=" * 128)
print(
    "STAGE 6C STEP 2B — CELL 6C-2B4 — "
    "FULL-GES TOP-RISK ENRICHMENT BOOTSTRAP INFERENCE"
)
print("=" * 128)

print()
print("BOOTSTRAP DESIGN")
print("-" * 128)
print(
    f"Evaluable records                   : "
    f"{N:,}"
)
print(
    f"Primary instability events         : "
    f"{TOTAL_EVENTS:,}"
)
print(
    f"Observed prevalence                : "
    f"{OVERALL_EVENT_RATE:.8f} "
    f"({OVERALL_EVENT_RATE * 100:.6f}%)"
)
print(
    f"Bootstrap replicates               : "
    f"{BOOTSTRAP_REPLICATES:,}"
)
print(
    f"Random seed                        : "
    f"{BOOTSTRAP_SEED}"
)
print(
    "Resampling unit                    : "
    "Individual frozen evaluable rows"
)
print(
    "Resampling relationship            : "
    "Identical draws across 5%, 10%, and 20% groups"
)
print(
    "Membership handling                : "
    "Fixed locked exact-rank membership"
)
print(
    "Cutoff-tie uncertainty             : "
    "Evaluated separately in Cell 6C-2B3"
)
print(
    f"Elapsed bootstrap time             : "
    f"{bootstrap_elapsed_seconds:.2f} seconds"
)
print(
    "Scientific artifact written        : No"
)

print()
print("LOCKED POINT ESTIMATES")
print("-" * 128)

display(
    stage6c_full_ges_enrichment_point_estimates_df
)

print()
print("KEY BOOTSTRAP INFERENCE")
print("-" * 128)

for target in TARGETS:

    target_percent = target[
        "target_percent"
    ]

    print()
    print(
        f"Lowest P(stable) {target_percent}%"
    )

    target_rows = (
        stage6c_full_ges_enrichment_bootstrap_key_inference_df
        .loc[
            stage6c_full_ges_enrichment_bootstrap_key_inference_df[
                "target_percent"
            ]
            == target_percent
        ]
    )

    for _, row in target_rows.iterrows():

        print(
            f"  {row['metric']}"
        )
        print(
            f"    Point estimate                 : "
            f"{row['point_estimate']:.8f}"
        )
        print(
            f"    Bootstrap mean                : "
            f"{row['bootstrap_mean']:.8f}"
        )
        print(
            f"    Bootstrap standard error      : "
            f"{row['bootstrap_standard_error']:.8f}"
        )
        print(
            f"    95% percentile interval       : "
            f"[{row['percentile_95_ci_lower']:.8f}, "
            f"{row['percentile_95_ci_upper']:.8f}]"
        )
        print(
            f"    Bootstrap support above null  : "
            f"{row['bootstrap_support_above_null']:.6f}"
        )
        print(
            f"    Interval excludes null above  : "
            f"{row['interval_excludes_null_above']}"
        )
        print(
            f"    Interval excludes null below  : "
            f"{row['interval_excludes_null_below']}"
        )

print()
print("COMPLETE BOOTSTRAP SUMMARY")
print("-" * 128)

display(
    stage6c_full_ges_enrichment_bootstrap_summary_df
)

print()
print("BOOTSTRAP COUNT QC")
print("-" * 128)

display(
    stage6c_full_ges_enrichment_bootstrap_count_qc_df
)

print()
print("RAW REPLICATE DATAFRAME")
print("-" * 128)
print(
    f"Shape: "
    f"{stage6c_full_ges_enrichment_bootstrap_replicates_df.shape}"
)

display(
    stage6c_full_ges_enrichment_bootstrap_replicates_df.head(
        10
    )
)

print()
print("CELL DECISION")
print("-" * 128)
print(
    "PASS_STAGE6C_FULL_GES_TOP_RISK_ENRICHMENT_"
    "BOOTSTRAP_INFERENCE_COMPLETE"
)
print(
    "The percentile intervals estimate sampling uncertainty "
    "conditional on the frozen exact-rank memberships."
)
print(
    "Cutoff-tie allocation sensitivity remains separately documented "
    "by Cell 6C-2B3."
)
print(
    "No score, outcome, cutoff, threshold, weight, ranking, or frozen "
    "scientific artifact was modified."
)

Completed 250/2,000 bootstrap replicates (4.4 seconds elapsed)
Completed 500/2,000 bootstrap replicates (8.1 seconds elapsed)
Completed 750/2,000 bootstrap replicates (10.8 seconds elapsed)
Completed 1,000/2,000 bootstrap replicates (14.4 seconds elapsed)
Completed 1,250/2,000 bootstrap replicates (17.1 seconds elapsed)
Completed 1,500/2,000 bootstrap replicates (21.1 seconds elapsed)
Completed 1,750/2,000 bootstrap replicates (23.1 seconds elapsed)
Completed 2,000/2,000 bootstrap replicates (24.6 seconds elapsed)

STAGE 6C STEP 2B — CELL 6C-2B4 — FULL-GES TOP-RISK ENRICHMENT BOOTSTRAP INFERENCE

BOOTSTRAP DESIGN
--------------------------------------------------------------------------------------------------------------------------------
Evaluable records                   : 66,636
Primary instability events         : 6,485
Observed prevalence                : 0.09731977 (9.731977%)
Bootstrap replicates               : 2,000
Random seed                        : 42
Resampling unit    

,target_percent,target_fraction,short_label,selected_records,selected_events,selected_event_rate,remaining_records,remaining_events,remaining_event_rate,event_rate_difference_vs_remaining,risk_ratio_vs_remaining,enrichment_vs_overall,event_capture_fraction
0,5,0.05,05pct,3332,420,0.126050,63304,6065,0.095808,0.030243,1.315663,1.295219,0.064765
1,10,0.10,10pct,6664,624,0.093637,59972,5861,0.097729,-0.004091,0.958134,0.962163,0.096222
2,20,0.20,20pct,13328,1335,0.100165,53308,5150,0.096608,0.003557,1.036815,1.029237,0.205860



KEY BOOTSTRAP INFERENCE
--------------------------------------------------------------------------------------------------------------------------------

Lowest P(stable) 5%
  enrichment_vs_overall
    Point estimate                 : 1.29521909
    Bootstrap mean                : 1.29381120
    Bootstrap standard error      : 0.05746378
    95% percentile interval       : [1.18209727, 1.40790567]
    Bootstrap support above null  : 1.000000
    Interval excludes null above  : True
    Interval excludes null below  : False
  event_rate_difference_vs_remaining
    Point estimate                 : 0.03024289
    Bootstrap mean                : 0.03009470
    Bootstrap standard error      : 0.00589713
    95% percentile interval       : [0.01863014, 0.04178359]
    Bootstrap support above null  : 1.000000
    Interval excludes null above  : True
    Interval excludes null below  : False
  risk_ratio_vs_remaining
    Point estimate                 : 1.31566295
    Bootstrap mean          

,target_percent,target_fraction,metric,point_estimate,bootstrap_mean,bootstrap_standard_error,percentile_95_ci_lower,percentile_95_ci_upper,null_value,bootstrap_support_above_null,interval_excludes_null_above,interval_excludes_null_below,bootstrap_replicates,bootstrap_seed,membership_definition
0,5,0.05,selected_event_rate,0.126050,0.125895,0.005782,0.114753,0.137538,NaN,NaN,False,False,2000,42,Fixed exact-rank membership
1,5,0.05,remaining_event_rate,0.095808,0.095801,0.001165,0.093560,0.098130,NaN,NaN,False,False,2000,42,Fixed exact-rank membership
2,5,0.05,event_rate_difference_vs_remaining,0.030243,0.030095,0.005897,0.018630,0.041784,0.0,1.0000,True,False,2000,42,Fixed exact-rank membership
3,5,0.05,risk_ratio_vs_remaining,1.315663,1.314331,0.062429,1.193559,1.438717,1.0,1.0000,True,False,2000,42,Fixed exact-rank membership
4,5,0.05,enrichment_vs_overall,1.295219,1.293811,0.057464,1.182097,1.407906,1.0,1.0000,True,False,2000,42,Fixed exact-rank membership
5,5,0.05,event_capture_fraction,0.064765,0.064702,0.003111,0.058631,0.070775,NaN,NaN,False,False,2000,42,Fixed exact-rank membership
6,10,0.10,selected_event_rate,0.093637,0.093567,0.003651,0.086289,0.100733,NaN,NaN,False,False,2000,42,Fixed exact-rank membership
7,10,0.10,remaining_event_rate,0.097729,0.097721,0.001213,0.095417,0.100158,NaN,NaN,False,False,2000,42,Fixed exact-rank membership
8,10,0.10,event_rate_difference_vs_remaining,-0.004091,-0.004154,0.003869,-0.011649,0.003233,0.0,0.1445,False,False,2000,42,Fixed exact-rank membership
9,10,0.10,risk_ratio_vs_remaining,0.958134,0.957650,0.039443,0.881662,1.033364,1.0,0.1445,False,False,2000,42,Fixed exact-rank membership



BOOTSTRAP COUNT QC
--------------------------------------------------------------------------------------------------------------------------------


,target_percent,locked_selected_records,bootstrap_mean_selected_records,bootstrap_minimum_selected_records,bootstrap_maximum_selected_records,bootstrap_mean_selected_events,bootstrap_minimum_selected_events,bootstrap_maximum_selected_events
0,5,3332,3332.3145,3149,3534,419.5370,354,490
1,10,6664,6664.9835,6369,6939,623.6365,544,715
2,20,13328,13327.7790,13025,13742,1333.7795,1215,1444



RAW REPLICATE DATAFRAME
--------------------------------------------------------------------------------------------------------------------------------
Shape: (2000, 27)


,bootstrap_replicate,bootstrap_total_events,bootstrap_overall_event_rate,selected_records_05pct,selected_events_05pct,selected_event_rate_05pct,remaining_event_rate_05pct,event_rate_difference_05pct,risk_ratio_vs_remaining_05pct,enrichment_vs_overall_05pct,...,enrichment_vs_overall_10pct,event_capture_fraction_10pct,selected_records_20pct,selected_events_20pct,selected_event_rate_20pct,remaining_event_rate_20pct,event_rate_difference_20pct,risk_ratio_vs_remaining_20pct,enrichment_vs_overall_20pct,event_capture_fraction_20pct
0,1,6507,0.097650,3359,414,0.123251,0.096291,0.026960,1.279985,1.262172,...,0.923619,0.091286,13307,1323,0.099421,0.097208,0.002213,1.022770,1.018141,0.203320
1,2,6490,0.097395,3278,393,0.119890,0.096231,0.023659,1.245859,1.230971,...,0.945527,0.093991,13216,1299,0.098290,0.097173,0.001117,1.011491,1.009191,0.200154
2,3,6444,0.096704,3300,391,0.118485,0.095570,0.022915,1.239775,1.225226,...,0.918317,0.091713,13243,1293,0.097636,0.096473,0.001163,1.012057,1.009638,0.200652
3,4,6512,0.097725,3332,422,0.126651,0.096202,0.030448,1.316501,1.295991,...,0.963413,0.096130,13433,1338,0.099605,0.097250,0.002355,1.024219,1.019243,0.205467
4,5,6498,0.097515,3275,455,0.138931,0.095374,0.043557,1.456698,1.424719,...,1.017973,0.102185,13328,1358,0.101891,0.096421,0.005470,1.056730,1.044874,0.208987
5,6,6481,0.097260,3254,399,0.122618,0.095958,0.026660,1.277835,1.260730,...,0.932459,0.093195,13528,1356,0.100237,0.096501,0.003735,1.038705,1.030607,0.209227
6,7,6605,0.099121,3341,430,0.128704,0.097559,0.031145,1.319242,1.298459,...,0.982628,0.097502,13293,1367,0.102836,0.098195,0.004641,1.047267,1.037484,0.206964
7,8,6510,0.097695,3398,456,0.134197,0.095734,0.038463,1.401771,1.373629,...,1.025697,0.104916,13544,1383,0.102112,0.096568,0.005543,1.057404,1.045209,0.212442
8,9,6584,0.098805,3327,462,0.138864,0.096700,0.042164,1.436023,1.405427,...,1.012397,0.100091,13155,1332,0.101254,0.098203,0.003051,1.031070,1.024784,0.202309
9,10,6376,0.095684,3285,420,0.127854,0.094016,0.033838,1.359918,1.336209,...,0.939793,0.093632,13433,1307,0.097298,0.095277,0.002021,1.021213,1.016865,0.204987



CELL DECISION
--------------------------------------------------------------------------------------------------------------------------------
PASS_STAGE6C_FULL_GES_TOP_RISK_ENRICHMENT_BOOTSTRAP_INFERENCE_COMPLETE
The percentile intervals estimate sampling uncertainty conditional on the frozen exact-rank memberships.
Cutoff-tie allocation sensitivity remains separately documented by Cell 6C-2B3.
No score, outcome, cutoff, threshold, weight, ranking, or frozen scientific artifact was modified.


In [19]:
# ==================================================================================================
# STAGE 6C — STEP 2C
# CELL 6C-2C1 — FROZEN 0.50 DECISION-THRESHOLD POINT ESTIMATES
#
# Purpose:
#   1. Reverify the immutable Stage 6B primary-evaluable cohort.
#   2. Reverify the frozen Stage 4C model specification and its 0.50 threshold.
#   3. Apply the original frozen reconstruction threshold to:
#        a. full GES;
#        b. no-star GES.
#   4. Calculate confusion-matrix counts and descriptive threshold metrics:
#        - sensitivity;
#        - specificity;
#        - positive predictive value;
#        - negative predictive value;
#        - accuracy;
#        - balanced accuracy;
#        - F1;
#        - Matthews correlation coefficient;
#        - predicted-positive fraction;
#        - event rate among predicted-positive records;
#        - event rate among predicted-negative records;
#        - relative event risk;
#        - event capture.
#
# Frozen threshold rule:
#   P(stable) >= 0.50  -> predicted stable
#   P(stable) <  0.50  -> predicted future instability
#
# Important scientific boundary:
#   - The 0.50 threshold was frozen during Stage 4C for weak-label reconstruction.
#   - It was not selected, optimized, or recalibrated using temporal outcomes.
#   - Results are descriptive threshold behavior, not clinical operating characteristics.
#   - No alternative cutoff is searched in this cell.
#   - No confidence interval is calculated in this cell.
#   - No scientific artifact is written or modified.
# ==================================================================================================

from pathlib import Path
import hashlib
import json
import math
import re

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from IPython.display import display


# --------------------------------------------------------------------------------------------------
# 1. Persistent paths
# --------------------------------------------------------------------------------------------------

DRIVE_ROOT = Path(
    "/content/drive/MyDrive"
)

if not DRIVE_ROOT.exists():

    from google.colab import drive

    drive.mount(
        "/content/drive",
        force_remount=False,
    )

assert DRIVE_ROOT.exists(), (
    "Google Drive is not mounted."
)

PROJECT_ROOT = (
    DRIVE_ROOT
    / "GES_RAG_Temporal_Study"
)

EVALUABLE_COHORT_PATH = (
    PROJECT_ROOT
    / "data_processed"
    / "stage6_temporal_validation"
    / "stage6b_locked_primary_evaluable_cohort_v1.parquet"
)

EVALUABLE_COHORT_SIDECAR_PATH = (
    EVALUABLE_COHORT_PATH.with_name(
        EVALUABLE_COHORT_PATH.name
        + ".sha256"
    )
)

MODEL_SPECIFICATION_PATH = (
    PROJECT_ROOT
    / "configs"
    / "stage4_ges"
    / "stage4c_ges_model_specification_v1.json"
)


# --------------------------------------------------------------------------------------------------
# 2. Frozen expected identities
# --------------------------------------------------------------------------------------------------

EXPECTED_EVALUABLE_SHA256 = (
    "c6ad50dfc376a746e1830d000dba975101d88763"
    "fd61b8d1983f6300edb6e038"
)

EXPECTED_MODEL_SPECIFICATION_SHA256 = (
    "d754c715c990f42cecd64259ca2c420427b9602"
    "c669dc7be9e80dee9554f61f6"
)

EXPECTED_ROWS = 66_636
EXPECTED_COLUMNS = 79
EXPECTED_EVENTS = 6_485
EXPECTED_NEGATIVES = 60_151
EXPECTED_THRESHOLD = 0.50

OUTCOME_COL = (
    "primary_future_instability"
)

FULL_P_STABLE_COL = (
    "full_ges_p_stable_t0"
)

FULL_RISK_COL = (
    "full_ges_instability_risk_t0"
)

NO_STAR_P_STABLE_COL = (
    "no_star_ges_p_stable_t0"
)

NO_STAR_RISK_COL = (
    "no_star_ges_instability_risk_t0"
)


# --------------------------------------------------------------------------------------------------
# 3. Cryptographic helpers
# --------------------------------------------------------------------------------------------------

def sha256_file(
    path: Path,
    chunk_size: int = 8 * 1024 * 1024,
) -> str:
    """Calculate a SHA-256 checksum without modifying the file."""

    digest = hashlib.sha256()

    with path.open("rb") as file_handle:

        while True:

            block = file_handle.read(
                chunk_size
            )

            if not block:
                break

            digest.update(
                block
            )

    return digest.hexdigest()


def read_sha256_sidecar(
    path: Path,
) -> str:
    """Read the first valid SHA-256 value from a sidecar."""

    sidecar_text = path.read_text(
        encoding="utf-8"
    ).strip()

    match = re.search(
        r"(?i)\b[0-9a-f]{64}\b",
        sidecar_text,
    )

    if match is None:

        raise AssertionError(
            "No valid SHA-256 value found in sidecar: "
            f"{path}"
        )

    return match.group(0).lower()


def recursively_find_key_values(
    object_value,
    target_key: str,
):
    """Find all values associated with a key in a nested JSON object."""

    found_values = []

    if isinstance(
        object_value,
        dict,
    ):

        for key, value in object_value.items():

            if key == target_key:

                found_values.append(
                    value
                )

            found_values.extend(
                recursively_find_key_values(
                    value,
                    target_key,
                )
            )

    elif isinstance(
        object_value,
        list,
    ):

        for value in object_value:

            found_values.extend(
                recursively_find_key_values(
                    value,
                    target_key,
                )
            )

    return found_values


# --------------------------------------------------------------------------------------------------
# 4. Fresh cryptographic verification
# --------------------------------------------------------------------------------------------------

assert EVALUABLE_COHORT_PATH.is_file(), (
    "Missing frozen Stage 6B evaluable cohort:\n"
    f"{EVALUABLE_COHORT_PATH}"
)

assert EVALUABLE_COHORT_SIDECAR_PATH.is_file(), (
    "Missing Stage 6B evaluable-cohort sidecar:\n"
    f"{EVALUABLE_COHORT_SIDECAR_PATH}"
)

assert MODEL_SPECIFICATION_PATH.is_file(), (
    "Missing frozen Stage 4C model specification:\n"
    f"{MODEL_SPECIFICATION_PATH}"
)

observed_evaluable_sha256 = sha256_file(
    EVALUABLE_COHORT_PATH
)

sidecar_evaluable_sha256 = read_sha256_sidecar(
    EVALUABLE_COHORT_SIDECAR_PATH
)

observed_model_specification_sha256 = sha256_file(
    MODEL_SPECIFICATION_PATH
)

assert (
    observed_evaluable_sha256
    == EXPECTED_EVALUABLE_SHA256
), (
    "The Stage 6B evaluable-cohort checksum does not "
    "match the accepted frozen checksum."
)

assert (
    sidecar_evaluable_sha256
    == EXPECTED_EVALUABLE_SHA256
), (
    "The Stage 6B evaluable-cohort sidecar does not "
    "contain the accepted frozen checksum."
)

assert (
    observed_model_specification_sha256
    == EXPECTED_MODEL_SPECIFICATION_SHA256
), (
    "The Stage 4C model-specification checksum does "
    "not match the accepted frozen checksum."
)


# --------------------------------------------------------------------------------------------------
# 5. Recover and verify the frozen model threshold
# --------------------------------------------------------------------------------------------------

with MODEL_SPECIFICATION_PATH.open(
    "r",
    encoding="utf-8",
) as file_handle:

    model_specification = json.load(
        file_handle
    )

threshold_values_raw = (
    recursively_find_key_values(
        model_specification,
        "decision_threshold",
    )
)

assert threshold_values_raw, (
    "No decision_threshold field was found in the "
    "frozen Stage 4C model specification."
)

threshold_values_numeric = []

for threshold_value in threshold_values_raw:

    try:

        threshold_values_numeric.append(
            float(
                threshold_value
            )
        )

    except (
        TypeError,
        ValueError,
    ):

        pass

assert threshold_values_numeric, (
    "The frozen decision threshold could not be "
    "converted to a numeric value."
)

unique_threshold_values = sorted(
    set(
        threshold_values_numeric
    )
)

assert all(
    np.isclose(
        threshold_value,
        EXPECTED_THRESHOLD,
        rtol=0.0,
        atol=1e-15,
    )
    for threshold_value
    in unique_threshold_values
), (
    "The Stage 4C model specification contains a "
    "decision threshold other than 0.50: "
    f"{unique_threshold_values}"
)

FROZEN_THRESHOLD = float(
    EXPECTED_THRESHOLD
)


# --------------------------------------------------------------------------------------------------
# 6. Inspect Parquet structure and identify the frozen key
# --------------------------------------------------------------------------------------------------

parquet_file = pq.ParquetFile(
    EVALUABLE_COHORT_PATH
)

parquet_metadata = (
    parquet_file.metadata
)

parquet_columns = (
    parquet_file.schema_arrow.names
)

assert (
    parquet_metadata.num_rows
    == EXPECTED_ROWS
)

assert (
    parquet_metadata.num_columns
    == EXPECTED_COLUMNS
)

key_candidates = (
    "t0_rcv_accession",
    "rcv_accession_t0",
    "rcv_accession",
)

KEY_COL = next(
    (
        candidate
        for candidate
        in key_candidates
        if candidate
        in parquet_columns
    ),
    None,
)

assert KEY_COL is not None, (
    "Could not identify the frozen T0 RCV key."
)

required_columns = [
    KEY_COL,
    OUTCOME_COL,
    FULL_P_STABLE_COL,
    FULL_RISK_COL,
    NO_STAR_P_STABLE_COL,
    NO_STAR_RISK_COL,
]

missing_columns = [
    column
    for column
    in required_columns
    if column
    not in parquet_columns
]

assert not missing_columns, (
    "Required decision-threshold columns are missing: "
    f"{missing_columns}"
)


# Frozen Stage 4C prediction columns may or may not have been retained
# in the Stage 6B joined cohort. Load and verify them when available.

optional_frozen_prediction_columns = [
    column
    for column
    in (
        "full_ges_predicted_stable_at_0_5",
        "no_star_ges_predicted_stable_at_0_5",
    )
    if column
    in parquet_columns
]

columns_to_load = (
    required_columns
    + optional_frozen_prediction_columns
)


# --------------------------------------------------------------------------------------------------
# 7. Load only the required frozen columns
# --------------------------------------------------------------------------------------------------

threshold_source_df = pd.read_parquet(
    EVALUABLE_COHORT_PATH,
    columns=columns_to_load,
)

assert len(
    threshold_source_df
) == EXPECTED_ROWS

assert threshold_source_df[
    KEY_COL
].notna().all()

assert threshold_source_df[
    KEY_COL
].is_unique

threshold_source_df[
    OUTCOME_COL
] = pd.to_numeric(
    threshold_source_df[
        OUTCOME_COL
    ],
    errors="raise",
).astype(
    "int8"
)

assert set(
    threshold_source_df[
        OUTCOME_COL
    ].unique().tolist()
) == {
    0,
    1,
}

for score_column in (
    FULL_P_STABLE_COL,
    FULL_RISK_COL,
    NO_STAR_P_STABLE_COL,
    NO_STAR_RISK_COL,
):

    threshold_source_df[
        score_column
    ] = pd.to_numeric(
        threshold_source_df[
            score_column
        ],
        errors="raise",
    ).astype(
        "float64"
    )

    assert np.isfinite(
        threshold_source_df[
            score_column
        ].to_numpy()
    ).all(), (
        f"{score_column} contains a nonfinite value."
    )

    assert threshold_source_df[
        score_column
    ].between(
        0.0,
        1.0,
        inclusive="both",
    ).all(), (
        f"{score_column} contains a value outside [0,1]."
    )


# --------------------------------------------------------------------------------------------------
# 8. Verify score complements
# --------------------------------------------------------------------------------------------------

assert np.allclose(
    (
        threshold_source_df[
            FULL_P_STABLE_COL
        ]
        + threshold_source_df[
            FULL_RISK_COL
        ]
    ).to_numpy(),
    1.0,
    rtol=0.0,
    atol=1e-10,
), (
    "Full-GES P(stable) and instability risk are "
    "not complements within tolerance."
)

assert np.allclose(
    (
        threshold_source_df[
            NO_STAR_P_STABLE_COL
        ]
        + threshold_source_df[
            NO_STAR_RISK_COL
        ]
    ).to_numpy(),
    1.0,
    rtol=0.0,
    atol=1e-10,
), (
    "No-star-GES P(stable) and instability risk are "
    "not complements within tolerance."
)


# --------------------------------------------------------------------------------------------------
# 9. Outcome accounting
# --------------------------------------------------------------------------------------------------

observed_events = int(
    threshold_source_df[
        OUTCOME_COL
    ].sum()
)

observed_negatives = int(
    len(
        threshold_source_df
    )
    - observed_events
)

observed_prevalence = (
    observed_events
    / len(
        threshold_source_df
    )
)

assert observed_events == EXPECTED_EVENTS

assert observed_negatives == EXPECTED_NEGATIVES


# --------------------------------------------------------------------------------------------------
# 10. Apply the exact frozen Stage 4C threshold
#
# Stage 4C definition:
#   predicted stable = P(stable) >= 0.50
#
# Temporal positive-class definition:
#   predicted instability = NOT predicted stable
#                         = P(stable) < 0.50
#
# Therefore an exact P(stable) value of 0.50 remains predicted stable.
# --------------------------------------------------------------------------------------------------

model_definitions = [
    {
        "model_name": "Full GES",
        "model_short_name": "full_ges",
        "p_stable_column": (
            FULL_P_STABLE_COL
        ),
        "instability_risk_column": (
            FULL_RISK_COL
        ),
        "optional_frozen_prediction_column": (
            "full_ges_predicted_stable_at_0_5"
        ),
    },
    {
        "model_name": "No-star GES",
        "model_short_name": "no_star_ges",
        "p_stable_column": (
            NO_STAR_P_STABLE_COL
        ),
        "instability_risk_column": (
            NO_STAR_RISK_COL
        ),
        "optional_frozen_prediction_column": (
            "no_star_ges_predicted_stable_at_0_5"
        ),
    },
]

stage6c_decision_threshold_membership_df = (
    threshold_source_df[
        [
            KEY_COL,
            OUTCOME_COL,
            FULL_P_STABLE_COL,
            FULL_RISK_COL,
            NO_STAR_P_STABLE_COL,
            NO_STAR_RISK_COL,
        ]
    ]
    .copy()
)

for model_definition in model_definitions:

    model_short_name = (
        model_definition[
            "model_short_name"
        ]
    )

    p_stable_column = (
        model_definition[
            "p_stable_column"
        ]
    )

    instability_risk_column = (
        model_definition[
            "instability_risk_column"
        ]
    )

    predicted_stable_column = (
        f"{model_short_name}_predicted_stable_at_frozen_0_5"
    )

    predicted_instability_column = (
        f"{model_short_name}_predicted_instability_at_frozen_0_5"
    )

    threshold_source_df[
        predicted_stable_column
    ] = (
        threshold_source_df[
            p_stable_column
        ]
        >= FROZEN_THRESHOLD
    )

    threshold_source_df[
        predicted_instability_column
    ] = (
        ~threshold_source_df[
            predicted_stable_column
        ]
    )

    # Equivalent instability-risk expression must use strict > 0.50,
    # because an exact P(stable) = 0.50 is classified as stable.
    risk_based_predicted_instability = (
        threshold_source_df[
            instability_risk_column
        ]
        > (
            1.0
            - FROZEN_THRESHOLD
        )
    )

    assert np.array_equal(
        threshold_source_df[
            predicted_instability_column
        ].to_numpy(),
        risk_based_predicted_instability.to_numpy(),
    ), (
        f"{model_definition['model_name']} P(stable)-based "
        "and risk-based threshold predictions disagree."
    )

    optional_prediction_column = (
        model_definition[
            "optional_frozen_prediction_column"
        ]
    )

    if (
        optional_prediction_column
        in threshold_source_df.columns
    ):

        frozen_prediction = (
            threshold_source_df[
                optional_prediction_column
            ]
            .astype(
                bool
            )
        )

        assert np.array_equal(
            frozen_prediction.to_numpy(),
            threshold_source_df[
                predicted_stable_column
            ].to_numpy(),
        ), (
            f"{model_definition['model_name']} frozen Stage 4C "
            "predictions do not reconstruct."
        )

    stage6c_decision_threshold_membership_df[
        predicted_stable_column
    ] = threshold_source_df[
        predicted_stable_column
    ]

    stage6c_decision_threshold_membership_df[
        predicted_instability_column
    ] = threshold_source_df[
        predicted_instability_column
    ]


# --------------------------------------------------------------------------------------------------
# 11. Metric helper functions
# --------------------------------------------------------------------------------------------------

def safe_divide(
    numerator,
    denominator,
):
    """Safely divide, returning NaN when the denominator is zero."""

    if denominator == 0:
        return np.nan

    return numerator / denominator


def calculate_matthews_correlation(
    true_positive,
    true_negative,
    false_positive,
    false_negative,
):
    """Calculate Matthews correlation coefficient."""

    numerator = (
        true_positive
        * true_negative
        - false_positive
        * false_negative
    )

    denominator_squared = (
        (true_positive + false_positive)
        * (true_positive + false_negative)
        * (true_negative + false_positive)
        * (true_negative + false_negative)
    )

    if denominator_squared <= 0:
        return np.nan

    return (
        numerator
        / math.sqrt(
            denominator_squared
        )
    )


# --------------------------------------------------------------------------------------------------
# 12. Calculate confusion matrices and threshold point estimates
# --------------------------------------------------------------------------------------------------

outcome_array = (
    threshold_source_df[
        OUTCOME_COL
    ]
    .to_numpy(
        dtype=np.int8
    )
)

metric_rows = []
confusion_rows = []
threshold_audit_rows = []

for model_definition in model_definitions:

    model_name = (
        model_definition[
            "model_name"
        ]
    )

    model_short_name = (
        model_definition[
            "model_short_name"
        ]
    )

    p_stable_column = (
        model_definition[
            "p_stable_column"
        ]
    )

    instability_risk_column = (
        model_definition[
            "instability_risk_column"
        ]
    )

    predicted_instability_column = (
        f"{model_short_name}_predicted_instability_at_frozen_0_5"
    )

    predicted_instability_array = (
        threshold_source_df[
            predicted_instability_column
        ]
        .to_numpy(
            dtype=bool
        )
    )

    observed_instability_array = (
        outcome_array
        == 1
    )

    true_positive = int(
        np.sum(
            predicted_instability_array
            & observed_instability_array
        )
    )

    false_positive = int(
        np.sum(
            predicted_instability_array
            & ~observed_instability_array
        )
    )

    true_negative = int(
        np.sum(
            ~predicted_instability_array
            & ~observed_instability_array
        )
    )

    false_negative = int(
        np.sum(
            ~predicted_instability_array
            & observed_instability_array
        )
    )

    predicted_positive_records = int(
        true_positive
        + false_positive
    )

    predicted_negative_records = int(
        true_negative
        + false_negative
    )

    sensitivity = safe_divide(
        true_positive,
        true_positive
        + false_negative,
    )

    specificity = safe_divide(
        true_negative,
        true_negative
        + false_positive,
    )

    positive_predictive_value = safe_divide(
        true_positive,
        true_positive
        + false_positive,
    )

    negative_predictive_value = safe_divide(
        true_negative,
        true_negative
        + false_negative,
    )

    accuracy = safe_divide(
        true_positive
        + true_negative,
        EXPECTED_ROWS,
    )

    balanced_accuracy = (
        (
            sensitivity
            + specificity
        )
        / 2.0
    )

    f1_score = safe_divide(
        2
        * true_positive,
        (
            2
            * true_positive
            + false_positive
            + false_negative
        ),
    )

    matthews_correlation = (
        calculate_matthews_correlation(
            true_positive=true_positive,
            true_negative=true_negative,
            false_positive=false_positive,
            false_negative=false_negative,
        )
    )

    false_positive_rate = (
        1.0
        - specificity
    )

    false_negative_rate = (
        1.0
        - sensitivity
    )

    positive_likelihood_ratio = (
        sensitivity
        / false_positive_rate
        if false_positive_rate > 0
        else np.inf
    )

    negative_likelihood_ratio = (
        false_negative_rate
        / specificity
        if specificity > 0
        else np.inf
    )

    predicted_positive_fraction = safe_divide(
        predicted_positive_records,
        EXPECTED_ROWS,
    )

    predicted_negative_fraction = safe_divide(
        predicted_negative_records,
        EXPECTED_ROWS,
    )

    positive_group_event_rate = (
        positive_predictive_value
    )

    negative_group_event_rate = safe_divide(
        false_negative,
        false_negative
        + true_negative,
    )

    relative_event_risk = (
        positive_group_event_rate
        / negative_group_event_rate
        if (
            np.isfinite(
                positive_group_event_rate
            )
            and np.isfinite(
                negative_group_event_rate
            )
            and negative_group_event_rate > 0
        )
        else np.nan
    )

    event_rate_difference = (
        positive_group_event_rate
        - negative_group_event_rate
    )

    enrichment_vs_overall = (
        positive_group_event_rate
        / observed_prevalence
        if observed_prevalence > 0
        else np.nan
    )

    youden_j = (
        sensitivity
        + specificity
        - 1.0
    )

    assert (
        true_positive
        + false_negative
        == EXPECTED_EVENTS
    )

    assert (
        true_negative
        + false_positive
        == EXPECTED_NEGATIVES
    )

    assert (
        true_positive
        + false_positive
        + true_negative
        + false_negative
        == EXPECTED_ROWS
    )

    metric_rows.append({
        "model": model_name,
        "threshold_source": (
            "Frozen Stage 4C reconstruction threshold"
        ),
        "p_stable_threshold": (
            FROZEN_THRESHOLD
        ),
        "predicted_instability_rule": (
            "P(stable) < 0.50"
        ),
        "predicted_positive_records": (
            predicted_positive_records
        ),
        "predicted_positive_fraction": (
            predicted_positive_fraction
        ),
        "predicted_negative_records": (
            predicted_negative_records
        ),
        "predicted_negative_fraction": (
            predicted_negative_fraction
        ),
        "true_positive": (
            true_positive
        ),
        "false_positive": (
            false_positive
        ),
        "true_negative": (
            true_negative
        ),
        "false_negative": (
            false_negative
        ),
        "sensitivity": (
            sensitivity
        ),
        "specificity": (
            specificity
        ),
        "positive_predictive_value": (
            positive_predictive_value
        ),
        "negative_predictive_value": (
            negative_predictive_value
        ),
        "accuracy": (
            accuracy
        ),
        "balanced_accuracy": (
            balanced_accuracy
        ),
        "f1_score": (
            f1_score
        ),
        "matthews_correlation": (
            matthews_correlation
        ),
        "false_positive_rate": (
            false_positive_rate
        ),
        "false_negative_rate": (
            false_negative_rate
        ),
        "positive_likelihood_ratio": (
            positive_likelihood_ratio
        ),
        "negative_likelihood_ratio": (
            negative_likelihood_ratio
        ),
        "youden_j": (
            youden_j
        ),
        "predicted_positive_event_rate": (
            positive_group_event_rate
        ),
        "predicted_negative_event_rate": (
            negative_group_event_rate
        ),
        "event_rate_difference": (
            event_rate_difference
        ),
        "relative_event_risk": (
            relative_event_risk
        ),
        "enrichment_vs_overall_prevalence": (
            enrichment_vs_overall
        ),
        "event_capture_fraction": (
            sensitivity
        ),
    })

    confusion_rows.extend([
        {
            "model": model_name,
            "observed_class": (
                "Future instability"
            ),
            "predicted_class": (
                "Predicted instability"
            ),
            "count": (
                true_positive
            ),
            "confusion_cell": "TP",
        },
        {
            "model": model_name,
            "observed_class": (
                "Future stability"
            ),
            "predicted_class": (
                "Predicted instability"
            ),
            "count": (
                false_positive
            ),
            "confusion_cell": "FP",
        },
        {
            "model": model_name,
            "observed_class": (
                "Future stability"
            ),
            "predicted_class": (
                "Predicted stable"
            ),
            "count": (
                true_negative
            ),
            "confusion_cell": "TN",
        },
        {
            "model": model_name,
            "observed_class": (
                "Future instability"
            ),
            "predicted_class": (
                "Predicted stable"
            ),
            "count": (
                false_negative
            ),
            "confusion_cell": "FN",
        },
    ])

    exact_threshold_records = int(
        np.sum(
            threshold_source_df[
                p_stable_column
            ].to_numpy()
            == FROZEN_THRESHOLD
        )
    )

    below_threshold_records = int(
        np.sum(
            threshold_source_df[
                p_stable_column
            ].to_numpy()
            < FROZEN_THRESHOLD
        )
    )

    above_or_equal_threshold_records = int(
        np.sum(
            threshold_source_df[
                p_stable_column
            ].to_numpy()
            >= FROZEN_THRESHOLD
        )
    )

    threshold_audit_rows.append({
        "model": model_name,
        "p_stable_column": (
            p_stable_column
        ),
        "instability_risk_column": (
            instability_risk_column
        ),
        "frozen_threshold": (
            FROZEN_THRESHOLD
        ),
        "predicted_stable_operator": (
            ">="
        ),
        "predicted_instability_operator": (
            "<"
        ),
        "records_below_threshold": (
            below_threshold_records
        ),
        "records_exactly_at_threshold": (
            exact_threshold_records
        ),
        "records_above_or_equal_threshold": (
            above_or_equal_threshold_records
        ),
        "minimum_p_stable": float(
            threshold_source_df[
                p_stable_column
            ].min()
        ),
        "maximum_p_stable": float(
            threshold_source_df[
                p_stable_column
            ].max()
        ),
        "mean_p_stable": float(
            threshold_source_df[
                p_stable_column
            ].mean()
        ),
        "median_p_stable": float(
            threshold_source_df[
                p_stable_column
            ].median()
        ),
    })


stage6c_decision_threshold_point_estimates_df = (
    pd.DataFrame(
        metric_rows
    )
)

stage6c_decision_threshold_confusion_matrix_df = (
    pd.DataFrame(
        confusion_rows
    )
)

stage6c_decision_threshold_audit_df = (
    pd.DataFrame(
        threshold_audit_rows
    )
)


# --------------------------------------------------------------------------------------------------
# 13. Compact required-metric table
# --------------------------------------------------------------------------------------------------

stage6c_decision_threshold_required_metrics_df = (
    stage6c_decision_threshold_point_estimates_df[
        [
            "model",
            "p_stable_threshold",
            "predicted_positive_records",
            "predicted_positive_fraction",
            "sensitivity",
            "specificity",
            "positive_predictive_value",
            "negative_predictive_value",
            "accuracy",
            "balanced_accuracy",
            "f1_score",
            "matthews_correlation",
            "predicted_positive_event_rate",
            "predicted_negative_event_rate",
            "relative_event_risk",
            "enrichment_vs_overall_prevalence",
        ]
    ]
    .copy()
)


# --------------------------------------------------------------------------------------------------
# 14. Final QC checks
# --------------------------------------------------------------------------------------------------

assert len(
    stage6c_decision_threshold_point_estimates_df
) == 2

assert len(
    stage6c_decision_threshold_confusion_matrix_df
) == 8

assert len(
    stage6c_decision_threshold_audit_df
) == 2

for metric_column in (
    "sensitivity",
    "specificity",
    "positive_predictive_value",
    "negative_predictive_value",
    "accuracy",
    "balanced_accuracy",
    "f1_score",
    "predicted_positive_fraction",
    "predicted_negative_fraction",
    "event_capture_fraction",
):

    metric_values = (
        stage6c_decision_threshold_point_estimates_df[
            metric_column
        ]
    )

    assert metric_values.between(
        0.0,
        1.0,
        inclusive="both",
    ).all(), (
        f"{metric_column} contains a value outside [0,1]."
    )

assert (
    stage6c_decision_threshold_point_estimates_df[
        "predicted_positive_records"
    ]
    + stage6c_decision_threshold_point_estimates_df[
        "predicted_negative_records"
    ]
    == EXPECTED_ROWS
).all()

assert np.allclose(
    (
        stage6c_decision_threshold_point_estimates_df[
            "predicted_positive_fraction"
        ]
        + stage6c_decision_threshold_point_estimates_df[
            "predicted_negative_fraction"
        ]
    ).to_numpy(),
    1.0,
    rtol=0.0,
    atol=1e-15,
)


# --------------------------------------------------------------------------------------------------
# 15. Display results
# --------------------------------------------------------------------------------------------------

print()
print("=" * 130)
print(
    "STAGE 6C STEP 2C — CELL 6C-2C1 — "
    "FROZEN 0.50 DECISION-THRESHOLD POINT ESTIMATES"
)
print("=" * 130)

print()
print("FROZEN INPUT VERIFICATION")
print("-" * 130)

print(
    "Stage 6B evaluable-cohort SHA-256 : "
    f"PASS ({observed_evaluable_sha256})"
)

print(
    "Stage 4C model-specification hash : "
    f"PASS ({observed_model_specification_sha256})"
)

print(
    "Parquet dimensions                : "
    f"PASS ({EXPECTED_ROWS:,} × {EXPECTED_COLUMNS})"
)

print(
    "Unique evaluable RCV keys         : "
    f"PASS ({threshold_source_df[KEY_COL].nunique():,})"
)

print(
    "Primary instability events        : "
    f"PASS ({observed_events:,})"
)

print(
    "Primary instability negatives     : "
    f"PASS ({observed_negatives:,})"
)

print(
    "Observed prevalence               : "
    f"{observed_prevalence:.8f} "
    f"({observed_prevalence * 100:.6f}%)"
)

print()
print("FROZEN THRESHOLD DEFINITION")
print("-" * 130)

print(
    "Frozen P(stable) threshold        : "
    f"{FROZEN_THRESHOLD:.2f}"
)

print(
    "Predicted stable                  : "
    "P(stable) >= 0.50"
)

print(
    "Predicted future instability      : "
    "P(stable) < 0.50"
)

print(
    "Threshold selected using T1       : No"
)

print(
    "Threshold optimized in Stage 6C   : No"
)

print(
    "Clinical operating threshold      : No"
)

print(
    "Scientific artifact written       : No"
)

print()
print("THRESHOLD AUDIT")
print("-" * 130)

display(
    stage6c_decision_threshold_audit_df
)

print()
print("REQUIRED DECISION-THRESHOLD METRICS")
print("-" * 130)

for row in metric_rows:

    print()
    print(
        row[
            "model"
        ]
    )

    print(
        "  Predicted instability records   : "
        f"{row['predicted_positive_records']:,} "
        f"({row['predicted_positive_fraction'] * 100:.6f}%)"
    )

    print(
        "  True positives                  : "
        f"{row['true_positive']:,}"
    )

    print(
        "  False positives                 : "
        f"{row['false_positive']:,}"
    )

    print(
        "  True negatives                  : "
        f"{row['true_negative']:,}"
    )

    print(
        "  False negatives                 : "
        f"{row['false_negative']:,}"
    )

    print(
        "  Sensitivity                     : "
        f"{row['sensitivity']:.8f} "
        f"({row['sensitivity'] * 100:.6f}%)"
    )

    print(
        "  Specificity                     : "
        f"{row['specificity']:.8f} "
        f"({row['specificity'] * 100:.6f}%)"
    )

    print(
        "  Positive predictive value       : "
        f"{row['positive_predictive_value']:.8f} "
        f"({row['positive_predictive_value'] * 100:.6f}%)"
    )

    print(
        "  Negative predictive value       : "
        f"{row['negative_predictive_value']:.8f} "
        f"({row['negative_predictive_value'] * 100:.6f}%)"
    )

    print(
        "  Balanced accuracy               : "
        f"{row['balanced_accuracy']:.8f}"
    )

    print(
        "  F1 score                        : "
        f"{row['f1_score']:.8f}"
    )

    print(
        "  Matthews correlation            : "
        f"{row['matthews_correlation']:.8f}"
    )

    print(
        "  Positive-group event rate       : "
        f"{row['predicted_positive_event_rate']:.8f} "
        f"({row['predicted_positive_event_rate'] * 100:.6f}%)"
    )

    print(
        "  Negative-group event rate       : "
        f"{row['predicted_negative_event_rate']:.8f} "
        f"({row['predicted_negative_event_rate'] * 100:.6f}%)"
    )

    print(
        "  Relative event risk             : "
        f"{row['relative_event_risk']:.8f}×"
    )

    print(
        "  Enrichment vs prevalence        : "
        f"{row['enrichment_vs_overall_prevalence']:.8f}×"
    )

print()
print("COMPACT REQUIRED-METRIC TABLE")
print("-" * 130)

display(
    stage6c_decision_threshold_required_metrics_df
)

print()
print("COMPLETE POINT-ESTIMATE TABLE")
print("-" * 130)

display(
    stage6c_decision_threshold_point_estimates_df
)

print()
print("CONFUSION-MATRIX LONG TABLE")
print("-" * 130)

display(
    stage6c_decision_threshold_confusion_matrix_df
)

print()
print("IN-MEMORY MEMBERSHIP TABLE")
print("-" * 130)

print(
    "Shape: "
    f"{stage6c_decision_threshold_membership_df.shape}"
)

display(
    stage6c_decision_threshold_membership_df.head(
        10
    )
)

print()
print("CELL DECISION")
print("-" * 130)

print(
    "PASS_STAGE6C_FROZEN_0_50_DECISION_THRESHOLD_"
    "POINT_ESTIMATES_COMPLETE"
)

print(
    "The original Stage 4C reconstruction threshold was "
    "evaluated descriptively against the frozen temporal outcome."
)

print(
    "No cutoff search, threshold optimization, recalibration, "
    "score modification, or scientific-artifact write occurred."
)

AssertionError: The frozen decision threshold could not be converted to a numeric value.

In [20]:
# ==================================================================================================
# STAGE 6C — STEP 2C
# CELL 6C-2C0 — READ-ONLY STAGE 4C THRESHOLD-SCHEMA DIAGNOSTIC
#
# Purpose:
#   Inspect how the frozen decision threshold is represented inside the verified Stage 4C
#   model-specification JSON.
#
# This cell:
#   - does not modify the JSON;
#   - does not assign a replacement threshold;
#   - does not calculate temporal performance;
#   - does not write any artifact.
# ==================================================================================================

import json
from pathlib import Path


# Recreate the path only if it is not still available from the failed cell.
if "MODEL_SPECIFICATION_PATH" not in globals():

    PROJECT_ROOT = Path(
        "/content/drive/MyDrive/GES_RAG_Temporal_Study"
    )

    MODEL_SPECIFICATION_PATH = (
        PROJECT_ROOT
        / "configs"
        / "stage4_ges"
        / "stage4c_ges_model_specification_v1.json"
    )


assert MODEL_SPECIFICATION_PATH.is_file(), (
    "Missing Stage 4C model specification:\n"
    f"{MODEL_SPECIFICATION_PATH}"
)


with MODEL_SPECIFICATION_PATH.open(
    "r",
    encoding="utf-8",
) as file_handle:

    model_specification_diagnostic = json.load(
        file_handle
    )


def walk_json(
    value,
    path="$",
):
    """Yield every JSON path, value, and Python type."""

    yield (
        path,
        value,
        type(value).__name__,
    )

    if isinstance(
        value,
        dict,
    ):

        for key, child_value in value.items():

            child_path = (
                f"{path}.{key}"
            )

            yield from walk_json(
                child_value,
                child_path,
            )

    elif isinstance(
        value,
        list,
    ):

        for index, child_value in enumerate(
            value
        ):

            child_path = (
                f"{path}[{index}]"
            )

            yield from walk_json(
                child_value,
                child_path,
            )


all_json_nodes = list(
    walk_json(
        model_specification_diagnostic
    )
)


search_terms = (
    "threshold",
    "cutoff",
    "decision",
    "predict",
    "stable",
)


matching_nodes = []

for path, value, value_type in all_json_nodes:

    normalized_path = path.lower()

    if any(
        search_term in normalized_path
        for search_term in search_terms
    ):

        matching_nodes.append({
            "path": path,
            "type": value_type,
            "value": value,
        })


print()
print("=" * 120)
print(
    "STAGE 6C STEP 2C — CELL 6C-2C0 — "
    "STAGE 4C THRESHOLD-SCHEMA DIAGNOSTIC"
)
print("=" * 120)

print()
print("MODEL SPECIFICATION")
print("-" * 120)
print(
    f"Path       : {MODEL_SPECIFICATION_PATH}"
)
print(
    f"Top-level type: "
    f"{type(model_specification_diagnostic).__name__}"
)

if isinstance(
    model_specification_diagnostic,
    dict,
):

    print(
        "Top-level keys:"
    )

    for key in model_specification_diagnostic.keys():

        print(
            f"  - {key}"
        )


print()
print("THRESHOLD- AND PREDICTION-RELATED JSON NODES")
print("-" * 120)

if not matching_nodes:

    print(
        "No matching nodes were found."
    )

else:

    for node_number, node in enumerate(
        matching_nodes,
        start=1,
    ):

        print()
        print(
            f"[{node_number}] Path : "
            f"{node['path']}"
        )

        print(
            f"    Type : "
            f"{node['type']}"
        )

        print(
            "    Value:"
        )

        print(
            json.dumps(
                node["value"],
                indent=2,
                ensure_ascii=False,
                default=str,
            )
        )


print()
print("EXACT decision_threshold KEYS")
print("-" * 120)

exact_decision_threshold_nodes = [
    node
    for node in matching_nodes
    if node[
        "path"
    ].lower().endswith(
        ".decision_threshold"
    )
]

if not exact_decision_threshold_nodes:

    print(
        "No exact decision_threshold key was found."
    )

else:

    for node_number, node in enumerate(
        exact_decision_threshold_nodes,
        start=1,
    ):

        print()
        print(
            f"[{node_number}] {node['path']}"
        )

        print(
            f"Type: {node['type']}"
        )

        print(
            json.dumps(
                node["value"],
                indent=2,
                ensure_ascii=False,
                default=str,
            )
        )


print()
print("CELL DECISION")
print("-" * 120)
print(
    "PASS_STAGE6C_STAGE4C_THRESHOLD_SCHEMA_"
    "DIAGNOSTIC_COMPLETE"
)
print(
    "No scientific artifact was modified or written."
)


STAGE 6C STEP 2C — CELL 6C-2C0 — STAGE 4C THRESHOLD-SCHEMA DIAGNOSTIC

MODEL SPECIFICATION
------------------------------------------------------------------------------------------------------------------------
Path       : /content/drive/MyDrive/GES_RAG_Temporal_Study/configs/stage4_ges/stage4c_ges_model_specification_v1.json
Top-level type: dict
Top-level keys:
  - created_at_utc
  - decision_threshold
  - full_model
  - logistic_regression
  - model_fitting_scope
  - no_star_model
  - preprocessing
  - random_seed
  - scientific_unit
  - software
  - specification_name
  - status
  - target_definition
  - version

THRESHOLD- AND PREDICTION-RELATED JSON NODES
------------------------------------------------------------------------------------------------------------------------

[1] Path : $.decision_threshold
    Type : dict
    Value:
{
  "additional_risk_tiers_selected": false,
  "purpose": "Fixed reconstruction decision threshold only.",
  "selected_using_t1": false,
  "tempora

In [21]:
# ==================================================================================================
# STAGE 6C — STEP 2C
# CELL 6C-2C1 — FROZEN 0.50 DECISION-THRESHOLD POINT ESTIMATES
#
# Frozen rule:
#   P(stable) >= 0.50  -> predicted stable
#   P(stable) <  0.50  -> predicted future instability
#
# Scientific boundary:
#   - Threshold recovered directly from the frozen Stage 4C model specification.
#   - Threshold was not selected using T1.
#   - No threshold search, optimization, or recalibration occurs.
#   - No scientific artifact is written.
# ==================================================================================================

from pathlib import Path
import hashlib
import json
import math
import re

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from IPython.display import display


# --------------------------------------------------------------------------------------------------
# 1. Frozen paths
# --------------------------------------------------------------------------------------------------

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/GES_RAG_Temporal_Study"
)

EVALUABLE_COHORT_PATH = (
    PROJECT_ROOT
    / "data_processed"
    / "stage6_temporal_validation"
    / "stage6b_locked_primary_evaluable_cohort_v1.parquet"
)

EVALUABLE_COHORT_SIDECAR_PATH = (
    EVALUABLE_COHORT_PATH.with_name(
        EVALUABLE_COHORT_PATH.name + ".sha256"
    )
)

MODEL_SPECIFICATION_PATH = (
    PROJECT_ROOT
    / "configs"
    / "stage4_ges"
    / "stage4c_ges_model_specification_v1.json"
)


# --------------------------------------------------------------------------------------------------
# 2. Expected frozen identities
# --------------------------------------------------------------------------------------------------

EXPECTED_EVALUABLE_SHA256 = (
    "c6ad50dfc376a746e1830d000dba975101d88763"
    "fd61b8d1983f6300edb6e038"
)

EXPECTED_MODEL_SPECIFICATION_SHA256 = (
    "d754c715c990f42cecd64259ca2c420427b9602"
    "c669dc7be9e80dee9554f61f6"
)

EXPECTED_ROWS = 66_636
EXPECTED_COLUMNS = 79
EXPECTED_EVENTS = 6_485
EXPECTED_NEGATIVES = 60_151
EXPECTED_THRESHOLD = 0.50

OUTCOME_COL = "primary_future_instability"

FULL_P_STABLE_COL = "full_ges_p_stable_t0"
FULL_RISK_COL = "full_ges_instability_risk_t0"

NO_STAR_P_STABLE_COL = "no_star_ges_p_stable_t0"
NO_STAR_RISK_COL = "no_star_ges_instability_risk_t0"


# --------------------------------------------------------------------------------------------------
# 3. Checksum helpers
# --------------------------------------------------------------------------------------------------

def sha256_file(
    path: Path,
    chunk_size: int = 8 * 1024 * 1024,
) -> str:
    """Calculate SHA-256 without modifying the file."""

    digest = hashlib.sha256()

    with path.open("rb") as file_handle:

        while True:

            block = file_handle.read(
                chunk_size
            )

            if not block:
                break

            digest.update(
                block
            )

    return digest.hexdigest()


def read_sha256_sidecar(
    path: Path,
) -> str:
    """Read the first valid SHA-256 value from a sidecar."""

    sidecar_text = path.read_text(
        encoding="utf-8"
    ).strip()

    match = re.search(
        r"(?i)\b[0-9a-f]{64}\b",
        sidecar_text,
    )

    if match is None:

        raise AssertionError(
            f"No SHA-256 value found in sidecar: {path}"
        )

    return match.group(0).lower()


# --------------------------------------------------------------------------------------------------
# 4. Fresh frozen-input verification
# --------------------------------------------------------------------------------------------------

assert EVALUABLE_COHORT_PATH.is_file(), (
    f"Missing evaluable cohort:\n{EVALUABLE_COHORT_PATH}"
)

assert EVALUABLE_COHORT_SIDECAR_PATH.is_file(), (
    f"Missing cohort sidecar:\n{EVALUABLE_COHORT_SIDECAR_PATH}"
)

assert MODEL_SPECIFICATION_PATH.is_file(), (
    f"Missing model specification:\n{MODEL_SPECIFICATION_PATH}"
)

observed_evaluable_sha256 = sha256_file(
    EVALUABLE_COHORT_PATH
)

sidecar_evaluable_sha256 = read_sha256_sidecar(
    EVALUABLE_COHORT_SIDECAR_PATH
)

observed_model_specification_sha256 = sha256_file(
    MODEL_SPECIFICATION_PATH
)

assert observed_evaluable_sha256 == EXPECTED_EVALUABLE_SHA256, (
    "Evaluable-cohort checksum mismatch."
)

assert sidecar_evaluable_sha256 == EXPECTED_EVALUABLE_SHA256, (
    "Evaluable-cohort sidecar checksum mismatch."
)

assert (
    observed_model_specification_sha256
    == EXPECTED_MODEL_SPECIFICATION_SHA256
), (
    "Stage 4C model-specification checksum mismatch."
)


# --------------------------------------------------------------------------------------------------
# 5. Recover the frozen threshold from its actual nested structure
# --------------------------------------------------------------------------------------------------

with MODEL_SPECIFICATION_PATH.open(
    "r",
    encoding="utf-8",
) as file_handle:

    model_specification = json.load(
        file_handle
    )

decision_threshold_block = model_specification.get(
    "decision_threshold"
)

assert isinstance(
    decision_threshold_block,
    dict,
), (
    "The decision_threshold field is not a dictionary."
)

assert "value" in decision_threshold_block, (
    "decision_threshold.value is missing."
)

FROZEN_THRESHOLD = float(
    decision_threshold_block[
        "value"
    ]
)

assert np.isclose(
    FROZEN_THRESHOLD,
    EXPECTED_THRESHOLD,
    rtol=0.0,
    atol=1e-15,
), (
    f"Unexpected frozen threshold: {FROZEN_THRESHOLD}"
)

assert (
    decision_threshold_block.get(
        "selected_using_t1"
    )
    is False
), (
    "The frozen specification does not confirm "
    "selected_using_t1=False."
)

assert (
    decision_threshold_block.get(
        "additional_risk_tiers_selected"
    )
    is False
), (
    "Unexpected additional frozen risk tiers were found."
)

threshold_purpose = decision_threshold_block.get(
    "purpose"
)

temporal_primary_use = decision_threshold_block.get(
    "temporal_evaluation_primary_use"
)


# --------------------------------------------------------------------------------------------------
# 6. Verify Parquet structure and identify the frozen key
# --------------------------------------------------------------------------------------------------

parquet_file = pq.ParquetFile(
    EVALUABLE_COHORT_PATH
)

parquet_metadata = parquet_file.metadata
parquet_columns = parquet_file.schema_arrow.names

assert parquet_metadata.num_rows == EXPECTED_ROWS

assert parquet_metadata.num_columns == EXPECTED_COLUMNS

key_candidates = (
    "t0_rcv_accession",
    "rcv_accession_t0",
    "rcv_accession",
)

KEY_COL = next(
    (
        candidate
        for candidate in key_candidates
        if candidate in parquet_columns
    ),
    None,
)

assert KEY_COL is not None, (
    "Could not identify the frozen T0 RCV key."
)

required_columns = [
    KEY_COL,
    OUTCOME_COL,
    FULL_P_STABLE_COL,
    FULL_RISK_COL,
    NO_STAR_P_STABLE_COL,
    NO_STAR_RISK_COL,
]

missing_columns = [
    column
    for column in required_columns
    if column not in parquet_columns
]

assert not missing_columns, (
    f"Required columns are missing: {missing_columns}"
)


# --------------------------------------------------------------------------------------------------
# 7. Load only required frozen columns
# --------------------------------------------------------------------------------------------------

threshold_source_df = pd.read_parquet(
    EVALUABLE_COHORT_PATH,
    columns=required_columns,
)

assert threshold_source_df.shape == (
    EXPECTED_ROWS,
    len(required_columns),
)

assert threshold_source_df[
    KEY_COL
].notna().all()

assert threshold_source_df[
    KEY_COL
].is_unique

threshold_source_df[
    OUTCOME_COL
] = pd.to_numeric(
    threshold_source_df[
        OUTCOME_COL
    ],
    errors="raise",
).astype(
    "int8"
)

assert set(
    threshold_source_df[
        OUTCOME_COL
    ].unique().tolist()
) == {
    0,
    1,
}

for score_column in (
    FULL_P_STABLE_COL,
    FULL_RISK_COL,
    NO_STAR_P_STABLE_COL,
    NO_STAR_RISK_COL,
):

    threshold_source_df[
        score_column
    ] = pd.to_numeric(
        threshold_source_df[
            score_column
        ],
        errors="raise",
    ).astype(
        "float64"
    )

    assert np.isfinite(
        threshold_source_df[
            score_column
        ].to_numpy()
    ).all(), (
        f"{score_column} contains nonfinite values."
    )

    assert threshold_source_df[
        score_column
    ].between(
        0.0,
        1.0,
        inclusive="both",
    ).all(), (
        f"{score_column} contains values outside [0,1]."
    )


# --------------------------------------------------------------------------------------------------
# 8. Verify P(stable) and instability-risk complements
# --------------------------------------------------------------------------------------------------

assert np.allclose(
    (
        threshold_source_df[
            FULL_P_STABLE_COL
        ]
        + threshold_source_df[
            FULL_RISK_COL
        ]
    ).to_numpy(),
    1.0,
    rtol=0.0,
    atol=1e-10,
), (
    "Full-GES score complements do not reconcile."
)

assert np.allclose(
    (
        threshold_source_df[
            NO_STAR_P_STABLE_COL
        ]
        + threshold_source_df[
            NO_STAR_RISK_COL
        ]
    ).to_numpy(),
    1.0,
    rtol=0.0,
    atol=1e-10,
), (
    "No-star-GES score complements do not reconcile."
)


# --------------------------------------------------------------------------------------------------
# 9. Outcome accounting
# --------------------------------------------------------------------------------------------------

observed_events = int(
    threshold_source_df[
        OUTCOME_COL
    ].sum()
)

observed_negatives = int(
    EXPECTED_ROWS
    - observed_events
)

observed_prevalence = (
    observed_events
    / EXPECTED_ROWS
)

assert observed_events == EXPECTED_EVENTS

assert observed_negatives == EXPECTED_NEGATIVES


# --------------------------------------------------------------------------------------------------
# 10. Helper functions
# --------------------------------------------------------------------------------------------------

def safe_divide(
    numerator,
    denominator,
):
    """Return NaN when denominator is zero."""

    if denominator == 0:
        return np.nan

    return numerator / denominator


def calculate_matthews_correlation(
    true_positive,
    true_negative,
    false_positive,
    false_negative,
):
    """Calculate Matthews correlation coefficient."""

    numerator = (
        true_positive
        * true_negative
        - false_positive
        * false_negative
    )

    denominator_squared = (
        (true_positive + false_positive)
        * (true_positive + false_negative)
        * (true_negative + false_positive)
        * (true_negative + false_negative)
    )

    if denominator_squared <= 0:
        return np.nan

    return numerator / math.sqrt(
        denominator_squared
    )


# --------------------------------------------------------------------------------------------------
# 11. Apply the exact frozen threshold
# --------------------------------------------------------------------------------------------------

model_definitions = [
    {
        "model": "Full GES",
        "short_name": "full_ges",
        "p_stable_column": FULL_P_STABLE_COL,
        "risk_column": FULL_RISK_COL,
    },
    {
        "model": "No-star GES",
        "short_name": "no_star_ges",
        "p_stable_column": NO_STAR_P_STABLE_COL,
        "risk_column": NO_STAR_RISK_COL,
    },
]

outcome_array = threshold_source_df[
    OUTCOME_COL
].to_numpy(
    dtype=np.int8
)

observed_positive = (
    outcome_array == 1
)

metric_rows = []
confusion_rows = []
threshold_audit_rows = []

stage6c_decision_threshold_membership_df = (
    threshold_source_df.copy()
)

for model_definition in model_definitions:

    model_name = model_definition[
        "model"
    ]

    short_name = model_definition[
        "short_name"
    ]

    p_stable_column = model_definition[
        "p_stable_column"
    ]

    risk_column = model_definition[
        "risk_column"
    ]

    predicted_stable_column = (
        f"{short_name}_predicted_stable_at_frozen_0_5"
    )

    predicted_instability_column = (
        f"{short_name}_predicted_instability_at_frozen_0_5"
    )

    predicted_stable = (
        threshold_source_df[
            p_stable_column
        ]
        >= FROZEN_THRESHOLD
    )

    predicted_instability = (
        threshold_source_df[
            p_stable_column
        ]
        < FROZEN_THRESHOLD
    )

    risk_based_prediction = (
        threshold_source_df[
            risk_column
        ]
        > (
            1.0
            - FROZEN_THRESHOLD
        )
    )

    assert np.array_equal(
        predicted_instability.to_numpy(),
        risk_based_prediction.to_numpy(),
    ), (
        f"{model_name}: P(stable)- and risk-based "
        "threshold predictions disagree."
    )

    assert np.array_equal(
        predicted_stable.to_numpy(),
        (~predicted_instability).to_numpy(),
    )

    stage6c_decision_threshold_membership_df[
        predicted_stable_column
    ] = predicted_stable

    stage6c_decision_threshold_membership_df[
        predicted_instability_column
    ] = predicted_instability

    predicted_positive = predicted_instability.to_numpy(
        dtype=bool
    )

    true_positive = int(
        np.sum(
            predicted_positive
            & observed_positive
        )
    )

    false_positive = int(
        np.sum(
            predicted_positive
            & ~observed_positive
        )
    )

    true_negative = int(
        np.sum(
            ~predicted_positive
            & ~observed_positive
        )
    )

    false_negative = int(
        np.sum(
            ~predicted_positive
            & observed_positive
        )
    )

    predicted_positive_records = (
        true_positive
        + false_positive
    )

    predicted_negative_records = (
        true_negative
        + false_negative
    )

    sensitivity = safe_divide(
        true_positive,
        true_positive
        + false_negative,
    )

    specificity = safe_divide(
        true_negative,
        true_negative
        + false_positive,
    )

    positive_predictive_value = safe_divide(
        true_positive,
        true_positive
        + false_positive,
    )

    negative_predictive_value = safe_divide(
        true_negative,
        true_negative
        + false_negative,
    )

    accuracy = safe_divide(
        true_positive
        + true_negative,
        EXPECTED_ROWS,
    )

    balanced_accuracy = (
        sensitivity
        + specificity
    ) / 2.0

    f1_score = safe_divide(
        2 * true_positive,
        (
            2 * true_positive
            + false_positive
            + false_negative
        ),
    )

    matthews_correlation = (
        calculate_matthews_correlation(
            true_positive=true_positive,
            true_negative=true_negative,
            false_positive=false_positive,
            false_negative=false_negative,
        )
    )

    predicted_positive_fraction = safe_divide(
        predicted_positive_records,
        EXPECTED_ROWS,
    )

    predicted_negative_fraction = safe_divide(
        predicted_negative_records,
        EXPECTED_ROWS,
    )

    predicted_positive_event_rate = (
        positive_predictive_value
    )

    predicted_negative_event_rate = safe_divide(
        false_negative,
        false_negative
        + true_negative,
    )

    event_rate_difference = (
        predicted_positive_event_rate
        - predicted_negative_event_rate
    )

    relative_event_risk = (
        predicted_positive_event_rate
        / predicted_negative_event_rate
        if predicted_negative_event_rate > 0
        else np.nan
    )

    enrichment_vs_prevalence = (
        predicted_positive_event_rate
        / observed_prevalence
        if observed_prevalence > 0
        else np.nan
    )

    false_positive_rate = (
        1.0 - specificity
    )

    false_negative_rate = (
        1.0 - sensitivity
    )

    positive_likelihood_ratio = (
        sensitivity / false_positive_rate
        if false_positive_rate > 0
        else np.inf
    )

    negative_likelihood_ratio = (
        false_negative_rate / specificity
        if specificity > 0
        else np.inf
    )

    youden_j = (
        sensitivity
        + specificity
        - 1.0
    )

    assert (
        true_positive
        + false_negative
        == EXPECTED_EVENTS
    )

    assert (
        true_negative
        + false_positive
        == EXPECTED_NEGATIVES
    )

    assert (
        true_positive
        + false_positive
        + true_negative
        + false_negative
        == EXPECTED_ROWS
    )

    metric_rows.append({
        "model": model_name,
        "threshold_source": (
            "Frozen Stage 4C reconstruction threshold"
        ),
        "p_stable_threshold": FROZEN_THRESHOLD,
        "predicted_instability_rule": (
            "P(stable) < 0.50"
        ),
        "predicted_positive_records": (
            predicted_positive_records
        ),
        "predicted_positive_fraction": (
            predicted_positive_fraction
        ),
        "predicted_negative_records": (
            predicted_negative_records
        ),
        "predicted_negative_fraction": (
            predicted_negative_fraction
        ),
        "true_positive": true_positive,
        "false_positive": false_positive,
        "true_negative": true_negative,
        "false_negative": false_negative,
        "sensitivity": sensitivity,
        "specificity": specificity,
        "positive_predictive_value": (
            positive_predictive_value
        ),
        "negative_predictive_value": (
            negative_predictive_value
        ),
        "accuracy": accuracy,
        "balanced_accuracy": (
            balanced_accuracy
        ),
        "f1_score": f1_score,
        "matthews_correlation": (
            matthews_correlation
        ),
        "false_positive_rate": (
            false_positive_rate
        ),
        "false_negative_rate": (
            false_negative_rate
        ),
        "positive_likelihood_ratio": (
            positive_likelihood_ratio
        ),
        "negative_likelihood_ratio": (
            negative_likelihood_ratio
        ),
        "youden_j": youden_j,
        "predicted_positive_event_rate": (
            predicted_positive_event_rate
        ),
        "predicted_negative_event_rate": (
            predicted_negative_event_rate
        ),
        "event_rate_difference": (
            event_rate_difference
        ),
        "relative_event_risk": (
            relative_event_risk
        ),
        "enrichment_vs_overall_prevalence": (
            enrichment_vs_prevalence
        ),
        "event_capture_fraction": (
            sensitivity
        ),
    })

    confusion_rows.extend([
        {
            "model": model_name,
            "confusion_cell": "TP",
            "observed_class": "Future instability",
            "predicted_class": "Predicted instability",
            "count": true_positive,
        },
        {
            "model": model_name,
            "confusion_cell": "FP",
            "observed_class": "Future stability",
            "predicted_class": "Predicted instability",
            "count": false_positive,
        },
        {
            "model": model_name,
            "confusion_cell": "TN",
            "observed_class": "Future stability",
            "predicted_class": "Predicted stable",
            "count": true_negative,
        },
        {
            "model": model_name,
            "confusion_cell": "FN",
            "observed_class": "Future instability",
            "predicted_class": "Predicted stable",
            "count": false_negative,
        },
    ])

    threshold_audit_rows.append({
        "model": model_name,
        "p_stable_column": p_stable_column,
        "risk_column": risk_column,
        "frozen_threshold": FROZEN_THRESHOLD,
        "predicted_stable_rule": (
            "P(stable) >= 0.50"
        ),
        "predicted_instability_rule": (
            "P(stable) < 0.50"
        ),
        "records_below_threshold": int(
            (
                threshold_source_df[
                    p_stable_column
                ]
                < FROZEN_THRESHOLD
            ).sum()
        ),
        "records_exactly_at_threshold": int(
            (
                threshold_source_df[
                    p_stable_column
                ]
                == FROZEN_THRESHOLD
            ).sum()
        ),
        "records_above_or_equal_threshold": int(
            (
                threshold_source_df[
                    p_stable_column
                ]
                >= FROZEN_THRESHOLD
            ).sum()
        ),
        "minimum_p_stable": float(
            threshold_source_df[
                p_stable_column
            ].min()
        ),
        "maximum_p_stable": float(
            threshold_source_df[
                p_stable_column
            ].max()
        ),
        "mean_p_stable": float(
            threshold_source_df[
                p_stable_column
            ].mean()
        ),
        "median_p_stable": float(
            threshold_source_df[
                p_stable_column
            ].median()
        ),
    })


# --------------------------------------------------------------------------------------------------
# 12. Create in-memory result tables
# --------------------------------------------------------------------------------------------------

stage6c_decision_threshold_point_estimates_df = (
    pd.DataFrame(
        metric_rows
    )
)

stage6c_decision_threshold_confusion_matrix_df = (
    pd.DataFrame(
        confusion_rows
    )
)

stage6c_decision_threshold_audit_df = (
    pd.DataFrame(
        threshold_audit_rows
    )
)

stage6c_decision_threshold_required_metrics_df = (
    stage6c_decision_threshold_point_estimates_df[
        [
            "model",
            "p_stable_threshold",
            "predicted_positive_records",
            "predicted_positive_fraction",
            "sensitivity",
            "specificity",
            "positive_predictive_value",
            "negative_predictive_value",
            "accuracy",
            "balanced_accuracy",
            "f1_score",
            "matthews_correlation",
            "predicted_positive_event_rate",
            "predicted_negative_event_rate",
            "relative_event_risk",
            "enrichment_vs_overall_prevalence",
        ]
    ]
    .copy()
)


# --------------------------------------------------------------------------------------------------
# 13. Final QC
# --------------------------------------------------------------------------------------------------

assert len(
    stage6c_decision_threshold_point_estimates_df
) == 2

assert len(
    stage6c_decision_threshold_confusion_matrix_df
) == 8

assert len(
    stage6c_decision_threshold_audit_df
) == 2

for metric_column in (
    "sensitivity",
    "specificity",
    "positive_predictive_value",
    "negative_predictive_value",
    "accuracy",
    "balanced_accuracy",
    "f1_score",
    "predicted_positive_fraction",
    "predicted_negative_fraction",
):

    assert (
        stage6c_decision_threshold_point_estimates_df[
            metric_column
        ]
        .between(
            0.0,
            1.0,
            inclusive="both",
        )
        .all()
    ), (
        f"{metric_column} contains a value outside [0,1]."
    )

assert (
    stage6c_decision_threshold_point_estimates_df[
        "predicted_positive_records"
    ]
    + stage6c_decision_threshold_point_estimates_df[
        "predicted_negative_records"
    ]
    == EXPECTED_ROWS
).all()


# --------------------------------------------------------------------------------------------------
# 14. Display
# --------------------------------------------------------------------------------------------------

print()
print("=" * 130)
print(
    "STAGE 6C STEP 2C — CELL 6C-2C1 — "
    "FROZEN 0.50 DECISION-THRESHOLD POINT ESTIMATES"
)
print("=" * 130)

print()
print("FROZEN INPUT VERIFICATION")
print("-" * 130)

print(
    "Stage 6B evaluable-cohort SHA-256 : "
    f"PASS ({observed_evaluable_sha256})"
)

print(
    "Stage 4C model-specification hash : "
    f"PASS ({observed_model_specification_sha256})"
)

print(
    "Parquet dimensions                : "
    f"PASS ({EXPECTED_ROWS:,} × {EXPECTED_COLUMNS})"
)

print(
    "Unique evaluable RCV keys         : "
    f"PASS ({threshold_source_df[KEY_COL].nunique():,})"
)

print(
    "Primary instability events        : "
    f"PASS ({observed_events:,})"
)

print(
    "Primary instability negatives     : "
    f"PASS ({observed_negatives:,})"
)

print(
    "Observed prevalence               : "
    f"{observed_prevalence:.8f} "
    f"({observed_prevalence * 100:.6f}%)"
)

print()
print("FROZEN THRESHOLD DEFINITION")
print("-" * 130)

print(
    "Frozen P(stable) threshold        : "
    f"{FROZEN_THRESHOLD:.2f}"
)

print(
    "Threshold purpose                 : "
    f"{threshold_purpose}"
)

print(
    "Temporal primary use              : "
    f"{temporal_primary_use}"
)

print(
    "Selected using T1                 : "
    f"{decision_threshold_block['selected_using_t1']}"
)

print(
    "Additional risk tiers selected    : "
    f"{decision_threshold_block['additional_risk_tiers_selected']}"
)

print(
    "Predicted stable                  : "
    "P(stable) >= 0.50"
)

print(
    "Predicted future instability      : "
    "P(stable) < 0.50"
)

print(
    "Threshold optimized in Stage 6C   : No"
)

print(
    "Clinical operating threshold      : No"
)

print(
    "Scientific artifact written       : No"
)

print()
print("THRESHOLD AUDIT")
print("-" * 130)

display(
    stage6c_decision_threshold_audit_df
)

print()
print("REQUIRED DECISION-THRESHOLD METRICS")
print("-" * 130)

for row in metric_rows:

    print()
    print(
        row[
            "model"
        ]
    )

    print(
        "  Predicted instability records   : "
        f"{row['predicted_positive_records']:,} "
        f"({row['predicted_positive_fraction'] * 100:.6f}%)"
    )

    print(
        "  True positives                  : "
        f"{row['true_positive']:,}"
    )

    print(
        "  False positives                 : "
        f"{row['false_positive']:,}"
    )

    print(
        "  True negatives                  : "
        f"{row['true_negative']:,}"
    )

    print(
        "  False negatives                 : "
        f"{row['false_negative']:,}"
    )

    print(
        "  Sensitivity                     : "
        f"{row['sensitivity']:.8f} "
        f"({row['sensitivity'] * 100:.6f}%)"
    )

    print(
        "  Specificity                     : "
        f"{row['specificity']:.8f} "
        f"({row['specificity'] * 100:.6f}%)"
    )

    print(
        "  Positive predictive value       : "
        f"{row['positive_predictive_value']:.8f} "
        f"({row['positive_predictive_value'] * 100:.6f}%)"
    )

    print(
        "  Negative predictive value       : "
        f"{row['negative_predictive_value']:.8f} "
        f"({row['negative_predictive_value'] * 100:.6f}%)"
    )

    print(
        "  Accuracy                        : "
        f"{row['accuracy']:.8f}"
    )

    print(
        "  Balanced accuracy               : "
        f"{row['balanced_accuracy']:.8f}"
    )

    print(
        "  F1 score                        : "
        f"{row['f1_score']:.8f}"
    )

    print(
        "  Matthews correlation            : "
        f"{row['matthews_correlation']:.8f}"
    )

    print(
        "  Positive-group event rate       : "
        f"{row['predicted_positive_event_rate']:.8f} "
        f"({row['predicted_positive_event_rate'] * 100:.6f}%)"
    )

    print(
        "  Negative-group event rate       : "
        f"{row['predicted_negative_event_rate']:.8f} "
        f"({row['predicted_negative_event_rate'] * 100:.6f}%)"
    )

    print(
        "  Relative event risk             : "
        f"{row['relative_event_risk']:.8f}×"
    )

    print(
        "  Enrichment vs prevalence        : "
        f"{row['enrichment_vs_overall_prevalence']:.8f}×"
    )

print()
print("COMPACT REQUIRED-METRIC TABLE")
print("-" * 130)

display(
    stage6c_decision_threshold_required_metrics_df
)

print()
print("COMPLETE POINT-ESTIMATE TABLE")
print("-" * 130)

display(
    stage6c_decision_threshold_point_estimates_df
)

print()
print("CONFUSION-MATRIX TABLE")
print("-" * 130)

display(
    stage6c_decision_threshold_confusion_matrix_df
)

print()
print("IN-MEMORY MEMBERSHIP TABLE")
print("-" * 130)

print(
    "Shape: "
    f"{stage6c_decision_threshold_membership_df.shape}"
)

display(
    stage6c_decision_threshold_membership_df.head(
        10
    )
)

print()
print("CELL DECISION")
print("-" * 130)

print(
    "PASS_STAGE6C_FROZEN_0_50_DECISION_THRESHOLD_"
    "POINT_ESTIMATES_COMPLETE"
)

print(
    "The original frozen Stage 4C reconstruction threshold "
    "was evaluated descriptively against the temporal outcome."
)

print(
    "No cutoff search, threshold optimization, recalibration, "
    "score modification, or scientific-artifact write occurred."
)


STAGE 6C STEP 2C — CELL 6C-2C1 — FROZEN 0.50 DECISION-THRESHOLD POINT ESTIMATES

FROZEN INPUT VERIFICATION
----------------------------------------------------------------------------------------------------------------------------------
Stage 6B evaluable-cohort SHA-256 : PASS (c6ad50dfc376a746e1830d000dba975101d88763fd61b8d1983f6300edb6e038)
Stage 4C model-specification hash : PASS (d754c715c990f42cecd64259ca2c420427b9602c669dc7be9e80dee9554f61f6)
Parquet dimensions                : PASS (66,636 × 79)
Unique evaluable RCV keys         : PASS (66,636)
Primary instability events        : PASS (6,485)
Primary instability negatives     : PASS (60,151)
Observed prevalence               : 0.09731977 (9.731977%)

FROZEN THRESHOLD DEFINITION
----------------------------------------------------------------------------------------------------------------------------------
Frozen P(stable) threshold        : 0.50
Threshold purpose                 : Fixed reconstruction decision threshold only.

,model,p_stable_column,risk_column,frozen_threshold,predicted_stable_rule,predicted_instability_rule,records_below_threshold,records_exactly_at_threshold,records_above_or_equal_threshold,minimum_p_stable,maximum_p_stable,mean_p_stable,median_p_stable
0,Full GES,full_ges_p_stable_t0,full_ges_instability_risk_t0,0.5,P(stable) >= 0.50,P(stable) < 0.50,4499,0,62137,2.174233e-15,1.0,0.939441,0.999998
1,No-star GES,no_star_ges_p_stable_t0,no_star_ges_instability_risk_t0,0.5,P(stable) >= 0.50,P(stable) < 0.50,2296,0,64340,2.176185e-16,1.0,0.965275,0.999999



REQUIRED DECISION-THRESHOLD METRICS
----------------------------------------------------------------------------------------------------------------------------------

Full GES
  Predicted instability records   : 4,499 (6.751606%)
  True positives                  : 442
  False positives                 : 4,057
  True negatives                  : 56,094
  False negatives                 : 6,043
  Sensitivity                     : 0.06815729 (6.815729%)
  Specificity                     : 0.93255307 (93.255307%)
  Positive predictive value       : 0.09824405 (9.824405%)
  Negative predictive value       : 0.90274716 (90.274716%)
  Accuracy                        : 0.84843028
  Balanced accuracy               : 0.50035518
  F1 score                        : 0.08048070
  Matthews correlation            : 0.00083912
  Positive-group event rate       : 0.09824405 (9.824405%)
  Negative-group event rate       : 0.09725284 (9.725284%)
  Relative event risk             : 1.01019209×
  Enrichm

,model,p_stable_threshold,predicted_positive_records,predicted_positive_fraction,sensitivity,specificity,positive_predictive_value,negative_predictive_value,accuracy,balanced_accuracy,f1_score,matthews_correlation,predicted_positive_event_rate,predicted_negative_event_rate,relative_event_risk,enrichment_vs_overall_prevalence
0,Full GES,0.5,4499,0.067516,0.068157,0.932553,0.098244,0.902747,0.848430,0.500355,0.080481,0.000839,0.098244,0.097253,1.010192,1.009497
1,No-star GES,0.5,2296,0.034456,0.056284,0.967897,0.158972,0.904880,0.879179,0.512091,0.083134,0.039294,0.158972,0.095120,1.671285,1.633503



COMPLETE POINT-ESTIMATE TABLE
----------------------------------------------------------------------------------------------------------------------------------


...


CONFUSION-MATRIX TABLE
----------------------------------------------------------------------------------------------------------------------------------


,model,confusion_cell,observed_class,predicted_class,count
0,Full GES,TP,Future instability,Predicted instability,442
1,Full GES,FP,Future stability,Predicted instability,4057
2,Full GES,TN,Future stability,Predicted stable,56094
3,Full GES,FN,Future instability,Predicted stable,6043
4,No-star GES,TP,Future instability,Predicted instability,365
5,No-star GES,FP,Future stability,Predicted instability,1931
6,No-star GES,TN,Future stability,Predicted stable,58220
7,No-star GES,FN,Future instability,Predicted stable,6120



IN-MEMORY MEMBERSHIP TABLE
----------------------------------------------------------------------------------------------------------------------------------
Shape: (66636, 10)


,rcv_accession,primary_future_instability,full_ges_p_stable_t0,full_ges_instability_risk_t0,no_star_ges_p_stable_t0,no_star_ges_instability_risk_t0,full_ges_predicted_stable_at_frozen_0_5,full_ges_predicted_instability_at_frozen_0_5,no_star_ges_predicted_stable_at_frozen_0_5,no_star_ges_predicted_instability_at_frozen_0_5
0,RCV000052656,0,0.953676,0.046324,0.991220,0.008780,True,False,True,False
1,RCV000053439,0,0.953676,0.046324,0.991220,0.008780,True,False,True,False
2,RCV000053440,0,0.953676,0.046324,0.991220,0.008780,True,False,True,False
3,RCV000053441,0,0.953676,0.046324,0.991220,0.008780,True,False,True,False
4,RCV000053532,0,0.953676,0.046324,0.991220,0.008780,True,False,True,False
5,RCV000053534,0,0.953676,0.046324,0.991220,0.008780,True,False,True,False
6,RCV000136092,0,0.910650,0.089350,0.981322,0.018678,True,False,True,False
7,RCV000139830,0,0.970656,0.029344,0.994740,0.005260,True,False,True,False
8,RCV000143077,0,0.986952,0.013048,0.997854,0.002146,True,False,True,False
9,RCV000206923,0,0.992705,0.007295,0.998866,0.001134,True,False,True,False



CELL DECISION
----------------------------------------------------------------------------------------------------------------------------------
PASS_STAGE6C_FROZEN_0_50_DECISION_THRESHOLD_POINT_ESTIMATES_COMPLETE
The original frozen Stage 4C reconstruction threshold was evaluated descriptively against the temporal outcome.
No cutoff search, threshold optimization, recalibration, score modification, or scientific-artifact write occurred.


In [22]:
# ==================================================================================================
# STAGE 6C — STEP 2C
# CELL 6C-2C2 — FROZEN 0.50 DECISION-THRESHOLD BOOTSTRAP INFERENCE
#
# Purpose:
#   1. Quantify uncertainty for the frozen 0.50 threshold results.
#   2. Use 2,000 nonparametric row-bootstrap replicates with seed 42.
#   3. Use identical resampled rows for full GES and no-star GES.
#   4. Report percentile 95% confidence intervals for:
#        - sensitivity;
#        - specificity;
#        - positive predictive value;
#        - negative predictive value;
#        - accuracy;
#        - balanced accuracy;
#        - F1 score;
#        - Matthews correlation coefficient;
#        - predicted-positive fraction;
#        - predicted-positive and predicted-negative event rates;
#        - event-rate difference;
#        - relative event risk;
#        - enrichment versus bootstrap prevalence.
#   5. Calculate paired full-GES-minus-no-star differences for key threshold metrics.
#
# Scientific boundary:
#   - The frozen threshold and record-level predictions remain unchanged.
#   - The threshold is not recalculated inside bootstrap samples.
#   - The same resampling sequence is used across both models.
#   - No threshold search, optimization, recalibration, or scientific-artifact write occurs.
# ==================================================================================================

import time

import numpy as np
import pandas as pd
from IPython.display import display


# --------------------------------------------------------------------------------------------------
# 1. Confirm that Cell 6C-2C1 objects remain available
# --------------------------------------------------------------------------------------------------

assert "stage6c_decision_threshold_membership_df" in globals(), (
    "The frozen-threshold membership dataframe is unavailable. "
    "Run Cell 6C-2C1 immediately before this cell."
)

assert "stage6c_decision_threshold_point_estimates_df" in globals(), (
    "The frozen-threshold point-estimate dataframe is unavailable. "
    "Run Cell 6C-2C1 immediately before this cell."
)

threshold_df = (
    stage6c_decision_threshold_membership_df
    .copy()
)

OUTCOME_COL_LOCAL = "primary_future_instability"

MODEL_DEFINITIONS = [
    {
        "model": "Full GES",
        "short_name": "full_ges",
        "prediction_column": (
            "full_ges_predicted_instability_at_frozen_0_5"
        ),
    },
    {
        "model": "No-star GES",
        "short_name": "no_star_ges",
        "prediction_column": (
            "no_star_ges_predicted_instability_at_frozen_0_5"
        ),
    },
]

required_columns = [
    OUTCOME_COL_LOCAL,
    *[
        model_definition["prediction_column"]
        for model_definition in MODEL_DEFINITIONS
    ],
]

missing_columns = [
    column
    for column in required_columns
    if column not in threshold_df.columns
]

assert not missing_columns, (
    f"Required threshold-membership columns are missing: "
    f"{missing_columns}"
)


# --------------------------------------------------------------------------------------------------
# 2. Locked constants
# --------------------------------------------------------------------------------------------------

N = int(
    len(threshold_df)
)

BOOTSTRAP_REPLICATES = 2_000
BOOTSTRAP_SEED = 42
BOOTSTRAP_BATCH_SIZE = 25

EXPECTED_ROWS = 66_636
EXPECTED_EVENTS = 6_485
EXPECTED_NEGATIVES = 60_151

assert N == EXPECTED_ROWS


# --------------------------------------------------------------------------------------------------
# 3. Prepare frozen outcome and prediction arrays
# --------------------------------------------------------------------------------------------------

outcome_array = pd.to_numeric(
    threshold_df[
        OUTCOME_COL_LOCAL
    ],
    errors="raise",
).to_numpy(
    dtype=np.int8
)

assert set(
    np.unique(
        outcome_array
    ).tolist()
) == {
    0,
    1,
}

observed_positive_array = (
    outcome_array == 1
)

prediction_matrix = np.column_stack([
    threshold_df[
        model_definition[
            "prediction_column"
        ]
    ].astype(
        bool
    ).to_numpy()
    for model_definition in MODEL_DEFINITIONS
])

assert prediction_matrix.shape == (
    N,
    2,
)

TOTAL_EVENTS = int(
    outcome_array.sum()
)

TOTAL_NEGATIVES = int(
    N - TOTAL_EVENTS
)

OBSERVED_PREVALENCE = (
    TOTAL_EVENTS / N
)

assert TOTAL_EVENTS == EXPECTED_EVENTS
assert TOTAL_NEGATIVES == EXPECTED_NEGATIVES


# --------------------------------------------------------------------------------------------------
# 4. Safe vectorized division
# --------------------------------------------------------------------------------------------------

def safe_array_divide(
    numerator,
    denominator,
):
    """Vectorized division returning NaN where the denominator is zero."""

    numerator_array = np.asarray(
        numerator,
        dtype=np.float64,
    )

    denominator_array = np.asarray(
        denominator,
        dtype=np.float64,
    )

    output_shape = np.broadcast_shapes(
        numerator_array.shape,
        denominator_array.shape,
    )

    result = np.full(
        output_shape,
        np.nan,
        dtype=np.float64,
    )

    np.divide(
        numerator_array,
        denominator_array,
        out=result,
        where=(
            denominator_array != 0
        ),
    )

    return result


# --------------------------------------------------------------------------------------------------
# 5. Recalculate locked point estimates independently
# --------------------------------------------------------------------------------------------------

point_true_positive = np.sum(
    prediction_matrix
    & observed_positive_array[
        :,
        np.newaxis,
    ],
    axis=0,
    dtype=np.int64,
)

point_false_positive = np.sum(
    prediction_matrix
    & ~observed_positive_array[
        :,
        np.newaxis,
    ],
    axis=0,
    dtype=np.int64,
)

point_true_negative = np.sum(
    ~prediction_matrix
    & ~observed_positive_array[
        :,
        np.newaxis,
    ],
    axis=0,
    dtype=np.int64,
)

point_false_negative = np.sum(
    ~prediction_matrix
    & observed_positive_array[
        :,
        np.newaxis,
    ],
    axis=0,
    dtype=np.int64,
)


def calculate_metric_arrays(
    true_positive,
    false_positive,
    true_negative,
    false_negative,
    total_events,
):
    """Calculate all threshold metrics from arrays of confusion counts."""

    true_positive = np.asarray(
        true_positive,
        dtype=np.float64,
    )

    false_positive = np.asarray(
        false_positive,
        dtype=np.float64,
    )

    true_negative = np.asarray(
        true_negative,
        dtype=np.float64,
    )

    false_negative = np.asarray(
        false_negative,
        dtype=np.float64,
    )

    predicted_positive = (
        true_positive
        + false_positive
    )

    predicted_negative = (
        true_negative
        + false_negative
    )

    sensitivity = safe_array_divide(
        true_positive,
        true_positive
        + false_negative,
    )

    specificity = safe_array_divide(
        true_negative,
        true_negative
        + false_positive,
    )

    positive_predictive_value = safe_array_divide(
        true_positive,
        predicted_positive,
    )

    negative_predictive_value = safe_array_divide(
        true_negative,
        predicted_negative,
    )

    accuracy = safe_array_divide(
        true_positive
        + true_negative,
        (
            true_positive
            + false_positive
            + true_negative
            + false_negative
        ),
    )

    balanced_accuracy = (
        sensitivity
        + specificity
    ) / 2.0

    f1_score = safe_array_divide(
        2.0
        * true_positive,
        (
            2.0
            * true_positive
            + false_positive
            + false_negative
        ),
    )

    mcc_numerator = (
        true_positive
        * true_negative
        - false_positive
        * false_negative
    )

    mcc_denominator = np.sqrt(
        (
            true_positive
            + false_positive
        )
        * (
            true_positive
            + false_negative
        )
        * (
            true_negative
            + false_positive
        )
        * (
            true_negative
            + false_negative
        )
    )

    matthews_correlation = safe_array_divide(
        mcc_numerator,
        mcc_denominator,
    )

    predicted_positive_fraction = safe_array_divide(
        predicted_positive,
        N,
    )

    predicted_negative_fraction = safe_array_divide(
        predicted_negative,
        N,
    )

    predicted_positive_event_rate = (
        positive_predictive_value
    )

    predicted_negative_event_rate = safe_array_divide(
        false_negative,
        predicted_negative,
    )

    event_rate_difference = (
        predicted_positive_event_rate
        - predicted_negative_event_rate
    )

    relative_event_risk = safe_array_divide(
        predicted_positive_event_rate,
        predicted_negative_event_rate,
    )

    total_events_array = np.asarray(
        total_events,
        dtype=np.float64,
    )

    bootstrap_prevalence = safe_array_divide(
        total_events_array,
        N,
    )

    if bootstrap_prevalence.ndim == 1:

        bootstrap_prevalence_for_models = (
            bootstrap_prevalence[
                :,
                np.newaxis,
            ]
        )

    else:

        bootstrap_prevalence_for_models = (
            bootstrap_prevalence
        )

    enrichment_vs_prevalence = safe_array_divide(
        predicted_positive_event_rate,
        bootstrap_prevalence_for_models,
    )

    event_capture_fraction = (
        sensitivity
    )

    youden_j = (
        sensitivity
        + specificity
        - 1.0
    )

    return {
        "sensitivity": sensitivity,
        "specificity": specificity,
        "positive_predictive_value": (
            positive_predictive_value
        ),
        "negative_predictive_value": (
            negative_predictive_value
        ),
        "accuracy": accuracy,
        "balanced_accuracy": (
            balanced_accuracy
        ),
        "f1_score": f1_score,
        "matthews_correlation": (
            matthews_correlation
        ),
        "predicted_positive_fraction": (
            predicted_positive_fraction
        ),
        "predicted_negative_fraction": (
            predicted_negative_fraction
        ),
        "predicted_positive_event_rate": (
            predicted_positive_event_rate
        ),
        "predicted_negative_event_rate": (
            predicted_negative_event_rate
        ),
        "event_rate_difference": (
            event_rate_difference
        ),
        "relative_event_risk": (
            relative_event_risk
        ),
        "enrichment_vs_overall_prevalence": (
            enrichment_vs_prevalence
        ),
        "event_capture_fraction": (
            event_capture_fraction
        ),
        "youden_j": youden_j,
    }


point_metrics = calculate_metric_arrays(
    true_positive=point_true_positive,
    false_positive=point_false_positive,
    true_negative=point_true_negative,
    false_negative=point_false_negative,
    total_events=np.asarray(
        [TOTAL_EVENTS],
        dtype=np.int64,
    ),
)

# Remove the singleton prevalence dimension introduced for point estimates.
for metric_name, metric_values in point_metrics.items():

    metric_values = np.asarray(
        metric_values
    )

    if metric_values.ndim == 2:

        assert metric_values.shape == (
            1,
            len(MODEL_DEFINITIONS),
        )

        point_metrics[
            metric_name
        ] = metric_values[
            0,
            :
        ]


# Reconcile with Cell 6C-2C1.
point_estimate_lookup = (
    stage6c_decision_threshold_point_estimates_df
    .set_index(
        "model"
    )
)

for model_index, model_definition in enumerate(
    MODEL_DEFINITIONS
):

    model_name = model_definition[
        "model"
    ]

    prior_row = point_estimate_lookup.loc[
        model_name
    ]

    assert int(
        prior_row[
            "true_positive"
        ]
    ) == int(
        point_true_positive[
            model_index
        ]
    )

    assert int(
        prior_row[
            "false_positive"
        ]
    ) == int(
        point_false_positive[
            model_index
        ]
    )

    assert int(
        prior_row[
            "true_negative"
        ]
    ) == int(
        point_true_negative[
            model_index
        ]
    )

    assert int(
        prior_row[
            "false_negative"
        ]
    ) == int(
        point_false_negative[
            model_index
        ]
    )

    for metric_name in (
        "sensitivity",
        "specificity",
        "positive_predictive_value",
        "negative_predictive_value",
        "accuracy",
        "balanced_accuracy",
        "f1_score",
        "matthews_correlation",
        "predicted_positive_fraction",
        "predicted_positive_event_rate",
        "predicted_negative_event_rate",
        "event_rate_difference",
        "relative_event_risk",
        "enrichment_vs_overall_prevalence",
    ):

        assert np.isclose(
            float(
                point_metrics[
                    metric_name
                ][
                    model_index
                ]
            ),
            float(
                prior_row[
                    metric_name
                ]
            ),
            rtol=0.0,
            atol=1e-12,
        ), (
            f"Point-estimate reconciliation failed for "
            f"{model_name}: {metric_name}"
        )


# --------------------------------------------------------------------------------------------------
# 6. Preallocate bootstrap confusion-count arrays
# --------------------------------------------------------------------------------------------------

NUMBER_OF_MODELS = len(
    MODEL_DEFINITIONS
)

bootstrap_true_positive = np.empty(
    (
        BOOTSTRAP_REPLICATES,
        NUMBER_OF_MODELS,
    ),
    dtype=np.int32,
)

bootstrap_false_positive = np.empty(
    (
        BOOTSTRAP_REPLICATES,
        NUMBER_OF_MODELS,
    ),
    dtype=np.int32,
)

bootstrap_true_negative = np.empty(
    (
        BOOTSTRAP_REPLICATES,
        NUMBER_OF_MODELS,
    ),
    dtype=np.int32,
)

bootstrap_false_negative = np.empty(
    (
        BOOTSTRAP_REPLICATES,
        NUMBER_OF_MODELS,
    ),
    dtype=np.int32,
)

bootstrap_total_events = np.empty(
    BOOTSTRAP_REPLICATES,
    dtype=np.int32,
)


# --------------------------------------------------------------------------------------------------
# 7. Run the paired nonparametric row bootstrap
#
# Seed and batching match the enrichment bootstrap cell so that the same deterministic row-draw
# stream is reproduced when both cells are run from their own newly initialized seed-42 generator.
# --------------------------------------------------------------------------------------------------

rng = np.random.default_rng(
    BOOTSTRAP_SEED
)

bootstrap_start_time = time.time()

for batch_start in range(
    0,
    BOOTSTRAP_REPLICATES,
    BOOTSTRAP_BATCH_SIZE,
):

    batch_stop = min(
        batch_start
        + BOOTSTRAP_BATCH_SIZE,
        BOOTSTRAP_REPLICATES,
    )

    current_batch_size = (
        batch_stop
        - batch_start
    )

    draw_indices = rng.integers(
        low=0,
        high=N,
        size=(
            current_batch_size,
            N,
        ),
        dtype=np.int32,
    )

    sampled_outcomes = (
        outcome_array[
            draw_indices
        ]
        == 1
    )

    sampled_predictions = (
        prediction_matrix[
            draw_indices,
            :,
        ]
    )

    true_positive_batch = np.sum(
        sampled_predictions
        & sampled_outcomes[
            :,
            :,
            np.newaxis,
        ],
        axis=1,
        dtype=np.int64,
    )

    false_positive_batch = np.sum(
        sampled_predictions
        & ~sampled_outcomes[
            :,
            :,
            np.newaxis,
        ],
        axis=1,
        dtype=np.int64,
    )

    true_negative_batch = np.sum(
        ~sampled_predictions
        & ~sampled_outcomes[
            :,
            :,
            np.newaxis,
        ],
        axis=1,
        dtype=np.int64,
    )

    false_negative_batch = np.sum(
        ~sampled_predictions
        & sampled_outcomes[
            :,
            :,
            np.newaxis,
        ],
        axis=1,
        dtype=np.int64,
    )

    total_events_batch = np.sum(
        sampled_outcomes,
        axis=1,
        dtype=np.int64,
    )

    bootstrap_true_positive[
        batch_start:batch_stop,
        :,
    ] = true_positive_batch

    bootstrap_false_positive[
        batch_start:batch_stop,
        :,
    ] = false_positive_batch

    bootstrap_true_negative[
        batch_start:batch_stop,
        :,
    ] = true_negative_batch

    bootstrap_false_negative[
        batch_start:batch_stop,
        :,
    ] = false_negative_batch

    bootstrap_total_events[
        batch_start:batch_stop
    ] = total_events_batch

    completed_replicates = batch_stop

    if (
        completed_replicates % 250 == 0
        or completed_replicates
        == BOOTSTRAP_REPLICATES
    ):

        elapsed_seconds = (
            time.time()
            - bootstrap_start_time
        )

        print(
            f"Completed "
            f"{completed_replicates:,}/"
            f"{BOOTSTRAP_REPLICATES:,} "
            f"threshold bootstrap replicates "
            f"({elapsed_seconds:.1f} seconds elapsed)"
        )


bootstrap_elapsed_seconds = (
    time.time()
    - bootstrap_start_time
)


# --------------------------------------------------------------------------------------------------
# 8. Bootstrap confusion-count QC
# --------------------------------------------------------------------------------------------------

assert (
    bootstrap_true_positive
    + bootstrap_false_negative
    == bootstrap_total_events[
        :,
        np.newaxis,
    ]
).all(), (
    "Bootstrap positive-outcome accounting failed."
)

assert (
    bootstrap_true_negative
    + bootstrap_false_positive
    == (
        N
        - bootstrap_total_events
    )[
        :,
        np.newaxis,
    ]
).all(), (
    "Bootstrap negative-outcome accounting failed."
)

assert (
    bootstrap_true_positive
    + bootstrap_false_positive
    + bootstrap_true_negative
    + bootstrap_false_negative
    == N
).all(), (
    "Bootstrap total-row accounting failed."
)

assert (
    bootstrap_total_events
    > 0
).all()

assert (
    bootstrap_total_events
    < N
).all()


# --------------------------------------------------------------------------------------------------
# 9. Calculate bootstrap metric arrays
# --------------------------------------------------------------------------------------------------

bootstrap_metrics = calculate_metric_arrays(
    true_positive=bootstrap_true_positive,
    false_positive=bootstrap_false_positive,
    true_negative=bootstrap_true_negative,
    false_negative=bootstrap_false_negative,
    total_events=bootstrap_total_events,
)

for metric_name, metric_values in bootstrap_metrics.items():

    assert metric_values.shape == (
        BOOTSTRAP_REPLICATES,
        NUMBER_OF_MODELS,
    ), (
        f"Unexpected bootstrap metric shape for "
        f"{metric_name}: {metric_values.shape}"
    )

    assert np.isfinite(
        metric_values
    ).all(), (
        f"Bootstrap metric contains nonfinite values: "
        f"{metric_name}"
    )


# --------------------------------------------------------------------------------------------------
# 10. Create complete bootstrap-summary table
# --------------------------------------------------------------------------------------------------

METRIC_NULL_VALUES = {
    "sensitivity": np.nan,
    "specificity": np.nan,
    "positive_predictive_value": np.nan,
    "negative_predictive_value": np.nan,
    "accuracy": np.nan,
    "balanced_accuracy": 0.50,
    "f1_score": np.nan,
    "matthews_correlation": 0.0,
    "predicted_positive_fraction": np.nan,
    "predicted_negative_fraction": np.nan,
    "predicted_positive_event_rate": np.nan,
    "predicted_negative_event_rate": np.nan,
    "event_rate_difference": 0.0,
    "relative_event_risk": 1.0,
    "enrichment_vs_overall_prevalence": 1.0,
    "event_capture_fraction": np.nan,
    "youden_j": 0.0,
}

summary_rows = []

for model_index, model_definition in enumerate(
    MODEL_DEFINITIONS
):

    model_name = model_definition[
        "model"
    ]

    for metric_name, metric_values in bootstrap_metrics.items():

        values = metric_values[
            :,
            model_index,
        ]

        point_estimate = float(
            point_metrics[
                metric_name
            ][
                model_index
            ]
        )

        percentile_lower, percentile_upper = np.quantile(
            values,
            [
                0.025,
                0.975,
            ],
        )

        null_value = METRIC_NULL_VALUES[
            metric_name
        ]

        if np.isfinite(
            null_value
        ):

            support_above_null = float(
                (
                    np.sum(
                        values
                        > null_value
                    )
                    + 0.5
                    * np.sum(
                        values
                        == null_value
                    )
                )
                / BOOTSTRAP_REPLICATES
            )

            interval_excludes_null_above = bool(
                percentile_lower
                > null_value
            )

            interval_excludes_null_below = bool(
                percentile_upper
                < null_value
            )

        else:

            support_above_null = np.nan
            interval_excludes_null_above = False
            interval_excludes_null_below = False

        summary_rows.append({
            "model": model_name,
            "metric": metric_name,
            "point_estimate": (
                point_estimate
            ),
            "bootstrap_mean": float(
                np.mean(
                    values
                )
            ),
            "bootstrap_standard_error": float(
                np.std(
                    values,
                    ddof=1,
                )
            ),
            "percentile_95_ci_lower": float(
                percentile_lower
            ),
            "percentile_95_ci_upper": float(
                percentile_upper
            ),
            "null_value": (
                null_value
            ),
            "bootstrap_support_above_null": (
                support_above_null
            ),
            "interval_excludes_null_above": (
                interval_excludes_null_above
            ),
            "interval_excludes_null_below": (
                interval_excludes_null_below
            ),
            "bootstrap_replicates": (
                BOOTSTRAP_REPLICATES
            ),
            "bootstrap_seed": (
                BOOTSTRAP_SEED
            ),
            "threshold_rule": (
                "P(stable) < 0.50 predicts instability"
            ),
        })


stage6c_decision_threshold_bootstrap_summary_df = (
    pd.DataFrame(
        summary_rows
    )
)


# --------------------------------------------------------------------------------------------------
# 11. Paired full-GES-minus-no-star comparisons
# --------------------------------------------------------------------------------------------------

PAIRED_COMPARISON_METRICS = [
    "sensitivity",
    "specificity",
    "positive_predictive_value",
    "negative_predictive_value",
    "balanced_accuracy",
    "f1_score",
    "matthews_correlation",
    "event_rate_difference",
]

paired_rows = []

for metric_name in PAIRED_COMPARISON_METRICS:

    full_values = (
        bootstrap_metrics[
            metric_name
        ][
            :,
            0,
        ]
    )

    no_star_values = (
        bootstrap_metrics[
            metric_name
        ][
            :,
            1,
        ]
    )

    paired_difference = (
        full_values
        - no_star_values
    )

    point_difference = float(
        point_metrics[
            metric_name
        ][0]
        - point_metrics[
            metric_name
        ][1]
    )

    percentile_lower, percentile_upper = np.quantile(
        paired_difference,
        [
            0.025,
            0.975,
        ],
    )

    directional_probability_above_zero = float(
        (
            np.sum(
                paired_difference
                > 0.0
            )
            + 0.5
            * np.sum(
                paired_difference
                == 0.0
            )
        )
        / BOOTSTRAP_REPLICATES
    )

    paired_rows.append({
        "comparison": (
            "Full GES minus No-star GES"
        ),
        "metric": metric_name,
        "point_difference": (
            point_difference
        ),
        "bootstrap_mean_difference": float(
            np.mean(
                paired_difference
            )
        ),
        "bootstrap_standard_error": float(
            np.std(
                paired_difference,
                ddof=1,
            )
        ),
        "percentile_95_ci_lower": float(
            percentile_lower
        ),
        "percentile_95_ci_upper": float(
            percentile_upper
        ),
        "directional_probability_above_zero": (
            directional_probability_above_zero
        ),
        "interval_excludes_zero_above": bool(
            percentile_lower > 0.0
        ),
        "interval_excludes_zero_below": bool(
            percentile_upper < 0.0
        ),
        "bootstrap_replicates": (
            BOOTSTRAP_REPLICATES
        ),
        "bootstrap_seed": (
            BOOTSTRAP_SEED
        ),
        "paired_resamples": True,
    })


stage6c_decision_threshold_paired_comparisons_df = (
    pd.DataFrame(
        paired_rows
    )
)


# --------------------------------------------------------------------------------------------------
# 12. Create raw bootstrap replicate table
# --------------------------------------------------------------------------------------------------

replicate_data = {
    "bootstrap_replicate": np.arange(
        1,
        BOOTSTRAP_REPLICATES + 1,
        dtype=np.int32,
    ),
    "bootstrap_total_events": (
        bootstrap_total_events
    ),
    "bootstrap_prevalence": (
        bootstrap_total_events
        / N
    ),
}

for model_index, model_definition in enumerate(
    MODEL_DEFINITIONS
):

    short_name = model_definition[
        "short_name"
    ]

    replicate_data[
        f"{short_name}_true_positive"
    ] = bootstrap_true_positive[
        :,
        model_index,
    ]

    replicate_data[
        f"{short_name}_false_positive"
    ] = bootstrap_false_positive[
        :,
        model_index,
    ]

    replicate_data[
        f"{short_name}_true_negative"
    ] = bootstrap_true_negative[
        :,
        model_index,
    ]

    replicate_data[
        f"{short_name}_false_negative"
    ] = bootstrap_false_negative[
        :,
        model_index,
    ]

    for metric_name, metric_values in bootstrap_metrics.items():

        replicate_data[
            f"{short_name}_{metric_name}"
        ] = metric_values[
            :,
            model_index,
        ]


stage6c_decision_threshold_bootstrap_replicates_df = (
    pd.DataFrame(
        replicate_data
    )
)

assert len(
    stage6c_decision_threshold_bootstrap_replicates_df
) == BOOTSTRAP_REPLICATES


# --------------------------------------------------------------------------------------------------
# 13. Count-distribution QC table
# --------------------------------------------------------------------------------------------------

count_qc_rows = []

for model_index, model_definition in enumerate(
    MODEL_DEFINITIONS
):

    count_qc_rows.append({
        "model": model_definition[
            "model"
        ],
        "locked_true_positive": int(
            point_true_positive[
                model_index
            ]
        ),
        "bootstrap_mean_true_positive": float(
            np.mean(
                bootstrap_true_positive[
                    :,
                    model_index,
                ]
            )
        ),
        "bootstrap_minimum_true_positive": int(
            np.min(
                bootstrap_true_positive[
                    :,
                    model_index,
                ]
            )
        ),
        "bootstrap_maximum_true_positive": int(
            np.max(
                bootstrap_true_positive[
                    :,
                    model_index,
                ]
            )
        ),
        "locked_predicted_positive": int(
            point_true_positive[
                model_index
            ]
            + point_false_positive[
                model_index
            ]
        ),
        "bootstrap_mean_predicted_positive": float(
            np.mean(
                bootstrap_true_positive[
                    :,
                    model_index,
                ]
                + bootstrap_false_positive[
                    :,
                    model_index,
                ]
            )
        ),
        "bootstrap_minimum_predicted_positive": int(
            np.min(
                bootstrap_true_positive[
                    :,
                    model_index,
                ]
                + bootstrap_false_positive[
                    :,
                    model_index,
                ]
            )
        ),
        "bootstrap_maximum_predicted_positive": int(
            np.max(
                bootstrap_true_positive[
                    :,
                    model_index,
                ]
                + bootstrap_false_positive[
                    :,
                    model_index,
                ]
            )
        ),
    })


stage6c_decision_threshold_bootstrap_count_qc_df = (
    pd.DataFrame(
        count_qc_rows
    )
)


# --------------------------------------------------------------------------------------------------
# 14. Compact required-metric bootstrap table
# --------------------------------------------------------------------------------------------------

REQUIRED_THRESHOLD_METRICS = [
    "sensitivity",
    "specificity",
    "positive_predictive_value",
    "negative_predictive_value",
    "balanced_accuracy",
    "f1_score",
    "matthews_correlation",
]

stage6c_decision_threshold_key_bootstrap_summary_df = (
    stage6c_decision_threshold_bootstrap_summary_df
    .loc[
        stage6c_decision_threshold_bootstrap_summary_df[
            "metric"
        ].isin(
            REQUIRED_THRESHOLD_METRICS
        )
    ]
    .copy()
    .sort_values(
        by=[
            "model",
            "metric",
        ],
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)


# --------------------------------------------------------------------------------------------------
# 15. Final QC
# --------------------------------------------------------------------------------------------------

assert len(
    stage6c_decision_threshold_bootstrap_summary_df
) == (
    len(
        MODEL_DEFINITIONS
    )
    * len(
        bootstrap_metrics
    )
)

assert len(
    stage6c_decision_threshold_paired_comparisons_df
) == len(
    PAIRED_COMPARISON_METRICS
)

assert (
    stage6c_decision_threshold_paired_comparisons_df[
        "paired_resamples"
    ]
    .all()
)

for model_name in (
    "Full GES",
    "No-star GES",
):

    model_key_rows = (
        stage6c_decision_threshold_key_bootstrap_summary_df
        .loc[
            stage6c_decision_threshold_key_bootstrap_summary_df[
                "model"
            ]
            == model_name
        ]
    )

    assert len(
        model_key_rows
    ) == len(
        REQUIRED_THRESHOLD_METRICS
    )


# --------------------------------------------------------------------------------------------------
# 16. Display results
# --------------------------------------------------------------------------------------------------

print()
print("=" * 132)
print(
    "STAGE 6C STEP 2C — CELL 6C-2C2 — "
    "FROZEN 0.50 DECISION-THRESHOLD BOOTSTRAP INFERENCE"
)
print("=" * 132)

print()
print("BOOTSTRAP DESIGN")
print("-" * 132)

print(
    f"Evaluable records                   : "
    f"{N:,}"
)

print(
    f"Primary instability events         : "
    f"{TOTAL_EVENTS:,}"
)

print(
    f"Observed prevalence                : "
    f"{OBSERVED_PREVALENCE:.8f} "
    f"({OBSERVED_PREVALENCE * 100:.6f}%)"
)

print(
    f"Bootstrap replicates               : "
    f"{BOOTSTRAP_REPLICATES:,}"
)

print(
    f"Random seed                        : "
    f"{BOOTSTRAP_SEED}"
)

print(
    "Resampling unit                    : "
    "Individual frozen evaluable rows"
)

print(
    "Resampling relationship            : "
    "Identical paired draws for full and no-star GES"
)

print(
    "Threshold rule                     : "
    "Fixed P(stable) < 0.50 predicts instability"
)

print(
    "Threshold reselected per replicate : No"
)

print(
    "Threshold optimized                : No"
)

print(
    f"Elapsed bootstrap time             : "
    f"{bootstrap_elapsed_seconds:.2f} seconds"
)

print(
    "Scientific artifact written        : No"
)

print()
print("KEY MODEL-SPECIFIC BOOTSTRAP INTERVALS")
print("-" * 132)

for model_definition in MODEL_DEFINITIONS:

    model_name = model_definition[
        "model"
    ]

    print()
    print(
        model_name
    )

    model_rows = (
        stage6c_decision_threshold_key_bootstrap_summary_df
        .loc[
            stage6c_decision_threshold_key_bootstrap_summary_df[
                "model"
            ]
            == model_name
        ]
    )

    for _, row in model_rows.iterrows():

        print(
            f"  {row['metric']}"
        )

        print(
            f"    Point estimate                 : "
            f"{row['point_estimate']:.8f}"
        )

        print(
            f"    Bootstrap mean                : "
            f"{row['bootstrap_mean']:.8f}"
        )

        print(
            f"    Bootstrap standard error      : "
            f"{row['bootstrap_standard_error']:.8f}"
        )

        print(
            f"    95% percentile interval       : "
            f"[{row['percentile_95_ci_lower']:.8f}, "
            f"{row['percentile_95_ci_upper']:.8f}]"
        )

        if np.isfinite(
            row[
                "null_value"
            ]
        ):

            print(
                f"    Null value                     : "
                f"{row['null_value']:.8f}"
            )

            print(
                f"    Bootstrap support above null  : "
                f"{row['bootstrap_support_above_null']:.6f}"
            )

            print(
                f"    Interval excludes null above  : "
                f"{row['interval_excludes_null_above']}"
            )

            print(
                f"    Interval excludes null below  : "
                f"{row['interval_excludes_null_below']}"
            )

print()
print("PAIRED FULL-GES-MINUS-NO-STAR COMPARISONS")
print("-" * 132)

for row in paired_rows:

    print()
    print(
        row[
            "metric"
        ]
    )

    print(
        f"  Point difference                 : "
        f"{row['point_difference']:+.8f}"
    )

    print(
        f"  Bootstrap mean difference        : "
        f"{row['bootstrap_mean_difference']:+.8f}"
    )

    print(
        f"  Bootstrap standard error         : "
        f"{row['bootstrap_standard_error']:.8f}"
    )

    print(
        f"  95% percentile interval          : "
        f"[{row['percentile_95_ci_lower']:+.8f}, "
        f"{row['percentile_95_ci_upper']:+.8f}]"
    )

    print(
        f"  Probability difference > 0       : "
        f"{row['directional_probability_above_zero']:.6f}"
    )

    print(
        f"  Interval excludes zero above     : "
        f"{row['interval_excludes_zero_above']}"
    )

    print(
        f"  Interval excludes zero below     : "
        f"{row['interval_excludes_zero_below']}"
    )

print()
print("COMPACT REQUIRED-METRIC SUMMARY")
print("-" * 132)

display(
    stage6c_decision_threshold_key_bootstrap_summary_df
)

print()
print("COMPLETE BOOTSTRAP SUMMARY")
print("-" * 132)

display(
    stage6c_decision_threshold_bootstrap_summary_df
)

print()
print("PAIRED-COMPARISON TABLE")
print("-" * 132)

display(
    stage6c_decision_threshold_paired_comparisons_df
)

print()
print("BOOTSTRAP COUNT QC")
print("-" * 132)

display(
    stage6c_decision_threshold_bootstrap_count_qc_df
)

print()
print("RAW REPLICATE DATAFRAME")
print("-" * 132)

print(
    "Shape: "
    f"{stage6c_decision_threshold_bootstrap_replicates_df.shape}"
)

display(
    stage6c_decision_threshold_bootstrap_replicates_df.head(
        10
    )
)

print()
print("CELL DECISION")
print("-" * 132)

print(
    "PASS_STAGE6C_FROZEN_0_50_DECISION_THRESHOLD_"
    "BOOTSTRAP_INFERENCE_COMPLETE"
)

print(
    "The intervals estimate sampling uncertainty for the "
    "unchanged frozen 0.50 threshold."
)

print(
    "Full and no-star GES were evaluated using identical "
    "paired bootstrap row samples."
)

print(
    "No threshold search, optimization, recalibration, score "
    "modification, or scientific-artifact write occurred."
)

Completed 250/2,000 threshold bootstrap replicates (7.3 seconds elapsed)
Completed 500/2,000 threshold bootstrap replicates (12.2 seconds elapsed)
Completed 750/2,000 threshold bootstrap replicates (16.5 seconds elapsed)
Completed 1,000/2,000 threshold bootstrap replicates (23.8 seconds elapsed)
Completed 1,250/2,000 threshold bootstrap replicates (28.1 seconds elapsed)
Completed 1,500/2,000 threshold bootstrap replicates (30.5 seconds elapsed)
Completed 1,750/2,000 threshold bootstrap replicates (33.0 seconds elapsed)
Completed 2,000/2,000 threshold bootstrap replicates (36.3 seconds elapsed)

STAGE 6C STEP 2C — CELL 6C-2C2 — FROZEN 0.50 DECISION-THRESHOLD BOOTSTRAP INFERENCE

BOOTSTRAP DESIGN
------------------------------------------------------------------------------------------------------------------------------------
Evaluable records                   : 66,636
Primary instability events         : 6,485
Observed prevalence                : 0.09731977 (9.731977%)
Bootstrap repli

,model,metric,point_estimate,bootstrap_mean,bootstrap_standard_error,percentile_95_ci_lower,percentile_95_ci_upper,null_value,bootstrap_support_above_null,interval_excludes_null_above,interval_excludes_null_below,bootstrap_replicates,bootstrap_seed,threshold_rule
0,Full GES,balanced_accuracy,0.500355,0.500349,0.001654,0.497158,0.503681,0.5,0.576,False,False,2000,42,P(stable) < 0.50 predicts instability
1,Full GES,f1_score,0.080481,0.080448,0.003617,0.073416,0.087738,NaN,NaN,False,False,2000,42,P(stable) < 0.50 predicts instability
2,Full GES,matthews_correlation,0.000839,0.000824,0.003909,-0.006689,0.008693,0.0,0.576,False,False,2000,42,P(stable) < 0.50 predicts instability
3,Full GES,negative_predictive_value,0.902747,0.902759,0.001185,0.900493,0.905103,NaN,NaN,False,False,2000,42,P(stable) < 0.50 predicts instability
4,Full GES,positive_predictive_value,0.098244,0.098214,0.004471,0.089629,0.107113,NaN,NaN,False,False,2000,42,P(stable) < 0.50 predicts instability
5,Full GES,sensitivity,0.068157,0.068135,0.003143,0.062067,0.074396,NaN,NaN,False,False,2000,42,P(stable) < 0.50 predicts instability
6,Full GES,specificity,0.932553,0.932563,0.001029,0.930585,0.934623,NaN,NaN,False,False,2000,42,P(stable) < 0.50 predicts instability
7,No-star GES,balanced_accuracy,0.512091,0.512105,0.001484,0.509245,0.515085,0.5,1.000,True,False,2000,42,P(stable) < 0.50 predicts instability
8,No-star GES,f1_score,0.083134,0.083146,0.004103,0.075327,0.091250,NaN,NaN,False,False,2000,42,P(stable) < 0.50 predicts instability
9,No-star GES,matthews_correlation,0.039294,0.039346,0.004785,0.030269,0.048853,0.0,1.000,True,False,2000,42,P(stable) < 0.50 predicts instability



COMPLETE BOOTSTRAP SUMMARY
------------------------------------------------------------------------------------------------------------------------------------


,model,metric,point_estimate,bootstrap_mean,bootstrap_standard_error,percentile_95_ci_lower,percentile_95_ci_upper,null_value,bootstrap_support_above_null,interval_excludes_null_above,interval_excludes_null_below,bootstrap_replicates,bootstrap_seed,threshold_rule
0,Full GES,sensitivity,0.068157,0.068135,0.003143,0.062067,0.074396,NaN,NaN,False,False,2000,42,P(stable) < 0.50 predicts instability
1,Full GES,specificity,0.932553,0.932563,0.001029,0.930585,0.934623,NaN,NaN,False,False,2000,42,P(stable) < 0.50 predicts instability
2,Full GES,positive_predictive_value,0.098244,0.098214,0.004471,0.089629,0.107113,NaN,NaN,False,False,2000,42,P(stable) < 0.50 predicts instability
3,Full GES,negative_predictive_value,0.902747,0.902759,0.001185,0.900493,0.905103,NaN,NaN,False,False,2000,42,P(stable) < 0.50 predicts instability
4,Full GES,accuracy,0.848430,0.848449,0.001393,0.845714,0.851162,NaN,NaN,False,False,2000,42,P(stable) < 0.50 predicts instability
5,Full GES,balanced_accuracy,0.500355,0.500349,0.001654,0.497158,0.503681,0.5,0.576,False,False,2000,42,P(stable) < 0.50 predicts instability
6,Full GES,f1_score,0.080481,0.080448,0.003617,0.073416,0.087738,NaN,NaN,False,False,2000,42,P(stable) < 0.50 predicts instability
7,Full GES,matthews_correlation,0.000839,0.000824,0.003909,-0.006689,0.008693,0.0,0.576,False,False,2000,42,P(stable) < 0.50 predicts instability
8,Full GES,predicted_positive_fraction,0.067516,0.067505,0.000977,0.065580,0.069452,NaN,NaN,False,False,2000,42,P(stable) < 0.50 predicts instability
9,Full GES,predicted_negative_fraction,0.932484,0.932495,0.000977,0.930548,0.934420,NaN,NaN,False,False,2000,42,P(stable) < 0.50 predicts instability



PAIRED-COMPARISON TABLE
------------------------------------------------------------------------------------------------------------------------------------


...


BOOTSTRAP COUNT QC
------------------------------------------------------------------------------------------------------------------------------------


,model,locked_true_positive,bootstrap_mean_true_positive,bootstrap_minimum_true_positive,bootstrap_maximum_true_positive,locked_predicted_positive,bootstrap_mean_predicted_positive,bootstrap_minimum_predicted_positive,bootstrap_maximum_predicted_positive
0,Full GES,442,441.7945,379,506,4499,4498.2685,4284,4765
1,No-star GES,365,365.0065,305,432,2296,2294.8030,2139,2456



RAW REPLICATE DATAFRAME
------------------------------------------------------------------------------------------------------------------------------------
Shape: (2000, 45)


,bootstrap_replicate,bootstrap_total_events,bootstrap_prevalence,full_ges_true_positive,full_ges_false_positive,full_ges_true_negative,full_ges_false_negative,full_ges_sensitivity,full_ges_specificity,full_ges_positive_predictive_value,...,no_star_ges_matthews_correlation,no_star_ges_predicted_positive_fraction,no_star_ges_predicted_negative_fraction,no_star_ges_predicted_positive_event_rate,no_star_ges_predicted_negative_event_rate,no_star_ges_event_rate_difference,no_star_ges_relative_event_risk,no_star_ges_enrichment_vs_overall_prevalence,no_star_ges_event_capture_fraction,no_star_ges_youden_j
0,1,6395,0.095969,426,4129,56112,5969,0.066615,0.931459,0.093524,...,0.039322,0.034681,0.965319,0.157075,0.093774,0.063301,1.675040,1.636722,0.056763,0.024426
1,2,6513,0.097740,426,4079,56044,6087,0.065408,0.932156,0.094562,...,0.035696,0.034546,0.965454,0.153779,0.095735,0.058045,1.606306,1.573352,0.054353,0.021953
2,3,6595,0.098971,462,4062,55979,6133,0.070053,0.932346,0.102122,...,0.044444,0.035866,0.964134,0.167782,0.096411,0.071372,1.740289,1.695277,0.060804,0.027676
3,4,6606,0.099136,433,4009,56021,6173,0.065546,0.933217,0.097479,...,0.037674,0.033465,0.966535,0.159641,0.097041,0.062601,1.645097,1.610332,0.053890,0.022673
4,5,6490,0.097395,474,4123,56023,6016,0.073035,0.931450,0.103111,...,0.043005,0.035776,0.964224,0.163591,0.094939,0.068652,1.723119,1.679665,0.060092,0.026940
5,6,6486,0.097335,470,3860,56290,6016,0.072464,0.935827,0.108545,...,0.047315,0.033450,0.966550,0.172723,0.094726,0.077997,1.823403,1.774527,0.059359,0.028702
6,7,6422,0.096374,415,3953,56261,6007,0.064622,0.934351,0.095009,...,0.038521,0.033090,0.966910,0.157823,0.094271,0.063552,1.674136,1.637605,0.054189,0.023349
7,8,6436,0.096584,449,4082,56118,5987,0.069764,0.932193,0.099095,...,0.039467,0.033690,0.966310,0.159020,0.094408,0.064612,1.684399,1.646436,0.055469,0.024107
8,9,6609,0.099181,462,4147,55880,6147,0.069905,0.930914,0.100239,...,0.039359,0.035071,0.964929,0.160890,0.096938,0.063952,1.659725,1.622192,0.056892,0.024223
9,10,6456,0.096885,445,4078,56102,6011,0.068928,0.932237,0.098386,...,0.040374,0.034621,0.965379,0.159948,0.094623,0.065325,1.690372,1.650913,0.057156,0.024953



CELL DECISION
------------------------------------------------------------------------------------------------------------------------------------
PASS_STAGE6C_FROZEN_0_50_DECISION_THRESHOLD_BOOTSTRAP_INFERENCE_COMPLETE
The intervals estimate sampling uncertainty for the unchanged frozen 0.50 threshold.
Full and no-star GES were evaluated using identical paired bootstrap row samples.
No threshold search, optimization, recalibration, score modification, or scientific-artifact write occurred.


In [23]:
# ==================================================================================================
# STAGE 6C — STEP 3A
# CELL 6C-3A1 — REMAINING-COMPARATOR PAIRED AUPRC/AUROC BOOTSTRAP INFERENCE
#
# Purpose:
#   1. Freshly verify the immutable Stage 6B primary-evaluable cohort.
#   2. Validate the five remaining prespecified comparator scores:
#        - conflict;
#        - recency;
#        - submitter support;
#        - classification entropy;
#        - additive risk.
#   3. Recalculate locked AUPRC and AUROC point estimates.
#   4. Run 2,000 paired nonparametric row-bootstrap replicates with seed 42.
#   5. Use identical bootstrap samples for full GES and every comparator.
#   6. Calculate:
#        - model-specific bootstrap confidence intervals;
#        - paired full-GES-minus-comparator differences;
#        - percentile confidence intervals for differences;
#        - directional bootstrap probabilities;
#        - two-sided bootstrap sign probabilities;
#        - Holm-adjusted secondary-comparison probabilities.
#
# Multiplicity policy:
#   - Five remaining comparator comparisons are treated as secondary.
#   - Holm correction is applied separately to:
#        a. the five AUPRC comparisons;
#        b. the five AUROC comparisons.
#   - Previously completed prespecified comparisons with review stars and combined metadata are not
#     mixed into this new secondary-comparison family.
#
# Scientific boundary:
#   - Higher score already means greater predicted future-instability risk.
#   - No comparator direction, formula, weight, threshold, or outcome is modified.
#   - No model is refitted.
#   - No threshold is selected.
#   - No scientific artifact is written.
# ==================================================================================================

from pathlib import Path
import hashlib
import re
import time

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from IPython.display import display
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
)


# --------------------------------------------------------------------------------------------------
# 1. Frozen path and identity
# --------------------------------------------------------------------------------------------------

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/GES_RAG_Temporal_Study"
)

EVALUABLE_COHORT_PATH = (
    PROJECT_ROOT
    / "data_processed"
    / "stage6_temporal_validation"
    / "stage6b_locked_primary_evaluable_cohort_v1.parquet"
)

EVALUABLE_COHORT_SIDECAR_PATH = (
    EVALUABLE_COHORT_PATH.with_name(
        EVALUABLE_COHORT_PATH.name
        + ".sha256"
    )
)

EXPECTED_EVALUABLE_SHA256 = (
    "c6ad50dfc376a746e1830d000dba975101d88763"
    "fd61b8d1983f6300edb6e038"
)

EXPECTED_ROWS = 66_636
EXPECTED_COLUMNS = 79
EXPECTED_EVENTS = 6_485
EXPECTED_NEGATIVES = 60_151

BOOTSTRAP_REPLICATES = 2_000
BOOTSTRAP_SEED = 42

OUTCOME_COL = (
    "primary_future_instability"
)


# --------------------------------------------------------------------------------------------------
# 2. Frozen comparator definitions
# --------------------------------------------------------------------------------------------------

MODEL_DEFINITIONS = [
    {
        "model": "Full GES",
        "short_name": "full_ges",
        "score_column": (
            "full_ges_instability_risk_t0"
        ),
        "role": "Reference model",
    },
    {
        "model": "Conflict",
        "short_name": "conflict",
        "score_column": (
            "conflict_instability_risk"
        ),
        "role": "Remaining comparator",
    },
    {
        "model": "Recency",
        "short_name": "recency",
        "score_column": (
            "recency_instability_risk"
        ),
        "role": "Remaining comparator",
    },
    {
        "model": "Submitter support",
        "short_name": "submitter",
        "score_column": (
            "submitter_instability_risk"
        ),
        "role": "Remaining comparator",
    },
    {
        "model": "Classification entropy",
        "short_name": "entropy",
        "score_column": (
            "entropy_instability_risk"
        ),
        "role": "Remaining comparator",
    },
    {
        "model": "Additive risk",
        "short_name": "additive",
        "score_column": (
            "additive_instability_risk"
        ),
        "role": "Remaining comparator",
    },
]

REFERENCE_MODEL_INDEX = 0
NUMBER_OF_MODELS = len(
    MODEL_DEFINITIONS
)

assert NUMBER_OF_MODELS == 6


# --------------------------------------------------------------------------------------------------
# 3. Cryptographic helpers
# --------------------------------------------------------------------------------------------------

def sha256_file(
    path: Path,
    chunk_size: int = 8 * 1024 * 1024,
) -> str:
    """Calculate a file SHA-256 checksum without modifying the file."""

    digest = hashlib.sha256()

    with path.open("rb") as file_handle:

        while True:

            block = file_handle.read(
                chunk_size
            )

            if not block:
                break

            digest.update(
                block
            )

    return digest.hexdigest()


def read_sha256_sidecar(
    path: Path,
) -> str:
    """Read the first valid SHA-256 value contained in a sidecar."""

    sidecar_text = path.read_text(
        encoding="utf-8"
    ).strip()

    match = re.search(
        r"(?i)\b[0-9a-f]{64}\b",
        sidecar_text,
    )

    if match is None:

        raise AssertionError(
            "No valid SHA-256 value was found in sidecar:\n"
            f"{path}"
        )

    return match.group(0).lower()


def holm_adjust(
    p_values,
):
    """
    Apply the Holm step-down family-wise-error correction.

    Returns adjusted probabilities in the original input order.
    """

    p_values_array = np.asarray(
        p_values,
        dtype=np.float64,
    )

    number_of_tests = len(
        p_values_array
    )

    assert number_of_tests > 0

    order = np.argsort(
        p_values_array,
        kind="mergesort",
    )

    ordered_p_values = (
        p_values_array[
            order
        ]
    )

    raw_adjusted_ordered = (
        (
            number_of_tests
            - np.arange(
                number_of_tests
            )
        )
        * ordered_p_values
    )

    monotonic_adjusted_ordered = (
        np.maximum.accumulate(
            raw_adjusted_ordered
        )
    )

    monotonic_adjusted_ordered = np.minimum(
        monotonic_adjusted_ordered,
        1.0,
    )

    adjusted_original_order = np.empty(
        number_of_tests,
        dtype=np.float64,
    )

    adjusted_original_order[
        order
    ] = monotonic_adjusted_ordered

    return adjusted_original_order


# --------------------------------------------------------------------------------------------------
# 4. Fresh frozen-input verification
# --------------------------------------------------------------------------------------------------

assert EVALUABLE_COHORT_PATH.is_file(), (
    "Missing frozen Stage 6B primary-evaluable cohort:\n"
    f"{EVALUABLE_COHORT_PATH}"
)

assert EVALUABLE_COHORT_SIDECAR_PATH.is_file(), (
    "Missing Stage 6B evaluable-cohort sidecar:\n"
    f"{EVALUABLE_COHORT_SIDECAR_PATH}"
)

observed_evaluable_sha256 = sha256_file(
    EVALUABLE_COHORT_PATH
)

sidecar_evaluable_sha256 = read_sha256_sidecar(
    EVALUABLE_COHORT_SIDECAR_PATH
)

assert (
    observed_evaluable_sha256
    == EXPECTED_EVALUABLE_SHA256
), (
    "The Stage 6B primary-evaluable cohort checksum "
    "does not match the accepted frozen checksum."
)

assert (
    sidecar_evaluable_sha256
    == EXPECTED_EVALUABLE_SHA256
), (
    "The Stage 6B evaluable-cohort sidecar does not "
    "contain the accepted frozen checksum."
)

parquet_file = pq.ParquetFile(
    EVALUABLE_COHORT_PATH
)

parquet_metadata = parquet_file.metadata
parquet_columns = parquet_file.schema_arrow.names

assert parquet_metadata.num_rows == EXPECTED_ROWS

assert parquet_metadata.num_columns == EXPECTED_COLUMNS


# --------------------------------------------------------------------------------------------------
# 5. Identify frozen key and verify required columns
# --------------------------------------------------------------------------------------------------

key_candidates = (
    "t0_rcv_accession",
    "rcv_accession_t0",
    "rcv_accession",
)

KEY_COL = next(
    (
        candidate
        for candidate in key_candidates
        if candidate in parquet_columns
    ),
    None,
)

assert KEY_COL is not None, (
    "Could not identify the frozen T0 RCV-accession key."
)

score_columns = [
    model_definition[
        "score_column"
    ]
    for model_definition in MODEL_DEFINITIONS
]

required_columns = [
    KEY_COL,
    OUTCOME_COL,
    *score_columns,
]

missing_columns = [
    column
    for column in required_columns
    if column not in parquet_columns
]

assert not missing_columns, (
    "Required comparator-inference columns are missing:\n"
    f"{missing_columns}"
)


# --------------------------------------------------------------------------------------------------
# 6. Load only the required frozen fields
# --------------------------------------------------------------------------------------------------

comparator_source_df = pd.read_parquet(
    EVALUABLE_COHORT_PATH,
    columns=required_columns,
)

assert comparator_source_df.shape == (
    EXPECTED_ROWS,
    len(required_columns),
)

assert comparator_source_df[
    KEY_COL
].notna().all()

assert comparator_source_df[
    KEY_COL
].is_unique

comparator_source_df[
    OUTCOME_COL
] = pd.to_numeric(
    comparator_source_df[
        OUTCOME_COL
    ],
    errors="raise",
).astype(
    "int8"
)

assert set(
    comparator_source_df[
        OUTCOME_COL
    ].unique().tolist()
) == {
    0,
    1,
}

observed_events = int(
    comparator_source_df[
        OUTCOME_COL
    ].sum()
)

observed_negatives = int(
    EXPECTED_ROWS
    - observed_events
)

observed_prevalence = (
    observed_events
    / EXPECTED_ROWS
)

assert observed_events == EXPECTED_EVENTS

assert observed_negatives == EXPECTED_NEGATIVES


# --------------------------------------------------------------------------------------------------
# 7. Validate all six score columns
# --------------------------------------------------------------------------------------------------

score_validation_rows = []

for model_definition in MODEL_DEFINITIONS:

    model_name = model_definition[
        "model"
    ]

    score_column = model_definition[
        "score_column"
    ]

    comparator_source_df[
        score_column
    ] = pd.to_numeric(
        comparator_source_df[
            score_column
        ],
        errors="raise",
    ).astype(
        "float64"
    )

    score_values = comparator_source_df[
        score_column
    ].to_numpy()

    missing_count = int(
        comparator_source_df[
            score_column
        ].isna().sum()
    )

    nonfinite_count = int(
        np.sum(
            ~np.isfinite(
                score_values
            )
        )
    )

    minimum_score = float(
        np.min(
            score_values
        )
    )

    maximum_score = float(
        np.max(
            score_values
        )
    )

    unique_score_count = int(
        comparator_source_df[
            score_column
        ].nunique(
            dropna=False
        )
    )

    within_unit_interval = bool(
        comparator_source_df[
            score_column
        ].between(
            0.0,
            1.0,
            inclusive="both",
        ).all()
    )

    assert missing_count == 0, (
        f"{model_name} contains missing scores."
    )

    assert nonfinite_count == 0, (
        f"{model_name} contains nonfinite scores."
    )

    assert within_unit_interval, (
        f"{model_name} contains a score outside [0,1]."
    )

    assert unique_score_count >= 2, (
        f"{model_name} does not contain at least two "
        "distinct score values."
    )

    score_validation_rows.append({
        "model": model_name,
        "role": model_definition[
            "role"
        ],
        "score_column": score_column,
        "rows": EXPECTED_ROWS,
        "missing": missing_count,
        "nonfinite": nonfinite_count,
        "minimum": minimum_score,
        "maximum": maximum_score,
        "unique_values": unique_score_count,
        "within_unit_interval": (
            within_unit_interval
        ),
        "score_direction": (
            "Higher = greater predicted instability risk"
        ),
    })


stage6c_remaining_comparator_score_validation_df = (
    pd.DataFrame(
        score_validation_rows
    )
)


# --------------------------------------------------------------------------------------------------
# 8. Prepare immutable numeric arrays
# --------------------------------------------------------------------------------------------------

outcome_array = comparator_source_df[
    OUTCOME_COL
].to_numpy(
    dtype=np.int8
)

score_matrix = comparator_source_df[
    score_columns
].to_numpy(
    dtype=np.float64
)

assert score_matrix.shape == (
    EXPECTED_ROWS,
    NUMBER_OF_MODELS,
)


# --------------------------------------------------------------------------------------------------
# 9. Locked point estimates
# --------------------------------------------------------------------------------------------------

point_auprc = np.empty(
    NUMBER_OF_MODELS,
    dtype=np.float64,
)

point_auroc = np.empty(
    NUMBER_OF_MODELS,
    dtype=np.float64,
)

point_rows = []

for model_index, model_definition in enumerate(
    MODEL_DEFINITIONS
):

    model_scores = score_matrix[
        :,
        model_index,
    ]

    model_auprc = float(
        average_precision_score(
            outcome_array,
            model_scores,
        )
    )

    model_auroc = float(
        roc_auc_score(
            outcome_array,
            model_scores,
        )
    )

    point_auprc[
        model_index
    ] = model_auprc

    point_auroc[
        model_index
    ] = model_auroc

    point_rows.append({
        "model": model_definition[
            "model"
        ],
        "role": model_definition[
            "role"
        ],
        "score_column": model_definition[
            "score_column"
        ],
        "auprc": model_auprc,
        "auprc_lift_vs_prevalence": (
            model_auprc
            / observed_prevalence
        ),
        "auroc": model_auroc,
        "delta_auprc_vs_full_ges": (
            model_auprc
            - point_auprc[
                REFERENCE_MODEL_INDEX
            ]
        ),
        "delta_auroc_vs_full_ges": (
            model_auroc
            - point_auroc[
                REFERENCE_MODEL_INDEX
            ]
        ),
    })


stage6c_remaining_comparator_point_estimates_df = (
    pd.DataFrame(
        point_rows
    )
)

# Recalculate reference deltas after all point estimates are available.
stage6c_remaining_comparator_point_estimates_df[
    "delta_auprc_vs_full_ges"
] = (
    stage6c_remaining_comparator_point_estimates_df[
        "auprc"
    ]
    - point_auprc[
        REFERENCE_MODEL_INDEX
    ]
)

stage6c_remaining_comparator_point_estimates_df[
    "delta_auroc_vs_full_ges"
] = (
    stage6c_remaining_comparator_point_estimates_df[
        "auroc"
    ]
    - point_auroc[
        REFERENCE_MODEL_INDEX
    ]
)

# Reconcile full-GES point estimates with the previously recorded locked results.
assert np.isclose(
    point_auprc[
        REFERENCE_MODEL_INDEX
    ],
    0.112444,
    rtol=0.0,
    atol=1e-6,
), (
    "Full-GES AUPRC does not reconcile with the "
    "previous locked Stage 6C result."
)

assert np.isclose(
    point_auroc[
        REFERENCE_MODEL_INDEX
    ],
    0.535812,
    rtol=0.0,
    atol=1e-6,
), (
    "Full-GES AUROC does not reconcile with the "
    "previous locked Stage 6C result."
)


# --------------------------------------------------------------------------------------------------
# 10. Preallocate paired-bootstrap result arrays
# --------------------------------------------------------------------------------------------------

bootstrap_auprc = np.empty(
    (
        BOOTSTRAP_REPLICATES,
        NUMBER_OF_MODELS,
    ),
    dtype=np.float64,
)

bootstrap_auroc = np.empty(
    (
        BOOTSTRAP_REPLICATES,
        NUMBER_OF_MODELS,
    ),
    dtype=np.float64,
)

bootstrap_event_counts = np.empty(
    BOOTSTRAP_REPLICATES,
    dtype=np.int32,
)


# --------------------------------------------------------------------------------------------------
# 11. Run 2,000 paired nonparametric row-bootstrap replicates
#
# The same resampled row indexes are used for all six scores and both metrics in each replicate.
# --------------------------------------------------------------------------------------------------

rng = np.random.default_rng(
    BOOTSTRAP_SEED
)

bootstrap_start_time = time.time()

for bootstrap_index in range(
    BOOTSTRAP_REPLICATES
):

    sampled_indexes = rng.integers(
        low=0,
        high=EXPECTED_ROWS,
        size=EXPECTED_ROWS,
        dtype=np.int32,
    )

    sampled_outcomes = outcome_array[
        sampled_indexes
    ]

    sampled_event_count = int(
        sampled_outcomes.sum()
    )

    sampled_negative_count = int(
        EXPECTED_ROWS
        - sampled_event_count
    )

    assert sampled_event_count > 0

    assert sampled_negative_count > 0

    bootstrap_event_counts[
        bootstrap_index
    ] = sampled_event_count

    for model_index in range(
        NUMBER_OF_MODELS
    ):

        sampled_scores = score_matrix[
            sampled_indexes,
            model_index,
        ]

        bootstrap_auprc[
            bootstrap_index,
            model_index,
        ] = average_precision_score(
            sampled_outcomes,
            sampled_scores,
        )

        bootstrap_auroc[
            bootstrap_index,
            model_index,
        ] = roc_auc_score(
            sampled_outcomes,
            sampled_scores,
        )

    completed_replicates = (
        bootstrap_index
        + 1
    )

    if (
        completed_replicates % 100 == 0
        or completed_replicates
        == BOOTSTRAP_REPLICATES
    ):

        elapsed_seconds = (
            time.time()
            - bootstrap_start_time
        )

        print(
            f"Completed "
            f"{completed_replicates:,}/"
            f"{BOOTSTRAP_REPLICATES:,} "
            f"remaining-comparator bootstrap replicates "
            f"({elapsed_seconds:.1f} seconds elapsed)"
        )


bootstrap_elapsed_seconds = (
    time.time()
    - bootstrap_start_time
)


# --------------------------------------------------------------------------------------------------
# 12. Bootstrap result QC
# --------------------------------------------------------------------------------------------------

assert np.isfinite(
    bootstrap_auprc
).all(), (
    "AUPRC bootstrap array contains nonfinite values."
)

assert np.isfinite(
    bootstrap_auroc
).all(), (
    "AUROC bootstrap array contains nonfinite values."
)

assert (
    bootstrap_auprc
    >= 0.0
).all()

assert (
    bootstrap_auprc
    <= 1.0
).all()

assert (
    bootstrap_auroc
    >= 0.0
).all()

assert (
    bootstrap_auroc
    <= 1.0
).all()

assert (
    bootstrap_event_counts
    > 0
).all()

assert (
    bootstrap_event_counts
    < EXPECTED_ROWS
).all()


# --------------------------------------------------------------------------------------------------
# 13. Model-specific bootstrap summaries
# --------------------------------------------------------------------------------------------------

model_summary_rows = []

metric_definitions = [
    {
        "metric": "AUPRC",
        "point_values": point_auprc,
        "bootstrap_values": bootstrap_auprc,
    },
    {
        "metric": "AUROC",
        "point_values": point_auroc,
        "bootstrap_values": bootstrap_auroc,
    },
]

for metric_definition in metric_definitions:

    metric_name = metric_definition[
        "metric"
    ]

    point_values = metric_definition[
        "point_values"
    ]

    bootstrap_values = metric_definition[
        "bootstrap_values"
    ]

    for model_index, model_definition in enumerate(
        MODEL_DEFINITIONS
    ):

        model_bootstrap_values = bootstrap_values[
            :,
            model_index,
        ]

        interval_lower, interval_upper = np.quantile(
            model_bootstrap_values,
            [
                0.025,
                0.975,
            ],
        )

        model_summary_rows.append({
            "model": model_definition[
                "model"
            ],
            "role": model_definition[
                "role"
            ],
            "metric": metric_name,
            "point_estimate": float(
                point_values[
                    model_index
                ]
            ),
            "bootstrap_mean": float(
                np.mean(
                    model_bootstrap_values
                )
            ),
            "bootstrap_standard_error": float(
                np.std(
                    model_bootstrap_values,
                    ddof=1,
                )
            ),
            "percentile_95_ci_lower": float(
                interval_lower
            ),
            "percentile_95_ci_upper": float(
                interval_upper
            ),
            "bootstrap_replicates": (
                BOOTSTRAP_REPLICATES
            ),
            "bootstrap_seed": (
                BOOTSTRAP_SEED
            ),
            "paired_resamples_across_models": True,
        })


stage6c_remaining_comparator_bootstrap_summary_df = (
    pd.DataFrame(
        model_summary_rows
    )
)


# --------------------------------------------------------------------------------------------------
# 14. Paired full-GES-minus-comparator differences
# --------------------------------------------------------------------------------------------------

paired_rows = []

for metric_definition in metric_definitions:

    metric_name = metric_definition[
        "metric"
    ]

    point_values = metric_definition[
        "point_values"
    ]

    bootstrap_values = metric_definition[
        "bootstrap_values"
    ]

    reference_bootstrap_values = bootstrap_values[
        :,
        REFERENCE_MODEL_INDEX,
    ]

    reference_point_value = float(
        point_values[
            REFERENCE_MODEL_INDEX
        ]
    )

    for comparator_index in range(
        1,
        NUMBER_OF_MODELS,
    ):

        comparator_definition = MODEL_DEFINITIONS[
            comparator_index
        ]

        paired_difference = (
            reference_bootstrap_values
            - bootstrap_values[
                :,
                comparator_index,
            ]
        )

        point_difference = (
            reference_point_value
            - float(
                point_values[
                    comparator_index
                ]
            )
        )

        interval_lower, interval_upper = np.quantile(
            paired_difference,
            [
                0.025,
                0.975,
            ],
        )

        number_above_zero = int(
            np.sum(
                paired_difference
                > 0.0
            )
        )

        number_below_zero = int(
            np.sum(
                paired_difference
                < 0.0
            )
        )

        number_equal_zero = int(
            np.sum(
                paired_difference
                == 0.0
            )
        )

        directional_probability_above_zero = float(
            (
                number_above_zero
                + 0.5
                * number_equal_zero
            )
            / BOOTSTRAP_REPLICATES
        )

        # Add-one finite-replicate correction.
        probability_nonpositive = (
            number_below_zero
            + number_equal_zero
            + 1
        ) / (
            BOOTSTRAP_REPLICATES
            + 1
        )

        probability_nonnegative = (
            number_above_zero
            + number_equal_zero
            + 1
        ) / (
            BOOTSTRAP_REPLICATES
            + 1
        )

        two_sided_bootstrap_sign_probability = min(
            1.0,
            2.0
            * min(
                probability_nonpositive,
                probability_nonnegative,
            ),
        )

        paired_rows.append({
            "metric": metric_name,
            "reference_model": "Full GES",
            "comparator_model": (
                comparator_definition[
                    "model"
                ]
            ),
            "comparison": (
                "Full GES minus "
                + comparator_definition[
                    "model"
                ]
            ),
            "point_reference": (
                reference_point_value
            ),
            "point_comparator": float(
                point_values[
                    comparator_index
                ]
            ),
            "point_difference": float(
                point_difference
            ),
            "bootstrap_mean_difference": float(
                np.mean(
                    paired_difference
                )
            ),
            "bootstrap_standard_error": float(
                np.std(
                    paired_difference,
                    ddof=1,
                )
            ),
            "percentile_95_ci_lower": float(
                interval_lower
            ),
            "percentile_95_ci_upper": float(
                interval_upper
            ),
            "directional_probability_full_above_comparator": (
                directional_probability_above_zero
            ),
            "two_sided_bootstrap_sign_probability": float(
                two_sided_bootstrap_sign_probability
            ),
            "interval_excludes_zero_above": bool(
                interval_lower > 0.0
            ),
            "interval_excludes_zero_below": bool(
                interval_upper < 0.0
            ),
            "bootstrap_replicates": (
                BOOTSTRAP_REPLICATES
            ),
            "bootstrap_seed": (
                BOOTSTRAP_SEED
            ),
            "paired_resamples": True,
        })


stage6c_remaining_comparator_paired_comparisons_df = (
    pd.DataFrame(
        paired_rows
    )
)


# --------------------------------------------------------------------------------------------------
# 15. Apply Holm correction separately within the AUPRC and AUROC comparison families
# --------------------------------------------------------------------------------------------------

stage6c_remaining_comparator_paired_comparisons_df[
    "holm_adjusted_probability"
] = np.nan

for metric_name in (
    "AUPRC",
    "AUROC",
):

    metric_mask = (
        stage6c_remaining_comparator_paired_comparisons_df[
            "metric"
        ]
        == metric_name
    )

    metric_probabilities = (
        stage6c_remaining_comparator_paired_comparisons_df
        .loc[
            metric_mask,
            "two_sided_bootstrap_sign_probability",
        ]
        .to_numpy(
            dtype=np.float64
        )
    )

    assert len(
        metric_probabilities
    ) == 5

    adjusted_probabilities = holm_adjust(
        metric_probabilities
    )

    stage6c_remaining_comparator_paired_comparisons_df.loc[
        metric_mask,
        "holm_adjusted_probability",
    ] = adjusted_probabilities


stage6c_remaining_comparator_paired_comparisons_df[
    "holm_reject_at_0_05"
] = (
    stage6c_remaining_comparator_paired_comparisons_df[
        "holm_adjusted_probability"
    ]
    < 0.05
)

stage6c_remaining_comparator_paired_comparisons_df[
    "multiple_comparison_family"
] = (
    stage6c_remaining_comparator_paired_comparisons_df[
        "metric"
    ]
    + " across five remaining comparators"
)


# --------------------------------------------------------------------------------------------------
# 16. Raw bootstrap replicate table
# --------------------------------------------------------------------------------------------------

replicate_data = {
    "bootstrap_replicate": np.arange(
        1,
        BOOTSTRAP_REPLICATES + 1,
        dtype=np.int32,
    ),
    "bootstrap_event_count": (
        bootstrap_event_counts
    ),
    "bootstrap_prevalence": (
        bootstrap_event_counts
        / EXPECTED_ROWS
    ),
}

for model_index, model_definition in enumerate(
    MODEL_DEFINITIONS
):

    short_name = model_definition[
        "short_name"
    ]

    replicate_data[
        f"{short_name}_auprc"
    ] = bootstrap_auprc[
        :,
        model_index,
    ]

    replicate_data[
        f"{short_name}_auroc"
    ] = bootstrap_auroc[
        :,
        model_index,
    ]


stage6c_remaining_comparator_bootstrap_replicates_df = (
    pd.DataFrame(
        replicate_data
    )
)

assert (
    stage6c_remaining_comparator_bootstrap_replicates_df.shape
    == (
        BOOTSTRAP_REPLICATES,
        3
        + 2
        * NUMBER_OF_MODELS,
    )
)


# --------------------------------------------------------------------------------------------------
# 17. Compact inference tables
# --------------------------------------------------------------------------------------------------

stage6c_remaining_comparator_auprc_comparisons_df = (
    stage6c_remaining_comparator_paired_comparisons_df
    .loc[
        stage6c_remaining_comparator_paired_comparisons_df[
            "metric"
        ]
        == "AUPRC"
    ]
    .copy()
    .sort_values(
        by="point_difference",
        ascending=False,
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)

stage6c_remaining_comparator_auroc_comparisons_df = (
    stage6c_remaining_comparator_paired_comparisons_df
    .loc[
        stage6c_remaining_comparator_paired_comparisons_df[
            "metric"
        ]
        == "AUROC"
    ]
    .copy()
    .sort_values(
        by="point_difference",
        ascending=False,
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)


# --------------------------------------------------------------------------------------------------
# 18. Final QC
# --------------------------------------------------------------------------------------------------

assert len(
    stage6c_remaining_comparator_score_validation_df
) == NUMBER_OF_MODELS

assert len(
    stage6c_remaining_comparator_point_estimates_df
) == NUMBER_OF_MODELS

assert len(
    stage6c_remaining_comparator_bootstrap_summary_df
) == (
    NUMBER_OF_MODELS
    * 2
)

assert len(
    stage6c_remaining_comparator_paired_comparisons_df
) == 10

assert (
    stage6c_remaining_comparator_paired_comparisons_df[
        "paired_resamples"
    ]
    .all()
)

assert (
    stage6c_remaining_comparator_paired_comparisons_df[
        "holm_adjusted_probability"
    ]
    .between(
        0.0,
        1.0,
        inclusive="both",
    )
    .all()
)

assert (
    stage6c_remaining_comparator_paired_comparisons_df
    .groupby(
        "metric"
    )
    .size()
    .to_dict()
    == {
        "AUPRC": 5,
        "AUROC": 5,
    }
)


# --------------------------------------------------------------------------------------------------
# 19. Display results
# --------------------------------------------------------------------------------------------------

print()
print("=" * 136)
print(
    "STAGE 6C STEP 3A — CELL 6C-3A1 — "
    "REMAINING-COMPARATOR PAIRED AUPRC/AUROC BOOTSTRAP INFERENCE"
)
print("=" * 136)

print()
print("FROZEN INPUT VERIFICATION")
print("-" * 136)

print(
    "Primary-evaluable cohort SHA-256 : "
    f"PASS ({observed_evaluable_sha256})"
)

print(
    "Parquet dimensions                : "
    f"PASS ({EXPECTED_ROWS:,} × {EXPECTED_COLUMNS})"
)

print(
    "Unique evaluable RCV keys         : "
    f"PASS ({comparator_source_df[KEY_COL].nunique():,})"
)

print(
    "Primary instability events        : "
    f"PASS ({observed_events:,})"
)

print(
    "Primary instability negatives     : "
    f"PASS ({observed_negatives:,})"
)

print(
    "Observed event prevalence         : "
    f"{observed_prevalence:.8f} "
    f"({observed_prevalence * 100:.6f}%)"
)

print()
print("BOOTSTRAP AND MULTIPLICITY DESIGN")
print("-" * 136)

print(
    f"Bootstrap replicates              : "
    f"{BOOTSTRAP_REPLICATES:,}"
)

print(
    f"Random seed                      : "
    f"{BOOTSTRAP_SEED}"
)

print(
    "Resampling unit                  : "
    "Individual frozen evaluable rows"
)

print(
    "Paired resampling                : "
    "Identical rows across all six scores and both metrics"
)

print(
    "Reference model                  : "
    "Full GES"
)

print(
    "Secondary comparison family      : "
    "Five remaining comparators"
)

print(
    "Multiplicity correction          : "
    "Holm, separately for AUPRC and AUROC"
)

print(
    "Score direction changed          : No"
)

print(
    "Threshold or weight optimized    : No"
)

print(
    f"Elapsed bootstrap time           : "
    f"{bootstrap_elapsed_seconds:.2f} seconds"
)

print(
    "Scientific artifact written      : No"
)

print()
print("SCORE VALIDATION")
print("-" * 136)

display(
    stage6c_remaining_comparator_score_validation_df
)

print()
print("LOCKED DISCRIMINATION POINT ESTIMATES")
print("-" * 136)

display(
    stage6c_remaining_comparator_point_estimates_df
    .sort_values(
        by="auprc",
        ascending=False,
        kind="mergesort",
    )
    .reset_index(
        drop=True
    )
)

print()
print("MODEL-SPECIFIC BOOTSTRAP INTERVALS")
print("-" * 136)

display(
    stage6c_remaining_comparator_bootstrap_summary_df
)

print()
print("PAIRED AUPRC COMPARISONS — FULL GES MINUS COMPARATOR")
print("-" * 136)

for _, row in (
    stage6c_remaining_comparator_auprc_comparisons_df
    .iterrows()
):

    print()
    print(
        row[
            "comparison"
        ]
    )

    print(
        f"  Point difference                : "
        f"{row['point_difference']:+.8f}"
    )

    print(
        f"  Bootstrap mean difference       : "
        f"{row['bootstrap_mean_difference']:+.8f}"
    )

    print(
        f"  Bootstrap standard error        : "
        f"{row['bootstrap_standard_error']:.8f}"
    )

    print(
        f"  95% percentile interval         : "
        f"[{row['percentile_95_ci_lower']:+.8f}, "
        f"{row['percentile_95_ci_upper']:+.8f}]"
    )

    print(
        f"  P(Full GES > comparator)        : "
        f"{row['directional_probability_full_above_comparator']:.6f}"
    )

    print(
        f"  Two-sided bootstrap probability : "
        f"{row['two_sided_bootstrap_sign_probability']:.8f}"
    )

    print(
        f"  Holm-adjusted probability       : "
        f"{row['holm_adjusted_probability']:.8f}"
    )

    print(
        f"  Holm reject at 0.05             : "
        f"{row['holm_reject_at_0_05']}"
    )

print()
print("PAIRED AUROC COMPARISONS — FULL GES MINUS COMPARATOR")
print("-" * 136)

for _, row in (
    stage6c_remaining_comparator_auroc_comparisons_df
    .iterrows()
):

    print()
    print(
        row[
            "comparison"
        ]
    )

    print(
        f"  Point difference                : "
        f"{row['point_difference']:+.8f}"
    )

    print(
        f"  Bootstrap mean difference       : "
        f"{row['bootstrap_mean_difference']:+.8f}"
    )

    print(
        f"  Bootstrap standard error        : "
        f"{row['bootstrap_standard_error']:.8f}"
    )

    print(
        f"  95% percentile interval         : "
        f"[{row['percentile_95_ci_lower']:+.8f}, "
        f"{row['percentile_95_ci_upper']:+.8f}]"
    )

    print(
        f"  P(Full GES > comparator)        : "
        f"{row['directional_probability_full_above_comparator']:.6f}"
    )

    print(
        f"  Two-sided bootstrap probability : "
        f"{row['two_sided_bootstrap_sign_probability']:.8f}"
    )

    print(
        f"  Holm-adjusted probability       : "
        f"{row['holm_adjusted_probability']:.8f}"
    )

    print(
        f"  Holm reject at 0.05             : "
        f"{row['holm_reject_at_0_05']}"
    )

print()
print("COMPLETE PAIRED-COMPARISON TABLE")
print("-" * 136)

display(
    stage6c_remaining_comparator_paired_comparisons_df
)

print()
print("RAW BOOTSTRAP REPLICATE TABLE")
print("-" * 136)

print(
    "Shape: "
    f"{stage6c_remaining_comparator_bootstrap_replicates_df.shape}"
)

display(
    stage6c_remaining_comparator_bootstrap_replicates_df.head(
        10
    )
)

print()
print("CELL DECISION")
print("-" * 136)

print(
    "PASS_STAGE6C_REMAINING_COMPARATOR_PAIRED_"
    "AUPRC_AUROC_BOOTSTRAP_INFERENCE_COMPLETE"
)

print(
    "Paired uncertainty inference is complete for conflict, recency, "
    "submitter support, classification entropy, and additive-risk comparators."
)

print(
    "Holm correction was applied separately to the five secondary AUPRC "
    "comparisons and the five secondary AUROC comparisons."
)

print(
    "No score, outcome, comparator formula, direction, weight, threshold, "
    "model, or frozen scientific artifact was modified."
)

Completed 100/2,000 remaining-comparator bootstrap replicates (25.2 seconds elapsed)
Completed 200/2,000 remaining-comparator bootstrap replicates (44.3 seconds elapsed)
Completed 300/2,000 remaining-comparator bootstrap replicates (58.9 seconds elapsed)
Completed 400/2,000 remaining-comparator bootstrap replicates (73.5 seconds elapsed)
Completed 500/2,000 remaining-comparator bootstrap replicates (88.2 seconds elapsed)
Completed 600/2,000 remaining-comparator bootstrap replicates (102.8 seconds elapsed)
Completed 700/2,000 remaining-comparator bootstrap replicates (117.4 seconds elapsed)
Completed 800/2,000 remaining-comparator bootstrap replicates (132.7 seconds elapsed)
Completed 900/2,000 remaining-comparator bootstrap replicates (147.2 seconds elapsed)
Completed 1,000/2,000 remaining-comparator bootstrap replicates (161.8 seconds elapsed)
Completed 1,100/2,000 remaining-comparator bootstrap replicates (176.4 seconds elapsed)
Completed 1,200/2,000 remaining-comparator bootstrap re

,model,role,score_column,rows,missing,nonfinite,minimum,maximum,unique_values,within_unit_interval,score_direction
0,Full GES,Reference model,full_ges_instability_risk_t0,66636,0,0,3.385514e-12,1.0,9504,True,Higher = greater predicted instability risk
1,Conflict,Remaining comparator,conflict_instability_risk,66636,0,0,0.000000e+00,1.0,2,True,Higher = greater predicted instability risk
2,Recency,Remaining comparator,recency_instability_risk,66636,0,0,7.607455e-04,1.0,3783,True,Higher = greater predicted instability risk
3,Submitter support,Remaining comparator,submitter_instability_risk,66636,0,0,0.000000e+00,1.0,22,True,Higher = greater predicted instability risk
4,Classification entropy,Remaining comparator,entropy_instability_risk,66636,0,0,0.000000e+00,1.0,39,True,Higher = greater predicted instability risk
5,Additive risk,Remaining comparator,additive_instability_risk,66636,0,0,0.000000e+00,0.5,4,True,Higher = greater predicted instability risk



LOCKED DISCRIMINATION POINT ESTIMATES
----------------------------------------------------------------------------------------------------------------------------------------


,model,role,score_column,auprc,auprc_lift_vs_prevalence,auroc,delta_auprc_vs_full_ges,delta_auroc_vs_full_ges
0,Full GES,Reference model,full_ges_instability_risk_t0,0.112444,1.155408,0.535812,0.000000,0.000000
1,Classification entropy,Remaining comparator,entropy_instability_risk,0.106189,1.091139,0.518451,-0.006255,-0.017361
2,Conflict,Remaining comparator,conflict_instability_risk,0.102047,1.048575,0.513057,-0.010397,-0.022755
3,Additive risk,Remaining comparator,additive_instability_risk,0.099511,1.022519,0.483180,-0.012933,-0.052633
4,Submitter support,Remaining comparator,submitter_instability_risk,0.092424,0.949689,0.471184,-0.020021,-0.064629
5,Recency,Remaining comparator,recency_instability_risk,0.082721,0.849995,0.437694,-0.029723,-0.098118



MODEL-SPECIFIC BOOTSTRAP INTERVALS
----------------------------------------------------------------------------------------------------------------------------------------


...


PAIRED AUPRC COMPARISONS — FULL GES MINUS COMPARATOR
----------------------------------------------------------------------------------------------------------------------------------------

Full GES minus Recency
  Point difference                : +0.02972278
  Bootstrap mean difference       : +0.02983578
  Bootstrap standard error        : 0.00177875
  95% percentile interval         : [+0.02652808, +0.03331336]
  P(Full GES > comparator)        : 1.000000
  Two-sided bootstrap probability : 0.00099950
  Holm-adjusted probability       : 0.00499750
  Holm reject at 0.05             : True

Full GES minus Submitter support
  Point difference                : +0.02002054
  Bootstrap mean difference       : +0.02021472
  Bootstrap standard error        : 0.00206072
  95% percentile interval         : [+0.01633368, +0.02435307]
  P(Full GES > comparator)        : 1.000000
  Two-sided bootstrap probability : 0.00099950
  Holm-adjusted probability       : 0.00499750
  Holm reject at 0.0

,metric,reference_model,comparator_model,comparison,point_reference,point_comparator,point_difference,bootstrap_mean_difference,bootstrap_standard_error,percentile_95_ci_lower,...,directional_probability_full_above_comparator,two_sided_bootstrap_sign_probability,interval_excludes_zero_above,interval_excludes_zero_below,bootstrap_replicates,bootstrap_seed,paired_resamples,holm_adjusted_probability,holm_reject_at_0_05,multiple_comparison_family
0,AUPRC,Full GES,Conflict,Full GES minus Conflict,0.112444,0.102047,0.010397,0.010555,0.001487,0.007793,...,1.0,0.001,True,False,2000,42,True,0.004998,True,AUPRC across five remaining comparators
1,AUPRC,Full GES,Recency,Full GES minus Recency,0.112444,0.082721,0.029723,0.029836,0.001779,0.026528,...,1.0,0.001,True,False,2000,42,True,0.004998,True,AUPRC across five remaining comparators
2,AUPRC,Full GES,Submitter support,Full GES minus Submitter support,0.112444,0.092424,0.020021,0.020215,0.002061,0.016334,...,1.0,0.001,True,False,2000,42,True,0.004998,True,AUPRC across five remaining comparators
3,AUPRC,Full GES,Classification entropy,Full GES minus Classification entropy,0.112444,0.106189,0.006255,0.006411,0.001494,0.003568,...,1.0,0.001,True,False,2000,42,True,0.004998,True,AUPRC across five remaining comparators
4,AUPRC,Full GES,Additive risk,Full GES minus Additive risk,0.112444,0.099511,0.012933,0.013145,0.001646,0.010068,...,1.0,0.001,True,False,2000,42,True,0.004998,True,AUPRC across five remaining comparators
5,AUROC,Full GES,Conflict,Full GES minus Conflict,0.535812,0.513057,0.022755,0.022628,0.003388,0.016064,...,1.0,0.001,True,False,2000,42,True,0.004998,True,AUROC across five remaining comparators
6,AUROC,Full GES,Recency,Full GES minus Recency,0.535812,0.437694,0.098118,0.098160,0.002984,0.092330,...,1.0,0.001,True,False,2000,42,True,0.004998,True,AUROC across five remaining comparators
7,AUROC,Full GES,Submitter support,Full GES minus Submitter support,0.535812,0.471184,0.064629,0.064588,0.003765,0.056923,...,1.0,0.001,True,False,2000,42,True,0.004998,True,AUROC across five remaining comparators
8,AUROC,Full GES,Classification entropy,Full GES minus Classification entropy,0.535812,0.518451,0.017361,0.017263,0.003365,0.010626,...,1.0,0.001,True,False,2000,42,True,0.004998,True,AUROC across five remaining comparators
9,AUROC,Full GES,Additive risk,Full GES minus Additive risk,0.535812,0.483180,0.052633,0.052629,0.002525,0.047415,...,1.0,0.001,True,False,2000,42,True,0.004998,True,AUROC across five remaining comparators



RAW BOOTSTRAP REPLICATE TABLE
----------------------------------------------------------------------------------------------------------------------------------------
Shape: (2000, 15)


,bootstrap_replicate,bootstrap_event_count,bootstrap_prevalence,full_ges_auprc,full_ges_auroc,conflict_auprc,conflict_auroc,recency_auprc,recency_auroc,submitter_auprc,submitter_auroc,entropy_auprc,entropy_auroc,additive_auprc,additive_auroc
0,1,6395,0.095969,0.108925,0.531908,0.100519,0.512705,0.081300,0.435620,0.091105,0.470964,0.104411,0.517397,0.096200,0.480385
1,2,6513,0.097740,0.113195,0.534142,0.102858,0.513747,0.082076,0.436377,0.092796,0.471093,0.106892,0.519081,0.099044,0.483289
2,3,6595,0.098971,0.115756,0.538252,0.104479,0.514574,0.084593,0.439622,0.093776,0.469389,0.109667,0.519460,0.100740,0.481726
3,4,6606,0.099136,0.113975,0.534637,0.103665,0.512404,0.084491,0.439381,0.094601,0.473986,0.107274,0.517057,0.100401,0.483648
4,5,6490,0.097395,0.113723,0.532015,0.102685,0.514247,0.081823,0.428858,0.092184,0.469189,0.106962,0.520366,0.100811,0.484638
5,6,6486,0.097335,0.116151,0.542631,0.103414,0.515321,0.083189,0.441453,0.092125,0.469361,0.107736,0.521781,0.100490,0.488092
6,7,6422,0.096374,0.109574,0.535566,0.100435,0.511793,0.082203,0.437938,0.091299,0.470017,0.105274,0.520231,0.096947,0.483012
7,8,6436,0.096584,0.112256,0.540307,0.101875,0.514013,0.081594,0.439949,0.091896,0.472535,0.104649,0.519132,0.099758,0.486852
8,9,6609,0.099181,0.117477,0.539314,0.104110,0.513305,0.085676,0.444746,0.093894,0.469523,0.108644,0.519631,0.102549,0.483683
9,10,6456,0.096885,0.114351,0.542333,0.101742,0.513375,0.083646,0.443497,0.092553,0.474441,0.106288,0.519146,0.099641,0.487000



CELL DECISION
----------------------------------------------------------------------------------------------------------------------------------------
PASS_STAGE6C_REMAINING_COMPARATOR_PAIRED_AUPRC_AUROC_BOOTSTRAP_INFERENCE_COMPLETE
Paired uncertainty inference is complete for conflict, recency, submitter support, classification entropy, and additive-risk comparators.
Holm correction was applied separately to the five secondary AUPRC comparisons and the five secondary AUROC comparisons.
No score, outcome, comparator formula, direction, weight, threshold, model, or frozen scientific artifact was modified.


In [24]:
# ==================================================================================================
# STAGE 6C STEP 3B — CELL 6C-3B0
# SAME-STAR ANALYSIS PREFLIGHT AND REVIEW-STAR STRATUM INVENTORY
#
# Purpose:
#   1. Freshly verify the frozen Stage 6B primary-evaluable cohort.
#   2. Identify the preserved T0 ClinVar review-star column.
#   3. Confirm that each review-star stratum contains valid outcomes and usable score variation.
#   4. Do NOT yet calculate subgroup AUPRC/AUROC or write scientific artifacts.
# ==================================================================================================

from pathlib import Path
import hashlib
import re

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

# --------------------------------------------------------------------------------------------------
# 1. MOUNT GOOGLE DRIVE
# --------------------------------------------------------------------------------------------------

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
except Exception as exc:
    raise RuntimeError(f"Google Drive could not be mounted: {exc}") from exc


# --------------------------------------------------------------------------------------------------
# 2. FROZEN INPUT PATHS AND EXPECTED IDENTITY
# --------------------------------------------------------------------------------------------------

STAGE6_DIR = Path(
    "/content/drive/MyDrive/GES_RAG_Temporal_Study/"
    "data_processed/stage6_temporal_validation"
)

EVALUABLE_PARQUET = (
    STAGE6_DIR / "stage6b_locked_primary_evaluable_cohort_v1.parquet"
)

EVALUABLE_SIDECAR = Path(str(EVALUABLE_PARQUET) + ".sha256")

EXPECTED_EVALUABLE_SHA256 = (
    "c6ad50dfc376a746e1830d000dba975101d88763fd61b8d1983f6300edb6e038"
)

EXPECTED_ROWS = 66_636
EXPECTED_COLUMNS = 79
EXPECTED_EVENTS = 6_485
EXPECTED_NEGATIVES = 60_151


# --------------------------------------------------------------------------------------------------
# 3. HELPERS
# --------------------------------------------------------------------------------------------------

def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()

    with path.open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)

    return digest.hexdigest()


def read_sidecar_hash(path: Path) -> str:
    text = path.read_text(encoding="utf-8").strip()
    matches = re.findall(r"\b[a-fA-F0-9]{64}\b", text)

    if not matches:
        raise ValueError(f"No SHA-256 value was found in sidecar: {path}")

    return matches[0].lower()


# --------------------------------------------------------------------------------------------------
# 4. CRYPTOGRAPHIC AND PARQUET-METADATA VERIFICATION
# --------------------------------------------------------------------------------------------------

if not EVALUABLE_PARQUET.exists():
    raise FileNotFoundError(f"Missing frozen evaluable cohort:\n{EVALUABLE_PARQUET}")

if not EVALUABLE_SIDECAR.exists():
    raise FileNotFoundError(f"Missing frozen evaluable-cohort sidecar:\n{EVALUABLE_SIDECAR}")

observed_sha256 = sha256_file(EVALUABLE_PARQUET)
sidecar_sha256 = read_sidecar_hash(EVALUABLE_SIDECAR)

if observed_sha256 != EXPECTED_EVALUABLE_SHA256:
    raise AssertionError(
        "Frozen evaluable-cohort SHA-256 does not match the expected Stage 6B value.\n"
        f"Expected: {EXPECTED_EVALUABLE_SHA256}\n"
        f"Observed: {observed_sha256}"
    )

if sidecar_sha256 != observed_sha256:
    raise AssertionError(
        "The evaluable-cohort sidecar does not match the calculated SHA-256.\n"
        f"Calculated: {observed_sha256}\n"
        f"Sidecar:    {sidecar_sha256}"
    )

parquet_file = pq.ParquetFile(EVALUABLE_PARQUET)
metadata = parquet_file.metadata
schema_columns = parquet_file.schema_arrow.names

if metadata.num_rows != EXPECTED_ROWS:
    raise AssertionError(
        f"Unexpected row count: {metadata.num_rows:,}; expected {EXPECTED_ROWS:,}"
    )

if metadata.num_columns != EXPECTED_COLUMNS:
    raise AssertionError(
        f"Unexpected column count: {metadata.num_columns}; expected {EXPECTED_COLUMNS}"
    )


# --------------------------------------------------------------------------------------------------
# 5. IDENTIFY THE PRESERVED T0 REVIEW-STAR COLUMN
# --------------------------------------------------------------------------------------------------

star_column_priority = [
    "t0_aggregate_review_stars",
    "aggregate_review_stars",
    "t0_review_stars",
    "review_stars",
]

star_column = next(
    (column for column in star_column_priority if column in schema_columns),
    None,
)

review_related_columns = [
    column
    for column in schema_columns
    if "star" in column.lower() or "review" in column.lower()
]

if star_column is None:
    print("Review/star-related columns found in the frozen schema:")
    for column in review_related_columns:
        print(f"  - {column}")

    raise KeyError(
        "The raw T0 review-star column could not be selected automatically. "
        "Review the candidate columns printed above."
    )


# --------------------------------------------------------------------------------------------------
# 6. LOAD ONLY THE COLUMNS REQUIRED FOR SAME-STAR PREFLIGHT
# --------------------------------------------------------------------------------------------------

required_columns = [
    "rcv_accession",
    "t0_row_order",
    "primary_future_instability",
    star_column,
    "review_stars_instability_risk",
    "full_ges_instability_risk_t0",
    "no_star_ges_instability_risk_t0",
    "combined_metadata_instability_risk",
]

missing_required = [
    column for column in required_columns if column not in schema_columns
]

if missing_required:
    raise KeyError(
        "Required same-star analysis columns are missing:\n"
        + "\n".join(f"  - {column}" for column in missing_required)
    )

same_star_df = pd.read_parquet(
    EVALUABLE_PARQUET,
    columns=required_columns,
).copy()


# --------------------------------------------------------------------------------------------------
# 7. STRUCTURAL, OUTCOME, STAR, AND SCORE VALIDATION
# --------------------------------------------------------------------------------------------------

if same_star_df.shape[0] != EXPECTED_ROWS:
    raise AssertionError(
        f"Loaded dataframe has {same_star_df.shape[0]:,} rows; expected {EXPECTED_ROWS:,}"
    )

if same_star_df["rcv_accession"].isna().any():
    raise AssertionError("Missing RCV accession detected.")

if same_star_df["rcv_accession"].astype(str).str.strip().eq("").any():
    raise AssertionError("Blank RCV accession detected.")

if same_star_df["rcv_accession"].nunique(dropna=False) != EXPECTED_ROWS:
    raise AssertionError("RCV accessions are not unique in the evaluable cohort.")

row_order = pd.to_numeric(
    same_star_df["t0_row_order"],
    errors="raise",
).to_numpy()

if len(np.unique(row_order)) != EXPECTED_ROWS:
    raise AssertionError("t0_row_order is not unique.")

if not np.all(np.diff(row_order) > 0):
    raise AssertionError(
        "The frozen evaluable cohort is not in monotonically increasing T0 row order."
    )

outcome = pd.to_numeric(
    same_star_df["primary_future_instability"],
    errors="raise",
).astype(int)

if outcome.isna().any():
    raise AssertionError("Missing primary outcome detected.")

if not set(outcome.unique()).issubset({0, 1}):
    raise AssertionError(
        f"Unexpected primary-outcome values: {sorted(outcome.unique().tolist())}"
    )

event_count = int(outcome.sum())
negative_count = int((outcome == 0).sum())

if event_count != EXPECTED_EVENTS:
    raise AssertionError(
        f"Unexpected event count: {event_count:,}; expected {EXPECTED_EVENTS:,}"
    )

if negative_count != EXPECTED_NEGATIVES:
    raise AssertionError(
        f"Unexpected negative count: {negative_count:,}; expected {EXPECTED_NEGATIVES:,}"
    )

same_star_df["_review_stars"] = pd.to_numeric(
    same_star_df[star_column],
    errors="raise",
)

if same_star_df["_review_stars"].isna().any():
    raise AssertionError("Missing T0 review-star values detected.")

if not np.all(
    np.isclose(
        same_star_df["_review_stars"],
        np.round(same_star_df["_review_stars"]),
    )
):
    raise AssertionError("Non-integer T0 review-star values detected.")

same_star_df["_review_stars"] = (
    same_star_df["_review_stars"].round().astype(int)
)

observed_star_levels = sorted(
    same_star_df["_review_stars"].unique().tolist()
)

if not set(observed_star_levels).issubset({0, 1, 2, 3, 4}):
    raise AssertionError(
        f"Unexpected ClinVar review-star levels: {observed_star_levels}"
    )

score_columns = [
    "review_stars_instability_risk",
    "full_ges_instability_risk_t0",
    "no_star_ges_instability_risk_t0",
    "combined_metadata_instability_risk",
]

for column in score_columns:
    values = pd.to_numeric(same_star_df[column], errors="raise").to_numpy(dtype=float)

    if np.isnan(values).any():
        raise AssertionError(f"Missing score values detected in {column}.")

    if not np.isfinite(values).all():
        raise AssertionError(f"Nonfinite score values detected in {column}.")

    if ((values < 0.0) | (values > 1.0)).any():
        raise AssertionError(f"Score outside [0,1] detected in {column}.")


# --------------------------------------------------------------------------------------------------
# 8. SAME-STAR STRATUM INVENTORY
# --------------------------------------------------------------------------------------------------

inventory_rows = []

for star_level in observed_star_levels:
    stratum = same_star_df.loc[
        same_star_df["_review_stars"] == star_level
    ].copy()

    y = stratum["primary_future_instability"].astype(int)

    inventory_rows.append(
        {
            "t0_review_stars": int(star_level),
            "rows": int(len(stratum)),
            "events": int(y.sum()),
            "negatives": int((y == 0).sum()),
            "event_prevalence": float(y.mean()),
            "both_outcome_classes_present": bool(y.nunique() == 2),
            "review_star_risk_unique_values": int(
                stratum["review_stars_instability_risk"].nunique()
            ),
            "full_ges_unique_values": int(
                stratum["full_ges_instability_risk_t0"].nunique()
            ),
            "no_star_ges_unique_values": int(
                stratum["no_star_ges_instability_risk_t0"].nunique()
            ),
            "combined_metadata_unique_values": int(
                stratum["combined_metadata_instability_risk"].nunique()
            ),
            "full_ges_auprc_auroc_estimable": bool(
                y.nunique() == 2
                and stratum["full_ges_instability_risk_t0"].nunique() >= 2
            ),
            "no_star_ges_auprc_auroc_estimable": bool(
                y.nunique() == 2
                and stratum["no_star_ges_instability_risk_t0"].nunique() >= 2
            ),
        }
    )

same_star_stratum_inventory = pd.DataFrame(inventory_rows)


# --------------------------------------------------------------------------------------------------
# 9. PRINT COMPLETE PREFLIGHT RECORD
# --------------------------------------------------------------------------------------------------

separator = "=" * 132
subseparator = "-" * 132

print("\n" + separator)
print(
    "STAGE 6C STEP 3B — CELL 6C-3B0 — "
    "SAME-STAR ANALYSIS PREFLIGHT AND REVIEW-STAR STRATUM INVENTORY"
)
print(separator)

print("\nFROZEN INPUT VERIFICATION")
print(subseparator)
print(f"Primary-evaluable cohort SHA-256 : PASS ({observed_sha256})")
print(
    f"Parquet dimensions                : PASS "
    f"({metadata.num_rows:,} × {metadata.num_columns})"
)
print(
    f"Unique evaluable RCV keys         : PASS "
    f"({same_star_df['rcv_accession'].nunique():,})"
)
print(f"Primary instability events        : PASS ({event_count:,})")
print(f"Primary instability negatives     : PASS ({negative_count:,})")
print(f"Observed event prevalence         : {outcome.mean():.8f} ({outcome.mean():.6%})")
print(f"Selected T0 review-star column    : {star_column}")
print(f"Observed review-star levels       : {observed_star_levels}")
print("Score direction changed           : No")
print("Threshold or weight optimized     : No")
print("Performance metric calculated     : No")
print("Scientific artifact written       : No")

print("\nREVIEW/STAR-RELATED FROZEN COLUMNS")
print(subseparator)
for column in review_related_columns:
    marker = "  <-- selected same-star stratifier" if column == star_column else ""
    print(f"{column}{marker}")

print("\nSAME-STAR STRATUM INVENTORY")
print(subseparator)

with pd.option_context(
    "display.max_columns", None,
    "display.width", 240,
    "display.float_format", lambda value: f"{value:.8f}",
):
    print(same_star_stratum_inventory.to_string(index=False))

print("\nACCOUNTING RECONCILIATION")
print(subseparator)
print(
    f"Stratum rows sum                  : "
    f"{same_star_stratum_inventory['rows'].sum():,}"
)
print(
    f"Stratum events sum                : "
    f"{same_star_stratum_inventory['events'].sum():,}"
)
print(
    f"Stratum negatives sum             : "
    f"{same_star_stratum_inventory['negatives'].sum():,}"
)

if same_star_stratum_inventory["rows"].sum() != EXPECTED_ROWS:
    raise AssertionError("Review-star stratum row counts do not reconcile.")

if same_star_stratum_inventory["events"].sum() != EXPECTED_EVENTS:
    raise AssertionError("Review-star stratum event counts do not reconcile.")

if same_star_stratum_inventory["negatives"].sum() != EXPECTED_NEGATIVES:
    raise AssertionError("Review-star stratum negative counts do not reconcile.")

nonestimable = same_star_stratum_inventory.loc[
    ~same_star_stratum_inventory["full_ges_auprc_auroc_estimable"]
]

print("\nCELL DECISION")
print(subseparator)

if len(nonestimable) == 0:
    print("PASS_STAGE6C_SAME_STAR_ANALYSIS_PREFLIGHT_COMPLETE")
    print(
        "Every observed review-star stratum contains both outcome classes "
        "and sufficient full-GES score variation for locked AUPRC/AUROC analysis."
    )
else:
    print("PASS_WITH_RESTRICTED_STRATA_STAGE6C_SAME_STAR_PREFLIGHT_COMPLETE")
    print(
        "Some review-star strata cannot support both AUPRC and AUROC. "
        "They must be retained descriptively and excluded only from "
        "non-estimable metrics."
    )
    print("\nNon-estimable full-GES strata:")
    print(nonestimable.to_string(index=False))

print(
    "\nNo score, outcome, review-star assignment, threshold, weight, "
    "cohort membership, or frozen scientific artifact was modified."
)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

STAGE 6C STEP 3B — CELL 6C-3B0 — SAME-STAR ANALYSIS PREFLIGHT AND REVIEW-STAR STRATUM INVENTORY

FROZEN INPUT VERIFICATION
------------------------------------------------------------------------------------------------------------------------------------
Primary-evaluable cohort SHA-256 : PASS (c6ad50dfc376a746e1830d000dba975101d88763fd61b8d1983f6300edb6e038)
Parquet dimensions                : PASS (66,636 × 79)
Unique evaluable RCV keys         : PASS (66,636)
Primary instability events        : PASS (6,485)
Primary instability negatives     : PASS (60,151)
Observed event prevalence         : 0.09731977 (9.731977%)
Selected T0 review-star column    : t0_aggregate_review_stars
Observed review-star levels       : [0, 1, 2, 3]
Score direction changed           : No
Threshold or weight optimized     : No
Performance metric calculated     : No
Scientific artif

In [25]:
# ==================================================================================================
# STAGE 6C STEP 3B — CELL 6C-3B1
# LOCKED SAME-STAR DISCRIMINATION POINT ESTIMATES
#
# Purpose:
#   1. Evaluate whether frozen GES scores discriminate future instability among records
#      having the same baseline ClinVar review-star level.
#   2. Calculate within-stratum AUPRC and AUROC for:
#         - Full GES
#         - No-star GES
#         - Combined metadata
#   3. Retain non-estimable strata descriptively.
#   4. Perform no bootstrap inference, threshold optimization, recalibration, or artifact writes.
# ==================================================================================================

from pathlib import Path
import hashlib
import re

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

from sklearn.metrics import average_precision_score, roc_auc_score


# --------------------------------------------------------------------------------------------------
# 1. FROZEN INPUT PATHS AND EXPECTED IDENTITY
# --------------------------------------------------------------------------------------------------

STAGE6_DIR = Path(
    "/content/drive/MyDrive/GES_RAG_Temporal_Study/"
    "data_processed/stage6_temporal_validation"
)

EVALUABLE_PARQUET = (
    STAGE6_DIR / "stage6b_locked_primary_evaluable_cohort_v1.parquet"
)

EVALUABLE_SIDECAR = Path(str(EVALUABLE_PARQUET) + ".sha256")

EXPECTED_EVALUABLE_SHA256 = (
    "c6ad50dfc376a746e1830d000dba975101d88763fd61b8d1983f6300edb6e038"
)

EXPECTED_ROWS = 66_636
EXPECTED_COLUMNS = 79
EXPECTED_EVENTS = 6_485
EXPECTED_NEGATIVES = 60_151

STAR_COLUMN = "t0_aggregate_review_stars"
OUTCOME_COLUMN = "primary_future_instability"

SCORE_SPECIFICATIONS = {
    "full_ges": {
        "column": "full_ges_instability_risk_t0",
        "display_name": "Full GES",
    },
    "no_star_ges": {
        "column": "no_star_ges_instability_risk_t0",
        "display_name": "No-star GES",
    },
    "combined_metadata": {
        "column": "combined_metadata_instability_risk",
        "display_name": "Combined metadata",
    },
}


# --------------------------------------------------------------------------------------------------
# 2. HELPERS
# --------------------------------------------------------------------------------------------------

def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()

    with path.open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)

    return digest.hexdigest()


def read_sidecar_hash(path: Path) -> str:
    text = path.read_text(encoding="utf-8").strip()
    matches = re.findall(r"\b[a-fA-F0-9]{64}\b", text)

    if not matches:
        raise ValueError(f"No SHA-256 value found in sidecar: {path}")

    return matches[0].lower()


def calculate_locked_metrics(
    y_true: np.ndarray,
    risk_score: np.ndarray,
) -> dict:
    """
    Calculate discrimination metrics only when both outcome classes
    and at least two distinct score values are present.
    """

    unique_outcomes = np.unique(y_true)
    unique_scores = np.unique(risk_score)

    if len(unique_outcomes) < 2:
        return {
            "estimable": False,
            "reason": "only_one_outcome_class",
            "auprc": np.nan,
            "auroc": np.nan,
        }

    if len(unique_scores) < 2:
        return {
            "estimable": False,
            "reason": "constant_score",
            "auprc": np.nan,
            "auroc": np.nan,
        }

    return {
        "estimable": True,
        "reason": "estimable",
        "auprc": float(average_precision_score(y_true, risk_score)),
        "auroc": float(roc_auc_score(y_true, risk_score)),
    }


# --------------------------------------------------------------------------------------------------
# 3. FRESH CRYPTOGRAPHIC AND STRUCTURAL VERIFICATION
# --------------------------------------------------------------------------------------------------

if not EVALUABLE_PARQUET.exists():
    raise FileNotFoundError(
        f"Missing frozen primary-evaluable cohort:\n{EVALUABLE_PARQUET}"
    )

if not EVALUABLE_SIDECAR.exists():
    raise FileNotFoundError(
        f"Missing frozen primary-evaluable cohort sidecar:\n{EVALUABLE_SIDECAR}"
    )

observed_sha256 = sha256_file(EVALUABLE_PARQUET)
sidecar_sha256 = read_sidecar_hash(EVALUABLE_SIDECAR)

if observed_sha256 != EXPECTED_EVALUABLE_SHA256:
    raise AssertionError(
        "Frozen Stage 6B evaluable-cohort SHA-256 mismatch.\n"
        f"Expected: {EXPECTED_EVALUABLE_SHA256}\n"
        f"Observed: {observed_sha256}"
    )

if sidecar_sha256 != observed_sha256:
    raise AssertionError(
        "Frozen Stage 6B evaluable-cohort sidecar mismatch.\n"
        f"Calculated: {observed_sha256}\n"
        f"Sidecar:    {sidecar_sha256}"
    )

parquet_file = pq.ParquetFile(EVALUABLE_PARQUET)
metadata = parquet_file.metadata
schema_columns = parquet_file.schema_arrow.names

if metadata.num_rows != EXPECTED_ROWS:
    raise AssertionError(
        f"Unexpected Parquet row count: {metadata.num_rows:,}; "
        f"expected {EXPECTED_ROWS:,}"
    )

if metadata.num_columns != EXPECTED_COLUMNS:
    raise AssertionError(
        f"Unexpected Parquet column count: {metadata.num_columns}; "
        f"expected {EXPECTED_COLUMNS}"
    )


# --------------------------------------------------------------------------------------------------
# 4. LOAD ONLY REQUIRED FROZEN COLUMNS
# --------------------------------------------------------------------------------------------------

required_columns = [
    "rcv_accession",
    "t0_row_order",
    OUTCOME_COLUMN,
    STAR_COLUMN,
    "review_stars_instability_risk",
] + [
    specification["column"]
    for specification in SCORE_SPECIFICATIONS.values()
]

missing_columns = [
    column for column in required_columns
    if column not in schema_columns
]

if missing_columns:
    raise KeyError(
        "Required same-star analysis columns are missing:\n"
        + "\n".join(f"  - {column}" for column in missing_columns)
    )

analysis_df = pd.read_parquet(
    EVALUABLE_PARQUET,
    columns=required_columns,
).copy()


# --------------------------------------------------------------------------------------------------
# 5. LOCKED COHORT VALIDATION
# --------------------------------------------------------------------------------------------------

if analysis_df.shape[0] != EXPECTED_ROWS:
    raise AssertionError(
        f"Loaded dataframe contains {analysis_df.shape[0]:,} rows; "
        f"expected {EXPECTED_ROWS:,}"
    )

if analysis_df["rcv_accession"].isna().any():
    raise AssertionError("Missing RCV accession detected.")

if analysis_df["rcv_accession"].astype(str).str.strip().eq("").any():
    raise AssertionError("Blank RCV accession detected.")

if analysis_df["rcv_accession"].nunique(dropna=False) != EXPECTED_ROWS:
    raise AssertionError("RCV accessions are not unique.")

row_order = pd.to_numeric(
    analysis_df["t0_row_order"],
    errors="raise",
).to_numpy()

if len(np.unique(row_order)) != EXPECTED_ROWS:
    raise AssertionError("t0_row_order is not unique.")

if not np.all(np.diff(row_order) > 0):
    raise AssertionError(
        "Frozen evaluable cohort is not in increasing T0 row order."
    )

analysis_df[OUTCOME_COLUMN] = pd.to_numeric(
    analysis_df[OUTCOME_COLUMN],
    errors="raise",
).astype(int)

if not set(analysis_df[OUTCOME_COLUMN].unique()).issubset({0, 1}):
    raise AssertionError(
        "Primary future-instability outcome is not binary."
    )

event_count = int(analysis_df[OUTCOME_COLUMN].sum())
negative_count = int((analysis_df[OUTCOME_COLUMN] == 0).sum())

if event_count != EXPECTED_EVENTS:
    raise AssertionError(
        f"Unexpected event count: {event_count:,}; expected {EXPECTED_EVENTS:,}"
    )

if negative_count != EXPECTED_NEGATIVES:
    raise AssertionError(
        f"Unexpected negative count: {negative_count:,}; "
        f"expected {EXPECTED_NEGATIVES:,}"
    )

analysis_df[STAR_COLUMN] = pd.to_numeric(
    analysis_df[STAR_COLUMN],
    errors="raise",
)

if analysis_df[STAR_COLUMN].isna().any():
    raise AssertionError("Missing baseline review-star values detected.")

if not np.all(
    np.isclose(
        analysis_df[STAR_COLUMN],
        np.round(analysis_df[STAR_COLUMN]),
    )
):
    raise AssertionError("Non-integer review-star values detected.")

analysis_df[STAR_COLUMN] = (
    analysis_df[STAR_COLUMN].round().astype(int)
)

observed_star_levels = sorted(
    analysis_df[STAR_COLUMN].unique().tolist()
)

if observed_star_levels != [0, 1, 2, 3]:
    raise AssertionError(
        f"Unexpected review-star levels: {observed_star_levels}"
    )

for specification in SCORE_SPECIFICATIONS.values():
    column = specification["column"]

    analysis_df[column] = pd.to_numeric(
        analysis_df[column],
        errors="raise",
    )

    values = analysis_df[column].to_numpy(dtype=float)

    if np.isnan(values).any():
        raise AssertionError(f"Missing score values detected in {column}.")

    if not np.isfinite(values).all():
        raise AssertionError(f"Nonfinite score values detected in {column}.")

    if ((values < 0.0) | (values > 1.0)).any():
        raise AssertionError(
            f"Score outside the frozen [0,1] range detected in {column}."
        )


# --------------------------------------------------------------------------------------------------
# 6. CONFIRM REVIEW-STAR BASELINE IS CONSTANT WITHIN EACH STAR STRATUM
# --------------------------------------------------------------------------------------------------

review_star_baseline_check = (
    analysis_df
    .groupby(STAR_COLUMN, sort=True)["review_stars_instability_risk"]
    .nunique(dropna=False)
)

if not (review_star_baseline_check == 1).all():
    raise AssertionError(
        "Review-star baseline is unexpectedly variable within a same-star stratum."
    )


# --------------------------------------------------------------------------------------------------
# 7. CALCULATE LOCKED SAME-STAR POINT ESTIMATES
# --------------------------------------------------------------------------------------------------

result_rows = []

for star_level in observed_star_levels:

    stratum_df = analysis_df.loc[
        analysis_df[STAR_COLUMN] == star_level
    ].copy()

    y_true = stratum_df[OUTCOME_COLUMN].to_numpy(dtype=int)

    stratum_rows = int(len(stratum_df))
    stratum_events = int(y_true.sum())
    stratum_negatives = int((y_true == 0).sum())
    stratum_prevalence = float(y_true.mean())

    for model_key, specification in SCORE_SPECIFICATIONS.items():

        score_column = specification["column"]
        display_name = specification["display_name"]

        risk_score = stratum_df[score_column].to_numpy(dtype=float)

        metric_result = calculate_locked_metrics(
            y_true=y_true,
            risk_score=risk_score,
        )

        event_mean_risk = (
            float(risk_score[y_true == 1].mean())
            if stratum_events > 0
            else np.nan
        )

        negative_mean_risk = (
            float(risk_score[y_true == 0].mean())
            if stratum_negatives > 0
            else np.nan
        )

        mean_risk_difference = (
            event_mean_risk - negative_mean_risk
            if np.isfinite(event_mean_risk)
            and np.isfinite(negative_mean_risk)
            else np.nan
        )

        auprc_lift_over_stratum_prevalence = (
            metric_result["auprc"] / stratum_prevalence
            if metric_result["estimable"]
            and stratum_prevalence > 0
            else np.nan
        )

        sparse_event_flag = bool(
            metric_result["estimable"]
            and min(stratum_events, stratum_negatives) < 20
        )

        result_rows.append(
            {
                "t0_review_stars": int(star_level),
                "model_key": model_key,
                "model": display_name,
                "score_column": score_column,
                "rows": stratum_rows,
                "events": stratum_events,
                "negatives": stratum_negatives,
                "stratum_prevalence": stratum_prevalence,
                "unique_score_values": int(
                    np.unique(risk_score).size
                ),
                "mean_risk_events": event_mean_risk,
                "mean_risk_negatives": negative_mean_risk,
                "mean_risk_difference_event_minus_negative": (
                    mean_risk_difference
                ),
                "auprc": metric_result["auprc"],
                "auprc_lift_over_stratum_prevalence": (
                    auprc_lift_over_stratum_prevalence
                ),
                "auroc": metric_result["auroc"],
                "estimable": bool(metric_result["estimable"]),
                "estimation_status": metric_result["reason"],
                "sparse_event_or_negative_flag": sparse_event_flag,
            }
        )

same_star_point_estimates = pd.DataFrame(result_rows)


# --------------------------------------------------------------------------------------------------
# 8. CREATE COMPACT DISPLAY TABLE
# --------------------------------------------------------------------------------------------------

same_star_display = same_star_point_estimates[
    [
        "t0_review_stars",
        "model",
        "rows",
        "events",
        "negatives",
        "stratum_prevalence",
        "unique_score_values",
        "mean_risk_events",
        "mean_risk_negatives",
        "mean_risk_difference_event_minus_negative",
        "auprc",
        "auprc_lift_over_stratum_prevalence",
        "auroc",
        "estimable",
        "sparse_event_or_negative_flag",
        "estimation_status",
    ]
].copy()


# --------------------------------------------------------------------------------------------------
# 9. SANITY AND ACCOUNTING CHECKS
# --------------------------------------------------------------------------------------------------

expected_result_rows = (
    len(observed_star_levels) * len(SCORE_SPECIFICATIONS)
)

if len(same_star_point_estimates) != expected_result_rows:
    raise AssertionError(
        f"Unexpected number of result rows: "
        f"{len(same_star_point_estimates)}; expected {expected_result_rows}"
    )

star_accounting = (
    analysis_df
    .groupby(STAR_COLUMN, sort=True)[OUTCOME_COLUMN]
    .agg(
        rows="size",
        events="sum",
    )
    .reset_index()
)

star_accounting["negatives"] = (
    star_accounting["rows"] - star_accounting["events"]
)

if int(star_accounting["rows"].sum()) != EXPECTED_ROWS:
    raise AssertionError("Same-star row accounting failed.")

if int(star_accounting["events"].sum()) != EXPECTED_EVENTS:
    raise AssertionError("Same-star event accounting failed.")

if int(star_accounting["negatives"].sum()) != EXPECTED_NEGATIVES:
    raise AssertionError("Same-star negative accounting failed.")

zero_star_results = same_star_point_estimates.loc[
    same_star_point_estimates["t0_review_stars"] == 0
]

if zero_star_results["estimable"].any():
    raise AssertionError(
        "The all-event 0-star stratum was incorrectly treated as estimable."
    )

estimable_star_levels = sorted(
    same_star_point_estimates.loc[
        same_star_point_estimates["estimable"],
        "t0_review_stars",
    ].unique().tolist()
)

if estimable_star_levels != [1, 2, 3]:
    raise AssertionError(
        f"Unexpected estimable star strata: {estimable_star_levels}"
    )


# --------------------------------------------------------------------------------------------------
# 10. PRINT COMPLETE LOCKED RESULT RECORD
# --------------------------------------------------------------------------------------------------

separator = "=" * 148
subseparator = "-" * 148

print("\n" + separator)
print(
    "STAGE 6C STEP 3B — CELL 6C-3B1 — "
    "LOCKED SAME-STAR DISCRIMINATION POINT ESTIMATES"
)
print(separator)

print("\nFROZEN INPUT VERIFICATION")
print(subseparator)
print(
    f"Stage 6B evaluable-cohort SHA-256 : PASS "
    f"({observed_sha256})"
)
print(
    f"Parquet dimensions                : PASS "
    f"({metadata.num_rows:,} × {metadata.num_columns})"
)
print(
    f"Unique evaluable RCV keys         : PASS "
    f"({analysis_df['rcv_accession'].nunique():,})"
)
print(
    f"Primary instability events        : PASS "
    f"({event_count:,})"
)
print(
    f"Primary instability negatives     : PASS "
    f"({negative_count:,})"
)
print(
    f"Overall event prevalence          : "
    f"{analysis_df[OUTCOME_COLUMN].mean():.8f} "
    f"({analysis_df[OUTCOME_COLUMN].mean():.6%})"
)
print(
    f"Observed review-star strata       : "
    f"{observed_star_levels}"
)
print(
    f"Estimable review-star strata      : "
    f"{estimable_star_levels}"
)
print("Review-star baseline within strata: Constant by construction")
print("Score direction changed           : No")
print("Threshold or weight optimized     : No")
print("Recalibration performed           : No")
print("Bootstrap inference performed     : No")
print("Scientific artifact written       : No")

print("\nSAME-STAR DISCRIMINATION POINT ESTIMATES")
print(subseparator)

with pd.option_context(
    "display.max_columns", None,
    "display.width", 280,
    "display.float_format", lambda value: f"{value:.8f}",
):
    print(same_star_display.to_string(index=False))

print("\nSTAR-LEVEL ACCOUNTING")
print(subseparator)
print(star_accounting.to_string(index=False))

print("\nINTERPRETATION FLAGS")
print(subseparator)
print(
    "0-star stratum : Descriptive only; all 36 records are events, "
    "so AUPRC/AUROC comparisons are not estimable."
)
print(
    "1-star stratum : Fully estimable."
)
print(
    "2-star stratum : Fully estimable."
)
print(
    "3-star stratum : Mathematically estimable but contains only "
    "2 events; point estimates must be interpreted as sparse-event results."
)

print("\nCELL DECISION")
print(subseparator)
print("PASS_STAGE6C_SAME_STAR_POINT_ESTIMATES_COMPLETE")
print(
    "Locked within-review-star AUPRC and AUROC point estimates were "
    "calculated for full GES, no-star GES, and combined metadata."
)
print(
    "The 0-star stratum was retained descriptively and was not assigned "
    "artificial discrimination metrics."
)
print(
    "No score, outcome, cohort membership, review-star assignment, "
    "threshold, weight, frozen model, or scientific artifact was modified."
)


STAGE 6C STEP 3B — CELL 6C-3B1 — LOCKED SAME-STAR DISCRIMINATION POINT ESTIMATES

FROZEN INPUT VERIFICATION
----------------------------------------------------------------------------------------------------------------------------------------------------
Stage 6B evaluable-cohort SHA-256 : PASS (c6ad50dfc376a746e1830d000dba975101d88763fd61b8d1983f6300edb6e038)
Parquet dimensions                : PASS (66,636 × 79)
Unique evaluable RCV keys         : PASS (66,636)
Primary instability events        : PASS (6,485)
Primary instability negatives     : PASS (60,151)
Overall event prevalence          : 0.09731977 (9.731977%)
Observed review-star strata       : [0, 1, 2, 3]
Estimable review-star strata      : [1, 2, 3]
Review-star baseline within strata: Constant by construction
Score direction changed           : No
Threshold or weight optimized     : No
Recalibration performed           : No
Bootstrap inference performed     : No
Scientific artifact written       : No

SAME-STAR DISCRIMIN

In [26]:
# ==================================================================================================
# STAGE 6C STEP 3B — CELL 6C-3B2
# LOCKED SAME-STAR PAIRED BOOTSTRAP INFERENCE
#
# Purpose:
#   1. Run 2,000 nonparametric row-bootstrap replicates separately within each estimable
#      baseline ClinVar review-star stratum.
#   2. Use identical bootstrap samples across Full GES, No-star GES, and Combined Metadata.
#   3. Calculate 95% percentile intervals for:
#         - AUPRC
#         - AUROC
#         - AUPRC minus bootstrap-sample prevalence
#         - AUROC minus 0.50
#   4. Calculate paired Full-GES-minus-comparator differences.
#   5. Apply Holm correction separately to the six AUPRC and six AUROC paired comparisons.
#   6. Retain the 0-star all-event stratum descriptively.
#
# Important:
#   - No score, threshold, model, weight, outcome, or cohort membership is changed.
#   - No recalibration or threshold optimization is performed.
#   - No scientific artifact is written by this cell.
# ==================================================================================================

from pathlib import Path
import hashlib
import re
import time

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

from sklearn.metrics import average_precision_score, roc_auc_score


# --------------------------------------------------------------------------------------------------
# 1. FROZEN INPUT IDENTITY AND ANALYSIS SPECIFICATION
# --------------------------------------------------------------------------------------------------

STAGE6_DIR = Path(
    "/content/drive/MyDrive/GES_RAG_Temporal_Study/"
    "data_processed/stage6_temporal_validation"
)

EVALUABLE_PARQUET = (
    STAGE6_DIR / "stage6b_locked_primary_evaluable_cohort_v1.parquet"
)

EVALUABLE_SIDECAR = Path(str(EVALUABLE_PARQUET) + ".sha256")

EXPECTED_EVALUABLE_SHA256 = (
    "c6ad50dfc376a746e1830d000dba975101d88763fd61b8d1983f6300edb6e038"
)

EXPECTED_ROWS = 66_636
EXPECTED_COLUMNS = 79
EXPECTED_EVENTS = 6_485
EXPECTED_NEGATIVES = 60_151

OUTCOME_COLUMN = "primary_future_instability"
STAR_COLUMN = "t0_aggregate_review_stars"

N_BOOTSTRAP = 2_000
RANDOM_SEED = 42
CI_LOWER_QUANTILE = 0.025
CI_UPPER_QUANTILE = 0.975
MINIMUM_VALID_REPLICATES = 1_000

MODEL_SPECIFICATIONS = {
    "full_ges": {
        "column": "full_ges_instability_risk_t0",
        "display_name": "Full GES",
    },
    "no_star_ges": {
        "column": "no_star_ges_instability_risk_t0",
        "display_name": "No-star GES",
    },
    "combined_metadata": {
        "column": "combined_metadata_instability_risk",
        "display_name": "Combined metadata",
    },
}

PAIRED_COMPARATORS = [
    "no_star_ges",
    "combined_metadata",
]

ANALYSIS_STAR_LEVELS = [1, 2, 3]


# --------------------------------------------------------------------------------------------------
# 2. HELPERS
# --------------------------------------------------------------------------------------------------

def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()

    with path.open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)

    return digest.hexdigest()


def read_sidecar_hash(path: Path) -> str:
    text = path.read_text(encoding="utf-8").strip()
    matches = re.findall(r"\b[a-fA-F0-9]{64}\b", text)

    if not matches:
        raise ValueError(f"No SHA-256 value found in sidecar: {path}")

    return matches[0].lower()


def percentile_interval(values: np.ndarray) -> tuple:
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]

    if len(values) == 0:
        return np.nan, np.nan

    lower, upper = np.quantile(
        values,
        [CI_LOWER_QUANTILE, CI_UPPER_QUANTILE],
    )

    return float(lower), float(upper)


def bootstrap_sign_pvalue(differences: np.ndarray) -> float:
    """
    Two-sided bootstrap sign probability with a plus-one correction.
    The null value is zero.
    """

    differences = np.asarray(differences, dtype=float)
    differences = differences[np.isfinite(differences)]

    if len(differences) == 0:
        return np.nan

    n = len(differences)

    lower_tail = (
        np.count_nonzero(differences <= 0.0) + 1
    ) / (n + 1)

    upper_tail = (
        np.count_nonzero(differences >= 0.0) + 1
    ) / (n + 1)

    return float(min(1.0, 2.0 * min(lower_tail, upper_tail)))


def holm_adjust(p_values: np.ndarray) -> np.ndarray:
    """
    Holm step-down family-wise-error correction.
    Missing values remain missing.
    """

    p_values = np.asarray(p_values, dtype=float)
    adjusted = np.full(len(p_values), np.nan, dtype=float)

    valid_positions = np.where(np.isfinite(p_values))[0]

    if len(valid_positions) == 0:
        return adjusted

    valid_p = p_values[valid_positions]
    order = np.argsort(valid_p)
    number_of_tests = len(valid_p)

    running_maximum = 0.0

    for ordered_rank, ordered_position in enumerate(order):
        original_position = valid_positions[ordered_position]

        raw_adjusted = (
            number_of_tests - ordered_rank
        ) * valid_p[ordered_position]

        running_maximum = max(running_maximum, raw_adjusted)

        adjusted[original_position] = min(
            1.0,
            running_maximum,
        )

    return adjusted


def interval_status(
    lower: float,
    upper: float,
    positive_label: str,
    negative_label: str,
) -> str:

    if not np.isfinite(lower) or not np.isfinite(upper):
        return "not_estimable"

    if lower > 0.0:
        return positive_label

    if upper < 0.0:
        return negative_label

    return "interval_includes_null"


# --------------------------------------------------------------------------------------------------
# 3. FRESH CRYPTOGRAPHIC AND STRUCTURAL VERIFICATION
# --------------------------------------------------------------------------------------------------

if not EVALUABLE_PARQUET.exists():
    raise FileNotFoundError(
        f"Missing frozen primary-evaluable cohort:\n{EVALUABLE_PARQUET}"
    )

if not EVALUABLE_SIDECAR.exists():
    raise FileNotFoundError(
        f"Missing frozen primary-evaluable cohort sidecar:\n{EVALUABLE_SIDECAR}"
    )

observed_sha256 = sha256_file(EVALUABLE_PARQUET)
sidecar_sha256 = read_sidecar_hash(EVALUABLE_SIDECAR)

if observed_sha256 != EXPECTED_EVALUABLE_SHA256:
    raise AssertionError(
        "Frozen Stage 6B evaluable-cohort SHA-256 mismatch.\n"
        f"Expected: {EXPECTED_EVALUABLE_SHA256}\n"
        f"Observed: {observed_sha256}"
    )

if sidecar_sha256 != observed_sha256:
    raise AssertionError(
        "Frozen Stage 6B evaluable-cohort sidecar mismatch.\n"
        f"Calculated: {observed_sha256}\n"
        f"Sidecar:    {sidecar_sha256}"
    )

parquet_file = pq.ParquetFile(EVALUABLE_PARQUET)
metadata = parquet_file.metadata
schema_columns = parquet_file.schema_arrow.names

if metadata.num_rows != EXPECTED_ROWS:
    raise AssertionError(
        f"Unexpected Parquet row count: {metadata.num_rows:,}; "
        f"expected {EXPECTED_ROWS:,}"
    )

if metadata.num_columns != EXPECTED_COLUMNS:
    raise AssertionError(
        f"Unexpected Parquet column count: {metadata.num_columns}; "
        f"expected {EXPECTED_COLUMNS}"
    )


# --------------------------------------------------------------------------------------------------
# 4. LOAD REQUIRED FROZEN COLUMNS
# --------------------------------------------------------------------------------------------------

required_columns = [
    "rcv_accession",
    "t0_row_order",
    OUTCOME_COLUMN,
    STAR_COLUMN,
] + [
    specification["column"]
    for specification in MODEL_SPECIFICATIONS.values()
]

missing_columns = [
    column
    for column in required_columns
    if column not in schema_columns
]

if missing_columns:
    raise KeyError(
        "Required same-star bootstrap columns are missing:\n"
        + "\n".join(f"  - {column}" for column in missing_columns)
    )

analysis_df = pd.read_parquet(
    EVALUABLE_PARQUET,
    columns=required_columns,
).copy()


# --------------------------------------------------------------------------------------------------
# 5. LOCKED COHORT VALIDATION
# --------------------------------------------------------------------------------------------------

if analysis_df.shape[0] != EXPECTED_ROWS:
    raise AssertionError(
        f"Loaded dataframe contains {analysis_df.shape[0]:,} rows; "
        f"expected {EXPECTED_ROWS:,}"
    )

if analysis_df["rcv_accession"].isna().any():
    raise AssertionError("Missing RCV accessions detected.")

if analysis_df["rcv_accession"].astype(str).str.strip().eq("").any():
    raise AssertionError("Blank RCV accessions detected.")

if analysis_df["rcv_accession"].nunique(dropna=False) != EXPECTED_ROWS:
    raise AssertionError("RCV accessions are not unique.")

row_order = pd.to_numeric(
    analysis_df["t0_row_order"],
    errors="raise",
).to_numpy()

if len(np.unique(row_order)) != EXPECTED_ROWS:
    raise AssertionError("t0_row_order is not unique.")

if not np.all(np.diff(row_order) > 0):
    raise AssertionError(
        "Frozen evaluable cohort is not in increasing T0 row order."
    )

analysis_df[OUTCOME_COLUMN] = pd.to_numeric(
    analysis_df[OUTCOME_COLUMN],
    errors="raise",
).astype(int)

if not set(analysis_df[OUTCOME_COLUMN].unique()).issubset({0, 1}):
    raise AssertionError(
        "Primary future-instability outcome is not binary."
    )

event_count = int(analysis_df[OUTCOME_COLUMN].sum())
negative_count = int((analysis_df[OUTCOME_COLUMN] == 0).sum())

if event_count != EXPECTED_EVENTS:
    raise AssertionError(
        f"Unexpected event count: {event_count:,}; "
        f"expected {EXPECTED_EVENTS:,}"
    )

if negative_count != EXPECTED_NEGATIVES:
    raise AssertionError(
        f"Unexpected negative count: {negative_count:,}; "
        f"expected {EXPECTED_NEGATIVES:,}"
    )

analysis_df[STAR_COLUMN] = pd.to_numeric(
    analysis_df[STAR_COLUMN],
    errors="raise",
)

if analysis_df[STAR_COLUMN].isna().any():
    raise AssertionError("Missing baseline review-star values detected.")

if not np.all(
    np.isclose(
        analysis_df[STAR_COLUMN],
        np.round(analysis_df[STAR_COLUMN]),
    )
):
    raise AssertionError("Non-integer review-star values detected.")

analysis_df[STAR_COLUMN] = (
    analysis_df[STAR_COLUMN].round().astype(int)
)

observed_star_levels = sorted(
    analysis_df[STAR_COLUMN].unique().tolist()
)

if observed_star_levels != [0, 1, 2, 3]:
    raise AssertionError(
        f"Unexpected baseline review-star levels: {observed_star_levels}"
    )

for specification in MODEL_SPECIFICATIONS.values():
    column = specification["column"]

    analysis_df[column] = pd.to_numeric(
        analysis_df[column],
        errors="raise",
    )

    values = analysis_df[column].to_numpy(dtype=float)

    if np.isnan(values).any():
        raise AssertionError(
            f"Missing score values detected in {column}."
        )

    if not np.isfinite(values).all():
        raise AssertionError(
            f"Nonfinite score values detected in {column}."
        )

    if ((values < 0.0) | (values > 1.0)).any():
        raise AssertionError(
            f"Score outside [0,1] detected in {column}."
        )


# --------------------------------------------------------------------------------------------------
# 6. VERIFY SAME-STAR OUTCOME ACCOUNTING
# --------------------------------------------------------------------------------------------------

star_accounting = (
    analysis_df
    .groupby(STAR_COLUMN, sort=True)[OUTCOME_COLUMN]
    .agg(
        rows="size",
        events="sum",
    )
    .reset_index()
)

star_accounting["negatives"] = (
    star_accounting["rows"] - star_accounting["events"]
)

expected_star_accounting = {
    0: {"rows": 36, "events": 36, "negatives": 0},
    1: {"rows": 49_224, "events": 4_941, "negatives": 44_283},
    2: {"rows": 9_223, "events": 1_506, "negatives": 7_717},
    3: {"rows": 8_153, "events": 2, "negatives": 8_151},
}

for star_level, expected in expected_star_accounting.items():
    observed_row = star_accounting.loc[
        star_accounting[STAR_COLUMN] == star_level
    ]

    if len(observed_row) != 1:
        raise AssertionError(
            f"Missing or duplicated accounting row for star level {star_level}."
        )

    observed_row = observed_row.iloc[0]

    for field in ["rows", "events", "negatives"]:
        if int(observed_row[field]) != expected[field]:
            raise AssertionError(
                f"Unexpected {field} count for star level {star_level}: "
                f"{int(observed_row[field]):,}; expected {expected[field]:,}"
            )


# --------------------------------------------------------------------------------------------------
# 7. POINT ESTIMATES AND WITHIN-STRATUM PAIRED BOOTSTRAP
# --------------------------------------------------------------------------------------------------

rng = np.random.default_rng(RANDOM_SEED)

point_estimate_rows = []
model_interval_rows = []
paired_difference_rows = []

bootstrap_storage = {}

analysis_start_time = time.time()

for star_level in ANALYSIS_STAR_LEVELS:

    stratum_df = analysis_df.loc[
        analysis_df[STAR_COLUMN] == star_level
    ].reset_index(drop=True)

    y_true = stratum_df[OUTCOME_COLUMN].to_numpy(dtype=int)
    n_rows = len(stratum_df)

    events = int(y_true.sum())
    negatives = int((y_true == 0).sum())
    prevalence = float(y_true.mean())

    if len(np.unique(y_true)) != 2:
        raise AssertionError(
            f"Star-{star_level} stratum does not contain both outcome classes."
        )

    score_arrays = {
        model_key: stratum_df[
            specification["column"]
        ].to_numpy(dtype=float)
        for model_key, specification in MODEL_SPECIFICATIONS.items()
    }

    point_metrics = {}

    for model_key, risk_score in score_arrays.items():
        point_auprc = float(
            average_precision_score(y_true, risk_score)
        )

        point_auroc = float(
            roc_auc_score(y_true, risk_score)
        )

        point_metrics[model_key] = {
            "auprc": point_auprc,
            "auroc": point_auroc,
        }

        point_estimate_rows.append(
            {
                "t0_review_stars": star_level,
                "model_key": model_key,
                "model": MODEL_SPECIFICATIONS[model_key]["display_name"],
                "rows": n_rows,
                "events": events,
                "negatives": negatives,
                "stratum_prevalence": prevalence,
                "auprc": point_auprc,
                "auprc_minus_prevalence": point_auprc - prevalence,
                "auroc": point_auroc,
                "auroc_minus_0_50": point_auroc - 0.50,
            }
        )

    bootstrap_metrics = {
        model_key: {
            "auprc": [],
            "auroc": [],
            "auprc_minus_prevalence": [],
            "auroc_minus_0_50": [],
        }
        for model_key in MODEL_SPECIFICATIONS
    }

    invalid_one_class_replicates = 0

    print(
        f"\nRunning star-{star_level} bootstrap: "
        f"{n_rows:,} rows, {events:,} events, {negatives:,} negatives"
    )

    stratum_start_time = time.time()

    for replicate in range(N_BOOTSTRAP):

        sampled_positions = rng.integers(
            low=0,
            high=n_rows,
            size=n_rows,
        )

        y_bootstrap = y_true[sampled_positions]

        if len(np.unique(y_bootstrap)) < 2:
            invalid_one_class_replicates += 1
            continue

        bootstrap_prevalence = float(y_bootstrap.mean())

        for model_key, original_scores in score_arrays.items():

            bootstrap_scores = original_scores[sampled_positions]

            bootstrap_auprc = float(
                average_precision_score(
                    y_bootstrap,
                    bootstrap_scores,
                )
            )

            bootstrap_auroc = float(
                roc_auc_score(
                    y_bootstrap,
                    bootstrap_scores,
                )
            )

            bootstrap_metrics[model_key]["auprc"].append(
                bootstrap_auprc
            )

            bootstrap_metrics[model_key]["auroc"].append(
                bootstrap_auroc
            )

            bootstrap_metrics[model_key][
                "auprc_minus_prevalence"
            ].append(
                bootstrap_auprc - bootstrap_prevalence
            )

            bootstrap_metrics[model_key][
                "auroc_minus_0_50"
            ].append(
                bootstrap_auroc - 0.50
            )

        if (
            (replicate + 1) % 250 == 0
            or replicate + 1 == N_BOOTSTRAP
        ):
            elapsed = time.time() - stratum_start_time

            print(
                f"  Completed {replicate + 1:,}/{N_BOOTSTRAP:,} "
                f"attempted replicates | elapsed {elapsed:.1f} seconds"
            )

    valid_replicates = len(
        bootstrap_metrics["full_ges"]["auprc"]
    )

    if valid_replicates < MINIMUM_VALID_REPLICATES:
        raise AssertionError(
            f"Star-{star_level} produced only {valid_replicates:,} valid "
            f"bootstrap replicates; minimum required is "
            f"{MINIMUM_VALID_REPLICATES:,}."
        )

    for model_key in MODEL_SPECIFICATIONS:
        for metric_name in bootstrap_metrics[model_key]:
            bootstrap_metrics[model_key][metric_name] = np.asarray(
                bootstrap_metrics[model_key][metric_name],
                dtype=float,
            )

            if (
                len(bootstrap_metrics[model_key][metric_name])
                != valid_replicates
            ):
                raise AssertionError(
                    f"Bootstrap-length mismatch for star {star_level}, "
                    f"model {model_key}, metric {metric_name}."
                )

    bootstrap_storage[star_level] = bootstrap_metrics

    for model_key, specification in MODEL_SPECIFICATIONS.items():

        auprc_values = bootstrap_metrics[model_key]["auprc"]
        auroc_values = bootstrap_metrics[model_key]["auroc"]

        auprc_null_difference_values = bootstrap_metrics[
            model_key
        ]["auprc_minus_prevalence"]

        auroc_null_difference_values = bootstrap_metrics[
            model_key
        ]["auroc_minus_0_50"]

        auprc_lower, auprc_upper = percentile_interval(
            auprc_values
        )

        auroc_lower, auroc_upper = percentile_interval(
            auroc_values
        )

        auprc_null_lower, auprc_null_upper = percentile_interval(
            auprc_null_difference_values
        )

        auroc_null_lower, auroc_null_upper = percentile_interval(
            auroc_null_difference_values
        )

        model_interval_rows.append(
            {
                "t0_review_stars": star_level,
                "model_key": model_key,
                "model": specification["display_name"],
                "rows": n_rows,
                "events": events,
                "negatives": negatives,
                "stratum_prevalence": prevalence,
                "point_auprc": point_metrics[model_key]["auprc"],
                "auprc_ci_lower": auprc_lower,
                "auprc_ci_upper": auprc_upper,
                "point_auprc_minus_prevalence": (
                    point_metrics[model_key]["auprc"] - prevalence
                ),
                "auprc_minus_prevalence_ci_lower": auprc_null_lower,
                "auprc_minus_prevalence_ci_upper": auprc_null_upper,
                "auprc_null_status": interval_status(
                    auprc_null_lower,
                    auprc_null_upper,
                    "supported_above_stratum_prevalence",
                    "supported_below_stratum_prevalence",
                ),
                "point_auroc": point_metrics[model_key]["auroc"],
                "auroc_ci_lower": auroc_lower,
                "auroc_ci_upper": auroc_upper,
                "point_auroc_minus_0_50": (
                    point_metrics[model_key]["auroc"] - 0.50
                ),
                "auroc_minus_0_50_ci_lower": auroc_null_lower,
                "auroc_minus_0_50_ci_upper": auroc_null_upper,
                "auroc_null_status": interval_status(
                    auroc_null_lower,
                    auroc_null_upper,
                    "supported_above_0_50",
                    "supported_below_0_50",
                ),
                "attempted_bootstrap_replicates": N_BOOTSTRAP,
                "valid_bootstrap_replicates": valid_replicates,
                "invalid_one_class_replicates": (
                    invalid_one_class_replicates
                ),
                "sparse_event_or_negative_flag": bool(
                    min(events, negatives) < 20
                ),
            }
        )

    for comparator_key in PAIRED_COMPARATORS:

        comparator_display = MODEL_SPECIFICATIONS[
            comparator_key
        ]["display_name"]

        for metric_name in ["auprc", "auroc"]:

            full_values = bootstrap_metrics[
                "full_ges"
            ][metric_name]

            comparator_values = bootstrap_metrics[
                comparator_key
            ][metric_name]

            paired_differences = (
                full_values - comparator_values
            )

            difference_lower, difference_upper = percentile_interval(
                paired_differences
            )

            point_difference = (
                point_metrics["full_ges"][metric_name]
                - point_metrics[comparator_key][metric_name]
            )

            paired_difference_rows.append(
                {
                    "metric": metric_name.upper(),
                    "t0_review_stars": star_level,
                    "comparison": (
                        f"Full GES minus {comparator_display}"
                    ),
                    "comparator_key": comparator_key,
                    "rows": n_rows,
                    "events": events,
                    "negatives": negatives,
                    "point_difference": point_difference,
                    "difference_ci_lower": difference_lower,
                    "difference_ci_upper": difference_upper,
                    "paired_interval_status": interval_status(
                        difference_lower,
                        difference_upper,
                        "full_ges_supported_higher",
                        "full_ges_supported_lower",
                    ),
                    "bootstrap_probability_full_greater": float(
                        np.mean(paired_differences > 0.0)
                    ),
                    "bootstrap_probability_equal": float(
                        np.mean(paired_differences == 0.0)
                    ),
                    "bootstrap_sign_p_value": bootstrap_sign_pvalue(
                        paired_differences
                    ),
                    "attempted_bootstrap_replicates": N_BOOTSTRAP,
                    "valid_bootstrap_replicates": valid_replicates,
                    "invalid_one_class_replicates": (
                        invalid_one_class_replicates
                    ),
                    "sparse_event_or_negative_flag": bool(
                        min(events, negatives) < 20
                    ),
                }
            )


# --------------------------------------------------------------------------------------------------
# 8. CREATE RESULT TABLES
# --------------------------------------------------------------------------------------------------

same_star_point_estimates = pd.DataFrame(
    point_estimate_rows
)

same_star_model_intervals = pd.DataFrame(
    model_interval_rows
)

same_star_paired_differences = pd.DataFrame(
    paired_difference_rows
)


# --------------------------------------------------------------------------------------------------
# 9. HOLM CORRECTION WITHIN AUPRC AND AUROC PAIRED-COMPARISON FAMILIES
# --------------------------------------------------------------------------------------------------

same_star_paired_differences[
    "holm_adjusted_bootstrap_sign_p"
] = np.nan

for metric_name in ["AUPRC", "AUROC"]:

    family_mask = (
        same_star_paired_differences["metric"] == metric_name
    )

    family_p_values = same_star_paired_differences.loc[
        family_mask,
        "bootstrap_sign_p_value",
    ].to_numpy(dtype=float)

    adjusted_values = holm_adjust(
        family_p_values
    )

    same_star_paired_differences.loc[
        family_mask,
        "holm_adjusted_bootstrap_sign_p",
    ] = adjusted_values

same_star_paired_differences[
    "holm_supported_at_0_05"
] = (
    same_star_paired_differences[
        "holm_adjusted_bootstrap_sign_p"
    ] < 0.05
)


# --------------------------------------------------------------------------------------------------
# 10. COMPLETENESS AND INTERNAL-CONSISTENCY CHECKS
# --------------------------------------------------------------------------------------------------

expected_model_rows = (
    len(ANALYSIS_STAR_LEVELS)
    * len(MODEL_SPECIFICATIONS)
)

if len(same_star_model_intervals) != expected_model_rows:
    raise AssertionError(
        f"Unexpected model-interval row count: "
        f"{len(same_star_model_intervals)}; "
        f"expected {expected_model_rows}."
    )

expected_paired_rows = (
    len(ANALYSIS_STAR_LEVELS)
    * len(PAIRED_COMPARATORS)
    * 2
)

if len(same_star_paired_differences) != expected_paired_rows:
    raise AssertionError(
        f"Unexpected paired-comparison row count: "
        f"{len(same_star_paired_differences)}; "
        f"expected {expected_paired_rows}."
    )

if (
    same_star_model_intervals[
        "valid_bootstrap_replicates"
    ] < MINIMUM_VALID_REPLICATES
).any():
    raise AssertionError(
        "At least one model/stratum has too few valid bootstrap replicates."
    )

if (
    same_star_paired_differences[
        "valid_bootstrap_replicates"
    ] < MINIMUM_VALID_REPLICATES
).any():
    raise AssertionError(
        "At least one paired comparison has too few valid bootstrap replicates."
    )

for metric_name in ["AUPRC", "AUROC"]:

    family_rows = same_star_paired_differences.loc[
        same_star_paired_differences["metric"] == metric_name
    ]

    if len(family_rows) != 6:
        raise AssertionError(
            f"{metric_name} Holm family contains {len(family_rows)} rows; "
            "expected 6."
        )

if same_star_paired_differences[
    "holm_adjusted_bootstrap_sign_p"
].isna().any():
    raise AssertionError(
        "Missing Holm-adjusted paired-comparison probability detected."
    )


# --------------------------------------------------------------------------------------------------
# 11. COMPACT DISPLAY TABLES
# --------------------------------------------------------------------------------------------------

model_display_columns = [
    "t0_review_stars",
    "model",
    "rows",
    "events",
    "negatives",
    "stratum_prevalence",
    "point_auprc",
    "auprc_ci_lower",
    "auprc_ci_upper",
    "point_auprc_minus_prevalence",
    "auprc_minus_prevalence_ci_lower",
    "auprc_minus_prevalence_ci_upper",
    "auprc_null_status",
    "point_auroc",
    "auroc_ci_lower",
    "auroc_ci_upper",
    "point_auroc_minus_0_50",
    "auroc_minus_0_50_ci_lower",
    "auroc_minus_0_50_ci_upper",
    "auroc_null_status",
    "valid_bootstrap_replicates",
    "invalid_one_class_replicates",
    "sparse_event_or_negative_flag",
]

paired_display_columns = [
    "metric",
    "t0_review_stars",
    "comparison",
    "rows",
    "events",
    "negatives",
    "point_difference",
    "difference_ci_lower",
    "difference_ci_upper",
    "paired_interval_status",
    "bootstrap_probability_full_greater",
    "bootstrap_sign_p_value",
    "holm_adjusted_bootstrap_sign_p",
    "holm_supported_at_0_05",
    "valid_bootstrap_replicates",
    "invalid_one_class_replicates",
    "sparse_event_or_negative_flag",
]

same_star_model_display = same_star_model_intervals[
    model_display_columns
].copy()

same_star_paired_display = same_star_paired_differences[
    paired_display_columns
].copy()


# --------------------------------------------------------------------------------------------------
# 12. PRINT COMPLETE LOCKED INFERENCE RECORD
# --------------------------------------------------------------------------------------------------

analysis_elapsed = time.time() - analysis_start_time

separator = "=" * 166
subseparator = "-" * 166

print("\n" + separator)
print(
    "STAGE 6C STEP 3B — CELL 6C-3B2 — "
    "LOCKED SAME-STAR PAIRED BOOTSTRAP INFERENCE"
)
print(separator)

print("\nFROZEN INPUT VERIFICATION")
print(subseparator)
print(
    f"Stage 6B evaluable-cohort SHA-256 : PASS "
    f"({observed_sha256})"
)
print(
    f"Parquet dimensions                : PASS "
    f"({metadata.num_rows:,} × {metadata.num_columns})"
)
print(
    f"Unique evaluable RCV keys         : PASS "
    f"({analysis_df['rcv_accession'].nunique():,})"
)
print(
    f"Primary instability events        : PASS "
    f"({event_count:,})"
)
print(
    f"Primary instability negatives     : PASS "
    f"({negative_count:,})"
)
print(
    f"Overall event prevalence          : "
    f"{analysis_df[OUTCOME_COLUMN].mean():.8f} "
    f"({analysis_df[OUTCOME_COLUMN].mean():.6%})"
)
print(
    f"Bootstrap attempts per stratum    : "
    f"{N_BOOTSTRAP:,}"
)
print(
    f"Bootstrap random seed             : "
    f"{RANDOM_SEED}"
)
print(
    f"Bootstrap design                  : "
    f"Nonparametric row bootstrap within each review-star stratum"
)
print(
    f"Paired resampling                 : "
    f"Identical indices across all three models within each stratum"
)
print(
    f"Percentile interval               : "
    f"95% ({CI_LOWER_QUANTILE:.3f}, {CI_UPPER_QUANTILE:.3f})"
)
print(
    f"Holm families                     : "
    f"Six AUPRC and six AUROC Full-GES-minus-comparator comparisons"
)
print("Score direction changed           : No")
print("Threshold or weight optimized     : No")
print("Recalibration performed           : No")
print("Scientific artifact written       : No")
print(
    f"Total analysis runtime            : "
    f"{analysis_elapsed:.1f} seconds"
)

print("\nSAME-STAR MODEL-SPECIFIC BOOTSTRAP INTERVALS")
print(subseparator)

with pd.option_context(
    "display.max_columns", None,
    "display.width", 360,
    "display.float_format", lambda value: f"{value:.8f}",
):
    print(
        same_star_model_display.to_string(
            index=False
        )
    )

print("\nPAIRED FULL-GES-MINUS-COMPARATOR INFERENCE")
print(subseparator)

with pd.option_context(
    "display.max_columns", None,
    "display.width", 340,
    "display.float_format", lambda value: f"{value:.8f}",
):
    print(
        same_star_paired_display.to_string(
            index=False
        )
    )

print("\nNON-ESTIMABLE AND SPARSE STRATA")
print(subseparator)
print(
    "0-star stratum : 36 events and 0 negatives. "
    "Retained descriptively; AUPRC and AUROC are not estimable."
)
print(
    "3-star stratum : 2 events and 8,151 negatives. "
    "Ordinary row-bootstrap samples lacking both classes were excluded, "
    "and the exact valid/invalid replicate counts are reported."
)
print(
    "The 3-star estimates remain mathematically valid only for the retained "
    "two-class replicates and must be interpreted as extremely sparse-event evidence."
)

print("\nCELL DECISION")
print(subseparator)
print("PASS_STAGE6C_SAME_STAR_BOOTSTRAP_INFERENCE_COMPLETE")
print(
    "Two-thousand attempted paired bootstrap replicates were completed "
    "within each estimable review-star stratum."
)
print(
    "Model-specific intervals, null-reference differences, paired "
    "Full-GES-minus-comparator intervals, bootstrap sign probabilities, "
    "and Holm-adjusted secondary-comparison probabilities were calculated."
)
print(
    "No score, outcome, review-star assignment, cohort membership, "
    "threshold, weight, frozen model, or scientific artifact was modified."
)


Running star-1 bootstrap: 49,224 rows, 4,941 events, 44,283 negatives


KeyboardInterrupt: 

In [27]:
# ==================================================================================================
# STAGE 6C STEP 3B — CELL 6C-3B2
# OPTIMIZED LOCKED SAME-STAR PAIRED BOOTSTRAP INFERENCE
#
# Exact analysis preserved:
#   - 2,000 ordinary nonparametric row-bootstrap samples per review-star stratum
#   - Multinomial bootstrap counts are mathematically identical to sampling rows with replacement
#   - Identical bootstrap samples are used across all models within each stratum
#   - Ties are handled exactly through grouped weighted AUPRC and AUROC calculations
#
# Models:
#   - Full GES
#   - No-star GES
#   - Combined metadata
#
# Strata:
#   - 0 stars: descriptive only because all 36 records are events
#   - 1, 2, and 3 stars: bootstrap inference
#
# No scientific artifact is written by this cell.
# ==================================================================================================

from pathlib import Path
import hashlib
import re
import time
import gc

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

from scipy.sparse import csr_matrix
from sklearn.metrics import average_precision_score, roc_auc_score


# --------------------------------------------------------------------------------------------------
# 1. FROZEN INPUT IDENTITY AND ANALYSIS SPECIFICATION
# --------------------------------------------------------------------------------------------------

STAGE6_DIR = Path(
    "/content/drive/MyDrive/GES_RAG_Temporal_Study/"
    "data_processed/stage6_temporal_validation"
)

EVALUABLE_PARQUET = (
    STAGE6_DIR / "stage6b_locked_primary_evaluable_cohort_v1.parquet"
)

EVALUABLE_SIDECAR = Path(str(EVALUABLE_PARQUET) + ".sha256")

EXPECTED_EVALUABLE_SHA256 = (
    "c6ad50dfc376a746e1830d000dba975101d88763fd61b8d1983f6300edb6e038"
)

EXPECTED_ROWS = 66_636
EXPECTED_COLUMNS = 79
EXPECTED_EVENTS = 6_485
EXPECTED_NEGATIVES = 60_151

OUTCOME_COLUMN = "primary_future_instability"
STAR_COLUMN = "t0_aggregate_review_stars"

N_BOOTSTRAP = 2_000
RANDOM_SEED = 42
BOOTSTRAP_BATCH_SIZE = 50

CI_LOWER_QUANTILE = 0.025
CI_UPPER_QUANTILE = 0.975
MINIMUM_VALID_REPLICATES = 1_000

ANALYSIS_STAR_LEVELS = [1, 2, 3]

MODEL_SPECIFICATIONS = {
    "full_ges": {
        "column": "full_ges_instability_risk_t0",
        "display_name": "Full GES",
    },
    "no_star_ges": {
        "column": "no_star_ges_instability_risk_t0",
        "display_name": "No-star GES",
    },
    "combined_metadata": {
        "column": "combined_metadata_instability_risk",
        "display_name": "Combined metadata",
    },
}

PAIRED_COMPARATORS = [
    "no_star_ges",
    "combined_metadata",
]


# --------------------------------------------------------------------------------------------------
# 2. GENERAL HELPERS
# --------------------------------------------------------------------------------------------------

def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()

    with path.open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


def read_sidecar_hash(path: Path) -> str:
    text = path.read_text(encoding="utf-8").strip()
    matches = re.findall(r"\b[a-fA-F0-9]{64}\b", text)

    if not matches:
        raise ValueError(
            f"No SHA-256 value was found in sidecar:\n{path}"
        )

    return matches[0].lower()


def percentile_interval(values: np.ndarray) -> tuple:
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]

    if len(values) == 0:
        return np.nan, np.nan

    lower, upper = np.quantile(
        values,
        [CI_LOWER_QUANTILE, CI_UPPER_QUANTILE],
    )

    return float(lower), float(upper)


def bootstrap_sign_pvalue(differences: np.ndarray) -> float:
    """
    Two-sided bootstrap sign probability with plus-one correction.
    """

    differences = np.asarray(differences, dtype=float)
    differences = differences[np.isfinite(differences)]

    if len(differences) == 0:
        return np.nan

    n_valid = len(differences)

    lower_tail = (
        np.count_nonzero(differences <= 0.0) + 1
    ) / (n_valid + 1)

    upper_tail = (
        np.count_nonzero(differences >= 0.0) + 1
    ) / (n_valid + 1)

    return float(
        min(
            1.0,
            2.0 * min(lower_tail, upper_tail),
        )
    )


def holm_adjust(p_values: np.ndarray) -> np.ndarray:
    """
    Holm step-down family-wise error correction.
    """

    p_values = np.asarray(p_values, dtype=float)
    adjusted = np.full(len(p_values), np.nan, dtype=float)

    valid_positions = np.where(np.isfinite(p_values))[0]

    if len(valid_positions) == 0:
        return adjusted

    valid_p_values = p_values[valid_positions]
    ordered_positions = np.argsort(valid_p_values)
    number_of_tests = len(valid_p_values)

    running_maximum = 0.0

    for ordered_rank, within_valid_position in enumerate(ordered_positions):

        original_position = valid_positions[within_valid_position]

        raw_adjusted_value = (
            number_of_tests - ordered_rank
        ) * valid_p_values[within_valid_position]

        running_maximum = max(
            running_maximum,
            raw_adjusted_value,
        )

        adjusted[original_position] = min(
            1.0,
            running_maximum,
        )

    return adjusted


def interval_status(
    lower: float,
    upper: float,
    positive_label: str,
    negative_label: str,
) -> str:

    if not np.isfinite(lower) or not np.isfinite(upper):
        return "not_estimable"

    if lower > 0.0:
        return positive_label

    if upper < 0.0:
        return negative_label

    return "interval_includes_null"


# --------------------------------------------------------------------------------------------------
# 3. FAST TIE-AWARE METRIC ENGINE
# --------------------------------------------------------------------------------------------------

def construct_score_group_cache(
    scores: np.ndarray,
    outcomes: np.ndarray,
) -> dict:
    """
    Construct sparse score-group matrices once for a model.

    np.unique returns score groups in ascending score order.

    The matrices permit bootstrap sample counts to be aggregated by
    tied score group without rebuilding or sorting every bootstrap sample.
    """

    scores = np.asarray(scores, dtype=float)
    outcomes = np.asarray(outcomes, dtype=int)

    unique_scores, group_index = np.unique(
        scores,
        return_inverse=True,
    )

    n_rows = len(scores)
    n_groups = len(unique_scores)
    row_positions = np.arange(n_rows)

    total_group_matrix = csr_matrix(
        (
            np.ones(n_rows, dtype=np.float64),
            (group_index, row_positions),
        ),
        shape=(n_groups, n_rows),
    )

    positive_positions = np.flatnonzero(outcomes == 1)

    positive_group_matrix = csr_matrix(
        (
            np.ones(len(positive_positions), dtype=np.float64),
            (
                group_index[positive_positions],
                positive_positions,
            ),
        ),
        shape=(n_groups, n_rows),
    )

    return {
        "unique_scores": unique_scores,
        "number_of_groups": n_groups,
        "total_group_matrix": total_group_matrix,
        "positive_group_matrix": positive_group_matrix,
    }


def calculate_grouped_weighted_metrics(
    cache: dict,
    count_matrix: np.ndarray,
    positive_totals: np.ndarray,
) -> tuple:
    """
    Calculate AUPRC and AUROC for multiple bootstrap samples simultaneously.

    count_matrix:
        Shape = bootstrap samples × original rows.

    Each row contains multinomial bootstrap multiplicities.

    AUPRC:
        Exact weighted average precision using descending tied-score groups.

    AUROC:
        Exact weighted pair probability:
        P(score_positive > score_negative) + 0.5 × P(tied scores).
    """

    count_matrix = np.asarray(count_matrix)
    positive_totals = np.asarray(
        positive_totals,
        dtype=np.float64,
    )

    sample_totals = count_matrix.sum(axis=1).astype(np.float64)
    negative_totals = sample_totals - positive_totals

    # Shape after multiplication: score groups × bootstrap samples.
    group_total_counts = np.asarray(
        cache["total_group_matrix"] @ count_matrix.T,
        dtype=np.float64,
    )

    group_positive_counts = np.asarray(
        cache["positive_group_matrix"] @ count_matrix.T,
        dtype=np.float64,
    )

    group_negative_counts = (
        group_total_counts - group_positive_counts
    )

    valid_mask = (
        (positive_totals > 0.0)
        & (negative_totals > 0.0)
    )

    # ----------------------------------------------------------------------------------------------
    # Weighted AUROC using ascending score groups
    # ----------------------------------------------------------------------------------------------

    cumulative_negatives_before_group = (
        np.cumsum(
            group_negative_counts,
            axis=0,
        )
        - group_negative_counts
    )

    concordant_pair_numerator = np.sum(
        group_positive_counts
        * (
            cumulative_negatives_before_group
            + 0.5 * group_negative_counts
        ),
        axis=0,
    )

    auc_denominator = (
        positive_totals * negative_totals
    )

    auroc = np.full(
        len(positive_totals),
        np.nan,
        dtype=np.float64,
    )

    np.divide(
        concordant_pair_numerator,
        auc_denominator,
        out=auroc,
        where=valid_mask,
    )

    # ----------------------------------------------------------------------------------------------
    # Weighted AUPRC using descending score groups
    # ----------------------------------------------------------------------------------------------

    positive_descending = group_positive_counts[::-1, :]
    total_descending = group_total_counts[::-1, :]

    cumulative_positive = np.cumsum(
        positive_descending,
        axis=0,
    )

    cumulative_total = np.cumsum(
        total_descending,
        axis=0,
    )

    precision_at_each_group = np.zeros_like(
        cumulative_positive,
        dtype=np.float64,
    )

    np.divide(
        cumulative_positive,
        cumulative_total,
        out=precision_at_each_group,
        where=cumulative_total > 0.0,
    )

    average_precision_numerator = np.sum(
        precision_at_each_group * positive_descending,
        axis=0,
    )

    auprc = np.full(
        len(positive_totals),
        np.nan,
        dtype=np.float64,
    )

    np.divide(
        average_precision_numerator,
        positive_totals,
        out=auprc,
        where=valid_mask,
    )

    return auprc, auroc


# --------------------------------------------------------------------------------------------------
# 4. FRESH CRYPTOGRAPHIC AND STRUCTURAL VERIFICATION
# --------------------------------------------------------------------------------------------------

if not EVALUABLE_PARQUET.exists():
    raise FileNotFoundError(
        f"Missing frozen primary-evaluable cohort:\n{EVALUABLE_PARQUET}"
    )

if not EVALUABLE_SIDECAR.exists():
    raise FileNotFoundError(
        f"Missing frozen evaluable-cohort sidecar:\n{EVALUABLE_SIDECAR}"
    )

observed_sha256 = sha256_file(EVALUABLE_PARQUET)
sidecar_sha256 = read_sidecar_hash(EVALUABLE_SIDECAR)

if observed_sha256 != EXPECTED_EVALUABLE_SHA256:
    raise AssertionError(
        "Frozen Stage 6B evaluable-cohort SHA-256 mismatch.\n"
        f"Expected: {EXPECTED_EVALUABLE_SHA256}\n"
        f"Observed: {observed_sha256}"
    )

if sidecar_sha256 != observed_sha256:
    raise AssertionError(
        "Frozen Stage 6B evaluable-cohort sidecar mismatch.\n"
        f"Calculated: {observed_sha256}\n"
        f"Sidecar:    {sidecar_sha256}"
    )

parquet_file = pq.ParquetFile(EVALUABLE_PARQUET)
metadata = parquet_file.metadata
schema_columns = parquet_file.schema_arrow.names

if metadata.num_rows != EXPECTED_ROWS:
    raise AssertionError(
        f"Unexpected Parquet row count: {metadata.num_rows:,}; "
        f"expected {EXPECTED_ROWS:,}"
    )

if metadata.num_columns != EXPECTED_COLUMNS:
    raise AssertionError(
        f"Unexpected Parquet column count: {metadata.num_columns}; "
        f"expected {EXPECTED_COLUMNS}"
    )


# --------------------------------------------------------------------------------------------------
# 5. LOAD REQUIRED FROZEN COLUMNS
# --------------------------------------------------------------------------------------------------

required_columns = [
    "rcv_accession",
    "t0_row_order",
    OUTCOME_COLUMN,
    STAR_COLUMN,
] + [
    specification["column"]
    for specification in MODEL_SPECIFICATIONS.values()
]

missing_columns = [
    column
    for column in required_columns
    if column not in schema_columns
]

if missing_columns:
    raise KeyError(
        "Required same-star bootstrap columns are missing:\n"
        + "\n".join(
            f"  - {column}"
            for column in missing_columns
        )
    )

analysis_df = pd.read_parquet(
    EVALUABLE_PARQUET,
    columns=required_columns,
).copy()


# --------------------------------------------------------------------------------------------------
# 6. LOCKED COHORT VALIDATION
# --------------------------------------------------------------------------------------------------

if analysis_df.shape[0] != EXPECTED_ROWS:
    raise AssertionError(
        f"Loaded dataframe contains {analysis_df.shape[0]:,} rows; "
        f"expected {EXPECTED_ROWS:,}"
    )

if analysis_df["rcv_accession"].isna().any():
    raise AssertionError("Missing RCV accession detected.")

if analysis_df["rcv_accession"].astype(str).str.strip().eq("").any():
    raise AssertionError("Blank RCV accession detected.")

if analysis_df["rcv_accession"].nunique(dropna=False) != EXPECTED_ROWS:
    raise AssertionError("RCV accessions are not unique.")

row_order = pd.to_numeric(
    analysis_df["t0_row_order"],
    errors="raise",
).to_numpy()

if len(np.unique(row_order)) != EXPECTED_ROWS:
    raise AssertionError("t0_row_order is not unique.")

if not np.all(np.diff(row_order) > 0):
    raise AssertionError(
        "Frozen evaluable cohort is not in increasing T0 row order."
    )

analysis_df[OUTCOME_COLUMN] = pd.to_numeric(
    analysis_df[OUTCOME_COLUMN],
    errors="raise",
).astype(int)

if not set(analysis_df[OUTCOME_COLUMN].unique()).issubset({0, 1}):
    raise AssertionError(
        "Primary future-instability outcome is not binary."
    )

event_count = int(
    analysis_df[OUTCOME_COLUMN].sum()
)

negative_count = int(
    (analysis_df[OUTCOME_COLUMN] == 0).sum()
)

if event_count != EXPECTED_EVENTS:
    raise AssertionError(
        f"Unexpected event count: {event_count:,}; "
        f"expected {EXPECTED_EVENTS:,}"
    )

if negative_count != EXPECTED_NEGATIVES:
    raise AssertionError(
        f"Unexpected negative count: {negative_count:,}; "
        f"expected {EXPECTED_NEGATIVES:,}"
    )

analysis_df[STAR_COLUMN] = pd.to_numeric(
    analysis_df[STAR_COLUMN],
    errors="raise",
)

if analysis_df[STAR_COLUMN].isna().any():
    raise AssertionError(
        "Missing baseline review-star value detected."
    )

if not np.all(
    np.isclose(
        analysis_df[STAR_COLUMN],
        np.round(analysis_df[STAR_COLUMN]),
    )
):
    raise AssertionError(
        "Non-integer review-star value detected."
    )

analysis_df[STAR_COLUMN] = (
    analysis_df[STAR_COLUMN]
    .round()
    .astype(int)
)

observed_star_levels = sorted(
    analysis_df[STAR_COLUMN].unique().tolist()
)

if observed_star_levels != [0, 1, 2, 3]:
    raise AssertionError(
        f"Unexpected baseline review-star levels: "
        f"{observed_star_levels}"
    )

for specification in MODEL_SPECIFICATIONS.values():

    score_column = specification["column"]

    analysis_df[score_column] = pd.to_numeric(
        analysis_df[score_column],
        errors="raise",
    )

    score_values = analysis_df[
        score_column
    ].to_numpy(dtype=float)

    if np.isnan(score_values).any():
        raise AssertionError(
            f"Missing values detected in {score_column}."
        )

    if not np.isfinite(score_values).all():
        raise AssertionError(
            f"Nonfinite values detected in {score_column}."
        )

    if (
        (score_values < 0.0)
        | (score_values > 1.0)
    ).any():
        raise AssertionError(
            f"Values outside [0,1] detected in {score_column}."
        )


# --------------------------------------------------------------------------------------------------
# 7. VERIFY STAR-LEVEL ACCOUNTING
# --------------------------------------------------------------------------------------------------

star_accounting = (
    analysis_df
    .groupby(STAR_COLUMN, sort=True)[OUTCOME_COLUMN]
    .agg(
        rows="size",
        events="sum",
    )
    .reset_index()
)

star_accounting["negatives"] = (
    star_accounting["rows"]
    - star_accounting["events"]
)

expected_star_accounting = {
    0: {
        "rows": 36,
        "events": 36,
        "negatives": 0,
    },
    1: {
        "rows": 49_224,
        "events": 4_941,
        "negatives": 44_283,
    },
    2: {
        "rows": 9_223,
        "events": 1_506,
        "negatives": 7_717,
    },
    3: {
        "rows": 8_153,
        "events": 2,
        "negatives": 8_151,
    },
}

for star_level, expected_values in expected_star_accounting.items():

    observed_row = star_accounting.loc[
        star_accounting[STAR_COLUMN] == star_level
    ]

    if len(observed_row) != 1:
        raise AssertionError(
            f"Missing or duplicated accounting row for "
            f"review-star level {star_level}."
        )

    observed_row = observed_row.iloc[0]

    for field_name in ["rows", "events", "negatives"]:

        observed_value = int(
            observed_row[field_name]
        )

        expected_value = expected_values[field_name]

        if observed_value != expected_value:
            raise AssertionError(
                f"Unexpected {field_name} count for star "
                f"{star_level}: {observed_value:,}; "
                f"expected {expected_value:,}"
            )


# --------------------------------------------------------------------------------------------------
# 8. RUN OPTIMIZED EXACT BOOTSTRAP
# --------------------------------------------------------------------------------------------------

rng = np.random.default_rng(RANDOM_SEED)

point_estimate_rows = []
model_interval_rows = []
paired_difference_rows = []

bootstrap_storage = {}

analysis_start_time = time.time()

for star_level in ANALYSIS_STAR_LEVELS:

    stratum_start_time = time.time()

    stratum_df = (
        analysis_df.loc[
            analysis_df[STAR_COLUMN] == star_level
        ]
        .reset_index(drop=True)
    )

    outcomes = stratum_df[
        OUTCOME_COLUMN
    ].to_numpy(dtype=np.int8)

    number_of_rows = len(stratum_df)
    number_of_events = int(outcomes.sum())
    number_of_negatives = int(
        number_of_rows - number_of_events
    )

    stratum_prevalence = float(
        number_of_events / number_of_rows
    )

    if len(np.unique(outcomes)) != 2:
        raise AssertionError(
            f"Star-{star_level} stratum does not contain "
            f"both outcome classes."
        )

    score_arrays = {
        model_key: stratum_df[
            specification["column"]
        ].to_numpy(dtype=np.float64)
        for model_key, specification
        in MODEL_SPECIFICATIONS.items()
    }

    # Build score-group caches once.
    score_group_caches = {
        model_key: construct_score_group_cache(
            scores=score_arrays[model_key],
            outcomes=outcomes,
        )
        for model_key in MODEL_SPECIFICATIONS
    }

    point_metrics = {}

    print(
        f"\nPreparing star-{star_level} stratum: "
        f"{number_of_rows:,} rows, "
        f"{number_of_events:,} events, "
        f"{number_of_negatives:,} negatives"
    )

    # ----------------------------------------------------------------------------------------------
    # Validate fast metric engine against scikit-learn before bootstrapping.
    # ----------------------------------------------------------------------------------------------

    original_count_matrix = np.ones(
        (1, number_of_rows),
        dtype=np.int16,
    )

    original_positive_total = np.array(
        [number_of_events],
        dtype=np.float64,
    )

    for model_key, specification in MODEL_SPECIFICATIONS.items():

        scores = score_arrays[model_key]

        sklearn_auprc = float(
            average_precision_score(
                outcomes,
                scores,
            )
        )

        sklearn_auroc = float(
            roc_auc_score(
                outcomes,
                scores,
            )
        )

        fast_auprc, fast_auroc = (
            calculate_grouped_weighted_metrics(
                cache=score_group_caches[model_key],
                count_matrix=original_count_matrix,
                positive_totals=original_positive_total,
            )
        )

        if not np.isclose(
            fast_auprc[0],
            sklearn_auprc,
            rtol=1e-11,
            atol=1e-12,
        ):
            raise AssertionError(
                f"Fast AUPRC validation failed for "
                f"star {star_level}, model {model_key}.\n"
                f"Scikit-learn: {sklearn_auprc:.15f}\n"
                f"Fast method:  {fast_auprc[0]:.15f}"
            )

        if not np.isclose(
            fast_auroc[0],
            sklearn_auroc,
            rtol=1e-11,
            atol=1e-12,
        ):
            raise AssertionError(
                f"Fast AUROC validation failed for "
                f"star {star_level}, model {model_key}.\n"
                f"Scikit-learn: {sklearn_auroc:.15f}\n"
                f"Fast method:  {fast_auroc[0]:.15f}"
            )

        point_metrics[model_key] = {
            "auprc": sklearn_auprc,
            "auroc": sklearn_auroc,
        }

        point_estimate_rows.append(
            {
                "t0_review_stars": star_level,
                "model_key": model_key,
                "model": specification["display_name"],
                "rows": number_of_rows,
                "events": number_of_events,
                "negatives": number_of_negatives,
                "stratum_prevalence": stratum_prevalence,
                "auprc": sklearn_auprc,
                "auprc_minus_prevalence": (
                    sklearn_auprc
                    - stratum_prevalence
                ),
                "auroc": sklearn_auroc,
                "auroc_minus_0_50": (
                    sklearn_auroc - 0.50
                ),
                "fast_metric_validation": "PASS",
            }
        )

    print(
        "  Fast metric validation against scikit-learn: PASS"
    )

    bootstrap_metrics = {
        model_key: {
            "auprc": np.full(
                N_BOOTSTRAP,
                np.nan,
                dtype=np.float64,
            ),
            "auroc": np.full(
                N_BOOTSTRAP,
                np.nan,
                dtype=np.float64,
            ),
        }
        for model_key in MODEL_SPECIFICATIONS
    }

    bootstrap_prevalence = np.full(
        N_BOOTSTRAP,
        np.nan,
        dtype=np.float64,
    )

    # Uniform multinomial probabilities.
    bootstrap_probabilities = np.full(
        number_of_rows,
        1.0 / number_of_rows,
        dtype=np.float64,
    )

    # Force exact normalization after floating-point construction.
    bootstrap_probabilities[-1] = (
        1.0
        - bootstrap_probabilities[:-1].sum()
    )

    if not np.isclose(
        bootstrap_probabilities.sum(),
        1.0,
        rtol=0.0,
        atol=1e-14,
    ):
        raise AssertionError(
            "Bootstrap probability vector does not sum to one."
        )

    for batch_start in range(
        0,
        N_BOOTSTRAP,
        BOOTSTRAP_BATCH_SIZE,
    ):

        batch_end = min(
            batch_start + BOOTSTRAP_BATCH_SIZE,
            N_BOOTSTRAP,
        )

        current_batch_size = (
            batch_end - batch_start
        )

        # Each row is one exact ordinary bootstrap represented by
        # multinomial multiplicities over the original records.
        bootstrap_counts = rng.multinomial(
            number_of_rows,
            bootstrap_probabilities,
            size=current_batch_size,
        )

        bootstrap_sample_totals = (
            bootstrap_counts.sum(axis=1)
        )

        if not np.all(
            bootstrap_sample_totals
            == number_of_rows
        ):
            raise AssertionError(
                "A bootstrap replicate did not preserve "
                "the required sample size."
            )

        bootstrap_positive_totals = (
            bootstrap_counts @ outcomes
        ).astype(np.float64)

        valid_outcome_mask = (
            (bootstrap_positive_totals > 0.0)
            & (
                bootstrap_positive_totals
                < number_of_rows
            )
        )

        bootstrap_prevalence[
            batch_start:batch_end
        ] = np.where(
            valid_outcome_mask,
            bootstrap_positive_totals
            / number_of_rows,
            np.nan,
        )

        for model_key in MODEL_SPECIFICATIONS:

            batch_auprc, batch_auroc = (
                calculate_grouped_weighted_metrics(
                    cache=score_group_caches[model_key],
                    count_matrix=bootstrap_counts,
                    positive_totals=bootstrap_positive_totals,
                )
            )

            bootstrap_metrics[
                model_key
            ]["auprc"][
                batch_start:batch_end
            ] = batch_auprc

            bootstrap_metrics[
                model_key
            ]["auroc"][
                batch_start:batch_end
            ] = batch_auroc

        completed = batch_end

        if (
            completed % 250 == 0
            or completed == N_BOOTSTRAP
        ):
            elapsed = (
                time.time() - stratum_start_time
            )

            valid_so_far = int(
                np.isfinite(
                    bootstrap_metrics[
                        "full_ges"
                    ]["auprc"][:completed]
                ).sum()
            )

            print(
                f"  Completed {completed:,}/{N_BOOTSTRAP:,} "
                f"replicates | valid {valid_so_far:,} | "
                f"elapsed {elapsed:.1f} seconds"
            )

        del bootstrap_counts
        gc.collect()

    valid_replicate_mask = np.isfinite(
        bootstrap_metrics[
            "full_ges"
        ]["auprc"]
    )

    valid_replicates = int(
        valid_replicate_mask.sum()
    )

    invalid_one_class_replicates = int(
        N_BOOTSTRAP - valid_replicates
    )

    if valid_replicates < MINIMUM_VALID_REPLICATES:
        raise AssertionError(
            f"Star-{star_level} produced only "
            f"{valid_replicates:,} valid bootstrap replicates; "
            f"minimum required is "
            f"{MINIMUM_VALID_REPLICATES:,}."
        )

    # All models must have exactly the same valid replicate positions.
    for model_key in MODEL_SPECIFICATIONS:

        for metric_name in ["auprc", "auroc"]:

            model_valid_mask = np.isfinite(
                bootstrap_metrics[
                    model_key
                ][metric_name]
            )

            if not np.array_equal(
                model_valid_mask,
                valid_replicate_mask,
            ):
                raise AssertionError(
                    f"Paired-bootstrap valid-replicate mismatch "
                    f"for star {star_level}, model {model_key}, "
                    f"metric {metric_name}."
                )

    bootstrap_storage[star_level] = {
        "model_metrics": bootstrap_metrics,
        "bootstrap_prevalence": bootstrap_prevalence,
        "valid_replicate_mask": valid_replicate_mask,
    }

    # ----------------------------------------------------------------------------------------------
    # Model-specific bootstrap intervals
    # ----------------------------------------------------------------------------------------------

    for model_key, specification in MODEL_SPECIFICATIONS.items():

        auprc_values = bootstrap_metrics[
            model_key
        ]["auprc"]

        auroc_values = bootstrap_metrics[
            model_key
        ]["auroc"]

        auprc_minus_prevalence_values = (
            auprc_values
            - bootstrap_prevalence
        )

        auroc_minus_half_values = (
            auroc_values - 0.50
        )

        auprc_lower, auprc_upper = (
            percentile_interval(
                auprc_values
            )
        )

        auroc_lower, auroc_upper = (
            percentile_interval(
                auroc_values
            )
        )

        (
            auprc_null_lower,
            auprc_null_upper,
        ) = percentile_interval(
            auprc_minus_prevalence_values
        )

        (
            auroc_null_lower,
            auroc_null_upper,
        ) = percentile_interval(
            auroc_minus_half_values
        )

        model_interval_rows.append(
            {
                "t0_review_stars": star_level,
                "model_key": model_key,
                "model": specification["display_name"],
                "rows": number_of_rows,
                "events": number_of_events,
                "negatives": number_of_negatives,
                "stratum_prevalence": stratum_prevalence,
                "point_auprc": point_metrics[
                    model_key
                ]["auprc"],
                "auprc_ci_lower": auprc_lower,
                "auprc_ci_upper": auprc_upper,
                "point_auprc_minus_prevalence": (
                    point_metrics[
                        model_key
                    ]["auprc"]
                    - stratum_prevalence
                ),
                "auprc_minus_prevalence_ci_lower": (
                    auprc_null_lower
                ),
                "auprc_minus_prevalence_ci_upper": (
                    auprc_null_upper
                ),
                "auprc_null_status": interval_status(
                    auprc_null_lower,
                    auprc_null_upper,
                    "supported_above_stratum_prevalence",
                    "supported_below_stratum_prevalence",
                ),
                "point_auroc": point_metrics[
                    model_key
                ]["auroc"],
                "auroc_ci_lower": auroc_lower,
                "auroc_ci_upper": auroc_upper,
                "point_auroc_minus_0_50": (
                    point_metrics[
                        model_key
                    ]["auroc"]
                    - 0.50
                ),
                "auroc_minus_0_50_ci_lower": (
                    auroc_null_lower
                ),
                "auroc_minus_0_50_ci_upper": (
                    auroc_null_upper
                ),
                "auroc_null_status": interval_status(
                    auroc_null_lower,
                    auroc_null_upper,
                    "supported_above_0_50",
                    "supported_below_0_50",
                ),
                "attempted_bootstrap_replicates": (
                    N_BOOTSTRAP
                ),
                "valid_bootstrap_replicates": (
                    valid_replicates
                ),
                "invalid_one_class_replicates": (
                    invalid_one_class_replicates
                ),
                "sparse_event_or_negative_flag": bool(
                    min(
                        number_of_events,
                        number_of_negatives,
                    ) < 20
                ),
            }
        )

    # ----------------------------------------------------------------------------------------------
    # Paired Full-GES-minus-comparator inference
    # ----------------------------------------------------------------------------------------------

    for comparator_key in PAIRED_COMPARATORS:

        comparator_display_name = (
            MODEL_SPECIFICATIONS[
                comparator_key
            ]["display_name"]
        )

        for metric_name in ["auprc", "auroc"]:

            full_values = bootstrap_metrics[
                "full_ges"
            ][metric_name]

            comparator_values = bootstrap_metrics[
                comparator_key
            ][metric_name]

            paired_differences = (
                full_values
                - comparator_values
            )

            (
                difference_lower,
                difference_upper,
            ) = percentile_interval(
                paired_differences
            )

            point_difference = (
                point_metrics[
                    "full_ges"
                ][metric_name]
                - point_metrics[
                    comparator_key
                ][metric_name]
            )

            finite_differences = (
                paired_differences[
                    np.isfinite(
                        paired_differences
                    )
                ]
            )

            paired_difference_rows.append(
                {
                    "metric": metric_name.upper(),
                    "t0_review_stars": star_level,
                    "comparison": (
                        f"Full GES minus "
                        f"{comparator_display_name}"
                    ),
                    "comparator_key": comparator_key,
                    "rows": number_of_rows,
                    "events": number_of_events,
                    "negatives": number_of_negatives,
                    "point_difference": point_difference,
                    "difference_ci_lower": difference_lower,
                    "difference_ci_upper": difference_upper,
                    "paired_interval_status": interval_status(
                        difference_lower,
                        difference_upper,
                        "full_ges_supported_higher",
                        "full_ges_supported_lower",
                    ),
                    "bootstrap_probability_full_greater": float(
                        np.mean(
                            finite_differences > 0.0
                        )
                    ),
                    "bootstrap_probability_equal": float(
                        np.mean(
                            finite_differences == 0.0
                        )
                    ),
                    "bootstrap_sign_p_value": (
                        bootstrap_sign_pvalue(
                            finite_differences
                        )
                    ),
                    "attempted_bootstrap_replicates": (
                        N_BOOTSTRAP
                    ),
                    "valid_bootstrap_replicates": (
                        valid_replicates
                    ),
                    "invalid_one_class_replicates": (
                        invalid_one_class_replicates
                    ),
                    "sparse_event_or_negative_flag": bool(
                        min(
                            number_of_events,
                            number_of_negatives,
                        ) < 20
                    ),
                }
            )

    star_elapsed = (
        time.time() - stratum_start_time
    )

    print(
        f"  Star-{star_level} completed: "
        f"{valid_replicates:,} valid, "
        f"{invalid_one_class_replicates:,} invalid "
        f"one-class replicates, "
        f"{star_elapsed:.1f} seconds"
    )

    del stratum_df
    del score_arrays
    del score_group_caches
    gc.collect()


# --------------------------------------------------------------------------------------------------
# 9. CREATE RESULT TABLES
# --------------------------------------------------------------------------------------------------

same_star_point_estimates = pd.DataFrame(
    point_estimate_rows
)

same_star_model_intervals = pd.DataFrame(
    model_interval_rows
)

same_star_paired_differences = pd.DataFrame(
    paired_difference_rows
)


# --------------------------------------------------------------------------------------------------
# 10. HOLM CORRECTION WITHIN AUPRC AND AUROC FAMILIES
# --------------------------------------------------------------------------------------------------

same_star_paired_differences[
    "holm_adjusted_bootstrap_sign_p"
] = np.nan

for metric_name in ["AUPRC", "AUROC"]:

    family_mask = (
        same_star_paired_differences[
            "metric"
        ] == metric_name
    )

    family_p_values = (
        same_star_paired_differences.loc[
            family_mask,
            "bootstrap_sign_p_value",
        ]
        .to_numpy(dtype=float)
    )

    adjusted_values = holm_adjust(
        family_p_values
    )

    same_star_paired_differences.loc[
        family_mask,
        "holm_adjusted_bootstrap_sign_p",
    ] = adjusted_values

same_star_paired_differences[
    "holm_supported_at_0_05"
] = (
    same_star_paired_differences[
        "holm_adjusted_bootstrap_sign_p"
    ] < 0.05
)


# --------------------------------------------------------------------------------------------------
# 11. COMPLETENESS CHECKS
# --------------------------------------------------------------------------------------------------

expected_model_rows = (
    len(ANALYSIS_STAR_LEVELS)
    * len(MODEL_SPECIFICATIONS)
)

if len(same_star_model_intervals) != expected_model_rows:
    raise AssertionError(
        f"Unexpected model-interval row count: "
        f"{len(same_star_model_intervals)}; "
        f"expected {expected_model_rows}."
    )

expected_paired_rows = (
    len(ANALYSIS_STAR_LEVELS)
    * len(PAIRED_COMPARATORS)
    * 2
)

if len(same_star_paired_differences) != expected_paired_rows:
    raise AssertionError(
        f"Unexpected paired-comparison row count: "
        f"{len(same_star_paired_differences)}; "
        f"expected {expected_paired_rows}."
    )

for metric_name in ["AUPRC", "AUROC"]:

    family_rows = same_star_paired_differences.loc[
        same_star_paired_differences[
            "metric"
        ] == metric_name
    ]

    if len(family_rows) != 6:
        raise AssertionError(
            f"{metric_name} Holm family contains "
            f"{len(family_rows)} rows; expected 6."
        )

if same_star_paired_differences[
    "holm_adjusted_bootstrap_sign_p"
].isna().any():
    raise AssertionError(
        "Missing Holm-adjusted bootstrap sign probability."
    )

if (
    same_star_model_intervals[
        "valid_bootstrap_replicates"
    ] < MINIMUM_VALID_REPLICATES
).any():
    raise AssertionError(
        "At least one stratum has too few valid "
        "bootstrap replicates."
    )


# --------------------------------------------------------------------------------------------------
# 12. COMPACT DISPLAY TABLES
# --------------------------------------------------------------------------------------------------

model_display_columns = [
    "t0_review_stars",
    "model",
    "rows",
    "events",
    "negatives",
    "stratum_prevalence",
    "point_auprc",
    "auprc_ci_lower",
    "auprc_ci_upper",
    "point_auprc_minus_prevalence",
    "auprc_minus_prevalence_ci_lower",
    "auprc_minus_prevalence_ci_upper",
    "auprc_null_status",
    "point_auroc",
    "auroc_ci_lower",
    "auroc_ci_upper",
    "point_auroc_minus_0_50",
    "auroc_minus_0_50_ci_lower",
    "auroc_minus_0_50_ci_upper",
    "auroc_null_status",
    "valid_bootstrap_replicates",
    "invalid_one_class_replicates",
    "sparse_event_or_negative_flag",
]

paired_display_columns = [
    "metric",
    "t0_review_stars",
    "comparison",
    "rows",
    "events",
    "negatives",
    "point_difference",
    "difference_ci_lower",
    "difference_ci_upper",
    "paired_interval_status",
    "bootstrap_probability_full_greater",
    "bootstrap_sign_p_value",
    "holm_adjusted_bootstrap_sign_p",
    "holm_supported_at_0_05",
    "valid_bootstrap_replicates",
    "invalid_one_class_replicates",
    "sparse_event_or_negative_flag",
]

same_star_model_display = (
    same_star_model_intervals[
        model_display_columns
    ].copy()
)

same_star_paired_display = (
    same_star_paired_differences[
        paired_display_columns
    ].copy()
)


# --------------------------------------------------------------------------------------------------
# 13. PRINT COMPLETE LOCKED INFERENCE RECORD
# --------------------------------------------------------------------------------------------------

analysis_elapsed = (
    time.time() - analysis_start_time
)

separator = "=" * 170
subseparator = "-" * 170

print("\n" + separator)
print(
    "STAGE 6C STEP 3B — CELL 6C-3B2 — "
    "OPTIMIZED LOCKED SAME-STAR PAIRED BOOTSTRAP INFERENCE"
)
print(separator)

print("\nFROZEN INPUT VERIFICATION")
print(subseparator)

print(
    f"Stage 6B evaluable-cohort SHA-256 : PASS "
    f"({observed_sha256})"
)

print(
    f"Parquet dimensions                : PASS "
    f"({metadata.num_rows:,} × {metadata.num_columns})"
)

print(
    f"Unique evaluable RCV keys         : PASS "
    f"({analysis_df['rcv_accession'].nunique():,})"
)

print(
    f"Primary instability events        : PASS "
    f"({event_count:,})"
)

print(
    f"Primary instability negatives     : PASS "
    f"({negative_count:,})"
)

print(
    f"Overall event prevalence          : "
    f"{analysis_df[OUTCOME_COLUMN].mean():.8f} "
    f"({analysis_df[OUTCOME_COLUMN].mean():.6%})"
)

print(
    f"Bootstrap attempts per stratum    : "
    f"{N_BOOTSTRAP:,}"
)

print(
    f"Bootstrap batch size              : "
    f"{BOOTSTRAP_BATCH_SIZE}"
)

print(
    f"Bootstrap random seed             : "
    f"{RANDOM_SEED}"
)

print(
    "Bootstrap representation         : "
    "Exact multinomial row multiplicities"
)

print(
    "Paired resampling                 : "
    "Identical bootstrap samples across all models"
)

print(
    "Tie handling                      : "
    "Exact grouped weighted AUPRC/AUROC"
)

print(
    "Fast metric validation            : "
    "Matched scikit-learn point estimates in every stratum/model"
)

print(
    f"Percentile interval               : "
    f"95% ({CI_LOWER_QUANTILE:.3f}, "
    f"{CI_UPPER_QUANTILE:.3f})"
)

print(
    "Holm families                     : "
    "Six AUPRC and six AUROC paired comparisons"
)

print("Score direction changed           : No")
print("Threshold or weight optimized     : No")
print("Recalibration performed           : No")
print("Scientific artifact written       : No")

print(
    f"Total analysis runtime            : "
    f"{analysis_elapsed:.1f} seconds"
)

print("\nSAME-STAR MODEL-SPECIFIC BOOTSTRAP INTERVALS")
print(subseparator)

with pd.option_context(
    "display.max_columns", None,
    "display.width", 380,
    "display.float_format",
    lambda value: f"{value:.8f}",
):
    print(
        same_star_model_display.to_string(
            index=False
        )
    )

print("\nPAIRED FULL-GES-MINUS-COMPARATOR INFERENCE")
print(subseparator)

with pd.option_context(
    "display.max_columns", None,
    "display.width", 360,
    "display.float_format",
    lambda value: f"{value:.8f}",
):
    print(
        same_star_paired_display.to_string(
            index=False
        )
    )

print("\nNON-ESTIMABLE AND SPARSE STRATA")
print(subseparator)

print(
    "0-star stratum : 36 events and 0 negatives. "
    "Retained descriptively; discrimination metrics are not estimable."
)

print(
    "3-star stratum : 2 events and 8,151 negatives. "
    "Bootstrap samples containing only one outcome class were excluded "
    "and counted explicitly."
)

print(
    "The 3-star intervals remain extremely sparse-event estimates and "
    "must not be interpreted as stable population-level performance."
)

print("\nCELL DECISION")
print(subseparator)

print(
    "PASS_STAGE6C_OPTIMIZED_SAME_STAR_BOOTSTRAP_INFERENCE_COMPLETE"
)

print(
    "Two thousand exact ordinary row-bootstrap attempts were completed "
    "within each estimable baseline review-star stratum."
)

print(
    "Model-specific percentile intervals, null-reference comparisons, "
    "paired Full-GES-minus-comparator intervals, bootstrap sign "
    "probabilities, and Holm-adjusted probabilities were calculated."
)

print(
    "No score, outcome, review-star assignment, cohort membership, "
    "threshold, weight, model, or frozen scientific artifact was modified."
)


Preparing star-1 stratum: 49,224 rows, 4,941 events, 44,283 negatives
  Fast metric validation against scikit-learn: PASS
  Completed 250/2,000 replicates | valid 250 | elapsed 3.5 seconds
  Completed 500/2,000 replicates | valid 500 | elapsed 6.8 seconds
  Completed 750/2,000 replicates | valid 750 | elapsed 9.9 seconds
  Completed 1,000/2,000 replicates | valid 1,000 | elapsed 14.5 seconds
  Completed 1,250/2,000 replicates | valid 1,250 | elapsed 18.8 seconds
  Completed 1,500/2,000 replicates | valid 1,500 | elapsed 21.9 seconds
  Completed 1,750/2,000 replicates | valid 1,750 | elapsed 25.0 seconds
  Completed 2,000/2,000 replicates | valid 2,000 | elapsed 30.3 seconds
  Star-1 completed: 2,000 valid, 0 invalid one-class replicates, 30.7 seconds

Preparing star-2 stratum: 9,223 rows, 1,506 events, 7,717 negatives
  Fast metric validation against scikit-learn: PASS
  Completed 250/2,000 replicates | valid 250 | elapsed 1.9 seconds
  Completed 500/2,000 replicates | valid 500 | ela

In [28]:
# ==================================================================================================
# STAGE 6C STEP 3C — CELL 6C-3C0
# GENE-LEVEL ANALYSIS PREFLIGHT AND SUBGROUP INVENTORY
#
# Purpose:
#   1. Freshly verify the frozen Stage 6B primary-evaluable cohort.
#   2. Identify and validate the preserved T0 target-gene field.
#   3. Inventory rows, events, negatives, prevalence, and score variation by gene.
#   4. Separate BRCA1, BRCA2, and MLH1 as primary analyses and EGFR as exploratory.
#   5. Determine which gene strata support AUPRC/AUROC analysis.
#
# This cell does NOT:
#   - calculate AUPRC or AUROC;
#   - run bootstrap inference;
#   - optimize thresholds or weights;
#   - recalibrate scores;
#   - write scientific artifacts.
# ==================================================================================================

from pathlib import Path
import hashlib
import json
import re

import numpy as np
import pandas as pd
import pyarrow.parquet as pq


# --------------------------------------------------------------------------------------------------
# 1. FROZEN INPUT PATHS AND EXPECTED IDENTITY
# --------------------------------------------------------------------------------------------------

STAGE6_DIR = Path(
    "/content/drive/MyDrive/GES_RAG_Temporal_Study/"
    "data_processed/stage6_temporal_validation"
)

EVALUABLE_PARQUET = (
    STAGE6_DIR / "stage6b_locked_primary_evaluable_cohort_v1.parquet"
)

EVALUABLE_SIDECAR = Path(
    str(EVALUABLE_PARQUET) + ".sha256"
)

EXPECTED_EVALUABLE_SHA256 = (
    "c6ad50dfc376a746e1830d000dba975101d88763fd61b8d1983f6300edb6e038"
)

EXPECTED_ROWS = 66_636
EXPECTED_COLUMNS = 79
EXPECTED_EVENTS = 6_485
EXPECTED_NEGATIVES = 60_151

OUTCOME_COLUMN = "primary_future_instability"

PRIMARY_GENES = [
    "BRCA1",
    "BRCA2",
    "MLH1",
]

EXPLORATORY_GENES = [
    "EGFR",
]

ALLOWED_GENES = set(
    PRIMARY_GENES + EXPLORATORY_GENES
)

SCORE_COLUMNS = {
    "full_ges": "full_ges_instability_risk_t0",
    "no_star_ges": "no_star_ges_instability_risk_t0",
    "review_stars": "review_stars_instability_risk",
    "combined_metadata": "combined_metadata_instability_risk",
}


# --------------------------------------------------------------------------------------------------
# 2. HELPERS
# --------------------------------------------------------------------------------------------------

def sha256_file(
    path: Path,
    chunk_size: int = 1024 * 1024,
) -> str:

    digest = hashlib.sha256()

    with path.open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


def read_sidecar_hash(path: Path) -> str:

    text = path.read_text(
        encoding="utf-8"
    ).strip()

    matches = re.findall(
        r"\b[a-fA-F0-9]{64}\b",
        text,
    )

    if not matches:
        raise ValueError(
            f"No SHA-256 value was found in sidecar:\n{path}"
        )

    return matches[0].lower()


def normalize_single_gene(value) -> str:
    """
    Normalize a scalar gene field or a serialized one-element gene list.

    Accepted examples:
        BRCA1
        "BRCA1"
        ["BRCA1"]
        "['BRCA1']"
        '["BRCA1"]'
    """

    if value is None:
        return ""

    if isinstance(value, float) and np.isnan(value):
        return ""

    if isinstance(value, (list, tuple, set, np.ndarray)):
        genes = [
            str(item).strip().upper()
            for item in value
            if str(item).strip()
        ]

    else:
        text = str(value).strip()

        if not text:
            return ""

        genes = None

        if (
            (text.startswith("[") and text.endswith("]"))
            or
            (text.startswith("{") and text.endswith("}"))
        ):
            try:
                parsed = json.loads(text)

                if isinstance(parsed, list):
                    genes = [
                        str(item).strip().upper()
                        for item in parsed
                        if str(item).strip()
                    ]

                elif isinstance(parsed, dict):
                    genes = [
                        str(item).strip().upper()
                        for item in parsed.values()
                        if str(item).strip()
                    ]

            except Exception:
                # Handle Python-style single-quoted list strings.
                cleaned = (
                    text
                    .strip("[]{}")
                    .replace('"', "")
                    .replace("'", "")
                )

                genes = [
                    item.strip().upper()
                    for item in cleaned.split(",")
                    if item.strip()
                ]

        if genes is None:
            genes = [text.upper()]

    genes = list(dict.fromkeys(genes))

    if len(genes) != 1:
        raise ValueError(
            f"Expected exactly one target gene but found: {genes}"
        )

    return genes[0]


# --------------------------------------------------------------------------------------------------
# 3. FRESH CRYPTOGRAPHIC AND PARQUET-METADATA VERIFICATION
# --------------------------------------------------------------------------------------------------

if not EVALUABLE_PARQUET.exists():
    raise FileNotFoundError(
        f"Missing frozen primary-evaluable cohort:\n"
        f"{EVALUABLE_PARQUET}"
    )

if not EVALUABLE_SIDECAR.exists():
    raise FileNotFoundError(
        f"Missing frozen evaluable-cohort sidecar:\n"
        f"{EVALUABLE_SIDECAR}"
    )

observed_sha256 = sha256_file(
    EVALUABLE_PARQUET
)

sidecar_sha256 = read_sidecar_hash(
    EVALUABLE_SIDECAR
)

if observed_sha256 != EXPECTED_EVALUABLE_SHA256:
    raise AssertionError(
        "Frozen Stage 6B evaluable-cohort SHA-256 mismatch.\n"
        f"Expected: {EXPECTED_EVALUABLE_SHA256}\n"
        f"Observed: {observed_sha256}"
    )

if sidecar_sha256 != observed_sha256:
    raise AssertionError(
        "Frozen Stage 6B evaluable-cohort sidecar mismatch.\n"
        f"Calculated: {observed_sha256}\n"
        f"Sidecar:    {sidecar_sha256}"
    )

parquet_file = pq.ParquetFile(
    EVALUABLE_PARQUET
)

metadata = parquet_file.metadata
schema_columns = parquet_file.schema_arrow.names

if metadata.num_rows != EXPECTED_ROWS:
    raise AssertionError(
        f"Unexpected Parquet row count: "
        f"{metadata.num_rows:,}; expected {EXPECTED_ROWS:,}"
    )

if metadata.num_columns != EXPECTED_COLUMNS:
    raise AssertionError(
        f"Unexpected Parquet column count: "
        f"{metadata.num_columns}; expected {EXPECTED_COLUMNS}"
    )


# --------------------------------------------------------------------------------------------------
# 4. IDENTIFY THE PRESERVED T0 TARGET-GENE COLUMN
# --------------------------------------------------------------------------------------------------

gene_column_priority = [
    "t0_target_gene",
    "target_gene",
    "t0_gene_symbol",
    "gene_symbol",
    "t0_gene",
    "gene",
    "t0_target_genes_json",
    "target_genes_json",
]

gene_column = next(
    (
        column
        for column in gene_column_priority
        if column in schema_columns
    ),
    None,
)

gene_related_columns = [
    column
    for column in schema_columns
    if "gene" in column.lower()
]

if gene_column is None:

    print(
        "Gene-related columns found in the frozen schema:"
    )

    for column in gene_related_columns:
        print(f"  - {column}")

    raise KeyError(
        "The frozen T0 target-gene column could not be "
        "selected automatically. Review the candidate "
        "columns printed above."
    )


# --------------------------------------------------------------------------------------------------
# 5. LOAD ONLY COLUMNS REQUIRED FOR GENE-LEVEL PREFLIGHT
# --------------------------------------------------------------------------------------------------

required_columns = [
    "rcv_accession",
    "t0_row_order",
    OUTCOME_COLUMN,
    gene_column,
] + list(SCORE_COLUMNS.values())

missing_columns = [
    column
    for column in required_columns
    if column not in schema_columns
]

if missing_columns:
    raise KeyError(
        "Required gene-analysis columns are missing:\n"
        + "\n".join(
            f"  - {column}"
            for column in missing_columns
        )
    )

gene_df = pd.read_parquet(
    EVALUABLE_PARQUET,
    columns=required_columns,
).copy()


# --------------------------------------------------------------------------------------------------
# 6. LOCKED COHORT STRUCTURAL VALIDATION
# --------------------------------------------------------------------------------------------------

if gene_df.shape[0] != EXPECTED_ROWS:
    raise AssertionError(
        f"Loaded dataframe contains "
        f"{gene_df.shape[0]:,} rows; "
        f"expected {EXPECTED_ROWS:,}"
    )

if gene_df["rcv_accession"].isna().any():
    raise AssertionError(
        "Missing RCV accession detected."
    )

if (
    gene_df["rcv_accession"]
    .astype(str)
    .str.strip()
    .eq("")
    .any()
):
    raise AssertionError(
        "Blank RCV accession detected."
    )

if (
    gene_df["rcv_accession"]
    .nunique(dropna=False)
    != EXPECTED_ROWS
):
    raise AssertionError(
        "RCV accessions are not unique."
    )

row_order = pd.to_numeric(
    gene_df["t0_row_order"],
    errors="raise",
).to_numpy()

if len(np.unique(row_order)) != EXPECTED_ROWS:
    raise AssertionError(
        "t0_row_order is not unique."
    )

if not np.all(np.diff(row_order) > 0):
    raise AssertionError(
        "Frozen evaluable cohort is not in "
        "increasing T0 row order."
    )


# --------------------------------------------------------------------------------------------------
# 7. OUTCOME VALIDATION
# --------------------------------------------------------------------------------------------------

gene_df[OUTCOME_COLUMN] = pd.to_numeric(
    gene_df[OUTCOME_COLUMN],
    errors="raise",
).astype(int)

if not set(
    gene_df[OUTCOME_COLUMN].unique()
).issubset({0, 1}):
    raise AssertionError(
        "Primary future-instability outcome is not binary."
    )

event_count = int(
    gene_df[OUTCOME_COLUMN].sum()
)

negative_count = int(
    (gene_df[OUTCOME_COLUMN] == 0).sum()
)

if event_count != EXPECTED_EVENTS:
    raise AssertionError(
        f"Unexpected event count: "
        f"{event_count:,}; expected {EXPECTED_EVENTS:,}"
    )

if negative_count != EXPECTED_NEGATIVES:
    raise AssertionError(
        f"Unexpected negative count: "
        f"{negative_count:,}; expected {EXPECTED_NEGATIVES:,}"
    )


# --------------------------------------------------------------------------------------------------
# 8. NORMALIZE AND VALIDATE TARGET GENES
# --------------------------------------------------------------------------------------------------

try:
    gene_df["_normalized_gene"] = (
        gene_df[gene_column]
        .map(normalize_single_gene)
    )

except Exception as exc:
    raise ValueError(
        f"Target-gene normalization failed for "
        f"column {gene_column}: {exc}"
    ) from exc

if gene_df["_normalized_gene"].eq("").any():
    missing_gene_rows = int(
        gene_df["_normalized_gene"].eq("").sum()
    )

    raise AssertionError(
        f"Missing normalized target gene detected in "
        f"{missing_gene_rows:,} rows."
    )

observed_genes = sorted(
    gene_df["_normalized_gene"]
    .unique()
    .tolist()
)

unexpected_genes = sorted(
    set(observed_genes) - ALLOWED_GENES
)

if unexpected_genes:
    raise AssertionError(
        f"Unexpected genes detected: {unexpected_genes}"
    )

missing_expected_genes = sorted(
    ALLOWED_GENES - set(observed_genes)
)

if missing_expected_genes:
    raise AssertionError(
        f"Expected study genes are absent: "
        f"{missing_expected_genes}"
    )


# --------------------------------------------------------------------------------------------------
# 9. VALIDATE FROZEN SCORE COLUMNS
# --------------------------------------------------------------------------------------------------

for model_key, score_column in SCORE_COLUMNS.items():

    gene_df[score_column] = pd.to_numeric(
        gene_df[score_column],
        errors="raise",
    )

    values = gene_df[
        score_column
    ].to_numpy(dtype=float)

    if np.isnan(values).any():
        raise AssertionError(
            f"Missing score values detected in "
            f"{score_column}."
        )

    if not np.isfinite(values).all():
        raise AssertionError(
            f"Nonfinite score values detected in "
            f"{score_column}."
        )

    if (
        (values < 0.0)
        | (values > 1.0)
    ).any():
        raise AssertionError(
            f"Score outside [0,1] detected in "
            f"{score_column}."
        )


# --------------------------------------------------------------------------------------------------
# 10. CONSTRUCT GENE-LEVEL SUBGROUP INVENTORY
# --------------------------------------------------------------------------------------------------

inventory_rows = []

gene_display_order = [
    "BRCA1",
    "BRCA2",
    "MLH1",
    "EGFR",
]

for gene in gene_display_order:

    stratum = gene_df.loc[
        gene_df["_normalized_gene"] == gene
    ].copy()

    outcomes = stratum[
        OUTCOME_COLUMN
    ].to_numpy(dtype=int)

    rows = int(len(stratum))
    events = int(outcomes.sum())
    negatives = int(
        rows - events
    )

    prevalence = (
        float(events / rows)
        if rows > 0
        else np.nan
    )

    both_classes_present = bool(
        len(np.unique(outcomes)) == 2
    )

    inventory_row = {
        "gene": gene,
        "analysis_role": (
            "primary"
            if gene in PRIMARY_GENES
            else "exploratory"
        ),
        "rows": rows,
        "events": events,
        "negatives": negatives,
        "event_prevalence": prevalence,
        "both_outcome_classes_present": (
            both_classes_present
        ),
    }

    for model_key, score_column in SCORE_COLUMNS.items():

        unique_score_values = int(
            stratum[score_column].nunique(
                dropna=False
            )
        )

        inventory_row[
            f"{model_key}_unique_score_values"
        ] = unique_score_values

        inventory_row[
            f"{model_key}_discrimination_estimable"
        ] = bool(
            both_classes_present
            and unique_score_values >= 2
        )

    inventory_row[
        "sparse_event_or_negative_flag"
    ] = bool(
        min(events, negatives) < 20
    )

    inventory_rows.append(
        inventory_row
    )

gene_subgroup_inventory = pd.DataFrame(
    inventory_rows
)


# --------------------------------------------------------------------------------------------------
# 11. ACCOUNTING AND ESTIMABILITY CHECKS
# --------------------------------------------------------------------------------------------------

if int(
    gene_subgroup_inventory["rows"].sum()
) != EXPECTED_ROWS:
    raise AssertionError(
        "Gene-stratum row counts do not reconcile "
        "with the frozen evaluable cohort."
    )

if int(
    gene_subgroup_inventory["events"].sum()
) != EXPECTED_EVENTS:
    raise AssertionError(
        "Gene-stratum event counts do not reconcile."
    )

if int(
    gene_subgroup_inventory["negatives"].sum()
) != EXPECTED_NEGATIVES:
    raise AssertionError(
        "Gene-stratum negative counts do not reconcile."
    )

nonestimable_full_ges = (
    gene_subgroup_inventory.loc[
        ~gene_subgroup_inventory[
            "full_ges_discrimination_estimable"
        ]
    ]
)

estimable_genes = (
    gene_subgroup_inventory.loc[
        gene_subgroup_inventory[
            "full_ges_discrimination_estimable"
        ],
        "gene",
    ]
    .tolist()
)


# --------------------------------------------------------------------------------------------------
# 12. PRINT COMPLETE PREFLIGHT RECORD
# --------------------------------------------------------------------------------------------------

separator = "=" * 156
subseparator = "-" * 156

print("\n" + separator)

print(
    "STAGE 6C STEP 3C — CELL 6C-3C0 — "
    "GENE-LEVEL ANALYSIS PREFLIGHT AND SUBGROUP INVENTORY"
)

print(separator)

print("\nFROZEN INPUT VERIFICATION")
print(subseparator)

print(
    f"Stage 6B evaluable-cohort SHA-256 : PASS "
    f"({observed_sha256})"
)

print(
    f"Parquet dimensions                : PASS "
    f"({metadata.num_rows:,} × {metadata.num_columns})"
)

print(
    f"Unique evaluable RCV keys         : PASS "
    f"({gene_df['rcv_accession'].nunique():,})"
)

print(
    f"Primary instability events        : PASS "
    f"({event_count:,})"
)

print(
    f"Primary instability negatives     : PASS "
    f"({negative_count:,})"
)

print(
    f"Overall event prevalence          : "
    f"{gene_df[OUTCOME_COLUMN].mean():.8f} "
    f"({gene_df[OUTCOME_COLUMN].mean():.6%})"
)

print(
    f"Selected frozen gene column       : "
    f"{gene_column}"
)

print(
    f"Observed normalized genes         : "
    f"{observed_genes}"
)

print(
    f"Primary genes                     : "
    f"{PRIMARY_GENES}"
)

print(
    f"Exploratory genes                 : "
    f"{EXPLORATORY_GENES}"
)

print("Performance metric calculated     : No")
print("Bootstrap inference performed     : No")
print("Score direction changed           : No")
print("Threshold or weight optimized     : No")
print("Scientific artifact written       : No")


print("\nGENE-RELATED FROZEN COLUMNS")
print(subseparator)

for column in gene_related_columns:

    marker = (
        "  <-- selected target-gene field"
        if column == gene_column
        else ""
    )

    print(f"{column}{marker}")


print("\nGENE-LEVEL SUBGROUP INVENTORY")
print(subseparator)

with pd.option_context(
    "display.max_columns", None,
    "display.width", 340,
    "display.float_format",
    lambda value: f"{value:.8f}",
):
    print(
        gene_subgroup_inventory.to_string(
            index=False
        )
    )


print("\nACCOUNTING RECONCILIATION")
print(subseparator)

print(
    f"Gene-stratum rows sum             : "
    f"{gene_subgroup_inventory['rows'].sum():,}"
)

print(
    f"Gene-stratum events sum           : "
    f"{gene_subgroup_inventory['events'].sum():,}"
)

print(
    f"Gene-stratum negatives sum        : "
    f"{gene_subgroup_inventory['negatives'].sum():,}"
)

print(
    f"Full-GES estimable genes          : "
    f"{estimable_genes}"
)


print("\nCELL DECISION")
print(subseparator)

if len(nonestimable_full_ges) == 0:

    print(
        "PASS_STAGE6C_GENE_LEVEL_ANALYSIS_PREFLIGHT_COMPLETE"
    )

    print(
        "All four prespecified gene strata contain both "
        "outcome classes and sufficient frozen Full-GES "
        "score variation for locked discrimination analysis."
    )

else:

    print(
        "PASS_WITH_RESTRICTED_GENES_STAGE6C_"
        "GENE_LEVEL_ANALYSIS_PREFLIGHT_COMPLETE"
    )

    print(
        "One or more gene strata cannot support both "
        "AUPRC and AUROC and must be retained descriptively."
    )

    print("\nNon-estimable Full-GES gene strata:")

    print(
        nonestimable_full_ges.to_string(
            index=False
        )
    )

print(
    "\nNo score, outcome, gene assignment, cohort membership, "
    "threshold, weight, frozen model, or scientific artifact "
    "was modified."
)


STAGE 6C STEP 3C — CELL 6C-3C0 — GENE-LEVEL ANALYSIS PREFLIGHT AND SUBGROUP INVENTORY

FROZEN INPUT VERIFICATION
------------------------------------------------------------------------------------------------------------------------------------------------------------
Stage 6B evaluable-cohort SHA-256 : PASS (c6ad50dfc376a746e1830d000dba975101d88763fd61b8d1983f6300edb6e038)
Parquet dimensions                : PASS (66,636 × 79)
Unique evaluable RCV keys         : PASS (66,636)
Primary instability events        : PASS (6,485)
Primary instability negatives     : PASS (60,151)
Overall event prevalence          : 0.09731977 (9.731977%)
Selected frozen gene column       : target_gene
Observed normalized genes         : ['BRCA1', 'BRCA2', 'EGFR', 'MLH1']
Primary genes                     : ['BRCA1', 'BRCA2', 'MLH1']
Exploratory genes                 : ['EGFR']
Performance metric calculated     : No
Bootstrap inference performed     : No
Score direction changed           : No
Threshold or w

In [29]:
# ==================================================================================================
# STAGE 6C STEP 3C — CELL 6C-3C1
# LOCKED GENE-LEVEL DISCRIMINATION POINT ESTIMATES
#
# Purpose:
#   1. Calculate locked within-gene AUPRC and AUROC point estimates.
#   2. Evaluate:
#         - Full GES
#         - No-star GES
#         - Review-stars baseline
#         - Combined-metadata baseline
#   3. Calculate AUPRC lift over gene-specific prevalence.
#   4. Calculate Full-GES-minus-comparator point differences.
#   5. Preserve EGFR as exploratory.
#
# This cell does NOT:
#   - run bootstrap inference;
#   - optimize thresholds or weights;
#   - recalibrate scores;
#   - modify frozen inputs;
#   - write scientific artifacts.
# ==================================================================================================

from pathlib import Path
import hashlib
import re

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

from sklearn.metrics import average_precision_score, roc_auc_score


# --------------------------------------------------------------------------------------------------
# 1. FROZEN INPUT IDENTITY
# --------------------------------------------------------------------------------------------------

STAGE6_DIR = Path(
    "/content/drive/MyDrive/GES_RAG_Temporal_Study/"
    "data_processed/stage6_temporal_validation"
)

EVALUABLE_PARQUET = (
    STAGE6_DIR / "stage6b_locked_primary_evaluable_cohort_v1.parquet"
)

EVALUABLE_SIDECAR = Path(
    str(EVALUABLE_PARQUET) + ".sha256"
)

EXPECTED_EVALUABLE_SHA256 = (
    "c6ad50dfc376a746e1830d000dba975101d88763fd61b8d1983f6300edb6e038"
)

EXPECTED_ROWS = 66_636
EXPECTED_COLUMNS = 79
EXPECTED_EVENTS = 6_485
EXPECTED_NEGATIVES = 60_151

OUTCOME_COLUMN = "primary_future_instability"
GENE_COLUMN = "target_gene"

PRIMARY_GENES = [
    "BRCA1",
    "BRCA2",
    "MLH1",
]

EXPLORATORY_GENES = [
    "EGFR",
]

GENE_DISPLAY_ORDER = [
    "BRCA1",
    "BRCA2",
    "MLH1",
    "EGFR",
]

MODEL_SPECIFICATIONS = {
    "full_ges": {
        "column": "full_ges_instability_risk_t0",
        "display_name": "Full GES",
    },
    "no_star_ges": {
        "column": "no_star_ges_instability_risk_t0",
        "display_name": "No-star GES",
    },
    "review_stars": {
        "column": "review_stars_instability_risk",
        "display_name": "Review stars",
    },
    "combined_metadata": {
        "column": "combined_metadata_instability_risk",
        "display_name": "Combined metadata",
    },
}

COMPARATOR_KEYS = [
    "no_star_ges",
    "review_stars",
    "combined_metadata",
]

EXPECTED_GENE_ACCOUNTING = {
    "BRCA1": {
        "rows": 21_594,
        "events": 2_023,
        "negatives": 19_571,
    },
    "BRCA2": {
        "rows": 34_152,
        "events": 3_960,
        "negatives": 30_192,
    },
    "MLH1": {
        "rows": 8_701,
        "events": 425,
        "negatives": 8_276,
    },
    "EGFR": {
        "rows": 2_189,
        "events": 77,
        "negatives": 2_112,
    },
}


# --------------------------------------------------------------------------------------------------
# 2. HELPERS
# --------------------------------------------------------------------------------------------------

def sha256_file(
    path: Path,
    chunk_size: int = 1024 * 1024,
) -> str:

    digest = hashlib.sha256()

    with path.open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


def read_sidecar_hash(path: Path) -> str:

    text = path.read_text(
        encoding="utf-8"
    ).strip()

    matches = re.findall(
        r"\b[a-fA-F0-9]{64}\b",
        text,
    )

    if not matches:
        raise ValueError(
            f"No SHA-256 value found in sidecar:\n{path}"
        )

    return matches[0].lower()


def normalize_gene(value) -> str:

    gene = str(value).strip().upper()

    if not gene:
        raise ValueError(
            "Blank target-gene value detected."
        )

    return gene


# --------------------------------------------------------------------------------------------------
# 3. FRESH CRYPTOGRAPHIC AND STRUCTURAL VERIFICATION
# --------------------------------------------------------------------------------------------------

if not EVALUABLE_PARQUET.exists():
    raise FileNotFoundError(
        f"Missing frozen primary-evaluable cohort:\n"
        f"{EVALUABLE_PARQUET}"
    )

if not EVALUABLE_SIDECAR.exists():
    raise FileNotFoundError(
        f"Missing frozen evaluable-cohort sidecar:\n"
        f"{EVALUABLE_SIDECAR}"
    )

observed_sha256 = sha256_file(
    EVALUABLE_PARQUET
)

sidecar_sha256 = read_sidecar_hash(
    EVALUABLE_SIDECAR
)

if observed_sha256 != EXPECTED_EVALUABLE_SHA256:
    raise AssertionError(
        "Frozen Stage 6B evaluable-cohort SHA-256 mismatch.\n"
        f"Expected: {EXPECTED_EVALUABLE_SHA256}\n"
        f"Observed: {observed_sha256}"
    )

if sidecar_sha256 != observed_sha256:
    raise AssertionError(
        "Frozen Stage 6B evaluable-cohort sidecar mismatch.\n"
        f"Calculated: {observed_sha256}\n"
        f"Sidecar:    {sidecar_sha256}"
    )

parquet_file = pq.ParquetFile(
    EVALUABLE_PARQUET
)

metadata = parquet_file.metadata
schema_columns = parquet_file.schema_arrow.names

if metadata.num_rows != EXPECTED_ROWS:
    raise AssertionError(
        f"Unexpected Parquet row count: "
        f"{metadata.num_rows:,}; expected {EXPECTED_ROWS:,}"
    )

if metadata.num_columns != EXPECTED_COLUMNS:
    raise AssertionError(
        f"Unexpected Parquet column count: "
        f"{metadata.num_columns}; expected {EXPECTED_COLUMNS}"
    )


# --------------------------------------------------------------------------------------------------
# 4. LOAD REQUIRED FROZEN COLUMNS
# --------------------------------------------------------------------------------------------------

required_columns = [
    "rcv_accession",
    "t0_row_order",
    OUTCOME_COLUMN,
    GENE_COLUMN,
] + [
    specification["column"]
    for specification in MODEL_SPECIFICATIONS.values()
]

missing_columns = [
    column
    for column in required_columns
    if column not in schema_columns
]

if missing_columns:
    raise KeyError(
        "Required gene-level analysis columns are missing:\n"
        + "\n".join(
            f"  - {column}"
            for column in missing_columns
        )
    )

analysis_df = pd.read_parquet(
    EVALUABLE_PARQUET,
    columns=required_columns,
).copy()


# --------------------------------------------------------------------------------------------------
# 5. LOCKED COHORT VALIDATION
# --------------------------------------------------------------------------------------------------

if analysis_df.shape[0] != EXPECTED_ROWS:
    raise AssertionError(
        f"Loaded dataframe contains "
        f"{analysis_df.shape[0]:,} rows; "
        f"expected {EXPECTED_ROWS:,}"
    )

if analysis_df["rcv_accession"].isna().any():
    raise AssertionError(
        "Missing RCV accession detected."
    )

if (
    analysis_df["rcv_accession"]
    .astype(str)
    .str.strip()
    .eq("")
    .any()
):
    raise AssertionError(
        "Blank RCV accession detected."
    )

if (
    analysis_df["rcv_accession"]
    .nunique(dropna=False)
    != EXPECTED_ROWS
):
    raise AssertionError(
        "RCV accessions are not unique."
    )

row_order = pd.to_numeric(
    analysis_df["t0_row_order"],
    errors="raise",
).to_numpy()

if len(np.unique(row_order)) != EXPECTED_ROWS:
    raise AssertionError(
        "t0_row_order is not unique."
    )

if not np.all(np.diff(row_order) > 0):
    raise AssertionError(
        "Frozen evaluable cohort is not in increasing "
        "T0 row order."
    )

analysis_df[OUTCOME_COLUMN] = pd.to_numeric(
    analysis_df[OUTCOME_COLUMN],
    errors="raise",
).astype(int)

if not set(
    analysis_df[OUTCOME_COLUMN].unique()
).issubset({0, 1}):
    raise AssertionError(
        "Primary future-instability outcome is not binary."
    )

event_count = int(
    analysis_df[OUTCOME_COLUMN].sum()
)

negative_count = int(
    (analysis_df[OUTCOME_COLUMN] == 0).sum()
)

if event_count != EXPECTED_EVENTS:
    raise AssertionError(
        f"Unexpected event count: "
        f"{event_count:,}; expected {EXPECTED_EVENTS:,}"
    )

if negative_count != EXPECTED_NEGATIVES:
    raise AssertionError(
        f"Unexpected negative count: "
        f"{negative_count:,}; expected {EXPECTED_NEGATIVES:,}"
    )

analysis_df[GENE_COLUMN] = (
    analysis_df[GENE_COLUMN]
    .map(normalize_gene)
)

observed_genes = sorted(
    analysis_df[GENE_COLUMN]
    .unique()
    .tolist()
)

if observed_genes != [
    "BRCA1",
    "BRCA2",
    "EGFR",
    "MLH1",
]:
    raise AssertionError(
        f"Unexpected normalized genes: {observed_genes}"
    )

for model_key, specification in MODEL_SPECIFICATIONS.items():

    score_column = specification["column"]

    analysis_df[score_column] = pd.to_numeric(
        analysis_df[score_column],
        errors="raise",
    )

    values = analysis_df[
        score_column
    ].to_numpy(dtype=float)

    if np.isnan(values).any():
        raise AssertionError(
            f"Missing score value detected in {score_column}."
        )

    if not np.isfinite(values).all():
        raise AssertionError(
            f"Nonfinite score value detected in {score_column}."
        )

    if (
        (values < 0.0)
        | (values > 1.0)
    ).any():
        raise AssertionError(
            f"Score outside [0,1] detected in {score_column}."
        )


# --------------------------------------------------------------------------------------------------
# 6. VERIFY GENE-LEVEL ACCOUNTING
# --------------------------------------------------------------------------------------------------

gene_accounting_rows = []

for gene in GENE_DISPLAY_ORDER:

    gene_df = analysis_df.loc[
        analysis_df[GENE_COLUMN] == gene
    ]

    rows = int(len(gene_df))
    events = int(
        gene_df[OUTCOME_COLUMN].sum()
    )
    negatives = int(
        rows - events
    )

    expected = EXPECTED_GENE_ACCOUNTING[gene]

    if rows != expected["rows"]:
        raise AssertionError(
            f"Unexpected row count for {gene}: "
            f"{rows:,}; expected {expected['rows']:,}"
        )

    if events != expected["events"]:
        raise AssertionError(
            f"Unexpected event count for {gene}: "
            f"{events:,}; expected {expected['events']:,}"
        )

    if negatives != expected["negatives"]:
        raise AssertionError(
            f"Unexpected negative count for {gene}: "
            f"{negatives:,}; expected {expected['negatives']:,}"
        )

    gene_accounting_rows.append(
        {
            "gene": gene,
            "analysis_role": (
                "primary"
                if gene in PRIMARY_GENES
                else "exploratory"
            ),
            "rows": rows,
            "events": events,
            "negatives": negatives,
            "event_prevalence": (
                events / rows
            ),
        }
    )

gene_accounting = pd.DataFrame(
    gene_accounting_rows
)

if int(
    gene_accounting["rows"].sum()
) != EXPECTED_ROWS:
    raise AssertionError(
        "Gene-level row accounting failed."
    )

if int(
    gene_accounting["events"].sum()
) != EXPECTED_EVENTS:
    raise AssertionError(
        "Gene-level event accounting failed."
    )

if int(
    gene_accounting["negatives"].sum()
) != EXPECTED_NEGATIVES:
    raise AssertionError(
        "Gene-level negative accounting failed."
    )


# --------------------------------------------------------------------------------------------------
# 7. CALCULATE LOCKED GENE-LEVEL POINT ESTIMATES
# --------------------------------------------------------------------------------------------------

model_result_rows = []
paired_difference_rows = []

for gene in GENE_DISPLAY_ORDER:

    stratum_df = analysis_df.loc[
        analysis_df[GENE_COLUMN] == gene
    ].copy()

    y_true = stratum_df[
        OUTCOME_COLUMN
    ].to_numpy(dtype=int)

    rows = int(len(stratum_df))
    events = int(y_true.sum())
    negatives = int(
        rows - events
    )
    prevalence = float(
        events / rows
    )

    if len(np.unique(y_true)) != 2:
        raise AssertionError(
            f"{gene} does not contain both outcome classes."
        )

    point_metrics = {}

    for model_key, specification in MODEL_SPECIFICATIONS.items():

        score_column = specification["column"]
        model_name = specification["display_name"]

        risk_score = stratum_df[
            score_column
        ].to_numpy(dtype=float)

        unique_score_values = int(
            np.unique(risk_score).size
        )

        if unique_score_values < 2:
            raise AssertionError(
                f"{model_name} is constant within {gene}."
            )

        auprc = float(
            average_precision_score(
                y_true,
                risk_score,
            )
        )

        auroc = float(
            roc_auc_score(
                y_true,
                risk_score,
            )
        )

        event_mean_risk = float(
            risk_score[
                y_true == 1
            ].mean()
        )

        negative_mean_risk = float(
            risk_score[
                y_true == 0
            ].mean()
        )

        point_metrics[model_key] = {
            "auprc": auprc,
            "auroc": auroc,
        }

        model_result_rows.append(
            {
                "gene": gene,
                "analysis_role": (
                    "primary"
                    if gene in PRIMARY_GENES
                    else "exploratory"
                ),
                "model_key": model_key,
                "model": model_name,
                "score_column": score_column,
                "rows": rows,
                "events": events,
                "negatives": negatives,
                "event_prevalence": prevalence,
                "unique_score_values": unique_score_values,
                "mean_risk_events": event_mean_risk,
                "mean_risk_negatives": negative_mean_risk,
                "mean_risk_difference_event_minus_negative": (
                    event_mean_risk
                    - negative_mean_risk
                ),
                "auprc": auprc,
                "auprc_minus_gene_prevalence": (
                    auprc - prevalence
                ),
                "auprc_lift_over_gene_prevalence": (
                    auprc / prevalence
                ),
                "auroc": auroc,
                "auroc_minus_0_50": (
                    auroc - 0.50
                ),
                "sparse_event_or_negative_flag": bool(
                    min(events, negatives) < 20
                ),
            }
        )

    for comparator_key in COMPARATOR_KEYS:

        comparator_name = (
            MODEL_SPECIFICATIONS[
                comparator_key
            ]["display_name"]
        )

        paired_difference_rows.append(
            {
                "gene": gene,
                "analysis_role": (
                    "primary"
                    if gene in PRIMARY_GENES
                    else "exploratory"
                ),
                "comparison": (
                    f"Full GES minus {comparator_name}"
                ),
                "comparator_key": comparator_key,
                "rows": rows,
                "events": events,
                "negatives": negatives,
                "auprc_difference": (
                    point_metrics["full_ges"]["auprc"]
                    - point_metrics[comparator_key]["auprc"]
                ),
                "auroc_difference": (
                    point_metrics["full_ges"]["auroc"]
                    - point_metrics[comparator_key]["auroc"]
                ),
                "inference_performed": False,
            }
        )

gene_level_point_estimates = pd.DataFrame(
    model_result_rows
)

gene_level_paired_point_differences = pd.DataFrame(
    paired_difference_rows
)


# --------------------------------------------------------------------------------------------------
# 8. COMPLETENESS CHECKS
# --------------------------------------------------------------------------------------------------

expected_model_rows = (
    len(GENE_DISPLAY_ORDER)
    * len(MODEL_SPECIFICATIONS)
)

if len(gene_level_point_estimates) != expected_model_rows:
    raise AssertionError(
        f"Unexpected model-result row count: "
        f"{len(gene_level_point_estimates)}; "
        f"expected {expected_model_rows}."
    )

expected_difference_rows = (
    len(GENE_DISPLAY_ORDER)
    * len(COMPARATOR_KEYS)
)

if (
    len(gene_level_paired_point_differences)
    != expected_difference_rows
):
    raise AssertionError(
        f"Unexpected paired-difference row count: "
        f"{len(gene_level_paired_point_differences)}; "
        f"expected {expected_difference_rows}."
    )

if gene_level_point_estimates[
    ["auprc", "auroc"]
].isna().any().any():
    raise AssertionError(
        "Missing gene-level discrimination result detected."
    )

if not gene_level_point_estimates[
    "auprc"
].between(0.0, 1.0).all():
    raise AssertionError(
        "Gene-level AUPRC outside [0,1] detected."
    )

if not gene_level_point_estimates[
    "auroc"
].between(0.0, 1.0).all():
    raise AssertionError(
        "Gene-level AUROC outside [0,1] detected."
    )


# --------------------------------------------------------------------------------------------------
# 9. COMPACT DISPLAY TABLES
# --------------------------------------------------------------------------------------------------

model_display_columns = [
    "gene",
    "analysis_role",
    "model",
    "rows",
    "events",
    "negatives",
    "event_prevalence",
    "unique_score_values",
    "mean_risk_events",
    "mean_risk_negatives",
    "mean_risk_difference_event_minus_negative",
    "auprc",
    "auprc_minus_gene_prevalence",
    "auprc_lift_over_gene_prevalence",
    "auroc",
    "auroc_minus_0_50",
    "sparse_event_or_negative_flag",
]

paired_display_columns = [
    "gene",
    "analysis_role",
    "comparison",
    "rows",
    "events",
    "negatives",
    "auprc_difference",
    "auroc_difference",
    "inference_performed",
]

gene_level_model_display = (
    gene_level_point_estimates[
        model_display_columns
    ].copy()
)

gene_level_paired_display = (
    gene_level_paired_point_differences[
        paired_display_columns
    ].copy()
)


# --------------------------------------------------------------------------------------------------
# 10. PRINT COMPLETE LOCKED RESULT RECORD
# --------------------------------------------------------------------------------------------------

separator = "=" * 166
subseparator = "-" * 166

print("\n" + separator)

print(
    "STAGE 6C STEP 3C — CELL 6C-3C1 — "
    "LOCKED GENE-LEVEL DISCRIMINATION POINT ESTIMATES"
)

print(separator)

print("\nFROZEN INPUT VERIFICATION")
print(subseparator)

print(
    f"Stage 6B evaluable-cohort SHA-256 : PASS "
    f"({observed_sha256})"
)

print(
    f"Parquet dimensions                : PASS "
    f"({metadata.num_rows:,} × {metadata.num_columns})"
)

print(
    f"Unique evaluable RCV keys         : PASS "
    f"({analysis_df['rcv_accession'].nunique():,})"
)

print(
    f"Primary instability events        : PASS "
    f"({event_count:,})"
)

print(
    f"Primary instability negatives     : PASS "
    f"({negative_count:,})"
)

print(
    f"Overall event prevalence          : "
    f"{analysis_df[OUTCOME_COLUMN].mean():.8f} "
    f"({analysis_df[OUTCOME_COLUMN].mean():.6%})"
)

print(
    f"Primary gene analyses             : "
    f"{PRIMARY_GENES}"
)

print(
    f"Exploratory gene analysis         : "
    f"{EXPLORATORY_GENES}"
)

print("Score direction changed           : No")
print("Threshold or weight optimized     : No")
print("Recalibration performed           : No")
print("Bootstrap inference performed     : No")
print("Scientific artifact written       : No")


print("\nGENE-LEVEL MODEL POINT ESTIMATES")
print(subseparator)

with pd.option_context(
    "display.max_columns", None,
    "display.width", 360,
    "display.float_format",
    lambda value: f"{value:.8f}",
):
    print(
        gene_level_model_display.to_string(
            index=False
        )
    )


print("\nFULL-GES-MINUS-COMPARATOR POINT DIFFERENCES")
print(subseparator)

with pd.option_context(
    "display.max_columns", None,
    "display.width", 300,
    "display.float_format",
    lambda value: f"{value:.8f}",
):
    print(
        gene_level_paired_display.to_string(
            index=False
        )
    )


print("\nGENE-LEVEL ACCOUNTING")
print(subseparator)

with pd.option_context(
    "display.float_format",
    lambda value: f"{value:.8f}",
):
    print(
        gene_accounting.to_string(
            index=False
        )
    )


print("\nCELL DECISION")
print(subseparator)

print(
    "PASS_STAGE6C_GENE_LEVEL_POINT_ESTIMATES_COMPLETE"
)

print(
    "Locked AUPRC and AUROC point estimates were calculated "
    "for Full GES, No-star GES, review stars, and combined "
    "metadata within BRCA1, BRCA2, MLH1, and EGFR."
)

print(
    "EGFR results remain exploratory and are not combined with "
    "the three prespecified primary-gene conclusions."
)

print(
    "No score, outcome, gene assignment, cohort membership, "
    "threshold, weight, frozen model, or scientific artifact "
    "was modified."
)


STAGE 6C STEP 3C — CELL 6C-3C1 — LOCKED GENE-LEVEL DISCRIMINATION POINT ESTIMATES

FROZEN INPUT VERIFICATION
----------------------------------------------------------------------------------------------------------------------------------------------------------------------
Stage 6B evaluable-cohort SHA-256 : PASS (c6ad50dfc376a746e1830d000dba975101d88763fd61b8d1983f6300edb6e038)
Parquet dimensions                : PASS (66,636 × 79)
Unique evaluable RCV keys         : PASS (66,636)
Primary instability events        : PASS (6,485)
Primary instability negatives     : PASS (60,151)
Overall event prevalence          : 0.09731977 (9.731977%)
Primary gene analyses             : ['BRCA1', 'BRCA2', 'MLH1']
Exploratory gene analysis         : ['EGFR']
Score direction changed           : No
Threshold or weight optimized     : No
Recalibration performed           : No
Bootstrap inference performed     : No
Scientific artifact written       : No

GENE-LEVEL MODEL POINT ESTIMATES
---------------

In [30]:
# ==================================================================================================
# STAGE 6C STEP 3C — CELL 6C-3C2
# OPTIMIZED LOCKED GENE-LEVEL PAIRED BOOTSTRAP INFERENCE
#
# Purpose:
#   1. Run 2,000 exact ordinary row-bootstrap replicates separately within:
#         - BRCA1
#         - BRCA2
#         - MLH1
#         - EGFR
#   2. Use identical bootstrap samples across all models within each gene.
#   3. Calculate model-specific 95% percentile intervals for:
#         - AUPRC
#         - AUPRC minus gene-specific bootstrap prevalence
#         - AUROC
#         - AUROC minus 0.50
#   4. Calculate paired Full-GES-minus-comparator inference for:
#         - No-star GES
#         - Review stars
#         - Combined metadata
#   5. Apply Holm correction separately to the nine PRIMARY-gene AUPRC comparisons
#      and the nine PRIMARY-gene AUROC comparisons.
#   6. Retain EGFR as exploratory and report its probabilities without including
#      them in the primary-gene multiplicity families.
#
# Bootstrap implementation:
#   - Exact multinomial row multiplicities
#   - Mathematically equivalent to sampling rows with replacement
#   - Vectorized tie-aware weighted AUPRC and AUROC
#
# This cell does NOT:
#   - modify scores, outcomes, genes, or cohort membership;
#   - optimize thresholds or weights;
#   - recalibrate scores;
#   - write scientific artifacts.
# ==================================================================================================

from pathlib import Path
import hashlib
import re
import time
import gc

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

from scipy.sparse import csr_matrix
from sklearn.metrics import average_precision_score, roc_auc_score


# --------------------------------------------------------------------------------------------------
# 1. FROZEN INPUT IDENTITY AND ANALYSIS SPECIFICATION
# --------------------------------------------------------------------------------------------------

STAGE6_DIR = Path(
    "/content/drive/MyDrive/GES_RAG_Temporal_Study/"
    "data_processed/stage6_temporal_validation"
)

EVALUABLE_PARQUET = (
    STAGE6_DIR / "stage6b_locked_primary_evaluable_cohort_v1.parquet"
)

EVALUABLE_SIDECAR = Path(
    str(EVALUABLE_PARQUET) + ".sha256"
)

EXPECTED_EVALUABLE_SHA256 = (
    "c6ad50dfc376a746e1830d000dba975101d88763fd61b8d1983f6300edb6e038"
)

EXPECTED_ROWS = 66_636
EXPECTED_COLUMNS = 79
EXPECTED_EVENTS = 6_485
EXPECTED_NEGATIVES = 60_151

OUTCOME_COLUMN = "primary_future_instability"
GENE_COLUMN = "target_gene"

N_BOOTSTRAP = 2_000
RANDOM_SEED = 42
BOOTSTRAP_BATCH_SIZE = 50
MINIMUM_VALID_REPLICATES = 1_000

CI_LOWER_QUANTILE = 0.025
CI_UPPER_QUANTILE = 0.975

PRIMARY_GENES = [
    "BRCA1",
    "BRCA2",
    "MLH1",
]

EXPLORATORY_GENES = [
    "EGFR",
]

GENE_DISPLAY_ORDER = [
    "BRCA1",
    "BRCA2",
    "MLH1",
    "EGFR",
]

MODEL_SPECIFICATIONS = {
    "full_ges": {
        "column": "full_ges_instability_risk_t0",
        "display_name": "Full GES",
    },
    "no_star_ges": {
        "column": "no_star_ges_instability_risk_t0",
        "display_name": "No-star GES",
    },
    "review_stars": {
        "column": "review_stars_instability_risk",
        "display_name": "Review stars",
    },
    "combined_metadata": {
        "column": "combined_metadata_instability_risk",
        "display_name": "Combined metadata",
    },
}

PAIRED_COMPARATORS = [
    "no_star_ges",
    "review_stars",
    "combined_metadata",
]

EXPECTED_GENE_ACCOUNTING = {
    "BRCA1": {
        "rows": 21_594,
        "events": 2_023,
        "negatives": 19_571,
    },
    "BRCA2": {
        "rows": 34_152,
        "events": 3_960,
        "negatives": 30_192,
    },
    "MLH1": {
        "rows": 8_701,
        "events": 425,
        "negatives": 8_276,
    },
    "EGFR": {
        "rows": 2_189,
        "events": 77,
        "negatives": 2_112,
    },
}


# --------------------------------------------------------------------------------------------------
# 2. GENERAL HELPERS
# --------------------------------------------------------------------------------------------------

def sha256_file(
    path: Path,
    chunk_size: int = 1024 * 1024,
) -> str:

    digest = hashlib.sha256()

    with path.open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


def read_sidecar_hash(path: Path) -> str:

    text = path.read_text(
        encoding="utf-8"
    ).strip()

    matches = re.findall(
        r"\b[a-fA-F0-9]{64}\b",
        text,
    )

    if not matches:
        raise ValueError(
            f"No SHA-256 value was found in sidecar:\n{path}"
        )

    return matches[0].lower()


def percentile_interval(
    values: np.ndarray,
) -> tuple:

    values = np.asarray(
        values,
        dtype=float,
    )

    values = values[
        np.isfinite(values)
    ]

    if len(values) == 0:
        return np.nan, np.nan

    lower, upper = np.quantile(
        values,
        [
            CI_LOWER_QUANTILE,
            CI_UPPER_QUANTILE,
        ],
    )

    return float(lower), float(upper)


def bootstrap_sign_pvalue(
    differences: np.ndarray,
) -> float:
    """
    Two-sided bootstrap sign probability with plus-one correction.
    Null value = zero.
    """

    differences = np.asarray(
        differences,
        dtype=float,
    )

    differences = differences[
        np.isfinite(differences)
    ]

    if len(differences) == 0:
        return np.nan

    number_valid = len(differences)

    lower_tail = (
        np.count_nonzero(
            differences <= 0.0
        ) + 1
    ) / (number_valid + 1)

    upper_tail = (
        np.count_nonzero(
            differences >= 0.0
        ) + 1
    ) / (number_valid + 1)

    return float(
        min(
            1.0,
            2.0 * min(
                lower_tail,
                upper_tail,
            ),
        )
    )


def holm_adjust(
    p_values: np.ndarray,
) -> np.ndarray:
    """
    Holm step-down family-wise error correction.
    """

    p_values = np.asarray(
        p_values,
        dtype=float,
    )

    adjusted = np.full(
        len(p_values),
        np.nan,
        dtype=float,
    )

    valid_positions = np.where(
        np.isfinite(p_values)
    )[0]

    if len(valid_positions) == 0:
        return adjusted

    valid_p_values = p_values[
        valid_positions
    ]

    ordered_positions = np.argsort(
        valid_p_values
    )

    number_of_tests = len(
        valid_p_values
    )

    running_maximum = 0.0

    for ordered_rank, within_valid_position in enumerate(
        ordered_positions
    ):

        original_position = valid_positions[
            within_valid_position
        ]

        raw_adjusted_value = (
            number_of_tests - ordered_rank
        ) * valid_p_values[
            within_valid_position
        ]

        running_maximum = max(
            running_maximum,
            raw_adjusted_value,
        )

        adjusted[
            original_position
        ] = min(
            1.0,
            running_maximum,
        )

    return adjusted


def interval_status(
    lower: float,
    upper: float,
    positive_label: str,
    negative_label: str,
) -> str:

    if (
        not np.isfinite(lower)
        or not np.isfinite(upper)
    ):
        return "not_estimable"

    if lower > 0.0:
        return positive_label

    if upper < 0.0:
        return negative_label

    return "interval_includes_null"


def normalize_gene(value) -> str:

    gene = str(value).strip().upper()

    if not gene:
        raise ValueError(
            "Blank target-gene value detected."
        )

    return gene


# --------------------------------------------------------------------------------------------------
# 3. FAST TIE-AWARE WEIGHTED METRIC ENGINE
# --------------------------------------------------------------------------------------------------

def construct_score_group_cache(
    scores: np.ndarray,
    outcomes: np.ndarray,
) -> dict:
    """
    Construct sparse score-group matrices once for each model and gene.

    np.unique returns score groups in ascending order.

    Bootstrap multiplicities can then be aggregated by tied-score group
    without sorting every bootstrap sample.
    """

    scores = np.asarray(
        scores,
        dtype=np.float64,
    )

    outcomes = np.asarray(
        outcomes,
        dtype=np.int8,
    )

    unique_scores, group_index = np.unique(
        scores,
        return_inverse=True,
    )

    number_of_rows = len(scores)
    number_of_groups = len(
        unique_scores
    )

    row_positions = np.arange(
        number_of_rows
    )

    total_group_matrix = csr_matrix(
        (
            np.ones(
                number_of_rows,
                dtype=np.float64,
            ),
            (
                group_index,
                row_positions,
            ),
        ),
        shape=(
            number_of_groups,
            number_of_rows,
        ),
    )

    positive_positions = np.flatnonzero(
        outcomes == 1
    )

    positive_group_matrix = csr_matrix(
        (
            np.ones(
                len(positive_positions),
                dtype=np.float64,
            ),
            (
                group_index[
                    positive_positions
                ],
                positive_positions,
            ),
        ),
        shape=(
            number_of_groups,
            number_of_rows,
        ),
    )

    return {
        "unique_scores": unique_scores,
        "number_of_groups": number_of_groups,
        "total_group_matrix": total_group_matrix,
        "positive_group_matrix": positive_group_matrix,
    }


def calculate_grouped_weighted_metrics(
    cache: dict,
    count_matrix: np.ndarray,
    positive_totals: np.ndarray,
) -> tuple:
    """
    Calculate exact weighted AUPRC and AUROC for multiple bootstrap
    samples simultaneously.

    count_matrix:
        bootstrap samples × original rows

    Each row contains multinomial bootstrap multiplicities.
    """

    count_matrix = np.asarray(
        count_matrix
    )

    positive_totals = np.asarray(
        positive_totals,
        dtype=np.float64,
    )

    sample_totals = (
        count_matrix
        .sum(axis=1)
        .astype(np.float64)
    )

    negative_totals = (
        sample_totals
        - positive_totals
    )

    group_total_counts = np.asarray(
        cache["total_group_matrix"]
        @ count_matrix.T,
        dtype=np.float64,
    )

    group_positive_counts = np.asarray(
        cache["positive_group_matrix"]
        @ count_matrix.T,
        dtype=np.float64,
    )

    group_negative_counts = (
        group_total_counts
        - group_positive_counts
    )

    valid_mask = (
        (positive_totals > 0.0)
        & (negative_totals > 0.0)
    )

    # ----------------------------------------------------------------------------------------------
    # AUROC: ascending score groups
    # ----------------------------------------------------------------------------------------------

    cumulative_negatives_before_group = (
        np.cumsum(
            group_negative_counts,
            axis=0,
        )
        - group_negative_counts
    )

    concordant_pair_numerator = np.sum(
        group_positive_counts
        * (
            cumulative_negatives_before_group
            + 0.5 * group_negative_counts
        ),
        axis=0,
    )

    auc_denominator = (
        positive_totals
        * negative_totals
    )

    auroc = np.full(
        len(positive_totals),
        np.nan,
        dtype=np.float64,
    )

    np.divide(
        concordant_pair_numerator,
        auc_denominator,
        out=auroc,
        where=valid_mask,
    )

    # ----------------------------------------------------------------------------------------------
    # AUPRC: descending score groups
    # ----------------------------------------------------------------------------------------------

    positive_descending = (
        group_positive_counts[::-1, :]
    )

    total_descending = (
        group_total_counts[::-1, :]
    )

    cumulative_positive = np.cumsum(
        positive_descending,
        axis=0,
    )

    cumulative_total = np.cumsum(
        total_descending,
        axis=0,
    )

    precision_at_group = np.zeros_like(
        cumulative_positive,
        dtype=np.float64,
    )

    np.divide(
        cumulative_positive,
        cumulative_total,
        out=precision_at_group,
        where=cumulative_total > 0.0,
    )

    average_precision_numerator = np.sum(
        precision_at_group
        * positive_descending,
        axis=0,
    )

    auprc = np.full(
        len(positive_totals),
        np.nan,
        dtype=np.float64,
    )

    np.divide(
        average_precision_numerator,
        positive_totals,
        out=auprc,
        where=valid_mask,
    )

    return auprc, auroc


# --------------------------------------------------------------------------------------------------
# 4. FRESH CRYPTOGRAPHIC AND STRUCTURAL VERIFICATION
# --------------------------------------------------------------------------------------------------

if not EVALUABLE_PARQUET.exists():
    raise FileNotFoundError(
        f"Missing frozen primary-evaluable cohort:\n"
        f"{EVALUABLE_PARQUET}"
    )

if not EVALUABLE_SIDECAR.exists():
    raise FileNotFoundError(
        f"Missing frozen evaluable-cohort sidecar:\n"
        f"{EVALUABLE_SIDECAR}"
    )

observed_sha256 = sha256_file(
    EVALUABLE_PARQUET
)

sidecar_sha256 = read_sidecar_hash(
    EVALUABLE_SIDECAR
)

if observed_sha256 != EXPECTED_EVALUABLE_SHA256:
    raise AssertionError(
        "Frozen Stage 6B evaluable-cohort SHA-256 mismatch.\n"
        f"Expected: {EXPECTED_EVALUABLE_SHA256}\n"
        f"Observed: {observed_sha256}"
    )

if sidecar_sha256 != observed_sha256:
    raise AssertionError(
        "Frozen Stage 6B evaluable-cohort sidecar mismatch.\n"
        f"Calculated: {observed_sha256}\n"
        f"Sidecar:    {sidecar_sha256}"
    )

parquet_file = pq.ParquetFile(
    EVALUABLE_PARQUET
)

metadata = parquet_file.metadata
schema_columns = parquet_file.schema_arrow.names

if metadata.num_rows != EXPECTED_ROWS:
    raise AssertionError(
        f"Unexpected Parquet row count: "
        f"{metadata.num_rows:,}; expected {EXPECTED_ROWS:,}"
    )

if metadata.num_columns != EXPECTED_COLUMNS:
    raise AssertionError(
        f"Unexpected Parquet column count: "
        f"{metadata.num_columns}; expected {EXPECTED_COLUMNS}"
    )


# --------------------------------------------------------------------------------------------------
# 5. LOAD REQUIRED FROZEN COLUMNS
# --------------------------------------------------------------------------------------------------

required_columns = [
    "rcv_accession",
    "t0_row_order",
    OUTCOME_COLUMN,
    GENE_COLUMN,
] + [
    specification["column"]
    for specification in MODEL_SPECIFICATIONS.values()
]

missing_columns = [
    column
    for column in required_columns
    if column not in schema_columns
]

if missing_columns:
    raise KeyError(
        "Required gene-bootstrap columns are missing:\n"
        + "\n".join(
            f"  - {column}"
            for column in missing_columns
        )
    )

analysis_df = pd.read_parquet(
    EVALUABLE_PARQUET,
    columns=required_columns,
).copy()


# --------------------------------------------------------------------------------------------------
# 6. LOCKED COHORT VALIDATION
# --------------------------------------------------------------------------------------------------

if analysis_df.shape[0] != EXPECTED_ROWS:
    raise AssertionError(
        f"Loaded dataframe contains "
        f"{analysis_df.shape[0]:,} rows; "
        f"expected {EXPECTED_ROWS:,}"
    )

if analysis_df["rcv_accession"].isna().any():
    raise AssertionError(
        "Missing RCV accession detected."
    )

if (
    analysis_df["rcv_accession"]
    .astype(str)
    .str.strip()
    .eq("")
    .any()
):
    raise AssertionError(
        "Blank RCV accession detected."
    )

if (
    analysis_df["rcv_accession"]
    .nunique(dropna=False)
    != EXPECTED_ROWS
):
    raise AssertionError(
        "RCV accessions are not unique."
    )

row_order = pd.to_numeric(
    analysis_df["t0_row_order"],
    errors="raise",
).to_numpy()

if len(np.unique(row_order)) != EXPECTED_ROWS:
    raise AssertionError(
        "t0_row_order is not unique."
    )

if not np.all(
    np.diff(row_order) > 0
):
    raise AssertionError(
        "Frozen evaluable cohort is not in "
        "increasing T0 row order."
    )

analysis_df[OUTCOME_COLUMN] = pd.to_numeric(
    analysis_df[OUTCOME_COLUMN],
    errors="raise",
).astype(int)

if not set(
    analysis_df[OUTCOME_COLUMN].unique()
).issubset({0, 1}):
    raise AssertionError(
        "Primary future-instability outcome is not binary."
    )

event_count = int(
    analysis_df[OUTCOME_COLUMN].sum()
)

negative_count = int(
    (analysis_df[OUTCOME_COLUMN] == 0).sum()
)

if event_count != EXPECTED_EVENTS:
    raise AssertionError(
        f"Unexpected event count: "
        f"{event_count:,}; expected {EXPECTED_EVENTS:,}"
    )

if negative_count != EXPECTED_NEGATIVES:
    raise AssertionError(
        f"Unexpected negative count: "
        f"{negative_count:,}; expected {EXPECTED_NEGATIVES:,}"
    )

analysis_df[GENE_COLUMN] = (
    analysis_df[GENE_COLUMN]
    .map(normalize_gene)
)

observed_genes = sorted(
    analysis_df[GENE_COLUMN]
    .unique()
    .tolist()
)

if observed_genes != [
    "BRCA1",
    "BRCA2",
    "EGFR",
    "MLH1",
]:
    raise AssertionError(
        f"Unexpected normalized genes: "
        f"{observed_genes}"
    )

for model_key, specification in MODEL_SPECIFICATIONS.items():

    score_column = specification["column"]

    analysis_df[score_column] = pd.to_numeric(
        analysis_df[score_column],
        errors="raise",
    )

    values = analysis_df[
        score_column
    ].to_numpy(dtype=float)

    if np.isnan(values).any():
        raise AssertionError(
            f"Missing score value detected in "
            f"{score_column}."
        )

    if not np.isfinite(values).all():
        raise AssertionError(
            f"Nonfinite score value detected in "
            f"{score_column}."
        )

    if (
        (values < 0.0)
        | (values > 1.0)
    ).any():
        raise AssertionError(
            f"Score outside [0,1] detected in "
            f"{score_column}."
        )


# --------------------------------------------------------------------------------------------------
# 7. VERIFY GENE-LEVEL ACCOUNTING
# --------------------------------------------------------------------------------------------------

gene_accounting_rows = []

for gene in GENE_DISPLAY_ORDER:

    gene_df = analysis_df.loc[
        analysis_df[GENE_COLUMN] == gene
    ]

    rows = int(
        len(gene_df)
    )

    events = int(
        gene_df[OUTCOME_COLUMN].sum()
    )

    negatives = int(
        rows - events
    )

    expected = EXPECTED_GENE_ACCOUNTING[
        gene
    ]

    if rows != expected["rows"]:
        raise AssertionError(
            f"Unexpected row count for {gene}: "
            f"{rows:,}; expected {expected['rows']:,}"
        )

    if events != expected["events"]:
        raise AssertionError(
            f"Unexpected event count for {gene}: "
            f"{events:,}; expected {expected['events']:,}"
        )

    if negatives != expected["negatives"]:
        raise AssertionError(
            f"Unexpected negative count for {gene}: "
            f"{negatives:,}; expected {expected['negatives']:,}"
        )

    gene_accounting_rows.append(
        {
            "gene": gene,
            "analysis_role": (
                "primary"
                if gene in PRIMARY_GENES
                else "exploratory"
            ),
            "rows": rows,
            "events": events,
            "negatives": negatives,
            "event_prevalence": (
                events / rows
            ),
        }
    )

gene_accounting = pd.DataFrame(
    gene_accounting_rows
)

if int(
    gene_accounting["rows"].sum()
) != EXPECTED_ROWS:
    raise AssertionError(
        "Gene-level row accounting failed."
    )

if int(
    gene_accounting["events"].sum()
) != EXPECTED_EVENTS:
    raise AssertionError(
        "Gene-level event accounting failed."
    )

if int(
    gene_accounting["negatives"].sum()
) != EXPECTED_NEGATIVES:
    raise AssertionError(
        "Gene-level negative accounting failed."
    )


# --------------------------------------------------------------------------------------------------
# 8. RUN OPTIMIZED EXACT GENE-LEVEL BOOTSTRAP
# --------------------------------------------------------------------------------------------------

rng = np.random.default_rng(
    RANDOM_SEED
)

point_estimate_rows = []
model_interval_rows = []
paired_difference_rows = []

analysis_start_time = time.time()

for gene in GENE_DISPLAY_ORDER:

    gene_start_time = time.time()

    analysis_role = (
        "primary"
        if gene in PRIMARY_GENES
        else "exploratory"
    )

    stratum_df = (
        analysis_df.loc[
            analysis_df[GENE_COLUMN] == gene
        ]
        .reset_index(drop=True)
    )

    outcomes = stratum_df[
        OUTCOME_COLUMN
    ].to_numpy(dtype=np.int8)

    number_of_rows = int(
        len(stratum_df)
    )

    number_of_events = int(
        outcomes.sum()
    )

    number_of_negatives = int(
        number_of_rows
        - number_of_events
    )

    gene_prevalence = float(
        number_of_events
        / number_of_rows
    )

    if len(
        np.unique(outcomes)
    ) != 2:
        raise AssertionError(
            f"{gene} does not contain both outcome classes."
        )

    score_arrays = {
        model_key: stratum_df[
            specification["column"]
        ].to_numpy(dtype=np.float64)
        for model_key, specification
        in MODEL_SPECIFICATIONS.items()
    }

    score_group_caches = {
        model_key: construct_score_group_cache(
            scores=score_arrays[
                model_key
            ],
            outcomes=outcomes,
        )
        for model_key
        in MODEL_SPECIFICATIONS
    }

    point_metrics = {}

    print(
        f"\nPreparing {gene} bootstrap: "
        f"{number_of_rows:,} rows, "
        f"{number_of_events:,} events, "
        f"{number_of_negatives:,} negatives, "
        f"role={analysis_role}"
    )

    # ----------------------------------------------------------------------------------------------
    # Validate optimized metrics against scikit-learn point estimates
    # ----------------------------------------------------------------------------------------------

    original_count_matrix = np.ones(
        (
            1,
            number_of_rows,
        ),
        dtype=np.int16,
    )

    original_positive_total = np.array(
        [
            number_of_events
        ],
        dtype=np.float64,
    )

    for model_key, specification in MODEL_SPECIFICATIONS.items():

        scores = score_arrays[
            model_key
        ]

        sklearn_auprc = float(
            average_precision_score(
                outcomes,
                scores,
            )
        )

        sklearn_auroc = float(
            roc_auc_score(
                outcomes,
                scores,
            )
        )

        fast_auprc, fast_auroc = (
            calculate_grouped_weighted_metrics(
                cache=score_group_caches[
                    model_key
                ],
                count_matrix=original_count_matrix,
                positive_totals=original_positive_total,
            )
        )

        if not np.isclose(
            fast_auprc[0],
            sklearn_auprc,
            rtol=1e-11,
            atol=1e-12,
        ):
            raise AssertionError(
                f"Fast AUPRC validation failed for "
                f"{gene}, model {model_key}.\n"
                f"Scikit-learn: {sklearn_auprc:.15f}\n"
                f"Fast method:  {fast_auprc[0]:.15f}"
            )

        if not np.isclose(
            fast_auroc[0],
            sklearn_auroc,
            rtol=1e-11,
            atol=1e-12,
        ):
            raise AssertionError(
                f"Fast AUROC validation failed for "
                f"{gene}, model {model_key}.\n"
                f"Scikit-learn: {sklearn_auroc:.15f}\n"
                f"Fast method:  {fast_auroc[0]:.15f}"
            )

        point_metrics[
            model_key
        ] = {
            "auprc": sklearn_auprc,
            "auroc": sklearn_auroc,
        }

        point_estimate_rows.append(
            {
                "gene": gene,
                "analysis_role": analysis_role,
                "model_key": model_key,
                "model": specification[
                    "display_name"
                ],
                "rows": number_of_rows,
                "events": number_of_events,
                "negatives": number_of_negatives,
                "gene_prevalence": gene_prevalence,
                "point_auprc": sklearn_auprc,
                "point_auprc_minus_prevalence": (
                    sklearn_auprc
                    - gene_prevalence
                ),
                "point_auroc": sklearn_auroc,
                "point_auroc_minus_0_50": (
                    sklearn_auroc
                    - 0.50
                ),
                "fast_metric_validation": "PASS",
            }
        )

    print(
        "  Fast metric validation against scikit-learn: PASS"
    )

    bootstrap_metrics = {
        model_key: {
            "auprc": np.full(
                N_BOOTSTRAP,
                np.nan,
                dtype=np.float64,
            ),
            "auroc": np.full(
                N_BOOTSTRAP,
                np.nan,
                dtype=np.float64,
            ),
        }
        for model_key
        in MODEL_SPECIFICATIONS
    }

    bootstrap_prevalence = np.full(
        N_BOOTSTRAP,
        np.nan,
        dtype=np.float64,
    )

    bootstrap_probabilities = np.full(
        number_of_rows,
        1.0 / number_of_rows,
        dtype=np.float64,
    )

    bootstrap_probabilities[-1] = (
        1.0
        - bootstrap_probabilities[:-1].sum()
    )

    if not np.isclose(
        bootstrap_probabilities.sum(),
        1.0,
        rtol=0.0,
        atol=1e-14,
    ):
        raise AssertionError(
            f"Bootstrap probability vector for {gene} "
            "does not sum to one."
        )

    for batch_start in range(
        0,
        N_BOOTSTRAP,
        BOOTSTRAP_BATCH_SIZE,
    ):

        batch_end = min(
            batch_start
            + BOOTSTRAP_BATCH_SIZE,
            N_BOOTSTRAP,
        )

        current_batch_size = (
            batch_end
            - batch_start
        )

        bootstrap_counts = rng.multinomial(
            number_of_rows,
            bootstrap_probabilities,
            size=current_batch_size,
        )

        bootstrap_sample_totals = (
            bootstrap_counts
            .sum(axis=1)
        )

        if not np.all(
            bootstrap_sample_totals
            == number_of_rows
        ):
            raise AssertionError(
                f"A {gene} bootstrap replicate did not "
                "preserve the required sample size."
            )

        bootstrap_positive_totals = (
            bootstrap_counts
            @ outcomes
        ).astype(np.float64)

        valid_outcome_mask = (
            (bootstrap_positive_totals > 0.0)
            & (
                bootstrap_positive_totals
                < number_of_rows
            )
        )

        bootstrap_prevalence[
            batch_start:batch_end
        ] = np.where(
            valid_outcome_mask,
            bootstrap_positive_totals
            / number_of_rows,
            np.nan,
        )

        for model_key in MODEL_SPECIFICATIONS:

            batch_auprc, batch_auroc = (
                calculate_grouped_weighted_metrics(
                    cache=score_group_caches[
                        model_key
                    ],
                    count_matrix=bootstrap_counts,
                    positive_totals=bootstrap_positive_totals,
                )
            )

            bootstrap_metrics[
                model_key
            ]["auprc"][
                batch_start:batch_end
            ] = batch_auprc

            bootstrap_metrics[
                model_key
            ]["auroc"][
                batch_start:batch_end
            ] = batch_auroc

        completed = batch_end

        if (
            completed % 250 == 0
            or completed == N_BOOTSTRAP
        ):

            elapsed = (
                time.time()
                - gene_start_time
            )

            valid_so_far = int(
                np.isfinite(
                    bootstrap_metrics[
                        "full_ges"
                    ]["auprc"][
                        :completed
                    ]
                ).sum()
            )

            print(
                f"  Completed {completed:,}/{N_BOOTSTRAP:,} "
                f"replicates | valid {valid_so_far:,} | "
                f"elapsed {elapsed:.1f} seconds"
            )

        del bootstrap_counts
        gc.collect()

    valid_replicate_mask = np.isfinite(
        bootstrap_metrics[
            "full_ges"
        ]["auprc"]
    )

    valid_replicates = int(
        valid_replicate_mask.sum()
    )

    invalid_one_class_replicates = int(
        N_BOOTSTRAP
        - valid_replicates
    )

    if valid_replicates < MINIMUM_VALID_REPLICATES:
        raise AssertionError(
            f"{gene} produced only "
            f"{valid_replicates:,} valid bootstrap replicates; "
            f"minimum required is "
            f"{MINIMUM_VALID_REPLICATES:,}."
        )

    for model_key in MODEL_SPECIFICATIONS:

        for metric_name in [
            "auprc",
            "auroc",
        ]:

            model_valid_mask = np.isfinite(
                bootstrap_metrics[
                    model_key
                ][metric_name]
            )

            if not np.array_equal(
                model_valid_mask,
                valid_replicate_mask,
            ):
                raise AssertionError(
                    f"Paired-bootstrap validity mismatch for "
                    f"{gene}, model {model_key}, "
                    f"metric {metric_name}."
                )

    # ----------------------------------------------------------------------------------------------
    # Model-specific intervals
    # ----------------------------------------------------------------------------------------------

    for model_key, specification in MODEL_SPECIFICATIONS.items():

        auprc_values = bootstrap_metrics[
            model_key
        ]["auprc"]

        auroc_values = bootstrap_metrics[
            model_key
        ]["auroc"]

        auprc_minus_prevalence_values = (
            auprc_values
            - bootstrap_prevalence
        )

        auroc_minus_half_values = (
            auroc_values
            - 0.50
        )

        (
            auprc_lower,
            auprc_upper,
        ) = percentile_interval(
            auprc_values
        )

        (
            auroc_lower,
            auroc_upper,
        ) = percentile_interval(
            auroc_values
        )

        (
            auprc_null_lower,
            auprc_null_upper,
        ) = percentile_interval(
            auprc_minus_prevalence_values
        )

        (
            auroc_null_lower,
            auroc_null_upper,
        ) = percentile_interval(
            auroc_minus_half_values
        )

        model_interval_rows.append(
            {
                "gene": gene,
                "analysis_role": analysis_role,
                "model_key": model_key,
                "model": specification[
                    "display_name"
                ],
                "rows": number_of_rows,
                "events": number_of_events,
                "negatives": number_of_negatives,
                "gene_prevalence": gene_prevalence,
                "point_auprc": point_metrics[
                    model_key
                ]["auprc"],
                "auprc_ci_lower": auprc_lower,
                "auprc_ci_upper": auprc_upper,
                "point_auprc_minus_prevalence": (
                    point_metrics[
                        model_key
                    ]["auprc"]
                    - gene_prevalence
                ),
                "auprc_minus_prevalence_ci_lower": (
                    auprc_null_lower
                ),
                "auprc_minus_prevalence_ci_upper": (
                    auprc_null_upper
                ),
                "auprc_null_status": interval_status(
                    auprc_null_lower,
                    auprc_null_upper,
                    "supported_above_gene_prevalence",
                    "supported_below_gene_prevalence",
                ),
                "point_auroc": point_metrics[
                    model_key
                ]["auroc"],
                "auroc_ci_lower": auroc_lower,
                "auroc_ci_upper": auroc_upper,
                "point_auroc_minus_0_50": (
                    point_metrics[
                        model_key
                    ]["auroc"]
                    - 0.50
                ),
                "auroc_minus_0_50_ci_lower": (
                    auroc_null_lower
                ),
                "auroc_minus_0_50_ci_upper": (
                    auroc_null_upper
                ),
                "auroc_null_status": interval_status(
                    auroc_null_lower,
                    auroc_null_upper,
                    "supported_above_0_50",
                    "supported_below_0_50",
                ),
                "attempted_bootstrap_replicates": (
                    N_BOOTSTRAP
                ),
                "valid_bootstrap_replicates": (
                    valid_replicates
                ),
                "invalid_one_class_replicates": (
                    invalid_one_class_replicates
                ),
                "sparse_event_or_negative_flag": bool(
                    min(
                        number_of_events,
                        number_of_negatives,
                    ) < 20
                ),
            }
        )

    # ----------------------------------------------------------------------------------------------
    # Paired Full-GES-minus-comparator inference
    # ----------------------------------------------------------------------------------------------

    for comparator_key in PAIRED_COMPARATORS:

        comparator_display_name = (
            MODEL_SPECIFICATIONS[
                comparator_key
            ]["display_name"]
        )

        for metric_name in [
            "auprc",
            "auroc",
        ]:

            full_values = bootstrap_metrics[
                "full_ges"
            ][metric_name]

            comparator_values = bootstrap_metrics[
                comparator_key
            ][metric_name]

            paired_differences = (
                full_values
                - comparator_values
            )

            finite_differences = (
                paired_differences[
                    np.isfinite(
                        paired_differences
                    )
                ]
            )

            (
                difference_lower,
                difference_upper,
            ) = percentile_interval(
                finite_differences
            )

            point_difference = (
                point_metrics[
                    "full_ges"
                ][metric_name]
                - point_metrics[
                    comparator_key
                ][metric_name]
            )

            paired_difference_rows.append(
                {
                    "metric": metric_name.upper(),
                    "gene": gene,
                    "analysis_role": analysis_role,
                    "comparison": (
                        f"Full GES minus "
                        f"{comparator_display_name}"
                    ),
                    "comparator_key": comparator_key,
                    "rows": number_of_rows,
                    "events": number_of_events,
                    "negatives": number_of_negatives,
                    "point_difference": point_difference,
                    "difference_ci_lower": difference_lower,
                    "difference_ci_upper": difference_upper,
                    "paired_interval_status": interval_status(
                        difference_lower,
                        difference_upper,
                        "full_ges_supported_higher",
                        "full_ges_supported_lower",
                    ),
                    "bootstrap_probability_full_greater": float(
                        np.mean(
                            finite_differences
                            > 0.0
                        )
                    ),
                    "bootstrap_probability_equal": float(
                        np.mean(
                            finite_differences
                            == 0.0
                        )
                    ),
                    "bootstrap_sign_p_value": (
                        bootstrap_sign_pvalue(
                            finite_differences
                        )
                    ),
                    "attempted_bootstrap_replicates": (
                        N_BOOTSTRAP
                    ),
                    "valid_bootstrap_replicates": (
                        valid_replicates
                    ),
                    "invalid_one_class_replicates": (
                        invalid_one_class_replicates
                    ),
                    "sparse_event_or_negative_flag": bool(
                        min(
                            number_of_events,
                            number_of_negatives,
                        ) < 20
                    ),
                }
            )

    gene_elapsed = (
        time.time()
        - gene_start_time
    )

    print(
        f"  {gene} completed: "
        f"{valid_replicates:,} valid, "
        f"{invalid_one_class_replicates:,} invalid "
        f"one-class replicates, "
        f"{gene_elapsed:.1f} seconds"
    )

    del stratum_df
    del score_arrays
    del score_group_caches
    del bootstrap_metrics
    del bootstrap_prevalence
    gc.collect()


# --------------------------------------------------------------------------------------------------
# 9. CREATE RESULT TABLES
# --------------------------------------------------------------------------------------------------

gene_level_point_estimates = pd.DataFrame(
    point_estimate_rows
)

gene_level_model_intervals = pd.DataFrame(
    model_interval_rows
)

gene_level_paired_differences = pd.DataFrame(
    paired_difference_rows
)


# --------------------------------------------------------------------------------------------------
# 10. PRIMARY-GENE HOLM CORRECTION
#
# Separate families:
#   - 9 primary-gene AUPRC comparisons
#   - 9 primary-gene AUROC comparisons
#
# EGFR is exploratory and excluded from these correction families.
# --------------------------------------------------------------------------------------------------

gene_level_paired_differences[
    "primary_gene_holm_adjusted_bootstrap_sign_p"
] = np.nan

gene_level_paired_differences[
    "multiplicity_family"
] = np.where(
    gene_level_paired_differences[
        "analysis_role"
    ] == "primary",
    "primary_gene_family",
    "exploratory_egfr_unadjusted",
)

for metric_name in [
    "AUPRC",
    "AUROC",
]:

    primary_family_mask = (
        (
            gene_level_paired_differences[
                "metric"
            ] == metric_name
        )
        & (
            gene_level_paired_differences[
                "analysis_role"
            ] == "primary"
        )
    )

    family_p_values = (
        gene_level_paired_differences.loc[
            primary_family_mask,
            "bootstrap_sign_p_value",
        ]
        .to_numpy(dtype=float)
    )

    if len(family_p_values) != 9:
        raise AssertionError(
            f"{metric_name} primary-gene multiplicity "
            f"family contains {len(family_p_values)} tests; "
            "expected 9."
        )

    adjusted_values = holm_adjust(
        family_p_values
    )

    gene_level_paired_differences.loc[
        primary_family_mask,
        "primary_gene_holm_adjusted_bootstrap_sign_p",
    ] = adjusted_values


gene_level_paired_differences[
    "primary_gene_holm_supported_at_0_05"
] = pd.Series(
    pd.NA,
    index=gene_level_paired_differences.index,
    dtype="boolean",
)

primary_rows_mask = (
    gene_level_paired_differences[
        "analysis_role"
    ] == "primary"
)

gene_level_paired_differences.loc[
    primary_rows_mask,
    "primary_gene_holm_supported_at_0_05",
] = (
    gene_level_paired_differences.loc[
        primary_rows_mask,
        "primary_gene_holm_adjusted_bootstrap_sign_p",
    ] < 0.05
)

gene_level_paired_differences[
    "exploratory_raw_p_supported_at_0_05"
] = pd.Series(
    pd.NA,
    index=gene_level_paired_differences.index,
    dtype="boolean",
)

exploratory_rows_mask = (
    gene_level_paired_differences[
        "analysis_role"
    ] == "exploratory"
)

gene_level_paired_differences.loc[
    exploratory_rows_mask,
    "exploratory_raw_p_supported_at_0_05",
] = (
    gene_level_paired_differences.loc[
        exploratory_rows_mask,
        "bootstrap_sign_p_value",
    ] < 0.05
)


# --------------------------------------------------------------------------------------------------
# 11. COMPLETENESS AND CONSISTENCY CHECKS
# --------------------------------------------------------------------------------------------------

expected_model_rows = (
    len(GENE_DISPLAY_ORDER)
    * len(MODEL_SPECIFICATIONS)
)

if len(
    gene_level_model_intervals
) != expected_model_rows:
    raise AssertionError(
        f"Unexpected model-interval row count: "
        f"{len(gene_level_model_intervals)}; "
        f"expected {expected_model_rows}."
    )

expected_paired_rows = (
    len(GENE_DISPLAY_ORDER)
    * len(PAIRED_COMPARATORS)
    * 2
)

if len(
    gene_level_paired_differences
) != expected_paired_rows:
    raise AssertionError(
        f"Unexpected paired-comparison row count: "
        f"{len(gene_level_paired_differences)}; "
        f"expected {expected_paired_rows}."
    )

if (
    gene_level_model_intervals[
        "valid_bootstrap_replicates"
    ] < MINIMUM_VALID_REPLICATES
).any():
    raise AssertionError(
        "At least one gene/model has too few "
        "valid bootstrap replicates."
    )

if (
    gene_level_paired_differences.loc[
        primary_rows_mask,
        "primary_gene_holm_adjusted_bootstrap_sign_p",
    ]
    .isna()
    .any()
):
    raise AssertionError(
        "Missing primary-gene Holm-adjusted "
        "bootstrap sign probability detected."
    )

if (
    gene_level_paired_differences.loc[
        exploratory_rows_mask,
        "primary_gene_holm_adjusted_bootstrap_sign_p",
    ]
    .notna()
    .any()
):
    raise AssertionError(
        "EGFR was incorrectly included in the "
        "primary-gene multiplicity family."
    )

for metric_name in [
    "AUPRC",
    "AUROC",
]:

    primary_metric_rows = (
        gene_level_paired_differences.loc[
            (
                gene_level_paired_differences[
                    "metric"
                ] == metric_name
            )
            & primary_rows_mask
        ]
    )

    if len(
        primary_metric_rows
    ) != 9:
        raise AssertionError(
            f"{metric_name} primary-gene family contains "
            f"{len(primary_metric_rows)} rows; expected 9."
        )


# --------------------------------------------------------------------------------------------------
# 12. COMPACT DISPLAY TABLES
# --------------------------------------------------------------------------------------------------

model_display_columns = [
    "gene",
    "analysis_role",
    "model",
    "rows",
    "events",
    "negatives",
    "gene_prevalence",
    "point_auprc",
    "auprc_ci_lower",
    "auprc_ci_upper",
    "point_auprc_minus_prevalence",
    "auprc_minus_prevalence_ci_lower",
    "auprc_minus_prevalence_ci_upper",
    "auprc_null_status",
    "point_auroc",
    "auroc_ci_lower",
    "auroc_ci_upper",
    "point_auroc_minus_0_50",
    "auroc_minus_0_50_ci_lower",
    "auroc_minus_0_50_ci_upper",
    "auroc_null_status",
    "valid_bootstrap_replicates",
    "invalid_one_class_replicates",
    "sparse_event_or_negative_flag",
]

paired_display_columns = [
    "metric",
    "gene",
    "analysis_role",
    "comparison",
    "rows",
    "events",
    "negatives",
    "point_difference",
    "difference_ci_lower",
    "difference_ci_upper",
    "paired_interval_status",
    "bootstrap_probability_full_greater",
    "bootstrap_sign_p_value",
    "primary_gene_holm_adjusted_bootstrap_sign_p",
    "primary_gene_holm_supported_at_0_05",
    "exploratory_raw_p_supported_at_0_05",
    "multiplicity_family",
    "valid_bootstrap_replicates",
    "invalid_one_class_replicates",
    "sparse_event_or_negative_flag",
]

gene_level_model_display = (
    gene_level_model_intervals[
        model_display_columns
    ].copy()
)

gene_level_paired_display = (
    gene_level_paired_differences[
        paired_display_columns
    ].copy()
)


# --------------------------------------------------------------------------------------------------
# 13. PRINT COMPLETE LOCKED INFERENCE RECORD
# --------------------------------------------------------------------------------------------------

analysis_elapsed = (
    time.time()
    - analysis_start_time
)

separator = "=" * 176
subseparator = "-" * 176

print("\n" + separator)

print(
    "STAGE 6C STEP 3C — CELL 6C-3C2 — "
    "OPTIMIZED LOCKED GENE-LEVEL PAIRED BOOTSTRAP INFERENCE"
)

print(separator)


print("\nFROZEN INPUT VERIFICATION")
print(subseparator)

print(
    f"Stage 6B evaluable-cohort SHA-256 : PASS "
    f"({observed_sha256})"
)

print(
    f"Parquet dimensions                : PASS "
    f"({metadata.num_rows:,} × {metadata.num_columns})"
)

print(
    f"Unique evaluable RCV keys         : PASS "
    f"({analysis_df['rcv_accession'].nunique():,})"
)

print(
    f"Primary instability events        : PASS "
    f"({event_count:,})"
)

print(
    f"Primary instability negatives     : PASS "
    f"({negative_count:,})"
)

print(
    f"Overall event prevalence          : "
    f"{analysis_df[OUTCOME_COLUMN].mean():.8f} "
    f"({analysis_df[OUTCOME_COLUMN].mean():.6%})"
)

print(
    f"Bootstrap attempts per gene       : "
    f"{N_BOOTSTRAP:,}"
)

print(
    f"Bootstrap batch size              : "
    f"{BOOTSTRAP_BATCH_SIZE}"
)

print(
    f"Bootstrap random seed             : "
    f"{RANDOM_SEED}"
)

print(
    "Bootstrap representation         : "
    "Exact multinomial row multiplicities"
)

print(
    "Paired resampling                 : "
    "Identical bootstrap samples across all models within each gene"
)

print(
    "Tie handling                      : "
    "Exact grouped weighted AUPRC/AUROC"
)

print(
    "Fast metric validation            : "
    "Matched scikit-learn point estimates for every gene/model"
)

print(
    f"Percentile interval               : "
    f"95% ({CI_LOWER_QUANTILE:.3f}, "
    f"{CI_UPPER_QUANTILE:.3f})"
)

print(
    "Primary multiplicity families     : "
    "Nine AUPRC and nine AUROC comparisons across BRCA1, BRCA2, and MLH1"
)

print(
    "EGFR multiplicity treatment       : "
    "Exploratory and excluded from primary-gene Holm correction"
)

print("Score direction changed           : No")
print("Threshold or weight optimized     : No")
print("Recalibration performed           : No")
print("Scientific artifact written       : No")

print(
    f"Total analysis runtime            : "
    f"{analysis_elapsed:.1f} seconds"
)


print("\nGENE-LEVEL MODEL-SPECIFIC BOOTSTRAP INTERVALS")
print(subseparator)

with pd.option_context(
    "display.max_columns", None,
    "display.width", 390,
    "display.float_format",
    lambda value: f"{value:.8f}",
):
    print(
        gene_level_model_display.to_string(
            index=False
        )
    )


print("\nPAIRED FULL-GES-MINUS-COMPARATOR INFERENCE")
print(subseparator)

with pd.option_context(
    "display.max_columns", None,
    "display.width", 390,
    "display.float_format",
    lambda value: f"{value:.8f}",
):
    print(
        gene_level_paired_display.to_string(
            index=False
        )
    )


print("\nGENE-LEVEL ACCOUNTING")
print(subseparator)

with pd.option_context(
    "display.float_format",
    lambda value: f"{value:.8f}",
):
    print(
        gene_accounting.to_string(
            index=False
        )
    )


print("\nINTERPRETATION BOUNDARY")
print(subseparator)

print(
    "BRCA1, BRCA2, and MLH1 constitute the prespecified "
    "primary-gene subgroup family."
)

print(
    "EGFR is retained as a separate exploratory subgroup because "
    "its interpretation context may include different somatic and "
    "germline processes."
)

print(
    "Gene-specific bootstrap intervals describe performance within "
    "the frozen cohort and do not constitute leave-one-gene-out "
    "model retraining or external validation."
)


print("\nCELL DECISION")
print(subseparator)

print(
    "PASS_STAGE6C_GENE_LEVEL_BOOTSTRAP_INFERENCE_COMPLETE"
)

print(
    "Two thousand exact ordinary row-bootstrap attempts were completed "
    "separately within BRCA1, BRCA2, MLH1, and EGFR."
)

print(
    "Model-specific intervals, null-reference comparisons, paired "
    "Full-GES-minus-comparator intervals, bootstrap sign probabilities, "
    "and primary-gene Holm-adjusted probabilities were calculated."
)

print(
    "EGFR remained exploratory and was not included in the "
    "primary-gene multiplicity families."
)

print(
    "No score, outcome, gene assignment, cohort membership, "
    "threshold, weight, frozen model, or scientific artifact "
    "was modified."
)


Preparing BRCA1 bootstrap: 21,594 rows, 2,023 events, 19,571 negatives, role=primary
  Fast metric validation against scikit-learn: PASS
  Completed 250/2,000 replicates | valid 250 | elapsed 2.1 seconds
  Completed 500/2,000 replicates | valid 500 | elapsed 4.8 seconds
  Completed 750/2,000 replicates | valid 750 | elapsed 6.6 seconds
  Completed 1,000/2,000 replicates | valid 1,000 | elapsed 9.4 seconds
  Completed 1,250/2,000 replicates | valid 1,250 | elapsed 12.9 seconds
  Completed 1,500/2,000 replicates | valid 1,500 | elapsed 14.8 seconds
  Completed 1,750/2,000 replicates | valid 1,750 | elapsed 16.8 seconds
  Completed 2,000/2,000 replicates | valid 2,000 | elapsed 18.8 seconds
  BRCA1 completed: 2,000 valid, 0 invalid one-class replicates, 19.0 seconds

Preparing BRCA2 bootstrap: 34,152 rows, 3,960 events, 30,192 negatives, role=primary
  Fast metric validation against scikit-learn: PASS
  Completed 250/2,000 replicates | valid 250 | elapsed 2.5 seconds
  Completed 500/2,00

In [31]:
# ==================================================================================================
# STAGE 6C STEP 3C — CELL 6C-3C2
# OPTIMIZED LOCKED GENE-LEVEL PAIRED BOOTSTRAP INFERENCE
#
# Runs 2,000 paired row-bootstrap replicates separately for BRCA1, BRCA2, MLH1, and EGFR.
# Produces gene-specific AUPRC/AUROC confidence intervals and Full-GES-minus-comparator
# paired inference. Holm correction is applied separately to the 9 primary-gene AUPRC
# comparisons and the 9 primary-gene AUROC comparisons. EGFR remains exploratory.
#
# This cell is read-only: it does not modify scores, outcomes, genes, thresholds, weights,
# cohort membership, or frozen scientific artifacts.
# ==================================================================================================

from pathlib import Path
import gc
import hashlib
import re
import time

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from scipy.sparse import csr_matrix
from sklearn.metrics import average_precision_score, roc_auc_score


# --------------------------------------------------------------------------------------------------
# 1. FROZEN INPUT AND ANALYSIS SPECIFICATION
# --------------------------------------------------------------------------------------------------

STAGE6_DIR = Path(
    "/content/drive/MyDrive/GES_RAG_Temporal_Study/"
    "data_processed/stage6_temporal_validation"
)

EVALUABLE_PARQUET = (
    STAGE6_DIR / "stage6b_locked_primary_evaluable_cohort_v1.parquet"
)
EVALUABLE_SIDECAR = Path(str(EVALUABLE_PARQUET) + ".sha256")

EXPECTED_EVALUABLE_SHA256 = (
    "c6ad50dfc376a746e1830d000dba975101d88763fd61b8d1983f6300edb6e038"
)

EXPECTED_ROWS = 66_636
EXPECTED_COLUMNS = 79
EXPECTED_EVENTS = 6_485
EXPECTED_NEGATIVES = 60_151

OUTCOME_COLUMN = "primary_future_instability"
GENE_COLUMN = "target_gene"

N_BOOTSTRAP = 2_000
RANDOM_SEED = 42
BOOTSTRAP_BATCH_SIZE = 50
MINIMUM_VALID_REPLICATES = 1_000
CI_QUANTILES = (0.025, 0.975)

PRIMARY_GENES = ["BRCA1", "BRCA2", "MLH1"]
GENE_DISPLAY_ORDER = ["BRCA1", "BRCA2", "MLH1", "EGFR"]

MODEL_SPECIFICATIONS = {
    "full_ges": {
        "column": "full_ges_instability_risk_t0",
        "display_name": "Full GES",
    },
    "no_star_ges": {
        "column": "no_star_ges_instability_risk_t0",
        "display_name": "No-star GES",
    },
    "review_stars": {
        "column": "review_stars_instability_risk",
        "display_name": "Review stars",
    },
    "combined_metadata": {
        "column": "combined_metadata_instability_risk",
        "display_name": "Combined metadata",
    },
}

PAIRED_COMPARATORS = ["no_star_ges", "review_stars", "combined_metadata"]

EXPECTED_GENE_ACCOUNTING = {
    "BRCA1": {"rows": 21_594, "events": 2_023, "negatives": 19_571},
    "BRCA2": {"rows": 34_152, "events": 3_960, "negatives": 30_192},
    "MLH1": {"rows": 8_701, "events": 425, "negatives": 8_276},
    "EGFR": {"rows": 2_189, "events": 77, "negatives": 2_112},
}


# --------------------------------------------------------------------------------------------------
# 2. HELPERS
# --------------------------------------------------------------------------------------------------

def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def read_sidecar_hash(path: Path) -> str:
    text = path.read_text(encoding="utf-8").strip()
    matches = re.findall(r"\b[a-fA-F0-9]{64}\b", text)
    if not matches:
        raise ValueError(f"No SHA-256 value found in sidecar: {path}")
    return matches[0].lower()


def percentile_interval(values: np.ndarray) -> tuple[float, float]:
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if len(values) == 0:
        return np.nan, np.nan
    lower, upper = np.quantile(values, CI_QUANTILES)
    return float(lower), float(upper)


def bootstrap_sign_pvalue(differences: np.ndarray) -> float:
    differences = np.asarray(differences, dtype=float)
    differences = differences[np.isfinite(differences)]
    if len(differences) == 0:
        return np.nan

    n = len(differences)
    lower_tail = (np.count_nonzero(differences <= 0.0) + 1) / (n + 1)
    upper_tail = (np.count_nonzero(differences >= 0.0) + 1) / (n + 1)
    return float(min(1.0, 2.0 * min(lower_tail, upper_tail)))


def holm_adjust(p_values: np.ndarray) -> np.ndarray:
    p_values = np.asarray(p_values, dtype=float)
    adjusted = np.full(len(p_values), np.nan, dtype=float)
    valid_positions = np.where(np.isfinite(p_values))[0]

    if len(valid_positions) == 0:
        return adjusted

    valid_p = p_values[valid_positions]
    order = np.argsort(valid_p)
    m = len(valid_p)
    running_max = 0.0

    for rank, position_within_valid in enumerate(order):
        original_position = valid_positions[position_within_valid]
        raw_adjusted = (m - rank) * valid_p[position_within_valid]
        running_max = max(running_max, raw_adjusted)
        adjusted[original_position] = min(1.0, running_max)

    return adjusted


def interval_status(
    lower: float,
    upper: float,
    positive_label: str,
    negative_label: str,
) -> str:
    if not np.isfinite(lower) or not np.isfinite(upper):
        return "not_estimable"
    if lower > 0.0:
        return positive_label
    if upper < 0.0:
        return negative_label
    return "interval_includes_null"


def construct_score_group_cache(scores: np.ndarray, outcomes: np.ndarray) -> dict:
    scores = np.asarray(scores, dtype=np.float64)
    outcomes = np.asarray(outcomes, dtype=np.int8)

    unique_scores, group_index = np.unique(scores, return_inverse=True)
    n_rows = len(scores)
    n_groups = len(unique_scores)
    row_positions = np.arange(n_rows)

    total_group_matrix = csr_matrix(
        (
            np.ones(n_rows, dtype=np.float64),
            (group_index, row_positions),
        ),
        shape=(n_groups, n_rows),
    )

    positive_positions = np.flatnonzero(outcomes == 1)
    positive_group_matrix = csr_matrix(
        (
            np.ones(len(positive_positions), dtype=np.float64),
            (group_index[positive_positions], positive_positions),
        ),
        shape=(n_groups, n_rows),
    )

    return {
        "unique_scores": unique_scores,
        "total_group_matrix": total_group_matrix,
        "positive_group_matrix": positive_group_matrix,
    }


def calculate_grouped_weighted_metrics(
    cache: dict,
    count_matrix: np.ndarray,
    positive_totals: np.ndarray,
) -> tuple[np.ndarray, np.ndarray]:
    count_matrix = np.asarray(count_matrix)
    positive_totals = np.asarray(positive_totals, dtype=np.float64)

    sample_totals = count_matrix.sum(axis=1).astype(np.float64)
    negative_totals = sample_totals - positive_totals

    group_total_counts = np.asarray(
        cache["total_group_matrix"] @ count_matrix.T,
        dtype=np.float64,
    )
    group_positive_counts = np.asarray(
        cache["positive_group_matrix"] @ count_matrix.T,
        dtype=np.float64,
    )
    group_negative_counts = group_total_counts - group_positive_counts

    valid = (positive_totals > 0.0) & (negative_totals > 0.0)

    # AUROC: ascending score groups, with 0.5 credit for ties.
    cumulative_negatives_before = (
        np.cumsum(group_negative_counts, axis=0) - group_negative_counts
    )
    concordant_numerator = np.sum(
        group_positive_counts
        * (cumulative_negatives_before + 0.5 * group_negative_counts),
        axis=0,
    )
    auc_denominator = positive_totals * negative_totals
    auroc = np.full(len(positive_totals), np.nan, dtype=np.float64)
    np.divide(
        concordant_numerator,
        auc_denominator,
        out=auroc,
        where=valid,
    )

    # AUPRC / average precision: descending score groups.
    positive_desc = group_positive_counts[::-1, :]
    total_desc = group_total_counts[::-1, :]
    cumulative_positive = np.cumsum(positive_desc, axis=0)
    cumulative_total = np.cumsum(total_desc, axis=0)

    precision = np.zeros_like(cumulative_positive, dtype=np.float64)
    np.divide(
        cumulative_positive,
        cumulative_total,
        out=precision,
        where=cumulative_total > 0.0,
    )

    ap_numerator = np.sum(precision * positive_desc, axis=0)
    auprc = np.full(len(positive_totals), np.nan, dtype=np.float64)
    np.divide(
        ap_numerator,
        positive_totals,
        out=auprc,
        where=valid,
    )

    return auprc, auroc


# --------------------------------------------------------------------------------------------------
# 3. FRESH CRYPTOGRAPHIC AND STRUCTURAL VERIFICATION
# --------------------------------------------------------------------------------------------------

if not EVALUABLE_PARQUET.exists():
    raise FileNotFoundError(f"Missing frozen evaluable cohort:\n{EVALUABLE_PARQUET}")

if not EVALUABLE_SIDECAR.exists():
    raise FileNotFoundError(f"Missing SHA-256 sidecar:\n{EVALUABLE_SIDECAR}")

observed_sha256 = sha256_file(EVALUABLE_PARQUET)
sidecar_sha256 = read_sidecar_hash(EVALUABLE_SIDECAR)

if observed_sha256 != EXPECTED_EVALUABLE_SHA256:
    raise AssertionError(
        "Frozen Stage 6B evaluable-cohort SHA-256 mismatch.\n"
        f"Expected: {EXPECTED_EVALUABLE_SHA256}\n"
        f"Observed: {observed_sha256}"
    )

if sidecar_sha256 != observed_sha256:
    raise AssertionError(
        "Frozen evaluable-cohort sidecar mismatch.\n"
        f"Calculated: {observed_sha256}\n"
        f"Sidecar:    {sidecar_sha256}"
    )

parquet_file = pq.ParquetFile(EVALUABLE_PARQUET)
metadata = parquet_file.metadata
schema_columns = parquet_file.schema_arrow.names

if metadata.num_rows != EXPECTED_ROWS:
    raise AssertionError(
        f"Unexpected row count: {metadata.num_rows:,}; expected {EXPECTED_ROWS:,}"
    )

if metadata.num_columns != EXPECTED_COLUMNS:
    raise AssertionError(
        f"Unexpected column count: {metadata.num_columns}; expected {EXPECTED_COLUMNS}"
    )


# --------------------------------------------------------------------------------------------------
# 4. LOAD AND VALIDATE REQUIRED FROZEN COLUMNS
# --------------------------------------------------------------------------------------------------

required_columns = [
    "rcv_accession",
    "t0_row_order",
    OUTCOME_COLUMN,
    GENE_COLUMN,
] + [spec["column"] for spec in MODEL_SPECIFICATIONS.values()]

missing_columns = [c for c in required_columns if c not in schema_columns]
if missing_columns:
    raise KeyError("Missing required columns:\n" + "\n".join(missing_columns))

analysis_df = pd.read_parquet(
    EVALUABLE_PARQUET,
    columns=required_columns,
).copy()

if len(analysis_df) != EXPECTED_ROWS:
    raise AssertionError("Loaded dataframe row count does not match the frozen cohort.")

if analysis_df["rcv_accession"].isna().any():
    raise AssertionError("Missing RCV accession detected.")

if analysis_df["rcv_accession"].astype(str).str.strip().eq("").any():
    raise AssertionError("Blank RCV accession detected.")

if analysis_df["rcv_accession"].nunique(dropna=False) != EXPECTED_ROWS:
    raise AssertionError("RCV accessions are not unique.")

row_order = pd.to_numeric(analysis_df["t0_row_order"], errors="raise").to_numpy()
if len(np.unique(row_order)) != EXPECTED_ROWS:
    raise AssertionError("t0_row_order is not unique.")
if not np.all(np.diff(row_order) > 0):
    raise AssertionError("Frozen evaluable cohort is not in increasing T0 row order.")

analysis_df[OUTCOME_COLUMN] = pd.to_numeric(
    analysis_df[OUTCOME_COLUMN],
    errors="raise",
).astype(int)

if not set(analysis_df[OUTCOME_COLUMN].unique()).issubset({0, 1}):
    raise AssertionError("Primary outcome is not binary.")

event_count = int(analysis_df[OUTCOME_COLUMN].sum())
negative_count = int((analysis_df[OUTCOME_COLUMN] == 0).sum())

if event_count != EXPECTED_EVENTS or negative_count != EXPECTED_NEGATIVES:
    raise AssertionError(
        f"Outcome accounting mismatch: events={event_count:,}, negatives={negative_count:,}"
    )

analysis_df[GENE_COLUMN] = (
    analysis_df[GENE_COLUMN].astype(str).str.strip().str.upper()
)

observed_genes = sorted(analysis_df[GENE_COLUMN].unique().tolist())
if observed_genes != ["BRCA1", "BRCA2", "EGFR", "MLH1"]:
    raise AssertionError(f"Unexpected genes: {observed_genes}")

for spec in MODEL_SPECIFICATIONS.values():
    column = spec["column"]
    analysis_df[column] = pd.to_numeric(analysis_df[column], errors="raise")
    values = analysis_df[column].to_numpy(dtype=float)

    if not np.isfinite(values).all():
        raise AssertionError(f"Missing/nonfinite score in {column}.")
    if ((values < 0.0) | (values > 1.0)).any():
        raise AssertionError(f"Score outside [0,1] in {column}.")


# --------------------------------------------------------------------------------------------------
# 5. VERIFY GENE ACCOUNTING
# --------------------------------------------------------------------------------------------------

gene_accounting_rows = []

for gene in GENE_DISPLAY_ORDER:
    gene_df = analysis_df.loc[analysis_df[GENE_COLUMN] == gene]
    rows = int(len(gene_df))
    events = int(gene_df[OUTCOME_COLUMN].sum())
    negatives = rows - events
    expected = EXPECTED_GENE_ACCOUNTING[gene]

    if (rows, events, negatives) != (
        expected["rows"],
        expected["events"],
        expected["negatives"],
    ):
        raise AssertionError(
            f"{gene} accounting mismatch: "
            f"observed {(rows, events, negatives)}, "
            f"expected {(expected['rows'], expected['events'], expected['negatives'])}"
        )

    gene_accounting_rows.append(
        {
            "gene": gene,
            "analysis_role": "primary" if gene in PRIMARY_GENES else "exploratory",
            "rows": rows,
            "events": events,
            "negatives": negatives,
            "event_prevalence": events / rows,
        }
    )

gene_accounting = pd.DataFrame(gene_accounting_rows)

if int(gene_accounting["rows"].sum()) != EXPECTED_ROWS:
    raise AssertionError("Gene row accounting failed.")
if int(gene_accounting["events"].sum()) != EXPECTED_EVENTS:
    raise AssertionError("Gene event accounting failed.")
if int(gene_accounting["negatives"].sum()) != EXPECTED_NEGATIVES:
    raise AssertionError("Gene negative accounting failed.")


# --------------------------------------------------------------------------------------------------
# 6. RUN OPTIMIZED EXACT GENE-LEVEL BOOTSTRAP
# --------------------------------------------------------------------------------------------------

rng = np.random.default_rng(RANDOM_SEED)

point_estimate_rows = []
model_interval_rows = []
paired_difference_rows = []
analysis_start_time = time.time()

for gene in GENE_DISPLAY_ORDER:
    gene_start_time = time.time()
    analysis_role = "primary" if gene in PRIMARY_GENES else "exploratory"

    stratum_df = (
        analysis_df.loc[analysis_df[GENE_COLUMN] == gene]
        .reset_index(drop=True)
    )

    outcomes = stratum_df[OUTCOME_COLUMN].to_numpy(dtype=np.int8)
    n_rows = int(len(stratum_df))
    n_events = int(outcomes.sum())
    n_negatives = n_rows - n_events
    prevalence = n_events / n_rows

    if len(np.unique(outcomes)) != 2:
        raise AssertionError(f"{gene} does not contain both outcome classes.")

    score_arrays = {
        key: stratum_df[spec["column"]].to_numpy(dtype=np.float64)
        for key, spec in MODEL_SPECIFICATIONS.items()
    }

    caches = {
        key: construct_score_group_cache(scores, outcomes)
        for key, scores in score_arrays.items()
    }

    point_metrics = {}

    print(
        f"\nPreparing {gene}: {n_rows:,} rows, {n_events:,} events, "
        f"{n_negatives:,} negatives, role={analysis_role}"
    )

    original_counts = np.ones((1, n_rows), dtype=np.int16)
    original_positive_total = np.array([n_events], dtype=np.float64)

    # Validate optimized metrics against scikit-learn and retain point estimates.
    for model_key, spec in MODEL_SPECIFICATIONS.items():
        scores = score_arrays[model_key]

        sklearn_auprc = float(average_precision_score(outcomes, scores))
        sklearn_auroc = float(roc_auc_score(outcomes, scores))

        fast_auprc, fast_auroc = calculate_grouped_weighted_metrics(
            caches[model_key],
            original_counts,
            original_positive_total,
        )

        if not np.isclose(fast_auprc[0], sklearn_auprc, rtol=1e-11, atol=1e-12):
            raise AssertionError(
                f"Fast AUPRC validation failed for {gene}/{model_key}: "
                f"{fast_auprc[0]:.15f} vs {sklearn_auprc:.15f}"
            )

        if not np.isclose(fast_auroc[0], sklearn_auroc, rtol=1e-11, atol=1e-12):
            raise AssertionError(
                f"Fast AUROC validation failed for {gene}/{model_key}: "
                f"{fast_auroc[0]:.15f} vs {sklearn_auroc:.15f}"
            )

        point_metrics[model_key] = {
            "auprc": sklearn_auprc,
            "auroc": sklearn_auroc,
        }

        point_estimate_rows.append(
            {
                "gene": gene,
                "analysis_role": analysis_role,
                "model_key": model_key,
                "model": spec["display_name"],
                "rows": n_rows,
                "events": n_events,
                "negatives": n_negatives,
                "gene_prevalence": prevalence,
                "point_auprc": sklearn_auprc,
                "point_auprc_minus_prevalence": sklearn_auprc - prevalence,
                "point_auroc": sklearn_auroc,
                "point_auroc_minus_0_50": sklearn_auroc - 0.50,
                "fast_metric_validation": "PASS",
            }
        )

    print("  Fast metric validation against scikit-learn: PASS")

    bootstrap_metrics = {
        model_key: {
            "auprc": np.full(N_BOOTSTRAP, np.nan, dtype=np.float64),
            "auroc": np.full(N_BOOTSTRAP, np.nan, dtype=np.float64),
        }
        for model_key in MODEL_SPECIFICATIONS
    }

    bootstrap_prevalence = np.full(N_BOOTSTRAP, np.nan, dtype=np.float64)

    probabilities = np.full(n_rows, 1.0 / n_rows, dtype=np.float64)
    probabilities[-1] = 1.0 - probabilities[:-1].sum()

    for batch_start in range(0, N_BOOTSTRAP, BOOTSTRAP_BATCH_SIZE):
        batch_end = min(batch_start + BOOTSTRAP_BATCH_SIZE, N_BOOTSTRAP)
        batch_size = batch_end - batch_start

        bootstrap_counts = rng.multinomial(
            n_rows,
            probabilities,
            size=batch_size,
        )

        if not np.all(bootstrap_counts.sum(axis=1) == n_rows):
            raise AssertionError(f"{gene} bootstrap sample-size preservation failed.")

        positive_totals = (bootstrap_counts @ outcomes).astype(np.float64)
        valid_outcomes = (positive_totals > 0.0) & (positive_totals < n_rows)

        bootstrap_prevalence[batch_start:batch_end] = np.where(
            valid_outcomes,
            positive_totals / n_rows,
            np.nan,
        )

        for model_key in MODEL_SPECIFICATIONS:
            batch_auprc, batch_auroc = calculate_grouped_weighted_metrics(
                caches[model_key],
                bootstrap_counts,
                positive_totals,
            )
            bootstrap_metrics[model_key]["auprc"][batch_start:batch_end] = batch_auprc
            bootstrap_metrics[model_key]["auroc"][batch_start:batch_end] = batch_auroc

        if batch_end % 250 == 0 or batch_end == N_BOOTSTRAP:
            valid_so_far = int(
                np.isfinite(
                    bootstrap_metrics["full_ges"]["auprc"][:batch_end]
                ).sum()
            )
            elapsed = time.time() - gene_start_time
            print(
                f"  Completed {batch_end:,}/{N_BOOTSTRAP:,} replicates | "
                f"valid {valid_so_far:,} | elapsed {elapsed:.1f}s"
            )

        del bootstrap_counts
        gc.collect()

    valid_mask = np.isfinite(bootstrap_metrics["full_ges"]["auprc"])
    valid_replicates = int(valid_mask.sum())
    invalid_one_class_replicates = N_BOOTSTRAP - valid_replicates

    if valid_replicates < MINIMUM_VALID_REPLICATES:
        raise AssertionError(
            f"{gene} produced only {valid_replicates:,} valid replicates."
        )

    # Confirm paired-validity masks match for every model and metric.
    for model_key in MODEL_SPECIFICATIONS:
        for metric_name in ["auprc", "auroc"]:
            model_valid = np.isfinite(bootstrap_metrics[model_key][metric_name])
            if not np.array_equal(model_valid, valid_mask):
                raise AssertionError(
                    f"Paired validity mismatch for {gene}/{model_key}/{metric_name}."
                )

    # Model-specific intervals.
    for model_key, spec in MODEL_SPECIFICATIONS.items():
        auprc_values = bootstrap_metrics[model_key]["auprc"]
        auroc_values = bootstrap_metrics[model_key]["auroc"]

        auprc_lower, auprc_upper = percentile_interval(auprc_values)
        auroc_lower, auroc_upper = percentile_interval(auroc_values)

        auprc_null_lower, auprc_null_upper = percentile_interval(
            auprc_values - bootstrap_prevalence
        )
        auroc_null_lower, auroc_null_upper = percentile_interval(
            auroc_values - 0.50
        )

        model_interval_rows.append(
            {
                "gene": gene,
                "analysis_role": analysis_role,
                "model_key": model_key,
                "model": spec["display_name"],
                "rows": n_rows,
                "events": n_events,
                "negatives": n_negatives,
                "gene_prevalence": prevalence,
                "point_auprc": point_metrics[model_key]["auprc"],
                "auprc_ci_lower": auprc_lower,
                "auprc_ci_upper": auprc_upper,
                "point_auprc_minus_prevalence": (
                    point_metrics[model_key]["auprc"] - prevalence
                ),
                "auprc_minus_prevalence_ci_lower": auprc_null_lower,
                "auprc_minus_prevalence_ci_upper": auprc_null_upper,
                "auprc_null_status": interval_status(
                    auprc_null_lower,
                    auprc_null_upper,
                    "supported_above_gene_prevalence",
                    "supported_below_gene_prevalence",
                ),
                "point_auroc": point_metrics[model_key]["auroc"],
                "auroc_ci_lower": auroc_lower,
                "auroc_ci_upper": auroc_upper,
                "point_auroc_minus_0_50": point_metrics[model_key]["auroc"] - 0.50,
                "auroc_minus_0_50_ci_lower": auroc_null_lower,
                "auroc_minus_0_50_ci_upper": auroc_null_upper,
                "auroc_null_status": interval_status(
                    auroc_null_lower,
                    auroc_null_upper,
                    "supported_above_0_50",
                    "supported_below_0_50",
                ),
                "attempted_bootstrap_replicates": N_BOOTSTRAP,
                "valid_bootstrap_replicates": valid_replicates,
                "invalid_one_class_replicates": invalid_one_class_replicates,
                "sparse_event_or_negative_flag": bool(
                    min(n_events, n_negatives) < 20
                ),
            }
        )

    # Paired Full-GES-minus-comparator inference.
    for comparator_key in PAIRED_COMPARATORS:
        comparator_name = MODEL_SPECIFICATIONS[comparator_key]["display_name"]

        for metric_name in ["auprc", "auroc"]:
            differences = (
                bootstrap_metrics["full_ges"][metric_name]
                - bootstrap_metrics[comparator_key][metric_name]
            )
            finite_differences = differences[np.isfinite(differences)]
            difference_lower, difference_upper = percentile_interval(
                finite_differences
            )

            point_difference = (
                point_metrics["full_ges"][metric_name]
                - point_metrics[comparator_key][metric_name]
            )

            paired_difference_rows.append(
                {
                    "metric": metric_name.upper(),
                    "gene": gene,
                    "analysis_role": analysis_role,
                    "comparison": f"Full GES minus {comparator_name}",
                    "comparator_key": comparator_key,
                    "rows": n_rows,
                    "events": n_events,
                    "negatives": n_negatives,
                    "point_difference": point_difference,
                    "difference_ci_lower": difference_lower,
                    "difference_ci_upper": difference_upper,
                    "paired_interval_status": interval_status(
                        difference_lower,
                        difference_upper,
                        "full_ges_supported_higher",
                        "full_ges_supported_lower",
                    ),
                    "bootstrap_probability_full_greater": float(
                        np.mean(finite_differences > 0.0)
                    ),
                    "bootstrap_probability_equal": float(
                        np.mean(finite_differences == 0.0)
                    ),
                    "bootstrap_sign_p_value": bootstrap_sign_pvalue(
                        finite_differences
                    ),
                    "attempted_bootstrap_replicates": N_BOOTSTRAP,
                    "valid_bootstrap_replicates": valid_replicates,
                    "invalid_one_class_replicates": invalid_one_class_replicates,
                    "sparse_event_or_negative_flag": bool(
                        min(n_events, n_negatives) < 20
                    ),
                }
            )

    print(
        f"  {gene} completed: {valid_replicates:,} valid, "
        f"{invalid_one_class_replicates:,} one-class invalid replicates, "
        f"{time.time() - gene_start_time:.1f}s"
    )

    del stratum_df, score_arrays, caches, bootstrap_metrics, bootstrap_prevalence
    gc.collect()


# --------------------------------------------------------------------------------------------------
# 7. CREATE RESULT TABLES AND APPLY PRIMARY-GENE HOLM CORRECTION
# --------------------------------------------------------------------------------------------------

gene_level_point_estimates = pd.DataFrame(point_estimate_rows)
gene_level_model_intervals = pd.DataFrame(model_interval_rows)
gene_level_paired_differences = pd.DataFrame(paired_difference_rows)

gene_level_paired_differences[
    "primary_gene_holm_adjusted_bootstrap_sign_p"
] = np.nan

gene_level_paired_differences["multiplicity_family"] = np.where(
    gene_level_paired_differences["analysis_role"] == "primary",
    "primary_gene_family",
    "exploratory_egfr_unadjusted",
)

for metric_name in ["AUPRC", "AUROC"]:
    family_mask = (
        (gene_level_paired_differences["metric"] == metric_name)
        & (gene_level_paired_differences["analysis_role"] == "primary")
    )

    family_p_values = gene_level_paired_differences.loc[
        family_mask,
        "bootstrap_sign_p_value",
    ].to_numpy(dtype=float)

    if len(family_p_values) != 9:
        raise AssertionError(
            f"{metric_name} primary family has {len(family_p_values)} tests; expected 9."
        )

    gene_level_paired_differences.loc[
        family_mask,
        "primary_gene_holm_adjusted_bootstrap_sign_p",
    ] = holm_adjust(family_p_values)

primary_rows_mask = (
    gene_level_paired_differences["analysis_role"] == "primary"
)
exploratory_rows_mask = (
    gene_level_paired_differences["analysis_role"] == "exploratory"
)

gene_level_paired_differences[
    "primary_gene_holm_supported_at_0_05"
] = pd.Series(
    pd.NA,
    index=gene_level_paired_differences.index,
    dtype="boolean",
)

gene_level_paired_differences.loc[
    primary_rows_mask,
    "primary_gene_holm_supported_at_0_05",
] = (
    gene_level_paired_differences.loc[
        primary_rows_mask,
        "primary_gene_holm_adjusted_bootstrap_sign_p",
    ] < 0.05
)

gene_level_paired_differences[
    "exploratory_raw_p_supported_at_0_05"
] = pd.Series(
    pd.NA,
    index=gene_level_paired_differences.index,
    dtype="boolean",
)

gene_level_paired_differences.loc[
    exploratory_rows_mask,
    "exploratory_raw_p_supported_at_0_05",
] = (
    gene_level_paired_differences.loc[
        exploratory_rows_mask,
        "bootstrap_sign_p_value",
    ] < 0.05
)


# --------------------------------------------------------------------------------------------------
# 8. COMPLETENESS CHECKS
# --------------------------------------------------------------------------------------------------

expected_model_rows = len(GENE_DISPLAY_ORDER) * len(MODEL_SPECIFICATIONS)
expected_paired_rows = len(GENE_DISPLAY_ORDER) * len(PAIRED_COMPARATORS) * 2

if len(gene_level_model_intervals) != expected_model_rows:
    raise AssertionError(
        f"Model interval rows={len(gene_level_model_intervals)}; "
        f"expected {expected_model_rows}."
    )

if len(gene_level_paired_differences) != expected_paired_rows:
    raise AssertionError(
        f"Paired rows={len(gene_level_paired_differences)}; "
        f"expected {expected_paired_rows}."
    )

if (
    gene_level_model_intervals["valid_bootstrap_replicates"]
    < MINIMUM_VALID_REPLICATES
).any():
    raise AssertionError("At least one gene/model has too few valid replicates.")

if gene_level_paired_differences.loc[
    primary_rows_mask,
    "primary_gene_holm_adjusted_bootstrap_sign_p",
].isna().any():
    raise AssertionError("Missing primary-gene Holm-adjusted value.")

if gene_level_paired_differences.loc[
    exploratory_rows_mask,
    "primary_gene_holm_adjusted_bootstrap_sign_p",
].notna().any():
    raise AssertionError("EGFR was incorrectly included in Holm correction.")

for metric_name in ["AUPRC", "AUROC"]:
    n_primary = int(
        (
            (gene_level_paired_differences["metric"] == metric_name)
            & primary_rows_mask
        ).sum()
    )
    if n_primary != 9:
        raise AssertionError(
            f"{metric_name} primary family contains {n_primary} rows; expected 9."
        )


# --------------------------------------------------------------------------------------------------
# 9. DISPLAY TABLES AND FINAL DECISION
# --------------------------------------------------------------------------------------------------

model_display_columns = [
    "gene",
    "analysis_role",
    "model",
    "rows",
    "events",
    "negatives",
    "gene_prevalence",
    "point_auprc",
    "auprc_ci_lower",
    "auprc_ci_upper",
    "point_auprc_minus_prevalence",
    "auprc_minus_prevalence_ci_lower",
    "auprc_minus_prevalence_ci_upper",
    "auprc_null_status",
    "point_auroc",
    "auroc_ci_lower",
    "auroc_ci_upper",
    "point_auroc_minus_0_50",
    "auroc_minus_0_50_ci_lower",
    "auroc_minus_0_50_ci_upper",
    "auroc_null_status",
    "valid_bootstrap_replicates",
    "invalid_one_class_replicates",
    "sparse_event_or_negative_flag",
]

paired_display_columns = [
    "metric",
    "gene",
    "analysis_role",
    "comparison",
    "rows",
    "events",
    "negatives",
    "point_difference",
    "difference_ci_lower",
    "difference_ci_upper",
    "paired_interval_status",
    "bootstrap_probability_full_greater",
    "bootstrap_sign_p_value",
    "primary_gene_holm_adjusted_bootstrap_sign_p",
    "primary_gene_holm_supported_at_0_05",
    "exploratory_raw_p_supported_at_0_05",
    "multiplicity_family",
    "valid_bootstrap_replicates",
    "invalid_one_class_replicates",
    "sparse_event_or_negative_flag",
]

gene_level_model_display = gene_level_model_intervals[
    model_display_columns
].copy()

gene_level_paired_display = gene_level_paired_differences[
    paired_display_columns
].copy()

analysis_elapsed = time.time() - analysis_start_time
separator = "=" * 150

print("\n" + separator)
print("STAGE 6C STEP 3C — CELL 6C-3C2 — LOCKED GENE-LEVEL BOOTSTRAP INFERENCE")
print(separator)
print(f"Frozen cohort SHA-256              : PASS ({observed_sha256})")
print(f"Frozen cohort dimensions           : PASS ({metadata.num_rows:,} × {metadata.num_columns})")
print(f"Unique RCV keys                    : PASS ({analysis_df['rcv_accession'].nunique():,})")
print(f"Events / negatives                 : PASS ({event_count:,} / {negative_count:,})")
print(f"Bootstrap attempts per gene        : {N_BOOTSTRAP:,}")
print(f"Random seed                        : {RANDOM_SEED}")
print("Primary Holm families              : 9 AUPRC + 9 AUROC comparisons")
print("EGFR                               : exploratory; excluded from primary Holm correction")
print(f"Elapsed time                       : {analysis_elapsed:.1f}s")

print("\nGENE ACCOUNTING")
print(gene_accounting.to_string(index=False))

print("\nMODEL-SPECIFIC GENE INTERVALS")
print(gene_level_model_display.to_string(index=False))

print("\nPAIRED FULL-GES-MINUS-COMPARATOR INFERENCE")
print(gene_level_paired_display.to_string(index=False))

print("\nCELL DECISION")
print("-" * 150)
print("PASS_STAGE6C_GENE_LEVEL_PAIRED_BOOTSTRAP_INFERENCE_COMPLETE")
print(
    "Gene-specific 95% intervals, paired Full-GES-minus-comparator inference, "
    "and primary-gene Holm correction are complete."
)
print(
    "No score, outcome, gene assignment, threshold, weight, cohort membership, "
    "or frozen scientific artifact was modified."
)


Preparing BRCA1: 21,594 rows, 2,023 events, 19,571 negatives, role=primary
  Fast metric validation against scikit-learn: PASS
  Completed 250/2,000 replicates | valid 250 | elapsed 1.8s
  Completed 500/2,000 replicates | valid 500 | elapsed 4.7s
  Completed 750/2,000 replicates | valid 750 | elapsed 7.8s
  Completed 1,000/2,000 replicates | valid 1,000 | elapsed 10.4s
  Completed 1,250/2,000 replicates | valid 1,250 | elapsed 14.6s
  Completed 1,500/2,000 replicates | valid 1,500 | elapsed 18.0s
  Completed 1,750/2,000 replicates | valid 1,750 | elapsed 22.9s
  Completed 2,000/2,000 replicates | valid 2,000 | elapsed 27.3s
  BRCA1 completed: 2,000 valid, 0 one-class invalid replicates, 27.6s

Preparing BRCA2: 34,152 rows, 3,960 events, 30,192 negatives, role=primary
  Fast metric validation against scikit-learn: PASS
  Completed 250/2,000 replicates | valid 250 | elapsed 3.7s
  Completed 500/2,000 replicates | valid 500 | elapsed 9.6s
  Completed 750/2,000 replicates | valid 750 | el

In [32]:
# ==================================================================================================
# STAGE 6C STEP 3D — CELL 6C-3D0
# EXACT-LINK-ONLY SENSITIVITY PREFLIGHT AND LOCKED POINT ESTIMATES
#
# Purpose:
#   1. Freshly verify the frozen 66,636-row Stage 6B primary-evaluable cohort.
#   2. Identify exact RCV links using frozen T0/T1 linkage fields without changing linkage decisions.
#   3. Exclude only evaluable records carried by the conservatively accepted non-exact linkage pathway.
#   4. Recalculate locked discrimination and top-risk enrichment point estimates.
#   5. Compare exact-link-only results with the complete frozen primary analysis.
#
# This cell is read-only. It does not modify scores, outcomes, linkage decisions, thresholds,
# weights, cohort membership in frozen artifacts, or any scientific file.
# ==================================================================================================

from pathlib import Path
import hashlib
import re

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from sklearn.metrics import average_precision_score, roc_auc_score


# --------------------------------------------------------------------------------------------------
# 1. FROZEN INPUT AND EXPECTED PRIMARY-ANALYSIS VALUES
# --------------------------------------------------------------------------------------------------

STAGE6_DIR = Path(
    "/content/drive/MyDrive/GES_RAG_Temporal_Study/"
    "data_processed/stage6_temporal_validation"
)

EVALUABLE_PARQUET = (
    STAGE6_DIR / "stage6b_locked_primary_evaluable_cohort_v1.parquet"
)
EVALUABLE_SIDECAR = Path(str(EVALUABLE_PARQUET) + ".sha256")

EXPECTED_EVALUABLE_SHA256 = (
    "c6ad50dfc376a746e1830d000dba975101d88763fd61b8d1983f6300edb6e038"
)

EXPECTED_ROWS = 66_636
EXPECTED_COLUMNS = 79
EXPECTED_EVENTS = 6_485
EXPECTED_NEGATIVES = 60_151
EXPECTED_MAX_ACCEPTED_NONEXACT_LINKS = 170

OUTCOME_COLUMN = "primary_future_instability"
T0_RCV_COLUMN = "rcv_accession"
ROW_ORDER_COLUMN = "t0_row_order"

SCORE_SPECIFICATIONS = {
    "full_ges": {
        "column": "full_ges_instability_risk_t0",
        "display_name": "Full GES",
    },
    "no_star_ges": {
        "column": "no_star_ges_instability_risk_t0",
        "display_name": "No-star GES",
    },
    "review_stars": {
        "column": "review_stars_instability_risk",
        "display_name": "Review stars",
    },
    "combined_metadata": {
        "column": "combined_metadata_instability_risk",
        "display_name": "Combined metadata",
    },
    "conflict": {
        "column": "conflict_instability_risk",
        "display_name": "Conflict",
    },
    "recency": {
        "column": "recency_instability_risk",
        "display_name": "Recency",
    },
    "submitter": {
        "column": "submitter_instability_risk",
        "display_name": "Submitter support",
    },
    "entropy": {
        "column": "entropy_instability_risk",
        "display_name": "Classification entropy",
    },
    "additive": {
        "column": "additive_instability_risk",
        "display_name": "Additive risk",
    },
}

PRIMARY_COMPARATORS = ["no_star_ges", "review_stars", "combined_metadata"]

EXPECTED_COMPLETE_COHORT_METRICS = {
    "full_ges": {
        "auprc": 0.112444,
        "auroc": 0.535812,
    },
    "no_star_ges": {
        "auprc": 0.096437,
        "auroc": 0.447722,
    },
    "review_stars": {
        "auprc": 0.108304,
    },
    "combined_metadata": {
        "auprc": 0.113503,
    },
}

RISK_FRACTIONS = [0.05, 0.10, 0.20]


# --------------------------------------------------------------------------------------------------
# 2. HELPERS
# --------------------------------------------------------------------------------------------------

def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def read_sidecar_hash(path: Path) -> str:
    text = path.read_text(encoding="utf-8").strip()
    matches = re.findall(r"\b[a-fA-F0-9]{64}\b", text)
    if not matches:
        raise ValueError(f"No SHA-256 value found in sidecar: {path}")
    return matches[0].lower()


def normalize_rcv_accession(series: pd.Series) -> pd.Series:
    # Extract the stable RCV base accession and ignore any optional version suffix.
    return (
        series.astype("string")
        .str.upper()
        .str.strip()
        .str.extract(r"(RCV\d+)", expand=False)
    )


def normalize_category(series: pd.Series) -> pd.Series:
    return (
        series.astype("string")
        .str.upper()
        .str.strip()
        .str.replace(r"[\s\-]+", "_", regex=True)
    )


def calculate_metrics(frame: pd.DataFrame) -> dict:
    y = frame[OUTCOME_COLUMN].to_numpy(dtype=int)
    prevalence = float(y.mean())

    if len(np.unique(y)) != 2:
        raise AssertionError("A discrimination cohort does not contain both outcome classes.")

    result = {
        "rows": int(len(frame)),
        "events": int(y.sum()),
        "negatives": int((y == 0).sum()),
        "prevalence": prevalence,
        "scores": {},
    }

    for model_key, specification in SCORE_SPECIFICATIONS.items():
        scores = frame[specification["column"]].to_numpy(dtype=float)
        result["scores"][model_key] = {
            "auprc": float(average_precision_score(y, scores)),
            "auroc": float(roc_auc_score(y, scores)),
        }

    return result


def calculate_exact_rank_enrichment(
    frame: pd.DataFrame,
    cohort_label: str,
) -> pd.DataFrame:
    ordered = (
        frame[
            [
                T0_RCV_COLUMN,
                ROW_ORDER_COLUMN,
                OUTCOME_COLUMN,
                SCORE_SPECIFICATIONS["full_ges"]["column"],
            ]
        ]
        .copy()
        .sort_values(
            [
                SCORE_SPECIFICATIONS["full_ges"]["column"],
                ROW_ORDER_COLUMN,
                T0_RCV_COLUMN,
            ],
            ascending=[False, True, True],
            kind="mergesort",
        )
        .reset_index(drop=True)
    )

    y = ordered[OUTCOME_COLUMN].to_numpy(dtype=int)
    risk = ordered[SCORE_SPECIFICATIONS["full_ges"]["column"]].to_numpy(dtype=float)
    cohort_prevalence = float(y.mean())
    n_rows = len(ordered)
    rows = []

    for fraction in RISK_FRACTIONS:
        n_selected = int(np.ceil(n_rows * fraction))
        selected_y = y[:n_selected]
        remaining_y = y[n_selected:]

        selected_events = int(selected_y.sum())
        remaining_events = int(remaining_y.sum())
        selected_rate = float(selected_y.mean())
        remaining_rate = float(remaining_y.mean()) if len(remaining_y) else np.nan
        enrichment = (
            selected_rate / cohort_prevalence
            if cohort_prevalence > 0
            else np.nan
        )
        risk_ratio = (
            selected_rate / remaining_rate
            if np.isfinite(remaining_rate) and remaining_rate > 0
            else np.nan
        )

        cutoff_score = float(risk[n_selected - 1])
        boundary_tie_size = int(np.count_nonzero(risk == cutoff_score))
        boundary_tie_selected = int(
            np.count_nonzero(risk[:n_selected] == cutoff_score)
        )

        rows.append(
            {
                "cohort": cohort_label,
                "risk_fraction": fraction,
                "selected_rows": n_selected,
                "selected_events": selected_events,
                "selected_event_rate": selected_rate,
                "remaining_rows": int(n_rows - n_selected),
                "remaining_events": remaining_events,
                "remaining_event_rate": remaining_rate,
                "cohort_prevalence": cohort_prevalence,
                "risk_ratio_vs_remaining": risk_ratio,
                "enrichment_over_prevalence": enrichment,
                "cutoff_instability_risk": cutoff_score,
                "boundary_tie_size": boundary_tie_size,
                "boundary_tie_selected": boundary_tie_selected,
            }
        )

    return pd.DataFrame(rows)


def mask_signature(mask: np.ndarray) -> bytes:
    return np.packbits(np.asarray(mask, dtype=np.uint8)).tobytes()


# --------------------------------------------------------------------------------------------------
# 3. FRESH CRYPTOGRAPHIC AND STRUCTURAL VERIFICATION
# --------------------------------------------------------------------------------------------------

if not EVALUABLE_PARQUET.exists():
    raise FileNotFoundError(f"Missing frozen evaluable cohort:\n{EVALUABLE_PARQUET}")

if not EVALUABLE_SIDECAR.exists():
    raise FileNotFoundError(f"Missing SHA-256 sidecar:\n{EVALUABLE_SIDECAR}")

observed_sha256 = sha256_file(EVALUABLE_PARQUET)
sidecar_sha256 = read_sidecar_hash(EVALUABLE_SIDECAR)

if observed_sha256 != EXPECTED_EVALUABLE_SHA256:
    raise AssertionError(
        "Frozen Stage 6B evaluable-cohort SHA-256 mismatch.\n"
        f"Expected: {EXPECTED_EVALUABLE_SHA256}\n"
        f"Observed: {observed_sha256}"
    )

if sidecar_sha256 != observed_sha256:
    raise AssertionError(
        "Frozen evaluable-cohort sidecar mismatch.\n"
        f"Calculated: {observed_sha256}\n"
        f"Sidecar:    {sidecar_sha256}"
    )

parquet_file = pq.ParquetFile(EVALUABLE_PARQUET)
metadata = parquet_file.metadata
schema_columns = parquet_file.schema_arrow.names

if metadata.num_rows != EXPECTED_ROWS:
    raise AssertionError(
        f"Unexpected row count: {metadata.num_rows:,}; expected {EXPECTED_ROWS:,}"
    )

if metadata.num_columns != EXPECTED_COLUMNS:
    raise AssertionError(
        f"Unexpected column count: {metadata.num_columns}; expected {EXPECTED_COLUMNS}"
    )

required_columns = [
    T0_RCV_COLUMN,
    ROW_ORDER_COLUMN,
    OUTCOME_COLUMN,
] + [specification["column"] for specification in SCORE_SPECIFICATIONS.values()]

missing_required_columns = [
    column for column in required_columns if column not in schema_columns
]
if missing_required_columns:
    raise KeyError(
        "Required frozen columns are missing:\n"
        + "\n".join(missing_required_columns)
    )

# Load the complete 79-column frozen dataframe because linkage-field names are
# intentionally discovered and verified rather than guessed.
cohort = pd.read_parquet(EVALUABLE_PARQUET).copy()

if len(cohort) != EXPECTED_ROWS:
    raise AssertionError("Loaded dataframe row count does not match Parquet metadata.")

if cohort[T0_RCV_COLUMN].isna().any():
    raise AssertionError("Missing T0 RCV accession detected.")

if cohort[T0_RCV_COLUMN].astype(str).str.strip().eq("").any():
    raise AssertionError("Blank T0 RCV accession detected.")

if cohort[T0_RCV_COLUMN].nunique(dropna=False) != EXPECTED_ROWS:
    raise AssertionError("T0 RCV accessions are not unique.")

row_order = pd.to_numeric(cohort[ROW_ORDER_COLUMN], errors="raise").to_numpy()
if len(np.unique(row_order)) != EXPECTED_ROWS:
    raise AssertionError("t0_row_order is not unique.")
if not np.all(np.diff(row_order) > 0):
    raise AssertionError("Frozen cohort is not in increasing T0 row order.")

cohort[OUTCOME_COLUMN] = pd.to_numeric(
    cohort[OUTCOME_COLUMN],
    errors="raise",
).astype(int)

if not set(cohort[OUTCOME_COLUMN].unique()).issubset({0, 1}):
    raise AssertionError("Primary outcome is not binary.")

event_count = int(cohort[OUTCOME_COLUMN].sum())
negative_count = int((cohort[OUTCOME_COLUMN] == 0).sum())

if event_count != EXPECTED_EVENTS or negative_count != EXPECTED_NEGATIVES:
    raise AssertionError(
        f"Outcome accounting mismatch: events={event_count:,}, "
        f"negatives={negative_count:,}"
    )

for specification in SCORE_SPECIFICATIONS.values():
    column = specification["column"]
    cohort[column] = pd.to_numeric(cohort[column], errors="raise")
    values = cohort[column].to_numpy(dtype=float)

    if not np.isfinite(values).all():
        raise AssertionError(f"Missing/nonfinite score in {column}.")
    if ((values < 0.0) | (values > 1.0)).any():
        raise AssertionError(f"Score outside [0,1] in {column}.")
    if len(np.unique(values)) < 2:
        raise AssertionError(f"Insufficient score variation in {column}.")


# --------------------------------------------------------------------------------------------------
# 4. IDENTIFY AND VERIFY THE EXACT-LINK MASK
# --------------------------------------------------------------------------------------------------

t0_rcv_base = normalize_rcv_accession(cohort[T0_RCV_COLUMN])
if t0_rcv_base.isna().any():
    raise AssertionError("T0 RCV normalization failed.")

candidate_records = []
candidate_masks = {}

# 4A. Preferred evidence: equality between the frozen T0 RCV and a linked T1 RCV field.
t1_rcv_candidate_columns = [
    column
    for column in cohort.columns
    if column != T0_RCV_COLUMN
    and "rcv" in column.lower()
    and (
        "t1" in column.lower()
        or "linked" in column.lower()
        or "matched" in column.lower()
    )
]

for column in t1_rcv_candidate_columns:
    t1_rcv_base = normalize_rcv_accession(cohort[column])
    nonmissing_count = int(t1_rcv_base.notna().sum())
    exact_mask = (t0_rcv_base == t1_rcv_base).fillna(False).to_numpy(dtype=bool)
    exact_count = int(exact_mask.sum())
    excluded_count = EXPECTED_ROWS - exact_count

    candidate_records.append(
        {
            "source_type": "t0_t1_rcv_equality",
            "column": column,
            "nonmissing_values": nonmissing_count,
            "exact_rows": exact_count,
            "excluded_rows": excluded_count,
            "plausible": bool(
                nonmissing_count == EXPECTED_ROWS
                and 0 <= excluded_count <= EXPECTED_MAX_ACCEPTED_NONEXACT_LINKS
            ),
        }
    )

    if (
        nonmissing_count == EXPECTED_ROWS
        and 0 <= excluded_count <= EXPECTED_MAX_ACCEPTED_NONEXACT_LINKS
    ):
        candidate_masks[
            ("t0_t1_rcv_equality", column)
        ] = exact_mask

# 4B. Independent cross-check: categorical or Boolean linkage fields.
linkage_candidate_columns = [
    column
    for column in cohort.columns
    if any(
        token in column.lower()
        for token in ["link", "match", "method", "category", "exact"]
    )
]

for column in linkage_candidate_columns:
    series = cohort[column]

    # Boolean/numeric exact-link indicator.
    if "exact" in column.lower():
        numeric = pd.to_numeric(series, errors="coerce")
        unique_numeric = set(
            numeric.dropna().astype(float).unique().tolist()
        )
        if unique_numeric and unique_numeric.issubset({0.0, 1.0}):
            exact_mask = numeric.fillna(0).eq(1).to_numpy(dtype=bool)
            exact_count = int(exact_mask.sum())
            excluded_count = EXPECTED_ROWS - exact_count

            candidate_records.append(
                {
                    "source_type": "boolean_exact_indicator",
                    "column": column,
                    "nonmissing_values": int(numeric.notna().sum()),
                    "exact_rows": exact_count,
                    "excluded_rows": excluded_count,
                    "plausible": bool(
                        0 <= excluded_count
                        <= EXPECTED_MAX_ACCEPTED_NONEXACT_LINKS
                    ),
                }
            )

            if 0 <= excluded_count <= EXPECTED_MAX_ACCEPTED_NONEXACT_LINKS:
                candidate_masks[
                    ("boolean_exact_indicator", column)
                ] = exact_mask

    # Categorical linkage method/category.
    normalized = normalize_category(series)
    exact_text_mask = (
        normalized.str.contains("EXACT", regex=False, na=False)
        & ~normalized.str.contains("NONEXACT", regex=False, na=False)
        & ~normalized.str.contains("NON_EXACT", regex=False, na=False)
        & ~normalized.str.contains("NOT_EXACT", regex=False, na=False)
    ).to_numpy(dtype=bool)

    exact_count = int(exact_text_mask.sum())
    excluded_count = EXPECTED_ROWS - exact_count

    if exact_count > 0:
        candidate_records.append(
            {
                "source_type": "categorical_exact_marker",
                "column": column,
                "nonmissing_values": int(normalized.notna().sum()),
                "exact_rows": exact_count,
                "excluded_rows": excluded_count,
                "plausible": bool(
                    0 <= excluded_count
                    <= EXPECTED_MAX_ACCEPTED_NONEXACT_LINKS
                ),
            }
        )

        if 0 <= excluded_count <= EXPECTED_MAX_ACCEPTED_NONEXACT_LINKS:
            candidate_masks[
                ("categorical_exact_marker", column)
            ] = exact_text_mask

candidate_diagnostics = pd.DataFrame(candidate_records)

if not candidate_masks:
    print("\nPotential linkage-related columns and value counts:")
    for column in linkage_candidate_columns:
        print(f"\nCOLUMN: {column}")
        print(cohort[column].astype("string").value_counts(dropna=False).head(20))
    raise AssertionError(
        "No defensible exact-link mask could be identified automatically. "
        "Preserve this diagnostic output and stop before analysis."
    )

# Deduplicate identical masks and retain the sources supporting each one.
mask_groups = {}
for source, mask in candidate_masks.items():
    signature = mask_signature(mask)
    mask_groups.setdefault(signature, {"mask": mask, "sources": []})
    mask_groups[signature]["sources"].append(source)

# Prefer T0/T1 RCV accession equality. All plausible accession-equality sources must agree.
accession_groups = [
    group
    for group in mask_groups.values()
    if any(source_type == "t0_t1_rcv_equality"
           for source_type, _ in group["sources"])
]

if accession_groups:
    accession_signatures = {
        mask_signature(group["mask"]) for group in accession_groups
    }
    if len(accession_signatures) != 1:
        raise AssertionError(
            "Plausible linked-T1 RCV columns produced conflicting exact-link masks."
        )
    selected_group = accession_groups[0]
    exact_mask = selected_group["mask"]
    exact_mask_basis = "frozen T0/T1 RCV accession equality"
else:
    if len(mask_groups) != 1:
        print(candidate_diagnostics.to_string(index=False))
        raise AssertionError(
            "Multiple nonidentical plausible linkage masks were found and no "
            "T0/T1 RCV equality field was available to resolve them."
        )
    selected_group = next(iter(mask_groups.values()))
    exact_mask = selected_group["mask"]
    exact_mask_basis = "frozen categorical/Boolean linkage field"

# Require every other plausible candidate to agree with the chosen mask, unless it represents
# a different non-exact concept. Differences are printed and must not be silently ignored.
mask_crosscheck_rows = []
for group in mask_groups.values():
    comparison_mask = group["mask"]
    mismatch_count = int(np.count_nonzero(comparison_mask != exact_mask))
    mask_crosscheck_rows.append(
        {
            "sources": "; ".join(
                f"{source_type}:{column}"
                for source_type, column in group["sources"]
            ),
            "exact_rows": int(comparison_mask.sum()),
            "mismatch_vs_selected_mask": mismatch_count,
        }
    )

mask_crosschecks = pd.DataFrame(mask_crosscheck_rows)

exact_count = int(exact_mask.sum())
accepted_nonexact_evaluable_count = EXPECTED_ROWS - exact_count

if accepted_nonexact_evaluable_count < 0:
    raise AssertionError("Exact-link count exceeded the frozen cohort size.")

if accepted_nonexact_evaluable_count > EXPECTED_MAX_ACCEPTED_NONEXACT_LINKS:
    raise AssertionError(
        f"Exact-link restriction excluded {accepted_nonexact_evaluable_count:,} rows, "
        f"which exceeds the 170 accepted non-exact links in the complete linkage."
    )

exact_cohort = cohort.loc[exact_mask].copy()
nonexact_evaluable = cohort.loc[~exact_mask].copy()

if exact_cohort[T0_RCV_COLUMN].nunique() != len(exact_cohort):
    raise AssertionError("Exact-link-only cohort has duplicate RCV keys.")

if not np.all(np.diff(exact_cohort[ROW_ORDER_COLUMN].to_numpy()) > 0):
    raise AssertionError("Exact-link-only cohort did not preserve frozen row order.")

if len(exact_cohort) + len(nonexact_evaluable) != EXPECTED_ROWS:
    raise AssertionError("Exact/non-exact partition does not reconstruct the full cohort.")


# --------------------------------------------------------------------------------------------------
# 5. RECALCULATE COMPLETE AND EXACT-LINK-ONLY DISCRIMINATION
# --------------------------------------------------------------------------------------------------

complete_metrics = calculate_metrics(cohort)
exact_metrics = calculate_metrics(exact_cohort)

# Reconfirm the previously reported complete-cohort principal metrics before sensitivity analysis.
for model_key, expected_values in EXPECTED_COMPLETE_COHORT_METRICS.items():
    for metric_name, expected_value in expected_values.items():
        observed_value = complete_metrics["scores"][model_key][metric_name]
        if not np.isclose(observed_value, expected_value, atol=5e-7, rtol=0.0):
            raise AssertionError(
                f"Complete-cohort {model_key} {metric_name} mismatch: "
                f"observed={observed_value:.9f}, expected≈{expected_value:.6f}"
            )

discrimination_rows = []

for model_key, specification in SCORE_SPECIFICATIONS.items():
    complete_auprc = complete_metrics["scores"][model_key]["auprc"]
    exact_auprc = exact_metrics["scores"][model_key]["auprc"]
    complete_auroc = complete_metrics["scores"][model_key]["auroc"]
    exact_auroc = exact_metrics["scores"][model_key]["auroc"]

    discrimination_rows.append(
        {
            "model_key": model_key,
            "model": specification["display_name"],
            "complete_rows": complete_metrics["rows"],
            "exact_link_rows": exact_metrics["rows"],
            "complete_prevalence": complete_metrics["prevalence"],
            "exact_link_prevalence": exact_metrics["prevalence"],
            "complete_auprc": complete_auprc,
            "exact_link_auprc": exact_auprc,
            "exact_minus_complete_auprc": exact_auprc - complete_auprc,
            "complete_auroc": complete_auroc,
            "exact_link_auroc": exact_auroc,
            "exact_minus_complete_auroc": exact_auroc - complete_auroc,
        }
    )

discrimination_sensitivity = pd.DataFrame(discrimination_rows)


# --------------------------------------------------------------------------------------------------
# 6. CHECK WHETHER CENTRAL POINT-ESTIMATE DIRECTIONS ARE PRESERVED
# --------------------------------------------------------------------------------------------------

direction_rows = []

for comparator_key in PRIMARY_COMPARATORS:
    comparator_name = SCORE_SPECIFICATIONS[comparator_key]["display_name"]

    for metric_name in ["auprc", "auroc"]:
        complete_difference = (
            complete_metrics["scores"]["full_ges"][metric_name]
            - complete_metrics["scores"][comparator_key][metric_name]
        )
        exact_difference = (
            exact_metrics["scores"]["full_ges"][metric_name]
            - exact_metrics["scores"][comparator_key][metric_name]
        )

        direction_rows.append(
            {
                "metric": metric_name.upper(),
                "comparison": f"Full GES minus {comparator_name}",
                "complete_difference": complete_difference,
                "exact_link_difference": exact_difference,
                "exact_minus_complete_difference": (
                    exact_difference - complete_difference
                ),
                "point_estimate_direction_preserved": bool(
                    np.sign(complete_difference) == np.sign(exact_difference)
                    or complete_difference == 0.0
                    or exact_difference == 0.0
                ),
            }
        )

principal_direction_sensitivity = pd.DataFrame(direction_rows)

null_reference_rows = [
    {
        "metric": "AUPRC",
        "comparison": "Full GES minus cohort prevalence",
        "complete_difference": (
            complete_metrics["scores"]["full_ges"]["auprc"]
            - complete_metrics["prevalence"]
        ),
        "exact_link_difference": (
            exact_metrics["scores"]["full_ges"]["auprc"]
            - exact_metrics["prevalence"]
        ),
    },
    {
        "metric": "AUROC",
        "comparison": "Full GES minus 0.50",
        "complete_difference": (
            complete_metrics["scores"]["full_ges"]["auroc"] - 0.50
        ),
        "exact_link_difference": (
            exact_metrics["scores"]["full_ges"]["auroc"] - 0.50
        ),
    },
]

null_reference_sensitivity = pd.DataFrame(null_reference_rows)
null_reference_sensitivity["exact_minus_complete_difference"] = (
    null_reference_sensitivity["exact_link_difference"]
    - null_reference_sensitivity["complete_difference"]
)
null_reference_sensitivity["point_estimate_direction_preserved"] = (
    np.sign(null_reference_sensitivity["complete_difference"])
    == np.sign(null_reference_sensitivity["exact_link_difference"])
)


# --------------------------------------------------------------------------------------------------
# 7. RECALCULATE FULL-GES EXACT-RANK ENRICHMENT
# --------------------------------------------------------------------------------------------------

complete_enrichment = calculate_exact_rank_enrichment(
    cohort,
    "complete_primary_evaluable",
)

exact_enrichment = calculate_exact_rank_enrichment(
    exact_cohort,
    "exact_link_only",
)

enrichment_sensitivity = (
    complete_enrichment.merge(
        exact_enrichment,
        on="risk_fraction",
        how="outer",
        suffixes=("_complete", "_exact"),
        validate="one_to_one",
    )
    .sort_values("risk_fraction")
    .reset_index(drop=True)
)

for measure in [
    "selected_event_rate",
    "remaining_event_rate",
    "risk_ratio_vs_remaining",
    "enrichment_over_prevalence",
]:
    enrichment_sensitivity[f"exact_minus_complete_{measure}"] = (
        enrichment_sensitivity[f"{measure}_exact"]
        - enrichment_sensitivity[f"{measure}_complete"]
    )


# --------------------------------------------------------------------------------------------------
# 8. LINKAGE PARTITION ACCOUNTING
# --------------------------------------------------------------------------------------------------

partition_accounting = pd.DataFrame(
    [
        {
            "linkage_subset": "complete_primary_evaluable",
            "rows": int(len(cohort)),
            "events": int(cohort[OUTCOME_COLUMN].sum()),
            "negatives": int((cohort[OUTCOME_COLUMN] == 0).sum()),
            "event_prevalence": float(cohort[OUTCOME_COLUMN].mean()),
        },
        {
            "linkage_subset": "exact_link_only",
            "rows": int(len(exact_cohort)),
            "events": int(exact_cohort[OUTCOME_COLUMN].sum()),
            "negatives": int((exact_cohort[OUTCOME_COLUMN] == 0).sum()),
            "event_prevalence": float(exact_cohort[OUTCOME_COLUMN].mean()),
        },
        {
            "linkage_subset": "accepted_nonexact_evaluable_excluded",
            "rows": int(len(nonexact_evaluable)),
            "events": int(nonexact_evaluable[OUTCOME_COLUMN].sum()),
            "negatives": int((nonexact_evaluable[OUTCOME_COLUMN] == 0).sum()),
            "event_prevalence": (
                float(nonexact_evaluable[OUTCOME_COLUMN].mean())
                if len(nonexact_evaluable)
                else np.nan
            ),
        },
    ]
)

gene_column_candidates = [
    column
    for column in cohort.columns
    if column.lower() in {"target_gene", "gene", "gene_symbol"}
]

if gene_column_candidates:
    gene_column = gene_column_candidates[0]
    linkage_gene_accounting = (
        cohort.assign(
            exact_link=exact_mask,
            _gene=cohort[gene_column].astype(str).str.upper().str.strip(),
        )
        .groupby(["_gene", "exact_link"], dropna=False)[OUTCOME_COLUMN]
        .agg(rows="size", events="sum")
        .reset_index()
        .rename(columns={"_gene": "gene"})
    )
    linkage_gene_accounting["negatives"] = (
        linkage_gene_accounting["rows"]
        - linkage_gene_accounting["events"]
    )
    linkage_gene_accounting["event_prevalence"] = (
        linkage_gene_accounting["events"]
        / linkage_gene_accounting["rows"]
    )
else:
    linkage_gene_accounting = pd.DataFrame()


# --------------------------------------------------------------------------------------------------
# 9. FINAL COMPLETENESS CHECKS AND DISPLAY
# --------------------------------------------------------------------------------------------------

if len(discrimination_sensitivity) != len(SCORE_SPECIFICATIONS):
    raise AssertionError("Nine-score discrimination table is incomplete.")

if len(principal_direction_sensitivity) != 6:
    raise AssertionError("Principal comparison direction table is incomplete.")

if len(enrichment_sensitivity) != 3:
    raise AssertionError("5%/10%/20% enrichment table is incomplete.")

if not np.isfinite(
    discrimination_sensitivity[
        [
            "complete_auprc",
            "exact_link_auprc",
            "complete_auroc",
            "exact_link_auroc",
        ]
    ].to_numpy(dtype=float)
).all():
    raise AssertionError("A nonfinite discrimination result was produced.")

separator = "=" * 150

print("\n" + separator)
print("STAGE 6C STEP 3D — CELL 6C-3D0 — EXACT-LINK-ONLY SENSITIVITY")
print(separator)
print(f"Frozen cohort SHA-256              : PASS ({observed_sha256})")
print(
    f"Frozen cohort dimensions           : PASS "
    f"({metadata.num_rows:,} × {metadata.num_columns})"
)
print(f"Unique RCV keys                    : PASS ({cohort[T0_RCV_COLUMN].nunique():,})")
print(f"Events / negatives                 : PASS ({event_count:,} / {negative_count:,})")
print(f"Exact-link mask basis              : {exact_mask_basis}")
print(
    "Exact-link mask supporting sources: "
    + "; ".join(
        f"{source_type}:{column}"
        for source_type, column in selected_group["sources"]
    )
)
print(f"Exact-link evaluable rows          : {exact_count:,}")
print(
    f"Accepted non-exact evaluable rows  : "
    f"{accepted_nonexact_evaluable_count:,} "
    f"(must be ≤ {EXPECTED_MAX_ACCEPTED_NONEXACT_LINKS})"
)
print("Frozen artifacts modified          : No")

print("\nPLAUSIBLE EXACT-LINK MASK DIAGNOSTICS")
if len(candidate_diagnostics):
    print(
        candidate_diagnostics.loc[
            candidate_diagnostics["plausible"]
        ].to_string(index=False)
    )
else:
    print("No candidate diagnostic rows were generated.")

print("\nMASK CROSS-CHECKS")
print(mask_crosschecks.to_string(index=False))

print("\nLINKAGE PARTITION ACCOUNTING")
print(partition_accounting.to_string(index=False))

if not linkage_gene_accounting.empty:
    print("\nLINKAGE PARTITION BY GENE")
    print(linkage_gene_accounting.to_string(index=False))

print("\nNINE-SCORE DISCRIMINATION SENSITIVITY")
print(discrimination_sensitivity.to_string(index=False))

print("\nCENTRAL FULL-GES COMPARISON DIRECTIONS")
print(principal_direction_sensitivity.to_string(index=False))

print("\nFULL-GES NULL-REFERENCE DIRECTIONS")
print(null_reference_sensitivity.to_string(index=False))

print("\nFULL-GES TOP-RISK ENRICHMENT SENSITIVITY")
print(enrichment_sensitivity.to_string(index=False))

print("\nCELL DECISION")
print("-" * 150)
print("PASS_STAGE6C_EXACT_LINK_ONLY_SENSITIVITY_POINT_ESTIMATES_COMPLETE")
print(
    "Exact-link-only accounting, nine-score discrimination point estimates, "
    "central comparison directions, and 5%/10%/20% enrichment point estimates are complete."
)
print(
    "Inference is not yet claimed. The next cell must run paired bootstrap inference "
    "within the exact-link-only cohort."
)
print(
    "No score, outcome, linkage decision, threshold, weight, cohort membership in a "
    "frozen artifact, or scientific file was modified."
)

KeyboardInterrupt: 

In [33]:
# ==================================================================================================
# STAGE 6C STEP 3D — CELL 6C-3D0
# EXACT-LINK-ONLY SENSITIVITY PREFLIGHT AND LOCKED POINT ESTIMATES
#
# Purpose:
#   1. Freshly verify the frozen 66,636-row Stage 6B primary-evaluable cohort.
#   2. Identify exact RCV links using frozen T0/T1 linkage fields without changing linkage decisions.
#   3. Exclude only evaluable records carried by the conservatively accepted non-exact linkage pathway.
#   4. Recalculate locked discrimination and top-risk enrichment point estimates.
#   5. Compare exact-link-only results with the complete frozen primary analysis.
#
# This cell is read-only. It does not modify scores, outcomes, linkage decisions, thresholds,
# weights, cohort membership in frozen artifacts, or any scientific file.
# ==================================================================================================

from pathlib import Path
import hashlib
import re

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from sklearn.metrics import average_precision_score, roc_auc_score


# --------------------------------------------------------------------------------------------------
# 1. FROZEN INPUT AND EXPECTED PRIMARY-ANALYSIS VALUES
# --------------------------------------------------------------------------------------------------

STAGE6_DIR = Path(
    "/content/drive/MyDrive/GES_RAG_Temporal_Study/"
    "data_processed/stage6_temporal_validation"
)

EVALUABLE_PARQUET = (
    STAGE6_DIR / "stage6b_locked_primary_evaluable_cohort_v1.parquet"
)
EVALUABLE_SIDECAR = Path(str(EVALUABLE_PARQUET) + ".sha256")

EXPECTED_EVALUABLE_SHA256 = (
    "c6ad50dfc376a746e1830d000dba975101d88763fd61b8d1983f6300edb6e038"
)

EXPECTED_ROWS = 66_636
EXPECTED_COLUMNS = 79
EXPECTED_EVENTS = 6_485
EXPECTED_NEGATIVES = 60_151
EXPECTED_MAX_ACCEPTED_NONEXACT_LINKS = 170

OUTCOME_COLUMN = "primary_future_instability"
T0_RCV_COLUMN = "rcv_accession"
ROW_ORDER_COLUMN = "t0_row_order"

SCORE_SPECIFICATIONS = {
    "full_ges": {
        "column": "full_ges_instability_risk_t0",
        "display_name": "Full GES",
    },
    "no_star_ges": {
        "column": "no_star_ges_instability_risk_t0",
        "display_name": "No-star GES",
    },
    "review_stars": {
        "column": "review_stars_instability_risk",
        "display_name": "Review stars",
    },
    "combined_metadata": {
        "column": "combined_metadata_instability_risk",
        "display_name": "Combined metadata",
    },
    "conflict": {
        "column": "conflict_instability_risk",
        "display_name": "Conflict",
    },
    "recency": {
        "column": "recency_instability_risk",
        "display_name": "Recency",
    },
    "submitter": {
        "column": "submitter_instability_risk",
        "display_name": "Submitter support",
    },
    "entropy": {
        "column": "entropy_instability_risk",
        "display_name": "Classification entropy",
    },
    "additive": {
        "column": "additive_instability_risk",
        "display_name": "Additive risk",
    },
}

PRIMARY_COMPARATORS = ["no_star_ges", "review_stars", "combined_metadata"]

EXPECTED_COMPLETE_COHORT_METRICS = {
    "full_ges": {
        "auprc": 0.112444,
        "auroc": 0.535812,
    },
    "no_star_ges": {
        "auprc": 0.096437,
        "auroc": 0.447722,
    },
    "review_stars": {
        "auprc": 0.108304,
    },
    "combined_metadata": {
        "auprc": 0.113503,
    },
}

RISK_FRACTIONS = [0.05, 0.10, 0.20]


# --------------------------------------------------------------------------------------------------
# 2. HELPERS
# --------------------------------------------------------------------------------------------------

def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def read_sidecar_hash(path: Path) -> str:
    text = path.read_text(encoding="utf-8").strip()
    matches = re.findall(r"\b[a-fA-F0-9]{64}\b", text)
    if not matches:
        raise ValueError(f"No SHA-256 value found in sidecar: {path}")
    return matches[0].lower()


def normalize_rcv_accession(series: pd.Series) -> pd.Series:
    # Extract the stable RCV base accession and ignore any optional version suffix.
    return (
        series.astype("string")
        .str.upper()
        .str.strip()
        .str.extract(r"(RCV\d+)", expand=False)
    )


def normalize_category(series: pd.Series) -> pd.Series:
    return (
        series.astype("string")
        .str.upper()
        .str.strip()
        .str.replace(r"[\s\-]+", "_", regex=True)
    )


def calculate_metrics(frame: pd.DataFrame) -> dict:
    y = frame[OUTCOME_COLUMN].to_numpy(dtype=int)
    prevalence = float(y.mean())

    if len(np.unique(y)) != 2:
        raise AssertionError("A discrimination cohort does not contain both outcome classes.")

    result = {
        "rows": int(len(frame)),
        "events": int(y.sum()),
        "negatives": int((y == 0).sum()),
        "prevalence": prevalence,
        "scores": {},
    }

    for model_key, specification in SCORE_SPECIFICATIONS.items():
        scores = frame[specification["column"]].to_numpy(dtype=float)
        result["scores"][model_key] = {
            "auprc": float(average_precision_score(y, scores)),
            "auroc": float(roc_auc_score(y, scores)),
        }

    return result


def calculate_exact_rank_enrichment(
    frame: pd.DataFrame,
    cohort_label: str,
) -> pd.DataFrame:
    ordered = (
        frame[
            [
                T0_RCV_COLUMN,
                ROW_ORDER_COLUMN,
                OUTCOME_COLUMN,
                SCORE_SPECIFICATIONS["full_ges"]["column"],
            ]
        ]
        .copy()
        .sort_values(
            [
                SCORE_SPECIFICATIONS["full_ges"]["column"],
                ROW_ORDER_COLUMN,
                T0_RCV_COLUMN,
            ],
            ascending=[False, True, True],
            kind="mergesort",
        )
        .reset_index(drop=True)
    )

    y = ordered[OUTCOME_COLUMN].to_numpy(dtype=int)
    risk = ordered[SCORE_SPECIFICATIONS["full_ges"]["column"]].to_numpy(dtype=float)
    cohort_prevalence = float(y.mean())
    n_rows = len(ordered)
    rows = []

    for fraction in RISK_FRACTIONS:
        n_selected = int(np.ceil(n_rows * fraction))
        selected_y = y[:n_selected]
        remaining_y = y[n_selected:]

        selected_events = int(selected_y.sum())
        remaining_events = int(remaining_y.sum())
        selected_rate = float(selected_y.mean())
        remaining_rate = float(remaining_y.mean()) if len(remaining_y) else np.nan
        enrichment = (
            selected_rate / cohort_prevalence
            if cohort_prevalence > 0
            else np.nan
        )
        risk_ratio = (
            selected_rate / remaining_rate
            if np.isfinite(remaining_rate) and remaining_rate > 0
            else np.nan
        )

        cutoff_score = float(risk[n_selected - 1])
        boundary_tie_size = int(np.count_nonzero(risk == cutoff_score))
        boundary_tie_selected = int(
            np.count_nonzero(risk[:n_selected] == cutoff_score)
        )

        rows.append(
            {
                "cohort": cohort_label,
                "risk_fraction": fraction,
                "selected_rows": n_selected,
                "selected_events": selected_events,
                "selected_event_rate": selected_rate,
                "remaining_rows": int(n_rows - n_selected),
                "remaining_events": remaining_events,
                "remaining_event_rate": remaining_rate,
                "cohort_prevalence": cohort_prevalence,
                "risk_ratio_vs_remaining": risk_ratio,
                "enrichment_over_prevalence": enrichment,
                "cutoff_instability_risk": cutoff_score,
                "boundary_tie_size": boundary_tie_size,
                "boundary_tie_selected": boundary_tie_selected,
            }
        )

    return pd.DataFrame(rows)


def mask_signature(mask: np.ndarray) -> bytes:
    return np.packbits(np.asarray(mask, dtype=np.uint8)).tobytes()


# --------------------------------------------------------------------------------------------------
# 3. FRESH CRYPTOGRAPHIC AND STRUCTURAL VERIFICATION
# --------------------------------------------------------------------------------------------------

if not EVALUABLE_PARQUET.exists():
    raise FileNotFoundError(f"Missing frozen evaluable cohort:\n{EVALUABLE_PARQUET}")

if not EVALUABLE_SIDECAR.exists():
    raise FileNotFoundError(f"Missing SHA-256 sidecar:\n{EVALUABLE_SIDECAR}")

observed_sha256 = sha256_file(EVALUABLE_PARQUET)
sidecar_sha256 = read_sidecar_hash(EVALUABLE_SIDECAR)

if observed_sha256 != EXPECTED_EVALUABLE_SHA256:
    raise AssertionError(
        "Frozen Stage 6B evaluable-cohort SHA-256 mismatch.\n"
        f"Expected: {EXPECTED_EVALUABLE_SHA256}\n"
        f"Observed: {observed_sha256}"
    )

if sidecar_sha256 != observed_sha256:
    raise AssertionError(
        "Frozen evaluable-cohort sidecar mismatch.\n"
        f"Calculated: {observed_sha256}\n"
        f"Sidecar:    {sidecar_sha256}"
    )

parquet_file = pq.ParquetFile(EVALUABLE_PARQUET)
metadata = parquet_file.metadata
schema_columns = parquet_file.schema_arrow.names

if metadata.num_rows != EXPECTED_ROWS:
    raise AssertionError(
        f"Unexpected row count: {metadata.num_rows:,}; expected {EXPECTED_ROWS:,}"
    )

if metadata.num_columns != EXPECTED_COLUMNS:
    raise AssertionError(
        f"Unexpected column count: {metadata.num_columns}; expected {EXPECTED_COLUMNS}"
    )

required_columns = [
    T0_RCV_COLUMN,
    ROW_ORDER_COLUMN,
    OUTCOME_COLUMN,
] + [specification["column"] for specification in SCORE_SPECIFICATIONS.values()]

missing_required_columns = [
    column for column in required_columns if column not in schema_columns
]
if missing_required_columns:
    raise KeyError(
        "Required frozen columns are missing:\n"
        + "\n".join(missing_required_columns)
    )

# Load the complete 79-column frozen dataframe because linkage-field names are
# intentionally discovered and verified rather than guessed.
cohort = pd.read_parquet(EVALUABLE_PARQUET).copy()

if len(cohort) != EXPECTED_ROWS:
    raise AssertionError("Loaded dataframe row count does not match Parquet metadata.")

if cohort[T0_RCV_COLUMN].isna().any():
    raise AssertionError("Missing T0 RCV accession detected.")

if cohort[T0_RCV_COLUMN].astype(str).str.strip().eq("").any():
    raise AssertionError("Blank T0 RCV accession detected.")

if cohort[T0_RCV_COLUMN].nunique(dropna=False) != EXPECTED_ROWS:
    raise AssertionError("T0 RCV accessions are not unique.")

row_order = pd.to_numeric(cohort[ROW_ORDER_COLUMN], errors="raise").to_numpy()
if len(np.unique(row_order)) != EXPECTED_ROWS:
    raise AssertionError("t0_row_order is not unique.")
if not np.all(np.diff(row_order) > 0):
    raise AssertionError("Frozen cohort is not in increasing T0 row order.")

cohort[OUTCOME_COLUMN] = pd.to_numeric(
    cohort[OUTCOME_COLUMN],
    errors="raise",
).astype(int)

if not set(cohort[OUTCOME_COLUMN].unique()).issubset({0, 1}):
    raise AssertionError("Primary outcome is not binary.")

event_count = int(cohort[OUTCOME_COLUMN].sum())
negative_count = int((cohort[OUTCOME_COLUMN] == 0).sum())

if event_count != EXPECTED_EVENTS or negative_count != EXPECTED_NEGATIVES:
    raise AssertionError(
        f"Outcome accounting mismatch: events={event_count:,}, "
        f"negatives={negative_count:,}"
    )

for specification in SCORE_SPECIFICATIONS.values():
    column = specification["column"]
    cohort[column] = pd.to_numeric(cohort[column], errors="raise")
    values = cohort[column].to_numpy(dtype=float)

    if not np.isfinite(values).all():
        raise AssertionError(f"Missing/nonfinite score in {column}.")
    if ((values < 0.0) | (values > 1.0)).any():
        raise AssertionError(f"Score outside [0,1] in {column}.")
    if len(np.unique(values)) < 2:
        raise AssertionError(f"Insufficient score variation in {column}.")


# --------------------------------------------------------------------------------------------------
# 4. IDENTIFY AND VERIFY THE EXACT-LINK MASK
# --------------------------------------------------------------------------------------------------
#
# The frozen Stage 5/6B schema already contains the authoritative linkage field:
#   linkage_decision_category
#
# Accepted exact links are labeled:
#   ACCEPTED_EXACT_RCV
#
# Accepted conservative non-exact links are labeled:
#   ACCEPTED_NONEXACT_TIER1_ONE_TO_ONE
#
# Using the frozen decision field is faster and scientifically stronger than scanning every
# RCV-like column and repeatedly applying string extraction over 66,636 rows.

LINKAGE_DECISION_COLUMN = "linkage_decision_category"
LINKAGE_STATUS_COLUMN = "linkage_status"
LINKAGE_METHOD_COLUMN = "linkage_method"
OUTCOME_T0_RCV_COLUMN = "t0_rcv_accession"
LINKED_T1_RCV_COLUMN = "linked_t1_rcv_accession"
OUTCOME_T1_RCV_COLUMN = "t1_rcv_accession"

linkage_required_columns = [
    LINKAGE_DECISION_COLUMN,
    LINKAGE_STATUS_COLUMN,
    LINKAGE_METHOD_COLUMN,
    OUTCOME_T0_RCV_COLUMN,
    LINKED_T1_RCV_COLUMN,
    OUTCOME_T1_RCV_COLUMN,
]

missing_linkage_columns = [
    column for column in linkage_required_columns if column not in cohort.columns
]
if missing_linkage_columns:
    raise KeyError(
        "Frozen cohort is missing required linkage columns:\n"
        + "\n".join(missing_linkage_columns)
    )

linkage_decision = (
    cohort[LINKAGE_DECISION_COLUMN]
    .astype("string")
    .str.strip()
)

observed_linkage_decisions = (
    linkage_decision.value_counts(dropna=False).to_dict()
)

allowed_evaluable_linkage_decisions = {
    "ACCEPTED_EXACT_RCV",
    "ACCEPTED_NONEXACT_TIER1_ONE_TO_ONE",
}

unexpected_linkage_decisions = sorted(
    set(linkage_decision.dropna().unique().tolist())
    - allowed_evaluable_linkage_decisions
)

if linkage_decision.isna().any():
    raise AssertionError(
        "Missing linkage_decision_category values were found in the evaluable cohort."
    )

if unexpected_linkage_decisions:
    raise AssertionError(
        "Unexpected linkage decision categories entered the primary-evaluable cohort: "
        + ", ".join(unexpected_linkage_decisions)
    )

exact_mask = (
    linkage_decision.eq("ACCEPTED_EXACT_RCV")
    .to_numpy(dtype=bool)
)

accepted_nonexact_mask = (
    linkage_decision.eq("ACCEPTED_NONEXACT_TIER1_ONE_TO_ONE")
    .to_numpy(dtype=bool)
)

if np.any(exact_mask & accepted_nonexact_mask):
    raise AssertionError("A row was classified as both exact and accepted non-exact.")

if not np.all(exact_mask | accepted_nonexact_mask):
    raise AssertionError(
        "The exact and accepted-non-exact masks do not cover the evaluable cohort."
    )

exact_count = int(exact_mask.sum())
accepted_nonexact_evaluable_count = int(accepted_nonexact_mask.sum())

if exact_count + accepted_nonexact_evaluable_count != EXPECTED_ROWS:
    raise AssertionError(
        "Exact/non-exact linkage accounting does not reconstruct the frozen cohort."
    )

if accepted_nonexact_evaluable_count > EXPECTED_MAX_ACCEPTED_NONEXACT_LINKS:
    raise AssertionError(
        f"The evaluable cohort contains {accepted_nonexact_evaluable_count:,} "
        "accepted non-exact rows, exceeding the 170 accepted non-exact links "
        "in the complete frozen Stage 3 linkage."
    )

# Fast accession cross-checks using only the three known scalar RCV fields.
stage6_key_rcv = (
    cohort[T0_RCV_COLUMN]
    .astype("string")
    .str.strip()
)
outcome_t0_rcv = (
    cohort[OUTCOME_T0_RCV_COLUMN]
    .astype("string")
    .str.strip()
)
linked_t1_rcv = (
    cohort[LINKED_T1_RCV_COLUMN]
    .astype("string")
    .str.strip()
)
outcome_t1_rcv = (
    cohort[OUTCOME_T1_RCV_COLUMN]
    .astype("string")
    .str.strip()
)

if stage6_key_rcv.isna().any() or outcome_t0_rcv.isna().any():
    raise AssertionError("Missing T0 RCV values were found.")

if linked_t1_rcv.isna().any() or outcome_t1_rcv.isna().any():
    raise AssertionError(
        "A primary-evaluable row is missing its accepted linked T1 RCV."
    )

if not stage6_key_rcv.eq(outcome_t0_rcv).all():
    mismatch_count = int((~stage6_key_rcv.eq(outcome_t0_rcv)).sum())
    raise AssertionError(
        f"{mismatch_count:,} Stage 6 keys disagree with frozen t0_rcv_accession."
    )

if not linked_t1_rcv.eq(outcome_t1_rcv).all():
    mismatch_count = int((~linked_t1_rcv.eq(outcome_t1_rcv)).sum())
    raise AssertionError(
        f"{mismatch_count:,} linked_t1_rcv_accession values disagree with "
        "t1_rcv_accession."
    )

exact_rcv_equality = outcome_t0_rcv.eq(outcome_t1_rcv).to_numpy(dtype=bool)

if not np.all(exact_rcv_equality[exact_mask]):
    mismatch_count = int(
        np.count_nonzero(~exact_rcv_equality[exact_mask])
    )
    raise AssertionError(
        f"{mismatch_count:,} ACCEPTED_EXACT_RCV rows have unequal T0/T1 RCV accessions."
    )

if np.any(exact_rcv_equality[accepted_nonexact_mask]):
    mismatch_count = int(
        np.count_nonzero(exact_rcv_equality[accepted_nonexact_mask])
    )
    raise AssertionError(
        f"{mismatch_count:,} accepted non-exact rows unexpectedly have equal "
        "T0/T1 RCV accessions."
    )

# Independent frozen-field summaries retained for audit display.
linkage_field_diagnostics = pd.DataFrame(
    [
        {
            "field": LINKAGE_DECISION_COLUMN,
            "nonmissing_rows": int(linkage_decision.notna().sum()),
            "unique_values": int(linkage_decision.nunique(dropna=True)),
            "exact_rows": exact_count,
            "accepted_nonexact_rows": accepted_nonexact_evaluable_count,
        },
        {
            "field": LINKAGE_STATUS_COLUMN,
            "nonmissing_rows": int(cohort[LINKAGE_STATUS_COLUMN].notna().sum()),
            "unique_values": int(cohort[LINKAGE_STATUS_COLUMN].nunique(dropna=True)),
            "exact_rows": np.nan,
            "accepted_nonexact_rows": np.nan,
        },
        {
            "field": LINKAGE_METHOD_COLUMN,
            "nonmissing_rows": int(cohort[LINKAGE_METHOD_COLUMN].notna().sum()),
            "unique_values": int(cohort[LINKAGE_METHOD_COLUMN].nunique(dropna=True)),
            "exact_rows": np.nan,
            "accepted_nonexact_rows": np.nan,
        },
    ]
)

linkage_value_counts = pd.concat(
    [
        cohort[LINKAGE_DECISION_COLUMN]
        .astype("string")
        .value_counts(dropna=False)
        .rename_axis("value")
        .reset_index(name="rows")
        .assign(field=LINKAGE_DECISION_COLUMN),

        cohort[LINKAGE_STATUS_COLUMN]
        .astype("string")
        .value_counts(dropna=False)
        .rename_axis("value")
        .reset_index(name="rows")
        .assign(field=LINKAGE_STATUS_COLUMN),

        cohort[LINKAGE_METHOD_COLUMN]
        .astype("string")
        .value_counts(dropna=False)
        .rename_axis("value")
        .reset_index(name="rows")
        .assign(field=LINKAGE_METHOD_COLUMN),
    ],
    ignore_index=True,
)[["field", "value", "rows"]]

exact_mask_basis = (
    "frozen linkage_decision_category == ACCEPTED_EXACT_RCV, "
    "cross-checked by T0/T1 RCV equality"
)

exact_cohort = cohort.loc[exact_mask].copy()
nonexact_evaluable = cohort.loc[accepted_nonexact_mask].copy()

if exact_cohort[T0_RCV_COLUMN].nunique() != len(exact_cohort):
    raise AssertionError("Exact-link-only cohort has duplicate RCV keys.")

if not np.all(np.diff(exact_cohort[ROW_ORDER_COLUMN].to_numpy()) > 0):
    raise AssertionError("Exact-link-only cohort did not preserve frozen row order.")

if len(exact_cohort) + len(nonexact_evaluable) != EXPECTED_ROWS:
    raise AssertionError("Exact/non-exact partition does not reconstruct the full cohort.")


# --------------------------------------------------------------------------------------------------
# 5. RECALCULATE COMPLETE AND EXACT-LINK-ONLY DISCRIMINATION
# --------------------------------------------------------------------------------------------------

complete_metrics = calculate_metrics(cohort)
exact_metrics = calculate_metrics(exact_cohort)

# Reconfirm the previously reported complete-cohort principal metrics before sensitivity analysis.
for model_key, expected_values in EXPECTED_COMPLETE_COHORT_METRICS.items():
    for metric_name, expected_value in expected_values.items():
        observed_value = complete_metrics["scores"][model_key][metric_name]
        if not np.isclose(observed_value, expected_value, atol=5e-7, rtol=0.0):
            raise AssertionError(
                f"Complete-cohort {model_key} {metric_name} mismatch: "
                f"observed={observed_value:.9f}, expected≈{expected_value:.6f}"
            )

discrimination_rows = []

for model_key, specification in SCORE_SPECIFICATIONS.items():
    complete_auprc = complete_metrics["scores"][model_key]["auprc"]
    exact_auprc = exact_metrics["scores"][model_key]["auprc"]
    complete_auroc = complete_metrics["scores"][model_key]["auroc"]
    exact_auroc = exact_metrics["scores"][model_key]["auroc"]

    discrimination_rows.append(
        {
            "model_key": model_key,
            "model": specification["display_name"],
            "complete_rows": complete_metrics["rows"],
            "exact_link_rows": exact_metrics["rows"],
            "complete_prevalence": complete_metrics["prevalence"],
            "exact_link_prevalence": exact_metrics["prevalence"],
            "complete_auprc": complete_auprc,
            "exact_link_auprc": exact_auprc,
            "exact_minus_complete_auprc": exact_auprc - complete_auprc,
            "complete_auroc": complete_auroc,
            "exact_link_auroc": exact_auroc,
            "exact_minus_complete_auroc": exact_auroc - complete_auroc,
        }
    )

discrimination_sensitivity = pd.DataFrame(discrimination_rows)


# --------------------------------------------------------------------------------------------------
# 6. CHECK WHETHER CENTRAL POINT-ESTIMATE DIRECTIONS ARE PRESERVED
# --------------------------------------------------------------------------------------------------

direction_rows = []

for comparator_key in PRIMARY_COMPARATORS:
    comparator_name = SCORE_SPECIFICATIONS[comparator_key]["display_name"]

    for metric_name in ["auprc", "auroc"]:
        complete_difference = (
            complete_metrics["scores"]["full_ges"][metric_name]
            - complete_metrics["scores"][comparator_key][metric_name]
        )
        exact_difference = (
            exact_metrics["scores"]["full_ges"][metric_name]
            - exact_metrics["scores"][comparator_key][metric_name]
        )

        direction_rows.append(
            {
                "metric": metric_name.upper(),
                "comparison": f"Full GES minus {comparator_name}",
                "complete_difference": complete_difference,
                "exact_link_difference": exact_difference,
                "exact_minus_complete_difference": (
                    exact_difference - complete_difference
                ),
                "point_estimate_direction_preserved": bool(
                    np.sign(complete_difference) == np.sign(exact_difference)
                    or complete_difference == 0.0
                    or exact_difference == 0.0
                ),
            }
        )

principal_direction_sensitivity = pd.DataFrame(direction_rows)

null_reference_rows = [
    {
        "metric": "AUPRC",
        "comparison": "Full GES minus cohort prevalence",
        "complete_difference": (
            complete_metrics["scores"]["full_ges"]["auprc"]
            - complete_metrics["prevalence"]
        ),
        "exact_link_difference": (
            exact_metrics["scores"]["full_ges"]["auprc"]
            - exact_metrics["prevalence"]
        ),
    },
    {
        "metric": "AUROC",
        "comparison": "Full GES minus 0.50",
        "complete_difference": (
            complete_metrics["scores"]["full_ges"]["auroc"] - 0.50
        ),
        "exact_link_difference": (
            exact_metrics["scores"]["full_ges"]["auroc"] - 0.50
        ),
    },
]

null_reference_sensitivity = pd.DataFrame(null_reference_rows)
null_reference_sensitivity["exact_minus_complete_difference"] = (
    null_reference_sensitivity["exact_link_difference"]
    - null_reference_sensitivity["complete_difference"]
)
null_reference_sensitivity["point_estimate_direction_preserved"] = (
    np.sign(null_reference_sensitivity["complete_difference"])
    == np.sign(null_reference_sensitivity["exact_link_difference"])
)


# --------------------------------------------------------------------------------------------------
# 7. RECALCULATE FULL-GES EXACT-RANK ENRICHMENT
# --------------------------------------------------------------------------------------------------

complete_enrichment = calculate_exact_rank_enrichment(
    cohort,
    "complete_primary_evaluable",
)

exact_enrichment = calculate_exact_rank_enrichment(
    exact_cohort,
    "exact_link_only",
)

enrichment_sensitivity = (
    complete_enrichment.merge(
        exact_enrichment,
        on="risk_fraction",
        how="outer",
        suffixes=("_complete", "_exact"),
        validate="one_to_one",
    )
    .sort_values("risk_fraction")
    .reset_index(drop=True)
)

for measure in [
    "selected_event_rate",
    "remaining_event_rate",
    "risk_ratio_vs_remaining",
    "enrichment_over_prevalence",
]:
    enrichment_sensitivity[f"exact_minus_complete_{measure}"] = (
        enrichment_sensitivity[f"{measure}_exact"]
        - enrichment_sensitivity[f"{measure}_complete"]
    )


# --------------------------------------------------------------------------------------------------
# 8. LINKAGE PARTITION ACCOUNTING
# --------------------------------------------------------------------------------------------------

partition_accounting = pd.DataFrame(
    [
        {
            "linkage_subset": "complete_primary_evaluable",
            "rows": int(len(cohort)),
            "events": int(cohort[OUTCOME_COLUMN].sum()),
            "negatives": int((cohort[OUTCOME_COLUMN] == 0).sum()),
            "event_prevalence": float(cohort[OUTCOME_COLUMN].mean()),
        },
        {
            "linkage_subset": "exact_link_only",
            "rows": int(len(exact_cohort)),
            "events": int(exact_cohort[OUTCOME_COLUMN].sum()),
            "negatives": int((exact_cohort[OUTCOME_COLUMN] == 0).sum()),
            "event_prevalence": float(exact_cohort[OUTCOME_COLUMN].mean()),
        },
        {
            "linkage_subset": "accepted_nonexact_evaluable_excluded",
            "rows": int(len(nonexact_evaluable)),
            "events": int(nonexact_evaluable[OUTCOME_COLUMN].sum()),
            "negatives": int((nonexact_evaluable[OUTCOME_COLUMN] == 0).sum()),
            "event_prevalence": (
                float(nonexact_evaluable[OUTCOME_COLUMN].mean())
                if len(nonexact_evaluable)
                else np.nan
            ),
        },
    ]
)

gene_column_candidates = [
    column
    for column in cohort.columns
    if column.lower() in {"target_gene", "gene", "gene_symbol"}
]

if gene_column_candidates:
    gene_column = gene_column_candidates[0]
    linkage_gene_accounting = (
        cohort.assign(
            exact_link=exact_mask,
            _gene=cohort[gene_column].astype(str).str.upper().str.strip(),
        )
        .groupby(["_gene", "exact_link"], dropna=False)[OUTCOME_COLUMN]
        .agg(rows="size", events="sum")
        .reset_index()
        .rename(columns={"_gene": "gene"})
    )
    linkage_gene_accounting["negatives"] = (
        linkage_gene_accounting["rows"]
        - linkage_gene_accounting["events"]
    )
    linkage_gene_accounting["event_prevalence"] = (
        linkage_gene_accounting["events"]
        / linkage_gene_accounting["rows"]
    )
else:
    linkage_gene_accounting = pd.DataFrame()


# --------------------------------------------------------------------------------------------------
# 9. FINAL COMPLETENESS CHECKS AND DISPLAY
# --------------------------------------------------------------------------------------------------

if len(discrimination_sensitivity) != len(SCORE_SPECIFICATIONS):
    raise AssertionError("Nine-score discrimination table is incomplete.")

if len(principal_direction_sensitivity) != 6:
    raise AssertionError("Principal comparison direction table is incomplete.")

if len(enrichment_sensitivity) != 3:
    raise AssertionError("5%/10%/20% enrichment table is incomplete.")

if not np.isfinite(
    discrimination_sensitivity[
        [
            "complete_auprc",
            "exact_link_auprc",
            "complete_auroc",
            "exact_link_auroc",
        ]
    ].to_numpy(dtype=float)
).all():
    raise AssertionError("A nonfinite discrimination result was produced.")

separator = "=" * 150

print("\n" + separator)
print("STAGE 6C STEP 3D — CELL 6C-3D0 — EXACT-LINK-ONLY SENSITIVITY")
print(separator)
print(f"Frozen cohort SHA-256              : PASS ({observed_sha256})")
print(
    f"Frozen cohort dimensions           : PASS "
    f"({metadata.num_rows:,} × {metadata.num_columns})"
)
print(f"Unique RCV keys                    : PASS ({cohort[T0_RCV_COLUMN].nunique():,})")
print(f"Events / negatives                 : PASS ({event_count:,} / {negative_count:,})")
print(f"Exact-link mask basis              : {exact_mask_basis}")
print(f"Exact-link evaluable rows          : {exact_count:,}")
print(
    f"Accepted non-exact evaluable rows  : "
    f"{accepted_nonexact_evaluable_count:,} "
    f"(must be ≤ {EXPECTED_MAX_ACCEPTED_NONEXACT_LINKS})"
)
print("Frozen artifacts modified          : No")

print("\nLINKAGE FIELD DIAGNOSTICS")
print(linkage_field_diagnostics.to_string(index=False))

print("\nLINKAGE FIELD VALUE COUNTS")
print(linkage_value_counts.to_string(index=False))

print("\nLINKAGE PARTITION ACCOUNTING")
print(partition_accounting.to_string(index=False))

if not linkage_gene_accounting.empty:
    print("\nLINKAGE PARTITION BY GENE")
    print(linkage_gene_accounting.to_string(index=False))

print("\nNINE-SCORE DISCRIMINATION SENSITIVITY")
print(discrimination_sensitivity.to_string(index=False))

print("\nCENTRAL FULL-GES COMPARISON DIRECTIONS")
print(principal_direction_sensitivity.to_string(index=False))

print("\nFULL-GES NULL-REFERENCE DIRECTIONS")
print(null_reference_sensitivity.to_string(index=False))

print("\nFULL-GES TOP-RISK ENRICHMENT SENSITIVITY")
print(enrichment_sensitivity.to_string(index=False))

print("\nCELL DECISION")
print("-" * 150)
print("PASS_STAGE6C_EXACT_LINK_ONLY_SENSITIVITY_POINT_ESTIMATES_COMPLETE")
print(
    "Exact-link-only accounting, nine-score discrimination point estimates, "
    "central comparison directions, and 5%/10%/20% enrichment point estimates are complete."
)
print(
    "Inference is not yet claimed. The next cell must run paired bootstrap inference "
    "within the exact-link-only cohort."
)
print(
    "No score, outcome, linkage decision, threshold, weight, cohort membership in a "
    "frozen artifact, or scientific file was modified."
)

KeyError: 'Frozen cohort is missing required linkage columns:\nt0_rcv_accession'

In [34]:
# ==================================================================================================
# STAGE 6C STEP 3D — CELL 6C-3D0
# EXACT-LINK-ONLY SENSITIVITY PREFLIGHT AND LOCKED POINT ESTIMATES
#
# Purpose:
#   1. Freshly verify the frozen 66,636-row Stage 6B primary-evaluable cohort.
#   2. Identify exact RCV links using frozen T0/T1 linkage fields without changing linkage decisions.
#   3. Exclude only evaluable records carried by the conservatively accepted non-exact linkage pathway.
#   4. Recalculate locked discrimination and top-risk enrichment point estimates.
#   5. Compare exact-link-only results with the complete frozen primary analysis.
#
# This cell is read-only. It does not modify scores, outcomes, linkage decisions, thresholds,
# weights, cohort membership in frozen artifacts, or any scientific file.
# ==================================================================================================

from pathlib import Path
import hashlib
import re

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from sklearn.metrics import average_precision_score, roc_auc_score


# --------------------------------------------------------------------------------------------------
# 1. FROZEN INPUT AND EXPECTED PRIMARY-ANALYSIS VALUES
# --------------------------------------------------------------------------------------------------

STAGE6_DIR = Path(
    "/content/drive/MyDrive/GES_RAG_Temporal_Study/"
    "data_processed/stage6_temporal_validation"
)

EVALUABLE_PARQUET = (
    STAGE6_DIR / "stage6b_locked_primary_evaluable_cohort_v1.parquet"
)
EVALUABLE_SIDECAR = Path(str(EVALUABLE_PARQUET) + ".sha256")

EXPECTED_EVALUABLE_SHA256 = (
    "c6ad50dfc376a746e1830d000dba975101d88763fd61b8d1983f6300edb6e038"
)

EXPECTED_ROWS = 66_636
EXPECTED_COLUMNS = 79
EXPECTED_EVENTS = 6_485
EXPECTED_NEGATIVES = 60_151
EXPECTED_MAX_ACCEPTED_NONEXACT_LINKS = 170

OUTCOME_COLUMN = "primary_future_instability"
T0_RCV_COLUMN = "rcv_accession"
ROW_ORDER_COLUMN = "t0_row_order"

SCORE_SPECIFICATIONS = {
    "full_ges": {
        "column": "full_ges_instability_risk_t0",
        "display_name": "Full GES",
    },
    "no_star_ges": {
        "column": "no_star_ges_instability_risk_t0",
        "display_name": "No-star GES",
    },
    "review_stars": {
        "column": "review_stars_instability_risk",
        "display_name": "Review stars",
    },
    "combined_metadata": {
        "column": "combined_metadata_instability_risk",
        "display_name": "Combined metadata",
    },
    "conflict": {
        "column": "conflict_instability_risk",
        "display_name": "Conflict",
    },
    "recency": {
        "column": "recency_instability_risk",
        "display_name": "Recency",
    },
    "submitter": {
        "column": "submitter_instability_risk",
        "display_name": "Submitter support",
    },
    "entropy": {
        "column": "entropy_instability_risk",
        "display_name": "Classification entropy",
    },
    "additive": {
        "column": "additive_instability_risk",
        "display_name": "Additive risk",
    },
}

PRIMARY_COMPARATORS = ["no_star_ges", "review_stars", "combined_metadata"]

EXPECTED_COMPLETE_COHORT_METRICS = {
    "full_ges": {
        "auprc": 0.112444,
        "auroc": 0.535812,
    },
    "no_star_ges": {
        "auprc": 0.096437,
        "auroc": 0.447722,
    },
    "review_stars": {
        "auprc": 0.108304,
    },
    "combined_metadata": {
        "auprc": 0.113503,
    },
}

RISK_FRACTIONS = [0.05, 0.10, 0.20]


# --------------------------------------------------------------------------------------------------
# 2. HELPERS
# --------------------------------------------------------------------------------------------------

def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def read_sidecar_hash(path: Path) -> str:
    text = path.read_text(encoding="utf-8").strip()
    matches = re.findall(r"\b[a-fA-F0-9]{64}\b", text)
    if not matches:
        raise ValueError(f"No SHA-256 value found in sidecar: {path}")
    return matches[0].lower()


def normalize_rcv_accession(series: pd.Series) -> pd.Series:
    # Extract the stable RCV base accession and ignore any optional version suffix.
    return (
        series.astype("string")
        .str.upper()
        .str.strip()
        .str.extract(r"(RCV\d+)", expand=False)
    )


def normalize_category(series: pd.Series) -> pd.Series:
    return (
        series.astype("string")
        .str.upper()
        .str.strip()
        .str.replace(r"[\s\-]+", "_", regex=True)
    )


def calculate_metrics(frame: pd.DataFrame) -> dict:
    y = frame[OUTCOME_COLUMN].to_numpy(dtype=int)
    prevalence = float(y.mean())

    if len(np.unique(y)) != 2:
        raise AssertionError("A discrimination cohort does not contain both outcome classes.")

    result = {
        "rows": int(len(frame)),
        "events": int(y.sum()),
        "negatives": int((y == 0).sum()),
        "prevalence": prevalence,
        "scores": {},
    }

    for model_key, specification in SCORE_SPECIFICATIONS.items():
        scores = frame[specification["column"]].to_numpy(dtype=float)
        result["scores"][model_key] = {
            "auprc": float(average_precision_score(y, scores)),
            "auroc": float(roc_auc_score(y, scores)),
        }

    return result


def calculate_exact_rank_enrichment(
    frame: pd.DataFrame,
    cohort_label: str,
) -> pd.DataFrame:
    ordered = (
        frame[
            [
                T0_RCV_COLUMN,
                ROW_ORDER_COLUMN,
                OUTCOME_COLUMN,
                SCORE_SPECIFICATIONS["full_ges"]["column"],
            ]
        ]
        .copy()
        .sort_values(
            [
                SCORE_SPECIFICATIONS["full_ges"]["column"],
                ROW_ORDER_COLUMN,
                T0_RCV_COLUMN,
            ],
            ascending=[False, True, True],
            kind="mergesort",
        )
        .reset_index(drop=True)
    )

    y = ordered[OUTCOME_COLUMN].to_numpy(dtype=int)
    risk = ordered[SCORE_SPECIFICATIONS["full_ges"]["column"]].to_numpy(dtype=float)
    cohort_prevalence = float(y.mean())
    n_rows = len(ordered)
    rows = []

    for fraction in RISK_FRACTIONS:
        n_selected = int(np.ceil(n_rows * fraction))
        selected_y = y[:n_selected]
        remaining_y = y[n_selected:]

        selected_events = int(selected_y.sum())
        remaining_events = int(remaining_y.sum())
        selected_rate = float(selected_y.mean())
        remaining_rate = float(remaining_y.mean()) if len(remaining_y) else np.nan
        enrichment = (
            selected_rate / cohort_prevalence
            if cohort_prevalence > 0
            else np.nan
        )
        risk_ratio = (
            selected_rate / remaining_rate
            if np.isfinite(remaining_rate) and remaining_rate > 0
            else np.nan
        )

        cutoff_score = float(risk[n_selected - 1])
        boundary_tie_size = int(np.count_nonzero(risk == cutoff_score))
        boundary_tie_selected = int(
            np.count_nonzero(risk[:n_selected] == cutoff_score)
        )

        rows.append(
            {
                "cohort": cohort_label,
                "risk_fraction": fraction,
                "selected_rows": n_selected,
                "selected_events": selected_events,
                "selected_event_rate": selected_rate,
                "remaining_rows": int(n_rows - n_selected),
                "remaining_events": remaining_events,
                "remaining_event_rate": remaining_rate,
                "cohort_prevalence": cohort_prevalence,
                "risk_ratio_vs_remaining": risk_ratio,
                "enrichment_over_prevalence": enrichment,
                "cutoff_instability_risk": cutoff_score,
                "boundary_tie_size": boundary_tie_size,
                "boundary_tie_selected": boundary_tie_selected,
            }
        )

    return pd.DataFrame(rows)


def mask_signature(mask: np.ndarray) -> bytes:
    return np.packbits(np.asarray(mask, dtype=np.uint8)).tobytes()


# --------------------------------------------------------------------------------------------------
# 3. FRESH CRYPTOGRAPHIC AND STRUCTURAL VERIFICATION
# --------------------------------------------------------------------------------------------------

if not EVALUABLE_PARQUET.exists():
    raise FileNotFoundError(f"Missing frozen evaluable cohort:\n{EVALUABLE_PARQUET}")

if not EVALUABLE_SIDECAR.exists():
    raise FileNotFoundError(f"Missing SHA-256 sidecar:\n{EVALUABLE_SIDECAR}")

observed_sha256 = sha256_file(EVALUABLE_PARQUET)
sidecar_sha256 = read_sidecar_hash(EVALUABLE_SIDECAR)

if observed_sha256 != EXPECTED_EVALUABLE_SHA256:
    raise AssertionError(
        "Frozen Stage 6B evaluable-cohort SHA-256 mismatch.\n"
        f"Expected: {EXPECTED_EVALUABLE_SHA256}\n"
        f"Observed: {observed_sha256}"
    )

if sidecar_sha256 != observed_sha256:
    raise AssertionError(
        "Frozen evaluable-cohort sidecar mismatch.\n"
        f"Calculated: {observed_sha256}\n"
        f"Sidecar:    {sidecar_sha256}"
    )

parquet_file = pq.ParquetFile(EVALUABLE_PARQUET)
metadata = parquet_file.metadata
schema_columns = parquet_file.schema_arrow.names

if metadata.num_rows != EXPECTED_ROWS:
    raise AssertionError(
        f"Unexpected row count: {metadata.num_rows:,}; expected {EXPECTED_ROWS:,}"
    )

if metadata.num_columns != EXPECTED_COLUMNS:
    raise AssertionError(
        f"Unexpected column count: {metadata.num_columns}; expected {EXPECTED_COLUMNS}"
    )

required_columns = [
    T0_RCV_COLUMN,
    ROW_ORDER_COLUMN,
    OUTCOME_COLUMN,
] + [specification["column"] for specification in SCORE_SPECIFICATIONS.values()]

missing_required_columns = [
    column for column in required_columns if column not in schema_columns
]
if missing_required_columns:
    raise KeyError(
        "Required frozen columns are missing:\n"
        + "\n".join(missing_required_columns)
    )

# Load the complete 79-column frozen dataframe because linkage-field names are
# intentionally discovered and verified rather than guessed.
cohort = pd.read_parquet(EVALUABLE_PARQUET).copy()

if len(cohort) != EXPECTED_ROWS:
    raise AssertionError("Loaded dataframe row count does not match Parquet metadata.")

if cohort[T0_RCV_COLUMN].isna().any():
    raise AssertionError("Missing T0 RCV accession detected.")

if cohort[T0_RCV_COLUMN].astype(str).str.strip().eq("").any():
    raise AssertionError("Blank T0 RCV accession detected.")

if cohort[T0_RCV_COLUMN].nunique(dropna=False) != EXPECTED_ROWS:
    raise AssertionError("T0 RCV accessions are not unique.")

row_order = pd.to_numeric(cohort[ROW_ORDER_COLUMN], errors="raise").to_numpy()
if len(np.unique(row_order)) != EXPECTED_ROWS:
    raise AssertionError("t0_row_order is not unique.")
if not np.all(np.diff(row_order) > 0):
    raise AssertionError("Frozen cohort is not in increasing T0 row order.")

cohort[OUTCOME_COLUMN] = pd.to_numeric(
    cohort[OUTCOME_COLUMN],
    errors="raise",
).astype(int)

if not set(cohort[OUTCOME_COLUMN].unique()).issubset({0, 1}):
    raise AssertionError("Primary outcome is not binary.")

event_count = int(cohort[OUTCOME_COLUMN].sum())
negative_count = int((cohort[OUTCOME_COLUMN] == 0).sum())

if event_count != EXPECTED_EVENTS or negative_count != EXPECTED_NEGATIVES:
    raise AssertionError(
        f"Outcome accounting mismatch: events={event_count:,}, "
        f"negatives={negative_count:,}"
    )

for specification in SCORE_SPECIFICATIONS.values():
    column = specification["column"]
    cohort[column] = pd.to_numeric(cohort[column], errors="raise")
    values = cohort[column].to_numpy(dtype=float)

    if not np.isfinite(values).all():
        raise AssertionError(f"Missing/nonfinite score in {column}.")
    if ((values < 0.0) | (values > 1.0)).any():
        raise AssertionError(f"Score outside [0,1] in {column}.")
    if len(np.unique(values)) < 2:
        raise AssertionError(f"Insufficient score variation in {column}.")


# --------------------------------------------------------------------------------------------------
# 4. IDENTIFY AND VERIFY THE EXACT-LINK MASK
# --------------------------------------------------------------------------------------------------
#
# The frozen Stage 6B evaluable cohort uses `rcv_accession` as its T0 key. It does not retain a
# separate `t0_rcv_accession` column. Exact-link sensitivity is therefore defined directly from
# the frozen linkage-decision field, which is the authoritative Stage 3/Stage 5 lineage variable.
#
# Only `linkage_decision_category` is required. Other linkage fields and the linked T1 RCV are
# used as optional audit cross-checks when they are present.

LINKAGE_DECISION_COLUMN = "linkage_decision_category"

if LINKAGE_DECISION_COLUMN not in cohort.columns:
    linkage_like_columns = [
        column
        for column in cohort.columns
        if any(
            token in column.lower()
            for token in ["link", "match", "decision", "method", "category"]
        )
    ]
    raise KeyError(
        "The frozen cohort does not contain linkage_decision_category.\n"
        "Available linkage-like columns:\n"
        + "\n".join(linkage_like_columns)
    )

linkage_decision_raw = (
    cohort[LINKAGE_DECISION_COLUMN]
    .astype("string")
    .str.strip()
)

if linkage_decision_raw.isna().any():
    raise AssertionError(
        "Missing linkage_decision_category values were found in the primary-evaluable cohort."
    )

# Normalize only this single categorical column. This is fast and avoids scanning every RCV-like
# field in the dataframe.
linkage_decision_normalized = (
    linkage_decision_raw
    .str.upper()
    .str.replace(r"[\s\-]+", "_", regex=True)
)

linkage_decision_value_counts = (
    pd.DataFrame(
        {
            "raw_value": linkage_decision_raw,
            "normalized_value": linkage_decision_normalized,
        }
    )
    .value_counts(dropna=False)
    .rename("rows")
    .reset_index()
    .sort_values(["rows", "normalized_value"], ascending=[False, True])
    .reset_index(drop=True)
)

# Prefer the exact frozen labels when available. A semantic fallback is retained only to make the
# cell robust to harmless capitalization/underscore differences.
exact_label_candidates = {
    "ACCEPTED_EXACT_RCV",
    "EXACT_RCV",
    "EXACT",
}

nonexact_label_candidates = {
    "ACCEPTED_NONEXACT_TIER1_ONE_TO_ONE",
    "ACCEPTED_NON_EXACT_TIER1_ONE_TO_ONE",
    "NONEXACT_TIER1_ONE_TO_ONE",
    "NON_EXACT_TIER1_ONE_TO_ONE",
}

exact_mask = linkage_decision_normalized.isin(
    exact_label_candidates
).to_numpy(dtype=bool)

accepted_nonexact_mask = linkage_decision_normalized.isin(
    nonexact_label_candidates
).to_numpy(dtype=bool)

# Fallback for a versioned category whose wording contains the same scientific meaning.
if exact_mask.sum() == 0:
    exact_mask = (
        linkage_decision_normalized.str.contains("EXACT", regex=False, na=False)
        & ~linkage_decision_normalized.str.contains("NONEXACT", regex=False, na=False)
        & ~linkage_decision_normalized.str.contains("NON_EXACT", regex=False, na=False)
        & linkage_decision_normalized.str.contains("ACCEPT", regex=False, na=False)
    ).to_numpy(dtype=bool)

if accepted_nonexact_mask.sum() == 0:
    accepted_nonexact_mask = (
        (
            linkage_decision_normalized.str.contains(
                "NONEXACT", regex=False, na=False
            )
            | linkage_decision_normalized.str.contains(
                "NON_EXACT", regex=False, na=False
            )
        )
        & linkage_decision_normalized.str.contains(
            "ACCEPT", regex=False, na=False
        )
    ).to_numpy(dtype=bool)

if np.any(exact_mask & accepted_nonexact_mask):
    raise AssertionError(
        "At least one row was classified as both exact and accepted non-exact."
    )

covered_mask = exact_mask | accepted_nonexact_mask
uncovered_count = int(np.count_nonzero(~covered_mask))

if uncovered_count:
    uncovered_values = (
        linkage_decision_value_counts.loc[
            ~linkage_decision_value_counts["normalized_value"].isin(
                set(
                    linkage_decision_normalized[
                        pd.Series(covered_mask, index=cohort.index)
                    ].unique().tolist()
                )
            )
        ]
    )
    print("\nObserved linkage-decision values:")
    print(linkage_decision_value_counts.to_string(index=False))
    raise AssertionError(
        f"{uncovered_count:,} primary-evaluable rows were not recognized as either "
        "accepted exact or accepted non-exact links. Stop before analysis."
    )

exact_count = int(exact_mask.sum())
accepted_nonexact_evaluable_count = int(accepted_nonexact_mask.sum())

if exact_count + accepted_nonexact_evaluable_count != EXPECTED_ROWS:
    raise AssertionError(
        "Exact/non-exact linkage accounting does not reconstruct the frozen cohort."
    )

if exact_count == 0:
    raise AssertionError("The exact-link-only cohort is empty.")

if accepted_nonexact_evaluable_count > EXPECTED_MAX_ACCEPTED_NONEXACT_LINKS:
    raise AssertionError(
        f"The evaluable cohort contains {accepted_nonexact_evaluable_count:,} accepted "
        "non-exact rows, exceeding the 170 accepted non-exact links in the complete "
        "frozen Stage 3 linkage."
    )

# Optional audit cross-checks. Their absence is acceptable because the authoritative decision
# category alone is sufficient to define this prespecified sensitivity analysis.
optional_audit_rows = []

for optional_column in ["linkage_status", "linkage_method"]:
    if optional_column in cohort.columns:
        optional_audit_rows.append(
            {
                "field": optional_column,
                "present": True,
                "nonmissing_rows": int(cohort[optional_column].notna().sum()),
                "unique_values": int(cohort[optional_column].nunique(dropna=True)),
                "crosscheck_result": "DESCRIPTIVE_ONLY",
            }
        )
    else:
        optional_audit_rows.append(
            {
                "field": optional_column,
                "present": False,
                "nonmissing_rows": np.nan,
                "unique_values": np.nan,
                "crosscheck_result": "NOT_RETAINED_IN_STAGE6B",
            }
        )

# `rcv_accession` is the frozen T0 key. Cross-check with a linked T1 RCV only if such a scalar
# field is present. No guessed T0 duplicate column is required.
linked_t1_candidates = [
    column
    for column in [
        "linked_t1_rcv_accession",
        "t1_rcv_accession",
        "matched_t1_rcv_accession",
    ]
    if column in cohort.columns
]

rcv_crosscheck_result = "NO_LINKED_T1_RCV_COLUMN_RETAINED"

if linked_t1_candidates:
    linked_t1_column = linked_t1_candidates[0]

    t0_rcv_base = (
        cohort[T0_RCV_COLUMN]
        .astype("string")
        .str.strip()
        .str.upper()
        .str.extract(r"(RCV\d+)", expand=False)
    )

    t1_rcv_base = (
        cohort[linked_t1_column]
        .astype("string")
        .str.strip()
        .str.upper()
        .str.extract(r"(RCV\d+)", expand=False)
    )

    if t0_rcv_base.isna().any():
        raise AssertionError("T0 rcv_accession normalization failed.")

    if t1_rcv_base.isna().any():
        raise AssertionError(
            f"{linked_t1_column} contains missing or malformed RCV values in the "
            "primary-evaluable cohort."
        )

    rcv_equal_mask = t0_rcv_base.eq(t1_rcv_base).to_numpy(dtype=bool)

    exact_equality_failures = int(
        np.count_nonzero(~rcv_equal_mask[exact_mask])
    )
    nonexact_equality_failures = int(
        np.count_nonzero(rcv_equal_mask[accepted_nonexact_mask])
    )

    if exact_equality_failures:
        raise AssertionError(
            f"{exact_equality_failures:,} accepted exact rows have unequal T0/T1 "
            "RCV base accessions."
        )

    if nonexact_equality_failures:
        raise AssertionError(
            f"{nonexact_equality_failures:,} accepted non-exact rows unexpectedly "
            "have equal T0/T1 RCV base accessions."
        )

    rcv_crosscheck_result = (
        f"PASS using {linked_t1_column}: exact rows equal; accepted non-exact rows differ"
    )

optional_audit_rows.append(
    {
        "field": "T0/T1 RCV equality",
        "present": bool(linked_t1_candidates),
        "nonmissing_rows": (
            EXPECTED_ROWS if linked_t1_candidates else np.nan
        ),
        "unique_values": np.nan,
        "crosscheck_result": rcv_crosscheck_result,
    }
)

linkage_field_diagnostics = pd.DataFrame(
    [
        {
            "field": LINKAGE_DECISION_COLUMN,
            "present": True,
            "nonmissing_rows": int(linkage_decision_raw.notna().sum()),
            "unique_values": int(linkage_decision_raw.nunique(dropna=True)),
            "crosscheck_result": (
                f"{exact_count:,} exact; "
                f"{accepted_nonexact_evaluable_count:,} accepted non-exact"
            ),
        }
    ]
    + optional_audit_rows
)

linkage_value_counts = linkage_decision_value_counts.rename(
    columns={
        "raw_value": "value",
        "normalized_value": "normalized",
    }
)

exact_mask_basis = (
    "frozen linkage_decision_category; rcv_accession is the retained T0 key"
)

exact_cohort = cohort.loc[exact_mask].copy()
nonexact_evaluable = cohort.loc[accepted_nonexact_mask].copy()

if exact_cohort[T0_RCV_COLUMN].nunique() != len(exact_cohort):
    raise AssertionError("Exact-link-only cohort has duplicate RCV keys.")

if not np.all(np.diff(exact_cohort[ROW_ORDER_COLUMN].to_numpy()) > 0):
    raise AssertionError(
        "Exact-link-only cohort did not preserve frozen T0 row order."
    )

if len(exact_cohort) + len(nonexact_evaluable) != EXPECTED_ROWS:
    raise AssertionError(
        "Exact/non-exact partition does not reconstruct the full evaluable cohort."
    )


# --------------------------------------------------------------------------------------------------
# 5. RECALCULATE COMPLETE AND EXACT-LINK-ONLY DISCRIMINATION
# --------------------------------------------------------------------------------------------------

complete_metrics = calculate_metrics(cohort)
exact_metrics = calculate_metrics(exact_cohort)

# Reconfirm the previously reported complete-cohort principal metrics before sensitivity analysis.
for model_key, expected_values in EXPECTED_COMPLETE_COHORT_METRICS.items():
    for metric_name, expected_value in expected_values.items():
        observed_value = complete_metrics["scores"][model_key][metric_name]
        if not np.isclose(observed_value, expected_value, atol=5e-6, rtol=0.0):
            raise AssertionError(
                f"Complete-cohort {model_key} {metric_name} mismatch: "
                f"observed={observed_value:.9f}, expected≈{expected_value:.6f}"
            )

discrimination_rows = []

for model_key, specification in SCORE_SPECIFICATIONS.items():
    complete_auprc = complete_metrics["scores"][model_key]["auprc"]
    exact_auprc = exact_metrics["scores"][model_key]["auprc"]
    complete_auroc = complete_metrics["scores"][model_key]["auroc"]
    exact_auroc = exact_metrics["scores"][model_key]["auroc"]

    discrimination_rows.append(
        {
            "model_key": model_key,
            "model": specification["display_name"],
            "complete_rows": complete_metrics["rows"],
            "exact_link_rows": exact_metrics["rows"],
            "complete_prevalence": complete_metrics["prevalence"],
            "exact_link_prevalence": exact_metrics["prevalence"],
            "complete_auprc": complete_auprc,
            "exact_link_auprc": exact_auprc,
            "exact_minus_complete_auprc": exact_auprc - complete_auprc,
            "complete_auroc": complete_auroc,
            "exact_link_auroc": exact_auroc,
            "exact_minus_complete_auroc": exact_auroc - complete_auroc,
        }
    )

discrimination_sensitivity = pd.DataFrame(discrimination_rows)


# --------------------------------------------------------------------------------------------------
# 6. CHECK WHETHER CENTRAL POINT-ESTIMATE DIRECTIONS ARE PRESERVED
# --------------------------------------------------------------------------------------------------

direction_rows = []

for comparator_key in PRIMARY_COMPARATORS:
    comparator_name = SCORE_SPECIFICATIONS[comparator_key]["display_name"]

    for metric_name in ["auprc", "auroc"]:
        complete_difference = (
            complete_metrics["scores"]["full_ges"][metric_name]
            - complete_metrics["scores"][comparator_key][metric_name]
        )
        exact_difference = (
            exact_metrics["scores"]["full_ges"][metric_name]
            - exact_metrics["scores"][comparator_key][metric_name]
        )

        direction_rows.append(
            {
                "metric": metric_name.upper(),
                "comparison": f"Full GES minus {comparator_name}",
                "complete_difference": complete_difference,
                "exact_link_difference": exact_difference,
                "exact_minus_complete_difference": (
                    exact_difference - complete_difference
                ),
                "point_estimate_direction_preserved": bool(
                    np.sign(complete_difference) == np.sign(exact_difference)
                    or complete_difference == 0.0
                    or exact_difference == 0.0
                ),
            }
        )

principal_direction_sensitivity = pd.DataFrame(direction_rows)

null_reference_rows = [
    {
        "metric": "AUPRC",
        "comparison": "Full GES minus cohort prevalence",
        "complete_difference": (
            complete_metrics["scores"]["full_ges"]["auprc"]
            - complete_metrics["prevalence"]
        ),
        "exact_link_difference": (
            exact_metrics["scores"]["full_ges"]["auprc"]
            - exact_metrics["prevalence"]
        ),
    },
    {
        "metric": "AUROC",
        "comparison": "Full GES minus 0.50",
        "complete_difference": (
            complete_metrics["scores"]["full_ges"]["auroc"] - 0.50
        ),
        "exact_link_difference": (
            exact_metrics["scores"]["full_ges"]["auroc"] - 0.50
        ),
    },
]

null_reference_sensitivity = pd.DataFrame(null_reference_rows)
null_reference_sensitivity["exact_minus_complete_difference"] = (
    null_reference_sensitivity["exact_link_difference"]
    - null_reference_sensitivity["complete_difference"]
)
null_reference_sensitivity["point_estimate_direction_preserved"] = (
    np.sign(null_reference_sensitivity["complete_difference"])
    == np.sign(null_reference_sensitivity["exact_link_difference"])
)


# --------------------------------------------------------------------------------------------------
# 7. RECALCULATE FULL-GES EXACT-RANK ENRICHMENT
# --------------------------------------------------------------------------------------------------

complete_enrichment = calculate_exact_rank_enrichment(
    cohort,
    "complete_primary_evaluable",
)

exact_enrichment = calculate_exact_rank_enrichment(
    exact_cohort,
    "exact_link_only",
)

enrichment_sensitivity = (
    complete_enrichment.merge(
        exact_enrichment,
        on="risk_fraction",
        how="outer",
        suffixes=("_complete", "_exact"),
        validate="one_to_one",
    )
    .sort_values("risk_fraction")
    .reset_index(drop=True)
)

for measure in [
    "selected_event_rate",
    "remaining_event_rate",
    "risk_ratio_vs_remaining",
    "enrichment_over_prevalence",
]:
    enrichment_sensitivity[f"exact_minus_complete_{measure}"] = (
        enrichment_sensitivity[f"{measure}_exact"]
        - enrichment_sensitivity[f"{measure}_complete"]
    )


# --------------------------------------------------------------------------------------------------
# 8. LINKAGE PARTITION ACCOUNTING
# --------------------------------------------------------------------------------------------------

partition_accounting = pd.DataFrame(
    [
        {
            "linkage_subset": "complete_primary_evaluable",
            "rows": int(len(cohort)),
            "events": int(cohort[OUTCOME_COLUMN].sum()),
            "negatives": int((cohort[OUTCOME_COLUMN] == 0).sum()),
            "event_prevalence": float(cohort[OUTCOME_COLUMN].mean()),
        },
        {
            "linkage_subset": "exact_link_only",
            "rows": int(len(exact_cohort)),
            "events": int(exact_cohort[OUTCOME_COLUMN].sum()),
            "negatives": int((exact_cohort[OUTCOME_COLUMN] == 0).sum()),
            "event_prevalence": float(exact_cohort[OUTCOME_COLUMN].mean()),
        },
        {
            "linkage_subset": "accepted_nonexact_evaluable_excluded",
            "rows": int(len(nonexact_evaluable)),
            "events": int(nonexact_evaluable[OUTCOME_COLUMN].sum()),
            "negatives": int((nonexact_evaluable[OUTCOME_COLUMN] == 0).sum()),
            "event_prevalence": (
                float(nonexact_evaluable[OUTCOME_COLUMN].mean())
                if len(nonexact_evaluable)
                else np.nan
            ),
        },
    ]
)

gene_column_candidates = [
    column
    for column in cohort.columns
    if column.lower() in {"target_gene", "gene", "gene_symbol"}
]

if gene_column_candidates:
    gene_column = gene_column_candidates[0]
    linkage_gene_accounting = (
        cohort.assign(
            exact_link=exact_mask,
            _gene=cohort[gene_column].astype(str).str.upper().str.strip(),
        )
        .groupby(["_gene", "exact_link"], dropna=False)[OUTCOME_COLUMN]
        .agg(rows="size", events="sum")
        .reset_index()
        .rename(columns={"_gene": "gene"})
    )
    linkage_gene_accounting["negatives"] = (
        linkage_gene_accounting["rows"]
        - linkage_gene_accounting["events"]
    )
    linkage_gene_accounting["event_prevalence"] = (
        linkage_gene_accounting["events"]
        / linkage_gene_accounting["rows"]
    )
else:
    linkage_gene_accounting = pd.DataFrame()


# --------------------------------------------------------------------------------------------------
# 9. FINAL COMPLETENESS CHECKS AND DISPLAY
# --------------------------------------------------------------------------------------------------

if len(discrimination_sensitivity) != len(SCORE_SPECIFICATIONS):
    raise AssertionError("Nine-score discrimination table is incomplete.")

if len(principal_direction_sensitivity) != 6:
    raise AssertionError("Principal comparison direction table is incomplete.")

if len(enrichment_sensitivity) != 3:
    raise AssertionError("5%/10%/20% enrichment table is incomplete.")

if not np.isfinite(
    discrimination_sensitivity[
        [
            "complete_auprc",
            "exact_link_auprc",
            "complete_auroc",
            "exact_link_auroc",
        ]
    ].to_numpy(dtype=float)
).all():
    raise AssertionError("A nonfinite discrimination result was produced.")

separator = "=" * 150

print("\n" + separator)
print("STAGE 6C STEP 3D — CELL 6C-3D0 — EXACT-LINK-ONLY SENSITIVITY")
print(separator)
print(f"Frozen cohort SHA-256              : PASS ({observed_sha256})")
print(
    f"Frozen cohort dimensions           : PASS "
    f"({metadata.num_rows:,} × {metadata.num_columns})"
)
print(f"Unique RCV keys                    : PASS ({cohort[T0_RCV_COLUMN].nunique():,})")
print(f"Events / negatives                 : PASS ({event_count:,} / {negative_count:,})")
print(f"Exact-link mask basis              : {exact_mask_basis}")
print(f"Exact-link evaluable rows          : {exact_count:,}")
print(
    f"Accepted non-exact evaluable rows  : "
    f"{accepted_nonexact_evaluable_count:,} "
    f"(must be ≤ {EXPECTED_MAX_ACCEPTED_NONEXACT_LINKS})"
)
print("Frozen artifacts modified          : No")

print("\nLINKAGE FIELD DIAGNOSTICS")
print(linkage_field_diagnostics.to_string(index=False))

print("\nLINKAGE FIELD VALUE COUNTS")
print(linkage_value_counts.to_string(index=False))

print("\nLINKAGE PARTITION ACCOUNTING")
print(partition_accounting.to_string(index=False))

if not linkage_gene_accounting.empty:
    print("\nLINKAGE PARTITION BY GENE")
    print(linkage_gene_accounting.to_string(index=False))

print("\nNINE-SCORE DISCRIMINATION SENSITIVITY")
print(discrimination_sensitivity.to_string(index=False))

print("\nCENTRAL FULL-GES COMPARISON DIRECTIONS")
print(principal_direction_sensitivity.to_string(index=False))

print("\nFULL-GES NULL-REFERENCE DIRECTIONS")
print(null_reference_sensitivity.to_string(index=False))

print("\nFULL-GES TOP-RISK ENRICHMENT SENSITIVITY")
print(enrichment_sensitivity.to_string(index=False))

print("\nCELL DECISION")
print("-" * 150)
print("PASS_STAGE6C_EXACT_LINK_ONLY_SENSITIVITY_POINT_ESTIMATES_COMPLETE")
print(
    "Exact-link-only accounting, nine-score discrimination point estimates, "
    "central comparison directions, and 5%/10%/20% enrichment point estimates are complete."
)
print(
    "Inference is not yet claimed. The next cell must run paired bootstrap inference "
    "within the exact-link-only cohort."
)
print(
    "No score, outcome, linkage decision, threshold, weight, cohort membership in a "
    "frozen artifact, or scientific file was modified."
)


STAGE 6C STEP 3D — CELL 6C-3D0 — EXACT-LINK-ONLY SENSITIVITY
Frozen cohort SHA-256              : PASS (c6ad50dfc376a746e1830d000dba975101d88763fd61b8d1983f6300edb6e038)
Frozen cohort dimensions           : PASS (66,636 × 79)
Unique RCV keys                    : PASS (66,636)
Events / negatives                 : PASS (6,485 / 60,151)
Exact-link mask basis              : frozen linkage_decision_category; rcv_accession is the retained T0 key
Exact-link evaluable rows          : 66,469
Accepted non-exact evaluable rows  : 167 (must be ≤ 170)
Frozen artifacts modified          : No

LINKAGE FIELD DIAGNOSTICS
                    field  present  nonmissing_rows  unique_values                                                                    crosscheck_result
linkage_decision_category     True            66636            2.0                                                 66,469 exact; 167 accepted non-exact
           linkage_status     True            66636            1.0                 

In [35]:
# ==================================================================================================
# STAGE 6C STEP 3D — CELL 6C-3D1
# EXACT-LINK-ONLY PAIRED BOOTSTRAP INFERENCE
#
# Purpose:
#   1. Freshly verify the frozen Stage 6B primary-evaluable cohort.
#   2. Reconstruct the prespecified exact-link-only sensitivity subset from the frozen
#      linkage_decision_category field.
#   3. Run 2,000 paired row-bootstrap replicates with seed 42 for all nine frozen scores.
#   4. Produce exact-link-only AUPRC/AUROC confidence intervals.
#   5. Compare Full GES with the three principal comparators and five remaining metadata
#      comparators using identical resamples.
#   6. Apply Holm correction separately to the five remaining-comparator AUPRC tests and
#      five remaining-comparator AUROC tests. Principal comparisons remain prespecified
#      paired sensitivity comparisons and are not mixed into that secondary family.
#
# This cell is read-only. It does not modify scores, outcomes, linkage decisions, thresholds,
# weights, cohort membership in frozen artifacts, or any scientific file.
# ==================================================================================================

from pathlib import Path
import gc
import hashlib
import re
import time

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from scipy.sparse import csr_matrix
from sklearn.metrics import average_precision_score, roc_auc_score


# --------------------------------------------------------------------------------------------------
# 1. FROZEN INPUT AND ANALYSIS SPECIFICATION
# --------------------------------------------------------------------------------------------------

STAGE6_DIR = Path(
    "/content/drive/MyDrive/GES_RAG_Temporal_Study/"
    "data_processed/stage6_temporal_validation"
)

EVALUABLE_PARQUET = (
    STAGE6_DIR / "stage6b_locked_primary_evaluable_cohort_v1.parquet"
)
EVALUABLE_SIDECAR = Path(str(EVALUABLE_PARQUET) + ".sha256")

EXPECTED_EVALUABLE_SHA256 = (
    "c6ad50dfc376a746e1830d000dba975101d88763fd61b8d1983f6300edb6e038"
)

EXPECTED_FULL_ROWS = 66_636
EXPECTED_FULL_COLUMNS = 79
EXPECTED_FULL_EVENTS = 6_485
EXPECTED_FULL_NEGATIVES = 60_151

# Verified by Cell 6C-3D0.
EXPECTED_EXACT_ROWS = 66_469
EXPECTED_EXACT_EVENTS = 6_433
EXPECTED_EXACT_NEGATIVES = 60_036
EXPECTED_ACCEPTED_NONEXACT_EVALUABLE = 167

OUTCOME_COLUMN = "primary_future_instability"
LINKAGE_DECISION_COLUMN = "linkage_decision_category"
RCV_COLUMN = "rcv_accession"
ROW_ORDER_COLUMN = "t0_row_order"

N_BOOTSTRAP = 2_000
RANDOM_SEED = 42
BOOTSTRAP_BATCH_SIZE = 50
MINIMUM_VALID_REPLICATES = 1_000
CI_QUANTILES = (0.025, 0.975)

SCORE_SPECIFICATIONS = {
    "full_ges": {
        "column": "full_ges_instability_risk_t0",
        "display_name": "Full GES",
    },
    "no_star_ges": {
        "column": "no_star_ges_instability_risk_t0",
        "display_name": "No-star GES",
    },
    "review_stars": {
        "column": "review_stars_instability_risk",
        "display_name": "Review stars",
    },
    "combined_metadata": {
        "column": "combined_metadata_instability_risk",
        "display_name": "Combined metadata",
    },
    "conflict": {
        "column": "conflict_instability_risk",
        "display_name": "Conflict",
    },
    "recency": {
        "column": "recency_instability_risk",
        "display_name": "Recency",
    },
    "submitter": {
        "column": "submitter_instability_risk",
        "display_name": "Submitter support",
    },
    "entropy": {
        "column": "entropy_instability_risk",
        "display_name": "Classification entropy",
    },
    "additive": {
        "column": "additive_instability_risk",
        "display_name": "Additive risk",
    },
}

PRINCIPAL_COMPARATORS = [
    "no_star_ges",
    "review_stars",
    "combined_metadata",
]

SECONDARY_COMPARATORS = [
    "conflict",
    "recency",
    "submitter",
    "entropy",
    "additive",
]


# --------------------------------------------------------------------------------------------------
# 2. HELPERS
# --------------------------------------------------------------------------------------------------

def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def read_sidecar_hash(path: Path) -> str:
    text = path.read_text(encoding="utf-8").strip()
    matches = re.findall(r"\b[a-fA-F0-9]{64}\b", text)
    if not matches:
        raise ValueError(f"No SHA-256 value found in sidecar: {path}")
    return matches[0].lower()


def percentile_interval(values: np.ndarray) -> tuple[float, float]:
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if len(values) == 0:
        return np.nan, np.nan
    lower, upper = np.quantile(values, CI_QUANTILES)
    return float(lower), float(upper)


def bootstrap_sign_pvalue(differences: np.ndarray) -> float:
    differences = np.asarray(differences, dtype=float)
    differences = differences[np.isfinite(differences)]
    if len(differences) == 0:
        return np.nan

    n = len(differences)
    lower_tail = (np.count_nonzero(differences <= 0.0) + 1) / (n + 1)
    upper_tail = (np.count_nonzero(differences >= 0.0) + 1) / (n + 1)
    return float(min(1.0, 2.0 * min(lower_tail, upper_tail)))


def holm_adjust(p_values: np.ndarray) -> np.ndarray:
    p_values = np.asarray(p_values, dtype=float)
    adjusted = np.full(len(p_values), np.nan, dtype=float)
    valid_positions = np.where(np.isfinite(p_values))[0]

    if len(valid_positions) == 0:
        return adjusted

    valid_p = p_values[valid_positions]
    order = np.argsort(valid_p)
    m = len(valid_p)
    running_max = 0.0

    for rank, position_within_valid in enumerate(order):
        original_position = valid_positions[position_within_valid]
        raw_adjusted = (m - rank) * valid_p[position_within_valid]
        running_max = max(running_max, raw_adjusted)
        adjusted[original_position] = min(1.0, running_max)

    return adjusted


def interval_status(
    lower: float,
    upper: float,
    positive_label: str,
    negative_label: str,
) -> str:
    if not np.isfinite(lower) or not np.isfinite(upper):
        return "not_estimable"
    if lower > 0.0:
        return positive_label
    if upper < 0.0:
        return negative_label
    return "interval_includes_null"


def construct_score_group_cache(
    scores: np.ndarray,
    outcomes: np.ndarray,
) -> dict:
    scores = np.asarray(scores, dtype=np.float64)
    outcomes = np.asarray(outcomes, dtype=np.int8)

    unique_scores, group_index = np.unique(scores, return_inverse=True)
    n_rows = len(scores)
    n_groups = len(unique_scores)
    row_positions = np.arange(n_rows)

    total_group_matrix = csr_matrix(
        (
            np.ones(n_rows, dtype=np.float64),
            (group_index, row_positions),
        ),
        shape=(n_groups, n_rows),
    )

    positive_positions = np.flatnonzero(outcomes == 1)
    positive_group_matrix = csr_matrix(
        (
            np.ones(len(positive_positions), dtype=np.float64),
            (group_index[positive_positions], positive_positions),
        ),
        shape=(n_groups, n_rows),
    )

    return {
        "unique_scores": unique_scores,
        "total_group_matrix": total_group_matrix,
        "positive_group_matrix": positive_group_matrix,
    }


def calculate_grouped_weighted_metrics(
    cache: dict,
    count_matrix: np.ndarray,
    positive_totals: np.ndarray,
) -> tuple[np.ndarray, np.ndarray]:
    count_matrix = np.asarray(count_matrix)
    positive_totals = np.asarray(positive_totals, dtype=np.float64)

    sample_totals = count_matrix.sum(axis=1).astype(np.float64)
    negative_totals = sample_totals - positive_totals

    group_total_counts = np.asarray(
        cache["total_group_matrix"] @ count_matrix.T,
        dtype=np.float64,
    )
    group_positive_counts = np.asarray(
        cache["positive_group_matrix"] @ count_matrix.T,
        dtype=np.float64,
    )
    group_negative_counts = group_total_counts - group_positive_counts

    valid = (positive_totals > 0.0) & (negative_totals > 0.0)

    # AUROC with exact 0.5 credit for tied positive-negative pairs.
    cumulative_negatives_before = (
        np.cumsum(group_negative_counts, axis=0)
        - group_negative_counts
    )
    concordant_numerator = np.sum(
        group_positive_counts
        * (
            cumulative_negatives_before
            + 0.5 * group_negative_counts
        ),
        axis=0,
    )

    auroc = np.full(len(positive_totals), np.nan, dtype=np.float64)
    np.divide(
        concordant_numerator,
        positive_totals * negative_totals,
        out=auroc,
        where=valid,
    )

    # Average precision / AUPRC using descending tied-score groups.
    positive_desc = group_positive_counts[::-1, :]
    total_desc = group_total_counts[::-1, :]
    cumulative_positive = np.cumsum(positive_desc, axis=0)
    cumulative_total = np.cumsum(total_desc, axis=0)

    precision = np.zeros_like(cumulative_positive, dtype=np.float64)
    np.divide(
        cumulative_positive,
        cumulative_total,
        out=precision,
        where=cumulative_total > 0.0,
    )

    ap_numerator = np.sum(precision * positive_desc, axis=0)
    auprc = np.full(len(positive_totals), np.nan, dtype=np.float64)
    np.divide(
        ap_numerator,
        positive_totals,
        out=auprc,
        where=valid,
    )

    return auprc, auroc


# --------------------------------------------------------------------------------------------------
# 3. FRESH CRYPTOGRAPHIC AND STRUCTURAL VERIFICATION
# --------------------------------------------------------------------------------------------------

if not EVALUABLE_PARQUET.exists():
    raise FileNotFoundError(
        f"Missing frozen evaluable cohort:\n{EVALUABLE_PARQUET}"
    )

if not EVALUABLE_SIDECAR.exists():
    raise FileNotFoundError(
        f"Missing SHA-256 sidecar:\n{EVALUABLE_SIDECAR}"
    )

observed_sha256 = sha256_file(EVALUABLE_PARQUET)
sidecar_sha256 = read_sidecar_hash(EVALUABLE_SIDECAR)

if observed_sha256 != EXPECTED_EVALUABLE_SHA256:
    raise AssertionError(
        "Frozen Stage 6B evaluable-cohort SHA-256 mismatch.\n"
        f"Expected: {EXPECTED_EVALUABLE_SHA256}\n"
        f"Observed: {observed_sha256}"
    )

if sidecar_sha256 != observed_sha256:
    raise AssertionError(
        "Frozen evaluable-cohort sidecar mismatch.\n"
        f"Calculated: {observed_sha256}\n"
        f"Sidecar:    {sidecar_sha256}"
    )

parquet_file = pq.ParquetFile(EVALUABLE_PARQUET)
metadata = parquet_file.metadata
schema_columns = parquet_file.schema_arrow.names

if metadata.num_rows != EXPECTED_FULL_ROWS:
    raise AssertionError(
        f"Unexpected full-cohort row count: {metadata.num_rows:,}; "
        f"expected {EXPECTED_FULL_ROWS:,}"
    )

if metadata.num_columns != EXPECTED_FULL_COLUMNS:
    raise AssertionError(
        f"Unexpected column count: {metadata.num_columns}; "
        f"expected {EXPECTED_FULL_COLUMNS}"
    )

required_columns = [
    RCV_COLUMN,
    ROW_ORDER_COLUMN,
    OUTCOME_COLUMN,
    LINKAGE_DECISION_COLUMN,
] + [
    specification["column"]
    for specification in SCORE_SPECIFICATIONS.values()
]

missing_columns = [
    column for column in required_columns
    if column not in schema_columns
]
if missing_columns:
    raise KeyError(
        "Missing required frozen columns:\n"
        + "\n".join(missing_columns)
    )

cohort = pd.read_parquet(
    EVALUABLE_PARQUET,
    columns=required_columns,
).copy()

if len(cohort) != EXPECTED_FULL_ROWS:
    raise AssertionError(
        "Loaded dataframe row count does not match the frozen cohort."
    )

if cohort[RCV_COLUMN].isna().any():
    raise AssertionError("Missing RCV accession detected.")

if cohort[RCV_COLUMN].astype(str).str.strip().eq("").any():
    raise AssertionError("Blank RCV accession detected.")

if cohort[RCV_COLUMN].nunique(dropna=False) != EXPECTED_FULL_ROWS:
    raise AssertionError("RCV accessions are not unique.")

row_order = pd.to_numeric(
    cohort[ROW_ORDER_COLUMN],
    errors="raise",
).to_numpy()

if len(np.unique(row_order)) != EXPECTED_FULL_ROWS:
    raise AssertionError("t0_row_order is not unique.")

if not np.all(np.diff(row_order) > 0):
    raise AssertionError(
        "Frozen evaluable cohort is not in increasing T0 row order."
    )

cohort[OUTCOME_COLUMN] = pd.to_numeric(
    cohort[OUTCOME_COLUMN],
    errors="raise",
).astype(int)

if not set(cohort[OUTCOME_COLUMN].unique()).issubset({0, 1}):
    raise AssertionError("Primary outcome is not binary.")

full_events = int(cohort[OUTCOME_COLUMN].sum())
full_negatives = int((cohort[OUTCOME_COLUMN] == 0).sum())

if (
    full_events != EXPECTED_FULL_EVENTS
    or full_negatives != EXPECTED_FULL_NEGATIVES
):
    raise AssertionError(
        f"Full outcome accounting mismatch: events={full_events:,}, "
        f"negatives={full_negatives:,}"
    )

for specification in SCORE_SPECIFICATIONS.values():
    column = specification["column"]
    cohort[column] = pd.to_numeric(
        cohort[column],
        errors="raise",
    )
    values = cohort[column].to_numpy(dtype=float)

    if not np.isfinite(values).all():
        raise AssertionError(
            f"Missing/nonfinite score in {column}."
        )

    if ((values < 0.0) | (values > 1.0)).any():
        raise AssertionError(
            f"Score outside [0,1] in {column}."
        )


# --------------------------------------------------------------------------------------------------
# 4. RECONSTRUCT AND VERIFY THE EXACT-LINK-ONLY COHORT
# --------------------------------------------------------------------------------------------------

linkage_decision = (
    cohort[LINKAGE_DECISION_COLUMN]
    .astype("string")
    .str.strip()
    .str.upper()
    .str.replace(r"[\s\-]+", "_", regex=True)
)

exact_mask = linkage_decision.eq(
    "ACCEPTED_EXACT_RCV"
).to_numpy(dtype=bool)

accepted_nonexact_mask = linkage_decision.eq(
    "ACCEPTED_NONEXACT_TIER1_ONE_TO_ONE"
).to_numpy(dtype=bool)

if np.any(exact_mask & accepted_nonexact_mask):
    raise AssertionError(
        "A row was classified as both exact and accepted non-exact."
    )

if not np.all(exact_mask | accepted_nonexact_mask):
    observed_values = (
        linkage_decision.value_counts(dropna=False)
        .rename_axis("linkage_decision_category")
        .reset_index(name="rows")
    )
    print(observed_values.to_string(index=False))
    raise AssertionError(
        "The exact and accepted-non-exact masks do not cover "
        "the complete primary-evaluable cohort."
    )

exact_cohort = (
    cohort.loc[exact_mask]
    .copy()
    .reset_index(drop=True)
)

accepted_nonexact_evaluable = (
    cohort.loc[accepted_nonexact_mask]
    .copy()
)

if len(exact_cohort) != EXPECTED_EXACT_ROWS:
    raise AssertionError(
        f"Exact-link row count={len(exact_cohort):,}; "
        f"expected {EXPECTED_EXACT_ROWS:,}"
    )

if len(accepted_nonexact_evaluable) != EXPECTED_ACCEPTED_NONEXACT_EVALUABLE:
    raise AssertionError(
        f"Accepted non-exact evaluable count="
        f"{len(accepted_nonexact_evaluable):,}; "
        f"expected {EXPECTED_ACCEPTED_NONEXACT_EVALUABLE:,}"
    )

exact_events = int(exact_cohort[OUTCOME_COLUMN].sum())
exact_negatives = int(
    (exact_cohort[OUTCOME_COLUMN] == 0).sum()
)

if (
    exact_events != EXPECTED_EXACT_EVENTS
    or exact_negatives != EXPECTED_EXACT_NEGATIVES
):
    raise AssertionError(
        f"Exact-link accounting mismatch: events={exact_events:,}, "
        f"negatives={exact_negatives:,}"
    )

if exact_cohort[RCV_COLUMN].nunique() != EXPECTED_EXACT_ROWS:
    raise AssertionError(
        "Exact-link-only RCV keys are not unique."
    )

if not np.all(
    np.diff(
        exact_cohort[ROW_ORDER_COLUMN].to_numpy()
    ) > 0
):
    raise AssertionError(
        "Exact-link-only cohort did not preserve frozen row order."
    )


# --------------------------------------------------------------------------------------------------
# 5. VALIDATE OPTIMIZED POINT METRICS AND PREPARE CACHES
# --------------------------------------------------------------------------------------------------

outcomes = exact_cohort[OUTCOME_COLUMN].to_numpy(
    dtype=np.int8
)

n_rows = int(len(exact_cohort))
n_events = int(outcomes.sum())
n_negatives = n_rows - n_events
prevalence = n_events / n_rows

if len(np.unique(outcomes)) != 2:
    raise AssertionError(
        "Exact-link-only cohort does not contain both outcome classes."
    )

score_arrays = {
    model_key: exact_cohort[
        specification["column"]
    ].to_numpy(dtype=np.float64)
    for model_key, specification
    in SCORE_SPECIFICATIONS.items()
}

caches = {
    model_key: construct_score_group_cache(
        scores,
        outcomes,
    )
    for model_key, scores in score_arrays.items()
}

point_metrics = {}
point_estimate_rows = []

original_counts = np.ones(
    (1, n_rows),
    dtype=np.int16,
)
original_positive_total = np.array(
    [n_events],
    dtype=np.float64,
)

for model_key, specification in SCORE_SPECIFICATIONS.items():
    scores = score_arrays[model_key]

    sklearn_auprc = float(
        average_precision_score(
            outcomes,
            scores,
        )
    )
    sklearn_auroc = float(
        roc_auc_score(
            outcomes,
            scores,
        )
    )

    fast_auprc, fast_auroc = (
        calculate_grouped_weighted_metrics(
            caches[model_key],
            original_counts,
            original_positive_total,
        )
    )

    if not np.isclose(
        fast_auprc[0],
        sklearn_auprc,
        rtol=1e-11,
        atol=1e-12,
    ):
        raise AssertionError(
            f"Fast AUPRC validation failed for {model_key}: "
            f"{fast_auprc[0]:.15f} vs {sklearn_auprc:.15f}"
        )

    if not np.isclose(
        fast_auroc[0],
        sklearn_auroc,
        rtol=1e-11,
        atol=1e-12,
    ):
        raise AssertionError(
            f"Fast AUROC validation failed for {model_key}: "
            f"{fast_auroc[0]:.15f} vs {sklearn_auroc:.15f}"
        )

    point_metrics[model_key] = {
        "auprc": sklearn_auprc,
        "auroc": sklearn_auroc,
    }

    point_estimate_rows.append(
        {
            "model_key": model_key,
            "model": specification["display_name"],
            "rows": n_rows,
            "events": n_events,
            "negatives": n_negatives,
            "prevalence": prevalence,
            "point_auprc": sklearn_auprc,
            "point_auprc_minus_prevalence": (
                sklearn_auprc - prevalence
            ),
            "point_auroc": sklearn_auroc,
            "point_auroc_minus_0_50": (
                sklearn_auroc - 0.50
            ),
            "fast_metric_validation": "PASS",
        }
    )

exact_link_point_estimates = pd.DataFrame(
    point_estimate_rows
)


# --------------------------------------------------------------------------------------------------
# 6. RUN 2,000 PAIRED EXACT-LINK-ONLY BOOTSTRAP REPLICATES
# --------------------------------------------------------------------------------------------------

rng = np.random.default_rng(RANDOM_SEED)

bootstrap_metrics = {
    model_key: {
        "auprc": np.full(
            N_BOOTSTRAP,
            np.nan,
            dtype=np.float64,
        ),
        "auroc": np.full(
            N_BOOTSTRAP,
            np.nan,
            dtype=np.float64,
        ),
    }
    for model_key in SCORE_SPECIFICATIONS
}

bootstrap_prevalence = np.full(
    N_BOOTSTRAP,
    np.nan,
    dtype=np.float64,
)

probabilities = np.full(
    n_rows,
    1.0 / n_rows,
    dtype=np.float64,
)
probabilities[-1] = 1.0 - probabilities[:-1].sum()

analysis_start_time = time.time()

print(
    f"Preparing exact-link-only bootstrap: "
    f"{n_rows:,} rows, {n_events:,} events, "
    f"{n_negatives:,} negatives"
)
print(
    "Fast metric validation against scikit-learn: PASS "
    "for all nine scores"
)

for batch_start in range(
    0,
    N_BOOTSTRAP,
    BOOTSTRAP_BATCH_SIZE,
):
    batch_end = min(
        batch_start + BOOTSTRAP_BATCH_SIZE,
        N_BOOTSTRAP,
    )
    batch_size = batch_end - batch_start

    bootstrap_counts = rng.multinomial(
        n_rows,
        probabilities,
        size=batch_size,
    )

    if not np.all(
        bootstrap_counts.sum(axis=1) == n_rows
    ):
        raise AssertionError(
            "Bootstrap sample-size preservation failed."
        )

    positive_totals = (
        bootstrap_counts @ outcomes
    ).astype(np.float64)

    valid_outcomes = (
        (positive_totals > 0.0)
        & (positive_totals < n_rows)
    )

    bootstrap_prevalence[
        batch_start:batch_end
    ] = np.where(
        valid_outcomes,
        positive_totals / n_rows,
        np.nan,
    )

    for model_key in SCORE_SPECIFICATIONS:
        batch_auprc, batch_auroc = (
            calculate_grouped_weighted_metrics(
                caches[model_key],
                bootstrap_counts,
                positive_totals,
            )
        )

        bootstrap_metrics[
            model_key
        ]["auprc"][
            batch_start:batch_end
        ] = batch_auprc

        bootstrap_metrics[
            model_key
        ]["auroc"][
            batch_start:batch_end
        ] = batch_auroc

    if (
        batch_end % 250 == 0
        or batch_end == N_BOOTSTRAP
    ):
        valid_so_far = int(
            np.isfinite(
                bootstrap_metrics[
                    "full_ges"
                ]["auprc"][:batch_end]
            ).sum()
        )
        elapsed = time.time() - analysis_start_time
        print(
            f"Completed {batch_end:,}/{N_BOOTSTRAP:,} "
            f"replicates | valid {valid_so_far:,} | "
            f"elapsed {elapsed:.1f}s"
        )

    del bootstrap_counts
    gc.collect()

valid_mask = np.isfinite(
    bootstrap_metrics["full_ges"]["auprc"]
)
valid_replicates = int(valid_mask.sum())
invalid_one_class_replicates = (
    N_BOOTSTRAP - valid_replicates
)

if valid_replicates < MINIMUM_VALID_REPLICATES:
    raise AssertionError(
        f"Only {valid_replicates:,} valid bootstrap "
        "replicates were produced."
    )

for model_key in SCORE_SPECIFICATIONS:
    for metric_name in ["auprc", "auroc"]:
        model_valid = np.isfinite(
            bootstrap_metrics[
                model_key
            ][metric_name]
        )
        if not np.array_equal(
            model_valid,
            valid_mask,
        ):
            raise AssertionError(
                f"Paired validity mismatch for "
                f"{model_key}/{metric_name}."
            )


# --------------------------------------------------------------------------------------------------
# 7. MODEL-SPECIFIC INTERVALS
# --------------------------------------------------------------------------------------------------

model_interval_rows = []

for model_key, specification in SCORE_SPECIFICATIONS.items():
    auprc_values = bootstrap_metrics[
        model_key
    ]["auprc"]

    auroc_values = bootstrap_metrics[
        model_key
    ]["auroc"]

    auprc_lower, auprc_upper = percentile_interval(
        auprc_values
    )
    auroc_lower, auroc_upper = percentile_interval(
        auroc_values
    )

    auprc_null_lower, auprc_null_upper = (
        percentile_interval(
            auprc_values - bootstrap_prevalence
        )
    )

    auroc_null_lower, auroc_null_upper = (
        percentile_interval(
            auroc_values - 0.50
        )
    )

    model_interval_rows.append(
        {
            "model_key": model_key,
            "model": specification["display_name"],
            "rows": n_rows,
            "events": n_events,
            "negatives": n_negatives,
            "prevalence": prevalence,
            "point_auprc": point_metrics[
                model_key
            ]["auprc"],
            "auprc_ci_lower": auprc_lower,
            "auprc_ci_upper": auprc_upper,
            "point_auprc_minus_prevalence": (
                point_metrics[model_key]["auprc"]
                - prevalence
            ),
            "auprc_minus_prevalence_ci_lower": (
                auprc_null_lower
            ),
            "auprc_minus_prevalence_ci_upper": (
                auprc_null_upper
            ),
            "auprc_null_status": interval_status(
                auprc_null_lower,
                auprc_null_upper,
                "supported_above_prevalence",
                "supported_below_prevalence",
            ),
            "point_auroc": point_metrics[
                model_key
            ]["auroc"],
            "auroc_ci_lower": auroc_lower,
            "auroc_ci_upper": auroc_upper,
            "point_auroc_minus_0_50": (
                point_metrics[model_key]["auroc"]
                - 0.50
            ),
            "auroc_minus_0_50_ci_lower": (
                auroc_null_lower
            ),
            "auroc_minus_0_50_ci_upper": (
                auroc_null_upper
            ),
            "auroc_null_status": interval_status(
                auroc_null_lower,
                auroc_null_upper,
                "supported_above_0_50",
                "supported_below_0_50",
            ),
            "attempted_bootstrap_replicates": (
                N_BOOTSTRAP
            ),
            "valid_bootstrap_replicates": (
                valid_replicates
            ),
            "invalid_one_class_replicates": (
                invalid_one_class_replicates
            ),
        }
    )

exact_link_model_intervals = pd.DataFrame(
    model_interval_rows
)


# --------------------------------------------------------------------------------------------------
# 8. PAIRED FULL-GES-MINUS-COMPARATOR INFERENCE
# --------------------------------------------------------------------------------------------------

paired_difference_rows = []

for comparison_family, comparator_keys in [
    ("principal_prespecified", PRINCIPAL_COMPARATORS),
    ("secondary_remaining_comparators", SECONDARY_COMPARATORS),
]:
    for comparator_key in comparator_keys:
        comparator_name = SCORE_SPECIFICATIONS[
            comparator_key
        ]["display_name"]

        for metric_name in ["auprc", "auroc"]:
            differences = (
                bootstrap_metrics[
                    "full_ges"
                ][metric_name]
                - bootstrap_metrics[
                    comparator_key
                ][metric_name]
            )

            finite_differences = differences[
                np.isfinite(differences)
            ]

            difference_lower, difference_upper = (
                percentile_interval(
                    finite_differences
                )
            )

            point_difference = (
                point_metrics[
                    "full_ges"
                ][metric_name]
                - point_metrics[
                    comparator_key
                ][metric_name]
            )

            paired_difference_rows.append(
                {
                    "metric": metric_name.upper(),
                    "comparison_family": comparison_family,
                    "comparison": (
                        f"Full GES minus {comparator_name}"
                    ),
                    "comparator_key": comparator_key,
                    "rows": n_rows,
                    "events": n_events,
                    "negatives": n_negatives,
                    "point_difference": point_difference,
                    "difference_ci_lower": (
                        difference_lower
                    ),
                    "difference_ci_upper": (
                        difference_upper
                    ),
                    "paired_interval_status": (
                        interval_status(
                            difference_lower,
                            difference_upper,
                            "full_ges_supported_higher",
                            "full_ges_supported_lower",
                        )
                    ),
                    "bootstrap_probability_full_greater": (
                        float(
                            np.mean(
                                finite_differences > 0.0
                            )
                        )
                    ),
                    "bootstrap_probability_equal": (
                        float(
                            np.mean(
                                finite_differences == 0.0
                            )
                        )
                    ),
                    "bootstrap_sign_p_value": (
                        bootstrap_sign_pvalue(
                            finite_differences
                        )
                    ),
                    "secondary_family_holm_adjusted_p": (
                        np.nan
                    ),
                    "secondary_family_holm_supported_at_0_05": (
                        pd.NA
                    ),
                    "attempted_bootstrap_replicates": (
                        N_BOOTSTRAP
                    ),
                    "valid_bootstrap_replicates": (
                        valid_replicates
                    ),
                    "invalid_one_class_replicates": (
                        invalid_one_class_replicates
                    ),
                }
            )

exact_link_paired_differences = pd.DataFrame(
    paired_difference_rows
)

exact_link_paired_differences[
    "secondary_family_holm_supported_at_0_05"
] = pd.Series(
    pd.NA,
    index=exact_link_paired_differences.index,
    dtype="boolean",
)

for metric_name in ["AUPRC", "AUROC"]:
    family_mask = (
        (
            exact_link_paired_differences[
                "metric"
            ] == metric_name
        )
        & (
            exact_link_paired_differences[
                "comparison_family"
            ] == "secondary_remaining_comparators"
        )
    )

    family_p_values = (
        exact_link_paired_differences.loc[
            family_mask,
            "bootstrap_sign_p_value",
        ].to_numpy(dtype=float)
    )

    if len(family_p_values) != 5:
        raise AssertionError(
            f"{metric_name} secondary family contains "
            f"{len(family_p_values)} tests; expected 5."
        )

    adjusted_values = holm_adjust(
        family_p_values
    )

    exact_link_paired_differences.loc[
        family_mask,
        "secondary_family_holm_adjusted_p",
    ] = adjusted_values

    exact_link_paired_differences.loc[
        family_mask,
        "secondary_family_holm_supported_at_0_05",
    ] = adjusted_values < 0.05


# --------------------------------------------------------------------------------------------------
# 9. COMPLETENESS CHECKS
# --------------------------------------------------------------------------------------------------

if len(exact_link_model_intervals) != 9:
    raise AssertionError(
        "Nine model-specific interval rows were not produced."
    )

if len(exact_link_paired_differences) != 16:
    raise AssertionError(
        "Expected 16 paired rows: "
        "8 comparators × 2 metrics."
    )

if (
    exact_link_model_intervals[
        "valid_bootstrap_replicates"
    ] < MINIMUM_VALID_REPLICATES
).any():
    raise AssertionError(
        "At least one model has too few valid replicates."
    )

secondary_mask = (
    exact_link_paired_differences[
        "comparison_family"
    ] == "secondary_remaining_comparators"
)

principal_mask = (
    exact_link_paired_differences[
        "comparison_family"
    ] == "principal_prespecified"
)

if exact_link_paired_differences.loc[
    secondary_mask,
    "secondary_family_holm_adjusted_p",
].isna().any():
    raise AssertionError(
        "A secondary comparison is missing its "
        "Holm-adjusted value."
    )

if exact_link_paired_differences.loc[
    principal_mask,
    "secondary_family_holm_adjusted_p",
].notna().any():
    raise AssertionError(
        "A principal comparison was incorrectly included "
        "in the secondary Holm family."
    )


# --------------------------------------------------------------------------------------------------
# 10. DISPLAY TABLES AND FINAL DECISION
# --------------------------------------------------------------------------------------------------

model_display_columns = [
    "model",
    "rows",
    "events",
    "negatives",
    "prevalence",
    "point_auprc",
    "auprc_ci_lower",
    "auprc_ci_upper",
    "point_auprc_minus_prevalence",
    "auprc_minus_prevalence_ci_lower",
    "auprc_minus_prevalence_ci_upper",
    "auprc_null_status",
    "point_auroc",
    "auroc_ci_lower",
    "auroc_ci_upper",
    "point_auroc_minus_0_50",
    "auroc_minus_0_50_ci_lower",
    "auroc_minus_0_50_ci_upper",
    "auroc_null_status",
    "valid_bootstrap_replicates",
    "invalid_one_class_replicates",
]

paired_display_columns = [
    "metric",
    "comparison_family",
    "comparison",
    "point_difference",
    "difference_ci_lower",
    "difference_ci_upper",
    "paired_interval_status",
    "bootstrap_probability_full_greater",
    "bootstrap_sign_p_value",
    "secondary_family_holm_adjusted_p",
    "secondary_family_holm_supported_at_0_05",
    "valid_bootstrap_replicates",
    "invalid_one_class_replicates",
]

exact_link_model_display = (
    exact_link_model_intervals[
        model_display_columns
    ].copy()
)

exact_link_paired_display = (
    exact_link_paired_differences[
        paired_display_columns
    ].copy()
)

analysis_elapsed = (
    time.time() - analysis_start_time
)

separator = "=" * 150

print("\n" + separator)
print(
    "STAGE 6C STEP 3D — CELL 6C-3D1 — "
    "EXACT-LINK-ONLY PAIRED BOOTSTRAP INFERENCE"
)
print(separator)
print(
    f"Frozen cohort SHA-256              : "
    f"PASS ({observed_sha256})"
)
print(
    f"Frozen full-cohort dimensions      : "
    f"PASS ({metadata.num_rows:,} × "
    f"{metadata.num_columns})"
)
print(
    f"Exact-link-only rows               : "
    f"{n_rows:,}"
)
print(
    f"Exact-link events / negatives      : "
    f"{n_events:,} / {n_negatives:,}"
)
print(
    f"Accepted non-exact rows excluded   : "
    f"{len(accepted_nonexact_evaluable):,}"
)
print(
    f"Bootstrap attempts                 : "
    f"{N_BOOTSTRAP:,}"
)
print(
    f"Valid / invalid replicates         : "
    f"{valid_replicates:,} / "
    f"{invalid_one_class_replicates:,}"
)
print(
    f"Random seed                        : "
    f"{RANDOM_SEED}"
)
print(
    "Multiplicity families             : "
    "five secondary AUPRC + five secondary AUROC; "
    "principal comparisons kept separate"
)
print(
    f"Elapsed time                       : "
    f"{analysis_elapsed:.1f}s"
)
print(
    "Frozen artifacts modified          : No"
)

print(
    "\nEXACT-LINK-ONLY MODEL-SPECIFIC INTERVALS"
)
print(
    exact_link_model_display.to_string(
        index=False
    )
)

print(
    "\nEXACT-LINK-ONLY PAIRED "
    "FULL-GES-MINUS-COMPARATOR INFERENCE"
)
print(
    exact_link_paired_display.to_string(
        index=False
    )
)

print("\nCELL DECISION")
print("-" * 150)
print(
    "PASS_STAGE6C_EXACT_LINK_ONLY_"
    "PAIRED_BOOTSTRAP_INFERENCE_COMPLETE"
)
print(
    "Exact-link-only model confidence intervals and "
    "paired Full-GES-minus-comparator inference are complete."
)
print(
    "Holm correction was applied separately to the five "
    "remaining-comparator AUPRC tests and five "
    "remaining-comparator AUROC tests."
)
print(
    "No score, outcome, linkage decision, threshold, weight, "
    "cohort membership in a frozen artifact, or scientific "
    "file was modified."
)

Preparing exact-link-only bootstrap: 66,469 rows, 6,433 events, 60,036 negatives
Fast metric validation against scikit-learn: PASS for all nine scores
Completed 250/2,000 replicates | valid 250 | elapsed 8.6s
Completed 500/2,000 replicates | valid 500 | elapsed 17.4s
Completed 750/2,000 replicates | valid 750 | elapsed 22.5s
Completed 1,000/2,000 replicates | valid 1,000 | elapsed 27.4s
Completed 1,250/2,000 replicates | valid 1,250 | elapsed 33.4s
Completed 1,500/2,000 replicates | valid 1,500 | elapsed 38.3s
Completed 1,750/2,000 replicates | valid 1,750 | elapsed 44.3s
Completed 2,000/2,000 replicates | valid 2,000 | elapsed 50.1s

STAGE 6C STEP 3D — CELL 6C-3D1 — EXACT-LINK-ONLY PAIRED BOOTSTRAP INFERENCE
Frozen cohort SHA-256              : PASS (c6ad50dfc376a746e1830d000dba975101d88763fd61b8d1983f6300edb6e038)
Frozen full-cohort dimensions      : PASS (66,636 × 79)
Exact-link-only rows               : 66,469
Exact-link events / negatives      : 6,433 / 60,036
Accepted non-exact r

In [36]:
# ==================================================================================================
# STAGE 6C STEP 3E — CELL 6C-3E0
# PRIMARY EVENT-COMPONENT PREFLIGHT, RECONSTRUCTION, AND LOCKED POINT ESTIMATES
#
# Purpose:
#   1. Freshly verify the frozen 66,636-row Stage 6B primary-evaluable cohort.
#   2. Identify the three frozen Stage 5 primary event-component columns without guessing
#      their exact names.
#   3. Reconstruct and verify the frozen primary outcome:
#          material clinical-group change: 1,405
#          new unresolved conflict:         4,789
#          material prior-conflict resolution: 297
#          material-change + new-conflict overlap: 6
#          unique primary events: 6,485
#   4. Calculate locked AUPRC/AUROC point estimates for all nine prespecified scores for
#      each component outcome.
#   5. Calculate Full-GES-minus-principal-comparator point differences and full-GES
#      5%/10%/20% risk-enrichment point estimates for each component.
#
# This cell is read-only. It does not modify the frozen primary outcome, component flags,
# scores, linkage decisions, thresholds, weights, cohort membership, or scientific files.
# ==================================================================================================

from pathlib import Path
import hashlib
import re

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from sklearn.metrics import average_precision_score, roc_auc_score


# --------------------------------------------------------------------------------------------------
# 1. FROZEN INPUT AND PRESPECIFIED COMPONENT ACCOUNTING
# --------------------------------------------------------------------------------------------------

STAGE6_DIR = Path(
    "/content/drive/MyDrive/GES_RAG_Temporal_Study/"
    "data_processed/stage6_temporal_validation"
)

EVALUABLE_PARQUET = (
    STAGE6_DIR / "stage6b_locked_primary_evaluable_cohort_v1.parquet"
)
EVALUABLE_SIDECAR = Path(str(EVALUABLE_PARQUET) + ".sha256")

EXPECTED_EVALUABLE_SHA256 = (
    "c6ad50dfc376a746e1830d000dba975101d88763fd61b8d1983f6300edb6e038"
)

EXPECTED_ROWS = 66_636
EXPECTED_COLUMNS = 79
EXPECTED_PRIMARY_EVENTS = 6_485
EXPECTED_PRIMARY_NEGATIVES = 60_151

EXPECTED_COMPONENT_COUNTS = {
    "material_group_change": 1_405,
    "new_unresolved_conflict": 4_789,
    "material_prior_conflict_resolution": 297,
}

EXPECTED_MATERIAL_AND_NEW_CONFLICT_OVERLAP = 6
EXPECTED_OTHER_PAIRWISE_OVERLAPS = 0
EXPECTED_TRIPLE_OVERLAP = 0

RCV_COLUMN = "rcv_accession"
ROW_ORDER_COLUMN = "t0_row_order"
PRIMARY_OUTCOME_COLUMN = "primary_future_instability"

RISK_FRACTIONS = [0.05, 0.10, 0.20]

SCORE_SPECIFICATIONS = {
    "full_ges": {
        "column": "full_ges_instability_risk_t0",
        "display_name": "Full GES",
    },
    "no_star_ges": {
        "column": "no_star_ges_instability_risk_t0",
        "display_name": "No-star GES",
    },
    "review_stars": {
        "column": "review_stars_instability_risk",
        "display_name": "Review stars",
    },
    "combined_metadata": {
        "column": "combined_metadata_instability_risk",
        "display_name": "Combined metadata",
    },
    "conflict": {
        "column": "conflict_instability_risk",
        "display_name": "Conflict",
    },
    "recency": {
        "column": "recency_instability_risk",
        "display_name": "Recency",
    },
    "submitter": {
        "column": "submitter_instability_risk",
        "display_name": "Submitter support",
    },
    "entropy": {
        "column": "entropy_instability_risk",
        "display_name": "Classification entropy",
    },
    "additive": {
        "column": "additive_instability_risk",
        "display_name": "Additive risk",
    },
}

PRINCIPAL_COMPARATORS = [
    "no_star_ges",
    "review_stars",
    "combined_metadata",
]

COMPONENT_DISPLAY_NAMES = {
    "material_group_change": "Material clinical-group change",
    "new_unresolved_conflict": "New unresolved conflict",
    "material_prior_conflict_resolution": "Material prior-conflict resolution",
}

SEMANTIC_TOKENS = {
    "material_group_change": {
        "positive": ["MATERIAL", "GROUP", "CHANGE"],
        "supportive": ["CLINICAL", "CLASSIFICATION", "CROSS"],
        "negative": ["CONFLICT", "RESOLUTION", "REVIEW", "STAR"],
    },
    "new_unresolved_conflict": {
        "positive": ["NEW", "CONFLICT"],
        "supportive": ["UNRESOLVED", "T1"],
        "negative": ["PRIOR", "RESOLUTION", "GROUP_CHANGE", "REVIEW", "STAR"],
    },
    "material_prior_conflict_resolution": {
        "positive": ["PRIOR", "CONFLICT", "RESOL"],
        "supportive": ["MATERIAL", "T1"],
        "negative": ["NEW_CONFLICT", "GROUP_CHANGE", "REVIEW", "STAR"],
    },
}


# --------------------------------------------------------------------------------------------------
# 2. HELPERS
# --------------------------------------------------------------------------------------------------

def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def read_sidecar_hash(path: Path) -> str:
    text = path.read_text(encoding="utf-8").strip()
    matches = re.findall(r"\b[a-fA-F0-9]{64}\b", text)
    if not matches:
        raise ValueError(f"No SHA-256 value found in sidecar: {path}")
    return matches[0].lower()


def normalize_name(value: str) -> str:
    return re.sub(r"[^A-Z0-9]+", "_", str(value).upper()).strip("_")


def semantic_name_score(column: str, component_key: str) -> int:
    normalized = normalize_name(column)
    token_spec = SEMANTIC_TOKENS[component_key]

    score = 0
    for token in token_spec["positive"]:
        if token in normalized:
            score += 5
    for token in token_spec["supportive"]:
        if token in normalized:
            score += 2
    for token in token_spec["negative"]:
        if token in normalized:
            score -= 4

    if "EVENT" in normalized:
        score += 1
    if "FLAG" in normalized:
        score += 1
    if "PRIMARY" in normalized:
        score += 1

    return score


def coerce_binary_series(series: pd.Series) -> np.ndarray | None:
    # Boolean columns.
    if pd.api.types.is_bool_dtype(series.dtype):
        if series.isna().any():
            return None
        return series.to_numpy(dtype=bool)

    # Numeric 0/1 columns.
    if pd.api.types.is_numeric_dtype(series.dtype):
        numeric = pd.to_numeric(series, errors="coerce")
        if numeric.isna().any():
            return None
        unique_values = set(numeric.unique().tolist())
        if unique_values.issubset({0, 1, 0.0, 1.0, False, True}):
            return numeric.eq(1).to_numpy(dtype=bool)
        return None

    # Small string-like binary columns only. Avoid costly normalization on arbitrary high-cardinality fields.
    if pd.api.types.is_string_dtype(series.dtype) or series.dtype == object:
        nonmissing = series.dropna()
        if len(nonmissing) != len(series):
            return None

        raw_unique = nonmissing.astype(str).str.strip().str.upper().unique().tolist()
        if len(raw_unique) > 4:
            return None

        true_values = {"TRUE", "T", "YES", "Y", "1"}
        false_values = {"FALSE", "F", "NO", "N", "0"}
        unique_set = set(raw_unique)

        if unique_set.issubset(true_values | false_values):
            normalized = series.astype(str).str.strip().str.upper()
            return normalized.isin(true_values).to_numpy(dtype=bool)

    return None


def calculate_component_metrics(
    frame: pd.DataFrame,
    component_key: str,
    outcome: np.ndarray,
) -> pd.DataFrame:
    rows = []
    prevalence = float(outcome.mean())
    events = int(outcome.sum())
    negatives = int(len(outcome) - events)

    if events == 0 or negatives == 0:
        raise AssertionError(
            f"{component_key} does not contain both outcome classes."
        )

    for model_key, specification in SCORE_SPECIFICATIONS.items():
        score = frame[specification["column"]].to_numpy(dtype=float)

        auprc = float(average_precision_score(outcome, score))
        auroc = float(roc_auc_score(outcome, score))

        rows.append(
            {
                "component_key": component_key,
                "component": COMPONENT_DISPLAY_NAMES[component_key],
                "model_key": model_key,
                "model": specification["display_name"],
                "rows": int(len(outcome)),
                "events": events,
                "negatives": negatives,
                "prevalence": prevalence,
                "point_auprc": auprc,
                "auprc_minus_prevalence": auprc - prevalence,
                "auprc_lift_over_prevalence": (
                    auprc / prevalence if prevalence > 0 else np.nan
                ),
                "point_auroc": auroc,
                "auroc_minus_0_50": auroc - 0.50,
            }
        )

    return pd.DataFrame(rows)


def calculate_principal_differences(
    component_metrics: pd.DataFrame,
) -> pd.DataFrame:
    rows = []

    for component_key in EXPECTED_COMPONENT_COUNTS:
        subset = component_metrics.loc[
            component_metrics["component_key"] == component_key
        ].set_index("model_key")

        for comparator_key in PRINCIPAL_COMPARATORS:
            comparator_name = SCORE_SPECIFICATIONS[comparator_key]["display_name"]

            for metric_name in ["point_auprc", "point_auroc"]:
                rows.append(
                    {
                        "component_key": component_key,
                        "component": COMPONENT_DISPLAY_NAMES[component_key],
                        "metric": (
                            "AUPRC" if metric_name == "point_auprc" else "AUROC"
                        ),
                        "comparison": f"Full GES minus {comparator_name}",
                        "comparator_key": comparator_key,
                        "point_difference": float(
                            subset.loc["full_ges", metric_name]
                            - subset.loc[comparator_key, metric_name]
                        ),
                    }
                )

    return pd.DataFrame(rows)


def calculate_component_enrichment(
    frame: pd.DataFrame,
    component_masks: dict[str, np.ndarray],
) -> pd.DataFrame:
    full_ges_column = SCORE_SPECIFICATIONS["full_ges"]["column"]

    ordered_positions = (
        frame[
            [RCV_COLUMN, ROW_ORDER_COLUMN, full_ges_column]
        ]
        .copy()
        .assign(_position=np.arange(len(frame)))
        .sort_values(
            [full_ges_column, ROW_ORDER_COLUMN, RCV_COLUMN],
            ascending=[False, True, True],
            kind="mergesort",
        )["_position"]
        .to_numpy(dtype=int)
    )

    ordered_risk = frame.iloc[ordered_positions][full_ges_column].to_numpy(
        dtype=float
    )

    rows = []

    for component_key, mask in component_masks.items():
        ordered_outcome = np.asarray(mask, dtype=np.int8)[ordered_positions]
        prevalence = float(ordered_outcome.mean())
        n_rows = len(ordered_outcome)

        for fraction in RISK_FRACTIONS:
            n_selected = int(np.ceil(n_rows * fraction))
            selected = ordered_outcome[:n_selected]
            remaining = ordered_outcome[n_selected:]

            selected_rate = float(selected.mean())
            remaining_rate = float(remaining.mean())
            enrichment = (
                selected_rate / prevalence if prevalence > 0 else np.nan
            )
            risk_ratio = (
                selected_rate / remaining_rate
                if remaining_rate > 0
                else np.nan
            )

            cutoff_score = float(ordered_risk[n_selected - 1])
            boundary_tie_size = int(
                np.count_nonzero(ordered_risk == cutoff_score)
            )
            boundary_tie_selected = int(
                np.count_nonzero(
                    ordered_risk[:n_selected] == cutoff_score
                )
            )

            rows.append(
                {
                    "component_key": component_key,
                    "component": COMPONENT_DISPLAY_NAMES[component_key],
                    "risk_fraction": fraction,
                    "selected_rows": n_selected,
                    "selected_events": int(selected.sum()),
                    "selected_event_rate": selected_rate,
                    "remaining_rows": int(len(remaining)),
                    "remaining_events": int(remaining.sum()),
                    "remaining_event_rate": remaining_rate,
                    "component_prevalence": prevalence,
                    "risk_ratio_vs_remaining": risk_ratio,
                    "enrichment_over_prevalence": enrichment,
                    "cutoff_instability_risk": cutoff_score,
                    "boundary_tie_size": boundary_tie_size,
                    "boundary_tie_selected": boundary_tie_selected,
                }
            )

    return pd.DataFrame(rows)


# --------------------------------------------------------------------------------------------------
# 3. FRESH CRYPTOGRAPHIC AND STRUCTURAL VERIFICATION
# --------------------------------------------------------------------------------------------------

if not EVALUABLE_PARQUET.exists():
    raise FileNotFoundError(
        f"Missing frozen evaluable cohort:\n{EVALUABLE_PARQUET}"
    )

if not EVALUABLE_SIDECAR.exists():
    raise FileNotFoundError(
        f"Missing SHA-256 sidecar:\n{EVALUABLE_SIDECAR}"
    )

observed_sha256 = sha256_file(EVALUABLE_PARQUET)
sidecar_sha256 = read_sidecar_hash(EVALUABLE_SIDECAR)

if observed_sha256 != EXPECTED_EVALUABLE_SHA256:
    raise AssertionError(
        "Frozen Stage 6B evaluable-cohort SHA-256 mismatch.\n"
        f"Expected: {EXPECTED_EVALUABLE_SHA256}\n"
        f"Observed: {observed_sha256}"
    )

if sidecar_sha256 != observed_sha256:
    raise AssertionError(
        "Frozen evaluable-cohort sidecar mismatch.\n"
        f"Calculated: {observed_sha256}\n"
        f"Sidecar:    {sidecar_sha256}"
    )

parquet_file = pq.ParquetFile(EVALUABLE_PARQUET)
metadata = parquet_file.metadata
schema_columns = parquet_file.schema_arrow.names

if metadata.num_rows != EXPECTED_ROWS:
    raise AssertionError(
        f"Unexpected row count: {metadata.num_rows:,}; expected {EXPECTED_ROWS:,}"
    )

if metadata.num_columns != EXPECTED_COLUMNS:
    raise AssertionError(
        f"Unexpected column count: {metadata.num_columns}; expected {EXPECTED_COLUMNS}"
    )

required_columns = [
    RCV_COLUMN,
    ROW_ORDER_COLUMN,
    PRIMARY_OUTCOME_COLUMN,
] + [specification["column"] for specification in SCORE_SPECIFICATIONS.values()]

missing_required_columns = [
    column for column in required_columns if column not in schema_columns
]
if missing_required_columns:
    raise KeyError(
        "Required frozen columns are missing:\n"
        + "\n".join(missing_required_columns)
    )

# Load all 79 columns because the three component field names are discovered and verified
# from the immutable schema rather than assumed.
cohort = pd.read_parquet(EVALUABLE_PARQUET).copy()

if len(cohort) != EXPECTED_ROWS:
    raise AssertionError("Loaded dataframe does not match frozen row count.")

if cohort[RCV_COLUMN].isna().any():
    raise AssertionError("Missing RCV accession detected.")

if cohort[RCV_COLUMN].nunique(dropna=False) != EXPECTED_ROWS:
    raise AssertionError("RCV accessions are not unique.")

row_order = pd.to_numeric(cohort[ROW_ORDER_COLUMN], errors="raise").to_numpy()
if len(np.unique(row_order)) != EXPECTED_ROWS:
    raise AssertionError("t0_row_order is not unique.")
if not np.all(np.diff(row_order) > 0):
    raise AssertionError("Frozen cohort is not in increasing T0 row order.")

cohort[PRIMARY_OUTCOME_COLUMN] = pd.to_numeric(
    cohort[PRIMARY_OUTCOME_COLUMN],
    errors="raise",
).astype(int)

primary_outcome = cohort[PRIMARY_OUTCOME_COLUMN].to_numpy(dtype=np.int8)

if not set(np.unique(primary_outcome)).issubset({0, 1}):
    raise AssertionError("Frozen primary outcome is not binary.")

if int(primary_outcome.sum()) != EXPECTED_PRIMARY_EVENTS:
    raise AssertionError("Frozen primary event count does not equal 6,485.")

if int((primary_outcome == 0).sum()) != EXPECTED_PRIMARY_NEGATIVES:
    raise AssertionError("Frozen primary negative count does not equal 60,151.")

for specification in SCORE_SPECIFICATIONS.values():
    column = specification["column"]
    cohort[column] = pd.to_numeric(cohort[column], errors="raise")
    values = cohort[column].to_numpy(dtype=float)

    if not np.isfinite(values).all():
        raise AssertionError(f"Missing/nonfinite score in {column}.")
    if ((values < 0.0) | (values > 1.0)).any():
        raise AssertionError(f"Score outside [0,1] in {column}.")
    if len(np.unique(values)) < 2:
        raise AssertionError(f"Insufficient score variation in {column}.")


# --------------------------------------------------------------------------------------------------
# 4. IDENTIFY THE THREE FROZEN COMPONENT FIELDS
# --------------------------------------------------------------------------------------------------

excluded_columns = {
    RCV_COLUMN,
    ROW_ORDER_COLUMN,
    PRIMARY_OUTCOME_COLUMN,
    *[specification["column"] for specification in SCORE_SPECIFICATIONS.values()],
}

binary_candidates = []

for column in cohort.columns:
    if column in excluded_columns:
        continue

    binary_mask = coerce_binary_series(cohort[column])
    if binary_mask is None:
        continue

    binary_candidates.append(
        {
            "column": column,
            "positive_count": int(binary_mask.sum()),
            "mask": binary_mask,
        }
    )

binary_candidate_summary = pd.DataFrame(
    [
        {
            "column": candidate["column"],
            "positive_count": candidate["positive_count"],
        }
        for candidate in binary_candidates
    ]
).sort_values(["positive_count", "column"]).reset_index(drop=True)

component_candidate_rows = []
component_candidate_masks = {}

for component_key, expected_count in EXPECTED_COMPONENT_COUNTS.items():
    matching = [
        candidate
        for candidate in binary_candidates
        if candidate["positive_count"] == expected_count
    ]

    if not matching:
        nearby = binary_candidate_summary.loc[
            binary_candidate_summary["positive_count"].between(
                max(0, expected_count - 10),
                expected_count + 10,
            )
        ]
        print("\nAll binary-column counts:")
        print(binary_candidate_summary.to_string(index=False))
        print(f"\nNearby counts for {component_key}:")
        print(nearby.to_string(index=False))
        raise AssertionError(
            f"No binary frozen field has the expected {expected_count:,} "
            f"positive rows for {component_key}."
        )

    ranked = sorted(
        matching,
        key=lambda candidate: (
            semantic_name_score(candidate["column"], component_key),
            candidate["column"],
        ),
        reverse=True,
    )

    top_score = semantic_name_score(ranked[0]["column"], component_key)
    top_candidates = [
        candidate
        for candidate in ranked
        if semantic_name_score(candidate["column"], component_key) == top_score
    ]

    # If names tie, later reconstruction checks will test combinations.
    for candidate in matching:
        component_candidate_rows.append(
            {
                "component_key": component_key,
                "expected_count": expected_count,
                "candidate_column": candidate["column"],
                "semantic_score": semantic_name_score(
                    candidate["column"],
                    component_key,
                ),
                "top_semantic_score": top_score,
            }
        )

    component_candidate_masks[component_key] = matching

component_candidate_diagnostics = pd.DataFrame(component_candidate_rows)


# --------------------------------------------------------------------------------------------------
# 5. SELECT THE UNIQUE COMBINATION THAT RECONSTRUCTS THE PRIMARY OUTCOME
# --------------------------------------------------------------------------------------------------

valid_combinations = []

for material_candidate in component_candidate_masks["material_group_change"]:
    for new_conflict_candidate in component_candidate_masks["new_unresolved_conflict"]:
        for prior_candidate in component_candidate_masks[
            "material_prior_conflict_resolution"
        ]:
            selected_columns = {
                material_candidate["column"],
                new_conflict_candidate["column"],
                prior_candidate["column"],
            }
            if len(selected_columns) != 3:
                continue

            material_mask = material_candidate["mask"]
            new_conflict_mask = new_conflict_candidate["mask"]
            prior_mask = prior_candidate["mask"]

            material_new_overlap = int(
                np.count_nonzero(material_mask & new_conflict_mask)
            )
            material_prior_overlap = int(
                np.count_nonzero(material_mask & prior_mask)
            )
            new_prior_overlap = int(
                np.count_nonzero(new_conflict_mask & prior_mask)
            )
            triple_overlap = int(
                np.count_nonzero(
                    material_mask & new_conflict_mask & prior_mask
                )
            )

            reconstructed = material_mask | new_conflict_mask | prior_mask
            reconstructed_events = int(reconstructed.sum())
            mismatch_vs_primary = int(
                np.count_nonzero(reconstructed.astype(np.int8) != primary_outcome)
            )

            if (
                material_new_overlap == EXPECTED_MATERIAL_AND_NEW_CONFLICT_OVERLAP
                and material_prior_overlap == EXPECTED_OTHER_PAIRWISE_OVERLAPS
                and new_prior_overlap == EXPECTED_OTHER_PAIRWISE_OVERLAPS
                and triple_overlap == EXPECTED_TRIPLE_OVERLAP
                and reconstructed_events == EXPECTED_PRIMARY_EVENTS
                and mismatch_vs_primary == 0
            ):
                semantic_total = (
                    semantic_name_score(
                        material_candidate["column"],
                        "material_group_change",
                    )
                    + semantic_name_score(
                        new_conflict_candidate["column"],
                        "new_unresolved_conflict",
                    )
                    + semantic_name_score(
                        prior_candidate["column"],
                        "material_prior_conflict_resolution",
                    )
                )

                valid_combinations.append(
                    {
                        "columns": {
                            "material_group_change": material_candidate["column"],
                            "new_unresolved_conflict": new_conflict_candidate["column"],
                            "material_prior_conflict_resolution": prior_candidate["column"],
                        },
                        "masks": {
                            "material_group_change": material_mask,
                            "new_unresolved_conflict": new_conflict_mask,
                            "material_prior_conflict_resolution": prior_mask,
                        },
                        "semantic_total": semantic_total,
                        "material_new_overlap": material_new_overlap,
                        "material_prior_overlap": material_prior_overlap,
                        "new_prior_overlap": new_prior_overlap,
                        "triple_overlap": triple_overlap,
                        "reconstructed_events": reconstructed_events,
                        "mismatch_vs_primary": mismatch_vs_primary,
                    }
                )

if not valid_combinations:
    print("\nComponent candidates:")
    print(component_candidate_diagnostics.to_string(index=False))
    raise AssertionError(
        "No candidate-column combination reconstructed the frozen 6,485-event "
        "primary outcome with the prespecified overlap pattern."
    )

valid_combinations = sorted(
    valid_combinations,
    key=lambda item: (
        item["semantic_total"],
        tuple(item["columns"].values()),
    ),
    reverse=True,
)

best_semantic_total = valid_combinations[0]["semantic_total"]
best_combinations = [
    item
    for item in valid_combinations
    if item["semantic_total"] == best_semantic_total
]

if len(best_combinations) != 1:
    print("\nEqually valid component combinations:")
    for combination in best_combinations:
        print(combination["columns"])
    raise AssertionError(
        "More than one equally supported component-column combination was found. "
        "Stop before performance analysis."
    )

selected = best_combinations[0]
component_columns = selected["columns"]
component_masks = selected["masks"]

component_identification = pd.DataFrame(
    [
        {
            "component_key": component_key,
            "component": COMPONENT_DISPLAY_NAMES[component_key],
            "selected_column": component_columns[component_key],
            "positive_rows": int(component_masks[component_key].sum()),
            "expected_positive_rows": EXPECTED_COMPONENT_COUNTS[component_key],
            "semantic_name_score": semantic_name_score(
                component_columns[component_key],
                component_key,
            ),
        }
        for component_key in EXPECTED_COMPONENT_COUNTS
    ]
)

component_combination_accounting = pd.DataFrame(
    [
        {
            "event_combination": "Material group change only",
            "records": int(
                np.count_nonzero(
                    component_masks["material_group_change"]
                    & ~component_masks["new_unresolved_conflict"]
                    & ~component_masks["material_prior_conflict_resolution"]
                )
            ),
            "expected_records": 1_399,
        },
        {
            "event_combination": "New unresolved conflict only",
            "records": int(
                np.count_nonzero(
                    component_masks["new_unresolved_conflict"]
                    & ~component_masks["material_group_change"]
                    & ~component_masks["material_prior_conflict_resolution"]
                )
            ),
            "expected_records": 4_783,
        },
        {
            "event_combination": "Prior conflict resolution only",
            "records": int(
                np.count_nonzero(
                    component_masks["material_prior_conflict_resolution"]
                    & ~component_masks["material_group_change"]
                    & ~component_masks["new_unresolved_conflict"]
                )
            ),
            "expected_records": 297,
        },
        {
            "event_combination": "Material group change plus new conflict",
            "records": int(
                np.count_nonzero(
                    component_masks["material_group_change"]
                    & component_masks["new_unresolved_conflict"]
                    & ~component_masks["material_prior_conflict_resolution"]
                )
            ),
            "expected_records": 6,
        },
        {
            "event_combination": "Unique primary events",
            "records": int(
                (
                    component_masks["material_group_change"]
                    | component_masks["new_unresolved_conflict"]
                    | component_masks["material_prior_conflict_resolution"]
                ).sum()
            ),
            "expected_records": 6_485,
        },
    ]
)

if not (
    component_combination_accounting["records"]
    == component_combination_accounting["expected_records"]
).all():
    raise AssertionError("Event-combination accounting failed.")


# --------------------------------------------------------------------------------------------------
# 6. CALCULATE LOCKED COMPONENT-SPECIFIC POINT ESTIMATES
# --------------------------------------------------------------------------------------------------

component_metric_tables = []

for component_key, mask in component_masks.items():
    component_metric_tables.append(
        calculate_component_metrics(
            cohort,
            component_key,
            mask.astype(np.int8),
        )
    )

component_discrimination_point_estimates = pd.concat(
    component_metric_tables,
    ignore_index=True,
)

component_principal_differences = calculate_principal_differences(
    component_discrimination_point_estimates
)

component_enrichment_point_estimates = calculate_component_enrichment(
    cohort,
    component_masks,
)


# --------------------------------------------------------------------------------------------------
# 7. COMPLETENESS CHECKS AND FINAL DISPLAY
# --------------------------------------------------------------------------------------------------

if len(component_discrimination_point_estimates) != 27:
    raise AssertionError(
        "Expected 27 discrimination rows: 3 components × 9 scores."
    )

if len(component_principal_differences) != 18:
    raise AssertionError(
        "Expected 18 principal-difference rows: "
        "3 components × 3 comparators × 2 metrics."
    )

if len(component_enrichment_point_estimates) != 9:
    raise AssertionError(
        "Expected 9 enrichment rows: 3 components × 3 risk fractions."
    )

numeric_result_columns = [
    "point_auprc",
    "point_auroc",
    "auprc_minus_prevalence",
    "auroc_minus_0_50",
]

if not np.isfinite(
    component_discrimination_point_estimates[
        numeric_result_columns
    ].to_numpy(dtype=float)
).all():
    raise AssertionError("A nonfinite component discrimination result was produced.")

separator = "=" * 150

print("\n" + separator)
print(
    "STAGE 6C STEP 3E — CELL 6C-3E0 — "
    "PRIMARY EVENT-COMPONENT POINT ESTIMATES"
)
print(separator)
print(f"Frozen cohort SHA-256              : PASS ({observed_sha256})")
print(
    f"Frozen cohort dimensions           : "
    f"PASS ({metadata.num_rows:,} × {metadata.num_columns})"
)
print(f"Unique RCV keys                    : PASS ({cohort[RCV_COLUMN].nunique():,})")
print(
    f"Primary events / negatives         : "
    f"PASS ({int(primary_outcome.sum()):,} / "
    f"{int((primary_outcome == 0).sum()):,})"
)
print("Frozen primary outcome modified    : No")
print("Frozen component fields modified   : No")
print("Scientific artifacts written       : No")

print("\nSELECTED FROZEN COMPONENT FIELDS")
print(component_identification.to_string(index=False))

print("\nCOMPONENT-CANDIDATE DIAGNOSTICS")
print(component_candidate_diagnostics.to_string(index=False))

print("\nPRIMARY EVENT-COMBINATION ACCOUNTING")
print(component_combination_accounting.to_string(index=False))

print("\nCOMPONENT-SPECIFIC NINE-SCORE DISCRIMINATION POINT ESTIMATES")
print(
    component_discrimination_point_estimates.to_string(index=False)
)

print("\nCOMPONENT-SPECIFIC FULL-GES PRINCIPAL COMPARISON DIFFERENCES")
print(component_principal_differences.to_string(index=False))

print("\nCOMPONENT-SPECIFIC FULL-GES 5%/10%/20% ENRICHMENT")
print(component_enrichment_point_estimates.to_string(index=False))

print("\nCELL DECISION")
print("-" * 150)
print(
    "PASS_STAGE6C_PRIMARY_EVENT_COMPONENT_"
    "POINT_ESTIMATES_COMPLETE"
)
print(
    "The three frozen primary event components were identified, "
    "reconstructed, and verified against the immutable 6,485-event "
    "primary outcome."
)
print(
    "Component-specific nine-score discrimination, principal-comparison "
    "differences, and full-GES risk-enrichment point estimates are complete."
)
print(
    "Inference is not yet claimed. The next cell must run paired bootstrap "
    "inference separately for the three component outcomes."
)
print(
    "No score, outcome, component flag, threshold, weight, cohort membership, "
    "or frozen scientific artifact was modified."
)


STAGE 6C STEP 3E — CELL 6C-3E0 — PRIMARY EVENT-COMPONENT POINT ESTIMATES
Frozen cohort SHA-256              : PASS (c6ad50dfc376a746e1830d000dba975101d88763fd61b8d1983f6300edb6e038)
Frozen cohort dimensions           : PASS (66,636 × 79)
Unique RCV keys                    : PASS (66,636)
Primary events / negatives         : PASS (6,485 / 60,151)
Frozen primary outcome modified    : No
Frozen component fields modified   : No
Scientific artifacts written       : No

SELECTED FROZEN COMPONENT FIELDS
                     component_key                          component                                 selected_column  positive_rows  expected_positive_rows  semantic_name_score
             material_group_change     Material clinical-group change            event_material_clinical_group_change           1405                    1405                   18
           new_unresolved_conflict            New unresolved conflict             event_new_unresolved_conflict_at_t1           4789         

In [37]:
# ==================================================================================================
# STAGE 6C STEP 3E — CELL 6C-3E1
# PRIMARY EVENT-COMPONENT PAIRED BOOTSTRAP INFERENCE
#
# Purpose:
#   1. Freshly verify the frozen 66,636-row Stage 6B primary-evaluable cohort.
#   2. Reverify the three frozen Stage 5 primary event-component fields and reconstruct
#      the immutable 6,485-event primary outcome.
#   3. Run 2,000 paired row-bootstrap replicates with seed 42 using identical resamples
#      across all three component outcomes and all nine frozen scores.
#   4. Produce component-specific AUPRC/AUROC confidence intervals.
#   5. Produce paired Full-GES-minus-comparator inference for the three principal
#      comparators and five remaining metadata comparators.
#   6. Apply Holm correction separately within each component and metric across the
#      five remaining-comparator tests.
#   7. Produce bootstrap intervals for Full-GES 5%/10%/20% component enrichment using
#      the frozen exact-rank membership defined from the original cohort.
#
# This cell is read-only. It does not modify outcomes, component flags, scores, linkage
# decisions, thresholds, weights, cohort membership, or frozen scientific artifacts.
# ==================================================================================================

from pathlib import Path
import gc
import hashlib
import re
import time

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from scipy.sparse import csr_matrix
from sklearn.metrics import average_precision_score, roc_auc_score


# --------------------------------------------------------------------------------------------------
# 1. FROZEN INPUT AND ANALYSIS SPECIFICATION
# --------------------------------------------------------------------------------------------------

STAGE6_DIR = Path(
    "/content/drive/MyDrive/GES_RAG_Temporal_Study/"
    "data_processed/stage6_temporal_validation"
)

EVALUABLE_PARQUET = (
    STAGE6_DIR / "stage6b_locked_primary_evaluable_cohort_v1.parquet"
)
EVALUABLE_SIDECAR = Path(str(EVALUABLE_PARQUET) + ".sha256")

EXPECTED_EVALUABLE_SHA256 = (
    "c6ad50dfc376a746e1830d000dba975101d88763fd61b8d1983f6300edb6e038"
)

EXPECTED_ROWS = 66_636
EXPECTED_COLUMNS = 79
EXPECTED_PRIMARY_EVENTS = 6_485
EXPECTED_PRIMARY_NEGATIVES = 60_151

RCV_COLUMN = "rcv_accession"
ROW_ORDER_COLUMN = "t0_row_order"
PRIMARY_OUTCOME_COLUMN = "primary_future_instability"

COMPONENT_SPECIFICATIONS = {
    "material_group_change": {
        "column": "event_material_clinical_group_change",
        "display_name": "Material clinical-group change",
        "expected_events": 1_405,
    },
    "new_unresolved_conflict": {
        "column": "event_new_unresolved_conflict_at_t1",
        "display_name": "New unresolved conflict",
        "expected_events": 4_789,
    },
    "material_prior_conflict_resolution": {
        "column": "event_prior_conflict_resolved_to_material_group",
        "display_name": "Material prior-conflict resolution",
        "expected_events": 297,
    },
}

EXPECTED_MATERIAL_AND_NEW_CONFLICT_OVERLAP = 6
EXPECTED_OTHER_PAIRWISE_OVERLAPS = 0
EXPECTED_TRIPLE_OVERLAP = 0

SCORE_SPECIFICATIONS = {
    "full_ges": {
        "column": "full_ges_instability_risk_t0",
        "display_name": "Full GES",
    },
    "no_star_ges": {
        "column": "no_star_ges_instability_risk_t0",
        "display_name": "No-star GES",
    },
    "review_stars": {
        "column": "review_stars_instability_risk",
        "display_name": "Review stars",
    },
    "combined_metadata": {
        "column": "combined_metadata_instability_risk",
        "display_name": "Combined metadata",
    },
    "conflict": {
        "column": "conflict_instability_risk",
        "display_name": "Conflict",
    },
    "recency": {
        "column": "recency_instability_risk",
        "display_name": "Recency",
    },
    "submitter": {
        "column": "submitter_instability_risk",
        "display_name": "Submitter support",
    },
    "entropy": {
        "column": "entropy_instability_risk",
        "display_name": "Classification entropy",
    },
    "additive": {
        "column": "additive_instability_risk",
        "display_name": "Additive risk",
    },
}

PRINCIPAL_COMPARATORS = [
    "no_star_ges",
    "review_stars",
    "combined_metadata",
]

SECONDARY_COMPARATORS = [
    "conflict",
    "recency",
    "submitter",
    "entropy",
    "additive",
]

RISK_FRACTIONS = [0.05, 0.10, 0.20]

N_BOOTSTRAP = 2_000
RANDOM_SEED = 42
BOOTSTRAP_BATCH_SIZE = 50
MINIMUM_VALID_REPLICATES = 1_000
CI_QUANTILES = (0.025, 0.975)


# --------------------------------------------------------------------------------------------------
# 2. HELPERS
# --------------------------------------------------------------------------------------------------

def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def read_sidecar_hash(path: Path) -> str:
    text = path.read_text(encoding="utf-8").strip()
    matches = re.findall(r"\b[a-fA-F0-9]{64}\b", text)
    if not matches:
        raise ValueError(f"No SHA-256 value found in sidecar: {path}")
    return matches[0].lower()


def percentile_interval(values: np.ndarray) -> tuple[float, float]:
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if len(values) == 0:
        return np.nan, np.nan
    lower, upper = np.quantile(values, CI_QUANTILES)
    return float(lower), float(upper)


def bootstrap_sign_pvalue(differences: np.ndarray) -> float:
    differences = np.asarray(differences, dtype=float)
    differences = differences[np.isfinite(differences)]
    if len(differences) == 0:
        return np.nan

    n = len(differences)
    lower_tail = (np.count_nonzero(differences <= 0.0) + 1) / (n + 1)
    upper_tail = (np.count_nonzero(differences >= 0.0) + 1) / (n + 1)
    return float(min(1.0, 2.0 * min(lower_tail, upper_tail)))


def holm_adjust(p_values: np.ndarray) -> np.ndarray:
    p_values = np.asarray(p_values, dtype=float)
    adjusted = np.full(len(p_values), np.nan, dtype=float)
    valid_positions = np.where(np.isfinite(p_values))[0]

    if len(valid_positions) == 0:
        return adjusted

    valid_p = p_values[valid_positions]
    order = np.argsort(valid_p)
    m = len(valid_p)
    running_max = 0.0

    for rank, position_within_valid in enumerate(order):
        original_position = valid_positions[position_within_valid]
        raw_adjusted = (m - rank) * valid_p[position_within_valid]
        running_max = max(running_max, raw_adjusted)
        adjusted[original_position] = min(1.0, running_max)

    return adjusted


def interval_status(
    lower: float,
    upper: float,
    positive_label: str,
    negative_label: str,
) -> str:
    if not np.isfinite(lower) or not np.isfinite(upper):
        return "not_estimable"
    if lower > 0.0:
        return positive_label
    if upper < 0.0:
        return negative_label
    return "interval_includes_null"


def construct_score_group_cache(
    scores: np.ndarray,
    outcomes: np.ndarray,
) -> dict:
    scores = np.asarray(scores, dtype=np.float64)
    outcomes = np.asarray(outcomes, dtype=np.int8)

    unique_scores, group_index = np.unique(scores, return_inverse=True)
    n_rows = len(scores)
    n_groups = len(unique_scores)
    row_positions = np.arange(n_rows)

    total_group_matrix = csr_matrix(
        (
            np.ones(n_rows, dtype=np.float64),
            (group_index, row_positions),
        ),
        shape=(n_groups, n_rows),
    )

    positive_positions = np.flatnonzero(outcomes == 1)
    positive_group_matrix = csr_matrix(
        (
            np.ones(len(positive_positions), dtype=np.float64),
            (group_index[positive_positions], positive_positions),
        ),
        shape=(n_groups, n_rows),
    )

    return {
        "total_group_matrix": total_group_matrix,
        "positive_group_matrix": positive_group_matrix,
    }


def calculate_grouped_weighted_metrics(
    cache: dict,
    count_matrix: np.ndarray,
    positive_totals: np.ndarray,
) -> tuple[np.ndarray, np.ndarray]:
    count_matrix = np.asarray(count_matrix)
    positive_totals = np.asarray(positive_totals, dtype=np.float64)

    sample_totals = count_matrix.sum(axis=1).astype(np.float64)
    negative_totals = sample_totals - positive_totals

    group_total_counts = np.asarray(
        cache["total_group_matrix"] @ count_matrix.T,
        dtype=np.float64,
    )
    group_positive_counts = np.asarray(
        cache["positive_group_matrix"] @ count_matrix.T,
        dtype=np.float64,
    )
    group_negative_counts = group_total_counts - group_positive_counts

    valid = (positive_totals > 0.0) & (negative_totals > 0.0)

    cumulative_negatives_before = (
        np.cumsum(group_negative_counts, axis=0)
        - group_negative_counts
    )
    concordant_numerator = np.sum(
        group_positive_counts
        * (
            cumulative_negatives_before
            + 0.5 * group_negative_counts
        ),
        axis=0,
    )

    auroc = np.full(len(positive_totals), np.nan, dtype=np.float64)
    np.divide(
        concordant_numerator,
        positive_totals * negative_totals,
        out=auroc,
        where=valid,
    )

    positive_desc = group_positive_counts[::-1, :]
    total_desc = group_total_counts[::-1, :]
    cumulative_positive = np.cumsum(positive_desc, axis=0)
    cumulative_total = np.cumsum(total_desc, axis=0)

    precision = np.zeros_like(cumulative_positive, dtype=np.float64)
    np.divide(
        cumulative_positive,
        cumulative_total,
        out=precision,
        where=cumulative_total > 0.0,
    )

    ap_numerator = np.sum(precision * positive_desc, axis=0)
    auprc = np.full(len(positive_totals), np.nan, dtype=np.float64)
    np.divide(
        ap_numerator,
        positive_totals,
        out=auprc,
        where=valid,
    )

    return auprc, auroc


def build_frozen_rank_membership(
    frame: pd.DataFrame,
) -> dict[float, np.ndarray]:
    full_ges_column = SCORE_SPECIFICATIONS["full_ges"]["column"]

    ordered_positions = (
        frame[
            [RCV_COLUMN, ROW_ORDER_COLUMN, full_ges_column]
        ]
        .copy()
        .assign(_position=np.arange(len(frame)))
        .sort_values(
            [full_ges_column, ROW_ORDER_COLUMN, RCV_COLUMN],
            ascending=[False, True, True],
            kind="mergesort",
        )["_position"]
        .to_numpy(dtype=int)
    )

    memberships = {}
    n_rows = len(frame)

    for fraction in RISK_FRACTIONS:
        selected_count = int(np.ceil(n_rows * fraction))
        membership = np.zeros(n_rows, dtype=np.int8)
        membership[ordered_positions[:selected_count]] = 1
        memberships[fraction] = membership

    return memberships


def calculate_weighted_enrichment(
    count_matrix: np.ndarray,
    outcome: np.ndarray,
    membership: np.ndarray,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    count_matrix = np.asarray(count_matrix, dtype=np.float64)
    outcome = np.asarray(outcome, dtype=np.float64)
    membership = np.asarray(membership, dtype=np.float64)

    sample_rows = count_matrix.sum(axis=1)
    sample_events = count_matrix @ outcome

    selected_rows = count_matrix @ membership
    selected_events = count_matrix @ (membership * outcome)

    remaining_rows = sample_rows - selected_rows
    remaining_events = sample_events - selected_events

    prevalence = np.full(len(sample_rows), np.nan, dtype=np.float64)
    selected_rate = np.full(len(sample_rows), np.nan, dtype=np.float64)
    remaining_rate = np.full(len(sample_rows), np.nan, dtype=np.float64)

    np.divide(
        sample_events,
        sample_rows,
        out=prevalence,
        where=sample_rows > 0,
    )
    np.divide(
        selected_events,
        selected_rows,
        out=selected_rate,
        where=selected_rows > 0,
    )
    np.divide(
        remaining_events,
        remaining_rows,
        out=remaining_rate,
        where=remaining_rows > 0,
    )

    enrichment = np.full(len(sample_rows), np.nan, dtype=np.float64)
    risk_ratio = np.full(len(sample_rows), np.nan, dtype=np.float64)

    np.divide(
        selected_rate,
        prevalence,
        out=enrichment,
        where=prevalence > 0,
    )
    np.divide(
        selected_rate,
        remaining_rate,
        out=risk_ratio,
        where=remaining_rate > 0,
    )

    return enrichment, risk_ratio, selected_rate


# --------------------------------------------------------------------------------------------------
# 3. FRESH CRYPTOGRAPHIC AND STRUCTURAL VERIFICATION
# --------------------------------------------------------------------------------------------------

if not EVALUABLE_PARQUET.exists():
    raise FileNotFoundError(
        f"Missing frozen evaluable cohort:\n{EVALUABLE_PARQUET}"
    )

if not EVALUABLE_SIDECAR.exists():
    raise FileNotFoundError(
        f"Missing SHA-256 sidecar:\n{EVALUABLE_SIDECAR}"
    )

observed_sha256 = sha256_file(EVALUABLE_PARQUET)
sidecar_sha256 = read_sidecar_hash(EVALUABLE_SIDECAR)

if observed_sha256 != EXPECTED_EVALUABLE_SHA256:
    raise AssertionError(
        "Frozen Stage 6B evaluable-cohort SHA-256 mismatch.\n"
        f"Expected: {EXPECTED_EVALUABLE_SHA256}\n"
        f"Observed: {observed_sha256}"
    )

if sidecar_sha256 != observed_sha256:
    raise AssertionError(
        "Frozen evaluable-cohort sidecar mismatch.\n"
        f"Calculated: {observed_sha256}\n"
        f"Sidecar:    {sidecar_sha256}"
    )

parquet_file = pq.ParquetFile(EVALUABLE_PARQUET)
metadata = parquet_file.metadata
schema_columns = parquet_file.schema_arrow.names

if metadata.num_rows != EXPECTED_ROWS:
    raise AssertionError(
        f"Unexpected row count: {metadata.num_rows:,}; expected {EXPECTED_ROWS:,}"
    )

if metadata.num_columns != EXPECTED_COLUMNS:
    raise AssertionError(
        f"Unexpected column count: {metadata.num_columns}; expected {EXPECTED_COLUMNS}"
    )

required_columns = [
    RCV_COLUMN,
    ROW_ORDER_COLUMN,
    PRIMARY_OUTCOME_COLUMN,
] + [
    specification["column"]
    for specification in COMPONENT_SPECIFICATIONS.values()
] + [
    specification["column"]
    for specification in SCORE_SPECIFICATIONS.values()
]

missing_columns = [
    column for column in required_columns
    if column not in schema_columns
]
if missing_columns:
    raise KeyError(
        "Missing required frozen columns:\n"
        + "\n".join(missing_columns)
    )

cohort = pd.read_parquet(
    EVALUABLE_PARQUET,
    columns=required_columns,
).copy()

if len(cohort) != EXPECTED_ROWS:
    raise AssertionError("Loaded dataframe row count mismatch.")

if cohort[RCV_COLUMN].isna().any():
    raise AssertionError("Missing RCV accession detected.")

if cohort[RCV_COLUMN].nunique(dropna=False) != EXPECTED_ROWS:
    raise AssertionError("RCV accessions are not unique.")

row_order = pd.to_numeric(
    cohort[ROW_ORDER_COLUMN],
    errors="raise",
).to_numpy()

if len(np.unique(row_order)) != EXPECTED_ROWS:
    raise AssertionError("t0_row_order is not unique.")

if not np.all(np.diff(row_order) > 0):
    raise AssertionError(
        "Frozen cohort is not in increasing T0 row order."
    )

cohort[PRIMARY_OUTCOME_COLUMN] = pd.to_numeric(
    cohort[PRIMARY_OUTCOME_COLUMN],
    errors="raise",
).astype(int)

primary_outcome = cohort[
    PRIMARY_OUTCOME_COLUMN
].to_numpy(dtype=np.int8)

if int(primary_outcome.sum()) != EXPECTED_PRIMARY_EVENTS:
    raise AssertionError("Primary event count mismatch.")

if int((primary_outcome == 0).sum()) != EXPECTED_PRIMARY_NEGATIVES:
    raise AssertionError("Primary negative count mismatch.")

component_outcomes = {}

for component_key, specification in COMPONENT_SPECIFICATIONS.items():
    column = specification["column"]
    cohort[column] = pd.to_numeric(
        cohort[column],
        errors="raise",
    ).astype(int)

    values = cohort[column].to_numpy(dtype=np.int8)

    if not set(np.unique(values)).issubset({0, 1}):
        raise AssertionError(
            f"{column} is not binary."
        )

    observed_events = int(values.sum())
    if observed_events != specification["expected_events"]:
        raise AssertionError(
            f"{column} has {observed_events:,} events; "
            f"expected {specification['expected_events']:,}."
        )

    component_outcomes[component_key] = values

for specification in SCORE_SPECIFICATIONS.values():
    column = specification["column"]
    cohort[column] = pd.to_numeric(
        cohort[column],
        errors="raise",
    )
    values = cohort[column].to_numpy(dtype=float)

    if not np.isfinite(values).all():
        raise AssertionError(
            f"Missing/nonfinite score in {column}."
        )
    if ((values < 0.0) | (values > 1.0)).any():
        raise AssertionError(
            f"Score outside [0,1] in {column}."
        )

material = component_outcomes["material_group_change"].astype(bool)
new_conflict = component_outcomes["new_unresolved_conflict"].astype(bool)
prior_resolution = component_outcomes[
    "material_prior_conflict_resolution"
].astype(bool)

if int(np.count_nonzero(material & new_conflict)) != EXPECTED_MATERIAL_AND_NEW_CONFLICT_OVERLAP:
    raise AssertionError(
        "Material-change/new-conflict overlap mismatch."
    )

if int(np.count_nonzero(material & prior_resolution)) != EXPECTED_OTHER_PAIRWISE_OVERLAPS:
    raise AssertionError(
        "Material-change/prior-resolution overlap mismatch."
    )

if int(np.count_nonzero(new_conflict & prior_resolution)) != EXPECTED_OTHER_PAIRWISE_OVERLAPS:
    raise AssertionError(
        "New-conflict/prior-resolution overlap mismatch."
    )

if int(np.count_nonzero(material & new_conflict & prior_resolution)) != EXPECTED_TRIPLE_OVERLAP:
    raise AssertionError("Triple-overlap mismatch.")

reconstructed_primary = (
    material | new_conflict | prior_resolution
).astype(np.int8)

if not np.array_equal(
    reconstructed_primary,
    primary_outcome,
):
    mismatch_count = int(
        np.count_nonzero(
            reconstructed_primary != primary_outcome
        )
    )
    raise AssertionError(
        f"Component reconstruction disagrees with the primary outcome "
        f"for {mismatch_count:,} rows."
    )


# --------------------------------------------------------------------------------------------------
# 4. PREPARE SCORE ARRAYS, COMPONENT-SPECIFIC CACHES, AND POINT ESTIMATES
# --------------------------------------------------------------------------------------------------

n_rows = len(cohort)

score_arrays = {
    model_key: cohort[
        specification["column"]
    ].to_numpy(dtype=np.float64)
    for model_key, specification
    in SCORE_SPECIFICATIONS.items()
}

frozen_rank_memberships = build_frozen_rank_membership(cohort)

point_estimate_rows = []
point_metrics = {}
component_caches = {}

original_counts = np.ones(
    (1, n_rows),
    dtype=np.int16,
)

for component_key, outcome in component_outcomes.items():
    component_name = COMPONENT_SPECIFICATIONS[
        component_key
    ]["display_name"]

    events = int(outcome.sum())
    negatives = n_rows - events
    prevalence = events / n_rows

    component_caches[component_key] = {}
    point_metrics[component_key] = {}

    for model_key, specification in SCORE_SPECIFICATIONS.items():
        scores = score_arrays[model_key]

        sklearn_auprc = float(
            average_precision_score(
                outcome,
                scores,
            )
        )
        sklearn_auroc = float(
            roc_auc_score(
                outcome,
                scores,
            )
        )

        cache = construct_score_group_cache(
            scores,
            outcome,
        )
        component_caches[
            component_key
        ][model_key] = cache

        fast_auprc, fast_auroc = (
            calculate_grouped_weighted_metrics(
                cache,
                original_counts,
                np.array(
                    [events],
                    dtype=np.float64,
                ),
            )
        )

        if not np.isclose(
            fast_auprc[0],
            sklearn_auprc,
            rtol=1e-11,
            atol=1e-12,
        ):
            raise AssertionError(
                f"Fast AUPRC validation failed for "
                f"{component_key}/{model_key}."
            )

        if not np.isclose(
            fast_auroc[0],
            sklearn_auroc,
            rtol=1e-11,
            atol=1e-12,
        ):
            raise AssertionError(
                f"Fast AUROC validation failed for "
                f"{component_key}/{model_key}."
            )

        point_metrics[
            component_key
        ][model_key] = {
            "auprc": sklearn_auprc,
            "auroc": sklearn_auroc,
        }

        point_estimate_rows.append(
            {
                "component_key": component_key,
                "component": component_name,
                "model_key": model_key,
                "model": specification["display_name"],
                "rows": n_rows,
                "events": events,
                "negatives": negatives,
                "prevalence": prevalence,
                "point_auprc": sklearn_auprc,
                "point_auroc": sklearn_auroc,
                "fast_metric_validation": "PASS",
            }
        )

component_point_estimates = pd.DataFrame(
    point_estimate_rows
)


# --------------------------------------------------------------------------------------------------
# 5. RUN 2,000 IDENTICAL PAIRED RESAMPLES ACROSS COMPONENTS AND MODELS
# --------------------------------------------------------------------------------------------------

rng = np.random.default_rng(RANDOM_SEED)

bootstrap_metrics = {
    component_key: {
        model_key: {
            "auprc": np.full(
                N_BOOTSTRAP,
                np.nan,
                dtype=np.float64,
            ),
            "auroc": np.full(
                N_BOOTSTRAP,
                np.nan,
                dtype=np.float64,
            ),
        }
        for model_key in SCORE_SPECIFICATIONS
    }
    for component_key in COMPONENT_SPECIFICATIONS
}

bootstrap_prevalence = {
    component_key: np.full(
        N_BOOTSTRAP,
        np.nan,
        dtype=np.float64,
    )
    for component_key in COMPONENT_SPECIFICATIONS
}

bootstrap_enrichment = {
    component_key: {
        fraction: {
            "enrichment": np.full(
                N_BOOTSTRAP,
                np.nan,
                dtype=np.float64,
            ),
            "risk_ratio": np.full(
                N_BOOTSTRAP,
                np.nan,
                dtype=np.float64,
            ),
            "selected_event_rate": np.full(
                N_BOOTSTRAP,
                np.nan,
                dtype=np.float64,
            ),
        }
        for fraction in RISK_FRACTIONS
    }
    for component_key in COMPONENT_SPECIFICATIONS
}

probabilities = np.full(
    n_rows,
    1.0 / n_rows,
    dtype=np.float64,
)
probabilities[-1] = (
    1.0 - probabilities[:-1].sum()
)

analysis_start_time = time.time()

print(
    "Preparing component bootstrap: "
    f"{n_rows:,} rows; "
    + ", ".join(
        f"{COMPONENT_SPECIFICATIONS[key]['display_name']}="
        f"{int(component_outcomes[key].sum()):,} events"
        for key in COMPONENT_SPECIFICATIONS
    )
)
print(
    "Fast metric validation against scikit-learn: "
    "PASS for all 27 component-score combinations"
)

for batch_start in range(
    0,
    N_BOOTSTRAP,
    BOOTSTRAP_BATCH_SIZE,
):
    batch_end = min(
        batch_start + BOOTSTRAP_BATCH_SIZE,
        N_BOOTSTRAP,
    )
    batch_size = batch_end - batch_start

    bootstrap_counts = rng.multinomial(
        n_rows,
        probabilities,
        size=batch_size,
    )

    if not np.all(
        bootstrap_counts.sum(axis=1) == n_rows
    ):
        raise AssertionError(
            "Bootstrap sample-size preservation failed."
        )

    for component_key, outcome in component_outcomes.items():
        positive_totals = (
            bootstrap_counts @ outcome
        ).astype(np.float64)

        valid_outcomes = (
            (positive_totals > 0.0)
            & (positive_totals < n_rows)
        )

        bootstrap_prevalence[
            component_key
        ][batch_start:batch_end] = np.where(
            valid_outcomes,
            positive_totals / n_rows,
            np.nan,
        )

        for model_key in SCORE_SPECIFICATIONS:
            batch_auprc, batch_auroc = (
                calculate_grouped_weighted_metrics(
                    component_caches[
                        component_key
                    ][model_key],
                    bootstrap_counts,
                    positive_totals,
                )
            )

            bootstrap_metrics[
                component_key
            ][model_key]["auprc"][
                batch_start:batch_end
            ] = batch_auprc

            bootstrap_metrics[
                component_key
            ][model_key]["auroc"][
                batch_start:batch_end
            ] = batch_auroc

        for fraction in RISK_FRACTIONS:
            enrichment, risk_ratio, selected_event_rate = (
                calculate_weighted_enrichment(
                    bootstrap_counts,
                    outcome,
                    frozen_rank_memberships[fraction],
                )
            )

            bootstrap_enrichment[
                component_key
            ][fraction]["enrichment"][
                batch_start:batch_end
            ] = enrichment

            bootstrap_enrichment[
                component_key
            ][fraction]["risk_ratio"][
                batch_start:batch_end
            ] = risk_ratio

            bootstrap_enrichment[
                component_key
            ][fraction]["selected_event_rate"][
                batch_start:batch_end
            ] = selected_event_rate

    if (
        batch_end % 250 == 0
        or batch_end == N_BOOTSTRAP
    ):
        valid_summary = ", ".join(
            f"{key}="
            f"{int(np.isfinite(bootstrap_metrics[key]['full_ges']['auprc'][:batch_end]).sum()):,}"
            for key in COMPONENT_SPECIFICATIONS
        )
        elapsed = time.time() - analysis_start_time
        print(
            f"Completed {batch_end:,}/{N_BOOTSTRAP:,} "
            f"replicates | valid [{valid_summary}] | "
            f"elapsed {elapsed:.1f}s"
        )

    del bootstrap_counts
    gc.collect()


# --------------------------------------------------------------------------------------------------
# 6. VERIFY PAIRED VALIDITY
# --------------------------------------------------------------------------------------------------

component_validity_rows = []
component_valid_masks = {}

for component_key in COMPONENT_SPECIFICATIONS:
    valid_mask = np.isfinite(
        bootstrap_metrics[
            component_key
        ]["full_ges"]["auprc"]
    )

    valid_replicates = int(valid_mask.sum())
    invalid_replicates = (
        N_BOOTSTRAP - valid_replicates
    )

    if valid_replicates < MINIMUM_VALID_REPLICATES:
        raise AssertionError(
            f"{component_key} produced only "
            f"{valid_replicates:,} valid replicates."
        )

    for model_key in SCORE_SPECIFICATIONS:
        for metric_name in ["auprc", "auroc"]:
            model_valid = np.isfinite(
                bootstrap_metrics[
                    component_key
                ][model_key][metric_name]
            )
            if not np.array_equal(
                model_valid,
                valid_mask,
            ):
                raise AssertionError(
                    f"Paired validity mismatch for "
                    f"{component_key}/{model_key}/{metric_name}."
                )

    component_valid_masks[
        component_key
    ] = valid_mask

    component_validity_rows.append(
        {
            "component_key": component_key,
            "component": COMPONENT_SPECIFICATIONS[
                component_key
            ]["display_name"],
            "attempted_replicates": N_BOOTSTRAP,
            "valid_replicates": valid_replicates,
            "invalid_one_class_replicates": invalid_replicates,
        }
    )

component_bootstrap_validity = pd.DataFrame(
    component_validity_rows
)


# --------------------------------------------------------------------------------------------------
# 7. MODEL-SPECIFIC COMPONENT INTERVALS
# --------------------------------------------------------------------------------------------------

model_interval_rows = []

for component_key, component_specification in COMPONENT_SPECIFICATIONS.items():
    prevalence = (
        component_specification["expected_events"]
        / EXPECTED_ROWS
    )
    valid_mask = component_valid_masks[
        component_key
    ]
    valid_replicates = int(valid_mask.sum())
    invalid_replicates = N_BOOTSTRAP - valid_replicates

    for model_key, model_specification in SCORE_SPECIFICATIONS.items():
        auprc_values = bootstrap_metrics[
            component_key
        ][model_key]["auprc"]

        auroc_values = bootstrap_metrics[
            component_key
        ][model_key]["auroc"]

        auprc_lower, auprc_upper = percentile_interval(
            auprc_values
        )
        auroc_lower, auroc_upper = percentile_interval(
            auroc_values
        )

        auprc_null_lower, auprc_null_upper = percentile_interval(
            auprc_values
            - bootstrap_prevalence[
                component_key
            ]
        )

        auroc_null_lower, auroc_null_upper = percentile_interval(
            auroc_values - 0.50
        )

        model_interval_rows.append(
            {
                "component_key": component_key,
                "component": component_specification["display_name"],
                "model_key": model_key,
                "model": model_specification["display_name"],
                "rows": EXPECTED_ROWS,
                "events": component_specification["expected_events"],
                "negatives": (
                    EXPECTED_ROWS
                    - component_specification["expected_events"]
                ),
                "prevalence": prevalence,
                "point_auprc": point_metrics[
                    component_key
                ][model_key]["auprc"],
                "auprc_ci_lower": auprc_lower,
                "auprc_ci_upper": auprc_upper,
                "point_auprc_minus_prevalence": (
                    point_metrics[
                        component_key
                    ][model_key]["auprc"]
                    - prevalence
                ),
                "auprc_minus_prevalence_ci_lower": auprc_null_lower,
                "auprc_minus_prevalence_ci_upper": auprc_null_upper,
                "auprc_null_status": interval_status(
                    auprc_null_lower,
                    auprc_null_upper,
                    "supported_above_prevalence",
                    "supported_below_prevalence",
                ),
                "point_auroc": point_metrics[
                    component_key
                ][model_key]["auroc"],
                "auroc_ci_lower": auroc_lower,
                "auroc_ci_upper": auroc_upper,
                "point_auroc_minus_0_50": (
                    point_metrics[
                        component_key
                    ][model_key]["auroc"]
                    - 0.50
                ),
                "auroc_minus_0_50_ci_lower": auroc_null_lower,
                "auroc_minus_0_50_ci_upper": auroc_null_upper,
                "auroc_null_status": interval_status(
                    auroc_null_lower,
                    auroc_null_upper,
                    "supported_above_0_50",
                    "supported_below_0_50",
                ),
                "attempted_bootstrap_replicates": N_BOOTSTRAP,
                "valid_bootstrap_replicates": valid_replicates,
                "invalid_one_class_replicates": invalid_replicates,
            }
        )

component_model_intervals = pd.DataFrame(
    model_interval_rows
)


# --------------------------------------------------------------------------------------------------
# 8. PAIRED FULL-GES-MINUS-COMPARATOR INFERENCE
# --------------------------------------------------------------------------------------------------

paired_difference_rows = []

for component_key, component_specification in COMPONENT_SPECIFICATIONS.items():
    valid_mask = component_valid_masks[
        component_key
    ]
    valid_replicates = int(valid_mask.sum())
    invalid_replicates = N_BOOTSTRAP - valid_replicates

    for comparison_family, comparator_keys in [
        ("principal_prespecified", PRINCIPAL_COMPARATORS),
        ("secondary_remaining_comparators", SECONDARY_COMPARATORS),
    ]:
        for comparator_key in comparator_keys:
            comparator_name = SCORE_SPECIFICATIONS[
                comparator_key
            ]["display_name"]

            for metric_name in ["auprc", "auroc"]:
                differences = (
                    bootstrap_metrics[
                        component_key
                    ]["full_ges"][metric_name]
                    - bootstrap_metrics[
                        component_key
                    ][comparator_key][metric_name]
                )

                finite_differences = differences[
                    np.isfinite(differences)
                ]

                lower, upper = percentile_interval(
                    finite_differences
                )

                point_difference = (
                    point_metrics[
                        component_key
                    ]["full_ges"][metric_name]
                    - point_metrics[
                        component_key
                    ][comparator_key][metric_name]
                )

                paired_difference_rows.append(
                    {
                        "component_key": component_key,
                        "component": component_specification["display_name"],
                        "metric": metric_name.upper(),
                        "comparison_family": comparison_family,
                        "comparison": (
                            f"Full GES minus {comparator_name}"
                        ),
                        "comparator_key": comparator_key,
                        "point_difference": point_difference,
                        "difference_ci_lower": lower,
                        "difference_ci_upper": upper,
                        "paired_interval_status": interval_status(
                            lower,
                            upper,
                            "full_ges_supported_higher",
                            "full_ges_supported_lower",
                        ),
                        "bootstrap_probability_full_greater": float(
                            np.mean(
                                finite_differences > 0.0
                            )
                        ),
                        "bootstrap_sign_p_value": bootstrap_sign_pvalue(
                            finite_differences
                        ),
                        "secondary_family_holm_adjusted_p": np.nan,
                        "secondary_family_holm_supported_at_0_05": pd.NA,
                        "attempted_bootstrap_replicates": N_BOOTSTRAP,
                        "valid_bootstrap_replicates": valid_replicates,
                        "invalid_one_class_replicates": invalid_replicates,
                    }
                )

component_paired_differences = pd.DataFrame(
    paired_difference_rows
)

component_paired_differences[
    "secondary_family_holm_supported_at_0_05"
] = pd.Series(
    pd.NA,
    index=component_paired_differences.index,
    dtype="boolean",
)

for component_key in COMPONENT_SPECIFICATIONS:
    for metric_name in ["AUPRC", "AUROC"]:
        family_mask = (
            (
                component_paired_differences[
                    "component_key"
                ] == component_key
            )
            & (
                component_paired_differences[
                    "metric"
                ] == metric_name
            )
            & (
                component_paired_differences[
                    "comparison_family"
                ] == "secondary_remaining_comparators"
            )
        )

        p_values = component_paired_differences.loc[
            family_mask,
            "bootstrap_sign_p_value",
        ].to_numpy(dtype=float)

        if len(p_values) != 5:
            raise AssertionError(
                f"{component_key}/{metric_name} secondary "
                f"family has {len(p_values)} tests; expected 5."
            )

        adjusted = holm_adjust(p_values)

        component_paired_differences.loc[
            family_mask,
            "secondary_family_holm_adjusted_p",
        ] = adjusted

        component_paired_differences.loc[
            family_mask,
            "secondary_family_holm_supported_at_0_05",
        ] = adjusted < 0.05


# --------------------------------------------------------------------------------------------------
# 9. FULL-GES COMPONENT ENRICHMENT INTERVALS
# --------------------------------------------------------------------------------------------------

enrichment_rows = []

for component_key, component_specification in COMPONENT_SPECIFICATIONS.items():
    outcome = component_outcomes[
        component_key
    ]
    prevalence = float(outcome.mean())

    ordered_memberships = frozen_rank_memberships

    for fraction in RISK_FRACTIONS:
        membership = ordered_memberships[fraction]
        selected_rows = int(membership.sum())
        selected_events = int(
            np.sum(outcome * membership)
        )
        selected_rate = (
            selected_events / selected_rows
        )

        remaining_rows = n_rows - selected_rows
        remaining_events = int(
            outcome.sum() - selected_events
        )
        remaining_rate = (
            remaining_events / remaining_rows
        )

        point_enrichment = (
            selected_rate / prevalence
        )
        point_risk_ratio = (
            selected_rate / remaining_rate
            if remaining_rate > 0
            else np.nan
        )

        enrichment_values = bootstrap_enrichment[
            component_key
        ][fraction]["enrichment"]

        risk_ratio_values = bootstrap_enrichment[
            component_key
        ][fraction]["risk_ratio"]

        selected_rate_values = bootstrap_enrichment[
            component_key
        ][fraction]["selected_event_rate"]

        enrichment_lower, enrichment_upper = percentile_interval(
            enrichment_values
        )
        risk_ratio_lower, risk_ratio_upper = percentile_interval(
            risk_ratio_values
        )
        selected_rate_lower, selected_rate_upper = percentile_interval(
            selected_rate_values
        )

        enrichment_rows.append(
            {
                "component_key": component_key,
                "component": component_specification["display_name"],
                "risk_fraction": fraction,
                "selected_rows": selected_rows,
                "selected_events": selected_events,
                "selected_event_rate": selected_rate,
                "selected_event_rate_ci_lower": selected_rate_lower,
                "selected_event_rate_ci_upper": selected_rate_upper,
                "remaining_rows": remaining_rows,
                "remaining_events": remaining_events,
                "remaining_event_rate": remaining_rate,
                "component_prevalence": prevalence,
                "point_risk_ratio_vs_remaining": point_risk_ratio,
                "risk_ratio_ci_lower": risk_ratio_lower,
                "risk_ratio_ci_upper": risk_ratio_upper,
                "point_enrichment_over_prevalence": point_enrichment,
                "enrichment_ci_lower": enrichment_lower,
                "enrichment_ci_upper": enrichment_upper,
                "enrichment_interval_status": interval_status(
                    enrichment_lower - 1.0,
                    enrichment_upper - 1.0,
                    "supported_above_1",
                    "supported_below_1",
                ),
                "attempted_bootstrap_replicates": N_BOOTSTRAP,
                "valid_enrichment_replicates": int(
                    np.isfinite(
                        enrichment_values
                    ).sum()
                ),
            }
        )

component_enrichment_intervals = pd.DataFrame(
    enrichment_rows
)


# --------------------------------------------------------------------------------------------------
# 10. COMPLETENESS CHECKS
# --------------------------------------------------------------------------------------------------

if len(component_model_intervals) != 27:
    raise AssertionError(
        "Expected 27 component-model interval rows."
    )

if len(component_paired_differences) != 48:
    raise AssertionError(
        "Expected 48 paired rows: "
        "3 components × 8 comparators × 2 metrics."
    )

if len(component_enrichment_intervals) != 9:
    raise AssertionError(
        "Expected 9 component-enrichment rows."
    )

secondary_rows = (
    component_paired_differences[
        "comparison_family"
    ] == "secondary_remaining_comparators"
)

principal_rows = (
    component_paired_differences[
        "comparison_family"
    ] == "principal_prespecified"
)

if component_paired_differences.loc[
    secondary_rows,
    "secondary_family_holm_adjusted_p",
].isna().any():
    raise AssertionError(
        "A secondary comparison is missing a Holm-adjusted value."
    )

if component_paired_differences.loc[
    principal_rows,
    "secondary_family_holm_adjusted_p",
].notna().any():
    raise AssertionError(
        "A principal comparison was incorrectly included "
        "in a secondary Holm family."
    )


# --------------------------------------------------------------------------------------------------
# 11. DISPLAY RESULTS AND FINAL DECISION
# --------------------------------------------------------------------------------------------------

model_display_columns = [
    "component",
    "model",
    "events",
    "prevalence",
    "point_auprc",
    "auprc_ci_lower",
    "auprc_ci_upper",
    "point_auprc_minus_prevalence",
    "auprc_minus_prevalence_ci_lower",
    "auprc_minus_prevalence_ci_upper",
    "auprc_null_status",
    "point_auroc",
    "auroc_ci_lower",
    "auroc_ci_upper",
    "point_auroc_minus_0_50",
    "auroc_minus_0_50_ci_lower",
    "auroc_minus_0_50_ci_upper",
    "auroc_null_status",
    "valid_bootstrap_replicates",
    "invalid_one_class_replicates",
]

paired_display_columns = [
    "component",
    "metric",
    "comparison_family",
    "comparison",
    "point_difference",
    "difference_ci_lower",
    "difference_ci_upper",
    "paired_interval_status",
    "bootstrap_probability_full_greater",
    "bootstrap_sign_p_value",
    "secondary_family_holm_adjusted_p",
    "secondary_family_holm_supported_at_0_05",
    "valid_bootstrap_replicates",
    "invalid_one_class_replicates",
]

enrichment_display_columns = [
    "component",
    "risk_fraction",
    "selected_rows",
    "selected_events",
    "selected_event_rate",
    "selected_event_rate_ci_lower",
    "selected_event_rate_ci_upper",
    "remaining_event_rate",
    "component_prevalence",
    "point_risk_ratio_vs_remaining",
    "risk_ratio_ci_lower",
    "risk_ratio_ci_upper",
    "point_enrichment_over_prevalence",
    "enrichment_ci_lower",
    "enrichment_ci_upper",
    "enrichment_interval_status",
    "valid_enrichment_replicates",
]

analysis_elapsed = time.time() - analysis_start_time
separator = "=" * 150

print("\n" + separator)
print(
    "STAGE 6C STEP 3E — CELL 6C-3E1 — "
    "PRIMARY EVENT-COMPONENT PAIRED BOOTSTRAP INFERENCE"
)
print(separator)
print(
    f"Frozen cohort SHA-256              : "
    f"PASS ({observed_sha256})"
)
print(
    f"Frozen cohort dimensions           : "
    f"PASS ({metadata.num_rows:,} × {metadata.num_columns})"
)
print(
    f"Primary outcome reconstruction     : "
    f"PASS ({EXPECTED_PRIMARY_EVENTS:,} events)"
)
print(
    f"Bootstrap attempts                 : "
    f"{N_BOOTSTRAP:,}"
)
print(
    f"Random seed                        : "
    f"{RANDOM_SEED}"
)
print(
    "Identical resamples                : "
    "Yes, across all components and scores"
)
print(
    "Secondary Holm families            : "
    "3 components × 2 metrics × 5 comparisons"
)
print(
    f"Elapsed time                       : "
    f"{analysis_elapsed:.1f}s"
)
print(
    "Frozen artifacts modified          : No"
)

print("\nCOMPONENT BOOTSTRAP VALIDITY")
print(
    component_bootstrap_validity.to_string(
        index=False
    )
)

print("\nCOMPONENT-SPECIFIC MODEL INTERVALS")
print(
    component_model_intervals[
        model_display_columns
    ].to_string(index=False)
)

print(
    "\nCOMPONENT-SPECIFIC PAIRED "
    "FULL-GES-MINUS-COMPARATOR INFERENCE"
)
print(
    component_paired_differences[
        paired_display_columns
    ].to_string(index=False)
)

print(
    "\nCOMPONENT-SPECIFIC FULL-GES "
    "5%/10%/20% ENRICHMENT INTERVALS"
)
print(
    component_enrichment_intervals[
        enrichment_display_columns
    ].to_string(index=False)
)

print("\nCELL DECISION")
print("-" * 150)
print(
    "PASS_STAGE6C_PRIMARY_EVENT_COMPONENT_"
    "PAIRED_BOOTSTRAP_INFERENCE_COMPLETE"
)
print(
    "Component-specific model intervals, paired "
    "Full-GES-minus-comparator inference, secondary-family "
    "Holm correction, and frozen-rank enrichment intervals "
    "are complete."
)
print(
    "No score, outcome, component flag, linkage decision, "
    "threshold, weight, cohort membership, or frozen scientific "
    "artifact was modified."
)

Preparing component bootstrap: 66,636 rows; Material clinical-group change=1,405 events, New unresolved conflict=4,789 events, Material prior-conflict resolution=297 events
Fast metric validation against scikit-learn: PASS for all 27 component-score combinations
Completed 250/2,000 replicates | valid [material_group_change=250, new_unresolved_conflict=250, material_prior_conflict_resolution=250] | elapsed 12.3s
Completed 500/2,000 replicates | valid [material_group_change=500, new_unresolved_conflict=500, material_prior_conflict_resolution=500] | elapsed 24.8s
Completed 750/2,000 replicates | valid [material_group_change=750, new_unresolved_conflict=750, material_prior_conflict_resolution=750] | elapsed 37.3s
Completed 1,000/2,000 replicates | valid [material_group_change=1,000, new_unresolved_conflict=1,000, material_prior_conflict_resolution=1,000] | elapsed 48.7s
Completed 1,250/2,000 replicates | valid [material_group_change=1,250, new_unresolved_conflict=1,250, material_prior_conf

In [38]:
# ==================================================================================================
# STAGE 6C STEP 3F — CELL 6C-3F0
# ALTERNATIVE-OUTCOME DEFINITION FREEZE, SECONDARY-DRIFT INVENTORY, AND LOCKED POINT ESTIMATES
#
# Prespecified alternative instability definitions:
#   A. Strict material instability
#        material clinical-group change OR material prior-conflict resolution
#        (new unresolved conflict excluded)
#   B. Conflict-transition instability
#        new unresolved conflict OR material prior-conflict resolution
#   C. Expanded instability/evidence drift
#        primary future instability OR any review-star change
#
# Prespecified secondary evidence-drift outcomes:
#   D. Any review-star change
#   E. Review-star increase
#   F. Review-star decrease
#   G. Any review-status change
#   H. New expert-panel involvement
#        T0 review stars < 3 AND T1 review stars >= 3
#
# Important interpretation boundary:
#   Review status/stars contribute to the original full GES pathway. Therefore, review-drift and
#   expert-panel outcomes are exploratory secondary evidence-drift outcomes, not independent
#   validation endpoints.
#
# This cell:
#   1. Freshly verifies the frozen 66,636-row Stage 6B primary-evaluable cohort.
#   2. Verifies the frozen primary components and review-drift fields.
#   3. Constructs the eight definitions above without examining model performance first.
#   4. Calculates locked AUPRC/AUROC point estimates for all nine frozen scores.
#   5. Calculates Full-GES-minus-principal-comparator point differences.
#   6. Calculates Full-GES 5%/10%/20% exact-rank enrichment point estimates.
#   7. Inventories classification direction, conflict transition, review transition, expert-panel
#      transition, and selected-group transition fields.
#
# This cell is read-only and writes no scientific artifact.
# ==================================================================================================

from pathlib import Path
import hashlib
import re

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from sklearn.metrics import average_precision_score, roc_auc_score


# --------------------------------------------------------------------------------------------------
# 1. FROZEN INPUT AND ANALYSIS SPECIFICATION
# --------------------------------------------------------------------------------------------------

STAGE6_DIR = Path(
    "/content/drive/MyDrive/GES_RAG_Temporal_Study/"
    "data_processed/stage6_temporal_validation"
)

EVALUABLE_PARQUET = (
    STAGE6_DIR / "stage6b_locked_primary_evaluable_cohort_v1.parquet"
)
EVALUABLE_SIDECAR = Path(str(EVALUABLE_PARQUET) + ".sha256")

EXPECTED_EVALUABLE_SHA256 = (
    "c6ad50dfc376a746e1830d000dba975101d88763fd61b8d1983f6300edb6e038"
)

EXPECTED_ROWS = 66_636
EXPECTED_COLUMNS = 79
EXPECTED_PRIMARY_EVENTS = 6_485
EXPECTED_PRIMARY_NEGATIVES = 60_151

RCV_COLUMN = "rcv_accession"
ROW_ORDER_COLUMN = "t0_row_order"
PRIMARY_OUTCOME_COLUMN = "primary_future_instability"

T0_REVIEW_STARS_COLUMN = "t0_aggregate_review_stars"
T1_REVIEW_STARS_COLUMN = "t1_aggregate_review_stars"
REVIEW_STAR_DELTA_COLUMN = "secondary_review_star_delta"
REVIEW_STAR_TRANSITION_COLUMN = "secondary_review_star_transition"
REVIEW_STATUS_CHANGED_COLUMN = "secondary_review_status_changed"
EXPERT_PANEL_TRANSITION_COLUMN = "secondary_expert_panel_transition"

MATERIAL_GROUP_CHANGE_COLUMN = "event_material_clinical_group_change"
NEW_CONFLICT_COLUMN = "event_new_unresolved_conflict_at_t1"
PRIOR_RESOLUTION_COLUMN = "event_prior_conflict_resolved_to_material_group"

DIRECTION_AND_TRANSITION_COLUMNS = [
    "material_group_change_direction",
    "prior_conflict_resolution_final_group",
    "secondary_conflict_transition",
    "secondary_review_star_transition",
    "secondary_expert_panel_transition",
    "secondary_selected_group_transition",
    "secondary_scv_disagreement_changed",
]

RISK_FRACTIONS = [0.05, 0.10, 0.20]

SCORE_SPECIFICATIONS = {
    "full_ges": {
        "column": "full_ges_instability_risk_t0",
        "display_name": "Full GES",
    },
    "no_star_ges": {
        "column": "no_star_ges_instability_risk_t0",
        "display_name": "No-star GES",
    },
    "review_stars": {
        "column": "review_stars_instability_risk",
        "display_name": "Review stars",
    },
    "combined_metadata": {
        "column": "combined_metadata_instability_risk",
        "display_name": "Combined metadata",
    },
    "conflict": {
        "column": "conflict_instability_risk",
        "display_name": "Conflict",
    },
    "recency": {
        "column": "recency_instability_risk",
        "display_name": "Recency",
    },
    "submitter": {
        "column": "submitter_instability_risk",
        "display_name": "Submitter support",
    },
    "entropy": {
        "column": "entropy_instability_risk",
        "display_name": "Classification entropy",
    },
    "additive": {
        "column": "additive_instability_risk",
        "display_name": "Additive risk",
    },
}

PRINCIPAL_COMPARATORS = [
    "no_star_ges",
    "review_stars",
    "combined_metadata",
]

OUTCOME_SPECIFICATIONS = {
    "strict_material_instability": {
        "display_name": "Strict material instability",
        "role": "alternative_primary_sensitivity",
        "definition": (
            "material clinical-group change OR material prior-conflict resolution; "
            "new unresolved conflict excluded"
        ),
        "independence_note": (
            "Alternative clinical-instability definition derived from frozen primary components."
        ),
    },
    "conflict_transition_instability": {
        "display_name": "Conflict-transition instability",
        "role": "alternative_primary_sensitivity",
        "definition": (
            "new unresolved conflict OR material prior-conflict resolution"
        ),
        "independence_note": (
            "Alternative conflict-dynamics definition derived from frozen primary components."
        ),
    },
    "expanded_primary_or_review_star_change": {
        "display_name": "Expanded primary-or-star-drift outcome",
        "role": "alternative_primary_sensitivity",
        "definition": (
            "primary future instability OR any review-star change"
        ),
        "independence_note": (
            "Expanded evidence-drift definition; not independent of review metadata."
        ),
    },
    "any_review_star_change": {
        "display_name": "Any review-star change",
        "role": "secondary_evidence_drift",
        "definition": "T1 review stars differ from T0 review stars",
        "independence_note": (
            "Exploratory only because review stars contribute to the full GES pathway."
        ),
    },
    "review_star_increase": {
        "display_name": "Review-star increase",
        "role": "secondary_evidence_drift",
        "definition": "T1 review stars greater than T0 review stars",
        "independence_note": (
            "Exploratory only because review stars contribute to the full GES pathway."
        ),
    },
    "review_star_decrease": {
        "display_name": "Review-star decrease",
        "role": "secondary_evidence_drift",
        "definition": "T1 review stars lower than T0 review stars",
        "independence_note": (
            "Exploratory only because review stars contribute to the full GES pathway."
        ),
    },
    "any_review_status_change": {
        "display_name": "Any review-status change",
        "role": "secondary_evidence_drift",
        "definition": "Frozen secondary_review_status_changed equals True",
        "independence_note": (
            "Exploratory only because review confidence contributes to full GES."
        ),
    },
    "new_expert_panel_involvement": {
        "display_name": "New expert-panel involvement",
        "role": "secondary_evidence_drift",
        "definition": "T0 review stars < 3 and T1 review stars >= 3",
        "independence_note": (
            "Exploratory only; expert-panel status is not an independent validation endpoint."
        ),
    },
}


# --------------------------------------------------------------------------------------------------
# 2. HELPERS
# --------------------------------------------------------------------------------------------------

def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def read_sidecar_hash(path: Path) -> str:
    text = path.read_text(encoding="utf-8").strip()
    matches = re.findall(r"\b[a-fA-F0-9]{64}\b", text)
    if not matches:
        raise ValueError(f"No SHA-256 value found in sidecar: {path}")
    return matches[0].lower()


def calculate_outcome_metrics(
    frame: pd.DataFrame,
    outcome_key: str,
    outcome: np.ndarray,
) -> pd.DataFrame:
    outcome = np.asarray(outcome, dtype=np.int8)
    events = int(outcome.sum())
    negatives = int(len(outcome) - events)
    prevalence = float(outcome.mean())

    if events == 0 or negatives == 0:
        raise AssertionError(
            f"{outcome_key} does not contain both outcome classes."
        )

    rows = []

    for model_key, model_specification in SCORE_SPECIFICATIONS.items():
        scores = frame[model_specification["column"]].to_numpy(dtype=float)

        auprc = float(average_precision_score(outcome, scores))
        auroc = float(roc_auc_score(outcome, scores))

        rows.append(
            {
                "outcome_key": outcome_key,
                "outcome": OUTCOME_SPECIFICATIONS[outcome_key]["display_name"],
                "outcome_role": OUTCOME_SPECIFICATIONS[outcome_key]["role"],
                "definition": OUTCOME_SPECIFICATIONS[outcome_key]["definition"],
                "independence_note": OUTCOME_SPECIFICATIONS[outcome_key][
                    "independence_note"
                ],
                "model_key": model_key,
                "model": model_specification["display_name"],
                "rows": int(len(outcome)),
                "events": events,
                "negatives": negatives,
                "prevalence": prevalence,
                "point_auprc": auprc,
                "auprc_minus_prevalence": auprc - prevalence,
                "auprc_lift_over_prevalence": (
                    auprc / prevalence if prevalence > 0 else np.nan
                ),
                "point_auroc": auroc,
                "auroc_minus_0_50": auroc - 0.50,
            }
        )

    return pd.DataFrame(rows)


def calculate_principal_differences(
    metric_table: pd.DataFrame,
) -> pd.DataFrame:
    rows = []

    for outcome_key in OUTCOME_SPECIFICATIONS:
        subset = metric_table.loc[
            metric_table["outcome_key"] == outcome_key
        ].set_index("model_key")

        for comparator_key in PRINCIPAL_COMPARATORS:
            comparator_name = SCORE_SPECIFICATIONS[
                comparator_key
            ]["display_name"]

            for metric_column, metric_name in [
                ("point_auprc", "AUPRC"),
                ("point_auroc", "AUROC"),
            ]:
                rows.append(
                    {
                        "outcome_key": outcome_key,
                        "outcome": OUTCOME_SPECIFICATIONS[
                            outcome_key
                        ]["display_name"],
                        "outcome_role": OUTCOME_SPECIFICATIONS[
                            outcome_key
                        ]["role"],
                        "metric": metric_name,
                        "comparison": f"Full GES minus {comparator_name}",
                        "comparator_key": comparator_key,
                        "point_difference": float(
                            subset.loc["full_ges", metric_column]
                            - subset.loc[comparator_key, metric_column]
                        ),
                    }
                )

    return pd.DataFrame(rows)


def calculate_exact_rank_enrichment(
    frame: pd.DataFrame,
    outcome_masks: dict[str, np.ndarray],
) -> pd.DataFrame:
    full_ges_column = SCORE_SPECIFICATIONS["full_ges"]["column"]

    ordered_positions = (
        frame[
            [RCV_COLUMN, ROW_ORDER_COLUMN, full_ges_column]
        ]
        .copy()
        .assign(_position=np.arange(len(frame)))
        .sort_values(
            [full_ges_column, ROW_ORDER_COLUMN, RCV_COLUMN],
            ascending=[False, True, True],
            kind="mergesort",
        )["_position"]
        .to_numpy(dtype=int)
    )

    ordered_risk = frame.iloc[
        ordered_positions
    ][full_ges_column].to_numpy(dtype=float)

    rows = []
    n_rows = len(frame)

    for outcome_key, mask in outcome_masks.items():
        ordered_outcome = np.asarray(
            mask,
            dtype=np.int8,
        )[ordered_positions]

        prevalence = float(ordered_outcome.mean())

        for fraction in RISK_FRACTIONS:
            selected_rows = int(np.ceil(n_rows * fraction))
            selected = ordered_outcome[:selected_rows]
            remaining = ordered_outcome[selected_rows:]

            selected_rate = float(selected.mean())
            remaining_rate = float(remaining.mean())
            enrichment = (
                selected_rate / prevalence
                if prevalence > 0
                else np.nan
            )
            risk_ratio = (
                selected_rate / remaining_rate
                if remaining_rate > 0
                else np.nan
            )

            cutoff_score = float(
                ordered_risk[selected_rows - 1]
            )
            tie_size = int(
                np.count_nonzero(
                    ordered_risk == cutoff_score
                )
            )
            tie_selected = int(
                np.count_nonzero(
                    ordered_risk[:selected_rows] == cutoff_score
                )
            )

            rows.append(
                {
                    "outcome_key": outcome_key,
                    "outcome": OUTCOME_SPECIFICATIONS[
                        outcome_key
                    ]["display_name"],
                    "outcome_role": OUTCOME_SPECIFICATIONS[
                        outcome_key
                    ]["role"],
                    "risk_fraction": fraction,
                    "selected_rows": selected_rows,
                    "selected_events": int(selected.sum()),
                    "selected_event_rate": selected_rate,
                    "remaining_rows": int(len(remaining)),
                    "remaining_events": int(remaining.sum()),
                    "remaining_event_rate": remaining_rate,
                    "outcome_prevalence": prevalence,
                    "risk_ratio_vs_remaining": risk_ratio,
                    "enrichment_over_prevalence": enrichment,
                    "cutoff_instability_risk": cutoff_score,
                    "boundary_tie_size": tie_size,
                    "boundary_tie_selected": tie_selected,
                }
            )

    return pd.DataFrame(rows)


def inventory_transition_columns(
    frame: pd.DataFrame,
) -> pd.DataFrame:
    tables = []

    for column in DIRECTION_AND_TRANSITION_COLUMNS:
        value_counts = (
            frame[column]
            .astype("string")
            .value_counts(dropna=False)
            .rename_axis("value")
            .reset_index(name="rows")
        )
        value_counts.insert(0, "field", column)
        tables.append(value_counts)

    return pd.concat(
        tables,
        ignore_index=True,
    )


# --------------------------------------------------------------------------------------------------
# 3. FRESH CRYPTOGRAPHIC AND STRUCTURAL VERIFICATION
# --------------------------------------------------------------------------------------------------

if not EVALUABLE_PARQUET.exists():
    raise FileNotFoundError(
        f"Missing frozen evaluable cohort:\n{EVALUABLE_PARQUET}"
    )

if not EVALUABLE_SIDECAR.exists():
    raise FileNotFoundError(
        f"Missing SHA-256 sidecar:\n{EVALUABLE_SIDECAR}"
    )

observed_sha256 = sha256_file(EVALUABLE_PARQUET)
sidecar_sha256 = read_sidecar_hash(EVALUABLE_SIDECAR)

if observed_sha256 != EXPECTED_EVALUABLE_SHA256:
    raise AssertionError(
        "Frozen Stage 6B evaluable-cohort SHA-256 mismatch.\n"
        f"Expected: {EXPECTED_EVALUABLE_SHA256}\n"
        f"Observed: {observed_sha256}"
    )

if sidecar_sha256 != observed_sha256:
    raise AssertionError(
        "Frozen evaluable-cohort sidecar mismatch.\n"
        f"Calculated: {observed_sha256}\n"
        f"Sidecar:    {sidecar_sha256}"
    )

parquet_file = pq.ParquetFile(EVALUABLE_PARQUET)
metadata = parquet_file.metadata
schema_columns = parquet_file.schema_arrow.names

if metadata.num_rows != EXPECTED_ROWS:
    raise AssertionError(
        f"Unexpected row count: {metadata.num_rows:,}; expected {EXPECTED_ROWS:,}"
    )

if metadata.num_columns != EXPECTED_COLUMNS:
    raise AssertionError(
        f"Unexpected column count: {metadata.num_columns}; expected {EXPECTED_COLUMNS}"
    )

required_columns = [
    RCV_COLUMN,
    ROW_ORDER_COLUMN,
    PRIMARY_OUTCOME_COLUMN,
    T0_REVIEW_STARS_COLUMN,
    T1_REVIEW_STARS_COLUMN,
    REVIEW_STAR_DELTA_COLUMN,
    REVIEW_STAR_TRANSITION_COLUMN,
    REVIEW_STATUS_CHANGED_COLUMN,
    EXPERT_PANEL_TRANSITION_COLUMN,
    MATERIAL_GROUP_CHANGE_COLUMN,
    NEW_CONFLICT_COLUMN,
    PRIOR_RESOLUTION_COLUMN,
] + DIRECTION_AND_TRANSITION_COLUMNS + [
    specification["column"]
    for specification in SCORE_SPECIFICATIONS.values()
]

required_columns = list(dict.fromkeys(required_columns))

missing_columns = [
    column
    for column in required_columns
    if column not in schema_columns
]

if missing_columns:
    raise KeyError(
        "Missing required frozen columns:\n"
        + "\n".join(missing_columns)
    )

cohort = pd.read_parquet(
    EVALUABLE_PARQUET,
    columns=required_columns,
).copy()

if len(cohort) != EXPECTED_ROWS:
    raise AssertionError("Loaded dataframe row count mismatch.")

if cohort[RCV_COLUMN].isna().any():
    raise AssertionError("Missing RCV accession detected.")

if cohort[RCV_COLUMN].nunique(dropna=False) != EXPECTED_ROWS:
    raise AssertionError("RCV accessions are not unique.")

row_order = pd.to_numeric(
    cohort[ROW_ORDER_COLUMN],
    errors="raise",
).to_numpy()

if len(np.unique(row_order)) != EXPECTED_ROWS:
    raise AssertionError("t0_row_order is not unique.")

if not np.all(np.diff(row_order) > 0):
    raise AssertionError(
        "Frozen cohort is not in increasing T0 row order."
    )

for column in [
    PRIMARY_OUTCOME_COLUMN,
    MATERIAL_GROUP_CHANGE_COLUMN,
    NEW_CONFLICT_COLUMN,
    PRIOR_RESOLUTION_COLUMN,
    REVIEW_STATUS_CHANGED_COLUMN,
]:
    cohort[column] = pd.to_numeric(
        cohort[column],
        errors="raise",
    ).astype(int)

    if not set(cohort[column].unique()).issubset({0, 1}):
        raise AssertionError(f"{column} is not binary.")

primary_outcome = cohort[
    PRIMARY_OUTCOME_COLUMN
].to_numpy(dtype=np.int8)

if int(primary_outcome.sum()) != EXPECTED_PRIMARY_EVENTS:
    raise AssertionError("Primary event count mismatch.")

if int((primary_outcome == 0).sum()) != EXPECTED_PRIMARY_NEGATIVES:
    raise AssertionError("Primary negative count mismatch.")

for column in [
    T0_REVIEW_STARS_COLUMN,
    T1_REVIEW_STARS_COLUMN,
    REVIEW_STAR_DELTA_COLUMN,
]:
    cohort[column] = pd.to_numeric(
        cohort[column],
        errors="raise",
    ).astype(int)

t0_stars = cohort[
    T0_REVIEW_STARS_COLUMN
].to_numpy(dtype=int)

t1_stars = cohort[
    T1_REVIEW_STARS_COLUMN
].to_numpy(dtype=int)

frozen_star_delta = cohort[
    REVIEW_STAR_DELTA_COLUMN
].to_numpy(dtype=int)

calculated_star_delta = t1_stars - t0_stars

if not np.array_equal(
    frozen_star_delta,
    calculated_star_delta,
):
    mismatch_count = int(
        np.count_nonzero(
            frozen_star_delta != calculated_star_delta
        )
    )
    raise AssertionError(
        f"Frozen review-star delta differs from T1-T0 for "
        f"{mismatch_count:,} rows."
    )

if ((t0_stars < 0) | (t0_stars > 4)).any():
    raise AssertionError("T0 review stars outside 0-4.")

if ((t1_stars < 0) | (t1_stars > 4)).any():
    raise AssertionError("T1 review stars outside 0-4.")

for specification in SCORE_SPECIFICATIONS.values():
    column = specification["column"]
    cohort[column] = pd.to_numeric(
        cohort[column],
        errors="raise",
    )
    values = cohort[column].to_numpy(dtype=float)

    if not np.isfinite(values).all():
        raise AssertionError(
            f"Missing/nonfinite score in {column}."
        )
    if ((values < 0.0) | (values > 1.0)).any():
        raise AssertionError(
            f"Score outside [0,1] in {column}."
        )


# --------------------------------------------------------------------------------------------------
# 4. REVERIFY PRIMARY COMPONENT RECONSTRUCTION
# --------------------------------------------------------------------------------------------------

material_group_change = cohort[
    MATERIAL_GROUP_CHANGE_COLUMN
].to_numpy(dtype=bool)

new_conflict = cohort[
    NEW_CONFLICT_COLUMN
].to_numpy(dtype=bool)

prior_resolution = cohort[
    PRIOR_RESOLUTION_COLUMN
].to_numpy(dtype=bool)

reconstructed_primary = (
    material_group_change
    | new_conflict
    | prior_resolution
).astype(np.int8)

if not np.array_equal(
    reconstructed_primary,
    primary_outcome,
):
    mismatch_count = int(
        np.count_nonzero(
            reconstructed_primary != primary_outcome
        )
    )
    raise AssertionError(
        f"Primary-component reconstruction mismatch for "
        f"{mismatch_count:,} rows."
    )


# --------------------------------------------------------------------------------------------------
# 5. CONSTRUCT THE PRESPECIFIED ALTERNATIVE AND SECONDARY OUTCOMES
# --------------------------------------------------------------------------------------------------

any_star_change = calculated_star_delta != 0
star_increase = calculated_star_delta > 0
star_decrease = calculated_star_delta < 0

review_status_change = cohort[
    REVIEW_STATUS_CHANGED_COLUMN
].to_numpy(dtype=bool)

new_expert_panel = (
    (t0_stars < 3)
    & (t1_stars >= 3)
)

outcome_masks = {
    "strict_material_instability": (
        material_group_change
        | prior_resolution
    ),
    "conflict_transition_instability": (
        new_conflict
        | prior_resolution
    ),
    "expanded_primary_or_review_star_change": (
        primary_outcome.astype(bool)
        | any_star_change
    ),
    "any_review_star_change": any_star_change,
    "review_star_increase": star_increase,
    "review_star_decrease": star_decrease,
    "any_review_status_change": review_status_change,
    "new_expert_panel_involvement": new_expert_panel,
}

for outcome_key, mask in outcome_masks.items():
    mask = np.asarray(mask, dtype=bool)
    if len(mask) != EXPECTED_ROWS:
        raise AssertionError(
            f"{outcome_key} row count mismatch."
        )
    if mask.sum() == 0:
        raise AssertionError(
            f"{outcome_key} has zero events."
        )
    if mask.sum() == EXPECTED_ROWS:
        raise AssertionError(
            f"{outcome_key} has no negatives."
        )
    outcome_masks[outcome_key] = mask

if np.any(
    star_increase & star_decrease
):
    raise AssertionError(
        "A row was simultaneously classified as star increase and decrease."
    )

if not np.array_equal(
    any_star_change,
    star_increase | star_decrease,
):
    raise AssertionError(
        "Any-star-change does not equal increase OR decrease."
    )

# Cross-check the frozen expert-panel transition field descriptively.
expert_panel_transition_inventory = (
    cohort[EXPERT_PANEL_TRANSITION_COLUMN]
    .astype("string")
    .value_counts(dropna=False)
    .rename_axis("secondary_expert_panel_transition")
    .reset_index(name="rows")
)

new_expert_panel_transition_counts = (
    cohort.loc[
        new_expert_panel,
        EXPERT_PANEL_TRANSITION_COLUMN,
    ]
    .astype("string")
    .value_counts(dropna=False)
    .rename_axis("transition_value")
    .reset_index(name="new_expert_panel_rows")
)


# --------------------------------------------------------------------------------------------------
# 6. LOCKED POINT ESTIMATES
# --------------------------------------------------------------------------------------------------

metric_tables = []

for outcome_key, mask in outcome_masks.items():
    metric_tables.append(
        calculate_outcome_metrics(
            cohort,
            outcome_key,
            mask.astype(np.int8),
        )
    )

alternative_outcome_point_estimates = pd.concat(
    metric_tables,
    ignore_index=True,
)

alternative_outcome_principal_differences = (
    calculate_principal_differences(
        alternative_outcome_point_estimates
    )
)

alternative_outcome_enrichment = (
    calculate_exact_rank_enrichment(
        cohort,
        outcome_masks,
    )
)


# --------------------------------------------------------------------------------------------------
# 7. ACCOUNTING AND TRANSITION INVENTORIES
# --------------------------------------------------------------------------------------------------

outcome_accounting = pd.DataFrame(
    [
        {
            "outcome_key": outcome_key,
            "outcome": OUTCOME_SPECIFICATIONS[
                outcome_key
            ]["display_name"],
            "outcome_role": OUTCOME_SPECIFICATIONS[
                outcome_key
            ]["role"],
            "definition": OUTCOME_SPECIFICATIONS[
                outcome_key
            ]["definition"],
            "events": int(mask.sum()),
            "negatives": int(
                EXPECTED_ROWS - mask.sum()
            ),
            "prevalence": float(mask.mean()),
            "overlap_with_primary_events": int(
                np.count_nonzero(
                    mask
                    & primary_outcome.astype(bool)
                )
            ),
            "events_outside_primary": int(
                np.count_nonzero(
                    mask
                    & ~primary_outcome.astype(bool)
                )
            ),
            "primary_events_not_in_outcome": int(
                np.count_nonzero(
                    primary_outcome.astype(bool)
                    & ~mask
                )
            ),
            "independence_note": OUTCOME_SPECIFICATIONS[
                outcome_key
            ]["independence_note"],
        }
        for outcome_key, mask in outcome_masks.items()
    ]
)

review_star_delta_inventory = (
    pd.Series(
        calculated_star_delta,
        name="secondary_review_star_delta",
    )
    .value_counts()
    .sort_index()
    .rename_axis("star_delta")
    .reset_index(name="rows")
)

review_star_cross_tab = pd.crosstab(
    pd.Series(
        t0_stars,
        name="t0_review_stars",
    ),
    pd.Series(
        t1_stars,
        name="t1_review_stars",
    ),
    dropna=False,
)

transition_inventory = inventory_transition_columns(
    cohort
)


# --------------------------------------------------------------------------------------------------
# 8. COMPLETENESS CHECKS
# --------------------------------------------------------------------------------------------------

expected_metric_rows = (
    len(OUTCOME_SPECIFICATIONS)
    * len(SCORE_SPECIFICATIONS)
)

expected_difference_rows = (
    len(OUTCOME_SPECIFICATIONS)
    * len(PRINCIPAL_COMPARATORS)
    * 2
)

expected_enrichment_rows = (
    len(OUTCOME_SPECIFICATIONS)
    * len(RISK_FRACTIONS)
)

if len(alternative_outcome_point_estimates) != expected_metric_rows:
    raise AssertionError(
        f"Point-estimate rows="
        f"{len(alternative_outcome_point_estimates)}; "
        f"expected {expected_metric_rows}."
    )

if (
    len(alternative_outcome_principal_differences)
    != expected_difference_rows
):
    raise AssertionError(
        f"Principal-difference rows="
        f"{len(alternative_outcome_principal_differences)}; "
        f"expected {expected_difference_rows}."
    )

if len(alternative_outcome_enrichment) != expected_enrichment_rows:
    raise AssertionError(
        f"Enrichment rows="
        f"{len(alternative_outcome_enrichment)}; "
        f"expected {expected_enrichment_rows}."
    )

if not np.isfinite(
    alternative_outcome_point_estimates[
        [
            "point_auprc",
            "point_auroc",
            "auprc_minus_prevalence",
            "auroc_minus_0_50",
        ]
    ].to_numpy(dtype=float)
).all():
    raise AssertionError(
        "A nonfinite alternative-outcome metric was produced."
    )


# --------------------------------------------------------------------------------------------------
# 9. DISPLAY RESULTS AND FINAL DECISION
# --------------------------------------------------------------------------------------------------

separator = "=" * 150

print("\n" + separator)
print(
    "STAGE 6C STEP 3F — CELL 6C-3F0 — "
    "ALTERNATIVE-OUTCOME AND SECONDARY-DRIFT POINT ESTIMATES"
)
print(separator)
print(
    f"Frozen cohort SHA-256              : "
    f"PASS ({observed_sha256})"
)
print(
    f"Frozen cohort dimensions           : "
    f"PASS ({metadata.num_rows:,} × "
    f"{metadata.num_columns})"
)
print(
    f"Primary outcome reconstruction     : "
    f"PASS ({EXPECTED_PRIMARY_EVENTS:,} events)"
)
print(
    "Review-star delta reconstruction   : "
    "PASS (secondary_review_star_delta = T1 − T0)"
)
print(
    "Outcome definitions selected after performance: No"
)
print(
    "Review-drift outcomes independent validation: No — exploratory only"
)
print(
    "Frozen artifacts modified          : No"
)
print(
    "Scientific artifacts written       : No"
)

print("\nALTERNATIVE/SECONDARY OUTCOME ACCOUNTING")
print(
    outcome_accounting.to_string(
        index=False
    )
)

print("\nREVIEW-STAR DELTA INVENTORY")
print(
    review_star_delta_inventory.to_string(
        index=False
    )
)

print("\nT0 × T1 REVIEW-STAR CROSS-TAB")
print(review_star_cross_tab.to_string())

print("\nFROZEN EXPERT-PANEL TRANSITION INVENTORY")
print(
    expert_panel_transition_inventory.to_string(
        index=False
    )
)

print(
    "\nTRANSITION VALUES AMONG DERIVED "
    "NEW-EXPERT-PANEL RECORDS"
)
print(
    new_expert_panel_transition_counts.to_string(
        index=False
    )
)

print("\nDIRECTION AND TRANSITION INVENTORY")
print(
    transition_inventory.to_string(
        index=False
    )
)

print(
    "\nALTERNATIVE/SECONDARY OUTCOME "
    "NINE-SCORE DISCRIMINATION POINT ESTIMATES"
)
print(
    alternative_outcome_point_estimates.to_string(
        index=False
    )
)

print(
    "\nFULL-GES PRINCIPAL-COMPARATOR "
    "POINT DIFFERENCES"
)
print(
    alternative_outcome_principal_differences.to_string(
        index=False
    )
)

print(
    "\nFULL-GES 5%/10%/20% "
    "ALTERNATIVE-OUTCOME ENRICHMENT"
)
print(
    alternative_outcome_enrichment.to_string(
        index=False
    )
)

print("\nCURRENT DATA-AVAILABILITY BOUNDARY")
print("-" * 150)
print(
    "New contradictory high-rigor submission and magnitude of "
    "submitter-distribution change are not represented as dedicated "
    "frozen Stage 5 outcome fields. They require a separately prespecified "
    "derivation from the frozen T0/T1 nested SCV evidence and must not be "
    "claimed as completed by this cell."
)

print("\nCELL DECISION")
print("-" * 150)
print(
    "PASS_STAGE6C_ALTERNATIVE_OUTCOME_"
    "POINT_ESTIMATES_COMPLETE"
)
print(
    "Three alternative instability definitions and five secondary "
    "evidence-drift outcomes were frozen in code, accounted, and evaluated "
    "with locked point estimates."
)
print(
    "Inference is not yet claimed. The next cell must run paired bootstrap "
    "inference for these eight outcomes."
)
print(
    "No score, outcome, component flag, review field, threshold, weight, "
    "cohort membership, or frozen scientific artifact was modified."
)


STAGE 6C STEP 3F — CELL 6C-3F0 — ALTERNATIVE-OUTCOME AND SECONDARY-DRIFT POINT ESTIMATES
Frozen cohort SHA-256              : PASS (c6ad50dfc376a746e1830d000dba975101d88763fd61b8d1983f6300edb6e038)
Frozen cohort dimensions           : PASS (66,636 × 79)
Primary outcome reconstruction     : PASS (6,485 events)
Review-star delta reconstruction   : PASS (secondary_review_star_delta = T1 − T0)
Outcome definitions selected after performance: No
Review-drift outcomes independent validation: No — exploratory only
Frozen artifacts modified          : No
Scientific artifacts written       : No

ALTERNATIVE/SECONDARY OUTCOME ACCOUNTING
                           outcome_key                                outcome                    outcome_role                                                                                             definition  events  negatives  prevalence  overlap_with_primary_events  events_outside_primary  primary_events_not_in_outcome                                      

In [39]:
# ==================================================================================================
# STAGE 6C STEP 3F — CELL 6C-3F1
# ALTERNATIVE-OUTCOME AND SECONDARY-DRIFT PAIRED BOOTSTRAP INFERENCE
#
# Purpose:
#   1. Freshly verify the frozen 66,636-row Stage 6B primary-evaluable cohort.
#   2. Reconstruct the three alternative instability definitions and five exploratory
#      secondary evidence-drift outcomes frozen in Cell 6C-3F0.
#   3. Run 2,000 ordinary paired row-bootstrap replicates with seed 42, using identical
#      resamples across all eight outcomes and all nine frozen scores.
#   4. Produce outcome-specific AUPRC/AUROC confidence intervals and null-reference intervals.
#   5. Produce paired Full-GES-minus-comparator inference for three principal comparators
#      and five remaining metadata comparators.
#   6. Apply:
#        a. Holm correction across the three alternative-primary outcomes separately for
#           each principal comparator and metric;
#        b. Holm correction across the five exploratory evidence-drift outcomes separately
#           for each principal comparator and metric;
#        c. Holm correction across the five remaining comparators separately within each
#           outcome and metric.
#   7. Produce bootstrap intervals for Full-GES 5%/10%/20% exact-rank enrichment.
#
# Interpretation boundary:
#   Review-star, review-status, and expert-panel outcomes are exploratory evidence-drift
#   outcomes, not independent validation endpoints, because review metadata contributes to
#   the original GES pathway. The new-expert-panel outcome contains only eight events and is
#   explicitly flagged as extremely sparse.
#
# This cell is read-only. It does not modify scores, outcomes, review fields, thresholds,
# weights, cohort membership, linkage decisions, or frozen scientific artifacts.
# ==================================================================================================

from pathlib import Path
import gc
import hashlib
import re
import time

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from scipy.sparse import csr_matrix
from sklearn.metrics import average_precision_score, roc_auc_score


# --------------------------------------------------------------------------------------------------
# 1. FROZEN INPUT AND ANALYSIS SPECIFICATION
# --------------------------------------------------------------------------------------------------

STAGE6_DIR = Path(
    "/content/drive/MyDrive/GES_RAG_Temporal_Study/"
    "data_processed/stage6_temporal_validation"
)

EVALUABLE_PARQUET = (
    STAGE6_DIR / "stage6b_locked_primary_evaluable_cohort_v1.parquet"
)
EVALUABLE_SIDECAR = Path(str(EVALUABLE_PARQUET) + ".sha256")

EXPECTED_EVALUABLE_SHA256 = (
    "c6ad50dfc376a746e1830d000dba975101d88763fd61b8d1983f6300edb6e038"
)

EXPECTED_ROWS = 66_636
EXPECTED_COLUMNS = 79
EXPECTED_PRIMARY_EVENTS = 6_485
EXPECTED_PRIMARY_NEGATIVES = 60_151

RCV_COLUMN = "rcv_accession"
ROW_ORDER_COLUMN = "t0_row_order"
PRIMARY_OUTCOME_COLUMN = "primary_future_instability"

T0_REVIEW_STARS_COLUMN = "t0_aggregate_review_stars"
T1_REVIEW_STARS_COLUMN = "t1_aggregate_review_stars"
REVIEW_STAR_DELTA_COLUMN = "secondary_review_star_delta"
REVIEW_STATUS_CHANGED_COLUMN = "secondary_review_status_changed"

MATERIAL_GROUP_CHANGE_COLUMN = "event_material_clinical_group_change"
NEW_CONFLICT_COLUMN = "event_new_unresolved_conflict_at_t1"
PRIOR_RESOLUTION_COLUMN = "event_prior_conflict_resolved_to_material_group"

N_BOOTSTRAP = 2_000
RANDOM_SEED = 42
BOOTSTRAP_BATCH_SIZE = 50
MINIMUM_VALID_REPLICATES = 1_000
CI_QUANTILES = (0.025, 0.975)
RISK_FRACTIONS = [0.05, 0.10, 0.20]

SCORE_SPECIFICATIONS = {
    "full_ges": {
        "column": "full_ges_instability_risk_t0",
        "display_name": "Full GES",
    },
    "no_star_ges": {
        "column": "no_star_ges_instability_risk_t0",
        "display_name": "No-star GES",
    },
    "review_stars": {
        "column": "review_stars_instability_risk",
        "display_name": "Review stars",
    },
    "combined_metadata": {
        "column": "combined_metadata_instability_risk",
        "display_name": "Combined metadata",
    },
    "conflict": {
        "column": "conflict_instability_risk",
        "display_name": "Conflict",
    },
    "recency": {
        "column": "recency_instability_risk",
        "display_name": "Recency",
    },
    "submitter": {
        "column": "submitter_instability_risk",
        "display_name": "Submitter support",
    },
    "entropy": {
        "column": "entropy_instability_risk",
        "display_name": "Classification entropy",
    },
    "additive": {
        "column": "additive_instability_risk",
        "display_name": "Additive risk",
    },
}

PRINCIPAL_COMPARATORS = [
    "no_star_ges",
    "review_stars",
    "combined_metadata",
]

SECONDARY_COMPARATORS = [
    "conflict",
    "recency",
    "submitter",
    "entropy",
    "additive",
]

OUTCOME_SPECIFICATIONS = {
    "strict_material_instability": {
        "display_name": "Strict material instability",
        "role": "alternative_primary_sensitivity",
        "expected_events": 1_702,
        "definition": (
            "material clinical-group change OR material prior-conflict resolution; "
            "new unresolved conflict excluded"
        ),
        "independence_note": (
            "Alternative clinical-instability definition derived from frozen primary components."
        ),
    },
    "conflict_transition_instability": {
        "display_name": "Conflict-transition instability",
        "role": "alternative_primary_sensitivity",
        "expected_events": 5_086,
        "definition": (
            "new unresolved conflict OR material prior-conflict resolution"
        ),
        "independence_note": (
            "Alternative conflict-dynamics definition derived from frozen primary components."
        ),
    },
    "expanded_primary_or_review_star_change": {
        "display_name": "Expanded primary-or-star-drift outcome",
        "role": "alternative_primary_sensitivity",
        "expected_events": 14_437,
        "definition": (
            "primary future instability OR any review-star change"
        ),
        "independence_note": (
            "Expanded evidence-drift definition; not independent of review metadata."
        ),
    },
    "any_review_star_change": {
        "display_name": "Any review-star change",
        "role": "secondary_evidence_drift",
        "expected_events": 9_946,
        "definition": "T1 review stars differ from T0 review stars",
        "independence_note": (
            "Exploratory only because review stars contribute to the full GES pathway."
        ),
    },
    "review_star_increase": {
        "display_name": "Review-star increase",
        "role": "secondary_evidence_drift",
        "expected_events": 3_675,
        "definition": "T1 review stars greater than T0 review stars",
        "independence_note": (
            "Exploratory only because review stars contribute to the full GES pathway."
        ),
    },
    "review_star_decrease": {
        "display_name": "Review-star decrease",
        "role": "secondary_evidence_drift",
        "expected_events": 6_271,
        "definition": "T1 review stars lower than T0 review stars",
        "independence_note": (
            "Exploratory only because review stars contribute to the full GES pathway."
        ),
    },
    "any_review_status_change": {
        "display_name": "Any review-status change",
        "role": "secondary_evidence_drift",
        "expected_events": 10_946,
        "definition": "Frozen secondary_review_status_changed equals True",
        "independence_note": (
            "Exploratory only because review confidence contributes to full GES."
        ),
    },
    "new_expert_panel_involvement": {
        "display_name": "New expert-panel involvement",
        "role": "secondary_evidence_drift",
        "expected_events": 8,
        "definition": "T0 review stars < 3 and T1 review stars >= 3",
        "independence_note": (
            "Exploratory only; eight events and not an independent validation endpoint."
        ),
    },
}

ALTERNATIVE_PRIMARY_OUTCOMES = [
    key
    for key, specification in OUTCOME_SPECIFICATIONS.items()
    if specification["role"] == "alternative_primary_sensitivity"
]

SECONDARY_DRIFT_OUTCOMES = [
    key
    for key, specification in OUTCOME_SPECIFICATIONS.items()
    if specification["role"] == "secondary_evidence_drift"
]


# --------------------------------------------------------------------------------------------------
# 2. HELPERS
# --------------------------------------------------------------------------------------------------

def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def read_sidecar_hash(path: Path) -> str:
    text = path.read_text(encoding="utf-8").strip()
    matches = re.findall(r"\b[a-fA-F0-9]{64}\b", text)
    if not matches:
        raise ValueError(f"No SHA-256 value found in sidecar: {path}")
    return matches[0].lower()


def percentile_interval(values: np.ndarray) -> tuple[float, float]:
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if len(values) == 0:
        return np.nan, np.nan
    lower, upper = np.quantile(values, CI_QUANTILES)
    return float(lower), float(upper)


def bootstrap_sign_pvalue(differences: np.ndarray) -> float:
    differences = np.asarray(differences, dtype=float)
    differences = differences[np.isfinite(differences)]
    if len(differences) == 0:
        return np.nan

    n = len(differences)
    lower_tail = (np.count_nonzero(differences <= 0.0) + 1) / (n + 1)
    upper_tail = (np.count_nonzero(differences >= 0.0) + 1) / (n + 1)
    return float(min(1.0, 2.0 * min(lower_tail, upper_tail)))


def holm_adjust(p_values: np.ndarray) -> np.ndarray:
    p_values = np.asarray(p_values, dtype=float)
    adjusted = np.full(len(p_values), np.nan, dtype=float)
    valid_positions = np.where(np.isfinite(p_values))[0]

    if len(valid_positions) == 0:
        return adjusted

    valid_p = p_values[valid_positions]
    order = np.argsort(valid_p)
    m = len(valid_p)
    running_max = 0.0

    for rank, position_within_valid in enumerate(order):
        original_position = valid_positions[position_within_valid]
        raw_adjusted = (m - rank) * valid_p[position_within_valid]
        running_max = max(running_max, raw_adjusted)
        adjusted[original_position] = min(1.0, running_max)

    return adjusted


def interval_status(
    lower: float,
    upper: float,
    positive_label: str,
    negative_label: str,
) -> str:
    if not np.isfinite(lower) or not np.isfinite(upper):
        return "not_estimable"
    if lower > 0.0:
        return positive_label
    if upper < 0.0:
        return negative_label
    return "interval_includes_null"


def construct_score_base_cache(
    scores: np.ndarray,
) -> dict:
    scores = np.asarray(scores, dtype=np.float64)

    unique_scores, group_index = np.unique(
        scores,
        return_inverse=True,
    )

    n_rows = len(scores)
    n_groups = len(unique_scores)
    row_positions = np.arange(n_rows)

    total_group_matrix = csr_matrix(
        (
            np.ones(n_rows, dtype=np.float64),
            (group_index, row_positions),
        ),
        shape=(n_groups, n_rows),
    )

    return {
        "group_index": group_index,
        "n_groups": n_groups,
        "n_rows": n_rows,
        "total_group_matrix": total_group_matrix,
    }


def construct_positive_group_matrix(
    base_cache: dict,
    outcomes: np.ndarray,
) -> csr_matrix:
    outcomes = np.asarray(outcomes, dtype=np.int8)
    positive_positions = np.flatnonzero(outcomes == 1)

    return csr_matrix(
        (
            np.ones(len(positive_positions), dtype=np.float64),
            (
                base_cache["group_index"][positive_positions],
                positive_positions,
            ),
        ),
        shape=(
            base_cache["n_groups"],
            base_cache["n_rows"],
        ),
    )


def calculate_grouped_weighted_metrics(
    total_group_matrix: csr_matrix,
    positive_group_matrix: csr_matrix,
    count_matrix: np.ndarray,
    positive_totals: np.ndarray,
) -> tuple[np.ndarray, np.ndarray]:
    count_matrix = np.asarray(count_matrix)
    positive_totals = np.asarray(positive_totals, dtype=np.float64)

    sample_totals = count_matrix.sum(axis=1).astype(np.float64)
    negative_totals = sample_totals - positive_totals

    group_total_counts = np.asarray(
        total_group_matrix @ count_matrix.T,
        dtype=np.float64,
    )
    group_positive_counts = np.asarray(
        positive_group_matrix @ count_matrix.T,
        dtype=np.float64,
    )
    group_negative_counts = group_total_counts - group_positive_counts

    valid = (positive_totals > 0.0) & (negative_totals > 0.0)

    # AUROC with exact 0.5 credit for tied positive-negative pairs.
    cumulative_negatives_before = (
        np.cumsum(group_negative_counts, axis=0)
        - group_negative_counts
    )

    concordant_numerator = np.sum(
        group_positive_counts
        * (
            cumulative_negatives_before
            + 0.5 * group_negative_counts
        ),
        axis=0,
    )

    auroc = np.full(
        len(positive_totals),
        np.nan,
        dtype=np.float64,
    )

    np.divide(
        concordant_numerator,
        positive_totals * negative_totals,
        out=auroc,
        where=valid,
    )

    # Average precision / grouped AUPRC using descending tied-score groups.
    positive_desc = group_positive_counts[::-1, :]
    total_desc = group_total_counts[::-1, :]
    cumulative_positive = np.cumsum(positive_desc, axis=0)
    cumulative_total = np.cumsum(total_desc, axis=0)

    precision = np.zeros_like(
        cumulative_positive,
        dtype=np.float64,
    )

    np.divide(
        cumulative_positive,
        cumulative_total,
        out=precision,
        where=cumulative_total > 0.0,
    )

    ap_numerator = np.sum(
        precision * positive_desc,
        axis=0,
    )

    auprc = np.full(
        len(positive_totals),
        np.nan,
        dtype=np.float64,
    )

    np.divide(
        ap_numerator,
        positive_totals,
        out=auprc,
        where=valid,
    )

    return auprc, auroc


def build_frozen_rank_membership(
    frame: pd.DataFrame,
) -> dict[float, np.ndarray]:
    full_ges_column = SCORE_SPECIFICATIONS["full_ges"]["column"]

    ordered_positions = (
        frame[
            [RCV_COLUMN, ROW_ORDER_COLUMN, full_ges_column]
        ]
        .copy()
        .assign(_position=np.arange(len(frame)))
        .sort_values(
            [full_ges_column, ROW_ORDER_COLUMN, RCV_COLUMN],
            ascending=[False, True, True],
            kind="mergesort",
        )["_position"]
        .to_numpy(dtype=int)
    )

    memberships = {}
    n_rows = len(frame)

    for fraction in RISK_FRACTIONS:
        selected_count = int(np.ceil(n_rows * fraction))
        membership = np.zeros(n_rows, dtype=np.int8)
        membership[ordered_positions[:selected_count]] = 1
        memberships[fraction] = membership

    return memberships


def calculate_weighted_enrichment(
    count_matrix: np.ndarray,
    outcome: np.ndarray,
    membership: np.ndarray,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    count_matrix = np.asarray(count_matrix, dtype=np.float64)
    outcome = np.asarray(outcome, dtype=np.float64)
    membership = np.asarray(membership, dtype=np.float64)

    sample_rows = count_matrix.sum(axis=1)
    sample_events = count_matrix @ outcome

    selected_rows = count_matrix @ membership
    selected_events = count_matrix @ (membership * outcome)

    remaining_rows = sample_rows - selected_rows
    remaining_events = sample_events - selected_events

    prevalence = np.full(
        len(sample_rows),
        np.nan,
        dtype=np.float64,
    )
    selected_rate = np.full(
        len(sample_rows),
        np.nan,
        dtype=np.float64,
    )
    remaining_rate = np.full(
        len(sample_rows),
        np.nan,
        dtype=np.float64,
    )

    np.divide(
        sample_events,
        sample_rows,
        out=prevalence,
        where=sample_rows > 0,
    )

    np.divide(
        selected_events,
        selected_rows,
        out=selected_rate,
        where=selected_rows > 0,
    )

    np.divide(
        remaining_events,
        remaining_rows,
        out=remaining_rate,
        where=remaining_rows > 0,
    )

    enrichment = np.full(
        len(sample_rows),
        np.nan,
        dtype=np.float64,
    )
    risk_ratio = np.full(
        len(sample_rows),
        np.nan,
        dtype=np.float64,
    )

    np.divide(
        selected_rate,
        prevalence,
        out=enrichment,
        where=prevalence > 0,
    )

    np.divide(
        selected_rate,
        remaining_rate,
        out=risk_ratio,
        where=remaining_rate > 0,
    )

    return enrichment, risk_ratio, selected_rate


# --------------------------------------------------------------------------------------------------
# 3. FRESH CRYPTOGRAPHIC AND STRUCTURAL VERIFICATION
# --------------------------------------------------------------------------------------------------

if not EVALUABLE_PARQUET.exists():
    raise FileNotFoundError(
        f"Missing frozen evaluable cohort:\n{EVALUABLE_PARQUET}"
    )

if not EVALUABLE_SIDECAR.exists():
    raise FileNotFoundError(
        f"Missing SHA-256 sidecar:\n{EVALUABLE_SIDECAR}"
    )

observed_sha256 = sha256_file(EVALUABLE_PARQUET)
sidecar_sha256 = read_sidecar_hash(EVALUABLE_SIDECAR)

if observed_sha256 != EXPECTED_EVALUABLE_SHA256:
    raise AssertionError(
        "Frozen Stage 6B evaluable-cohort SHA-256 mismatch.\n"
        f"Expected: {EXPECTED_EVALUABLE_SHA256}\n"
        f"Observed: {observed_sha256}"
    )

if sidecar_sha256 != observed_sha256:
    raise AssertionError(
        "Frozen evaluable-cohort sidecar mismatch.\n"
        f"Calculated: {observed_sha256}\n"
        f"Sidecar:    {sidecar_sha256}"
    )

parquet_file = pq.ParquetFile(EVALUABLE_PARQUET)
metadata = parquet_file.metadata
schema_columns = parquet_file.schema_arrow.names

if metadata.num_rows != EXPECTED_ROWS:
    raise AssertionError(
        f"Unexpected row count: {metadata.num_rows:,}; "
        f"expected {EXPECTED_ROWS:,}"
    )

if metadata.num_columns != EXPECTED_COLUMNS:
    raise AssertionError(
        f"Unexpected column count: {metadata.num_columns}; "
        f"expected {EXPECTED_COLUMNS}"
    )

required_columns = [
    RCV_COLUMN,
    ROW_ORDER_COLUMN,
    PRIMARY_OUTCOME_COLUMN,
    T0_REVIEW_STARS_COLUMN,
    T1_REVIEW_STARS_COLUMN,
    REVIEW_STAR_DELTA_COLUMN,
    REVIEW_STATUS_CHANGED_COLUMN,
    MATERIAL_GROUP_CHANGE_COLUMN,
    NEW_CONFLICT_COLUMN,
    PRIOR_RESOLUTION_COLUMN,
] + [
    specification["column"]
    for specification in SCORE_SPECIFICATIONS.values()
]

missing_columns = [
    column
    for column in required_columns
    if column not in schema_columns
]

if missing_columns:
    raise KeyError(
        "Missing required frozen columns:\n"
        + "\n".join(missing_columns)
    )

cohort = pd.read_parquet(
    EVALUABLE_PARQUET,
    columns=required_columns,
).copy()

if len(cohort) != EXPECTED_ROWS:
    raise AssertionError("Loaded dataframe row count mismatch.")

if cohort[RCV_COLUMN].isna().any():
    raise AssertionError("Missing RCV accession detected.")

if cohort[RCV_COLUMN].nunique(dropna=False) != EXPECTED_ROWS:
    raise AssertionError("RCV accessions are not unique.")

row_order = pd.to_numeric(
    cohort[ROW_ORDER_COLUMN],
    errors="raise",
).to_numpy()

if len(np.unique(row_order)) != EXPECTED_ROWS:
    raise AssertionError("t0_row_order is not unique.")

if not np.all(np.diff(row_order) > 0):
    raise AssertionError(
        "Frozen cohort is not in increasing T0 row order."
    )

for column in [
    PRIMARY_OUTCOME_COLUMN,
    MATERIAL_GROUP_CHANGE_COLUMN,
    NEW_CONFLICT_COLUMN,
    PRIOR_RESOLUTION_COLUMN,
    REVIEW_STATUS_CHANGED_COLUMN,
]:
    cohort[column] = pd.to_numeric(
        cohort[column],
        errors="raise",
    ).astype(int)

    if not set(cohort[column].unique()).issubset({0, 1}):
        raise AssertionError(f"{column} is not binary.")

for column in [
    T0_REVIEW_STARS_COLUMN,
    T1_REVIEW_STARS_COLUMN,
    REVIEW_STAR_DELTA_COLUMN,
]:
    cohort[column] = pd.to_numeric(
        cohort[column],
        errors="raise",
    ).astype(int)

for specification in SCORE_SPECIFICATIONS.values():
    column = specification["column"]
    cohort[column] = pd.to_numeric(
        cohort[column],
        errors="raise",
    )

    values = cohort[column].to_numpy(dtype=float)

    if not np.isfinite(values).all():
        raise AssertionError(
            f"Missing/nonfinite score in {column}."
        )

    if ((values < 0.0) | (values > 1.0)).any():
        raise AssertionError(
            f"Score outside [0,1] in {column}."
        )


# --------------------------------------------------------------------------------------------------
# 4. RECONSTRUCT AND VERIFY ALL EIGHT OUTCOMES
# --------------------------------------------------------------------------------------------------

primary_outcome = cohort[
    PRIMARY_OUTCOME_COLUMN
].to_numpy(dtype=np.int8)

if int(primary_outcome.sum()) != EXPECTED_PRIMARY_EVENTS:
    raise AssertionError("Primary event count mismatch.")

if int((primary_outcome == 0).sum()) != EXPECTED_PRIMARY_NEGATIVES:
    raise AssertionError("Primary negative count mismatch.")

material_group_change = cohort[
    MATERIAL_GROUP_CHANGE_COLUMN
].to_numpy(dtype=bool)

new_conflict = cohort[
    NEW_CONFLICT_COLUMN
].to_numpy(dtype=bool)

prior_resolution = cohort[
    PRIOR_RESOLUTION_COLUMN
].to_numpy(dtype=bool)

reconstructed_primary = (
    material_group_change
    | new_conflict
    | prior_resolution
).astype(np.int8)

if not np.array_equal(
    reconstructed_primary,
    primary_outcome,
):
    mismatch_count = int(
        np.count_nonzero(
            reconstructed_primary != primary_outcome
        )
    )
    raise AssertionError(
        f"Primary-component reconstruction mismatch for "
        f"{mismatch_count:,} rows."
    )

t0_stars = cohort[
    T0_REVIEW_STARS_COLUMN
].to_numpy(dtype=int)

t1_stars = cohort[
    T1_REVIEW_STARS_COLUMN
].to_numpy(dtype=int)

frozen_star_delta = cohort[
    REVIEW_STAR_DELTA_COLUMN
].to_numpy(dtype=int)

calculated_star_delta = t1_stars - t0_stars

if not np.array_equal(
    frozen_star_delta,
    calculated_star_delta,
):
    mismatch_count = int(
        np.count_nonzero(
            frozen_star_delta != calculated_star_delta
        )
    )
    raise AssertionError(
        f"Frozen review-star delta mismatch for "
        f"{mismatch_count:,} rows."
    )

any_star_change = calculated_star_delta != 0
star_increase = calculated_star_delta > 0
star_decrease = calculated_star_delta < 0

review_status_change = cohort[
    REVIEW_STATUS_CHANGED_COLUMN
].to_numpy(dtype=bool)

new_expert_panel = (
    (t0_stars < 3)
    & (t1_stars >= 3)
)

outcome_arrays = {
    "strict_material_instability": (
        material_group_change
        | prior_resolution
    ).astype(np.int8),

    "conflict_transition_instability": (
        new_conflict
        | prior_resolution
    ).astype(np.int8),

    "expanded_primary_or_review_star_change": (
        primary_outcome.astype(bool)
        | any_star_change
    ).astype(np.int8),

    "any_review_star_change": (
        any_star_change.astype(np.int8)
    ),

    "review_star_increase": (
        star_increase.astype(np.int8)
    ),

    "review_star_decrease": (
        star_decrease.astype(np.int8)
    ),

    "any_review_status_change": (
        review_status_change.astype(np.int8)
    ),

    "new_expert_panel_involvement": (
        new_expert_panel.astype(np.int8)
    ),
}

if np.any(star_increase & star_decrease):
    raise AssertionError(
        "A row was simultaneously classified as star increase and decrease."
    )

if not np.array_equal(
    any_star_change,
    star_increase | star_decrease,
):
    raise AssertionError(
        "Any-star-change does not equal increase OR decrease."
    )

outcome_accounting_rows = []

for outcome_key, outcome in outcome_arrays.items():
    expected_events = OUTCOME_SPECIFICATIONS[
        outcome_key
    ]["expected_events"]

    observed_events = int(outcome.sum())
    observed_negatives = int(len(outcome) - observed_events)

    if observed_events != expected_events:
        raise AssertionError(
            f"{outcome_key} has {observed_events:,} events; "
            f"expected {expected_events:,}."
        )

    if observed_events == 0 or observed_negatives == 0:
        raise AssertionError(
            f"{outcome_key} does not contain both outcome classes."
        )

    outcome_accounting_rows.append(
        {
            "outcome_key": outcome_key,
            "outcome": OUTCOME_SPECIFICATIONS[
                outcome_key
            ]["display_name"],
            "outcome_role": OUTCOME_SPECIFICATIONS[
                outcome_key
            ]["role"],
            "events": observed_events,
            "negatives": observed_negatives,
            "prevalence": observed_events / EXPECTED_ROWS,
            "sparse_event_flag": observed_events < 50,
            "independence_note": OUTCOME_SPECIFICATIONS[
                outcome_key
            ]["independence_note"],
        }
    )

alternative_outcome_accounting = pd.DataFrame(
    outcome_accounting_rows
)


# --------------------------------------------------------------------------------------------------
# 5. PREPARE SHARED SCORE CACHES AND VALIDATE POINT METRICS
# --------------------------------------------------------------------------------------------------

n_rows = len(cohort)

score_arrays = {
    model_key: cohort[
        specification["column"]
    ].to_numpy(dtype=np.float64)
    for model_key, specification
    in SCORE_SPECIFICATIONS.items()
}

score_base_caches = {
    model_key: construct_score_base_cache(
        scores
    )
    for model_key, scores
    in score_arrays.items()
}

positive_group_matrices = {
    outcome_key: {
        model_key: construct_positive_group_matrix(
            score_base_caches[model_key],
            outcome,
        )
        for model_key in SCORE_SPECIFICATIONS
    }
    for outcome_key, outcome in outcome_arrays.items()
}

frozen_rank_memberships = build_frozen_rank_membership(
    cohort
)

original_counts = np.ones(
    (1, n_rows),
    dtype=np.int16,
)

point_metrics = {}
point_estimate_rows = []

for outcome_key, outcome in outcome_arrays.items():
    outcome_specification = OUTCOME_SPECIFICATIONS[
        outcome_key
    ]

    events = int(outcome.sum())
    negatives = n_rows - events
    prevalence = events / n_rows

    point_metrics[outcome_key] = {}

    for model_key, model_specification in SCORE_SPECIFICATIONS.items():
        scores = score_arrays[model_key]

        sklearn_auprc = float(
            average_precision_score(
                outcome,
                scores,
            )
        )

        sklearn_auroc = float(
            roc_auc_score(
                outcome,
                scores,
            )
        )

        fast_auprc, fast_auroc = (
            calculate_grouped_weighted_metrics(
                score_base_caches[
                    model_key
                ]["total_group_matrix"],
                positive_group_matrices[
                    outcome_key
                ][model_key],
                original_counts,
                np.array(
                    [events],
                    dtype=np.float64,
                ),
            )
        )

        if not np.isclose(
            fast_auprc[0],
            sklearn_auprc,
            rtol=1e-11,
            atol=1e-12,
        ):
            raise AssertionError(
                f"Fast AUPRC validation failed for "
                f"{outcome_key}/{model_key}: "
                f"{fast_auprc[0]:.15f} vs "
                f"{sklearn_auprc:.15f}"
            )

        if not np.isclose(
            fast_auroc[0],
            sklearn_auroc,
            rtol=1e-11,
            atol=1e-12,
        ):
            raise AssertionError(
                f"Fast AUROC validation failed for "
                f"{outcome_key}/{model_key}: "
                f"{fast_auroc[0]:.15f} vs "
                f"{sklearn_auroc:.15f}"
            )

        point_metrics[
            outcome_key
        ][model_key] = {
            "auprc": sklearn_auprc,
            "auroc": sklearn_auroc,
        }

        point_estimate_rows.append(
            {
                "outcome_key": outcome_key,
                "outcome": outcome_specification[
                    "display_name"
                ],
                "outcome_role": outcome_specification[
                    "role"
                ],
                "model_key": model_key,
                "model": model_specification[
                    "display_name"
                ],
                "rows": n_rows,
                "events": events,
                "negatives": negatives,
                "prevalence": prevalence,
                "point_auprc": sklearn_auprc,
                "point_auroc": sklearn_auroc,
                "fast_metric_validation": "PASS",
                "sparse_event_flag": events < 50,
            }
        )

alternative_outcome_point_estimates = pd.DataFrame(
    point_estimate_rows
)


# --------------------------------------------------------------------------------------------------
# 6. RUN 2,000 IDENTICAL PAIRED RESAMPLES ACROSS ALL OUTCOMES AND SCORES
# --------------------------------------------------------------------------------------------------

bootstrap_metrics = {
    outcome_key: {
        model_key: {
            "auprc": np.full(
                N_BOOTSTRAP,
                np.nan,
                dtype=np.float64,
            ),
            "auroc": np.full(
                N_BOOTSTRAP,
                np.nan,
                dtype=np.float64,
            ),
        }
        for model_key in SCORE_SPECIFICATIONS
    }
    for outcome_key in OUTCOME_SPECIFICATIONS
}

bootstrap_prevalence = {
    outcome_key: np.full(
        N_BOOTSTRAP,
        np.nan,
        dtype=np.float64,
    )
    for outcome_key in OUTCOME_SPECIFICATIONS
}

bootstrap_enrichment = {
    outcome_key: {
        fraction: {
            "enrichment": np.full(
                N_BOOTSTRAP,
                np.nan,
                dtype=np.float64,
            ),
            "risk_ratio": np.full(
                N_BOOTSTRAP,
                np.nan,
                dtype=np.float64,
            ),
            "selected_event_rate": np.full(
                N_BOOTSTRAP,
                np.nan,
                dtype=np.float64,
            ),
        }
        for fraction in RISK_FRACTIONS
    }
    for outcome_key in OUTCOME_SPECIFICATIONS
}

rng = np.random.default_rng(RANDOM_SEED)

probabilities = np.full(
    n_rows,
    1.0 / n_rows,
    dtype=np.float64,
)
probabilities[-1] = (
    1.0 - probabilities[:-1].sum()
)

analysis_start_time = time.time()

print(
    "Preparing alternative-outcome bootstrap: "
    f"{n_rows:,} rows; "
    + ", ".join(
        f"{OUTCOME_SPECIFICATIONS[key]['display_name']}="
        f"{OUTCOME_SPECIFICATIONS[key]['expected_events']:,}"
        for key in OUTCOME_SPECIFICATIONS
    )
)

print(
    "Fast metric validation against scikit-learn: "
    "PASS for all 72 outcome-score combinations"
)

for batch_start in range(
    0,
    N_BOOTSTRAP,
    BOOTSTRAP_BATCH_SIZE,
):
    batch_end = min(
        batch_start + BOOTSTRAP_BATCH_SIZE,
        N_BOOTSTRAP,
    )

    batch_size = batch_end - batch_start

    bootstrap_counts = rng.multinomial(
        n_rows,
        probabilities,
        size=batch_size,
    )

    if not np.all(
        bootstrap_counts.sum(axis=1) == n_rows
    ):
        raise AssertionError(
            "Bootstrap sample-size preservation failed."
        )

    for outcome_key, outcome in outcome_arrays.items():
        positive_totals = (
            bootstrap_counts @ outcome
        ).astype(np.float64)

        valid_outcomes = (
            (positive_totals > 0.0)
            & (positive_totals < n_rows)
        )

        bootstrap_prevalence[
            outcome_key
        ][batch_start:batch_end] = np.where(
            valid_outcomes,
            positive_totals / n_rows,
            np.nan,
        )

        for model_key in SCORE_SPECIFICATIONS:
            batch_auprc, batch_auroc = (
                calculate_grouped_weighted_metrics(
                    score_base_caches[
                        model_key
                    ]["total_group_matrix"],
                    positive_group_matrices[
                        outcome_key
                    ][model_key],
                    bootstrap_counts,
                    positive_totals,
                )
            )

            bootstrap_metrics[
                outcome_key
            ][model_key]["auprc"][
                batch_start:batch_end
            ] = batch_auprc

            bootstrap_metrics[
                outcome_key
            ][model_key]["auroc"][
                batch_start:batch_end
            ] = batch_auroc

        for fraction in RISK_FRACTIONS:
            (
                enrichment,
                risk_ratio,
                selected_event_rate,
            ) = calculate_weighted_enrichment(
                bootstrap_counts,
                outcome,
                frozen_rank_memberships[
                    fraction
                ],
            )

            bootstrap_enrichment[
                outcome_key
            ][fraction]["enrichment"][
                batch_start:batch_end
            ] = enrichment

            bootstrap_enrichment[
                outcome_key
            ][fraction]["risk_ratio"][
                batch_start:batch_end
            ] = risk_ratio

            bootstrap_enrichment[
                outcome_key
            ][fraction]["selected_event_rate"][
                batch_start:batch_end
            ] = selected_event_rate

    if (
        batch_end % 250 == 0
        or batch_end == N_BOOTSTRAP
    ):
        valid_summary = ", ".join(
            f"{key}="
            f"{int(np.isfinite(bootstrap_metrics[key]['full_ges']['auprc'][:batch_end]).sum()):,}"
            for key in OUTCOME_SPECIFICATIONS
        )

        elapsed = time.time() - analysis_start_time

        print(
            f"Completed {batch_end:,}/{N_BOOTSTRAP:,} "
            f"replicates | valid [{valid_summary}] | "
            f"elapsed {elapsed:.1f}s"
        )

    del bootstrap_counts
    gc.collect()


# --------------------------------------------------------------------------------------------------
# 7. VERIFY PAIRED VALIDITY
# --------------------------------------------------------------------------------------------------

outcome_valid_masks = {}
validity_rows = []

for outcome_key, outcome_specification in OUTCOME_SPECIFICATIONS.items():
    valid_mask = np.isfinite(
        bootstrap_metrics[
            outcome_key
        ]["full_ges"]["auprc"]
    )

    valid_replicates = int(
        valid_mask.sum()
    )

    invalid_one_class_replicates = (
        N_BOOTSTRAP - valid_replicates
    )

    if valid_replicates < MINIMUM_VALID_REPLICATES:
        raise AssertionError(
            f"{outcome_key} produced only "
            f"{valid_replicates:,} valid replicates."
        )

    for model_key in SCORE_SPECIFICATIONS:
        for metric_name in ["auprc", "auroc"]:
            model_valid = np.isfinite(
                bootstrap_metrics[
                    outcome_key
                ][model_key][metric_name]
            )

            if not np.array_equal(
                model_valid,
                valid_mask,
            ):
                raise AssertionError(
                    f"Paired validity mismatch for "
                    f"{outcome_key}/{model_key}/{metric_name}."
                )

    outcome_valid_masks[
        outcome_key
    ] = valid_mask

    validity_rows.append(
        {
            "outcome_key": outcome_key,
            "outcome": outcome_specification[
                "display_name"
            ],
            "outcome_role": outcome_specification[
                "role"
            ],
            "events": outcome_specification[
                "expected_events"
            ],
            "attempted_replicates": N_BOOTSTRAP,
            "valid_replicates": valid_replicates,
            "invalid_one_class_replicates": (
                invalid_one_class_replicates
            ),
            "sparse_event_flag": (
                outcome_specification[
                    "expected_events"
                ] < 50
            ),
        }
    )

alternative_outcome_bootstrap_validity = pd.DataFrame(
    validity_rows
)


# --------------------------------------------------------------------------------------------------
# 8. MODEL-SPECIFIC OUTCOME INTERVALS
# --------------------------------------------------------------------------------------------------

model_interval_rows = []

for outcome_key, outcome_specification in OUTCOME_SPECIFICATIONS.items():
    events = outcome_specification[
        "expected_events"
    ]
    negatives = EXPECTED_ROWS - events
    prevalence = events / EXPECTED_ROWS

    valid_replicates = int(
        outcome_valid_masks[
            outcome_key
        ].sum()
    )

    invalid_replicates = (
        N_BOOTSTRAP - valid_replicates
    )

    for model_key, model_specification in SCORE_SPECIFICATIONS.items():
        auprc_values = bootstrap_metrics[
            outcome_key
        ][model_key]["auprc"]

        auroc_values = bootstrap_metrics[
            outcome_key
        ][model_key]["auroc"]

        auprc_lower, auprc_upper = (
            percentile_interval(
                auprc_values
            )
        )

        auroc_lower, auroc_upper = (
            percentile_interval(
                auroc_values
            )
        )

        (
            auprc_null_lower,
            auprc_null_upper,
        ) = percentile_interval(
            auprc_values
            - bootstrap_prevalence[
                outcome_key
            ]
        )

        (
            auroc_null_lower,
            auroc_null_upper,
        ) = percentile_interval(
            auroc_values - 0.50
        )

        model_interval_rows.append(
            {
                "outcome_key": outcome_key,
                "outcome": outcome_specification[
                    "display_name"
                ],
                "outcome_role": outcome_specification[
                    "role"
                ],
                "model_key": model_key,
                "model": model_specification[
                    "display_name"
                ],
                "rows": EXPECTED_ROWS,
                "events": events,
                "negatives": negatives,
                "prevalence": prevalence,
                "point_auprc": point_metrics[
                    outcome_key
                ][model_key]["auprc"],
                "auprc_ci_lower": auprc_lower,
                "auprc_ci_upper": auprc_upper,
                "point_auprc_minus_prevalence": (
                    point_metrics[
                        outcome_key
                    ][model_key]["auprc"]
                    - prevalence
                ),
                "auprc_minus_prevalence_ci_lower": (
                    auprc_null_lower
                ),
                "auprc_minus_prevalence_ci_upper": (
                    auprc_null_upper
                ),
                "auprc_null_status": interval_status(
                    auprc_null_lower,
                    auprc_null_upper,
                    "supported_above_prevalence",
                    "supported_below_prevalence",
                ),
                "point_auroc": point_metrics[
                    outcome_key
                ][model_key]["auroc"],
                "auroc_ci_lower": auroc_lower,
                "auroc_ci_upper": auroc_upper,
                "point_auroc_minus_0_50": (
                    point_metrics[
                        outcome_key
                    ][model_key]["auroc"]
                    - 0.50
                ),
                "auroc_minus_0_50_ci_lower": (
                    auroc_null_lower
                ),
                "auroc_minus_0_50_ci_upper": (
                    auroc_null_upper
                ),
                "auroc_null_status": interval_status(
                    auroc_null_lower,
                    auroc_null_upper,
                    "supported_above_0_50",
                    "supported_below_0_50",
                ),
                "attempted_bootstrap_replicates": (
                    N_BOOTSTRAP
                ),
                "valid_bootstrap_replicates": (
                    valid_replicates
                ),
                "invalid_one_class_replicates": (
                    invalid_replicates
                ),
                "sparse_event_flag": events < 50,
                "independence_note": outcome_specification[
                    "independence_note"
                ],
            }
        )

alternative_outcome_model_intervals = pd.DataFrame(
    model_interval_rows
)


# --------------------------------------------------------------------------------------------------
# 9. PAIRED FULL-GES-MINUS-COMPARATOR INFERENCE
# --------------------------------------------------------------------------------------------------

paired_rows = []

for outcome_key, outcome_specification in OUTCOME_SPECIFICATIONS.items():
    valid_replicates = int(
        outcome_valid_masks[
            outcome_key
        ].sum()
    )

    invalid_replicates = (
        N_BOOTSTRAP - valid_replicates
    )

    for comparison_family, comparator_keys in [
        ("principal_prespecified", PRINCIPAL_COMPARATORS),
        (
            "secondary_remaining_comparators",
            SECONDARY_COMPARATORS,
        ),
    ]:
        for comparator_key in comparator_keys:
            comparator_name = SCORE_SPECIFICATIONS[
                comparator_key
            ]["display_name"]

            for metric_name in ["auprc", "auroc"]:
                differences = (
                    bootstrap_metrics[
                        outcome_key
                    ]["full_ges"][metric_name]
                    - bootstrap_metrics[
                        outcome_key
                    ][comparator_key][metric_name]
                )

                finite_differences = differences[
                    np.isfinite(differences)
                ]

                lower, upper = percentile_interval(
                    finite_differences
                )

                point_difference = (
                    point_metrics[
                        outcome_key
                    ]["full_ges"][metric_name]
                    - point_metrics[
                        outcome_key
                    ][comparator_key][metric_name]
                )

                paired_rows.append(
                    {
                        "outcome_key": outcome_key,
                        "outcome": outcome_specification[
                            "display_name"
                        ],
                        "outcome_role": outcome_specification[
                            "role"
                        ],
                        "metric": metric_name.upper(),
                        "comparison_family": comparison_family,
                        "comparison": (
                            f"Full GES minus {comparator_name}"
                        ),
                        "comparator_key": comparator_key,
                        "point_difference": point_difference,
                        "difference_ci_lower": lower,
                        "difference_ci_upper": upper,
                        "paired_interval_status": interval_status(
                            lower,
                            upper,
                            "full_ges_supported_higher",
                            "full_ges_supported_lower",
                        ),
                        "bootstrap_probability_full_greater": float(
                            np.mean(
                                finite_differences > 0.0
                            )
                        ),
                        "bootstrap_sign_p_value": (
                            bootstrap_sign_pvalue(
                                finite_differences
                            )
                        ),
                        "principal_outcome_family_holm_adjusted_p": np.nan,
                        "principal_outcome_family_holm_supported_at_0_05": pd.NA,
                        "secondary_comparator_family_holm_adjusted_p": np.nan,
                        "secondary_comparator_family_holm_supported_at_0_05": pd.NA,
                        "attempted_bootstrap_replicates": N_BOOTSTRAP,
                        "valid_bootstrap_replicates": valid_replicates,
                        "invalid_one_class_replicates": invalid_replicates,
                        "sparse_event_flag": (
                            outcome_specification[
                                "expected_events"
                            ] < 50
                        ),
                        "independence_note": outcome_specification[
                            "independence_note"
                        ],
                    }
                )

alternative_outcome_paired_differences = pd.DataFrame(
    paired_rows
)

alternative_outcome_paired_differences[
    "principal_outcome_family_holm_supported_at_0_05"
] = pd.Series(
    pd.NA,
    index=alternative_outcome_paired_differences.index,
    dtype="boolean",
)

alternative_outcome_paired_differences[
    "secondary_comparator_family_holm_supported_at_0_05"
] = pd.Series(
    pd.NA,
    index=alternative_outcome_paired_differences.index,
    dtype="boolean",
)


# 9A. Principal-comparator outcome-family correction:
#     - three alternative-primary outcomes per comparator and metric;
#     - five exploratory evidence-drift outcomes per comparator and metric.
for role_name, role_outcomes in [
    (
        "alternative_primary_sensitivity",
        ALTERNATIVE_PRIMARY_OUTCOMES,
    ),
    (
        "secondary_evidence_drift",
        SECONDARY_DRIFT_OUTCOMES,
    ),
]:
    for comparator_key in PRINCIPAL_COMPARATORS:
        for metric_name in ["AUPRC", "AUROC"]:
            family_mask = (
                alternative_outcome_paired_differences[
                    "outcome_key"
                ].isin(role_outcomes)
                & (
                    alternative_outcome_paired_differences[
                        "outcome_role"
                    ] == role_name
                )
                & (
                    alternative_outcome_paired_differences[
                        "comparison_family"
                    ] == "principal_prespecified"
                )
                & (
                    alternative_outcome_paired_differences[
                        "comparator_key"
                    ] == comparator_key
                )
                & (
                    alternative_outcome_paired_differences[
                        "metric"
                    ] == metric_name
                )
            )

            p_values = (
                alternative_outcome_paired_differences.loc[
                    family_mask,
                    "bootstrap_sign_p_value",
                ].to_numpy(dtype=float)
            )

            expected_family_size = len(
                role_outcomes
            )

            if len(p_values) != expected_family_size:
                raise AssertionError(
                    f"{role_name}/{comparator_key}/{metric_name} "
                    f"contains {len(p_values)} tests; "
                    f"expected {expected_family_size}."
                )

            adjusted = holm_adjust(
                p_values
            )

            alternative_outcome_paired_differences.loc[
                family_mask,
                "principal_outcome_family_holm_adjusted_p",
            ] = adjusted

            alternative_outcome_paired_differences.loc[
                family_mask,
                "principal_outcome_family_holm_supported_at_0_05",
            ] = adjusted < 0.05


# 9B. Remaining-comparator correction:
#     five secondary comparators within each outcome and metric.
for outcome_key in OUTCOME_SPECIFICATIONS:
    for metric_name in ["AUPRC", "AUROC"]:
        family_mask = (
            (
                alternative_outcome_paired_differences[
                    "outcome_key"
                ] == outcome_key
            )
            & (
                alternative_outcome_paired_differences[
                    "metric"
                ] == metric_name
            )
            & (
                alternative_outcome_paired_differences[
                    "comparison_family"
                ] == "secondary_remaining_comparators"
            )
        )

        p_values = (
            alternative_outcome_paired_differences.loc[
                family_mask,
                "bootstrap_sign_p_value",
            ].to_numpy(dtype=float)
        )

        if len(p_values) != 5:
            raise AssertionError(
                f"{outcome_key}/{metric_name} secondary "
                f"comparator family has {len(p_values)} tests; "
                "expected 5."
            )

        adjusted = holm_adjust(
            p_values
        )

        alternative_outcome_paired_differences.loc[
            family_mask,
            "secondary_comparator_family_holm_adjusted_p",
        ] = adjusted

        alternative_outcome_paired_differences.loc[
            family_mask,
            "secondary_comparator_family_holm_supported_at_0_05",
        ] = adjusted < 0.05


# --------------------------------------------------------------------------------------------------
# 10. FULL-GES 5%/10%/20% ENRICHMENT INTERVALS
# --------------------------------------------------------------------------------------------------

enrichment_rows = []

for outcome_key, outcome_specification in OUTCOME_SPECIFICATIONS.items():
    outcome = outcome_arrays[
        outcome_key
    ]

    prevalence = float(
        outcome.mean()
    )

    for fraction in RISK_FRACTIONS:
        membership = frozen_rank_memberships[
            fraction
        ]

        selected_rows = int(
            membership.sum()
        )

        selected_events = int(
            np.sum(
                outcome * membership
            )
        )

        selected_event_rate = (
            selected_events
            / selected_rows
        )

        remaining_rows = (
            n_rows - selected_rows
        )

        remaining_events = int(
            outcome.sum()
            - selected_events
        )

        remaining_event_rate = (
            remaining_events
            / remaining_rows
        )

        point_enrichment = (
            selected_event_rate
            / prevalence
        )

        point_risk_ratio = (
            selected_event_rate
            / remaining_event_rate
            if remaining_event_rate > 0
            else np.nan
        )

        enrichment_values = (
            bootstrap_enrichment[
                outcome_key
            ][fraction]["enrichment"]
        )

        risk_ratio_values = (
            bootstrap_enrichment[
                outcome_key
            ][fraction]["risk_ratio"]
        )

        selected_rate_values = (
            bootstrap_enrichment[
                outcome_key
            ][fraction]["selected_event_rate"]
        )

        (
            enrichment_lower,
            enrichment_upper,
        ) = percentile_interval(
            enrichment_values
        )

        (
            risk_ratio_lower,
            risk_ratio_upper,
        ) = percentile_interval(
            risk_ratio_values
        )

        (
            selected_rate_lower,
            selected_rate_upper,
        ) = percentile_interval(
            selected_rate_values
        )

        enrichment_rows.append(
            {
                "outcome_key": outcome_key,
                "outcome": outcome_specification[
                    "display_name"
                ],
                "outcome_role": outcome_specification[
                    "role"
                ],
                "risk_fraction": fraction,
                "selected_rows": selected_rows,
                "selected_events": selected_events,
                "selected_event_rate": selected_event_rate,
                "selected_event_rate_ci_lower": selected_rate_lower,
                "selected_event_rate_ci_upper": selected_rate_upper,
                "remaining_rows": remaining_rows,
                "remaining_events": remaining_events,
                "remaining_event_rate": remaining_event_rate,
                "outcome_prevalence": prevalence,
                "point_risk_ratio_vs_remaining": point_risk_ratio,
                "risk_ratio_ci_lower": risk_ratio_lower,
                "risk_ratio_ci_upper": risk_ratio_upper,
                "point_enrichment_over_prevalence": point_enrichment,
                "enrichment_ci_lower": enrichment_lower,
                "enrichment_ci_upper": enrichment_upper,
                "enrichment_interval_status": interval_status(
                    enrichment_lower - 1.0,
                    enrichment_upper - 1.0,
                    "supported_above_1",
                    "supported_below_1",
                ),
                "attempted_bootstrap_replicates": N_BOOTSTRAP,
                "valid_enrichment_replicates": int(
                    np.isfinite(
                        enrichment_values
                    ).sum()
                ),
                "sparse_event_flag": (
                    outcome_specification[
                        "expected_events"
                    ] < 50
                ),
                "independence_note": outcome_specification[
                    "independence_note"
                ],
            }
        )

alternative_outcome_enrichment_intervals = pd.DataFrame(
    enrichment_rows
)


# --------------------------------------------------------------------------------------------------
# 11. COMPLETENESS CHECKS
# --------------------------------------------------------------------------------------------------

expected_model_rows = (
    len(OUTCOME_SPECIFICATIONS)
    * len(SCORE_SPECIFICATIONS)
)

expected_paired_rows = (
    len(OUTCOME_SPECIFICATIONS)
    * (
        len(PRINCIPAL_COMPARATORS)
        + len(SECONDARY_COMPARATORS)
    )
    * 2
)

expected_enrichment_rows = (
    len(OUTCOME_SPECIFICATIONS)
    * len(RISK_FRACTIONS)
)

if len(alternative_outcome_model_intervals) != expected_model_rows:
    raise AssertionError(
        f"Model-interval rows="
        f"{len(alternative_outcome_model_intervals)}; "
        f"expected {expected_model_rows}."
    )

if len(alternative_outcome_paired_differences) != expected_paired_rows:
    raise AssertionError(
        f"Paired-comparison rows="
        f"{len(alternative_outcome_paired_differences)}; "
        f"expected {expected_paired_rows}."
    )

if len(alternative_outcome_enrichment_intervals) != expected_enrichment_rows:
    raise AssertionError(
        f"Enrichment rows="
        f"{len(alternative_outcome_enrichment_intervals)}; "
        f"expected {expected_enrichment_rows}."
    )

principal_rows = (
    alternative_outcome_paired_differences[
        "comparison_family"
    ] == "principal_prespecified"
)

secondary_rows = (
    alternative_outcome_paired_differences[
        "comparison_family"
    ] == "secondary_remaining_comparators"
)

if alternative_outcome_paired_differences.loc[
    principal_rows,
    "principal_outcome_family_holm_adjusted_p",
].isna().any():
    raise AssertionError(
        "A principal comparison is missing its "
        "outcome-family Holm-adjusted p-value."
    )

if alternative_outcome_paired_differences.loc[
    secondary_rows,
    "secondary_comparator_family_holm_adjusted_p",
].isna().any():
    raise AssertionError(
        "A remaining-comparator comparison is missing its "
        "within-outcome Holm-adjusted p-value."
    )

if alternative_outcome_paired_differences.loc[
    principal_rows,
    "secondary_comparator_family_holm_adjusted_p",
].notna().any():
    raise AssertionError(
        "A principal comparison was incorrectly included "
        "in a remaining-comparator family."
    )

if alternative_outcome_paired_differences.loc[
    secondary_rows,
    "principal_outcome_family_holm_adjusted_p",
].notna().any():
    raise AssertionError(
        "A remaining-comparator comparison was incorrectly "
        "included in a principal outcome family."
    )


# --------------------------------------------------------------------------------------------------
# 12. DISPLAY RESULTS AND FINAL DECISION
# --------------------------------------------------------------------------------------------------

model_display_columns = [
    "outcome",
    "outcome_role",
    "model",
    "events",
    "prevalence",
    "point_auprc",
    "auprc_ci_lower",
    "auprc_ci_upper",
    "point_auprc_minus_prevalence",
    "auprc_minus_prevalence_ci_lower",
    "auprc_minus_prevalence_ci_upper",
    "auprc_null_status",
    "point_auroc",
    "auroc_ci_lower",
    "auroc_ci_upper",
    "point_auroc_minus_0_50",
    "auroc_minus_0_50_ci_lower",
    "auroc_minus_0_50_ci_upper",
    "auroc_null_status",
    "valid_bootstrap_replicates",
    "invalid_one_class_replicates",
    "sparse_event_flag",
]

paired_display_columns = [
    "outcome",
    "outcome_role",
    "metric",
    "comparison_family",
    "comparison",
    "point_difference",
    "difference_ci_lower",
    "difference_ci_upper",
    "paired_interval_status",
    "bootstrap_probability_full_greater",
    "bootstrap_sign_p_value",
    "principal_outcome_family_holm_adjusted_p",
    "principal_outcome_family_holm_supported_at_0_05",
    "secondary_comparator_family_holm_adjusted_p",
    "secondary_comparator_family_holm_supported_at_0_05",
    "valid_bootstrap_replicates",
    "invalid_one_class_replicates",
    "sparse_event_flag",
]

enrichment_display_columns = [
    "outcome",
    "outcome_role",
    "risk_fraction",
    "selected_rows",
    "selected_events",
    "selected_event_rate",
    "selected_event_rate_ci_lower",
    "selected_event_rate_ci_upper",
    "remaining_event_rate",
    "outcome_prevalence",
    "point_risk_ratio_vs_remaining",
    "risk_ratio_ci_lower",
    "risk_ratio_ci_upper",
    "point_enrichment_over_prevalence",
    "enrichment_ci_lower",
    "enrichment_ci_upper",
    "enrichment_interval_status",
    "valid_enrichment_replicates",
    "sparse_event_flag",
]

analysis_elapsed = (
    time.time()
    - analysis_start_time
)

separator = "=" * 150

print("\n" + separator)
print(
    "STAGE 6C STEP 3F — CELL 6C-3F1 — "
    "ALTERNATIVE-OUTCOME AND SECONDARY-DRIFT "
    "PAIRED BOOTSTRAP INFERENCE"
)
print(separator)

print(
    f"Frozen cohort SHA-256              : "
    f"PASS ({observed_sha256})"
)

print(
    f"Frozen cohort dimensions           : "
    f"PASS ({metadata.num_rows:,} × "
    f"{metadata.num_columns})"
)

print(
    f"Primary outcome reconstruction     : "
    f"PASS ({EXPECTED_PRIMARY_EVENTS:,} events)"
)

print(
    "Alternative/secondary accounting  : "
    "PASS (all eight event counts match Cell 6C-3F0)"
)

print(
    f"Bootstrap attempts                 : "
    f"{N_BOOTSTRAP:,}"
)

print(
    f"Random seed                        : "
    f"{RANDOM_SEED}"
)

print(
    "Identical resamples                : "
    "Yes, across all eight outcomes and nine scores"
)

print(
    "Principal multiplicity policy      : "
    "Holm across 3 alternative outcomes or 5 exploratory drift "
    "outcomes, separately by comparator and metric"
)

print(
    "Remaining-comparator policy        : "
    "Holm across 5 comparators within each outcome and metric"
)

print(
    "New-expert-panel outcome           : "
    "Exploratory and extremely sparse (8 events)"
)

print(
    f"Elapsed time                       : "
    f"{analysis_elapsed:.1f}s"
)

print(
    "Frozen artifacts modified          : No"
)

print("\nALTERNATIVE/SECONDARY OUTCOME ACCOUNTING")
print(
    alternative_outcome_accounting.to_string(
        index=False
    )
)

print("\nOUTCOME-SPECIFIC BOOTSTRAP VALIDITY")
print(
    alternative_outcome_bootstrap_validity.to_string(
        index=False
    )
)

print("\nOUTCOME-SPECIFIC MODEL INTERVALS")
print(
    alternative_outcome_model_intervals[
        model_display_columns
    ].to_string(index=False)
)

print(
    "\nOUTCOME-SPECIFIC PAIRED "
    "FULL-GES-MINUS-COMPARATOR INFERENCE"
)
print(
    alternative_outcome_paired_differences[
        paired_display_columns
    ].to_string(index=False)
)

print(
    "\nOUTCOME-SPECIFIC FULL-GES "
    "5%/10%/20% ENRICHMENT INTERVALS"
)
print(
    alternative_outcome_enrichment_intervals[
        enrichment_display_columns
    ].to_string(index=False)
)

print("\nINTERPRETATION BOUNDARY")
print("-" * 150)

print(
    "Review-star, review-status, and expert-panel outcomes remain "
    "exploratory evidence-drift analyses because review metadata contributes "
    "to the original GES pathway. New expert-panel involvement has only "
    "eight events and must not be treated as stable confirmatory evidence."
)

print(
    "New contradictory high-rigor submission and magnitude of "
    "submitter-distribution change still require a separately prespecified "
    "derivation from frozen nested T0/T1 SCV evidence."
)

print("\nCELL DECISION")
print("-" * 150)

print(
    "PASS_STAGE6C_ALTERNATIVE_OUTCOME_"
    "PAIRED_BOOTSTRAP_INFERENCE_COMPLETE"
)

print(
    "Outcome-specific model intervals, paired Full-GES-minus-comparator "
    "inference, role-aware principal-outcome multiplicity correction, "
    "within-outcome remaining-comparator Holm correction, and frozen-rank "
    "enrichment intervals are complete for all eight definitions."
)

print(
    "No score, outcome, component flag, review field, threshold, weight, "
    "cohort membership, linkage decision, or frozen scientific artifact "
    "was modified."
)

Preparing alternative-outcome bootstrap: 66,636 rows; Strict material instability=1,702, Conflict-transition instability=5,086, Expanded primary-or-star-drift outcome=14,437, Any review-star change=9,946, Review-star increase=3,675, Review-star decrease=6,271, Any review-status change=10,946, New expert-panel involvement=8
Fast metric validation against scikit-learn: PASS for all 72 outcome-score combinations
Completed 250/2,000 replicates | valid [strict_material_instability=250, conflict_transition_instability=250, expanded_primary_or_review_star_change=250, any_review_star_change=250, review_star_increase=250, review_star_decrease=250, any_review_status_change=250, new_expert_panel_involvement=250] | elapsed 30.4s
Completed 500/2,000 replicates | valid [strict_material_instability=500, conflict_transition_instability=500, expanded_primary_or_review_star_change=500, any_review_star_change=500, review_star_increase=500, review_star_decrease=500, any_review_status_change=500, new_exper

In [40]:
# ==================================================================================================
# STAGE 6C STEP 3G — CELL 6C-3G0A
# NESTED-SCV DERIVATION PREFLIGHT AND SCHEMA FREEZE DIAGNOSTIC
#
# Purpose:
#   1. Freshly verify the frozen Stage 6B primary-evaluable cohort.
#   2. Locate and cryptographically verify the accepted T0 and T1 RCV Parquet artifacts.
#   3. Map every evaluable Stage 6 row to its frozen T0 RCV and accepted linked T1 RCV.
#   4. Parse the nested SCV evidence in read-only mode.
#   5. Inventory the exact nested keys, data types, nonempty coverage, sample values, and
#      semantic candidates needed to prespecify:
#         a. newly appearing contradictory high-rigor submissions;
#         b. magnitude of submitter-distribution change.
#   6. Produce no performance result and create no derived outcome.
#
# Why this preflight is required:
#   The Stage 5 table does not contain dedicated frozen columns for these two outcomes.
#   Their operational definitions must be based on the actual accepted nested-SCV schema,
#   frozen before any association with GES scores is evaluated.
#
# This cell is read-only. It does not modify or write any scientific artifact.
# ==================================================================================================

from pathlib import Path
from collections import Counter, defaultdict
import hashlib
import json
import math
import re

import numpy as np
import pandas as pd
import pyarrow.parquet as pq


# --------------------------------------------------------------------------------------------------
# 1. FROZEN INPUT SPECIFICATION
# --------------------------------------------------------------------------------------------------

PROJECT_DIR = Path(
    "/content/drive/MyDrive/GES_RAG_Temporal_Study"
)

STAGE6_DIR = (
    PROJECT_DIR
    / "data_processed"
    / "stage6_temporal_validation"
)

EVALUABLE_PARQUET = (
    STAGE6_DIR
    / "stage6b_locked_primary_evaluable_cohort_v1.parquet"
)

EVALUABLE_SIDECAR = Path(
    str(EVALUABLE_PARQUET) + ".sha256"
)

EXPECTED_EVALUABLE_SHA256 = (
    "c6ad50dfc376a746e1830d000dba975101d88763fd61b8d1983f6300edb6e038"
)

T0_FILENAME = (
    "t0_rcv_target_genes_corrected_v1_2.parquet"
)

T1_FILENAME = (
    "t1_rcv_target_genes_harmonized_v1.parquet"
)

EXPECTED_T0_SHA256 = (
    "f6b6760b2ad6e4352e3bdecdeaf89827e8a514b031abf2373bb17568d999466d"
)

EXPECTED_T1_SHA256 = (
    "5713a11bdbf4804758cc011f9b2f302afc91fa1f88c1b178d675c28bb277d37c"
)

EXPECTED_T0_ROWS = 71_659
EXPECTED_T0_COLUMNS = 34
EXPECTED_T1_ROWS = 100_920
EXPECTED_T1_COLUMNS = 36

EXPECTED_EVALUABLE_ROWS = 66_636
EXPECTED_EVALUABLE_COLUMNS = 79
EXPECTED_PRIMARY_EVENTS = 6_485
EXPECTED_PRIMARY_NEGATIVES = 60_151

STAGE6_T0_RCV_COLUMN = "rcv_accession"
STAGE6_T1_RCV_COLUMN = "linked_t1_rcv_accession"
STAGE6_ROW_ORDER_COLUMN = "t0_row_order"
STAGE6_OUTCOME_COLUMN = "primary_future_instability"
STAGE6_LINKAGE_DECISION_COLUMN = "linkage_decision_category"

SOURCE_RCV_COLUMN = "rcv_accession"
NESTED_SCV_COLUMN = "scv_records_json"

MAX_SAMPLE_VALUES_PER_KEY = 3
MAX_SAMPLE_CHARACTERS = 240

SEMANTIC_CANDIDATE_GROUPS = {
    "scv_identity": [
        "SCV",
        "ACCESSION",
        "SUBMISSION",
        "ID",
    ],
    "submitter_identity": [
        "SUBMITTER",
        "ORGANIZATION",
        "ORG",
        "ORGID",
        "INSTITUTION",
    ],
    "classification": [
        "CLASSIFICATION",
        "CLINSIG",
        "SIGNIFICANCE",
        "INTERPRETATION",
        "DESCRIPTION",
    ],
    "classification_group": [
        "GROUP",
        "NORMALIZED",
        "BROAD",
    ],
    "review_rigor": [
        "REVIEW",
        "STAR",
        "EXPERT",
        "GUIDELINE",
        "PRACTICE",
        "CRITERIA",
    ],
    "submission_date": [
        "DATE",
        "UPDATED",
        "EVALUATED",
        "CREATED",
        "SUBMITTED",
    ],
}


# --------------------------------------------------------------------------------------------------
# 2. HELPERS
# --------------------------------------------------------------------------------------------------

def sha256_file(
    path: Path,
    chunk_size: int = 1024 * 1024,
) -> str:
    digest = hashlib.sha256()

    with path.open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


def read_sidecar_hash(path: Path) -> str:
    text = path.read_text(
        encoding="utf-8"
    ).strip()

    matches = re.findall(
        r"\b[a-fA-F0-9]{64}\b",
        text,
    )

    if not matches:
        raise ValueError(
            f"No SHA-256 value found in sidecar: {path}"
        )

    return matches[0].lower()


def normalize_rcv(
    series: pd.Series,
) -> pd.Series:
    return (
        series.astype("string")
        .str.upper()
        .str.strip()
        .str.extract(
            r"(RCV\d+)",
            expand=False,
        )
    )


def normalize_key_name(
    value: str,
) -> str:
    return re.sub(
        r"[^A-Z0-9]+",
        "_",
        str(value).upper(),
    ).strip("_")


def is_nonempty(
    value,
) -> bool:
    if value is None:
        return False

    if isinstance(value, str):
        return value.strip() != ""

    if isinstance(value, (list, tuple, dict, set)):
        return len(value) > 0

    try:
        return not bool(pd.isna(value))
    except Exception:
        return True


def compact_sample(
    value,
) -> str:
    if isinstance(value, (dict, list, tuple)):
        try:
            text = json.dumps(
                value,
                ensure_ascii=False,
                sort_keys=True,
            )
        except Exception:
            text = repr(value)
    else:
        text = str(value)

    text = re.sub(
        r"\s+",
        " ",
        text,
    ).strip()

    if len(text) > MAX_SAMPLE_CHARACTERS:
        text = (
            text[:MAX_SAMPLE_CHARACTERS]
            + "…"
        )

    return text


def parse_nested_scv_value(
    value,
    row_label: str,
) -> list[dict]:
    if isinstance(value, list):
        parsed = value

    elif isinstance(value, tuple):
        parsed = list(value)

    elif isinstance(value, np.ndarray):
        parsed = value.tolist()

    elif isinstance(value, str):
        text = value.strip()

        if text == "":
            raise ValueError(
                f"Blank nested SCV JSON at {row_label}"
            )

        try:
            parsed = json.loads(text)
        except Exception as error:
            raise ValueError(
                f"Malformed nested SCV JSON at {row_label}: "
                f"{error}"
            ) from error

    else:
        try:
            if pd.isna(value):
                raise ValueError(
                    f"Missing nested SCV value at {row_label}"
                )
        except Exception:
            pass

        raise TypeError(
            f"Unsupported nested SCV type at {row_label}: "
            f"{type(value).__name__}"
        )

    if not isinstance(parsed, list):
        raise TypeError(
            f"Nested SCV value is not a list at {row_label}: "
            f"{type(parsed).__name__}"
        )

    for position, record in enumerate(parsed):
        if not isinstance(record, dict):
            raise TypeError(
                f"Nested SCV record is not a dictionary at "
                f"{row_label}, position {position}: "
                f"{type(record).__name__}"
            )

    return parsed


def locate_verified_artifact(
    project_dir: Path,
    filename: str,
    expected_sha256: str,
) -> tuple[Path, pd.DataFrame]:
    candidates = sorted(
        project_dir.rglob(filename)
    )

    if not candidates:
        raise FileNotFoundError(
            f"Could not locate {filename} beneath:\n"
            f"{project_dir}"
        )

    diagnostic_rows = []
    verified_candidates = []

    for candidate in candidates:
        observed_hash = sha256_file(
            candidate
        )

        diagnostic_rows.append(
            {
                "candidate_path": str(candidate),
                "sha256": observed_hash,
                "matches_expected_sha256": (
                    observed_hash
                    == expected_sha256
                ),
            }
        )

        if observed_hash == expected_sha256:
            verified_candidates.append(
                candidate
            )

    diagnostics = pd.DataFrame(
        diagnostic_rows
    )

    if not verified_candidates:
        print(
            f"\nCandidate diagnostics for {filename}:"
        )
        print(
            diagnostics.to_string(
                index=False
            )
        )

        raise AssertionError(
            f"No located copy of {filename} has the "
            "accepted frozen SHA-256."
        )

    # Multiple byte-identical copies are acceptable. Select the shortest stable path
    # deterministically and print all candidates for auditability.
    selected = sorted(
        verified_candidates,
        key=lambda item: (
            len(str(item)),
            str(item),
        ),
    )[0]

    return selected, diagnostics


def profile_nested_records(
    frame: pd.DataFrame,
    timepoint: str,
) -> tuple[pd.DataFrame, pd.DataFrame, dict]:
    key_presence = Counter()
    key_nonempty = Counter()
    key_type_counts = defaultdict(Counter)
    key_samples = defaultdict(list)

    total_rows = 0
    total_records = 0
    zero_scv_rows = 0
    duplicate_dictionary_key_failures = 0

    row_scv_counts = []
    parsed_by_rcv = {}

    for row in frame.itertuples(index=False):
        rcv = getattr(
            row,
            SOURCE_RCV_COLUMN,
        )

        raw_nested = getattr(
            row,
            NESTED_SCV_COLUMN,
        )

        records = parse_nested_scv_value(
            raw_nested,
            f"{timepoint}:{rcv}",
        )

        total_rows += 1
        total_records += len(records)
        row_scv_counts.append(
            len(records)
        )

        if len(records) == 0:
            zero_scv_rows += 1

        parsed_by_rcv[rcv] = records

        for record in records:
            if len(record) != len(
                set(record.keys())
            ):
                duplicate_dictionary_key_failures += 1

            for key, value in record.items():
                key_presence[key] += 1

                value_type = type(
                    value
                ).__name__

                key_type_counts[
                    key
                ][value_type] += 1

                if is_nonempty(value):
                    key_nonempty[
                        key
                    ] += 1

                    sample = compact_sample(
                        value
                    )

                    if (
                        sample
                        not in key_samples[key]
                        and len(
                            key_samples[key]
                        )
                        < MAX_SAMPLE_VALUES_PER_KEY
                    ):
                        key_samples[
                            key
                        ].append(sample)

    if duplicate_dictionary_key_failures:
        raise AssertionError(
            f"{timepoint} contained "
            f"{duplicate_dictionary_key_failures:,} "
            "nested dictionary key failures."
        )

    key_rows = []

    for key in sorted(
        key_presence
    ):
        presence = key_presence[
            key
        ]

        nonempty = key_nonempty[
            key
        ]

        key_rows.append(
            {
                "timepoint": timepoint,
                "nested_key": key,
                "normalized_key": normalize_key_name(
                    key
                ),
                "records_with_key": presence,
                "records_with_nonempty_value": nonempty,
                "nonempty_percent_of_present": (
                    100.0
                    * nonempty
                    / presence
                    if presence
                    else np.nan
                ),
                "observed_python_types": "; ".join(
                    f"{type_name}:{count}"
                    for type_name, count
                    in sorted(
                        key_type_counts[key].items()
                    )
                ),
                "sample_values": " || ".join(
                    key_samples[key]
                ),
            }
        )

    key_summary = pd.DataFrame(
        key_rows
    )

    row_count_distribution = (
        pd.Series(
            row_scv_counts,
            name="nested_scv_count",
        )
        .value_counts()
        .sort_index()
        .rename_axis(
            "nested_scv_count"
        )
        .reset_index(
            name="rcv_rows"
        )
    )

    row_count_distribution.insert(
        0,
        "timepoint",
        timepoint,
    )

    accounting = {
        "timepoint": timepoint,
        "rcv_rows": total_rows,
        "nested_scv_records": total_records,
        "zero_scv_rows": zero_scv_rows,
        "minimum_scv_records_per_rcv": (
            int(min(row_scv_counts))
            if row_scv_counts
            else 0
        ),
        "maximum_scv_records_per_rcv": (
            int(max(row_scv_counts))
            if row_scv_counts
            else 0
        ),
        "mean_scv_records_per_rcv": (
            float(np.mean(row_scv_counts))
            if row_scv_counts
            else np.nan
        ),
    }

    return (
        key_summary,
        row_count_distribution,
        {
            "accounting": accounting,
            "parsed_by_rcv": parsed_by_rcv,
        },
    )


def rank_semantic_candidates(
    combined_key_summary: pd.DataFrame,
) -> pd.DataFrame:
    rows = []

    shared_keys = (
        combined_key_summary.groupby(
            "nested_key"
        )["timepoint"]
        .nunique()
    )

    for key in sorted(
        combined_key_summary[
            "nested_key"
        ].unique()
    ):
        normalized = normalize_key_name(
            key
        )

        for candidate_group, tokens in SEMANTIC_CANDIDATE_GROUPS.items():
            token_hits = [
                token
                for token in tokens
                if token in normalized
            ]

            score = len(
                token_hits
            )

            if candidate_group == "scv_identity":
                if "SCV" in normalized:
                    score += 5
                if "ACCESSION" in normalized:
                    score += 4

            if candidate_group == "submitter_identity":
                if "ORGID" in normalized:
                    score += 6
                if "SUBMITTER" in normalized:
                    score += 4
                if "ORGANIZATION" in normalized:
                    score += 4

            if candidate_group == "classification":
                if "CLASSIFICATION" in normalized:
                    score += 5
                if "CLINSIG" in normalized:
                    score += 5

            if candidate_group == "review_rigor":
                if "REVIEW" in normalized:
                    score += 5
                if "STAR" in normalized:
                    score += 5

            if candidate_group == "submission_date":
                if "DATE" in normalized:
                    score += 4

            if score <= 0:
                continue

            key_rows = combined_key_summary.loc[
                combined_key_summary[
                    "nested_key"
                ] == key
            ]

            rows.append(
                {
                    "candidate_group": candidate_group,
                    "nested_key": key,
                    "normalized_key": normalized,
                    "semantic_score": score,
                    "token_hits": ", ".join(
                        token_hits
                    ),
                    "present_at_both_timepoints": (
                        int(
                            shared_keys.get(
                                key,
                                0,
                            )
                        )
                        == 2
                    ),
                    "t0_nonempty_records": int(
                        key_rows.loc[
                            key_rows[
                                "timepoint"
                            ] == "T0",
                            "records_with_nonempty_value",
                        ].sum()
                    ),
                    "t1_nonempty_records": int(
                        key_rows.loc[
                            key_rows[
                                "timepoint"
                            ] == "T1",
                            "records_with_nonempty_value",
                        ].sum()
                    ),
                    "sample_values": " || ".join(
                        key_rows[
                            "sample_values"
                        ].dropna().astype(str).tolist()
                    ),
                }
            )

    if not rows:
        return pd.DataFrame(
            columns=[
                "candidate_group",
                "nested_key",
                "normalized_key",
                "semantic_score",
                "token_hits",
                "present_at_both_timepoints",
                "t0_nonempty_records",
                "t1_nonempty_records",
                "sample_values",
            ]
        )

    return (
        pd.DataFrame(rows)
        .sort_values(
            [
                "candidate_group",
                "semantic_score",
                "present_at_both_timepoints",
                "nested_key",
            ],
            ascending=[
                True,
                False,
                False,
                True,
            ],
        )
        .reset_index(drop=True)
    )


# --------------------------------------------------------------------------------------------------
# 3. VERIFY THE FROZEN STAGE 6B EVALUABLE COHORT
# --------------------------------------------------------------------------------------------------

if not EVALUABLE_PARQUET.exists():
    raise FileNotFoundError(
        f"Missing frozen Stage 6B evaluable cohort:\n"
        f"{EVALUABLE_PARQUET}"
    )

if not EVALUABLE_SIDECAR.exists():
    raise FileNotFoundError(
        f"Missing Stage 6B SHA-256 sidecar:\n"
        f"{EVALUABLE_SIDECAR}"
    )

observed_evaluable_sha256 = sha256_file(
    EVALUABLE_PARQUET
)

sidecar_evaluable_sha256 = read_sidecar_hash(
    EVALUABLE_SIDECAR
)

if (
    observed_evaluable_sha256
    != EXPECTED_EVALUABLE_SHA256
):
    raise AssertionError(
        "Frozen Stage 6B evaluable cohort SHA-256 mismatch.\n"
        f"Expected: {EXPECTED_EVALUABLE_SHA256}\n"
        f"Observed: {observed_evaluable_sha256}"
    )

if (
    sidecar_evaluable_sha256
    != observed_evaluable_sha256
):
    raise AssertionError(
        "Stage 6B evaluable cohort sidecar mismatch."
    )

evaluable_parquet_file = pq.ParquetFile(
    EVALUABLE_PARQUET
)

evaluable_metadata = (
    evaluable_parquet_file.metadata
)

evaluable_schema = (
    evaluable_parquet_file.schema_arrow.names
)

if (
    evaluable_metadata.num_rows
    != EXPECTED_EVALUABLE_ROWS
):
    raise AssertionError(
        f"Unexpected Stage 6B row count: "
        f"{evaluable_metadata.num_rows:,}"
    )

if (
    evaluable_metadata.num_columns
    != EXPECTED_EVALUABLE_COLUMNS
):
    raise AssertionError(
        f"Unexpected Stage 6B column count: "
        f"{evaluable_metadata.num_columns}"
    )

required_stage6_columns = [
    STAGE6_T0_RCV_COLUMN,
    STAGE6_T1_RCV_COLUMN,
    STAGE6_ROW_ORDER_COLUMN,
    STAGE6_OUTCOME_COLUMN,
    STAGE6_LINKAGE_DECISION_COLUMN,
]

missing_stage6_columns = [
    column
    for column in required_stage6_columns
    if column not in evaluable_schema
]

if missing_stage6_columns:
    raise KeyError(
        "Missing required Stage 6B columns:\n"
        + "\n".join(
            missing_stage6_columns
        )
    )

stage6 = pd.read_parquet(
    EVALUABLE_PARQUET,
    columns=required_stage6_columns,
).copy()

stage6[
    STAGE6_T0_RCV_COLUMN
] = normalize_rcv(
    stage6[
        STAGE6_T0_RCV_COLUMN
    ]
)

stage6[
    STAGE6_T1_RCV_COLUMN
] = normalize_rcv(
    stage6[
        STAGE6_T1_RCV_COLUMN
    ]
)

if stage6[
    STAGE6_T0_RCV_COLUMN
].isna().any():
    raise AssertionError(
        "A Stage 6B T0 RCV failed normalization."
    )

if stage6[
    STAGE6_T1_RCV_COLUMN
].isna().any():
    raise AssertionError(
        "A Stage 6B linked T1 RCV failed normalization."
    )

if (
    stage6[
        STAGE6_T0_RCV_COLUMN
    ].nunique()
    != EXPECTED_EVALUABLE_ROWS
):
    raise AssertionError(
        "Stage 6B T0 RCV keys are not unique."
    )

if (
    stage6[
        STAGE6_T1_RCV_COLUMN
    ].nunique()
    != EXPECTED_EVALUABLE_ROWS
):
    raise AssertionError(
        "Stage 6B linked T1 RCV keys are not unique."
    )

stage6[
    STAGE6_OUTCOME_COLUMN
] = pd.to_numeric(
    stage6[
        STAGE6_OUTCOME_COLUMN
    ],
    errors="raise",
).astype(int)

if (
    int(
        stage6[
            STAGE6_OUTCOME_COLUMN
        ].sum()
    )
    != EXPECTED_PRIMARY_EVENTS
):
    raise AssertionError(
        "Stage 6B primary event count mismatch."
    )

if (
    int(
        (
            stage6[
                STAGE6_OUTCOME_COLUMN
            ]
            == 0
        ).sum()
    )
    != EXPECTED_PRIMARY_NEGATIVES
):
    raise AssertionError(
        "Stage 6B primary negative count mismatch."
    )


# --------------------------------------------------------------------------------------------------
# 4. LOCATE AND VERIFY THE ACCEPTED T0 AND T1 PARQUETS
# --------------------------------------------------------------------------------------------------

(
    t0_path,
    t0_location_diagnostics,
) = locate_verified_artifact(
    PROJECT_DIR,
    T0_FILENAME,
    EXPECTED_T0_SHA256,
)

(
    t1_path,
    t1_location_diagnostics,
) = locate_verified_artifact(
    PROJECT_DIR,
    T1_FILENAME,
    EXPECTED_T1_SHA256,
)

t0_parquet_file = pq.ParquetFile(
    t0_path
)

t1_parquet_file = pq.ParquetFile(
    t1_path
)

t0_metadata = t0_parquet_file.metadata
t1_metadata = t1_parquet_file.metadata

t0_schema = t0_parquet_file.schema_arrow.names
t1_schema = t1_parquet_file.schema_arrow.names

if (
    t0_metadata.num_rows
    != EXPECTED_T0_ROWS
    or t0_metadata.num_columns
    != EXPECTED_T0_COLUMNS
):
    raise AssertionError(
        "Accepted T0 Parquet dimensions mismatch."
    )

if (
    t1_metadata.num_rows
    != EXPECTED_T1_ROWS
    or t1_metadata.num_columns
    != EXPECTED_T1_COLUMNS
):
    raise AssertionError(
        "Accepted T1 Parquet dimensions mismatch."
    )

for label, schema in [
    ("T0", t0_schema),
    ("T1", t1_schema),
]:
    missing = [
        column
        for column in [
            SOURCE_RCV_COLUMN,
            NESTED_SCV_COLUMN,
        ]
        if column not in schema
    ]

    if missing:
        raise KeyError(
            f"{label} is missing required columns:\n"
            + "\n".join(missing)
        )


# --------------------------------------------------------------------------------------------------
# 5. LOAD ONLY IDENTIFIERS AND NESTED SCV EVIDENCE, THEN MAP TO STAGE 6B
# --------------------------------------------------------------------------------------------------

t0_source = pd.read_parquet(
    t0_path,
    columns=[
        SOURCE_RCV_COLUMN,
        NESTED_SCV_COLUMN,
    ],
).copy()

t1_source = pd.read_parquet(
    t1_path,
    columns=[
        SOURCE_RCV_COLUMN,
        NESTED_SCV_COLUMN,
    ],
).copy()

t0_source[
    SOURCE_RCV_COLUMN
] = normalize_rcv(
    t0_source[
        SOURCE_RCV_COLUMN
    ]
)

t1_source[
    SOURCE_RCV_COLUMN
] = normalize_rcv(
    t1_source[
        SOURCE_RCV_COLUMN
    ]
)

for label, frame, expected_rows in [
    (
        "T0",
        t0_source,
        EXPECTED_T0_ROWS,
    ),
    (
        "T1",
        t1_source,
        EXPECTED_T1_ROWS,
    ),
]:
    if frame[
        SOURCE_RCV_COLUMN
    ].isna().any():
        raise AssertionError(
            f"{label} contains malformed RCV accessions."
        )

    if frame[
        SOURCE_RCV_COLUMN
    ].nunique() != expected_rows:
        raise AssertionError(
            f"{label} RCV accessions are not unique."
        )

t0_evaluable = (
    stage6[
        [
            STAGE6_T0_RCV_COLUMN,
            STAGE6_ROW_ORDER_COLUMN,
        ]
    ]
    .merge(
        t0_source,
        left_on=STAGE6_T0_RCV_COLUMN,
        right_on=SOURCE_RCV_COLUMN,
        how="left",
        validate="one_to_one",
        indicator=True,
    )
    .sort_values(
        STAGE6_ROW_ORDER_COLUMN,
        kind="mergesort",
    )
    .reset_index(drop=True)
)

t1_evaluable = (
    stage6[
        [
            STAGE6_T1_RCV_COLUMN,
            STAGE6_ROW_ORDER_COLUMN,
        ]
    ]
    .merge(
        t1_source,
        left_on=STAGE6_T1_RCV_COLUMN,
        right_on=SOURCE_RCV_COLUMN,
        how="left",
        validate="one_to_one",
        indicator=True,
    )
    .sort_values(
        STAGE6_ROW_ORDER_COLUMN,
        kind="mergesort",
    )
    .reset_index(drop=True)
)

if not t0_evaluable[
    "_merge"
].eq("both").all():
    raise AssertionError(
        "At least one evaluable T0 RCV did not map "
        "to the accepted T0 Parquet."
    )

if not t1_evaluable[
    "_merge"
].eq("both").all():
    raise AssertionError(
        "At least one evaluable linked T1 RCV did not map "
        "to the accepted T1 Parquet."
    )

if len(t0_evaluable) != EXPECTED_EVALUABLE_ROWS:
    raise AssertionError(
        "Mapped T0 evaluable row count mismatch."
    )

if len(t1_evaluable) != EXPECTED_EVALUABLE_ROWS:
    raise AssertionError(
        "Mapped T1 evaluable row count mismatch."
    )

if not np.array_equal(
    t0_evaluable[
        STAGE6_ROW_ORDER_COLUMN
    ].to_numpy(),
    t1_evaluable[
        STAGE6_ROW_ORDER_COLUMN
    ].to_numpy(),
):
    raise AssertionError(
        "T0 and T1 mapped rows are not aligned "
        "in frozen T0 row order."
    )

t0_evaluable = (
    t0_evaluable[
        [
            SOURCE_RCV_COLUMN,
            NESTED_SCV_COLUMN,
        ]
    ]
    .copy()
)

t1_evaluable = (
    t1_evaluable[
        [
            SOURCE_RCV_COLUMN,
            NESTED_SCV_COLUMN,
        ]
    ]
    .copy()
)


# --------------------------------------------------------------------------------------------------
# 6. PROFILE THE COMPLETE NESTED SCV EVIDENCE FOR THE EVALUABLE COHORT
# --------------------------------------------------------------------------------------------------

(
    t0_key_summary,
    t0_scv_count_distribution,
    t0_profile,
) = profile_nested_records(
    t0_evaluable,
    "T0",
)

(
    t1_key_summary,
    t1_scv_count_distribution,
    t1_profile,
) = profile_nested_records(
    t1_evaluable,
    "T1",
)

combined_key_summary = pd.concat(
    [
        t0_key_summary,
        t1_key_summary,
    ],
    ignore_index=True,
)

semantic_candidate_ranking = (
    rank_semantic_candidates(
        combined_key_summary
    )
)

nested_scv_accounting = pd.DataFrame(
    [
        t0_profile[
            "accounting"
        ],
        t1_profile[
            "accounting"
        ],
    ]
)

shared_key_accounting = (
    combined_key_summary.groupby(
        "nested_key"
    )
    .agg(
        timepoints_present=(
            "timepoint",
            "nunique",
        ),
        total_records_with_key=(
            "records_with_key",
            "sum",
        ),
        total_nonempty_records=(
            "records_with_nonempty_value",
            "sum",
        ),
    )
    .reset_index()
)

shared_key_accounting[
    "present_at_both_timepoints"
] = (
    shared_key_accounting[
        "timepoints_present"
    ]
    == 2
)

shared_nested_keys = sorted(
    shared_key_accounting.loc[
        shared_key_accounting[
            "present_at_both_timepoints"
        ],
        "nested_key",
    ].tolist()
)

t0_only_nested_keys = sorted(
    set(
        t0_key_summary[
            "nested_key"
        ]
    )
    - set(
        t1_key_summary[
            "nested_key"
        ]
    )
)

t1_only_nested_keys = sorted(
    set(
        t1_key_summary[
            "nested_key"
        ]
    )
    - set(
        t0_key_summary[
            "nested_key"
        ]
    )
)


# --------------------------------------------------------------------------------------------------
# 7. REQUIRED CANDIDATE-GROUP COVERAGE CHECK
# --------------------------------------------------------------------------------------------------

required_candidate_groups = [
    "scv_identity",
    "submitter_identity",
    "classification",
    "review_rigor",
]

missing_candidate_groups = []

for candidate_group in required_candidate_groups:
    group_rows = semantic_candidate_ranking.loc[
        (
            semantic_candidate_ranking[
                "candidate_group"
            ]
            == candidate_group
        )
        & (
            semantic_candidate_ranking[
                "present_at_both_timepoints"
            ]
        )
    ]

    if group_rows.empty:
        missing_candidate_groups.append(
            candidate_group
        )

if missing_candidate_groups:
    print(
        "\nCOMPLETE COMBINED NESTED KEY SUMMARY"
    )
    print(
        combined_key_summary.to_string(
            index=False
        )
    )

    raise AssertionError(
        "No shared nested-key candidate was found for: "
        + ", ".join(
            missing_candidate_groups
        )
    )


# --------------------------------------------------------------------------------------------------
# 8. DISPLAY PREFLIGHT RESULTS
# --------------------------------------------------------------------------------------------------

separator = "=" * 150

print("\n" + separator)

print(
    "STAGE 6C STEP 3G — CELL 6C-3G0A — "
    "NESTED-SCV DERIVATION PREFLIGHT"
)

print(separator)

print(
    f"Stage 6B evaluable SHA-256         : "
    f"PASS ({observed_evaluable_sha256})"
)

print(
    f"Stage 6B dimensions                : "
    f"PASS ({evaluable_metadata.num_rows:,} × "
    f"{evaluable_metadata.num_columns})"
)

print(
    f"Stage 6B events / negatives        : "
    f"PASS ({EXPECTED_PRIMARY_EVENTS:,} / "
    f"{EXPECTED_PRIMARY_NEGATIVES:,})"
)

print(
    f"Accepted T0 path                   : "
    f"{t0_path}"
)

print(
    f"Accepted T0 SHA-256                : "
    f"PASS ({EXPECTED_T0_SHA256})"
)

print(
    f"Accepted T0 dimensions             : "
    f"PASS ({t0_metadata.num_rows:,} × "
    f"{t0_metadata.num_columns})"
)

print(
    f"Accepted T1 path                   : "
    f"{t1_path}"
)

print(
    f"Accepted T1 SHA-256                : "
    f"PASS ({EXPECTED_T1_SHA256})"
)

print(
    f"Accepted T1 dimensions             : "
    f"PASS ({t1_metadata.num_rows:,} × "
    f"{t1_metadata.num_columns})"
)

print(
    "Evaluable T0 mapping              : "
    f"PASS ({len(t0_evaluable):,}/{EXPECTED_EVALUABLE_ROWS:,})"
)

print(
    "Evaluable T1 mapping              : "
    f"PASS ({len(t1_evaluable):,}/{EXPECTED_EVALUABLE_ROWS:,})"
)

print(
    "GES scores loaded                 : No"
)

print(
    "Derived outcomes created          : No"
)

print(
    "Scientific artifacts written      : No"
)

print(
    "\nT0 ARTIFACT-LOCATION DIAGNOSTICS"
)

print(
    t0_location_diagnostics.to_string(
        index=False
    )
)

print(
    "\nT1 ARTIFACT-LOCATION DIAGNOSTICS"
)

print(
    t1_location_diagnostics.to_string(
        index=False
    )
)

print(
    "\nEVALUABLE NESTED-SCV ACCOUNTING"
)

print(
    nested_scv_accounting.to_string(
        index=False
    )
)

print(
    "\nT0 NESTED-SCV COUNT DISTRIBUTION"
)

print(
    t0_scv_count_distribution.to_string(
        index=False
    )
)

print(
    "\nT1 NESTED-SCV COUNT DISTRIBUTION"
)

print(
    t1_scv_count_distribution.to_string(
        index=False
    )
)

print(
    "\nCOMPLETE T0/T1 NESTED-KEY SUMMARY"
)

print(
    combined_key_summary.to_string(
        index=False
    )
)

print(
    "\nSEMANTIC CANDIDATE RANKING"
)

print(
    semantic_candidate_ranking.to_string(
        index=False
    )
)

print(
    "\nSHARED NESTED KEYS"
)

print(
    shared_nested_keys
)

print(
    "\nT0-ONLY NESTED KEYS"
)

print(
    t0_only_nested_keys
)

print(
    "\nT1-ONLY NESTED KEYS"
)

print(
    t1_only_nested_keys
)

print(
    "\nSCIENTIFIC FREEZE BOUNDARY"
)

print("-" * 150)

print(
    "This preflight intentionally does not define high-rigor, contradiction, "
    "new-submission identity, or submitter-distribution magnitude. Those "
    "definitions must be frozen in the next cell using the exact shared keys "
    "and value vocabularies printed above, before loading any GES score."
)

print(
    "\nCELL DECISION"
)

print("-" * 150)

print(
    "PASS_STAGE6C_NESTED_SCV_DERIVATION_"
    "PREFLIGHT_COMPLETE"
)

print(
    "The accepted T0/T1 artifacts were cryptographically verified, all "
    "66,636 evaluable T0/T1 links were mapped, the complete nested-SCV "
    "schema was profiled, and candidate fields for the two remaining "
    "SCV-derived outcomes were identified without performance analysis."
)

print(
    "No score, outcome, nested evidence, linkage decision, threshold, "
    "weight, cohort membership, or frozen scientific artifact was modified."
)


STAGE 6C STEP 3G — CELL 6C-3G0A — NESTED-SCV DERIVATION PREFLIGHT
Stage 6B evaluable SHA-256         : PASS (c6ad50dfc376a746e1830d000dba975101d88763fd61b8d1983f6300edb6e038)
Stage 6B dimensions                : PASS (66,636 × 79)
Stage 6B events / negatives        : PASS (6,485 / 60,151)
Accepted T0 path                   : /content/drive/MyDrive/GES_RAG_Temporal_Study/data_interim/t0_rcv_target_genes_corrected_v1_2.parquet
Accepted T0 SHA-256                : PASS (f6b6760b2ad6e4352e3bdecdeaf89827e8a514b031abf2373bb17568d999466d)
Accepted T0 dimensions             : PASS (71,659 × 34)
Accepted T1 path                   : /content/drive/MyDrive/GES_RAG_Temporal_Study/data_interim/t1_rcv_target_genes_harmonized_v1.parquet
Accepted T1 SHA-256                : PASS (5713a11bdbf4804758cc011f9b2f302afc91fa1f88c1b178d675c28bb277d37c)
Accepted T1 dimensions             : PASS (100,920 × 36)
Evaluable T0 mapping              : PASS (66,636/66,636)
Evaluable T1 mapping              : PASS (66

In [41]:
# ==================================================================================================
# STAGE 6C STEP 3G — CELL 6C-3G0B
# NESTED-SCV SECONDARY-OUTCOME POLICY FREEZE AND DERIVABILITY AUDIT
#
# Purpose:
#   1. Freshly verify the immutable Stage 6B evaluable cohort and accepted T0/T1 RCV sources.
#   2. Freeze exact, score-blind operational definitions for:
#        a. a new contradictory high-rigor SCV submission;
#        b. magnitude of submitter-classification distribution change.
#   3. Write and freshly reverify only the deterministic policy JSON and SHA-256 sidecar.
#   4. Derive the two outcomes in memory solely to audit whether the accepted nested schema
#      supports them and to determine evaluable/censored accounting.
#   5. Load no GES or comparator score and calculate no predictive-performance result.
#
# Scientific boundary:
#   T1 nested review_status is mostly missing, and T1 nested classification_group may contain
#   the explicit "Missing" sentinel. A negative label is therefore permitted only when the
#   evidence needed to rule out the event is observed. Otherwise the record is censored.
#
# This cell does NOT write a record-level outcome table. That can occur only after this
# policy and its derivability accounting pass.
# ==================================================================================================

from pathlib import Path
from collections import Counter
import hashlib
import json
import os
import re
import tempfile

import numpy as np
import pandas as pd
import pyarrow.parquet as pq


# --------------------------------------------------------------------------------------------------
# 1. FROZEN PATHS, HASHES, AND SCHEMA
# --------------------------------------------------------------------------------------------------

PROJECT_DIR = Path(
    "/content/drive/MyDrive/GES_RAG_Temporal_Study"
)

STAGE6_DIR = (
    PROJECT_DIR
    / "data_processed"
    / "stage6_temporal_validation"
)

EVALUABLE_PARQUET = (
    STAGE6_DIR
    / "stage6b_locked_primary_evaluable_cohort_v1.parquet"
)

EVALUABLE_SIDECAR = Path(
    str(EVALUABLE_PARQUET) + ".sha256"
)

T0_PARQUET = (
    PROJECT_DIR
    / "data_interim"
    / "t0_rcv_target_genes_corrected_v1_2.parquet"
)

T1_PARQUET = (
    PROJECT_DIR
    / "data_interim"
    / "t1_rcv_target_genes_harmonized_v1.parquet"
)

POLICY_DIR = (
    PROJECT_DIR
    / "configs"
    / "stage6c_nested_scv_secondary_outcomes"
)

POLICY_JSON = (
    POLICY_DIR
    / "stage6c_nested_scv_secondary_outcome_policy_v1.json"
)

POLICY_SIDECAR = Path(
    str(POLICY_JSON) + ".sha256"
)

EXPECTED_EVALUABLE_SHA256 = (
    "c6ad50dfc376a746e1830d000dba975101d88763fd61b8d1983f6300edb6e038"
)

EXPECTED_T0_SHA256 = (
    "f6b6760b2ad6e4352e3bdecdeaf89827e8a514b031abf2373bb17568d999466d"
)

EXPECTED_T1_SHA256 = (
    "5713a11bdbf4804758cc011f9b2f302afc91fa1f88c1b178d675c28bb277d37c"
)

EXPECTED_EVALUABLE_ROWS = 66_636
EXPECTED_EVALUABLE_COLUMNS = 79
EXPECTED_T0_ROWS = 71_659
EXPECTED_T0_COLUMNS = 34
EXPECTED_T1_ROWS = 100_920
EXPECTED_T1_COLUMNS = 36

STAGE6_T0_RCV_COLUMN = "rcv_accession"
STAGE6_T1_RCV_COLUMN = "linked_t1_rcv_accession"
STAGE6_ROW_ORDER_COLUMN = "t0_row_order"
STAGE6_BASELINE_GROUP_COLUMN = "t0_canonical_classification_group"

SOURCE_RCV_COLUMN = "rcv_accession"
NESTED_SCV_COLUMN = "scv_records_json"

SCV_ID_KEY = "scv_accession"
SUBMITTER_KEY = "submitter"
CLASSIFICATION_GROUP_KEY = "classification_group"
REVIEW_STATUS_KEY = "review_status"

INFORMATIVE_GROUPS = ("BLB", "VUS", "PLP")
MAJOR_DISTRIBUTION_SHIFT_THRESHOLD = 0.50


# --------------------------------------------------------------------------------------------------
# 2. HELPERS
# --------------------------------------------------------------------------------------------------

def sha256_file(
    path: Path,
    chunk_size: int = 1024 * 1024,
) -> str:
    digest = hashlib.sha256()

    with path.open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


def sha256_bytes(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()


def read_sidecar_hash(path: Path) -> str:
    text = path.read_text(
        encoding="utf-8"
    ).strip()

    matches = re.findall(
        r"\b[a-fA-F0-9]{64}\b",
        text,
    )

    if not matches:
        raise ValueError(
            f"No SHA-256 value found in sidecar: {path}"
        )

    return matches[0].lower()


def normalize_rcv(
    series: pd.Series,
) -> pd.Series:
    return (
        series.astype("string")
        .str.upper()
        .str.strip()
        .str.extract(
            r"(RCV\d+)",
            expand=False,
        )
    )


def normalize_scv_accession(
    value,
) -> str | None:
    if value is None:
        return None

    text = str(value).upper().strip()

    match = re.search(
        r"(SCV\d+)",
        text,
    )

    if not match:
        return None

    return match.group(1)


def normalize_text(
    value,
) -> str | None:
    if value is None:
        return None

    text = re.sub(
        r"\s+",
        " ",
        str(value),
    ).strip()

    if text == "":
        return None

    return text


def normalize_review_status(
    value,
) -> str | None:
    text = normalize_text(value)

    if text is None:
        return None

    return text.lower()


def is_high_rigor_review_status(
    normalized_status: str | None,
) -> bool:
    if normalized_status is None:
        return False

    # Exclusion precedence is explicit so that
    # "no assertion criteria provided" is never misclassified merely
    # because it contains the phrase "criteria provided".
    if (
        normalized_status.startswith(
            "no assertion criteria"
        )
        or normalized_status.startswith(
            "no assertion provided"
        )
    ):
        return False

    return (
        normalized_status.startswith(
            "criteria provided"
        )
        or "reviewed by expert panel"
        in normalized_status
        or "practice guideline"
        in normalized_status
    )


def normalize_classification_group(
    value,
) -> str:
    text = normalize_text(value)

    if text is None:
        return "MISSING"

    normalized = (
        text.lower()
        .replace("_", " ")
        .replace("-", " ")
    )

    normalized = re.sub(
        r"\s+",
        " ",
        normalized,
    ).strip()

    if normalized in {
        "blb",
        "benign",
        "likely benign",
        "benign/likely benign",
        "benign likely benign",
    }:
        return "BLB"

    if normalized in {
        "vus",
        "uncertain significance",
        "uncertain",
    }:
        return "VUS"

    if normalized in {
        "plp",
        "pathogenic",
        "likely pathogenic",
        "pathogenic/likely pathogenic",
        "pathogenic likely pathogenic",
    }:
        return "PLP"

    if normalized in {
        "missing",
        "none",
        "nan",
        "not provided",
        "no classification",
        "noclassification",
    }:
        return "MISSING"

    if "conflict" in normalized:
        return "CONFLICTING"

    return "OTHER"


def parse_nested_scv_value(
    value,
    row_label: str,
) -> list[dict]:
    if isinstance(value, list):
        parsed = value

    elif isinstance(value, tuple):
        parsed = list(value)

    elif isinstance(value, np.ndarray):
        parsed = value.tolist()

    elif isinstance(value, str):
        text = value.strip()

        if text == "":
            raise ValueError(
                f"Blank nested SCV JSON at {row_label}"
            )

        try:
            parsed = json.loads(text)
        except Exception as error:
            raise ValueError(
                f"Malformed nested SCV JSON at "
                f"{row_label}: {error}"
            ) from error

    else:
        try:
            if pd.isna(value):
                raise ValueError(
                    f"Missing nested SCV evidence at {row_label}"
                )
        except Exception:
            pass

        raise TypeError(
            f"Unsupported nested SCV type at "
            f"{row_label}: {type(value).__name__}"
        )

    if not isinstance(parsed, list):
        raise TypeError(
            f"Nested SCV value is not a list at "
            f"{row_label}: {type(parsed).__name__}"
        )

    for position, record in enumerate(parsed):
        if not isinstance(record, dict):
            raise TypeError(
                f"Nested SCV record is not a dictionary "
                f"at {row_label}, position {position}."
            )

    return parsed


def canonical_json_bytes(
    value: dict,
) -> bytes:
    text = json.dumps(
        value,
        ensure_ascii=False,
        sort_keys=True,
        indent=2,
    ) + "\n"

    return text.encode("utf-8")


def immutable_write_or_verify(
    path: Path,
    expected_bytes: bytes,
) -> str:
    expected_hash = sha256_bytes(
        expected_bytes
    )

    if path.exists():
        observed_bytes = path.read_bytes()

        if observed_bytes != expected_bytes:
            raise FileExistsError(
                f"An existing immutable artifact differs "
                f"from the expected content:\n{path}"
            )

        observed_hash = sha256_bytes(
            observed_bytes
        )

        if observed_hash != expected_hash:
            raise AssertionError(
                f"Existing artifact hash mismatch:\n{path}"
            )

        return observed_hash

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    with tempfile.NamedTemporaryFile(
        mode="wb",
        dir=str(path.parent),
        prefix=path.name + ".tmp.",
        delete=False,
    ) as temporary:
        temporary.write(
            expected_bytes
        )
        temporary.flush()
        os.fsync(
            temporary.fileno()
        )
        temporary_path = Path(
            temporary.name
        )

    try:
        os.replace(
            temporary_path,
            path,
        )
    finally:
        if temporary_path.exists():
            temporary_path.unlink()

    observed_hash = sha256_file(
        path
    )

    if observed_hash != expected_hash:
        raise AssertionError(
            f"Fresh artifact write/readback mismatch:\n{path}"
        )

    return observed_hash


def distribution_vector(
    normalized_groups: list[str],
) -> np.ndarray:
    counts = Counter(
        normalized_groups
    )

    total = sum(
        counts[group]
        for group in INFORMATIVE_GROUPS
    )

    if total <= 0:
        raise ValueError(
            "Cannot construct an informative distribution "
            "with zero BLB/VUS/PLP SCVs."
        )

    return np.array(
        [
            counts[group] / total
            for group in INFORMATIVE_GROUPS
        ],
        dtype=float,
    )


def total_variation_distance(
    first: np.ndarray,
    second: np.ndarray,
) -> float:
    return float(
        0.5
        * np.abs(
            np.asarray(first, dtype=float)
            - np.asarray(second, dtype=float)
        ).sum()
    )


# --------------------------------------------------------------------------------------------------
# 3. CRYPTOGRAPHIC AND STRUCTURAL VERIFICATION
# --------------------------------------------------------------------------------------------------

for path in [
    EVALUABLE_PARQUET,
    EVALUABLE_SIDECAR,
    T0_PARQUET,
    T1_PARQUET,
]:
    if not path.exists():
        raise FileNotFoundError(
            f"Missing required frozen artifact:\n{path}"
        )

observed_evaluable_sha256 = sha256_file(
    EVALUABLE_PARQUET
)

if (
    observed_evaluable_sha256
    != EXPECTED_EVALUABLE_SHA256
):
    raise AssertionError(
        "Stage 6B evaluable SHA-256 mismatch."
    )

if (
    read_sidecar_hash(
        EVALUABLE_SIDECAR
    )
    != observed_evaluable_sha256
):
    raise AssertionError(
        "Stage 6B evaluable sidecar mismatch."
    )

observed_t0_sha256 = sha256_file(
    T0_PARQUET
)

observed_t1_sha256 = sha256_file(
    T1_PARQUET
)

if observed_t0_sha256 != EXPECTED_T0_SHA256:
    raise AssertionError(
        "Accepted T0 SHA-256 mismatch."
    )

if observed_t1_sha256 != EXPECTED_T1_SHA256:
    raise AssertionError(
        "Accepted T1 SHA-256 mismatch."
    )

stage6_file = pq.ParquetFile(
    EVALUABLE_PARQUET
)

t0_file = pq.ParquetFile(
    T0_PARQUET
)

t1_file = pq.ParquetFile(
    T1_PARQUET
)

if (
    stage6_file.metadata.num_rows
    != EXPECTED_EVALUABLE_ROWS
    or stage6_file.metadata.num_columns
    != EXPECTED_EVALUABLE_COLUMNS
):
    raise AssertionError(
        "Stage 6B dimensions mismatch."
    )

if (
    t0_file.metadata.num_rows
    != EXPECTED_T0_ROWS
    or t0_file.metadata.num_columns
    != EXPECTED_T0_COLUMNS
):
    raise AssertionError(
        "Accepted T0 dimensions mismatch."
    )

if (
    t1_file.metadata.num_rows
    != EXPECTED_T1_ROWS
    or t1_file.metadata.num_columns
    != EXPECTED_T1_COLUMNS
):
    raise AssertionError(
        "Accepted T1 dimensions mismatch."
    )

required_stage6_columns = [
    STAGE6_T0_RCV_COLUMN,
    STAGE6_T1_RCV_COLUMN,
    STAGE6_ROW_ORDER_COLUMN,
    STAGE6_BASELINE_GROUP_COLUMN,
]

missing_stage6_columns = [
    column
    for column in required_stage6_columns
    if column
    not in stage6_file.schema_arrow.names
]

if missing_stage6_columns:
    raise KeyError(
        "Missing required Stage 6B columns:\n"
        + "\n".join(
            missing_stage6_columns
        )
    )

for label, schema in [
    (
        "T0",
        t0_file.schema_arrow.names,
    ),
    (
        "T1",
        t1_file.schema_arrow.names,
    ),
]:
    missing = [
        column
        for column in [
            SOURCE_RCV_COLUMN,
            NESTED_SCV_COLUMN,
        ]
        if column not in schema
    ]

    if missing:
        raise KeyError(
            f"{label} source is missing:\n"
            + "\n".join(missing)
        )


# --------------------------------------------------------------------------------------------------
# 4. CREATE AND FREEZE THE SCORE-BLIND OPERATIONAL POLICY
# --------------------------------------------------------------------------------------------------

policy = {
    "policy_name": (
        "stage6c_nested_scv_secondary_outcome_policy"
    ),
    "policy_version": "1.0",
    "status": (
        "FROZEN_BEFORE_NESTED_SCV_SCORE_ANALYSIS"
    ),
    "study_unit": (
        "RCV-level variant-condition aggregate record"
    ),
    "accepted_sources": {
        "stage6b_primary_evaluable_cohort": {
            "filename": EVALUABLE_PARQUET.name,
            "sha256": EXPECTED_EVALUABLE_SHA256,
            "rows": EXPECTED_EVALUABLE_ROWS,
            "columns": EXPECTED_EVALUABLE_COLUMNS,
        },
        "t0_rcv_source": {
            "filename": T0_PARQUET.name,
            "sha256": EXPECTED_T0_SHA256,
            "rows": EXPECTED_T0_ROWS,
            "columns": EXPECTED_T0_COLUMNS,
        },
        "t1_rcv_source": {
            "filename": T1_PARQUET.name,
            "sha256": EXPECTED_T1_SHA256,
            "rows": EXPECTED_T1_ROWS,
            "columns": EXPECTED_T1_COLUMNS,
        },
    },
    "nested_keys": {
        "scv_identity": SCV_ID_KEY,
        "submitter_identity": SUBMITTER_KEY,
        "classification_group": CLASSIFICATION_GROUP_KEY,
        "review_rigor": REVIEW_STATUS_KEY,
    },
    "classification_group_mapping": {
        "BLB": [
            "Benign",
            "Likely benign",
            "Benign/Likely benign",
        ],
        "VUS": [
            "Uncertain significance",
            "VUS",
        ],
        "PLP": [
            "Pathogenic",
            "Likely pathogenic",
            "Pathogenic/Likely pathogenic",
        ],
        "MISSING": [
            "Missing",
            "NoClassification",
            "null/blank",
        ],
        "CONFLICTING": [
            "values containing conflict",
        ],
        "OTHER": [
            "all remaining values",
        ],
    },
    "high_rigor_rule": {
        "positive_review_status_logic": [
            "normalized status starts with 'criteria provided'",
            "normalized status contains 'reviewed by expert panel'",
            "normalized status contains 'practice guideline'",
        ],
        "explicit_exclusion_precedence": [
            "normalized status starts with 'no assertion criteria'",
            "normalized status starts with 'no assertion provided'",
        ],
        "missing_review_status": (
            "unknown, never assumed low-rigor"
        ),
    },
    "new_contradictory_high_rigor_submission": {
        "new_submission": (
            "normalized T1 SCV accession absent from the "
            "same linked RCV's T0 SCV-accession set"
        ),
        "baseline_interpretation": (
            "frozen Stage 6B "
            "t0_canonical_classification_group"
        ),
        "contradiction": (
            "new T1 SCV normalized group is BLB, VUS, or PLP "
            "and differs from the informative T0 baseline group"
        ),
        "positive": (
            "at least one new T1 SCV is high-rigor and contradictory"
        ),
        "negative": (
            "no new T1 SCV exists, or every new T1 SCV can be "
            "fully evaluated and none is both high-rigor and contradictory"
        ),
        "censored": (
            "no positive event is observed and at least one new T1 SCV "
            "has missing review status, or a high-rigor new SCV has a "
            "noninformative classification group"
        ),
        "updated_existing_scv": (
            "an existing SCV accession with a later version is not a "
            "new submission and is excluded from this outcome"
        ),
    },
    "submitter_classification_distribution_change": {
        "distribution_unit": (
            "one nested SCV classification assertion"
        ),
        "categories": list(
            INFORMATIVE_GROUPS
        ),
        "complete_case_evaluability": (
            "every nested SCV at both T0 and T1 must map to BLB, VUS, "
            "or PLP; otherwise the distribution magnitude is censored"
        ),
        "magnitude": (
            "total variation distance = 0.5 * sum(abs(p_T1 - p_T0))"
        ),
        "range": [
            0.0,
            1.0,
        ],
        "major_shift_binary_sensitivity": (
            "total variation distance >= 0.50"
        ),
        "major_shift_threshold": (
            MAJOR_DISTRIBUTION_SHIFT_THRESHOLD
        ),
        "threshold_rationale": (
            "a fixed, interpretable threshold requiring at least half "
            "of classification probability mass to move; not selected "
            "from GES performance"
        ),
    },
    "analysis_role": {
        "new_contradictory_high_rigor_submission": (
            "exploratory secondary evidence-drift outcome"
        ),
        "distribution_magnitude": (
            "exploratory continuous secondary evidence-drift measure"
        ),
        "major_distribution_shift": (
            "exploratory binary sensitivity outcome"
        ),
    },
    "leakage_protection": {
        "ges_or_comparator_scores_loaded_during_policy_freeze": False,
        "threshold_selected_from_performance": False,
        "outcome_definition_selected_from_performance": False,
    },
}

policy_bytes = canonical_json_bytes(
    policy
)

policy_sha256 = immutable_write_or_verify(
    POLICY_JSON,
    policy_bytes,
)

sidecar_bytes = (
    f"{policy_sha256}  {POLICY_JSON.name}\n"
).encode("utf-8")

sidecar_sha256 = immutable_write_or_verify(
    POLICY_SIDECAR,
    sidecar_bytes,
)

if sha256_file(
    POLICY_JSON
) != policy_sha256:
    raise AssertionError(
        "Policy fresh readback verification failed."
    )

if read_sidecar_hash(
    POLICY_SIDECAR
) != policy_sha256:
    raise AssertionError(
        "Policy sidecar fresh readback verification failed."
    )


# --------------------------------------------------------------------------------------------------
# 5. LOAD ONLY IDENTIFIERS, BASELINE GROUP, AND NESTED EVIDENCE
# --------------------------------------------------------------------------------------------------

stage6 = pd.read_parquet(
    EVALUABLE_PARQUET,
    columns=required_stage6_columns,
).copy()

stage6[
    STAGE6_T0_RCV_COLUMN
] = normalize_rcv(
    stage6[
        STAGE6_T0_RCV_COLUMN
    ]
)

stage6[
    STAGE6_T1_RCV_COLUMN
] = normalize_rcv(
    stage6[
        STAGE6_T1_RCV_COLUMN
    ]
)

stage6[
    STAGE6_BASELINE_GROUP_COLUMN
] = (
    stage6[
        STAGE6_BASELINE_GROUP_COLUMN
    ]
    .map(
        normalize_classification_group
    )
)

if stage6[
    STAGE6_T0_RCV_COLUMN
].isna().any():
    raise AssertionError(
        "A Stage 6B T0 RCV failed normalization."
    )

if stage6[
    STAGE6_T1_RCV_COLUMN
].isna().any():
    raise AssertionError(
        "A Stage 6B T1 RCV failed normalization."
    )

if (
    stage6[
        STAGE6_T0_RCV_COLUMN
    ].nunique()
    != EXPECTED_EVALUABLE_ROWS
):
    raise AssertionError(
        "Stage 6B T0 RCV keys are not unique."
    )

if (
    stage6[
        STAGE6_T1_RCV_COLUMN
    ].nunique()
    != EXPECTED_EVALUABLE_ROWS
):
    raise AssertionError(
        "Stage 6B linked T1 RCV keys are not unique."
    )

if not stage6[
    STAGE6_BASELINE_GROUP_COLUMN
].isin(
    INFORMATIVE_GROUPS
).all():
    baseline_counts = (
        stage6[
            STAGE6_BASELINE_GROUP_COLUMN
        ]
        .value_counts(
            dropna=False
        )
    )

    print(
        "\nBASELINE GROUP INVENTORY"
    )

    print(
        baseline_counts.to_string()
    )

    raise AssertionError(
        "A primary-evaluable record has a "
        "noninformative T0 baseline group."
    )

t0_source = pd.read_parquet(
    T0_PARQUET,
    columns=[
        SOURCE_RCV_COLUMN,
        NESTED_SCV_COLUMN,
    ],
).copy()

t1_source = pd.read_parquet(
    T1_PARQUET,
    columns=[
        SOURCE_RCV_COLUMN,
        NESTED_SCV_COLUMN,
    ],
).copy()

t0_source[
    SOURCE_RCV_COLUMN
] = normalize_rcv(
    t0_source[
        SOURCE_RCV_COLUMN
    ]
)

t1_source[
    SOURCE_RCV_COLUMN
] = normalize_rcv(
    t1_source[
        SOURCE_RCV_COLUMN
    ]
)

if (
    t0_source[
        SOURCE_RCV_COLUMN
    ].nunique()
    != EXPECTED_T0_ROWS
):
    raise AssertionError(
        "T0 source RCV keys are not unique."
    )

if (
    t1_source[
        SOURCE_RCV_COLUMN
    ].nunique()
    != EXPECTED_T1_ROWS
):
    raise AssertionError(
        "T1 source RCV keys are not unique."
    )

mapped = (
    stage6.merge(
        t0_source.rename(
            columns={
                SOURCE_RCV_COLUMN: (
                    "_t0_source_rcv"
                ),
                NESTED_SCV_COLUMN: (
                    "_t0_scv_records_json"
                ),
            }
        ),
        left_on=STAGE6_T0_RCV_COLUMN,
        right_on="_t0_source_rcv",
        how="left",
        validate="one_to_one",
        indicator="_t0_merge",
    )
    .merge(
        t1_source.rename(
            columns={
                SOURCE_RCV_COLUMN: (
                    "_t1_source_rcv"
                ),
                NESTED_SCV_COLUMN: (
                    "_t1_scv_records_json"
                ),
            }
        ),
        left_on=STAGE6_T1_RCV_COLUMN,
        right_on="_t1_source_rcv",
        how="left",
        validate="one_to_one",
        indicator="_t1_merge",
    )
    .sort_values(
        STAGE6_ROW_ORDER_COLUMN,
        kind="mergesort",
    )
    .reset_index(drop=True)
)

if not mapped[
    "_t0_merge"
].eq("both").all():
    raise AssertionError(
        "An evaluable T0 RCV failed source mapping."
    )

if not mapped[
    "_t1_merge"
].eq("both").all():
    raise AssertionError(
        "An evaluable T1 RCV failed source mapping."
    )

if len(mapped) != EXPECTED_EVALUABLE_ROWS:
    raise AssertionError(
        "Mapped evaluable row count mismatch."
    )


# --------------------------------------------------------------------------------------------------
# 6. SCORE-BLIND IN-MEMORY DERIVATION AND DERIVABILITY AUDIT
# --------------------------------------------------------------------------------------------------

t0_review_status_counts = Counter()
t1_review_status_counts = Counter()
t0_group_counts = Counter()
t1_group_counts = Counter()

record_rows = []

for row in mapped.itertuples(index=False):
    t0_rcv = getattr(
        row,
        STAGE6_T0_RCV_COLUMN,
    )

    t1_rcv = getattr(
        row,
        STAGE6_T1_RCV_COLUMN,
    )

    baseline_group = getattr(
        row,
        STAGE6_BASELINE_GROUP_COLUMN,
    )

    t0_records = parse_nested_scv_value(
        getattr(
            row,
            "_t0_scv_records_json",
        ),
        f"T0:{t0_rcv}",
    )

    t1_records = parse_nested_scv_value(
        getattr(
            row,
            "_t1_scv_records_json",
        ),
        f"T1:{t1_rcv}",
    )

    t0_accessions = set()

    t0_normalized_groups = []
    t1_normalized_groups = []

    for record in t0_records:
        accession = normalize_scv_accession(
            record.get(
                SCV_ID_KEY
            )
        )

        if accession is None:
            raise AssertionError(
                f"T0 nested SCV lacks a valid accession: "
                f"{t0_rcv}"
            )

        if accession in t0_accessions:
            raise AssertionError(
                f"Duplicate T0 SCV accession within RCV "
                f"{t0_rcv}: {accession}"
            )

        t0_accessions.add(
            accession
        )

        status = normalize_review_status(
            record.get(
                REVIEW_STATUS_KEY
            )
        )

        t0_review_status_counts[
            status
            if status is not None
            else "<MISSING>"
        ] += 1

        group = normalize_classification_group(
            record.get(
                CLASSIFICATION_GROUP_KEY
            )
        )

        t0_group_counts[
            group
        ] += 1

        t0_normalized_groups.append(
            group
        )

    t1_accessions = set()

    new_scv_count = 0
    new_scv_missing_review_status = 0
    new_high_rigor_count = 0
    new_high_rigor_noninformative_group = 0
    new_high_rigor_contradictory_count = 0
    new_high_rigor_concordant_count = 0
    updated_existing_scv_count = 0

    t0_version_by_accession = {
        normalize_scv_accession(
            record.get(
                SCV_ID_KEY
            )
        ): str(
            record.get(
                "scv_version"
            )
        )
        for record in t0_records
    }

    for record in t1_records:
        accession = normalize_scv_accession(
            record.get(
                SCV_ID_KEY
            )
        )

        if accession is None:
            raise AssertionError(
                f"T1 nested SCV lacks a valid accession: "
                f"{t1_rcv}"
            )

        if accession in t1_accessions:
            raise AssertionError(
                f"Duplicate T1 SCV accession within RCV "
                f"{t1_rcv}: {accession}"
            )

        t1_accessions.add(
            accession
        )

        status = normalize_review_status(
            record.get(
                REVIEW_STATUS_KEY
            )
        )

        t1_review_status_counts[
            status
            if status is not None
            else "<MISSING>"
        ] += 1

        group = normalize_classification_group(
            record.get(
                CLASSIFICATION_GROUP_KEY
            )
        )

        t1_group_counts[
            group
        ] += 1

        t1_normalized_groups.append(
            group
        )

        if accession in t0_accessions:
            t0_version = t0_version_by_accession.get(
                accession
            )

            t1_version = str(
                record.get(
                    "scv_version"
                )
            )

            if t0_version != t1_version:
                updated_existing_scv_count += 1

            continue

        new_scv_count += 1

        if status is None:
            new_scv_missing_review_status += 1
            continue

        if not is_high_rigor_review_status(
            status
        ):
            continue

        new_high_rigor_count += 1

        if group not in INFORMATIVE_GROUPS:
            new_high_rigor_noninformative_group += 1
            continue

        if group != baseline_group:
            new_high_rigor_contradictory_count += 1
        else:
            new_high_rigor_concordant_count += 1

    if (
        new_high_rigor_contradictory_count
        > 0
    ):
        high_rigor_outcome_status = "POSITIVE"
        high_rigor_outcome = 1

    elif (
        new_scv_missing_review_status
        > 0
        or new_high_rigor_noninformative_group
        > 0
    ):
        high_rigor_outcome_status = (
            "CENSORED_REQUIRED_NESTED_FIELD_MISSING"
        )
        high_rigor_outcome = pd.NA

    else:
        high_rigor_outcome_status = "NEGATIVE"
        high_rigor_outcome = 0

    t0_complete_distribution = all(
        group in INFORMATIVE_GROUPS
        for group in t0_normalized_groups
    )

    t1_complete_distribution = all(
        group in INFORMATIVE_GROUPS
        for group in t1_normalized_groups
    )

    distribution_evaluable = (
        t0_complete_distribution
        and t1_complete_distribution
        and len(
            t0_normalized_groups
        ) > 0
        and len(
            t1_normalized_groups
        ) > 0
    )

    if distribution_evaluable:
        t0_distribution = distribution_vector(
            t0_normalized_groups
        )

        t1_distribution = distribution_vector(
            t1_normalized_groups
        )

        distribution_tvd = total_variation_distance(
            t0_distribution,
            t1_distribution,
        )

        major_distribution_shift = int(
            distribution_tvd
            >= MAJOR_DISTRIBUTION_SHIFT_THRESHOLD
        )

        distribution_status = "EVALUABLE"

    else:
        distribution_tvd = np.nan
        major_distribution_shift = pd.NA
        distribution_status = (
            "CENSORED_NONINFORMATIVE_NESTED_CLASSIFICATION"
        )

    record_rows.append(
        {
            "t0_rcv_accession": t0_rcv,
            "linked_t1_rcv_accession": t1_rcv,
            "t0_row_order": getattr(
                row,
                STAGE6_ROW_ORDER_COLUMN,
            ),
            "t0_baseline_group": baseline_group,
            "t0_scv_count": len(
                t0_records
            ),
            "t1_scv_count": len(
                t1_records
            ),
            "new_scv_count": new_scv_count,
            "updated_existing_scv_count": (
                updated_existing_scv_count
            ),
            "new_scv_missing_review_status": (
                new_scv_missing_review_status
            ),
            "new_high_rigor_scv_count": (
                new_high_rigor_count
            ),
            "new_high_rigor_noninformative_group_count": (
                new_high_rigor_noninformative_group
            ),
            "new_high_rigor_concordant_count": (
                new_high_rigor_concordant_count
            ),
            "new_high_rigor_contradictory_count": (
                new_high_rigor_contradictory_count
            ),
            "new_contradictory_high_rigor_status": (
                high_rigor_outcome_status
            ),
            "new_contradictory_high_rigor_submission": (
                high_rigor_outcome
            ),
            "distribution_status": (
                distribution_status
            ),
            "submitter_classification_tvd": (
                distribution_tvd
            ),
            "major_submitter_distribution_shift": (
                major_distribution_shift
            ),
        }
    )

derived_audit = pd.DataFrame(
    record_rows
)

if len(derived_audit) != EXPECTED_EVALUABLE_ROWS:
    raise AssertionError(
        "In-memory derivation row count mismatch."
    )

if (
    derived_audit[
        "t0_rcv_accession"
    ].nunique()
    != EXPECTED_EVALUABLE_ROWS
):
    raise AssertionError(
        "In-memory derivation T0 keys are not unique."
    )

if not np.all(
    np.diff(
        derived_audit[
            "t0_row_order"
        ].to_numpy()
    ) > 0
):
    raise AssertionError(
        "In-memory derivation did not preserve "
        "frozen T0 row order."
    )


# --------------------------------------------------------------------------------------------------
# 7. AUDIT TABLES
# --------------------------------------------------------------------------------------------------

nested_vocabulary = pd.concat(
    [
        pd.DataFrame(
            {
                "timepoint": "T0",
                "field": REVIEW_STATUS_KEY,
                "value": list(
                    t0_review_status_counts.keys()
                ),
                "nested_records": list(
                    t0_review_status_counts.values()
                ),
            }
        ),
        pd.DataFrame(
            {
                "timepoint": "T1",
                "field": REVIEW_STATUS_KEY,
                "value": list(
                    t1_review_status_counts.keys()
                ),
                "nested_records": list(
                    t1_review_status_counts.values()
                ),
            }
        ),
        pd.DataFrame(
            {
                "timepoint": "T0",
                "field": CLASSIFICATION_GROUP_KEY,
                "value": list(
                    t0_group_counts.keys()
                ),
                "nested_records": list(
                    t0_group_counts.values()
                ),
            }
        ),
        pd.DataFrame(
            {
                "timepoint": "T1",
                "field": CLASSIFICATION_GROUP_KEY,
                "value": list(
                    t1_group_counts.keys()
                ),
                "nested_records": list(
                    t1_group_counts.values()
                ),
            }
        ),
    ],
    ignore_index=True,
).sort_values(
    [
        "field",
        "timepoint",
        "nested_records",
        "value",
    ],
    ascending=[
        True,
        True,
        False,
        True,
    ],
).reset_index(drop=True)

high_rigor_status_accounting = (
    derived_audit[
        "new_contradictory_high_rigor_status"
    ]
    .value_counts(
        dropna=False
    )
    .rename_axis(
        "outcome_status"
    )
    .reset_index(
        name="rcv_rows"
    )
)

distribution_status_accounting = (
    derived_audit[
        "distribution_status"
    ]
    .value_counts(
        dropna=False
    )
    .rename_axis(
        "distribution_status"
    )
    .reset_index(
        name="rcv_rows"
    )
)

scv_transition_summary = pd.DataFrame(
    [
        {
            "measure": "RCVs with at least one new SCV",
            "rcv_rows": int(
                (
                    derived_audit[
                        "new_scv_count"
                    ] > 0
                ).sum()
            ),
            "nested_scv_total": int(
                derived_audit[
                    "new_scv_count"
                ].sum()
            ),
        },
        {
            "measure": (
                "RCVs with a new SCV missing review_status"
            ),
            "rcv_rows": int(
                (
                    derived_audit[
                        "new_scv_missing_review_status"
                    ] > 0
                ).sum()
            ),
            "nested_scv_total": int(
                derived_audit[
                    "new_scv_missing_review_status"
                ].sum()
            ),
        },
        {
            "measure": (
                "RCVs with at least one new high-rigor SCV"
            ),
            "rcv_rows": int(
                (
                    derived_audit[
                        "new_high_rigor_scv_count"
                    ] > 0
                ).sum()
            ),
            "nested_scv_total": int(
                derived_audit[
                    "new_high_rigor_scv_count"
                ].sum()
            ),
        },
        {
            "measure": (
                "RCVs with at least one new contradictory high-rigor SCV"
            ),
            "rcv_rows": int(
                (
                    derived_audit[
                        "new_high_rigor_contradictory_count"
                    ] > 0
                ).sum()
            ),
            "nested_scv_total": int(
                derived_audit[
                    "new_high_rigor_contradictory_count"
                ].sum()
            ),
        },
        {
            "measure": (
                "RCVs with an updated existing SCV version"
            ),
            "rcv_rows": int(
                (
                    derived_audit[
                        "updated_existing_scv_count"
                    ] > 0
                ).sum()
            ),
            "nested_scv_total": int(
                derived_audit[
                    "updated_existing_scv_count"
                ].sum()
            ),
        },
    ]
)

distribution_evaluable = derived_audit.loc[
    derived_audit[
        "distribution_status"
    ] == "EVALUABLE"
].copy()

if len(distribution_evaluable):
    tvd_summary = pd.DataFrame(
        [
            {
                "evaluable_rcv_rows": len(
                    distribution_evaluable
                ),
                "mean_tvd": float(
                    distribution_evaluable[
                        "submitter_classification_tvd"
                    ].mean()
                ),
                "median_tvd": float(
                    distribution_evaluable[
                        "submitter_classification_tvd"
                    ].median()
                ),
                "minimum_tvd": float(
                    distribution_evaluable[
                        "submitter_classification_tvd"
                    ].min()
                ),
                "maximum_tvd": float(
                    distribution_evaluable[
                        "submitter_classification_tvd"
                    ].max()
                ),
                "major_shift_threshold": (
                    MAJOR_DISTRIBUTION_SHIFT_THRESHOLD
                ),
                "major_shift_events": int(
                    distribution_evaluable[
                        "major_submitter_distribution_shift"
                    ].astype(int).sum()
                ),
                "major_shift_negatives": int(
                    len(
                        distribution_evaluable
                    )
                    - distribution_evaluable[
                        "major_submitter_distribution_shift"
                    ].astype(int).sum()
                ),
            }
        ]
    )

    tvd_bins = pd.cut(
        distribution_evaluable[
            "submitter_classification_tvd"
        ],
        bins=[
            -1e-12,
            0.0,
            0.25,
            0.50,
            0.75,
            1.0,
        ],
        labels=[
            "0",
            "(0,0.25]",
            "(0.25,0.50]",
            "(0.50,0.75]",
            "(0.75,1.00]",
        ],
        include_lowest=True,
    )

    tvd_distribution = (
        tvd_bins.value_counts(
            sort=False,
            dropna=False,
        )
        .rename_axis(
            "tvd_interval"
        )
        .reset_index(
            name="rcv_rows"
        )
    )

else:
    tvd_summary = pd.DataFrame(
        [
            {
                "evaluable_rcv_rows": 0,
                "mean_tvd": np.nan,
                "median_tvd": np.nan,
                "minimum_tvd": np.nan,
                "maximum_tvd": np.nan,
                "major_shift_threshold": (
                    MAJOR_DISTRIBUTION_SHIFT_THRESHOLD
                ),
                "major_shift_events": 0,
                "major_shift_negatives": 0,
            }
        ]
    )

    tvd_distribution = pd.DataFrame(
        columns=[
            "tvd_interval",
            "rcv_rows",
        ]
    )


# --------------------------------------------------------------------------------------------------
# 8. FINAL SCIENTIFIC DECISION LOGIC
# --------------------------------------------------------------------------------------------------

high_rigor_positive_count = int(
    (
        derived_audit[
            "new_contradictory_high_rigor_status"
        ]
        == "POSITIVE"
    ).sum()
)

high_rigor_negative_count = int(
    (
        derived_audit[
            "new_contradictory_high_rigor_status"
        ]
        == "NEGATIVE"
    ).sum()
)

high_rigor_censored_count = int(
    (
        derived_audit[
            "new_contradictory_high_rigor_status"
        ]
        .str.startswith(
            "CENSORED",
            na=False,
        )
    ).sum()
)

high_rigor_binary_estimable = (
    high_rigor_positive_count > 0
    and high_rigor_negative_count > 0
)

distribution_binary_estimable = (
    len(
        distribution_evaluable
    ) > 0
    and distribution_evaluable[
        "major_submitter_distribution_shift"
    ].astype(int).nunique() == 2
)

distribution_continuous_estimable = (
    len(
        distribution_evaluable
    ) >= 50
    and distribution_evaluable[
        "submitter_classification_tvd"
    ].nunique() >= 2
)

derivability_decision = pd.DataFrame(
    [
        {
            "target": (
                "new_contradictory_high_rigor_submission"
            ),
            "requested_analysis_form": (
                "binary secondary outcome"
            ),
            "evaluable_rows": (
                high_rigor_positive_count
                + high_rigor_negative_count
            ),
            "positive_events": (
                high_rigor_positive_count
            ),
            "negative_rows": (
                high_rigor_negative_count
            ),
            "censored_rows": (
                high_rigor_censored_count
            ),
            "estimable_for_discrimination": (
                high_rigor_binary_estimable
            ),
            "decision": (
                "ESTIMABLE"
                if high_rigor_binary_estimable
                else (
                    "NOT_ESTIMABLE_NO_POSITIVE_EVENTS"
                    if high_rigor_positive_count == 0
                    else "NOT_ESTIMABLE_INSUFFICIENT_CLASS_VARIATION"
                )
            ),
        },
        {
            "target": (
                "major_submitter_distribution_shift"
            ),
            "requested_analysis_form": (
                "binary sensitivity outcome, TVD >= 0.50"
            ),
            "evaluable_rows": len(
                distribution_evaluable
            ),
            "positive_events": int(
                distribution_evaluable[
                    "major_submitter_distribution_shift"
                ].fillna(0).astype(int).sum()
            ),
            "negative_rows": int(
                len(
                    distribution_evaluable
                )
                - distribution_evaluable[
                    "major_submitter_distribution_shift"
                ].fillna(0).astype(int).sum()
            ),
            "censored_rows": int(
                EXPECTED_EVALUABLE_ROWS
                - len(
                    distribution_evaluable
                )
            ),
            "estimable_for_discrimination": (
                distribution_binary_estimable
            ),
            "decision": (
                "ESTIMABLE"
                if distribution_binary_estimable
                else "NOT_ESTIMABLE_INSUFFICIENT_COMPLETE_NESTED_CLASSIFICATION"
            ),
        },
        {
            "target": (
                "submitter_classification_tvd"
            ),
            "requested_analysis_form": (
                "continuous secondary drift magnitude"
            ),
            "evaluable_rows": len(
                distribution_evaluable
            ),
            "positive_events": pd.NA,
            "negative_rows": pd.NA,
            "censored_rows": int(
                EXPECTED_EVALUABLE_ROWS
                - len(
                    distribution_evaluable
                )
            ),
            "estimable_for_discrimination": (
                distribution_continuous_estimable
            ),
            "decision": (
                "ESTIMABLE_AS_CONTINUOUS"
                if distribution_continuous_estimable
                else "NOT_ESTIMABLE_INSUFFICIENT_COMPLETE_NESTED_CLASSIFICATION"
            ),
        },
    ]
)


# --------------------------------------------------------------------------------------------------
# 9. DISPLAY AND PASS MARKER
# --------------------------------------------------------------------------------------------------

separator = "=" * 150

print("\n" + separator)

print(
    "STAGE 6C STEP 3G — CELL 6C-3G0B — "
    "NESTED-SCV POLICY FREEZE AND DERIVABILITY AUDIT"
)

print(separator)

print(
    f"Stage 6B evaluable SHA-256         : "
    f"PASS ({observed_evaluable_sha256})"
)

print(
    f"Accepted T0 SHA-256                : "
    f"PASS ({observed_t0_sha256})"
)

print(
    f"Accepted T1 SHA-256                : "
    f"PASS ({observed_t1_sha256})"
)

print(
    f"Mapped evaluable T0/T1 rows        : "
    f"PASS ({len(mapped):,}/{EXPECTED_EVALUABLE_ROWS:,})"
)

print(
    f"Policy path                        : "
    f"{POLICY_JSON}"
)

print(
    f"Policy SHA-256                     : "
    f"PASS ({policy_sha256})"
)

print(
    f"Policy sidecar file SHA-256        : "
    f"PASS ({sidecar_sha256})"
)

print(
    "GES or comparator scores loaded    : No"
)

print(
    "Predictive performance calculated  : No"
)

print(
    "Record-level derived artifact written: No"
)

print(
    "\nFROZEN OPERATIONAL DEFINITIONS"
)

print(
    json.dumps(
        policy,
        ensure_ascii=False,
        sort_keys=True,
        indent=2,
    )
)

print(
    "\nNESTED REVIEW-STATUS AND "
    "CLASSIFICATION-GROUP VOCABULARY"
)

print(
    nested_vocabulary.to_string(
        index=False
    )
)

print(
    "\nSCV TRANSITION SUMMARY"
)

print(
    scv_transition_summary.to_string(
        index=False
    )
)

print(
    "\nNEW CONTRADICTORY HIGH-RIGOR "
    "OUTCOME STATUS ACCOUNTING"
)

print(
    high_rigor_status_accounting.to_string(
        index=False
    )
)

print(
    "\nSUBMITTER-CLASSIFICATION "
    "DISTRIBUTION STATUS ACCOUNTING"
)

print(
    distribution_status_accounting.to_string(
        index=False
    )
)

print(
    "\nTOTAL-VARIATION-DISTANCE SUMMARY"
)

print(
    tvd_summary.to_string(
        index=False
    )
)

print(
    "\nTOTAL-VARIATION-DISTANCE DISTRIBUTION"
)

if len(
    tvd_distribution
):
    print(
        tvd_distribution.to_string(
            index=False
        )
    )
else:
    print(
        "No complete-case distribution rows were available."
    )

print(
    "\nDERIVABILITY DECISION"
)

print(
    derivability_decision.to_string(
        index=False
    )
)

print(
    "\nINTERPRETATION BOUNDARY"
)

print("-" * 150)

print(
    "Missing nested T1 review_status or classification_group evidence "
    "is never converted into a negative outcome. High-rigor and "
    "submitter-distribution analyses may therefore be censored or declared "
    "non-estimable rather than filled from aggregate RCV fields."
)

print(
    "This preserves the prespecified submission-level meaning and avoids "
    "manufacturing evidence from aggregate review stars or aggregate "
    "classification."
)

print(
    "\nCELL DECISION"
)

print("-" * 150)

print(
    "PASS_STAGE6C_NESTED_SCV_OUTCOME_POLICY_"
    "FROZEN_AND_DERIVABILITY_AUDITED"
)

print(
    "The exact nested-SCV operational policy was checksum-frozen before "
    "score access, and both remaining secondary targets were derived in "
    "memory solely to determine honest evaluability, censoring, and class "
    "variation."
)

print(
    "The next cell may freeze a record-level nested-SCV outcome package "
    "only for targets declared estimable above; non-estimable targets must "
    "be reported as data-limited rather than forced into performance analysis."
)



BASELINE GROUP INVENTORY
t0_canonical_classification_group
VUS            26048
PLP            20136
BLB            18939
CONFLICTING     1477
OTHER             36


AssertionError: A primary-evaluable record has a noninformative T0 baseline group.

In [42]:
# ==================================================================================================
# STAGE 6C STEP 3G — CELL 6C-3G0B v1.1
# NESTED-SCV SECONDARY-OUTCOME POLICY FREEZE AND DERIVABILITY AUDIT
#
# Purpose:
#   1. Freshly verify the immutable Stage 6B evaluable cohort and accepted T0/T1 RCV sources.
#   2. Freeze exact, score-blind operational definitions for:
#        a. a new contradictory high-rigor SCV submission;
#        b. magnitude of submitter-classification distribution change.
#   3. Write and freshly reverify only the deterministic policy JSON and SHA-256 sidecar.
#   4. Derive the two outcomes in memory solely to audit whether the accepted nested schema
#      supports them and to determine evaluable/censored accounting.
#   5. Load no GES or comparator score and calculate no predictive-performance result.
#
# Scientific boundary:
#   T1 nested review_status is mostly missing, and T1 nested classification_group may contain
#   the explicit "Missing" sentinel. A negative label is therefore permitted only when the
#   evidence needed to rule out the event is observed. Otherwise the record is censored.
#
# This cell does NOT write a record-level outcome table. That can occur only after this
# policy and its derivability accounting pass.
# ==================================================================================================

from pathlib import Path
from collections import Counter
import hashlib
import json
import os
import re
import tempfile

import numpy as np
import pandas as pd
import pyarrow.parquet as pq


# --------------------------------------------------------------------------------------------------
# 1. FROZEN PATHS, HASHES, AND SCHEMA
# --------------------------------------------------------------------------------------------------

PROJECT_DIR = Path(
    "/content/drive/MyDrive/GES_RAG_Temporal_Study"
)

STAGE6_DIR = (
    PROJECT_DIR
    / "data_processed"
    / "stage6_temporal_validation"
)

EVALUABLE_PARQUET = (
    STAGE6_DIR
    / "stage6b_locked_primary_evaluable_cohort_v1.parquet"
)

EVALUABLE_SIDECAR = Path(
    str(EVALUABLE_PARQUET) + ".sha256"
)

T0_PARQUET = (
    PROJECT_DIR
    / "data_interim"
    / "t0_rcv_target_genes_corrected_v1_2.parquet"
)

T1_PARQUET = (
    PROJECT_DIR
    / "data_interim"
    / "t1_rcv_target_genes_harmonized_v1.parquet"
)

POLICY_DIR = (
    PROJECT_DIR
    / "configs"
    / "stage6c_nested_scv_secondary_outcomes"
)

PRIOR_POLICY_JSON = (
    POLICY_DIR
    / "stage6c_nested_scv_secondary_outcome_policy_v1.json"
)

PRIOR_POLICY_SIDECAR = Path(
    str(PRIOR_POLICY_JSON) + ".sha256"
)

POLICY_JSON = (
    POLICY_DIR
    / "stage6c_nested_scv_secondary_outcome_policy_v1_1.json"
)

POLICY_SIDECAR = Path(
    str(POLICY_JSON) + ".sha256"
)

EXPECTED_EVALUABLE_SHA256 = (
    "c6ad50dfc376a746e1830d000dba975101d88763fd61b8d1983f6300edb6e038"
)

EXPECTED_T0_SHA256 = (
    "f6b6760b2ad6e4352e3bdecdeaf89827e8a514b031abf2373bb17568d999466d"
)

EXPECTED_T1_SHA256 = (
    "5713a11bdbf4804758cc011f9b2f302afc91fa1f88c1b178d675c28bb277d37c"
)

EXPECTED_EVALUABLE_ROWS = 66_636
EXPECTED_EVALUABLE_COLUMNS = 79
EXPECTED_T0_ROWS = 71_659
EXPECTED_T0_COLUMNS = 34
EXPECTED_T1_ROWS = 100_920
EXPECTED_T1_COLUMNS = 36

STAGE6_T0_RCV_COLUMN = "rcv_accession"
STAGE6_T1_RCV_COLUMN = "linked_t1_rcv_accession"
STAGE6_ROW_ORDER_COLUMN = "t0_row_order"
STAGE6_BASELINE_GROUP_COLUMN = "t0_canonical_classification_group"

SOURCE_RCV_COLUMN = "rcv_accession"
NESTED_SCV_COLUMN = "scv_records_json"

SCV_ID_KEY = "scv_accession"
SUBMITTER_KEY = "submitter"
CLASSIFICATION_GROUP_KEY = "classification_group"
REVIEW_STATUS_KEY = "review_status"

INFORMATIVE_GROUPS = ("BLB", "VUS", "PLP")
MAJOR_DISTRIBUTION_SHIFT_THRESHOLD = 0.50


# --------------------------------------------------------------------------------------------------
# 2. HELPERS
# --------------------------------------------------------------------------------------------------

def sha256_file(
    path: Path,
    chunk_size: int = 1024 * 1024,
) -> str:
    digest = hashlib.sha256()

    with path.open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


def sha256_bytes(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()


def read_sidecar_hash(path: Path) -> str:
    text = path.read_text(
        encoding="utf-8"
    ).strip()

    matches = re.findall(
        r"\b[a-fA-F0-9]{64}\b",
        text,
    )

    if not matches:
        raise ValueError(
            f"No SHA-256 value found in sidecar: {path}"
        )

    return matches[0].lower()


def normalize_rcv(
    series: pd.Series,
) -> pd.Series:
    return (
        series.astype("string")
        .str.upper()
        .str.strip()
        .str.extract(
            r"(RCV\d+)",
            expand=False,
        )
    )


def normalize_scv_accession(
    value,
) -> str | None:
    if value is None:
        return None

    text = str(value).upper().strip()

    match = re.search(
        r"(SCV\d+)",
        text,
    )

    if not match:
        return None

    return match.group(1)


def normalize_text(
    value,
) -> str | None:
    if value is None:
        return None

    text = re.sub(
        r"\s+",
        " ",
        str(value),
    ).strip()

    if text == "":
        return None

    return text


def normalize_review_status(
    value,
) -> str | None:
    text = normalize_text(value)

    if text is None:
        return None

    return text.lower()


def is_high_rigor_review_status(
    normalized_status: str | None,
) -> bool:
    if normalized_status is None:
        return False

    # Exclusion precedence is explicit so that
    # "no assertion criteria provided" is never misclassified merely
    # because it contains the phrase "criteria provided".
    if (
        normalized_status.startswith(
            "no assertion criteria"
        )
        or normalized_status.startswith(
            "no assertion provided"
        )
    ):
        return False

    return (
        normalized_status.startswith(
            "criteria provided"
        )
        or "reviewed by expert panel"
        in normalized_status
        or "practice guideline"
        in normalized_status
    )


def normalize_classification_group(
    value,
) -> str:
    text = normalize_text(value)

    if text is None:
        return "MISSING"

    normalized = (
        text.lower()
        .replace("_", " ")
        .replace("-", " ")
    )

    normalized = re.sub(
        r"\s+",
        " ",
        normalized,
    ).strip()

    if normalized in {
        "blb",
        "benign",
        "likely benign",
        "benign/likely benign",
        "benign likely benign",
    }:
        return "BLB"

    if normalized in {
        "vus",
        "uncertain significance",
        "uncertain",
    }:
        return "VUS"

    if normalized in {
        "plp",
        "pathogenic",
        "likely pathogenic",
        "pathogenic/likely pathogenic",
        "pathogenic likely pathogenic",
    }:
        return "PLP"

    if normalized in {
        "missing",
        "none",
        "nan",
        "not provided",
        "no classification",
        "noclassification",
    }:
        return "MISSING"

    if "conflict" in normalized:
        return "CONFLICTING"

    return "OTHER"


def parse_nested_scv_value(
    value,
    row_label: str,
) -> list[dict]:
    if isinstance(value, list):
        parsed = value

    elif isinstance(value, tuple):
        parsed = list(value)

    elif isinstance(value, np.ndarray):
        parsed = value.tolist()

    elif isinstance(value, str):
        text = value.strip()

        if text == "":
            raise ValueError(
                f"Blank nested SCV JSON at {row_label}"
            )

        try:
            parsed = json.loads(text)
        except Exception as error:
            raise ValueError(
                f"Malformed nested SCV JSON at "
                f"{row_label}: {error}"
            ) from error

    else:
        try:
            if pd.isna(value):
                raise ValueError(
                    f"Missing nested SCV evidence at {row_label}"
                )
        except Exception:
            pass

        raise TypeError(
            f"Unsupported nested SCV type at "
            f"{row_label}: {type(value).__name__}"
        )

    if not isinstance(parsed, list):
        raise TypeError(
            f"Nested SCV value is not a list at "
            f"{row_label}: {type(parsed).__name__}"
        )

    for position, record in enumerate(parsed):
        if not isinstance(record, dict):
            raise TypeError(
                f"Nested SCV record is not a dictionary "
                f"at {row_label}, position {position}."
            )

    return parsed


def canonical_json_bytes(
    value: dict,
) -> bytes:
    text = json.dumps(
        value,
        ensure_ascii=False,
        sort_keys=True,
        indent=2,
    ) + "\n"

    return text.encode("utf-8")


def immutable_write_or_verify(
    path: Path,
    expected_bytes: bytes,
) -> str:
    expected_hash = sha256_bytes(
        expected_bytes
    )

    if path.exists():
        observed_bytes = path.read_bytes()

        if observed_bytes != expected_bytes:
            raise FileExistsError(
                f"An existing immutable artifact differs "
                f"from the expected content:\n{path}"
            )

        observed_hash = sha256_bytes(
            observed_bytes
        )

        if observed_hash != expected_hash:
            raise AssertionError(
                f"Existing artifact hash mismatch:\n{path}"
            )

        return observed_hash

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    with tempfile.NamedTemporaryFile(
        mode="wb",
        dir=str(path.parent),
        prefix=path.name + ".tmp.",
        delete=False,
    ) as temporary:
        temporary.write(
            expected_bytes
        )
        temporary.flush()
        os.fsync(
            temporary.fileno()
        )
        temporary_path = Path(
            temporary.name
        )

    try:
        os.replace(
            temporary_path,
            path,
        )
    finally:
        if temporary_path.exists():
            temporary_path.unlink()

    observed_hash = sha256_file(
        path
    )

    if observed_hash != expected_hash:
        raise AssertionError(
            f"Fresh artifact write/readback mismatch:\n{path}"
        )

    return observed_hash


def distribution_vector(
    normalized_groups: list[str],
) -> np.ndarray:
    counts = Counter(
        normalized_groups
    )

    total = sum(
        counts[group]
        for group in INFORMATIVE_GROUPS
    )

    if total <= 0:
        raise ValueError(
            "Cannot construct an informative distribution "
            "with zero BLB/VUS/PLP SCVs."
        )

    return np.array(
        [
            counts[group] / total
            for group in INFORMATIVE_GROUPS
        ],
        dtype=float,
    )


def total_variation_distance(
    first: np.ndarray,
    second: np.ndarray,
) -> float:
    return float(
        0.5
        * np.abs(
            np.asarray(first, dtype=float)
            - np.asarray(second, dtype=float)
        ).sum()
    )


# --------------------------------------------------------------------------------------------------
# 3. CRYPTOGRAPHIC AND STRUCTURAL VERIFICATION
# --------------------------------------------------------------------------------------------------

for path in [
    EVALUABLE_PARQUET,
    EVALUABLE_SIDECAR,
    T0_PARQUET,
    T1_PARQUET,
]:
    if not path.exists():
        raise FileNotFoundError(
            f"Missing required frozen artifact:\n{path}"
        )

observed_evaluable_sha256 = sha256_file(
    EVALUABLE_PARQUET
)

if (
    observed_evaluable_sha256
    != EXPECTED_EVALUABLE_SHA256
):
    raise AssertionError(
        "Stage 6B evaluable SHA-256 mismatch."
    )

if (
    read_sidecar_hash(
        EVALUABLE_SIDECAR
    )
    != observed_evaluable_sha256
):
    raise AssertionError(
        "Stage 6B evaluable sidecar mismatch."
    )

observed_t0_sha256 = sha256_file(
    T0_PARQUET
)

observed_t1_sha256 = sha256_file(
    T1_PARQUET
)

if observed_t0_sha256 != EXPECTED_T0_SHA256:
    raise AssertionError(
        "Accepted T0 SHA-256 mismatch."
    )

if observed_t1_sha256 != EXPECTED_T1_SHA256:
    raise AssertionError(
        "Accepted T1 SHA-256 mismatch."
    )

stage6_file = pq.ParquetFile(
    EVALUABLE_PARQUET
)

t0_file = pq.ParquetFile(
    T0_PARQUET
)

t1_file = pq.ParquetFile(
    T1_PARQUET
)

if (
    stage6_file.metadata.num_rows
    != EXPECTED_EVALUABLE_ROWS
    or stage6_file.metadata.num_columns
    != EXPECTED_EVALUABLE_COLUMNS
):
    raise AssertionError(
        "Stage 6B dimensions mismatch."
    )

if (
    t0_file.metadata.num_rows
    != EXPECTED_T0_ROWS
    or t0_file.metadata.num_columns
    != EXPECTED_T0_COLUMNS
):
    raise AssertionError(
        "Accepted T0 dimensions mismatch."
    )

if (
    t1_file.metadata.num_rows
    != EXPECTED_T1_ROWS
    or t1_file.metadata.num_columns
    != EXPECTED_T1_COLUMNS
):
    raise AssertionError(
        "Accepted T1 dimensions mismatch."
    )

required_stage6_columns = [
    STAGE6_T0_RCV_COLUMN,
    STAGE6_T1_RCV_COLUMN,
    STAGE6_ROW_ORDER_COLUMN,
    STAGE6_BASELINE_GROUP_COLUMN,
]

missing_stage6_columns = [
    column
    for column in required_stage6_columns
    if column
    not in stage6_file.schema_arrow.names
]

if missing_stage6_columns:
    raise KeyError(
        "Missing required Stage 6B columns:\n"
        + "\n".join(
            missing_stage6_columns
        )
    )

for label, schema in [
    (
        "T0",
        t0_file.schema_arrow.names,
    ),
    (
        "T1",
        t1_file.schema_arrow.names,
    ),
]:
    missing = [
        column
        for column in [
            SOURCE_RCV_COLUMN,
            NESTED_SCV_COLUMN,
        ]
        if column not in schema
    ]

    if missing:
        raise KeyError(
            f"{label} source is missing:\n"
            + "\n".join(missing)
        )


# --------------------------------------------------------------------------------------------------
# 4. CREATE AND FREEZE THE SCORE-BLIND OPERATIONAL POLICY
# --------------------------------------------------------------------------------------------------

# Cell 6C-3G0B v1 wrote its policy before the derivability audit discovered that the
# primary-evaluable cohort legitimately contains CONFLICTING and OTHER baseline groups.
# That earlier artifact is preserved unchanged. Version 1.1 explicitly defines how those
# baseline groups are handled and never overwrites the prior policy.
if PRIOR_POLICY_JSON.exists():
    prior_policy_sha256 = sha256_file(
        PRIOR_POLICY_JSON
    )

    if PRIOR_POLICY_SIDECAR.exists():
        prior_sidecar_target_sha256 = read_sidecar_hash(
            PRIOR_POLICY_SIDECAR
        )

        if prior_sidecar_target_sha256 != prior_policy_sha256:
            raise AssertionError(
                "Prior v1 policy sidecar does not match the prior policy artifact."
            )
    else:
        prior_sidecar_target_sha256 = None

    prior_policy_provenance = {
        "filename": PRIOR_POLICY_JSON.name,
        "sha256": prior_policy_sha256,
        "sidecar_present": PRIOR_POLICY_SIDECAR.exists(),
        "status": "PRESERVED_UNCHANGED",
    }
else:
    prior_policy_provenance = {
        "filename": PRIOR_POLICY_JSON.name,
        "sha256": None,
        "sidecar_present": False,
        "status": "NOT_PRESENT_IN_THIS_RUNTIME",
    }

policy = {
    "policy_name": (
        "stage6c_nested_scv_secondary_outcome_policy"
    ),
    "policy_version": "1.1",
    "status": (
        "FROZEN_BEFORE_NESTED_SCV_SCORE_ANALYSIS"
    ),
    "supersedes": {
        "prior_policy": prior_policy_provenance,
        "reason": (
            "The failed score-blind derivability audit showed that the frozen "
            "primary-evaluable cohort includes valid CONFLICTING and OTHER "
            "baseline groups. Version 1.1 explicitly censors new-submission "
            "contradiction assessment when a new SCV exists but the baseline "
            "group is not BLB, VUS, or PLP. The prior policy is preserved."
        ),
    },
    "study_unit": (
        "RCV-level variant-condition aggregate record"
    ),
    "accepted_sources": {
        "stage6b_primary_evaluable_cohort": {
            "filename": EVALUABLE_PARQUET.name,
            "sha256": EXPECTED_EVALUABLE_SHA256,
            "rows": EXPECTED_EVALUABLE_ROWS,
            "columns": EXPECTED_EVALUABLE_COLUMNS,
        },
        "t0_rcv_source": {
            "filename": T0_PARQUET.name,
            "sha256": EXPECTED_T0_SHA256,
            "rows": EXPECTED_T0_ROWS,
            "columns": EXPECTED_T0_COLUMNS,
        },
        "t1_rcv_source": {
            "filename": T1_PARQUET.name,
            "sha256": EXPECTED_T1_SHA256,
            "rows": EXPECTED_T1_ROWS,
            "columns": EXPECTED_T1_COLUMNS,
        },
    },
    "nested_keys": {
        "scv_identity": SCV_ID_KEY,
        "submitter_identity": SUBMITTER_KEY,
        "classification_group": CLASSIFICATION_GROUP_KEY,
        "review_rigor": REVIEW_STATUS_KEY,
    },
    "classification_group_mapping": {
        "BLB": [
            "Benign",
            "Likely benign",
            "Benign/Likely benign",
        ],
        "VUS": [
            "Uncertain significance",
            "VUS",
        ],
        "PLP": [
            "Pathogenic",
            "Likely pathogenic",
            "Pathogenic/Likely pathogenic",
        ],
        "MISSING": [
            "Missing",
            "NoClassification",
            "null/blank",
        ],
        "CONFLICTING": [
            "values containing conflict",
        ],
        "OTHER": [
            "all remaining values",
        ],
    },
    "high_rigor_rule": {
        "positive_review_status_logic": [
            "normalized status starts with 'criteria provided'",
            "normalized status contains 'reviewed by expert panel'",
            "normalized status contains 'practice guideline'",
        ],
        "explicit_exclusion_precedence": [
            "normalized status starts with 'no assertion criteria'",
            "normalized status starts with 'no assertion provided'",
        ],
        "missing_review_status": (
            "unknown, never assumed low-rigor"
        ),
    },
    "new_contradictory_high_rigor_submission": {
        "new_submission": (
            "normalized T1 SCV accession absent from the "
            "same linked RCV's T0 SCV-accession set"
        ),
        "baseline_interpretation": (
            "frozen Stage 6B t0_canonical_classification_group; BLB, VUS, "
            "and PLP are informative single-group baselines; CONFLICTING, "
            "OTHER, and MISSING are noninformative for contradiction assessment"
        ),
        "contradiction": (
            "for an informative BLB/VUS/PLP baseline, a new T1 SCV normalized "
            "group is BLB, VUS, or PLP and differs from that baseline group"
        ),
        "positive": (
            "at least one new T1 SCV is high-rigor and contradictory"
        ),
        "negative": (
            "no new T1 SCV exists regardless of baseline group, or the baseline "
            "is informative and every new T1 SCV can be fully evaluated and none "
            "is both high-rigor and contradictory"
        ),
        "censored": (
            "when at least one new T1 SCV exists: the T0 baseline group is "
            "CONFLICTING/OTHER/MISSING, or no positive event is observed and at "
            "least one new T1 SCV has missing review status, or a high-rigor new "
            "SCV has a noninformative classification group"
        ),
        "updated_existing_scv": (
            "an existing SCV accession with a later version is not a "
            "new submission and is excluded from this outcome"
        ),
    },
    "submitter_classification_distribution_change": {
        "distribution_unit": (
            "one nested SCV classification assertion"
        ),
        "categories": list(
            INFORMATIVE_GROUPS
        ),
        "complete_case_evaluability": (
            "every nested SCV at both T0 and T1 must map to BLB, VUS, "
            "or PLP; otherwise the distribution magnitude is censored"
        ),
        "magnitude": (
            "total variation distance = 0.5 * sum(abs(p_T1 - p_T0))"
        ),
        "range": [
            0.0,
            1.0,
        ],
        "major_shift_binary_sensitivity": (
            "total variation distance >= 0.50"
        ),
        "major_shift_threshold": (
            MAJOR_DISTRIBUTION_SHIFT_THRESHOLD
        ),
        "threshold_rationale": (
            "a fixed, interpretable threshold requiring at least half "
            "of classification probability mass to move; not selected "
            "from GES performance"
        ),
    },
    "analysis_role": {
        "new_contradictory_high_rigor_submission": (
            "exploratory secondary evidence-drift outcome"
        ),
        "distribution_magnitude": (
            "exploratory continuous secondary evidence-drift measure"
        ),
        "major_distribution_shift": (
            "exploratory binary sensitivity outcome"
        ),
    },
    "leakage_protection": {
        "ges_or_comparator_scores_loaded_during_policy_freeze": False,
        "threshold_selected_from_performance": False,
        "outcome_definition_selected_from_performance": False,
    },
}

policy_bytes = canonical_json_bytes(
    policy
)

policy_sha256 = immutable_write_or_verify(
    POLICY_JSON,
    policy_bytes,
)

sidecar_bytes = (
    f"{policy_sha256}  {POLICY_JSON.name}\n"
).encode("utf-8")

sidecar_sha256 = immutable_write_or_verify(
    POLICY_SIDECAR,
    sidecar_bytes,
)

if sha256_file(
    POLICY_JSON
) != policy_sha256:
    raise AssertionError(
        "Policy fresh readback verification failed."
    )

if read_sidecar_hash(
    POLICY_SIDECAR
) != policy_sha256:
    raise AssertionError(
        "Policy sidecar fresh readback verification failed."
    )


# --------------------------------------------------------------------------------------------------
# 5. LOAD ONLY IDENTIFIERS, BASELINE GROUP, AND NESTED EVIDENCE
# --------------------------------------------------------------------------------------------------

stage6 = pd.read_parquet(
    EVALUABLE_PARQUET,
    columns=required_stage6_columns,
).copy()

stage6[
    STAGE6_T0_RCV_COLUMN
] = normalize_rcv(
    stage6[
        STAGE6_T0_RCV_COLUMN
    ]
)

stage6[
    STAGE6_T1_RCV_COLUMN
] = normalize_rcv(
    stage6[
        STAGE6_T1_RCV_COLUMN
    ]
)

stage6[
    STAGE6_BASELINE_GROUP_COLUMN
] = (
    stage6[
        STAGE6_BASELINE_GROUP_COLUMN
    ]
    .map(
        normalize_classification_group
    )
)

if stage6[
    STAGE6_T0_RCV_COLUMN
].isna().any():
    raise AssertionError(
        "A Stage 6B T0 RCV failed normalization."
    )

if stage6[
    STAGE6_T1_RCV_COLUMN
].isna().any():
    raise AssertionError(
        "A Stage 6B T1 RCV failed normalization."
    )

if (
    stage6[
        STAGE6_T0_RCV_COLUMN
    ].nunique()
    != EXPECTED_EVALUABLE_ROWS
):
    raise AssertionError(
        "Stage 6B T0 RCV keys are not unique."
    )

if (
    stage6[
        STAGE6_T1_RCV_COLUMN
    ].nunique()
    != EXPECTED_EVALUABLE_ROWS
):
    raise AssertionError(
        "Stage 6B linked T1 RCV keys are not unique."
    )

baseline_group_accounting = (
    stage6[
        STAGE6_BASELINE_GROUP_COLUMN
    ]
    .value_counts(
        dropna=False
    )
    .rename_axis(
        "t0_baseline_group"
    )
    .reset_index(
        name="rcv_rows"
    )
)

baseline_group_accounting[
    "informative_for_contradiction"
] = baseline_group_accounting[
    "t0_baseline_group"
].isin(
    INFORMATIVE_GROUPS
)

allowed_baseline_groups = set(
    INFORMATIVE_GROUPS
) | {
    "CONFLICTING",
    "OTHER",
    "MISSING",
}

unexpected_baseline_groups = sorted(
    set(
        stage6[
            STAGE6_BASELINE_GROUP_COLUMN
        ].unique().tolist()
    )
    - allowed_baseline_groups
)

if unexpected_baseline_groups:
    print(
        "\nBASELINE GROUP INVENTORY"
    )

    print(
        baseline_group_accounting.to_string(
            index=False
        )
    )

    raise AssertionError(
        "Unexpected normalized T0 baseline groups: "
        + ", ".join(
            unexpected_baseline_groups
        )
    )

t0_source = pd.read_parquet(
    T0_PARQUET,
    columns=[
        SOURCE_RCV_COLUMN,
        NESTED_SCV_COLUMN,
    ],
).copy()

t1_source = pd.read_parquet(
    T1_PARQUET,
    columns=[
        SOURCE_RCV_COLUMN,
        NESTED_SCV_COLUMN,
    ],
).copy()

t0_source[
    SOURCE_RCV_COLUMN
] = normalize_rcv(
    t0_source[
        SOURCE_RCV_COLUMN
    ]
)

t1_source[
    SOURCE_RCV_COLUMN
] = normalize_rcv(
    t1_source[
        SOURCE_RCV_COLUMN
    ]
)

if (
    t0_source[
        SOURCE_RCV_COLUMN
    ].nunique()
    != EXPECTED_T0_ROWS
):
    raise AssertionError(
        "T0 source RCV keys are not unique."
    )

if (
    t1_source[
        SOURCE_RCV_COLUMN
    ].nunique()
    != EXPECTED_T1_ROWS
):
    raise AssertionError(
        "T1 source RCV keys are not unique."
    )

mapped = (
    stage6.merge(
        t0_source.rename(
            columns={
                SOURCE_RCV_COLUMN: (
                    "_t0_source_rcv"
                ),
                NESTED_SCV_COLUMN: (
                    "_t0_scv_records_json"
                ),
            }
        ),
        left_on=STAGE6_T0_RCV_COLUMN,
        right_on="_t0_source_rcv",
        how="left",
        validate="one_to_one",
        indicator="_t0_merge",
    )
    .merge(
        t1_source.rename(
            columns={
                SOURCE_RCV_COLUMN: (
                    "_t1_source_rcv"
                ),
                NESTED_SCV_COLUMN: (
                    "_t1_scv_records_json"
                ),
            }
        ),
        left_on=STAGE6_T1_RCV_COLUMN,
        right_on="_t1_source_rcv",
        how="left",
        validate="one_to_one",
        indicator="_t1_merge",
    )
    .sort_values(
        STAGE6_ROW_ORDER_COLUMN,
        kind="mergesort",
    )
    .reset_index(drop=True)
)

if not mapped[
    "_t0_merge"
].eq("both").all():
    raise AssertionError(
        "An evaluable T0 RCV failed source mapping."
    )

if not mapped[
    "_t1_merge"
].eq("both").all():
    raise AssertionError(
        "An evaluable T1 RCV failed source mapping."
    )

if len(mapped) != EXPECTED_EVALUABLE_ROWS:
    raise AssertionError(
        "Mapped evaluable row count mismatch."
    )


# --------------------------------------------------------------------------------------------------
# 6. SCORE-BLIND IN-MEMORY DERIVATION AND DERIVABILITY AUDIT
# --------------------------------------------------------------------------------------------------

t0_review_status_counts = Counter()
t1_review_status_counts = Counter()
t0_group_counts = Counter()
t1_group_counts = Counter()

record_rows = []

derivation_columns = [
    STAGE6_T0_RCV_COLUMN,
    STAGE6_T1_RCV_COLUMN,
    STAGE6_ROW_ORDER_COLUMN,
    STAGE6_BASELINE_GROUP_COLUMN,
    "_t0_scv_records_json",
    "_t1_scv_records_json",
]

for (
    t0_rcv,
    t1_rcv,
    t0_row_order,
    baseline_group,
    t0_scv_records_json,
    t1_scv_records_json,
) in mapped[
    derivation_columns
].itertuples(
    index=False,
    name=None,
):
    baseline_group_informative = (
        baseline_group
        in INFORMATIVE_GROUPS
    )

    t0_records = parse_nested_scv_value(
        t0_scv_records_json,
        f"T0:{t0_rcv}",
    )

    t1_records = parse_nested_scv_value(
        t1_scv_records_json,
        f"T1:{t1_rcv}",
    )

    t0_accessions = set()

    t0_normalized_groups = []
    t1_normalized_groups = []

    for record in t0_records:
        accession = normalize_scv_accession(
            record.get(
                SCV_ID_KEY
            )
        )

        if accession is None:
            raise AssertionError(
                f"T0 nested SCV lacks a valid accession: "
                f"{t0_rcv}"
            )

        if accession in t0_accessions:
            raise AssertionError(
                f"Duplicate T0 SCV accession within RCV "
                f"{t0_rcv}: {accession}"
            )

        t0_accessions.add(
            accession
        )

        status = normalize_review_status(
            record.get(
                REVIEW_STATUS_KEY
            )
        )

        t0_review_status_counts[
            status
            if status is not None
            else "<MISSING>"
        ] += 1

        group = normalize_classification_group(
            record.get(
                CLASSIFICATION_GROUP_KEY
            )
        )

        t0_group_counts[
            group
        ] += 1

        t0_normalized_groups.append(
            group
        )

    t1_accessions = set()

    new_scv_count = 0
    new_scv_missing_review_status = 0
    new_high_rigor_count = 0
    new_high_rigor_noninformative_group = 0
    new_high_rigor_contradictory_count = 0
    new_high_rigor_concordant_count = 0
    new_high_rigor_baseline_not_comparable_count = 0
    updated_existing_scv_count = 0

    t0_version_by_accession = {
        normalize_scv_accession(
            record.get(
                SCV_ID_KEY
            )
        ): str(
            record.get(
                "scv_version"
            )
        )
        for record in t0_records
    }

    for record in t1_records:
        accession = normalize_scv_accession(
            record.get(
                SCV_ID_KEY
            )
        )

        if accession is None:
            raise AssertionError(
                f"T1 nested SCV lacks a valid accession: "
                f"{t1_rcv}"
            )

        if accession in t1_accessions:
            raise AssertionError(
                f"Duplicate T1 SCV accession within RCV "
                f"{t1_rcv}: {accession}"
            )

        t1_accessions.add(
            accession
        )

        status = normalize_review_status(
            record.get(
                REVIEW_STATUS_KEY
            )
        )

        t1_review_status_counts[
            status
            if status is not None
            else "<MISSING>"
        ] += 1

        group = normalize_classification_group(
            record.get(
                CLASSIFICATION_GROUP_KEY
            )
        )

        t1_group_counts[
            group
        ] += 1

        t1_normalized_groups.append(
            group
        )

        if accession in t0_accessions:
            t0_version = t0_version_by_accession.get(
                accession
            )

            t1_version = str(
                record.get(
                    "scv_version"
                )
            )

            if t0_version != t1_version:
                updated_existing_scv_count += 1

            continue

        new_scv_count += 1

        if status is None:
            new_scv_missing_review_status += 1
            continue

        if not is_high_rigor_review_status(
            status
        ):
            continue

        new_high_rigor_count += 1

        if group not in INFORMATIVE_GROUPS:
            new_high_rigor_noninformative_group += 1
            continue

        if not baseline_group_informative:
            new_high_rigor_baseline_not_comparable_count += 1
            continue

        if group != baseline_group:
            new_high_rigor_contradictory_count += 1
        else:
            new_high_rigor_concordant_count += 1

    if new_scv_count == 0:
        # No new submission means the requested event cannot occur, even when
        # the aggregate T0 baseline group is CONFLICTING or OTHER.
        high_rigor_outcome_status = "NEGATIVE_NO_NEW_SCV"
        high_rigor_outcome = 0

    elif not baseline_group_informative:
        high_rigor_outcome_status = (
            "CENSORED_NONINFORMATIVE_BASELINE_GROUP"
        )
        high_rigor_outcome = pd.NA

    elif (
        new_high_rigor_contradictory_count
        > 0
    ):
        high_rigor_outcome_status = "POSITIVE"
        high_rigor_outcome = 1

    elif (
        new_scv_missing_review_status
        > 0
        or new_high_rigor_noninformative_group
        > 0
    ):
        high_rigor_outcome_status = (
            "CENSORED_REQUIRED_NESTED_FIELD_MISSING"
        )
        high_rigor_outcome = pd.NA

    else:
        high_rigor_outcome_status = "NEGATIVE_EVALUABLE_NEW_SCV"
        high_rigor_outcome = 0

    t0_complete_distribution = all(
        group in INFORMATIVE_GROUPS
        for group in t0_normalized_groups
    )

    t1_complete_distribution = all(
        group in INFORMATIVE_GROUPS
        for group in t1_normalized_groups
    )

    distribution_evaluable = (
        t0_complete_distribution
        and t1_complete_distribution
        and len(
            t0_normalized_groups
        ) > 0
        and len(
            t1_normalized_groups
        ) > 0
    )

    if distribution_evaluable:
        t0_distribution = distribution_vector(
            t0_normalized_groups
        )

        t1_distribution = distribution_vector(
            t1_normalized_groups
        )

        distribution_tvd = total_variation_distance(
            t0_distribution,
            t1_distribution,
        )

        major_distribution_shift = int(
            distribution_tvd
            >= MAJOR_DISTRIBUTION_SHIFT_THRESHOLD
        )

        distribution_status = "EVALUABLE"

    else:
        distribution_tvd = np.nan
        major_distribution_shift = pd.NA
        distribution_status = (
            "CENSORED_NONINFORMATIVE_NESTED_CLASSIFICATION"
        )

    record_rows.append(
        {
            "t0_rcv_accession": t0_rcv,
            "linked_t1_rcv_accession": t1_rcv,
            "t0_row_order": t0_row_order,
            "t0_baseline_group": baseline_group,
            "t0_baseline_group_informative_for_contradiction": (
                baseline_group_informative
            ),
            "t0_scv_count": len(
                t0_records
            ),
            "t1_scv_count": len(
                t1_records
            ),
            "new_scv_count": new_scv_count,
            "updated_existing_scv_count": (
                updated_existing_scv_count
            ),
            "new_scv_missing_review_status": (
                new_scv_missing_review_status
            ),
            "new_high_rigor_scv_count": (
                new_high_rigor_count
            ),
            "new_high_rigor_noninformative_group_count": (
                new_high_rigor_noninformative_group
            ),
            "new_high_rigor_concordant_count": (
                new_high_rigor_concordant_count
            ),
            "new_high_rigor_baseline_not_comparable_count": (
                new_high_rigor_baseline_not_comparable_count
            ),
            "new_high_rigor_contradictory_count": (
                new_high_rigor_contradictory_count
            ),
            "new_contradictory_high_rigor_status": (
                high_rigor_outcome_status
            ),
            "new_contradictory_high_rigor_submission": (
                high_rigor_outcome
            ),
            "distribution_status": (
                distribution_status
            ),
            "submitter_classification_tvd": (
                distribution_tvd
            ),
            "major_submitter_distribution_shift": (
                major_distribution_shift
            ),
        }
    )

derived_audit = pd.DataFrame(
    record_rows
)

if len(derived_audit) != EXPECTED_EVALUABLE_ROWS:
    raise AssertionError(
        "In-memory derivation row count mismatch."
    )

if (
    derived_audit[
        "t0_rcv_accession"
    ].nunique()
    != EXPECTED_EVALUABLE_ROWS
):
    raise AssertionError(
        "In-memory derivation T0 keys are not unique."
    )

if not np.all(
    np.diff(
        derived_audit[
            "t0_row_order"
        ].to_numpy()
    ) > 0
):
    raise AssertionError(
        "In-memory derivation did not preserve "
        "frozen T0 row order."
    )


# --------------------------------------------------------------------------------------------------
# 7. AUDIT TABLES
# --------------------------------------------------------------------------------------------------

nested_vocabulary = pd.concat(
    [
        pd.DataFrame(
            {
                "timepoint": "T0",
                "field": REVIEW_STATUS_KEY,
                "value": list(
                    t0_review_status_counts.keys()
                ),
                "nested_records": list(
                    t0_review_status_counts.values()
                ),
            }
        ),
        pd.DataFrame(
            {
                "timepoint": "T1",
                "field": REVIEW_STATUS_KEY,
                "value": list(
                    t1_review_status_counts.keys()
                ),
                "nested_records": list(
                    t1_review_status_counts.values()
                ),
            }
        ),
        pd.DataFrame(
            {
                "timepoint": "T0",
                "field": CLASSIFICATION_GROUP_KEY,
                "value": list(
                    t0_group_counts.keys()
                ),
                "nested_records": list(
                    t0_group_counts.values()
                ),
            }
        ),
        pd.DataFrame(
            {
                "timepoint": "T1",
                "field": CLASSIFICATION_GROUP_KEY,
                "value": list(
                    t1_group_counts.keys()
                ),
                "nested_records": list(
                    t1_group_counts.values()
                ),
            }
        ),
    ],
    ignore_index=True,
).sort_values(
    [
        "field",
        "timepoint",
        "nested_records",
        "value",
    ],
    ascending=[
        True,
        True,
        False,
        True,
    ],
).reset_index(drop=True)

high_rigor_status_accounting = (
    derived_audit[
        "new_contradictory_high_rigor_status"
    ]
    .value_counts(
        dropna=False
    )
    .rename_axis(
        "outcome_status"
    )
    .reset_index(
        name="rcv_rows"
    )
)

distribution_status_accounting = (
    derived_audit[
        "distribution_status"
    ]
    .value_counts(
        dropna=False
    )
    .rename_axis(
        "distribution_status"
    )
    .reset_index(
        name="rcv_rows"
    )
)

scv_transition_summary = pd.DataFrame(
    [
        {
            "measure": "RCVs with at least one new SCV",
            "rcv_rows": int(
                (
                    derived_audit[
                        "new_scv_count"
                    ] > 0
                ).sum()
            ),
            "nested_scv_total": int(
                derived_audit[
                    "new_scv_count"
                ].sum()
            ),
        },
        {
            "measure": (
                "RCVs with a new SCV missing review_status"
            ),
            "rcv_rows": int(
                (
                    derived_audit[
                        "new_scv_missing_review_status"
                    ] > 0
                ).sum()
            ),
            "nested_scv_total": int(
                derived_audit[
                    "new_scv_missing_review_status"
                ].sum()
            ),
        },
        {
            "measure": (
                "RCVs with at least one new high-rigor SCV"
            ),
            "rcv_rows": int(
                (
                    derived_audit[
                        "new_high_rigor_scv_count"
                    ] > 0
                ).sum()
            ),
            "nested_scv_total": int(
                derived_audit[
                    "new_high_rigor_scv_count"
                ].sum()
            ),
        },
        {
            "measure": (
                "RCVs with at least one new contradictory high-rigor SCV"
            ),
            "rcv_rows": int(
                (
                    derived_audit[
                        "new_high_rigor_contradictory_count"
                    ] > 0
                ).sum()
            ),
            "nested_scv_total": int(
                derived_audit[
                    "new_high_rigor_contradictory_count"
                ].sum()
            ),
        },
        {
            "measure": (
                "RCVs with an updated existing SCV version"
            ),
            "rcv_rows": int(
                (
                    derived_audit[
                        "updated_existing_scv_count"
                    ] > 0
                ).sum()
            ),
            "nested_scv_total": int(
                derived_audit[
                    "updated_existing_scv_count"
                ].sum()
            ),
        },
    ]
)

distribution_evaluable = derived_audit.loc[
    derived_audit[
        "distribution_status"
    ] == "EVALUABLE"
].copy()

if len(distribution_evaluable):
    tvd_summary = pd.DataFrame(
        [
            {
                "evaluable_rcv_rows": len(
                    distribution_evaluable
                ),
                "mean_tvd": float(
                    distribution_evaluable[
                        "submitter_classification_tvd"
                    ].mean()
                ),
                "median_tvd": float(
                    distribution_evaluable[
                        "submitter_classification_tvd"
                    ].median()
                ),
                "minimum_tvd": float(
                    distribution_evaluable[
                        "submitter_classification_tvd"
                    ].min()
                ),
                "maximum_tvd": float(
                    distribution_evaluable[
                        "submitter_classification_tvd"
                    ].max()
                ),
                "major_shift_threshold": (
                    MAJOR_DISTRIBUTION_SHIFT_THRESHOLD
                ),
                "major_shift_events": int(
                    distribution_evaluable[
                        "major_submitter_distribution_shift"
                    ].astype(int).sum()
                ),
                "major_shift_negatives": int(
                    len(
                        distribution_evaluable
                    )
                    - distribution_evaluable[
                        "major_submitter_distribution_shift"
                    ].astype(int).sum()
                ),
            }
        ]
    )

    tvd_bins = pd.cut(
        distribution_evaluable[
            "submitter_classification_tvd"
        ],
        bins=[
            -1e-12,
            0.0,
            0.25,
            0.50,
            0.75,
            1.0,
        ],
        labels=[
            "0",
            "(0,0.25]",
            "(0.25,0.50]",
            "(0.50,0.75]",
            "(0.75,1.00]",
        ],
        include_lowest=True,
    )

    tvd_distribution = (
        tvd_bins.value_counts(
            sort=False,
            dropna=False,
        )
        .rename_axis(
            "tvd_interval"
        )
        .reset_index(
            name="rcv_rows"
        )
    )

else:
    tvd_summary = pd.DataFrame(
        [
            {
                "evaluable_rcv_rows": 0,
                "mean_tvd": np.nan,
                "median_tvd": np.nan,
                "minimum_tvd": np.nan,
                "maximum_tvd": np.nan,
                "major_shift_threshold": (
                    MAJOR_DISTRIBUTION_SHIFT_THRESHOLD
                ),
                "major_shift_events": 0,
                "major_shift_negatives": 0,
            }
        ]
    )

    tvd_distribution = pd.DataFrame(
        columns=[
            "tvd_interval",
            "rcv_rows",
        ]
    )


# --------------------------------------------------------------------------------------------------
# 8. FINAL SCIENTIFIC DECISION LOGIC
# --------------------------------------------------------------------------------------------------

high_rigor_positive_count = int(
    (
        derived_audit[
            "new_contradictory_high_rigor_status"
        ]
        == "POSITIVE"
    ).sum()
)

high_rigor_negative_count = int(
    (
        derived_audit[
            "new_contradictory_high_rigor_status"
        ]
        == "NEGATIVE"
    ).sum()
)

high_rigor_censored_count = int(
    (
        derived_audit[
            "new_contradictory_high_rigor_status"
        ]
        .str.startswith(
            "CENSORED",
            na=False,
        )
    ).sum()
)

high_rigor_binary_estimable = (
    high_rigor_positive_count > 0
    and high_rigor_negative_count > 0
)

distribution_binary_estimable = (
    len(
        distribution_evaluable
    ) > 0
    and distribution_evaluable[
        "major_submitter_distribution_shift"
    ].astype(int).nunique() == 2
)

distribution_continuous_estimable = (
    len(
        distribution_evaluable
    ) >= 50
    and distribution_evaluable[
        "submitter_classification_tvd"
    ].nunique() >= 2
)

derivability_decision = pd.DataFrame(
    [
        {
            "target": (
                "new_contradictory_high_rigor_submission"
            ),
            "requested_analysis_form": (
                "binary secondary outcome"
            ),
            "evaluable_rows": (
                high_rigor_positive_count
                + high_rigor_negative_count
            ),
            "positive_events": (
                high_rigor_positive_count
            ),
            "negative_rows": (
                high_rigor_negative_count
            ),
            "censored_rows": (
                high_rigor_censored_count
            ),
            "estimable_for_discrimination": (
                high_rigor_binary_estimable
            ),
            "decision": (
                "ESTIMABLE"
                if high_rigor_binary_estimable
                else (
                    "NOT_ESTIMABLE_NO_POSITIVE_EVENTS"
                    if high_rigor_positive_count == 0
                    else "NOT_ESTIMABLE_INSUFFICIENT_CLASS_VARIATION"
                )
            ),
        },
        {
            "target": (
                "major_submitter_distribution_shift"
            ),
            "requested_analysis_form": (
                "binary sensitivity outcome, TVD >= 0.50"
            ),
            "evaluable_rows": len(
                distribution_evaluable
            ),
            "positive_events": int(
                distribution_evaluable[
                    "major_submitter_distribution_shift"
                ].fillna(0).astype(int).sum()
            ),
            "negative_rows": int(
                len(
                    distribution_evaluable
                )
                - distribution_evaluable[
                    "major_submitter_distribution_shift"
                ].fillna(0).astype(int).sum()
            ),
            "censored_rows": int(
                EXPECTED_EVALUABLE_ROWS
                - len(
                    distribution_evaluable
                )
            ),
            "estimable_for_discrimination": (
                distribution_binary_estimable
            ),
            "decision": (
                "ESTIMABLE"
                if distribution_binary_estimable
                else "NOT_ESTIMABLE_INSUFFICIENT_COMPLETE_NESTED_CLASSIFICATION"
            ),
        },
        {
            "target": (
                "submitter_classification_tvd"
            ),
            "requested_analysis_form": (
                "continuous secondary drift magnitude"
            ),
            "evaluable_rows": len(
                distribution_evaluable
            ),
            "positive_events": pd.NA,
            "negative_rows": pd.NA,
            "censored_rows": int(
                EXPECTED_EVALUABLE_ROWS
                - len(
                    distribution_evaluable
                )
            ),
            "estimable_for_discrimination": (
                distribution_continuous_estimable
            ),
            "decision": (
                "ESTIMABLE_AS_CONTINUOUS"
                if distribution_continuous_estimable
                else "NOT_ESTIMABLE_INSUFFICIENT_COMPLETE_NESTED_CLASSIFICATION"
            ),
        },
    ]
)


# --------------------------------------------------------------------------------------------------
# 9. DISPLAY AND PASS MARKER
# --------------------------------------------------------------------------------------------------

separator = "=" * 150

print("\n" + separator)

print(
    "STAGE 6C STEP 3G — CELL 6C-3G0B v1.1 — "
    "NESTED-SCV POLICY FREEZE AND DERIVABILITY AUDIT"
)

print(separator)

print(
    f"Stage 6B evaluable SHA-256         : "
    f"PASS ({observed_evaluable_sha256})"
)

print(
    f"Accepted T0 SHA-256                : "
    f"PASS ({observed_t0_sha256})"
)

print(
    f"Accepted T1 SHA-256                : "
    f"PASS ({observed_t1_sha256})"
)

print(
    f"Mapped evaluable T0/T1 rows        : "
    f"PASS ({len(mapped):,}/{EXPECTED_EVALUABLE_ROWS:,})"
)

print(
    f"Policy path                        : "
    f"{POLICY_JSON}"
)

print(
    f"Policy SHA-256                     : "
    f"PASS ({policy_sha256})"
)

print(
    f"Policy sidecar file SHA-256        : "
    f"PASS ({sidecar_sha256})"
)

print(
    "GES or comparator scores loaded    : No"
)

print(
    "Predictive performance calculated  : No"
)

print(
    "Record-level derived artifact written: No"
)

print(
    "\nFROZEN OPERATIONAL DEFINITIONS"
)

print(
    json.dumps(
        policy,
        ensure_ascii=False,
        sort_keys=True,
        indent=2,
    )
)

print(
    "\nT0 BASELINE-GROUP ACCOUNTING"
)

print(
    baseline_group_accounting.to_string(
        index=False
    )
)

print(
    "\nNESTED REVIEW-STATUS AND "
    "CLASSIFICATION-GROUP VOCABULARY"
)

print(
    nested_vocabulary.to_string(
        index=False
    )
)

print(
    "\nSCV TRANSITION SUMMARY"
)

print(
    scv_transition_summary.to_string(
        index=False
    )
)

print(
    "\nNEW CONTRADICTORY HIGH-RIGOR "
    "OUTCOME STATUS ACCOUNTING"
)

print(
    high_rigor_status_accounting.to_string(
        index=False
    )
)

print(
    "\nSUBMITTER-CLASSIFICATION "
    "DISTRIBUTION STATUS ACCOUNTING"
)

print(
    distribution_status_accounting.to_string(
        index=False
    )
)

print(
    "\nTOTAL-VARIATION-DISTANCE SUMMARY"
)

print(
    tvd_summary.to_string(
        index=False
    )
)

print(
    "\nTOTAL-VARIATION-DISTANCE DISTRIBUTION"
)

if len(
    tvd_distribution
):
    print(
        tvd_distribution.to_string(
            index=False
        )
    )
else:
    print(
        "No complete-case distribution rows were available."
    )

print(
    "\nDERIVABILITY DECISION"
)

print(
    derivability_decision.to_string(
        index=False
    )
)

print(
    "\nINTERPRETATION BOUNDARY"
)

print("-" * 150)

print(
    "Missing nested T1 review_status or classification_group evidence "
    "is never converted into a negative outcome. High-rigor and "
    "submitter-distribution analyses may therefore be censored or declared "
    "non-estimable rather than filled from aggregate RCV fields."
)

print(
    "This preserves the prespecified submission-level meaning and avoids "
    "manufacturing evidence from aggregate review stars or aggregate "
    "classification."
)

print(
    "\nCELL DECISION"
)

print("-" * 150)

print(
    "PASS_STAGE6C_NESTED_SCV_OUTCOME_POLICY_V1_1_"
    "FROZEN_AND_DERIVABILITY_AUDITED"
)

print(
    "The exact nested-SCV operational policy was checksum-frozen before "
    "score access, and both remaining secondary targets were derived in "
    "memory solely to determine honest evaluability, censoring, and class "
    "variation."
)

print(
    "The next cell may freeze a record-level nested-SCV outcome package "
    "only for targets declared estimable above; non-estimable targets must "
    "be reported as data-limited rather than forced into performance analysis."
)


STAGE 6C STEP 3G — CELL 6C-3G0B v1.1 — NESTED-SCV POLICY FREEZE AND DERIVABILITY AUDIT
Stage 6B evaluable SHA-256         : PASS (c6ad50dfc376a746e1830d000dba975101d88763fd61b8d1983f6300edb6e038)
Accepted T0 SHA-256                : PASS (f6b6760b2ad6e4352e3bdecdeaf89827e8a514b031abf2373bb17568d999466d)
Accepted T1 SHA-256                : PASS (5713a11bdbf4804758cc011f9b2f302afc91fa1f88c1b178d675c28bb277d37c)
Mapped evaluable T0/T1 rows        : PASS (66,636/66,636)
Policy path                        : /content/drive/MyDrive/GES_RAG_Temporal_Study/configs/stage6c_nested_scv_secondary_outcomes/stage6c_nested_scv_secondary_outcome_policy_v1_1.json
Policy SHA-256                     : PASS (0d11d2eec8a3dcc0116a0ab45862a73b46ce130b79df1d9583910232bedb93aa)
Policy sidecar file SHA-256        : PASS (7c5d12dabe1a0ce18216a61bc3addae7f9835e842140372d7d723aab17886e4e)
GES or comparator scores loaded    : No
Predictive performance calculated  : No
Record-level derived artifact written: No

FRO

/tmp/ipykernel_527/3514243261.py:1935: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  ].fillna(0).astype(int).sum()
/tmp/ipykernel_527/3514243261.py:1943: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  ].fillna(0).astype(int).sum()


In [43]:
# ==================================================================================================
# STAGE 6C STEP 3G — CELL 6C-3G0B v1.1A
# NESTED-SCV SECONDARY-OUTCOME POLICY FREEZE AND DERIVABILITY AUDIT
#
# Purpose:
#   1. Freshly verify the immutable Stage 6B evaluable cohort and accepted T0/T1 RCV sources.
#   2. Freeze exact, score-blind operational definitions for:
#        a. a new contradictory high-rigor SCV submission;
#        b. magnitude of submitter-classification distribution change.
#   3. Write and freshly reverify only the deterministic policy JSON and SHA-256 sidecar.
#   4. Derive the two outcomes in memory solely to audit whether the accepted nested schema
#      supports them and to determine evaluable/censored accounting.
#   5. Load no GES or comparator score and calculate no predictive-performance result.
#
# Scientific boundary:
#   T1 nested review_status is mostly missing, and T1 nested classification_group may contain
#   the explicit "Missing" sentinel. A negative label is therefore permitted only when the
#   evidence needed to rule out the event is observed. Otherwise the record is censored.
#
# This cell does NOT write a record-level outcome table. That can occur only after this
# policy and its derivability accounting pass.
# ==================================================================================================

from pathlib import Path
from collections import Counter
import hashlib
import json
import os
import re
import tempfile

import numpy as np
import pandas as pd
import pyarrow.parquet as pq


# --------------------------------------------------------------------------------------------------
# 1. FROZEN PATHS, HASHES, AND SCHEMA
# --------------------------------------------------------------------------------------------------

PROJECT_DIR = Path(
    "/content/drive/MyDrive/GES_RAG_Temporal_Study"
)

STAGE6_DIR = (
    PROJECT_DIR
    / "data_processed"
    / "stage6_temporal_validation"
)

EVALUABLE_PARQUET = (
    STAGE6_DIR
    / "stage6b_locked_primary_evaluable_cohort_v1.parquet"
)

EVALUABLE_SIDECAR = Path(
    str(EVALUABLE_PARQUET) + ".sha256"
)

T0_PARQUET = (
    PROJECT_DIR
    / "data_interim"
    / "t0_rcv_target_genes_corrected_v1_2.parquet"
)

T1_PARQUET = (
    PROJECT_DIR
    / "data_interim"
    / "t1_rcv_target_genes_harmonized_v1.parquet"
)

POLICY_DIR = (
    PROJECT_DIR
    / "configs"
    / "stage6c_nested_scv_secondary_outcomes"
)

PRIOR_POLICY_JSON = (
    POLICY_DIR
    / "stage6c_nested_scv_secondary_outcome_policy_v1.json"
)

PRIOR_POLICY_SIDECAR = Path(
    str(PRIOR_POLICY_JSON) + ".sha256"
)

POLICY_JSON = (
    POLICY_DIR
    / "stage6c_nested_scv_secondary_outcome_policy_v1_1.json"
)

POLICY_SIDECAR = Path(
    str(POLICY_JSON) + ".sha256"
)

EXPECTED_EVALUABLE_SHA256 = (
    "c6ad50dfc376a746e1830d000dba975101d88763fd61b8d1983f6300edb6e038"
)

EXPECTED_T0_SHA256 = (
    "f6b6760b2ad6e4352e3bdecdeaf89827e8a514b031abf2373bb17568d999466d"
)

EXPECTED_T1_SHA256 = (
    "5713a11bdbf4804758cc011f9b2f302afc91fa1f88c1b178d675c28bb277d37c"
)

EXPECTED_EVALUABLE_ROWS = 66_636
EXPECTED_EVALUABLE_COLUMNS = 79
EXPECTED_T0_ROWS = 71_659
EXPECTED_T0_COLUMNS = 34
EXPECTED_T1_ROWS = 100_920
EXPECTED_T1_COLUMNS = 36

STAGE6_T0_RCV_COLUMN = "rcv_accession"
STAGE6_T1_RCV_COLUMN = "linked_t1_rcv_accession"
STAGE6_ROW_ORDER_COLUMN = "t0_row_order"
STAGE6_BASELINE_GROUP_COLUMN = "t0_canonical_classification_group"

SOURCE_RCV_COLUMN = "rcv_accession"
NESTED_SCV_COLUMN = "scv_records_json"

SCV_ID_KEY = "scv_accession"
SUBMITTER_KEY = "submitter"
CLASSIFICATION_GROUP_KEY = "classification_group"
REVIEW_STATUS_KEY = "review_status"

INFORMATIVE_GROUPS = ("BLB", "VUS", "PLP")
MAJOR_DISTRIBUTION_SHIFT_THRESHOLD = 0.50


# --------------------------------------------------------------------------------------------------
# 2. HELPERS
# --------------------------------------------------------------------------------------------------

def sha256_file(
    path: Path,
    chunk_size: int = 1024 * 1024,
) -> str:
    digest = hashlib.sha256()

    with path.open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


def sha256_bytes(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()


def read_sidecar_hash(path: Path) -> str:
    text = path.read_text(
        encoding="utf-8"
    ).strip()

    matches = re.findall(
        r"\b[a-fA-F0-9]{64}\b",
        text,
    )

    if not matches:
        raise ValueError(
            f"No SHA-256 value found in sidecar: {path}"
        )

    return matches[0].lower()


def normalize_rcv(
    series: pd.Series,
) -> pd.Series:
    return (
        series.astype("string")
        .str.upper()
        .str.strip()
        .str.extract(
            r"(RCV\d+)",
            expand=False,
        )
    )


def normalize_scv_accession(
    value,
) -> str | None:
    if value is None:
        return None

    text = str(value).upper().strip()

    match = re.search(
        r"(SCV\d+)",
        text,
    )

    if not match:
        return None

    return match.group(1)


def normalize_text(
    value,
) -> str | None:
    if value is None:
        return None

    text = re.sub(
        r"\s+",
        " ",
        str(value),
    ).strip()

    if text == "":
        return None

    return text


def normalize_review_status(
    value,
) -> str | None:
    text = normalize_text(value)

    if text is None:
        return None

    return text.lower()


def is_high_rigor_review_status(
    normalized_status: str | None,
) -> bool:
    if normalized_status is None:
        return False

    # Exclusion precedence is explicit so that
    # "no assertion criteria provided" is never misclassified merely
    # because it contains the phrase "criteria provided".
    if (
        normalized_status.startswith(
            "no assertion criteria"
        )
        or normalized_status.startswith(
            "no assertion provided"
        )
    ):
        return False

    return (
        normalized_status.startswith(
            "criteria provided"
        )
        or "reviewed by expert panel"
        in normalized_status
        or "practice guideline"
        in normalized_status
    )


def normalize_classification_group(
    value,
) -> str:
    text = normalize_text(value)

    if text is None:
        return "MISSING"

    normalized = (
        text.lower()
        .replace("_", " ")
        .replace("-", " ")
    )

    normalized = re.sub(
        r"\s+",
        " ",
        normalized,
    ).strip()

    if normalized in {
        "blb",
        "benign",
        "likely benign",
        "benign/likely benign",
        "benign likely benign",
    }:
        return "BLB"

    if normalized in {
        "vus",
        "uncertain significance",
        "uncertain",
    }:
        return "VUS"

    if normalized in {
        "plp",
        "pathogenic",
        "likely pathogenic",
        "pathogenic/likely pathogenic",
        "pathogenic likely pathogenic",
    }:
        return "PLP"

    if normalized in {
        "missing",
        "none",
        "nan",
        "not provided",
        "no classification",
        "noclassification",
    }:
        return "MISSING"

    if "conflict" in normalized:
        return "CONFLICTING"

    return "OTHER"


def parse_nested_scv_value(
    value,
    row_label: str,
) -> list[dict]:
    if isinstance(value, list):
        parsed = value

    elif isinstance(value, tuple):
        parsed = list(value)

    elif isinstance(value, np.ndarray):
        parsed = value.tolist()

    elif isinstance(value, str):
        text = value.strip()

        if text == "":
            raise ValueError(
                f"Blank nested SCV JSON at {row_label}"
            )

        try:
            parsed = json.loads(text)
        except Exception as error:
            raise ValueError(
                f"Malformed nested SCV JSON at "
                f"{row_label}: {error}"
            ) from error

    else:
        try:
            if pd.isna(value):
                raise ValueError(
                    f"Missing nested SCV evidence at {row_label}"
                )
        except Exception:
            pass

        raise TypeError(
            f"Unsupported nested SCV type at "
            f"{row_label}: {type(value).__name__}"
        )

    if not isinstance(parsed, list):
        raise TypeError(
            f"Nested SCV value is not a list at "
            f"{row_label}: {type(parsed).__name__}"
        )

    for position, record in enumerate(parsed):
        if not isinstance(record, dict):
            raise TypeError(
                f"Nested SCV record is not a dictionary "
                f"at {row_label}, position {position}."
            )

    return parsed


def canonical_json_bytes(
    value: dict,
) -> bytes:
    text = json.dumps(
        value,
        ensure_ascii=False,
        sort_keys=True,
        indent=2,
    ) + "\n"

    return text.encode("utf-8")


def immutable_write_or_verify(
    path: Path,
    expected_bytes: bytes,
) -> str:
    expected_hash = sha256_bytes(
        expected_bytes
    )

    if path.exists():
        observed_bytes = path.read_bytes()

        if observed_bytes != expected_bytes:
            raise FileExistsError(
                f"An existing immutable artifact differs "
                f"from the expected content:\n{path}"
            )

        observed_hash = sha256_bytes(
            observed_bytes
        )

        if observed_hash != expected_hash:
            raise AssertionError(
                f"Existing artifact hash mismatch:\n{path}"
            )

        return observed_hash

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    with tempfile.NamedTemporaryFile(
        mode="wb",
        dir=str(path.parent),
        prefix=path.name + ".tmp.",
        delete=False,
    ) as temporary:
        temporary.write(
            expected_bytes
        )
        temporary.flush()
        os.fsync(
            temporary.fileno()
        )
        temporary_path = Path(
            temporary.name
        )

    try:
        os.replace(
            temporary_path,
            path,
        )
    finally:
        if temporary_path.exists():
            temporary_path.unlink()

    observed_hash = sha256_file(
        path
    )

    if observed_hash != expected_hash:
        raise AssertionError(
            f"Fresh artifact write/readback mismatch:\n{path}"
        )

    return observed_hash


def distribution_vector(
    normalized_groups: list[str],
) -> np.ndarray:
    counts = Counter(
        normalized_groups
    )

    total = sum(
        counts[group]
        for group in INFORMATIVE_GROUPS
    )

    if total <= 0:
        raise ValueError(
            "Cannot construct an informative distribution "
            "with zero BLB/VUS/PLP SCVs."
        )

    return np.array(
        [
            counts[group] / total
            for group in INFORMATIVE_GROUPS
        ],
        dtype=float,
    )


def total_variation_distance(
    first: np.ndarray,
    second: np.ndarray,
) -> float:
    return float(
        0.5
        * np.abs(
            np.asarray(first, dtype=float)
            - np.asarray(second, dtype=float)
        ).sum()
    )


# --------------------------------------------------------------------------------------------------
# 3. CRYPTOGRAPHIC AND STRUCTURAL VERIFICATION
# --------------------------------------------------------------------------------------------------

for path in [
    EVALUABLE_PARQUET,
    EVALUABLE_SIDECAR,
    T0_PARQUET,
    T1_PARQUET,
]:
    if not path.exists():
        raise FileNotFoundError(
            f"Missing required frozen artifact:\n{path}"
        )

observed_evaluable_sha256 = sha256_file(
    EVALUABLE_PARQUET
)

if (
    observed_evaluable_sha256
    != EXPECTED_EVALUABLE_SHA256
):
    raise AssertionError(
        "Stage 6B evaluable SHA-256 mismatch."
    )

if (
    read_sidecar_hash(
        EVALUABLE_SIDECAR
    )
    != observed_evaluable_sha256
):
    raise AssertionError(
        "Stage 6B evaluable sidecar mismatch."
    )

observed_t0_sha256 = sha256_file(
    T0_PARQUET
)

observed_t1_sha256 = sha256_file(
    T1_PARQUET
)

if observed_t0_sha256 != EXPECTED_T0_SHA256:
    raise AssertionError(
        "Accepted T0 SHA-256 mismatch."
    )

if observed_t1_sha256 != EXPECTED_T1_SHA256:
    raise AssertionError(
        "Accepted T1 SHA-256 mismatch."
    )

stage6_file = pq.ParquetFile(
    EVALUABLE_PARQUET
)

t0_file = pq.ParquetFile(
    T0_PARQUET
)

t1_file = pq.ParquetFile(
    T1_PARQUET
)

if (
    stage6_file.metadata.num_rows
    != EXPECTED_EVALUABLE_ROWS
    or stage6_file.metadata.num_columns
    != EXPECTED_EVALUABLE_COLUMNS
):
    raise AssertionError(
        "Stage 6B dimensions mismatch."
    )

if (
    t0_file.metadata.num_rows
    != EXPECTED_T0_ROWS
    or t0_file.metadata.num_columns
    != EXPECTED_T0_COLUMNS
):
    raise AssertionError(
        "Accepted T0 dimensions mismatch."
    )

if (
    t1_file.metadata.num_rows
    != EXPECTED_T1_ROWS
    or t1_file.metadata.num_columns
    != EXPECTED_T1_COLUMNS
):
    raise AssertionError(
        "Accepted T1 dimensions mismatch."
    )

required_stage6_columns = [
    STAGE6_T0_RCV_COLUMN,
    STAGE6_T1_RCV_COLUMN,
    STAGE6_ROW_ORDER_COLUMN,
    STAGE6_BASELINE_GROUP_COLUMN,
]

missing_stage6_columns = [
    column
    for column in required_stage6_columns
    if column
    not in stage6_file.schema_arrow.names
]

if missing_stage6_columns:
    raise KeyError(
        "Missing required Stage 6B columns:\n"
        + "\n".join(
            missing_stage6_columns
        )
    )

for label, schema in [
    (
        "T0",
        t0_file.schema_arrow.names,
    ),
    (
        "T1",
        t1_file.schema_arrow.names,
    ),
]:
    missing = [
        column
        for column in [
            SOURCE_RCV_COLUMN,
            NESTED_SCV_COLUMN,
        ]
        if column not in schema
    ]

    if missing:
        raise KeyError(
            f"{label} source is missing:\n"
            + "\n".join(missing)
        )


# --------------------------------------------------------------------------------------------------
# 4. CREATE AND FREEZE THE SCORE-BLIND OPERATIONAL POLICY
# --------------------------------------------------------------------------------------------------

# Cell 6C-3G0B v1 wrote its policy before the derivability audit discovered that the
# primary-evaluable cohort legitimately contains CONFLICTING and OTHER baseline groups.
# That earlier artifact is preserved unchanged. Version 1.1 explicitly defines how those
# baseline groups are handled and never overwrites the prior policy.
if PRIOR_POLICY_JSON.exists():
    prior_policy_sha256 = sha256_file(
        PRIOR_POLICY_JSON
    )

    if PRIOR_POLICY_SIDECAR.exists():
        prior_sidecar_target_sha256 = read_sidecar_hash(
            PRIOR_POLICY_SIDECAR
        )

        if prior_sidecar_target_sha256 != prior_policy_sha256:
            raise AssertionError(
                "Prior v1 policy sidecar does not match the prior policy artifact."
            )
    else:
        prior_sidecar_target_sha256 = None

    prior_policy_provenance = {
        "filename": PRIOR_POLICY_JSON.name,
        "sha256": prior_policy_sha256,
        "sidecar_present": PRIOR_POLICY_SIDECAR.exists(),
        "status": "PRESERVED_UNCHANGED",
    }
else:
    prior_policy_provenance = {
        "filename": PRIOR_POLICY_JSON.name,
        "sha256": None,
        "sidecar_present": False,
        "status": "NOT_PRESENT_IN_THIS_RUNTIME",
    }

policy = {
    "policy_name": (
        "stage6c_nested_scv_secondary_outcome_policy"
    ),
    "policy_version": "1.1",
    "status": (
        "FROZEN_BEFORE_NESTED_SCV_SCORE_ANALYSIS"
    ),
    "supersedes": {
        "prior_policy": prior_policy_provenance,
        "reason": (
            "The failed score-blind derivability audit showed that the frozen "
            "primary-evaluable cohort includes valid CONFLICTING and OTHER "
            "baseline groups. Version 1.1 explicitly censors new-submission "
            "contradiction assessment when a new SCV exists but the baseline "
            "group is not BLB, VUS, or PLP. The prior policy is preserved."
        ),
    },
    "study_unit": (
        "RCV-level variant-condition aggregate record"
    ),
    "accepted_sources": {
        "stage6b_primary_evaluable_cohort": {
            "filename": EVALUABLE_PARQUET.name,
            "sha256": EXPECTED_EVALUABLE_SHA256,
            "rows": EXPECTED_EVALUABLE_ROWS,
            "columns": EXPECTED_EVALUABLE_COLUMNS,
        },
        "t0_rcv_source": {
            "filename": T0_PARQUET.name,
            "sha256": EXPECTED_T0_SHA256,
            "rows": EXPECTED_T0_ROWS,
            "columns": EXPECTED_T0_COLUMNS,
        },
        "t1_rcv_source": {
            "filename": T1_PARQUET.name,
            "sha256": EXPECTED_T1_SHA256,
            "rows": EXPECTED_T1_ROWS,
            "columns": EXPECTED_T1_COLUMNS,
        },
    },
    "nested_keys": {
        "scv_identity": SCV_ID_KEY,
        "submitter_identity": SUBMITTER_KEY,
        "classification_group": CLASSIFICATION_GROUP_KEY,
        "review_rigor": REVIEW_STATUS_KEY,
    },
    "classification_group_mapping": {
        "BLB": [
            "Benign",
            "Likely benign",
            "Benign/Likely benign",
        ],
        "VUS": [
            "Uncertain significance",
            "VUS",
        ],
        "PLP": [
            "Pathogenic",
            "Likely pathogenic",
            "Pathogenic/Likely pathogenic",
        ],
        "MISSING": [
            "Missing",
            "NoClassification",
            "null/blank",
        ],
        "CONFLICTING": [
            "values containing conflict",
        ],
        "OTHER": [
            "all remaining values",
        ],
    },
    "high_rigor_rule": {
        "positive_review_status_logic": [
            "normalized status starts with 'criteria provided'",
            "normalized status contains 'reviewed by expert panel'",
            "normalized status contains 'practice guideline'",
        ],
        "explicit_exclusion_precedence": [
            "normalized status starts with 'no assertion criteria'",
            "normalized status starts with 'no assertion provided'",
        ],
        "missing_review_status": (
            "unknown, never assumed low-rigor"
        ),
    },
    "new_contradictory_high_rigor_submission": {
        "new_submission": (
            "normalized T1 SCV accession absent from the "
            "same linked RCV's T0 SCV-accession set"
        ),
        "baseline_interpretation": (
            "frozen Stage 6B t0_canonical_classification_group; BLB, VUS, "
            "and PLP are informative single-group baselines; CONFLICTING, "
            "OTHER, and MISSING are noninformative for contradiction assessment"
        ),
        "contradiction": (
            "for an informative BLB/VUS/PLP baseline, a new T1 SCV normalized "
            "group is BLB, VUS, or PLP and differs from that baseline group"
        ),
        "positive": (
            "at least one new T1 SCV is high-rigor and contradictory"
        ),
        "negative": (
            "no new T1 SCV exists regardless of baseline group, or the baseline "
            "is informative and every new T1 SCV can be fully evaluated and none "
            "is both high-rigor and contradictory"
        ),
        "censored": (
            "when at least one new T1 SCV exists: the T0 baseline group is "
            "CONFLICTING/OTHER/MISSING, or no positive event is observed and at "
            "least one new T1 SCV has missing review status, or a high-rigor new "
            "SCV has a noninformative classification group"
        ),
        "updated_existing_scv": (
            "an existing SCV accession with a later version is not a "
            "new submission and is excluded from this outcome"
        ),
    },
    "submitter_classification_distribution_change": {
        "distribution_unit": (
            "one nested SCV classification assertion"
        ),
        "categories": list(
            INFORMATIVE_GROUPS
        ),
        "complete_case_evaluability": (
            "every nested SCV at both T0 and T1 must map to BLB, VUS, "
            "or PLP; otherwise the distribution magnitude is censored"
        ),
        "magnitude": (
            "total variation distance = 0.5 * sum(abs(p_T1 - p_T0))"
        ),
        "range": [
            0.0,
            1.0,
        ],
        "major_shift_binary_sensitivity": (
            "total variation distance >= 0.50"
        ),
        "major_shift_threshold": (
            MAJOR_DISTRIBUTION_SHIFT_THRESHOLD
        ),
        "threshold_rationale": (
            "a fixed, interpretable threshold requiring at least half "
            "of classification probability mass to move; not selected "
            "from GES performance"
        ),
    },
    "analysis_role": {
        "new_contradictory_high_rigor_submission": (
            "exploratory secondary evidence-drift outcome"
        ),
        "distribution_magnitude": (
            "exploratory continuous secondary evidence-drift measure"
        ),
        "major_distribution_shift": (
            "exploratory binary sensitivity outcome"
        ),
    },
    "leakage_protection": {
        "ges_or_comparator_scores_loaded_during_policy_freeze": False,
        "threshold_selected_from_performance": False,
        "outcome_definition_selected_from_performance": False,
    },
}

policy_bytes = canonical_json_bytes(
    policy
)

policy_sha256 = immutable_write_or_verify(
    POLICY_JSON,
    policy_bytes,
)

sidecar_bytes = (
    f"{policy_sha256}  {POLICY_JSON.name}\n"
).encode("utf-8")

sidecar_sha256 = immutable_write_or_verify(
    POLICY_SIDECAR,
    sidecar_bytes,
)

if sha256_file(
    POLICY_JSON
) != policy_sha256:
    raise AssertionError(
        "Policy fresh readback verification failed."
    )

if read_sidecar_hash(
    POLICY_SIDECAR
) != policy_sha256:
    raise AssertionError(
        "Policy sidecar fresh readback verification failed."
    )


# --------------------------------------------------------------------------------------------------
# 5. LOAD ONLY IDENTIFIERS, BASELINE GROUP, AND NESTED EVIDENCE
# --------------------------------------------------------------------------------------------------

stage6 = pd.read_parquet(
    EVALUABLE_PARQUET,
    columns=required_stage6_columns,
).copy()

stage6[
    STAGE6_T0_RCV_COLUMN
] = normalize_rcv(
    stage6[
        STAGE6_T0_RCV_COLUMN
    ]
)

stage6[
    STAGE6_T1_RCV_COLUMN
] = normalize_rcv(
    stage6[
        STAGE6_T1_RCV_COLUMN
    ]
)

stage6[
    STAGE6_BASELINE_GROUP_COLUMN
] = (
    stage6[
        STAGE6_BASELINE_GROUP_COLUMN
    ]
    .map(
        normalize_classification_group
    )
)

if stage6[
    STAGE6_T0_RCV_COLUMN
].isna().any():
    raise AssertionError(
        "A Stage 6B T0 RCV failed normalization."
    )

if stage6[
    STAGE6_T1_RCV_COLUMN
].isna().any():
    raise AssertionError(
        "A Stage 6B T1 RCV failed normalization."
    )

if (
    stage6[
        STAGE6_T0_RCV_COLUMN
    ].nunique()
    != EXPECTED_EVALUABLE_ROWS
):
    raise AssertionError(
        "Stage 6B T0 RCV keys are not unique."
    )

if (
    stage6[
        STAGE6_T1_RCV_COLUMN
    ].nunique()
    != EXPECTED_EVALUABLE_ROWS
):
    raise AssertionError(
        "Stage 6B linked T1 RCV keys are not unique."
    )

baseline_group_accounting = (
    stage6[
        STAGE6_BASELINE_GROUP_COLUMN
    ]
    .value_counts(
        dropna=False
    )
    .rename_axis(
        "t0_baseline_group"
    )
    .reset_index(
        name="rcv_rows"
    )
)

baseline_group_accounting[
    "informative_for_contradiction"
] = baseline_group_accounting[
    "t0_baseline_group"
].isin(
    INFORMATIVE_GROUPS
)

allowed_baseline_groups = set(
    INFORMATIVE_GROUPS
) | {
    "CONFLICTING",
    "OTHER",
    "MISSING",
}

unexpected_baseline_groups = sorted(
    set(
        stage6[
            STAGE6_BASELINE_GROUP_COLUMN
        ].unique().tolist()
    )
    - allowed_baseline_groups
)

if unexpected_baseline_groups:
    print(
        "\nBASELINE GROUP INVENTORY"
    )

    print(
        baseline_group_accounting.to_string(
            index=False
        )
    )

    raise AssertionError(
        "Unexpected normalized T0 baseline groups: "
        + ", ".join(
            unexpected_baseline_groups
        )
    )

t0_source = pd.read_parquet(
    T0_PARQUET,
    columns=[
        SOURCE_RCV_COLUMN,
        NESTED_SCV_COLUMN,
    ],
).copy()

t1_source = pd.read_parquet(
    T1_PARQUET,
    columns=[
        SOURCE_RCV_COLUMN,
        NESTED_SCV_COLUMN,
    ],
).copy()

t0_source[
    SOURCE_RCV_COLUMN
] = normalize_rcv(
    t0_source[
        SOURCE_RCV_COLUMN
    ]
)

t1_source[
    SOURCE_RCV_COLUMN
] = normalize_rcv(
    t1_source[
        SOURCE_RCV_COLUMN
    ]
)

if (
    t0_source[
        SOURCE_RCV_COLUMN
    ].nunique()
    != EXPECTED_T0_ROWS
):
    raise AssertionError(
        "T0 source RCV keys are not unique."
    )

if (
    t1_source[
        SOURCE_RCV_COLUMN
    ].nunique()
    != EXPECTED_T1_ROWS
):
    raise AssertionError(
        "T1 source RCV keys are not unique."
    )

mapped = (
    stage6.merge(
        t0_source.rename(
            columns={
                SOURCE_RCV_COLUMN: (
                    "_t0_source_rcv"
                ),
                NESTED_SCV_COLUMN: (
                    "_t0_scv_records_json"
                ),
            }
        ),
        left_on=STAGE6_T0_RCV_COLUMN,
        right_on="_t0_source_rcv",
        how="left",
        validate="one_to_one",
        indicator="_t0_merge",
    )
    .merge(
        t1_source.rename(
            columns={
                SOURCE_RCV_COLUMN: (
                    "_t1_source_rcv"
                ),
                NESTED_SCV_COLUMN: (
                    "_t1_scv_records_json"
                ),
            }
        ),
        left_on=STAGE6_T1_RCV_COLUMN,
        right_on="_t1_source_rcv",
        how="left",
        validate="one_to_one",
        indicator="_t1_merge",
    )
    .sort_values(
        STAGE6_ROW_ORDER_COLUMN,
        kind="mergesort",
    )
    .reset_index(drop=True)
)

if not mapped[
    "_t0_merge"
].eq("both").all():
    raise AssertionError(
        "An evaluable T0 RCV failed source mapping."
    )

if not mapped[
    "_t1_merge"
].eq("both").all():
    raise AssertionError(
        "An evaluable T1 RCV failed source mapping."
    )

if len(mapped) != EXPECTED_EVALUABLE_ROWS:
    raise AssertionError(
        "Mapped evaluable row count mismatch."
    )


# --------------------------------------------------------------------------------------------------
# 6. SCORE-BLIND IN-MEMORY DERIVATION AND DERIVABILITY AUDIT
# --------------------------------------------------------------------------------------------------

t0_review_status_counts = Counter()
t1_review_status_counts = Counter()
t0_group_counts = Counter()
t1_group_counts = Counter()

record_rows = []

derivation_columns = [
    STAGE6_T0_RCV_COLUMN,
    STAGE6_T1_RCV_COLUMN,
    STAGE6_ROW_ORDER_COLUMN,
    STAGE6_BASELINE_GROUP_COLUMN,
    "_t0_scv_records_json",
    "_t1_scv_records_json",
]

for (
    t0_rcv,
    t1_rcv,
    t0_row_order,
    baseline_group,
    t0_scv_records_json,
    t1_scv_records_json,
) in mapped[
    derivation_columns
].itertuples(
    index=False,
    name=None,
):
    baseline_group_informative = (
        baseline_group
        in INFORMATIVE_GROUPS
    )

    t0_records = parse_nested_scv_value(
        t0_scv_records_json,
        f"T0:{t0_rcv}",
    )

    t1_records = parse_nested_scv_value(
        t1_scv_records_json,
        f"T1:{t1_rcv}",
    )

    t0_accessions = set()

    t0_normalized_groups = []
    t1_normalized_groups = []

    for record in t0_records:
        accession = normalize_scv_accession(
            record.get(
                SCV_ID_KEY
            )
        )

        if accession is None:
            raise AssertionError(
                f"T0 nested SCV lacks a valid accession: "
                f"{t0_rcv}"
            )

        if accession in t0_accessions:
            raise AssertionError(
                f"Duplicate T0 SCV accession within RCV "
                f"{t0_rcv}: {accession}"
            )

        t0_accessions.add(
            accession
        )

        status = normalize_review_status(
            record.get(
                REVIEW_STATUS_KEY
            )
        )

        t0_review_status_counts[
            status
            if status is not None
            else "<MISSING>"
        ] += 1

        group = normalize_classification_group(
            record.get(
                CLASSIFICATION_GROUP_KEY
            )
        )

        t0_group_counts[
            group
        ] += 1

        t0_normalized_groups.append(
            group
        )

    t1_accessions = set()

    new_scv_count = 0
    new_scv_missing_review_status = 0
    new_high_rigor_count = 0
    new_high_rigor_noninformative_group = 0
    new_high_rigor_contradictory_count = 0
    new_high_rigor_concordant_count = 0
    new_high_rigor_baseline_not_comparable_count = 0
    updated_existing_scv_count = 0

    t0_version_by_accession = {
        normalize_scv_accession(
            record.get(
                SCV_ID_KEY
            )
        ): str(
            record.get(
                "scv_version"
            )
        )
        for record in t0_records
    }

    for record in t1_records:
        accession = normalize_scv_accession(
            record.get(
                SCV_ID_KEY
            )
        )

        if accession is None:
            raise AssertionError(
                f"T1 nested SCV lacks a valid accession: "
                f"{t1_rcv}"
            )

        if accession in t1_accessions:
            raise AssertionError(
                f"Duplicate T1 SCV accession within RCV "
                f"{t1_rcv}: {accession}"
            )

        t1_accessions.add(
            accession
        )

        status = normalize_review_status(
            record.get(
                REVIEW_STATUS_KEY
            )
        )

        t1_review_status_counts[
            status
            if status is not None
            else "<MISSING>"
        ] += 1

        group = normalize_classification_group(
            record.get(
                CLASSIFICATION_GROUP_KEY
            )
        )

        t1_group_counts[
            group
        ] += 1

        t1_normalized_groups.append(
            group
        )

        if accession in t0_accessions:
            t0_version = t0_version_by_accession.get(
                accession
            )

            t1_version = str(
                record.get(
                    "scv_version"
                )
            )

            if t0_version != t1_version:
                updated_existing_scv_count += 1

            continue

        new_scv_count += 1

        if status is None:
            new_scv_missing_review_status += 1
            continue

        if not is_high_rigor_review_status(
            status
        ):
            continue

        new_high_rigor_count += 1

        if group not in INFORMATIVE_GROUPS:
            new_high_rigor_noninformative_group += 1
            continue

        if not baseline_group_informative:
            new_high_rigor_baseline_not_comparable_count += 1
            continue

        if group != baseline_group:
            new_high_rigor_contradictory_count += 1
        else:
            new_high_rigor_concordant_count += 1

    if new_scv_count == 0:
        # No new submission means the requested event cannot occur, even when
        # the aggregate T0 baseline group is CONFLICTING or OTHER.
        high_rigor_outcome_status = "NEGATIVE_NO_NEW_SCV"
        high_rigor_outcome = 0

    elif not baseline_group_informative:
        high_rigor_outcome_status = (
            "CENSORED_NONINFORMATIVE_BASELINE_GROUP"
        )
        high_rigor_outcome = pd.NA

    elif (
        new_high_rigor_contradictory_count
        > 0
    ):
        high_rigor_outcome_status = "POSITIVE"
        high_rigor_outcome = 1

    elif (
        new_scv_missing_review_status
        > 0
        or new_high_rigor_noninformative_group
        > 0
    ):
        high_rigor_outcome_status = (
            "CENSORED_REQUIRED_NESTED_FIELD_MISSING"
        )
        high_rigor_outcome = pd.NA

    else:
        high_rigor_outcome_status = "NEGATIVE_EVALUABLE_NEW_SCV"
        high_rigor_outcome = 0

    t0_complete_distribution = all(
        group in INFORMATIVE_GROUPS
        for group in t0_normalized_groups
    )

    t1_complete_distribution = all(
        group in INFORMATIVE_GROUPS
        for group in t1_normalized_groups
    )

    distribution_evaluable = (
        t0_complete_distribution
        and t1_complete_distribution
        and len(
            t0_normalized_groups
        ) > 0
        and len(
            t1_normalized_groups
        ) > 0
    )

    if distribution_evaluable:
        t0_distribution = distribution_vector(
            t0_normalized_groups
        )

        t1_distribution = distribution_vector(
            t1_normalized_groups
        )

        distribution_tvd = total_variation_distance(
            t0_distribution,
            t1_distribution,
        )

        major_distribution_shift = int(
            distribution_tvd
            >= MAJOR_DISTRIBUTION_SHIFT_THRESHOLD
        )

        distribution_status = "EVALUABLE"

    else:
        distribution_tvd = np.nan
        major_distribution_shift = pd.NA
        distribution_status = (
            "CENSORED_NONINFORMATIVE_NESTED_CLASSIFICATION"
        )

    record_rows.append(
        {
            "t0_rcv_accession": t0_rcv,
            "linked_t1_rcv_accession": t1_rcv,
            "t0_row_order": t0_row_order,
            "t0_baseline_group": baseline_group,
            "t0_baseline_group_informative_for_contradiction": (
                baseline_group_informative
            ),
            "t0_scv_count": len(
                t0_records
            ),
            "t1_scv_count": len(
                t1_records
            ),
            "new_scv_count": new_scv_count,
            "updated_existing_scv_count": (
                updated_existing_scv_count
            ),
            "new_scv_missing_review_status": (
                new_scv_missing_review_status
            ),
            "new_high_rigor_scv_count": (
                new_high_rigor_count
            ),
            "new_high_rigor_noninformative_group_count": (
                new_high_rigor_noninformative_group
            ),
            "new_high_rigor_concordant_count": (
                new_high_rigor_concordant_count
            ),
            "new_high_rigor_baseline_not_comparable_count": (
                new_high_rigor_baseline_not_comparable_count
            ),
            "new_high_rigor_contradictory_count": (
                new_high_rigor_contradictory_count
            ),
            "new_contradictory_high_rigor_status": (
                high_rigor_outcome_status
            ),
            "new_contradictory_high_rigor_submission": (
                high_rigor_outcome
            ),
            "distribution_status": (
                distribution_status
            ),
            "submitter_classification_tvd": (
                distribution_tvd
            ),
            "major_submitter_distribution_shift": (
                major_distribution_shift
            ),
        }
    )

derived_audit = pd.DataFrame(
    record_rows
)

if len(derived_audit) != EXPECTED_EVALUABLE_ROWS:
    raise AssertionError(
        "In-memory derivation row count mismatch."
    )

if (
    derived_audit[
        "t0_rcv_accession"
    ].nunique()
    != EXPECTED_EVALUABLE_ROWS
):
    raise AssertionError(
        "In-memory derivation T0 keys are not unique."
    )

if not np.all(
    np.diff(
        derived_audit[
            "t0_row_order"
        ].to_numpy()
    ) > 0
):
    raise AssertionError(
        "In-memory derivation did not preserve "
        "frozen T0 row order."
    )


# --------------------------------------------------------------------------------------------------
# 7. AUDIT TABLES
# --------------------------------------------------------------------------------------------------

nested_vocabulary = pd.concat(
    [
        pd.DataFrame(
            {
                "timepoint": "T0",
                "field": REVIEW_STATUS_KEY,
                "value": list(
                    t0_review_status_counts.keys()
                ),
                "nested_records": list(
                    t0_review_status_counts.values()
                ),
            }
        ),
        pd.DataFrame(
            {
                "timepoint": "T1",
                "field": REVIEW_STATUS_KEY,
                "value": list(
                    t1_review_status_counts.keys()
                ),
                "nested_records": list(
                    t1_review_status_counts.values()
                ),
            }
        ),
        pd.DataFrame(
            {
                "timepoint": "T0",
                "field": CLASSIFICATION_GROUP_KEY,
                "value": list(
                    t0_group_counts.keys()
                ),
                "nested_records": list(
                    t0_group_counts.values()
                ),
            }
        ),
        pd.DataFrame(
            {
                "timepoint": "T1",
                "field": CLASSIFICATION_GROUP_KEY,
                "value": list(
                    t1_group_counts.keys()
                ),
                "nested_records": list(
                    t1_group_counts.values()
                ),
            }
        ),
    ],
    ignore_index=True,
).sort_values(
    [
        "field",
        "timepoint",
        "nested_records",
        "value",
    ],
    ascending=[
        True,
        True,
        False,
        True,
    ],
).reset_index(drop=True)

high_rigor_status_accounting = (
    derived_audit[
        "new_contradictory_high_rigor_status"
    ]
    .value_counts(
        dropna=False
    )
    .rename_axis(
        "outcome_status"
    )
    .reset_index(
        name="rcv_rows"
    )
)

distribution_status_accounting = (
    derived_audit[
        "distribution_status"
    ]
    .value_counts(
        dropna=False
    )
    .rename_axis(
        "distribution_status"
    )
    .reset_index(
        name="rcv_rows"
    )
)

scv_transition_summary = pd.DataFrame(
    [
        {
            "measure": "RCVs with at least one new SCV",
            "rcv_rows": int(
                (
                    derived_audit[
                        "new_scv_count"
                    ] > 0
                ).sum()
            ),
            "nested_scv_total": int(
                derived_audit[
                    "new_scv_count"
                ].sum()
            ),
        },
        {
            "measure": (
                "RCVs with a new SCV missing review_status"
            ),
            "rcv_rows": int(
                (
                    derived_audit[
                        "new_scv_missing_review_status"
                    ] > 0
                ).sum()
            ),
            "nested_scv_total": int(
                derived_audit[
                    "new_scv_missing_review_status"
                ].sum()
            ),
        },
        {
            "measure": (
                "RCVs with at least one new high-rigor SCV"
            ),
            "rcv_rows": int(
                (
                    derived_audit[
                        "new_high_rigor_scv_count"
                    ] > 0
                ).sum()
            ),
            "nested_scv_total": int(
                derived_audit[
                    "new_high_rigor_scv_count"
                ].sum()
            ),
        },
        {
            "measure": (
                "RCVs with at least one new contradictory high-rigor SCV"
            ),
            "rcv_rows": int(
                (
                    derived_audit[
                        "new_high_rigor_contradictory_count"
                    ] > 0
                ).sum()
            ),
            "nested_scv_total": int(
                derived_audit[
                    "new_high_rigor_contradictory_count"
                ].sum()
            ),
        },
        {
            "measure": (
                "RCVs with an updated existing SCV version"
            ),
            "rcv_rows": int(
                (
                    derived_audit[
                        "updated_existing_scv_count"
                    ] > 0
                ).sum()
            ),
            "nested_scv_total": int(
                derived_audit[
                    "updated_existing_scv_count"
                ].sum()
            ),
        },
    ]
)

distribution_evaluable = derived_audit.loc[
    derived_audit[
        "distribution_status"
    ] == "EVALUABLE"
].copy()

if len(distribution_evaluable):
    tvd_summary = pd.DataFrame(
        [
            {
                "evaluable_rcv_rows": len(
                    distribution_evaluable
                ),
                "mean_tvd": float(
                    distribution_evaluable[
                        "submitter_classification_tvd"
                    ].mean()
                ),
                "median_tvd": float(
                    distribution_evaluable[
                        "submitter_classification_tvd"
                    ].median()
                ),
                "minimum_tvd": float(
                    distribution_evaluable[
                        "submitter_classification_tvd"
                    ].min()
                ),
                "maximum_tvd": float(
                    distribution_evaluable[
                        "submitter_classification_tvd"
                    ].max()
                ),
                "major_shift_threshold": (
                    MAJOR_DISTRIBUTION_SHIFT_THRESHOLD
                ),
                "major_shift_events": int(
                    distribution_evaluable[
                        "major_submitter_distribution_shift"
                    ].astype(int).sum()
                ),
                "major_shift_negatives": int(
                    len(
                        distribution_evaluable
                    )
                    - distribution_evaluable[
                        "major_submitter_distribution_shift"
                    ].astype(int).sum()
                ),
            }
        ]
    )

    tvd_bins = pd.cut(
        distribution_evaluable[
            "submitter_classification_tvd"
        ],
        bins=[
            -1e-12,
            0.0,
            0.25,
            0.50,
            0.75,
            1.0,
        ],
        labels=[
            "0",
            "(0,0.25]",
            "(0.25,0.50]",
            "(0.50,0.75]",
            "(0.75,1.00]",
        ],
        include_lowest=True,
    )

    tvd_distribution = (
        tvd_bins.value_counts(
            sort=False,
            dropna=False,
        )
        .rename_axis(
            "tvd_interval"
        )
        .reset_index(
            name="rcv_rows"
        )
    )

else:
    tvd_summary = pd.DataFrame(
        [
            {
                "evaluable_rcv_rows": 0,
                "mean_tvd": np.nan,
                "median_tvd": np.nan,
                "minimum_tvd": np.nan,
                "maximum_tvd": np.nan,
                "major_shift_threshold": (
                    MAJOR_DISTRIBUTION_SHIFT_THRESHOLD
                ),
                "major_shift_events": 0,
                "major_shift_negatives": 0,
            }
        ]
    )

    tvd_distribution = pd.DataFrame(
        columns=[
            "tvd_interval",
            "rcv_rows",
        ]
    )


# --------------------------------------------------------------------------------------------------
# 8. FINAL SCIENTIFIC DECISION LOGIC
# --------------------------------------------------------------------------------------------------

high_rigor_positive_count = int(
    (
        derived_audit[
            "new_contradictory_high_rigor_status"
        ]
        == "POSITIVE"
    ).sum()
)

high_rigor_negative_count = int(
    derived_audit[
        "new_contradictory_high_rigor_status"
    ]
    .astype("string")
    .str.startswith(
        "NEGATIVE",
        na=False,
    )
    .sum()
)

high_rigor_censored_count = int(
    (
        derived_audit[
            "new_contradictory_high_rigor_status"
        ]
        .str.startswith(
            "CENSORED",
            na=False,
        )
    ).sum()
)

high_rigor_binary_estimable = (
    high_rigor_positive_count > 0
    and high_rigor_negative_count > 0
)

distribution_binary_values = pd.to_numeric(
    distribution_evaluable[
        "major_submitter_distribution_shift"
    ],
    errors="raise",
).astype("int64")

distribution_major_shift_events = int(
    distribution_binary_values.sum()
)

distribution_major_shift_negatives = int(
    len(distribution_binary_values)
    - distribution_major_shift_events
)

distribution_binary_estimable = (
    len(distribution_binary_values) > 0
    and distribution_binary_values.nunique() == 2
)

distribution_continuous_estimable = (
    len(
        distribution_evaluable
    ) >= 50
    and distribution_evaluable[
        "submitter_classification_tvd"
    ].nunique() >= 2
)

# Corrected accounting assertions. The score-blind v1.1 derivation previously
# exposed a reporting bug: NEGATIVE_NO_NEW_SCV was not counted by an exact
# equality check for "NEGATIVE". The scientific policy and row labels were
# correct; only the summary count was wrong.
if (
    high_rigor_positive_count
    + high_rigor_negative_count
    + high_rigor_censored_count
    != EXPECTED_EVALUABLE_ROWS
):
    raise AssertionError(
        "High-rigor status accounting does not sum to the frozen cohort."
    )

EXPECTED_HIGH_RIGOR_POSITIVES = 0
EXPECTED_HIGH_RIGOR_NEGATIVES = 54_228
EXPECTED_HIGH_RIGOR_CENSORED = 12_408
EXPECTED_DISTRIBUTION_EVALUABLE = 37
EXPECTED_DISTRIBUTION_MAJOR_SHIFT_EVENTS = 21
EXPECTED_DISTRIBUTION_MAJOR_SHIFT_NEGATIVES = 16
EXPECTED_DISTRIBUTION_CENSORED = 66_599

if high_rigor_positive_count != EXPECTED_HIGH_RIGOR_POSITIVES:
    raise AssertionError(
        f"High-rigor positives={high_rigor_positive_count:,}; "
        f"expected {EXPECTED_HIGH_RIGOR_POSITIVES:,}."
    )

if high_rigor_negative_count != EXPECTED_HIGH_RIGOR_NEGATIVES:
    raise AssertionError(
        f"High-rigor negatives={high_rigor_negative_count:,}; "
        f"expected {EXPECTED_HIGH_RIGOR_NEGATIVES:,}."
    )

if high_rigor_censored_count != EXPECTED_HIGH_RIGOR_CENSORED:
    raise AssertionError(
        f"High-rigor censored={high_rigor_censored_count:,}; "
        f"expected {EXPECTED_HIGH_RIGOR_CENSORED:,}."
    )

if len(distribution_evaluable) != EXPECTED_DISTRIBUTION_EVALUABLE:
    raise AssertionError(
        f"Distribution-evaluable rows={len(distribution_evaluable):,}; "
        f"expected {EXPECTED_DISTRIBUTION_EVALUABLE:,}."
    )

if (
    distribution_major_shift_events
    != EXPECTED_DISTRIBUTION_MAJOR_SHIFT_EVENTS
):
    raise AssertionError(
        f"Major-shift events={distribution_major_shift_events:,}; "
        f"expected {EXPECTED_DISTRIBUTION_MAJOR_SHIFT_EVENTS:,}."
    )

if (
    distribution_major_shift_negatives
    != EXPECTED_DISTRIBUTION_MAJOR_SHIFT_NEGATIVES
):
    raise AssertionError(
        f"Major-shift negatives={distribution_major_shift_negatives:,}; "
        f"expected {EXPECTED_DISTRIBUTION_MAJOR_SHIFT_NEGATIVES:,}."
    )

if (
    EXPECTED_EVALUABLE_ROWS - len(distribution_evaluable)
    != EXPECTED_DISTRIBUTION_CENSORED
):
    raise AssertionError(
        "Distribution-censored row count mismatch."
    )

derivability_decision = pd.DataFrame(
    [
        {
            "target": (
                "new_contradictory_high_rigor_submission"
            ),
            "requested_analysis_form": (
                "binary secondary outcome"
            ),
            "evaluable_rows": (
                high_rigor_positive_count
                + high_rigor_negative_count
            ),
            "positive_events": (
                high_rigor_positive_count
            ),
            "negative_rows": (
                high_rigor_negative_count
            ),
            "censored_rows": (
                high_rigor_censored_count
            ),
            "estimable_for_discrimination": (
                high_rigor_binary_estimable
            ),
            "decision": (
                "ESTIMABLE"
                if high_rigor_binary_estimable
                else (
                    "NOT_ESTIMABLE_NO_POSITIVE_EVENTS"
                    if high_rigor_positive_count == 0
                    else "NOT_ESTIMABLE_INSUFFICIENT_CLASS_VARIATION"
                )
            ),
        },
        {
            "target": (
                "major_submitter_distribution_shift"
            ),
            "requested_analysis_form": (
                "binary sensitivity outcome, TVD >= 0.50"
            ),
            "evaluable_rows": len(
                distribution_evaluable
            ),
            "positive_events": (
                distribution_major_shift_events
            ),
            "negative_rows": (
                distribution_major_shift_negatives
            ),
            "censored_rows": int(
                EXPECTED_EVALUABLE_ROWS
                - len(
                    distribution_evaluable
                )
            ),
            "estimable_for_discrimination": (
                distribution_binary_estimable
            ),
            "decision": (
                "ESTIMABLE"
                if distribution_binary_estimable
                else "NOT_ESTIMABLE_INSUFFICIENT_COMPLETE_NESTED_CLASSIFICATION"
            ),
        },
        {
            "target": (
                "submitter_classification_tvd"
            ),
            "requested_analysis_form": (
                "continuous secondary drift magnitude"
            ),
            "evaluable_rows": len(
                distribution_evaluable
            ),
            "positive_events": pd.NA,
            "negative_rows": pd.NA,
            "censored_rows": int(
                EXPECTED_EVALUABLE_ROWS
                - len(
                    distribution_evaluable
                )
            ),
            "estimable_for_discrimination": (
                distribution_continuous_estimable
            ),
            "decision": (
                "ESTIMABLE_AS_CONTINUOUS"
                if distribution_continuous_estimable
                else "NOT_ESTIMABLE_INSUFFICIENT_COMPLETE_NESTED_CLASSIFICATION"
            ),
        },
    ]
)


# --------------------------------------------------------------------------------------------------
# 9. DISPLAY AND PASS MARKER
# --------------------------------------------------------------------------------------------------

separator = "=" * 150

print("\n" + separator)

print(
    "STAGE 6C STEP 3G — CELL 6C-3G0B v1.1A — "
    "NESTED-SCV POLICY FREEZE AND CORRECTED DERIVABILITY AUDIT"
)

print(separator)

print(
    f"Stage 6B evaluable SHA-256         : "
    f"PASS ({observed_evaluable_sha256})"
)

print(
    f"Accepted T0 SHA-256                : "
    f"PASS ({observed_t0_sha256})"
)

print(
    f"Accepted T1 SHA-256                : "
    f"PASS ({observed_t1_sha256})"
)

print(
    f"Mapped evaluable T0/T1 rows        : "
    f"PASS ({len(mapped):,}/{EXPECTED_EVALUABLE_ROWS:,})"
)

print(
    f"Policy path                        : "
    f"{POLICY_JSON}"
)

print(
    f"Policy SHA-256                     : "
    f"PASS ({policy_sha256})"
)

print(
    f"Policy sidecar file SHA-256        : "
    f"PASS ({sidecar_sha256})"
)

print(
    "GES or comparator scores loaded    : No"
)

print(
    "Predictive performance calculated  : No"
)

print(
    "Record-level derived artifact written: No"
)

print(
    "\nFROZEN OPERATIONAL DEFINITIONS"
)

print(
    json.dumps(
        policy,
        ensure_ascii=False,
        sort_keys=True,
        indent=2,
    )
)

print(
    "\nT0 BASELINE-GROUP ACCOUNTING"
)

print(
    baseline_group_accounting.to_string(
        index=False
    )
)

print(
    "\nNESTED REVIEW-STATUS AND "
    "CLASSIFICATION-GROUP VOCABULARY"
)

print(
    nested_vocabulary.to_string(
        index=False
    )
)

print(
    "\nSCV TRANSITION SUMMARY"
)

print(
    scv_transition_summary.to_string(
        index=False
    )
)

print(
    "\nNEW CONTRADICTORY HIGH-RIGOR "
    "OUTCOME STATUS ACCOUNTING"
)

print(
    high_rigor_status_accounting.to_string(
        index=False
    )
)

print(
    "\nSUBMITTER-CLASSIFICATION "
    "DISTRIBUTION STATUS ACCOUNTING"
)

print(
    distribution_status_accounting.to_string(
        index=False
    )
)

print(
    "\nTOTAL-VARIATION-DISTANCE SUMMARY"
)

print(
    tvd_summary.to_string(
        index=False
    )
)

print(
    "\nTOTAL-VARIATION-DISTANCE DISTRIBUTION"
)

if len(
    tvd_distribution
):
    print(
        tvd_distribution.to_string(
            index=False
        )
    )
else:
    print(
        "No complete-case distribution rows were available."
    )

print(
    "\nDERIVABILITY DECISION"
)

print(
    derivability_decision.to_string(
        index=False
    )
)

print(
    "\nINTERPRETATION BOUNDARY"
)

print("-" * 150)

print(
    "Missing nested T1 review_status or classification_group evidence "
    "is never converted into a negative outcome. High-rigor and "
    "submitter-distribution analyses may therefore be censored or declared "
    "non-estimable rather than filled from aggregate RCV fields."
)

print(
    "This preserves the prespecified submission-level meaning and avoids "
    "manufacturing evidence from aggregate review stars or aggregate "
    "classification."
)

print(
    "\nCELL DECISION"
)

print("-" * 150)

print(
    "PASS_STAGE6C_NESTED_SCV_OUTCOME_POLICY_V1_1_"
    "CORRECTED_AUDIT_COMPLETE"
)

print(
    "The existing policy v1.1 remains checksum-frozen before score access. "
    "This corrected audit fixes only the summary classification of "
    "NEGATIVE_NO_NEW_SCV records; no record-level label or policy changed."
)

print(
    "Corrected high-rigor accounting: 0 positive, 54,228 negative, "
    "and 12,408 censored records. The no-positive-event conclusion is "
    "unchanged, but evaluable negatives are now reported correctly."
)

print(
    "The next cell may freeze a record-level nested-SCV outcome package "
    "only for targets declared estimable above; non-estimable targets must "
    "be reported as data-limited rather than forced into performance analysis."
)


STAGE 6C STEP 3G — CELL 6C-3G0B v1.1A — NESTED-SCV POLICY FREEZE AND CORRECTED DERIVABILITY AUDIT
Stage 6B evaluable SHA-256         : PASS (c6ad50dfc376a746e1830d000dba975101d88763fd61b8d1983f6300edb6e038)
Accepted T0 SHA-256                : PASS (f6b6760b2ad6e4352e3bdecdeaf89827e8a514b031abf2373bb17568d999466d)
Accepted T1 SHA-256                : PASS (5713a11bdbf4804758cc011f9b2f302afc91fa1f88c1b178d675c28bb277d37c)
Mapped evaluable T0/T1 rows        : PASS (66,636/66,636)
Policy path                        : /content/drive/MyDrive/GES_RAG_Temporal_Study/configs/stage6c_nested_scv_secondary_outcomes/stage6c_nested_scv_secondary_outcome_policy_v1_1.json
Policy SHA-256                     : PASS (0d11d2eec8a3dcc0116a0ab45862a73b46ce130b79df1d9583910232bedb93aa)
Policy sidecar file SHA-256        : PASS (7c5d12dabe1a0ce18216a61bc3addae7f9835e842140372d7d723aab17886e4e)
GES or comparator scores loaded    : No
Predictive performance calculated  : No
Record-level derived artifact writt

In [45]:
STAGE 6C STEP 3G — CELL 6C-3G0B v1.1A — NESTED-SCV POLICY FREEZE AND CORRECTED DERIVABILITY AUDIT
======================================================================================================================================================
Stage 6B evaluable SHA-256         : PASS (c6ad50dfc376a746e1830d000dba975101d88763fd61b8d1983f6300edb6e038)
Accepted T0 SHA-256                : PASS (f6b6760b2ad6e4352e3bdecdeaf89827e8a514b031abf2373bb17568d999466d)
Accepted T1 SHA-256                : PASS (5713a11bdbf4804758cc011f9b2f302afc91fa1f88c1b178d675c28bb277d37c)
Mapped evaluable T0/T1 rows        : PASS (66,636/66,636)
Policy path                        : /content/drive/MyDrive/GES_RAG_Temporal_Study/configs/stage6c_nested_scv_secondary_outcomes/stage6c_nested_scv_secondary_outcome_policy_v1_1.json
Policy SHA-256                     : PASS (0d11d2eec8a3dcc0116a0ab45862a73b46ce130b79df1d9583910232bedb93aa)
Policy sidecar file SHA-256        : PASS (7c5d12dabe1a0ce18216a61bc3addae7f9835e842140372d7d723aab17886e4e)
GES or comparator scores loaded    : No
Predictive performance calculated  : No
Record-level derived artifact written: No

FROZEN OPERATIONAL DEFINITIONS
{
  "accepted_sources": {
    "stage6b_primary_evaluable_cohort": {
      "columns": 79,
      "filename": "stage6b_locked_primary_evaluable_cohort_v1.parquet",
      "rows": 66636,
      "sha256": "c6ad50dfc376a746e1830d000dba975101d88763fd61b8d1983f6300edb6e038"
    },
    "t0_rcv_source": {
      "columns": 34,
      "filename": "t0_rcv_target_genes_corrected_v1_2.parquet",
      "rows": 71659,
      "sha256": "f6b6760b2ad6e4352e3bdecdeaf89827e8a514b031abf2373bb17568d999466d"
    },
    "t1_rcv_source": {
      "columns": 36,
      "filename": "t1_rcv_target_genes_harmonized_v1.parquet",
      "rows": 100920,
      "sha256": "5713a11bdbf4804758cc011f9b2f302afc91fa1f88c1b178d675c28bb277d37c"
    }
  },
  "analysis_role": {
    "distribution_magnitude": "exploratory continuous secondary evidence-drift measure",
    "major_distribution_shift": "exploratory binary sensitivity outcome",
    "new_contradictory_high_rigor_submission": "exploratory secondary evidence-drift outcome"
  },
  "classification_group_mapping": {
    "BLB": [
      "Benign",
      "Likely benign",
      "Benign/Likely benign"
    ],
    "CONFLICTING": [
      "values containing conflict"
    ],
    "MISSING": [
      "Missing",
      "NoClassification",
      "null/blank"
    ],
    "OTHER": [
      "all remaining values"
    ],
    "PLP": [
      "Pathogenic",
      "Likely pathogenic",
      "Pathogenic/Likely pathogenic"
    ],
    "VUS": [
      "Uncertain significance",
      "VUS"
    ]
  },
  "high_rigor_rule": {
    "explicit_exclusion_precedence": [
      "normalized status starts with 'no assertion criteria'",
      "normalized status starts with 'no assertion provided'"
    ],
    "missing_review_status": "unknown, never assumed low-rigor",
    "positive_review_status_logic": [
      "normalized status starts with 'criteria provided'",
      "normalized status contains 'reviewed by expert panel'",
      "normalized status contains 'practice guideline'"
    ]
  },
  "leakage_protection": {
    "ges_or_comparator_scores_loaded_during_policy_freeze": false,
    "outcome_definition_selected_from_performance": false,
    "threshold_selected_from_performance": false
  },
  "nested_keys": {
    "classification_group": "classification_group",
    "review_rigor": "review_status",
    "scv_identity": "scv_accession",
    "submitter_identity": "submitter"
  },
  "new_contradictory_high_rigor_submission": {
    "baseline_interpretation": "frozen Stage 6B t0_canonical_classification_group; BLB, VUS, and PLP are informative single-group baselines; CONFLICTING, OTHER, and MISSING are noninformative for contradiction assessment",
    "censored": "when at least one new T1 SCV exists: the T0 baseline group is CONFLICTING/OTHER/MISSING, or no positive event is observed and at least one new T1 SCV has missing review status, or a high-rigor new SCV has a noninformative classification group",
    "contradiction": "for an informative BLB/VUS/PLP baseline, a new T1 SCV normalized group is BLB, VUS, or PLP and differs from that baseline group",
    "negative": "no new T1 SCV exists regardless of baseline group, or the baseline is informative and every new T1 SCV can be fully evaluated and none is both high-rigor and contradictory",
    "new_submission": "normalized T1 SCV accession absent from the same linked RCV's T0 SCV-accession set",
    "positive": "at least one new T1 SCV is high-rigor and contradictory",
    "updated_existing_scv": "an existing SCV accession with a later version is not a new submission and is excluded from this outcome"
  },
  "policy_name": "stage6c_nested_scv_secondary_outcome_policy",
  "policy_version": "1.1",
  "status": "FROZEN_BEFORE_NESTED_SCV_SCORE_ANALYSIS",
  "study_unit": "RCV-level variant-condition aggregate record",
  "submitter_classification_distribution_change": {
    "categories": [
      "BLB",
      "VUS",
      "PLP"
    ],
    "complete_case_evaluability": "every nested SCV at both T0 and T1 must map to BLB, VUS, or PLP; otherwise the distribution magnitude is censored",
    "distribution_unit": "one nested SCV classification assertion",
    "magnitude": "total variation distance = 0.5 * sum(abs(p_T1 - p_T0))",
    "major_shift_binary_sensitivity": "total variation distance >= 0.50",
    "major_shift_threshold": 0.5,
    "range": [
      0.0,
      1.0
    ],
    "threshold_rationale": "a fixed, interpretable threshold requiring at least half of classification probability mass to move; not selected from GES performance"
  },
  "supersedes": {
    "prior_policy": {
      "filename": "stage6c_nested_scv_secondary_outcome_policy_v1.json",
      "sha256": "f4d54e47667740ea0f6542db7959241caca8e8e1cb6d2d92a7064559132fd703",
      "sidecar_present": true,
      "status": "PRESERVED_UNCHANGED"
    },
    "reason": "The failed score-blind derivability audit showed that the frozen primary-evaluable cohort includes valid CONFLICTING and OTHER baseline groups. Version 1.1 explicitly censors new-submission contradiction assessment when a new SCV exists but the baseline group is not BLB, VUS, or PLP. The prior policy is preserved."
  }
}

T0 BASELINE-GROUP ACCOUNTING
t0_baseline_group  rcv_rows  informative_for_contradiction
              VUS     26048                           True
              PLP     20136                           True
              BLB     18939                           True
      CONFLICTING      1477                          False
            OTHER        36                          False

NESTED REVIEW-STATUS AND CLASSIFICATION-GROUP VOCABULARY
timepoint                field                               value  nested_records
       T0 classification_group                                 VUS           33114
       T0 classification_group                                 PLP           32915
       T0 classification_group                                 BLB           28631
       T0 classification_group                               OTHER             932
       T1 classification_group                             MISSING          109180
       T1 classification_group                                 VUS              98
       T1 classification_group                                 BLB              82
       T1 classification_group                                 PLP              63
       T0        review_status criteria provided, single submitter           72930
       T0        review_status      no assertion criteria provided           13630
       T0        review_status            reviewed by expert panel            8156
       T0        review_status               no assertion provided             876
       T1        review_status                           <MISSING>          109179
       T1        review_status      no assertion criteria provided             244

SCV TRANSITION SUMMARY
                                                measure  rcv_rows  nested_scv_total
                         RCVs with at least one new SCV     12408             14827
              RCVs with a new SCV missing review_status     12408             14827
              RCVs with at least one new high-rigor SCV         0                 0
RCVs with at least one new contradictory high-rigor SCV         0                 0
              RCVs with an updated existing SCV version     44091             48328

NEW CONTRADICTORY HIGH-RIGOR OUTCOME STATUS ACCOUNTING
                        outcome_status  rcv_rows
                   NEGATIVE_NO_NEW_SCV     54228
CENSORED_REQUIRED_NESTED_FIELD_MISSING     11838
CENSORED_NONINFORMATIVE_BASELINE_GROUP       570

SUBMITTER-CLASSIFICATION DISTRIBUTION STATUS ACCOUNTING
                          distribution_status  rcv_rows
CENSORED_NONINFORMATIVE_NESTED_CLASSIFICATION     66599
                                    EVALUABLE        37

TOTAL-VARIATION-DISTANCE SUMMARY
 evaluable_rcv_rows  mean_tvd  median_tvd  minimum_tvd  maximum_tvd  major_shift_threshold  major_shift_events  major_shift_negatives
                 37  0.567568         1.0          0.0          1.0                    0.5                  21                     16

TOTAL-VARIATION-DISTANCE DISTRIBUTION
tvd_interval  rcv_rows
           0        16
    (0,0.25]         0
 (0.25,0.50]         0
 (0.50,0.75]         0
 (0.75,1.00]        21

DERIVABILITY DECISION
                                 target                 requested_analysis_form  evaluable_rows positive_events negative_rows  censored_rows  estimable_for_discrimination                                                  decision
new_contradictory_high_rigor_submission                binary secondary outcome           54228               0         54228          12408                         False                          NOT_ESTIMABLE_NO_POSITIVE_EVENTS
     major_submitter_distribution_shift binary sensitivity outcome, TVD >= 0.50              37              21            16          66599                          True                                                 ESTIMABLE
           submitter_classification_tvd    continuous secondary drift magnitude              37            <NA>          <NA>          66599                         False NOT_ESTIMABLE_INSUFFICIENT_COMPLETE_NESTED_CLASSIFICATION

INTERPRETATION BOUNDARY
------------------------------------------------------------------------------------------------------------------------------------------------------
Missing nested T1 review_status or classification_group evidence is never converted into a negative outcome. High-rigor and submitter-distribution analyses may therefore be censored or declared non-estimable rather than filled from aggregate RCV fields.
This preserves the prespecified submission-level meaning and avoids manufacturing evidence from aggregate review stars or aggregate classification.

CELL DECISION
------------------------------------------------------------------------------------------------------------------------------------------------------
PASS_STAGE6C_NESTED_SCV_OUTCOME_POLICY_V1_1_CORRECTED_AUDIT_COMPLETE
The existing policy v1.1 remains checksum-frozen before score access. This corrected audit fixes only the summary classification of NEGATIVE_NO_NEW_SCV records; no record-level label or policy changed.
Corrected high-rigor accounting: 0 positive, 54,228 negative, and 12,408 censored records. The no-positive-event conclusion is unchanged, but evaluable negatives are now reported correctly.
The next cell may freeze a record-level nested-SCV outcome package only for targets declared estimable above; non-estimable targets must be reported as data-limited rather than forced into performance analysis.

IndentationError: unindent does not match any outer indentation level (<tokenize>, line 137)

In [46]:
# ==================================================================================================
# STAGE 6C STEP 3G — CELL 6C-3G0C
# SCORE-BLIND FREEZE OF RECORD-LEVEL NESTED-SCV SECONDARY OUTCOME PACKAGE
#
# Purpose:
#   1. Freshly verify the immutable Stage 6B cohort, accepted T0/T1 sources, and frozen
#      nested-SCV policy v1.1.
#   2. Re-derive the record-level nested-SCV secondary outcomes without loading any GES
#      or comparator score.
#   3. Freeze an auditable 66,636-row Parquet package containing:
#        - new-SCV and updated-SCV accounting;
#        - the non-estimable high-rigor contradiction status;
#        - T0/T1 nested classification-group counts;
#        - complete-case submitter-classification TVD;
#        - the estimable 37-row major-distribution-shift sensitivity outcome.
#   4. Write SHA-256 sidecars and a checksum-controlled manifest.
#   5. Read every written artifact back and freshly reverify structure, row accounting,
#      hashes, nullable outcomes, and exact class counts.
#
# Scientific boundary:
#   - No GES or comparator score is loaded.
#   - No predictive-performance metric is calculated.
#   - The high-rigor contradiction target remains non-estimable: 0 positive, 54,228 negative,
#     and 12,408 censored records.
#   - The binary major-distribution-shift target is evaluable in only 37 complete-case records:
#     21 events and 16 negatives. It remains highly exploratory.
#
# This cell creates the first record-level scientific artifact for Stage 6C Step 3G.
# Existing differing artifacts are never overwritten.
# ==================================================================================================

from pathlib import Path
from collections import Counter
import hashlib
import json
import os
import re
import tempfile

import numpy as np
import pandas as pd
import pyarrow.parquet as pq


# --------------------------------------------------------------------------------------------------
# 1. FROZEN PATHS, HASHES, AND EXPECTED ACCOUNTING
# --------------------------------------------------------------------------------------------------

PROJECT_DIR = Path(
    "/content/drive/MyDrive/GES_RAG_Temporal_Study"
)

STAGE6_DIR = (
    PROJECT_DIR
    / "data_processed"
    / "stage6_temporal_validation"
)

EVALUABLE_PARQUET = (
    STAGE6_DIR
    / "stage6b_locked_primary_evaluable_cohort_v1.parquet"
)

EVALUABLE_SIDECAR = Path(
    str(EVALUABLE_PARQUET) + ".sha256"
)

T0_PARQUET = (
    PROJECT_DIR
    / "data_interim"
    / "t0_rcv_target_genes_corrected_v1_2.parquet"
)

T1_PARQUET = (
    PROJECT_DIR
    / "data_interim"
    / "t1_rcv_target_genes_harmonized_v1.parquet"
)

POLICY_JSON = (
    PROJECT_DIR
    / "configs"
    / "stage6c_nested_scv_secondary_outcomes"
    / "stage6c_nested_scv_secondary_outcome_policy_v1_1.json"
)

POLICY_SIDECAR = Path(
    str(POLICY_JSON) + ".sha256"
)

OUTPUT_DIR = (
    STAGE6_DIR
    / "nested_scv_secondary_outcomes"
)

PACKAGE_PARQUET = (
    OUTPUT_DIR
    / "stage6c_nested_scv_secondary_outcomes_record_level_v1.parquet"
)

PACKAGE_SIDECAR = Path(
    str(PACKAGE_PARQUET) + ".sha256"
)

MANIFEST_JSON = (
    OUTPUT_DIR
    / "stage6c_nested_scv_secondary_outcomes_manifest_v1.json"
)

MANIFEST_SIDECAR = Path(
    str(MANIFEST_JSON) + ".sha256"
)

EXPECTED_EVALUABLE_SHA256 = (
    "c6ad50dfc376a746e1830d000dba975101d88763fd61b8d1983f6300edb6e038"
)

EXPECTED_T0_SHA256 = (
    "f6b6760b2ad6e4352e3bdecdeaf89827e8a514b031abf2373bb17568d999466d"
)

EXPECTED_T1_SHA256 = (
    "5713a11bdbf4804758cc011f9b2f302afc91fa1f88c1b178d675c28bb277d37c"
)

EXPECTED_POLICY_SHA256 = (
    "0d11d2eec8a3dcc0116a0ab45862a73b46ce130b79df1d9583910232bedb93aa"
)

EXPECTED_EVALUABLE_ROWS = 66_636
EXPECTED_EVALUABLE_COLUMNS = 79
EXPECTED_T0_ROWS = 71_659
EXPECTED_T0_COLUMNS = 34
EXPECTED_T1_ROWS = 100_920
EXPECTED_T1_COLUMNS = 36

EXPECTED_HIGH_RIGOR_POSITIVES = 0
EXPECTED_HIGH_RIGOR_NEGATIVES = 54_228
EXPECTED_HIGH_RIGOR_CENSORED_MISSING = 11_838
EXPECTED_HIGH_RIGOR_CENSORED_BASELINE = 570
EXPECTED_HIGH_RIGOR_CENSORED_TOTAL = 12_408

EXPECTED_DISTRIBUTION_EVALUABLE = 37
EXPECTED_DISTRIBUTION_CENSORED = 66_599
EXPECTED_MAJOR_SHIFT_EVENTS = 21
EXPECTED_MAJOR_SHIFT_NEGATIVES = 16

EXPECTED_NEW_SCV_RCVS = 12_408
EXPECTED_NEW_SCV_TOTAL = 14_827
EXPECTED_NEW_HIGH_RIGOR_RCVS = 0
EXPECTED_NEW_HIGH_RIGOR_TOTAL = 0
EXPECTED_UPDATED_EXISTING_RCVS = 44_091
EXPECTED_UPDATED_EXISTING_TOTAL = 48_328

STAGE6_T0_RCV_COLUMN = "rcv_accession"
STAGE6_T1_RCV_COLUMN = "linked_t1_rcv_accession"
STAGE6_ROW_ORDER_COLUMN = "t0_row_order"
STAGE6_BASELINE_GROUP_COLUMN = "t0_canonical_classification_group"

SOURCE_RCV_COLUMN = "rcv_accession"
NESTED_SCV_COLUMN = "scv_records_json"

SCV_ID_KEY = "scv_accession"
CLASSIFICATION_GROUP_KEY = "classification_group"
REVIEW_STATUS_KEY = "review_status"
SCV_VERSION_KEY = "scv_version"

INFORMATIVE_GROUPS = ("BLB", "VUS", "PLP")
MAJOR_SHIFT_THRESHOLD = 0.50

EXPECTED_PACKAGE_COLUMNS = [
    "t0_row_order",
    "rcv_accession",
    "linked_t1_rcv_accession",
    "t0_canonical_classification_group",
    "t0_scv_count",
    "t1_scv_count",
    "new_scv_count",
    "updated_existing_scv_count",
    "new_scv_missing_review_status",
    "new_high_rigor_scv_count",
    "new_high_rigor_noninformative_group_count",
    "new_high_rigor_concordant_count",
    "new_high_rigor_contradictory_count",
    "new_contradictory_high_rigor_status",
    "new_contradictory_high_rigor_submission",
    "t0_blb_scv_count",
    "t0_vus_scv_count",
    "t0_plp_scv_count",
    "t0_noninformative_scv_count",
    "t1_blb_scv_count",
    "t1_vus_scv_count",
    "t1_plp_scv_count",
    "t1_noninformative_scv_count",
    "distribution_status",
    "submitter_classification_tvd",
    "major_submitter_distribution_shift",
    "policy_version",
    "policy_sha256",
]


# --------------------------------------------------------------------------------------------------
# 2. HELPERS
# --------------------------------------------------------------------------------------------------

def sha256_file(
    path: Path,
    chunk_size: int = 1024 * 1024,
) -> str:
    digest = hashlib.sha256()

    with path.open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


def sha256_bytes(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()


def read_sidecar_hash(path: Path) -> str:
    text = path.read_text(
        encoding="utf-8"
    ).strip()

    matches = re.findall(
        r"\b[a-fA-F0-9]{64}\b",
        text,
    )

    if not matches:
        raise ValueError(
            f"No SHA-256 value found in sidecar: {path}"
        )

    return matches[0].lower()


def canonical_json_bytes(value: dict) -> bytes:
    return (
        json.dumps(
            value,
            ensure_ascii=False,
            sort_keys=True,
            indent=2,
        )
        + "\n"
    ).encode("utf-8")


def normalize_rcv(series: pd.Series) -> pd.Series:
    return (
        series.astype("string")
        .str.upper()
        .str.strip()
        .str.extract(
            r"(RCV\d+)",
            expand=False,
        )
    )


def normalize_scv_accession(value) -> str | None:
    if value is None:
        return None

    match = re.search(
        r"(SCV\d+)",
        str(value).upper().strip(),
    )

    return (
        match.group(1)
        if match
        else None
    )


def normalize_text(value) -> str | None:
    if value is None:
        return None

    text = re.sub(
        r"\s+",
        " ",
        str(value),
    ).strip()

    return (
        text
        if text
        else None
    )


def normalize_review_status(value) -> str | None:
    text = normalize_text(value)

    return (
        text.lower()
        if text is not None
        else None
    )


def is_high_rigor_review_status(
    normalized_status: str | None,
) -> bool:
    if normalized_status is None:
        return False

    if (
        normalized_status.startswith(
            "no assertion criteria"
        )
        or normalized_status.startswith(
            "no assertion provided"
        )
    ):
        return False

    return (
        normalized_status.startswith(
            "criteria provided"
        )
        or "reviewed by expert panel"
        in normalized_status
        or "practice guideline"
        in normalized_status
    )


def normalize_classification_group(value) -> str:
    text = normalize_text(value)

    if text is None:
        return "MISSING"

    normalized = (
        text.lower()
        .replace("_", " ")
        .replace("-", " ")
    )

    normalized = re.sub(
        r"\s+",
        " ",
        normalized,
    ).strip()

    if normalized in {
        "blb",
        "benign",
        "likely benign",
        "benign/likely benign",
        "benign likely benign",
    }:
        return "BLB"

    if normalized in {
        "vus",
        "uncertain significance",
        "uncertain",
    }:
        return "VUS"

    if normalized in {
        "plp",
        "pathogenic",
        "likely pathogenic",
        "pathogenic/likely pathogenic",
        "pathogenic likely pathogenic",
    }:
        return "PLP"

    if normalized in {
        "missing",
        "none",
        "nan",
        "not provided",
        "no classification",
        "noclassification",
    }:
        return "MISSING"

    if "conflict" in normalized:
        return "CONFLICTING"

    return "OTHER"


def parse_nested_scv_value(
    value,
    row_label: str,
) -> list[dict]:
    if isinstance(value, list):
        parsed = value

    elif isinstance(value, tuple):
        parsed = list(value)

    elif isinstance(value, np.ndarray):
        parsed = value.tolist()

    elif isinstance(value, str):
        text = value.strip()

        if text == "":
            raise ValueError(
                f"Blank nested SCV JSON at {row_label}"
            )

        try:
            parsed = json.loads(text)
        except Exception as error:
            raise ValueError(
                f"Malformed nested SCV JSON at "
                f"{row_label}: {error}"
            ) from error

    else:
        try:
            if pd.isna(value):
                raise ValueError(
                    f"Missing nested SCV evidence at "
                    f"{row_label}"
                )
        except Exception:
            pass

        raise TypeError(
            f"Unsupported nested SCV type at "
            f"{row_label}: {type(value).__name__}"
        )

    if not isinstance(parsed, list):
        raise TypeError(
            f"Nested SCV value is not a list at "
            f"{row_label}: {type(parsed).__name__}"
        )

    for position, record in enumerate(parsed):
        if not isinstance(record, dict):
            raise TypeError(
                f"Nested SCV record is not a dictionary "
                f"at {row_label}, position {position}."
            )

    return parsed


def group_count_dict(
    records: list[dict],
) -> dict[str, int]:
    counts = Counter(
        normalize_classification_group(
            record.get(
                CLASSIFICATION_GROUP_KEY
            )
        )
        for record in records
    )

    return {
        "BLB": int(counts["BLB"]),
        "VUS": int(counts["VUS"]),
        "PLP": int(counts["PLP"]),
        "NONINFORMATIVE": int(
            sum(
                count
                for group, count
                in counts.items()
                if group
                not in INFORMATIVE_GROUPS
            )
        ),
    }


def distribution_vector(
    counts: dict[str, int],
) -> np.ndarray:
    total = sum(
        counts[group]
        for group in INFORMATIVE_GROUPS
    )

    if total <= 0:
        raise ValueError(
            "Informative SCV distribution has zero total."
        )

    return np.array(
        [
            counts[group] / total
            for group in INFORMATIVE_GROUPS
        ],
        dtype=float,
    )


def total_variation_distance(
    first: np.ndarray,
    second: np.ndarray,
) -> float:
    return float(
        0.5
        * np.abs(
            np.asarray(first, dtype=float)
            - np.asarray(second, dtype=float)
        ).sum()
    )


def write_text_immutable_or_verify(
    path: Path,
    expected_bytes: bytes,
) -> str:
    expected_hash = sha256_bytes(
        expected_bytes
    )

    if path.exists():
        observed_bytes = path.read_bytes()

        if observed_bytes != expected_bytes:
            raise FileExistsError(
                "An existing immutable text artifact "
                f"differs from the expected content:\n{path}"
            )

        return sha256_bytes(
            observed_bytes
        )

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    with tempfile.NamedTemporaryFile(
        mode="wb",
        dir=str(path.parent),
        prefix=path.name + ".tmp.",
        delete=False,
    ) as temporary:
        temporary.write(
            expected_bytes
        )
        temporary.flush()
        os.fsync(
            temporary.fileno()
        )
        temporary_path = Path(
            temporary.name
        )

    try:
        os.replace(
            temporary_path,
            path,
        )
    finally:
        if temporary_path.exists():
            temporary_path.unlink()

    observed_hash = sha256_file(
        path
    )

    if observed_hash != expected_hash:
        raise AssertionError(
            f"Fresh text-artifact readback mismatch:\n{path}"
        )

    return observed_hash


def write_parquet_immutable_or_verify(
    frame: pd.DataFrame,
    path: Path,
) -> str:
    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    with tempfile.NamedTemporaryFile(
        suffix=".parquet",
        dir=str(path.parent),
        prefix=path.name + ".tmp.",
        delete=False,
    ) as temporary:
        temporary_path = Path(
            temporary.name
        )

    try:
        frame.to_parquet(
            temporary_path,
            engine="pyarrow",
            index=False,
            compression="snappy",
        )

        temporary_hash = sha256_file(
            temporary_path
        )

        if path.exists():
            existing_hash = sha256_file(
                path
            )

            existing_frame = pd.read_parquet(
                path
            )

            try:
                pd.testing.assert_frame_equal(
                    existing_frame,
                    frame,
                    check_dtype=True,
                    check_like=False,
                )
            except AssertionError as error:
                raise FileExistsError(
                    "An existing immutable Parquet artifact "
                    "differs semantically from the freshly "
                    f"derived package:\n{path}"
                ) from error

            return existing_hash

        os.replace(
            temporary_path,
            path,
        )

        temporary_path = None

        observed_hash = sha256_file(
            path
        )

        if observed_hash != temporary_hash:
            raise AssertionError(
                f"Fresh Parquet write/readback hash mismatch:\n{path}"
            )

        return observed_hash

    finally:
        if (
            temporary_path is not None
            and temporary_path.exists()
        ):
            temporary_path.unlink()


# --------------------------------------------------------------------------------------------------
# 3. VERIFY ALL FROZEN INPUTS AND POLICY
# --------------------------------------------------------------------------------------------------

for path in [
    EVALUABLE_PARQUET,
    EVALUABLE_SIDECAR,
    T0_PARQUET,
    T1_PARQUET,
    POLICY_JSON,
    POLICY_SIDECAR,
]:
    if not path.exists():
        raise FileNotFoundError(
            f"Missing required frozen artifact:\n{path}"
        )

observed_evaluable_sha256 = sha256_file(
    EVALUABLE_PARQUET
)

observed_t0_sha256 = sha256_file(
    T0_PARQUET
)

observed_t1_sha256 = sha256_file(
    T1_PARQUET
)

observed_policy_sha256 = sha256_file(
    POLICY_JSON
)

if observed_evaluable_sha256 != EXPECTED_EVALUABLE_SHA256:
    raise AssertionError(
        "Stage 6B evaluable SHA-256 mismatch."
    )

if read_sidecar_hash(
    EVALUABLE_SIDECAR
) != observed_evaluable_sha256:
    raise AssertionError(
        "Stage 6B evaluable sidecar mismatch."
    )

if observed_t0_sha256 != EXPECTED_T0_SHA256:
    raise AssertionError(
        "Accepted T0 SHA-256 mismatch."
    )

if observed_t1_sha256 != EXPECTED_T1_SHA256:
    raise AssertionError(
        "Accepted T1 SHA-256 mismatch."
    )

if observed_policy_sha256 != EXPECTED_POLICY_SHA256:
    raise AssertionError(
        "Frozen policy v1.1 SHA-256 mismatch."
    )

if read_sidecar_hash(
    POLICY_SIDECAR
) != observed_policy_sha256:
    raise AssertionError(
        "Frozen policy v1.1 sidecar mismatch."
    )

policy = json.loads(
    POLICY_JSON.read_text(
        encoding="utf-8"
    )
)

if policy.get(
    "policy_version"
) != "1.1":
    raise AssertionError(
        "Unexpected nested-SCV policy version."
    )

if policy.get(
    "status"
) != "FROZEN_BEFORE_NESTED_SCV_SCORE_ANALYSIS":
    raise AssertionError(
        "Nested-SCV policy freeze status mismatch."
    )

if policy[
    "leakage_protection"
][
    "ges_or_comparator_scores_loaded_during_policy_freeze"
] is not False:
    raise AssertionError(
        "Policy leakage-protection declaration mismatch."
    )

stage6_file = pq.ParquetFile(
    EVALUABLE_PARQUET
)

t0_file = pq.ParquetFile(
    T0_PARQUET
)

t1_file = pq.ParquetFile(
    T1_PARQUET
)

if (
    stage6_file.metadata.num_rows
    != EXPECTED_EVALUABLE_ROWS
    or stage6_file.metadata.num_columns
    != EXPECTED_EVALUABLE_COLUMNS
):
    raise AssertionError(
        "Stage 6B dimensions mismatch."
    )

if (
    t0_file.metadata.num_rows
    != EXPECTED_T0_ROWS
    or t0_file.metadata.num_columns
    != EXPECTED_T0_COLUMNS
):
    raise AssertionError(
        "Accepted T0 dimensions mismatch."
    )

if (
    t1_file.metadata.num_rows
    != EXPECTED_T1_ROWS
    or t1_file.metadata.num_columns
    != EXPECTED_T1_COLUMNS
):
    raise AssertionError(
        "Accepted T1 dimensions mismatch."
    )

required_stage6_columns = [
    STAGE6_T0_RCV_COLUMN,
    STAGE6_T1_RCV_COLUMN,
    STAGE6_ROW_ORDER_COLUMN,
    STAGE6_BASELINE_GROUP_COLUMN,
]

missing_stage6_columns = [
    column
    for column in required_stage6_columns
    if column
    not in stage6_file.schema_arrow.names
]

if missing_stage6_columns:
    raise KeyError(
        "Missing required Stage 6B columns:\n"
        + "\n".join(
            missing_stage6_columns
        )
    )


# --------------------------------------------------------------------------------------------------
# 4. LOAD ONLY SCORE-BLIND FIELDS AND MAP ALL EVALUABLE T0/T1 RECORDS
# --------------------------------------------------------------------------------------------------

stage6 = pd.read_parquet(
    EVALUABLE_PARQUET,
    columns=required_stage6_columns,
).copy()

stage6[
    STAGE6_T0_RCV_COLUMN
] = normalize_rcv(
    stage6[
        STAGE6_T0_RCV_COLUMN
    ]
)

stage6[
    STAGE6_T1_RCV_COLUMN
] = normalize_rcv(
    stage6[
        STAGE6_T1_RCV_COLUMN
    ]
)

stage6[
    STAGE6_BASELINE_GROUP_COLUMN
] = (
    stage6[
        STAGE6_BASELINE_GROUP_COLUMN
    ]
    .map(
        normalize_classification_group
    )
)

if stage6[
    STAGE6_T0_RCV_COLUMN
].isna().any():
    raise AssertionError(
        "A Stage 6B T0 RCV failed normalization."
    )

if stage6[
    STAGE6_T1_RCV_COLUMN
].isna().any():
    raise AssertionError(
        "A Stage 6B linked T1 RCV failed normalization."
    )

if stage6[
    STAGE6_T0_RCV_COLUMN
].nunique() != EXPECTED_EVALUABLE_ROWS:
    raise AssertionError(
        "Stage 6B T0 RCV keys are not unique."
    )

if stage6[
    STAGE6_T1_RCV_COLUMN
].nunique() != EXPECTED_EVALUABLE_ROWS:
    raise AssertionError(
        "Stage 6B linked T1 RCV keys are not unique."
    )

t0_source = pd.read_parquet(
    T0_PARQUET,
    columns=[
        SOURCE_RCV_COLUMN,
        NESTED_SCV_COLUMN,
    ],
).copy()

t1_source = pd.read_parquet(
    T1_PARQUET,
    columns=[
        SOURCE_RCV_COLUMN,
        NESTED_SCV_COLUMN,
    ],
).copy()

t0_source[
    SOURCE_RCV_COLUMN
] = normalize_rcv(
    t0_source[
        SOURCE_RCV_COLUMN
    ]
)

t1_source[
    SOURCE_RCV_COLUMN
] = normalize_rcv(
    t1_source[
        SOURCE_RCV_COLUMN
    ]
)

mapped = (
    stage6.merge(
        t0_source.rename(
            columns={
                SOURCE_RCV_COLUMN: "_t0_source_rcv",
                NESTED_SCV_COLUMN: "_t0_scv_records_json",
            }
        ),
        left_on=STAGE6_T0_RCV_COLUMN,
        right_on="_t0_source_rcv",
        how="left",
        validate="one_to_one",
        indicator="_t0_merge",
    )
    .merge(
        t1_source.rename(
            columns={
                SOURCE_RCV_COLUMN: "_t1_source_rcv",
                NESTED_SCV_COLUMN: "_t1_scv_records_json",
            }
        ),
        left_on=STAGE6_T1_RCV_COLUMN,
        right_on="_t1_source_rcv",
        how="left",
        validate="one_to_one",
        indicator="_t1_merge",
    )
    .sort_values(
        STAGE6_ROW_ORDER_COLUMN,
        kind="mergesort",
    )
    .reset_index(drop=True)
)

if not mapped[
    "_t0_merge"
].eq("both").all():
    raise AssertionError(
        "An evaluable T0 RCV failed source mapping."
    )

if not mapped[
    "_t1_merge"
].eq("both").all():
    raise AssertionError(
        "An evaluable T1 RCV failed source mapping."
    )

if len(mapped) != EXPECTED_EVALUABLE_ROWS:
    raise AssertionError(
        "Mapped evaluable row count mismatch."
    )


# --------------------------------------------------------------------------------------------------
# 5. RE-DERIVE THE RECORD-LEVEL PACKAGE WITHOUT SCORE ACCESS
# --------------------------------------------------------------------------------------------------

rows = []

for (
    t0_row_order,
    t0_rcv,
    t1_rcv,
    baseline_group,
    t0_raw,
    t1_raw,
) in zip(
    mapped[
        STAGE6_ROW_ORDER_COLUMN
    ].to_numpy(),
    mapped[
        STAGE6_T0_RCV_COLUMN
    ].to_numpy(),
    mapped[
        STAGE6_T1_RCV_COLUMN
    ].to_numpy(),
    mapped[
        STAGE6_BASELINE_GROUP_COLUMN
    ].to_numpy(),
    mapped[
        "_t0_scv_records_json"
    ].to_numpy(),
    mapped[
        "_t1_scv_records_json"
    ].to_numpy(),
):
    t0_records = parse_nested_scv_value(
        t0_raw,
        f"T0:{t0_rcv}",
    )

    t1_records = parse_nested_scv_value(
        t1_raw,
        f"T1:{t1_rcv}",
    )

    t0_accessions = set()
    t0_version_by_accession = {}

    for record in t0_records:
        accession = normalize_scv_accession(
            record.get(
                SCV_ID_KEY
            )
        )

        if accession is None:
            raise AssertionError(
                f"T0 nested SCV lacks a valid accession: {t0_rcv}"
            )

        if accession in t0_accessions:
            raise AssertionError(
                f"Duplicate T0 SCV accession in {t0_rcv}: {accession}"
            )

        t0_accessions.add(
            accession
        )

        t0_version_by_accession[
            accession
        ] = str(
            record.get(
                SCV_VERSION_KEY
            )
        )

    t1_accessions = set()

    new_scv_count = 0
    updated_existing_scv_count = 0
    new_scv_missing_review_status = 0
    new_high_rigor_scv_count = 0
    new_high_rigor_noninformative_group_count = 0
    new_high_rigor_concordant_count = 0
    new_high_rigor_contradictory_count = 0

    for record in t1_records:
        accession = normalize_scv_accession(
            record.get(
                SCV_ID_KEY
            )
        )

        if accession is None:
            raise AssertionError(
                f"T1 nested SCV lacks a valid accession: {t1_rcv}"
            )

        if accession in t1_accessions:
            raise AssertionError(
                f"Duplicate T1 SCV accession in {t1_rcv}: {accession}"
            )

        t1_accessions.add(
            accession
        )

        if accession in t0_accessions:
            if (
                t0_version_by_accession[
                    accession
                ]
                != str(
                    record.get(
                        SCV_VERSION_KEY
                    )
                )
            ):
                updated_existing_scv_count += 1

            continue

        new_scv_count += 1

        review_status = normalize_review_status(
            record.get(
                REVIEW_STATUS_KEY
            )
        )

        if review_status is None:
            new_scv_missing_review_status += 1
            continue

        if not is_high_rigor_review_status(
            review_status
        ):
            continue

        new_high_rigor_scv_count += 1

        group = normalize_classification_group(
            record.get(
                CLASSIFICATION_GROUP_KEY
            )
        )

        if group not in INFORMATIVE_GROUPS:
            new_high_rigor_noninformative_group_count += 1
            continue

        if (
            baseline_group
            in INFORMATIVE_GROUPS
        ):
            if group != baseline_group:
                new_high_rigor_contradictory_count += 1
            else:
                new_high_rigor_concordant_count += 1

    if new_scv_count == 0:
        high_rigor_status = (
            "NEGATIVE_NO_NEW_SCV"
        )
        high_rigor_outcome = 0

    elif baseline_group not in INFORMATIVE_GROUPS:
        high_rigor_status = (
            "CENSORED_NONINFORMATIVE_BASELINE_GROUP"
        )
        high_rigor_outcome = pd.NA

    elif new_high_rigor_contradictory_count > 0:
        high_rigor_status = "POSITIVE"
        high_rigor_outcome = 1

    elif (
        new_scv_missing_review_status > 0
        or new_high_rigor_noninformative_group_count > 0
    ):
        high_rigor_status = (
            "CENSORED_REQUIRED_NESTED_FIELD_MISSING"
        )
        high_rigor_outcome = pd.NA

    else:
        high_rigor_status = "NEGATIVE"
        high_rigor_outcome = 0

    t0_counts = group_count_dict(
        t0_records
    )

    t1_counts = group_count_dict(
        t1_records
    )

    distribution_evaluable = (
        t0_counts["NONINFORMATIVE"] == 0
        and t1_counts["NONINFORMATIVE"] == 0
        and sum(
            t0_counts[group]
            for group in INFORMATIVE_GROUPS
        ) > 0
        and sum(
            t1_counts[group]
            for group in INFORMATIVE_GROUPS
        ) > 0
    )

    if distribution_evaluable:
        tvd = total_variation_distance(
            distribution_vector(
                t0_counts
            ),
            distribution_vector(
                t1_counts
            ),
        )

        major_shift = int(
            tvd
            >= MAJOR_SHIFT_THRESHOLD
        )

        distribution_status = "EVALUABLE"

    else:
        tvd = np.nan
        major_shift = pd.NA
        distribution_status = (
            "CENSORED_NONINFORMATIVE_NESTED_CLASSIFICATION"
        )

    rows.append(
        {
            "t0_row_order": int(
                t0_row_order
            ),
            "rcv_accession": str(
                t0_rcv
            ),
            "linked_t1_rcv_accession": str(
                t1_rcv
            ),
            "t0_canonical_classification_group": (
                str(
                    baseline_group
                )
            ),
            "t0_scv_count": int(
                len(
                    t0_records
                )
            ),
            "t1_scv_count": int(
                len(
                    t1_records
                )
            ),
            "new_scv_count": int(
                new_scv_count
            ),
            "updated_existing_scv_count": int(
                updated_existing_scv_count
            ),
            "new_scv_missing_review_status": int(
                new_scv_missing_review_status
            ),
            "new_high_rigor_scv_count": int(
                new_high_rigor_scv_count
            ),
            "new_high_rigor_noninformative_group_count": int(
                new_high_rigor_noninformative_group_count
            ),
            "new_high_rigor_concordant_count": int(
                new_high_rigor_concordant_count
            ),
            "new_high_rigor_contradictory_count": int(
                new_high_rigor_contradictory_count
            ),
            "new_contradictory_high_rigor_status": (
                high_rigor_status
            ),
            "new_contradictory_high_rigor_submission": (
                high_rigor_outcome
            ),
            "t0_blb_scv_count": int(
                t0_counts["BLB"]
            ),
            "t0_vus_scv_count": int(
                t0_counts["VUS"]
            ),
            "t0_plp_scv_count": int(
                t0_counts["PLP"]
            ),
            "t0_noninformative_scv_count": int(
                t0_counts["NONINFORMATIVE"]
            ),
            "t1_blb_scv_count": int(
                t1_counts["BLB"]
            ),
            "t1_vus_scv_count": int(
                t1_counts["VUS"]
            ),
            "t1_plp_scv_count": int(
                t1_counts["PLP"]
            ),
            "t1_noninformative_scv_count": int(
                t1_counts["NONINFORMATIVE"]
            ),
            "distribution_status": (
                distribution_status
            ),
            "submitter_classification_tvd": (
                tvd
            ),
            "major_submitter_distribution_shift": (
                major_shift
            ),
            "policy_version": "1.1",
            "policy_sha256": (
                EXPECTED_POLICY_SHA256
            ),
        }
    )

package = pd.DataFrame(
    rows,
    columns=EXPECTED_PACKAGE_COLUMNS,
)

# Stable, explicit dtypes.
integer_columns = [
    "t0_row_order",
    "t0_scv_count",
    "t1_scv_count",
    "new_scv_count",
    "updated_existing_scv_count",
    "new_scv_missing_review_status",
    "new_high_rigor_scv_count",
    "new_high_rigor_noninformative_group_count",
    "new_high_rigor_concordant_count",
    "new_high_rigor_contradictory_count",
    "t0_blb_scv_count",
    "t0_vus_scv_count",
    "t0_plp_scv_count",
    "t0_noninformative_scv_count",
    "t1_blb_scv_count",
    "t1_vus_scv_count",
    "t1_plp_scv_count",
    "t1_noninformative_scv_count",
]

for column in integer_columns:
    package[column] = pd.to_numeric(
        package[column],
        errors="raise",
    ).astype("int64")

for column in [
    "new_contradictory_high_rigor_submission",
    "major_submitter_distribution_shift",
]:
    package[column] = pd.array(
        package[column],
        dtype="Int8",
    )

package[
    "submitter_classification_tvd"
] = pd.to_numeric(
    package[
        "submitter_classification_tvd"
    ],
    errors="coerce",
).astype("float64")

for column in [
    "rcv_accession",
    "linked_t1_rcv_accession",
    "t0_canonical_classification_group",
    "new_contradictory_high_rigor_status",
    "distribution_status",
    "policy_version",
    "policy_sha256",
]:
    package[column] = package[
        column
    ].astype("string")


# --------------------------------------------------------------------------------------------------
# 6. PRE-WRITE VALIDATION AGAINST THE CORRECTED AUDIT
# --------------------------------------------------------------------------------------------------

if package.shape != (
    EXPECTED_EVALUABLE_ROWS,
    len(
        EXPECTED_PACKAGE_COLUMNS
    ),
):
    raise AssertionError(
        f"Unexpected package dimensions: "
        f"{package.shape}"
    )

if package[
    "rcv_accession"
].nunique() != EXPECTED_EVALUABLE_ROWS:
    raise AssertionError(
        "Package T0 RCV keys are not unique."
    )

if package[
    "linked_t1_rcv_accession"
].nunique() != EXPECTED_EVALUABLE_ROWS:
    raise AssertionError(
        "Package linked T1 RCV keys are not unique."
    )

if not np.all(
    np.diff(
        package[
            "t0_row_order"
        ].to_numpy()
    ) > 0
):
    raise AssertionError(
        "Package is not in increasing T0 row order."
    )

status_counts = package[
    "new_contradictory_high_rigor_status"
].value_counts(
    dropna=False
)

if int(
    status_counts.get(
        "POSITIVE",
        0,
    )
) != EXPECTED_HIGH_RIGOR_POSITIVES:
    raise AssertionError(
        "Unexpected high-rigor positive count."
    )

if int(
    status_counts.get(
        "NEGATIVE_NO_NEW_SCV",
        0,
    )
) != EXPECTED_HIGH_RIGOR_NEGATIVES:
    raise AssertionError(
        "Unexpected high-rigor negative count."
    )

if int(
    status_counts.get(
        "CENSORED_REQUIRED_NESTED_FIELD_MISSING",
        0,
    )
) != EXPECTED_HIGH_RIGOR_CENSORED_MISSING:
    raise AssertionError(
        "Unexpected missing-field censor count."
    )

if int(
    status_counts.get(
        "CENSORED_NONINFORMATIVE_BASELINE_GROUP",
        0,
    )
) != EXPECTED_HIGH_RIGOR_CENSORED_BASELINE:
    raise AssertionError(
        "Unexpected baseline-group censor count."
    )

if int(
    package[
        "new_contradictory_high_rigor_submission"
    ].notna().sum()
) != EXPECTED_HIGH_RIGOR_NEGATIVES:
    raise AssertionError(
        "Unexpected evaluable high-rigor outcome count."
    )

if int(
    package[
        "new_contradictory_high_rigor_submission"
    ].fillna(0).astype("int8").sum()
) != EXPECTED_HIGH_RIGOR_POSITIVES:
    raise AssertionError(
        "Unexpected high-rigor event count."
    )

distribution_evaluable_mask = package[
    "distribution_status"
].eq(
    "EVALUABLE"
)

if int(
    distribution_evaluable_mask.sum()
) != EXPECTED_DISTRIBUTION_EVALUABLE:
    raise AssertionError(
        "Unexpected distribution-evaluable count."
    )

if int(
    (~distribution_evaluable_mask).sum()
) != EXPECTED_DISTRIBUTION_CENSORED:
    raise AssertionError(
        "Unexpected distribution-censored count."
    )

major_shift_evaluable = package.loc[
    distribution_evaluable_mask,
    "major_submitter_distribution_shift",
].astype("int8")

if int(
    major_shift_evaluable.sum()
) != EXPECTED_MAJOR_SHIFT_EVENTS:
    raise AssertionError(
        "Unexpected major-shift event count."
    )

if int(
    len(
        major_shift_evaluable
    )
    - major_shift_evaluable.sum()
) != EXPECTED_MAJOR_SHIFT_NEGATIVES:
    raise AssertionError(
        "Unexpected major-shift negative count."
    )

tvd_evaluable = package.loc[
    distribution_evaluable_mask,
    "submitter_classification_tvd",
]

if not tvd_evaluable.notna().all():
    raise AssertionError(
        "An evaluable distribution row lacks TVD."
    )

if package.loc[
    ~distribution_evaluable_mask,
    "submitter_classification_tvd",
].notna().any():
    raise AssertionError(
        "A censored distribution row has a TVD value."
    )

if not np.allclose(
    np.sort(
        tvd_evaluable.unique()
    ),
    np.array(
        [0.0, 1.0]
    ),
):
    raise AssertionError(
        "Unexpected complete-case TVD support."
    )

if int(
    package[
        "new_scv_count"
    ].gt(0).sum()
) != EXPECTED_NEW_SCV_RCVS:
    raise AssertionError(
        "Unexpected number of RCVs with new SCVs."
    )

if int(
    package[
        "new_scv_count"
    ].sum()
) != EXPECTED_NEW_SCV_TOTAL:
    raise AssertionError(
        "Unexpected total number of new SCVs."
    )

if int(
    package[
        "new_high_rigor_scv_count"
    ].gt(0).sum()
) != EXPECTED_NEW_HIGH_RIGOR_RCVS:
    raise AssertionError(
        "Unexpected number of RCVs with new high-rigor SCVs."
    )

if int(
    package[
        "new_high_rigor_scv_count"
    ].sum()
) != EXPECTED_NEW_HIGH_RIGOR_TOTAL:
    raise AssertionError(
        "Unexpected total number of new high-rigor SCVs."
    )

if int(
    package[
        "updated_existing_scv_count"
    ].gt(0).sum()
) != EXPECTED_UPDATED_EXISTING_RCVS:
    raise AssertionError(
        "Unexpected number of RCVs with updated existing SCVs."
    )

if int(
    package[
        "updated_existing_scv_count"
    ].sum()
) != EXPECTED_UPDATED_EXISTING_TOTAL:
    raise AssertionError(
        "Unexpected total number of updated existing SCVs."
    )

if not package[
    "policy_sha256"
].eq(
    EXPECTED_POLICY_SHA256
).all():
    raise AssertionError(
        "Package policy hash column mismatch."
    )


# --------------------------------------------------------------------------------------------------
# 7. WRITE PACKAGE, SIDECAR, AND MANIFEST IMMUTABLY
# --------------------------------------------------------------------------------------------------

package_sha256 = write_parquet_immutable_or_verify(
    package,
    PACKAGE_PARQUET,
)

package_sidecar_bytes = (
    f"{package_sha256}  {PACKAGE_PARQUET.name}\n"
).encode("utf-8")

package_sidecar_sha256 = (
    write_text_immutable_or_verify(
        PACKAGE_SIDECAR,
        package_sidecar_bytes,
    )
)

manifest = {
    "artifact_name": (
        "stage6c_nested_scv_secondary_outcomes_record_level"
    ),
    "artifact_version": "1.0",
    "analysis_stage": "Stage 6C Step 3G",
    "scientific_role": (
        "score-blind frozen record-level nested-SCV secondary outcome package"
    ),
    "package": {
        "path": str(
            PACKAGE_PARQUET
        ),
        "filename": (
            PACKAGE_PARQUET.name
        ),
        "sha256": package_sha256,
        "rows": int(
            len(
                package
            )
        ),
        "columns": int(
            package.shape[1]
        ),
        "column_order": list(
            package.columns
        ),
    },
    "policy": {
        "path": str(
            POLICY_JSON
        ),
        "filename": POLICY_JSON.name,
        "version": "1.1",
        "sha256": EXPECTED_POLICY_SHA256,
    },
    "frozen_sources": {
        "stage6b_primary_evaluable_cohort": {
            "filename": EVALUABLE_PARQUET.name,
            "sha256": EXPECTED_EVALUABLE_SHA256,
            "rows": EXPECTED_EVALUABLE_ROWS,
            "columns": EXPECTED_EVALUABLE_COLUMNS,
        },
        "t0_rcv_source": {
            "filename": T0_PARQUET.name,
            "sha256": EXPECTED_T0_SHA256,
            "rows": EXPECTED_T0_ROWS,
            "columns": EXPECTED_T0_COLUMNS,
        },
        "t1_rcv_source": {
            "filename": T1_PARQUET.name,
            "sha256": EXPECTED_T1_SHA256,
            "rows": EXPECTED_T1_ROWS,
            "columns": EXPECTED_T1_COLUMNS,
        },
    },
    "outcome_accounting": {
        "new_contradictory_high_rigor_submission": {
            "positive_events": (
                EXPECTED_HIGH_RIGOR_POSITIVES
            ),
            "negative_rows": (
                EXPECTED_HIGH_RIGOR_NEGATIVES
            ),
            "censored_rows": (
                EXPECTED_HIGH_RIGOR_CENSORED_TOTAL
            ),
            "estimable_for_discrimination": False,
            "decision": (
                "NOT_ESTIMABLE_NO_POSITIVE_EVENTS"
            ),
        },
        "major_submitter_distribution_shift": {
            "evaluable_rows": (
                EXPECTED_DISTRIBUTION_EVALUABLE
            ),
            "positive_events": (
                EXPECTED_MAJOR_SHIFT_EVENTS
            ),
            "negative_rows": (
                EXPECTED_MAJOR_SHIFT_NEGATIVES
            ),
            "censored_rows": (
                EXPECTED_DISTRIBUTION_CENSORED
            ),
            "threshold": (
                MAJOR_SHIFT_THRESHOLD
            ),
            "estimable_for_discrimination": True,
            "interpretation": (
                "highly exploratory complete-case sensitivity analysis"
            ),
        },
        "submitter_classification_tvd": {
            "evaluable_rows": (
                EXPECTED_DISTRIBUTION_EVALUABLE
            ),
            "censored_rows": (
                EXPECTED_DISTRIBUTION_CENSORED
            ),
            "unique_evaluable_values": [
                0.0,
                1.0,
            ],
            "decision": (
                "NOT_ESTIMABLE_AS_STABLE_CONTINUOUS_ANALYSIS"
            ),
        },
    },
    "leakage_protection": {
        "ges_or_comparator_scores_loaded": False,
        "predictive_performance_calculated": False,
        "outcome_definition_changed_after_policy_freeze": False,
    },
}

manifest_bytes = canonical_json_bytes(
    manifest
)

manifest_sha256 = (
    write_text_immutable_or_verify(
        MANIFEST_JSON,
        manifest_bytes,
    )
)

manifest_sidecar_bytes = (
    f"{manifest_sha256}  {MANIFEST_JSON.name}\n"
).encode("utf-8")

manifest_sidecar_sha256 = (
    write_text_immutable_or_verify(
        MANIFEST_SIDECAR,
        manifest_sidecar_bytes,
    )
)


# --------------------------------------------------------------------------------------------------
# 8. FRESH READBACK AND REVERIFICATION
# --------------------------------------------------------------------------------------------------

if read_sidecar_hash(
    PACKAGE_SIDECAR
) != package_sha256:
    raise AssertionError(
        "Package sidecar readback mismatch."
    )

if sha256_file(
    PACKAGE_PARQUET
) != package_sha256:
    raise AssertionError(
        "Package Parquet fresh hash mismatch."
    )

if read_sidecar_hash(
    MANIFEST_SIDECAR
) != manifest_sha256:
    raise AssertionError(
        "Manifest sidecar readback mismatch."
    )

if sha256_file(
    MANIFEST_JSON
) != manifest_sha256:
    raise AssertionError(
        "Manifest JSON fresh hash mismatch."
    )

readback = pd.read_parquet(
    PACKAGE_PARQUET
)

pd.testing.assert_frame_equal(
    readback,
    package,
    check_dtype=True,
    check_like=False,
)

readback_manifest = json.loads(
    MANIFEST_JSON.read_text(
        encoding="utf-8"
    )
)

if (
    readback_manifest[
        "package"
    ][
        "sha256"
    ]
    != package_sha256
):
    raise AssertionError(
        "Manifest package hash mismatch."
    )

if (
    readback_manifest[
        "policy"
    ][
        "sha256"
    ]
    != EXPECTED_POLICY_SHA256
):
    raise AssertionError(
        "Manifest policy hash mismatch."
    )

if (
    readback_manifest[
        "outcome_accounting"
    ][
        "major_submitter_distribution_shift"
    ][
        "positive_events"
    ]
    != EXPECTED_MAJOR_SHIFT_EVENTS
):
    raise AssertionError(
        "Manifest major-shift event count mismatch."
    )


# --------------------------------------------------------------------------------------------------
# 9. DISPLAY AND PASS MARKER
# --------------------------------------------------------------------------------------------------

high_rigor_status_table = (
    readback[
        "new_contradictory_high_rigor_status"
    ]
    .value_counts(
        dropna=False
    )
    .rename_axis(
        "outcome_status"
    )
    .reset_index(
        name="rcv_rows"
    )
)

distribution_status_table = (
    readback[
        "distribution_status"
    ]
    .value_counts(
        dropna=False
    )
    .rename_axis(
        "distribution_status"
    )
    .reset_index(
        name="rcv_rows"
    )
)

major_shift_table = (
    readback.loc[
        readback[
            "distribution_status"
        ].eq(
            "EVALUABLE"
        ),
        "major_submitter_distribution_shift",
    ]
    .astype("int8")
    .value_counts()
    .sort_index()
    .rename_axis(
        "major_submitter_distribution_shift"
    )
    .reset_index(
        name="rcv_rows"
    )
)

tvd_table = (
    readback.loc[
        readback[
            "distribution_status"
        ].eq(
            "EVALUABLE"
        ),
        "submitter_classification_tvd",
    ]
    .value_counts()
    .sort_index()
    .rename_axis(
        "submitter_classification_tvd"
    )
    .reset_index(
        name="rcv_rows"
    )
)

separator = "=" * 150

print("\n" + separator)

print(
    "STAGE 6C STEP 3G — CELL 6C-3G0C — "
    "RECORD-LEVEL NESTED-SCV OUTCOME PACKAGE FREEZE"
)

print(separator)

print(
    f"Stage 6B evaluable SHA-256         : "
    f"PASS ({observed_evaluable_sha256})"
)

print(
    f"Accepted T0 SHA-256                : "
    f"PASS ({observed_t0_sha256})"
)

print(
    f"Accepted T1 SHA-256                : "
    f"PASS ({observed_t1_sha256})"
)

print(
    f"Frozen policy v1.1 SHA-256         : "
    f"PASS ({observed_policy_sha256})"
)

print(
    f"Mapped and derived rows            : "
    f"PASS ({len(package):,})"
)

print(
    f"Package dimensions                 : "
    f"PASS ({package.shape[0]:,} × "
    f"{package.shape[1]})"
)

print(
    f"Package path                       : "
    f"{PACKAGE_PARQUET}"
)

print(
    f"Package SHA-256                    : "
    f"PASS ({package_sha256})"
)

print(
    f"Package sidecar file SHA-256       : "
    f"PASS ({package_sidecar_sha256})"
)

print(
    f"Manifest path                      : "
    f"{MANIFEST_JSON}"
)

print(
    f"Manifest SHA-256                   : "
    f"PASS ({manifest_sha256})"
)

print(
    f"Manifest sidecar file SHA-256      : "
    f"PASS ({manifest_sidecar_sha256})"
)

print(
    "Fresh package readback             : PASS"
)

print(
    "Fresh manifest readback            : PASS"
)

print(
    "GES or comparator scores loaded    : No"
)

print(
    "Predictive performance calculated  : No"
)

print(
    "\nNEW CONTRADICTORY HIGH-RIGOR STATUS"
)

print(
    high_rigor_status_table.to_string(
        index=False
    )
)

print(
    "\nSUBMITTER-DISTRIBUTION STATUS"
)

print(
    distribution_status_table.to_string(
        index=False
    )
)

print(
    "\nMAJOR DISTRIBUTION-SHIFT OUTCOME "
    "AMONG 37 COMPLETE-CASE RECORDS"
)

print(
    major_shift_table.to_string(
        index=False
    )
)

print(
    "\nTOTAL-VARIATION-DISTANCE VALUES "
    "AMONG 37 COMPLETE-CASE RECORDS"
)

print(
    tvd_table.to_string(
        index=False
    )
)

print(
    "\nSCIENTIFIC DECISION"
)

print("-" * 150)

print(
    "The high-rigor contradiction outcome is frozen as "
    "NOT_ESTIMABLE_NO_POSITIVE_EVENTS and will not be forced "
    "into discrimination analysis."
)

print(
    "The major submitter-distribution-shift outcome is frozen "
    "as technically estimable in 37 complete-case records "
    "(21 events, 16 negatives), but any subsequent performance "
    "analysis must be labeled highly exploratory and data-limited."
)

print(
    "\nCELL DECISION"
)

print("-" * 150)

print(
    "PASS_STAGE6C_NESTED_SCV_RECORD_LEVEL_"
    "OUTCOME_PACKAGE_FROZEN_AND_REVERIFIED"
)

print(
    "The score-blind 66,636-row nested-SCV secondary outcome "
    "package, SHA-256 sidecar, manifest, and manifest sidecar "
    "were written immutably and freshly reverified."
)

print(
    "No GES score, comparator score, policy definition, linkage "
    "decision, threshold, cohort membership, or frozen source "
    "artifact was modified."
)


STAGE 6C STEP 3G — CELL 6C-3G0C — RECORD-LEVEL NESTED-SCV OUTCOME PACKAGE FREEZE
Stage 6B evaluable SHA-256         : PASS (c6ad50dfc376a746e1830d000dba975101d88763fd61b8d1983f6300edb6e038)
Accepted T0 SHA-256                : PASS (f6b6760b2ad6e4352e3bdecdeaf89827e8a514b031abf2373bb17568d999466d)
Accepted T1 SHA-256                : PASS (5713a11bdbf4804758cc011f9b2f302afc91fa1f88c1b178d675c28bb277d37c)
Frozen policy v1.1 SHA-256         : PASS (0d11d2eec8a3dcc0116a0ab45862a73b46ce130b79df1d9583910232bedb93aa)
Mapped and derived rows            : PASS (66,636)
Package dimensions                 : PASS (66,636 × 28)
Package path                       : /content/drive/MyDrive/GES_RAG_Temporal_Study/data_processed/stage6_temporal_validation/nested_scv_secondary_outcomes/stage6c_nested_scv_secondary_outcomes_record_level_v1.parquet
Package SHA-256                    : PASS (18b3d75b62e8d1b891dcd88eb951b4aa767aa6c88570369004bd08b9c3de7fc2)
Package sidecar file SHA-256       : PASS (eb4f78

In [47]:
# ==================================================================================================
# STAGE 6C STEP 3G — CELL 6C-3G1
# HIGHLY EXPLORATORY LOCKED SCORE ANALYSIS OF THE 37 COMPLETE-CASE
# MAJOR SUBMITTER-DISTRIBUTION-SHIFT RECORDS
#
# Boundary:
# - Analyze only the frozen 37-record endpoint (21 events, 16 negatives).
# - Do not analyze the high-rigor contradiction endpoint (0 positives).
# - No score/outcome/model/threshold/policy/linkage/cohort change.
# - No artifact is written; final freezing occurs after remaining Stage 6C work.
# ==================================================================================================

from pathlib import Path
from collections import OrderedDict
import hashlib, json, re

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from sklearn.metrics import average_precision_score, roc_auc_score

# --------------------------------------------------------------------------------------------------
# 1. FROZEN PATHS AND EXPECTATIONS
# --------------------------------------------------------------------------------------------------

PROJECT = Path("/content/drive/MyDrive/GES_RAG_Temporal_Study")
STAGE6 = PROJECT / "data_processed" / "stage6_temporal_validation"

EVAL = STAGE6 / "stage6b_locked_primary_evaluable_cohort_v1.parquet"
EVAL_SHA = Path(str(EVAL) + ".sha256")

NESTED_DIR = STAGE6 / "nested_scv_secondary_outcomes"
PACKAGE = NESTED_DIR / "stage6c_nested_scv_secondary_outcomes_record_level_v1.parquet"
PACKAGE_SHA = Path(str(PACKAGE) + ".sha256")
MANIFEST = NESTED_DIR / "stage6c_nested_scv_secondary_outcomes_manifest_v1.json"
MANIFEST_SHA = Path(str(MANIFEST) + ".sha256")

EXPECTED = {
    "eval_hash": "c6ad50dfc376a746e1830d000dba975101d88763fd61b8d1983f6300edb6e038",
    "package_hash": "18b3d75b62e8d1b891dcd88eb951b4aa767aa6c88570369004bd08b9c3de7fc2",
    "manifest_hash": "b89626648af773b79053515f81dc1b7afee1ca3e516510a2bf52aa5110c5edc5",
    "policy_hash": "0d11d2eec8a3dcc0116a0ab45862a73b46ce130b79df1d9583910232bedb93aa",
    "total_rows": 66_636,
    "eval_cols": 79,
    "package_cols": 28,
    "complete_rows": 37,
    "events": 21,
    "negatives": 16,
    "censored": 66_599,
    "high_rigor_events": 0,
    "high_rigor_negatives": 54_228,
    "high_rigor_censored": 12_408,
}

BOOTSTRAPS = 2_000
SEED = 42
KEYS = ["t0_row_order", "rcv_accession"]

SCORES = OrderedDict([
    ("full_ges", ("full_ges_instability_risk_t0", "Full GES")),
    ("no_star_ges", ("no_star_ges_instability_risk_t0", "No-star GES")),
    ("review_stars", ("review_stars_instability_risk", "Review stars")),
    ("combined_metadata", ("combined_metadata_instability_risk", "Combined metadata")),
    ("conflict", ("conflict_instability_risk", "Conflict")),
    ("recency", ("recency_instability_risk", "Recency")),
    ("submitter", ("submitter_instability_risk", "Submitter support")),
    ("entropy", ("entropy_instability_risk", "Classification entropy")),
    ("additive", ("additive_instability_risk", "Additive risk")),
])

PRINCIPAL_COMPARATORS = ["no_star_ges", "review_stars", "combined_metadata"]

# --------------------------------------------------------------------------------------------------
# 2. HELPERS
# --------------------------------------------------------------------------------------------------

def sha256_file(path, chunk_size=1024 * 1024):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()

def sidecar_hash(path):
    text = Path(path).read_text(encoding="utf-8")
    matches = re.findall(r"\b[a-fA-F0-9]{64}\b", text)
    if not matches:
        raise ValueError(f"No SHA-256 found in sidecar: {path}")
    return matches[0].lower()

def normalize_rcv(series):
    return (
        series.astype("string")
        .str.upper()
        .str.strip()
        .str.extract(r"(RCV\d+)", expand=False)
    )

def ci(values):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if len(values) == 0:
        return np.nan, np.nan
    low, high = np.percentile(values, [2.5, 97.5])
    return float(low), float(high)

# --------------------------------------------------------------------------------------------------
# 3. CRYPTOGRAPHIC AND STRUCTURAL PREFLIGHT
# --------------------------------------------------------------------------------------------------

for path in [EVAL, EVAL_SHA, PACKAGE, PACKAGE_SHA, MANIFEST, MANIFEST_SHA]:
    if not path.exists():
        raise FileNotFoundError(f"Missing frozen artifact:\n{path}")

eval_hash = sha256_file(EVAL)
package_hash = sha256_file(PACKAGE)
manifest_hash = sha256_file(MANIFEST)

assert eval_hash == EXPECTED["eval_hash"], "Stage 6B evaluable SHA-256 mismatch."
assert sidecar_hash(EVAL_SHA) == eval_hash, "Stage 6B evaluable sidecar mismatch."
assert package_hash == EXPECTED["package_hash"], "Nested-SCV package SHA-256 mismatch."
assert sidecar_hash(PACKAGE_SHA) == package_hash, "Nested-SCV package sidecar mismatch."
assert manifest_hash == EXPECTED["manifest_hash"], "Nested-SCV manifest SHA-256 mismatch."
assert sidecar_hash(MANIFEST_SHA) == manifest_hash, "Nested-SCV manifest sidecar mismatch."

eval_pf = pq.ParquetFile(EVAL)
package_pf = pq.ParquetFile(PACKAGE)

assert (
    eval_pf.metadata.num_rows,
    eval_pf.metadata.num_columns,
) == (
    EXPECTED["total_rows"],
    EXPECTED["eval_cols"],
), "Unexpected Stage 6B evaluable dimensions."

assert (
    package_pf.metadata.num_rows,
    package_pf.metadata.num_columns,
) == (
    EXPECTED["total_rows"],
    EXPECTED["package_cols"],
), "Unexpected nested-SCV package dimensions."

# Confirm manifest can be parsed after hash verification.
_ = json.loads(MANIFEST.read_text(encoding="utf-8"))

# --------------------------------------------------------------------------------------------------
# 4. VERIFY THE FROZEN OUTCOMES BEFORE LOADING SCORES
# --------------------------------------------------------------------------------------------------

nested_columns = [
    "t0_row_order",
    "rcv_accession",
    "new_contradictory_high_rigor_status",
    "new_contradictory_high_rigor_submission",
    "distribution_status",
    "submitter_classification_tvd",
    "major_submitter_distribution_shift",
    "policy_version",
    "policy_sha256",
]

nested = pd.read_parquet(PACKAGE, columns=nested_columns).copy()
nested["rcv_accession"] = normalize_rcv(nested["rcv_accession"])
nested["t0_row_order"] = pd.to_numeric(nested["t0_row_order"], errors="raise").astype("int64")

assert len(nested) == EXPECTED["total_rows"]
assert nested["rcv_accession"].nunique() == EXPECTED["total_rows"]
assert nested["t0_row_order"].nunique() == EXPECTED["total_rows"]
assert nested["policy_version"].astype("string").eq("1.1").all()
assert nested["policy_sha256"].astype("string").eq(EXPECTED["policy_hash"]).all()

high_rigor = nested["new_contradictory_high_rigor_submission"]
hr_events = int(high_rigor.fillna(0).astype("int8").sum())
hr_evaluable = int(high_rigor.notna().sum())
hr_negatives = hr_evaluable - hr_events
hr_censored = len(nested) - hr_evaluable

assert hr_events == EXPECTED["high_rigor_events"]
assert hr_negatives == EXPECTED["high_rigor_negatives"]
assert hr_censored == EXPECTED["high_rigor_censored"]

complete = nested.loc[nested["distribution_status"].eq("EVALUABLE")].copy()
assert len(complete) == EXPECTED["complete_rows"]
assert int(nested["distribution_status"].ne("EVALUABLE").sum()) == EXPECTED["censored"]
assert complete["major_submitter_distribution_shift"].notna().all()

y = complete["major_submitter_distribution_shift"].astype("int8").to_numpy()
assert int(y.sum()) == EXPECTED["events"]
assert int(len(y) - y.sum()) == EXPECTED["negatives"]

tvd_values = set(
    np.round(complete["submitter_classification_tvd"].to_numpy(float), 12).tolist()
)
assert tvd_values == {0.0, 1.0}, f"Unexpected TVD values: {tvd_values}"

# --------------------------------------------------------------------------------------------------
# 5. LOAD THE FROZEN SCORES ONLY AFTER THE SCORE-BLIND PACKAGE PASSES
# --------------------------------------------------------------------------------------------------

score_columns = [column for column, _ in SCORES.values()]
required_score_columns = KEYS + score_columns
missing = [c for c in required_score_columns if c not in eval_pf.schema_arrow.names]
if missing:
    raise KeyError("Missing Stage 6B fields:\n" + "\n".join(missing))

scores = pd.read_parquet(EVAL, columns=required_score_columns).copy()
scores["rcv_accession"] = normalize_rcv(scores["rcv_accession"])
scores["t0_row_order"] = pd.to_numeric(scores["t0_row_order"], errors="raise").astype("int64")

assert len(scores) == EXPECTED["total_rows"]
assert scores["rcv_accession"].nunique() == EXPECTED["total_rows"]
assert scores["t0_row_order"].nunique() == EXPECTED["total_rows"]

for column in score_columns:
    scores[column] = pd.to_numeric(scores[column], errors="raise").astype("float64")
    values = scores[column].to_numpy(float)
    assert np.isfinite(values).all(), f"Nonfinite values in {column}"
    assert values.min() >= 0 and values.max() <= 1, f"{column} is outside [0,1]"

analysis = complete.merge(
    scores,
    on=KEYS,
    how="left",
    validate="one_to_one",
    indicator=True,
)
assert analysis["_merge"].eq("both").all()
analysis = (
    analysis.drop(columns="_merge")
    .sort_values("t0_row_order", kind="mergesort")
    .reset_index(drop=True)
)
assert len(analysis) == EXPECTED["complete_rows"]
y = analysis["major_submitter_distribution_shift"].astype("int8").to_numpy()
prevalence = float(y.mean())

# --------------------------------------------------------------------------------------------------
# 6. DESCRIPTIVE POINT ESTIMATES
# --------------------------------------------------------------------------------------------------

point_rows = []
for key, (column, label) in SCORES.items():
    s = analysis[column].to_numpy(float)
    ap = float(average_precision_score(y, s))
    auc = float(roc_auc_score(y, s))
    point_rows.append({
        "model_key": key,
        "model": label,
        "unique_score_values": int(np.unique(s).size),
        "point_auprc": ap,
        "auprc_minus_prevalence": ap - prevalence,
        "auprc_lift_over_prevalence": ap / prevalence,
        "point_auroc": auc,
        "auroc_minus_0_50": auc - 0.50,
    })

point = pd.DataFrame(point_rows)
point_lookup = point.set_index("model_key")

# --------------------------------------------------------------------------------------------------
# 7. 2,000 IDENTICAL PAIRED ORDINARY ROW-BOOTSTRAP ATTEMPTS
# --------------------------------------------------------------------------------------------------

rng = np.random.default_rng(SEED)
bootstrap = {
    key: {"auprc": [], "auroc": []}
    for key in SCORES
}
valid = 0

for _ in range(BOOTSTRAPS):
    idx = rng.integers(0, len(y), size=len(y))
    y_b = y[idx]
    if np.unique(y_b).size < 2:
        continue

    valid += 1
    for key, (column, _) in SCORES.items():
        s_b = analysis[column].to_numpy(float)[idx]
        bootstrap[key]["auprc"].append(average_precision_score(y_b, s_b))
        bootstrap[key]["auroc"].append(roc_auc_score(y_b, s_b))

for key in bootstrap:
    for metric in bootstrap[key]:
        bootstrap[key][metric] = np.asarray(bootstrap[key][metric], dtype=float)

invalid = BOOTSTRAPS - valid
assert valid > 0, "No valid two-class bootstrap replicate."

summary_rows = []
for key, (_, label) in SCORES.items():
    ap_low, ap_high = ci(bootstrap[key]["auprc"])
    auc_low, auc_high = ci(bootstrap[key]["auroc"])
    summary_rows.append({
        "model_key": key,
        "model": label,
        "auprc_bootstrap_95_low": ap_low,
        "auprc_bootstrap_95_high": ap_high,
        "auroc_bootstrap_95_low": auc_low,
        "auroc_bootstrap_95_high": auc_high,
    })

bootstrap_summary = pd.DataFrame(summary_rows)

display_metrics = point.merge(
    bootstrap_summary,
    on=["model_key", "model"],
    how="left",
    validate="one_to_one",
)

paired_rows = []
for comparator in PRINCIPAL_COMPARATORS:
    for metric, label, point_column in [
        ("auprc", "AUPRC", "point_auprc"),
        ("auroc", "AUROC", "point_auroc"),
    ]:
        differences = bootstrap["full_ges"][metric] - bootstrap[comparator][metric]
        low, high = ci(differences)
        paired_rows.append({
            "metric": label,
            "comparison": f"Full GES minus {SCORES[comparator][1]}",
            "point_difference": float(
                point_lookup.loc["full_ges", point_column]
                - point_lookup.loc[comparator, point_column]
            ),
            "bootstrap_95_low": low,
            "bootstrap_95_high": high,
            "fraction_bootstrap_difference_gt_0": float(np.mean(differences > 0)),
        })

paired = pd.DataFrame(paired_rows)

numeric_metric_columns = [
    "point_auprc",
    "auprc_minus_prevalence",
    "auprc_lift_over_prevalence",
    "point_auroc",
    "auroc_minus_0_50",
    "auprc_bootstrap_95_low",
    "auprc_bootstrap_95_high",
    "auroc_bootstrap_95_low",
    "auroc_bootstrap_95_high",
]
display_metrics[numeric_metric_columns] = display_metrics[numeric_metric_columns].round(6)

paired_numeric_columns = [
    "point_difference",
    "bootstrap_95_low",
    "bootstrap_95_high",
    "fraction_bootstrap_difference_gt_0",
]
paired[paired_numeric_columns] = paired[paired_numeric_columns].round(6)

# --------------------------------------------------------------------------------------------------
# 8. DISPLAY AND PASS MARKER
# --------------------------------------------------------------------------------------------------

separator = "=" * 150
censoring_fraction = EXPECTED["censored"] / EXPECTED["total_rows"]

print("\n" + separator)
print(
    "STAGE 6C STEP 3G — CELL 6C-3G1 — "
    "HIGHLY EXPLORATORY 37-RECORD NESTED-SCV SCORE ANALYSIS"
)
print(separator)

print(f"Stage 6B evaluable SHA-256         : PASS ({eval_hash})")
print(f"Nested-SCV package SHA-256         : PASS ({package_hash})")
print(f"Nested-SCV manifest SHA-256        : PASS ({manifest_hash})")
print(
    "High-rigor contradiction endpoint  : NOT ESTIMABLE "
    f"({hr_events} events, {hr_negatives:,} negatives, {hr_censored:,} censored)"
)
print(f"Distribution endpoint rows         : {len(analysis)}")
print(f"Major-shift events / negatives     : {int(y.sum())} / {int(len(y)-y.sum())}")
print(f"Major-shift prevalence             : {prevalence:.6f}")
print(
    f"Distribution censoring             : {EXPECTED['censored']:,} / "
    f"{EXPECTED['total_rows']:,} ({100*censoring_fraction:.4f}%)"
)
print(f"Bootstrap attempts                 : {BOOTSTRAPS:,}")
print(f"Valid / invalid replicates         : {valid:,} / {invalid:,}")

print("\nDESCRIPTIVE DISCRIMINATION RESULTS")
print(
    display_metrics[
        [
            "model",
            "unique_score_values",
            "point_auprc",
            "auprc_bootstrap_95_low",
            "auprc_bootstrap_95_high",
            "auprc_minus_prevalence",
            "auprc_lift_over_prevalence",
            "point_auroc",
            "auroc_bootstrap_95_low",
            "auroc_bootstrap_95_high",
            "auroc_minus_0_50",
        ]
    ].to_string(index=False)
)

print("\nPAIRED FULL-GES MINUS PRINCIPAL-COMPARATOR DIFFERENCES")
print(paired.to_string(index=False))

print("\nSCIENTIFIC INTERPRETATION BOUNDARY")
print("-" * 150)
print(
    "These estimates are highly exploratory because only 37 of 66,636 records are "
    "evaluable and 99.94% are censored. Bootstrap intervals describe instability under "
    "resampling of this tiny complete-case subset; they do not establish generalizable "
    "superiority, calibration, or clinical utility."
)
print(
    "The high-rigor contradiction endpoint was not analyzed because it has zero "
    "observable positive events under the frozen policy."
)

print("\nCELL DECISION")
print("-" * 150)
print("PASS_STAGE6C_NESTED_SCV_37_RECORD_EXPLORATORY_SCORE_ANALYSIS_COMPLETE")
print(
    "Frozen scores were joined only after successful reverification of the score-blind "
    "nested-SCV outcome package. No input artifact, score, outcome, threshold, policy, "
    "model, linkage decision, or cohort membership was modified. No artifact was written."
)


STAGE 6C STEP 3G — CELL 6C-3G1 — HIGHLY EXPLORATORY 37-RECORD NESTED-SCV SCORE ANALYSIS
Stage 6B evaluable SHA-256         : PASS (c6ad50dfc376a746e1830d000dba975101d88763fd61b8d1983f6300edb6e038)
Nested-SCV package SHA-256         : PASS (18b3d75b62e8d1b891dcd88eb951b4aa767aa6c88570369004bd08b9c3de7fc2)
Nested-SCV manifest SHA-256        : PASS (b89626648af773b79053515f81dc1b7afee1ca3e516510a2bf52aa5110c5edc5)
High-rigor contradiction endpoint  : NOT ESTIMABLE (0 events, 54,228 negatives, 12,408 censored)
Distribution endpoint rows         : 37
Major-shift events / negatives     : 21 / 16
Major-shift prevalence             : 0.567568
Distribution censoring             : 66,599 / 66,636 (99.9445%)
Bootstrap attempts                 : 2,000
Valid / invalid replicates         : 2,000 / 0

DESCRIPTIVE DISCRIMINATION RESULTS
                 model  unique_score_values  point_auprc  auprc_bootstrap_95_low  auprc_bootstrap_95_high  auprc_minus_prevalence  auprc_lift_over_prevalence  point_a

In [48]:
# ==================================================================================================
# STAGE 6C STEP 3H — CELL 6C-3H0
# LOCKED LEAVE-ONE-GENE-OUT (LOGO) TEMPORAL GENERALIZATION VALIDATION
#
# Primary cross-gene design:
#   1. Train on BRCA2 + MLH1; test on BRCA1.
#   2. Train on BRCA1 + MLH1; test on BRCA2.
#   3. Train on BRCA1 + BRCA2; test on MLH1.
#
# For each split, the full-GES and no-star-GES pipelines are cloned from the checksum-verified,
# frozen Stage 4C joblib artifacts and refitted using only Stage 4B T0 weak-label rows from the two
# training genes. Predictions are generated before the Stage 6B temporal outcome is loaded.
#
# The held-out-gene evaluation uses the frozen Stage 6B primary-evaluable outcome and compares:
#   - LOGO Full GES
#   - LOGO No-star GES
#   - Frozen review-stars comparator
#   - Frozen combined-metadata comparator
#
# AUPRC is primary. AUROC is secondary. The cell runs 2,000 paired test-set row-bootstrap
# replicates per held-out gene, reports 95% percentile intervals, and applies Holm correction
# separately across the 9 primary-gene AUPRC comparisons and the 9 AUROC comparisons.
#
# Scientific boundary:
#   - No T1 outcome is used to fit, tune, calibrate, threshold, or select either LOGO model.
#   - EGFR is not pooled into the primary LOGO experiment.
#   - No score, weak label, outcome, threshold, feature definition, gene assignment, or frozen
#     artifact is modified.
#   - This cell is read-only and writes no scientific artifact.
# ==================================================================================================

from pathlib import Path
import gc
import hashlib
import json
import re
import time

import joblib
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from scipy.sparse import csr_matrix
from sklearn.base import clone
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


# --------------------------------------------------------------------------------------------------
# 1. GOOGLE DRIVE AND FROZEN PATHS
# --------------------------------------------------------------------------------------------------

DRIVE_ROOT = Path("/content/drive/MyDrive")
if not DRIVE_ROOT.exists():
    from google.colab import drive
    drive.mount("/content/drive")

if not DRIVE_ROOT.exists():
    raise FileNotFoundError("Google Drive is not mounted at /content/drive/MyDrive.")

PROJECT_DIR = DRIVE_ROOT / "GES_RAG_Temporal_Study"

STAGE4_DATA_DIR = PROJECT_DIR / "data_processed" / "stage4_ges"
STAGE4_MODEL_DIR = PROJECT_DIR / "models" / "stage4_ges"
STAGE4_CONFIG_DIR = PROJECT_DIR / "configs" / "stage4_ges"
STAGE6_DIR = PROJECT_DIR / "data_processed" / "stage6_temporal_validation"

WEAK_LABEL_TABLE = (
    STAGE4_DATA_DIR / "stage4b_t0_feature_transforms_and_weak_labels_v1.parquet"
)
STAGE4B_MANIFEST = STAGE4_CONFIG_DIR / "stage4b_weak_label_freeze_manifest_v1.json"
FULL_MODEL_PATH = STAGE4_MODEL_DIR / "stage4c_full_ges_logistic_model_v1.joblib"
NO_STAR_MODEL_PATH = STAGE4_MODEL_DIR / "stage4c_no_star_ges_logistic_model_v1.joblib"
STAGE4C_MANIFEST = STAGE4_CONFIG_DIR / "stage4c_ges_model_freeze_manifest_v1.json"

EVALUABLE_PARQUET = STAGE6_DIR / "stage6b_locked_primary_evaluable_cohort_v1.parquet"
EVALUABLE_SIDECAR = Path(str(EVALUABLE_PARQUET) + ".sha256")

EXPECTED_HASHES = {
    "stage4b_weak_label_table": (
        "c2e9e4f96cd1f61d3962e88d28557dfa1221729b8f79f8fd4d2dafb34847bdb8"
    ),
    "stage4b_manifest": (
        "e766061442e6d4610f661f44a619e41b45dc6363b28f6176d3f9a71f8215c63f"
    ),
    "stage4c_full_model": (
        "0b4a87b16f768484cbdae168fc226cec5e978521172fb03510bfb1ea3e78fa30"
    ),
    "stage4c_no_star_model": (
        "6c3fe4fc7fe8fdde7b8f0f0d608c48e66a07945effb8c67c6b98d35e1955257c"
    ),
    "stage4c_manifest": (
        "c0d8008a4db80c67f5b1c568ddba3496b2411b29bce0db1eb20e63f26613d4ee"
    ),
    "stage6b_evaluable": (
        "c6ad50dfc376a746e1830d000dba975101d88763fd61b8d1983f6300edb6e038"
    ),
}

EXPECTED_STAGE4B_ROWS = 71_659
EXPECTED_STAGE4B_COLUMNS = 43
EXPECTED_STAGE6B_ROWS = 66_636
EXPECTED_STAGE6B_COLUMNS = 79
EXPECTED_EVENTS = 6_485
EXPECTED_NEGATIVES = 60_151

EXPECTED_WEAK_LABEL_ACCOUNTING = {
    "full": {
        "stable": 61_842,
        "unstable": 5_723,
        "unlabeled": 4_094,
        "eligible": 67_565,
    },
    "no_star": {
        "stable": 61_298,
        "unstable": 1_850,
        "unlabeled": 8_511,
        "eligible": 63_148,
    },
}

EXPECTED_HELD_OUT_ACCOUNTING = {
    "BRCA1": {"rows": 21_594, "events": 2_023, "negatives": 19_571},
    "BRCA2": {"rows": 34_152, "events": 3_960, "negatives": 30_192},
    "MLH1": {"rows": 8_701, "events": 425, "negatives": 8_276},
}

PRIMARY_GENES = ["BRCA1", "BRCA2", "MLH1"]
EXPLORATORY_GENE = "EGFR"

OUTCOME_COLUMN = "primary_future_instability"
GENE_COLUMN = "target_gene"
KEY_COLUMN = "rcv_accession"
ROW_ORDER_COLUMN = "t0_row_order"

FULL_FEATURES = [
    "recency_score",
    "recency_missing_flag",
    "submitter_diversity_score",
    "review_confidence",
    "aggregate_conflict_flag",
    "scv_group_entropy_normalized",
]

NO_STAR_FEATURES = [
    "recency_score",
    "recency_missing_flag",
    "submitter_diversity_score",
    "aggregate_conflict_flag",
    "scv_group_entropy_normalized",
]

MODEL_SPECS = {
    "logo_full_ges": {
        "display_name": "LOGO Full GES",
        "score_column": "logo_full_ges_instability_risk",
    },
    "logo_no_star_ges": {
        "display_name": "LOGO No-star GES",
        "score_column": "logo_no_star_ges_instability_risk",
    },
    "review_stars": {
        "display_name": "Review stars",
        "score_column": "review_stars_instability_risk",
    },
    "combined_metadata": {
        "display_name": "Combined metadata",
        "score_column": "combined_metadata_instability_risk",
    },
}

PAIRED_COMPARATORS = ["logo_no_star_ges", "review_stars", "combined_metadata"]

N_BOOTSTRAP = 2_000
RANDOM_SEED = 42
BOOTSTRAP_BATCH_SIZE = 50
CI_QUANTILES = (0.025, 0.975)
MINIMUM_VALID_REPLICATES = 1_000


# --------------------------------------------------------------------------------------------------
# 2. GENERAL HELPERS
# --------------------------------------------------------------------------------------------------

def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def read_sidecar_hash(path: Path) -> str:
    text = path.read_text(encoding="utf-8").strip()
    matches = re.findall(r"\b[a-fA-F0-9]{64}\b", text)
    if not matches:
        raise ValueError(f"No SHA-256 value found in sidecar: {path}")
    return matches[0].lower()


def verify_hash(path: Path, expected: str, label: str) -> str:
    if not path.exists():
        raise FileNotFoundError(f"Missing frozen artifact for {label}:\n{path}")
    observed = sha256_file(path)
    if observed != expected:
        raise AssertionError(
            f"SHA-256 mismatch for {label}.\nExpected: {expected}\nObserved: {observed}"
        )
    return observed


def normalize_gene(value) -> str:
    text = str(value).strip().upper()
    if text in {"BRCA1", "BRCA2", "MLH1", "EGFR"}:
        return text
    return ""


def gene_from_json(value) -> str:
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return ""

    parsed = value
    if isinstance(value, str):
        text = value.strip()
        if not text:
            return ""
        try:
            parsed = json.loads(text)
        except json.JSONDecodeError:
            parsed = [part.strip() for part in re.split(r"[,;|]", text) if part.strip()]

    if isinstance(parsed, dict):
        candidates = list(parsed.keys()) + list(parsed.values())
    elif isinstance(parsed, (list, tuple, set, np.ndarray, pd.Series)):
        candidates = list(parsed)
    else:
        candidates = [parsed]

    genes = []
    for candidate in candidates:
        if isinstance(candidate, (list, tuple, set, dict)):
            nested = gene_from_json(candidate)
            if nested:
                genes.append(nested)
            continue
        gene = normalize_gene(candidate)
        if gene:
            genes.append(gene)

    genes = sorted(set(genes))
    if len(genes) != 1:
        return ""
    return genes[0]


def find_pipeline_component(pipeline: Pipeline, expected_type):
    matches = [step for _, step in pipeline.steps if isinstance(step, expected_type)]
    if len(matches) != 1:
        raise AssertionError(
            f"Expected exactly one {expected_type.__name__} in pipeline; found {len(matches)}."
        )
    return matches[0]


def stable_class_probability(model: Pipeline, features: pd.DataFrame) -> np.ndarray:
    probabilities = model.predict_proba(features)
    classifier = find_pipeline_component(model, LogisticRegression)
    classes = np.asarray(classifier.classes_)
    stable_positions = np.flatnonzero(classes == 1)
    if len(stable_positions) != 1:
        raise AssertionError(f"Model classes do not contain exactly one stable class 1: {classes}")
    return probabilities[:, int(stable_positions[0])]


def percentile_interval(values: np.ndarray) -> tuple[float, float]:
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if len(values) == 0:
        return np.nan, np.nan
    lower, upper = np.quantile(values, CI_QUANTILES)
    return float(lower), float(upper)


def bootstrap_sign_pvalue(differences: np.ndarray) -> float:
    differences = np.asarray(differences, dtype=float)
    differences = differences[np.isfinite(differences)]
    if len(differences) == 0:
        return np.nan

    n = len(differences)
    lower_tail = (np.count_nonzero(differences <= 0.0) + 1) / (n + 1)
    upper_tail = (np.count_nonzero(differences >= 0.0) + 1) / (n + 1)
    return float(min(1.0, 2.0 * min(lower_tail, upper_tail)))


def holm_adjust(p_values: np.ndarray) -> np.ndarray:
    p_values = np.asarray(p_values, dtype=float)
    adjusted = np.full(len(p_values), np.nan, dtype=float)
    valid_positions = np.where(np.isfinite(p_values))[0]

    if len(valid_positions) == 0:
        return adjusted

    valid_p = p_values[valid_positions]
    order = np.argsort(valid_p)
    m = len(valid_p)
    running_max = 0.0

    for rank, position_within_valid in enumerate(order):
        original_position = valid_positions[position_within_valid]
        raw_adjusted = (m - rank) * valid_p[position_within_valid]
        running_max = max(running_max, raw_adjusted)
        adjusted[original_position] = min(1.0, running_max)

    return adjusted


def interval_status(lower: float, upper: float, positive_label: str, negative_label: str) -> str:
    if not np.isfinite(lower) or not np.isfinite(upper):
        return "not_estimable"
    if lower > 0.0:
        return positive_label
    if upper < 0.0:
        return negative_label
    return "interval_includes_null"


# --------------------------------------------------------------------------------------------------
# 3. EXACT TIE-AWARE METRIC CACHE USED FOR PAIRED BOOTSTRAP
# --------------------------------------------------------------------------------------------------

def construct_score_group_cache(scores: np.ndarray, outcomes: np.ndarray) -> dict:
    scores = np.asarray(scores, dtype=np.float64)
    outcomes = np.asarray(outcomes, dtype=np.int8)

    unique_scores, group_index = np.unique(scores, return_inverse=True)
    n_rows = len(scores)
    n_groups = len(unique_scores)
    row_positions = np.arange(n_rows)

    total_group_matrix = csr_matrix(
        (
            np.ones(n_rows, dtype=np.float64),
            (group_index, row_positions),
        ),
        shape=(n_groups, n_rows),
    )

    positive_positions = np.flatnonzero(outcomes == 1)
    positive_group_matrix = csr_matrix(
        (
            np.ones(len(positive_positions), dtype=np.float64),
            (group_index[positive_positions], positive_positions),
        ),
        shape=(n_groups, n_rows),
    )

    return {
        "unique_scores": unique_scores,
        "total_group_matrix": total_group_matrix,
        "positive_group_matrix": positive_group_matrix,
    }


def calculate_grouped_weighted_metrics(
    cache: dict,
    count_matrix: np.ndarray,
    positive_totals: np.ndarray,
) -> tuple[np.ndarray, np.ndarray]:
    count_matrix = np.asarray(count_matrix)
    positive_totals = np.asarray(positive_totals, dtype=np.float64)

    sample_totals = count_matrix.sum(axis=1).astype(np.float64)
    negative_totals = sample_totals - positive_totals

    group_total_counts = np.asarray(
        cache["total_group_matrix"] @ count_matrix.T,
        dtype=np.float64,
    )
    group_positive_counts = np.asarray(
        cache["positive_group_matrix"] @ count_matrix.T,
        dtype=np.float64,
    )
    group_negative_counts = group_total_counts - group_positive_counts

    valid = (positive_totals > 0.0) & (negative_totals > 0.0)

    cumulative_negatives_before = (
        np.cumsum(group_negative_counts, axis=0) - group_negative_counts
    )
    concordant_numerator = np.sum(
        group_positive_counts
        * (cumulative_negatives_before + 0.5 * group_negative_counts),
        axis=0,
    )
    auc_denominator = positive_totals * negative_totals
    auroc = np.full(len(positive_totals), np.nan, dtype=np.float64)
    np.divide(concordant_numerator, auc_denominator, out=auroc, where=valid)

    positive_desc = group_positive_counts[::-1, :]
    total_desc = group_total_counts[::-1, :]
    cumulative_positive = np.cumsum(positive_desc, axis=0)
    cumulative_total = np.cumsum(total_desc, axis=0)

    precision = np.zeros_like(cumulative_positive, dtype=np.float64)
    np.divide(
        cumulative_positive,
        cumulative_total,
        out=precision,
        where=cumulative_total > 0.0,
    )

    ap_numerator = np.sum(precision * positive_desc, axis=0)
    auprc = np.full(len(positive_totals), np.nan, dtype=np.float64)
    np.divide(ap_numerator, positive_totals, out=auprc, where=valid)

    return auprc, auroc


# --------------------------------------------------------------------------------------------------
# 4. FRESHLY VERIFY STAGE 4B AND STAGE 4C BEFORE ANY TEMPORAL OUTCOME IS LOADED
# --------------------------------------------------------------------------------------------------

observed_hashes = {}
observed_hashes["stage4b_weak_label_table"] = verify_hash(
    WEAK_LABEL_TABLE,
    EXPECTED_HASHES["stage4b_weak_label_table"],
    "Stage 4B weak-label table",
)
observed_hashes["stage4b_manifest"] = verify_hash(
    STAGE4B_MANIFEST,
    EXPECTED_HASHES["stage4b_manifest"],
    "Stage 4B manifest",
)
observed_hashes["stage4c_full_model"] = verify_hash(
    FULL_MODEL_PATH,
    EXPECTED_HASHES["stage4c_full_model"],
    "Stage 4C full GES model",
)
observed_hashes["stage4c_no_star_model"] = verify_hash(
    NO_STAR_MODEL_PATH,
    EXPECTED_HASHES["stage4c_no_star_model"],
    "Stage 4C no-star GES model",
)
observed_hashes["stage4c_manifest"] = verify_hash(
    STAGE4C_MANIFEST,
    EXPECTED_HASHES["stage4c_manifest"],
    "Stage 4C model-freeze manifest",
)

stage4b_parquet = pq.ParquetFile(WEAK_LABEL_TABLE)
if stage4b_parquet.metadata.num_rows != EXPECTED_STAGE4B_ROWS:
    raise AssertionError(
        f"Stage 4B row count is {stage4b_parquet.metadata.num_rows:,}; "
        f"expected {EXPECTED_STAGE4B_ROWS:,}."
    )
if stage4b_parquet.metadata.num_columns != EXPECTED_STAGE4B_COLUMNS:
    raise AssertionError(
        f"Stage 4B column count is {stage4b_parquet.metadata.num_columns}; "
        f"expected {EXPECTED_STAGE4B_COLUMNS}."
    )

stage4b_columns = stage4b_parquet.schema_arrow.names
required_stage4b_columns = [
    KEY_COLUMN,
    ROW_ORDER_COLUMN,
    "full_training_eligible",
    "full_weak_label_binary",
    "no_star_training_eligible",
    "no_star_weak_label_binary",
] + sorted(set(FULL_FEATURES + NO_STAR_FEATURES))

missing_stage4b = [column for column in required_stage4b_columns if column not in stage4b_columns]
if missing_stage4b:
    raise KeyError("Missing required Stage 4B columns:\n" + "\n".join(missing_stage4b))

if GENE_COLUMN in stage4b_columns:
    gene_source_column = GENE_COLUMN
elif "target_genes_json" in stage4b_columns:
    gene_source_column = "target_genes_json"
else:
    raise KeyError(
        "Stage 4B has neither target_gene nor target_genes_json; gene assignment cannot be reconstructed."
    )

load_stage4b_columns = list(dict.fromkeys(required_stage4b_columns + [gene_source_column]))
weak = pd.read_parquet(WEAK_LABEL_TABLE, columns=load_stage4b_columns).copy()

if len(weak) != EXPECTED_STAGE4B_ROWS:
    raise AssertionError("Loaded Stage 4B dataframe row count is incorrect.")
if weak[KEY_COLUMN].isna().any() or weak[KEY_COLUMN].astype(str).str.strip().eq("").any():
    raise AssertionError("Missing or blank Stage 4B RCV key detected.")
if weak[KEY_COLUMN].nunique(dropna=False) != EXPECTED_STAGE4B_ROWS:
    raise AssertionError("Stage 4B RCV keys are not unique.")

weak[KEY_COLUMN] = weak[KEY_COLUMN].astype(str).str.strip().str.upper()
weak[ROW_ORDER_COLUMN] = pd.to_numeric(weak[ROW_ORDER_COLUMN], errors="raise").astype(np.int64)
if weak[ROW_ORDER_COLUMN].nunique() != EXPECTED_STAGE4B_ROWS:
    raise AssertionError("Stage 4B t0_row_order is not unique.")
if not np.all(np.diff(weak[ROW_ORDER_COLUMN].to_numpy()) > 0):
    raise AssertionError("Stage 4B rows are not in strictly increasing T0 order.")

if gene_source_column == GENE_COLUMN:
    weak[GENE_COLUMN] = weak[gene_source_column].map(normalize_gene)
else:
    weak[GENE_COLUMN] = weak[gene_source_column].map(gene_from_json)

if weak[GENE_COLUMN].eq("").any():
    bad = int(weak[GENE_COLUMN].eq("").sum())
    raise AssertionError(f"Could not derive exactly one target gene for {bad:,} Stage 4B rows.")

observed_genes = sorted(weak[GENE_COLUMN].unique().tolist())
if observed_genes != ["BRCA1", "BRCA2", "EGFR", "MLH1"]:
    raise AssertionError(f"Unexpected Stage 4B genes: {observed_genes}")

for feature in sorted(set(FULL_FEATURES + NO_STAR_FEATURES)):
    weak[feature] = pd.to_numeric(weak[feature], errors="coerce").astype(float)

for eligibility_column in ["full_training_eligible", "no_star_training_eligible"]:
    weak[eligibility_column] = weak[eligibility_column].fillna(False).astype(bool)

for label_column in ["full_weak_label_binary", "no_star_weak_label_binary"]:
    weak[label_column] = pd.to_numeric(weak[label_column], errors="coerce")
    nonmissing = weak[label_column].dropna().unique()
    if not set(nonmissing).issubset({0, 1, 0.0, 1.0}):
        raise AssertionError(f"Unexpected values in {label_column}: {sorted(nonmissing.tolist())}")

weak_label_accounting_rows = []
for model_key, eligibility_column, label_column in [
    ("full", "full_training_eligible", "full_weak_label_binary"),
    ("no_star", "no_star_training_eligible", "no_star_weak_label_binary"),
]:
    eligible = weak[eligibility_column]
    labels = weak[label_column]
    if labels.loc[eligible].isna().any():
        raise AssertionError(f"Eligible {model_key} weak-label rows contain missing labels.")
    if labels.loc[~eligible].notna().any():
        # A noneligible row may retain an intermediate numeric value in some exports; only the frozen
        # eligibility flag controls training. Record but do not silently use it.
        noneligible_labeled = int(labels.loc[~eligible].notna().sum())
    else:
        noneligible_labeled = 0

    stable = int((labels == 1).sum())
    unstable = int((labels == 0).sum())
    unlabeled = int(labels.isna().sum())
    eligible_count = int(eligible.sum())
    expected = EXPECTED_WEAK_LABEL_ACCOUNTING[model_key]

    if (stable, unstable, unlabeled, eligible_count) != (
        expected["stable"],
        expected["unstable"],
        expected["unlabeled"],
        expected["eligible"],
    ):
        raise AssertionError(
            f"{model_key} weak-label accounting mismatch: "
            f"observed={(stable, unstable, unlabeled, eligible_count)}, "
            f"expected={(expected['stable'], expected['unstable'], expected['unlabeled'], expected['eligible'])}"
        )

    weak_label_accounting_rows.append(
        {
            "model_pathway": model_key,
            "stable_labels": stable,
            "unstable_labels": unstable,
            "unlabeled_rows": unlabeled,
            "training_eligible_rows": eligible_count,
            "noneligible_rows_with_numeric_label": noneligible_labeled,
        }
    )

weak_label_accounting = pd.DataFrame(weak_label_accounting_rows)

frozen_full_pipeline = joblib.load(FULL_MODEL_PATH)
frozen_no_star_pipeline = joblib.load(NO_STAR_MODEL_PATH)

for pipeline_name, pipeline, expected_features in [
    ("full", frozen_full_pipeline, FULL_FEATURES),
    ("no_star", frozen_no_star_pipeline, NO_STAR_FEATURES),
]:
    if not isinstance(pipeline, Pipeline):
        raise TypeError(f"Frozen {pipeline_name} artifact is not an sklearn Pipeline.")
    find_pipeline_component(pipeline, SimpleImputer)
    find_pipeline_component(pipeline, StandardScaler)
    classifier = find_pipeline_component(pipeline, LogisticRegression)
    if getattr(pipeline, "n_features_in_", len(expected_features)) != len(expected_features):
        raise AssertionError(
            f"Frozen {pipeline_name} pipeline expects {getattr(pipeline, 'n_features_in_', None)} "
            f"features; expected {len(expected_features)}."
        )
    if set(np.asarray(classifier.classes_).tolist()) != {0, 1}:
        raise AssertionError(f"Unexpected classes in frozen {pipeline_name} classifier.")

full_classifier = find_pipeline_component(frozen_full_pipeline, LogisticRegression)
no_star_classifier = find_pipeline_component(frozen_no_star_pipeline, LogisticRegression)

frozen_model_settings = {
    "full_solver": full_classifier.solver,
    "full_penalty": full_classifier.penalty,
    "full_C": float(full_classifier.C),
    "full_max_iter": int(full_classifier.max_iter),
    "full_tol": float(full_classifier.tol),
    "full_class_weight": full_classifier.class_weight,
    "full_random_state": full_classifier.random_state,
    "no_star_solver": no_star_classifier.solver,
    "no_star_penalty": no_star_classifier.penalty,
    "no_star_C": float(no_star_classifier.C),
    "no_star_max_iter": int(no_star_classifier.max_iter),
    "no_star_tol": float(no_star_classifier.tol),
    "no_star_class_weight": no_star_classifier.class_weight,
    "no_star_random_state": no_star_classifier.random_state,
}


# --------------------------------------------------------------------------------------------------
# 5. FIT ALL SIX LOGO MODELS USING T0 WEAK LABELS ONLY
#    IMPORTANT: STAGE 6B OUTCOME HAS NOT BEEN OPENED YET.
# --------------------------------------------------------------------------------------------------

training_accounting_rows = []
prediction_frames = []
fitted_models = {}
fit_start = time.time()

for held_out_gene in PRIMARY_GENES:
    training_genes = [gene for gene in PRIMARY_GENES if gene != held_out_gene]
    training_gene_text = "+".join(training_genes)

    split_predictions = weak.loc[
        weak[GENE_COLUMN].eq(held_out_gene),
        [KEY_COLUMN, ROW_ORDER_COLUMN, GENE_COLUMN] + sorted(set(FULL_FEATURES + NO_STAR_FEATURES)),
    ].copy()

    if split_predictions.empty:
        raise AssertionError(f"No Stage 4B rows found for held-out gene {held_out_gene}.")

    for pathway, template_pipeline, features, eligibility_column, label_column, output_column in [
        (
            "full_ges",
            frozen_full_pipeline,
            FULL_FEATURES,
            "full_training_eligible",
            "full_weak_label_binary",
            "logo_full_ges_instability_risk",
        ),
        (
            "no_star_ges",
            frozen_no_star_pipeline,
            NO_STAR_FEATURES,
            "no_star_training_eligible",
            "no_star_weak_label_binary",
            "logo_no_star_ges_instability_risk",
        ),
    ]:
        train_mask = weak[GENE_COLUMN].isin(training_genes) & weak[eligibility_column]
        train = weak.loc[train_mask].copy()
        y_train = train[label_column].astype(int).to_numpy()

        if len(np.unique(y_train)) != 2:
            raise AssertionError(
                f"{held_out_gene}/{pathway} training labels do not contain both classes."
            )

        stable_labels = int((y_train == 1).sum())
        unstable_labels = int((y_train == 0).sum())

        model = clone(template_pipeline)
        model.fit(train[features], y_train)

        fitted_classifier = find_pipeline_component(model, LogisticRegression)
        n_iter = int(np.max(np.asarray(fitted_classifier.n_iter_)))
        converged = n_iter < int(fitted_classifier.max_iter)
        if not converged:
            raise AssertionError(
                f"{held_out_gene}/{pathway} reached max_iter={fitted_classifier.max_iter}."
            )

        p_stable = stable_class_probability(model, split_predictions[features])
        risk = 1.0 - p_stable
        if not np.isfinite(risk).all() or ((risk < 0.0) | (risk > 1.0)).any():
            raise AssertionError(f"Invalid LOGO risk generated for {held_out_gene}/{pathway}.")

        split_predictions[output_column] = risk
        fitted_models[(held_out_gene, pathway)] = model

        training_accounting_rows.append(
            {
                "held_out_gene": held_out_gene,
                "training_genes": training_gene_text,
                "model_pathway": pathway,
                "training_rows": int(len(train)),
                "stable_weak_labels": stable_labels,
                "unstable_weak_labels": unstable_labels,
                "training_prevalence_stable": stable_labels / len(train),
                "features": ";".join(features),
                "solver": fitted_classifier.solver,
                "penalty": fitted_classifier.penalty,
                "C": float(fitted_classifier.C),
                "max_iter": int(fitted_classifier.max_iter),
                "iterations_used": n_iter,
                "converged": converged,
                "t1_outcome_loaded_during_fit": False,
            }
        )

        del train, y_train
        gc.collect()

    prediction_frames.append(
        split_predictions[
            [
                KEY_COLUMN,
                ROW_ORDER_COLUMN,
                GENE_COLUMN,
                "logo_full_ges_instability_risk",
                "logo_no_star_ges_instability_risk",
            ]
        ].copy()
    )

logo_predictions_all_t0 = pd.concat(prediction_frames, ignore_index=True)
training_accounting = pd.DataFrame(training_accounting_rows)

if logo_predictions_all_t0[KEY_COLUMN].duplicated().any():
    raise AssertionError("LOGO prediction table contains duplicate RCV keys.")
if sorted(logo_predictions_all_t0[GENE_COLUMN].unique().tolist()) != PRIMARY_GENES:
    raise AssertionError("LOGO prediction table does not contain exactly the three primary genes.")

fit_elapsed = time.time() - fit_start


# --------------------------------------------------------------------------------------------------
# 6. ONLY NOW VERIFY AND LOAD THE FROZEN STAGE 6B TEMPORAL OUTCOME
# --------------------------------------------------------------------------------------------------

observed_hashes["stage6b_evaluable"] = verify_hash(
    EVALUABLE_PARQUET,
    EXPECTED_HASHES["stage6b_evaluable"],
    "Stage 6B primary-evaluable cohort",
)
if not EVALUABLE_SIDECAR.exists():
    raise FileNotFoundError(f"Missing Stage 6B SHA-256 sidecar:\n{EVALUABLE_SIDECAR}")
if read_sidecar_hash(EVALUABLE_SIDECAR) != observed_hashes["stage6b_evaluable"]:
    raise AssertionError("Stage 6B evaluable-cohort SHA-256 sidecar does not match the file.")

stage6b_parquet = pq.ParquetFile(EVALUABLE_PARQUET)
if stage6b_parquet.metadata.num_rows != EXPECTED_STAGE6B_ROWS:
    raise AssertionError("Unexpected Stage 6B row count.")
if stage6b_parquet.metadata.num_columns != EXPECTED_STAGE6B_COLUMNS:
    raise AssertionError("Unexpected Stage 6B column count.")

stage6b_columns = stage6b_parquet.schema_arrow.names
required_stage6b_columns = [
    KEY_COLUMN,
    ROW_ORDER_COLUMN,
    GENE_COLUMN,
    OUTCOME_COLUMN,
    "review_stars_instability_risk",
    "combined_metadata_instability_risk",
]
missing_stage6b = [column for column in required_stage6b_columns if column not in stage6b_columns]
if missing_stage6b:
    raise KeyError("Missing required Stage 6B columns:\n" + "\n".join(missing_stage6b))

outcome = pd.read_parquet(EVALUABLE_PARQUET, columns=required_stage6b_columns).copy()
outcome[KEY_COLUMN] = outcome[KEY_COLUMN].astype(str).str.strip().str.upper()
outcome[ROW_ORDER_COLUMN] = pd.to_numeric(outcome[ROW_ORDER_COLUMN], errors="raise").astype(np.int64)
outcome[GENE_COLUMN] = outcome[GENE_COLUMN].map(normalize_gene)
outcome[OUTCOME_COLUMN] = pd.to_numeric(outcome[OUTCOME_COLUMN], errors="raise").astype(int)

if len(outcome) != EXPECTED_STAGE6B_ROWS:
    raise AssertionError("Loaded Stage 6B row count is incorrect.")
if outcome[KEY_COLUMN].nunique(dropna=False) != EXPECTED_STAGE6B_ROWS:
    raise AssertionError("Stage 6B RCV keys are not unique.")
if not set(outcome[OUTCOME_COLUMN].unique()).issubset({0, 1}):
    raise AssertionError("Stage 6B primary outcome is not binary.")

observed_events = int(outcome[OUTCOME_COLUMN].sum())
observed_negatives = int((outcome[OUTCOME_COLUMN] == 0).sum())
if (observed_events, observed_negatives) != (EXPECTED_EVENTS, EXPECTED_NEGATIVES):
    raise AssertionError(
        f"Stage 6B outcome accounting mismatch: {observed_events:,}/{observed_negatives:,}."
    )

for column in ["review_stars_instability_risk", "combined_metadata_instability_risk"]:
    outcome[column] = pd.to_numeric(outcome[column], errors="raise").astype(float)
    values = outcome[column].to_numpy()
    if not np.isfinite(values).all() or ((values < 0.0) | (values > 1.0)).any():
        raise AssertionError(f"Invalid frozen comparator values in {column}.")

primary_outcome = outcome.loc[outcome[GENE_COLUMN].isin(PRIMARY_GENES)].copy()

analysis = primary_outcome.merge(
    logo_predictions_all_t0,
    on=[KEY_COLUMN, ROW_ORDER_COLUMN, GENE_COLUMN],
    how="left",
    validate="one_to_one",
    indicator=True,
)

if not analysis["_merge"].eq("both").all():
    missing = int((analysis["_merge"] != "both").sum())
    raise AssertionError(f"{missing:,} primary-gene outcome rows lack LOGO predictions.")
analysis = analysis.drop(columns=["_merge"])

for column in [
    "logo_full_ges_instability_risk",
    "logo_no_star_ges_instability_risk",
]:
    values = pd.to_numeric(analysis[column], errors="raise").to_numpy(dtype=float)
    if not np.isfinite(values).all() or ((values < 0.0) | (values > 1.0)).any():
        raise AssertionError(f"Invalid merged LOGO score in {column}.")

held_out_accounting_rows = []
for gene in PRIMARY_GENES:
    gene_df = analysis.loc[analysis[GENE_COLUMN].eq(gene)]
    rows = int(len(gene_df))
    events = int(gene_df[OUTCOME_COLUMN].sum())
    negatives = rows - events
    expected = EXPECTED_HELD_OUT_ACCOUNTING[gene]
    if (rows, events, negatives) != (
        expected["rows"],
        expected["events"],
        expected["negatives"],
    ):
        raise AssertionError(
            f"Held-out {gene} accounting mismatch: {(rows, events, negatives)}."
        )
    held_out_accounting_rows.append(
        {
            "held_out_gene": gene,
            "rows": rows,
            "events": events,
            "negatives": negatives,
            "event_prevalence": events / rows,
        }
    )

held_out_accounting = pd.DataFrame(held_out_accounting_rows)


# --------------------------------------------------------------------------------------------------
# 7. HELD-OUT-GENE POINT ESTIMATES AND 2,000-REPLICATE PAIRED BOOTSTRAP
# --------------------------------------------------------------------------------------------------

rng = np.random.default_rng(RANDOM_SEED)
model_interval_rows = []
paired_difference_rows = []
score_summary_rows = []
bootstrap_start = time.time()

for held_out_gene in PRIMARY_GENES:
    gene_start = time.time()
    test = analysis.loc[analysis[GENE_COLUMN].eq(held_out_gene)].reset_index(drop=True)
    y = test[OUTCOME_COLUMN].to_numpy(dtype=np.int8)
    n_rows = int(len(test))
    n_events = int(y.sum())
    n_negatives = n_rows - n_events
    prevalence = n_events / n_rows

    if len(np.unique(y)) != 2:
        raise AssertionError(f"Held-out gene {held_out_gene} does not contain both outcome classes.")

    scores = {
        key: test[spec["score_column"]].to_numpy(dtype=np.float64)
        for key, spec in MODEL_SPECS.items()
    }
    caches = {key: construct_score_group_cache(value, y) for key, value in scores.items()}

    original_counts = np.ones((1, n_rows), dtype=np.int16)
    original_positive_total = np.array([n_events], dtype=np.float64)
    point_metrics = {}

    print(
        f"\nPreparing held-out {held_out_gene}: {n_rows:,} rows, {n_events:,} events, "
        f"{n_negatives:,} negatives"
    )

    for model_key, spec in MODEL_SPECS.items():
        score = scores[model_key]
        sklearn_auprc = float(average_precision_score(y, score))
        sklearn_auroc = float(roc_auc_score(y, score))
        fast_auprc, fast_auroc = calculate_grouped_weighted_metrics(
            caches[model_key],
            original_counts,
            original_positive_total,
        )
        if not np.isclose(fast_auprc[0], sklearn_auprc, rtol=1e-11, atol=1e-12):
            raise AssertionError(f"Fast AUPRC validation failed for {held_out_gene}/{model_key}.")
        if not np.isclose(fast_auroc[0], sklearn_auroc, rtol=1e-11, atol=1e-12):
            raise AssertionError(f"Fast AUROC validation failed for {held_out_gene}/{model_key}.")

        point_metrics[model_key] = {"auprc": sklearn_auprc, "auroc": sklearn_auroc}
        score_summary_rows.append(
            {
                "held_out_gene": held_out_gene,
                "model_key": model_key,
                "model": spec["display_name"],
                "unique_score_values": int(len(np.unique(score))),
                "score_min": float(np.min(score)),
                "score_max": float(np.max(score)),
                "score_mean": float(np.mean(score)),
                "mean_score_events": float(np.mean(score[y == 1])),
                "mean_score_negatives": float(np.mean(score[y == 0])),
                "point_auprc": sklearn_auprc,
                "point_auroc": sklearn_auroc,
                "fast_metric_validation": "PASS",
            }
        )

    print("  Exact tie-aware metric validation against scikit-learn: PASS")

    bootstrap_metrics = {
        model_key: {
            "auprc": np.full(N_BOOTSTRAP, np.nan, dtype=np.float64),
            "auroc": np.full(N_BOOTSTRAP, np.nan, dtype=np.float64),
        }
        for model_key in MODEL_SPECS
    }
    bootstrap_prevalence = np.full(N_BOOTSTRAP, np.nan, dtype=np.float64)

    probabilities = np.full(n_rows, 1.0 / n_rows, dtype=np.float64)
    probabilities[-1] = 1.0 - probabilities[:-1].sum()

    for batch_start in range(0, N_BOOTSTRAP, BOOTSTRAP_BATCH_SIZE):
        batch_end = min(batch_start + BOOTSTRAP_BATCH_SIZE, N_BOOTSTRAP)
        batch_size = batch_end - batch_start
        counts = rng.multinomial(n_rows, probabilities, size=batch_size)
        if not np.all(counts.sum(axis=1) == n_rows):
            raise AssertionError(f"Bootstrap sample-size preservation failed for {held_out_gene}.")

        positive_totals = (counts @ y).astype(np.float64)
        valid = (positive_totals > 0.0) & (positive_totals < n_rows)
        bootstrap_prevalence[batch_start:batch_end] = np.where(
            valid,
            positive_totals / n_rows,
            np.nan,
        )

        for model_key in MODEL_SPECS:
            auprc_values, auroc_values = calculate_grouped_weighted_metrics(
                caches[model_key],
                counts,
                positive_totals,
            )
            bootstrap_metrics[model_key]["auprc"][batch_start:batch_end] = auprc_values
            bootstrap_metrics[model_key]["auroc"][batch_start:batch_end] = auroc_values

        if batch_end % 250 == 0 or batch_end == N_BOOTSTRAP:
            valid_so_far = int(
                np.isfinite(bootstrap_metrics["logo_full_ges"]["auprc"][:batch_end]).sum()
            )
            print(
                f"  Completed {batch_end:,}/{N_BOOTSTRAP:,} replicates | "
                f"valid {valid_so_far:,} | elapsed {time.time() - gene_start:.1f}s"
            )

        del counts
        gc.collect()

    valid_mask = np.isfinite(bootstrap_metrics["logo_full_ges"]["auprc"])
    valid_replicates = int(valid_mask.sum())
    invalid_replicates = N_BOOTSTRAP - valid_replicates
    if valid_replicates < MINIMUM_VALID_REPLICATES:
        raise AssertionError(
            f"Held-out {held_out_gene} produced only {valid_replicates:,} valid replicates."
        )

    for model_key in MODEL_SPECS:
        for metric in ["auprc", "auroc"]:
            if not np.array_equal(
                np.isfinite(bootstrap_metrics[model_key][metric]),
                valid_mask,
            ):
                raise AssertionError(
                    f"Paired-validity mismatch for {held_out_gene}/{model_key}/{metric}."
                )

    for model_key, spec in MODEL_SPECS.items():
        auprc_values = bootstrap_metrics[model_key]["auprc"]
        auroc_values = bootstrap_metrics[model_key]["auroc"]
        auprc_low, auprc_high = percentile_interval(auprc_values)
        auroc_low, auroc_high = percentile_interval(auroc_values)
        auprc_null_low, auprc_null_high = percentile_interval(
            auprc_values - bootstrap_prevalence
        )
        auroc_null_low, auroc_null_high = percentile_interval(auroc_values - 0.50)

        model_interval_rows.append(
            {
                "held_out_gene": held_out_gene,
                "training_genes": "+".join(
                    [gene for gene in PRIMARY_GENES if gene != held_out_gene]
                ),
                "model_key": model_key,
                "model": spec["display_name"],
                "rows": n_rows,
                "events": n_events,
                "negatives": n_negatives,
                "held_out_prevalence": prevalence,
                "point_auprc": point_metrics[model_key]["auprc"],
                "auprc_ci_lower": auprc_low,
                "auprc_ci_upper": auprc_high,
                "point_auprc_minus_prevalence": point_metrics[model_key]["auprc"] - prevalence,
                "auprc_minus_prevalence_ci_lower": auprc_null_low,
                "auprc_minus_prevalence_ci_upper": auprc_null_high,
                "auprc_null_status": interval_status(
                    auprc_null_low,
                    auprc_null_high,
                    "supported_above_held_out_prevalence",
                    "supported_below_held_out_prevalence",
                ),
                "point_auroc": point_metrics[model_key]["auroc"],
                "auroc_ci_lower": auroc_low,
                "auroc_ci_upper": auroc_high,
                "point_auroc_minus_0_50": point_metrics[model_key]["auroc"] - 0.50,
                "auroc_minus_0_50_ci_lower": auroc_null_low,
                "auroc_minus_0_50_ci_upper": auroc_null_high,
                "auroc_null_status": interval_status(
                    auroc_null_low,
                    auroc_null_high,
                    "supported_above_0_50",
                    "supported_below_0_50",
                ),
                "attempted_bootstrap_replicates": N_BOOTSTRAP,
                "valid_bootstrap_replicates": valid_replicates,
                "invalid_one_class_replicates": invalid_replicates,
            }
        )

    for comparator_key in PAIRED_COMPARATORS:
        comparator_name = MODEL_SPECS[comparator_key]["display_name"]
        for metric in ["auprc", "auroc"]:
            differences = (
                bootstrap_metrics["logo_full_ges"][metric]
                - bootstrap_metrics[comparator_key][metric]
            )
            finite = differences[np.isfinite(differences)]
            difference_low, difference_high = percentile_interval(finite)
            point_difference = (
                point_metrics["logo_full_ges"][metric]
                - point_metrics[comparator_key][metric]
            )

            paired_difference_rows.append(
                {
                    "metric": metric.upper(),
                    "held_out_gene": held_out_gene,
                    "training_genes": "+".join(
                        [gene for gene in PRIMARY_GENES if gene != held_out_gene]
                    ),
                    "comparison": f"LOGO Full GES minus {comparator_name}",
                    "comparator_key": comparator_key,
                    "rows": n_rows,
                    "events": n_events,
                    "negatives": n_negatives,
                    "point_difference": point_difference,
                    "difference_ci_lower": difference_low,
                    "difference_ci_upper": difference_high,
                    "paired_interval_status": interval_status(
                        difference_low,
                        difference_high,
                        "logo_full_ges_supported_higher",
                        "logo_full_ges_supported_lower",
                    ),
                    "bootstrap_probability_logo_full_greater": float(
                        np.mean(finite > 0.0)
                    ),
                    "bootstrap_probability_equal": float(np.mean(finite == 0.0)),
                    "bootstrap_sign_p_value": bootstrap_sign_pvalue(finite),
                    "attempted_bootstrap_replicates": N_BOOTSTRAP,
                    "valid_bootstrap_replicates": valid_replicates,
                    "invalid_one_class_replicates": invalid_replicates,
                }
            )

    print(
        f"  Held-out {held_out_gene} complete: {valid_replicates:,} valid, "
        f"{invalid_replicates:,} invalid one-class replicates"
    )

    del test, y, scores, caches, bootstrap_metrics, bootstrap_prevalence
    gc.collect()

bootstrap_elapsed = time.time() - bootstrap_start


# --------------------------------------------------------------------------------------------------
# 8. RESULT TABLES AND HOLM CORRECTION
# --------------------------------------------------------------------------------------------------

logo_model_intervals = pd.DataFrame(model_interval_rows)
logo_paired_differences = pd.DataFrame(paired_difference_rows)
logo_score_summary = pd.DataFrame(score_summary_rows)

expected_model_rows = len(PRIMARY_GENES) * len(MODEL_SPECS)
expected_paired_rows = len(PRIMARY_GENES) * len(PAIRED_COMPARATORS) * 2
if len(logo_model_intervals) != expected_model_rows:
    raise AssertionError(
        f"LOGO model interval table has {len(logo_model_intervals)} rows; "
        f"expected {expected_model_rows}."
    )
if len(logo_paired_differences) != expected_paired_rows:
    raise AssertionError(
        f"LOGO paired table has {len(logo_paired_differences)} rows; "
        f"expected {expected_paired_rows}."
    )

logo_paired_differences["holm_adjusted_bootstrap_sign_p"] = np.nan
for metric in ["AUPRC", "AUROC"]:
    mask = logo_paired_differences["metric"].eq(metric)
    p_values = logo_paired_differences.loc[mask, "bootstrap_sign_p_value"].to_numpy(float)
    if len(p_values) != 9:
        raise AssertionError(f"{metric} LOGO multiplicity family has {len(p_values)} tests; expected 9.")
    logo_paired_differences.loc[mask, "holm_adjusted_bootstrap_sign_p"] = holm_adjust(p_values)

logo_paired_differences["holm_supported_at_0_05"] = (
    logo_paired_differences["holm_adjusted_bootstrap_sign_p"] < 0.05
)
logo_paired_differences["multiplicity_family"] = (
    "three_primary_held_out_genes_x_three_comparators"
)

if logo_paired_differences["holm_adjusted_bootstrap_sign_p"].isna().any():
    raise AssertionError("At least one LOGO Holm-adjusted value is missing.")
if (logo_model_intervals["valid_bootstrap_replicates"] < MINIMUM_VALID_REPLICATES).any():
    raise AssertionError("At least one held-out-gene/model interval has too few valid replicates.")


# --------------------------------------------------------------------------------------------------
# 9. DISPLAY AND FINAL DECISION
# --------------------------------------------------------------------------------------------------

model_display_columns = [
    "held_out_gene",
    "training_genes",
    "model",
    "rows",
    "events",
    "negatives",
    "held_out_prevalence",
    "point_auprc",
    "auprc_ci_lower",
    "auprc_ci_upper",
    "point_auprc_minus_prevalence",
    "auprc_minus_prevalence_ci_lower",
    "auprc_minus_prevalence_ci_upper",
    "auprc_null_status",
    "point_auroc",
    "auroc_ci_lower",
    "auroc_ci_upper",
    "point_auroc_minus_0_50",
    "auroc_minus_0_50_ci_lower",
    "auroc_minus_0_50_ci_upper",
    "auroc_null_status",
    "valid_bootstrap_replicates",
    "invalid_one_class_replicates",
]

paired_display_columns = [
    "metric",
    "held_out_gene",
    "training_genes",
    "comparison",
    "rows",
    "events",
    "negatives",
    "point_difference",
    "difference_ci_lower",
    "difference_ci_upper",
    "paired_interval_status",
    "bootstrap_probability_logo_full_greater",
    "bootstrap_sign_p_value",
    "holm_adjusted_bootstrap_sign_p",
    "holm_supported_at_0_05",
    "multiplicity_family",
    "valid_bootstrap_replicates",
    "invalid_one_class_replicates",
]

separator = "=" * 150
print("\n" + separator)
print("STAGE 6C STEP 3H — CELL 6C-3H0 — LOCKED LEAVE-ONE-GENE-OUT VALIDATION")
print(separator)
print(f"Stage 4B weak-label SHA-256        : PASS ({observed_hashes['stage4b_weak_label_table']})")
print(f"Stage 4B manifest SHA-256          : PASS ({observed_hashes['stage4b_manifest']})")
print(f"Stage 4C full-model SHA-256        : PASS ({observed_hashes['stage4c_full_model']})")
print(f"Stage 4C no-star-model SHA-256     : PASS ({observed_hashes['stage4c_no_star_model']})")
print(f"Stage 4C manifest SHA-256          : PASS ({observed_hashes['stage4c_manifest']})")
print(f"Stage 6B evaluable SHA-256         : PASS ({observed_hashes['stage6b_evaluable']})")
print(f"Stage 4B dimensions                : PASS ({EXPECTED_STAGE4B_ROWS:,} × {EXPECTED_STAGE4B_COLUMNS})")
print(f"Stage 6B dimensions                : PASS ({EXPECTED_STAGE6B_ROWS:,} × {EXPECTED_STAGE6B_COLUMNS})")
print(f"Stage 6B events / negatives        : PASS ({observed_events:,} / {observed_negatives:,})")
print("Temporal outcome loaded during fit: No")
print("Held-out model tuning              : None")
print("EGFR pooled into primary LOGO      : No")
print(f"Bootstrap attempts per held-out gene: {N_BOOTSTRAP:,}")
print(f"Random seed                        : {RANDOM_SEED}")
print("Holm families                      : 9 AUPRC + 9 AUROC paired comparisons")
print(f"Model fitting elapsed              : {fit_elapsed:.1f}s")
print(f"Bootstrap elapsed                  : {bootstrap_elapsed:.1f}s")

print("\nFROZEN PIPELINE SETTINGS RECOVERED FROM CHECKSUM-VERIFIED JOBLIBS")
for key, value in frozen_model_settings.items():
    print(f"{key:34s}: {value}")

print("\nGLOBAL WEAK-LABEL ACCOUNTING")
print(weak_label_accounting.to_string(index=False))

print("\nLOGO TRAINING ACCOUNTING")
print(training_accounting.to_string(index=False))

print("\nHELD-OUT TEST ACCOUNTING")
print(held_out_accounting.to_string(index=False))

print("\nHELD-OUT SCORE AND POINT-ESTIMATE SUMMARY")
print(logo_score_summary.to_string(index=False))

print("\nHELD-OUT MODEL-SPECIFIC INTERVALS")
print(logo_model_intervals[model_display_columns].to_string(index=False))

print("\nPAIRED LOGO FULL-GES-MINUS-COMPARATOR INFERENCE")
print(logo_paired_differences[paired_display_columns].to_string(index=False))

print("\nSCIENTIFIC INTERPRETATION BOUNDARY")
print("-" * 150)
print(
    "This is the prespecified cross-gene generalization test. Each GES model was refitted only on T0 weak-label "
    "records from two primary genes and evaluated on the third gene using the locked T0-to-T1 temporal outcome."
)
print(
    "A positive held-out result requires attention to absolute discrimination, confidence intervals, and performance "
    "against both review stars and the strong combined-metadata baseline; statistical separation alone does not imply "
    "clinical utility or calibration."
)
print(
    "EGFR remains separate and exploratory. No downstream RAG benefit, patient outcome, treatment safety, or clinical "
    "deployment claim is evaluated in this cell."
)

print("\nCELL DECISION")
print("-" * 150)
print("PASS_STAGE6C_LEAVE_ONE_GENE_OUT_PAIRED_BOOTSTRAP_VALIDATION_COMPLETE")
print(
    "All three prespecified train-two/test-one primary-gene splits, 2,000 paired bootstrap attempts per split, "
    "and separate 9-test Holm corrections for AUPRC and AUROC are complete."
)
print(
    "No frozen score, weak label, outcome, feature definition, threshold, gene assignment, cohort membership, "
    "or scientific artifact was modified. No artifact was written."
)

TypeError: Frozen full artifact is not an sklearn Pipeline.

In [49]:
# ==================================================================================================
# STAGE 6C STEP 3H — CELL 6C-3H0
# LOCKED LEAVE-ONE-GENE-OUT (LOGO) TEMPORAL GENERALIZATION VALIDATION
#
# Primary cross-gene design:
#   1. Train on BRCA2 + MLH1; test on BRCA1.
#   2. Train on BRCA1 + MLH1; test on BRCA2.
#   3. Train on BRCA1 + BRCA2; test on MLH1.
#
# For each split, the full-GES and no-star-GES pipelines are cloned from the checksum-verified,
# frozen Stage 4C joblib artifacts and refitted using only Stage 4B T0 weak-label rows from the two
# training genes. Predictions are generated before the Stage 6B temporal outcome is loaded.
#
# The held-out-gene evaluation uses the frozen Stage 6B primary-evaluable outcome and compares:
#   - LOGO Full GES
#   - LOGO No-star GES
#   - Frozen review-stars comparator
#   - Frozen combined-metadata comparator
#
# AUPRC is primary. AUROC is secondary. The cell runs 2,000 paired test-set row-bootstrap
# replicates per held-out gene, reports 95% percentile intervals, and applies Holm correction
# separately across the 9 primary-gene AUPRC comparisons and the 9 AUROC comparisons.
#
# Scientific boundary:
#   - No T1 outcome is used to fit, tune, calibrate, threshold, or select either LOGO model.
#   - EGFR is not pooled into the primary LOGO experiment.
#   - No score, weak label, outcome, threshold, feature definition, gene assignment, or frozen
#     artifact is modified.
#   - This cell is read-only and writes no scientific artifact.
# ==================================================================================================

from pathlib import Path
import gc
import hashlib
import json
import re
import time

import joblib
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from scipy.sparse import csr_matrix
from sklearn.base import clone
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


# --------------------------------------------------------------------------------------------------
# 1. GOOGLE DRIVE AND FROZEN PATHS
# --------------------------------------------------------------------------------------------------

DRIVE_ROOT = Path("/content/drive/MyDrive")
if not DRIVE_ROOT.exists():
    from google.colab import drive
    drive.mount("/content/drive")

if not DRIVE_ROOT.exists():
    raise FileNotFoundError("Google Drive is not mounted at /content/drive/MyDrive.")

PROJECT_DIR = DRIVE_ROOT / "GES_RAG_Temporal_Study"

STAGE4_DATA_DIR = PROJECT_DIR / "data_processed" / "stage4_ges"
STAGE4_MODEL_DIR = PROJECT_DIR / "models" / "stage4_ges"
STAGE4_CONFIG_DIR = PROJECT_DIR / "configs" / "stage4_ges"
STAGE6_DIR = PROJECT_DIR / "data_processed" / "stage6_temporal_validation"

WEAK_LABEL_TABLE = (
    STAGE4_DATA_DIR / "stage4b_t0_feature_transforms_and_weak_labels_v1.parquet"
)
STAGE4B_MANIFEST = STAGE4_CONFIG_DIR / "stage4b_weak_label_freeze_manifest_v1.json"
FULL_MODEL_PATH = STAGE4_MODEL_DIR / "stage4c_full_ges_logistic_model_v1.joblib"
NO_STAR_MODEL_PATH = STAGE4_MODEL_DIR / "stage4c_no_star_ges_logistic_model_v1.joblib"
STAGE4C_MANIFEST = STAGE4_CONFIG_DIR / "stage4c_ges_model_freeze_manifest_v1.json"

EVALUABLE_PARQUET = STAGE6_DIR / "stage6b_locked_primary_evaluable_cohort_v1.parquet"
EVALUABLE_SIDECAR = Path(str(EVALUABLE_PARQUET) + ".sha256")

EXPECTED_HASHES = {
    "stage4b_weak_label_table": (
        "c2e9e4f96cd1f61d3962e88d28557dfa1221729b8f79f8fd4d2dafb34847bdb8"
    ),
    "stage4b_manifest": (
        "e766061442e6d4610f661f44a619e41b45dc6363b28f6176d3f9a71f8215c63f"
    ),
    "stage4c_full_model": (
        "0b4a87b16f768484cbdae168fc226cec5e978521172fb03510bfb1ea3e78fa30"
    ),
    "stage4c_no_star_model": (
        "6c3fe4fc7fe8fdde7b8f0f0d608c48e66a07945effb8c67c6b98d35e1955257c"
    ),
    "stage4c_manifest": (
        "c0d8008a4db80c67f5b1c568ddba3496b2411b29bce0db1eb20e63f26613d4ee"
    ),
    "stage6b_evaluable": (
        "c6ad50dfc376a746e1830d000dba975101d88763fd61b8d1983f6300edb6e038"
    ),
}

EXPECTED_STAGE4B_ROWS = 71_659
EXPECTED_STAGE4B_COLUMNS = 43
EXPECTED_STAGE6B_ROWS = 66_636
EXPECTED_STAGE6B_COLUMNS = 79
EXPECTED_EVENTS = 6_485
EXPECTED_NEGATIVES = 60_151

EXPECTED_WEAK_LABEL_ACCOUNTING = {
    "full": {
        "stable": 61_842,
        "unstable": 5_723,
        "unlabeled": 4_094,
        "eligible": 67_565,
    },
    "no_star": {
        "stable": 61_298,
        "unstable": 1_850,
        "unlabeled": 8_511,
        "eligible": 63_148,
    },
}

EXPECTED_HELD_OUT_ACCOUNTING = {
    "BRCA1": {"rows": 21_594, "events": 2_023, "negatives": 19_571},
    "BRCA2": {"rows": 34_152, "events": 3_960, "negatives": 30_192},
    "MLH1": {"rows": 8_701, "events": 425, "negatives": 8_276},
}

PRIMARY_GENES = ["BRCA1", "BRCA2", "MLH1"]
EXPLORATORY_GENE = "EGFR"

OUTCOME_COLUMN = "primary_future_instability"
GENE_COLUMN = "target_gene"
KEY_COLUMN = "rcv_accession"
ROW_ORDER_COLUMN = "t0_row_order"

FULL_FEATURES = [
    "recency_score",
    "recency_missing_flag",
    "submitter_diversity_score",
    "review_confidence",
    "aggregate_conflict_flag",
    "scv_group_entropy_normalized",
]

NO_STAR_FEATURES = [
    "recency_score",
    "recency_missing_flag",
    "submitter_diversity_score",
    "aggregate_conflict_flag",
    "scv_group_entropy_normalized",
]

MODEL_SPECS = {
    "logo_full_ges": {
        "display_name": "LOGO Full GES",
        "score_column": "logo_full_ges_instability_risk",
    },
    "logo_no_star_ges": {
        "display_name": "LOGO No-star GES",
        "score_column": "logo_no_star_ges_instability_risk",
    },
    "review_stars": {
        "display_name": "Review stars",
        "score_column": "review_stars_instability_risk",
    },
    "combined_metadata": {
        "display_name": "Combined metadata",
        "score_column": "combined_metadata_instability_risk",
    },
}

PAIRED_COMPARATORS = ["logo_no_star_ges", "review_stars", "combined_metadata"]

N_BOOTSTRAP = 2_000
RANDOM_SEED = 42
BOOTSTRAP_BATCH_SIZE = 50
CI_QUANTILES = (0.025, 0.975)
MINIMUM_VALID_REPLICATES = 1_000


# --------------------------------------------------------------------------------------------------
# 2. GENERAL HELPERS
# --------------------------------------------------------------------------------------------------

def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def read_sidecar_hash(path: Path) -> str:
    text = path.read_text(encoding="utf-8").strip()
    matches = re.findall(r"\b[a-fA-F0-9]{64}\b", text)
    if not matches:
        raise ValueError(f"No SHA-256 value found in sidecar: {path}")
    return matches[0].lower()


def verify_hash(path: Path, expected: str, label: str) -> str:
    if not path.exists():
        raise FileNotFoundError(f"Missing frozen artifact for {label}:\n{path}")
    observed = sha256_file(path)
    if observed != expected:
        raise AssertionError(
            f"SHA-256 mismatch for {label}.\nExpected: {expected}\nObserved: {observed}"
        )
    return observed


def normalize_gene(value) -> str:
    text = str(value).strip().upper()
    if text in {"BRCA1", "BRCA2", "MLH1", "EGFR"}:
        return text
    return ""


def gene_from_json(value) -> str:
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return ""

    parsed = value
    if isinstance(value, str):
        text = value.strip()
        if not text:
            return ""
        try:
            parsed = json.loads(text)
        except json.JSONDecodeError:
            parsed = [part.strip() for part in re.split(r"[,;|]", text) if part.strip()]

    if isinstance(parsed, dict):
        candidates = list(parsed.keys()) + list(parsed.values())
    elif isinstance(parsed, (list, tuple, set, np.ndarray, pd.Series)):
        candidates = list(parsed)
    else:
        candidates = [parsed]

    genes = []
    for candidate in candidates:
        if isinstance(candidate, (list, tuple, set, dict)):
            nested = gene_from_json(candidate)
            if nested:
                genes.append(nested)
            continue
        gene = normalize_gene(candidate)
        if gene:
            genes.append(gene)

    genes = sorted(set(genes))
    if len(genes) != 1:
        return ""
    return genes[0]


def extract_sklearn_pipeline(artifact, artifact_name: str):
    """Extract exactly one sklearn Pipeline from a checksum-verified joblib bundle.

    Stage 4C joblib files may serialize a metadata bundle rather than placing the Pipeline at the
    top level. This helper is read-only: it unwraps the existing fitted Pipeline without altering
    the frozen artifact or reconstructing model settings from assumptions.
    """
    if isinstance(artifact, Pipeline):
        return artifact, "<top-level>"

    preferred_keys = (
        "pipeline",
        "model_pipeline",
        "fitted_pipeline",
        "sklearn_pipeline",
        "model",
        "estimator",
        "classifier",
    )

    # First honor common explicit bundle keys in deterministic order.
    if isinstance(artifact, dict):
        for key in preferred_keys:
            if key in artifact and isinstance(artifact[key], Pipeline):
                return artifact[key], f"[{key!r}]"

    # Then recursively inspect containers and selected object attributes.
    matches = []
    visited = set()

    def walk(obj, location: str, depth: int = 0):
        if depth > 8:
            return
        obj_id = id(obj)
        if obj_id in visited:
            return
        visited.add(obj_id)

        if isinstance(obj, Pipeline):
            matches.append((location, obj))
            return

        if isinstance(obj, dict):
            # Preferred keys first, followed by all remaining keys in stable text order.
            ordered_keys = [key for key in preferred_keys if key in obj]
            ordered_keys += sorted(
                [key for key in obj.keys() if key not in ordered_keys], key=lambda value: str(value)
            )
            for key in ordered_keys:
                walk(obj[key], f"{location}[{key!r}]", depth + 1)
            return

        if isinstance(obj, (list, tuple)):
            for index, value in enumerate(obj):
                walk(value, f"{location}[{index}]", depth + 1)
            return

        # Some joblib bundles are lightweight custom objects with model/pipeline attributes.
        for attr in preferred_keys:
            if hasattr(obj, attr):
                try:
                    value = getattr(obj, attr)
                except Exception:
                    continue
                walk(value, f"{location}.{attr}", depth + 1)

    walk(artifact, "<top-level>")

    # Deduplicate aliases that point to the same Pipeline object.
    unique = {}
    for location, pipeline in matches:
        unique.setdefault(id(pipeline), (location, pipeline))
    unique_matches = list(unique.values())

    if len(unique_matches) == 1:
        return unique_matches[0][1], unique_matches[0][0]

    artifact_type = f"{type(artifact).__module__}.{type(artifact).__name__}"
    if isinstance(artifact, dict):
        structure = f"top-level keys={sorted(map(str, artifact.keys()))}"
    elif isinstance(artifact, (list, tuple)):
        structure = f"top-level length={len(artifact)}"
    else:
        structure = f"top-level attributes checked={list(preferred_keys)}"

    if not unique_matches:
        raise TypeError(
            f"Frozen {artifact_name} artifact is {artifact_type}, not a top-level sklearn Pipeline, "
            f"and no nested Pipeline was found ({structure})."
        )

    locations = [location for location, _ in unique_matches]
    raise TypeError(
        f"Frozen {artifact_name} artifact contains multiple distinct sklearn Pipelines at "
        f"{locations}; refusing to choose one ambiguously."
    )


def find_pipeline_component(pipeline: Pipeline, expected_type):
    matches = [step for _, step in pipeline.steps if isinstance(step, expected_type)]
    if len(matches) != 1:
        raise AssertionError(
            f"Expected exactly one {expected_type.__name__} in pipeline; found {len(matches)}."
        )
    return matches[0]


def stable_class_probability(model: Pipeline, features: pd.DataFrame) -> np.ndarray:
    probabilities = model.predict_proba(features)
    classifier = find_pipeline_component(model, LogisticRegression)
    classes = np.asarray(classifier.classes_)
    stable_positions = np.flatnonzero(classes == 1)
    if len(stable_positions) != 1:
        raise AssertionError(f"Model classes do not contain exactly one stable class 1: {classes}")
    return probabilities[:, int(stable_positions[0])]


def percentile_interval(values: np.ndarray) -> tuple[float, float]:
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if len(values) == 0:
        return np.nan, np.nan
    lower, upper = np.quantile(values, CI_QUANTILES)
    return float(lower), float(upper)


def bootstrap_sign_pvalue(differences: np.ndarray) -> float:
    differences = np.asarray(differences, dtype=float)
    differences = differences[np.isfinite(differences)]
    if len(differences) == 0:
        return np.nan

    n = len(differences)
    lower_tail = (np.count_nonzero(differences <= 0.0) + 1) / (n + 1)
    upper_tail = (np.count_nonzero(differences >= 0.0) + 1) / (n + 1)
    return float(min(1.0, 2.0 * min(lower_tail, upper_tail)))


def holm_adjust(p_values: np.ndarray) -> np.ndarray:
    p_values = np.asarray(p_values, dtype=float)
    adjusted = np.full(len(p_values), np.nan, dtype=float)
    valid_positions = np.where(np.isfinite(p_values))[0]

    if len(valid_positions) == 0:
        return adjusted

    valid_p = p_values[valid_positions]
    order = np.argsort(valid_p)
    m = len(valid_p)
    running_max = 0.0

    for rank, position_within_valid in enumerate(order):
        original_position = valid_positions[position_within_valid]
        raw_adjusted = (m - rank) * valid_p[position_within_valid]
        running_max = max(running_max, raw_adjusted)
        adjusted[original_position] = min(1.0, running_max)

    return adjusted


def interval_status(lower: float, upper: float, positive_label: str, negative_label: str) -> str:
    if not np.isfinite(lower) or not np.isfinite(upper):
        return "not_estimable"
    if lower > 0.0:
        return positive_label
    if upper < 0.0:
        return negative_label
    return "interval_includes_null"


# --------------------------------------------------------------------------------------------------
# 3. EXACT TIE-AWARE METRIC CACHE USED FOR PAIRED BOOTSTRAP
# --------------------------------------------------------------------------------------------------

def construct_score_group_cache(scores: np.ndarray, outcomes: np.ndarray) -> dict:
    scores = np.asarray(scores, dtype=np.float64)
    outcomes = np.asarray(outcomes, dtype=np.int8)

    unique_scores, group_index = np.unique(scores, return_inverse=True)
    n_rows = len(scores)
    n_groups = len(unique_scores)
    row_positions = np.arange(n_rows)

    total_group_matrix = csr_matrix(
        (
            np.ones(n_rows, dtype=np.float64),
            (group_index, row_positions),
        ),
        shape=(n_groups, n_rows),
    )

    positive_positions = np.flatnonzero(outcomes == 1)
    positive_group_matrix = csr_matrix(
        (
            np.ones(len(positive_positions), dtype=np.float64),
            (group_index[positive_positions], positive_positions),
        ),
        shape=(n_groups, n_rows),
    )

    return {
        "unique_scores": unique_scores,
        "total_group_matrix": total_group_matrix,
        "positive_group_matrix": positive_group_matrix,
    }


def calculate_grouped_weighted_metrics(
    cache: dict,
    count_matrix: np.ndarray,
    positive_totals: np.ndarray,
) -> tuple[np.ndarray, np.ndarray]:
    count_matrix = np.asarray(count_matrix)
    positive_totals = np.asarray(positive_totals, dtype=np.float64)

    sample_totals = count_matrix.sum(axis=1).astype(np.float64)
    negative_totals = sample_totals - positive_totals

    group_total_counts = np.asarray(
        cache["total_group_matrix"] @ count_matrix.T,
        dtype=np.float64,
    )
    group_positive_counts = np.asarray(
        cache["positive_group_matrix"] @ count_matrix.T,
        dtype=np.float64,
    )
    group_negative_counts = group_total_counts - group_positive_counts

    valid = (positive_totals > 0.0) & (negative_totals > 0.0)

    cumulative_negatives_before = (
        np.cumsum(group_negative_counts, axis=0) - group_negative_counts
    )
    concordant_numerator = np.sum(
        group_positive_counts
        * (cumulative_negatives_before + 0.5 * group_negative_counts),
        axis=0,
    )
    auc_denominator = positive_totals * negative_totals
    auroc = np.full(len(positive_totals), np.nan, dtype=np.float64)
    np.divide(concordant_numerator, auc_denominator, out=auroc, where=valid)

    positive_desc = group_positive_counts[::-1, :]
    total_desc = group_total_counts[::-1, :]
    cumulative_positive = np.cumsum(positive_desc, axis=0)
    cumulative_total = np.cumsum(total_desc, axis=0)

    precision = np.zeros_like(cumulative_positive, dtype=np.float64)
    np.divide(
        cumulative_positive,
        cumulative_total,
        out=precision,
        where=cumulative_total > 0.0,
    )

    ap_numerator = np.sum(precision * positive_desc, axis=0)
    auprc = np.full(len(positive_totals), np.nan, dtype=np.float64)
    np.divide(ap_numerator, positive_totals, out=auprc, where=valid)

    return auprc, auroc


# --------------------------------------------------------------------------------------------------
# 4. FRESHLY VERIFY STAGE 4B AND STAGE 4C BEFORE ANY TEMPORAL OUTCOME IS LOADED
# --------------------------------------------------------------------------------------------------

observed_hashes = {}
observed_hashes["stage4b_weak_label_table"] = verify_hash(
    WEAK_LABEL_TABLE,
    EXPECTED_HASHES["stage4b_weak_label_table"],
    "Stage 4B weak-label table",
)
observed_hashes["stage4b_manifest"] = verify_hash(
    STAGE4B_MANIFEST,
    EXPECTED_HASHES["stage4b_manifest"],
    "Stage 4B manifest",
)
observed_hashes["stage4c_full_model"] = verify_hash(
    FULL_MODEL_PATH,
    EXPECTED_HASHES["stage4c_full_model"],
    "Stage 4C full GES model",
)
observed_hashes["stage4c_no_star_model"] = verify_hash(
    NO_STAR_MODEL_PATH,
    EXPECTED_HASHES["stage4c_no_star_model"],
    "Stage 4C no-star GES model",
)
observed_hashes["stage4c_manifest"] = verify_hash(
    STAGE4C_MANIFEST,
    EXPECTED_HASHES["stage4c_manifest"],
    "Stage 4C model-freeze manifest",
)

stage4b_parquet = pq.ParquetFile(WEAK_LABEL_TABLE)
if stage4b_parquet.metadata.num_rows != EXPECTED_STAGE4B_ROWS:
    raise AssertionError(
        f"Stage 4B row count is {stage4b_parquet.metadata.num_rows:,}; "
        f"expected {EXPECTED_STAGE4B_ROWS:,}."
    )
if stage4b_parquet.metadata.num_columns != EXPECTED_STAGE4B_COLUMNS:
    raise AssertionError(
        f"Stage 4B column count is {stage4b_parquet.metadata.num_columns}; "
        f"expected {EXPECTED_STAGE4B_COLUMNS}."
    )

stage4b_columns = stage4b_parquet.schema_arrow.names
required_stage4b_columns = [
    KEY_COLUMN,
    ROW_ORDER_COLUMN,
    "full_training_eligible",
    "full_weak_label_binary",
    "no_star_training_eligible",
    "no_star_weak_label_binary",
] + sorted(set(FULL_FEATURES + NO_STAR_FEATURES))

missing_stage4b = [column for column in required_stage4b_columns if column not in stage4b_columns]
if missing_stage4b:
    raise KeyError("Missing required Stage 4B columns:\n" + "\n".join(missing_stage4b))

if GENE_COLUMN in stage4b_columns:
    gene_source_column = GENE_COLUMN
elif "target_genes_json" in stage4b_columns:
    gene_source_column = "target_genes_json"
else:
    raise KeyError(
        "Stage 4B has neither target_gene nor target_genes_json; gene assignment cannot be reconstructed."
    )

load_stage4b_columns = list(dict.fromkeys(required_stage4b_columns + [gene_source_column]))
weak = pd.read_parquet(WEAK_LABEL_TABLE, columns=load_stage4b_columns).copy()

if len(weak) != EXPECTED_STAGE4B_ROWS:
    raise AssertionError("Loaded Stage 4B dataframe row count is incorrect.")
if weak[KEY_COLUMN].isna().any() or weak[KEY_COLUMN].astype(str).str.strip().eq("").any():
    raise AssertionError("Missing or blank Stage 4B RCV key detected.")
if weak[KEY_COLUMN].nunique(dropna=False) != EXPECTED_STAGE4B_ROWS:
    raise AssertionError("Stage 4B RCV keys are not unique.")

weak[KEY_COLUMN] = weak[KEY_COLUMN].astype(str).str.strip().str.upper()
weak[ROW_ORDER_COLUMN] = pd.to_numeric(weak[ROW_ORDER_COLUMN], errors="raise").astype(np.int64)
if weak[ROW_ORDER_COLUMN].nunique() != EXPECTED_STAGE4B_ROWS:
    raise AssertionError("Stage 4B t0_row_order is not unique.")
if not np.all(np.diff(weak[ROW_ORDER_COLUMN].to_numpy()) > 0):
    raise AssertionError("Stage 4B rows are not in strictly increasing T0 order.")

if gene_source_column == GENE_COLUMN:
    weak[GENE_COLUMN] = weak[gene_source_column].map(normalize_gene)
else:
    weak[GENE_COLUMN] = weak[gene_source_column].map(gene_from_json)

if weak[GENE_COLUMN].eq("").any():
    bad = int(weak[GENE_COLUMN].eq("").sum())
    raise AssertionError(f"Could not derive exactly one target gene for {bad:,} Stage 4B rows.")

observed_genes = sorted(weak[GENE_COLUMN].unique().tolist())
if observed_genes != ["BRCA1", "BRCA2", "EGFR", "MLH1"]:
    raise AssertionError(f"Unexpected Stage 4B genes: {observed_genes}")

for feature in sorted(set(FULL_FEATURES + NO_STAR_FEATURES)):
    weak[feature] = pd.to_numeric(weak[feature], errors="coerce").astype(float)

for eligibility_column in ["full_training_eligible", "no_star_training_eligible"]:
    weak[eligibility_column] = weak[eligibility_column].fillna(False).astype(bool)

for label_column in ["full_weak_label_binary", "no_star_weak_label_binary"]:
    weak[label_column] = pd.to_numeric(weak[label_column], errors="coerce")
    nonmissing = weak[label_column].dropna().unique()
    if not set(nonmissing).issubset({0, 1, 0.0, 1.0}):
        raise AssertionError(f"Unexpected values in {label_column}: {sorted(nonmissing.tolist())}")

weak_label_accounting_rows = []
for model_key, eligibility_column, label_column in [
    ("full", "full_training_eligible", "full_weak_label_binary"),
    ("no_star", "no_star_training_eligible", "no_star_weak_label_binary"),
]:
    eligible = weak[eligibility_column]
    labels = weak[label_column]
    if labels.loc[eligible].isna().any():
        raise AssertionError(f"Eligible {model_key} weak-label rows contain missing labels.")
    if labels.loc[~eligible].notna().any():
        # A noneligible row may retain an intermediate numeric value in some exports; only the frozen
        # eligibility flag controls training. Record but do not silently use it.
        noneligible_labeled = int(labels.loc[~eligible].notna().sum())
    else:
        noneligible_labeled = 0

    stable = int((labels == 1).sum())
    unstable = int((labels == 0).sum())
    unlabeled = int(labels.isna().sum())
    eligible_count = int(eligible.sum())
    expected = EXPECTED_WEAK_LABEL_ACCOUNTING[model_key]

    if (stable, unstable, unlabeled, eligible_count) != (
        expected["stable"],
        expected["unstable"],
        expected["unlabeled"],
        expected["eligible"],
    ):
        raise AssertionError(
            f"{model_key} weak-label accounting mismatch: "
            f"observed={(stable, unstable, unlabeled, eligible_count)}, "
            f"expected={(expected['stable'], expected['unstable'], expected['unlabeled'], expected['eligible'])}"
        )

    weak_label_accounting_rows.append(
        {
            "model_pathway": model_key,
            "stable_labels": stable,
            "unstable_labels": unstable,
            "unlabeled_rows": unlabeled,
            "training_eligible_rows": eligible_count,
            "noneligible_rows_with_numeric_label": noneligible_labeled,
        }
    )

weak_label_accounting = pd.DataFrame(weak_label_accounting_rows)

frozen_full_artifact = joblib.load(FULL_MODEL_PATH)
frozen_no_star_artifact = joblib.load(NO_STAR_MODEL_PATH)

frozen_full_pipeline, full_pipeline_location = extract_sklearn_pipeline(
    frozen_full_artifact, "full"
)
frozen_no_star_pipeline, no_star_pipeline_location = extract_sklearn_pipeline(
    frozen_no_star_artifact, "no-star"
)

print(
    "Frozen Stage 4C artifact unwrapping  : "
    f"full={type(frozen_full_artifact).__name__}{full_pipeline_location}; "
    f"no-star={type(frozen_no_star_artifact).__name__}{no_star_pipeline_location}"
)

for pipeline_name, pipeline, expected_features in [
    ("full", frozen_full_pipeline, FULL_FEATURES),
    ("no_star", frozen_no_star_pipeline, NO_STAR_FEATURES),
]:
    find_pipeline_component(pipeline, SimpleImputer)
    find_pipeline_component(pipeline, StandardScaler)
    classifier = find_pipeline_component(pipeline, LogisticRegression)
    if getattr(pipeline, "n_features_in_", len(expected_features)) != len(expected_features):
        raise AssertionError(
            f"Frozen {pipeline_name} pipeline expects {getattr(pipeline, 'n_features_in_', None)} "
            f"features; expected {len(expected_features)}."
        )
    if set(np.asarray(classifier.classes_).tolist()) != {0, 1}:
        raise AssertionError(f"Unexpected classes in frozen {pipeline_name} classifier.")

full_classifier = find_pipeline_component(frozen_full_pipeline, LogisticRegression)
no_star_classifier = find_pipeline_component(frozen_no_star_pipeline, LogisticRegression)

frozen_model_settings = {
    "full_solver": full_classifier.solver,
    "full_penalty": full_classifier.penalty,
    "full_C": float(full_classifier.C),
    "full_max_iter": int(full_classifier.max_iter),
    "full_tol": float(full_classifier.tol),
    "full_class_weight": full_classifier.class_weight,
    "full_random_state": full_classifier.random_state,
    "no_star_solver": no_star_classifier.solver,
    "no_star_penalty": no_star_classifier.penalty,
    "no_star_C": float(no_star_classifier.C),
    "no_star_max_iter": int(no_star_classifier.max_iter),
    "no_star_tol": float(no_star_classifier.tol),
    "no_star_class_weight": no_star_classifier.class_weight,
    "no_star_random_state": no_star_classifier.random_state,
}


# --------------------------------------------------------------------------------------------------
# 5. FIT ALL SIX LOGO MODELS USING T0 WEAK LABELS ONLY
#    IMPORTANT: STAGE 6B OUTCOME HAS NOT BEEN OPENED YET.
# --------------------------------------------------------------------------------------------------

training_accounting_rows = []
prediction_frames = []
fitted_models = {}
fit_start = time.time()

for held_out_gene in PRIMARY_GENES:
    training_genes = [gene for gene in PRIMARY_GENES if gene != held_out_gene]
    training_gene_text = "+".join(training_genes)

    split_predictions = weak.loc[
        weak[GENE_COLUMN].eq(held_out_gene),
        [KEY_COLUMN, ROW_ORDER_COLUMN, GENE_COLUMN] + sorted(set(FULL_FEATURES + NO_STAR_FEATURES)),
    ].copy()

    if split_predictions.empty:
        raise AssertionError(f"No Stage 4B rows found for held-out gene {held_out_gene}.")

    for pathway, template_pipeline, features, eligibility_column, label_column, output_column in [
        (
            "full_ges",
            frozen_full_pipeline,
            FULL_FEATURES,
            "full_training_eligible",
            "full_weak_label_binary",
            "logo_full_ges_instability_risk",
        ),
        (
            "no_star_ges",
            frozen_no_star_pipeline,
            NO_STAR_FEATURES,
            "no_star_training_eligible",
            "no_star_weak_label_binary",
            "logo_no_star_ges_instability_risk",
        ),
    ]:
        train_mask = weak[GENE_COLUMN].isin(training_genes) & weak[eligibility_column]
        train = weak.loc[train_mask].copy()
        y_train = train[label_column].astype(int).to_numpy()

        if len(np.unique(y_train)) != 2:
            raise AssertionError(
                f"{held_out_gene}/{pathway} training labels do not contain both classes."
            )

        stable_labels = int((y_train == 1).sum())
        unstable_labels = int((y_train == 0).sum())

        model = clone(template_pipeline)
        model.fit(train[features], y_train)

        fitted_classifier = find_pipeline_component(model, LogisticRegression)
        n_iter = int(np.max(np.asarray(fitted_classifier.n_iter_)))
        converged = n_iter < int(fitted_classifier.max_iter)
        if not converged:
            raise AssertionError(
                f"{held_out_gene}/{pathway} reached max_iter={fitted_classifier.max_iter}."
            )

        p_stable = stable_class_probability(model, split_predictions[features])
        risk = 1.0 - p_stable
        if not np.isfinite(risk).all() or ((risk < 0.0) | (risk > 1.0)).any():
            raise AssertionError(f"Invalid LOGO risk generated for {held_out_gene}/{pathway}.")

        split_predictions[output_column] = risk
        fitted_models[(held_out_gene, pathway)] = model

        training_accounting_rows.append(
            {
                "held_out_gene": held_out_gene,
                "training_genes": training_gene_text,
                "model_pathway": pathway,
                "training_rows": int(len(train)),
                "stable_weak_labels": stable_labels,
                "unstable_weak_labels": unstable_labels,
                "training_prevalence_stable": stable_labels / len(train),
                "features": ";".join(features),
                "solver": fitted_classifier.solver,
                "penalty": fitted_classifier.penalty,
                "C": float(fitted_classifier.C),
                "max_iter": int(fitted_classifier.max_iter),
                "iterations_used": n_iter,
                "converged": converged,
                "t1_outcome_loaded_during_fit": False,
            }
        )

        del train, y_train
        gc.collect()

    prediction_frames.append(
        split_predictions[
            [
                KEY_COLUMN,
                ROW_ORDER_COLUMN,
                GENE_COLUMN,
                "logo_full_ges_instability_risk",
                "logo_no_star_ges_instability_risk",
            ]
        ].copy()
    )

logo_predictions_all_t0 = pd.concat(prediction_frames, ignore_index=True)
training_accounting = pd.DataFrame(training_accounting_rows)

if logo_predictions_all_t0[KEY_COLUMN].duplicated().any():
    raise AssertionError("LOGO prediction table contains duplicate RCV keys.")
if sorted(logo_predictions_all_t0[GENE_COLUMN].unique().tolist()) != PRIMARY_GENES:
    raise AssertionError("LOGO prediction table does not contain exactly the three primary genes.")

fit_elapsed = time.time() - fit_start


# --------------------------------------------------------------------------------------------------
# 6. ONLY NOW VERIFY AND LOAD THE FROZEN STAGE 6B TEMPORAL OUTCOME
# --------------------------------------------------------------------------------------------------

observed_hashes["stage6b_evaluable"] = verify_hash(
    EVALUABLE_PARQUET,
    EXPECTED_HASHES["stage6b_evaluable"],
    "Stage 6B primary-evaluable cohort",
)
if not EVALUABLE_SIDECAR.exists():
    raise FileNotFoundError(f"Missing Stage 6B SHA-256 sidecar:\n{EVALUABLE_SIDECAR}")
if read_sidecar_hash(EVALUABLE_SIDECAR) != observed_hashes["stage6b_evaluable"]:
    raise AssertionError("Stage 6B evaluable-cohort SHA-256 sidecar does not match the file.")

stage6b_parquet = pq.ParquetFile(EVALUABLE_PARQUET)
if stage6b_parquet.metadata.num_rows != EXPECTED_STAGE6B_ROWS:
    raise AssertionError("Unexpected Stage 6B row count.")
if stage6b_parquet.metadata.num_columns != EXPECTED_STAGE6B_COLUMNS:
    raise AssertionError("Unexpected Stage 6B column count.")

stage6b_columns = stage6b_parquet.schema_arrow.names
required_stage6b_columns = [
    KEY_COLUMN,
    ROW_ORDER_COLUMN,
    GENE_COLUMN,
    OUTCOME_COLUMN,
    "review_stars_instability_risk",
    "combined_metadata_instability_risk",
]
missing_stage6b = [column for column in required_stage6b_columns if column not in stage6b_columns]
if missing_stage6b:
    raise KeyError("Missing required Stage 6B columns:\n" + "\n".join(missing_stage6b))

outcome = pd.read_parquet(EVALUABLE_PARQUET, columns=required_stage6b_columns).copy()
outcome[KEY_COLUMN] = outcome[KEY_COLUMN].astype(str).str.strip().str.upper()
outcome[ROW_ORDER_COLUMN] = pd.to_numeric(outcome[ROW_ORDER_COLUMN], errors="raise").astype(np.int64)
outcome[GENE_COLUMN] = outcome[GENE_COLUMN].map(normalize_gene)
outcome[OUTCOME_COLUMN] = pd.to_numeric(outcome[OUTCOME_COLUMN], errors="raise").astype(int)

if len(outcome) != EXPECTED_STAGE6B_ROWS:
    raise AssertionError("Loaded Stage 6B row count is incorrect.")
if outcome[KEY_COLUMN].nunique(dropna=False) != EXPECTED_STAGE6B_ROWS:
    raise AssertionError("Stage 6B RCV keys are not unique.")
if not set(outcome[OUTCOME_COLUMN].unique()).issubset({0, 1}):
    raise AssertionError("Stage 6B primary outcome is not binary.")

observed_events = int(outcome[OUTCOME_COLUMN].sum())
observed_negatives = int((outcome[OUTCOME_COLUMN] == 0).sum())
if (observed_events, observed_negatives) != (EXPECTED_EVENTS, EXPECTED_NEGATIVES):
    raise AssertionError(
        f"Stage 6B outcome accounting mismatch: {observed_events:,}/{observed_negatives:,}."
    )

for column in ["review_stars_instability_risk", "combined_metadata_instability_risk"]:
    outcome[column] = pd.to_numeric(outcome[column], errors="raise").astype(float)
    values = outcome[column].to_numpy()
    if not np.isfinite(values).all() or ((values < 0.0) | (values > 1.0)).any():
        raise AssertionError(f"Invalid frozen comparator values in {column}.")

primary_outcome = outcome.loc[outcome[GENE_COLUMN].isin(PRIMARY_GENES)].copy()

analysis = primary_outcome.merge(
    logo_predictions_all_t0,
    on=[KEY_COLUMN, ROW_ORDER_COLUMN, GENE_COLUMN],
    how="left",
    validate="one_to_one",
    indicator=True,
)

if not analysis["_merge"].eq("both").all():
    missing = int((analysis["_merge"] != "both").sum())
    raise AssertionError(f"{missing:,} primary-gene outcome rows lack LOGO predictions.")
analysis = analysis.drop(columns=["_merge"])

for column in [
    "logo_full_ges_instability_risk",
    "logo_no_star_ges_instability_risk",
]:
    values = pd.to_numeric(analysis[column], errors="raise").to_numpy(dtype=float)
    if not np.isfinite(values).all() or ((values < 0.0) | (values > 1.0)).any():
        raise AssertionError(f"Invalid merged LOGO score in {column}.")

held_out_accounting_rows = []
for gene in PRIMARY_GENES:
    gene_df = analysis.loc[analysis[GENE_COLUMN].eq(gene)]
    rows = int(len(gene_df))
    events = int(gene_df[OUTCOME_COLUMN].sum())
    negatives = rows - events
    expected = EXPECTED_HELD_OUT_ACCOUNTING[gene]
    if (rows, events, negatives) != (
        expected["rows"],
        expected["events"],
        expected["negatives"],
    ):
        raise AssertionError(
            f"Held-out {gene} accounting mismatch: {(rows, events, negatives)}."
        )
    held_out_accounting_rows.append(
        {
            "held_out_gene": gene,
            "rows": rows,
            "events": events,
            "negatives": negatives,
            "event_prevalence": events / rows,
        }
    )

held_out_accounting = pd.DataFrame(held_out_accounting_rows)


# --------------------------------------------------------------------------------------------------
# 7. HELD-OUT-GENE POINT ESTIMATES AND 2,000-REPLICATE PAIRED BOOTSTRAP
# --------------------------------------------------------------------------------------------------

rng = np.random.default_rng(RANDOM_SEED)
model_interval_rows = []
paired_difference_rows = []
score_summary_rows = []
bootstrap_start = time.time()

for held_out_gene in PRIMARY_GENES:
    gene_start = time.time()
    test = analysis.loc[analysis[GENE_COLUMN].eq(held_out_gene)].reset_index(drop=True)
    y = test[OUTCOME_COLUMN].to_numpy(dtype=np.int8)
    n_rows = int(len(test))
    n_events = int(y.sum())
    n_negatives = n_rows - n_events
    prevalence = n_events / n_rows

    if len(np.unique(y)) != 2:
        raise AssertionError(f"Held-out gene {held_out_gene} does not contain both outcome classes.")

    scores = {
        key: test[spec["score_column"]].to_numpy(dtype=np.float64)
        for key, spec in MODEL_SPECS.items()
    }
    caches = {key: construct_score_group_cache(value, y) for key, value in scores.items()}

    original_counts = np.ones((1, n_rows), dtype=np.int16)
    original_positive_total = np.array([n_events], dtype=np.float64)
    point_metrics = {}

    print(
        f"\nPreparing held-out {held_out_gene}: {n_rows:,} rows, {n_events:,} events, "
        f"{n_negatives:,} negatives"
    )

    for model_key, spec in MODEL_SPECS.items():
        score = scores[model_key]
        sklearn_auprc = float(average_precision_score(y, score))
        sklearn_auroc = float(roc_auc_score(y, score))
        fast_auprc, fast_auroc = calculate_grouped_weighted_metrics(
            caches[model_key],
            original_counts,
            original_positive_total,
        )
        if not np.isclose(fast_auprc[0], sklearn_auprc, rtol=1e-11, atol=1e-12):
            raise AssertionError(f"Fast AUPRC validation failed for {held_out_gene}/{model_key}.")
        if not np.isclose(fast_auroc[0], sklearn_auroc, rtol=1e-11, atol=1e-12):
            raise AssertionError(f"Fast AUROC validation failed for {held_out_gene}/{model_key}.")

        point_metrics[model_key] = {"auprc": sklearn_auprc, "auroc": sklearn_auroc}
        score_summary_rows.append(
            {
                "held_out_gene": held_out_gene,
                "model_key": model_key,
                "model": spec["display_name"],
                "unique_score_values": int(len(np.unique(score))),
                "score_min": float(np.min(score)),
                "score_max": float(np.max(score)),
                "score_mean": float(np.mean(score)),
                "mean_score_events": float(np.mean(score[y == 1])),
                "mean_score_negatives": float(np.mean(score[y == 0])),
                "point_auprc": sklearn_auprc,
                "point_auroc": sklearn_auroc,
                "fast_metric_validation": "PASS",
            }
        )

    print("  Exact tie-aware metric validation against scikit-learn: PASS")

    bootstrap_metrics = {
        model_key: {
            "auprc": np.full(N_BOOTSTRAP, np.nan, dtype=np.float64),
            "auroc": np.full(N_BOOTSTRAP, np.nan, dtype=np.float64),
        }
        for model_key in MODEL_SPECS
    }
    bootstrap_prevalence = np.full(N_BOOTSTRAP, np.nan, dtype=np.float64)

    probabilities = np.full(n_rows, 1.0 / n_rows, dtype=np.float64)
    probabilities[-1] = 1.0 - probabilities[:-1].sum()

    for batch_start in range(0, N_BOOTSTRAP, BOOTSTRAP_BATCH_SIZE):
        batch_end = min(batch_start + BOOTSTRAP_BATCH_SIZE, N_BOOTSTRAP)
        batch_size = batch_end - batch_start
        counts = rng.multinomial(n_rows, probabilities, size=batch_size)
        if not np.all(counts.sum(axis=1) == n_rows):
            raise AssertionError(f"Bootstrap sample-size preservation failed for {held_out_gene}.")

        positive_totals = (counts @ y).astype(np.float64)
        valid = (positive_totals > 0.0) & (positive_totals < n_rows)
        bootstrap_prevalence[batch_start:batch_end] = np.where(
            valid,
            positive_totals / n_rows,
            np.nan,
        )

        for model_key in MODEL_SPECS:
            auprc_values, auroc_values = calculate_grouped_weighted_metrics(
                caches[model_key],
                counts,
                positive_totals,
            )
            bootstrap_metrics[model_key]["auprc"][batch_start:batch_end] = auprc_values
            bootstrap_metrics[model_key]["auroc"][batch_start:batch_end] = auroc_values

        if batch_end % 250 == 0 or batch_end == N_BOOTSTRAP:
            valid_so_far = int(
                np.isfinite(bootstrap_metrics["logo_full_ges"]["auprc"][:batch_end]).sum()
            )
            print(
                f"  Completed {batch_end:,}/{N_BOOTSTRAP:,} replicates | "
                f"valid {valid_so_far:,} | elapsed {time.time() - gene_start:.1f}s"
            )

        del counts
        gc.collect()

    valid_mask = np.isfinite(bootstrap_metrics["logo_full_ges"]["auprc"])
    valid_replicates = int(valid_mask.sum())
    invalid_replicates = N_BOOTSTRAP - valid_replicates
    if valid_replicates < MINIMUM_VALID_REPLICATES:
        raise AssertionError(
            f"Held-out {held_out_gene} produced only {valid_replicates:,} valid replicates."
        )

    for model_key in MODEL_SPECS:
        for metric in ["auprc", "auroc"]:
            if not np.array_equal(
                np.isfinite(bootstrap_metrics[model_key][metric]),
                valid_mask,
            ):
                raise AssertionError(
                    f"Paired-validity mismatch for {held_out_gene}/{model_key}/{metric}."
                )

    for model_key, spec in MODEL_SPECS.items():
        auprc_values = bootstrap_metrics[model_key]["auprc"]
        auroc_values = bootstrap_metrics[model_key]["auroc"]
        auprc_low, auprc_high = percentile_interval(auprc_values)
        auroc_low, auroc_high = percentile_interval(auroc_values)
        auprc_null_low, auprc_null_high = percentile_interval(
            auprc_values - bootstrap_prevalence
        )
        auroc_null_low, auroc_null_high = percentile_interval(auroc_values - 0.50)

        model_interval_rows.append(
            {
                "held_out_gene": held_out_gene,
                "training_genes": "+".join(
                    [gene for gene in PRIMARY_GENES if gene != held_out_gene]
                ),
                "model_key": model_key,
                "model": spec["display_name"],
                "rows": n_rows,
                "events": n_events,
                "negatives": n_negatives,
                "held_out_prevalence": prevalence,
                "point_auprc": point_metrics[model_key]["auprc"],
                "auprc_ci_lower": auprc_low,
                "auprc_ci_upper": auprc_high,
                "point_auprc_minus_prevalence": point_metrics[model_key]["auprc"] - prevalence,
                "auprc_minus_prevalence_ci_lower": auprc_null_low,
                "auprc_minus_prevalence_ci_upper": auprc_null_high,
                "auprc_null_status": interval_status(
                    auprc_null_low,
                    auprc_null_high,
                    "supported_above_held_out_prevalence",
                    "supported_below_held_out_prevalence",
                ),
                "point_auroc": point_metrics[model_key]["auroc"],
                "auroc_ci_lower": auroc_low,
                "auroc_ci_upper": auroc_high,
                "point_auroc_minus_0_50": point_metrics[model_key]["auroc"] - 0.50,
                "auroc_minus_0_50_ci_lower": auroc_null_low,
                "auroc_minus_0_50_ci_upper": auroc_null_high,
                "auroc_null_status": interval_status(
                    auroc_null_low,
                    auroc_null_high,
                    "supported_above_0_50",
                    "supported_below_0_50",
                ),
                "attempted_bootstrap_replicates": N_BOOTSTRAP,
                "valid_bootstrap_replicates": valid_replicates,
                "invalid_one_class_replicates": invalid_replicates,
            }
        )

    for comparator_key in PAIRED_COMPARATORS:
        comparator_name = MODEL_SPECS[comparator_key]["display_name"]
        for metric in ["auprc", "auroc"]:
            differences = (
                bootstrap_metrics["logo_full_ges"][metric]
                - bootstrap_metrics[comparator_key][metric]
            )
            finite = differences[np.isfinite(differences)]
            difference_low, difference_high = percentile_interval(finite)
            point_difference = (
                point_metrics["logo_full_ges"][metric]
                - point_metrics[comparator_key][metric]
            )

            paired_difference_rows.append(
                {
                    "metric": metric.upper(),
                    "held_out_gene": held_out_gene,
                    "training_genes": "+".join(
                        [gene for gene in PRIMARY_GENES if gene != held_out_gene]
                    ),
                    "comparison": f"LOGO Full GES minus {comparator_name}",
                    "comparator_key": comparator_key,
                    "rows": n_rows,
                    "events": n_events,
                    "negatives": n_negatives,
                    "point_difference": point_difference,
                    "difference_ci_lower": difference_low,
                    "difference_ci_upper": difference_high,
                    "paired_interval_status": interval_status(
                        difference_low,
                        difference_high,
                        "logo_full_ges_supported_higher",
                        "logo_full_ges_supported_lower",
                    ),
                    "bootstrap_probability_logo_full_greater": float(
                        np.mean(finite > 0.0)
                    ),
                    "bootstrap_probability_equal": float(np.mean(finite == 0.0)),
                    "bootstrap_sign_p_value": bootstrap_sign_pvalue(finite),
                    "attempted_bootstrap_replicates": N_BOOTSTRAP,
                    "valid_bootstrap_replicates": valid_replicates,
                    "invalid_one_class_replicates": invalid_replicates,
                }
            )

    print(
        f"  Held-out {held_out_gene} complete: {valid_replicates:,} valid, "
        f"{invalid_replicates:,} invalid one-class replicates"
    )

    del test, y, scores, caches, bootstrap_metrics, bootstrap_prevalence
    gc.collect()

bootstrap_elapsed = time.time() - bootstrap_start


# --------------------------------------------------------------------------------------------------
# 8. RESULT TABLES AND HOLM CORRECTION
# --------------------------------------------------------------------------------------------------

logo_model_intervals = pd.DataFrame(model_interval_rows)
logo_paired_differences = pd.DataFrame(paired_difference_rows)
logo_score_summary = pd.DataFrame(score_summary_rows)

expected_model_rows = len(PRIMARY_GENES) * len(MODEL_SPECS)
expected_paired_rows = len(PRIMARY_GENES) * len(PAIRED_COMPARATORS) * 2
if len(logo_model_intervals) != expected_model_rows:
    raise AssertionError(
        f"LOGO model interval table has {len(logo_model_intervals)} rows; "
        f"expected {expected_model_rows}."
    )
if len(logo_paired_differences) != expected_paired_rows:
    raise AssertionError(
        f"LOGO paired table has {len(logo_paired_differences)} rows; "
        f"expected {expected_paired_rows}."
    )

logo_paired_differences["holm_adjusted_bootstrap_sign_p"] = np.nan
for metric in ["AUPRC", "AUROC"]:
    mask = logo_paired_differences["metric"].eq(metric)
    p_values = logo_paired_differences.loc[mask, "bootstrap_sign_p_value"].to_numpy(float)
    if len(p_values) != 9:
        raise AssertionError(f"{metric} LOGO multiplicity family has {len(p_values)} tests; expected 9.")
    logo_paired_differences.loc[mask, "holm_adjusted_bootstrap_sign_p"] = holm_adjust(p_values)

logo_paired_differences["holm_supported_at_0_05"] = (
    logo_paired_differences["holm_adjusted_bootstrap_sign_p"] < 0.05
)
logo_paired_differences["multiplicity_family"] = (
    "three_primary_held_out_genes_x_three_comparators"
)

if logo_paired_differences["holm_adjusted_bootstrap_sign_p"].isna().any():
    raise AssertionError("At least one LOGO Holm-adjusted value is missing.")
if (logo_model_intervals["valid_bootstrap_replicates"] < MINIMUM_VALID_REPLICATES).any():
    raise AssertionError("At least one held-out-gene/model interval has too few valid replicates.")


# --------------------------------------------------------------------------------------------------
# 9. DISPLAY AND FINAL DECISION
# --------------------------------------------------------------------------------------------------

model_display_columns = [
    "held_out_gene",
    "training_genes",
    "model",
    "rows",
    "events",
    "negatives",
    "held_out_prevalence",
    "point_auprc",
    "auprc_ci_lower",
    "auprc_ci_upper",
    "point_auprc_minus_prevalence",
    "auprc_minus_prevalence_ci_lower",
    "auprc_minus_prevalence_ci_upper",
    "auprc_null_status",
    "point_auroc",
    "auroc_ci_lower",
    "auroc_ci_upper",
    "point_auroc_minus_0_50",
    "auroc_minus_0_50_ci_lower",
    "auroc_minus_0_50_ci_upper",
    "auroc_null_status",
    "valid_bootstrap_replicates",
    "invalid_one_class_replicates",
]

paired_display_columns = [
    "metric",
    "held_out_gene",
    "training_genes",
    "comparison",
    "rows",
    "events",
    "negatives",
    "point_difference",
    "difference_ci_lower",
    "difference_ci_upper",
    "paired_interval_status",
    "bootstrap_probability_logo_full_greater",
    "bootstrap_sign_p_value",
    "holm_adjusted_bootstrap_sign_p",
    "holm_supported_at_0_05",
    "multiplicity_family",
    "valid_bootstrap_replicates",
    "invalid_one_class_replicates",
]

separator = "=" * 150
print("\n" + separator)
print("STAGE 6C STEP 3H — CELL 6C-3H0 — LOCKED LEAVE-ONE-GENE-OUT VALIDATION")
print(separator)
print(f"Stage 4B weak-label SHA-256        : PASS ({observed_hashes['stage4b_weak_label_table']})")
print(f"Stage 4B manifest SHA-256          : PASS ({observed_hashes['stage4b_manifest']})")
print(f"Stage 4C full-model SHA-256        : PASS ({observed_hashes['stage4c_full_model']})")
print(f"Stage 4C no-star-model SHA-256     : PASS ({observed_hashes['stage4c_no_star_model']})")
print(f"Stage 4C manifest SHA-256          : PASS ({observed_hashes['stage4c_manifest']})")
print(f"Stage 6B evaluable SHA-256         : PASS ({observed_hashes['stage6b_evaluable']})")
print(f"Stage 4B dimensions                : PASS ({EXPECTED_STAGE4B_ROWS:,} × {EXPECTED_STAGE4B_COLUMNS})")
print(f"Stage 6B dimensions                : PASS ({EXPECTED_STAGE6B_ROWS:,} × {EXPECTED_STAGE6B_COLUMNS})")
print(f"Stage 6B events / negatives        : PASS ({observed_events:,} / {observed_negatives:,})")
print("Temporal outcome loaded during fit: No")
print("Held-out model tuning              : None")
print("EGFR pooled into primary LOGO      : No")
print(f"Bootstrap attempts per held-out gene: {N_BOOTSTRAP:,}")
print(f"Random seed                        : {RANDOM_SEED}")
print("Holm families                      : 9 AUPRC + 9 AUROC paired comparisons")
print(f"Model fitting elapsed              : {fit_elapsed:.1f}s")
print(f"Bootstrap elapsed                  : {bootstrap_elapsed:.1f}s")

print("\nFROZEN PIPELINE SETTINGS RECOVERED FROM CHECKSUM-VERIFIED JOBLIBS")
for key, value in frozen_model_settings.items():
    print(f"{key:34s}: {value}")

print("\nGLOBAL WEAK-LABEL ACCOUNTING")
print(weak_label_accounting.to_string(index=False))

print("\nLOGO TRAINING ACCOUNTING")
print(training_accounting.to_string(index=False))

print("\nHELD-OUT TEST ACCOUNTING")
print(held_out_accounting.to_string(index=False))

print("\nHELD-OUT SCORE AND POINT-ESTIMATE SUMMARY")
print(logo_score_summary.to_string(index=False))

print("\nHELD-OUT MODEL-SPECIFIC INTERVALS")
print(logo_model_intervals[model_display_columns].to_string(index=False))

print("\nPAIRED LOGO FULL-GES-MINUS-COMPARATOR INFERENCE")
print(logo_paired_differences[paired_display_columns].to_string(index=False))

print("\nSCIENTIFIC INTERPRETATION BOUNDARY")
print("-" * 150)
print(
    "This is the prespecified cross-gene generalization test. Each GES model was refitted only on T0 weak-label "
    "records from two primary genes and evaluated on the third gene using the locked T0-to-T1 temporal outcome."
)
print(
    "A positive held-out result requires attention to absolute discrimination, confidence intervals, and performance "
    "against both review stars and the strong combined-metadata baseline; statistical separation alone does not imply "
    "clinical utility or calibration."
)
print(
    "EGFR remains separate and exploratory. No downstream RAG benefit, patient outcome, treatment safety, or clinical "
    "deployment claim is evaluated in this cell."
)

print("\nCELL DECISION")
print("-" * 150)
print("PASS_STAGE6C_LEAVE_ONE_GENE_OUT_PAIRED_BOOTSTRAP_VALIDATION_COMPLETE")
print(
    "All three prespecified train-two/test-one primary-gene splits, 2,000 paired bootstrap attempts per split, "
    "and separate 9-test Holm corrections for AUPRC and AUROC are complete."
)
print(
    "No frozen score, weak label, outcome, feature definition, threshold, gene assignment, cohort membership, "
    "or scientific artifact was modified. No artifact was written."
)

Frozen Stage 4C artifact unwrapping  : full=dict['pipeline']; no-star=dict['pipeline']

Preparing held-out BRCA1: 21,594 rows, 2,023 events, 19,571 negatives
  Exact tie-aware metric validation against scikit-learn: PASS


KeyboardInterrupt: 

In [50]:
# ==================================================================================================
# STAGE 6C STEP 3H — CELL 6C-3H0
# LOCKED LEAVE-ONE-GENE-OUT (LOGO) TEMPORAL GENERALIZATION VALIDATION
#
# Primary cross-gene design:
#   1. Train on BRCA2 + MLH1; test on BRCA1.
#   2. Train on BRCA1 + MLH1; test on BRCA2.
#   3. Train on BRCA1 + BRCA2; test on MLH1.
#
# For each split, the full-GES and no-star-GES pipelines are cloned from the checksum-verified,
# frozen Stage 4C joblib artifacts and refitted using only Stage 4B T0 weak-label rows from the two
# training genes. Predictions are generated before the Stage 6B temporal outcome is loaded.
#
# The held-out-gene evaluation uses the frozen Stage 6B primary-evaluable outcome and compares:
#   - LOGO Full GES
#   - LOGO No-star GES
#   - Frozen review-stars comparator
#   - Frozen combined-metadata comparator
#
# AUPRC is primary. AUROC is secondary. The cell runs 2,000 paired test-set row-bootstrap
# replicates per held-out gene, reports 95% percentile intervals, and applies Holm correction
# separately across the 9 primary-gene AUPRC comparisons and the 9 AUROC comparisons.
#
# Scientific boundary:
#   - No T1 outcome is used to fit, tune, calibrate, threshold, or select either LOGO model.
#   - EGFR is not pooled into the primary LOGO experiment.
#   - No score, weak label, outcome, threshold, feature definition, gene assignment, or frozen
#     artifact is modified.
#   - This cell is read-only and writes no scientific artifact.
# ==================================================================================================
# Runtime revision v3: removes explicit cyclic-garbage-collection calls from the numerical bootstrap
# loop and prints progress after every 50 replicates. Scientific logic, hashes, models, outcomes,
# resampling seed, replicate count, metrics, and multiplicity correction are unchanged.
# ==================================================================================================

from pathlib import Path
import hashlib
import json
import re
import time

import joblib
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from scipy.sparse import csr_matrix
from sklearn.base import clone
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


# --------------------------------------------------------------------------------------------------
# 1. GOOGLE DRIVE AND FROZEN PATHS
# --------------------------------------------------------------------------------------------------

DRIVE_ROOT = Path("/content/drive/MyDrive")
if not DRIVE_ROOT.exists():
    from google.colab import drive
    drive.mount("/content/drive")

if not DRIVE_ROOT.exists():
    raise FileNotFoundError("Google Drive is not mounted at /content/drive/MyDrive.")

PROJECT_DIR = DRIVE_ROOT / "GES_RAG_Temporal_Study"

STAGE4_DATA_DIR = PROJECT_DIR / "data_processed" / "stage4_ges"
STAGE4_MODEL_DIR = PROJECT_DIR / "models" / "stage4_ges"
STAGE4_CONFIG_DIR = PROJECT_DIR / "configs" / "stage4_ges"
STAGE6_DIR = PROJECT_DIR / "data_processed" / "stage6_temporal_validation"

WEAK_LABEL_TABLE = (
    STAGE4_DATA_DIR / "stage4b_t0_feature_transforms_and_weak_labels_v1.parquet"
)
STAGE4B_MANIFEST = STAGE4_CONFIG_DIR / "stage4b_weak_label_freeze_manifest_v1.json"
FULL_MODEL_PATH = STAGE4_MODEL_DIR / "stage4c_full_ges_logistic_model_v1.joblib"
NO_STAR_MODEL_PATH = STAGE4_MODEL_DIR / "stage4c_no_star_ges_logistic_model_v1.joblib"
STAGE4C_MANIFEST = STAGE4_CONFIG_DIR / "stage4c_ges_model_freeze_manifest_v1.json"

EVALUABLE_PARQUET = STAGE6_DIR / "stage6b_locked_primary_evaluable_cohort_v1.parquet"
EVALUABLE_SIDECAR = Path(str(EVALUABLE_PARQUET) + ".sha256")

EXPECTED_HASHES = {
    "stage4b_weak_label_table": (
        "c2e9e4f96cd1f61d3962e88d28557dfa1221729b8f79f8fd4d2dafb34847bdb8"
    ),
    "stage4b_manifest": (
        "e766061442e6d4610f661f44a619e41b45dc6363b28f6176d3f9a71f8215c63f"
    ),
    "stage4c_full_model": (
        "0b4a87b16f768484cbdae168fc226cec5e978521172fb03510bfb1ea3e78fa30"
    ),
    "stage4c_no_star_model": (
        "6c3fe4fc7fe8fdde7b8f0f0d608c48e66a07945effb8c67c6b98d35e1955257c"
    ),
    "stage4c_manifest": (
        "c0d8008a4db80c67f5b1c568ddba3496b2411b29bce0db1eb20e63f26613d4ee"
    ),
    "stage6b_evaluable": (
        "c6ad50dfc376a746e1830d000dba975101d88763fd61b8d1983f6300edb6e038"
    ),
}

EXPECTED_STAGE4B_ROWS = 71_659
EXPECTED_STAGE4B_COLUMNS = 43
EXPECTED_STAGE6B_ROWS = 66_636
EXPECTED_STAGE6B_COLUMNS = 79
EXPECTED_EVENTS = 6_485
EXPECTED_NEGATIVES = 60_151

EXPECTED_WEAK_LABEL_ACCOUNTING = {
    "full": {
        "stable": 61_842,
        "unstable": 5_723,
        "unlabeled": 4_094,
        "eligible": 67_565,
    },
    "no_star": {
        "stable": 61_298,
        "unstable": 1_850,
        "unlabeled": 8_511,
        "eligible": 63_148,
    },
}

EXPECTED_HELD_OUT_ACCOUNTING = {
    "BRCA1": {"rows": 21_594, "events": 2_023, "negatives": 19_571},
    "BRCA2": {"rows": 34_152, "events": 3_960, "negatives": 30_192},
    "MLH1": {"rows": 8_701, "events": 425, "negatives": 8_276},
}

PRIMARY_GENES = ["BRCA1", "BRCA2", "MLH1"]
EXPLORATORY_GENE = "EGFR"

OUTCOME_COLUMN = "primary_future_instability"
GENE_COLUMN = "target_gene"
KEY_COLUMN = "rcv_accession"
ROW_ORDER_COLUMN = "t0_row_order"

FULL_FEATURES = [
    "recency_score",
    "recency_missing_flag",
    "submitter_diversity_score",
    "review_confidence",
    "aggregate_conflict_flag",
    "scv_group_entropy_normalized",
]

NO_STAR_FEATURES = [
    "recency_score",
    "recency_missing_flag",
    "submitter_diversity_score",
    "aggregate_conflict_flag",
    "scv_group_entropy_normalized",
]

MODEL_SPECS = {
    "logo_full_ges": {
        "display_name": "LOGO Full GES",
        "score_column": "logo_full_ges_instability_risk",
    },
    "logo_no_star_ges": {
        "display_name": "LOGO No-star GES",
        "score_column": "logo_no_star_ges_instability_risk",
    },
    "review_stars": {
        "display_name": "Review stars",
        "score_column": "review_stars_instability_risk",
    },
    "combined_metadata": {
        "display_name": "Combined metadata",
        "score_column": "combined_metadata_instability_risk",
    },
}

PAIRED_COMPARATORS = ["logo_no_star_ges", "review_stars", "combined_metadata"]

N_BOOTSTRAP = 2_000
RANDOM_SEED = 42
BOOTSTRAP_BATCH_SIZE = 50
BOOTSTRAP_PROGRESS_INTERVAL = 50
CI_QUANTILES = (0.025, 0.975)
MINIMUM_VALID_REPLICATES = 1_000


# --------------------------------------------------------------------------------------------------
# 2. GENERAL HELPERS
# --------------------------------------------------------------------------------------------------

def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def read_sidecar_hash(path: Path) -> str:
    text = path.read_text(encoding="utf-8").strip()
    matches = re.findall(r"\b[a-fA-F0-9]{64}\b", text)
    if not matches:
        raise ValueError(f"No SHA-256 value found in sidecar: {path}")
    return matches[0].lower()


def verify_hash(path: Path, expected: str, label: str) -> str:
    if not path.exists():
        raise FileNotFoundError(f"Missing frozen artifact for {label}:\n{path}")
    observed = sha256_file(path)
    if observed != expected:
        raise AssertionError(
            f"SHA-256 mismatch for {label}.\nExpected: {expected}\nObserved: {observed}"
        )
    return observed


def normalize_gene(value) -> str:
    text = str(value).strip().upper()
    if text in {"BRCA1", "BRCA2", "MLH1", "EGFR"}:
        return text
    return ""


def gene_from_json(value) -> str:
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return ""

    parsed = value
    if isinstance(value, str):
        text = value.strip()
        if not text:
            return ""
        try:
            parsed = json.loads(text)
        except json.JSONDecodeError:
            parsed = [part.strip() for part in re.split(r"[,;|]", text) if part.strip()]

    if isinstance(parsed, dict):
        candidates = list(parsed.keys()) + list(parsed.values())
    elif isinstance(parsed, (list, tuple, set, np.ndarray, pd.Series)):
        candidates = list(parsed)
    else:
        candidates = [parsed]

    genes = []
    for candidate in candidates:
        if isinstance(candidate, (list, tuple, set, dict)):
            nested = gene_from_json(candidate)
            if nested:
                genes.append(nested)
            continue
        gene = normalize_gene(candidate)
        if gene:
            genes.append(gene)

    genes = sorted(set(genes))
    if len(genes) != 1:
        return ""
    return genes[0]


def extract_sklearn_pipeline(artifact, artifact_name: str):
    """Extract exactly one sklearn Pipeline from a checksum-verified joblib bundle.

    Stage 4C joblib files may serialize a metadata bundle rather than placing the Pipeline at the
    top level. This helper is read-only: it unwraps the existing fitted Pipeline without altering
    the frozen artifact or reconstructing model settings from assumptions.
    """
    if isinstance(artifact, Pipeline):
        return artifact, "<top-level>"

    preferred_keys = (
        "pipeline",
        "model_pipeline",
        "fitted_pipeline",
        "sklearn_pipeline",
        "model",
        "estimator",
        "classifier",
    )

    # First honor common explicit bundle keys in deterministic order.
    if isinstance(artifact, dict):
        for key in preferred_keys:
            if key in artifact and isinstance(artifact[key], Pipeline):
                return artifact[key], f"[{key!r}]"

    # Then recursively inspect containers and selected object attributes.
    matches = []
    visited = set()

    def walk(obj, location: str, depth: int = 0):
        if depth > 8:
            return
        obj_id = id(obj)
        if obj_id in visited:
            return
        visited.add(obj_id)

        if isinstance(obj, Pipeline):
            matches.append((location, obj))
            return

        if isinstance(obj, dict):
            # Preferred keys first, followed by all remaining keys in stable text order.
            ordered_keys = [key for key in preferred_keys if key in obj]
            ordered_keys += sorted(
                [key for key in obj.keys() if key not in ordered_keys], key=lambda value: str(value)
            )
            for key in ordered_keys:
                walk(obj[key], f"{location}[{key!r}]", depth + 1)
            return

        if isinstance(obj, (list, tuple)):
            for index, value in enumerate(obj):
                walk(value, f"{location}[{index}]", depth + 1)
            return

        # Some joblib bundles are lightweight custom objects with model/pipeline attributes.
        for attr in preferred_keys:
            if hasattr(obj, attr):
                try:
                    value = getattr(obj, attr)
                except Exception:
                    continue
                walk(value, f"{location}.{attr}", depth + 1)

    walk(artifact, "<top-level>")

    # Deduplicate aliases that point to the same Pipeline object.
    unique = {}
    for location, pipeline in matches:
        unique.setdefault(id(pipeline), (location, pipeline))
    unique_matches = list(unique.values())

    if len(unique_matches) == 1:
        return unique_matches[0][1], unique_matches[0][0]

    artifact_type = f"{type(artifact).__module__}.{type(artifact).__name__}"
    if isinstance(artifact, dict):
        structure = f"top-level keys={sorted(map(str, artifact.keys()))}"
    elif isinstance(artifact, (list, tuple)):
        structure = f"top-level length={len(artifact)}"
    else:
        structure = f"top-level attributes checked={list(preferred_keys)}"

    if not unique_matches:
        raise TypeError(
            f"Frozen {artifact_name} artifact is {artifact_type}, not a top-level sklearn Pipeline, "
            f"and no nested Pipeline was found ({structure})."
        )

    locations = [location for location, _ in unique_matches]
    raise TypeError(
        f"Frozen {artifact_name} artifact contains multiple distinct sklearn Pipelines at "
        f"{locations}; refusing to choose one ambiguously."
    )


def find_pipeline_component(pipeline: Pipeline, expected_type):
    matches = [step for _, step in pipeline.steps if isinstance(step, expected_type)]
    if len(matches) != 1:
        raise AssertionError(
            f"Expected exactly one {expected_type.__name__} in pipeline; found {len(matches)}."
        )
    return matches[0]


def stable_class_probability(model: Pipeline, features: pd.DataFrame) -> np.ndarray:
    probabilities = model.predict_proba(features)
    classifier = find_pipeline_component(model, LogisticRegression)
    classes = np.asarray(classifier.classes_)
    stable_positions = np.flatnonzero(classes == 1)
    if len(stable_positions) != 1:
        raise AssertionError(f"Model classes do not contain exactly one stable class 1: {classes}")
    return probabilities[:, int(stable_positions[0])]


def percentile_interval(values: np.ndarray) -> tuple[float, float]:
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if len(values) == 0:
        return np.nan, np.nan
    lower, upper = np.quantile(values, CI_QUANTILES)
    return float(lower), float(upper)


def bootstrap_sign_pvalue(differences: np.ndarray) -> float:
    differences = np.asarray(differences, dtype=float)
    differences = differences[np.isfinite(differences)]
    if len(differences) == 0:
        return np.nan

    n = len(differences)
    lower_tail = (np.count_nonzero(differences <= 0.0) + 1) / (n + 1)
    upper_tail = (np.count_nonzero(differences >= 0.0) + 1) / (n + 1)
    return float(min(1.0, 2.0 * min(lower_tail, upper_tail)))


def holm_adjust(p_values: np.ndarray) -> np.ndarray:
    p_values = np.asarray(p_values, dtype=float)
    adjusted = np.full(len(p_values), np.nan, dtype=float)
    valid_positions = np.where(np.isfinite(p_values))[0]

    if len(valid_positions) == 0:
        return adjusted

    valid_p = p_values[valid_positions]
    order = np.argsort(valid_p)
    m = len(valid_p)
    running_max = 0.0

    for rank, position_within_valid in enumerate(order):
        original_position = valid_positions[position_within_valid]
        raw_adjusted = (m - rank) * valid_p[position_within_valid]
        running_max = max(running_max, raw_adjusted)
        adjusted[original_position] = min(1.0, running_max)

    return adjusted


def interval_status(lower: float, upper: float, positive_label: str, negative_label: str) -> str:
    if not np.isfinite(lower) or not np.isfinite(upper):
        return "not_estimable"
    if lower > 0.0:
        return positive_label
    if upper < 0.0:
        return negative_label
    return "interval_includes_null"


# --------------------------------------------------------------------------------------------------
# 3. EXACT TIE-AWARE METRIC CACHE USED FOR PAIRED BOOTSTRAP
# --------------------------------------------------------------------------------------------------

def construct_score_group_cache(scores: np.ndarray, outcomes: np.ndarray) -> dict:
    scores = np.asarray(scores, dtype=np.float64)
    outcomes = np.asarray(outcomes, dtype=np.int8)

    unique_scores, group_index = np.unique(scores, return_inverse=True)
    n_rows = len(scores)
    n_groups = len(unique_scores)
    row_positions = np.arange(n_rows)

    total_group_matrix = csr_matrix(
        (
            np.ones(n_rows, dtype=np.float64),
            (group_index, row_positions),
        ),
        shape=(n_groups, n_rows),
    )

    positive_positions = np.flatnonzero(outcomes == 1)
    positive_group_matrix = csr_matrix(
        (
            np.ones(len(positive_positions), dtype=np.float64),
            (group_index[positive_positions], positive_positions),
        ),
        shape=(n_groups, n_rows),
    )

    return {
        "unique_scores": unique_scores,
        "total_group_matrix": total_group_matrix,
        "positive_group_matrix": positive_group_matrix,
    }


def calculate_grouped_weighted_metrics(
    cache: dict,
    count_matrix: np.ndarray,
    positive_totals: np.ndarray,
) -> tuple[np.ndarray, np.ndarray]:
    count_matrix = np.asarray(count_matrix)
    positive_totals = np.asarray(positive_totals, dtype=np.float64)

    sample_totals = count_matrix.sum(axis=1).astype(np.float64)
    negative_totals = sample_totals - positive_totals

    group_total_counts = np.asarray(
        cache["total_group_matrix"] @ count_matrix.T,
        dtype=np.float64,
    )
    group_positive_counts = np.asarray(
        cache["positive_group_matrix"] @ count_matrix.T,
        dtype=np.float64,
    )
    group_negative_counts = group_total_counts - group_positive_counts

    valid = (positive_totals > 0.0) & (negative_totals > 0.0)

    cumulative_negatives_before = (
        np.cumsum(group_negative_counts, axis=0) - group_negative_counts
    )
    concordant_numerator = np.sum(
        group_positive_counts
        * (cumulative_negatives_before + 0.5 * group_negative_counts),
        axis=0,
    )
    auc_denominator = positive_totals * negative_totals
    auroc = np.full(len(positive_totals), np.nan, dtype=np.float64)
    np.divide(concordant_numerator, auc_denominator, out=auroc, where=valid)

    positive_desc = group_positive_counts[::-1, :]
    total_desc = group_total_counts[::-1, :]
    cumulative_positive = np.cumsum(positive_desc, axis=0)
    cumulative_total = np.cumsum(total_desc, axis=0)

    precision = np.zeros_like(cumulative_positive, dtype=np.float64)
    np.divide(
        cumulative_positive,
        cumulative_total,
        out=precision,
        where=cumulative_total > 0.0,
    )

    ap_numerator = np.sum(precision * positive_desc, axis=0)
    auprc = np.full(len(positive_totals), np.nan, dtype=np.float64)
    np.divide(ap_numerator, positive_totals, out=auprc, where=valid)

    return auprc, auroc


# --------------------------------------------------------------------------------------------------
# 4. FRESHLY VERIFY STAGE 4B AND STAGE 4C BEFORE ANY TEMPORAL OUTCOME IS LOADED
# --------------------------------------------------------------------------------------------------

observed_hashes = {}
observed_hashes["stage4b_weak_label_table"] = verify_hash(
    WEAK_LABEL_TABLE,
    EXPECTED_HASHES["stage4b_weak_label_table"],
    "Stage 4B weak-label table",
)
observed_hashes["stage4b_manifest"] = verify_hash(
    STAGE4B_MANIFEST,
    EXPECTED_HASHES["stage4b_manifest"],
    "Stage 4B manifest",
)
observed_hashes["stage4c_full_model"] = verify_hash(
    FULL_MODEL_PATH,
    EXPECTED_HASHES["stage4c_full_model"],
    "Stage 4C full GES model",
)
observed_hashes["stage4c_no_star_model"] = verify_hash(
    NO_STAR_MODEL_PATH,
    EXPECTED_HASHES["stage4c_no_star_model"],
    "Stage 4C no-star GES model",
)
observed_hashes["stage4c_manifest"] = verify_hash(
    STAGE4C_MANIFEST,
    EXPECTED_HASHES["stage4c_manifest"],
    "Stage 4C model-freeze manifest",
)

stage4b_parquet = pq.ParquetFile(WEAK_LABEL_TABLE)
if stage4b_parquet.metadata.num_rows != EXPECTED_STAGE4B_ROWS:
    raise AssertionError(
        f"Stage 4B row count is {stage4b_parquet.metadata.num_rows:,}; "
        f"expected {EXPECTED_STAGE4B_ROWS:,}."
    )
if stage4b_parquet.metadata.num_columns != EXPECTED_STAGE4B_COLUMNS:
    raise AssertionError(
        f"Stage 4B column count is {stage4b_parquet.metadata.num_columns}; "
        f"expected {EXPECTED_STAGE4B_COLUMNS}."
    )

stage4b_columns = stage4b_parquet.schema_arrow.names
required_stage4b_columns = [
    KEY_COLUMN,
    ROW_ORDER_COLUMN,
    "full_training_eligible",
    "full_weak_label_binary",
    "no_star_training_eligible",
    "no_star_weak_label_binary",
] + sorted(set(FULL_FEATURES + NO_STAR_FEATURES))

missing_stage4b = [column for column in required_stage4b_columns if column not in stage4b_columns]
if missing_stage4b:
    raise KeyError("Missing required Stage 4B columns:\n" + "\n".join(missing_stage4b))

if GENE_COLUMN in stage4b_columns:
    gene_source_column = GENE_COLUMN
elif "target_genes_json" in stage4b_columns:
    gene_source_column = "target_genes_json"
else:
    raise KeyError(
        "Stage 4B has neither target_gene nor target_genes_json; gene assignment cannot be reconstructed."
    )

load_stage4b_columns = list(dict.fromkeys(required_stage4b_columns + [gene_source_column]))
weak = pd.read_parquet(WEAK_LABEL_TABLE, columns=load_stage4b_columns).copy()

if len(weak) != EXPECTED_STAGE4B_ROWS:
    raise AssertionError("Loaded Stage 4B dataframe row count is incorrect.")
if weak[KEY_COLUMN].isna().any() or weak[KEY_COLUMN].astype(str).str.strip().eq("").any():
    raise AssertionError("Missing or blank Stage 4B RCV key detected.")
if weak[KEY_COLUMN].nunique(dropna=False) != EXPECTED_STAGE4B_ROWS:
    raise AssertionError("Stage 4B RCV keys are not unique.")

weak[KEY_COLUMN] = weak[KEY_COLUMN].astype(str).str.strip().str.upper()
weak[ROW_ORDER_COLUMN] = pd.to_numeric(weak[ROW_ORDER_COLUMN], errors="raise").astype(np.int64)
if weak[ROW_ORDER_COLUMN].nunique() != EXPECTED_STAGE4B_ROWS:
    raise AssertionError("Stage 4B t0_row_order is not unique.")
if not np.all(np.diff(weak[ROW_ORDER_COLUMN].to_numpy()) > 0):
    raise AssertionError("Stage 4B rows are not in strictly increasing T0 order.")

if gene_source_column == GENE_COLUMN:
    weak[GENE_COLUMN] = weak[gene_source_column].map(normalize_gene)
else:
    weak[GENE_COLUMN] = weak[gene_source_column].map(gene_from_json)

if weak[GENE_COLUMN].eq("").any():
    bad = int(weak[GENE_COLUMN].eq("").sum())
    raise AssertionError(f"Could not derive exactly one target gene for {bad:,} Stage 4B rows.")

observed_genes = sorted(weak[GENE_COLUMN].unique().tolist())
if observed_genes != ["BRCA1", "BRCA2", "EGFR", "MLH1"]:
    raise AssertionError(f"Unexpected Stage 4B genes: {observed_genes}")

for feature in sorted(set(FULL_FEATURES + NO_STAR_FEATURES)):
    weak[feature] = pd.to_numeric(weak[feature], errors="coerce").astype(float)

for eligibility_column in ["full_training_eligible", "no_star_training_eligible"]:
    weak[eligibility_column] = weak[eligibility_column].fillna(False).astype(bool)

for label_column in ["full_weak_label_binary", "no_star_weak_label_binary"]:
    weak[label_column] = pd.to_numeric(weak[label_column], errors="coerce")
    nonmissing = weak[label_column].dropna().unique()
    if not set(nonmissing).issubset({0, 1, 0.0, 1.0}):
        raise AssertionError(f"Unexpected values in {label_column}: {sorted(nonmissing.tolist())}")

weak_label_accounting_rows = []
for model_key, eligibility_column, label_column in [
    ("full", "full_training_eligible", "full_weak_label_binary"),
    ("no_star", "no_star_training_eligible", "no_star_weak_label_binary"),
]:
    eligible = weak[eligibility_column]
    labels = weak[label_column]
    if labels.loc[eligible].isna().any():
        raise AssertionError(f"Eligible {model_key} weak-label rows contain missing labels.")
    if labels.loc[~eligible].notna().any():
        # A noneligible row may retain an intermediate numeric value in some exports; only the frozen
        # eligibility flag controls training. Record but do not silently use it.
        noneligible_labeled = int(labels.loc[~eligible].notna().sum())
    else:
        noneligible_labeled = 0

    stable = int((labels == 1).sum())
    unstable = int((labels == 0).sum())
    unlabeled = int(labels.isna().sum())
    eligible_count = int(eligible.sum())
    expected = EXPECTED_WEAK_LABEL_ACCOUNTING[model_key]

    if (stable, unstable, unlabeled, eligible_count) != (
        expected["stable"],
        expected["unstable"],
        expected["unlabeled"],
        expected["eligible"],
    ):
        raise AssertionError(
            f"{model_key} weak-label accounting mismatch: "
            f"observed={(stable, unstable, unlabeled, eligible_count)}, "
            f"expected={(expected['stable'], expected['unstable'], expected['unlabeled'], expected['eligible'])}"
        )

    weak_label_accounting_rows.append(
        {
            "model_pathway": model_key,
            "stable_labels": stable,
            "unstable_labels": unstable,
            "unlabeled_rows": unlabeled,
            "training_eligible_rows": eligible_count,
            "noneligible_rows_with_numeric_label": noneligible_labeled,
        }
    )

weak_label_accounting = pd.DataFrame(weak_label_accounting_rows)

frozen_full_artifact = joblib.load(FULL_MODEL_PATH)
frozen_no_star_artifact = joblib.load(NO_STAR_MODEL_PATH)

frozen_full_pipeline, full_pipeline_location = extract_sklearn_pipeline(
    frozen_full_artifact, "full"
)
frozen_no_star_pipeline, no_star_pipeline_location = extract_sklearn_pipeline(
    frozen_no_star_artifact, "no-star"
)

print(
    "Frozen Stage 4C artifact unwrapping  : "
    f"full={type(frozen_full_artifact).__name__}{full_pipeline_location}; "
    f"no-star={type(frozen_no_star_artifact).__name__}{no_star_pipeline_location}"
)

for pipeline_name, pipeline, expected_features in [
    ("full", frozen_full_pipeline, FULL_FEATURES),
    ("no_star", frozen_no_star_pipeline, NO_STAR_FEATURES),
]:
    find_pipeline_component(pipeline, SimpleImputer)
    find_pipeline_component(pipeline, StandardScaler)
    classifier = find_pipeline_component(pipeline, LogisticRegression)
    if getattr(pipeline, "n_features_in_", len(expected_features)) != len(expected_features):
        raise AssertionError(
            f"Frozen {pipeline_name} pipeline expects {getattr(pipeline, 'n_features_in_', None)} "
            f"features; expected {len(expected_features)}."
        )
    if set(np.asarray(classifier.classes_).tolist()) != {0, 1}:
        raise AssertionError(f"Unexpected classes in frozen {pipeline_name} classifier.")

full_classifier = find_pipeline_component(frozen_full_pipeline, LogisticRegression)
no_star_classifier = find_pipeline_component(frozen_no_star_pipeline, LogisticRegression)

frozen_model_settings = {
    "full_solver": full_classifier.solver,
    "full_penalty": full_classifier.penalty,
    "full_C": float(full_classifier.C),
    "full_max_iter": int(full_classifier.max_iter),
    "full_tol": float(full_classifier.tol),
    "full_class_weight": full_classifier.class_weight,
    "full_random_state": full_classifier.random_state,
    "no_star_solver": no_star_classifier.solver,
    "no_star_penalty": no_star_classifier.penalty,
    "no_star_C": float(no_star_classifier.C),
    "no_star_max_iter": int(no_star_classifier.max_iter),
    "no_star_tol": float(no_star_classifier.tol),
    "no_star_class_weight": no_star_classifier.class_weight,
    "no_star_random_state": no_star_classifier.random_state,
}


# --------------------------------------------------------------------------------------------------
# 5. FIT ALL SIX LOGO MODELS USING T0 WEAK LABELS ONLY
#    IMPORTANT: STAGE 6B OUTCOME HAS NOT BEEN OPENED YET.
# --------------------------------------------------------------------------------------------------

training_accounting_rows = []
prediction_frames = []
fitted_models = {}
fit_start = time.time()

for held_out_gene in PRIMARY_GENES:
    training_genes = [gene for gene in PRIMARY_GENES if gene != held_out_gene]
    training_gene_text = "+".join(training_genes)

    split_predictions = weak.loc[
        weak[GENE_COLUMN].eq(held_out_gene),
        [KEY_COLUMN, ROW_ORDER_COLUMN, GENE_COLUMN] + sorted(set(FULL_FEATURES + NO_STAR_FEATURES)),
    ].copy()

    if split_predictions.empty:
        raise AssertionError(f"No Stage 4B rows found for held-out gene {held_out_gene}.")

    for pathway, template_pipeline, features, eligibility_column, label_column, output_column in [
        (
            "full_ges",
            frozen_full_pipeline,
            FULL_FEATURES,
            "full_training_eligible",
            "full_weak_label_binary",
            "logo_full_ges_instability_risk",
        ),
        (
            "no_star_ges",
            frozen_no_star_pipeline,
            NO_STAR_FEATURES,
            "no_star_training_eligible",
            "no_star_weak_label_binary",
            "logo_no_star_ges_instability_risk",
        ),
    ]:
        train_mask = weak[GENE_COLUMN].isin(training_genes) & weak[eligibility_column]
        train = weak.loc[train_mask].copy()
        y_train = train[label_column].astype(int).to_numpy()

        if len(np.unique(y_train)) != 2:
            raise AssertionError(
                f"{held_out_gene}/{pathway} training labels do not contain both classes."
            )

        stable_labels = int((y_train == 1).sum())
        unstable_labels = int((y_train == 0).sum())

        model = clone(template_pipeline)
        model.fit(train[features], y_train)

        fitted_classifier = find_pipeline_component(model, LogisticRegression)
        n_iter = int(np.max(np.asarray(fitted_classifier.n_iter_)))
        converged = n_iter < int(fitted_classifier.max_iter)
        if not converged:
            raise AssertionError(
                f"{held_out_gene}/{pathway} reached max_iter={fitted_classifier.max_iter}."
            )

        p_stable = stable_class_probability(model, split_predictions[features])
        risk = 1.0 - p_stable
        if not np.isfinite(risk).all() or ((risk < 0.0) | (risk > 1.0)).any():
            raise AssertionError(f"Invalid LOGO risk generated for {held_out_gene}/{pathway}.")

        split_predictions[output_column] = risk
        fitted_models[(held_out_gene, pathway)] = model

        training_accounting_rows.append(
            {
                "held_out_gene": held_out_gene,
                "training_genes": training_gene_text,
                "model_pathway": pathway,
                "training_rows": int(len(train)),
                "stable_weak_labels": stable_labels,
                "unstable_weak_labels": unstable_labels,
                "training_prevalence_stable": stable_labels / len(train),
                "features": ";".join(features),
                "solver": fitted_classifier.solver,
                "penalty": fitted_classifier.penalty,
                "C": float(fitted_classifier.C),
                "max_iter": int(fitted_classifier.max_iter),
                "iterations_used": n_iter,
                "converged": converged,
                "t1_outcome_loaded_during_fit": False,
            }
        )

        del train, y_train

    prediction_frames.append(
        split_predictions[
            [
                KEY_COLUMN,
                ROW_ORDER_COLUMN,
                GENE_COLUMN,
                "logo_full_ges_instability_risk",
                "logo_no_star_ges_instability_risk",
            ]
        ].copy()
    )

logo_predictions_all_t0 = pd.concat(prediction_frames, ignore_index=True)
training_accounting = pd.DataFrame(training_accounting_rows)

if logo_predictions_all_t0[KEY_COLUMN].duplicated().any():
    raise AssertionError("LOGO prediction table contains duplicate RCV keys.")
if sorted(logo_predictions_all_t0[GENE_COLUMN].unique().tolist()) != PRIMARY_GENES:
    raise AssertionError("LOGO prediction table does not contain exactly the three primary genes.")

fit_elapsed = time.time() - fit_start


# --------------------------------------------------------------------------------------------------
# 6. ONLY NOW VERIFY AND LOAD THE FROZEN STAGE 6B TEMPORAL OUTCOME
# --------------------------------------------------------------------------------------------------

observed_hashes["stage6b_evaluable"] = verify_hash(
    EVALUABLE_PARQUET,
    EXPECTED_HASHES["stage6b_evaluable"],
    "Stage 6B primary-evaluable cohort",
)
if not EVALUABLE_SIDECAR.exists():
    raise FileNotFoundError(f"Missing Stage 6B SHA-256 sidecar:\n{EVALUABLE_SIDECAR}")
if read_sidecar_hash(EVALUABLE_SIDECAR) != observed_hashes["stage6b_evaluable"]:
    raise AssertionError("Stage 6B evaluable-cohort SHA-256 sidecar does not match the file.")

stage6b_parquet = pq.ParquetFile(EVALUABLE_PARQUET)
if stage6b_parquet.metadata.num_rows != EXPECTED_STAGE6B_ROWS:
    raise AssertionError("Unexpected Stage 6B row count.")
if stage6b_parquet.metadata.num_columns != EXPECTED_STAGE6B_COLUMNS:
    raise AssertionError("Unexpected Stage 6B column count.")

stage6b_columns = stage6b_parquet.schema_arrow.names
required_stage6b_columns = [
    KEY_COLUMN,
    ROW_ORDER_COLUMN,
    GENE_COLUMN,
    OUTCOME_COLUMN,
    "review_stars_instability_risk",
    "combined_metadata_instability_risk",
]
missing_stage6b = [column for column in required_stage6b_columns if column not in stage6b_columns]
if missing_stage6b:
    raise KeyError("Missing required Stage 6B columns:\n" + "\n".join(missing_stage6b))

outcome = pd.read_parquet(EVALUABLE_PARQUET, columns=required_stage6b_columns).copy()
outcome[KEY_COLUMN] = outcome[KEY_COLUMN].astype(str).str.strip().str.upper()
outcome[ROW_ORDER_COLUMN] = pd.to_numeric(outcome[ROW_ORDER_COLUMN], errors="raise").astype(np.int64)
outcome[GENE_COLUMN] = outcome[GENE_COLUMN].map(normalize_gene)
outcome[OUTCOME_COLUMN] = pd.to_numeric(outcome[OUTCOME_COLUMN], errors="raise").astype(int)

if len(outcome) != EXPECTED_STAGE6B_ROWS:
    raise AssertionError("Loaded Stage 6B row count is incorrect.")
if outcome[KEY_COLUMN].nunique(dropna=False) != EXPECTED_STAGE6B_ROWS:
    raise AssertionError("Stage 6B RCV keys are not unique.")
if not set(outcome[OUTCOME_COLUMN].unique()).issubset({0, 1}):
    raise AssertionError("Stage 6B primary outcome is not binary.")

observed_events = int(outcome[OUTCOME_COLUMN].sum())
observed_negatives = int((outcome[OUTCOME_COLUMN] == 0).sum())
if (observed_events, observed_negatives) != (EXPECTED_EVENTS, EXPECTED_NEGATIVES):
    raise AssertionError(
        f"Stage 6B outcome accounting mismatch: {observed_events:,}/{observed_negatives:,}."
    )

for column in ["review_stars_instability_risk", "combined_metadata_instability_risk"]:
    outcome[column] = pd.to_numeric(outcome[column], errors="raise").astype(float)
    values = outcome[column].to_numpy()
    if not np.isfinite(values).all() or ((values < 0.0) | (values > 1.0)).any():
        raise AssertionError(f"Invalid frozen comparator values in {column}.")

primary_outcome = outcome.loc[outcome[GENE_COLUMN].isin(PRIMARY_GENES)].copy()

analysis = primary_outcome.merge(
    logo_predictions_all_t0,
    on=[KEY_COLUMN, ROW_ORDER_COLUMN, GENE_COLUMN],
    how="left",
    validate="one_to_one",
    indicator=True,
)

if not analysis["_merge"].eq("both").all():
    missing = int((analysis["_merge"] != "both").sum())
    raise AssertionError(f"{missing:,} primary-gene outcome rows lack LOGO predictions.")
analysis = analysis.drop(columns=["_merge"])

for column in [
    "logo_full_ges_instability_risk",
    "logo_no_star_ges_instability_risk",
]:
    values = pd.to_numeric(analysis[column], errors="raise").to_numpy(dtype=float)
    if not np.isfinite(values).all() or ((values < 0.0) | (values > 1.0)).any():
        raise AssertionError(f"Invalid merged LOGO score in {column}.")

held_out_accounting_rows = []
for gene in PRIMARY_GENES:
    gene_df = analysis.loc[analysis[GENE_COLUMN].eq(gene)]
    rows = int(len(gene_df))
    events = int(gene_df[OUTCOME_COLUMN].sum())
    negatives = rows - events
    expected = EXPECTED_HELD_OUT_ACCOUNTING[gene]
    if (rows, events, negatives) != (
        expected["rows"],
        expected["events"],
        expected["negatives"],
    ):
        raise AssertionError(
            f"Held-out {gene} accounting mismatch: {(rows, events, negatives)}."
        )
    held_out_accounting_rows.append(
        {
            "held_out_gene": gene,
            "rows": rows,
            "events": events,
            "negatives": negatives,
            "event_prevalence": events / rows,
        }
    )

held_out_accounting = pd.DataFrame(held_out_accounting_rows)


# --------------------------------------------------------------------------------------------------
# 7. HELD-OUT-GENE POINT ESTIMATES AND 2,000-REPLICATE PAIRED BOOTSTRAP
# --------------------------------------------------------------------------------------------------

rng = np.random.default_rng(RANDOM_SEED)
model_interval_rows = []
paired_difference_rows = []
score_summary_rows = []
bootstrap_start = time.time()

for held_out_gene in PRIMARY_GENES:
    gene_start = time.time()
    test = analysis.loc[analysis[GENE_COLUMN].eq(held_out_gene)].reset_index(drop=True)
    y = test[OUTCOME_COLUMN].to_numpy(dtype=np.int8)
    n_rows = int(len(test))
    n_events = int(y.sum())
    n_negatives = n_rows - n_events
    prevalence = n_events / n_rows

    if len(np.unique(y)) != 2:
        raise AssertionError(f"Held-out gene {held_out_gene} does not contain both outcome classes.")

    scores = {
        key: test[spec["score_column"]].to_numpy(dtype=np.float64)
        for key, spec in MODEL_SPECS.items()
    }
    caches = {key: construct_score_group_cache(value, y) for key, value in scores.items()}

    original_counts = np.ones((1, n_rows), dtype=np.int16)
    original_positive_total = np.array([n_events], dtype=np.float64)
    point_metrics = {}

    print(
        f"\nPreparing held-out {held_out_gene}: {n_rows:,} rows, {n_events:,} events, "
        f"{n_negatives:,} negatives"
    )

    for model_key, spec in MODEL_SPECS.items():
        score = scores[model_key]
        sklearn_auprc = float(average_precision_score(y, score))
        sklearn_auroc = float(roc_auc_score(y, score))
        fast_auprc, fast_auroc = calculate_grouped_weighted_metrics(
            caches[model_key],
            original_counts,
            original_positive_total,
        )
        if not np.isclose(fast_auprc[0], sklearn_auprc, rtol=1e-11, atol=1e-12):
            raise AssertionError(f"Fast AUPRC validation failed for {held_out_gene}/{model_key}.")
        if not np.isclose(fast_auroc[0], sklearn_auroc, rtol=1e-11, atol=1e-12):
            raise AssertionError(f"Fast AUROC validation failed for {held_out_gene}/{model_key}.")

        point_metrics[model_key] = {"auprc": sklearn_auprc, "auroc": sklearn_auroc}
        score_summary_rows.append(
            {
                "held_out_gene": held_out_gene,
                "model_key": model_key,
                "model": spec["display_name"],
                "unique_score_values": int(len(np.unique(score))),
                "score_min": float(np.min(score)),
                "score_max": float(np.max(score)),
                "score_mean": float(np.mean(score)),
                "mean_score_events": float(np.mean(score[y == 1])),
                "mean_score_negatives": float(np.mean(score[y == 0])),
                "point_auprc": sklearn_auprc,
                "point_auroc": sklearn_auroc,
                "fast_metric_validation": "PASS",
            }
        )

    print("  Exact tie-aware metric validation against scikit-learn: PASS")

    bootstrap_metrics = {
        model_key: {
            "auprc": np.full(N_BOOTSTRAP, np.nan, dtype=np.float64),
            "auroc": np.full(N_BOOTSTRAP, np.nan, dtype=np.float64),
        }
        for model_key in MODEL_SPECS
    }
    bootstrap_prevalence = np.full(N_BOOTSTRAP, np.nan, dtype=np.float64)

    probabilities = np.full(n_rows, 1.0 / n_rows, dtype=np.float64)
    probabilities[-1] = 1.0 - probabilities[:-1].sum()

    for batch_start in range(0, N_BOOTSTRAP, BOOTSTRAP_BATCH_SIZE):
        batch_end = min(batch_start + BOOTSTRAP_BATCH_SIZE, N_BOOTSTRAP)
        batch_size = batch_end - batch_start
        counts = rng.multinomial(n_rows, probabilities, size=batch_size)
        if not np.all(counts.sum(axis=1) == n_rows):
            raise AssertionError(f"Bootstrap sample-size preservation failed for {held_out_gene}.")

        positive_totals = (counts @ y).astype(np.float64)
        valid = (positive_totals > 0.0) & (positive_totals < n_rows)
        bootstrap_prevalence[batch_start:batch_end] = np.where(
            valid,
            positive_totals / n_rows,
            np.nan,
        )

        for model_key in MODEL_SPECS:
            auprc_values, auroc_values = calculate_grouped_weighted_metrics(
                caches[model_key],
                counts,
                positive_totals,
            )
            bootstrap_metrics[model_key]["auprc"][batch_start:batch_end] = auprc_values
            bootstrap_metrics[model_key]["auroc"][batch_start:batch_end] = auroc_values

        if batch_end % BOOTSTRAP_PROGRESS_INTERVAL == 0 or batch_end == N_BOOTSTRAP:
            valid_so_far = int(
                np.isfinite(bootstrap_metrics["logo_full_ges"]["auprc"][:batch_end]).sum()
            )
            print(
                f"  Completed {batch_end:,}/{N_BOOTSTRAP:,} replicates | "
                f"valid {valid_so_far:,} | elapsed {time.time() - gene_start:.1f}s"
            )

        # NumPy arrays are released immediately by reference counting. Explicit gc.collect()
        # is intentionally avoided here because it can stall Colab on large sklearn/pandas graphs.
        del counts, positive_totals

    valid_mask = np.isfinite(bootstrap_metrics["logo_full_ges"]["auprc"])
    valid_replicates = int(valid_mask.sum())
    invalid_replicates = N_BOOTSTRAP - valid_replicates
    if valid_replicates < MINIMUM_VALID_REPLICATES:
        raise AssertionError(
            f"Held-out {held_out_gene} produced only {valid_replicates:,} valid replicates."
        )

    for model_key in MODEL_SPECS:
        for metric in ["auprc", "auroc"]:
            if not np.array_equal(
                np.isfinite(bootstrap_metrics[model_key][metric]),
                valid_mask,
            ):
                raise AssertionError(
                    f"Paired-validity mismatch for {held_out_gene}/{model_key}/{metric}."
                )

    for model_key, spec in MODEL_SPECS.items():
        auprc_values = bootstrap_metrics[model_key]["auprc"]
        auroc_values = bootstrap_metrics[model_key]["auroc"]
        auprc_low, auprc_high = percentile_interval(auprc_values)
        auroc_low, auroc_high = percentile_interval(auroc_values)
        auprc_null_low, auprc_null_high = percentile_interval(
            auprc_values - bootstrap_prevalence
        )
        auroc_null_low, auroc_null_high = percentile_interval(auroc_values - 0.50)

        model_interval_rows.append(
            {
                "held_out_gene": held_out_gene,
                "training_genes": "+".join(
                    [gene for gene in PRIMARY_GENES if gene != held_out_gene]
                ),
                "model_key": model_key,
                "model": spec["display_name"],
                "rows": n_rows,
                "events": n_events,
                "negatives": n_negatives,
                "held_out_prevalence": prevalence,
                "point_auprc": point_metrics[model_key]["auprc"],
                "auprc_ci_lower": auprc_low,
                "auprc_ci_upper": auprc_high,
                "point_auprc_minus_prevalence": point_metrics[model_key]["auprc"] - prevalence,
                "auprc_minus_prevalence_ci_lower": auprc_null_low,
                "auprc_minus_prevalence_ci_upper": auprc_null_high,
                "auprc_null_status": interval_status(
                    auprc_null_low,
                    auprc_null_high,
                    "supported_above_held_out_prevalence",
                    "supported_below_held_out_prevalence",
                ),
                "point_auroc": point_metrics[model_key]["auroc"],
                "auroc_ci_lower": auroc_low,
                "auroc_ci_upper": auroc_high,
                "point_auroc_minus_0_50": point_metrics[model_key]["auroc"] - 0.50,
                "auroc_minus_0_50_ci_lower": auroc_null_low,
                "auroc_minus_0_50_ci_upper": auroc_null_high,
                "auroc_null_status": interval_status(
                    auroc_null_low,
                    auroc_null_high,
                    "supported_above_0_50",
                    "supported_below_0_50",
                ),
                "attempted_bootstrap_replicates": N_BOOTSTRAP,
                "valid_bootstrap_replicates": valid_replicates,
                "invalid_one_class_replicates": invalid_replicates,
            }
        )

    for comparator_key in PAIRED_COMPARATORS:
        comparator_name = MODEL_SPECS[comparator_key]["display_name"]
        for metric in ["auprc", "auroc"]:
            differences = (
                bootstrap_metrics["logo_full_ges"][metric]
                - bootstrap_metrics[comparator_key][metric]
            )
            finite = differences[np.isfinite(differences)]
            difference_low, difference_high = percentile_interval(finite)
            point_difference = (
                point_metrics["logo_full_ges"][metric]
                - point_metrics[comparator_key][metric]
            )

            paired_difference_rows.append(
                {
                    "metric": metric.upper(),
                    "held_out_gene": held_out_gene,
                    "training_genes": "+".join(
                        [gene for gene in PRIMARY_GENES if gene != held_out_gene]
                    ),
                    "comparison": f"LOGO Full GES minus {comparator_name}",
                    "comparator_key": comparator_key,
                    "rows": n_rows,
                    "events": n_events,
                    "negatives": n_negatives,
                    "point_difference": point_difference,
                    "difference_ci_lower": difference_low,
                    "difference_ci_upper": difference_high,
                    "paired_interval_status": interval_status(
                        difference_low,
                        difference_high,
                        "logo_full_ges_supported_higher",
                        "logo_full_ges_supported_lower",
                    ),
                    "bootstrap_probability_logo_full_greater": float(
                        np.mean(finite > 0.0)
                    ),
                    "bootstrap_probability_equal": float(np.mean(finite == 0.0)),
                    "bootstrap_sign_p_value": bootstrap_sign_pvalue(finite),
                    "attempted_bootstrap_replicates": N_BOOTSTRAP,
                    "valid_bootstrap_replicates": valid_replicates,
                    "invalid_one_class_replicates": invalid_replicates,
                }
            )

    print(
        f"  Held-out {held_out_gene} complete: {valid_replicates:,} valid, "
        f"{invalid_replicates:,} invalid one-class replicates"
    )

    del test, y, scores, caches, bootstrap_metrics, bootstrap_prevalence

bootstrap_elapsed = time.time() - bootstrap_start


# --------------------------------------------------------------------------------------------------
# 8. RESULT TABLES AND HOLM CORRECTION
# --------------------------------------------------------------------------------------------------

logo_model_intervals = pd.DataFrame(model_interval_rows)
logo_paired_differences = pd.DataFrame(paired_difference_rows)
logo_score_summary = pd.DataFrame(score_summary_rows)

expected_model_rows = len(PRIMARY_GENES) * len(MODEL_SPECS)
expected_paired_rows = len(PRIMARY_GENES) * len(PAIRED_COMPARATORS) * 2
if len(logo_model_intervals) != expected_model_rows:
    raise AssertionError(
        f"LOGO model interval table has {len(logo_model_intervals)} rows; "
        f"expected {expected_model_rows}."
    )
if len(logo_paired_differences) != expected_paired_rows:
    raise AssertionError(
        f"LOGO paired table has {len(logo_paired_differences)} rows; "
        f"expected {expected_paired_rows}."
    )

logo_paired_differences["holm_adjusted_bootstrap_sign_p"] = np.nan
for metric in ["AUPRC", "AUROC"]:
    mask = logo_paired_differences["metric"].eq(metric)
    p_values = logo_paired_differences.loc[mask, "bootstrap_sign_p_value"].to_numpy(float)
    if len(p_values) != 9:
        raise AssertionError(f"{metric} LOGO multiplicity family has {len(p_values)} tests; expected 9.")
    logo_paired_differences.loc[mask, "holm_adjusted_bootstrap_sign_p"] = holm_adjust(p_values)

logo_paired_differences["holm_supported_at_0_05"] = (
    logo_paired_differences["holm_adjusted_bootstrap_sign_p"] < 0.05
)
logo_paired_differences["multiplicity_family"] = (
    "three_primary_held_out_genes_x_three_comparators"
)

if logo_paired_differences["holm_adjusted_bootstrap_sign_p"].isna().any():
    raise AssertionError("At least one LOGO Holm-adjusted value is missing.")
if (logo_model_intervals["valid_bootstrap_replicates"] < MINIMUM_VALID_REPLICATES).any():
    raise AssertionError("At least one held-out-gene/model interval has too few valid replicates.")


# --------------------------------------------------------------------------------------------------
# 9. DISPLAY AND FINAL DECISION
# --------------------------------------------------------------------------------------------------

model_display_columns = [
    "held_out_gene",
    "training_genes",
    "model",
    "rows",
    "events",
    "negatives",
    "held_out_prevalence",
    "point_auprc",
    "auprc_ci_lower",
    "auprc_ci_upper",
    "point_auprc_minus_prevalence",
    "auprc_minus_prevalence_ci_lower",
    "auprc_minus_prevalence_ci_upper",
    "auprc_null_status",
    "point_auroc",
    "auroc_ci_lower",
    "auroc_ci_upper",
    "point_auroc_minus_0_50",
    "auroc_minus_0_50_ci_lower",
    "auroc_minus_0_50_ci_upper",
    "auroc_null_status",
    "valid_bootstrap_replicates",
    "invalid_one_class_replicates",
]

paired_display_columns = [
    "metric",
    "held_out_gene",
    "training_genes",
    "comparison",
    "rows",
    "events",
    "negatives",
    "point_difference",
    "difference_ci_lower",
    "difference_ci_upper",
    "paired_interval_status",
    "bootstrap_probability_logo_full_greater",
    "bootstrap_sign_p_value",
    "holm_adjusted_bootstrap_sign_p",
    "holm_supported_at_0_05",
    "multiplicity_family",
    "valid_bootstrap_replicates",
    "invalid_one_class_replicates",
]

separator = "=" * 150
print("\n" + separator)
print("STAGE 6C STEP 3H — CELL 6C-3H0 — LOCKED LEAVE-ONE-GENE-OUT VALIDATION")
print(separator)
print(f"Stage 4B weak-label SHA-256        : PASS ({observed_hashes['stage4b_weak_label_table']})")
print(f"Stage 4B manifest SHA-256          : PASS ({observed_hashes['stage4b_manifest']})")
print(f"Stage 4C full-model SHA-256        : PASS ({observed_hashes['stage4c_full_model']})")
print(f"Stage 4C no-star-model SHA-256     : PASS ({observed_hashes['stage4c_no_star_model']})")
print(f"Stage 4C manifest SHA-256          : PASS ({observed_hashes['stage4c_manifest']})")
print(f"Stage 6B evaluable SHA-256         : PASS ({observed_hashes['stage6b_evaluable']})")
print(f"Stage 4B dimensions                : PASS ({EXPECTED_STAGE4B_ROWS:,} × {EXPECTED_STAGE4B_COLUMNS})")
print(f"Stage 6B dimensions                : PASS ({EXPECTED_STAGE6B_ROWS:,} × {EXPECTED_STAGE6B_COLUMNS})")
print(f"Stage 6B events / negatives        : PASS ({observed_events:,} / {observed_negatives:,})")
print("Temporal outcome loaded during fit: No")
print("Held-out model tuning              : None")
print("EGFR pooled into primary LOGO      : No")
print(f"Bootstrap attempts per held-out gene: {N_BOOTSTRAP:,}")
print(f"Random seed                        : {RANDOM_SEED}")
print("Holm families                      : 9 AUPRC + 9 AUROC paired comparisons")
print(f"Model fitting elapsed              : {fit_elapsed:.1f}s")
print(f"Bootstrap elapsed                  : {bootstrap_elapsed:.1f}s")

print("\nFROZEN PIPELINE SETTINGS RECOVERED FROM CHECKSUM-VERIFIED JOBLIBS")
for key, value in frozen_model_settings.items():
    print(f"{key:34s}: {value}")

print("\nGLOBAL WEAK-LABEL ACCOUNTING")
print(weak_label_accounting.to_string(index=False))

print("\nLOGO TRAINING ACCOUNTING")
print(training_accounting.to_string(index=False))

print("\nHELD-OUT TEST ACCOUNTING")
print(held_out_accounting.to_string(index=False))

print("\nHELD-OUT SCORE AND POINT-ESTIMATE SUMMARY")
print(logo_score_summary.to_string(index=False))

print("\nHELD-OUT MODEL-SPECIFIC INTERVALS")
print(logo_model_intervals[model_display_columns].to_string(index=False))

print("\nPAIRED LOGO FULL-GES-MINUS-COMPARATOR INFERENCE")
print(logo_paired_differences[paired_display_columns].to_string(index=False))

print("\nSCIENTIFIC INTERPRETATION BOUNDARY")
print("-" * 150)
print(
    "This is the prespecified cross-gene generalization test. Each GES model was refitted only on T0 weak-label "
    "records from two primary genes and evaluated on the third gene using the locked T0-to-T1 temporal outcome."
)
print(
    "A positive held-out result requires attention to absolute discrimination, confidence intervals, and performance "
    "against both review stars and the strong combined-metadata baseline; statistical separation alone does not imply "
    "clinical utility or calibration."
)
print(
    "EGFR remains separate and exploratory. No downstream RAG benefit, patient outcome, treatment safety, or clinical "
    "deployment claim is evaluated in this cell."
)

print("\nCELL DECISION")
print("-" * 150)
print("PASS_STAGE6C_LEAVE_ONE_GENE_OUT_PAIRED_BOOTSTRAP_VALIDATION_COMPLETE")
print(
    "All three prespecified train-two/test-one primary-gene splits, 2,000 paired bootstrap attempts per split, "
    "and separate 9-test Holm corrections for AUPRC and AUROC are complete."
)
print(
    "No frozen score, weak label, outcome, feature definition, threshold, gene assignment, cohort membership, "
    "or scientific artifact was modified. No artifact was written."
)

Frozen Stage 4C artifact unwrapping  : full=dict['pipeline']; no-star=dict['pipeline']

Preparing held-out BRCA1: 21,594 rows, 2,023 events, 19,571 negatives
  Exact tie-aware metric validation against scikit-learn: PASS
  Completed 50/2,000 replicates | valid 50 | elapsed 0.2s
  Completed 100/2,000 replicates | valid 100 | elapsed 0.4s
  Completed 150/2,000 replicates | valid 150 | elapsed 0.5s
  Completed 200/2,000 replicates | valid 200 | elapsed 0.7s
  Completed 250/2,000 replicates | valid 250 | elapsed 0.8s
  Completed 300/2,000 replicates | valid 300 | elapsed 1.0s
  Completed 350/2,000 replicates | valid 350 | elapsed 1.1s
  Completed 400/2,000 replicates | valid 400 | elapsed 1.3s
  Completed 450/2,000 replicates | valid 450 | elapsed 1.4s
  Completed 500/2,000 replicates | valid 500 | elapsed 1.6s
  Completed 550/2,000 replicates | valid 550 | elapsed 1.7s
  Completed 600/2,000 replicates | valid 600 | elapsed 1.9s
  Completed 650/2,000 replicates | valid 650 | elapsed 2.1s
 



1.   List item
2.   List item

